In [ ]:
#@title Setup/Initialization
# ---
# ABCD Study -- Biopsychosocial Predictive Modelling Pipeline
# Adolescent Brain Cognitive Development Study, Release 5.1
# Combined N ~23,760 (adolescents age 9-14 + parents, mean age ~40)
# ---
# Architecture : Stacking ensemble (CatBoost, XGBoost, Random Forest,
#                TabPFN) with linear / logistic meta-learner
#
# Targets : Multiple depression operationalizations (dimensional,
# RCI-based, percentile thresholds), anxiety, ADHD,
# externalizing, social problems, cognitive/educational
# outcomes across five annual assessment waves (T0-T4).
# Parcellation : ~1,078 biopsychosocial variables organized into
# child and parent ontological categories for comparison (Fig 1)
# and across-domain analyses with circular-feature exclusions.
# ---

!pip install -q shap smogn imbalanced-learn catboost mapie optuna optuna-integration tabpfn xgboost pygam networkx statsmodels

# Core data manipulation and modelling
import pandas as pd
import numpy as np

# IterativeImputer is experimental; this enable import must precede the imputer import.
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer, KNNImputer

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.utils import resample

import matplotlib.pyplot as plt
import seaborn as sns

# ---
# REPORT STATE INITIALIZATION & PIPELINE STUBS
# ---

# ---
# Lightweight report state container (_RS) used by later cells
# to track which section is active and collect figures/metrics for
# downstream reporting cells.
# ---

from pathlib import Path
CACHE_ROOT = Path('.')  # base directory for reports/figures

_RS = {
    'section':    'init',
    'run_key':    None,
    'figures':    [],
    'metrics':    {},
    'tables':     [],
    'fig_seq':    {},
    'data_info':  {},
    '_patched':   False,
}

# Pipeline stubs referenced by validation cells.

def make_run_key(extra=None):
    import time
    return f"run_{int(time.time())}"

def cache_load(tag, key=None):
    return None

def fold_load_all(key, tag):
    return {}

def cache_save(data, cell_tag=None, key=None, label=None):
    pass

def report_save_state(key=None):
    pass

def _pipeline_cleanup(label='', verbose=False):
    pass

def report_init(key=None):
    _RS['run_key'] = key
    _RS['section'] = 'init'

def report_load_state(key=None):
    return False

def report_log_data_info(X_train=None, X_test=None, y_train=None, y_test=None):
    try:
        _RS['data_info'] = {
            'n_train': len(X_train) if X_train is not None else 0,
            'n_test': len(X_test) if X_test is not None else 0,
            'n_features': X_train.shape[1] if X_train is not None and hasattr(X_train, 'shape') else 0,
        }
    except Exception:
        pass

def report_log_table(df, title='', section=''):
    try:
        _RS['tables'].append({'title': title, 'df': df, 'section': section})
    except Exception:
        pass



In [ ]:
#@title Variable Registry: Within & Across Categories (Children, Parents, All TPs)

# Defines ~1,078 predictor variables across 5 timepoints (T0-T4)
# contains both within and across category feature-sets used in adolescents and parents
# for all regression and classification target outcomes.
# Use the Main Analysis Runner section to choose within- or across-category feature sets and specific targets
# T1 and T3 were not used in Parents given insufficient data collection
# ---
'''
Variable Registry -- Predictor & Outcome Definitions (ABCD 5.1, T0-T4)
Ontological parcellation of ~1,078 biopsychosocial variables into
theoretically motivated or pragmatically contrastive categories
for hierarchical ML comparison.

Contents
--------
  binary_or_categorical_outcome_variables
      Classification targets -- RCI-based onset/remission (threshold 2.3),
      CBCL percentile cutoffs (top 5th/10th), KSADS diagnosis, latent classes.
  continuous_outcome_variables
      Regression targets -- CBCL syndrome subscales, SBT depression dimensions
      (core, guilt/hopelessness, somatic, etc.), cognitive tasks, grades.
  child_variables_for_each_time_point
      {timepoint -> {category -> [variables]}} for within-category ML.
      27 categories at T0 for children (Fig 1A); 37 unique across all five waves.
  parent_variables_for_each_time_point
      21 categories at T0 for parent-target analyses (Fig 1B); 32 unique across waves.
  t0_variables … t4_variables
      Flat variable lists for across-category analyses at each wave.
  CIRCULAR_EXCLUSIONS  (see circularity exclusions cell)
      Per-target exclusion sets preventing predictor-outcome content overlap.
      circularity between CBCL composites
      and their constituent items (see supplement for full audit).

Usage
  Run this cell once before any modelling cell. The active variable
  structure and exclusion set are resolved at runtime based on
  target_options; no manual editing is required when switching targets.
'''

binary_or_categorical_outcome_variables = [
    "dep_onset_rci_1.96", "dep_onset_rci_2.3",
    "dep_remission_rci_1.96", "dep_remission_rci_2.3",
    "dep_increase_2sd", "dep_increase_1.5sd",
    "dep_decrease_2sd", "dep_decrease_1.5sd",
    "top_10_depression", "top_5_depression",
    "latent_class_depression",
    "top_10_sbt_core_depression", "top_5_sbt_core_depression",
    "sbt_core_dep_onset_rci_2.3", "sbt_core_dep_decrease_2sd",
    "sbt_core_dep_onset_rci_1.96", "sbt_core_dep_remission_rci_1.96",
    "sbt_core_dep_remission_rci_2.3",
    "sbt_core_dep_decrease_1.5sd", "sbt_core_dep_increase_2sd",
    "sbt_core_dep_increase_1.5sd",
    "MDD_KSADS_C", "depressed_mood_B_k", "anhedonia_B_k",
    "wish_dead_present_B_k", "parent_wish_dead_present_B_p",
    "depadhd_c", "asdadhd_c",
    "parent_dep_onset_rci_2.3",
    "top_10_depression_parent", "top_5_depression_parent",
    # -- Child KSADS classification (added for Formal DepAnx NeuroCog batch) --
    "two_more_depression_B_k", "GAD_present",
    "excessive_worry_B_k", "social_anxiety_disorder_B_k",
]

continuous_outcome_variables = [
    "depress_D_p", "depress_D_p_rev", "depress_update_p",
    "internal_D_p", "external_D_p", "adhd_D_p", "anxdisord_D_p",
    "social_problems_D_p", "thought_disorder_D_p",
    "sbt_core_depression", "sbt_core_depression_raw",
    "sbt_guilt_hopelessness", "sbt_social_withdrawal",
    "sbt_fatigue_sleep", "sbt_somatic_depression",
    "sbt_anxiety_depression", "sbt_emotional_dysregulation",
    "sbt_avoidance_fear", "sbt_aggression_irritability",
    "sbt_academic_cognitive", "sbt_perfectionism_achievement", "sbt_well_being",
    "delta_depress_D_p", "delta_sbt_core_depression",
    "sbt_core_depression_delta_t0_t3",
    "parent_depress_D_p", "parent_anxdisord_D_p", "parent_adhd_D_p",
    "parent_depressed_p", "parent_happy_person_p",
    "parent_worries_a_lot_p", "parent_concentration_trouble_p",
    "parent_bad_relationships_p", "parent_not_liked_by_others_p",
    "parent_friendship_trouble_p", "parent_bad_family_relationship_p",
    "parent_suicidal_thoughts_p",
    "delta_parent_depressed_p", "delta_parent_worries_a_lot_p",
    "tb_flanker", "tb_reading", "tb_fluid", "tb_cryst",
    "bad_grades", "pa_sum_k",
    "not_liked_p", "doesnt_get_along_p", "enjoys_little_p", "suicidal_p",
    "parent_income", "area_deprivation_idx",
    "asd_ssrs_sum", "gd_riskybets", "bdefs_distract_upset_p",
    "compulsions_p", "impulsive_p",
    "parent_impulsive_p", "parent_tobacco_use_frequency_p",
    "parent_drinks_frequency_p", "parent_drunk_days_p",
]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def save_plot_2(coefs, name, metric, metric_score, tp):
    plt.figure(figsize=(8, 6))
    sns.barplot(x=coefs.values, y=coefs.index, palette="gray", hue=coefs.index, legend=False)
    plt.title(f"{name} ({metric} = {metric_score:.4f}, t = {tp})")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.yticks(fontsize=7)
    filename = f"dep_{name}.png"
    plt.savefig(filename, dpi=300)
    plt.close()

# ---
# WITHIN-CATEGORY VARIABLES {timepoint -> {category -> [variables]}}
# ---
# Two parallel dicts: child_variables_for_each_time_point (child targets)
# and parent_variables_for_each_time_point (parent targets). The pipeline
# selects the correct dict via is_parent_target() at runtime and drops
# categories in within_exclude for the active target.
#
# Categories span biological (neuroimaging PCs, genetic ancestry, cognitive
# tasks, physical/activity), psychological (personality, psychopathology,
# emotional regulation, cognitive style), social (relationship quality,
# school dynamics, family dynamics), and environmental (SES, residential,
# adverse life events) domains -- following the biopsychosocial gradient
# used in Figures 1 and 4c.
#
# Dual-naming in within_exclude (e.g. {'Anxiety', 'Child Anxiety'})
# ensures the same exclusion set resolves against both dict variants.
# ---

# -- Child target structure (used when target is a child outcome) ---
child_variables_for_each_time_point = {
    0: {  # T0 Variables
        'Cognitive Task Parameters': [
            # NIH Toolbox
            'tb_picvocab', 'tb_flanker', 'tb_list', 'tb_cardsort', 'tb_pattern', 'tb_picture',
            'tb_reading', 'tb_fluid', 'tb_cryst',
            # RAVLT
            'ravlt_s_total', 'ravlt_s_repitition', 'ravlt_s_intrusions',
            'ravlt_l_total', 'ravlt_l_repitition', 'ravlt_l_intrusions',
            # Matrix Reasoning
            'mr_total', 'mr_matrix', 'mr_serial',
            # Cash Choice Task
            'cct',
            # Little Man Task
            'lmt_accuracy', 'lmt_efficiency', 'lmt_mrt', 'lmt_incorrect_mrt',
            # N-Back
            'nb_correct_nt', 'nb_correct_mrt', 'nb_correct_nt_2back', 'nb_correct_mrt_2back',
            'nb_correct_nt_pos', 'nb_correct_mrt_pos', 'nb_correct_nt_neutral', 'nb_correct_mrt_neutral',
            'nb_correct_nt_neg', 'nb_correct_mrt_neg',
            'nb2_accuracy_pos', 'nb2_resp_bias_pos', 'nb2_D_prime_pos',
            'nb2_accuracy_neg', 'nb2_resp_bias_neg', 'nb2_D_prime_neg',
            # SST behavioral
            'sst_ssrt_mean_est', 'sst_ssrt_int_est', 'sst_acceptable_performance',
            # MID
            'mid_acceptable_performance', 'mid_total_payout', 'mid_mrt_smrw', 'mid_mrt_lgrw',
            # SST PRAD model parameters
            'sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS',
            'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
            'sst_kappa0', 'sst_xM', 'sst_bG', 'sst_pp', 'sst_factor1',
            'sst_median_ssrt', 'sst_median_PDR', 'sst_median_SD', 'sst_median_SDr',
            'sst_median_PDRg', 'sst_median_betaS', 'sst_median_bS2',
            'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV',
            'sst_sdSD', 'sst_sdCV',
            'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV',
            'sst_pdrgSD', 'sst_pdrgCV',
            'sst_absdeltaRV', 'sst_absdeltaSD', 'sst_absdeltaCV',
            'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV',
        ],

        'Other Personality Features': [
            'loquacious_p', 'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p',
            'easily_embarrassed_p', 'secretive_p', 'perfectionist_p', 'easily_offended_p',
            'blames_others_p', 'sociable_p', 'school_excitement_p', 'not_critical_others_p',
            'disagreeable_p', 'goal_continuity_p', 'strange_ideas_p', 'stubborn_p',
            'up_negative_urgency_ss_k', 'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k',
            'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k', 'bis_behav_inhibition_ss_k',
            'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k',
            'sex_orient_y'],
        'Medical/Somatic Problems': ['body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p',
            'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p',
            'wets_bed_p'],
        'Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],
        'Physical Activity/Features': ['height', 'weight', 'waist', 'saliva_DHEA', 'saliva_testosterone', 'puberty_k',
            'sex', 'birth_weight_p'],
        'Technology Use': ['screentime_weekday_ss_k', 'screentime_weekend_ss_k'],
        'Social Relationship Quality': ['not_liked_p', 'doesnt_get_along_p', 'prosocial_ss_p', 'close_boy_friends_k',
            'close_girl_friends_k', 'prosocial_ss_k'],
        'Family Dynamics & Parenting': ['fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k',
            'fam_no_lose_temps_k', 'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k',
            'fam_try_one_up_k', 'fam_no_raise_voices_k', 'family_peaceful_p', 'family_lose_temper_rare_p',
            'family_believe_not_raise_voice_p', 'frequent_family_conflict_p', 'family_conflict_ss_p',
            'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
            'religious_service_frequency', 'relig_importance', ],
        'Parent Anxiety and Fear-Related Issues': [
            'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
            'parent_worries_about_future_p', 'parent_worries_about_family_p', 'parent_worries_a_lot_p',
            'parent_relationship_concerns_p',
            'parent_anxdisord_D_p', 'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p'
        ],
        'Parent Mood Issues': [
            'parent_cries_a_lot_p', 'parent_lonely_p', 'parent_feels_unloved_p', 'parent_paranoid_p',
            'parent_feels_inferior_p', 'parent_depressed_p', 'parent_feels_unsuccessful_p',
            'parent_tired_no_reason_p', 'parent_low_energy_p', 'parent_sleep_trouble_p',
            'parent_enjoys_little_p', 'parent_sudden_mood_changes_p', 'parent_suicidal_thoughts_p', 'parent_happy_person_p'
        ],
        'Parent Impulsivity and Behavior Regulation': [
            'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
            'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
            'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
            'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
            'parent_illegal_behavior_p', 'parent_self_harm_p', 'parent_doesnt_eat_well_p'
        ],
        'Parent Interpersonal Relationships and Social Functioning': [
            'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
            'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
            'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
            'parent_teases_others_p', 'parent_stands_up_rights_p'
        ],
        'Parent Cognitive and Attention Issues': [
            'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
            'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p',
            'parent_repeats_acts_p', 'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p',
            'parent_decision_trouble_p', 'parent_priority_trouble_p',
            'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p'
        ],
        'Parent Personality': [
            'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p',
            'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p',
            'parent_louder_than_others_p', 'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p',
            'parent_easily_bored_p', 'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p',
            'parent_prefers_to_be_alone_p', 'parent_no_guilt_p', 'parent_sense_of_fairness_p',
            'parent_high_sleep_duration_p',
            'parent_opposite_sex_wish_p'
        ],
        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'couldnt_afford_phone',
            'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off', 'parent_work_absences_p', 'parent_financial_trouble_p', 'parent_fails_to_pay_debts_p'],
        'Residential Characteristics': ['neighborhood_safety_ss_p', 'area_deprivation_idx', 'neighborhood_safe_y',
            'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],
        'School Dynamics': ['getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k', 'enjoys_school_k',
            'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k', 'school_disengagement_ss_k',
            'repeated_grade', 'grades_dropped', 'school_detension_suspension', 'finds_schoolboring_k'],
        'Task and Resting State': [
            *[f'pc_SSTmri{i}' for i in range(1, 76)],
            *[f'pc_nbackmri{i}' for i in range(1, 76)],
            *[f'pc_midfmri{i}' for i in range(1, 76)],
            *[f'pc_rsFMRI{i}' for i in range(1, 76)],
        ],
        'Structural and DTI': [
            *[f'pc_struct{i}' for i in range(1, 76)],
            *[f'pc_DTI{i}' for i in range(1, 76)],
        ],
        'Anxiety': ['social_fear_present_PK', 'worries_p', 'clings_to_adults_p', 'nervous_general_p',
            'nervous_twitching_p', 'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p'],
        'ADHD': ['cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p'],
        'Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p'],
        'Family Psychopathology Aggregated': ['parent_internal_D_p', 'parent_anhedonia_B_p', 'obsessions_present_B_p', 'poor_eye_contact_B_p','nightmares_B_p',
            'parent_elevated_mood_B_p', 'parent_excessive_worry_B_p', 'parent_lying_B_p', 'parent_social_anxiety_disorder_B_p',
            'parent_sleep_problem_B_p', 'parent_bulimia_B_p', 'parent_attention_D_p',
            'parent_aggressive_D_p', 'parent_external_D_p',
            'parent_anxdisord_D_p', 'parent_antisocial_D_p', 'parent_hyperactive_D_p',
            'parent_somatic_problems_D_p', 'parent_intrusive_thoughts_D_p', 'parent_avoidant_person_D_p', 'parent_personal_strength_D_p',
            'd_grandfather_dep', 'd_grandmother_dep', 'm_grandfather_dep', 'm_grandmother_dep', 'father_mania', 'mother_mania',
            'father_trouble', 'parent_hospitalized_emo', 'parent_therapy_emo'],
        'Family Drug Use': [
            'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
            'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse',
            'cigs_during_pregnancy_p', 'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p',
            'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
            'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p',
            'weed_before_pregnancy_p', 'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p',
            'drugs_before_pregnancy_p', 'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p',
            'caffeine_during_pregnancy_p', 'parent_tobacco_use_frequency_p', 'parent_drug_use_p', 'parent_drinks_too_much_p',
            'parent_drinks_frequency_p', 'parent_drunk_days_p', 'parent_drug_days_nonmedical_p',
            'parent_alcohol_excess_p', 'parent_alcohol_freq_p', 'parent_alcohol_days_drunk_p', 'parent_drug_days_p', 'parent_tobacco_use_p'
        ],
        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            *[f'pc_gene_aces{i}' for i in range(1, 33)]],
        'Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p', 'car_accident_hurt_p',
            'big_accident_need_treatment_p', 'fire_victim_p', 'natural_disaster_victim_p', 'terrorism_victim_p',
            'war_death_witness_p', 'stabbing_shooting_witness_p', 'stabbing_shooting_victim_community_p',
            'stabbing_shooting_victim_home_p', 'beating_victim_home_p', 'stranger_threatened_child_victim_p',
            'family_threatened_child_victim_p', 'adult_family_fighting_victim_p', 'domestic_child_sexually_abuse_victim_p',
            'foreign_child_sexually_abuse_victim_p', 'peer_child_sexually_abuse_victim_p', 'sudden_death_in_family_p'],
        'Religion': [
              'child_religion', 'religious_service_frequency', 'relig_importance'],
    },

1: {  # T1 Variables
        'Cognitive Task Parameters': [
            # Stroop
            'str_accuracy', 'str_accuracy_ic', 'str_accuracy_c',
            'str_mrt', 'str_mrt_ic', 'str_mrt_c',
            'str_stroop_mrt', 'str_stroop_acc',
            # Delay Discounting
            'ddt_rew_amnt', 'ddt_mrt', 'ddt_mrt_delayed', 'ddt_mrt_immediate',
        ],
        'Cognitive Style': ['problem_solving_ss_k', 'strange_ideas_p', 'avoids_eyecontact_p',
            'bad_conversational_flow_p', 'narrow_interests_p', 'sensory_sensitivity_p', 'concentration_on_parts_p',
            'face_understanding'],
        'Delta Other Psychopathology': ["delta_adhd_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
            "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
            "delta_somatic_problems_D_p",
            "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p"],
        'Other Personality Features': ['loquacious_p', 'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p',
            'easily_embarrassed_p', 'secretive_p', 'perfectionist_p', 'easily_offended_p',
            'blames_others_p', 'sociable_p', 'school_excitement_p', 'not_critical_others_p', 'stubborn_p',
            'disagreeable_p', 'goal_continuity_p', 'strange_ideas_p','mania_7up_ss_k', 'sex_orient_y'],
        'Medical/Somatic Problems': ['medhx_p', 'medhx_doctorvisit_p', 'medhx_asthma_p', 'medhx_allergies_p',
            'medhx_brain_p', 'medhx_diabetes_p', 'medhx_epilepsy_p', 'medhx_heart_p', 'medhx_headaches_p',
            'medhx_emergencyroom_p', 'medhx_brokenbones_p', 'medhx_seriousinjury_p', 'seriously_sick_lastyear_k',
            'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p', 'skin_problems_p',
            'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p', 'wets_bed_p'],
        'Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],
        'Physical Activity/Features': ['height', 'weight', 'waist', 'saliva_DHEA', 'saliva_testosterone', 'puberty_k',
            'sex', 'no_sports_activities_p', 'birth_weight_p'],
        'Technology Use': ['screentime_weekday_ss_k', 'screentime_weekend_ss_k'],
        'Social Relationship Quality': ['not_liked_p', 'doesnt_get_along_p', 'prosocial_ss_p',
            'difficulty_making_friends_p', 'regarded_weird_p', 'discrimination_ss_k', 'feels_discriminated_k',
            'senses_racism_k', 'doesnt_feel_accepted_k', 'prosocial_ss_k', 'feels_discriminated_teachers_k',
            'feels_discriminated_adults_not_school_k', 'feels_discriminated_students_k',
            'feels_unwanted_american_society_k', 'feels_discriminated_americans_k'],
        'Family Dynamics & Parenting': ['fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k',
            'fam_no_lose_temps_k', 'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k',
            'fam_try_one_up_k', 'fam_no_raise_voices_k', 'family_peaceful_p', 'family_lose_temper_rare_p',
            'family_believe_not_raise_voice_p', 'frequent_family_conflict_p', 'family_conflict_ss_p',
            'parents_argue_more_p', 'family_emotionprob_p', 'parents_divorced_p', 'death_in_family_p',
            'family_move_p', 'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'marital_status', 'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
            'religious_service_frequency', 'relig_importance'],
        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'positive_finance_p',
            'couldnt_afford_phone', 'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off'
        ],
        'Residential Characteristics': ['neighborhood_safety_ss_p', 'neighborhood_safe_y',  'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],
        'School Dynamics': ['disobeys_at_school_k', 'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k',
            'enjoys_school_k', 'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k',
            'school_disengagement_ss_k', 'repeated_grade', 'grades_dropped', 'school_detension_suspension',
            'child_newschool_p', 'finds_schoolboring_k'],
        'Anxiety': ['worries_p', 'clings_to_adults_p', 'nervous_general_p', 'nervous_twitching_p',
            'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p', 'self_conscious_k',
            'anxious_fearful_k'],
        'ADHD': ['trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
            'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p'],
        'Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p', 'acts_immature_k', 'destroys_others_things_k',
            'disobeys_parents_k', 'stubborn_k', 'hot_temper_k'],
        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            *[f'pc_gene_aces{i}' for i in range(1, 33)]],
        'Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'experienced_crime_p', 'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p'],
        'Religion': [
              'child_religion', 'religious_service_frequency', 'relig_importance'],
    },

2: {  # T2 Variables
        'Cognitive Task Parameters': [
            # NIH Toolbox
            'tb_picvocab', 'tb_flanker', 'tb_pattern', 'tb_picture', 'tb_reading', 'tb_cryst',
            # Game of Dice
            'gd_safebets', 'gd_riskybets',
            # RAVLT
            'ravlt_s_total', 'ravlt_s_repitition', 'ravlt_s_intrusions',
            'ravlt_l_total', 'ravlt_l_repitition', 'ravlt_l_intrusions',
            # N-Back
            'nb_correct_nt', 'nb_correct_mrt', 'nb_correct_nt_2back', 'nb_correct_mrt_2back',
            'nb_correct_nt_pos', 'nb_correct_mrt_pos', 'nb_correct_nt_neutral', 'nb_correct_mrt_neutral',
            'nb_correct_nt_neg', 'nb_correct_mrt_neg',
            'nb2_accuracy_pos', 'nb2_resp_bias_pos', 'nb2_D_prime_pos',
            'nb2_accuracy_neg', 'nb2_resp_bias_neg', 'nb2_D_prime_neg',
            # SST behavioral
            'sst_ssrt_mean_est', 'sst_ssrt_int_est', 'sst_acceptable_performance',
            # MID
            'mid_mrt_smrw', 'mid_mrt_lgrw', 'mid_total_payout',
            'mid_acceptable_performance', 'mid_num_trials',
            # Little Man Task
            'lmt_accuracy', 'lmt_mrt', 'lmt_efficiency',
            # SST BEESTS model parameters (no factors at T2)
            'sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS',
            'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
            'sst_kappa0', 'sst_xM', 'sst_bG', 'sst_pp',
            # SST BEESTS medians
            'sst_median_ssrt', 'sst_median_PDR', 'sst_median_SD', 'sst_median_SDr',
            'sst_median_PDRg', 'sst_median_betaS', 'sst_median_bS2',
            # SST BEESTS variability
            'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV',
            'sst_sdSD', 'sst_sdCV',
            'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV',
            'sst_pdrgSD', 'sst_pdrgCV',
            'sst_absdeltaRV', 'sst_absdeltaSD', 'sst_absdeltaCV',
            'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV',
        ],
        'Dynamic Cognitive Control Parameters': ['sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS', 'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
            'sst_kappa0', 'sst_xM', 'sst_sM', 'sst_bG', 'sst_pp', 'sst_factor1', 'sst_factor2', 'sst_factor3', 'sst_mean_ssrt',
            'sst_median_ssrt', 'sst_mean_PDR', 'sst_median_PDR', 'sst_mean_SD', 'sst_median_SD', 'sst_mean_SDr', 'sst_median_SDr',
            'sst_mean_PDRg', 'sst_median_PDRg', 'sst_mean_betaS', 'sst_median_betaS', 'sst_mean_bS2', 'sst_median_bS2',
            'sst_mean_absdelta', 'sst_median_absdelta', 'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV', 'sst_sdRV', 'sst_sdSD', 'sst_sdCV',
            'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV', 'sst_pdrgV', 'sst_pdrgSD', 'sst_pdrgCV', 'sst_betasV', 'sst_betasSD',
            'sst_betasCV', 'sst_absdeltaRV', 'sst_absdeltaSD', 'sst_absdeltaCV', 'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV'],
        'Delta Other Psychopathology': ["delta_adhd_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
            "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
            "delta_somatic_problems_D_p",
            "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p",
            "delta_anxdisord_D_p"],
        'Other Personality Features': ['loquacious_p', 'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p',
            'easily_embarrassed_p', 'secretive_p', 'perfectionist_p', 'easily_offended_p',
            'blames_others_p', 'sociable_p', 'school_excitement_p', 'not_critical_others_p', 'stubborn_p',
            'disagreeable_p', 'goal_continuity_p', 'strange_ideas_p', 'up_negative_urgency_ss_k',
            'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k', 'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k',
            'bis_behav_inhibition_ss_k', 'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k',
            'sex_orient_y'],
        'Medical/Somatic Problems': ['medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p', 'pain_last_month_k',
            'seriously_sick_lastyear_k', 'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p',
            'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p',
            'wets_bed_p'],
        'Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'fallsleeptime',
            'wakeuptime', 'wakesleepcalc', 'chronotype', 'nightmares_p',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],
        'Diet/Nutrition': ['fruit_intake', 'vegetable_intake', 'protein_sources_intake', 'legume_intake',
            'added_sugar', 'sugary_beverage_freq', 'dairy_intake', 'whole_grain_intake', 'total_calories',
            'protein_intake', 'carbohydrate_intake', 'fiber_intake', 'sodium_intake', 'potassium_intake',
            'total_sugar', 'saturated_fat', 'bad_diet_p'],
        'Physical Activity/Features': ['height', 'weight', 'waist', 'puberty_k', 'sex', 'no_sports_activities_p',
            'birth_weight_p', 'fitbit_resting_hr', 'fitbit_steps', 'fitbit_sedentary_mins', 'fitbit_lightlyactive_mins', 'fitbit_fairlyactive_mins', 'fitbit_veryactive_mins'],
        'Technology Use': ['socialmedia_daysperweek_k', 'videogames_daysperweek_k', 'bullied_on_internet_k', 'vgame_thinking'],
        'Social Relationship Quality': [
            'not_liked_p', 'doesnt_get_along_p', 'prosocial_ss_p', 'close_boy_friends_k', 'close_girl_friends_k',
            'peer_net_protective_ss_k', 'peers_beh_prosocial_ss_k', 'peers_beh_delinquent_ss_k', 'feels_leftout_k', 'not_invited_k',
            'excluded_k', 'otherkids_spreadneg_rumors_k', 'otherkids_gossip_k', 'feels_threatned_k', 'saysmeanthings_others_k',
            'otherkids_saymeanthings_k', 'discrimination_ss_k', 'feels_discriminated_k', 'senses_racism_k', 'doesnt_feel_accepted_k',
            'bullied_on_internet_k', 'prosocial_ss_k', 'socialinfluence_meanfinal_k', 'relational_victimization_ss_k', 'reputational_aggression_ss_k',
            'reputational_victimization_ss_k', 'overt_aggression_ss_k', 'overt_victimization_ss_k', 'relational_aggression_ss_k',
            'feels_discriminated_teachers_k', 'feels_discriminated_adults_not_school_k', 'feels_discriminated_students_k',
            'feels_unwanted_american_society_k', 'feels_discriminated_americans_k'
        ],
        'Family Dynamics & Parenting': ['p_comm_cohesion_ss', 'p_comm_ctrl_ss', 'p_comm_collective_efficacy_ss',
            'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k', 'fam_no_lose_temps_k',
            'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k', 'fam_try_one_up_k',
            'fam_no_raise_voices_k', 'family_not_talk_aboutfeelings_p', 'family_peaceful_p',
            'family_open_discussing_anything_p', 'family_lose_temper_rare_p', 'family_believe_not_raise_voice_p',
            'frequent_family_conflict_p', 'family_conflict_ss_p', 'family_expression_ss_p', 'family_intellectual_ss_p',
            'family_activities_ss_p', 'family_organisation_ss_p', 'parents_argue_more_p', 'family_emotionprob_p',
            'parents_divorced_p', 'death_in_family_p', 'family_move_p', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'marital_status', 'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
            'religious_service_frequency', 'relig_importance', ],
        'Parent Anxiety': [
            'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
            'parent_worries_about_future_p', 'parent_worries_about_family_p', 'parent_worries_a_lot_p',
            'parent_relationship_concerns_p',
            'parent_anxdisord_D_p', 'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p'
        ],
        'Parent Mood Issues': [
            'parent_cries_a_lot_p', 'parent_lonely_p', 'parent_feels_unloved_p', 'parent_paranoid_p',
            'parent_feels_inferior_p', 'parent_depressed_p', 'parent_feels_unsuccessful_p',
            'parent_tired_no_reason_p', 'parent_low_energy_p', 'parent_sleep_trouble_p',
            'parent_enjoys_little_p', 'parent_sudden_mood_changes_p', 'parent_suicidal_thoughts_p', 'parent_happy_person_p'
        ],
        'Parent Impulsivity and Behavior Regulation': [
            'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
            'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
            'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
            'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
            'parent_illegal_behavior_p', 'parent_self_harm_p', 'parent_doesnt_eat_well_p'
        ],
        'Parent Social Functioning': [
            'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
            'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
            'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
            'parent_teases_others_p', 'parent_stands_up_rights_p'
        ],
        'Parent Cognitive and Attention Issues': [
            'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
            'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p',
            'parent_repeats_acts_p', 'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p',
            'parent_decision_trouble_p', 'parent_priority_trouble_p',
            'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p'
        ],
        'Parent Personality': [
            'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p',
            'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p',
            'parent_louder_than_others_p', 'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p',
            'parent_easily_bored_p', 'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p',
            'parent_prefers_to_be_alone_p', 'parent_no_guilt_p', 'parent_sense_of_fairness_p',
            'parent_high_sleep_duration_p',
            'parent_opposite_sex_wish_p'
        ],
        'Parent Delta Psychopathology': [
            'delta_parent_sleep_trouble_p', 'delta_parent_worries_about_family_p',
            'delta_parent_friendship_trouble_p', 'delta_parent_poor_work_performance_p',
            'delta_parent_aches_pains_p', 'delta_parent_not_liked_by_others_p',
            'delta_parent_feels_overwhelmed_p', 'delta_parent_feels_unloved_p',
            'delta_parent_bad_family_relationship_p', 'delta_parent_worries_about_future_p',
            'delta_parent_worries_a_lot_p', 'delta_parent_depressed_p',
            'delta_parent_concentration_trouble_p', 'delta_parent_stubborn_irritable_p',
            'delta_parent_drinks_too_much_p', 'delta_parent_financial_failures_p',
            'delta_parent_meets_family_duties_p', 'delta_parent_planning_trouble_p',
            'delta_parent_bad_relationships_p', 'delta_parent_drug_use_p',
            'delta_parent_drug_days_nonmedical_p', 'delta_parent_financial_trouble_p',
            'delta_parent_internal_D_p', 'delta_parent_somatic_problems_D_p'
        ],
        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'positive_finance_p', 'parent_work_absences_p', 'parent_financial_trouble_p', 'parent_fails_to_pay_debts_p',
            'couldnt_afford_phone', 'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off'],
        'Residential Characteristics': ['neighborhood_safety_ss_p', 'neighborhood_safe_y',  'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],
        'School Dynamics': ['disobeys_at_school_k', 'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k',
            'enjoys_school_k', 'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k',
            'school_disengagement_ss_k', 'bad_grades', 'repeated_grade', 'grades_dropped', 'school_detension_suspension',
            'child_newschool_p', 'finds_schoolboring_k'],
        'Anxiety': ['social_fear_present_PK', 'worries_p', 'clings_to_adults_p', 'nervous_general_p',
            'nervous_twitching_p', 'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p',
            'fear_becoming_obese_present_PK', 'self_conscious_k', 'anxious_fearful_k'],
        'ADHD': ['trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
            'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p', 'easily_distracted_p'],
        'Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p', 'conduct_physical_fights_present_PK',
            'acts_immature_k', 'destroys_others_things_k', 'disobeys_parents_k', 'stubborn_k', 'hot_temper_k'],
        'Family Psychopathology Aggregated': ['parent_internal_D_p', 'parent_anhedonia_B_p', 'obsessions_present_B_p', 'poor_eye_contact_B_p','nightmares_B_p',
            'parent_elevated_mood_B_p', 'parent_excessive_worry_B_p', 'parent_lying_B_p', 'parent_social_anxiety_disorder_B_p',
            'parent_sleep_problem_B_p', 'parent_bulimia_B_p', 'parent_attention_D_p',
            'parent_aggressive_D_p', 'parent_external_D_p',
            'parent_anxdisord_D_p', 'parent_antisocial_D_p', 'parent_hyperactive_D_p',
            'parent_somatic_problems_D_p', 'parent_intrusive_thoughts_D_p', 'parent_avoidant_person_D_p', 'parent_personal_strength_D_p',
            'd_grandfather_dep', 'd_grandmother_dep', 'm_grandfather_dep', 'm_grandmother_dep', 'father_mania', 'mother_mania',
            'father_trouble', 'parent_hospitalized_emo', 'parent_therapy_emo'],
        'Family Drug Use': [
            'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
            'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse',
            'cigs_during_pregnancy_p', 'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p',
            'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
            'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p',
            'weed_before_pregnancy_p', 'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p',
            'drugs_before_pregnancy_p', 'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p',
            'caffeine_during_pregnancy_p', 'parent_tobacco_use_frequency_p', 'parent_drug_use_p', 'parent_drinks_too_much_p',
            'parent_drinks_frequency_p', 'parent_drunk_days_p', 'parent_drug_days_nonmedical_p',
            'parent_alcohol_days_drunk_p', 'parent_alcohol_excess_p', 'parent_alcohol_freq_p',
            'parent_drug_days_p', 'parent_tobacco_use_p',
            'druguse_alcohol_p', 'druguse_other_p', 'druguse_tobacco'],
        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            *[f'pc_gene_aces{i}' for i in range(1, 33)]],
        'Task and Resting State': [
            *[f'pc_SSTmri{i}' for i in range(1, 76)],
            *[f'pc_nbackmri{i}' for i in range(1, 76)],
            *[f'pc_midfmri{i}' for i in range(1, 76)],
            *[f'pc_rsFMRI{i}' for i in range(1, 76)],
        ],
        'Structural and DTI': [
            *[f'pc_struct{i}' for i in range(1, 76)],
            *[f'pc_DTI{i}' for i in range(1, 76)],
        ],
        'Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'experienced_crime_p', 'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p',
            'car_accident_hurt_p', 'big_accident_need_treatment_p', 'fire_victim_p', 'natural_disaster_victim_p',
            'terrorism_victim_p', 'war_death_witness_p', 'stabbing_shooting_witness_p',
            'stabbing_shooting_victim_community_p', 'stabbing_shooting_victim_home_p', 'beating_victim_home_p',
            'stranger_threatened_child_victim_p', 'family_threatened_child_victim_p', 'adult_family_fighting_victim_p',
            'domestic_child_sexually_abuse_victim_p', 'foreign_child_sexually_abuse_victim_p',
            'peer_child_sexually_abuse_victim_p', 'sudden_death_in_family_p'],
        'Religion': [
            'child_religion', 'religious_service_frequency', 'relig_importance'],
        'Parent Psychopathology Composite': ['parent_anxdep_D_p', 'parent_depressed_mood_B_p',
            'parent_dysthymia_B_p', 'parent_suicide_past_B_p', 'ptsd_diagnosis_present_B_p'],
    },

3: {  # T3 Variables
              'Cognitive Task Parameters': [
            # Stroop
            'str_accuracy', 'str_accuracy_ic', 'str_accuracy_c',
            'str_mrt', 'str_mrt_ic', 'str_mrt_c',
            'str_stroop_mrt', 'str_stroop_acc',
            # Delay Discounting
            'ddt_rew_amnt', 'ddt_mrt', 'ddt_mrt_delayed', 'ddt_mrt_immediate',
            # Arithmetic / Enumeration
            'correctRT_singlearith', 'num_correct_singlearith',
            'accu_mixeddigitarith', 'totalcorrect_arith', 'totalcorrect_enum',
        ],
      'Cognitive Style': ['bdefs_explain_idea_p', 'bdefs_explain_pt_p', 'bdefs_explain_seq_p',
            'bdefs_impulsive_action_p', 'bdefs_inconsistant_p', 'bdefs_process_info_p', 'bdefs_rechannel_p',
            'bdefs_sense_time_p', 'bdefs_shortcuts_p', 'bdefs_stop_think_p', 'problem_solving_ss_k', 'strange_ideas_p'],
        'Emotional Regulation Strategies': ['emoreg_sup_ss_k', 'emoreg_reapp_ss_k', 'emoreg_sup_control_k',
            'emoreg_reapp_happy_k', 'emoreg_sup_hide_k', 'emoreg_reapp_less_bad_k', 'emoreg_sup_self_k',
            'emoreg_reapp_thoughts_k', 'bdefs_calm_down_p', 'bdefs_consequences_p', 'bdefs_distract_upset_p',
            'child_selfaware_p', 'child_clear_feelings_p', 'child_emotion_overwhelm_p', 'child_feelings_attentive_p',
            'child_feelings_care_p', 'child_feelings_know_p', 'child_upset_acknowledge_p', 'child_upset_angry_p',
            'child_upset_ashamed_p', 'child_upset_control_p', 'child_upset_bad_behavior_p', 'child_upset_better_p',
            'child_upset_poor_concentrate_p', 'child_upset_no_control_p', 'child_upset_unproductive_p',
            'child_upset_embarrassed_p', 'child_upset_bad_esteem_p',
            'child_upset_nothing_better_p', 'child_upset_fixation_p', 'child_upset_bad_focus_p',
            'child_upset_guilty_p', 'child_upset_irritated_p', 'child_upset_longtime_better_p',
            'child_upset_lose_control_p', 'child_upset_out_control_p', 'child_upset_longtime_p',
            'child_upset_weak_p'],
        'Delta Other Psychopathology': ["delta_adhd_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
            "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
            "delta_somatic_problems_D_p",
            "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p"],
        'Other Personality Features': ['loquacious_p', 'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p',
            'easily_embarrassed_p', 'secretive_p', 'perfectionist_p', 'easily_offended_p',
            'blames_others_p', 'sociable_p', 'school_excitement_p', 'not_critical_others_p', 'stubborn_p',
            'disagreeable_p', 'goal_continuity_p', 'strange_ideas_p','bdefs_lazy_p', 'mania_7up_ss_k', 'enjoys_music_k', 'sex_orient_y'],
        'Medical/Somatic Problems': ['medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p', 'pain_last_month_k',
            'seriously_sick_lastyear_k', 'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p',
            'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p',
            'wets_bed_p'],
        'Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p', 'fallsleeptime',
            'wakeuptime', 'wakesleepcalc', 'chronotype',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],
        'Physical Activity/Features': ['puberty_k', 'sex', 'no_sports_activities_p', 'birth_weight_p'],
        'Technology Use': ['sphone_Interupt', 'sphone_noreason', 'sphone_distress', 'sphone_attachment',
            'vgame_thinking', 'socialmedia_daysperweek_k', 'videogames_daysperweek_k', 'bullied_on_internet_k'],
        'Social Relationship Quality': [
            'not_liked_p', 'doesnt_get_along_p', 'prosocial_ss_p', 'close_boy_friends_k',
            'close_girl_friends_k', 'peer_net_protective_ss_k', 'peers_beh_prosocial_ss_k', 'peers_beh_delinquent_ss_k',
            'feels_leftout_k', 'not_invited_k', 'excluded_k', 'otherkids_spreadneg_rumors_k',
            'otherkids_gossip_k', 'feels_threatned_k', 'saysmeanthings_others_k', 'otherkids_saymeanthings_k',
            'bullied_on_internet_k', 'prosocial_ss_k', 'relational_victimization_ss_k', 'reputational_aggression_ss_k',
            'reputational_victimization_ss_k', 'overt_aggression_ss_k', 'overt_victimization_ss_k', 'relational_aggression_ss_k'],
        'Family Dynamics & Parenting': ['fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k',
            'fam_no_lose_temps_k', 'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k',
            'fam_try_one_up_k', 'fam_no_raise_voices_k', 'parent_care_misbehave_k', 'parent_care_whereabouts_k',
            'parent_care_friends_k', 'parent_helphomework_k', 'parent_safeplay_k', 'parent_gotoschool_k',
            'parent_troubleschool_k', 'parent_helpunderstanding_k', 'family_not_talk_aboutfeelings_p',
            'family_peaceful_p', 'family_open_discussing_anything_p', 'family_lose_temper_rare_p',
            'family_believe_not_raise_voice_p', 'frequent_family_conflict_p', 'family_conflict_ss_p',
            'family_expression_ss_p', 'family_intellectual_ss_p', 'family_activities_ss_p', 'family_organisation_ss_p',
            'parents_argue_more_p', 'family_emotionprob_p', 'parents_divorced_p', 'death_in_family_p',
            'family_move_p', 'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'parent_cares_ss_k', 'marital_status', 'parent_age', 'sex_P', 'num_brothers_p',
            'num_sisters_p', 'religious_service_frequency', 'relig_importance'],
        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'positive_finance_p',
            'couldnt_afford_phone', 'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off'
        ],
        'School Dynamics': ['disobeys_at_school_k', 'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k',
            'enjoys_school_k', 'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k',
            'school_disengagement_ss_k', 'bad_grades', 'repeated_grade', 'grades_dropped',
            'school_detension_suspension', 'child_newschool_p', 'finds_schoolboring_k'],
        'Anxiety': ['worries_p', 'clings_to_adults_p', 'nervous_general_p', 'nervous_twitching_p',
            'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p', 'self_conscious_k',
            'anxious_fearful_k'],
        'ADHD': ['trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
            'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p', 'easily_distracted_p'],
        'Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p', 'acts_immature_k', 'destroys_others_things_k',
            'disobeys_parents_k', 'stubborn_k', 'hot_temper_k'],
        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            *[f'pc_gene_aces{i}' for i in range(1, 33)]],
        'Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'experienced_crime_p', 'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p'],
        'Religion': [
            'child_religion', 'religious_service_frequency', 'relig_importance'],
        'Residential Characteristics': ['neighborhood_safety_ss_p', 'neighborhood_safe_y', 'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],
    },

4: {  # T4 Variables
        'Cognitive Task Parameters': [
            # NIH Toolbox
            'tb_picvocab', 'tb_flanker', 'tb_list', 'tb_pattern', 'tb_picture', 'tb_reading', 'tb_cryst',
            # Game of Dice
            'gd_safebets', 'gd_riskybets',
            # N-Back
            'nb_correct_nt', 'nb_correct_mrt', 'nb_correct_nt_2back', 'nb_correct_mrt_2back',
            'nb_correct_nt_pos', 'nb_correct_mrt_pos', 'nb_correct_nt_neutral', 'nb_correct_mrt_neutral',
            'nb_correct_nt_neg', 'nb_correct_mrt_neg',
            'nb2_accuracy_pos', 'nb2_resp_bias_pos', 'nb2_D_prime_pos',
            'nb2_accuracy_neg', 'nb2_resp_bias_neg', 'nb2_D_prime_neg',
            # SST behavioral
            'sst_ssrt_mean_est', 'sst_ssrt_int_est', 'sst_acceptable_performance',
            # MID
            'mid_mrt_smrw', 'mid_mrt_lgrw', 'mid_total_payout',
            'mid_acceptable_performance', 'mid_num_trials',
            # Little Man Task
            'lmt_accuracy', 'lmt_mrt', 'lmt_efficiency',
            # SST BEESTS model parameters (no factors at T4)
            'sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS',
            'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
            'sst_kappa0', 'sst_xM', 'sst_bG', 'sst_pp',
            # SST BEESTS medians
            'sst_median_ssrt', 'sst_median_PDR', 'sst_median_SD', 'sst_median_SDr',
            'sst_median_PDRg', 'sst_median_betaS', 'sst_median_bS2',
            # SST BEESTS variability
            'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV',
            'sst_sdSD', 'sst_sdCV',
            'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV',
            'sst_pdrgSD', 'sst_pdrgCV',
            'sst_absdeltaRV', 'sst_absdeltaSD', 'sst_absdeltaCV',
            'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV',
        ],
        'Delta Other Psychopathology': ["delta_adhd_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
            "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
            "delta_somatic_problems_D_p",
            "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p"],
        'Parent Delta Psychopathology': [
            'delta_parent_sleep_trouble_p', 'delta_parent_worries_about_family_p',
            'delta_parent_friendship_trouble_p', 'delta_parent_poor_work_performance_p',
            'delta_parent_aches_pains_p', 'delta_parent_not_liked_by_others_p',
            'delta_parent_feels_overwhelmed_p', 'delta_parent_feels_unloved_p',
            'delta_parent_bad_family_relationship_p', 'delta_parent_worries_about_future_p',
            'delta_parent_worries_a_lot_p', 'delta_parent_depressed_p',
            'delta_parent_concentration_trouble_p', 'delta_parent_stubborn_irritable_p',
            'delta_parent_drinks_too_much_p', 'delta_parent_financial_failures_p',
            'delta_parent_meets_family_duties_p', 'delta_parent_planning_trouble_p',
            'delta_parent_bad_relationships_p', 'delta_parent_drug_use_p',
            'delta_parent_internal_D_p', 'delta_parent_somatic_problems_D_p'
        ],
        'Emotional Regulation Strategies': ['emoreg_sup_ss_k', 'emoreg_reapp_ss_k', 'emoreg_sup_control_k',
            'emoreg_reapp_happy_k', 'emoreg_sup_hide_k', 'emoreg_reapp_less_bad_k', 'emoreg_sup_self_k',
            'emoreg_reapp_thoughts_k', 'child_selfaware_p', 'child_clear_feelings_p', 'child_emotion_overwhelm_p',
            'child_feelings_attentive_p', 'child_feelings_care_p', 'child_feelings_know_p', 'child_upset_acknowledge_p',
            'child_upset_angry_p', 'child_upset_ashamed_p', 'child_upset_control_p', 'child_upset_bad_behavior_p',
            'child_upset_better_p', 'child_upset_poor_concentrate_p', 'child_upset_no_control_p',
            'child_upset_unproductive_p', 'child_upset_embarrassed_p', 'child_upset_emotions_overwhelm_p',
            'child_upset_bad_esteem_p', 'child_upset_nothing_better_p', 'child_upset_fixation_p',
            'child_upset_bad_focus_p', 'child_upset_guilty_p', 'child_upset_irritated_p',
            'child_upset_longtime_better_p', 'child_upset_lose_control_p', 'child_upset_out_control_p',
            'child_upset_longtime_p', 'child_upset_weak_p'],
        'Other Personality Features': ['loquacious_p', 'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p',
            'easily_embarrassed_p', 'secretive_p', 'perfectionist_p', 'easily_offended_p',
            'blames_others_p', 'sociable_p', 'school_excitement_p', 'not_critical_others_p', 'stubborn_p',
            'disagreeable_p', 'goal_continuity_p', 'strange_ideas_p','up_negative_urgency_ss_k', 'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k',
            'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k', 'bis_behav_inhibition_ss_k',
            'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k', 'enjoys_music_k',
            'sex_orient_y'],
        'Medical/Somatic Problems': ['medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p', 'pain_last_month_k',
            'seriously_sick_lastyear_k', 'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p',
            'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p',
            'wets_bed_p'],
        'Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p', 'fallsleeptime',
            'wakeuptime', 'wakesleepcalc', 'chronotype',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],
        'Physical Activity/Features': ['height', 'weight', 'puberty_k', 'sex', 'no_sports_activities_p',
            'birth_weight_p', 'fitbit_resting_hr', 'fitbit_steps', 'fitbit_sedentary_mins', 'fitbit_lightlyactive_mins', 'fitbit_fairlyactive_mins', 'fitbit_veryactive_mins'],
        'Technology Use': ['sphone_Interupt', 'sphone_noreason', 'sphone_distress', 'sphone_attachment', 'bullied_on_internet_k', 'vgame_thinking'],
        'Social Relationship Quality': ['nonBfriends_k', 'close_nonB_friends_k', 'not_liked_p', 'doesnt_get_along_p',
            'prosocial_ss_p', 'close_boy_friends_k', 'close_girl_friends_k', 'peer_net_protective_ss_k',
            'peers_beh_prosocial_ss_k', 'peers_beh_delinquent_ss_k', 'feels_leftout_k', 'not_invited_k',
            'excluded_k', 'otherkids_spreadneg_rumors_k', 'otherkids_gossip_k', 'feels_threatned_k',
            'saysmeanthings_others_k', 'otherkids_saymeanthings_k', 'discrimination_ss_k', 'feels_discriminated_k',
            'senses_racism_k', 'doesnt_feel_accepted_k', 'bullied_on_internet_k', 'prosocial_ss_k',
            'relational_victimization_ss_k', 'reputational_aggression_ss_k', 'reputational_victimization_ss_k',
            'overt_aggression_ss_k', 'overt_victimization_ss_k', 'relational_aggression_ss_k',
            'feels_discriminated_teachers_k', 'feels_discriminated_adults_not_school_k',
            'feels_discriminated_students_k', 'feels_unwanted_american_society_k', 'feels_discriminated_americans_k'],
        'Family Dynamics & Parenting': ['p_comm_cohesion_ss', 'p_comm_ctrl_ss', 'p_comm_collective_efficacy_ss',
            'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k', 'fam_no_lose_temps_k',
            'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k', 'fam_try_one_up_k',
            'fam_no_raise_voices_k', 'family_not_talk_aboutfeelings_p', 'family_peaceful_p',
            'family_open_discussing_anything_p', 'family_lose_temper_rare_p', 'family_believe_not_raise_voice_p',
            'frequent_family_conflict_p', 'family_conflict_ss_p', 'family_expression_ss_p',
            'family_intellectual_ss_p', 'family_activities_ss_p', 'family_organisation_ss_p', 'parents_argue_more_p',
            'family_emotionprob_p', 'parents_divorced_p', 'death_in_family_p', 'family_move_p',
            'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'marital_status', 'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
            'religious_service_frequency', 'relig_importance', ],
        'Parent Anxiety': [
            'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
            'parent_worries_about_future_p', 'parent_worries_about_family_p', 'parent_worries_a_lot_p',
            'parent_relationship_concerns_p',
            'parent_anxdisord_D_p', 'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p'
        ],
        'Parent Mood Issues': [
            'parent_cries_a_lot_p', 'parent_lonely_p', 'parent_feels_unloved_p', 'parent_paranoid_p',
            'parent_feels_inferior_p', 'parent_depressed_p', 'parent_feels_unsuccessful_p',
            'parent_tired_no_reason_p', 'parent_low_energy_p', 'parent_sleep_trouble_p',
            'parent_enjoys_little_p', 'parent_sudden_mood_changes_p', 'parent_happy_person_p', 'parent_suicidal_thoughts_p'
        ],
        'Parent Impulsivity and Behavior Regulation': [
            'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
            'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
            'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
            'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
            'parent_illegal_behavior_p', 'parent_self_harm_p', 'parent_doesnt_eat_well_p'
        ],
        'Parent Social Functioning': [
            'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
            'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
            'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
            'parent_teases_others_p', 'parent_stands_up_rights_p'
        ],
        'Parent Cognitive and Attention Issues': [
            'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
            'parent_doesnt_eat_well_p', 'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p',
            'parent_repeats_acts_p', 'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p',
            'parent_decision_trouble_p', 'parent_priority_trouble_p',
            'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p'
        ],
        'Parent Personality': [
            'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p',
            'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p',
            'parent_louder_than_others_p', 'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p',
            'parent_easily_bored_p', 'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p',
            'parent_prefers_to_be_alone_p', 'parent_no_guilt_p', 'parent_sense_of_fairness_p',
            'parent_high_sleep_duration_p',
            'parent_opposite_sex_wish_p'
        ],
        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'positive_finance_p', 'parent_work_absences_p', 'parent_financial_trouble_p', 'parent_fails_to_pay_debts_p',
            'couldnt_afford_phone', 'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off'
        ],
        'Residential Characteristics': ['neighborhood_safety_ss_p', 'neighborhood_safe_y', 'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],
        'School Dynamics': ['disobeys_at_school_k', 'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k',
            'enjoys_school_k', 'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k',
            'school_disengagement_ss_k', 'bad_grades', 'school_detension_suspension', 'child_newschool_p',
            'finds_schoolboring_k'],
        'Anxiety': ['worries_p', 'clings_to_adults_p', 'nervous_general_p', 'nervous_twitching_p',
            'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p', 'self_conscious_k',
            'anxious_fearful_k'],
        'ADHD': ['trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
            'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p', 'easily_distracted_p'],
        'Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p', 'acts_immature_k', 'destroys_others_things_k',
            'disobeys_parents_k', 'stubborn_k', 'hot_temper_k'],
        'Family Psychopathology Aggregated': ['parent_internal_D_p', 'parent_anhedonia_B_p', 'obsessions_present_B_p', 'poor_eye_contact_B_p','nightmares_B_p',
            'parent_elevated_mood_B_p', 'parent_excessive_worry_B_p', 'parent_lying_B_p', 'parent_social_anxiety_disorder_B_p',
            'parent_sleep_problem_B_p', 'parent_bulimia_B_p', 'parent_attention_D_p',
            'parent_aggressive_D_p', 'parent_external_D_p',
            'parent_anxdisord_D_p', 'parent_antisocial_D_p', 'parent_hyperactive_D_p',
            'parent_somatic_problems_D_p', 'parent_intrusive_thoughts_D_p', 'parent_avoidant_person_D_p', 'parent_personal_strength_D_p',
            'd_grandfather_dep', 'd_grandmother_dep', 'm_grandfather_dep', 'm_grandmother_dep', 'father_mania', 'mother_mania',
            'father_trouble', 'parent_hospitalized_emo', 'parent_therapy_emo'],
        'Family Drug Use': [
            'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
            'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse',
            'cigs_during_pregnancy_p', 'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p',
            'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
            'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p',
            'weed_before_pregnancy_p', 'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p',
            'drugs_before_pregnancy_p', 'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p',
            'caffeine_during_pregnancy_p', 'parent_tobacco_use_frequency_p', 'parent_drug_use_p', 'parent_drinks_too_much_p',
            'parent_drinks_frequency_p', 'parent_drunk_days_p', 'parent_drug_days_nonmedical_p',
            'parent_alcohol_excess_p', 'parent_alcohol_freq_p', 'parent_alcohol_days_drunk_p', 'parent_drug_days_p', 'parent_tobacco_use_p'
        ],
        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            *[f'pc_gene_aces{i}' for i in range(1, 33)]],
        'Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'experienced_crime_p', 'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p'],
        'Religion': [
            'child_religion', 'religious_service_frequency', 'relig_importance'],
        'Inter': [
            'sex', 'sex_P', 'parent_income', 'num_brothers_p', 'num_sisters_p',
            'L_site_id', 'weight_grouped',
            'parent_education_grouped', 'parent_age_grouped', 'child_white',
            'feels_leftout_k',
            'sex_orient_y', 'trans_id_y', 'screentime_daysperweek_k',
            'screentime_weekday_ss_k', 'puberty_k', 'close_boy_friends_k',
            'close_girl_friends_k', 'cct',
            'nb_correct_nt_2back', 'sst_ssrt_mean_est', 'mid_total_payout',
            'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'crysflu_c', 'child_religion', 'relig_importance'
        ],
    }
}
# Default to child structure; overridden at runtime for parent targets

# ---
# PARENT TARGETS -- Variables Within & Across Categories (All TPs)
# ---

# ---
# Parent within-category dict + flat variable lists (t0-t4).
# ---

# -- Parent target structure (used when target is a parent outcome) ---
parent_variables_for_each_time_point = {
    0: {  # T0 Variables
        'Child Personality Features': ['up_negative_urgency_ss_k', 'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k',
            'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k', 'bis_behav_inhibition_ss_k',
            'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k', 'loquacious_p', 'bragadocious_p',
            'easily_jealous_p', 'wishes_other_sex_p', 'easily_embarrassed_p', 'secretive_p', 'perfectionist_p',
            'sex_orient_y'],

        'Child Medical/Somatic Problems': ['body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p',
            'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p',
            'wets_bed_p'],

        'Child Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],

        'Child Physical Activity/Features': ['height', 'weight', 'waist', 'saliva_DHEA', 'saliva_testosterone', 'puberty_k',
            'sex', 'birth_weight_p'],

        'Child Social Relationship Quality': ['not_liked_p', 'doesnt_get_along_p', 'prosocial_ss_p', 'close_boy_friends_k',
            'close_girl_friends_k', 'prosocial_ss_k'],

        'Child Mood Issues': ['enjoys_little_p', 'sad_p', 'suicidal_p', 'guilty_p', 'withdrawn_p'],

        'Family Dynamics & Parenting': ['fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k',
            'fam_no_lose_temps_k', 'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k',
            'fam_try_one_up_k', 'fam_no_raise_voices_k', 'family_peaceful_p', 'family_lose_temper_rare_p',
            'family_believe_not_raise_voice_p', 'frequent_family_conflict_p', 'family_conflict_ss_p',
            'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
            'religious_service_frequency', 'relig_importance'],

        'Parent Anxiety': [
            'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
            'parent_worries_about_family_p', 'parent_worries_a_lot_p',
            'parent_relationship_concerns_p',
            'parent_anxdisord_D_p', 'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p'
        ],

        'Parent Impulsivity and Behavior Regulation': [
            'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
            'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
            'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
            'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
            'parent_illegal_behavior_p', 'parent_doesnt_eat_well_p', 'parent_self_harm_p',
        ],

        'Parent Social Functioning': [
            'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
            'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
            'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
            'parent_teases_others_p', 'parent_stands_up_rights_p'
        ],

        'Parent Cognitive and Attention Issues': [
            'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
            'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p',
            'parent_repeats_acts_p', 'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p',
            'parent_decision_trouble_p', 'parent_priority_trouble_p'
        ],

        'Parent Personality': [
            'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p',
            'parent_clumsy_p', 'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p',
            'parent_louder_than_others_p', 'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p',
            'parent_easily_bored_p', 'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p',
            'parent_prefers_to_be_alone_p', 'parent_no_guilt_p', 'parent_sense_of_fairness_p',
            'parent_high_sleep_duration_p', 'parent_opposite_sex_wish_p'
        ],

        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'couldnt_afford_phone',
            'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off', 'parent_work_absences_p', 'parent_financial_trouble_p', 'parent_fails_to_pay_debts_p'
        ],

        'Residential Characteristics': ['neighborhood_safety_ss_p', 'area_deprivation_idx', 'neighborhood_safe_y',
            'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],

        'Child School Dynamics': ['getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k', 'enjoys_school_k',
            'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k', 'school_disengagement_ss_k',
            'repeated_grade', 'grades_dropped', 'school_detension_suspension', 'finds_schoolboring_k'],

        'Child Anxiety': ['social_fear_present_PK', 'worries_p', 'clings_to_adults_p', 'nervous_general_p',
            'nervous_twitching_p', 'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p'],

        'Child ADHD': ['cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p', 'easily_distracted_p'],

        'Child Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'attacks_others_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p'],

        'Parent Drug Use': [
          'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
          'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse',
          'cigs_during_pregnancy_p', 'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p',
          'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
          'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p',
          'weed_before_pregnancy_p', 'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p',
          'drugs_before_pregnancy_p', 'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p',
          'caffeine_during_pregnancy_p', 'parent_tobacco_use_p','parent_alcohol_excess_p', 'parent_alcohol_freq_p',
          'parent_alcohol_days_drunk_p', 'parent_drug_days_p'],

        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            'pc_gene_aces1', 'pc_gene_aces2', 'pc_gene_aces3', 'pc_gene_aces4',
            'pc_gene_aces5', 'pc_gene_aces6', 'pc_gene_aces7', 'pc_gene_aces8',
            'pc_gene_aces9', 'pc_gene_aces10', 'pc_gene_aces11', 'pc_gene_aces12',
            'pc_gene_aces13', 'pc_gene_aces14', 'pc_gene_aces15', 'pc_gene_aces16',
            'pc_gene_aces17', 'pc_gene_aces18', 'pc_gene_aces19', 'pc_gene_aces20',
            'pc_gene_aces21', 'pc_gene_aces22', 'pc_gene_aces23', 'pc_gene_aces24',
            'pc_gene_aces25', 'pc_gene_aces26', 'pc_gene_aces27', 'pc_gene_aces28',
            'pc_gene_aces29', 'pc_gene_aces30', 'pc_gene_aces31', 'pc_gene_aces32'],

        'Familial Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p', 'car_accident_hurt_p',
            'big_accident_need_treatment_p', 'fire_victim_p', 'natural_disaster_victim_p', 'terrorism_victim_p',
            'war_death_witness_p', 'stabbing_shooting_witness_p', 'stabbing_shooting_victim_community_p',
            'stabbing_shooting_victim_home_p', 'beating_victim_home_p', 'stranger_threatened_child_victim_p',
            'family_threatened_child_victim_p', 'adult_family_fighting_victim_p', 'domestic_child_sexually_abuse_victim_p',
            'foreign_child_sexually_abuse_victim_p', 'peer_child_sexually_abuse_victim_p', 'sudden_death_in_family_p'],

          'Religion Proxy': [
              'child_religion', 'religious_service_frequency', 'relig_importance'],
    },

1: {  # T1 Variables
        'Child Cognitive Task Outcomes': ['str_accuracy', 'str_accuracy_ic', 'str_accuracy_c', 'str_mrt', 'str_mrt_ic',
            'str_mrt_c', 'str_stroop_mrt', 'str_stroop_acc'],

        'Inter': [
            'sex', 'sex_P', 'parent_income', 'num_brothers_p', 'num_sisters_p',
            'L_site_id', 'weight_grouped',
            'parent_education_grouped', 'parent_age_grouped', 'child_white',
            'feels_leftout_k',
            'sex_orient_y', 'trans_id_y', 'screentime_daysperweek_k',
            'screentime_weekday_ss_k', 'puberty_k', 'close_boy_friends_k',
            'close_girl_friends_k',
            'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'crysflu_c', 'child_religion', 'relig_importance'
        ],
    },

2: {  # T2 Variables

        'Parent Personality': ['parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p',
            'parent_clumsy_p', 'parent_strange_thoughts_p', 'parent_uses_opportunities_p',
            'parent_louder_than_others_p', 'parent_yells_a_lot_p',
            'parent_easily_bored_p', 'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p',
            'parent_no_guilt_p', 'parent_sense_of_fairness_p',
            'parent_opposite_sex_wish_p'],

        'Parent Cognitive and Attention Issues': [
            'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
            'parent_not_good_at_details_p',
            'parent_repeats_acts_p', 'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p',
            'parent_decision_trouble_p', 'parent_priority_trouble_p', 'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p'
        ],

        'Inter': [
            'sex', 'sex_P', 'parent_income', 'parent_income_new', 'num_brothers_p', 'num_sisters_p',
            'L_site_id', 'weight', 'social_problems_D_p', 'cant_concentrate_p', 'delta_not_liked_p',
            'parent_age_grouped', 'child_white',
            'feels_leftout_k',
            'sex_orient_y', 'trans_id_y', 'screentime_daysperweek_k',
            'screentime_weekday_ss_k', 'puberty_k', 'close_boy_friends_k',
            'close_girl_friends_k', 'cct', 'bad_diet_p', 'bad_grades',
            'nb_correct_nt_2back', 'tb_cryst', 'sst_ssrt_mean_est', 'mid_total_payout',
            'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k', 'b_lifeevents_ss_p',
            'crysflu_c', 'child_religion', 'relig_importance'
        ],
    },

3: {  # T3 Variables
        'Child Cognitive Style': ['bdefs_explain_idea_p', 'bdefs_explain_pt_p', 'bdefs_explain_seq_p',
            'bdefs_impulsive_action_p', 'bdefs_inconsistant_p', 'bdefs_process_info_p', 'bdefs_rechannel_p',
            'bdefs_sense_time_p', 'bdefs_shortcuts_p', 'bdefs_stop_think_p', 'problem_solving_ss_k', 'strange_ideas_p'],

        'Inter': [
            'sex', 'sex_P', 'parent_income', 'num_brothers_p', 'num_sisters_p',
            'L_site_id', 'weight_grouped',
            'parent_education_grouped', 'parent_age_grouped', 'child_white',
            'feels_leftout_k',
            'sex_orient_y', 'trans_id_y', 'screentime_daysperweek_k',
            'screentime_weekday_ss_k', 'puberty_k', 'close_boy_friends_k',
            'close_girl_friends_k', 'cct',
            'nb_correct_nt_2back',
            'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'crysflu_c', 'child_religion', 'relig_importance'
        ],
    },

4: {  # T4 Variables
        'Child Cognitive Task Outcomes': ['tb_picvocab', 'tb_list', 'tb_picture', 'tb_reading', 'lmt_accuracy',
            'lmt_efficiency', 'gd_safebets', 'gd_riskybets'],

        'Child Delta': ["delta_anxdisord_D_p",
            "delta_parent_internal_D_p", "delta_adhd_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
            "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades", "delta_social_problems_D_p",
            "delta_somatic_problems_D_p", "delta_cant_concentrate_p",
            "delta_bad_diet_p", "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p"],

        'Child Mood Issues': ['enjoys_little_p', 'sad_p', 'suicidal_p', 'guilty_p', 'withdrawn_p'],

        'Child Emotional Regulation Strategies': ['emoreg_sup_ss_k', 'emoreg_reapp_ss_k', 'emoreg_sup_control_k',
            'emoreg_reapp_happy_k', 'emoreg_sup_hide_k', 'emoreg_reapp_less_bad_k', 'emoreg_sup_self_k',
            'emoreg_reapp_thoughts_k', 'child_selfaware_p', 'child_clear_feelings_p', 'child_emotion_overwhelm_p',
            'child_feelings_attentive_p', 'child_feelings_care_p', 'child_feelings_know_p', 'child_upset_acknowledge_p',
            'child_upset_angry_p', 'child_upset_ashamed_p', 'child_upset_control_p', 'child_upset_bad_behavior_p',
            'child_upset_better_p', 'child_upset_poor_concentrate_p', 'child_upset_no_control_p',
            'child_upset_unproductive_p', 'child_upset_embarrassed_p', 'child_upset_emotions_overwhelm_p',
            'child_upset_bad_esteem_p', 'child_upset_nothing_better_p', 'child_upset_fixation_p',
            'child_upset_bad_focus_p', 'child_upset_guilty_p', 'child_upset_irritated_p',
            'child_upset_longtime_better_p', 'child_upset_lose_control_p', 'child_upset_out_control_p',
            'child_upset_longtime_p', 'child_upset_weak_p'],

        'Child Other Personality Features': ['up_negative_urgency_ss_k', 'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k',
            'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k', 'bis_behav_inhibition_ss_k',
            'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k', 'enjoys_music_k', 'loquacious_p',
            'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p', 'easily_embarrassed_p', 'secretive_p',
            'perfectionist_p', 'sex_orient_y'],

        'Child Medical/Somatic Problems': ['medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p', 'pain_last_month_k',
            'seriously_sick_lastyear_k', 'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p',
            'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p', 'constipated_p', 'bad_toilet_habits_p',
            'wets_bed_p'],

        'Child Sleep Problems': ['difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p', 'fallsleeptime',
            'wakeuptime', 'wakesleepcalc', 'chronotype',
            'sleeps_more_p', 'sleeps_less_p', 'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p'],

        'Child Physical Activity/Features': ['height', 'weight', 'puberty_k', 'sex', 'no_sports_activities_p',
            'birth_weight_p', 'fitbit_resting_hr', 'fitbit_steps', 'fitbit_sedentary_mins', 'fitbit_lightlyactive_mins', 'fitbit_fairlyactive_mins', 'fitbit_veryactive_mins'],

        'Child Technology Use': ['sphone_Interupt', 'sphone_noreason', 'sphone_distress', 'sphone_attachment', 'bullied_on_internet_k', 'vgame_thinking'],

        'Child Social Relationship Quality': ['nonBfriends_k', 'close_nonB_friends_k', 'not_liked_p', 'doesnt_get_along_p',
            'prosocial_ss_p', 'close_boy_friends_k', 'close_girl_friends_k', 'peer_net_protective_ss_k',
            'peers_beh_prosocial_ss_k', 'peers_beh_delinquent_ss_k', 'feels_leftout_k', 'not_invited_k',
            'excluded_k', 'otherkids_spreadneg_rumors_k', 'otherkids_gossip_k', 'feels_threatned_k',
            'saysmeanthings_others_k', 'otherkids_saymeanthings_k', 'discrimination_ss_k', 'feels_discriminated_k',
            'senses_racism_k', 'doesnt_feel_accepted_k', 'bullied_on_internet_k', 'prosocial_ss_k',
            'relational_victimization_ss_k', 'reputational_aggression_ss_k', 'reputational_victimization_ss_k',
            'overt_aggression_ss_k', 'overt_victimization_ss_k', 'relational_aggression_ss_k',
            'feels_discriminated_teachers_k', 'feels_discriminated_adults_not_school_k',
            'feels_discriminated_students_k', 'feels_unwanted_american_society_k', 'feels_discriminated_americans_k'],

        'Family Dynamics & Parenting': ['p_comm_cohesion_ss', 'p_comm_ctrl_ss', 'p_comm_collective_efficacy_ss',
            'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k', 'fam_no_lose_temps_k',
            'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_keep_peace_k', 'fam_try_one_up_k',
            'fam_no_raise_voices_k', 'family_not_talk_aboutfeelings_p', 'family_peaceful_p',
            'family_open_discussing_anything_p', 'family_lose_temper_rare_p', 'family_believe_not_raise_voice_p',
            'frequent_family_conflict_p', 'family_conflict_ss_p', 'family_expression_ss_p',
            'family_intellectual_ss_p', 'family_activities_ss_p', 'family_organisation_ss_p', 'parents_argue_more_p',
            'family_emotionprob_p', 'parents_divorced_p', 'death_in_family_p', 'family_move_p',
            'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi', 'family_conflict_ss_k',
            'parent_monitoring_ss_k', 'marital_status', 'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
            'religious_service_frequency', 'relig_importance'],

        'Parent Personality': [
            'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p',
            'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p',
            'parent_louder_than_others_p', 'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p',
            'parent_easily_bored_p', 'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p',
            'parent_prefers_to_be_alone_p', 'parent_no_guilt_p', 'parent_sense_of_fairness_p',
            'parent_high_sleep_duration_p', 'parent_opposite_sex_wish_p'
        ],

        'Parent Anxiety': [
            'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
            'parent_worries_about_family_p', 'parent_worries_a_lot_p',
            'parent_relationship_concerns_p',
            'parent_anxdisord_D_p', 'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p'
        ],

        'Parent Impulsivity and Behavior Regulation': [
            'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
            'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
            'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
            'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
            'parent_illegal_behavior_p', 'parent_doesnt_eat_well_p', 'parent_self_harm_p',
        ],

        'Parent Social Functioning': [
            'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
            'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
            'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
            'parent_teases_others_p', 'parent_stands_up_rights_p'
        ],

        'Parent Cognitive and Attention Issues': [
            'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
            'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p',
            'parent_repeats_acts_p', 'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p',
            'parent_decision_trouble_p', 'parent_priority_trouble_p', 'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p',
        ],

        'Parent Delta Psychopathology': [
            'delta_parent_sleep_trouble_p', 'delta_parent_worries_about_family_p',
            'delta_parent_friendship_trouble_p', 'delta_parent_poor_work_performance_p',
            'delta_parent_aches_pains_p', 'delta_parent_not_liked_by_others_p',
            'delta_parent_feels_overwhelmed_p', 'delta_parent_feels_unloved_p',
            'delta_parent_bad_family_relationship_p',
            'delta_parent_worries_a_lot_p', 'delta_parent_somatic_problems_D_p',
            'delta_parent_concentration_trouble_p', 'delta_parent_stubborn_irritable_p',
            'delta_parent_drinks_too_much_p',
            'delta_parent_meets_family_duties_p', 'delta_parent_planning_trouble_p',
            'delta_parent_bad_relationships_p', 'delta_parent_drug_use_p',
            'delta_parent_internal_D_p'
        ],

        'SES & Mobility': ['parent_education', 'parent_income', 'struggle_food_expenses', 'couldnt_afford_phone',
            'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off', 'parent_work_absences_p', 'parent_financial_trouble_p', 'parent_fails_to_pay_debts_p'
        ],

        'Residential Characteristics': ['neighborhood_safety_ss_p', 'neighborhood_safe_y', 'resid_density', 'resid_walkability', 'resid_prox_roads', 'resid_crime_tot', 'resid_crime_violent',
            'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism', 'resid_sex_orient_bias',
            'resid_immigrant_bias', 'resid_racism', 'L_site_id'],

        'Child School Dynamics': ['disobeys_at_school_k', 'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k',
            'enjoys_school_k', 'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k',
            'school_disengagement_ss_k', 'bad_grades', 'school_detension_suspension', 'child_newschool_p',
            'finds_schoolboring_k'],

        'Child Anxiety': ['worries_p', 'clings_to_adults_p', 'nervous_general_p', 'nervous_twitching_p',
            'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p', 'self_conscious_k',
            'anxious_fearful_k'],

        'Child ADHD': ['trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
            'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p', 'easily_distracted_p'],

        'Child Externalizing': ['argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p', 'acts_immature_k', 'destroys_others_things_k',
            'disobeys_parents_k', 'stubborn_k', 'hot_temper_k'],

        'Child Social Personality Psychopathology Sum': [
            'easily_offended_p', 'blames_others_p', 'sociable_p', 'school_excitement_p',
            'not_critical_others_p', 'scared_dark_p', 'disagreeable_p', 'goal_continuity_p',
            'up_negative_urgency_ss_k', 'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k',
            'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k', 'bis_behav_inhibition_ss_k',
            'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k',
            'loquacious_p', 'bragadocious_p', 'easily_jealous_p', 'wishes_other_sex_p',
            'easily_embarrassed_p', 'secretive_p', 'perfectionist_p', 'sex_orient_y',
            'medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p', 'pain_last_month_k',
            'seriously_sick_lastyear_k', 'body_aches_p', 'frequent_headaches_p', 'nausea_p',
            'eye_problems_p', 'skin_problems_p', 'frequent_stomachaches_p', 'vomiting_p',
            'constipated_p', 'bad_toilet_habits_p', 'wets_bed_p',
            'difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p', 'fallsleeptime',
            'wakeuptime', 'wakesleepcalc', 'chronotype',
            'not_liked_p', 'doesnt_get_along_p', 'prosocial_ss_p', 'close_boy_friends_k',
            'close_girl_friends_k', 'peer_net_protective_ss_k', 'peers_beh_prosocial_ss_k', 'peers_beh_delinquent_ss_k',
            'feels_leftout_k', 'not_invited_k', 'excluded_k', 'otherkids_spreadneg_rumors_k', 'otherkids_gossip_k',
            'feels_threatned_k', 'saysmeanthings_others_k', 'otherkids_saymeanthings_k', 'discrimination_ss_k',
            'feels_discriminated_k', 'senses_racism_k', 'doesnt_feel_accepted_k', 'bullied_on_internet_k',
            'socialinfluence_meanfinal_k', 'relational_victimization_ss_k',
            'reputational_aggression_ss_k', 'reputational_victimization_ss_k', 'overt_aggression_ss_k',
            'overt_victimization_ss_k', 'relational_aggression_ss_k',
            'feels_discriminated_teachers_k', 'feels_discriminated_adults_not_school_k', 'feels_discriminated_students_k',
            'feels_unwanted_american_society_k', 'feels_discriminated_americans_k',
            'social_fear_present_PK', 'worries_p', 'clings_to_adults_p', 'nervous_general_p',
            'nervous_twitching_p', 'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p',
            'paranoid_p', 'fear_becoming_obese_present_PK', 'self_conscious_k', 'anxious_fearful_k',
            'trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
            'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p',
            'argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p',
            'destroys_own_things_p', 'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
            'breaks_rules_p', 'fights_p', 'lying_p', 'steals_home_p', 'steals_outside_p',
            'threatens_others_p', 'whines_p', 'demands_attention_p', 'conduct_physical_fights_present_PK',
            'acts_immature_k', 'destroys_others_things_k', 'disobeys_parents_k', 'stubborn_k', 'hot_temper_k'
          ],

        'Parent Other Psychopathology': ['obsessions_present_B_p', 'poor_eye_contact_B_p','nightmares_B_p',
            'parent_elevated_mood_B_p', 'parent_excessive_worry_B_p', 'parent_lying_B_p', 'parent_social_anxiety_disorder_B_p',
            'parent_sleep_problem_B_p', 'parent_bulimia_B_p', 'parent_attention_D_p',
            'parent_aggressive_D_p', 'parent_external_D_p',
            'parent_anxdisord_D_p', 'parent_antisocial_D_p', 'parent_hyperactive_D_p',
            'parent_somatic_problems_D_p', 'parent_intrusive_thoughts_D_p', 'parent_avoidant_person_D_p',
            'd_grandfather_dep', 'd_grandmother_dep', 'm_grandfather_dep', 'm_grandmother_dep', 'father_mania', 'mother_mania',
            'father_trouble', 'parent_hospitalized_emo', 'parent_therapy_emo'],

        'Parent Drug Use': [
          'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
          'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse',
          'cigs_during_pregnancy_p', 'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p',
          'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
          'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p',
          'weed_before_pregnancy_p', 'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p',
          'drugs_before_pregnancy_p', 'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p',
          'caffeine_during_pregnancy_p', 'parent_tobacco_use_p','parent_alcohol_excess_p', 'parent_alcohol_freq_p',
          'parent_alcohol_days_drunk_p', 'parent_drug_days_p'],

        'Ancestral-Genetic PCs/Ethnicity': ['desc_african_AFR_B', 'desc_native_american_AMR_B', 'desc_alaska_native_AMR_B',
            'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_vietnamese_EAS_B',
            'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B', 'desc_latin_B',
            'pc_gene_aces1', 'pc_gene_aces2', 'pc_gene_aces3', 'pc_gene_aces4',
            'pc_gene_aces5', 'pc_gene_aces6', 'pc_gene_aces7', 'pc_gene_aces8',
            'pc_gene_aces9', 'pc_gene_aces10', 'pc_gene_aces11', 'pc_gene_aces12',
            'pc_gene_aces13', 'pc_gene_aces14', 'pc_gene_aces15', 'pc_gene_aces16',
            'pc_gene_aces17', 'pc_gene_aces18', 'pc_gene_aces19', 'pc_gene_aces20',
            'pc_gene_aces21', 'pc_gene_aces22', 'pc_gene_aces23', 'pc_gene_aces24',
            'pc_gene_aces25', 'pc_gene_aces26', 'pc_gene_aces27', 'pc_gene_aces28',
            'pc_gene_aces29', 'pc_gene_aces30', 'pc_gene_aces31', 'pc_gene_aces32'],

        'Familial Adverse Life Events': ['g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'experienced_crime_p', 'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p'],

        'Religion Proxy': [
            'child_religion', 'religious_service_frequency', 'relig_importance'],

        'Inter': [
            'sex', 'sex_P', 'parent_income', 'num_brothers_p', 'num_sisters_p',
            'L_site_id', 'weight_grouped',
            'parent_education_grouped', 'parent_age_grouped', 'child_white',
            'feels_leftout_k',
            'sex_orient_y', 'trans_id_y', 'screentime_daysperweek_k',
            'screentime_weekday_ss_k', 'puberty_k', 'close_boy_friends_k',
            'close_girl_friends_k', 'cct',
            'nb_correct_nt_2back', 'sst_ssrt_mean_est', 'mid_total_payout',
            'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k',
            'crysflu_c', 'child_religion', 'relig_importance'
        ],
    }
}

# Default to child structure; overridden at runtime for parent targets
variables_for_each_time_point = child_variables_for_each_time_point

# ---
# ACROSS-CATEGORY VARIABLES (flat lists per timepoint)
# ---
# Full predictor pools for across-category analysis at each wave.
# Circular variables are filtered at runtime via get_circular_exclusions().
# Variables absent from the loaded DataFrame are silently skipped.
# Genetic ancestry PCs (pc_gene_aces1-32) appear in every timepoint.
# ---

# -- T0: Baseline (n=756) ---
t0_variables = [
    # Child Cognitive Task Outcomes
    "tb_flanker", "tb_cardsort", "lmt_efficiency", "nb_correct_nt",
    "nb_correct_mrt", "nb_correct_nt_2back", "nb_correct_mrt_2back", "nb_correct_nt_pos",
    "nb_correct_mrt_pos", "nb_correct_nt_neutral", "nb_correct_mrt_neutral", "nb_correct_nt_neg",
    "nb_correct_mrt_neg", "nb2_accuracy_pos", "nb2_resp_bias_pos", "nb2_D_prime_pos", "nb2_accuracy_neg",
    "nb2_resp_bias_neg", "nb2_D_prime_neg", "sst_ssrt_mean_est", "sst_ssrt_int_est",
    "sst_acceptable_performance", "mid_acceptable_performance", "mid_total_payout", "cct", "mr_total",
    "mr_matrix", "ravlt_s_total", "ravlt_s_repitition", "ravlt_s_intrusions", "ravlt_l_total",
    "ravlt_l_repitition", "ravlt_l_intrusions", "sst_theta0", "sst_theta1", "sst_mu", "sst_aS", "sst_dS",
    "sst_gamma0", "sst_gamma1", "sst_dG", "sst_aG1", "sst_aG2", "sst_kappa0", "sst_xM", "sst_sM",
    "sst_bG", "sst_pp", "sst_factor1", "sst_factor2", "sst_factor3", "sst_median_ssrt",
    "sst_median_PDR", "sst_median_SD", "sst_median_SDr", "sst_median_PDRg",
    "sst_median_betaS", "sst_median_bS2", "sst_median_absdelta", "sst_pdrV", "sst_pdrSD", "sst_pdrCV",
    "sst_sdRV", "sst_sdSD", "sst_sdCV", "sst_sdrRV", "sst_sdrSD", "sst_sdrCV", "sst_pdrgV", "sst_pdrgSD",
    "sst_pdrgCV", "sst_betasV", "sst_betasSD", "sst_betasCV", "sst_absdeltaRV", "sst_absdeltaSD",
    "sst_absdeltaCV", "sst_bs2V", "sst_bs2SD", "sst_bs2CV", "strange_ideas_p",
    # Child Personality Features
    "up_negative_urgency_ss_k", "up_lackofplanning_ss_k", "up_sensationseeking_ss_k",
    "up_positiveurgency_ss_k", "up_lackperseverance_ss_k", "bis_behav_inhibition_ss_k",
    "bis_reward_responsive_ss_k", "bis_drive_ss_k", "bis_funseeking_ss_k", "loquacious_p",
    "bragadocious_p", "easily_jealous_p", "wishes_other_sex_p", "easily_embarrassed_p", "secretive_p",
    "perfectionist_p", "sex_orient_y",
    # Child Medical/Somatic Problems
    "body_aches_p", "frequent_headaches_p", "nausea_p", "eye_problems_p", "skin_problems_p",
    "frequent_stomachaches_p", "vomiting_p", "constipated_p", "bad_toilet_habits_p", "wets_bed_p",
    # Child Sleep Problems
    "difficulty_goingtosleep_p", "difficulty_wakingup_p", "nightmares_p",
    # Child Physical Activity/Features
    "height", "weight", "waist", "saliva_DHEA", "saliva_testosterone", "puberty_k", "sex",
    "birth_weight_p",
    # Child Technology Use
    "screentime_weekday_ss_k", "screentime_weekend_ss_k",
    # Child Social Relationship Quality
    "not_liked_p", "doesnt_get_along_p", "prosocial_ss_p", "close_boy_friends_k", "close_girl_friends_k",
    "prosocial_ss_k",
    # Family Dynamics & Parenting
    "fam_fight_often_k", "fam_no_open_anger_k", "fam_throw_things_k", "fam_no_lose_temps_k",
    "fam_criticize_often_k", "fam_hit_each_other_k", "fam_keep_peace_k", "fam_try_one_up_k",
    "fam_no_raise_voices_k", "family_peaceful_p", "family_lose_temper_rare_p",
    "family_believe_not_raise_voice_p", "frequent_family_conflict_p", "family_conflict_ss_p",
    "y_acceptance_ss_p_crpbi", "y_acceptance_ss_caregiver_crpbi", "family_conflict_ss_k",
    "parent_monitoring_ss_k", "parent_age", "sex_P", "num_brothers_p", "num_sisters_p",
    "religious_service_frequency", "relig_importance",
    # Parent Anxiety
    "parent_fearful_or_anxious_p", "parent_specific_fears_p", "parent_fear_of_bad_thoughts_p",
    "parent_worries_about_future_p", "parent_worries_about_family_p", "parent_relationship_concerns_p",
    # Parent Mood Issues
    "parent_cries_a_lot_p", "parent_lonely_p", "parent_feels_unloved_p", "parent_paranoid_p",
    "parent_feels_inferior_p", "parent_depressed_p", "parent_feels_unsuccessful_p",
    "parent_tired_no_reason_p", "parent_low_energy_p", "parent_sleep_trouble_p",
    "parent_enjoys_little_p", "parent_sudden_mood_changes_p", "parent_happy_person_p",
    # Parent Impulsivity and Behavior Regulation
    "parent_impulsive_p", "parent_risky_decisions_p", "parent_drives_too_fast_p", "parent_tardy_often_p",
    "parent_money_management_trouble_p", "parent_priority_trouble_p", "parent_behavior_changeable_p",
    "parent_hot_temper_p", "parent_attention_seeking_p", "parent_destroys_own_things_p",
    "parent_destroys_others_things_p", "parent_doesnt_finish_tasks_p", "parent_strange_behavior_p",
    "parent_illegal_behavior_p", "parent_self_harm_p", "parent_suicidal_thoughts_p",
    # Parent Social Functioning
    "parent_bad_relationships_p", "parent_bad_family_relationship_p", "parent_not_liked_by_others_p",
    "parent_friendship_trouble_p", "parent_prefers_older_people_p", "parent_associates_with_trouble_p",
    "parent_bad_opposite_sex_relationship_p", "parent_meets_family_duties_p",
    "parent_clowns_or_shows_off_p", "parent_teases_others_p", "parent_stands_up_rights_p",
    # Parent Cognitive and Attention Issues
    "parent_forgetful_p", "parent_concentration_trouble_p", "parent_confused_p",
    "parent_planning_trouble_p", "parent_doesnt_eat_well_p", "parent_not_good_at_details_p",
    "parent_obsessive_thoughts_p", "parent_repeats_acts_p", "parent_max_effort_p",
    "parent_disorganized_p", "parent_loses_things_p", "parent_decision_trouble_p",
    # Parent Personality
    "parent_bragging_p", "parent_honest_p", "parent_secretive_p", "parent_stubborn_irritable_p",
    "parent_clumsy_p", "parent_strange_thoughts_p", "parent_self_conscious_p",
    "parent_uses_opportunities_p", "parent_louder_than_others_p", "parent_yells_a_lot_p",
    "parent_shy_or_timid_p", "parent_restless_p", "parent_easily_bored_p", "parent_hyperactive_p",
    "parent_talks_too_much_p", "parent_avoids_talking_p", "parent_prefers_to_be_alone_p",
    "parent_no_guilt_p", "parent_sense_of_fairness_p", "parent_high_sleep_duration_p",
    # Parent Other
    "parent_physical_attacks_p", "parent_picks_skin_p", "parent_heart_racing_p", "parent_numbness_p",
    "parent_sees_things_p", "parent_hears_voices_p", "parent_speech_problems_p",
    "parent_opposite_sex_wish_p",
    # Child School Dynamics
    "getalong_teachers_k", "feelsafe_at_school_k", "feels_smart_k", "enjoys_school_k",
    "grades_important_k", "school_environment_ss_k", "school_involvement_ss_k",
    "school_disengagement_ss_k", "repeated_grade", "grades_dropped", "school_detension_suspension",
    "finds_schoolboring_k",
    # Child Anxiety
    "social_fear_present_PK", "worries_p", "clings_to_adults_p", "nervous_general_p",
    "nervous_twitching_p", "fears_excl_school_p", "fears_school_p", "paranoid_p",
    # Child ADHD
    "cant_concentrate_p", "doesnt_finish_p", "hyperactive_p", "impulsive_p",
    # Child Externalizing
    "argues_p", "stubborn_p", "temper_tantrums_p", "bullies_others_p", "destroys_own_things_p",
    "destroys_others_things_p", "disobedient_home_p", "disobedient_school_p", "breaks_rules_p",
    "fights_p", "lying_p", "steals_home_p", "steals_outside_p", "threatens_others_p", "whines_p",
    "demands_attention_p", "parent_personal_strength_D_p",
    # Parent Drug Use
    "father_alcohol", "mother_alcohol", "father_druguse", "mother_druguse",
    "cigs_during_pregnancy_p", "alcohol_during_pregnancy_p", "weed_during_pregnancy_p",
    "cocaine_during_pregnancy_p", "heroin_during_pregnancy_p",
    # Family Psychopathology Aggregated
    "d_grandfather_dep", "d_grandmother_dep", "m_grandfather_dep", "m_grandmother_dep",
    "father_mania", "mother_mania", "father_trouble", "parent_hospitalized_emo", "parent_therapy_emo",
    # Ancestral-Genetic PCs/Ethnicity
    "desc_african_AFR_B", "desc_native_american_AMR_B", "desc_alaska_native_AMR_B", "desc_chinese_EAS_B",
    "desc_japanese_EAS_B", "desc_korean_EAS_B", "desc_vietnamese_EAS_B", "desc_european_EUR_B",
    "desc_asian_indian_SAS_B", "desc_other_south_asian_SAS_B", "desc_latin_B",
    *[f"pc_gene_aces{i}" for i in range(1, 33)],
    # Familial Adverse Life Events
    "g_lifeevents_ss_k", "b_lifeevents_ss_k", "b_lifeevents_affected_ss_k", "g_lifeevents_ss_p",
    "b_lifeevents_ss_p", "b_lifeevents_affected_ss_p", "car_accident_hurt_p",
    "big_accident_need_treatment_p", "fire_victim_p", "natural_disaster_victim_p", "terrorism_victim_p",
    "war_death_witness_p", "stabbing_shooting_witness_p", "stabbing_shooting_victim_community_p",
    "stabbing_shooting_victim_home_p", "beating_victim_home_p", "stranger_threatened_child_victim_p",
    "family_threatened_child_victim_p", "adult_family_fighting_victim_p",
    "domestic_child_sexually_abuse_victim_p", "foreign_child_sexually_abuse_victim_p",
    "peer_child_sexually_abuse_victim_p", "sudden_death_in_family_p",
    # Child Cognitive Task Outcomes (NIH Toolbox)
    "tb_picvocab", "tb_list", "tb_pattern", "tb_picture", "tb_reading", "tb_cryst",
    # SES & Mobility
    "parent_education", "parent_income", "struggle_food_expenses", "couldnt_afford_phone",
    "couldnt_afford_rent_mortgage", "evicted", "gas_electric_oil_turned_off",
    # Residential Characteristics
    "neighborhood_safety_ss_p", "area_deprivation_idx", "neighborhood_safe_y", "resid_density",
    "resid_walkability", "resid_prox_roads", "resid_crime_tot", "resid_crime_violent",
    "resid_crime_drug", "resid_crime_dui", "resid_lead_risk_poverty", "resid_lead_risk_houses_perc",
    "resid_no2_avg", "resid_pm25_avg", "resid_sexism", "resid_sex_orient_bias",
    "resid_immigrant_bias", "resid_racism", "L_site_id", "fears_being_bad_p",
    # Parent Other Psychopathology
    "obsessions_present_B_p", "poor_eye_contact_B_p",
    "hallucinogen_use_history_B_p", "hallucinogen_current_B_p", "sedative_hypnotic_anxiolytic_use_B_p",
    "nightmares_B_p", "ptsd_diagnosis_present_B_p", "parent_anhedonia_B_p", "parent_elevated_mood_B_p",
    "parent_lying_B_p", "parent_wish_dead_present_B_p",
    "parent_suicide_past_B_p", "parent_dysthymia_B_p",
    "parent_social_anxiety_disorder_B_p", "parent_bulimia_B_p",
    "parent_attention_D_p", "parent_aggressive_D_p", "parent_external_D_p", "parent_depress_D_p",
    "parent_antisocial_D_p", "parent_hyperactive_D_p", "parent_somatic_problems_D_p",
    "parent_intrusive_thoughts_D_p", "parent_avoidant_person_D_p", "parent_depressed_mood_B_p",
    "parent_adhd_D_p",
    # Child Mood Issues
    "enjoys_little_p", "sad_p", "suicidal_p", "guilty_p", "withdrawn_p",
    # Parent Drug Use (prenatal)
    "prescriptionmed_pregnancy_p", "cigs_before_pregnancy_p", "alcohol_before_pregnancy_p",
    "weed_before_pregnancy_p", "cocaine_before_pregnancy_p", "heroin_before_pregnancy_p",
    "drugs_before_pregnancy_p", "drinksperweek_during_pregnancy_p", "drugs_during_pregnancy_p",
    "caffeine_during_pregnancy_p",
]

# -- T1: Year 1 (n=284) ---
t1_variables = [
    # Child Cognitive Task Outcomes
    "str_accuracy", "str_accuracy_ic", "str_accuracy_c", "str_mrt", "str_mrt_ic", "str_mrt_c",
    "str_stroop_mrt", "str_stroop_acc",
    # Child Delta
    "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
    "delta_somatic_problems_D_p", "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p",
    # Child Personality Features
    "mania_7up_ss_k", "loquacious_p", "bragadocious_p", "easily_jealous_p", "wishes_other_sex_p",
    "easily_embarrassed_p", "secretive_p", "perfectionist_p", "sex_orient_y",
    # Child Medical/Somatic Problems
    "medhx_p", "medhx_doctorvisit_p", "medhx_asthma_p", "medhx_allergies_p", "medhx_brain_p",
    "medhx_diabetes_p", "medhx_epilepsy_p", "medhx_heart_p", "medhx_headaches_p",
    "medhx_emergencyroom_p", "medhx_brokenbones_p", "medhx_seriousinjury_p", "seriously_sick_lastyear_k",
    "body_aches_p", "frequent_headaches_p", "nausea_p", "eye_problems_p", "skin_problems_p",
    "frequent_stomachaches_p", "vomiting_p", "constipated_p", "bad_toilet_habits_p", "wets_bed_p",
    # Child Sleep Problems
    "difficulty_goingtosleep_p", "difficulty_wakingup_p", "nightmares_p",
    # Child Physical Activity/Features
    "height", "weight", "waist", "saliva_DHEA", "saliva_testosterone", "puberty_k", "sex",
    "no_sports_activities_p", "birth_weight_p",
    # Child Technology Use
    "screentime_weekday_ss_k", "screentime_weekend_ss_k",
    # Child Social Relationship Quality
    "discrimination_ss_k", "feels_discriminated_k", "senses_racism_k", "doesnt_feel_accepted_k",
    "prosocial_ss_k", "feels_discriminated_teachers_k", "feels_discriminated_adults_not_school_k",
    "feels_discriminated_students_k", "feels_unwanted_american_society_k",
    "feels_discriminated_americans_k",
    # Family Dynamics & Parenting
    "fam_fight_often_k", "fam_no_open_anger_k", "fam_throw_things_k", "fam_no_lose_temps_k",
    "fam_criticize_often_k", "fam_hit_each_other_k", "fam_keep_peace_k", "fam_try_one_up_k",
    "fam_no_raise_voices_k", "family_peaceful_p", "family_lose_temper_rare_p",
    "family_believe_not_raise_voice_p", "frequent_family_conflict_p", "family_conflict_ss_p",
    "parents_argue_more_p", "family_emotionprob_p", "parents_divorced_p", "death_in_family_p",
    "family_move_p", "y_acceptance_ss_p_crpbi", "y_acceptance_ss_caregiver_crpbi",
    "family_conflict_ss_k", "parent_monitoring_ss_k", "marital_status", "parent_age", "sex_P",
    "num_brothers_p", "num_sisters_p", "religious_service_frequency", "relig_importance",
    # SES & Mobility
    "parent_education", "parent_income", "struggle_food_expenses", "positive_finance_p",
    # Residential Characteristics
    "neighborhood_safety_ss_p", "neighborhood_safe_y", "L_site_id",
    # Child School Dynamics
    "disobeys_at_school_k", "getalong_teachers_k", "feelsafe_at_school_k", "feels_smart_k",
    "enjoys_school_k", "grades_important_k", "school_environment_ss_k", "school_involvement_ss_k",
    "school_disengagement_ss_k", "repeated_grade", "grades_dropped", "school_detension_suspension",
    "child_newschool_p", "finds_schoolboring_k",
    # Child Externalizing
    "argues_p", "bullies_others_p", "destroys_own_things_p", "destroys_others_things_p",
    "disobedient_home_p", "disobedient_school_p", "breaks_rules_p", "fights_p", "lying_p",
    "steals_home_p", "steals_outside_p", "threatens_others_p", "whines_p", "demands_attention_p",
    "acts_immature_k", "destroys_others_things_k", "disobeys_parents_k", "stubborn_k", "hot_temper_k",
    # Parent Other Psychopathology
    "parent_lying_B_p", "parent_bulimia_B_p",
    # Parent Drug Use
    "father_alcohol", "mother_alcohol", "father_druguse", "mother_druguse",
    # Family Psychopathology Aggregated
    "d_grandfather_dep", "d_grandmother_dep", "m_grandfather_dep", "m_grandmother_dep",
    "cigs_during_pregnancy_p", "alcohol_during_pregnancy_p", "weed_during_pregnancy_p",
    "cocaine_during_pregnancy_p", "heroin_during_pregnancy_p",
    "father_mania", "mother_mania", "father_trouble", "parent_hospitalized_emo", "parent_therapy_emo",
    # Ancestral-Genetic PCs/Ethnicity
    "desc_african_AFR_B", "desc_native_american_AMR_B", "desc_alaska_native_AMR_B", "desc_chinese_EAS_B",
    "desc_japanese_EAS_B", "desc_korean_EAS_B", "desc_vietnamese_EAS_B", "desc_european_EUR_B",
    "desc_asian_indian_SAS_B", "desc_other_south_asian_SAS_B", "desc_latin_B",
    *[f"pc_gene_aces{i}" for i in range(1, 33)],
    # Familial Adverse Life Events
    "g_lifeevents_ss_k", "b_lifeevents_ss_k", "b_lifeevents_affected_ss_k", "experienced_crime_p",
    "g_lifeevents_ss_p", "b_lifeevents_ss_p", "b_lifeevents_affected_ss_p",
    # Child Cognitive Style
    "problem_solving_ss_k", "strange_ideas_p", "avoids_eyecontact_p", "bad_conversational_flow_p",
    "narrow_interests_p", "sensory_sensitivity_p", "concentration_on_parts_p", "face_understanding",
    # Child Delta
    "delta_anxdisord_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
    "delta_social_problems_D_p",
    # Child Social Relationship Quality
    "not_liked_p", "doesnt_get_along_p", "prosocial_ss_p", "difficulty_making_friends_p",
    "regarded_weird_p",
    # Child Anxiety
    "worries_p", "clings_to_adults_p", "nervous_general_p", "nervous_twitching_p", "fears_excl_school_p",
    "fears_school_p", "fears_being_bad_p", "paranoid_p", "self_conscious_k", "anxious_fearful_k",
    # Child ADHD
    "trouble_concentrating_k", "difficulty_finishing_tasks_k", "easily_distracted_k",
    "cant_concentrate_p", "doesnt_finish_p", "hyperactive_p", "impulsive_p",
    # Child Externalizing
    "stubborn_p", "temper_tantrums_p",
    # Parent Other Psychopathology
    "obsessions_present_B_p", "poor_eye_contact_B_p",
    "hallucinogen_use_history_B_p", "hallucinogen_current_B_p", "sedative_hypnotic_anxiolytic_use_B_p",
    "nightmares_B_p", "parent_elevated_mood_B_p",
    "parent_social_anxiety_disorder_B_p", "parent_attention_D_p",
    "parent_aggressive_D_p", "parent_external_D_p", "parent_antisocial_D_p", "parent_hyperactive_D_p",
    "parent_somatic_problems_D_p", "parent_avoidant_person_D_p", "parent_personal_strength_D_p",
    # Child Mood Issues
    "enjoys_little_p", "sad_p", "suicidal_p", "guilty_p", "withdrawn_p",
    # Parent Drug Use (prenatal)
    "prescriptionmed_pregnancy_p", "cigs_before_pregnancy_p", "alcohol_before_pregnancy_p",
    "weed_before_pregnancy_p", "cocaine_before_pregnancy_p", "heroin_before_pregnancy_p",
    "drugs_before_pregnancy_p", "drinksperweek_during_pregnancy_p", "drugs_during_pregnancy_p",
    "caffeine_during_pregnancy_p",
]

# -- T2: Year 2 (n=617) ---
t2_variables = [
    # Child Cognitive Task Outcomes
    "tb_picvocab", "tb_picture", "tb_reading", "tb_flanker", "tb_list", "tb_cardsort", "tb_pattern",
    "gd_safebets", "gd_riskybets",
    "ravlt_s_total", "ravlt_s_repitition", "ravlt_s_intrusions",
    "ravlt_l_total", "ravlt_l_repitition", "ravlt_l_intrusions",
    "nb_correct_nt", "nb_correct_mrt", "nb_correct_nt_2back", "nb_correct_mrt_2back",
    "nb_correct_nt_pos", "nb_correct_mrt_pos", "nb_correct_nt_neutral", "nb_correct_mrt_neutral",
    "nb_correct_nt_neg", "nb_correct_mrt_neg",
    "nb2_accuracy_pos", "nb2_resp_bias_pos", "nb2_D_prime_pos",
    "nb2_accuracy_neg", "nb2_resp_bias_neg", "nb2_D_prime_neg",
    "sst_ssrt_mean_est", "sst_ssrt_int_est", "sst_acceptable_performance",
    "mid_mrt_lgrw", "mid_total_payout", "mid_acceptable_performance", "mid_num_trials",
    "lmt_correct_nt", "lmt_mrt", "lmt_correct_mrt", "lmt_efficiency",
    # Dynamic Cognitive Control Parameters
    "sst_theta0", "sst_theta1", "sst_mu", "sst_aS", "sst_dS",
    "sst_gamma0", "sst_gamma1", "sst_dG", "sst_aG1", "sst_aG2",
    "sst_kappa0", "sst_xM", "sst_sM", "sst_bG", "sst_pp",
    "sst_factor1", "sst_factor2", "sst_factor3",
    "sst_median_ssrt", "sst_median_PDR", "sst_median_SD",
    "sst_median_SDr", "sst_median_PDRg",
    "sst_median_betaS", "sst_median_bS2", "sst_median_absdelta",
    "sst_pdrV", "sst_pdrSD", "sst_pdrCV",
    "sst_sdRV", "sst_sdSD", "sst_sdCV",
    "sst_sdrRV", "sst_sdrSD", "sst_sdrCV",
    "sst_pdrgV", "sst_pdrgSD", "sst_pdrgCV",
    "sst_betasV", "sst_betasSD", "sst_betasCV",
    "sst_absdeltaRV", "sst_absdeltaSD", "sst_absdeltaCV",
    "sst_bs2V", "sst_bs2SD", "sst_bs2CV",
    # Child Delta
    "delta_parent_internal_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
    "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
    "delta_social_problems_D_p", "delta_somatic_problems_D_p", "delta_anxdisord_D_p",
    "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p", "delta_cant_concentrate_p",
    # Child Positive Affect
    "pa_attentive", "pa_delighted", "pa_calm", "pa_at_ease",
    "pa_enthusiastic", "pa_interested", "pa_confident", "pa_energetic", "pa_concentrate",
    # Child Personality Features
    "easily_offended_p", "blames_others_p", "sociable_p", "school_excitement_p",
    "not_critical_others_p", "scared_dark_p", "disagreeable_p", "goal_continuity_p",
    "up_negative_urgency_ss_k", "up_lackofplanning_ss_k", "up_sensationseeking_ss_k",
    "up_positiveurgency_ss_k", "up_lackperseverance_ss_k",
    "bis_behav_inhibition_ss_k", "bis_reward_responsive_ss_k", "bis_drive_ss_k", "bis_funseeking_ss_k",
    "loquacious_p", "bragadocious_p", "easily_jealous_p", "wishes_other_sex_p",
    "easily_embarrassed_p", "secretive_p", "perfectionist_p", "sex_orient_y", "strange_ideas_p",
    # Child Medical/Somatic Problems
    "medhx_p", "medhx_doctorvisit_p", "medhx_emergencyroom_p", "pain_last_month_k",
    "seriously_sick_lastyear_k", "body_aches_p", "frequent_headaches_p", "nausea_p",
    "eye_problems_p", "skin_problems_p", "frequent_stomachaches_p", "vomiting_p",
    "constipated_p", "bad_toilet_habits_p", "wets_bed_p",
    # Child Sleep Problems
    "difficulty_goingtosleep_p", "difficulty_wakingup_p", "nightmares_p",
    "fallsleeptime", "wakeuptime", "wakesleepcalc", "chronotype",
    # Diet/Nutrition
    "fruit_intake", "vegetable_intake", "protein_sources_intake", "legume_intake",
    "added_sugar", "sugary_beverage_freq", "dairy_intake", "whole_grain_intake",
    "total_calories", "protein_intake", "carbohydrate_intake", "fiber_intake",
    "sodium_intake", "potassium_intake", "total_sugar", "saturated_fat",
    # Child Physical Activity/Features
    "height", "weight", "waist", "puberty_k", "sex", "no_sports_activities_p", "birth_weight_p",
    "fitbit_resting_hr", "fitbit_steps", "fitbit_sedentary_mins",
    "fitbit_lightlyactive_mins", "fitbit_fairlyactive_mins", "fitbit_veryactive_mins",
    # Child Technology Use
    "socialmedia_daysperweek_k", "videogames_daysperweek_k", "bullied_on_internet_k", "vgame_thinking",
    # Child Social Relationship Quality
    "not_liked_p", "doesnt_get_along_p", "prosocial_ss_p",
    "close_boy_friends_k", "close_girl_friends_k",
    "peer_net_protective_ss_k", "peers_beh_prosocial_ss_k", "peers_beh_delinquent_ss_k",
    "feels_leftout_k", "not_invited_k", "excluded_k",
    "otherkids_spreadneg_rumors_k", "otherkids_gossip_k", "feels_threatned_k",
    "saysmeanthings_others_k", "otherkids_saymeanthings_k",
    "discrimination_ss_k", "feels_discriminated_k", "senses_racism_k", "doesnt_feel_accepted_k",
    "prosocial_ss_k", "socialinfluence_meanfinal_k",
    "relational_victimization_ss_k", "reputational_aggression_ss_k", "reputational_victimization_ss_k",
    "overt_aggression_ss_k", "overt_victimization_ss_k", "relational_aggression_ss_k",
    "feels_discriminated_teachers_k", "feels_discriminated_adults_not_school_k",
    "feels_discriminated_students_k", "feels_unwanted_american_society_k",
    "feels_discriminated_americans_k", "social_problems_D_p",
    # Family Dynamics & Parenting
    "p_comm_cohesion_ss", "p_comm_ctrl_ss", "p_comm_collective_efficacy_ss",
    "fam_fight_often_k", "fam_no_open_anger_k", "fam_throw_things_k", "fam_no_lose_temps_k",
    "fam_criticize_often_k", "fam_hit_each_other_k", "fam_keep_peace_k",
    "fam_try_one_up_k", "fam_no_raise_voices_k",
    "family_not_talk_aboutfeelings_p", "family_peaceful_p", "family_open_discussing_anything_p",
    "family_lose_temper_rare_p", "family_believe_not_raise_voice_p", "frequent_family_conflict_p",
    "family_conflict_ss_p", "family_expression_ss_p", "family_intellectual_ss_p",
    "family_activities_ss_p", "family_organisation_ss_p",
    "parents_argue_more_p", "family_emotionprob_p", "parents_divorced_p",
    "death_in_family_p", "family_move_p",
    "family_conflict_ss_k", "parent_monitoring_ss_k",
    "marital_status", "parent_age", "sex_P", "num_brothers_p", "num_sisters_p",
    "parent_family_responsibilities_p",
    # Parent Anxiety
    "parent_fearful_or_anxious_p", "parent_specific_fears_p", "parent_fear_of_bad_thoughts_p",
    "parent_worries_about_future_p", "parent_worries_about_family_p", "parent_relationship_concerns_p",
    # Parent Mood Issues
    "parent_cries_a_lot_p", "parent_lonely_p", "parent_feels_unloved_p", "parent_paranoid_p",
    "parent_feels_inferior_p", "parent_depressed_p", "parent_feels_unsuccessful_p",
    "parent_tired_no_reason_p", "parent_low_energy_p", "parent_sleep_trouble_p",
    "parent_enjoys_little_p", "parent_sudden_mood_changes_p", "parent_happy_person_p",
    "parent_suicidal_thoughts_p",
    # Parent Impulsivity and Behavior Regulation
    "parent_impulsive_p", "parent_risky_decisions_p", "parent_drives_too_fast_p",
    "parent_tardy_often_p", "parent_money_management_trouble_p", "parent_priority_trouble_p",
    "parent_behavior_changeable_p", "parent_hot_temper_p", "parent_attention_seeking_p",
    "parent_destroys_own_things_p", "parent_destroys_others_things_p",
    "parent_doesnt_finish_tasks_p", "parent_strange_behavior_p",
    "parent_illegal_behavior_p", "parent_self_harm_p", "parent_doesnt_eat_well_p",
    # Parent Social Functioning
    "parent_bad_relationships_p", "parent_bad_family_relationship_p", "parent_not_liked_by_others_p",
    "parent_friendship_trouble_p", "parent_prefers_older_people_p", "parent_associates_with_trouble_p",
    "parent_bad_opposite_sex_relationship_p", "parent_meets_family_duties_p",
    "parent_clowns_or_shows_off_p", "parent_teases_others_p", "parent_stands_up_rights_p",
    # Parent Cognitive and Attention Issues
    "parent_forgetful_p", "parent_concentration_trouble_p", "parent_confused_p",
    "parent_planning_trouble_p", "parent_not_good_at_details_p",
    "parent_obsessive_thoughts_p", "parent_repeats_acts_p", "parent_max_effort_p",
    "parent_disorganized_p", "parent_loses_things_p", "parent_decision_trouble_p",
    # Parent Personality
    "parent_bragging_p", "parent_honest_p", "parent_secretive_p", "parent_stubborn_irritable_p",
    "parent_clumsy_p", "parent_strange_thoughts_p", "parent_self_conscious_p",
    "parent_uses_opportunities_p", "parent_louder_than_others_p", "parent_yells_a_lot_p",
    "parent_shy_or_timid_p", "parent_restless_p", "parent_easily_bored_p", "parent_hyperactive_p",
    "parent_talks_too_much_p", "parent_avoids_talking_p", "parent_prefers_to_be_alone_p",
    "parent_no_guilt_p", "parent_sense_of_fairness_p", "parent_high_sleep_duration_p",
    # Parent Other
    "parent_physical_attacks_p", "parent_picks_skin_p", "parent_heart_racing_p", "parent_numbness_p",
    "parent_sees_things_p", "parent_hears_voices_p", "parent_speech_problems_p",
    "parent_opposite_sex_wish_p",
    # Parent Delta Psychopathology
    "delta_parent_sleep_trouble_p", "delta_parent_worries_about_family_p",
    "delta_parent_friendship_trouble_p", "delta_parent_poor_work_performance_p",
    "delta_parent_aches_pains_p", "delta_parent_not_liked_by_others_p",
    "delta_parent_feels_overwhelmed_p", "delta_parent_feels_unloved_p",
    "delta_parent_bad_family_relationship_p", "delta_parent_worries_about_future_p",
    "delta_parent_worries_a_lot_p", "delta_parent_depressed_p",
    "delta_parent_concentration_trouble_p", "delta_parent_stubborn_irritable_p",
    "delta_parent_drinks_too_much_p",
    "delta_parent_meets_family_duties_p", "delta_parent_planning_trouble_p",
    "delta_parent_bad_relationships_p", "delta_parent_drug_use_p",
    "delta_parent_drug_days_nonmedical_p", "delta_parent_financial_trouble_p",
    "delta_parent_somatic_problems_D_p",
    # SES & Mobility
    "parent_education", "parent_income",
    "struggle_food_expenses", "positive_finance_p",
    "couldnt_afford_phone", "couldnt_afford_rent_mortgage", "evicted", "gas_electric_oil_turned_off",
    "parent_work_absences_p", "parent_financial_trouble_p", "parent_fails_to_pay_debts_p",
    # Residential Characteristics
    "neighborhood_safety_ss_p", "neighborhood_safe_y",
    "resid_density", "resid_walkability", "resid_prox_roads",
    "resid_crime_tot", "resid_crime_violent", "resid_crime_drug", "resid_crime_dui",
    "resid_lead_risk_poverty", "resid_lead_risk_houses_perc",
    "resid_no2_avg", "resid_pm25_avg",
    "resid_sexism", "resid_sex_orient_bias", "resid_immigrant_bias", "resid_racism", "L_site_id",
    # Child School Dynamics
    "disobeys_at_school_k", "getalong_teachers_k", "feelsafe_at_school_k", "feels_smart_k",
    "enjoys_school_k", "grades_important_k", "school_environment_ss_k", "school_involvement_ss_k",
    "school_disengagement_ss_k", "bad_grades", "repeated_grade", "grades_dropped",
    "school_detension_suspension", "child_newschool_p", "finds_schoolboring_k",
    # Child Anxiety
    "social_fear_present_PK", "worries_p", "clings_to_adults_p", "nervous_general_p",
    "nervous_twitching_p", "fears_excl_school_p", "fears_school_p", "fears_being_bad_p",
    "paranoid_p", "fear_becoming_obese_present_PK", "self_conscious_k", "anxious_fearful_k",
    # Child ADHD
    "trouble_concentrating_k", "difficulty_finishing_tasks_k", "easily_distracted_k",
    "cant_concentrate_p", "doesnt_finish_p", "hyperactive_p", "impulsive_p",
    # Child Externalizing
    "argues_p", "stubborn_p", "temper_tantrums_p", "bullies_others_p",
    "destroys_own_things_p", "destroys_others_things_p", "disobedient_home_p", "disobedient_school_p",
    "breaks_rules_p", "fights_p", "lying_p", "steals_home_p", "steals_outside_p",
    "threatens_others_p", "whines_p", "demands_attention_p",
    "druguse_alcohol_p", "druguse_tobacco", "druguse_other_p",
    "conduct_physical_fights_present_PK",
    "acts_immature_k", "destroys_others_things_k", "disobeys_parents_k", "stubborn_k", "hot_temper_k",
    # Parent Other Psychopathology
    "poor_eye_contact_B_p", "nightmares_B_p", "obsessions_present_B_p",
    "parent_anhedonia_B_p", "parent_elevated_mood_B_p",
    "parent_lying_B_p", "parent_social_anxiety_disorder_B_p", "parent_bulimia_B_p",
    "parent_attention_D_p", "parent_aggressive_D_p", "parent_external_D_p",
    "parent_antisocial_D_p", "parent_hyperactive_D_p",
    "parent_somatic_problems_D_p", "parent_intrusive_thoughts_D_p", "parent_avoidant_person_D_p",
    "parent_personal_strength_D_p",
    # Family Psychopathology Aggregated
    "d_grandfather_dep", "d_grandmother_dep", "m_grandfather_dep", "m_grandmother_dep",
    "father_mania", "mother_mania", "father_trouble",
    "parent_hospitalized_emo", "parent_therapy_emo",
    "parent_depressed_mood_B_p", "parent_dysthymia_B_p", "parent_suicide_past_B_p",
    "ptsd_diagnosis_present_B_p",
    # Parent Drug Use
    "hallucinogen_use_history_B_p", "hallucinogen_current_B_p", "sedative_hypnotic_anxiolytic_use_B_p",
    "father_alcohol", "mother_alcohol", "father_druguse", "mother_druguse",
    "cigs_during_pregnancy_p", "alcohol_during_pregnancy_p", "weed_during_pregnancy_p",
    "cocaine_during_pregnancy_p", "heroin_during_pregnancy_p",
    "prescriptionmed_pregnancy_p", "cigs_before_pregnancy_p", "alcohol_before_pregnancy_p",
    "weed_before_pregnancy_p", "cocaine_before_pregnancy_p", "heroin_before_pregnancy_p",
    "drugs_before_pregnancy_p", "drinksperweek_during_pregnancy_p", "drugs_during_pregnancy_p",
    "caffeine_during_pregnancy_p",
    "parent_tobacco_use_frequency_p", "parent_drug_use_p",
    "parent_drug_days_nonmedical_p",
    "parent_alcohol_excess_p", "parent_alcohol_freq_p", "parent_alcohol_days_drunk_p",
    # Ancestral-Genetic PCs/Ethnicity
    "desc_african_AFR_B", "desc_native_american_AMR_B", "desc_alaska_native_AMR_B",
    "desc_chinese_EAS_B", "desc_japanese_EAS_B", "desc_korean_EAS_B", "desc_vietnamese_EAS_B",
    "desc_european_EUR_B", "desc_asian_indian_SAS_B", "desc_other_south_asian_SAS_B", "desc_latin_B",
    *[f"pc_gene_aces{i}" for i in range(1, 33)],
    # Familial Adverse Life Events
    "g_lifeevents_ss_k", "b_lifeevents_ss_k", "b_lifeevents_affected_ss_k", "experienced_crime_p",
    "g_lifeevents_ss_p", "b_lifeevents_ss_p", "b_lifeevents_affected_ss_p",
    "car_accident_hurt_p", "big_accident_need_treatment_p", "fire_victim_p",
    "natural_disaster_victim_p", "terrorism_victim_p", "war_death_witness_p",
    "stabbing_shooting_witness_p", "stabbing_shooting_victim_community_p",
    "stabbing_shooting_victim_home_p", "beating_victim_home_p",
    "stranger_threatened_child_victim_p", "family_threatened_child_victim_p",
    "adult_family_fighting_victim_p", "domestic_child_sexually_abuse_victim_p",
    "foreign_child_sexually_abuse_victim_p", "peer_child_sexually_abuse_victim_p",
    "sudden_death_in_family_p",
    # Religion Proxy
    "child_religion", "religious_service_frequency", "relig_importance",
]

# -- T3: Year 3 (n=350) ---
t3_variables = [
    # Child Cognitive Style
    "bdefs_explain_idea_p", "bdefs_explain_pt_p", "bdefs_explain_seq_p", "bdefs_impulsive_action_p",
    "bdefs_inconsistant_p", "bdefs_process_info_p", "bdefs_rechannel_p", "bdefs_sense_time_p",
    "bdefs_shortcuts_p", "bdefs_stop_think_p", "problem_solving_ss_k", "strange_ideas_p",
    # Child Positive Affect
    "pa_attentive", "pa_delighted", "pa_calm", "pa_at_ease", "pa_enthusiastic", "pa_interested",
    "pa_confident", "pa_energetic", "pa_concentrate", "pa_sum_k",
    # Child Delta
    "delta_anxdisord_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
    "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
    "delta_social_problems_D_p", "delta_somatic_problems_D_p", "delta_bad_diet_p",
    "delta_atschool_total_problems_ss_t", "delta_b_lifeevents_ss_p",
    # Child Emotional Regulation Strategies
    "emoreg_sup_ss_k", "emoreg_reapp_ss_k", "emoreg_sup_control_k", "emoreg_reapp_happy_k",
    "emoreg_sup_hide_k", "emoreg_reapp_less_bad_k", "emoreg_sup_self_k", "emoreg_reapp_thoughts_k",
    "bdefs_calm_down_p", "bdefs_consequences_p", "bdefs_distract_upset_p", "child_selfaware_p",
    "child_clear_feelings_p", "child_feelings_attentive_p", "child_feelings_care_p",
    "child_feelings_know_p", "child_upset_acknowledge_p", "child_upset_angry_p", "child_upset_ashamed_p",
    "child_upset_control_p", "child_upset_bad_behavior_p", "child_upset_better_p",
    "child_upset_poor_concentrate_p", "child_upset_no_control_p", "child_upset_unproductive_p",
    "child_upset_embarrassed_p", "child_upset_bad_esteem_p", "child_upset_nothing_better_p",
    "child_upset_fixation_p", "child_upset_bad_focus_p", "child_upset_guilty_p",
    "child_upset_irritated_p", "child_upset_longtime_better_p", "child_upset_lose_control_p",
    "child_upset_out_control_p", "child_upset_longtime_p", "child_upset_weak_p",
    # Child Personality Features
    "bdefs_lazy_p", "mania_7up_ss_k", "enjoys_music_k", "loquacious_p", "bragadocious_p",
    "easily_jealous_p", "wishes_other_sex_p", "easily_embarrassed_p", "secretive_p", "perfectionist_p",
    "sex_orient_y",
    # Child Medical/Somatic Problems
    "medhx_p", "medhx_doctorvisit_p", "medhx_emergencyroom_p", "pain_last_month_k",
    "seriously_sick_lastyear_k", "body_aches_p", "frequent_headaches_p", "nausea_p", "eye_problems_p",
    "skin_problems_p", "frequent_stomachaches_p", "vomiting_p", "constipated_p", "bad_toilet_habits_p",
    "wets_bed_p",
    # Child Sleep Problems
    "difficulty_goingtosleep_p", "difficulty_wakingup_p", "nightmares_p", "fallsleeptime", "wakeuptime",
    "wakesleepcalc", "chronotype",
    # Child Physical Activity/Features
    "puberty_k", "sex", "no_sports_activities_p", "birth_weight_p",
    # Child Technology Use
    "sphone_Interupt", "sphone_noreason", "sphone_distress", "sphone_attachment", "vgame_thinking",
    "socialmedia_daysperweek_k", "videogames_daysperweek_k",
    # Child Social Relationship Quality
    "not_liked_p", "doesnt_get_along_p", "prosocial_ss_p", "close_boy_friends_k", "close_girl_friends_k",
    "peer_net_protective_ss_k", "peers_beh_prosocial_ss_k", "peers_beh_delinquent_ss_k",
    "feels_leftout_k", "not_invited_k", "excluded_k", "otherkids_spreadneg_rumors_k",
    "otherkids_gossip_k", "feels_threatned_k", "saysmeanthings_others_k", "otherkids_saymeanthings_k",
    "bullied_on_internet_k", "prosocial_ss_k", "relational_victimization_ss_k",
    "reputational_aggression_ss_k", "reputational_victimization_ss_k", "overt_aggression_ss_k",
    "overt_victimization_ss_k", "relational_aggression_ss_k",
    # Family Dynamics & Parenting
    "fam_fight_often_k", "fam_no_open_anger_k", "fam_throw_things_k", "fam_no_lose_temps_k",
    "fam_criticize_often_k", "fam_hit_each_other_k", "fam_keep_peace_k", "fam_try_one_up_k",
    "fam_no_raise_voices_k", "parent_care_misbehave_k", "parent_care_whereabouts_k",
    "parent_care_friends_k", "parent_helphomework_k", "parent_safeplay_k", "parent_gotoschool_k",
    "parent_troubleschool_k", "parent_helpunderstanding_k", "family_not_talk_aboutfeelings_p",
    "family_peaceful_p", "family_open_discussing_anything_p", "family_lose_temper_rare_p",
    "family_believe_not_raise_voice_p", "frequent_family_conflict_p", "family_conflict_ss_p",
    "family_expression_ss_p", "family_intellectual_ss_p", "family_activities_ss_p",
    "family_organisation_ss_p", "parents_argue_more_p", "family_emotionprob_p", "parents_divorced_p",
    "death_in_family_p", "family_move_p", "y_acceptance_ss_p_crpbi", "y_acceptance_ss_caregiver_crpbi",
    "family_conflict_ss_k", "parent_monitoring_ss_k", "parent_cares_ss_k", "marital_status",
    "parent_age", "sex_P", "num_brothers_p", "num_sisters_p", "religious_service_frequency",
    "relig_importance",
    # SES & Mobility
    "parent_education", "parent_income", "struggle_food_expenses", "positive_finance_p", "L_site_id",
    # Child School Dynamics
    "disobeys_at_school_k", "getalong_teachers_k", "feelsafe_at_school_k", "feels_smart_k",
    "enjoys_school_k", "grades_important_k", "school_environment_ss_k", "school_involvement_ss_k",
    "school_disengagement_ss_k", "bad_grades", "repeated_grade", "grades_dropped",
    "school_detension_suspension", "child_newschool_p", "finds_schoolboring_k",
    # Child Anxiety
    "worries_p", "clings_to_adults_p", "nervous_general_p", "nervous_twitching_p", "fears_excl_school_p",
    "fears_school_p", "fears_being_bad_p", "paranoid_p", "self_conscious_k", "anxious_fearful_k",
    # Child ADHD
    "trouble_concentrating_k", "difficulty_finishing_tasks_k", "easily_distracted_k",
    "cant_concentrate_p", "doesnt_finish_p", "hyperactive_p", "impulsive_p",
    # Child Externalizing
    "argues_p", "stubborn_p", "temper_tantrums_p", "bullies_others_p", "destroys_own_things_p",
    "destroys_others_things_p", "disobedient_home_p", "disobedient_school_p", "breaks_rules_p",
    "fights_p", "lying_p", "steals_home_p", "steals_outside_p", "threatens_others_p", "whines_p",
    "demands_attention_p", "acts_immature_k", "destroys_others_things_k", "disobeys_parents_k",
    "stubborn_k", "hot_temper_k",
    # Parent Drug Use
    "father_alcohol", "mother_alcohol", "father_druguse", "mother_druguse",
    # Family Psychopathology Aggregated
    "d_grandfather_dep", "d_grandmother_dep", "m_grandfather_dep", "m_grandmother_dep",
    "cigs_during_pregnancy_p", "alcohol_during_pregnancy_p", "weed_during_pregnancy_p",
    "cocaine_during_pregnancy_p", "heroin_during_pregnancy_p",
    "father_mania", "mother_mania", "father_trouble", "parent_hospitalized_emo", "parent_therapy_emo",
    # Ancestral-Genetic PCs/Ethnicity
    "desc_african_AFR_B", "desc_native_american_AMR_B", "desc_alaska_native_AMR_B", "desc_chinese_EAS_B",
    "desc_japanese_EAS_B", "desc_korean_EAS_B", "desc_vietnamese_EAS_B", "desc_european_EUR_B",
    "desc_asian_indian_SAS_B", "desc_other_south_asian_SAS_B", "desc_latin_B",
    *[f"pc_gene_aces{i}" for i in range(1, 33)],
    # Familial Adverse Life Events
    "g_lifeevents_ss_k", "b_lifeevents_ss_k", "b_lifeevents_affected_ss_k", "experienced_crime_p",
    "g_lifeevents_ss_p", "b_lifeevents_ss_p", "b_lifeevents_affected_ss_p",
    # Child Emotional Regulation Strategies
    "child_emotion_overwhelm_p", "child_upset_emotions_overwhelm_p",
    # Parent Other Psychopathology
    "obsessions_present_B_p", "poor_eye_contact_B_p",
    "hallucinogen_use_history_B_p", "hallucinogen_current_B_p", "sedative_hypnotic_anxiolytic_use_B_p",
    "nightmares_B_p", "parent_elevated_mood_B_p", "parent_lying_B_p",
    "parent_social_anxiety_disorder_B_p", "parent_bulimia_B_p",
    "parent_attention_D_p", "parent_aggressive_D_p", "parent_external_D_p", "parent_antisocial_D_p",
    "parent_hyperactive_D_p", "parent_somatic_problems_D_p", "parent_avoidant_person_D_p",
    "parent_personal_strength_D_p",
    # Child Mood Issues
    "enjoys_little_p", "sad_p", "suicidal_p", "guilty_p", "withdrawn_p",
    # Parent Drug Use (prenatal)
    "prescriptionmed_pregnancy_p", "cigs_before_pregnancy_p", "alcohol_before_pregnancy_p",
    "weed_before_pregnancy_p", "cocaine_before_pregnancy_p", "heroin_before_pregnancy_p",
    "drugs_before_pregnancy_p", "drinksperweek_during_pregnancy_p", "drugs_during_pregnancy_p",
    "caffeine_during_pregnancy_p",
]

# -- T4: Year 4 (n=459) ---
t4_variables = [
    # Child Cognitive Task Outcomes
    "tb_picvocab", "tb_list", "tb_picture", "tb_reading", "lmt_efficiency",
    "gd_safebets", "gd_riskybets", "strange_ideas_p",
    # Child Delta
    "delta_anxdisord_D_p", "delta_not_liked_p", "delta_doesnt_get_along_p",
    "delta_family_conflict_ss_p", "delta_family_conflict_ss_k", "delta_bad_grades",
    "delta_social_problems_D_p", "delta_somatic_problems_D_p", "delta_atschool_total_problems_ss_t",
    "delta_b_lifeevents_ss_p",
    # Child Emotional Regulation Strategies
    "emoreg_sup_ss_k", "emoreg_reapp_ss_k", "emoreg_sup_control_k", "emoreg_reapp_happy_k",
    "emoreg_sup_hide_k", "emoreg_reapp_less_bad_k", "emoreg_sup_self_k", "emoreg_reapp_thoughts_k",
    "child_selfaware_p", "child_clear_feelings_p", "child_feelings_attentive_p", "child_feelings_care_p",
    "child_feelings_know_p", "child_upset_acknowledge_p", "child_upset_angry_p", "child_upset_ashamed_p",
    "child_upset_control_p", "child_upset_bad_behavior_p", "child_upset_better_p",
    "child_upset_poor_concentrate_p", "child_upset_no_control_p", "child_upset_unproductive_p",
    "child_upset_embarrassed_p", "child_upset_bad_esteem_p", "child_upset_nothing_better_p",
    "child_upset_fixation_p", "child_upset_bad_focus_p", "child_upset_guilty_p",
    "child_upset_irritated_p", "child_upset_longtime_better_p", "child_upset_lose_control_p",
    "child_upset_out_control_p", "child_upset_longtime_p", "child_upset_weak_p",
    # Child Personality Features
    "up_negative_urgency_ss_k", "up_lackofplanning_ss_k", "up_sensationseeking_ss_k",
    "up_positiveurgency_ss_k", "up_lackperseverance_ss_k", "bis_behav_inhibition_ss_k",
    "bis_reward_responsive_ss_k", "bis_drive_ss_k", "bis_funseeking_ss_k", "enjoys_music_k",
    "loquacious_p", "bragadocious_p", "easily_jealous_p", "wishes_other_sex_p", "easily_embarrassed_p",
    "secretive_p", "perfectionist_p", "sex_orient_y",
    # Child Medical/Somatic Problems
    "medhx_p", "medhx_doctorvisit_p", "medhx_emergencyroom_p", "pain_last_month_k",
    "seriously_sick_lastyear_k", "body_aches_p", "frequent_headaches_p", "nausea_p", "eye_problems_p",
    "skin_problems_p", "frequent_stomachaches_p", "vomiting_p", "constipated_p", "bad_toilet_habits_p",
    "wets_bed_p",
    # Child Sleep Problems
    "difficulty_goingtosleep_p", "difficulty_wakingup_p", "nightmares_p", "fallsleeptime", "wakeuptime",
    "wakesleepcalc", "chronotype",
    # Child Physical Activity/Features
    "height", "weight", "puberty_k", "sex", "no_sports_activities_p", "birth_weight_p",
    # Child Technology Use
    "sphone_Interupt", "sphone_noreason", "sphone_distress", "sphone_attachment",
    # Child Social Relationship Quality
    "nonBfriends_k", "close_nonB_friends_k", "not_liked_p", "doesnt_get_along_p", "prosocial_ss_p",
    "close_boy_friends_k", "close_girl_friends_k", "peer_net_protective_ss_k",
    "peers_beh_prosocial_ss_k", "peers_beh_delinquent_ss_k", "feels_leftout_k", "not_invited_k",
    "excluded_k", "otherkids_spreadneg_rumors_k", "otherkids_gossip_k", "feels_threatned_k",
    "saysmeanthings_others_k", "otherkids_saymeanthings_k", "discrimination_ss_k",
    "feels_discriminated_k", "senses_racism_k", "doesnt_feel_accepted_k", "bullied_on_internet_k",
    "prosocial_ss_k", "relational_victimization_ss_k", "reputational_aggression_ss_k",
    "reputational_victimization_ss_k", "overt_aggression_ss_k", "overt_victimization_ss_k",
    "relational_aggression_ss_k", "feels_discriminated_teachers_k",
    "feels_discriminated_adults_not_school_k", "feels_discriminated_students_k",
    "feels_unwanted_american_society_k", "feels_discriminated_americans_k",
    # Family Dynamics & Parenting
    "p_comm_cohesion_ss", "p_comm_ctrl_ss", "p_comm_collective_efficacy_ss", "fam_fight_often_k",
    "fam_no_open_anger_k", "fam_throw_things_k", "fam_no_lose_temps_k", "fam_criticize_often_k",
    "fam_hit_each_other_k", "fam_keep_peace_k", "fam_try_one_up_k", "fam_no_raise_voices_k",
    "family_not_talk_aboutfeelings_p", "family_peaceful_p", "family_open_discussing_anything_p",
    "family_lose_temper_rare_p", "family_believe_not_raise_voice_p", "frequent_family_conflict_p",
    "family_conflict_ss_p", "family_expression_ss_p", "family_intellectual_ss_p",
    "family_activities_ss_p", "family_organisation_ss_p", "parents_argue_more_p", "family_emotionprob_p",
    "parents_divorced_p", "death_in_family_p", "family_move_p", "y_acceptance_ss_p_crpbi",
    "y_acceptance_ss_caregiver_crpbi", "family_conflict_ss_k", "parent_monitoring_ss_k",
    "marital_status", "parent_age", "sex_P", "num_brothers_p", "num_sisters_p",
    "religious_service_frequency", "relig_importance",
    # Parent Anxiety
    "parent_fearful_or_anxious_p", "parent_specific_fears_p", "parent_fear_of_bad_thoughts_p",
    "parent_worries_about_future_p", "parent_worries_about_family_p", "parent_relationship_concerns_p",
    # Parent Mood Issues
    "parent_cries_a_lot_p", "parent_lonely_p", "parent_feels_unloved_p", "parent_paranoid_p",
    "parent_feels_inferior_p", "parent_depressed_p", "parent_feels_unsuccessful_p",
    "parent_tired_no_reason_p", "parent_low_energy_p", "parent_sleep_trouble_p",
    "parent_enjoys_little_p", "parent_sudden_mood_changes_p", "parent_happy_person_p",
    # Parent Impulsivity and Behavior Regulation
    "parent_impulsive_p", "parent_risky_decisions_p", "parent_drives_too_fast_p", "parent_tardy_often_p",
    "parent_money_management_trouble_p", "parent_priority_trouble_p", "parent_behavior_changeable_p",
    "parent_hot_temper_p", "parent_attention_seeking_p", "parent_destroys_own_things_p",
    "parent_destroys_others_things_p", "parent_doesnt_finish_tasks_p", "parent_strange_behavior_p",
    "parent_illegal_behavior_p", "parent_self_harm_p", "parent_suicidal_thoughts_p",
    # Parent Social Functioning
    "parent_bad_relationships_p", "parent_bad_family_relationship_p", "parent_not_liked_by_others_p",
    "parent_friendship_trouble_p", "parent_prefers_older_people_p", "parent_associates_with_trouble_p",
    "parent_bad_opposite_sex_relationship_p", "parent_meets_family_duties_p",
    "parent_clowns_or_shows_off_p", "parent_teases_others_p", "parent_stands_up_rights_p",
    # Parent Cognitive and Attention Issues
    "parent_forgetful_p", "parent_concentration_trouble_p", "parent_confused_p",
    "parent_planning_trouble_p", "parent_doesnt_eat_well_p", "parent_not_good_at_details_p",
    "parent_obsessive_thoughts_p", "parent_repeats_acts_p", "parent_max_effort_p",
    "parent_disorganized_p", "parent_loses_things_p", "parent_decision_trouble_p",
    # Parent Personality
    "parent_bragging_p", "parent_honest_p", "parent_secretive_p", "parent_stubborn_irritable_p",
    "parent_clumsy_p", "parent_strange_thoughts_p", "parent_self_conscious_p",
    "parent_uses_opportunities_p", "parent_louder_than_others_p", "parent_yells_a_lot_p",
    "parent_shy_or_timid_p", "parent_restless_p", "parent_easily_bored_p", "parent_hyperactive_p",
    "parent_talks_too_much_p", "parent_avoids_talking_p", "parent_prefers_to_be_alone_p",
    "parent_no_guilt_p", "parent_sense_of_fairness_p", "parent_high_sleep_duration_p",
    # Parent Other
    "parent_physical_attacks_p", "parent_picks_skin_p", "parent_heart_racing_p", "parent_numbness_p",
    "parent_sees_things_p", "parent_hears_voices_p", "parent_speech_problems_p",
    "parent_opposite_sex_wish_p",
    # Parent Delta Psychopathology
    "delta_parent_sleep_trouble_p", "delta_parent_worries_about_family_p",
    "delta_parent_friendship_trouble_p", "delta_parent_poor_work_performance_p",
    "delta_parent_aches_pains_p", "delta_parent_not_liked_by_others_p",
    "delta_parent_feels_overwhelmed_p", "delta_parent_feels_unloved_p",
    "delta_parent_bad_family_relationship_p", "delta_parent_worries_about_future_p",
    "delta_parent_worries_a_lot_p", "delta_parent_depressed_p", "delta_parent_concentration_trouble_p",
    "delta_parent_stubborn_irritable_p", "delta_parent_drinks_too_much_p",
    "delta_parent_meets_family_duties_p",
    "delta_parent_planning_trouble_p", "delta_parent_bad_relationships_p", "delta_parent_drug_use_p",
    # SES & Mobility
    "parent_education", "parent_income", "struggle_food_expenses", "positive_finance_p",
    "couldnt_afford_phone", "couldnt_afford_rent_mortgage", "evicted", "gas_electric_oil_turned_off",
    # Residential Characteristics
    "neighborhood_safety_ss_p", "neighborhood_safe_y", "L_site_id",
    # Child School Dynamics
    "disobeys_at_school_k", "getalong_teachers_k", "feelsafe_at_school_k", "feels_smart_k",
    "enjoys_school_k", "grades_important_k", "school_environment_ss_k", "school_involvement_ss_k",
    "school_disengagement_ss_k", "bad_grades", "school_detension_suspension", "child_newschool_p",
    "finds_schoolboring_k",
    # Child Anxiety
    "worries_p", "clings_to_adults_p", "nervous_general_p", "nervous_twitching_p", "fears_excl_school_p",
    "fears_school_p", "fears_being_bad_p", "paranoid_p", "self_conscious_k", "anxious_fearful_k",
    # Child ADHD
    "trouble_concentrating_k", "difficulty_finishing_tasks_k", "easily_distracted_k",
    "cant_concentrate_p", "doesnt_finish_p", "hyperactive_p", "impulsive_p",
    # Child Externalizing
    "argues_p", "stubborn_p", "temper_tantrums_p", "bullies_others_p", "destroys_own_things_p",
    "destroys_others_things_p", "disobedient_home_p", "disobedient_school_p", "breaks_rules_p",
    "fights_p", "lying_p", "steals_home_p", "steals_outside_p", "threatens_others_p", "whines_p",
    "demands_attention_p", "acts_immature_k", "destroys_others_things_k", "disobeys_parents_k",
    "stubborn_k", "hot_temper_k", "parent_personal_strength_D_p",
    # Parent Drug Use
    "father_alcohol", "mother_alcohol", "father_druguse", "mother_druguse",
    # Family Psychopathology Aggregated
    "d_grandfather_dep", "d_grandmother_dep", "m_grandfather_dep", "m_grandmother_dep",
    "cigs_during_pregnancy_p", "alcohol_during_pregnancy_p", "weed_during_pregnancy_p",
    "cocaine_during_pregnancy_p", "heroin_during_pregnancy_p",
    "father_mania", "mother_mania", "father_trouble", "parent_hospitalized_emo", "parent_therapy_emo",
    # Ancestral-Genetic PCs/Ethnicity
    "desc_african_AFR_B", "desc_native_american_AMR_B", "desc_alaska_native_AMR_B", "desc_chinese_EAS_B",
    "desc_japanese_EAS_B", "desc_korean_EAS_B", "desc_vietnamese_EAS_B", "desc_european_EUR_B",
    "desc_asian_indian_SAS_B", "desc_other_south_asian_SAS_B", "desc_latin_B",
    *[f"pc_gene_aces{i}" for i in range(1, 33)],
    # Familial Adverse Life Events
    "g_lifeevents_ss_k", "b_lifeevents_ss_k", "b_lifeevents_affected_ss_k", "experienced_crime_p",
    "g_lifeevents_ss_p", "b_lifeevents_ss_p", "b_lifeevents_affected_ss_p",
    # Child Emotional Regulation Strategies
    "child_emotion_overwhelm_p", "child_upset_emotions_overwhelm_p",
    # Parent Other Psychopathology
    "parent_attention_D_p", "parent_aggressive_D_p", "parent_internal_D_p", "parent_external_D_p",
    "parent_depress_D_p", "parent_adhd_D_p", "parent_antisocial_D_p", "parent_hyperactive_D_p",
    "parent_somatic_problems_D_p", "parent_intrusive_thoughts_D_p", "parent_avoidant_person_D_p",
    "obsessions_present_B_p", "poor_eye_contact_B_p",
    "hallucinogen_use_history_B_p", "hallucinogen_current_B_p", "sedative_hypnotic_anxiolytic_use_B_p",
    "nightmares_B_p", "parent_elevated_mood_B_p", "parent_lying_B_p",
    "parent_social_anxiety_disorder_B_p", "parent_bulimia_B_p",
    # Child Mood Issues
    "enjoys_little_p", "sad_p", "suicidal_p", "guilty_p", "withdrawn_p",
    "delta_parent_somatic_problems_D_p",
    # Parent Drug Use (prenatal)
    "prescriptionmed_pregnancy_p", "cigs_before_pregnancy_p", "alcohol_before_pregnancy_p",
    "weed_before_pregnancy_p", "cocaine_before_pregnancy_p", "heroin_before_pregnancy_p",
    "drugs_before_pregnancy_p", "drinksperweek_during_pregnancy_p", "drugs_during_pregnancy_p",
    "caffeine_during_pregnancy_p",
    # Parent Delta Psychopathology
    "delta_parent_financial_trouble_p", "delta_parent_drug_days_nonmedical_p",
]

# ---
# CIRCULARITY EXCLUSION DEFINITIONS (All TPs)
# ---

# ---
# Per-target circularity exclusions for predictor-outcome content overlap.
# Each target maps to: (a) specific variables to drop from within-category
# and across-category models.
#
# Exclusion rationale: variables are removed if they (1) are subsumed in
# CBCL composite scores used as targets, (2) have construct overlap that
# would be difficult for parent raters to distinguish from the target, or
# (3) are theoretically tautological with the outcome. A full variable-
# level audit is provided in the supplementary materials (see S1.1).
# ---
# CIRCULAR EXCLUSION SYSTEM
# ---
# Prevents feature-target circularity for CBCL/ASR-derived outcomes by
# removing predictors that directly constitute, sum into, or are algebraically
# derived from the target. Two exclusion layers are applied per target:
#
# across_exclude  -- individual variable names dropped from the flat predictor pool
# within_exclude  -- category names dropped from the within-category dict entirely
#
# Items listed that are absent from the active feature pool are silently skipped,
# so these dicts can be kept full without per-timepoint bookkeeping.
# ---

# ---
# SHARED BUILDING BLOCKS
# ---

# Excluded from all internalizing composite scores which have overlapping
# subfeatures or theoretically ambiguous overlap.
_GENERAL_INTERNALIZING_EXCLUDE = {
    'sulks_p', 'fears_being_bad_p', 'nightmares_p', 'mood_fluctuations_p',
    'sleeps_more_p', 'sleeps_less_p', 'guilty_p', 'nervous_general_p', 'prosocial_ss_p',
}

# Change scores and raw precursor forms are algebraically derived from their
# parent subscale; including them allows partial reconstruction of the target.
_DELTA_INTERNALIZING_EXCLUDE = {
    'delta_internal_D_p', 'delta_anxdisord_D_p', 'delta_somatic_problems_D_p',
    'delta_depress_D_p', 'depress_D_p_raw',
    'delta_sbt_core_depression', 'sbt_core_depression_delta_t0_t3',
    'delta_worries_p',
}
_DELTA_EXTERNALIZING_EXCLUDE = {
    'delta_external_D_p', 'delta_adhd_D_p',
    'delta_aggressive_D_p', 'delta_rulebreaking_D_p',
}

# CBCL items correlated with ADHD but not formal DSM-5 scale constituents.
_ADHD_EXTRA_GENERAL = {'clumsy_p', 'disobedient_p'}

# -- CBCL syndrome scale constituents ---
_ANXDEP_SYNDROME = {
    'cries_p', 'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p',
    'perfectionist_p', 'feels_unloved_p', 'feels_inferior_p', 'nervous_general_p',
    'anxious_p', 'guilty_p', 'easily_embarrassed_p', 'suicidal_p', 'worries_p',
}
_WITHDRAWN_SYNDROME = {
    'enjoys_little_p', 'prefer_alone_p', 'doesnt_talk_p', 'secretive_p',
    'shy_p', 'low_energy_p', 'sad_p', 'withdrawn_p',
}
_SOMATIC_SYNDROME = {
    'nightmares_p', 'constipated_p', 'dizzy_p', 'overtired_p', 'body_aches_p',
    'frequent_headaches_p', 'nausea_p', 'eye_problems_p', 'skin_problems_p',
    'frequent_stomachaches_p', 'vomiting_p',
}
_AGGRESSIVE_SYNDROME = {
    'argues_p', 'bullies_others_p', 'demands_attention_p', 'destroys_own_things_p',
    'destroys_others_things_p', 'disobedient_home_p', 'disobedient_school_p',
    'fights_p', 'screams_p', 'stubborn_p', 'mood_fluctuations_p',
    'sulks_p', 'suspicious_p', 'teases_p', 'temper_tantrums_p', 'threatens_others_p',
}
_RULEBREAKING_SYNDROME = {
    'druguse_alcohol_p', 'no_guilt_p', 'breaks_rules_p', 'bad_friends_p', 'lying_p',
    'prefer_older_p', 'runs_away_p', 'sets_fires_p', 'sexual_problems_p',
    'steals_home_p', 'steals_outside_p', 'obscene_language_p', 'sexual_thoughts_p',
    'druguse_tobacco', 'truancy_p', 'whines_p', 'vandalism_p',
}
_EXTERNALIZING_THEORETICAL = {
    'doesnt_get_along_p', 'cruel_animals_p', 'easily_jealous_p',
    'disobedient_friends_p', 'plays_genitals_public_p',
    'plays_genitals_excessive_p', 'druguse_other_p',
}

# Shared ASR items excluded across parent internalizing targets; these reflect
# avoidance, somatic-anxiety, and cognitive features that co-vary with mood in the ASR.
_PARENT_GENERAL_MOOD_EXCLUDE = {
    'parent_self_conscious_p', 'parent_prefers_to_be_alone_p',
    'parent_avoids_talking_p', 'parent_shy_or_timid_p',
    'parent_high_sleep_duration_p', 'parent_heart_racing_p', 'parent_numbness_p',
    'parent_fearful_or_anxious_p', 'parent_confused_p',
}

# All SBT subtypes are downstream aggregations of CBCL items that overlap with
# any depression target; the full set must be excluded to prevent indirect
# reconstruction via a constituent composite.
_DEPRESSION_SBT = {
    'sbt_core_depression', 'sbt_core_depression_raw', 'sbt_guilt_hopelessness',
    'sbt_social_withdrawal', 'sbt_fatigue_sleep', 'sbt_somatic_depression',
    'sbt_anxiety_depression', 'sbt_emotional_dysregulation',
    'latent_class_depression', 'depress_D_p_rev', 'depress_D_p_raw',
    'top_5_depression', 'top_10_depression', 'top_5_sbt_core_depression',
    'top_10_sbt_core_depression',
}

# BDEFS and upset-response items unavailable at T0/T2; merged into across_exclude
# only at T1/T3/T4 via get_circular_exclusions(timepoint=).
_DEPRESSION_T1T3T4_EXTRA = {
    'bdefs_calm_down_p', 'bdefs_distract_upset_p',
    'child_upset_guilty_p', 'child_upset_ashamed_p', 'child_upset_bad_esteem_p',
    'child_upset_weak_p', 'child_upset_nothing_better_p', 'child_upset_fixation_p',
    'child_upset_longtime_p', 'child_upset_longtime_better_p',
    'child_upset_unproductive_p', 'child_upset_bad_focus_p',
    'child_upset_poor_concentrate_p', 'child_upset_bad_behavior_p',
}
_ANXIETY_T1T3T4_EXTRA = {
    'bdefs_calm_down_p', 'bdefs_stop_think_p',
    'child_upset_control_p', 'child_upset_no_control_p',
    'child_upset_out_control_p', 'child_upset_lose_control_p',
    'child_upset_embarrassed_p', 'child_upset_irritated_p',
}

# -- SBT subtype-specific constituent items ---
# Each set contains only the CBCL items that sum into that subtype but are NOT
# already covered by _DEPRESSION_ACROSS. sbt_core_depression and sbt_fatigue_sleep
# are fully covered by _DEPRESSION_ACROSS and therefore have no supplementary set.

_SBT_GUILT_HOPELESSNESS_ITEMS    = {'confused_p'}

_SBT_ANXIETY_DEPRESSION_ITEMS    = {
    'fears_school_p', 'nervous_general_p', 'clings_to_adults_p',
    'easily_embarrassed_p'
}               # #1: was missing; constituent of this subtype

_SBT_SOCIAL_WITHDRAWAL_ITEMS     = {
    'doesnt_get_along_p', 'not_liked_p',
    'delta_not_liked_p', 'delta_social_problems_D_p', 'delta_doesnt_get_along_p'
}  # delta items overlap with constituents

_SBT_AVOIDANCE_FEAR_ITEMS        = {
    'fears_excl_school_p', 'fears_school_p', 'clings_to_adults_p',
    'nervous_general_p', 'anxious_p', 'easily_embarrassed_p',
    'obsessions_p'
}                       # #3: was missing; constituent of this subtype

_SBT_WELL_BEING_ITEMS            = {
    'doesnt_get_along_p', 'not_liked_p',
    'delta_not_liked_p', 'delta_doesnt_get_along_p'
}  # delta items overlap with constituents

_SBT_SOMATIC_DEPRESSION_ITEMS = {
    'frequent_headaches_p', 'frequent_stomachaches_p', 'nausea_p', 'eye_problems_p',
    'skin_problems_p', 'constipated_p', 'unknown_physical_problems_p',
    'medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p',
    'pain_last_month_k', 'seriously_sick_lastyear_k',
    'body_aches_p', 'vomiting_p',
    'bad_toilet_habits_p', 'wets_bed_p',
}

_SBT_EMOTIONAL_DYSREGULATION_ITEMS = {
    'stubborn_p', 'temper_tantrums_p', 'screams_p', 'demands_attention_p',
    'argues_p', 'disobedient_home_p', 'destroys_others_things_p', 'whines_p', 'impulsive_p',
    'mood_fluctuations_p',               # also in _GENERAL_INTERNALIZING_EXCLUDE; duplicates are harmless
    'easily_jealous_p', 'prosocial_ss_p', 'doesnt_get_along_p',
}

_SBT_AGGRESSION_IRRITABILITY_ITEMS = {
    'argues_p', 'bullies_others_p', 'destroys_own_things_p',
    'destroys_others_things_p', 'disobedient_home_p', 'fights_p',
    'attacks_others_p', 'threatens_others_p',
    'breaks_rules_p', 'lying_p',         # present in flat across-category pools
    'temper_tantrums_p', 'stubborn_p', 'screams_p', 'cruel_animals_p',  # #4: were missing; constituents of this subtype
}

_SBT_ACADEMIC_COGNITIVE_ITEMS   = {
    'cant_concentrate_p', 'doesnt_finish_p', 'poor_schoolwork_p',
    'poor_schoolwork_p_reverse', 'easily_distracted_p',
    'confused_p', 'hyperactive_p', 'bad_grades_p', 'impulsive_p',
    'delta_cant_concentrate_p',          # delta item overlaps with constituent
}

_SBT_PERFECTIONISM_ITEMS        = {
    'perfectionist_p', 'poor_schoolwork_p_reverse', 'bad_grades', 'fears_being_bad_p', 'confused_p', 'easily_embarrassed_p',
}

In [ ]:
#@title Target Circularity Exclusions

# Builds the per-target circularity exclusion dictionary and defines
# get_circular_exclusions(). Removes construct-overlapping, tautological,
# and reverse-causation predictors independently for each target -- necessary
# given high covariation among CBCL items where raters may conflate related
# constructs (e.g., anxiety/depression, ADHD/externalizing). Applies to
# both within-category (category-level skips) and across-category
# (variable-level drops) analysis modes.

# ---
# Additional Standardized exclusions to remove predictors from composite targets
#
# Internalizing / Depression
# _GENERAL_INTERNALIZING_EXCLUDE     _ANXDEP_SYNDROME
# _WITHDRAWN_SYNDROME                _SOMATIC_SYNDROME
# _DELTA_INTERNALIZING_EXCLUDE       _DEPRESSION_SBT
# _DEPRESSION_T1T3T4_EXTRA           _ANXIETY_T1T3T4_EXTRA
#
# Externalizing / ADHD
# _DELTA_EXTERNALIZING_EXCLUDE       _AGGRESSIVE_SYNDROME
# _RULEBREAKING_SYNDROME             _EXTERNALIZING_THEORETICAL
# _ADHD_EXTRA_GENERAL                _ADHD_T1T3T4_EXTRA  (defined in this cell)
#
# SBT subtype item-sets
# _SBT_GUILT_HOPELESSNESS_ITEMS      _SBT_ANXIETY_DEPRESSION_ITEMS
# _SBT_SOCIAL_WITHDRAWAL_ITEMS       _SBT_SOMATIC_DEPRESSION_ITEMS
# _SBT_AVOIDANCE_FEAR_ITEMS          _SBT_PERFECTIONISM_ITEMS
# _SBT_WELL_BEING_ITEMS              _SBT_EMOTIONAL_DYSREGULATION_ITEMS
# _SBT_AGGRESSION_IRRITABILITY_ITEMS _SBT_ACADEMIC_COGNITIVE_ITEMS
#

# ---
# INDEX of Defined Targets Below
# CHILD TARGETS -- PRIMARY
# 1.  Internalizing          internal_D_p
# 2.  Depression             depress_D_p + 28 targets, SBT subtypes
# 3.  Anxiety                anxdisord_D_p
# 4.  Externalizing          external_D_p
# 5.  ADHD                   adhd_D_p, depadhd_c, asdadhd_c
# 6.  Social Problems        social_problems_D_p, not_liked_p, doesnt_get_along_p
# CHILD TARGETS -- SECONDARY
# 7.  Grade Outcomes         bad_grades
# 8.  NIH Toolbox            tb_flanker, tb_reading, tb_fluid, tb_cryst
# 9.  Thought Disorder       thought_disorder_D_p
# 10.  Child OCD              ocd_D_p
# 11.  Child Obsessions       obsessions_p
# 12.  Child Perfectionism    perfectionist_p
# 13.  Child Impulsivity      impulsive_p
# 14.  Child Compulsivity     compulsions_p
# PARENT TARGETS -- PRIMARY
# 15.  Parent Depression      parent_depress_D_p + 2 targets, delta variant
# 16.  Parent Anxiety         parent_anxdisord_D_p, parent_worries_a_lot_p, delta variant
# 17.  Parent ADHD            parent_adhd_D_p, parent_concentration_trouble_p
# PARENT TARGETS -- SECONDARY
# 18.  Parent Social Quality  parent_bad_relationships_p + 3 targets
# 19.  Parent Happiness       parent_happy_person_p
# 20.  Parent Suicidal        parent_suicidal_thoughts_p
# 21.  Parent Impulsivity     parent_impulsive_p
# 22.  Parent Substance Use   parent_tobacco_use_frequency_p, parent_drinks_frequency_p,
# parent_drunk_days_p
# OTHER TARGETS
# 23.  SES / Neighbourhood    parent_income, area_deprivation_idx
# 24.  Miscellaneous          gd_riskybets, nb_correct_nt_2back, pa_sum_k,
# bdefs_distract_upset_p, asd_ssrs_sum
# 25.  Body Weight            weight
# ---

CIRCULAR_EXCLUSIONS = {}

# ---
# Extend helper sets from Cell 1 based on full registry audit. These augment
# the original sets in-place before they are used by target entries below.
# ---

# Depression: 10 T3/T4 child_upset_* items not in original set --
# irritable/dysphoric affect, loss-of-control, and emotional-overwhelm items
# that operationalise DSM youth MDD criteria in the ERC-P instrument.
_DEPRESSION_T1T3T4_EXTRA = _DEPRESSION_T1T3T4_EXTRA | {
    'child_upset_angry_p', 'child_upset_irritated_p',
    'child_upset_embarrassed_p', 'child_upset_emotions_overwhelm_p',
    'child_upset_control_p', 'child_upset_no_control_p',
    'child_upset_lose_control_p', 'child_upset_out_control_p',
    'child_upset_acknowledge_p', 'child_upset_better_p',
}

# Anxiety: 3 T3/T4 items not in original set --
# angry/irritable arousal, avoidant acknowledgement, and emotional flooding
# are indistinguishable from anxious agitation at the parent-report level.
_ANXIETY_T1T3T4_EXTRA = _ANXIETY_T1T3T4_EXTRA | {
    'child_upset_angry_p', 'child_upset_acknowledge_p',
    'child_upset_emotions_overwhelm_p',
}

# ADHD: new T1T3T4 set -- ERC-P attention/control items and BDEFS executive-
# function items that load directly on the ADHD EF/dysregulation pathway.
_ADHD_T1T3T4_EXTRA = {
    'child_upset_bad_focus_p', 'child_upset_poor_concentrate_p',
    'child_upset_control_p', 'child_upset_no_control_p',
    'child_upset_lose_control_p', 'child_upset_out_control_p',
    'bdefs_explain_idea_p', 'bdefs_explain_pt_p', 'bdefs_explain_seq_p',
    'bdefs_inconsistant_p', 'bdefs_lazy_p', 'bdefs_process_info_p',
    'bdefs_rechannel_p', 'bdefs_consequences_p',
}

# ---
# CHILD TARGETS -- PRIMARY
# ---

# -- 1. Internalizing ---
# All three CBCL syndrome subscales constituting the internalizing broadband are
# excluded; SBT composites are excluded to prevent indirect reconstruction.
_INTERNAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE
    | _ANXDEP_SYNDROME | _WITHDRAWN_SYNDROME | _SOMATIC_SYNDROME
    | _DELTA_INTERNALIZING_EXCLUDE | _DEPRESSION_SBT
    | {'clings_to_adults_p'}                                            # cbcl_q11 Anxious/Depressed constituent (safety-net; verify _ANXDEP_SYNDROME in Cell 1)
    | {'social_problems_D_p', 'self_conscious_k', 'anxious_fearful_k', 'feels_worthless_k',
       'difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'sleep_problems_p',
       'sleeps_little_p', 'sleeps_alot_p', 'overeating_p', 'sociable_p', 'adhd_d_p',
       'parent_depress_D_p', 'bdefs_calm_down_p', 'bdefs_consequences_p', 'bdefs_distract_upset_p', 'bdefs_explain_idea_p', 'bdefs_explain_pt_p',
'bdefs_explain_seq_p', 'bdefs_impulsive_action_p', 'bdefs_inconsistant_p', 'bdefs_lazy_p', 'bdefs_process_info_p',
'bdefs_rechannel_p', 'bdefs_sense_time_p', 'bdefs_shortcuts_p', 'bdefs_stop_think_p', 'child_emotion_overwhelm_p',
'child_clear_feelings_p', 'child_feelings_attentive_p', 'child_feelings_care_p', 'child_feelings_know_p',
'child_upset_acknowledge_p', 'child_upset_angry_p', 'child_upset_ashamed_p', 'child_upset_bad_behavior_p',
'child_upset_bad_esteem_p', 'child_upset_bad_focus_p', 'child_upset_better_p', 'child_upset_control_p',
'child_upset_embarrassed_p', 'child_upset_emotions_overwhelm_p', 'child_upset_fixation_p', 'child_upset_guilty_p',
'child_upset_irritated_p', 'child_upset_longtime_better_p', 'child_upset_longtime_p', 'child_upset_lose_control_p',
'child_upset_no_control_p', 'child_upset_nothing_better_p', 'child_upset_out_control_p',
'child_upset_poor_concentrate_p', 'child_upset_unproductive_p', 'child_upset_weak_p',
       'nervous_general_p', 'paranoid_p', 'scared_dark_p', 'nervous_twitching_p'}
    | {'parent_anhedonia_B_p'}                                          # DSM MDD core criterion; subsumed by internalizing composite
)
_INTERNAL_WITHIN = {'Anxiety', 'Child Anxiety', 'ADHD', 'Child ADHD', 'Externalizing', 'Child Externalizing'}
CIRCULAR_EXCLUSIONS['internal_D_p'] = {'across_exclude': _INTERNAL_ACROSS, 'within_exclude': _INTERNAL_WITHIN}

# -- 2. Depression ---
# Covers CBCL Withdrawn and Anxious/Depressed items that operationalise DSM depression
# criteria, sleep/appetite disturbance items, child self-report mood indices, and all
# SBT composites. Anxiety category excluded within-category due to overlapping affect items.
_DEPRESSION_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _DELTA_INTERNALIZING_EXCLUDE | _DEPRESSION_SBT
    | {'social_problems_D_p', 'delta_parent_depressed_p'}
    | {'parent_anhedonia_B_p', 'parent_dysthymia_B_p'}                  # DSM MDD core criterion / depressive disorder diagnosis
    | {'enjoys_little_p', 'cries_p', 'selfharm_p', 'doesnt_eat_well_p',
       'feels_unloved_p', 'feels_inferior_p', 'guilty_p', 'overtired_p',
       'sleeps_less_p', 'sleeps_more_p', 'suicidal_p', 'low_energy_p',
       'sad_p', 'parent_wish_dead_present_B_p'}
    | {'withdrawn_p', 'prefer_alone_p', 'doesnt_talk_p',
       'secretive_p', 'shy_p', 'lonely_p'}
    | {'sleeps_little_p', 'sleeps_alot_p', 'sleep_problems_p', 'overeating_p'}
    | {'feels_worthless_k', 'anxious_fearful_k'}
    | {'difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'paranoid_p',
       'nervous_twitching_p', 'worries_p'}
    # KSADS depression-domain predictors in child feature pool
    | {'parent_depressed_mood_B_p', 'parent_suicide_past_B_p',
       'nightmares_B_p', 'ptsd_diagnosis_present_B_p'}
    # Parent mood items allowing indirect depression severity reconstruction
    | {'parent_suicidal_thoughts_p', 'parent_self_harm_p'}
    # Emotional regulation / clarity items circular with depression construct
    | {'bdefs_rechannel_p', 'child_emotion_overwhelm_p',
       'child_clear_feelings_p', 'parent_depress_D_p'}
    # Executive-function emotion-regulation items that index depression severity
    | {'bdefs_calm_down_p', 'bdefs_distract_upset_p',
       'bdefs_lazy_p', 'bdefs_stop_think_p'}
    # Child emotion regulation scale -- items directly indexing mood/affect dysregulation
    | {'child_upset_angry_p', 'child_upset_ashamed_p', 'child_upset_bad_behavior_p',
       'child_upset_bad_esteem_p', 'child_upset_bad_focus_p', 'child_upset_better_p',
       'child_upset_control_p', 'child_upset_embarrassed_p',
       'child_upset_emotions_overwhelm_p', 'child_upset_fixation_p',
       'child_upset_guilty_p', 'child_upset_irritated_p',
       'child_upset_longtime_better_p', 'child_upset_longtime_p',
       'child_upset_lose_control_p', 'child_upset_no_control_p',
       'child_upset_nothing_better_p', 'child_upset_out_control_p',
       'child_upset_poor_concentrate_p', 'child_upset_unproductive_p',
       'child_upset_weak_p', 'child_upset_acknowledge_p',
       'child_feelings_attentive_p', 'child_feelings_care_p',
       'child_feelings_know_p'}
)
_DEPRESSION_WITHIN = {'Anxiety', 'Child Anxiety'}

# Extended within_exclude for subtypes that draw on cross-domain CBCL items.
_SBT_EMODSYREG_AGGRESSION_WITHIN = _DEPRESSION_WITHIN | {'Externalizing', 'Child Externalizing'}
_SBT_ACADEMIC_WITHIN             = _DEPRESSION_WITHIN | {'ADHD', 'Child ADHD'}

_DEP_BASE = {'across_exclude': _DEPRESSION_ACROSS,
             'within_exclude': _DEPRESSION_WITHIN,
             'across_exclude_t1t3t4': _DEPRESSION_T1T3T4_EXTRA}

for target in ['depress_D_p', 'depress_D_p_rev', 'depress_update_p',
               'delta_depress_D_p', 'delta_sbt_core_depression',
               'sbt_core_depression', 'sbt_core_depression_raw',
               'top_10_sbt_core_depression', 'top_5_sbt_core_depression',
               'sbt_core_depression_delta_t0_t3', 'sbt_fatigue_sleep',
               'dep_onset_rci_2.3', 'dep_remission_rci_2.3',
               'dep_increase_2sd', 'dep_increase_1.5sd', 'dep_increase_1sd',
               'dep_decrease_2sd', 'dep_decrease_1.5sd', 'dep_decrease_1sd',
               'sbt_core_dep_onset_rci_2.3', 'sbt_core_dep_decrease_2sd',
               'sbt_core_dep_remission_rci_2.3',
               'sbt_core_dep_decrease_1.5sd', 'sbt_core_dep_increase_2sd',
               'sbt_core_dep_increase_1.5sd',
               'top_10_depression', 'top_5_depression', 'latent_class_depression',
               'enjoys_little_p', 'suicidal_p',
               'MDD_KSADS_C', 'depressed_mood_B_k', 'anhedonia_B_k',
               'parent_wish_dead_present_B_p', 'wish_dead_present_B_k',
               'hopeless_B_k', 'two_more_depression_B_k', 'ksads_1_840_t']:
    CIRCULAR_EXCLUSIONS[target] = _DEP_BASE

# --- Delta-specific overrides (depression family) ---
# The bulk loop above assigned _DEP_BASE (whose across_exclude is _DEPRESSION_ACROSS)
# to every depression target, including the two depression deltas. But
# _DEPRESSION_ACROSS does not contain the raw baseline composites used to compute
# those deltas (depress_D_p, sbt_core_depression at baseline trivially reconstruct
# the change score), nor the broadband internalizing raw composite. Override the
# two delta entries here with expanded exclusions. Non-delta depression targets
# keep the plain _DEP_BASE assigned above.
CIRCULAR_EXCLUSIONS['delta_depress_D_p'] = {
    **_DEP_BASE,
    'across_exclude': _DEPRESSION_ACROSS | {
        'depress_D_p',         # raw baseline/endpoint source of the delta
        'depress_update_p',    # depression severity update item
        'internal_D_p',        # broadband internalizing contains depress_D_p
    },
}
CIRCULAR_EXCLUSIONS['delta_sbt_core_depression'] = {
    **_DEP_BASE,
    'across_exclude': _DEPRESSION_ACROSS | {
        'depress_D_p',         # shares constituent items with sbt_core_depression
        'internal_D_p',        # broadband internalizing contains depression content
    },
}

# Per-subtype: base _DEPRESSION_ACROSS plus the subtype's unique constituent items.
CIRCULAR_EXCLUSIONS['sbt_guilt_hopelessness']        = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_GUILT_HOPELESSNESS_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_anxiety_depression']        = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_ANXIETY_DEPRESSION_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_social_withdrawal']         = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_SOCIAL_WITHDRAWAL_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_somatic_depression']        = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_SOMATIC_DEPRESSION_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_avoidance_fear']            = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_AVOIDANCE_FEAR_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_perfectionism_achievement'] = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_PERFECTIONISM_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_well_being']                = {**_DEP_BASE, 'across_exclude': _DEPRESSION_ACROSS | _SBT_WELL_BEING_ITEMS}
CIRCULAR_EXCLUSIONS['sbt_emotional_dysregulation']   = {**_DEP_BASE,
    'across_exclude': _DEPRESSION_ACROSS | _SBT_EMOTIONAL_DYSREGULATION_ITEMS
                      | {'breaks_rules_p'},
    'within_exclude': _SBT_EMODSYREG_AGGRESSION_WITHIN}
CIRCULAR_EXCLUSIONS['sbt_aggression_irritability']   = {**_DEP_BASE,
    'across_exclude': _DEPRESSION_ACROSS | _SBT_AGGRESSION_IRRITABILITY_ITEMS
                      | {'impulsive_p', 'not_liked_p', 'doesnt_get_along_p'},
    'within_exclude': _SBT_EMODSYREG_AGGRESSION_WITHIN}
CIRCULAR_EXCLUSIONS['sbt_academic_cognitive']        = {**_DEP_BASE,
    'across_exclude': (
        _DEPRESSION_ACROSS
        | _SBT_ACADEMIC_COGNITIVE_ITEMS
        | {
            'goal_continuity_p',
            'bad_grades',
            'delta_bad_grades',
            'disobedient_school_p',
            'disobedient_home_p',
            'breaks_rules_p',
            'grades_dropped',
            'up_lackperseverance_ss_k',
            'trouble_concentrating_k',
            'strange_ideas_p',
        }
    ),
    'within_exclude': _SBT_ACADEMIC_WITHIN,
}

# -- 3. Anxiety ---
# CBCL Anxious/Depressed items plus specific-fear, somatic-anxiety, and avoidance
# items excluded. KSADS construct measures and ABCD PK items excluded where present.
_ANXIETY_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _WITHDRAWN_SYNDROME | _DELTA_INTERNALIZING_EXCLUDE
    | {'social_problems_D_p', 'self_conscious_k', 'anxious_fearful_k'}
    | {'clings_to_adults_p', 'fears_excl_school_p', 'fears_school_p',
       'nervous_general_p', 'anxious_p', 'worries_p', 'secretive_p'}
    | {'fears_being_bad_p', 'paranoid_p', 'nervous_twitching_p', 'perfectionist_p',
       'easily_embarrassed_p', 'Doesnt_finish_p', 'nightmares_p'}
    | {'social_fear_present_PK', 'scared_dark_p', 'fear_becoming_obese_present_PK',
       'obsessions_present_B_p', 'difficulty_goingtosleep_p', 'difficulty_wakingup_p'}
    | {'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p',
       'parent_excessive_worry_B_p'}                                       # direct anxiety constructs / KSADS predictors
    # Emotional regulation / clarity items circular with anxiety construct
    | {'bdefs_rechannel_p', 'child_emotion_overwhelm_p',
       'child_clear_feelings_p', 'parent_depress_D_p'}
    # Construct overlap ambiguity
    | {'bdefs_calm_down_p', 'bdefs_distract_upset_p',
       'bdefs_stop_think_p'}
    # Child emotion regulation scale -- items directly indexing mood/affect dysregulation
    | {'child_upset_angry_p', 'child_upset_ashamed_p', 'child_upset_bad_behavior_p',
       'child_upset_bad_esteem_p', 'child_upset_bad_focus_p', 'child_upset_better_p',
       'child_upset_control_p', 'child_upset_embarrassed_p',
       'child_upset_emotions_overwhelm_p', 'child_upset_fixation_p',
       'child_upset_guilty_p', 'child_upset_irritated_p',
       'child_upset_longtime_better_p', 'child_upset_longtime_p',
       'child_upset_lose_control_p', 'child_upset_no_control_p',
       'child_upset_nothing_better_p', 'child_upset_out_control_p',
       'child_upset_poor_concentrate_p', 'child_upset_unproductive_p',
       'child_upset_weak_p', 'child_upset_acknowledge_p',
       'child_feelings_attentive_p', 'child_feelings_care_p',
       'child_feelings_know_p'}
)
CIRCULAR_EXCLUSIONS['anxdisord_D_p'] = {
    'across_exclude': _ANXIETY_ACROSS,
    'within_exclude': {'Anxiety', 'Child Anxiety'},
    'across_exclude_t1t3t4': _ANXIETY_T1T3T4_EXTRA,
}
# Child Only Deltas -- inherits anxiety base
CIRCULAR_EXCLUSIONS['delta_anxdisord_D_p'] = {
    # Base _ANXIETY_ACROSS excludes delta-counterpart and broadband-delta composites
    # but NOT the raw baseline composite used to compute the delta. For a delta
    # target, the raw composite (at either timepoint) trivially reconstructs the
    # target, so it must be explicitly excluded. Same for broadband internalizing
    # (which contains anxdisord_D_p) and cross-informant parent_anxdisord_D_p.
    'across_exclude': _ANXIETY_ACROSS | {
        'anxdisord_D_p',          # raw baseline/endpoint source of the delta
        'parent_anxdisord_D_p',   # cross-informant anxiety composite (highly correlated)
        'internal_D_p',           # broadband internalizing contains anxdisord_D_p
    },
    'within_exclude': {'Anxiety', 'Child Anxiety'},
    'across_exclude_t1t3t4': _ANXIETY_T1T3T4_EXTRA,
}

# -- 4. Externalizing ---
# three CBCL externalizing syndrome scales are excluded; ABCD youth
# and parent report analogues are excluded alongside CBCL-derived broadband deltas.
_EXTERNAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _DELTA_EXTERNALIZING_EXCLUDE
    | _AGGRESSIVE_SYNDROME | _RULEBREAKING_SYNDROME | _EXTERNALIZING_THEORETICAL
    | {'acts_immature_k', 'destroys_others_things_k', 'disobeys_parents_k',
       'stubborn_k', 'hot_temper_k', 'disobeys_at_school_k',
       'secretive_p', 'social_problems_D_p', 'easily_offended_p', 'bdefs_lazy_p', 'child_clear_feelings_p',
       'disagreeable_p', 'blames_others_p', 'prosocial_ss_p', 'not_critical_others_p',
'parent_depress_D_p', 'bdefs_calm_down_p', 'bdefs_consequences_p', 'bdefs_distract_upset_p', 'bdefs_explain_idea_p', 'bdefs_explain_pt_p',
'bdefs_explain_seq_p', 'bdefs_impulsive_action_p', 'bdefs_inconsistant_p', 'bdefs_lazy_p', 'bdefs_process_info_p',
'bdefs_rechannel_p', 'bdefs_sense_time_p', 'bdefs_shortcuts_p', 'bdefs_stop_think_p', 'child_emotion_overwhelm_p',
'child_clear_feelings_p', 'child_feelings_attentive_p', 'child_feelings_care_p', 'child_feelings_know_p',
'child_upset_acknowledge_p', 'child_upset_angry_p', 'child_upset_ashamed_p', 'child_upset_bad_behavior_p',
'child_upset_bad_esteem_p', 'child_upset_bad_focus_p', 'child_upset_better_p', 'child_upset_control_p',
'child_upset_embarrassed_p', 'child_upset_emotions_overwhelm_p', 'child_upset_fixation_p', 'child_upset_guilty_p',
'child_upset_irritated_p', 'child_upset_longtime_better_p', 'child_upset_longtime_p', 'child_upset_lose_control_p',
'child_upset_no_control_p', 'child_upset_nothing_better_p', 'child_upset_out_control_p',
'child_upset_poor_concentrate_p', 'child_upset_unproductive_p', 'child_upset_weak_p',
       'doesnt_finish_p', 'adhd_d_p', 'sociable_p', 'bis_behav_inhibition_ss_k'}
    | {'delta_doesnt_get_along_p', 'delta_not_liked_p', 'not_liked_p'}
    | {'conduct_physical_fights_present_PK', 'bragadocious_p', 'easily_jealous_p', 'impulsive_p'}
    | {'demands_attention_p'}
    | {'hyperactive_p'}
)
CIRCULAR_EXCLUSIONS['external_D_p'] = {
    'across_exclude': _EXTERNAL_ACROSS,
    'within_exclude': {'Externalizing', 'Child Externalizing'},
}

# -- 5. ADHD ---
# All nine CBCL ADHD DSM-5 scale items excluded; ODD/CD-adjacent items excluded
# due to inclusion in the scale's hyperactive-impulsive subscale definition.
# Child self-report attention/impulse items excluded for construct-level overlap.
_ADHD_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _ADHD_EXTRA_GENERAL | _DELTA_EXTERNALIZING_EXCLUDE
    | {'acts_young_p', 'doesnt_finish_p', 'cant_concentrate_p', 'hyperactive_p',
       'impulsive_p', 'easily_distracted_p', 'loquacious_p', 'goal_continuity_p'}
    | {'stubborn_p', 'temper_tantrums_p', 'social_problems_D_p',
       'disobedient_home_p', 'disobedient_school_p', 'breaks_rules_p', 'poor_schoolwork_p'}
    | {'bad_grades', 'grades_dropped', 'delta_cant_concentrate_p',
       'delta_bad_grades', 'bdefs_impulsive_action_p', 'bdefs_sense_time_p'}
    | {'trouble_concentrating_k', 'difficulty_finishing_tasks_k', 'easily_distracted_k',
       'acts_immature_k', 'destroys_others_things_k', 'disobeys_parents_k',
       'up_lackperseverance_ss_k', 'up_lackofplanning_ss_k', 'stubborn_k',
       'hot_temper_k', 'disobeys_at_school_k'}
)
for target in ['adhd_D_p', 'depadhd_c', 'asdadhd_c']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _ADHD_ACROSS,
        'within_exclude': {'ADHD', 'Child ADHD', 'Externalizing', 'Child Externalizing'},
        'across_exclude_t1t3t4': _ADHD_T1T3T4_EXTRA,
    }

# Child Only Deltas -- needs T1T3T4 extras for child_upset + BDEFS items
CIRCULAR_EXCLUSIONS['delta_adhd_D_p'] = {
    # See delta_anxdisord_D_p note: base _ADHD_ACROSS excludes delta composites
    # but not the raw baseline ones. Add raw source + broadband externalizing +
    # the combined-class targets (depadhd_c, asdadhd_c) that include ADHD status.
    'across_exclude': _ADHD_ACROSS | {
        'adhd_D_p',               # raw baseline/endpoint source of the delta
        'external_D_p',           # broadband externalizing contains adhd_D_p
        'depadhd_c',              # latent class: combined depression+ADHD
        'asdadhd_c',              # latent class: combined ASD+ADHD
    },
    'within_exclude': {'ADHD', 'Child ADHD', 'Externalizing', 'Child Externalizing'},
    'across_exclude_t1t3t4': _ADHD_T1T3T4_EXTRA,
}

# -- 6. Social Problems ---
# Targets (social_problems_D_p, not_liked_p, doesnt_get_along_p) shares
# enough item overlap that all three are mutually excluded across the group.
_SOCIAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE
    | {'social_problems_D_p', 'delta_social_problems_D_p', 'not_liked_p', 'regarded_weird_p', 'difficulty_making_friends_p',
       'doesnt_get_along_p', 'delta_not_liked_p', 'delta_doesnt_get_along_p'}
    | {'clings_to_adults_p', 'lonely_p', 'easily_jealous_p', 'paranoid_p', 'Doesnt_finish_p',
       'gets_hurt_p', 'teased_p', 'clumsy_p', 'prefer_younger_p', 'speech_problems_p'}
    | {'shy_p', 'prefer_alone_p', 'disobedient_friends_p', 'self_conscious_k'}
)
for target in ['social_problems_D_p', 'not_liked_p', 'doesnt_get_along_p',
               # Child Only Deltas
               'delta_social_problems_D_p']:
    CIRCULAR_EXCLUSIONS[target] = {'across_exclude': _SOCIAL_ACROSS, 'within_exclude': set()}

# ---
# CHILD TARGETS -- SECONDARY
# ---

# -- 7. Grade Outcomes ---
# bad_grades and grades_dropped are near-identical constructs; ADHD dropped given
# parents likely proxy with attention features.
# T1/T3/T4 extras: BDEFS executive-function items and ERC-P concentration items
# that operationalise the cognitive substrate of academic performance at later waves.
_BAD_GRADES_T1T3T4_EXTRA = {
    'bdefs_explain_idea_p', 'bdefs_explain_pt_p', 'bdefs_explain_seq_p',   # verbal/working-memory executive
    'bdefs_inconsistant_p', 'bdefs_lazy_p', 'bdefs_process_info_p',        # effortful-control executive
    'bdefs_rechannel_p', 'bdefs_sense_time_p', 'bdefs_shortcuts_p',        # time-management / task-switching executive
    'child_upset_bad_focus_p', 'child_upset_poor_concentrate_p',           # ERC-P concentration under distress
    'child_upset_unproductive_p',                                          # task-completion under distress
    'finds_schoolboring_k', 'enjoys_school_k',                             # school disengagement proxies
}
CIRCULAR_EXCLUSIONS['bad_grades'] = {
    'across_exclude': {
        'bad_grades', 'grades_dropped', 'delta_bad_grades',
        'poor_schoolwork_p', 'skips_school_p', 'cant_concentrate_p', 'disobedient_school_p', 'doesnt_finish_p',
        'goal_continuity_p', 'atschool_total_problems_ss_t', 'atschool_attention_ss_t', 'delta_cant_concentrate_p',
        'school_detension_suspension',
        'atschool_external_ss_t', 'atschool_internal_ss_t', 'delta_atschool_total_problems_ss_t',
    },
    'within_exclude': set(),
    'across_exclude_t1t3t4': _BAD_GRADES_T1T3T4_EXTRA,
}
# Child Only Deltas (shared dict reference -- inherits T1T3T4 extras automatically)
CIRCULAR_EXCLUSIONS['delta_bad_grades'] = CIRCULAR_EXCLUSIONS['bad_grades']

# -- 8. NIH Toolbox Composite Scores ---
# tb_total aggregates all TB scores.
# Constituent scores are mutually excluded within each composite.
# tasks measuring largely same general constructs excluded from each other
CIRCULAR_EXCLUSIONS['tb_flanker'] = {
    'across_exclude': {
        'tb_flanker', 'tb_fluid', 'tb_fluid_grouped', 'tb_total', 'lmt_mrt',
        'sst_mean_ssrt', 'sst_median_ssrt', 'sst_ssrt_int_est', 'sst_ssrt_mean_est',
        'tb_pattern',  # Pattern Comparison Processing Speed (NIH) -- shared latent construct
    },
    'within_exclude': set(),
}
CIRCULAR_EXCLUSIONS['tb_reading'] = {
    # tb_reading and tb_picvocab are the two constituent scores of tb_cryst.
    'across_exclude': {'tb_reading', 'tb_picvocab', 'tb_cryst', 'tb_total'},
    'within_exclude': set(),
}
CIRCULAR_EXCLUSIONS['tb_fluid'] = {
    # tb_fluid = tb_flanker + tb_pattern + tb_cardsort + tb_picture + tb_list
    'across_exclude': {
        'tb_fluid', 'tb_fluid_grouped', 'tb_total', 'tb_flanker', 'tb_cryst',
        'tb_pattern', 'tb_cardsort', 'tb_picture', 'tb_list', 'lmt_mrt', 'tb_reading', 'tb_picvocab',
    },
    'within_exclude': set(),
}
CIRCULAR_EXCLUSIONS['tb_cryst'] = {
    # tb_cryst = tb_reading + tb_picvocab
    'across_exclude': {'tb_cryst', 'tb_total', 'tb_reading', 'tb_picvocab'},
    'within_exclude': set(),
}

# -- 8b. Verbal Episodic Memory (RAVLT) ---
# RAVLT short- and long-delay recall (total / repetitions / intrusions) all
# index the same auditory-verbal LTM construct, so all six subscales are
# mutually excluded. tb_picture (NIH Picture Sequence Memory) shares the
# episodic-memory construct via a different modality and is also excluded,
# along with composite parents (tb_fluid, tb_total).
CIRCULAR_EXCLUSIONS['ravlt_l_total'] = {
    'across_exclude': {
        'ravlt_l_total', 'ravlt_l_repitition', 'ravlt_l_intrusions',
        'ravlt_s_total', 'ravlt_s_repitition', 'ravlt_s_intrusions',
        'tb_picture',                       # NIH Picture Sequence Memory -- shared episodic-memory construct
        'tb_fluid', 'tb_fluid_grouped',     # composite parent
        'tb_total',                         # top-level composite
    },
    'within_exclude': set(),
}

# -- 8c. Visuospatial Mental Rotation (LMT) ---
# All LMT subscales (accuracy, efficiency, RT variants, correct-trial counts)
# index the same mental-rotation construct and are mutually excluded.
# Matrix Reasoning (mr_*) is the closest visuospatial-reasoning analogue and
# tb_pattern (NIH Pattern Comparison) shares a processing-speed component, so
# both families are excluded.
CIRCULAR_EXCLUSIONS['lmt_efficiency'] = {
    'across_exclude': {
        'lmt_efficiency', 'lmt_accuracy',
        'lmt_correct_nt', 'lmt_correct_mrt', 'lmt_incorrect_mrt', 'lmt_mrt',
        'mr_total', 'mr_matrix', 'mr_serial',  # Matrix Reasoning -- shared visuospatial-reasoning construct
        'tb_pattern',                          # NIH Pattern Comparison -- shared processing-speed component
    },
    'within_exclude': set(),
}

# -- 8d. Stop Signal Task (SST) ---
# All SST behavioural summaries and computational-model parameters index
# the same response-inhibition construct (or its variability/uncertainty),
# so the full SST family is mutually excluded. tb_flanker (NIH inhibitory
# control via conflict monitoring) shares the response-inhibition construct
# via a different paradigm and is excluded, along with the fluid composite
# parents and lmt_mrt (shared RT/speed confound).
CIRCULAR_EXCLUSIONS['sst_mean_ssrt'] = {
    'across_exclude': {
        # Primary SSRT estimates
        'sst_mean_ssrt', 'sst_median_ssrt',
        'sst_ssrt_mean_est', 'sst_ssrt_int_est',
        # QC flag
        'sst_acceptable_performance',
        # Computational-model parameters (BEESTS / general SST modelling)
        'sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS',
        'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
        'sst_kappa0', 'sst_xM', 'sst_sM', 'sst_bG', 'sst_pp',
        'sst_factor1', 'sst_factor2', 'sst_factor3',
        # Trial-level central-tendency summaries
        'sst_mean_PDR', 'sst_median_PDR',
        'sst_mean_SD', 'sst_median_SD',
        'sst_mean_SDr', 'sst_median_SDr',
        'sst_mean_PDRg', 'sst_median_PDRg',
        'sst_mean_betaS', 'sst_median_betaS',
        'sst_mean_bS2', 'sst_median_bS2',
        'sst_mean_absdelta', 'sst_median_absdelta',
        # Variance / SD / CV summaries
        'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV',
        'sst_sdRV', 'sst_sdSD', 'sst_sdCV',
        'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV',
        'sst_pdrgV', 'sst_pdrgSD', 'sst_pdrgCV',
        'sst_betasV', 'sst_betasSD', 'sst_betasCV',
        'sst_absdeltaRV', 'sst_absdeltaSD', 'sst_absdeltaCV',
        'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV',
        # Cross-paradigm overlap
        'tb_flanker',                       # NIH inhibitory control -- shared response-inhibition construct
        'tb_fluid', 'tb_fluid_grouped',     # composite parent (contains tb_flanker)
        'tb_total',                         # top-level composite
        'lmt_mrt',                          # shared RT / processing-speed confound
    },
    'within_exclude': set(),
}

# -- 8e. EN-back Working Memory ---
# All n-back behavioural metrics (0-/2-back accuracy, RT, and 2-back signal-
# detection indices across positive/neutral/negative valence conditions) index
# the same emotional-WM construct and are mutually excluded. tb_list (NIH
# List Sorting Working Memory) shares the WM construct via a non-emotional
# paradigm and is also excluded, along with composite parents.
CIRCULAR_EXCLUSIONS['nb_correct_nt_2back'] = {
    'across_exclude': {
        # 0-back / 2-back number correct & mean RT (overall + by valence)
        'nb_correct_nt_2back', 'nb_correct_mrt_2back',
        'nb_correct_nt',         'nb_correct_mrt',
        'nb_correct_nt_pos',     'nb_correct_mrt_pos',
        'nb_correct_nt_neutral', 'nb_correct_mrt_neutral',
        'nb_correct_nt_neg',     'nb_correct_mrt_neg',
        # 2-back signal-detection metrics by valence
        'nb2_accuracy_pos', 'nb2_resp_bias_pos', 'nb2_D_prime_pos',
        'nb2_accuracy_neg', 'nb2_resp_bias_neg', 'nb2_D_prime_neg',
        # QC flag
        'nb_acceptable_performance',
        # Cross-paradigm overlap
        'tb_list',                          # NIH List Sorting Working Memory -- shared WM construct
        'tb_fluid', 'tb_fluid_grouped',     # composite parent (contains tb_list)
        'tb_total',                         # top-level composite
    },
    'within_exclude': set(),
}

# -- 9. Thought Disorder ---
# All nine CBCL Thought Problems syndrome items excluded; this subscale has no
# overlapping items with internalizing/externalizing syndromes, so within_exclude is empty.
CIRCULAR_EXCLUSIONS['thought_disorder_D_p'] = {
    'across_exclude': {
        'thought_disorder_D_p', 'delta_thought_disorder_D_p', 'strange_behavior_p',
        'strange_ideas_p', 'hears_voices_p', 'sees_things_p',
        'repetitive_acts_p', 'hoarding_p', 'daydreams_p',
        'stares_blankly_p', 'sleep_walks_p', 'nervous_twitching_p',
        'picks_things_p', 'plays_genitals_public_p',
    },
    'within_exclude': set(),
}

# -- 10. Child OCD ---
# Provisional Testing Target
# CBCL OCD scale comprises cbcl_q09 (obsessions), cbcl_q66 (compulsions), cbcl_q83 (hoarding).
# Thought Problems broadband excluded as its constituent items substantially overlap with OCD.
# Body-focused repetitive behaviour and tic-adjacent items excluded for redundnacy/overlap
#
CIRCULAR_EXCLUSIONS['ocd_D_p'] = {
    'across_exclude': (
        {'ocd_D_p', 'delta_ocd_D_p'}
        | {'obsessions_p', 'repetitive_acts_p', 'hordes_p'}              # CBCL OCD scale items (q09, q66, q83)
        | {'thought_disorder_D_p', 'delta_thought_disorder_D_p'}         # broadband containing OCD scale
        | {'picks_skin_p', 'nervous_twitching_p', 'picks_things_p',      # body-focused repetitive / tic-adjacent
           'blank_stare_p', 'strange_ideas_p', 'strange_behavior_p'}
        | {'obsessions_present_B_p'}                                      # KSADS direct-construct item
        | _DEPRESSION_SBT
    ),
    'within_exclude': set(),
}

# -- 11. Child Obsessions ---
# cbcl_q09 ("Can't get his/her mind off certain thoughts; obsessions") is a
# constituent of the CBCL OCD scale and Thought Problems broadband. OCD co-
# constituents (compulsions_p, hordes_p) and body-focused repetitive/tic-adjacent
# items excluded for phenotypic proximity.
# hyperactive_p excluded: parent raters cannot cleanly distinguish restless/driven
# behaviour from obsessive mental preoccupation.
CIRCULAR_EXCLUSIONS['obsessions_p'] = {
    'across_exclude': (
        {'obsessions_p', 'delta_obsessions_p'}
        | {'compulsions_p', 'repetitive_acts_p', 'hordes_p'}             # OCD scale co-constituents (q66, q83)
        | {'ocd_D_p', 'delta_ocd_D_p'}                                   # OCD composite sums this item
        | {'thought_disorder_D_p', 'delta_thought_disorder_D_p'}         # Thought Problems broadband contains q09
        | {'nervous_twitching_p', 'picks_skin_p', 'picks_things_p',
           'strange_behavior_p', 'strange_ideas_p', 'blank_stare_p'}     # body-focused repetitive / tic-adjacent
        | {'obsessions_present_B_p'}                                      # KSADS direct-construct item
        | {'hyperactive_p'}                                               # theoretical overlap: restless/driven vs obsessive preoccupation
        | {'parent_picks_skin_p'}                                         # body-focused repetitive behaviour proxy (parent analogue)
        | {'narrow_interests_p', 'concentration_on_parts_p',             # ASD cognitive style: rater conflation with OCD/obsessive style
           'sensory_sensitivity_p', 'avoids_eyecontact_p',
           'bad_conversational_flow_p', 'regarded_weird_p'}
        | {'mania_7up_ss_k'}                                              # racing/driven cognition indistinguishable from obsessive ideation
        | _DEPRESSION_SBT
    ),
    'within_exclude': set(),
}

# -- 12. Child Perfectionism ---
# cbcl_q32 ("Feels he/she has to be perfect") is a constituent of the CBCL
# Anxious/Depressed syndrome (→ anxdisord_D_p, internal_D_p) and the SBT
# Perfectionism/Achievement subtype. Co-constituents of sbt_perfectionism
# (fears_being_bad_p, poor_schoolwork_p_reverse, confused_p) are excluded to
# prevent indirect reconstruction. Self-evaluative items (easily_embarrassed_p,
# self_conscious_k) excluded for construct overlap with evaluative concern
# perfectionism.
# hyperactive_p excluded: parent raters cannot cleanly distinguish restless/driven
# behaviour from perfectionistic rigidity.
CIRCULAR_EXCLUSIONS['perfectionist_p'] = {
    'across_exclude': (
        {'perfectionist_p', 'delta_perfectionist_p'}
        | {'fears_being_bad_p', 'confused_p', 'worries_p',
           'poor_schoolwork_p', 'poor_schoolwork_p_reverse'}              # sbt_perfectionism_achievement co-constituents
        | {'sbt_perfectionism_achievement'}                               # SBT subtype that sums this item directly
        | {'easily_embarrassed_p', 'self_conscious_k'}                    # evaluative-concern perfectionism overlap
        | {'anxdisord_D_p', 'delta_anxdisord_D_p'}                       # Anxious/Depressed syndrome contains this item
        | {'internal_D_p', 'delta_internal_D_p'}                         # Internalizing broadband (via anxdep subscale)
        | {'hyperactive_p'}                                               # theoretical overlap: driven/restless vs perfectionistic rigidity
        | {'mania_7up_ss_k'}                                              # perfectionistic striving vs manic grandiosity indistinguishable at item level
        | _DEPRESSION_SBT
    ),
    'within_exclude': {'Anxiety', 'Child Anxiety'},
}

# -- 13. Child Impulsivity ---
# cbcl_q41 ("Acts without thinking") is a constituent of the CBCL ADHD DSM-5 scale
# (hyperactive/impulsive subscale). The exclusion set is intentionally narrower than
# _ADHD_ACROSS: acts_young_p and goal_continuity_p are excluded because they belong
# to the same ADHD subscale grouping and share variance with impulsive dyscontrol.
# _DEPRESSION_SBT excluded because sbt_emotional_dysregulation aggregates
# impulse-control failure items, creating indirect circularity.
_IMPULSIVE_T1T3T4_EXTRA = {
    'bdefs_impulsive_action_p', 'bdefs_stop_think_p', 'bdefs_shortcuts_p',
    'child_upset_bad_behavior_p', 'child_upset_lose_control_p',
    'child_upset_out_control_p', 'child_upset_no_control_p',
}
CIRCULAR_EXCLUSIONS['impulsive_p'] = {
    'across_exclude': (
        {'impulsive_p', 'delta_impulsive_p'}
        | {'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'easily_distracted_p',
           'loquacious_p', 'acts_young_p', 'goal_continuity_p'}
        | {'adhd_D_p', 'delta_adhd_D_p', 'external_D_p',
           'delta_external_D_p', 'depadhd_c', 'asdadhd_c'}
        | {'demands_attention_p', 'temper_tantrums_p', 'argues_p',
           'screams_p', 'breaks_rules_p', 'no_guilt_p',
           'stubborn_p', 'disobedient_home_p', 'disobedient_school_p'}
        | {'social_problems_D_p', 'poor_schoolwork_p', 'bad_grades', 'grades_dropped',
           'delta_cant_concentrate_p', 'delta_bad_grades',
           'bdefs_impulsive_action_p', 'bdefs_sense_time_p'}
        | {'trouble_concentrating_k', 'difficulty_finishing_tasks_k',
           'easily_distracted_k', 'acts_immature_k',
           'destroys_others_things_k', 'disobeys_parents_k',
           'up_lackperseverance_ss_k', 'up_lackofplanning_ss_k',
           'stubborn_k', 'hot_temper_k', 'disobeys_at_school_k'}
        | _DELTA_EXTERNALIZING_EXCLUDE | _DEPRESSION_SBT
    ),
    'within_exclude': {'ADHD', 'Child ADHD', 'Externalizing', 'Child Externalizing'},
    'across_exclude_t1t3t4': _IMPULSIVE_T1T3T4_EXTRA,
}

# -- 14. Child Compulsivity ---
# cbcl_q66 ("Repeats certain acts over and over; compulsions") is a CBCL OCD scale
# constituent. repetitive_acts_p is excluded as a likely dataset alias for the same
# item. Thought Problems broadband is excluded because it sums q66 directly.
# Body-focused repetitive and tic-adjacent items excluded for phenotypic proximity.
# hyperactive_p excluded: parent raters cannot cleanly distinguish restless/driven
# behaviour from compulsive repetition.
# T1/T3/T4 extras: ERC-P emotional-fixation/control items and BDEFS executive-function
# items that operationalise perseverative/stuck cognition at the later waves.
_COMPULSIONS_T1T3T4_EXTRA = {
    'child_upset_fixation_p',                                             # emotional fixation/perseveration
    'child_upset_longtime_p', 'child_upset_longtime_better_p',            # prolonged emotional stuck state
    'child_upset_nothing_better_p',                                        # inability to disengage from distress
    'bdefs_stop_think_p', 'bdefs_rechannel_p',                            # executive inhibition / shifting
    'bdefs_consequences_p',                                                # repetitive behaviour despite aversive consequences
}
CIRCULAR_EXCLUSIONS['compulsions_p'] = {
    'across_exclude': (
        {'compulsions_p', 'delta_compulsions_p'}
        | {'repetitive_acts_p'}                                           # dataset alias for cbcl_q66
        | {'obsessions_p', 'impulsive_p', 'hordes_p'}                    # CBCL OCD scale co-constituents (q09, q83)
        | {'ocd_D_p', 'delta_ocd_D_p'}                                   # OCD composite sums this item directly
        | {'thought_disorder_D_p', 'delta_thought_disorder_D_p'}         # Thought Problems broadband contains q66
        | {'nervous_twitching_p', 'picks_skin_p', 'picks_things_p',
           'strange_behavior_p', 'strange_ideas_p', 'blank_stare_p'}     # body-focused repetitive / tic-adjacent
        | {'obsessions_present_B_p'}                                      # KSADS direct-construct item
        | {'goal_continuity_p', 'up_lackperseverance_ss_k', 'up_lackofplanning_ss_k'}  # stickiness/perseveration proxies
        | {'hyperactive_p'}                                               # theoretical overlap: restless/driven vs compulsive repetition
        | {'parent_picks_skin_p'}                                         # body-focused repetitive behaviour proxy (parent analogue)
        | {'narrow_interests_p', 'concentration_on_parts_p',             # ASD cognitive style: rater conflation with compulsive/OCD style
           'sensory_sensitivity_p', 'avoids_eyecontact_p',
           'bad_conversational_flow_p', 'regarded_weird_p'}
        | _DEPRESSION_SBT
    ),
    'within_exclude': set(),
    'across_exclude_t1t3t4': _COMPULSIONS_T1T3T4_EXTRA,
}

# ---
# SHARED BATCH-WIDE EXCLUSIONS  (Parent Social/Impulsivity/Compulsivity batch)
# ---
# Applied to all 5 targets in the Parent Social/Impulsivity/Compulsivity
# bundle (parent_avoidant_person_D_p, parent_not_liked_by_others_p,
# parent_impulsive_p, parent_repeats_acts_p, parent_obsessive_thoughts_p).
#
# These broad ASR composites and the mood-swing item are removed from every
# predictor pool in this batch even where item-level circularity is absent.
# Rationale: composite-subsumption / reconstruction-via-subtraction risk
# (Internalizing + Externalizing + Other = Total Problems), construct-overlap
# at the broadband level, and Mood Swings as a transdiagnostic affect-
# dysregulation confound that cuts across all 5 target dimensions.
_PARENT_SOCIAL_IMPCOMP_BATCH_EXCLUDE = {
    # ASR Externalizing broadband + sub-syndromes
    'parent_external_D_p', 'parent_aggressive_D_p', 'parent_antisocial_D_p',
    'parent_hyperactive_D_p',
    # ASR Attention/ADHD composites
    'parent_attention_D_p', 'parent_adhd_D_p',
    # ASR Depression Syndrome composite (parent self-report)
    'parent_depress_D_p', 'parent_depressed_p',
    # ASR 87 Mood Swings -- transdiagnostic affect-dysregulation confound
    'parent_sudden_mood_changes_p',
    # Delta derivatives (future-proof; no-op if not present in predictor pool)
    'delta_parent_external_D_p', 'delta_parent_aggressive_D_p',
    'delta_parent_antisocial_D_p', 'delta_parent_hyperactive_D_p',
    'delta_parent_attention_D_p', 'delta_parent_adhd_D_p',
    'delta_parent_depress_D_p', 'delta_parent_sudden_mood_changes_p',
}

# ---
# PARENT TARGETS -- PRIMARY
# ---

# -- 15. Parent Depression ---
# ASR internalizing broadband and subscale composites excluded; ASR items that
# operationalise DSM MDD criteria (appetite, sleep, energy, anhedonia, hopelessness)
# are individually excluded. Change scores for mood and somatic subscales included.
_PARENT_DEP_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _PARENT_GENERAL_MOOD_EXCLUDE
    | {'parent_depressed_p', 'parent_doesnt_eat_well_p', 'parent_depress_D_p',
       'parent_self_harm_p', 'parent_suicidal_thoughts_p'}
    | {'parent_anxdep_D_p', 'parent_internal_D_p', 'parent_external_D_p',
       'parent_somatic_problems_D_p', 'parent_depressed_mood_B_p'}       # ASR broadband/subscale composites
    | {'parent_anhedonia_B_p', 'parent_dysthymia_B_p'}                   # DSM MDD core criterion / depressive disorder diagnosis
    | {'parent_paranoid_p'}                                               # paranoid ideation co-occurs with psychotic depression; rater conflation
    | {'parent_enjoys_little_p', 'parent_low_energy_p', 'parent_tired_no_reason_p', 'parent_avoidant_person_D_p',
       'parent_attention_D_p', 'parent_cries_a_lot_p', 'parent_lonely_p', 'parent_aggressive_D_p', 'parent_stubborn_irritable_p',
       'parent_feels_unsuccessful_p', 'parent_happy_person_p', 'parent_obsessive_thoughts_p', 'parent_restless_p',
       'parent_avoidant_person_D_p', 'parent_worries_about_future_p', 'parent_planning_trouble_p',
       'parent_personal_strength_D_p', 'parent_worries_a_lot_p', 'parent_decision_trouble_p', 'delta_parent_stubborn_irritable_p', 'parent_secretive_p',
       'parent_feels_inferior_p', 'parent_sudden_mood_changes_p', 'parent_antisocial_D_p', 'parent_hyperactive_D_p', 'parent_adhd_D_p',
       'parent_therapy_emo', 'family_emotionprob_p', 'parent_strange_thoughts_p', 'parent_intrusive_thoughts_D_p', 'delta_parent_worries_about_future_p',
       'parent_self_conscious_p', 'parent_clumsy_p', 'parent_sleep_trouble_p'}
    | {'delta_parent_depressed_p', 'delta_parent_feels_overwhelmed_p',
       'delta_parent_worries_a_lot_p', 'delta_parent_internal_D_p', 'tb_picvocab',
       'delta_parent_somatic_problems_D_p', 'delta_parent_stubborn_irritable_p',
       'delta_parent_planning_trouble_p', 'delta_parent_sleep_trouble_p'}
)
_PARENT_DEP_WITHIN = {'Parent Mood Issues', 'Parent Anxiety', 'Parent Anxiety and Fear-Related Issues'}

for target in ['parent_depress_D_p', 'parent_depressed_p', 'delta_parent_depress_D_p', 'delta_parent_depressed_D_p',
               'parent_dep_onset_rci_2.3', 'parent_dep_onset_rci_1.96',
               'parent_dep_remission_rci_2.3', 'parent_dep_remission_rci_1.96',
               'parent_dep_increase_2sd', 'parent_dep_increase_1.5sd',
               'parent_dep_decrease_2sd', 'parent_dep_decrease_1.5sd',
               'top_10_depression_parent', 'top_5_depression_parent',
               # Parent Only Deltas -- mood-domain change scores
               'delta_parent_sleep_trouble_p', 'delta_parent_feels_unloved_p']:
    CIRCULAR_EXCLUSIONS[target] = {'across_exclude': _PARENT_DEP_ACROSS, 'within_exclude': _PARENT_DEP_WITHIN}

# delta_parent_depressed_p gets additional SST exclusion (sst_bG identified as
# circular predictor for depression change scores via inhibitory-control confound).
CIRCULAR_EXCLUSIONS['delta_parent_depressed_p'] = {
    'across_exclude': _PARENT_DEP_ACROSS | {'sst_bG'},
    'within_exclude': _PARENT_DEP_WITHIN,
}

# -- 16. Parent Anxiety ---
# ASR anxiety subscale items excluded individually; obsessive-compulsive and
# avoidance items included due to their role in generalized/social anxiety constructs.
_PARENT_ANX_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _PARENT_GENERAL_MOOD_EXCLUDE
    | {'parent_anxdisord_D_p', 'parent_anxdep_D_p',
       'parent_internal_D_p', 'parent_sleep_trouble_p'}
    | {'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_depress_D_p', 'parent_restless_p',
       'parent_fear_of_bad_thoughts_p', 'parent_worries_about_future_p', 'delta_parent_depressed_p',
       'parent_worries_about_family_p', 'parent_worries_a_lot_p', 'parent_avoidant_person_D_p',
       'parent_avoidant_person_D_p', 'parent_aggressive_D_p', 'parent_happy_person_p', 'parent_stubborn_irritable_p',
       'parent_therapy_emo', 'family_emotionprob_p', 'parent_intrusive_thoughts_D_p', 'parent_personal_strength_D_p',
       'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_relationship_concerns_p'}
    | {'parent_obsessive_thoughts_p', 'parent_repeats_acts_p'}
    | {'parent_social_anxiety_disorder_B_p', 'parent_intrusive_thoughts_D_p'}  # direct anxiety disorder construct measures
    | {'parent_paranoid_p'}                                               # fearful/paranoid ideation indistinguishable from severe anxiety
    | {'delta_parent_worries_a_lot_p', 'delta_parent_worries_about_future_p',
       'delta_parent_worries_about_family_p', 'delta_parent_feels_overwhelmed_p', 'delta_parent_sleep_trouble_p'}
)
_PARENT_ANX_WITHIN = {'Parent Anxiety', 'Parent Anxiety and Fear-Related Issues', 'Parent Mood Issues'}

for target in ['parent_anxdisord_D_p', 'parent_worries_a_lot_p', 'delta_parent_worries_a_lot_p',
               # Parent Only Deltas -- anxiety-domain change scores
               'delta_parent_worries_about_family_p', 'delta_parent_worries_about_future_p']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _PARENT_ANX_ACROSS,
        'within_exclude': _PARENT_ANX_WITHIN,
    }

# parent_anxdisord_D_p-specific additions: avoidant coping, inferiority, and depressed
# affect are excluded as ASR items with construct overlap that parent raters cannot
# readily distinguish from anxiety disorder symptomatology (see S1.1 audit).
CIRCULAR_EXCLUSIONS['parent_anxdisord_D_p']['across_exclude'] = (
    _PARENT_ANX_ACROSS
    | {'parent_avoidant_person_D_p', 'parent_feels_inferior_p', 'parent_depressed_p'}
)

# -- 17. Parent ADHD ---
# Both inattentive and hyperactive/impulsive ASR subscale items are excluded, along
# with behavioural-consequence items (risky decisions, time management) that are
# functionally dependent on executive dysregulation rather than independent predictors.
_PARENT_ADHD_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE
    | {'parent_adhd_D_p', 'parent_attention_D_p', 'parent_hyperactive_D_p'}
    | {'parent_concentration_trouble_p', 'parent_forgetful_p',
       'parent_planning_trouble_p', 'parent_not_good_at_details_p',
       'parent_max_effort_p', 'parent_disorganized_p', 'parent_depress_D_p', 'parent_internal_D_p',
       'parent_loses_things_p', 'parent_decision_trouble_p',
       'parent_priority_trouble_p', 'parent_doesnt_finish_tasks_p'}
    | {'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_restless_p',
       'parent_impulsive_p', 'parent_intrusive_thoughts_D_p', 'parent_easily_bored_p', 'parent_tardy_often_p'}
    | {'parent_stubborn_irritable_p', 'parent_behavior_changeable_p', 'parent_strange_behavior_p',
       'parent_antisocial_D_p', 'family_organisation_ss_p',
       'parent_money_management_trouble_p', 'parent_personal_strength_D_p',
       'parent_attention_seeking_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p'}
    | {'delta_parent_concentration_trouble_p', 'delta_parent_poor_work_performance_p',
       'delta_parent_planning_trouble_p', 'delta_parent_stubborn_irritable_p'}
    | {'parent_aggressive_D_p', 'parent_clumsy_p', 'parent_impulsive_p',  # ASR items with construct overlap
       'parent_confused_p', 'parent_external_D_p'}                        # ASR externalizing broadband aggregates ADHD items
    | {'parent_hot_temper_p', 'parent_yells_a_lot_p'}                    # emotional dysregulation / ADHD emotional impulsivity subdomain
    | {'parent_destroys_own_things_p', 'parent_destroys_others_things_p'}  # downstream ADHD impulse-dyscontrol consequences
    | {'parent_elevated_mood_B_p'}                                        # mania vs ADHD hyperactivity diagnostic ambiguity
    | {'parent_sees_things_p', 'parent_hears_voices_p'}                  # psychosis items misclassified in cognitive/attention category
)
for target in ['parent_adhd_D_p', 'parent_concentration_trouble_p',
               # Parent Only Deltas -- ADHD-domain change score
               'delta_parent_concentration_trouble_p']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _PARENT_ADHD_ACROSS,
        'within_exclude': {'Parent Cognitive and Attention Issues',
                           'Parent Impulsivity and Behavior Regulation'},
    }

# ---
# PARENT TARGETS -- SECONDARY
# ---

# -- 18. Parent Social Quality ---
_PARENT_SOCIAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE
    | {'parent_bad_relationships_p', 'parent_bad_family_relationship_p',
       'parent_not_liked_by_others_p', 'parent_friendship_trouble_p', 'parent_external_D_p',
       'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p',
       'parent_enjoys_little_p', 'parent_feels_unloved_p', 'parent_depress_D_p', 'parent_internal_D_p',
       'parent_clowns_or_shows_off_p', 'parent_teases_others_p', 'parent_antisocial_D_p',
       'parent_stands_up_rights_p', 'parent_associates_with_trouble_p', 'parent_avoidant_person_D_p',
       'parent_therapy_emo', 'family_emotionprob_p', 'parent_intrusive_thoughts_D_p', 'parent_aggressive_D_p',
       'parent_strange_thoughts_p', 'parent_prefers_older_people_p'}
    | {'parent_prefers_to_be_alone_p', 'parent_shy_or_timid_p',
       'parent_avoids_talking_p', 'parent_speech_problems_p'}
    | {'delta_parent_bad_relationships_p', 'delta_parent_bad_family_relationship_p',
       'delta_parent_not_liked_by_others_p', 'parent_personal_strength_D_p',
       'delta_parent_friendship_trouble_p', 'delta_parent_meets_family_duties_p'}
    | _PARENT_SOCIAL_IMPCOMP_BATCH_EXCLUDE                                  # batch-wide broad composites + mood-swings
)
_PARENT_SOCIAL_WITHIN = {'Parent Social Functioning',
                          'Parent Interpersonal Relationships and Social Functioning'}

for target in ['parent_bad_relationships_p', 'parent_not_liked_by_others_p',
               'parent_friendship_trouble_p', 'parent_bad_family_relationship_p',
               # Parent Only Deltas -- social-domain change scores
               'delta_parent_not_liked_by_others_p', 'delta_parent_bad_relationships_p']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _PARENT_SOCIAL_ACROSS,
        'within_exclude': _PARENT_SOCIAL_WITHIN,
    }

# -- 19. Parent Happiness / Wellbeing ---
# parent_happy_person_p is an inverse-mood item; parent depression and internalizing
# composites are excluded as they represent the same latent dimension in reverse.
_PARENT_HAPPY_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _PARENT_GENERAL_MOOD_EXCLUDE
    | {'parent_happy_person_p', 'parent_enjoys_little_p',
       'parent_doesnt_eat_well_p', 'parent_therapy_emo', 'parent_internal_D_p',
       'family_emotionprob_p', 'parent_strange_thoughts_p', 'parent_avoidant_person_D_p',
       'parent_self_conscious_p', 'parent_intrusive_thoughts_D_p', 'parent_self_harm_p'}
    | {'parent_depressed_p', 'parent_internal_D_p', 'parent_depress_D_p',
       'parent_anxdep_D_p', 'parent_somatic_problems_D_p'}
    | {'delta_parent_depressed_p', 'parent_personal_strength_D_p',
       'delta_parent_feels_overwhelmed_p', 'delta_parent_internal_D_p',
       'delta_parent_somatic_problems_D_p'}
)
CIRCULAR_EXCLUSIONS['parent_happy_person_p'] = {
    'across_exclude': _PARENT_HAPPY_ACROSS,
    'within_exclude': {'Parent Mood Issues', 'Parent Anxiety', 'Parent Anxiety and Fear-Related Issues'},
}

# -- 20. Parent Suicidal Thoughts ---
_PARENT_SUICIDAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _PARENT_GENERAL_MOOD_EXCLUDE
    | {'parent_suicidal_thoughts_p', 'parent_self_harm_p',
       'parent_depressed_p', 'parent_feels_unloved_p', 'parent_depress_D_p',
       'parent_feels_overwhelmed_p', 'parent_therapy_emo', 'parent_intrusive_thoughts_D_p',
       'family_emotionprob_p', 'parent_strange_thoughts_p', 'parent_fear_of_bad_thoughts_p',
       'parent_self_conscious_p', 'parent_doesnt_eat_well_p',
       'parent_sudden_mood_changes_p', 'parent_obsessive_thoughts_p',
       'parent_aggressive_D_p', 'parent_avoidant_person_D_p',
       'parent_internal_D_p', 'parent_external_D_p', 'parent_anxdep_D_p',
       'parent_depressed_mood_B_p', 'parent_happy_person_p',
       'delta_parent_internal_D_p', 'parent_tired_no_reason_p',
       'parent_personal_strength_D_p', 'parent_confused_p'}
    | {'delta_parent_depressed_p', 'delta_parent_feels_overwhelmed_p', 'delta_parent_feels_unloved_p'}
    | {'parent_suicide_past_B_p'}                                         # past suicidal act is near-equivalent to current ideation (criterion 1/2)
    | {'parent_dysthymia_B_p', 'parent_anhedonia_B_p'}                   # depressive disorder features causally upstream of suicidal ideation
    | {'parent_paranoid_p'}                                               # paranoid ideation co-present with suicidality; rater conflation
)
CIRCULAR_EXCLUSIONS['parent_suicidal_thoughts_p'] = {
    'across_exclude': _PARENT_SUICIDAL_ACROSS,
    'within_exclude': {'Parent Mood Issues', 'Parent Anxiety', 'Parent Anxiety and Fear-Related Issues'},
}

# -- 21. Parent Impulsivity ---
# Intentionally narrower than parent_adhd_D_p: inattentive-subscale items
# (concentration, planning, forgetfulness) are retained because they do not
# directly constitute impulsivity. Only the hyperactive/impulsive subscale, its
# behavioural-consequence items, and the ADHD broadband composites that aggregate
# this item are excluded.
CIRCULAR_EXCLUSIONS['parent_impulsive_p'] = {
    'across_exclude': (
        {'parent_impulsive_p', 'delta_parent_impulsive_p'}
        | {'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_restless_p',
           'parent_concentration_trouble_p', 'parent_external_D_p', 'parent_aggressive_D_p',
           'parent_easily_bored_p', 'parent_tardy_often_p'}              # ASR hyperactive/impulsive subscale co-constituents
        | {'parent_risky_decisions_p', 'parent_personal_strength_D_p',
           'parent_drives_too_fast_p', 'parent_money_management_trouble_p', 'parent_intrusive_thoughts_D_p',
           'parent_attention_seeking_p'}                                  # downstream impulsivity consequences
        | {'parent_stubborn_irritable_p', 'parent_behavior_changeable_p',
           'delta_parent_stubborn_irritable_p'}                           # emotional dysregulation overlap
        | {'parent_adhd_D_p', 'parent_hyperactive_D_p'}                  # ASR composites that aggregate this item
        | {'parent_hot_temper_p', 'parent_yells_a_lot_p',                # emotional/behavioural impulsivity proxies
           'parent_louder_than_others_p'}
        | {'parent_destroys_own_things_p', 'parent_destroys_others_things_p',  # impulse-dyscontrol consequences
           'parent_illegal_behavior_p'}
        | {'parent_no_guilt_p'}                                           # antisocial/ADHD emotional blunting overlap
        | {'parent_strange_behavior_p'}                                   # disinhibited acts: impulsivity vs thought disorder (conservative)
        | _PARENT_SOCIAL_IMPCOMP_BATCH_EXCLUDE                              # batch-wide broad composites + mood-swings
    ),
    'within_exclude': {'Parent Impulsivity and Behavior Regulation'},
}

# -- 22. Parent Substance Use ---
# ASR substance use frequency items. Tobacco and alcohol are mutually excluded
# across each other's entries: they occupy the same Parent Substance Use feature
# category, and their high co-occurrence introduces near-circularity in within-category
# models even absent a direct item-composite relationship.
_SUBSTANCE_SHARED = {
    'parent_druguse_other_p', 'parent_rulebreaking_D_p',
    'parent_external_D_p', 'parent_aggressive_D_p', 'parent_intrusive_thoughts_D_p',
    'delta_parent_rulebreaking_D_p', 'delta_parent_external_D_p',
}
_SUBSTANCE_WITHIN = {'Parent Drug Use', 'Parent Externalizing'}           # updated category names

CIRCULAR_EXCLUSIONS['parent_tobacco_use_frequency_p'] = {
    'across_exclude': (
        {'parent_tobacco_use_frequency_p', 'cigs_during_pregnancy_p', 'delta_parent_tobacco_use_frequency_p'}
        | {'parent_tobacco_use_p'}                                        # binary tobacco use item -- near-circular with frequency
        | {'parent_drinks_frequency_p', 'delta_parent_drinks_frequency_p'}
        | _SUBSTANCE_SHARED
    ),
    'within_exclude': _SUBSTANCE_WITHIN,
}
CIRCULAR_EXCLUSIONS['parent_drinks_frequency_p'] = {
    'across_exclude': (
        {'parent_drinks_frequency_p', 'delta_parent_drinks_frequency_p'}
        | {'parent_tobacco_use_frequency_p', 'delta_parent_tobacco_use_frequency_p'}
        | {'parent_drinks_too_much_p', 'parent_drunk_days_p',             # alcohol quantity/severity items -- near-circular with frequency
           'parent_alcohol_excess_p', 'parent_alcohol_freq_p', 'parent_alcohol_days_drunk_p'}
        | _SUBSTANCE_SHARED
    ),
    'within_exclude': _SUBSTANCE_WITHIN,
}
CIRCULAR_EXCLUSIONS['parent_drunk_days_p'] = {
    'across_exclude': (
        {'parent_drunk_days_p'}
        | {'parent_drinks_frequency_p', 'delta_parent_drinks_frequency_p'}
        | {'parent_tobacco_use_frequency_p', 'delta_parent_tobacco_use_frequency_p'}
        | {'parent_drinks_too_much_p', 'parent_alcohol_excess_p',         # alcohol severity items -- near-circular with days drunk
           'parent_alcohol_freq_p', 'parent_alcohol_days_drunk_p'}
        | _SUBSTANCE_SHARED
    ),
    'within_exclude': _SUBSTANCE_WITHIN,
}

# ---
# OTHER TARGETS
# ---

# -- 22b. Parent Avoidant Personality (ASR DSM Composite) ---
# parent_avoidant_person_D_p is the ASR DSM-oriented Avoidant Personality scale
# -- the closest available analog of the CBCL Social Problems syndrome at the
# parent self-report measurement level. Its item constituents span social
# inhibition, self-consciousness, social withdrawal, and feelings of inferiority.
# Exclusions cover (a) the constituent items, (b) ASR Internalizing/Withdrawn
# syndrome composites that subsume it, (c) social-functioning items that index
# the same latent dimension, and (d) KSADS social-anxiety diagnostic proxies.
_PARENT_AVOIDANT_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _PARENT_GENERAL_MOOD_EXCLUDE
    | {'parent_avoidant_person_D_p', 'delta_parent_avoidant_person_D_p'}
    # ASR DSM composites that subsume or substantially overlap with Avoidant Personality
    | {'parent_internal_D_p', 'parent_anxdep_D_p', 'parent_anxdisord_D_p',
       'parent_personal_strength_D_p', 'parent_intrusive_thoughts_D_p',
       'parent_somatic_problems_D_p', 'parent_depress_D_p',
       'parent_depressed_mood_B_p'}
    # ASR Avoidant Personality scale primary item constituents
    | {'parent_self_conscious_p',                                          # ASR 71 Self-conscious / easily embarrassed
       'parent_shy_or_timid_p',                                            # ASR 75 Shy or timid
       'parent_feels_inferior_p',                                          # ASR 35 Feels worthless or inferior
       'parent_avoids_talking_p',                                          # ASR 65 Refuses to talk
       'parent_prefers_to_be_alone_p',                                     # ASR 42 Prefers to be alone
       'parent_relationship_concerns_p',                                   # ASR 113 Worries about relationships
       'parent_fearful_or_anxious_p'}                                      # ASR 50 Too fearful or anxious
    # ASR Withdrawn syndrome co-constituents (composite subsumption + construct overlap)
    | {'parent_bad_relationships_p',                                       # ASR 25
       'parent_bad_opposite_sex_relationship_p',                           # ASR 30
       'parent_not_liked_by_others_p',                                     # ASR 48
       'parent_friendship_trouble_p',                                      # ASR 67
       'parent_secretive_p',                                               # ASR 69
       'parent_enjoys_little_p'}                                           # ASR 60
    # KSADS social-anxiety diagnostic proxies
    | {'parent_social_anxiety_disorder_B_p'}
    # Adaptive functioning / reverse-coded social wellbeing
    | {'parent_happy_person_p',
       'parent_meets_family_duties_p', 'parent_stands_up_rights_p',
       'parent_uses_opportunities_p'}
    # Treatment-seeking / family environment
    | {'parent_therapy_emo', 'family_emotionprob_p', 'parent_strange_thoughts_p'}
    # Delta derivatives
    | {'delta_parent_internal_D_p', 'delta_parent_not_liked_by_others_p',
       'delta_parent_friendship_trouble_p', 'delta_parent_bad_relationships_p',
       'delta_parent_meets_family_duties_p',
       'delta_parent_bad_opposite_sex_relationship_p'}
    | _PARENT_SOCIAL_IMPCOMP_BATCH_EXCLUDE                                  # batch-wide broad composites + mood-swings
    | _DEPRESSION_SBT
)
CIRCULAR_EXCLUSIONS['parent_avoidant_person_D_p'] = {
    'across_exclude': _PARENT_AVOIDANT_ACROSS,
    'within_exclude': {'Parent Mood Issues', 'Parent Anxiety',
                       'Parent Anxiety and Fear-Related Issues',
                       'Parent Social Functioning',
                       'Parent Interpersonal Relationships and Social Functioning',
                       'Parent Personality',
                       'Parent Other Psychopathology',
                       'Parent Psychopathology Composite'},
}

# -- 23. SES / Neighbourhood ---
# parent_income and area_deprivation_idx are bidirectionally confounded at the
# census-tract level; each is excluded from the other's predictor pool.
CIRCULAR_EXCLUSIONS['parent_income']        = {'across_exclude': {'parent_income', 'area_deprivation_idx'}, 'within_exclude': set()}
CIRCULAR_EXCLUSIONS['area_deprivation_idx'] = {'across_exclude': {'area_deprivation_idx', 'parent_income'},  'within_exclude': set()}

# -- 24. Miscellaneous Single-Item Targets ---
CIRCULAR_EXCLUSIONS['gd_riskybets'] = {'across_exclude': {'gd_riskybets'}, 'within_exclude': set()}

CIRCULAR_EXCLUSIONS['nb_correct_nt_2back'] = {
    'across_exclude': {
        'nb_correct_nt_2back',
        'nb_correct_nt', 'nb_correct_nt_pos', 'nb_correct_nt_neg', 'nb_correct_nt_neutral',
        'nb_correct_mrt_2back', 'nb_correct_mrt', 'nb_correct_mrt_pos',
        'nb_correct_mrt_neg', 'nb_correct_mrt_neutral',
        'nb2_accuracy_pos', 'nb2_accuracy_neg', 'nb2_D_prime_pos', 'nb2_D_prime_neg',
        'nb2_resp_bias_pos', 'nb2_resp_bias_neg',
    },
    'within_exclude': set(),
}

CIRCULAR_EXCLUSIONS['pa_sum_k'] = {'across_exclude': {'pa_sum_k'}, 'within_exclude': set()}

# bdefs_distract_upset_p is only present in the T3 predictor pool;
# bdefs_calm_down_p and bdefs_stop_think_p reflect overlapping emotion-regulation strategies.
CIRCULAR_EXCLUSIONS['bdefs_distract_upset_p'] = {
    'across_exclude': {'bdefs_distract_upset_p', 'bdefs_calm_down_p', 'bdefs_stop_think_p'},
    'within_exclude': set(),
}

# asd_ssrs_sum was missing -- without this entry SSRS subscale items
# could circularly predict the SSRS composite.
CIRCULAR_EXCLUSIONS['asd_ssrs_sum'] = {
    'across_exclude': {'asd_ssrs_sum'},
    'within_exclude': set(),
}

# -- 25. Body Weight ---
# height, waist, and birth_weight_p excluded as near-collinear body-size indicators.
CIRCULAR_EXCLUSIONS['weight'] = {
    'across_exclude': {
        'weight', 'height', 'waist', 'birth_weight_p',
        'weight_grouped', 'sex', 'bad_diet_p',
    },
    'within_exclude': set(),
}

# -- 26. RCI Threshold Aliases (1.96 ↔ 2.3) ---
# RCI-1.96 and RCI-2.3 variants measure the same construct at different confidence
# thresholds; they share identical exclusion sets.
for _t, _b in [('dep_onset_rci_1.96',              'dep_onset_rci_2.3'),
               ('dep_onset_rci_1.7',                'dep_onset_rci_2.3'),
               ('dep_remission_rci_1.96',           'dep_remission_rci_2.3'),
               ('dep_remission_rci_1.7',            'dep_remission_rci_2.3'),
               ('sbt_core_dep_onset_rci_1.96',      'sbt_core_dep_onset_rci_2.3'),
               ('sbt_core_dep_onset_rci_1.7',       'sbt_core_dep_onset_rci_2.3'),
               ('sbt_core_dep_remission_rci_1.96',  'sbt_core_dep_remission_rci_2.3'),
               ('sbt_core_dep_remission_rci_1.7',   'sbt_core_dep_remission_rci_2.3')]:
    CIRCULAR_EXCLUSIONS[_t] = CIRCULAR_EXCLUSIONS[_b].copy()

# ---
# 27. CHILD KSADS CLASSIFICATION TARGETS (19 targets)
# ---
# Targets already inheriting _DEP_BASE above (section 2):
# depressed_mood_B_k, anhedonia_B_k, hopeless_B_k,
# two_more_depression_B_k, ksads_1_840_t, MDD_KSADS_C
# ---

# -- 27a. Child Suicidality (KSADS) ---

_CHILD_SUICIDALITY_ACROSS = (
    _DEPRESSION_ACROSS
    | {'parent_suicidal_thoughts_p', 'parent_self_harm_p'}
)

for target in ['suicidal_ideation_present_B_k', 'suicide_attempt_present_B_k', 'suicide_attempt_past_B_k']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _CHILD_SUICIDALITY_ACROSS,
        'within_exclude': _DEPRESSION_WITHIN,
        'across_exclude_t1t3t4': _DEPRESSION_T1T3T4_EXTRA,
    }

# -- 27b. Child Fatigue (KSADS) ---

CIRCULAR_EXCLUSIONS['fatigue_present_B_k'] = {
    'across_exclude': (
        _DEPRESSION_ACROSS
        | {'parent_tired_no_reason_p', 'parent_low_energy_p'}
        | {'chronotype', 'fallsleeptime', 'wakeuptime', 'wakesleepcalc'}
        | {'no_sports_activities_p', 'fitbit_sedentary_mins', 'fitbit_resting_hr'}
    ),
    'within_exclude': _DEPRESSION_WITHIN,
    'across_exclude_t1t3t4': _DEPRESSION_T1T3T4_EXTRA,
}

# -- 27c. Elevated Mood (KSADS) ---

CIRCULAR_EXCLUSIONS['elevated_mood_B_k'] = {
    'across_exclude': (
        _DEPRESSION_ACROSS
        | {'hyperactive_p', 'impulsive_p', 'loquacious_p',
           'demands_attention_p', 'temper_tantrums_p'}
        | {'bragadocious_p', 'easily_jealous_p',
           'up_positiveurgency_ss_k', 'up_sensationseeking_ss_k',
           'bis_funseeking_ss_k', 'bis_drive_ss_k'}
        | {'mania_7up_ss_k'}
        | {'parent_elevated_mood_B_p', 'parent_hyperactive_D_p', 'parent_restless_p',
           'parent_behavior_changeable_p', 'parent_sudden_mood_changes_p'}
    ),
    'within_exclude': {'Anxiety', 'Child Anxiety', 'ADHD', 'Child ADHD'},
    'across_exclude_t1t3t4': _DEPRESSION_T1T3T4_EXTRA,
}

# -- 27d. Child Anxiety-domain (KSADS) ---

CIRCULAR_EXCLUSIONS['excessive_worry_B_k'] = {
    'across_exclude': _ANXIETY_ACROSS,
    'within_exclude': {'Anxiety', 'Child Anxiety'},
    'across_exclude_t1t3t4': _ANXIETY_T1T3T4_EXTRA,
}

CIRCULAR_EXCLUSIONS['GAD_present'] = {
    'across_exclude': (
        _ANXIETY_ACROSS
        | {'body_aches_p', 'frequent_headaches_p', 'frequent_stomachaches_p'}
        | {'difficulty_goingtosleep_p', 'sleep_problems_p'}
    ),
    'within_exclude': {'Anxiety', 'Child Anxiety'},
    'across_exclude_t1t3t4': _ANXIETY_T1T3T4_EXTRA,
}

_SOCIAL_ANXIETY_CHILD_EXTRA = {
    'shy_p', 'prefer_alone_p', 'lonely_p', 'poor_eye_contact_B_p',
    'difficulty_making_friends_p', 'doesnt_feel_accepted_k',
    'parent_shy_or_timid_p', 'parent_prefers_to_be_alone_p', 'parent_avoids_talking_p',
}

for target in ['socially_anxious_B_k', 'social_anxiety_disorder_B_k']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _ANXIETY_ACROSS | _SOCIAL_ANXIETY_CHILD_EXTRA,
        'within_exclude': {'Anxiety', 'Child Anxiety'},
        'across_exclude_t1t3t4': _ANXIETY_T1T3T4_EXTRA,
    }

# -- 27e. Child OCD (KSADS) ---

CIRCULAR_EXCLUSIONS['OCD_present'] = {
    'across_exclude': (
        CIRCULAR_EXCLUSIONS['ocd_D_p']['across_exclude']
        | {'parent_obsessive_thoughts_p', 'parent_repeats_acts_p', 'perfectionist_p'}
    ),
    'within_exclude': set(),
}

# -- 27f. Child Sleep Problems (KSADS) ---

CIRCULAR_EXCLUSIONS['sleep_problem_B_k'] = {
    'across_exclude': (
        _DEPRESSION_ACROSS
        | {'nightmares_p', 'nightmares_B_p', 'parent_sleep_problem_B_p'}
        | {'parent_sleep_trouble_p', 'parent_high_sleep_duration_p'}
        | {'parent_tired_no_reason_p', 'parent_low_energy_p'}
        | {'chronotype', 'fallsleeptime', 'wakeuptime', 'wakesleepcalc'}
        | {'fitbit_sedentary_mins', 'fitbit_resting_hr'}
    ),
    'within_exclude': {'Anxiety', 'Child Anxiety'},
    'across_exclude_t1t3t4': _DEPRESSION_T1T3T4_EXTRA,
}

# -- 27g. Child Lying (KSADS) ---

CIRCULAR_EXCLUSIONS['lying_B_k'] = {
    'across_exclude': (
        _EXTERNAL_ACROSS
        | {'parent_lying_B_p', 'parent_antisocial_D_p'}
        | {'no_guilt_p', 'parent_no_guilt_p', 'parent_honest_p'}
        | {'school_detension_suspension'}
    ),
    'within_exclude': {'Externalizing', 'Child Externalizing'},
}

# -- 27h. Child Eating Disorders (KSADS) ---

_EATING_ACROSS = (
    {'anorexia_B_k', 'bulimia_B_k'}
    | {'doesnt_eat_well_p', 'overeating_p'}
    | {'nausea_p', 'vomiting_p', 'frequent_stomachaches_p', 'constipated_p'}
    | {'somatic_problems_D_p', 'delta_somatic_problems_D_p'}
    | {'fear_becoming_obese_present_PK', 'weight', 'waist', 'bad_diet_p'}
    | {'height', 'saliva_DHEA', 'saliva_testosterone', 'puberty_k'}
    | {'perfectionist_p', 'fears_being_bad_p'}
    | {'total_calories', 'protein_intake', 'carbohydrate_intake', 'fiber_intake',
       'sodium_intake', 'potassium_intake', 'total_sugar', 'saturated_fat',
       'fruit_intake', 'vegetable_intake', 'sugary_beverage_freq',
       'protein_sources_intake', 'legume_intake', 'added_sugar',
       'dairy_intake', 'whole_grain_intake'}
    | {'parent_bulimia_B_p', 'parent_doesnt_eat_well_p'}
)

for target in ['anorexia_B_k', 'bulimia_B_k']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _EATING_ACROSS,
        'within_exclude': {'Diet/Nutrition'},
    }

# ---
# 28. PARENT KSADS CLASSIFICATION TARGETS (15 targets)
# ---

# -- 28a. Parent Depression-domain (KSADS) ---

_PARENT_DEP_KSADS_ACROSS = (
    _PARENT_DEP_ACROSS
    | {'parent_two_more_depression_B_p', 'hopeless_B_p', 'parent_low_self_esteem_B_p'}
    | {'parent_wish_dead_present_B_p', 'parent_suicide_past_B_p', 'ptsd_diagnosis_present_B_p'}
    | {'parent_shy_or_timid_p', 'parent_prefers_to_be_alone_p', 'parent_avoids_talking_p'}
    | {'parent_confused_p', 'parent_clumsy_p'}
    | {'parent_heart_racing_p', 'parent_numbness_p'}
    | {'delta_parent_poor_work_performance_p'}
)

for target in [
    'parent_two_more_depression_B_p', 'parent_depressed_mood_B_p',
    'parent_anhedonia_B_p', 'hopeless_B_p',
    'parent_dysthymia_B_p', 'parent_low_self_esteem_B_p',
]:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _PARENT_DEP_KSADS_ACROSS,
        'within_exclude': _PARENT_DEP_WITHIN,
    }

# -- 28b. Parent Anxiety-domain (KSADS) ---

_PARENT_ANX_KSADS_ACROSS = (
    _PARENT_ANX_ACROSS
    | {'parent_excessive_worry_B_p', 'parent_socially_anxious_B_p',
       'parent_distress_due_to_social_anxiety_B_p', 'parent_obsessions_present_B_p'}
    | {'nightmares_B_p', 'obsessions_present_B_p'}
    | {'parent_shy_or_timid_p', 'parent_prefers_to_be_alone_p', 'parent_avoids_talking_p',
       'parent_self_conscious_p'}
    | {'parent_heart_racing_p', 'parent_numbness_p'}
)

_PARENT_SA_SOCIAL_CONTAMINATION = {
    'parent_bad_relationships_p', 'parent_friendship_trouble_p',
    'parent_not_liked_by_others_p', 'parent_bad_opposite_sex_relationship_p',
    'delta_parent_friendship_trouble_p', 'delta_parent_not_liked_by_others_p',
    'delta_parent_bad_relationships_p',
}

CIRCULAR_EXCLUSIONS['parent_excessive_worry_B_p'] = {
    'across_exclude': _PARENT_ANX_KSADS_ACROSS,
    'within_exclude': _PARENT_ANX_WITHIN,
}

for target in ['parent_socially_anxious_B_p', 'parent_social_anxiety_disorder_B_p']:
    CIRCULAR_EXCLUSIONS[target] = {
        'across_exclude': _PARENT_ANX_KSADS_ACROSS | _PARENT_SA_SOCIAL_CONTAMINATION,
        'within_exclude': _PARENT_ANX_WITHIN | {
            'Parent Social Functioning',
            'Parent Interpersonal Relationships and Social Functioning'},
    }

CIRCULAR_EXCLUSIONS['parent_distress_due_to_social_anxiety_B_p'] = {
    'across_exclude': (
        _PARENT_ANX_KSADS_ACROSS | _PARENT_SA_SOCIAL_CONTAMINATION
        | {'parent_bad_family_relationship_p', 'parent_meets_family_duties_p',
           'delta_parent_meets_family_duties_p', 'delta_parent_poor_work_performance_p',
           'parent_prefers_older_people_p', 'parent_clowns_or_shows_off_p'}
    ),
    'within_exclude': _PARENT_ANX_WITHIN | {
        'Parent Social Functioning',
        'Parent Interpersonal Relationships and Social Functioning'},
}

# -- 28c. Parent Concentration (KSADS) ---

CIRCULAR_EXCLUSIONS['parent_concentration_issues_B_p'] = {
    'across_exclude': (
        _PARENT_ADHD_ACROSS
        | {'parent_concentration_issues_B_p'}
        | {'bdefs_explain_idea_p', 'bdefs_explain_pt_p', 'bdefs_explain_seq_p',
           'bdefs_inconsistant_p', 'bdefs_process_info_p', 'bdefs_impulsive_action_p'}
        | {'delta_parent_poor_work_performance_p'}
    ),
    'within_exclude': {'Parent Cognitive and Attention Issues',
                       'Parent Impulsivity and Behavior Regulation'},
}

# -- 28d. Parent Irritability (KSADS) ---

CIRCULAR_EXCLUSIONS['parent_easily_annoyed_B_p'] = {
    'across_exclude': (
        _GENERAL_INTERNALIZING_EXCLUDE
        | {'parent_easily_annoyed_B_p'}
        | {'parent_stubborn_irritable_p', 'delta_parent_stubborn_irritable_p',
           'parent_hot_temper_p', 'parent_yells_a_lot_p', 'parent_louder_than_others_p'}
        | {'parent_sudden_mood_changes_p', 'parent_behavior_changeable_p'}
        | {'parent_aggressive_D_p', 'parent_external_D_p',
           'parent_depressed_p', 'parent_anxdep_D_p'}
        | {'parent_impulsive_p', 'parent_hyperactive_p', 'parent_restless_p'}
        | {'parent_physical_attacks_p',
           'parent_destroys_own_things_p', 'parent_destroys_others_things_p'}
        | {'parent_elevated_mood_B_p'}
        | {'delta_parent_depressed_p', 'delta_parent_feels_overwhelmed_p'}
        | {'frequent_family_conflict_p', 'family_conflict_ss_p', 'family_conflict_ss_k',
           'family_lose_temper_rare_p', 'family_believe_not_raise_voice_p',
           'parents_argue_more_p'}
        | {'fam_fight_often_k', 'fam_no_lose_temps_k',
           'fam_criticize_often_k', 'fam_no_raise_voices_k'}
        | {'argues_p', 'stubborn_p', 'temper_tantrums_p',
           'disobedient_home_p', 'whines_p'}
    ),
    'within_exclude': {'Parent Impulsivity and Behavior Regulation', 'Parent Mood Issues'},
}

# -- 28e. Parent Sleep Problems (KSADS) ---

CIRCULAR_EXCLUSIONS['parent_sleep_problem_B_p'] = {
    'across_exclude': (
        _PARENT_DEP_ACROSS
        | {'parent_sleep_problem_B_p', 'parent_high_sleep_duration_p', 'nightmares_B_p'}
        | {'parent_tired_no_reason_p', 'parent_low_energy_p'}
        | {'parent_confused_p', 'parent_forgetful_p', 'parent_concentration_trouble_p'}
        | {'difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p'}
        | {'parent_heart_racing_p'}
    ),
    'within_exclude': _PARENT_DEP_WITHIN,
}

# -- 28f. Parent PTSD (KSADS) ---

CIRCULAR_EXCLUSIONS['ptsd_diagnosis_present_B_p'] = {
    'across_exclude': (
        _PARENT_DEP_ACROSS | _PARENT_ANX_ACROSS
        | {'ptsd_diagnosis_present_B_p'}
        | {'parent_heart_racing_p', 'parent_numbness_p', 'parent_physical_attacks_p',
           'parent_hot_temper_p', 'parent_yells_a_lot_p'}
        | {'nightmares_B_p'}
        | {'parent_prefers_to_be_alone_p', 'parent_shy_or_timid_p'}
        | {'parent_drinks_too_much_p', 'parent_drunk_days_p',
           'parent_drug_use_p', 'parent_drug_days_p'}
        | {'parent_strange_behavior_p', 'parent_strange_thoughts_p'}
        | {'car_accident_hurt_p', 'big_accident_need_treatment_p',
           'fire_victim_p', 'natural_disaster_victim_p', 'terrorism_victim_p',
           'war_death_witness_p', 'stabbing_shooting_witness_p',
           'stabbing_shooting_victim_community_p', 'stabbing_shooting_victim_home_p',
           'beating_victim_home_p', 'stranger_threatened_child_victim_p',
           'family_threatened_child_victim_p', 'adult_family_fighting_victim_p',
           'domestic_child_sexually_abuse_victim_p', 'foreign_child_sexually_abuse_victim_p',
           'peer_child_sexually_abuse_victim_p', 'sudden_death_in_family_p'}
    ),
    'within_exclude': (
        _PARENT_DEP_WITHIN | _PARENT_ANX_WITHIN
        | {'Parent Impulsivity and Behavior Regulation'}
    ),
}

# -- 28g. Parent Obsessions (KSADS) ---

CIRCULAR_EXCLUSIONS['parent_obsessions_present_B_p'] = {
    'across_exclude': (
        {'parent_obsessions_present_B_p'}
        | {'parent_obsessive_thoughts_p', 'parent_repeats_acts_p'}
        | {'obsessions_present_B_p'}
        | {'parent_anxdep_D_p', 'parent_internal_D_p', 'parent_intrusive_thoughts_D_p'}
        | {'parent_strange_thoughts_p', 'parent_strange_behavior_p',
           'parent_sees_things_p', 'parent_hears_voices_p'}
        | {'parent_picks_skin_p'}
        | {'parent_hyperactive_p', 'parent_restless_p', 'parent_max_effort_p'}
        | _DEPRESSION_SBT
    ),
    'within_exclude': {'Parent Cognitive and Attention Issues'},
}

# -- 28h. Parent Compulsions (ASR Repeats Acts) ---
# parent_repeats_acts_p is the ASR analog of child cbcl_q66 ("Repeats certain
# acts over and over"). Mirrors child compulsions_p (§14) adapted to the
# parent self-report measurement level.
CIRCULAR_EXCLUSIONS['parent_repeats_acts_p'] = {
    'across_exclude': (
        {'parent_repeats_acts_p', 'delta_parent_repeats_acts_p'}
        | {'parent_obsessive_thoughts_p',                                   # ASR OCD scale co-constituent
           'parent_picks_skin_p'}                                           # body-focused repetitive behaviour
        | {'parent_intrusive_thoughts_D_p'}                                 # ASR DSM OCD composite that aggregates this item
        | {'parent_obsessions_present_B_p'}                                 # KSADS OCD diagnostic proxy
        | {'parent_strange_behavior_p', 'parent_strange_thoughts_p',        # thought disorder / OCD-anxiety overlap
           'parent_sees_things_p', 'parent_hears_voices_p'}
        | {'parent_fear_of_bad_thoughts_p'}                                 # ASR 31 -- intrusive-thoughts construct overlap
        | {'parent_internal_D_p', 'parent_anxdep_D_p'}                      # ASR Internalizing broadband (subsumes OCD scale via Thought Problems overlap)
        | {'parent_hyperactive_p', 'parent_restless_p',                     # restless/driven vs compulsive repetition
           'parent_max_effort_p'}
        | {'compulsions_p'}                                                 # cross-rater child analog
        | _PARENT_SOCIAL_IMPCOMP_BATCH_EXCLUDE                              # batch-wide broad composites + mood-swings
        | _DEPRESSION_SBT
    ),
    'within_exclude': {'Parent Cognitive and Attention Issues',
                       'Parent Anxiety and Fear-Related Issues',
                       'Parent Other Psychopathology',
                       'Parent Psychopathology Composite'},
}

# -- 28i. Parent Obsessions (ASR Obsessive Thoughts) ---
# parent_obsessive_thoughts_p is the ASR analog of child cbcl_q09
# ("Obsessions / can't get mind off certain thoughts"). Mirrors child
# obsessions_p (§11) adapted to the parent self-report measurement level.
# Attention items (concentration, decision trouble) intentionally NOT excluded
# to align with child §11 convention -- attention/rumination conflation is
# permitted as cross-domain prediction signal.
CIRCULAR_EXCLUSIONS['parent_obsessive_thoughts_p'] = {
    'across_exclude': (
        {'parent_obsessive_thoughts_p', 'delta_parent_obsessive_thoughts_p'}
        | {'parent_repeats_acts_p',                                         # ASR OCD scale co-constituent
           'parent_picks_skin_p'}                                           # body-focused repetitive behaviour
        | {'parent_intrusive_thoughts_D_p'}                                 # ASR DSM OCD composite that aggregates this item
        | {'parent_obsessions_present_B_p'}                                 # KSADS OCD diagnostic proxy
        | {'parent_strange_thoughts_p', 'parent_strange_behavior_p',        # thought disorder overlap
           'parent_sees_things_p', 'parent_hears_voices_p'}
        | {'parent_fear_of_bad_thoughts_p'}                                 # ASR 31 -- intrusive-thoughts construct overlap
        | {'parent_internal_D_p', 'parent_anxdep_D_p'}                      # ASR Internalizing broadband (subsumes OCD via Thought Problems overlap)
        | {'parent_hyperactive_p', 'parent_restless_p',                     # restless/driven vs obsessive preoccupation
           'parent_max_effort_p'}
        | {'obsessions_p'}                                                  # cross-rater child analog
        | _PARENT_SOCIAL_IMPCOMP_BATCH_EXCLUDE                              # batch-wide broad composites + mood-swings
        | _DEPRESSION_SBT
    ),
    'within_exclude': {'Parent Cognitive and Attention Issues',
                       'Parent Anxiety and Fear-Related Issues',
                       'Parent Other Psychopathology',
                       'Parent Psychopathology Composite'},
}

# -- Parent Internalizing (ASR Broadband) ---
# ASR Internalizing = Anxious/Depressed (18 items) + Withdrawn (9 items)
# + Somatic Complaints (12 items).
# ALL constituent items of these three subscales must be excluded, plus
# broadband composites, diagnostic proxies, and change-score derivatives.
#
# _PARENT_GENERAL_MOOD_EXCLUDE (Cell 1) already covers:
# ASR Anxious/Depressed:  parent_self_conscious_p (71), parent_fearful_or_anxious_p (50),
# parent_confused_p (13)
# ASR Withdrawn:          parent_prefers_to_be_alone_p (42), parent_avoids_talking_p (65)
# ASR Somatic Complaints: parent_heart_racing_p (56h), parent_numbness_p (56i)
# Other Problems:         parent_shy_or_timid_p (75), parent_high_sleep_duration_p (77)
_PARENT_INTERNAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE | _PARENT_GENERAL_MOOD_EXCLUDE
    # -- ASR broadband & syndrome composites ---
    | {'parent_internal_D_p', 'parent_anxdep_D_p', 'parent_adhd_D_p',
       'parent_somatic_problems_D_p', 'parent_depressed_mood_B_p'}       # ASR internalizing subscales
    | {'parent_external_D_p', 'parent_aggressive_D_p',
       'parent_antisocial_D_p', 'parent_hyperactive_D_p'}                # ASR externalizing broadband (composite subsumption: prevents reconstruction via subtraction)
    | {'parent_depress_D_p', 'parent_anxdisord_D_p'}                     # CBCL-style DSM syndrome composites
    | {'parent_avoidant_person_D_p', 'parent_personal_strength_D_p',
       'parent_attention_D_p'}                                            # ASR adaptive/attention composites with internalizing loading
    # -- ASR Anxious/Depressed subscale (18 items → Internalizing) ---
    # Items already covered by _PARENT_GENERAL_MOOD_EXCLUDE above:
    # parent_self_conscious_p (71), parent_fearful_or_anxious_p (50),
    # parent_confused_p (13)
    # Remaining Anxious/Depressed items:
    | {'parent_depressed_p',                                              # ASR 103 Unhappy, sad, or depressed
       'parent_cries_a_lot_p',                                            # ASR 14 Cries a lot
       'parent_lonely_p',                                                 # ASR 12 Lonely
       'parent_feels_unloved_p',                                          # ASR 33 Feels no one loves me
       'parent_feels_inferior_p',                                         # ASR 35 Feels worthless or inferior
       'parent_feels_unsuccessful_p',                                     # ASR 107 Feels can't succeed
       'parent_paranoid_p',                                               # ASR 34 Others are out to get me
       'parent_worries_a_lot_p',                                          # ASR 112 Worries
       'parent_worries_about_future_p',                                   # ASR 22 Worries about future
       'parent_relationship_concerns_p',                                  # ASR 113 Worries about opposite sex relations
       'parent_fear_of_bad_thoughts_p',                                   # ASR 31 Afraid might think/do something bad
       'parent_suicidal_thoughts_p'}                                      # ASR 91 Thinks about killing self
    # -- ASR Withdrawn subscale (9 items → Internalizing) ---
    # Items already covered by _PARENT_GENERAL_MOOD_EXCLUDE above:
    # parent_prefers_to_be_alone_p (42), parent_avoids_talking_p (65)
    # Remaining Withdrawn items:
    | {'parent_bad_relationships_p',                                      # ASR 25 Don't get along with other people
       'parent_bad_opposite_sex_relationship_p',                          # ASR 30 Relations with opposite sex are poor
       'parent_not_liked_by_others_p',                                    # ASR 48 Not liked by others
       'parent_friendship_trouble_p',                                     # ASR 67 Trouble making or keeping friends
       'parent_enjoys_little_p',                                          # ASR 60 Very little that I enjoy
       'parent_secretive_p'}                                              # ASR 69 Secretive, keeps things to self
    # ASR 111 (Withdrawn) not in ABCD dataset
    # -- ASR Somatic Complaints subscale (12 items → Internalizing) ---
    # Items already covered by _PARENT_GENERAL_MOOD_EXCLUDE above:
    # parent_heart_racing_p (56h), parent_numbness_p (56i)
    # Remaining Somatic items:
    | {'parent_tired_no_reason_p',                                        # ASR 54 Tired without good reason
       'parent_sleep_trouble_p',                                          # ASR 100 Trouble sleeping
       'parent_picks_skin_p'}                                             # ASR 58 Picks skin (Other Problems, but somatic/BFR-adjacent)
    # parent_doesnt_eat_well_p (ASR 24, Other Problems) intentionally
    # NOT excluded -- it is not an Internalizing subscale constituent.
    # ASR 51 (dizzy), 56a-56g (aches/headaches/nausea/eye/skin/
    # stomach/vomiting) not individually named in ABCD -- they are
    # captured via the parent_somatic_problems_D_p composite above.
    # -- Cross-scale items excluded for construct overlap ---
    # These are NOT Internalizing subscale constituents but index mood/
    # energy/anhedonia constructs that are inseparable from depression.
    | {'parent_happy_person_p',                                           # Adaptive: reverse-coded anhedonia proxy
       'parent_sudden_mood_changes_p',                                    # ASR 87 Aggressive Behavior -- mood dysregulation confound
       'parent_decision_trouble_p',                                       # ASR 78 Attention Problems -- depression-linked indecisiveness
       'parent_restless_p',                                               # Other Problems -- anxious agitation confound
       'parent_low_energy_p'}                                             # ASR 102 Attention Problems -- fatigue/anergia confound
    # -- Thought disorder / OCD-anxiety overlap ---
    | {'parent_intrusive_thoughts_D_p', 'parent_obsessive_thoughts_p'}    # ASR 9 Thought Problems -- obsessional anxiety overlap
    # -- KSADS / diagnostic items ---
    | {'parent_anhedonia_B_p', 'parent_dysthymia_B_p'}                   # DSM MDD core criterion / depressive disorder diagnosis
    | {'parent_self_harm_p',                                              # suicidality subsumed by internalizing
       'parent_suicide_past_B_p'}                                         # KSADS past suicide attempt
    | {'ptsd_diagnosis_present_B_p'}                                      # trauma/PTSD diagnostic -- internalizing spectrum
    | {'parent_social_anxiety_disorder_B_p'}                              # KSADS social anxiety diagnosis
    | {'parent_specific_fears_p',                                         # ASR 29 Other Problems -- specific phobia-adjacent
       'parent_worries_about_family_p'}                                   # ASR 72 Other Problems -- generalised worry proxy
    # -- Family/environment confounds ---
    | {'parent_therapy_emo', 'family_emotionprob_p',
       'parent_strange_thoughts_p'}                                       # treatment-seeking / family distress confounds
    # -- Delta (change-score) derivatives ---
    | {'delta_parent_internal_D_p', 'delta_parent_depressed_p',
       'delta_parent_worries_a_lot_p', 'delta_parent_somatic_problems_D_p',
       'delta_parent_feels_overwhelmed_p', 'delta_parent_sleep_trouble_p',
       'delta_parent_feels_unloved_p', 'delta_parent_stubborn_irritable_p',
       'delta_parent_worries_about_family_p', 'delta_parent_worries_about_future_p',
       'delta_parent_aches_pains_p',
       'delta_parent_planning_trouble_p',
       'delta_parent_bad_relationships_p',                                # Withdrawn deltas
       'delta_parent_bad_opposite_sex_relationship_p',
       'delta_parent_not_liked_by_others_p',
       'delta_parent_friendship_trouble_p'}
    | _DEPRESSION_SBT
)
CIRCULAR_EXCLUSIONS['parent_internal_D_p'] = {
    'across_exclude': _PARENT_INTERNAL_ACROSS,
    'within_exclude': {'Parent Mood Issues', 'Parent Anxiety',
                       'Parent Anxiety and Fear-Related Issues',
                       'Parent Other Psychopathology',
                       'Parent Psychopathology Composite'},
}

# -- Parent Externalizing (ASR Broadband) ---
# ASR Externalizing = Aggressive Behavior (15 items) + Rule-Breaking
# Behavior (14 items) + Intrusive (6 items).
# ALL constituent items of these three subscales must be excluded, plus
# broadband composites, diagnostic proxies, and change-score derivatives.
_PARENT_EXTERNAL_ACROSS = (
    _GENERAL_INTERNALIZING_EXCLUDE
    # -- ASR broadband & syndrome composites ---
    | {'parent_external_D_p', 'parent_aggressive_D_p',
       'parent_antisocial_D_p', 'parent_hyperactive_D_p'}               # ASR externalizing subscales
    | {'parent_internal_D_p', 'parent_anxdep_D_p', 'parent_depress_D_p',
       'parent_somatic_problems_D_p'}                                    # ASR internalizing broadband (composite subsumption)
    | {'parent_adhd_D_p', 'parent_attention_D_p'}                        # ADHD composites (externalizing ADHD item overlap)
    | {'parent_personal_strength_D_p', 'parent_intrusive_thoughts_D_p',
       'parent_avoidant_person_D_p'}                                     # ASR composites with externalizing loading
    # -- ASR Aggressive Behavior items (15 items → Externalizing) ---
    | {'parent_stubborn_irritable_p',                                     # ASR 86 Stubborn, sullen, or irritable
       'parent_hot_temper_p',                                             # ASR 95 Hot temper
       'parent_yells_a_lot_p',                                            # ASR 68 Screams or yells a lot
       'parent_sudden_mood_changes_p',                                    # ASR 87 Moods or feelings change suddenly
       'parent_behavior_changeable_p',                                    # ASR 81 Behavior is very changeable
       'parent_destroys_own_things_p',                                    # ASR 20 Destroys own things
       'parent_destroys_others_things_p',                                 # ASR 21 Destroys others' things (via ASR item 21)
       'parent_physical_attacks_p',                                       # ASR 57 Physically attacks people
       'parent_depressed_p'}                                              # ASR 103 -- excluded for Aggressive mood-swing confound
    # remaining Aggressive items (3-argues, 5-blames, 16-mean,
    # 28-badly_family, 37-fights, 55-elate_depress, 97-threaten,
    # 116-upset, 118-impatient) may not be individually named in ABCD
    # but are captured via parent_aggressive_D_p composite above.
    # -- ASR Rule-Breaking Behavior items (14 items → Externalizing) --
    | {'parent_impulsive_p',                                              # ASR 41 Impulsive, acts without thinking
       'parent_no_guilt_p',                                               # ASR 26 Doesn't feel guilty
       'parent_illegal_behavior_p',                                       # ASR 92 Does things that may cause trouble with law
       'parent_drug_use_p',                                               # ASR 6 Uses drugs for nonmedical purposes
       'parent_drinks_too_much_p',                                        # ASR 90 Drinks too much / gets drunk
       'parent_money_management_trouble_p',                               # ASR 117 Trouble managing money or credit cards
       'parent_doesnt_finish_tasks_p',                                    # ASR 59 Fails to finish things (also Attention)
       'parent_risky_decisions_p',                                        # ASR 76 Behavior is irresponsible
       'parent_strange_behavior_p'}                                       # ASR 84 Does things others think are strange (also Thought)
    # parent_fails_to_pay_debts_p (ASR 114) and
    # parent_drives_too_fast_p (ASR 120, Other) not reliably in ABCD
    # but are named below for safety-net coverage.
    | {'parent_fails_to_pay_debts_p',                                     # ASR 114 Fails to pay debts (Rule-Breaking)
       'parent_drives_too_fast_p'}                                        # ASR 120 Other Problems -- reckless-impulsivity confound
    # parent_tardy_often_p (ASR 121, Attention) intentionally NOT
    # excluded -- symmetrical with PI treatment.
    # -- ASR Intrusive items (6 items → Externalizing) ---
    | {'parent_bragging_p',                                               # ASR 7 Brags
       'parent_attention_seeking_p',                                      # ASR 19 Tries to get a lot of attention
       'parent_clowns_or_shows_off_p',                                    # ASR 74 Shows off or clowns
       'parent_talks_too_much_p',                                         # ASR 93 Talks too much
       'parent_teases_others_p',                                          # ASR 94 Teases others
       'parent_louder_than_others_p'}                                     # ASR 104 Louder than others
    # -- ASR Withdrawn items (Internalizing -- excluded for composite subsumption) --
    # These are Internalizing constituents, not Externalizing. Excluding
    # them prevents indirect reconstruction of Externalizing via
    # Total − Internalizing arithmetic and removes social-functioning
    # confounds that covary with externalizing consequences.
    | {'parent_bad_relationships_p',                                      # ASR 25 Withdrawn → INT
       'parent_bad_family_relationship_p',                                # ASR 28 Aggressive → EXT (gets along badly with family)
       'parent_not_liked_by_others_p',                                    # ASR 48 Withdrawn → INT
       'parent_friendship_trouble_p',                                     # ASR 67 Withdrawn → INT
       'parent_bad_opposite_sex_relationship_p',                          # ASR 30 Withdrawn → INT
       'parent_meets_family_duties_p',                                    # Adaptive functioning (reverse-coded) -- interpersonal confound
       'parent_stands_up_rights_p',                                       # Adaptive functioning -- assertiveness confound
       'parent_associates_with_trouble_p',                                # ASR 39 Rule-Breaking → EXT
       'parent_prefers_older_people_p'}                                   # ASR 63 Thought Problems -- social-deviance confound
    # -- Cross-scale items excluded for construct overlap ---
    | {'parent_secretive_p',                                              # ASR 69 Withdrawn -- covert antisocial confound
       'parent_restless_p',                                               # Other Problems -- hyperactive-impulsivity confound
       'parent_hyperactive_p',                                            # Other Problems -- ADHD-hyperactivity confound
       'parent_strange_thoughts_p'}                                       # ASR 85 Thought Problems -- impulsive-chaotic cognition
    # parent_priority_trouble_p (ASR 64) and parent_easily_bored_p
    # (ASR 83) intentionally NOT excluded -- symmetrical with PI treatment.
    # ASR Attention Problems items (parent_concentration_trouble_p,
    # parent_forgetful_p, parent_planning_trouble_p, parent_disorganized_p,
    # parent_loses_things_p) intentionally NOT excluded -- they are not
    # Externalizing subscale constituents and are allowed as cross-domain
    # predictors (symmetrical with Parent Internalizing treatment).
    # -- KSADS / diagnostic items with externalizing overlap ---
    | {'parent_lying_B_p', 'parent_elevated_mood_B_p'}                   # antisocial/mania-hyperactivity ambiguity
    # -- Family/environment confounds ---
    | {'family_conflict_ss_p', 'family_organisation_ss_p',
       'family_believe_not_raise_voice_p', 'family_lose_temper_rare_p',
       'frequent_family_conflict_p'}                                      # family behavioural environment confounds
    # -- Delta (change-score) derivatives ---
    | {'delta_parent_stubborn_irritable_p',
       'delta_parent_bad_relationships_p', 'delta_parent_bad_family_relationship_p',
       'delta_parent_not_liked_by_others_p', 'delta_parent_friendship_trouble_p',
       'delta_parent_meets_family_duties_p',
       'delta_parent_poor_work_performance_p',
       'delta_parent_bad_opposite_sex_relationship_p',                     # Withdrawn delta
       'delta_parent_drug_use_p', 'delta_parent_drinks_too_much_p',       # Rule-Breaking substance deltas
       'delta_parent_fails_to_pay_debts_p',                               # Rule-Breaking delta
       'delta_family_conflict_ss_p'}
)
CIRCULAR_EXCLUSIONS['parent_external_D_p'] = {
    'across_exclude': _PARENT_EXTERNAL_ACROSS,
    'within_exclude': {'Parent Impulsivity and Behavior Regulation',
                       'Parent Interpersonal Relationships and Social Functioning',
                       'Parent Social Functioning',
                       'Parent Cognitive and Attention Issues',
                       'Parent Other Psychopathology'},
}

# ---
# LOOKUP FUNCTIONS
# ---



# -- Child Only Deltas -- entries ---

# delta_family_conflict_ss_k: Youth-report family conflict change score.
# Same FES conflict items as the parent-report version.
CIRCULAR_EXCLUSIONS['delta_family_conflict_ss_k'] = {
    'across_exclude': (
        {'family_conflict_ss_p', 'family_conflict_ss_k', 'delta_family_conflict_ss_p',
         'delta_family_conflict_ss_k',
         'frequent_family_conflict_p', 'family_lose_temper_rare_p',
         'family_peaceful_p', 'family_believe_not_raise_voice_p',
         'parents_argue_more_p'}
        | {'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_no_lose_temps_k',
           'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_throw_things_k',
           'fam_keep_peace_k', 'fam_try_one_up_k', 'fam_no_raise_voices_k'}
    ),
    'within_exclude': {'Family Dynamics & Parenting'},
}

# delta_somatic_problems_D_p: CBCL Somatic Complaints syndrome change score.
# Exclude all somatic syndrome constituent items plus the internalizing broadband
# composites that subsume somatic complaints.
CIRCULAR_EXCLUSIONS['delta_somatic_problems_D_p'] = {
    'across_exclude': (
        _SOMATIC_SYNDROME
        | {'somatic_problems_D_p', 'delta_somatic_problems_D_p',
           'internal_D_p', 'delta_internal_D_p'}                          # broadband composites containing somatic
        | {'pain_last_month_k'}                                            # youth self-report pain item
        | _GENERAL_INTERNALIZING_EXCLUDE
    ),
    'within_exclude': {'Medical/Somatic Problems'},
}

# -- Parent Only Deltas -- entries ---

# delta_family_conflict_ss_p: Family conflict summary score change.
# Exclude family conflict items and related family-dynamics predictors that
# directly constitute or reconstruct the composite.
CIRCULAR_EXCLUSIONS['delta_family_conflict_ss_p'] = {
    'across_exclude': (
        {'family_conflict_ss_p', 'family_conflict_ss_k', 'delta_family_conflict_ss_p',
         'delta_family_conflict_ss_k',
         'frequent_family_conflict_p', 'family_lose_temper_rare_p',
         'family_peaceful_p', 'family_believe_not_raise_voice_p',
         'parents_argue_more_p'}                                          # parent arguing -- directly taps family conflict
        | {'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_no_lose_temps_k',
           'fam_criticize_often_k', 'fam_hit_each_other_k', 'fam_throw_things_k',
           'fam_keep_peace_k', 'fam_try_one_up_k', 'fam_no_raise_voices_k'}  # all 9 FES conflict subscale items (youth)
    ),
    'within_exclude': {'Family Dynamics & Parenting'},
}

# delta_parent_financial_trouble_p: SES / financial hardship change score.
# Exclude financial hardship items that operationalise the same construct.
CIRCULAR_EXCLUSIONS['delta_parent_financial_trouble_p'] = {
    'across_exclude': (
        {'parent_financial_trouble_p', 'delta_parent_financial_trouble_p',
         'parent_financial_failures_p', 'delta_parent_financial_failures_p'}
        | {'struggle_food_expenses', 'couldnt_afford_rent_mortgage',
           'couldnt_afford_phone', 'evicted', 'gas_electric_oil_turned_off'}  # all material-hardship indicators
        | {'parent_fails_to_pay_debts_p', 'parent_money_management_trouble_p',
           'positive_finance_p'}                                               # ASR financial items + inverse construct
        | {'parent_income', 'area_deprivation_idx'}                            # SES composites that subsume financial hardship
    ),
    'within_exclude': {'SES & Mobility'},
}

# delta_parent_somatic_problems_D_p: ASR Somatic Complaints broadband change.
# Inherits from _PARENT_INTERNAL_ACROSS since it is a constituent of the
# internalizing broadband; within-exclude covers mood and somatic categories.
CIRCULAR_EXCLUSIONS['delta_parent_somatic_problems_D_p'] = {
    'across_exclude': _PARENT_INTERNAL_ACROSS,
    'within_exclude': {'Parent Mood Issues', 'Parent Anxiety',
                       'Parent Anxiety and Fear-Related Issues',
                       'Parent Other Psychopathology',
                       'Parent Psychopathology Composite'},
}

def get_circular_exclusions(target, timepoint=None):
    """
    Return (across_exclude_set, within_exclude_set) for a given target.

    Targets absent from CIRCULAR_EXCLUSIONS return empty sets -- a safe default
    that applies no exclusions rather than failing silently on novel targets.

    Parameters
    target    : str       Target variable name (e.g. 'depress_D_p', 'adhd_D_p').
    timepoint : int|None  When provided, T1/T3/T4-specific extras are merged in.

    Returns
    across_exclude : set  Variable names to remove from the flat predictor pool.
    within_exclude : set  Category names to skip in within-category analysis.
    """
    entry  = CIRCULAR_EXCLUSIONS.get(target, {'across_exclude': set(), 'within_exclude': set()})
    across = set(entry.get('across_exclude', set()))
    within = set(entry.get('within_exclude', set()))
    if timepoint in (1, 3, 4):
        across |= entry.get('across_exclude_t1t3t4', set())
    return across, within

_PARENT_TARGET_OVERRIDES = {'hopeless_B_p', 'ptsd_diagnosis_present_B_p'}

def is_parent_target(target):
    """Return True if the target uses the parent feature structure (ASR-derived).
    NOTE: hopeless_B_p and ptsd_diagnosis_present_B_p are parent targets that
    do not follow the parent_ prefix convention."""
    return (target.startswith('parent_')
            or target.startswith('delta_parent_')
            or target in _PARENT_TARGET_OVERRIDES)

In [ ]:
#@title [Alt] Feature-Set: Objective/Subjective Spectrum (T0 and T2 Within-Category Loader)

# Run only if these feature sets are used downstream.
# Alternative parcellation arranging the T0 and T2 within-category structure
# along an 8-category objective-to-subjective continuum (Fig 4c). Addresses
# the predictive discrepancy between objective biophysical measures and
# parent-rated psychopathology/personality for psychiatric outcomes.
#
# Categories:
#   Objective No-Report  : cognitive tasks, neuroimaging PCs, genetic PCs,
#                          fitbit/physiological, physical measurements
#   Objective Rated      : SES, residential, grades, diet
#   Child-Self Measures  : school, family, social (child report)
#   Parent-to-Child      : CBCL psychopathology, personality, social
#   Parent-Self Measures : ASR psychopathology, personality, family, drugs
#   Child & Parent       : adverse life events (joint ratings)
#
# This cell overrides child_variables_for_each_time_point[0] and [2],
# variables_for_each_time_point, t0_variables, and t2_variables. Run after
# Cells 0-2 instead of using the default T0/T2 categories. All circularity
# exclusions, outcome variables, save_plot_2, and helper functions from
# earlier cells remain intact.
#
# Circularity is handled entirely through variable-level exclusions
# (across_exclude) in get_circular_exclusions(), because the within_exclude
# sets reference fine-grained category names (e.g. 'Anxiety',
# 'Externalizing') that do not match these broad spectrum categories.
#
# T0 vs T2 differences:
#   T0 lacks : fitbit, diet, delta variables, community dynamics,
#              extended social/discrimination scales
#   T0 adds  : saliva biomarkers, family psych history, parent diagnostic
#              composites (D_p / B_p), child mood items, sleep items,
#              additional SES hardship items, additional cognitive tasks,
#              parenting acceptance scales
# ---

# --- Verify Cells 2 & 3 have been run ---
_required = ['child_variables_for_each_time_point', 'variables_for_each_time_point',
             'parent_variables_for_each_time_point', 'get_circular_exclusions',
             'CIRCULAR_EXCLUSIONS', 'is_parent_target',
             'binary_or_categorical_outcome_variables', 'continuous_outcome_variables',
             'save_plot_2', 't0_variables', 't2_variables']
_missing = [r for r in _required if r not in dir()]
if _missing:
    raise RuntimeError(
        f"Cells 2 (FE: Variable Registry) and 3 (FE: Circularity Exclusion Definitions) must run BEFORE this cell.\n"
        f"Missing: {_missing}"
    )
print("Cells 2 & 3 dependencies verified.\n")

# ---
# T0 Objective / Subjective Spectrum -- 8 Categories
# ---

_obj_sub_t0 = {

# ---
    # 1. Objective No-Report
    # T0 vs T2: NO fitbit
    # T0 adds:  tb_fluid, tb_cryst, tb_total, cct, mr_total, mr_matrix,
    # lmt_efficiency, nb_correct_nt_neutral, nb_correct_mrt_neutral,
    # saliva_DHEA, saliva_testosterone, puberty_k
    # T0 lacks: mid_mrt_smrw, mid_mrt_lgrw, mid_num_trials, lmt_accuracy,
    # lmt_correct_nt, lmt_mrt, lmt_correct_mrt,
    # sst_mean_ssrt..sst_mean_absdelta (only median at T0),
    # no_sports_activities_p, fitbit_*
# ---
    'Objective No-Report (Cog Tasks, Neuro, Physio, Ances Genes)': [
        # NIH Toolbox
        'tb_picvocab', 'tb_picture', 'tb_reading', 'tb_flanker', 'tb_list',
        'tb_cardsort', 'tb_pattern', 'tb_fluid', 'tb_cryst', 'tb_total',
        # RAVLT
        'ravlt_s_total', 'ravlt_s_repitition', 'ravlt_s_intrusions',
        'ravlt_l_total', 'ravlt_l_repitition', 'ravlt_l_intrusions',
        # N-Back (includes T0-specific neutral condition)
        'nb_correct_nt', 'nb_correct_mrt', 'nb_correct_nt_2back', 'nb_correct_mrt_2back',
        'nb_correct_nt_pos', 'nb_correct_mrt_pos',
        'nb_correct_nt_neutral', 'nb_correct_mrt_neutral',
        'nb_correct_nt_neg', 'nb_correct_mrt_neg',
        'nb2_accuracy_pos', 'nb2_resp_bias_pos', 'nb2_D_prime_pos',
        'nb2_accuracy_neg', 'nb2_resp_bias_neg', 'nb2_D_prime_neg',
        # SST (no sst_mean_* at T0 -- median only)
        'sst_ssrt_mean_est', 'sst_ssrt_int_est', 'sst_acceptable_performance',
        'sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS', 'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
        'sst_kappa0', 'sst_xM', 'sst_sM', 'sst_bG', 'sst_pp', 'sst_factor1', 'sst_factor2', 'sst_factor3',
        'sst_median_ssrt', 'sst_median_PDR', 'sst_median_SD', 'sst_median_SDr',
        'sst_median_PDRg', 'sst_median_betaS', 'sst_median_bS2', 'sst_median_absdelta',
        'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV', 'sst_sdRV', 'sst_sdSD', 'sst_sdCV',
        'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV', 'sst_pdrgV', 'sst_pdrgSD', 'sst_pdrgCV',
        'sst_betasV', 'sst_betasSD', 'sst_betasCV', 'sst_absdeltaRV', 'sst_absdeltaSD',
        'sst_absdeltaCV', 'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV',
        # MID
        'mid_total_payout', 'mid_acceptable_performance',
        # LMT (T0 has efficiency only, not accuracy/mrt breakdown)
        'lmt_efficiency',
        # CCT & Matrix Reasoning (T0-specific)
        'cct', 'mr_total', 'mr_matrix',
        # Structural
        'pc_struct1', 'pc_struct2', 'pc_struct3', 'pc_struct4', 'pc_struct5', 'pc_struct6', 'pc_struct7', 'pc_struct8',
        'pc_struct9', 'pc_struct10', 'pc_struct11', 'pc_struct12', 'pc_struct13', 'pc_struct14', 'pc_struct15',
        'pc_struct16', 'pc_struct17', 'pc_struct18', 'pc_struct19', 'pc_struct20', 'pc_struct21', 'pc_struct22',
        'pc_struct23', 'pc_struct24', 'pc_struct25', 'pc_struct26', 'pc_struct27', 'pc_struct28', 'pc_struct29',
        'pc_struct30', 'pc_struct31', 'pc_struct32', 'pc_struct33', 'pc_struct34', 'pc_struct35', 'pc_struct36',
        'pc_struct37', 'pc_struct38', 'pc_struct39', 'pc_struct40', 'pc_struct41', 'pc_struct42', 'pc_struct43',
        'pc_struct44', 'pc_struct45', 'pc_struct46', 'pc_struct47', 'pc_struct48', 'pc_struct49', 'pc_struct50',
        'pc_struct51', 'pc_struct52', 'pc_struct53', 'pc_struct54', 'pc_struct55', 'pc_struct56', 'pc_struct57',
        'pc_struct58', 'pc_struct59', 'pc_struct60', 'pc_struct61', 'pc_struct62', 'pc_struct63', 'pc_struct64',
        'pc_struct65', 'pc_struct66', 'pc_struct67', 'pc_struct68', 'pc_struct69', 'pc_struct70', 'pc_struct71',
        'pc_struct72', 'pc_struct73', 'pc_struct74', 'pc_struct75',
        # DTI
        'pc_DTI1', 'pc_DTI2', 'pc_DTI3', 'pc_DTI4', 'pc_DTI5', 'pc_DTI6', 'pc_DTI7', 'pc_DTI8', 'pc_DTI9', 'pc_DTI10',
        'pc_DTI11', 'pc_DTI12', 'pc_DTI13', 'pc_DTI14', 'pc_DTI15', 'pc_DTI16', 'pc_DTI17', 'pc_DTI18', 'pc_DTI19',
        'pc_DTI20', 'pc_DTI21', 'pc_DTI22', 'pc_DTI23', 'pc_DTI24', 'pc_DTI25', 'pc_DTI26', 'pc_DTI27', 'pc_DTI28',
        'pc_DTI29', 'pc_DTI30', 'pc_DTI31', 'pc_DTI32', 'pc_DTI33', 'pc_DTI34', 'pc_DTI35', 'pc_DTI36', 'pc_DTI37',
        'pc_DTI38', 'pc_DTI39', 'pc_DTI40', 'pc_DTI41', 'pc_DTI42', 'pc_DTI43', 'pc_DTI44', 'pc_DTI45', 'pc_DTI46',
        'pc_DTI47', 'pc_DTI48', 'pc_DTI49', 'pc_DTI50', 'pc_DTI51', 'pc_DTI52', 'pc_DTI53', 'pc_DTI54', 'pc_DTI55',
        'pc_DTI56', 'pc_DTI57', 'pc_DTI58', 'pc_DTI59', 'pc_DTI60', 'pc_DTI61', 'pc_DTI62', 'pc_DTI63', 'pc_DTI64',
        'pc_DTI65', 'pc_DTI66', 'pc_DTI67', 'pc_DTI68', 'pc_DTI69', 'pc_DTI70', 'pc_DTI71', 'pc_DTI72', 'pc_DTI73',
        'pc_DTI74', 'pc_DTI75',
        # Resting State
        'pc_rsFMRI1', 'pc_rsFMRI2', 'pc_rsFMRI3', 'pc_rsFMRI4', 'pc_rsFMRI5', 'pc_rsFMRI6', 'pc_rsFMRI7', 'pc_rsFMRI8',
        'pc_rsFMRI9', 'pc_rsFMRI10', 'pc_rsFMRI11', 'pc_rsFMRI12', 'pc_rsFMRI13', 'pc_rsFMRI14', 'pc_rsFMRI15',
        'pc_rsFMRI16', 'pc_rsFMRI17', 'pc_rsFMRI18', 'pc_rsFMRI19', 'pc_rsFMRI20', 'pc_rsFMRI21', 'pc_rsFMRI22',
        'pc_rsFMRI23', 'pc_rsFMRI24', 'pc_rsFMRI25', 'pc_rsFMRI26', 'pc_rsFMRI27', 'pc_rsFMRI28', 'pc_rsFMRI29',
        'pc_rsFMRI30', 'pc_rsFMRI31', 'pc_rsFMRI32', 'pc_rsFMRI33', 'pc_rsFMRI34', 'pc_rsFMRI35', 'pc_rsFMRI36',
        'pc_rsFMRI37', 'pc_rsFMRI38', 'pc_rsFMRI39', 'pc_rsFMRI40', 'pc_rsFMRI41', 'pc_rsFMRI42', 'pc_rsFMRI43',
        'pc_rsFMRI44', 'pc_rsFMRI45', 'pc_rsFMRI46', 'pc_rsFMRI47', 'pc_rsFMRI48', 'pc_rsFMRI49', 'pc_rsFMRI50',
        'pc_rsFMRI51', 'pc_rsFMRI52', 'pc_rsFMRI53', 'pc_rsFMRI54', 'pc_rsFMRI55', 'pc_rsFMRI56', 'pc_rsFMRI57',
        'pc_rsFMRI58', 'pc_rsFMRI59', 'pc_rsFMRI60', 'pc_rsFMRI61', 'pc_rsFMRI62', 'pc_rsFMRI63', 'pc_rsFMRI64',
        'pc_rsFMRI65', 'pc_rsFMRI66', 'pc_rsFMRI67', 'pc_rsFMRI68', 'pc_rsFMRI69', 'pc_rsFMRI70', 'pc_rsFMRI71',
        'pc_rsFMRI72', 'pc_rsFMRI73', 'pc_rsFMRI74', 'pc_rsFMRI75',
        # Ancestral-Genetic PCs
        *[f'pc_gene_aces{i}' for i in range(1, 33)],
        'desc_african_AFR_B', 'desc_native_american_AMR_B',
        'desc_alaska_native_AMR_B', 'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B',
        'desc_vietnamese_EAS_B', 'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B',
        'desc_latin_B',
        # Physical Measurements & Biomarkers
        'height', 'weight', 'waist', 'sex', 'birth_weight_p',
        'saliva_DHEA', 'saliva_testosterone', 'puberty_k'],

# ---
    # 2. Objective Rated
    # T0 vs T2: NO diet variables
    # T0 adds:  area_deprivation_idx, couldnt_afford_phone,
    # couldnt_afford_rent_mortgage, evicted, gas_electric_oil_turned_off
    # T0 lacks: fruit_intake..sodium_intake, resid_lead_risk,
    # positive_finance_p, parent_work_absences_p,
    # parent_financial_trouble_p, parent_fails_to_pay_debts_p,
    # marital_status, child_religion
# ---
    'Objective Rated (SES, Resid)': [
        # Residential Characteristics
        'neighborhood_safety_ss_p', 'neighborhood_safe_y', 'resid_density', 'resid_walkability', 'resid_prox_roads',
        'resid_crime_tot', 'resid_crime_violent', 'resid_crime_drug', 'resid_crime_dui',
        'resid_lead_risk_poverty', 'resid_lead_risk_houses_perc',
        'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism',
        'resid_sex_orient_bias', 'resid_immigrant_bias', 'resid_racism',
        'area_deprivation_idx',
        'L_site_id',
        # SES & Financial Hardship
        'parent_education', 'parent_income', 'struggle_food_expenses',
        'couldnt_afford_phone', 'couldnt_afford_rent_mortgage', 'evicted', 'gas_electric_oil_turned_off',
        'parent_age', 'sex_P', 'num_brothers_p', 'num_sisters_p',
        # Religion
        'religious_service_frequency', 'relig_importance'],

# ---
    # 3. Child-Self Measures
    # T0 vs T2: NO community dynamics, NO extended social/discrimination
    # scales, NO disobeys_at_school_k, NO child_newschool_p
    # T0 adds:  grades_dropped, school_detension_suspension,
    # screentime_weekday_ss_k, screentime_weekend_ss_k
# ---
    'Child-Self Measures (School, Family, Social)': [
        # School Dynamics
        'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k', 'enjoys_school_k',
        'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k', 'school_disengagement_ss_k',
        'repeated_grade', 'finds_schoolboring_k',
        'grades_dropped', 'school_detension_suspension',
        # Family Conflict (child-rated)
        'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k', 'fam_no_lose_temps_k', 'fam_criticize_often_k',
        'fam_hit_each_other_k', 'fam_keep_peace_k', 'fam_try_one_up_k', 'fam_no_raise_voices_k',
        # Social Quality (limited at T0)
        'prosocial_ss_p', 'close_boy_friends_k', 'close_girl_friends_k', 'prosocial_ss_k',
        # Technology Use
        'screentime_weekday_ss_k', 'screentime_weekend_ss_k',
        # Personality (child self-report)
        'up_negative_urgency_ss_k',
        'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k', 'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k',
        'bis_behav_inhibition_ss_k', 'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k'],

# ---
    # 4. Parent-to-Child Measures
    # T0 vs T2: NO scared_dark_p, NO goal_continuity_p, NO medhx_*,
    # NO easily_offended_p, blames_others_p, sociable_p,
    # school_excitement_p, not_critical_others_p, disagreeable_p
    # T0 adds:  child mood (enjoys_little_p, sad_p, suicidal_p, guilty_p,
    # withdrawn_p), sleep (difficulty_goingtosleep_p, etc.),
    # somatic extras (bad_toilet_habits_p, wets_bed_p),
    # strange_ideas_p, fears_being_bad_p,
    # parent_personal_strength_D_p
# ---
    'Parent-to-Child Measures (Psychopathology, Personality, Social)': [
        # General
        'not_liked_p', 'doesnt_get_along_p',
        # Anxiety
        'social_fear_present_PK', 'worries_p', 'clings_to_adults_p', 'nervous_general_p', 'nervous_twitching_p',
        'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p',
        # Mood (T0-specific -- not in T2 spectrum)
        'enjoys_little_p', 'sad_p', 'suicidal_p', 'guilty_p', 'withdrawn_p',
        # ADHD
        'cant_concentrate_p', 'doesnt_finish_p', 'hyperactive_p', 'impulsive_p',
        # Externalizing
        'argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p', 'destroys_own_things_p', 'destroys_others_things_p',
        'disobedient_home_p', 'disobedient_school_p', 'breaks_rules_p', 'fights_p', 'lying_p',
        'steals_home_p', 'steals_outside_p', 'threatens_others_p', 'whines_p', 'demands_attention_p',
        # Personality Features (parent-rated)
        'loquacious_p', 'bragadocious_p', 'easily_jealous_p',
        'wishes_other_sex_p', 'easily_embarrassed_p', 'secretive_p', 'perfectionist_p',
        'strange_ideas_p',
        # Sleep Problems (T0-specific)
        'difficulty_goingtosleep_p', 'difficulty_wakingup_p', 'nightmares_p',
        # Medical/Somatic Problems (T0 adds bad_toilet_habits_p, wets_bed_p)
        'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p', 'skin_problems_p',
        'frequent_stomachaches_p', 'vomiting_p', 'constipated_p',
        'bad_toilet_habits_p', 'wets_bed_p'],

# ---
    # 5. Child & Parent Ratings
    # T0 vs T2: NO experienced_crime_p
    # T0 adds:  b_lifeevents_affected_ss_p
# ---
    'Child & Parent Ratings (Adverse Life Events)': [
        'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p',
        'car_accident_hurt_p', 'big_accident_need_treatment_p', 'fire_victim_p', 'natural_disaster_victim_p',
        'terrorism_victim_p', 'war_death_witness_p', 'stabbing_shooting_witness_p',
        'stabbing_shooting_victim_community_p', 'stabbing_shooting_victim_home_p', 'beating_victim_home_p',
        'stranger_threatened_child_victim_p', 'family_threatened_child_victim_p', 'adult_family_fighting_victim_p',
        'domestic_child_sexually_abuse_victim_p', 'foreign_child_sexually_abuse_victim_p',
        'peer_child_sexually_abuse_victim_p', 'sudden_death_in_family_p',
        'g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k'],

# ---
    # 6. Parent-Self Measures (Psychopathology, Personality, Social)
    # T0 vs T2: NO parent_worries_a_lot_p, NO delta_* variables
    # T0 adds:  parent_suicidal_thoughts_p, family psych history
    # (d_grandfather_dep, father_mania, etc.),
    # parent diagnostic composites (parent_attention_D_p, etc.),
    # parent diagnostic binary items (obsessions_present_B_p, etc.)
# ---
    'Parent-Self Measures (Psychopathology, Personality, Social)': [
        # Parent Anxiety
        'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
        'parent_worries_about_future_p', 'parent_worries_about_family_p',
        'parent_relationship_concerns_p',
        # Parent Mood Issues
        'parent_cries_a_lot_p', 'parent_lonely_p', 'parent_feels_unloved_p', 'parent_paranoid_p',
        'parent_feels_inferior_p', 'parent_depressed_p', 'parent_feels_unsuccessful_p', 'parent_tired_no_reason_p',
        'parent_low_energy_p', 'parent_sleep_trouble_p', 'parent_enjoys_little_p', 'parent_sudden_mood_changes_p',
        'parent_suicidal_thoughts_p', 'parent_happy_person_p',
        # Parent Impulsivity and Behavior Regulation
        'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
        'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
        'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
        'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
        'parent_illegal_behavior_p', 'parent_self_harm_p', 'parent_doesnt_eat_well_p',
        # Parent Cognitive and Attention Issues
        'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
        'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p', 'parent_repeats_acts_p',
        'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p', 'parent_decision_trouble_p',
        # Parent Personality
        'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p', 'parent_clumsy_p',
        'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p', 'parent_louder_than_others_p',
        'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p', 'parent_easily_bored_p',
        'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p', 'parent_prefers_to_be_alone_p',
        'parent_no_guilt_p', 'parent_sense_of_fairness_p', 'parent_high_sleep_duration_p',
        # Parent Other
        'parent_physical_attacks_p', 'parent_picks_skin_p', 'parent_heart_racing_p', 'parent_numbness_p',
        'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p', 'parent_opposite_sex_wish_p',
        # Parent Social Functioning
        'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
        'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
        'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
        'parent_teases_others_p', 'parent_stands_up_rights_p',
        # -- T0-Specific: Parent Diagnostic Composites (ASR-derived D scales) --
        'parent_attention_D_p', 'parent_aggressive_D_p', 'parent_external_D_p', 'parent_depress_D_p',
        'parent_antisocial_D_p', 'parent_hyperactive_D_p', 'parent_somatic_problems_D_p',
        'parent_intrusive_thoughts_D_p', 'parent_avoidant_person_D_p', 'parent_depressed_mood_B_p',
        'parent_adhd_D_p',
        # -- T0-Specific: Parent Diagnostic Binary Items (KSADS/clinical) --
        'obsessions_present_B_p', 'poor_eye_contact_B_p',
        'nightmares_B_p', 'ptsd_diagnosis_present_B_p', 'parent_anhedonia_B_p', 'parent_elevated_mood_B_p',
        'parent_lying_B_p', 'parent_wish_dead_present_B_p',
        'parent_suicide_past_B_p', 'parent_dysthymia_B_p',
        'parent_social_anxiety_disorder_B_p', 'parent_bulimia_B_p',
        # -- T0-Specific: Family Psychopathology History --
        'd_grandfather_dep', 'd_grandmother_dep', 'm_grandfather_dep', 'm_grandmother_dep',
        'father_mania', 'mother_mania', 'father_trouble', 'parent_hospitalized_emo', 'parent_therapy_emo'],

# ---
    # 7. Parent-Self Measures (Family Style/Dynamics, Drug Use)
    # T0 vs T2: NO family_not_talk_aboutfeelings_p,
    # family_open_discussing_anything_p, family_expression_ss_p,
    # family_intellectual_ss_p, family_activities_ss_p,
    # family_organisation_ss_p, parents_divorced_p,
    # death_in_family_p, family_move_p,
    # parent_family_responsibilities_p,
    # parent_tobacco_use_frequency_p, parent_drug_use_p,
    # parent_drinks_too_much_p, parent_drinks_frequency_p,
    # parent_drunk_days_p, parent_drug_days_nonmedical_p
    # T0 adds:  y_acceptance_ss_p_crpbi, y_acceptance_ss_caregiver_crpbi,
    # full prenatal substance set (before + during pregnancy)
# ---
    'Parent-Self Measures (Family Style/Dynamics, Drug Use)': [
        # Family Dynamics
        'family_peaceful_p', 'family_lose_temper_rare_p',
        'family_believe_not_raise_voice_p', 'frequent_family_conflict_p',
        'family_conflict_ss_p', 'family_conflict_ss_k',
        'parent_monitoring_ss_k',
        # Parenting Style (T0-specific)
        'y_acceptance_ss_p_crpbi', 'y_acceptance_ss_caregiver_crpbi',
        # Drug Use and Substance History
        'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
        'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse',
        'cigs_during_pregnancy_p', 'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p',
        'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
        'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p',
        'weed_before_pregnancy_p', 'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p',
        'drugs_before_pregnancy_p', 'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p',
        'caffeine_during_pregnancy_p'],

# ---
    # 8. Inter
    # T0 vs T2: NO social_problems_D_p, delta_not_liked_p, child_white,
    # trans_id_y, screentime_daysperweek_k, bad_grades,
    # gd_riskybets, child_religion, parent_age_grouped
    # T0 adds:  screentime_weekend_ss_k, mr_total, tb_total
# ---
    'Inter': [
        'sex', 'sex_P', 'parent_income', 'num_brothers_p', 'num_sisters_p',
        'L_site_id', 'weight',
        'sex_orient_y',
        'screentime_weekday_ss_k', 'screentime_weekend_ss_k',
        'puberty_k', 'close_boy_friends_k', 'close_girl_friends_k',
        'cct', 'nb_correct_nt_2back', 'tb_cryst', 'tb_total',
        'sst_ssrt_mean_est', 'mid_total_payout', 'mr_total',
        'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k', 'b_lifeevents_ss_p',
        'relig_importance'],
}

# ---
# T2 Objective / Subjective Spectrum -- 8 Categories
# ---

_obj_sub_t2 = {

# ---
    # 1. Objective No-Report
# ---
    'Objective No-Report (Cog Tasks, Neuro, Physio, Ances Genes)': ['tb_picvocab', 'tb_picture', 'tb_reading', 'tb_flanker', 'tb_list',
            'tb_cardsort', 'tb_pattern', 'gd_safebets', 'gd_riskybets',
            'ravlt_s_total', 'ravlt_s_repitition', 'ravlt_s_intrusions',
            'ravlt_l_total', 'ravlt_l_repitition', 'ravlt_l_intrusions',
            'nb_correct_nt', 'nb_correct_mrt', 'nb_correct_nt_2back', 'nb_correct_mrt_2back',
            'nb_correct_nt_pos', 'nb_correct_mrt_pos', 'nb_correct_nt_neg', 'nb_correct_mrt_neg',
            'nb2_accuracy_pos', 'nb2_resp_bias_pos', 'nb2_D_prime_pos',
            'nb2_accuracy_neg', 'nb2_resp_bias_neg', 'nb2_D_prime_neg',
            'sst_ssrt_mean_est', 'sst_ssrt_int_est', 'sst_acceptable_performance',
            'mid_mrt_smrw', 'mid_mrt_lgrw', 'mid_total_payout',
            'mid_acceptable_performance', 'mid_num_trials',
            'lmt_accuracy', 'lmt_correct_nt', 'lmt_mrt',
            'lmt_correct_mrt', 'lmt_efficiency', 'sst_theta0', 'sst_theta1', 'sst_mu', 'sst_aS', 'sst_dS', 'sst_gamma0', 'sst_gamma1', 'sst_dG', 'sst_aG1', 'sst_aG2',
            'sst_kappa0', 'sst_xM', 'sst_sM', 'sst_bG', 'sst_pp', 'sst_factor1', 'sst_factor2', 'sst_factor3', 'sst_mean_ssrt',
            'sst_median_ssrt', 'sst_mean_PDR', 'sst_median_PDR', 'sst_mean_SD', 'sst_median_SD', 'sst_mean_SDr', 'sst_median_SDr',
            'sst_mean_PDRg', 'sst_median_PDRg', 'sst_mean_betaS', 'sst_median_betaS', 'sst_mean_bS2', 'sst_median_bS2',
            'sst_mean_absdelta', 'sst_median_absdelta', 'sst_pdrV', 'sst_pdrSD', 'sst_pdrCV', 'sst_sdRV', 'sst_sdSD', 'sst_sdCV',
            'sst_sdrRV', 'sst_sdrSD', 'sst_sdrCV', 'sst_pdrgV', 'sst_pdrgSD', 'sst_pdrgCV', 'sst_betasV', 'sst_betasSD',
            'sst_betasCV', 'sst_absdeltaRV', 'sst_absdeltaSD', 'sst_absdeltaCV', 'sst_bs2V', 'sst_bs2SD', 'sst_bs2CV',
        # Structural
        'pc_struct1', 'pc_struct2', 'pc_struct3', 'pc_struct4', 'pc_struct5', 'pc_struct6', 'pc_struct7', 'pc_struct8',
        'pc_struct9', 'pc_struct10', 'pc_struct11', 'pc_struct12', 'pc_struct13', 'pc_struct14', 'pc_struct15',
        'pc_struct16', 'pc_struct17', 'pc_struct18', 'pc_struct19', 'pc_struct20', 'pc_struct21', 'pc_struct22',
        'pc_struct23', 'pc_struct24', 'pc_struct25', 'pc_struct26', 'pc_struct27', 'pc_struct28', 'pc_struct29',
        'pc_struct30', 'pc_struct31', 'pc_struct32', 'pc_struct33', 'pc_struct34', 'pc_struct35', 'pc_struct36',
        'pc_struct37', 'pc_struct38', 'pc_struct39', 'pc_struct40', 'pc_struct41', 'pc_struct42', 'pc_struct43',
        'pc_struct44', 'pc_struct45', 'pc_struct46', 'pc_struct47', 'pc_struct48', 'pc_struct49', 'pc_struct50',
        'pc_struct51', 'pc_struct52', 'pc_struct53', 'pc_struct54', 'pc_struct55', 'pc_struct56', 'pc_struct57',
        'pc_struct58', 'pc_struct59', 'pc_struct60', 'pc_struct61', 'pc_struct62', 'pc_struct63', 'pc_struct64',
        'pc_struct65', 'pc_struct66', 'pc_struct67', 'pc_struct68', 'pc_struct69', 'pc_struct70', 'pc_struct71',
        'pc_struct72', 'pc_struct73', 'pc_struct74', 'pc_struct75',
        # DTI
        'pc_DTI1', 'pc_DTI2', 'pc_DTI3', 'pc_DTI4', 'pc_DTI5', 'pc_DTI6', 'pc_DTI7', 'pc_DTI8', 'pc_DTI9', 'pc_DTI10',
        'pc_DTI11', 'pc_DTI12', 'pc_DTI13', 'pc_DTI14', 'pc_DTI15', 'pc_DTI16', 'pc_DTI17', 'pc_DTI18', 'pc_DTI19',
        'pc_DTI20', 'pc_DTI21', 'pc_DTI22', 'pc_DTI23', 'pc_DTI24', 'pc_DTI25', 'pc_DTI26', 'pc_DTI27', 'pc_DTI28',
        'pc_DTI29', 'pc_DTI30', 'pc_DTI31', 'pc_DTI32', 'pc_DTI33', 'pc_DTI34', 'pc_DTI35', 'pc_DTI36', 'pc_DTI37',
        'pc_DTI38', 'pc_DTI39', 'pc_DTI40', 'pc_DTI41', 'pc_DTI42', 'pc_DTI43', 'pc_DTI44', 'pc_DTI45', 'pc_DTI46',
        'pc_DTI47', 'pc_DTI48', 'pc_DTI49', 'pc_DTI50', 'pc_DTI51', 'pc_DTI52', 'pc_DTI53', 'pc_DTI54', 'pc_DTI55',
        'pc_DTI56', 'pc_DTI57', 'pc_DTI58', 'pc_DTI59', 'pc_DTI60', 'pc_DTI61', 'pc_DTI62', 'pc_DTI63', 'pc_DTI64',
        'pc_DTI65', 'pc_DTI66', 'pc_DTI67', 'pc_DTI68', 'pc_DTI69', 'pc_DTI70', 'pc_DTI71', 'pc_DTI72', 'pc_DTI73',
        'pc_DTI74', 'pc_DTI75',
        # Resting State
        'pc_rsFMRI1', 'pc_rsFMRI2', 'pc_rsFMRI3', 'pc_rsFMRI4', 'pc_rsFMRI5', 'pc_rsFMRI6', 'pc_rsFMRI7', 'pc_rsFMRI8',
        'pc_rsFMRI9', 'pc_rsFMRI10', 'pc_rsFMRI11', 'pc_rsFMRI12', 'pc_rsFMRI13', 'pc_rsFMRI14', 'pc_rsFMRI15',
        'pc_rsFMRI16', 'pc_rsFMRI17', 'pc_rsFMRI18', 'pc_rsFMRI19', 'pc_rsFMRI20', 'pc_rsFMRI21', 'pc_rsFMRI22',
        'pc_rsFMRI23', 'pc_rsFMRI24', 'pc_rsFMRI25', 'pc_rsFMRI26', 'pc_rsFMRI27', 'pc_rsFMRI28', 'pc_rsFMRI29',
        'pc_rsFMRI30', 'pc_rsFMRI31', 'pc_rsFMRI32', 'pc_rsFMRI33', 'pc_rsFMRI34', 'pc_rsFMRI35', 'pc_rsFMRI36',
        'pc_rsFMRI37', 'pc_rsFMRI38', 'pc_rsFMRI39', 'pc_rsFMRI40', 'pc_rsFMRI41', 'pc_rsFMRI42', 'pc_rsFMRI43',
        'pc_rsFMRI44', 'pc_rsFMRI45', 'pc_rsFMRI46', 'pc_rsFMRI47', 'pc_rsFMRI48', 'pc_rsFMRI49', 'pc_rsFMRI50',
        'pc_rsFMRI51', 'pc_rsFMRI52', 'pc_rsFMRI53', 'pc_rsFMRI54', 'pc_rsFMRI55', 'pc_rsFMRI56', 'pc_rsFMRI57',
        'pc_rsFMRI58', 'pc_rsFMRI59', 'pc_rsFMRI60', 'pc_rsFMRI61', 'pc_rsFMRI62', 'pc_rsFMRI63', 'pc_rsFMRI64',
        'pc_rsFMRI65', 'pc_rsFMRI66', 'pc_rsFMRI67', 'pc_rsFMRI68', 'pc_rsFMRI69', 'pc_rsFMRI70', 'pc_rsFMRI71',
        'pc_rsFMRI72', 'pc_rsFMRI73', 'pc_rsFMRI74', 'pc_rsFMRI75',
        # Ethnicity/Nationality
        'pc_gene_aces1', 'pc_gene_aces2', 'pc_gene_aces3', 'pc_gene_aces4', 'pc_gene_aces5', 'pc_gene_aces6',
        'pc_gene_aces7', 'pc_gene_aces8', 'pc_gene_aces9', 'pc_gene_aces10', 'pc_gene_aces11', 'pc_gene_aces12',
        'pc_gene_aces13', 'pc_gene_aces14', 'pc_gene_aces15', 'pc_gene_aces16', 'pc_gene_aces17', 'pc_gene_aces18',
        'pc_gene_aces19', 'pc_gene_aces20', 'pc_gene_aces21', 'pc_gene_aces22', 'pc_gene_aces23', 'pc_gene_aces24',
        'pc_gene_aces25', 'pc_gene_aces26', 'pc_gene_aces27', 'pc_gene_aces28', 'pc_gene_aces29', 'pc_gene_aces30',
        'pc_gene_aces31', 'pc_gene_aces32', 'desc_african_AFR_B', 'desc_native_american_AMR_B',
        'desc_alaska_native_AMR_B', 'desc_chinese_EAS_B', 'desc_japanese_EAS_B', 'desc_korean_EAS_B',
        'desc_vietnamese_EAS_B', 'desc_european_EUR_B', 'desc_asian_indian_SAS_B', 'desc_other_south_asian_SAS_B',
        'desc_latin_B',
        # Physical Measurements
        'height', 'weight', 'waist', 'sex', 'no_sports_activities_p', 'birth_weight_p', 'fitbit_resting_hr',
        'fitbit_steps', 'fitbit_sedentary_mins', 'fitbit_lightlyactive_mins', 'fitbit_fairlyactive_mins',
        'fitbit_veryactive_mins'],

# ---
    # 2. Objective Rated
# ---
    'Objective Rated (SES, Resid, Grades, Diet)': [
        # Residential Characteristics
        'neighborhood_safety_ss_p', 'neighborhood_safe_y', 'resid_density', 'resid_walkability', 'resid_prox_roads',
        'resid_crime_tot', 'resid_crime_violent', 'resid_crime_drug', 'resid_crime_dui', 'resid_lead_risk_poverty',
        'resid_lead_risk_houses_perc', 'resid_lead_risk', 'resid_no2_avg', 'resid_pm25_avg', 'resid_sexism',
        'resid_sex_orient_bias', 'resid_immigrant_bias', 'resid_racism', 'L_site_id',
        # Diet
        'fruit_intake', 'vegetable_intake', 'protein_sources_intake', 'legume_intake', 'added_sugar',
        'sugary_beverage_freq', 'dairy_intake', 'whole_grain_intake', 'total_calories', 'protein_intake',
        'carbohydrate_intake', 'fiber_intake', 'sodium_intake', 'potassium_intake', 'total_sugar', 'saturated_fat',
        # SES
        'parent_education', 'parent_income', 'struggle_food_expenses', 'positive_finance_p', 'parent_work_absences_p',
        'parent_financial_trouble_p', 'parent_fails_to_pay_debts_p', 'marital_status', 'parent_age', 'sex_P',
        'num_brothers_p', 'num_sisters_p',
        # Religion
        'child_religion', 'religious_service_frequency', 'relig_importance'],

# ---
    # 3. Child-Self Measures
# ---
    'Child-Self Measures (School, Family, Social)': [
        # School Dynamics
        'disobeys_at_school_k', 'getalong_teachers_k', 'feelsafe_at_school_k', 'feels_smart_k', 'enjoys_school_k',
        'grades_important_k', 'school_environment_ss_k', 'school_involvement_ss_k', 'school_disengagement_ss_k',
        'repeated_grade', 'child_newschool_p', 'finds_schoolboring_k',
        # Community Dynamics
        'p_comm_cohesion_ss', 'p_comm_ctrl_ss', 'p_comm_collective_efficacy_ss',
        # Family Conflict
        'fam_fight_often_k', 'fam_no_open_anger_k', 'fam_throw_things_k', 'fam_no_lose_temps_k', 'fam_criticize_often_k',
        'fam_hit_each_other_k', 'fam_keep_peace_k', 'fam_try_one_up_k', 'fam_no_raise_voices_k',
        # Social Quality
        'prosocial_ss_p', 'close_boy_friends_k',
        'close_girl_friends_k', 'peer_net_protective_ss_k', 'peers_beh_prosocial_ss_k', 'peers_beh_delinquent_ss_k',
        'feels_leftout_k', 'not_invited_k', 'excluded_k', 'otherkids_spreadneg_rumors_k', 'otherkids_gossip_k',
        'feels_threatned_k', 'saysmeanthings_others_k', 'otherkids_saymeanthings_k', 'discrimination_ss_k',
        'feels_discriminated_k', 'senses_racism_k', 'doesnt_feel_accepted_k', 'bullied_on_internet_k',
        'prosocial_ss_k', 'socialinfluence_meanfinal_k', 'relational_victimization_ss_k',
        'reputational_aggression_ss_k', 'reputational_victimization_ss_k', 'overt_aggression_ss_k',
        'overt_victimization_ss_k', 'relational_aggression_ss_k', 'peer_net_protective_ss_k',
        'relational_victimization_ss_k', 'overt_aggression_ss_k', 'relational_aggression_ss_k',
        'feels_discriminated_teachers_k', 'feels_discriminated_adults_not_school_k', 'feels_discriminated_students_k',
        'feels_unwanted_american_society_k', 'feels_discriminated_americans_k',
        # Personality
        'up_negative_urgency_ss_k',
        'up_lackofplanning_ss_k', 'up_sensationseeking_ss_k', 'up_positiveurgency_ss_k', 'up_lackperseverance_ss_k',
        'bis_behav_inhibition_ss_k', 'bis_reward_responsive_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k'],

# ---
    # 4. Parent-to-Child Measures
# ---
    'Parent-to-Child Measures (Psychopathology, Personality, Social)': [
        # General
        'not_liked_p', 'doesnt_get_along_p',
        # Anxiety
        'social_fear_present_PK', 'worries_p', 'clings_to_adults_p', 'nervous_general_p', 'nervous_twitching_p',
        'fears_excl_school_p', 'fears_school_p', 'fears_being_bad_p', 'paranoid_p', 'scared_dark_p',
        # ADHD
        'cant_concentrate_p',
        'doesnt_finish_p', 'hyperactive_p', 'impulsive_p',
        'goal_continuity_p',
        # Externalizing
        'argues_p', 'stubborn_p', 'temper_tantrums_p', 'bullies_others_p', 'destroys_own_things_p', 'destroys_others_things_p',
        'disobedient_home_p', 'disobedient_school_p', 'breaks_rules_p', 'fights_p', 'lying_p', 'attacks_others_p',
        'steals_home_p', 'steals_outside_p', 'threatens_others_p', 'whines_p', 'demands_attention_p',
        # Other Personality Features
        'easily_offended_p', 'blames_others_p', 'sociable_p', 'school_excitement_p', 'not_critical_others_p',
        'disagreeable_p', 'loquacious_p', 'bragadocious_p', 'easily_jealous_p',
        'wishes_other_sex_p', 'easily_embarrassed_p', 'secretive_p', 'perfectionist_p',
        # Medical/Somatic Problems
        'medhx_p', 'medhx_doctorvisit_p', 'medhx_emergencyroom_p',
        'body_aches_p', 'frequent_headaches_p', 'nausea_p', 'eye_problems_p', 'skin_problems_p', 'frequent_stomachaches_p',
        'vomiting_p', 'constipated_p'],

# ---
    # 5. Child & Parent Ratings
# ---
    'Child & Parent Ratings (Adverse Life Events)': ['experienced_crime_p', 'g_lifeevents_ss_p', 'b_lifeevents_ss_p', 'b_lifeevents_affected_ss_p',
            'car_accident_hurt_p', 'big_accident_need_treatment_p', 'fire_victim_p', 'natural_disaster_victim_p',
            'terrorism_victim_p', 'war_death_witness_p', 'stabbing_shooting_witness_p',
            'stabbing_shooting_victim_community_p', 'stabbing_shooting_victim_home_p', 'beating_victim_home_p',
            'stranger_threatened_child_victim_p', 'family_threatened_child_victim_p', 'adult_family_fighting_victim_p',
            'domestic_child_sexually_abuse_victim_p', 'foreign_child_sexually_abuse_victim_p',
            'peer_child_sexually_abuse_victim_p', 'sudden_death_in_family_p',
            'g_lifeevents_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k'],

# ---
    # 6. Parent-Self Measures (Psychopathology, Personality, Social)
# ---
    'Parent-Self Measures (Psychopathology, Personality, Social)': [
        # Parent Anxiety
        'parent_fearful_or_anxious_p', 'parent_specific_fears_p', 'parent_fear_of_bad_thoughts_p',
        'parent_worries_about_future_p', 'parent_worries_about_family_p', 'parent_worries_a_lot_p',
        'parent_relationship_concerns_p',
        # Parent Mood Issues
        'parent_cries_a_lot_p', 'parent_lonely_p', 'parent_feels_unloved_p', 'parent_paranoid_p',
        'parent_feels_inferior_p', 'parent_depressed_p', 'parent_feels_unsuccessful_p', 'parent_tired_no_reason_p',
        'parent_low_energy_p', 'parent_sleep_trouble_p', 'parent_enjoys_little_p', 'parent_sudden_mood_changes_p',
        'parent_suicidal_thoughts_p', 'parent_happy_person_p',
        # Parent Impulsivity and Behavior Regulation
        'parent_impulsive_p', 'parent_risky_decisions_p', 'parent_drives_too_fast_p', 'parent_tardy_often_p',
        'parent_money_management_trouble_p', 'parent_priority_trouble_p', 'parent_behavior_changeable_p',
        'parent_hot_temper_p', 'parent_attention_seeking_p', 'parent_destroys_own_things_p',
        'parent_destroys_others_things_p', 'parent_doesnt_finish_tasks_p', 'parent_strange_behavior_p',
        'parent_illegal_behavior_p', 'parent_self_harm_p', 'parent_doesnt_eat_well_p',
        # Parent Cognitive and Attention Issues
        'parent_forgetful_p', 'parent_concentration_trouble_p', 'parent_confused_p', 'parent_planning_trouble_p',
        'parent_not_good_at_details_p', 'parent_obsessive_thoughts_p', 'parent_repeats_acts_p',
        'parent_max_effort_p', 'parent_disorganized_p', 'parent_loses_things_p', 'parent_decision_trouble_p',
        'parent_priority_trouble_p',
        # Parent Personality
        'parent_bragging_p', 'parent_honest_p', 'parent_secretive_p', 'parent_stubborn_irritable_p', 'parent_clumsy_p',
        'parent_strange_thoughts_p', 'parent_self_conscious_p', 'parent_uses_opportunities_p', 'parent_louder_than_others_p',
        'parent_yells_a_lot_p', 'parent_shy_or_timid_p', 'parent_restless_p', 'parent_easily_bored_p',
        'parent_hyperactive_p', 'parent_talks_too_much_p', 'parent_avoids_talking_p', 'parent_prefers_to_be_alone_p',
        'parent_no_guilt_p', 'parent_sense_of_fairness_p', 'parent_high_sleep_duration_p',
        # Parent Other
        'parent_physical_attacks_p', 'parent_picks_skin_p', 'parent_heart_racing_p', 'parent_numbness_p',
        'parent_sees_things_p', 'parent_hears_voices_p', 'parent_speech_problems_p', 'parent_opposite_sex_wish_p',
        # Parent Social Functioning
        'parent_bad_relationships_p', 'parent_bad_family_relationship_p', 'parent_not_liked_by_others_p',
        'parent_friendship_trouble_p', 'parent_prefers_older_people_p', 'parent_associates_with_trouble_p',
        'parent_bad_opposite_sex_relationship_p', 'parent_meets_family_duties_p', 'parent_clowns_or_shows_off_p',
        'parent_teases_others_p', 'parent_stands_up_rights_p',
        # Parent Deltas
        'delta_parent_sleep_trouble_p', 'delta_parent_worries_about_family_p',
        'delta_parent_friendship_trouble_p', 'delta_parent_poor_work_performance_p',
        'delta_parent_aches_pains_p', 'delta_parent_not_liked_by_others_p',
        'delta_parent_feels_overwhelmed_p', 'delta_parent_feels_unloved_p',
        'delta_parent_bad_family_relationship_p',
        'delta_parent_worries_a_lot_p', 'delta_parent_somatic_problems_D_p',
        'delta_parent_concentration_trouble_p', 'delta_parent_stubborn_irritable_p',
        'delta_parent_drinks_too_much_p', 'delta_parent_financial_failures_p',
        'delta_parent_meets_family_duties_p', 'delta_parent_planning_trouble_p',
        'delta_parent_bad_relationships_p', 'delta_parent_drug_use_p'],

# ---
    # 7. Parent-Self Measures (Family Style/Dynamics, Drug Use)
# ---
    'Parent-Self Measures (Family Style/Dynamics, Drug Use)': [
        # Family Dynamics
        'family_not_talk_aboutfeelings_p', 'family_peaceful_p', 'family_open_discussing_anything_p',
        'family_lose_temper_rare_p', 'family_believe_not_raise_voice_p', 'frequent_family_conflict_p',
        'family_conflict_ss_p', 'family_expression_ss_p', 'family_intellectual_ss_p', 'family_activities_ss_p',
        'family_organisation_ss_p', 'parents_divorced_p', 'death_in_family_p', 'family_move_p', 'family_conflict_ss_k',
        'parent_monitoring_ss_k', 'parent_family_responsibilities_p',
        # Drug Use and Substance History
        'hallucinogen_use_history_B_p', 'hallucinogen_current_B_p', 'sedative_hypnotic_anxiolytic_use_B_p',
        'father_alcohol', 'mother_alcohol', 'father_druguse', 'mother_druguse', 'cigs_during_pregnancy_p',
        'alcohol_during_pregnancy_p', 'weed_during_pregnancy_p', 'cocaine_during_pregnancy_p', 'heroin_during_pregnancy_p',
        'prescriptionmed_pregnancy_p', 'cigs_before_pregnancy_p', 'alcohol_before_pregnancy_p', 'weed_before_pregnancy_p',
        'cocaine_before_pregnancy_p', 'heroin_before_pregnancy_p', 'drugs_before_pregnancy_p',
        'drinksperweek_during_pregnancy_p', 'drugs_during_pregnancy_p', 'caffeine_during_pregnancy_p',
        'parent_tobacco_use_frequency_p', 'parent_drug_use_p', 'parent_drinks_too_much_p', 'parent_drinks_frequency_p',
        'parent_drunk_days_p', 'parent_drug_days_nonmedical_p'],

# ---
    # 8. Inter
# ---
    'Inter': [
        'sex', 'sex_P', 'parent_income', 'num_brothers_p', 'num_sisters_p',
        'L_site_id', 'weight', 'social_problems_D_p', 'delta_not_liked_p',
        'parent_age_grouped', 'child_white',
        'feels_leftout_k',
        'sex_orient_y', 'trans_id_y', 'screentime_daysperweek_k',
        'screentime_weekday_ss_k', 'puberty_k', 'close_boy_friends_k',
        'close_girl_friends_k', 'cct', 'bad_grades',
        'nb_correct_nt_2back', 'tb_cryst', 'sst_ssrt_mean_est', 'mid_total_payout',
        'family_conflict_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_affected_ss_k', 'b_lifeevents_ss_p',
        'gd_riskybets', 'child_religion', 'relig_importance'],
}

# ---
# Deduplicate, override, and build flat lists -- both timepoints
# ---

_spectra = {0: _obj_sub_t0, 2: _obj_sub_t2}

for _tp, _spectrum in _spectra.items():

    # -- Deduplicate within each category --
    for _cat in _spectrum:
        _seen = set()
        _deduped = []
        for _v in _spectrum[_cat]:
            if _v not in _seen:
                _seen.add(_v)
                _deduped.append(_v)
        if len(_deduped) < len(_spectrum[_cat]):
            print(f"  Warning: T{_tp} Deduplicated '{_cat}': {len(_spectrum[_cat])} → {len(_deduped)}")
        _spectrum[_cat] = _deduped

    # -- Override within-category structure --
    child_variables_for_each_time_point[_tp] = _spectrum
    if _tp in parent_variables_for_each_time_point:
        parent_variables_for_each_time_point[_tp] = {}

    # -- Build flat variable list --
    _all_vars = []
    _seen_flat = set()
    for _vars in _spectrum.values():
        for _v in _vars:
            if _v not in _seen_flat:
                _seen_flat.add(_v)
                _all_vars.append(_v)

    if _tp == 0:
        t0_variables = _all_vars
    elif _tp == 2:
        t2_variables = _all_vars

# Refresh the alias used by the Preprocessing cell
variables_for_each_time_point = child_variables_for_each_time_point

# ---
# Summary
# ---
for _tp, _spectrum in _spectra.items():
    _flat = t0_variables if _tp == 0 else t2_variables
    print("=" * 70)
    print(f"FE: Obj/Sub Spectrum -- T{_tp} Within-Category Structure Loaded")
    print("=" * 70)
    _total = 0
    for _cat, _vars in _spectrum.items():
        print(f"  {_cat:<60s} {len(_vars):>4d} features")
        _total += len(_vars)
    print(f"  {'-' * 60} {'-' * 4}")
    print(f"  {'TOTAL (unique across categories)':<60s} {len(_flat):>4d} features")
    print(f"  {'TOTAL (with cross-category overlap)':<60s} {_total:>4d} entries")
    print(f"  t{_tp}_variables (flat): {len(_flat)}")
    print()

print(f"Timepoints available: {sorted(child_variables_for_each_time_point.keys())}")
print("=" * 70)

In [ ]:
#@title Preprocessing, Natural Language Variable Renaming, and Train-Test Partitioning

# Data loading, imputation, scaling, train-test partitioning, and natural
# language variable renaming. All preprocessing fit on training data only
# and applied post-split. Provides within_categories_data() and
# across_categories_data() slicing functions that apply circular exclusions
# at runtime based on the active target. Delta variables enter as pre-
# computed columns from the source data.
# ---
# Functions are grouped as follows:
#
# Task-type detection  is_regression(), is_classification()
# Class encoding       map_classes(), inverse_map_classes()
# Class rebalancing    simple_balance_classes()
# Imputation factory   get_imputer()
# Missing-data QC      drop_high_missing_columns()
# Data splitting       within_categories_data(), across_categories_data()
# within_categories_data_combinations()
# K-fold variants      within_categories_data_kfold(),
# across_categories_data_kfold(),
# train_and_evaluate_model_kfold()
# Quantile utilities   segment_data_by_quantiles(),
# calculate_quantile_weights()
# ---

import pandas as pd
from scipy import stats
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import resample
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, accuracy_score, classification_report, confusion_matrix, make_scorer
from sklearn.base import clone
from collections import defaultdict

# -- DATA PATH -- update to your local ABCD 5.1 panel file location --
DATA_PATH = "CLEAN_ABCD_5.1_panel_20241022.csv"

# Load cleaned ABCD 5.1 panel (N ~23,760 after merging adolescent + parent
# data across 5 annual waves).
sample = pd.read_csv(DATA_PATH, low_memory=False)
# Drop legacy unnamed index columns from older exports (e.g., from df.to_csv(index=True))
_unnamed_cols = [c for c in sample.columns if str(c).startswith('Unnamed:')]
if _unnamed_cols:
    sample = sample.drop(columns=_unnamed_cols)

# Registry of continuous-outcome targets. Dimensional targets (CBCL subscales,
# SBT depression composites, cognitive scores) are modelled as regressions;
# binary/categorical targets default to classification with stratified splits.
def is_regression(target_options):
    regression_targets = [
    "depress_D_p", "delta_depress_D_p",
    "internal_D_p", "external_D_p", "anxdisord_D_p", "adhd_D_p",
    "social_problems_D_p", "thought_disorder_D_p",
    "sbt_core_depression", "delta_sbt_core_depression", "sbt_core_depression_delta_t0_t3",
    "sbt_anxiety_depression", "sbt_guilt_hopelessness", "sbt_emotional_dysregulation", "sbt_well_being",
    "sbt_fatigue_sleep", "sbt_somatic_depression",
    "sbt_academic_cognitive", "sbt_perfectionism_achievement",
    "sbt_social_withdrawal", "sbt_avoidance_fear", "sbt_aggression_irritability",
    "suicidal_p", "wish_dead_present_B_k",
    "asd_ssrs_sum", 'nb_correct_nt_2back',
    "bdefs_distract_upset_p", "gd_riskybets", "pa_sum_k",
    "tb_fluid", "tb_cryst", "tb_reading", "tb_flanker",
    # T0 Cognitive Task Battery additions:
    "ravlt_l_total", "lmt_efficiency", "sst_mean_ssrt",
    "bad_grades", "area_deprivation_idx", "weight",
    "parent_income",
    'compulsions_p', 'obsessions_p', "impulsive_p", 'perfectionist_p',
    # -- Child CBCL Social Problems items ---
    "not_liked_p", "doesnt_get_along_p", "enjoys_little_p",
    "parent_depressed_p", "parent_suicidal_thoughts_p",
    "parent_depress_D_p",
    "parent_adhd_D_p", "parent_anxdisord_D_p",
    "delta_anxdisord_D_p",
    "parent_worries_a_lot_p", "parent_concentration_trouble_p",
    "parent_happy_person_p", "parent_bad_relationships_p",
    "parent_not_liked_by_others_p",
    "delta_parent_depressed_p", "delta_parent_worries_a_lot_p",
    "parent_impulsive_p", "parent_tobacco_use_frequency_p",
    "parent_drinks_frequency_p", "parent_drunk_days_p",
    # -- Parent Social/Impulsivity/Compulsivity batch ---
    "parent_avoidant_person_D_p", "parent_repeats_acts_p", "parent_obsessive_thoughts_p",
    # -- Parent broadband (ASR Internalizing / Externalizing) --
    "parent_internal_D_p", "parent_external_D_p",
    "delta_parent_depress_D_p",   # parent broadband Depression Syndrome change score
    # -- Child Only Deltas batch ---
    "delta_adhd_D_p",
    "delta_family_conflict_ss_k",
    "delta_bad_grades",
    "delta_social_problems_D_p",
    "delta_somatic_problems_D_p",
    # -- Parent Only Deltas batch ---
    "delta_parent_sleep_trouble_p",
    "delta_parent_worries_about_family_p",
    "delta_parent_not_liked_by_others_p",
    "delta_parent_feels_unloved_p",
    "delta_family_conflict_ss_p",
    "delta_parent_worries_about_future_p",
    "delta_parent_concentration_trouble_p",
    "delta_parent_bad_relationships_p",
    "delta_parent_financial_trouble_p",
    "delta_parent_somatic_problems_D_p",
    ]
    return target_options in regression_targets

def is_classification(y):
    """Heuristic check: targets with ≤10 unique observed values are treated
    as classification tasks.  Used for stratified splitting and metric selection."""
    try:
        unique_values = y.nunique() if hasattr(y, 'nunique') else len(set(y.values.flatten()))
        if isinstance(unique_values, pd.Series):
            unique_values = unique_values.iloc[0]
        return bool(unique_values <= 10)
    except Exception as e:
        print(f"[WARNING] {e}")
        return False

# Canonical task-type detector used by validation/reporting cells (14-18).
# Defined here so there is a single authoritative copy; later cells
# rely on earlier registry cells having been executed.
# -- Canonical task-type detector ---
def detect_task_type(y, threshold=10):
    """Determine whether a target is regression or classification.

    Uses the same logic as is_regression / is_classification defined below in this cell,
    but returns a string for downstream branching.

    Parameters
    y : array-like
        Target variable (Series, ndarray, or DataFrame column).
    threshold : int, default 10
        If the number of unique non-NaN values ≤ threshold, treat as
        classification.  Otherwise regression.

    Returns
    str : 'classification' or 'regression'
    """
    # Honour whitelist for targets with known types
    try:
        if 'is_regression' in globals() and 'target_options' in globals():
            if is_regression(target_options):
                return 'regression'
    except Exception:
        pass
    _arr = np.asarray(y).ravel()
    _arr = _arr[~np.isnan(_arr)] if np.issubdtype(_arr.dtype, np.floating) else _arr
    n_unique = len(np.unique(_arr))
    return 'classification' if n_unique <= threshold else 'regression'

def map_classes(y):
    if target_options == 'depadhd_c':
      class_mapping = {'other':0,
                        'lowdep_lowadhd':1,
                        'highdep_highadhd':2, 'highdep_lowadhd':3,
                        'lowdep_highadhd': 4}
    elif target_options == 'asdadhd_c':
      class_mapping = {
          'other': 0,
          'lowasd_lowadhd': 1,
          'highasd_highadhd': 2,
          'highasd_lowadhd': 3,
          'lowasd_highadhd': 4
      }
    else:
      class_mapping = {'falling': 0, 'low': 1, 'rising': 2, 'high': 3}

    print(f"For {target_options}, mapping of classes are as follows: {class_mapping}")

    if isinstance(y, pd.DataFrame):
        return y.map(lambda x: class_mapping.get(x, x))
    elif isinstance(y, pd.Series):
        return y.map(class_mapping)
    else:
        raise TypeError("Input must be a pandas Series or DataFrame")

# Reverse the integer encoding produced by map_classes() for display and
# downstream evaluation against the original string labels.
def inverse_map_classes(y):
    if target_options == 'depadhd_c':
        inverse_class_mapping = {0: 'other', 1: 'lowdep_lowadhd',
                                 2: 'highdep_highadhd', 3: 'highdep_lowadhd',
                                 4: 'lowdep_highadhd'}
    elif target_options == 'asdadhd_c':
        inverse_class_mapping = {
            0: 'other',
            1: 'lowasd_lowadhd',
            2: 'highasd_highadhd',
            3: 'highasd_lowadhd',
            4: 'lowasd_highadhd'
        }
    else:
        inverse_class_mapping = {0: 'falling', 1: 'low', 2: 'rising', 3: 'high'}

    if isinstance(y, pd.DataFrame):
        return y.map(lambda x: inverse_class_mapping.get(x, x))
    elif isinstance(y, pd.Series):
        return y.map(inverse_class_mapping)
    elif isinstance(y, (int, np.integer)):  # Handle integer input
        return inverse_class_mapping.get(y, y)
    else:
        raise TypeError("Input must be a pandas Series, DataFrame, or integer")

# Rebalance class frequencies for multi-class trajectory targets (latent_class_
# depression, depadhd_c, asdadhd_c). Majority classes are downsampled and
# minority classes upsampled with replacement to equalize representation.
# Applied post-split to training data only in each data-preparation function.
def simple_balance_classes(df, target_column='latent_class_depression'):
    # Note: In the main pipeline (within/across_categories_data), this function
    # is called AFTER the train/test split on training data only.
    # The kfold variant still applies rebalancing pre-split.



    # Get the current class distribution
    class_counts = df[target_column].value_counts()
    print("Original class distribution:")
    print(class_counts)

    # Determine the desired number of samples per class
    n_samples = len(df)
    n_classes = len(class_counts)
    samples_per_class = n_samples // n_classes

    # Collect resampled class chunks; build DataFrame once to avoid pandas 2.x FutureWarning
    chunks = []

    for class_label in class_counts.index:
        class_df = df[df[target_column] == class_label]

        if len(class_df) > samples_per_class:
            # Undersample majority class
            resampled = resample(class_df,
                                replace=False,
                                n_samples=samples_per_class,
                                random_state=42)
        else:
            # Oversample minority class
            resampled = resample(class_df,
                                replace=True,
                                n_samples=samples_per_class,
                                random_state=42)

        chunks.append(resampled)

    balanced_df = pd.concat(chunks, ignore_index=True)

    # Shuffle the balanced dataframe
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\nBalanced class distribution:")
    print(balanced_df[target_column].value_counts())

    return balanced_df

# Factory function that instantiates the selected missing-data imputer.
# All options are fit exclusively on training data and applied to both
# train and test splits to prevent information leakage.
# "mean"   -- unconditional mean imputation
# "median" -- unconditional median imputation
# "knn"    -- k-nearest-neighbour imputation (k = 5)
# "mice"   -- multiple imputation by chained equations
def get_imputer(imputer_type):
    if imputer_type == "mean":
        return SimpleImputer(strategy='mean')
    elif imputer_type == "median":
        return SimpleImputer(strategy='median')
    elif imputer_type == "knn":
        return KNNImputer(n_neighbors=5)
    elif imputer_type == "mice":
        return IterativeImputer(random_state=0)
    else:
        raise ValueError(f"Unknown imputer type: {imputer_type}")

def drop_high_missing_columns(df, threshold=0.7, verbose=False):
    """
    Drop columns with more than threshold% missing values

    Args:
        df: pandas DataFrame
        threshold: float between 0 and 1, representing maximum allowed missing percentage
        verbose: bool, whether to print dropped columns

    Returns:
        DataFrame with high-missing columns dropped
    """
    # Calculate percentage of missing values
    missing_percentages = df.isna().sum() / len(df)

    # Get columns to drop
    cols_to_drop = missing_percentages[missing_percentages > threshold].index.tolist()

    if verbose and cols_to_drop:
        print(f"\nDropping {len(cols_to_drop)} columns with >{threshold*100}% missing values:")
        for col in cols_to_drop:
            print(f"{col}: {missing_percentages[col]*100:.1f}%")

    return df.drop(columns=cols_to_drop)

# ---
# within_categories_data()
#
# Prepares train/test splits for within-category analyses. Each key in the
# returned dictionary corresponds to one predictor category (e.g., "Anxiety",
# "Task fMRI"); the value is the (X_train, X_test, y_train, y_test) tuple for
# that category's variable subset.
#
# Processing order (leakage-free):
# 1. Select child or parent variable structure based on target.
# 2. Apply category-level circular exclusions from CIRCULAR_EXCLUSIONS.
# 3. Drop columns with >70 % missing values.
# 4. One-hot-encode nominal covariates.
# 5. Split raw data 70/30 (stratified for classification).
# 6. Fit imputer and optional scaler on training fold only; transform both.
# 7. Return per-category slices of the imputed/scaled arrays.
# ---
def within_categories_data(t, targets, group=None, scale_x=True, scale_y=True, verbose=False, class_rebalancing=False, feature_tp=None):

  # -- Resolve feature timepoint ---
  feat_t = feature_tp if feature_tp is not None else t
  _cross_tp = (feature_tp is not None and feature_tp != t)

  # Route to child or parent variable hierarchy depending on the target.
  global variables_for_each_time_point
  target_name = targets[0] if targets else ''
  if is_parent_target(target_name):
      variables_for_each_time_point = parent_variables_for_each_time_point
      print(f"[VARS] Using PARENT within-category structure for target: {target_name}")
  else:
      variables_for_each_time_point = child_variables_for_each_time_point
      print(f"[VARS] Using CHILD within-category structure for target: {target_name}")

  # -- Build active categories from the FEATURE timepoint ---
  # Circular exclusions are always keyed on the TARGET timepoint (t).
  across_excl, within_excl = get_circular_exclusions(target_name, timepoint=t)
  active_vfet_tp = {
      cat: vars_list for cat, vars_list in variables_for_each_time_point[feat_t].items()
      if cat not in within_excl
  }
  if within_excl:
      skipped = within_excl & set(variables_for_each_time_point[feat_t].keys())
      if skipped:
          print(f"[CIRCULAR] Excluded within-categories for {target_name}: {skipped}")

  # Flatten retained categories to a single variable list, removing any
  # individually excluded circular variables within those categories.
  within_categories = [
        x
        for xs in active_vfet_tp.values()
        for x in xs
        if x not in across_excl
      ] + targets
  if across_excl:
      individual_removed = len([x for xs in active_vfet_tp.values() for x in xs if x in across_excl])
      if individual_removed > 0:
          print(f"[CIRCULAR] Also removed {individual_removed} individual circular variables from within-category features for {target_name}")

  within_categories = list(set(within_categories))

  # Get available columns in the DataFrame
  available_columns = set(sample.columns)

  # Remove columns from within_categories that are not in available_columns
  within_categories = [var for var in within_categories if var in available_columns]

  # If any columns were removed, notify the user
  dropped_columns = set([x for xs in variables_for_each_time_point[feat_t].values() for x in xs] + targets).difference(available_columns)

  if dropped_columns:
      print(f"WARNING: The following columns were not found in the dataset and have been dropped: {dropped_columns}")

  # Remove peer-victimisation variables absent/unreliable at T1 and T3 (feature side)
  if int(feat_t) == 3 or int(feat_t) == 1:
      variables_to_remove = [
          'peq_ss_relational_victim',
          'peq_ss_reputation_aggs',
          'peq_ss_reputation_victim',
          'peq_ss_overt_aggression',
          'peq_ss_overt_victim',
          'peq_ss_relational_aggs'
      ]
      within_categories = [var for var in within_categories if var not in variables_to_remove]

  # Apply circular exclusions so kfold path is same as standard path.
  _across_excl_kw, _within_excl_kw = get_circular_exclusions(targets[0] if targets else '', timepoint=t)
  if _within_excl_kw:
      within_categories = [
          x for x in within_categories
          if x in targets or not any(x in variables_for_each_time_point[feat_t].get(cat, []) for cat in _within_excl_kw)
      ]
  if _across_excl_kw:
      within_categories = [x for x in within_categories if x not in _across_excl_kw or x in targets]
      if verbose:
          print(f"[CIRCULAR/kfold] Excluded {len(_across_excl_kw)} circular variables for {targets[0] if targets else '?'}")

  # -- Build all_data ---
  _feat_only_vars = [v for v in within_categories if v not in targets]

  if _cross_tp:
      # -- Cross-timepoint path: features from feat_t, targets from t ---
      # Group filter applied to TARGET wave so subgroup membership is
      # defined at the outcome timepoint.
      tgt_rows = sample[sample['time'] == t].copy()
      if group:
          if group == 'high_ale':
              tgt_rows = tgt_rows[tgt_rows['high_ale'] == True]
          elif group == 'high_ale_severe_p_mh':
              tgt_rows = tgt_rows[tgt_rows['high_ale_severe_p_mh'] == True]
          elif group == 'low_ale':
              tgt_rows = tgt_rows[tgt_rows['low_ale_children_p'] == True]
          elif group == 'Female':
              tgt_rows = tgt_rows[tgt_rows['sex'] == 2]
          elif group == 'Male':
              tgt_rows = tgt_rows[tgt_rows['sex'] == 1]

      valid_subjects = tgt_rows['subject'].values

      # Feature-wave rows, restricted to group-filtered subjects
      feat_rows = sample[
          (sample['time'] == feat_t) &
          (sample['subject'].isin(valid_subjects))
      ].copy()

      # Merge: features from feat_t, targets from t, on subject
      feat_cols_avail = [c for c in _feat_only_vars if c in feat_rows.columns]
      tgt_cols_avail  = [c for c in targets if c in tgt_rows.columns]

      _feat_df = feat_rows[['subject'] + feat_cols_avail].drop_duplicates('subject')
      _tgt_df  = tgt_rows[['subject'] + tgt_cols_avail].drop_duplicates('subject')

      all_data = _feat_df.merge(_tgt_df, on='subject', how='inner').drop(columns='subject')

      if verbose:
          print(f"[Cross-TP within] feat_tp={feat_t}, target_tp={t}, "
                f"group={group if group else 'full'}, merged N={len(all_data)}")

  else:
      # -- Same-timepoint path: original logic preserved exactly ---
      if group:
        if group == 'high_ale':
          all_vars = within_categories + [f"{group}"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data[all_data['high_ale'] == True] # high_ale (old)

        elif group == 'high_ale_severe_p_mh':
          all_vars = within_categories + [f"{group}"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data[all_data['high_ale_severe_p_mh'] == True] # high_ale (old)

        elif group == 'low_ale':
          group = 'low_ale_children_p'
          all_vars =  within_categories + [f"{group}"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data[all_data['low_ale_children_p'] == True]

        elif group == 'Female':
          all_vars =  within_categories + [f"sex"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data.loc[:, ~all_data.columns.duplicated()]
          all_data = all_data[all_data['sex'] == 2]

        elif group == 'Male':
          all_vars =  within_categories + [f"sex"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data.loc[:, ~all_data.columns.duplicated()]
          all_data = all_data[all_data['sex'] == 1]

        else:
          all_vars = within_categories
          all_data = sample.query(f"time == {t}")[all_vars]

      else:
        all_vars = within_categories
        all_data = sample.query(f"time == {t}")[all_vars]

  if verbose:
    print(f"Gathered within category data from {all_data.shape[0]} subjects for {group if group else 'full'} group at t = {t}")

  if 'depress_D_p' not in targets and 'depress_D_p_rev' not in targets and "external_D_p" not in targets and "adhd_D_p" not in targets: # ("latent_class_depression" in targets):
    if "latent_class_depression" in targets:
      # latent_class_depression has sporadic missing/empty entries; drop them.
      all_data = all_data[(all_data["latent_class_depression"] != "")]
    # Class rebalancing deferred to AFTER split
    # to prevent oversampled duplicates from appearing in both train and test.
    _needs_rebalance = class_rebalancing

  available = drop_high_missing_columns(all_data, threshold=0.7, verbose=verbose)
  dropped_columns = set(all_data.columns).difference(set(available.columns))

  if dropped_columns:
    print(f"WARNING: Dropped columns at t = {t} due to entirely missing data: {dropped_columns}")

  raw_X = available.drop(targets, axis=1)
  raw_y = available[targets]

  # -- Drop rows where target is NaN (parent ASR items can have sporadic missingness) --
  _target_notna = raw_y.iloc[:, 0].notna()
  if (~_target_notna).any():
      _n_dropped = (~_target_notna).sum()
      print(f"Dropped {_n_dropped} rows with missing target values")
      raw_X = raw_X[_target_notna]
      raw_y = raw_y[_target_notna]

  # Before imputation and scaling
  # One Hot Encoding
  # OHE is fit on pre-split data. For categorical columns with
  # handle_unknown='ignore', this is not statistical leakage (no learned
  # statistics), but fit-on-train-only would be stricter.
  one_hot_features = ['sex', 'sex_P', 'L_site_id', 'marital_status', 'child_religion', 'sex_orient_y', 'trans_id_y', 'crysflu_c', 'anxadhd_c']
  encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

  for feature in one_hot_features:
      if feature in raw_X.columns:
          print(f"Encoding {feature} for one hot encoding..")
          # Reshape data for encoding
          raw_X_cat = raw_X[[feature]].astype(str)

          # Fit and transform
          encoded_data = encoder.fit_transform(raw_X_cat)

          # Create column names for encoded features
          unique_values = encoder.categories_[0][1:]
          encoded_columns = [f"{feature}_{val}" for val in unique_values]

          # Convert to DataFrame
          encoded_df = pd.DataFrame(
              encoded_data,
              columns=encoded_columns,
              index=raw_X.index
          )

          # Replace original column with encoded columns
          raw_X = pd.concat([raw_X.drop(feature, axis=1), encoded_df], axis=1)
          print(f"Encoded {feature} into {len(encoded_columns)} categories")

  # -- Preserve original index for downstream demographic lookups --
  # Store the original index BEFORE resetting, so later cells can
  # align X_train/X_test rows back to the `sample` DataFrame.
  _original_index = raw_X.index.copy()
  raw_X = raw_X.reset_index(drop=True)
  raw_y = raw_y.reset_index(drop=True)

  # -- Leakage-free split → impute → scale ---
  # Raw data is split before any preprocessing estimator is fit. Imputation
  # and standardisation parameters are estimated on X_train_raw only.
  print("Splitting data prior to preprocessing...")
  X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
      raw_X, raw_y, test_size=0.3, random_state=42,
      stratify=raw_y if (not is_regression(targets[0] if isinstance(targets, list) else targets) and is_classification(raw_y)) else None
  )
  # -- Apply class rebalancing to TRAINING data only --
  if '_needs_rebalance' in dir() and _needs_rebalance:
      _train_combined = pd.concat([X_train_raw, y_train_raw], axis=1)
      _train_balanced = simple_balance_classes(_train_combined, target_column=str(targets[0]) if isinstance(targets, list) else str(targets))
      X_train_raw = _train_balanced.drop(targets, axis=1, errors='ignore')
      y_train_raw = _train_balanced[targets] if isinstance(targets, list) else _train_balanced[[targets]]
      print(f"  Class rebalancing applied to TRAINING data only ({len(X_train_raw)} samples)")

  # Fit imputer on TRAINING ONLY
  selected_imputer = get_imputer(imputer)
  print("Imputing the data (fit on train only)...")
  selected_imputer.fit(X_train_raw)

  X_train = pd.DataFrame(
      selected_imputer.transform(X_train_raw),
      columns=X_train_raw.columns, index=X_train_raw.index
  )
  X_test = pd.DataFrame(
      selected_imputer.transform(X_test_raw),
      columns=X_test_raw.columns, index=X_test_raw.index
  )

  # Handle y imputation (separate imputer -- avoids X-fitted statistics on y)
  y_train = y_train_raw.copy()
  y_test = y_test_raw.copy()
  try:
      from sklearn.impute import SimpleImputer as _SI
      _y_imputer = _SI(strategy='median')
      _y_imputer.fit(y_train_raw)
      y_train = pd.DataFrame(
          _y_imputer.transform(y_train_raw),
          columns=y_train_raw.columns, index=y_train_raw.index
      )
      y_test = pd.DataFrame(
          _y_imputer.transform(y_test_raw),
          columns=y_test_raw.columns, index=y_test_raw.index
      )
  except Exception as e:
      print("raw_y does not need to be imputed")

  # Scale X if requested
  print("Scaling the data (fit on train only)...")
  if scale_x:
      scaler_x = StandardScaler()
      try:
          scaler_x.fit(X_train)
          X_train = pd.DataFrame(
              scaler_x.transform(X_train),
              columns=X_train.columns, index=X_train.index
          )
          X_test = pd.DataFrame(
              scaler_x.transform(X_test),
              columns=X_test.columns, index=X_test.index
          )
      except Exception as e:
          print("raw_X does not need to be scaled")

  # Scale y if requested (skip for classification targets)
  if scale_y and not is_classification(raw_y):
      scaler_y = StandardScaler()
      try:
          scaler_y.fit(y_train.values.reshape(-1, 1))
          y_train = pd.Series(
              scaler_y.transform(y_train.values.reshape(-1, 1)).flatten(),
              index=y_train.index,
              name=y_train.name if hasattr(y_train, 'name') else None
          )
          y_test = pd.Series(
              scaler_y.transform(y_test.values.reshape(-1, 1)).flatten(),
              index=y_test.index,
              name=y_test.name if hasattr(y_test, 'name') else None
          )
      except Exception as e:
          print("raw_y does not need to be scaled")

  # ========================================================================

  ret = {}

  try:
    for (category, vars) in active_vfet_tp.items():
      available_vars = list(set(vars).difference(dropped_columns))
      ret[category] = (
          X_train[available_vars], X_test[available_vars],
          y_train, y_test
      )
  except KeyError:
    # Category variable list references a column not yet in imputed X; rebuild defensively.
    for (category, vars) in active_vfet_tp.items():
        # Get the available variables after removing dropped columns
        available_vars = list(set(vars).difference(dropped_columns))

        # Further filter to ensure available_vars exist in both X_train and X_test
        available_vars = [var for var in available_vars if var in X_train.columns and var in X_test.columns]

        if available_vars:
            ret[category] = (
                X_train[available_vars], X_test[available_vars],
                y_train, y_test
            )
        else:
            print(f"WARNING: No available variables left for category: {category}")

  return ret

# ---
# across_categories_data()
#
# Prepares train/test splits for across-category analyses, drawing from the
# full flat predictor list (t0_variables … t4_variables) at the selected wave.
# Returns a single (X_train, X_test, y_train, y_test) tuple.
#
# Applies the same leakage-free preprocessing sequence as
# within_categories_data(): circular exclusion → missing-data filtering →
# one-hot encoding → raw split → imputation (train-only) → optional scaling.
# ---
def across_categories_data(t, targets, group=None, scale_x=True, scale_y=True, verbose=False, class_rebalancing=False, feature_tp=None):
  # feature_tp: if set, pull FEATURES from this timepoint and TARGETS from t.
  # None (default) preserves existing behaviour (features and targets same wave).
  feat_t = feature_tp if feature_tp is not None else t

  available_columns = set(sample.columns)

  if feat_t == 0:
    across_categories = list(t0_variables)
  elif feat_t == 1:
    across_categories = list(t1_variables)
  elif feat_t == 2:
    across_categories = list(t2_variables)
  elif feat_t == 3:
    across_categories = list(t3_variables)
  elif feat_t == 4:
    across_categories = list(t4_variables)
  else:
    raise ValueError(f"Invalid feature time point: {feat_t}")

  # Circular exclusions are always keyed on the TARGET timepoint (t), not feat_t.
  # Cross-tp runs (e.g. T0 features → T2 target) are inherently safe by design,
  # but any target-wave exclusions still apply to keep the pool clean.
  target_name = targets[0] if targets else ''
  across_excl, _ = get_circular_exclusions(target_name, timepoint=t)
  if across_excl:
      before_count = len(across_categories)
      across_categories = [v for v in across_categories if v not in across_excl]
      removed_count = before_count - len(across_categories)
      if removed_count > 0:
          print(f"[CIRCULAR] Removed {removed_count} circular variables from across-category pool for target: {target_name}")

  # Remove columns not present in dataset (based on feat_t variable space)
  across_categories = [var for var in across_categories if var in available_columns]

  # Warn about any expected variables missing from the dataset (use feat_t)
  dropped_columns = set([x for xs in variables_for_each_time_point[feat_t].values() for x in xs] + targets).difference(available_columns)
  if dropped_columns:
      print(f"WARNING: The following columns were not found in the dataset and have been dropped: {dropped_columns}")

  # Remove peer-victimisation variables absent/unreliable at T1 and T3 (feature side)
  if int(feat_t) == 3 or int(feat_t) == 1:
      variables_to_remove = [
          'peq_ss_relational_victim',
          'peq_ss_reputation_aggs',
          'peq_ss_reputation_victim',
          'peq_ss_overt_aggression',
          'peq_ss_overt_victim',
          'peq_ss_relational_aggs'
      ]
      across_categories = [var for var in across_categories if var not in variables_to_remove]

  # -- Build all_data ---
  # Cross-timepoint path: features from feat_t rows, targets from t rows,
  # merged on subject ID. Group filter is applied to the TARGET wave so that
  # subgroup membership (e.g. sex, ALE) is defined at the outcome timepoint.
  if feature_tp is not None and feature_tp != t:
      # 1. Get target-wave rows and apply any group filter
      tgt_rows = sample[sample['time'] == t].copy()
      if group == 'high_ale':
          tgt_rows = tgt_rows[tgt_rows['high_ale'] == True]
      elif group == 'high_ale_severe_p_mh':
          tgt_rows = tgt_rows[tgt_rows['high_ale_severe_p_mh'] == True]
      elif group == 'low_ale':
          tgt_rows = tgt_rows[tgt_rows['low_ale_children_p'] == True]
      elif group == 'Female':
          tgt_rows = tgt_rows[tgt_rows['sex'] == 2]
      elif group == 'Male':
          tgt_rows = tgt_rows[tgt_rows['sex'] == 1]

      valid_subjects = tgt_rows['subject'].values

      # 2. Get feature-wave rows, restricted to the group-filtered subjects
      feat_rows = sample[
          (sample['time'] == feat_t) &
          (sample['subject'].isin(valid_subjects))
      ].copy()

      # 3. Merge: features from feat_t, target from t, on subject
      feat_cols_avail = [c for c in across_categories if c in feat_rows.columns]
      tgt_cols_avail  = [c for c in targets if c in tgt_rows.columns]

      _feat_df = feat_rows[['subject'] + feat_cols_avail].drop_duplicates('subject')
      _tgt_df  = tgt_rows[['subject'] + tgt_cols_avail].drop_duplicates('subject')

      all_data = _feat_df.merge(_tgt_df, on='subject', how='inner').drop(columns='subject')

      if verbose:
          print(f"[Cross-TP] feat_tp={feat_t}, target_tp={t}, "
                f"group={group if group else 'full'}, merged N={len(all_data)}")

  # -- Same-timepoint path: original logic preserved exactly ---
  else:
      if group:
          if group == 'high_ale':
              all_vars =  across_categories + targets + [f"{group}"]
              all_data = sample[sample['time']==t][all_vars]
              all_data = all_data[all_data['high_ale'] == True] # high_ale

          elif group == 'high_ale_severe_p_mh':
              all_vars =  across_categories + targets + [f"{group}"]
              all_data = sample[sample['time']==t][all_vars]
              all_data = all_data[all_data['high_ale_severe_p_mh'] == True] # high_ale

          elif group == 'low_ale':
              all_vars =  across_categories + targets + ['low_ale_children_p']
              all_data = sample[sample['time']==t][all_vars]
              all_data = all_data[all_data['low_ale_children_p'] == True]

          elif group == 'Female':
              all_vars =  across_categories + targets + ["sex"]
              all_data = sample[sample['time']==t][all_vars]
              all_data = all_data.loc[:, ~all_data.columns.duplicated()]
              all_data = all_data[all_data['sex'] == 2]

          elif group == 'Male':
              all_vars =  across_categories + targets + ["sex"]
              all_data = sample[sample['time']==t][all_vars]
              all_data = all_data.loc[:, ~all_data.columns.duplicated()]
              all_data = all_data[all_data['sex'] == 1]

          else:
              all_vars = across_categories + targets
              all_data = sample.query(f"time == {t}")[all_vars]

      else:
          all_vars = across_categories + targets
          all_data = sample.query(f"time == {t}")[all_vars]

      if verbose:
          print(f"Gathered across category data from {all_data.shape[0]} subjects for {group if group else 'full'} group at t = {t}")

  # -- Downstream processing ---
  if 'depress_D_p' not in targets and 'depress_D_p_rev' not in targets and "external_D_p" not in targets and "adhd_D_p" not in targets: # ("latent_class_depression" in targets):
    # latent_class_depression has sporadic missing/empty entries; drop them.
    if 'latent_class_depression' in targets:
      all_data = all_data[(all_data["latent_class_depression"] != "")]
    # Class rebalancing deferred to AFTER split
    # to prevent oversampled duplicates from appearing in both train and test.
    _needs_rebalance = class_rebalancing

  available = drop_high_missing_columns(all_data, threshold=0.7, verbose=verbose)
  dropped_columns = set(all_data.columns).difference(set(available.columns))

  if dropped_columns:
    print(f"WARNING: Dropped columns at t = {t} due to entirely missing data: {dropped_columns}")

  raw_X = available.drop(targets, axis=1)
  raw_y = available[targets]

  # -- Drop rows where target is NaN (parent ASR items can have sporadic missingness) --
  _target_notna = raw_y.iloc[:, 0].notna()
  if (~_target_notna).any():
      _n_dropped = (~_target_notna).sum()
      print(f"Dropped {_n_dropped} rows with missing target values")
      raw_X = raw_X[_target_notna]
      raw_y = raw_y[_target_notna]

  # Before imputation and scaling
  # One Hot Encoding
  one_hot_features = ['sex', 'sex_P', 'L_site_id', 'marital_status', 'child_religion', 'sex_orient_y', 'trans_id_y', 'crysflu_c', 'anxadhd_c']
  encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

  for feature in one_hot_features:
      if feature in raw_X.columns:
          print(f"Encoding {feature} for one hot encoding..")
          # Reshape data for encoding
          raw_X_cat = raw_X[[feature]].astype(str)

          # Fit and transform
          encoded_data = encoder.fit_transform(raw_X_cat)

          # Create column names for encoded features
          unique_values = encoder.categories_[0][1:]
          encoded_columns = [f"{feature}_{val}" for val in unique_values]

          # Convert to DataFrame
          encoded_df = pd.DataFrame(
              encoded_data,
              columns=encoded_columns,
              index=raw_X.index
          )

          # Replace original column with encoded columns
          raw_X = pd.concat([raw_X.drop(feature, axis=1), encoded_df], axis=1)
          print(f"Encoded {feature} into {len(encoded_columns)} categories")

  # -- Preserve original index for downstream demographic lookups --
  _original_index = raw_X.index.copy()
  raw_X = raw_X.reset_index(drop=True)
  raw_y = raw_y.reset_index(drop=True)

  # Split before fitting any preprocessing estimator to prevent leakage.
  # Imputer and scaler are fit on X_train_raw only, then applied to both splits.

  # Split raw data FIRST
  print("Splitting data (before preprocessing to prevent leakage)...")
  X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
      raw_X, raw_y, test_size=0.3, random_state=42,
      stratify=raw_y if (not is_regression(targets[0] if isinstance(targets, list) else targets) and is_classification(raw_y)) else None
  )
  # -- Apply class rebalancing to TRAINING data only --
  if '_needs_rebalance' in dir() and _needs_rebalance:
      _train_combined = pd.concat([X_train_raw, y_train_raw], axis=1)
      _train_balanced = simple_balance_classes(_train_combined, target_column=str(targets[0]) if isinstance(targets, list) else str(targets))
      X_train_raw = _train_balanced.drop(targets, axis=1, errors='ignore')
      y_train_raw = _train_balanced[targets] if isinstance(targets, list) else _train_balanced[[targets]]
      print(f"  Class rebalancing applied to TRAINING data only ({len(X_train_raw)} samples)")

  # Fit imputer on TRAINING ONLY
  selected_imputer = get_imputer(imputer)
  print("Imputing the data (fit on train only)...")
  selected_imputer.fit(X_train_raw)

  X_train = pd.DataFrame(
      selected_imputer.transform(X_train_raw),
      columns=X_train_raw.columns, index=X_train_raw.index
  )
  X_test = pd.DataFrame(
      selected_imputer.transform(X_test_raw),
      columns=X_test_raw.columns, index=X_test_raw.index
  )

  # Handle y imputation (separate imputer -- avoids X-fitted statistics on y)
  y_train = y_train_raw.copy()
  y_test = y_test_raw.copy()
  try:
      from sklearn.impute import SimpleImputer as _SI
      _y_imputer = _SI(strategy='median')
      _y_imputer.fit(y_train_raw)
      y_train = pd.DataFrame(
          _y_imputer.transform(y_train_raw),
          columns=y_train_raw.columns, index=y_train_raw.index
      )
      y_test = pd.DataFrame(
          _y_imputer.transform(y_test_raw),
          columns=y_test_raw.columns, index=y_test_raw.index
      )
  except Exception as e:
      print("raw_y does not need to be imputed")

  # Scale X if requested (fit on TRAINING ONLY)
  print("Scaling the data (fit on train only)...")
  if scale_x:
      scaler_x = StandardScaler()
      try:
          scaler_x.fit(X_train)
          X_train = pd.DataFrame(
              scaler_x.transform(X_train),
              columns=X_train.columns, index=X_train.index
          )
          X_test = pd.DataFrame(
              scaler_x.transform(X_test),
              columns=X_test.columns, index=X_test.index
          )
      except Exception as e:
          print("raw_X does not need to be scaled")

  # Scale y if requested (skip for classification targets)
  if scale_y and not is_classification(raw_y):
      scaler_y = StandardScaler()
      try:
          scaler_y.fit(y_train)
          y_train = pd.DataFrame(
              scaler_y.transform(y_train),
              columns=y_train.columns, index=y_train.index
          )
          y_test = pd.DataFrame(
              scaler_y.transform(y_test),
              columns=y_test.columns, index=y_test.index
          )
      except Exception as e:
          print("raw_y does not need to be scaled")

  return X_train, X_test, y_train, y_test

# ---
# within_categories_data_combinations()
#
# Exploratory function: pairs a single base category with each remaining
# category in turn and prepares a merged train/test split for each pair.
# Useful for assessing the marginal contribution of individual categories
# when combined with a fixed reference domain (e.g., neuroimaging + each
# psychosocial domain).
# ---
def within_categories_data_combinations(t, targets, base_category, group=None, scale_x=True, scale_y=True, verbose=False, class_rebalancing=False):

  # -- Route to child or parent variable hierarchy depending on target --
  global variables_for_each_time_point
  target_name = targets[0] if targets else ''
  if is_parent_target(target_name):
      variables_for_each_time_point = parent_variables_for_each_time_point
      print(f"[VARS] Using PARENT within-category structure for target: {target_name}")
  else:
      variables_for_each_time_point = child_variables_for_each_time_point
      print(f"[VARS] Using CHILD within-category structure for target: {target_name}")

  # -- Apply circular exclusions --
  across_excl, within_excl = get_circular_exclusions(target_name, timepoint=t)
  active_vfet_tp = {
      cat: vars_list for cat, vars_list in variables_for_each_time_point[t].items()
      if cat not in within_excl
  }
  if within_excl:
      skipped = within_excl & set(variables_for_each_time_point[t].keys())
      if skipped:
          print(f"[CIRCULAR] Excluded within-categories for {target_name}: {skipped}")

  # Flatten retained categories, removing individually excluded circular variables.
  within_categories = [
        x
        for xs in active_vfet_tp.values()
        for x in xs
        if x not in across_excl
      ] + targets
  if across_excl:
      individual_removed = len([x for xs in active_vfet_tp.values() for x in xs if x in across_excl])
      if individual_removed > 0:
          print(f"[CIRCULAR] Also removed {individual_removed} individual circular variables from within-category features for {target_name}")

  within_categories = list(set(within_categories))

  # Get available columns in the DataFrame
  available_columns = set(sample.columns)

  # Remove columns from within_categories that are not in available_columns
  within_categories = [var for var in within_categories if var in available_columns]

  # If any columns were removed, notify the user
  dropped_columns = set([x for xs in variables_for_each_time_point[t].values() for x in xs] + targets).difference(available_columns)

  if dropped_columns:
      print(f"WARNING: The following columns were not found in the dataset and have been dropped: {dropped_columns}")

  # Remove specific variables
  if int(t) == 3 or int(t) == 1:
      variables_to_remove = [
          'peq_ss_relational_victim',
          'peq_ss_reputation_aggs',
          'peq_ss_reputation_victim',
          'peq_ss_overt_aggression',
          'peq_ss_overt_victim',
          'peq_ss_relational_aggs'
      ]
      within_categories = [var for var in within_categories if var not in variables_to_remove]

  ret = {}
  if base_category:
    # Use actual timepoint t (not hardcoded 0) for category enumeration.
    # Filter out circular-excluded categories from pairing candidates.
    other_categories = [cat for cat in active_vfet_tp.keys() if cat != base_category]
    print(f"Mixing {base_category} with: {other_categories}")

    # Avoid mutating the group parameter across loop iterations.
    _resolved_group = 'low_ale_children_p' if group == 'low_ale' else group

    for category in other_categories:

      if _resolved_group:
        if _resolved_group == 'high_ale':
          all_vars = within_categories + [f"{_resolved_group}"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data[all_data['high_ale'] == True]

        elif _resolved_group == 'high_ale_severe_p_mh':
          all_vars = within_categories + [f"{_resolved_group}"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data[all_data['high_ale_severe_p_mh'] == True]

        elif _resolved_group == 'low_ale_children_p':
          all_vars =  within_categories + ['low_ale_children_p']
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data[all_data['low_ale_children_p'] == True]

        elif _resolved_group == 'Female':
          all_vars =  within_categories + ["sex"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data.loc[:, ~all_data.columns.duplicated()]
          all_data = all_data[all_data['sex'] == 2]

        elif _resolved_group == 'Male':
          all_vars =  within_categories + ["sex"]
          all_data = sample[sample['time']==t][all_vars]
          all_data = all_data.loc[:, ~all_data.columns.duplicated()]
          all_data = all_data[all_data['sex'] == 1]

        else:
          all_vars = within_categories
          all_data = sample.query(f"time == {t}")[all_vars]

      else:
        all_vars = within_categories
        all_data = sample.query(f"time == {t}")[all_vars]

      try:
        variables_total = list(active_vfet_tp[f'{base_category}']) + list(active_vfet_tp[f'{category}'])
      except Exception as e:
        print(f"  Skipped: '{category}' not available for this tp/target/group combination.")
        continue

      # Apply across_excl to the per-combo variable list as well.
      if across_excl:
          variables_total = [v for v in variables_total if v not in across_excl]

      variables_total += targets

      if verbose:
          print(f"\nProcessing {base_category} + {category}")
          print(f"Total variables before filtering: {len(variables_total)}")

      # Filter out variables that aren't in the dataset
      available_vars = [var for var in variables_total if var in sample.columns]
      missing_vars = set(variables_total) - set(available_vars)

      if missing_vars and verbose:
          print(f"WARNING: The following variables were not found in the dataset:\n{sorted(missing_vars)}")
          print(f"Continuing with {len(available_vars)} available variables")

      # Subset only the available vars
      available_vars = list(dict.fromkeys(available_vars))  # Remove duplicates while preserving order
      all_data = all_data[available_vars]

      if 'depress_D_p' not in targets and 'depress_D_p_rev' not in targets and "external_D_p" not in targets and "adhd_D_p" not in targets:
        if "latent_class_depression" in targets:
          all_data = all_data[(all_data["latent_class_depression"] != "")]
        if class_rebalancing:
          all_data = simple_balance_classes(all_data, target_column = f"{target_options}")
          display(all_data[f'{target_options}'].value_counts())

      available = drop_high_missing_columns(all_data, threshold=0.7, verbose=verbose)
      dropped_columns = set(all_data.columns).difference(set(available.columns))

      if dropped_columns:
        print(f"WARNING: Dropped columns at t = {t} due to entirely missing data: {dropped_columns}")

      raw_X = available.drop(targets, axis=1)
      raw_y = available[targets]

      # -- Drop rows where target is NaN --
      _target_notna = raw_y.iloc[:, 0].notna()
      if (~_target_notna).any():
          _n_dropped = (~_target_notna).sum()
          print(f"Dropped {_n_dropped} rows with missing target values")
          raw_X = raw_X[_target_notna]
          raw_y = raw_y[_target_notna]

      # One Hot Encoding
      one_hot_features = ['sex', 'sex_P', 'L_site_id', 'marital_status', 'child_religion', 'sex_orient_y', 'trans_id_y', 'crysflu_c', 'anxadhd_c']
      encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

      for feature in one_hot_features:
          if feature in raw_X.columns:
              print(f"Encoding {feature} for one hot encoding..")
              raw_X_cat = raw_X[[feature]].astype(str)
              encoded_data = encoder.fit_transform(raw_X_cat)
              unique_values = encoder.categories_[0][1:]
              encoded_columns = [f"{feature}_{val}" for val in unique_values]
              encoded_df = pd.DataFrame(
                  encoded_data,
                  columns=encoded_columns,
                  index=raw_X.index
              )
              raw_X = pd.concat([raw_X.drop(feature, axis=1), encoded_df], axis=1)
              print(f"Encoded {feature} into {len(encoded_columns)} categories")

      # Reset to RangeIndex so positional indices align
      raw_X = raw_X.reset_index(drop=True)
      raw_y = raw_y.reset_index(drop=True)

      # Split raw data before fitting any preprocessing estimator.
      print("Splitting data prior to preprocessing...")
      X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
          raw_X, raw_y, test_size=0.3, random_state=42,
          stratify=raw_y if (not is_regression(targets[0] if isinstance(targets, list) else targets) and is_classification(raw_y)) else None
      )

      # Fit imputer on TRAINING ONLY
      selected_imputer = get_imputer(imputer)
      print("Imputing the data (fit on train only)...")
      selected_imputer.fit(X_train_raw)

      X_train = pd.DataFrame(
          selected_imputer.transform(X_train_raw),
          columns=X_train_raw.columns, index=X_train_raw.index
      )
      X_test = pd.DataFrame(
          selected_imputer.transform(X_test_raw),
          columns=X_test_raw.columns, index=X_test_raw.index
      )

      # Handle y imputation (separate imputer)
      y_train = y_train_raw.copy()
      y_test = y_test_raw.copy()
      try:
          from sklearn.impute import SimpleImputer as _SI
          _y_imputer = _SI(strategy='median')
          _y_imputer.fit(y_train_raw)
          y_train = pd.DataFrame(
              _y_imputer.transform(y_train_raw),
              columns=y_train_raw.columns, index=y_train_raw.index
          )
          y_test = pd.DataFrame(
              _y_imputer.transform(y_test_raw),
              columns=y_test_raw.columns, index=y_test_raw.index
          )
      except Exception as e:
          print("raw_y does not need to be imputed")

      # Scale X if requested (fit on TRAINING ONLY)
      print("Scaling the data (fit on train only)...")
      if scale_x:
          scaler_x = StandardScaler()
          try:
              scaler_x.fit(X_train)
              X_train = pd.DataFrame(
                  scaler_x.transform(X_train),
                  columns=X_train.columns, index=X_train.index
              )
              X_test = pd.DataFrame(
                  scaler_x.transform(X_test),
                  columns=X_test.columns, index=X_test.index
              )
          except Exception as e:
              print("raw_X does not need to be scaled")

      # Scale y if requested (fit on TRAINING ONLY)
      if scale_y:
          scaler_y = StandardScaler()
          try:
              scaler_y.fit(y_train)
              y_train = pd.DataFrame(
                  scaler_y.transform(y_train),
                  columns=y_train.columns, index=y_train.index
              )
              y_test = pd.DataFrame(
                  scaler_y.transform(y_test),
                  columns=y_test.columns, index=y_test.index
              )
          except Exception as e:
              print("raw_y does not need to be scaled")

      # Store results
      combination_name = f"{base_category} + {category}"
      ret[combination_name] = (X_train, X_test, y_train, y_test)

  return ret

# ---
# Quantile Analysis Utilities
#
# These functions segment data by predictor or target quantiles to examine
# whether predictive performance varies across the distribution of a given
# feature or outcome. Used for exploratory heterogeneity analyses.
# ---

def analyze_category_quantiles(data_dict):
    """Compute Q1/Q2/Q3 quantile boundaries and bin counts for every feature
    within each category.  Returns a nested dict keyed by category → feature.

    Parameters
    data_dict : dict
        {category: (X_train, X_test, y_train, y_test)} as returned by
        within_categories_data().

    Returns
    dict
        Nested structure: {category: {'total_samples': int,
        'feature_quantiles': {feature: {Q1/Q2/Q3 counts and values}}}}
    """
    results = {}

    for category, (X_train, X_test, y_train, y_test) in data_dict.items():
        # Combine train and test data for complete picture
        X_combined = pd.concat([X_train, X_test])

        # Calculate quantiles
        quantiles = X_combined.quantile([0.25, 0.5, 0.75])

        # Count data points in each quantile range
        quantile_counts = {}

        # For each column in the dataset
        for column in X_combined.columns:
            q1, q2, q3 = quantiles.loc[0.25, column], quantiles.loc[0.5, column], quantiles.loc[0.75, column]

            counts = {
                'Q1 (0-25%)': len(X_combined[X_combined[column] <= q1]),
                'Q2 (25-50%)': len(X_combined[(X_combined[column] > q1) & (X_combined[column] <= q2)]),
                'Q3 (50-75%)': len(X_combined[(X_combined[column] > q2) & (X_combined[column] <= q3)]),
                'Q4 (75-100%)': len(X_combined[X_combined[column] > q3]),
                'quantile_values': {
                    'Q1': q1,
                    'Q2 (median)': q2,
                    'Q3': q3
                }
            }
            quantile_counts[column] = counts

        results[category] = {
            'total_samples': len(X_combined),
            'feature_quantiles': quantile_counts
        }

    return results

def print_quantile_analysis(results):
    """
    Print the quantile analysis results in a readable format
    """
    for category, analysis in results.items():
        print(f"\n{'='*80}")
        print(f"Category: {category}")
        print(f"Total samples: {analysis['total_samples']}")
        print(f"{'='*80}")

        for feature, counts in analysis['feature_quantiles'].items():
            print(f"\nFeature: {feature}")
            print("Quantile values:")
            for q_name, value in counts['quantile_values'].items():
                print(f"  {q_name}: {value:.3f}")
            print("Data points in each quantile:")
            for q_name, count in {k:v for k,v in counts.items() if k != 'quantile_values'}.items():
                print(f"  {q_name}: {count}")
            print("-" * 40)

def segment_data_by_quantiles(X_train, X_test, y_train, y_test, feature_name):
    """Split (X_train, X_test, y_train, y_test) into four quartile bins
    defined by the distribution of feature_name, preserving the original
    train/test membership within each bin.
    """
    # Reset indices before concatenation to avoid duplicates
    X_train_reset = X_train.reset_index(drop=True)
    X_test_reset = X_test.reset_index(drop=True)
    y_train_reset = y_train.reset_index(drop=True)
    y_test_reset = y_test.reset_index(drop=True)

    # Create a marker column to identify train/test split
    X_train_reset['is_train'] = True
    X_test_reset['is_train'] = False

    # Combine train and test
    X_combined = pd.concat([X_train_reset, X_test_reset], axis=0, ignore_index=True)
    y_combined = pd.concat([y_train_reset, y_test_reset], axis=0, ignore_index=True)

    # Calculate quantile values
    q1, q2, q3 = X_combined[feature_name].quantile([0.25, 0.5, 0.75])

    # Create masks for each quantile
    q1_mask = X_combined[feature_name] <= q1
    q2_mask = (X_combined[feature_name] > q1) & (X_combined[feature_name] <= q2)
    q3_mask = (X_combined[feature_name] > q2) & (X_combined[feature_name] <= q3)
    q4_mask = X_combined[feature_name] > q3

    segments = {}
    for q_name, mask in [('Q1', q1_mask), ('Q2', q2_mask), ('Q3', q3_mask), ('Q4', q4_mask)]:
        # Get the masked data
        X_segment = X_combined[mask].copy()
        y_segment = y_combined[mask].copy()

        if len(X_segment) == 0:
            print(f"Warning: No data in {q_name} segment for feature {feature_name}")
            continue

        # Split back into train and test using the marker column
        train_mask = X_segment['is_train']
        test_mask = ~train_mask

        # Remove the marker column
        X_segment = X_segment.drop('is_train', axis=1)

        segments[q_name] = {
            'X_train': X_segment[train_mask],
            'X_test': X_segment[test_mask],
            'y_train': y_segment[train_mask],
            'y_test': y_segment[test_mask]
        }

    return segments

def segment_data_by_target_quantiles(X_train, X_test, y_train, y_test):
    """
    Segments data into quantiles based on the target variable.
    """
    # Reset indices before concatenation to avoid duplicates
    X_train_reset = X_train.reset_index(drop=True)
    X_test_reset = X_test.reset_index(drop=True)
    y_train_reset = y_train.reset_index(drop=True)
    y_test_reset = y_test.reset_index(drop=True)

    # Create a marker column to identify train/test split
    X_train_reset['is_train'] = True
    X_test_reset['is_train'] = False

    # Combine train and test
    X_combined = pd.concat([X_train_reset, X_test_reset], axis=0, ignore_index=True)
    y_combined = pd.concat([y_train_reset, y_test_reset], axis=0, ignore_index=True)

    # Calculate quantile values for the target variable
    q1, q2, q3 = y_combined.iloc[:, 0].quantile([0.25, 0.5, 0.75])

    # Create masks for each quantile
    q1_mask = y_combined.iloc[:, 0] <= q1
    q2_mask = (y_combined.iloc[:, 0] > q1) & (y_combined.iloc[:, 0] <= q2)
    q3_mask = (y_combined.iloc[:, 0] > q2) & (y_combined.iloc[:, 0] <= q3)
    q4_mask = y_combined.iloc[:, 0] > q3

    segments = {}
    for q_name, mask in [('Q1', q1_mask), ('Q2', q2_mask), ('Q3', q3_mask), ('Q4', q4_mask)]:
        # Get the masked data
        X_segment = X_combined[mask].copy()
        y_segment = y_combined[mask].copy()

        if len(X_segment) == 0:
            print(f"Warning: No data in {q_name} segment")
            continue

        # Split back into train and test using the marker column
        train_mask = X_segment['is_train']
        test_mask = ~train_mask

        # Remove the marker column
        X_segment = X_segment.drop('is_train', axis=1)

        segments[q_name] = {
            'X_train': X_segment[train_mask],
            'X_test': X_segment[test_mask],
            'y_train': y_segment[train_mask],
            'y_test': y_segment[test_mask],
            'quantile_range': (q1, q2, q3)  # Store quantile boundaries for reference
        }

        print(f"{q_name} segment ranges for target:")
        if q_name == 'Q1':
            print(f"  Range: <= {q1:.3f}")
        elif q_name == 'Q2':
            print(f"  Range: {q1:.3f} - {q2:.3f}")
        elif q_name == 'Q3':
            print(f"  Range: {q2:.3f} - {q3:.3f}")
        else:
            print(f"  Range: > {q3:.3f}")
        print(f"  Samples - Train: {len(segments[q_name]['X_train'])}, Test: {len(segments[q_name]['X_test'])}")

    return segments

def run_quantile_analysis(data_dict, tp, target_options, quantile_segment=False):
    """
    Runs CatBoost analysis on each category, segmenting by target variable quantiles.
    """
    results = {}

    for category, (X_train, X_test, y_train, y_test) in data_dict.items():
        print(f"\nAnalyzing category: {category}")

        if not quantile_segment:
            # Regular analysis without quantile segmentation
            model, _, _ = run_catboost_analysis(
                X_train, X_test, y_train, y_test,
                target_options, tp, category
            )
            results[category] = {
                'model': model,
                'feature_importance': getattr(model, 'feature_importances_', None)
            }
        else:
            print(f"\nSegmenting by {target_options} quantiles")
            try:
                segments = segment_data_by_target_quantiles(X_train, X_test, y_train, y_test)

                quantile_results = {}
                for quantile, data in segments.items():
                    print(f"\nAnalyzing {category} for {target_options} {quantile}")

                    # Check if we have enough samples in this segment
                    if len(data['X_train']) < 50 or len(data['X_test']) < 20:
                        print(f"Skipping {quantile} due to insufficient samples")
                        continue

                    try:
                        model, _, _ = run_catboost_analysis(
                            data['X_train'], data['X_test'],
                            data['y_train'], data['y_test'],
                            target_options, tp,
                            f"{category}_{quantile}"
                        )

                        quantile_results[quantile] = {
                            'model': model,
                            'feature_importance': getattr(model, 'feature_importances_', None),
                            'n_samples': len(data['X_train']) + len(data['X_test']),
                            'quantile_range': data['quantile_range']
                        }
                    except Exception as e:
                        print(f"Error in analyzing {quantile}: {str(e)}")
                        continue

                if quantile_results:  # Only store if we have results
                    results[category] = quantile_results
            except Exception as e:
                print(f"Error processing category {category}: {str(e)}")
                continue

    return results

# Assign sample weights based on target quartile membership so that higher
# outcome values receive greater emphasis during training. Weights are
# passed directly to the CatBoost model via the sample_weight parameter.
def calculate_quantile_weights(y_values):
    """Compute sample weights by target quartile with adaptive boundaries.

    Falls back to equal-size grouping if any bin would otherwise be empty.

    Parameters
    y_values : array-like
        Target variable values (regression outcomes).

    Returns
    numpy.ndarray
        Per-sample weights in {1.0, 2.0, 3.0, 4.0}.
    """
    # Convert to numpy array and flatten
    y_flat = np.array(y_values).ravel()

    # Get unique values and their counts
    unique_vals = np.unique(y_flat)

    if len(unique_vals) < 4:
        print("Warning: Less than 4 unique values in target. Using available unique values for grouping.")
        weights = np.ones(len(y_flat))
        for i, val in enumerate(unique_vals, 1):
            weights[y_flat == val] = i

        # Print distribution
        print("\nUnique value weights:")
        for i, val in enumerate(unique_vals, 1):
            count = np.sum(y_flat == val)
            print(f"Value {val:.3f}: weight {i}.0, {count} samples")

        return weights

    # Find adaptive quantile boundaries that ensure samples in each group
    n_samples = len(y_flat)
    sorted_vals = np.sort(y_flat)

    # Try to get roughly equal groups while ensuring no empty groups
    target_size = n_samples // 4

    # Find boundaries that create non-empty groups
    q1_idx = max(target_size - target_size//2, 0)
    q2_idx = 2 * target_size
    q3_idx = 3 * target_size

    # indices don't exceed array bounds
    q1_idx = min(q1_idx, n_samples-3)
    q2_idx = min(q2_idx, n_samples-2)
    q3_idx = min(q3_idx, n_samples-1)

    # Get the actual values at these positions
    q1 = sorted_vals[q1_idx]
    q2 = sorted_vals[q2_idx]
    q3 = sorted_vals[q3_idx]

    # Initialize weights array
    weights = np.ones(len(y_flat))

    # Assign weights based on adaptive quantiles
    weights[y_flat > q1] = 2.0
    weights[y_flat > q2] = 3.0
    weights[y_flat > q3] = 4.0

    # Print distribution information
    print("\nAdaptive quantile ranges and weights:")
    print(f"Q1 (≤{q1:.3f}): weight 1.0, {np.sum(y_flat <= q1)} samples")
    print(f"Q2 ({q1:.3f}-{q2:.3f}): weight 2.0, {np.sum((y_flat > q1) & (y_flat <= q2))} samples")
    print(f"Q3 ({q2:.3f}-{q3:.3f}): weight 3.0, {np.sum((y_flat > q2) & (y_flat <= q3))} samples")
    print(f"Q4 (>{q3:.3f}): weight 4.0, {np.sum(y_flat > q3)} samples")

    # Verify no empty groups
    for i, (lower, upper) in enumerate([
        (-np.inf, q1),
        (q1, q2),
        (q2, q3),
        (q3, np.inf)
    ]):
        count = np.sum((y_flat > lower) & (y_flat <= upper)) if i < 3 else np.sum(y_flat > lower)
        if count == 0:
            print(f"\nWarning: Found empty group {i+1}, adjusting boundaries...")
            return calculate_quantile_weights_equal_size(y_flat)

    return weights

def calculate_quantile_weights_equal_size(y_values):
    """Fallback: rank samples and assign weights based on equal-size quantile bins."""
    n_samples = len(y_values)
    ranks = pd.Series(y_values).rank(method='first')
    group_size = n_samples // 4

    weights = np.ones(n_samples)
    weights[ranks > group_size] = 2.0
    weights[ranks > 2 * group_size] = 3.0
    weights[ranks > 3 * group_size] = 4.0

    # Print distribution information
    print("\nEqual-size group weights:")
    print(f"Group 1: weight 1.0, {np.sum(weights == 1.0)} samples")
    print(f"Group 2: weight 2.0, {np.sum(weights == 2.0)} samples")
    print(f"Group 3: weight 3.0, {np.sum(weights == 3.0)} samples")
    print(f"Group 4: weight 4.0, {np.sum(weights == 4.0)} samples")

    return weights

def calculate_top_quartile_weights(y_values):
    """Apply a binary weighting scheme that up-weights samples in the top 20 %
    of the target distribution (weight = 1.25) relative to the remainder
    (weight = 1.0).  Intended for use when clinical interest is concentrated
    in the high-symptom tail.

    Parameters
    y_values : array-like
        Continuous target variable values.

    Returns
    numpy.ndarray
        Per-sample weights in {1.0, 1.25}.
    """
    # Convert to numpy array and flatten
    y_flat = np.array(y_values).ravel()

    # Calculate 80th percentile
    q3 = np.percentile(y_flat, 80)

    # Initialize all weights to 1.0
    weights = np.ones(len(y_flat))

    # Set weight 1.25 for top 20%
    weights[y_flat > q3] = 1.25

    # Print distribution information
    print("\nBinary weighting scheme:")
    print(f"Bottom 80% (≤{q3:.3f}): weight 1.0, {np.sum(y_flat <= q3)} samples")
    print(f"Top 20% (>{q3:.3f}): weight 1.25, {np.sum(y_flat > q3)} samples")

    # Verify distribution
    weight_dist = pd.Series(weights).value_counts()
    print("\nWeight distribution:")
    for weight, count in weight_dist.items():
        print(f"Weight {weight:.1f}: {count} samples ({count/len(weights)*100:.1f}%)")

    return weights

# ---
# K-Fold Variants
#
# These functions return raw (un-imputed) feature matrices together with a
# KFold/StratifiedKFold splitter object. Imputation, scaling, and model
# fitting are deferred to train_and_evaluate_model_kfold() so that the full
# preprocessing pipeline is applied independently within each fold, preventing
# any leakage from test data into fitted preprocessing parameters.
# ---

from sklearn.model_selection import KFold, StratifiedKFold

# Within-category variant: returns {category: (X_raw, y_raw, kf, scale_x, scale_y)}
def within_categories_data_kfold(t, targets, n_splits=5, group=None, scale_x=False, scale_y=False, verbose=False, class_rebalancing=False):

  # -- Route to child or parent variable hierarchy depending on target --
  global variables_for_each_time_point
  target_name = targets[0] if targets else ''
  if is_parent_target(target_name):
      variables_for_each_time_point = parent_variables_for_each_time_point
      print(f"[VARS] Using PARENT within-category structure for target: {target_name}")
  else:
      variables_for_each_time_point = child_variables_for_each_time_point
      print(f"[VARS] Using CHILD within-category structure for target: {target_name}")

  # -- Apply circular exclusions --
  across_excl, within_excl = get_circular_exclusions(target_name, timepoint=t)
  active_vfet_tp = {
      cat: vars_list for cat, vars_list in variables_for_each_time_point[t].items()
      if cat not in within_excl
  }
  if within_excl:
      skipped = within_excl & set(variables_for_each_time_point[t].keys())
      if skipped:
          print(f"[CIRCULAR] Excluded within-categories for {target_name}: {skipped}")

  # Flatten retained categories, removing individually excluded circular variables.
  within_categories = [
        x
        for xs in active_vfet_tp.values()
        for x in xs
        if x not in across_excl
      ] + targets
  if across_excl:
      individual_removed = len([x for xs in active_vfet_tp.values() for x in xs if x in across_excl])
      if individual_removed > 0:
          print(f"[CIRCULAR] Also removed {individual_removed} individual circular variables for {target_name}")

  within_categories = list(set(within_categories))

  # Get available columns in the DataFrame
  available_columns = set(sample.columns)

  # Remove columns from within_categories that are not in available_columns
  within_categories = [var for var in within_categories if var in available_columns]

  # If any columns were removed, notify the user
  dropped_columns = set([x for xs in variables_for_each_time_point[t].values() for x in xs] + targets).difference(available_columns)

  if dropped_columns:
      print(f"WARNING: The following columns were not found in the dataset and have been dropped: {dropped_columns}")

  # Remove specific variables
  if int(t) == 3 or int(t) == 1:
      variables_to_remove = [
          'peq_ss_relational_victim',
          'peq_ss_reputation_aggs',
          'peq_ss_reputation_victim',
          'peq_ss_overt_aggression',
          'peq_ss_overt_victim',
          'peq_ss_relational_aggs'
      ]
      within_categories = [var for var in within_categories if var not in variables_to_remove]

  # Avoid mutating the group parameter.
  _resolved_group = 'low_ale_children_p' if group == 'low_ale' else group

  if _resolved_group:
    if _resolved_group == 'high_ale':
      all_vars = within_categories + [f"{_resolved_group}"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data[all_data['high_ale'] == True]

    elif _resolved_group == 'high_ale_severe_p_mh':
      all_vars = within_categories + [f"{_resolved_group}"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data[all_data['high_ale_severe_p_mh'] == True]

    elif _resolved_group == 'low_ale_children_p':
      all_vars =  within_categories + ['low_ale_children_p']
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data[all_data['low_ale_children_p'] == True]

    elif _resolved_group == 'Female':
      all_vars =  within_categories + ["sex"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data.loc[:, ~all_data.columns.duplicated()]
      all_data = all_data[all_data['sex'] == 2]

    elif _resolved_group == 'Male':
      all_vars =  within_categories + ["sex"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data.loc[:, ~all_data.columns.duplicated()]
      all_data = all_data[all_data['sex'] == 1]

    else:
      all_vars = within_categories
      all_data = sample.query(f"time == {t}")[all_vars]

  else:
    all_vars = within_categories
    all_data = sample.query(f"time == {t}")[all_vars]

  if verbose:
    print(f"Gathered within category data from {all_data.shape[0]} subjects for {_resolved_group if _resolved_group else 'full'} group at t = {t}")

  if 'depress_D_p' not in targets and 'depress_D_p_rev' not in targets and "external_D_p" not in targets and "adhd_D_p" not in targets:
    if "latent_class_depression" in targets:
      all_data = all_data[(all_data["latent_class_depression"] != "")]
    # Class rebalancing deferred to AFTER split
    _needs_rebalance = class_rebalancing or (target_options == 'latent_class_depression' or target_options == 'depadhd_c' or target_options == 'asdadhd_c')

  available = drop_high_missing_columns(all_data, threshold=0.7, verbose=verbose)
  dropped_columns = set(all_data.columns).difference(set(available.columns))

  if dropped_columns:
    print(f"WARNING: Dropped columns at t = {t} due to entirely missing data: {dropped_columns}")

  raw_X = available.drop(targets, axis=1)
  raw_y = available[targets]

  # -- Drop rows where target is NaN (parent ASR items can have sporadic missingness) --
  _target_notna = raw_y.iloc[:, 0].notna()
  if (~_target_notna).any():
      _n_dropped = (~_target_notna).sum()
      print(f"Dropped {_n_dropped} rows with missing target values")
      raw_X = raw_X[_target_notna]
      raw_y = raw_y[_target_notna]

  # One-hot encoding is applied to the full dataset before folding because it
  # uses only category membership (deterministic; no data-dependent statistics).
  # Imputation and scaling are deferred to train_and_evaluate_model_kfold().
  one_hot_features = ['sex', 'sex_P', 'L_site_id', 'marital_status', 'child_religion', 'sex_orient_y', 'trans_id_y', 'crysflu_c', 'anxadhd_c']
  encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

  for feature in one_hot_features:
      if feature in raw_X.columns:
          print(f"Encoding {feature} for one hot encoding..")
          raw_X_cat = raw_X[[feature]].astype(str)
          encoded_data = encoder.fit_transform(raw_X_cat)
          unique_values = encoder.categories_[0][1:]
          encoded_columns = [f"{feature}_{val}" for val in unique_values]
          encoded_df = pd.DataFrame(
              encoded_data,
              columns=encoded_columns,
              index=raw_X.index
          )
          raw_X = pd.concat([raw_X.drop(feature, axis=1), encoded_df], axis=1)
          print(f"Encoded {feature} into {len(encoded_columns)} categories")

  ret = {}

  if is_regression(targets[0]):
      kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
  else:
      kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

  # Iterate over the circular-exclusion-filtered category dict (active_vfet_tp)
  # so that excluded categories never appear in the returned results.
  for (category, cat_vars) in active_vfet_tp.items():
      # Also filter individual across_excl variables from each category slice.
      filtered_vars = [v for v in cat_vars if v not in across_excl]
      available_vars = list(set(filtered_vars).difference(dropped_columns))
      available_vars = [var for var in available_vars if var in raw_X.columns]

      if available_vars:
          ret[category] = (raw_X[available_vars], raw_y, kf, scale_x, scale_y)
      else:
          print(f"WARNING: No available variables left for category: {category}")

  return ret

# Across-category variant: returns (X_raw, y_raw, kf, scale_x, scale_y)
def across_categories_data_kfold(t, targets, n_splits=5, group=None, scale_x=False, scale_y=False, verbose=False, class_rebalancing=False):
  available_columns = set(sample.columns)

  if t == 0:
    across_categories = list(t0_variables)
  elif t == 1:
    across_categories = list(t1_variables)
  elif t == 2:
    across_categories = list(t2_variables)
  elif t == 3:
    across_categories = list(t3_variables)
  elif t == 4:
    across_categories = list(t4_variables)
  else:
    raise ValueError(f"Invalid time point: {t}")

  # -- Apply circular exclusions --
  target_name = targets[0] if targets else ''
  across_excl, _ = get_circular_exclusions(target_name, timepoint=t)
  if across_excl:
      before_count = len(across_categories)
      across_categories = [v for v in across_categories if v not in across_excl]
      removed_count = before_count - len(across_categories)
      if removed_count > 0:
          print(f"[CIRCULAR] Removed {removed_count} circular variables from across-category pool for target: {target_name}")

  # Remove columns from across_categories that are not in available_columns
  across_categories = [var for var in across_categories if var in available_columns]

  # If any columns were removed, notify the user
  dropped_columns = set([x for xs in variables_for_each_time_point[t].values() for x in xs] + targets).difference(available_columns)

  if dropped_columns:
      print(f"WARNING: The following columns were not found in the dataset and have been dropped: {dropped_columns}")

  # Remove specific variables when t == 1 or t == 3
  if int(t) == 3 or int(t) == 1:
      variables_to_remove = [
          'peq_ss_relational_victim',
          'peq_ss_reputation_aggs',
          'peq_ss_reputation_victim',
          'peq_ss_overt_aggression',
          'peq_ss_overt_victim',
          'peq_ss_relational_aggs'
      ]
      across_categories = [var for var in across_categories if var not in variables_to_remove]

  # Avoid mutating the group parameter.
  _resolved_group = 'low_ale_children_p' if group == 'low_ale' else group

  if _resolved_group:
    if _resolved_group == 'high_ale':
      all_vars =  across_categories + targets + [f"{_resolved_group}"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data[all_data['high_ale'] == True]

    elif _resolved_group == 'high_ale_severe_p_mh':
      all_vars =  across_categories + targets + [f"{_resolved_group}"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data[all_data['high_ale_severe_p_mh'] == True]

    elif _resolved_group == 'low_ale_children_p':
      all_vars =  across_categories + targets + ['low_ale_children_p']
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data[all_data['low_ale_children_p'] == True]

    elif _resolved_group == 'Female':
      all_vars =  across_categories + targets + ["sex"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data.loc[:, ~all_data.columns.duplicated()]
      all_data = all_data[all_data['sex'] == 2]

    elif _resolved_group == 'Male':
      all_vars =  across_categories + targets + ["sex"]
      all_data = sample[sample['time']==t][all_vars]
      all_data = all_data.loc[:, ~all_data.columns.duplicated()]
      all_data = all_data[all_data['sex'] == 1]

    else:
      all_vars = across_categories + targets
      all_data = sample.query(f"time == {t}")[all_vars]

  else:
    all_vars = across_categories + targets
    all_data = sample.query(f"time == {t}")[all_vars]

  if verbose:
    print(f"Gathered across category data from {all_data.shape[0]} subjects for {_resolved_group if _resolved_group else 'full'} group at t = {t}")

  if 'depress_D_p' not in targets and 'depress_D_p_rev' not in targets and "external_D_p" not in targets and "adhd_D_p" not in targets:
    if 'latent_class_depression' in targets:
      all_data = all_data[(all_data["latent_class_depression"] != "")]
    # Class rebalancing deferred to AFTER split
    _needs_rebalance = class_rebalancing or (target_options == 'latent_class_depression' or target_options == 'depadhd_c' or target_options == 'asdadhd_c')

  available = drop_high_missing_columns(all_data, threshold=0.7, verbose=verbose)
  dropped_columns = set(all_data.columns).difference(set(available.columns))

  if dropped_columns:
    print(f"WARNING: Dropped columns at t = {t} due to entirely missing data: {dropped_columns}")

  raw_X = available.drop(targets, axis=1)
  raw_y = available[targets]

  # -- Drop rows where target is NaN (parent ASR items can have sporadic missingness) --
  _target_notna = raw_y.iloc[:, 0].notna()
  if (~_target_notna).any():
      _n_dropped = (~_target_notna).sum()
      print(f"Dropped {_n_dropped} rows with missing target values")
      raw_X = raw_X[_target_notna]
      raw_y = raw_y[_target_notna]

  # Same encoding strategy as the within-category kfold function above.
  one_hot_features = ['sex', 'sex_P', 'L_site_id', 'marital_status', 'child_religion', 'sex_orient_y', 'trans_id_y', 'crysflu_c', 'anxadhd_c']
  encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

  for feature in one_hot_features:
      if feature in raw_X.columns:
          print(f"Encoding {feature} for one hot encoding..")
          raw_X_cat = raw_X[[feature]].astype(str)
          encoded_data = encoder.fit_transform(raw_X_cat)
          unique_values = encoder.categories_[0][1:]
          encoded_columns = [f"{feature}_{val}" for val in unique_values]
          encoded_df = pd.DataFrame(
              encoded_data,
              columns=encoded_columns,
              index=raw_X.index
          )
          raw_X = pd.concat([raw_X.drop(feature, axis=1), encoded_df], axis=1)
          print(f"Encoded {feature} into {len(encoded_columns)} categories")

  # Instead of train_test_split, we'll return the full dataset and a KFold object
  if is_regression(targets[0]):
      kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
  else:
      kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

  return raw_X, raw_y, kf, scale_x, scale_y

def train_and_evaluate_model_kfold(X, y, kf, model, scale_x=False, scale_y=False):
    """Train and evaluate a model under K-fold cross-validation with per-fold
    preprocessing to ensure no leakage from held-out folds.

    Imputation and optional standardisation are fit on the training fold only
    and applied to the corresponding test fold within every iteration.

    Parameters
    X, y      : Feature matrix and target returned by a kfold data function.
    kf        : Pre-configured KFold or StratifiedKFold splitter.
    model     : Unfitted estimator (will be cloned each fold).
    scale_x   : Standardise features if True.
    scale_y   : Standardise target if True (regression only).

    Returns
    (mean_score, std_score) : Mean and standard deviation of fold-wise scores.
    """
    scores = []
    for train_index, test_index in kf.split(X, y):
        X_train_fold, X_test_fold = X.iloc[train_index].copy(), X.iloc[test_index].copy()
        y_train_fold, y_test_fold = y.iloc[train_index].copy(), y.iloc[test_index].copy()

        # Impute per fold (fit on train only)
        fold_imputer = get_imputer(imputer)
        fold_imputer.fit(X_train_fold)
        X_train_fold = pd.DataFrame(
            fold_imputer.transform(X_train_fold),
            columns=X_train_fold.columns, index=X_train_fold.index
        )
        X_test_fold = pd.DataFrame(
            fold_imputer.transform(X_test_fold),
            columns=X_test_fold.columns, index=X_test_fold.index
        )

        # Scale per fold if requested (fit on train only)
        if scale_x:
            fold_scaler_x = StandardScaler()
            fold_scaler_x.fit(X_train_fold)
            X_train_fold = pd.DataFrame(
                fold_scaler_x.transform(X_train_fold),
                columns=X_train_fold.columns, index=X_train_fold.index
            )
            X_test_fold = pd.DataFrame(
                fold_scaler_x.transform(X_test_fold),
                columns=X_test_fold.columns, index=X_test_fold.index
            )

        if scale_y:
            fold_scaler_y = StandardScaler()
            fold_scaler_y.fit(y_train_fold)
            y_train_fold = pd.DataFrame(
                fold_scaler_y.transform(y_train_fold),
                columns=y_train_fold.columns, index=y_train_fold.index
            )
            y_test_fold = pd.DataFrame(
                fold_scaler_y.transform(y_test_fold),
                columns=y_test_fold.columns, index=y_test_fold.index
            )

        model_clone = clone(model)
        model_clone.fit(X_train_fold, y_train_fold)
        score = model_clone.score(X_test_fold, y_test_fold)
        scores.append(score)

    return np.mean(scores), np.std(scores)

# Usage examples:
# a = within_categories_data(2, ['depress_D_p'], group='high_ale', verbose=True)
# X_tr, X_te, y_tr, y_te = a['Anxiety']
#
# b = across_categories_data(1, ['depress_D_p'], group='high_ale', verbose=True)
# X_train, X_test, y_train, y_test = b

# ---
# Shared Validation Utilities
# ---

# -- Standard validation constants ---
VAL_N_OUTER       = 5       # Outer CV folds for all validation analyses
VAL_N_BOOTSTRAP   = 1000    # Bootstrap iterations for CIs
VAL_RANDOM_STATE  = 42      # Master seed
VAL_DPI           = 300     # Figure DPI for Nature
VAL_FIGSIZE_SINGLE = (3.50, 3.50)   # Nature single-column (89 mm)
VAL_FIGSIZE_DOUBLE = (7.20, 4.50)   # Nature double-column (183 mm)

# -- Lightweight validation model factory ---
def create_lightweight_model(is_classification=True, n_features=None):
    """Return a quick-training model suitable for validation sweeps.

    Uses CatBoost with moderate iterations (300) and early stopping.
    Intended for PCA baseline, temporal holdout, ablation, and TP sweeps
    where full stacking ensemble is too expensive.
    """
    try:
        if is_classification:
            from catboost import CatBoostClassifier
            return CatBoostClassifier(
                iterations=300, depth=6, learning_rate=0.08,
                early_stopping_rounds=30, verbose=0,
                random_state=42, thread_count=-1,
                auto_class_weights='Balanced'
            )
        else:
            from catboost import CatBoostRegressor
            return CatBoostRegressor(
                iterations=300, depth=6, learning_rate=0.08,
                early_stopping_rounds=30, verbose=0,
                random_state=42, thread_count=-1
            )
    except ImportError:
        from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
        if is_classification:
            return GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
        else:
            return GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)

# -- Helper: bootstrap CI ---
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=VAL_N_BOOTSTRAP,
                 ci=0.95, seed=VAL_RANDOM_STATE):
    """Compute bootstrap confidence interval for a metric.

    Returns (point_estimate, ci_lower, ci_upper).
    """
    rng = np.random.RandomState(seed)
    point = metric_fn(y_true, y_pred)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            continue
    scores = np.array(scores)
    alpha = 1 - ci
    lo = np.percentile(scores, 100 * alpha / 2)
    hi = np.percentile(scores, 100 * (1 - alpha / 2))
    return point, lo, hi

print("Shared Validation Utilities loaded (create_lightweight_model, bootstrap_ci)")

# ---
# Variable Display Names
# ---

# ---
# Translates raw ABCD variable names into readable labels for feature
# importance figures. The rename_dict is consumed by FI plotting functions
# in this cell.
# ---
# DISPLAY_NAMES -- T2 Within-Category Variable Renaming Guide
# ---
# Conventions:
# (Y) at end  = youth self-report -- ONLY when raw name ends in _k
# (_p) parent ratings of children
# (_parent)  = parent self-report (mostly from ASR)
# ---

DISPLAY_NAMES = {

# ---
    # ADHD (8)
# ---
    'trouble_concentrating_k':       'Concentration Trouble (Y)',
    'difficulty_finishing_tasks_k':   'Task Continuity  (Y)',
    'easily_distracted_k':           'Distracted Easily (Y)',
    'cant_concentrate_p':            'Concentration Difficulty',
    'doesnt_finish_p':               'Failure to Finish Tasks',
    'hyperactive_p':                 'Hyperactive',
    'impulsive_p':                   'Impulsive',
    'easily_distracted_p':           'Distracted Easily',

# ---
    # Adverse Life Events (24)
# ---
    'g_lifeevents_ss_k':                       'Positive Events (Y)',
    'b_lifeevents_ss_k':                       'Adverse Life Events (Y)',
    'b_lifeevents_affected_ss_k':              'Adverse Events Impact (Y)',
    'experienced_crime_p':                     'Crime Victim',
    'g_lifeevents_ss_p':                       'Positive Events',
    'b_lifeevents_ss_p':                       'Adverse Life Events',
    'b_lifeevents_affected_ss_p':              'Adverse Events Impact',
    'car_accident_hurt_p':                     'Car Accident',
    'big_accident_need_treatment_p':           'Major Accident',
    'fire_victim_p':                           'Fire Victim',
    'natural_disaster_victim_p':               'Nat. Disaster',
    'terrorism_victim_p':                      'Terrorism Exp.',
    'war_death_witness_p':                     'War Witness',
    'stabbing_shooting_witness_p':             'Violence Witness',
    'stabbing_shooting_victim_community_p':    'Community Violence',
    'stabbing_shooting_victim_home_p':         'Home Violence',
    'beating_victim_home_p':                   'Beaten at Home',
    'stranger_threatened_child_victim_p':      'Stranger Threat',
    'family_threatened_child_victim_p':        'Family Threat',
    'adult_family_fighting_victim_p':          'Adult Fighting',
    'domestic_child_sexually_abuse_victim_p':  'Dom. Sex Abuse',
    'foreign_child_sexually_abuse_victim_p':   'Non-Dom. Sex Abuse',
    'peer_child_sexually_abuse_victim_p':      'Peer Sex Abuse',
    'sudden_death_in_family_p':                'Sudden Death',

# ---
    # Anxiety (12)
# ---
    'social_fear_present_PK':       'Social Fear',
    'worries_p':                    'Worries',
    'clings_to_adults_p':           'Clingy',
    'nervous_general_p':            'Generally Nervous',
    'nervous_twitching_p':          'Nervous Twitching',
    'fears_excl_school_p':          'Fears Exclusion',
    'fears_school_p':               'School Fear',
    'fears_being_bad_p':            'Guilt Fear',
    'paranoid_p':                   'Paranoid',
    'fear_becoming_obese_present_PK': 'Weight Fear',
    'self_conscious_k':             'Self-Conscious (Y)',
    'anxious_fearful_k':            'Anxious (Y)',

# ---
    # Cognitive Task Outcomes (44)
# ---
    'tb_picvocab':                'Picture Vocabulary Task (NIH)',
    'tb_picture':                 'Picture Sequence Memory Task (NIH)',
    'tb_reading':                 'Oral Reading Recognition Task (NIH)',
    'tb_flanker':                 'Flanker Inhibitory Control Task (NIH)',
    'tb_list':                    'List Sorting Working Memory Task (NIH)',
    'tb_cardsort':                'Dimensional Change Card Sort Task (NIH)',
    'tb_pattern':                 'Pattern Comparison Speed Task (NIH)',
    'gd_safebets':                'Safe Bets (GDT)',
    'gd_riskybets':               'Game of Dice Task -- Risky Bets',
    'ravlt_s_total':              'RAVLT Short Tot.',
    'ravlt_s_repitition':         'RAVLT Short Rep.',
    'ravlt_s_intrusions':         'RAVLT Short Intr.',
    'ravlt_l_total':              'RAVLT Long Tot.',
    'ravlt_l_repitition':         'RAVLT Long Rep.',
    'ravlt_l_intrusions':         'RAVLT Long Intr.',
    'nb_correct_nt':              'N-Back Acc.',
    'nb_correct_mrt':             'N-Back RT',
    'nb_correct_nt_2back':        '2-Back Acc.',
    'nb_correct_mrt_2back':       '2-Back RT',
    'nb_correct_nt_pos':          'N-Back Acc. Pos',
    'nb_correct_mrt_pos':         'N-Back RT Pos',
    'nb_correct_nt_neg':          'N-Back Acc. Neg',
    'nb_correct_mrt_neg':         'N-Back RT Neg',
    'nb2_accuracy_pos':           '2B Acc. Pos',
    'nb2_resp_bias_pos':          '2B Bias Pos',
    'nb2_D_prime_pos':            "2B d' Pos",
    'nb2_accuracy_neg':           '2B Acc. Neg',
    'nb2_resp_bias_neg':          '2B Bias Neg',
    'nb2_D_prime_neg':            "2B d' Neg",
    'sst_ssrt_mean_est':          'SSRT Mean',
    'sst_ssrt_int_est':           'SSRT Integration',
    'sst_acceptable_performance':'SST Task Valid',
    'mid_mrt_smrw':               'MID RT Sm. Rew.',
    'mid_mrt_lgrw':               'MID RT Lg. Rew.',
    'mid_total_payout':           'MID Earnings',
    'mid_acceptable_performance': 'MID Valid',
    'mid_num_trials':             'MID Trials',
    'lmt_accuracy':               'LMT Accuracy',
    'lmt_correct_nt':             'LMT Correct N',
    'lmt_mrt':                    'LMT Mean RT',
    'lmt_correct_mrt':            'LMT Correct RT',
    'lmt_efficiency':             'LMT Efficiency',
    'nb_correct_mrt_neutral':     'N-Back RT Neut',
    'nb_correct_nt_neutral':      'N-Back Acc. Neut',

# ---
    # Delta (13)
# ---
    'delta_adhd_D_p':               'Change in ADHD',
    'delta_not_liked_p':            'Change in Unliked',
    'delta_doesnt_get_along_p':     'Change in Peer Conflict',
    'delta_family_conflict_ss_p':   'Change in Fam. Conflict',
    'delta_family_conflict_ss_k':   'Change in Fam. Conflict (Y)',
    'delta_bad_grades':             'Change in Bad Grades',
    'delta_social_problems_D_p':    'Change in Social Problems',
    'delta_somatic_problems_D_p':   'Change in Somatic Issues',
    'delta_bad_diet_p':             'Change in Bad Diet',
    'delta_atschool_total_problems_ss_t': 'Change in School Prob.',
    'delta_b_lifeevents_ss_p':      'Change in Bad Events',
    'delta_anxdisord_D_p':          'Change in Anxiety',
    'delta_cant_concentrate_p':     'Change in Concentration Difficulty',

# ---
    # Diet / Nutrition (17)
# ---
    'fruit_intake':          'Fruit Intake',
    'vegetable_intake':      'Vegetable Intake',
    'protein_sources_intake':'Protein Sources',
    'legume_intake':         'Legume Intake',
    'added_sugar':           'Added Sugar',
    'sugary_beverage_freq':  'Sugary Drinks',
    'dairy_intake':          'Dairy Intake',
    'whole_grain_intake':    'Whole Grains',
    'total_calories':        'Total Calories',
    'protein_intake':        'Protein (g)',
    'carbohydrate_intake':   'Carbs (g)',
    'fiber_intake':          'Fiber (g)',
    'sodium_intake':         'Sodium (mg)',
    'potassium_intake':      'Potassium (mg)',
    'total_sugar':           'Total Sugar',
    'saturated_fat':         'Saturated Fat',
    'bad_diet_p':            'Bad Diet',

# ---
    # Dynamic Cognitive Control Parameters (55)
    # Computational Modelling of SST Task
    # from Percy Mistry (see https://www.biorxiv.org/content/10.1101/2024.10.01.615613v1.full for overview)
    # ALL prefixed with SST
# ---
    'sst_theta0':'SST Task Theta0',
    'sst_theta1':'SST Task Theta1',
    'sst_mu':'SST Task Mu',
    'sst_aS':'SST Task AlphaS',
    'sst_dS':'SST Task DeltaS',
    'sst_gamma0':'SST Task Gamma0',
    'sst_gamma1':'SST Task Gamma1',
    'sst_dG':'SST Task DeltaG',
    'sst_aG1':'SST Task AlphaG1',
    'sst_aG2':'SST Task AlphaG2',
    'sst_kappa0':'SST Task Kappa0',
    'sst_xM':'SST Task xM',
    'sst_sM':'SST Task sM',
    'sst_bG':'SST Task BetaG',
    'sst_pp':'SST Task pp',
    'sst_factor1':'SST Task Factor 1',
    'sst_factor2':'SST Task Factor 2',
    'sst_factor3':'SST Task Factor 3',
    'sst_mean_ssrt':'SST Task Mean SSRT',
    'sst_median_ssrt':'SST Task Median SSRT',
    'sst_mean_PDR':'SST Task Mean PDR',
    'sst_median_PDR':'SST Task Median PDR',
    'sst_mean_SD':'SST Task Mean SD',
    'sst_median_SD':'SST Task Median SD',
    'sst_mean_SDr':'SST Task Mean SDr',
    'sst_median_SDr':'SST Task Median SDr',
    'sst_mean_PDRg':'SST Task Mean PDRg',
    'sst_median_PDRg':'SST Task Median PDRg',
    'sst_mean_betaS':'SST Task Mean BetaS',
    'sst_median_betaS':'SST Task Median BetaS',
    'sst_mean_bS2':'SST Task Mean BetaS2',
    'sst_median_bS2':'SST Task Median BetaS2',
    'sst_mean_absdelta':'SST Task Mean AbsDelta',
    'sst_median_absdelta':'SST Task Med. AbsDelta',
    'sst_pdrV':'SST Task PDR Var.',
    'sst_pdrSD':'SST Task PDR SD',
    'sst_pdrCV':'SST Task PDR CV',
    'sst_sdRV':'SST Task SD Var.',
    'sst_sdSD':'SST Task SD StdDev',
    'sst_sdCV':'SST Task SD CV',
    'sst_sdrRV':'SST Task SDr Var.',
    'sst_sdrSD':'SST Task SDr SD',
    'sst_sdrCV':'SST Task SDr CV',
    'sst_pdrgV':'SST Task PDRg Var.',
    'sst_pdrgSD':'SST Task PDRg SD',
    'sst_pdrgCV':'SST Task PDRg CV',
    'sst_betasV':'SST Task BetaS Var.',
    'sst_betasSD':'SST Task BetaS SD',
    'sst_betasCV':'SST Task BetaS CV',
    'sst_absdeltaRV':'SST Task AbsDelta Var.',
    'sst_absdeltaSD':'SST Task AbsDelta SD',
    'sst_absdeltaCV':'SST Task AbsDelta CV',
    'sst_bs2V':'SST Task BetaS2 Var.',
    'sst_bs2SD':'SST Task BetaS2 SD',
    'sst_bs2CV':'SST Task BetaS2 CV',

# ---
    # Ethnicity / Nationality (43)
# ---
    'desc_african_AFR_B':          'African Desc.',
    'desc_native_american_AMR_B':  'Native Amer.',
    'desc_alaska_native_AMR_B':    'Alaska Native',
    'desc_chinese_EAS_B':          'Chinese Desc.',
    'desc_japanese_EAS_B':         'Japanese Desc.',
    'desc_korean_EAS_B':           'Korean Desc.',
    'desc_vietnamese_EAS_B':       'Vietnamese Desc.',
    'desc_european_EUR_B':         'European Desc.',
    'desc_asian_indian_SAS_B':     'South Asian Desc.',
    'desc_other_south_asian_SAS_B':'Oth. S. Asian',
    'desc_latin_B':                'Latino Desc.',
    **{f'pc_gene_aces{i}': f'Ancestry PC{i}' for i in range(1, 33)},

# ---
    # Externalizing / Other (23)
# ---
    'argues_p':                          'Argumentative',
    'stubborn_p':                        'Stubborn',
    'temper_tantrums_p':                 'Tantrums',
    'bullies_others_p':                  'Bullies Others',
    'destroys_own_things_p':             'Destroys Own Stuff',
    'destroys_others_things_p':          'Destroys Others Stuff',
    'disobedient_home_p':                'Disobedient Home',
    'disobedient_school_p':              'Disobedient School',
    'breaks_rules_p':                    'Rule-Breaking',
    'fights_p':                          'Fights',
    'lying_p':                           'Dishonest/Deceptive',
    'steals_home_p':                     'Steals Home',
    'steals_outside_p':                  'Steals Outside',
    'threatens_others_p':                'Threatens',
    'whines_p':                          'Whines',
    'demands_attention_p':               'Demands Attention',
    'conduct_physical_fights_present_PK':'Phys. Fights',
    'acts_immature_k':                   'Immature (Y)',
    'destroys_others_things_k':          'Destroys Others (Y)',
    'disobeys_parents_k':                'Par. Disobeys (Y)',
    'stubborn_k':                        'Stubborn (Y)',
    'hot_temper_k':                      'Hot Temper (Y)',

# ---
    # Family Drug Use (36)
# ---
    'hallucinogen_use_history_B_p':            'Hallucinogen Hx',
    'hallucinogen_current_B_p':                'Hallucinogen Cur.',
    'sedative_hypnotic_anxiolytic_use_B_p':    'Sedative Use',
    'father_alcohol':                          'Father Alcohol',
    'mother_alcohol':                          'Mother Alcohol',
    'father_druguse':                          'Father Drugs',
    'mother_druguse':                          'Mother Drugs',
    'cigs_during_pregnancy_p':                 'Prenatal Cigs',
    'alcohol_during_pregnancy_p':              'Prenatal Use Alcohol',
    'weed_during_pregnancy_p':                 'Prenatal Use Cannabis',
    'cocaine_during_pregnancy_p':              'Prenatal Use Cocaine',
    'heroin_during_pregnancy_p':               'Prenatal Use Heroin',
    'prescriptionmed_pregnancy_p':             'Prenatal Use Rx',
    'cigs_before_pregnancy_p':                 'Pre-Preg. Cigs',
    'alcohol_before_pregnancy_p':              'Pre-Preg. Alcohol',
    'weed_before_pregnancy_p':                 'Pre-Preg. Cannabis',
    'cocaine_before_pregnancy_p':              'Pre-Preg. Cocaine',
    'heroin_before_pregnancy_p':               'Pre-Preg. Heroin',
    'drugs_before_pregnancy_p':                'Pre-Preg. Drugs',
    'drinksperweek_during_pregnancy_p':        'Prenatal Use Drinks/Wk',
    'drugs_during_pregnancy_p':                'Prenatal Use Drugs',
    'caffeine_during_pregnancy_p':             'Prenatal Use Caffeine',
    'parent_tobacco_use_frequency_p':          'Par. Tobacco Freq.',
    'parent_drug_use_p':                       'Par. Drug Use',
    'parent_drinks_too_much_p':                'Par. Excess Drink',
    'parent_drinks_frequency_p':               'Par. Drink Freq.',
    'parent_drunk_days_p':                     'Par. Drunk Days',
    'parent_drug_days_nonmedical_p':           'Par. Rec. Drug Days',
    'parent_alcohol_days_drunk_p':             'Par. Intox. Days',
    'parent_alcohol_excess_p':                 'Par. Binge Drink',
    'parent_alcohol_freq_p':                   'Par. Alcohol Freq.',
    'parent_drug_days_p':                      'Par. Drug Days',
    'parent_tobacco_use_p':                    'Par. Tobacco Use',
    'druguse_alcohol_p':                       'Child Alcohol',
    'druguse_other_p':                         'Child Oth. Drug',
    'druguse_tobacco':                         'Child Tobacco',

# ---
    # Family Dynamics & Parenting (38)
    # ALL prefixed with Fam.
# ---
    'p_comm_cohesion_ss':               'Fam. Comm. Cohesion',
    'p_comm_ctrl_ss':                   'Fam. Comm. Control',
    'p_comm_collective_efficacy_ss':    'Fam. Collect. Efficacy',
    'fam_fight_often_k':                'Fam. Fight Often (Y)',
    'fam_no_open_anger_k':              'Fam. No Open Anger (Y)',
    'fam_throw_things_k':               'Fam. Throw Things (Y)',
    'fam_no_lose_temps_k':              'Fam. Keep Temper (Y)',
    'fam_criticize_often_k':            'Fam. Criticize (Y)',
    'fam_hit_each_other_k':             'Fam. Hitting (Y)',
    'fam_keep_peace_k':                 'Fam. Keep Peace (Y)',
    'fam_try_one_up_k':                 'Fam. One-Upmanship (Y)',
    'fam_no_raise_voices_k':            'Fam. No Yelling (Y)',
    'family_not_talk_aboutfeelings_p':  'Fam. No Feeling Talk',
    'family_peaceful_p':                'Fam. Peaceful',
    'family_open_discussing_anything_p':'Fam. Open Discussion',
    'family_lose_temper_rare_p':        'Fam. Rare Temper',
    'family_believe_not_raise_voice_p': 'Fam. No Yelling',
    'frequent_family_conflict_p':       'Fam. Freq. Conflict',
    'family_conflict_ss_p':             'Fam. Conflict SS',
    'family_expression_ss_p':           'Fam. Expression SS',
    'family_intellectual_ss_p':         'Fam. Intellectual SS',
    'family_activities_ss_p':           'Fam. Activities SS',
    'family_organisation_ss_p':         'Fam. Organization SS',
    'parents_argue_more_p':             'Fam. Par. Argue',
    'family_emotionprob_p':             'Fam. Emot. Prob.',
    'parents_divorced_p':               'Fam. Par. Divorced',
    'death_in_family_p':                'Fam. Death',
    'family_move_p':                    'Fam. Moved',
    'family_conflict_ss_k':             'Fam. Conflict SS (Y)',
    'parent_monitoring_ss_k':           'Fam. Par. Monitoring (Y)',
    'marital_status':                   'Fam. Marital Status',
    'parent_age':                       'Fam. Par. Age',
    'sex_P_2':                          'Female (Par. Sex)',
    'sex_P_3':                          'Intersex-Male (Par. Sex)',
    'sex_P_4':                          'Intersex-Female (Par. Sex)',
    'num_brothers_p':                   'Fam. Num. Brothers',
    'num_sisters_p':                    'Fam. Num. Sisters',
    'religious_service_frequency':      'Fam. Service Freq.',
    'relig_importance':                 'Fam. Relig. Import.',
    'parent_family_responsibilities_p': 'Fam. Par. Duties',

# ---
    # Family Psychopathology & Well-Being (30)
# ---
    'parent_internal_D_p':              'Par. Internalizing',
    'parent_anhedonia_B_p':             'Par. Anhedonia',
    'obsessions_present_B_p':           'Par. Obsessions',
    'poor_eye_contact_B_p':             'Par. Eye Contact',
    'nightmares_B_p':                   'Par. Nightmares',
    'parent_elevated_mood_B_p':         'Par. Elev. Mood',
    'parent_excessive_worry_B_p':       'Par. Chronic Worry',
    'parent_lying_B_p':                 'Par. Lying',
    'parent_social_anxiety_disorder_B_p':'Par. Social Anx.',
    'parent_sleep_problem_B_p':         'Par. Sleep Prob.',
    'parent_bulimia_B_p':               'Par. Bulimia',
    'parent_attention_D_p':             'Par. Attention Dysregulation',
    'parent_aggressive_D_p':            'Par. Aggressive/Impulsive',
    'parent_external_D_p':              'Par. Externalizing',
    'parent_anxdisord_D_p':             'Par. Anx. Disorder',
    'parent_antisocial_D_p':            'Par. Antisocial',
    'parent_hyperactive_D_p':           'Par. Hyperactivity',
    'parent_somatic_problems_D_p':      'Par. Somatic Issues',
    'parent_intrusive_thoughts_D_p':    'Par. Intrusive Thoughts',
    'parent_avoidant_person_D_p':       'Par. Avoidant',
    'parent_personal_strength_D_p':     'Par. Strengths',
    'd_grandfather_dep':                'Pat. Grandpa Dep.',
    'd_grandmother_dep':                'Pat. Grandma Dep.',
    'm_grandfather_dep':                'Mat. Grandpa Dep.',
    'm_grandmother_dep':                'Mat. Grandma Dep.',
    'father_mania':                     'Father Mania',
    'mother_mania':                     'Mother Mania',
    'father_trouble':                   'Father Trouble',
    'parent_hospitalized_emo':          'Par. Hospitalized',
    'parent_therapy_emo':               'Par. Therapy',

    'suicidal_p':                     'Suicidal Ideation',
    'parent_adhd_D_p':                'Par. ADHD',
    'sex_orient_y_3.0':              'Sex Orientation (3.0)',

# ---
    # Inter (35)
# ---
    'sex_2':                         'Female (Child Sex)',
    'sex_3':                         'Intersex-Male (Child Sex)',
    'sex_4':                         'Intersex-Female (Child Sex)',
    'parent_income':                'Par. Income',
    'parent_income_new':            'Par. Income Adj.',
    'L_site_id':                    'ABCD Site',
    'weight':                       'Weight',
    'social_problems_D_p':          'Social Problems',
    'delta_not_liked_p':            'Change in Unliked',
    'parent_age_grouped':           'Par. Age Group',
    'child_white':                  'White',
    'feels_leftout_k':              'Left Out (Y)',
    'sex_orient_y':                 'Sex Orientation',
    'trans_id_y':                   'Trans Identity',
    'screentime_daysperweek_k':     'Screen Days/Wk (Y)',
    'screentime_weekday_ss_k':      'Weekday Screen (Y)',
    'puberty_k':                    'Puberty Stage (Y)',
    'close_boy_friends_k':          'Close Boy Friends (Y)',
    'close_girl_friends_k':         'Close Girl Friends (Y)',
    'cct':                          'Cash Choice',
    'bad_grades':                   'Bad Grades',
    'tb_cryst':                     'Crystallized Composite (NIH)',
    'crysflu_c':                    'Cryst./Fluid Comp.',
    'child_religion':               'Child Religion',

# ---
    # Medical / Somatic Problems (15)
# ---
    'medhx_p':                  'Med. History',
    'medhx_doctorvisit_p':      'Doctor Visits',
    'medhx_emergencyroom_p':    'ER Visits',
    'pain_last_month_k':        'Recent Pain (Y)',
    'seriously_sick_lastyear_k': 'Sick Last Year (Y)',
    'body_aches_p':             'Body Aches',
    'frequent_headaches_p':     'Headaches',
    'nausea_p':                 'Nausea',
    'eye_problems_p':           'Eye Problems',
    'skin_problems_p':          'Skin Problems',
    'frequent_stomachaches_p':  'Stomachaches',
    'vomiting_p':               'Vomiting',
    'constipated_p':            'Constipation',
    'bad_toilet_habits_p':      'Encopresis',
    'wets_bed_p':               'Enuresis',

# ---
    # Other Personality Features (26)
# ---
    'easily_offended_p':             'Easily Offended',
    'blames_others_p':               'Blames Others',
    'sociable_p':                    'Sociable',
    'school_excitement_p':           'School Excite.',
    'not_critical_others_p':         'Non-Critical',
    'scared_dark_p':                 'Scared Dark',
    'disagreeable_p':                'Disagreeable',
    'goal_continuity_p':             'Goal Persistence',
    'up_negative_urgency_ss_k':      'Neg. Urgency (Y)',
    'up_lackofplanning_ss_k':        'Lack Planning (Y)',
    'up_sensationseeking_ss_k':      'Sensation Seek. (Y)',
    'up_positiveurgency_ss_k':       'Pos. Urgency (Y)',
    'up_lackperseverance_ss_k':      'Lack Persever. (Y)',
    'bis_behav_inhibition_ss_k':     'Behav. Inhibit. (Y)',
    'bis_reward_responsive_ss_k':    'Reward Resp. (Y)',
    'bis_drive_ss_k':                'Drive (Y)',
    'bis_funseeking_ss_k':           'Fun Seeking (Y)',
    'loquacious_p':                  'Loquacious',
    'bragadocious_p':                'Bragging',
    'easily_jealous_p':              'Easily Jealous',
    'wishes_other_sex_p':            'Other Gender Wish',
    'easily_embarrassed_p':          'Easily Embarrassed',
    'secretive_p':                   'Secretive',
    'perfectionist_p':               'Perfectionist',
    'strange_ideas_p':               'Strange Ideas',

# ---
    # Parent Anxiety (7)
# ---
    'parent_fearful_or_anxious_p':       'Par. Fearful',
    'parent_specific_fears_p':           'Par. Phobias',
    'parent_fear_of_bad_thoughts_p':     'Par. Bad Thoughts',
    'parent_worries_about_future_p':     'Par. Future Worry',
    'parent_worries_about_family_p':     'Par. Family Specific Worry',
    'parent_worries_a_lot_p':            'Par. Worries',
    'parent_relationship_concerns_p':    'Par. Relat. Specific Worry',

# ---
    # Parent Cognitive and Attention Issues (15)
# ---
    'parent_forgetful_p':               'Par. Forgetful',
    'parent_concentration_trouble_p':   'Par. Concentration Difficulty',
    'parent_confused_p':                'Par. Confused',
    'parent_planning_trouble_p':        'Par. Planing Trouble',
    'parent_not_good_at_details_p':     'Par. Struggles with Details',
    'parent_obsessive_thoughts_p':      'Par. Obsessive Thinking',
    'parent_repeats_acts_p':            'Par. Compulsions',
    'parent_max_effort_p':              'Par. Low Effort',
    'parent_disorganized_p':            'Par. Disorganized',
    'parent_loses_things_p':            'Par. Loses Things',
    'parent_decision_trouble_p':        'Par. Indecisive',
    'parent_priority_trouble_p':        'Par. Prioritization Difficulty',
    'parent_sees_things_p':             'Par. Vis. Halluc.',
    'parent_hears_voices_p':            'Par. Aud. Halluc.',
    'parent_speech_problems_p':         'Par. Speech Prob.',

# ---
    # Parent Delta Psychopathology (24)
# ---
    'delta_parent_sleep_trouble_p':           'Change in Par. Sleep',
    'delta_parent_worries_about_family_p':    'Change in Par. Fam. Worry',
    'delta_parent_friendship_trouble_p':      'Change in Par. Friend Prob.',
    'delta_parent_poor_work_performance_p':   'Change in Par. Work Performance',
    'delta_parent_aches_pains_p':             'Change in Par. Aches',
    'delta_parent_not_liked_by_others_p':     'Change in Par. Unliked',
    'delta_parent_feels_overwhelmed_p':       'Change in Par. Overwhelmed',
    'delta_parent_feels_unloved_p':           'Change in Par. Feels Unloved',
    'delta_parent_bad_family_relationship_p': 'Change in Par. Fam. Relations',
    'delta_parent_worries_about_future_p':    'Change in Par. Future Worry',
    'delta_parent_worries_a_lot_p':           'Change in Par. Worries',
    'delta_parent_depressed_p':               'Change in Par. Depressed',
    'delta_parent_depress_D_p':               'Parent Depression Syndrome (Change Score)',
    'delta_parent_concentration_trouble_p':   'Change in Par. Concentration',
    'delta_parent_stubborn_irritable_p':      'Change in Par. Stubborn/Irritable',
    'delta_parent_drinks_too_much_p':         'Change in Par. Drinking',
    'delta_parent_financial_failures_p':      'Change in Par. Financial Failures',
    'delta_parent_meets_family_duties_p':     'Change in Par. Fulfills Fam. Responsibilities',
    'delta_parent_planning_trouble_p':        'Change in Par. Planning Trouble',
    'delta_parent_bad_relationships_p':       'Change in Par. General Relationship Quality',
    'delta_parent_drug_use_p':                'Change in Par. Drug Use',
    'delta_parent_drug_days_nonmedical_p':    'Change in Par. Rec. Drugs',
    'delta_parent_financial_trouble_p':       'Change in Par. Finances',
    'delta_parent_internal_D_p':              'Change in Par. Internal.',
    'delta_parent_somatic_problems_D_p':      'Change in Par. Somatic',

# ---
    # Parent Impulsivity and Behavior Regulation (16)
# ---
    'parent_impulsive_p':                  'Par. Impulsive',
    'parent_risky_decisions_p':            'Par. Risky Decisions',
    'parent_drives_too_fast_p':            'Par. Drives Fast',
    'parent_tardy_often_p':                'Par. Tardy',
    'parent_money_management_trouble_p':   'Par. Money Management Prob.',
    'parent_behavior_changeable_p':        'Par. Behavior Labile',
    'parent_hot_temper_p':                 'Par. Hot Temper',
    'parent_attention_seeking_p':          'Par. Attention Seeking',
    'parent_destroys_own_things_p':        'Par. Destroys Own',
    'parent_destroys_others_things_p':     'Par. Destroys Oth.',
    'parent_doesnt_finish_tasks_p':        'Par. Task Completion',
    'parent_strange_behavior_p':           'Par. Strange Behavior',
    'parent_illegal_behavior_p':           'Par. Illegal Behav.',
    'parent_self_harm_p':                  'Par. Self-Harm',
    'parent_doesnt_eat_well_p':            'Par. Poor Eating',

# ---
    # Parent Mood Issues (14)
# ---
    'parent_cries_a_lot_p':           'Par. Cries',
    'parent_lonely_p':                'Par. Lonely',
    'parent_feels_unloved_p':         'Par. Unloved',
    'parent_paranoid_p':              'Par. Paranoid',
    'parent_feels_inferior_p':        'Par. Feels Inferior',
    'parent_depressed_p':             'Par. Depressed',
    'parent_feels_unsuccessful_p':    'Par. Feels Unsuccessful',
    'parent_tired_no_reason_p':       'Par. Fatigue',
    'parent_low_energy_p':            'Par. Low Energy',
    'parent_sleep_trouble_p':         'Par. Sleep Trouble',
    'parent_enjoys_little_p':         'Par. No Pleasure',
    'parent_sudden_mood_changes_p':   'Par. Mood Swings',
    'parent_suicidal_thoughts_p':     'Par. Suicidal',
    'parent_happy_person_p':          'Par. Happiness',

# ---
    # Parent Other (4)
# ---
    'parent_heart_racing_p':      'Par. Heart Racing',
    'parent_numbness_p':          'Par. Numbness',
    'parent_physical_attacks_p':  'Par. Panic Attacks',
    'parent_picks_skin_p':        'Par. Skin Picking',

# ---
    # Parent Personality (21)
# ---
    'parent_bragging_p':               'Par. Bragging',
    'parent_honest_p':                 'Par. Honest',
    'parent_secretive_p':              'Par. Secretive',
    'parent_stubborn_irritable_p':     'Par. Stubborn/Irritable',
    'parent_clumsy_p':                 'Par. Clumsy',
    'parent_strange_thoughts_p':       'Par. Odd Thoughts',
    'parent_self_conscious_p':         'Par. Low Self Confidence',
    'parent_uses_opportunities_p':     'Par. Opportunistic',
    'parent_louder_than_others_p':     'Par. Loud',
    'parent_yells_a_lot_p':            'Par. Yells',
    'parent_shy_or_timid_p':           'Par. Shy',
    'parent_restless_p':               'Par. Restless',
    'parent_easily_bored_p':           'Par. Easily Bored',
    'parent_hyperactive_p':            'Par. Hyperactive',
    'parent_talks_too_much_p':         'Par. Talkative',
    'parent_avoids_talking_p':         'Par. Reticent',
    'parent_prefers_to_be_alone_p':    'Par. Prefers to be Alone',
    'parent_no_guilt_p':               'Par. No Guilt',
    'parent_sense_of_fairness_p':      'Par. Fairness',
    'parent_high_sleep_duration_p':    'Par. Long Sleep Duration',
    'parent_opposite_sex_wish_p':      'Par. Gender Wish',

# ---
    # Parent Psychopathology Composite (5)
# ---
    'parent_anxdep_D_p':          'Par. Anx./Dep.',
    'parent_depressed_mood_B_p':  'Par. Dep. Mood',
    'parent_dysthymia_B_p':       'Par. Dysthymia',
    'parent_suicide_past_B_p':    'Par. Past Suicide',
    'ptsd_diagnosis_present_B_p': 'Par. PTSD',

# ---
    # Parent Social Functioning (11)
# ---
    'parent_bad_relationships_p':               'Par. General Bad Relationships',
    'parent_bad_family_relationship_p':         'Par. Bad Family Relations',
    'parent_not_liked_by_others_p':             'Par. Unliked by Others',
    'parent_friendship_trouble_p':              'Par. Friendship Problems',
    'parent_prefers_older_people_p':            'Par. Pref. Older Friends',
    'parent_associates_with_trouble_p':         'Par. Gets in Trouble',
    'parent_bad_opposite_sex_relationship_p':   'Par. Opposite Sex Relationships',
    'parent_meets_family_duties_p':             'Par. Fulfills Fam. Responsibilities',
    'parent_clowns_or_shows_off_p':             'Par. Socially Performative',
    'parent_teases_others_p':                   'Par. Teases',
    'parent_stands_up_rights_p':                'Par. Assertive',

# ---
    # Physical Activity / Features (13)
# ---
    'height':                    'Height',
    'waist':                     'Waist Circumference',
    'no_sports_activities_p':    'No Sports',
    'birth_weight_p':            'Birth Weight',
    'fitbit_resting_hr':         'Resting HR',
    'fitbit_steps':              'Daily Steps',
    'fitbit_sedentary_mins':     'Sedentary Min.',
    'fitbit_lightlyactive_mins': 'Light Active Min.',
    'fitbit_fairlyactive_mins':  'Moderate Active Min.',
    'fitbit_veryactive_mins':    'Vigorous Active Min.',

# ---
    # Positive Affect (9)
# ---
    'pa_at_ease':       'At Ease',
    'pa_attentive':     'Attentive',
    'pa_calm':          'Calm',
    'pa_concentrate':   'Concentrating',
    'pa_confident':     'Confident',
    'pa_delighted':     'Delighted',
    'pa_energetic':     'Energetic',
    'pa_enthusiastic':  'Enthusiastic',
    'pa_interested':    'Interested',

# ---
    # Residential Characteristics (19)
# ---
    'neighborhood_safety_ss_p':        'Nbhd. Safety',
    'neighborhood_safe_y':             'Nbhd. Safe Youth',
    'resid_density':                   'Pop. Density',
    'resid_walkability':               'Walkability',
    'resid_prox_roads':                'Road Proximity',
    'resid_crime_tot':                 'Total Crime',
    'resid_crime_violent':             'Violent Crime',
    'resid_crime_drug':                'Drug Crime',
    'resid_crime_dui':                 'DUI Rate',
    'resid_lead_risk_poverty':         'Lead Risk Pov.',
    'resid_lead_risk_houses_perc':     'Lead Risk Hsg.',
    'resid_lead_risk':                 'Lead Risk',
    'resid_no2_avg':                   'NO2 Exposure',
    'resid_pm25_avg':                  'PM2.5 Exposure',
    'resid_sexism':                    'Area Sexism',
    'resid_sex_orient_bias':           'Area SO Bias',
    'resid_immigrant_bias':            'Immigrant Bias',
    'resid_racism':                    'Area Racism',

# ---
    # SES & Mobility (11)
# ---
    'parent_education':               'Par. Education',
    'struggle_food_expenses':         'Food Hardship',
    #this one was reverse coded on accident
    'positive_finance_p':             'Negative Financial Dynamics',
    'parent_work_absences_p':         'Par. Work Absences',
    'parent_financial_trouble_p':     'Par. Fin. Trouble',
    'parent_fails_to_pay_debts_p':    'Par. Debt Default',
    'couldnt_afford_phone':           'No Phone',
    'couldnt_afford_rent_mortgage':   'Housing Hardship',
    'evicted':                        'Evicted',
    'gas_electric_oil_turned_off':    'Utilities Cutoff',

# ---
    # School Dynamics (15)
# ---
    'disobeys_at_school_k':         'Disobeys School (Y)',
    'getalong_teachers_k':          'Teacher Rapport (Y)',
    'feelsafe_at_school_k':         'School Safety (Y)',
    'feels_smart_k':                'Feels Smart (Y)',
    'enjoys_school_k':              'Enjoys School (Y)',
    'grades_important_k':           'Grades Matter (Y)',
    'school_environment_ss_k':      'School Environ. (Y)',
    'school_involvement_ss_k':      'School Involve. (Y)',
    'school_disengagement_ss_k':    'Disengagement (Y)',
    'repeated_grade':               'Repeated Grade',
    'grades_dropped':               'Grades Dropped',
    'school_detension_suspension':  'Detention/Susp.',
    'child_newschool_p':            'New School',
    'finds_schoolboring_k':         'School Boring (Y)',

# ---
    # Sleep Problems (12)
# ---
    'difficulty_goingtosleep_p':  'Onset Difficulty',
    'difficulty_wakingup_p':      'Wake Difficulty',
    'fallsleeptime':              'Sleep Onset Time',
    'wakeuptime':                 'Wake Time',
    'wakesleepcalc':              'Sleep Duration',
    'chronotype':                 'Chronotype',
    'nightmares_p':               'Nightmares',
    'sleeps_more_p':              'Sleeps More',
    'sleeps_less_p':              'Sleeps Less',
    'sleeps_little_p':            'Little Sleep',
    'sleeps_alot_p':              'Excess Sleep',
    'sleep_problems_p':           'Sleep Problems',

# ---
    # Social Relationship Quality (34)
# ---
    'not_liked_p':                              'Not Liked By Others',
    'doesnt_get_along_p':                       'Peer Conflict',
    'prosocial_ss_p':                           'Prosocial',
    'peer_net_protective_ss_k':                 'Peer Protection (Y)',
    'peers_beh_prosocial_ss_k':                 'Peer Prosocial (Y)',
    'peers_beh_delinquent_ss_k':                'Peer Delinquency (Y)',
    'not_invited_k':                            'Not Invited (Y)',
    'excluded_k':                               'Excluded (Y)',
    'otherkids_spreadneg_rumors_k':             'Rumors (Y)',
    'otherkids_gossip_k':                       'Gossip (Y)',
    'feels_threatned_k':                        'Threatened (Y)',
    'saysmeanthings_others_k':                  'Says Mean (Y)',
    'otherkids_saymeanthings_k':                'Mean to Me (Y)',
    'discrimination_ss_k':                      'Discrimination (Y)',
    'feels_discriminated_k':                    'Feels Discrim. (Y)',
    'senses_racism_k':                          'Senses Racism (Y)',
    'doesnt_feel_accepted_k':                   'Not Accepted (Y)',
    'bullied_on_internet_k':                    'Cyberbullied (Y)',
    'prosocial_ss_k':                           'Prosocial (Y)',
    'socialinfluence_meanfinal_k':              'Social Influence (Y)',
    'relational_victimization_ss_k':            'Relat. Victim. (Y)',
    'reputational_aggression_ss_k':             'Reput. Aggress. (Y)',
    'reputational_victimization_ss_k':          'Reput. Victim. (Y)',
    'overt_aggression_ss_k':                    'Overt Aggress. (Y)',
    'overt_victimization_ss_k':                 'Overt Victim. (Y)',
    'relational_aggression_ss_k':               'Relat. Aggress. (Y)',
    'feels_discriminated_teachers_k':           'Discrim. Teachers (Y)',
    'feels_discriminated_adults_not_school_k':  'Discrim. Adults (Y)',
    'feels_discriminated_students_k':           'Discrim. Students (Y)',
    'feels_unwanted_american_society_k':        'Feels Unwanted (Y)',
    'feels_discriminated_americans_k':          'Discrim. Society (Y)',

# ---
    # Technology Use (4)
# ---
    'socialmedia_daysperweek_k':  'Social Media (Y)',
    'videogames_daysperweek_k':   'Gaming Days (Y)',
    'vgame_thinking':             'Gaming Preoccup.',

# ---
    # Structural and DTI (150)
# ---
    **{f'pc_struct{i}': f'Struct PC{i}' for i in range(1, 76)},
    **{f'pc_DTI{i}': f'DTI PC{i}' for i in range(1, 76)},

# ---
    # Task and Resting State (300)
# ---
    **{f'pc_SSTmri{i}': f'SST-fMRI PC{i}' for i in range(1, 76)},
    **{f'pc_nbackmri{i}': f'NBack-fMRI PC{i}' for i in range(1, 76)},
    **{f'pc_midfmri{i}': f'MID-fMRI PC{i}' for i in range(1, 76)},
    **{f'pc_rsFMRI{i}': f'RS-fMRI PC{i}' for i in range(1, 76)},

# ---
    # Social Responsiveness (SSRS) -- Parent Report
# ---
    'avoids_eyecontact_p':          'Avoids Eye Contact',
    'difficulty_making_friends_p':  'Difficulty Making Friends',
    'regarded_weird_p':             'Regarded as Odd',
    'bad_conversational_flow_p':    'Poor Conversational Flow',
    'narrow_interests_p':           'Narrow Interests',
    'sensory_sensitivity_p':        'Sensory Sensitivity',
    'concentration_on_parts_p':     'Detail Fixation',
    'face_understanding':           'Poor Social Cue Reading',

# ---
    # Barkley Deficits in Executive Functioning (BDEFS) -- Parent Report
# ---
    'bdefs_calm_down_p':            'Difficulty Calming Down',
    'bdefs_consequences_p':         'Ignores Consequences',
    'bdefs_distract_upset_p':       'Emotional Refocusing Deficit',
    'bdefs_explain_idea_p':         'Poor Idea Expression',
    'bdefs_explain_pt_p':           'Unfocused Explanations',
    'bdefs_explain_seq_p':          'Poor Sequential Narration',
    'bdefs_impulsive_action_p':     'Impulsive Actions',
    'bdefs_inconsistant_p':         'Inconsistent Performance',
    'bdefs_lazy_p':                 'Appears Unmotivated',
    'bdefs_process_info_p':         'Slow Info Processing',
    'bdefs_rechannel_p':            'Poor Emotion Redirection',
    'bdefs_sense_time_p':           'Poor Time Sense',
    'bdefs_shortcuts_p':            'Takes Shortcuts',
    'bdefs_stop_think_p':           'Acts Before Thinking',

# ---
    # Child Emotional Regulation
# ---
    'child_emotion_overwhelm_p':    'Emotion Overwhelm',
    'child_clear_feelings_p':       'Emotion Clarity',

# ---
    # Child Emotion Regulation (ERC/DERS) -- Parent Report
# ---
    'child_feelings_attentive_p':       'Attentive to Feelings',
    'child_feelings_care_p':            'Cares About Feelings',
    'child_feelings_know_p':            'Knows Own Feelings',
    'child_upset_acknowledge_p':        'Acknowledges Upset',
    'child_upset_angry_p':              'Upset: Gets Angry',
    'child_upset_ashamed_p':            'Upset: Feels Ashamed',
    'child_upset_bad_behavior_p':       'Upset: Bad Behavior',
    'child_upset_bad_esteem_p':         'Upset: Low Self-Esteem',
    'child_upset_bad_focus_p':          'Upset: Poor Focus',
    'child_upset_better_p':             'Upset: Recovers Quickly',
    'child_upset_control_p':            'Upset: Maintains Control',
    'child_upset_embarrassed_p':        'Upset: Embarrassed',
    'child_upset_emotions_overwhelm_p': 'Upset: Emotionally Overwhelmed',
    'child_upset_fixation_p':           'Upset: Ruminative Fixation',
    'child_upset_guilty_p':             'Upset: Feels Guilty',
    'child_upset_irritated_p':          'Upset: Irritable',
    'child_upset_longtime_better_p':    'Upset: Slow to Recover',
    'child_upset_longtime_p':           'Upset: Prolonged Duration',
    'child_upset_lose_control_p':       'Upset: Loses Control',
    'child_upset_no_control_p':         'Upset: No Self-Control',
    'child_upset_nothing_better_p':     'Upset: Feels Hopeless',
    'child_upset_out_control_p':        'Upset: Out of Control',
    'child_upset_poor_concentrate_p':   'Upset: Can\'t Concentrate',
    'child_upset_unproductive_p':       'Upset: Unproductive',
    'child_upset_weak_p':               'Upset: Feels Weak',

# ---
    # Emotion Regulation Strategies -- Youth Self-Report (T3/T4)
# ---
    'emoreg_reapp_ss_k':            'Reappraisal (Sum)',
    'emoreg_reapp_happy_k':         'Reappraises: Happy',
    'emoreg_reapp_less_bad_k':      'Reappraises: Less Bad',
    'emoreg_reapp_thoughts_k':      'Reappraises: Thoughts',
    'emoreg_sup_ss_k':              'Suppression (Sum)',
    'emoreg_sup_control_k':         'Suppresses: Control',
    'emoreg_sup_hide_k':            'Suppresses: Hides',
    'emoreg_sup_self_k':            'Suppresses: Self',
    'child_selfaware_p':            'Self-Awareness',

# ---
    # Smartphone Use (T3/T4)
# ---
    'sphone_Interupt':              'Phone Interruptions',
    'sphone_attachment':            'Phone Attachment',
    'sphone_distress':              'Phone Distress',
    'sphone_noreason':              'Phone: No Reason',

# ---
    # Parental Monitoring -- Youth Self-Report (T3)
# ---
    'parent_care_friends_k':        'Par. Knows Friends (Y)',
    'parent_care_misbehave_k':      'Par. Monitors Behavior (Y)',
    'parent_care_whereabouts_k':    'Par. Knows Whereabouts (Y)',
    'parent_cares_ss_k':            'Parental Monitoring (Y)',
    'parent_gotoschool_k':          'Par. School Attendance (Y)',
    'parent_helphomework_k':        'Par. Homework Help (Y)',
    'parent_helpunderstanding_k':   'Par. Helps Understand (Y)',
    'parent_safeplay_k':            'Par. Safe Play (Y)',
    'parent_troubleschool_k':       'Par. School Trouble (Y)',

# ---
    # Parental Acceptance -- Youth Report (T3/T4)
# ---
    'y_acceptance_ss_caregiver_crpbi': 'Caregiver Acceptance (Y)',
    'y_acceptance_ss_p_crpbi':         'Parent Acceptance (Y)',

# ---
    # Social / Friendship (T3/T4)
# ---
    'close_nonB_friends_k':         'Close Friends (Y)',
    'nonBfriends_k':                'Number of Friends (Y)',

# ---
    # Other T3/T4-Specific
# ---
    'mania_7up_ss_k':               'Mania Symptoms (Y)',
    'problem_solving_ss_k':         'Problem Solving (Y)',
    'enjoys_music_k':               'Enjoys Music (Y)',

# ---
    # CBCL Items (present in T3/T4 flat lists)
# ---
    'enjoys_little_p':              'Enjoys Little',
    'guilty_p':                     'Feels Guilty',
    'sad_p':                        'Sad',
    'withdrawn_p':                  'Withdrawn',

# ---
    # Cognitive Tasks -- Matrix Reasoning (T0)
# ---
    'mr_matrix':                    'Matrix Reasoning',
    'mr_total':                     'Matrix Reasoning Total',

# ---
    # Stroop Task (T1)
# ---
    'str_accuracy':                 'Stroop Accuracy',
    'str_accuracy_c':               'Stroop Accuracy (Cong.)',
    'str_accuracy_ic':              'Stroop Accuracy (Incong.)',
    'str_mrt':                      'Stroop Reaction Time',
    'str_mrt_c':                    'Stroop RT (Congruent)',
    'str_mrt_ic':                   'Stroop RT (Incongruent)',
    'str_stroop_acc':               'Stroop Effect (Acc.)',
    'str_stroop_mrt':               'Stroop Effect (RT)',

# ---
    # Medical History -- Parent Report (T1)
# ---
    'medhx_allergies_p':            'Hx: Allergies',
    'medhx_asthma_p':               'Hx: Asthma',
    'medhx_brain_p':                'Hx: Brain Condition',
    'medhx_brokenbones_p':          'Hx: Broken Bones',
    'medhx_diabetes_p':             'Hx: Diabetes',
    'medhx_epilepsy_p':             'Hx: Epilepsy',
    'medhx_headaches_p':            'Hx: Headaches',
    'medhx_heart_p':                'Hx: Heart Condition',
    'medhx_seriousinjury_p':        'Hx: Serious Injury',

# ---
    # Biomarkers / Hormones (T0/T1)
# ---
    'saliva_DHEA':                  'Salivary DHEA',
    'saliva_testosterone':          'Salivary Testosterone',

# ---
    # Other T0/T1-Specific
# ---
    'screentime_weekend_ss_k':      'Weekend Screen Time (Y)',
    'sex_P':                        'Biological Sex',
}

# ---
# Validation
# ---
print(f"DISPLAY_NAMES contains {len(DISPLAY_NAMES)} mappings")

# ---
# TARGET_DISPLAY_NAMES -- Outcome Variable Renaming for Figures
# ---
# Translates raw target variable names into publication-ready labels for
# heatmap axes, figure titles, and summary tables. used by
# downstream visualization cells (Cross-Target Heatmap, Residual
# Diagnostics, Manuscript Figure Generator) and optionally by the Batch Metrics cell.
#
# Naming conventions:
# "Mood/Depression (Primary Features)"  -- SBT core depression
# "Depression Syndrome (w/ Secondary Features)"  -- CBCL depress_D_p
# "Depression Subtype (X)"  -- SBT subscales
# "(Change Score)"  -- delta / longitudinal change
# "(RCI ≥ X)"  -- Reliable Change Index thresholds
# ---

TARGET_DISPLAY_NAMES = {

    # -- Core Syndrome Scales (Continuous) ---
    'sbt_core_depression':          'Mood/Depression (Primary Features)',
    'depress_D_p':                  'Depression Syndrome (w/ Secondary Features)',
    'depress_D_p_rev':              'Depression Syndrome (Reversed)',
    'depress_update_p':             'Depression (Updated Composite)',
    'internal_D_p':                 'Internalizing Spectra',
    'external_D_p':                 'Externalizing Spectra',
    'adhd_D_p':                     'ADHD (Syndrome)',
    'anxdisord_D_p':                'Anxiety (Syndrome)',
    'social_problems_D_p':          'Social Problems (Syndrome)',
    'thought_disorder_D_p':         'Thought Disorder (Syndrome)',

    # -- SBT Depression Subtypes (Continuous) ---
    'sbt_guilt_hopelessness':       'Depression Subtype (Guilt/Hopelessness)',
    'sbt_social_withdrawal':        'Depression Subtype (Social Withdrawal)',
    'sbt_fatigue_sleep':            'Depression Subtype (Fatigue/Sleep)',
    'sbt_somatic_depression':       'Depression Subtype (Somatic)',
    'sbt_anxiety_depression':       'Depression Subtype (Anxiety/Depression Overlap)',
    'sbt_emotional_dysregulation':  'Depression Subtype (Emotional Dysregulation)',
    'sbt_avoidance_fear':           'Depression Subtype (Avoidance/Fear)',
    'sbt_aggression_irritability':  'Depression Subtype (Aggression/Irritability)',
    'sbt_academic_cognitive':       'Depression Subtype (Poor Academic/Cognitive)',
    'sbt_perfectionism_achievement':'Depression Subtype (Perfectionism/High-Achievement)',
    'sbt_well_being':               'Depression Subtype (Well-Being)',

    # -- Change Scores (Continuous) ---
    'delta_depress_D_p':            'Depression Syndrome (Change Score)',
    'delta_sbt_core_depression':    'Mood/Depression (Change Score)',
    'sbt_core_depression_delta_t0_t3': 'Mood/Depression (Change T0→T3)',
    'delta_anxdisord_D_p':          'Anxiety (Change Score)',
    'delta_parent_depressed_p':     'Parent Depression (Change Score)',
    'delta_parent_worries_a_lot_p': 'Parent Worry (Change Score)',

    # -- RCI-Based Binary Outcomes ---
    'dep_onset_rci_2.3':            'Depression Onset (RCI ≥ 2.3)',
    'dep_onset_rci_1.96':           'Depression Onset (RCI ≥ 1.96)',
    'dep_remission_rci_2.3':        'Depression Remission (RCI ≥ 2.3)',
    'dep_remission_rci_1.96':       'Depression Remission (RCI ≥ 1.96)',
    'sbt_core_dep_onset_rci_2.3':   'Mood/Depression Onset (RCI ≥ 2.3)',
    'sbt_core_dep_onset_rci_1.96':  'Mood/Depression Onset (RCI ≥ 1.96)',
    'sbt_core_dep_remission_rci_2.3':  'Mood/Depression Remission (RCI ≥ 2.3)',
    'sbt_core_dep_remission_rci_1.96': 'Mood/Depression Remission (RCI ≥ 1.96)',

    # -- Threshold-Based Binary Outcomes ---
    'dep_increase_2sd':             'Depression Increase (≥ 2 SD)',
    'dep_increase_1.5sd':           'Depression Increase (≥ 1.5 SD)',
    'dep_decrease_2sd':             'Depression Decrease (≥ 2 SD)',
    'dep_decrease_1.5sd':           'Depression Decrease (≥ 1.5 SD)',
    'sbt_core_dep_increase_2sd':    'Mood/Depression Increase (≥ 2 SD)',
    'sbt_core_dep_increase_1.5sd':  'Mood/Depression Increase (≥ 1.5 SD)',
    'sbt_core_dep_decrease_2sd':    'Mood/Depression Decrease (≥ 2 SD)',
    'sbt_core_dep_decrease_1.5sd':  'Mood/Depression Decrease (≥ 1.5 SD)',
    'top_10_depression':            'Depression (Top 10th Percentile)',
    'top_5_depression':             'Depression (Top 5th Percentile)',
    'top_10_depression_parent':      'Parent Depression (Top 10th Percentile)',
    'top_5_depression_parent':       'Parent Depression (Top 5th Percentile)',
    'top_10_sbt_core_depression':   'Mood/Depression (Top 10th Percentile)',
    'top_5_sbt_core_depression':    'Mood/Depression (Top 5th Percentile)',

    # -- Clinical / Diagnostic Binary Outcomes ---
    'MDD_KSADS_C':                  'MDD Diagnosis (KSADS)',
    'depressed_mood_B_k':           'Depressed Mood (KSADS Item)',
    'anhedonia_B_k':                'Anhedonia (KSADS Item)',
    'wish_dead_present_B_k':        'Wish to Be Dead (Child Report)',
    'parent_wish_dead_present_B_p': 'Wish to Be Dead (Parent Report)',
    'latent_class_depression':      'Depression Latent Class',
    'depadhd_c':                    'Depression × ADHD Comorbidity Class',
    'asdadhd_c':                    'ASD × ADHD Comorbidity Class',

    # -- Parent Outcomes ---
    'parent_depress_D_p':           'Parent Depression (Syndrome)',
    'parent_anxdisord_D_p':         'Parent Anxiety',
    'parent_adhd_D_p':              'Parent ADHD',
    # -- Parent Social/Impulsivity/Compulsivity batch ---
    'parent_avoidant_person_D_p':   'Parent Avoidant Personality (Syndrome)',
    'parent_not_liked_by_others_p': 'Parent Not Liked by Others',
    'parent_repeats_acts_p':        'Parent Compulsivity (Repeats Acts)',
    'parent_obsessive_thoughts_p':  'Parent Obsessions (Obsessive Thoughts)',
    'parent_dep_onset_rci_2.3':     'Parent Depression Onset (RCI ≥ 2.3)',
    'parent_bad_relationships_p':   'Parent Relationship Problems',
    'parent_suicidal_thoughts_p':   'Parent Suicidal Ideation',
    'parent_depressed_p':           'Parent Mood/Depression (Primary Features)',
    'parent_worries_a_lot_p':       'Parent Worry (Item)',
    'parent_concentration_trouble_p': 'Parent Concentration Trouble (Item)',
    'parent_happy_person_p':        'Parent Happiness (Item)',

    # -- Other Targets ---
    'bad_grades':                   'Poor Academic Grades',
    'tb_flanker':                   'Cognitive Control (Flanker)',
    'tb_fluid':                     'Fluid Intelligence',
    'tb_cryst':                     'Crystallized Intelligence',
    'tb_reading':                   'Reading Ability',
    # -- T0 Cognitive Task Battery additions ---
    'ravlt_l_total':                'Verbal Episodic Memory (RAVLT)',
    'lmt_efficiency':               'Visuospatial Mental Rotation (LMT)',
    'sst_mean_ssrt':                'Response Inhibition (SST SSRT)',
    'nb_correct_nt_2back':          'Emotional Working Memory (2-Back)',
    'suicidal_p':                   'Suicidal Ideation (Child)',
    'compulsive_p':                 'Compulsive Behavior',
    'impulsive_p':                  'Impulsive Behavior',
    'parent_impulsive_p':           'Parent Impulsive Behavior',
    'parent_tobacco_use_frequency_p': 'Parent Tobacco Use',
    'parent_drinks_frequency_p':    'Parent Alcohol Use',
    'parent_drunk_days_p':          'Parent Heavy Drinking',
    'parent_income':                'Parent Income',
    'area_deprivation_idx':         'Area Deprivation Index',
    'asd_ssrs_sum':                 'ASD Symptoms (SSRS Total)',
    'bdefs_distract_upset_p':       'Executive Dysfunction (Distract/Upset)',
    'gd_riskybets':                 'Risk-Taking (Game of Dice)',
    'pa_sum_k':                     'Physical Activity (Child Report)',
    'weight':                        'Body Weight',

    #Parent Spectra
    'parent_internal_D_p':              'Parent Internalizing Spectra',
    'parent_external_D_p':              'Parent Externalizing Spectra',

    # -- RCI 1.7 Variants (Child) ---
    'dep_onset_rci_1.7':            'Depression Onset (RCI ≥ 1.7)',
    'dep_remission_rci_1.7':        'Depression Remission (RCI ≥ 1.7)',
    'sbt_core_dep_onset_rci_1.7':   'Mood/Depression Onset (RCI ≥ 1.7)',
    'sbt_core_dep_remission_rci_1.7': 'Mood/Depression Remission (RCI ≥ 1.7)',

    # -- 1 SD Variants (Child) ---
    'dep_increase_1sd':             'Depression Worsened (≥ 1 SD)',
    'dep_decrease_1sd':             'Depression Improved (≥ 1 SD)',

    # -- Parent Classification Targets ---
    'parent_dep_onset_rci_1.96':    'Parent Depression Onset (RCI ≥ 1.96)',
    'parent_dep_remission_rci_1.96':'Parent Depression Remission (RCI ≥ 1.96)',
    'parent_dep_remission_rci_2.3': 'Parent Depression Remission (RCI ≥ 2.3)',
    'parent_dep_increase_2sd':      'Parent Depression Changes (≥ 2 SD)',
    'parent_dep_increase_1.5sd':    'Parent Depression Changes (≥ 1.5 SD)',
    'parent_dep_decrease_2sd':      'Parent Depression Improved (≥ 2 SD)',
    'parent_dep_decrease_1.5sd':    'Parent Depression Improved (≥ 1.5 SD)',

    # -- KSADS Child -- Depression / Mood ---
    'anhedonia_B_k':                  'Anhedonia (KSADS)',
    'depressed_mood_B_k':             'Depressed Mood (KSADS)',
    'elevated_mood_B_k':              'Elevated Mood (KSADS)',
    'fatigue_present_B_k':            'Fatigue (KSADS)',
    'hopeless_B_k':                   'Hopelessness (KSADS)',
    'ksads_1_840_t':                  'Major Depressive Disorder (KSADS)',
    'two_more_depression_B_k':        '≥2 Depressive Symptoms (KSADS)',

    # -- KSADS Child -- Suicidality ---
    'suicidal_ideation_present_B_k':  'Suicidal Ideation (KSADS)',
    'suicide_attempt_past_B_k':       'Past Suicide Attempt (KSADS)',
    'suicide_attempt_present_B_k':    'Suicide Attempt (KSADS)',

    # -- KSADS Child -- Anxiety ---
    'excessive_worry_B_k':            'Excessive Worry (KSADS)',
    'GAD_present':                    'Generalized Anxiety Disorder (KSADS)',
    'OCD_present':                    'OCD (KSADS)',
    'social_anxiety_disorder_B_k':    'Social Anxiety Disorder (KSADS)',
    'socially_anxious_B_k':           'Social Anxiety Symptoms (KSADS)',

    # -- KSADS Child -- Other ---
    'anorexia_B_k':                   'Anorexia Nervosa (KSADS)',
    'bulimia_B_k':                    'Bulimia Nervosa (KSADS)',
    'lying_B_k':                      'Frequent Lying (KSADS)',
    'sleep_problem_B_k':              'Sleep Problems (KSADS)',

    # -- KSADS Parent -- Depression ---
    'hopeless_B_p':                   'Parent Hopelessness (KSADS)',
    'parent_anhedonia_B_p':           'Parent Anhedonia (KSADS)',
    'parent_depressed_mood_B_p':      'Parent Depressed Mood (KSADS)',
    'parent_dysthymia_B_p':           'Parent Dysthymia (KSADS)',
    'parent_low_self_esteem_B_p':     'Parent Low Self-Esteem (KSADS)',
    'parent_two_more_depression_B_p': 'Parent ≥2 Depressive Symptoms (KSADS)',

    # -- KSADS Parent -- Anxiety ---
    'parent_distress_due_to_social_anxiety_B_p': 'Parent Social Anxiety Distress (KSADS)',
    'parent_excessive_worry_B_p':     'Parent Excessive Worry (KSADS)',
    'parent_obsessions_present_B_p':  'Parent Obsessions (KSADS)',
    'parent_social_anxiety_disorder_B_p': 'Parent Social Anxiety Disorder (KSADS)',
    'parent_socially_anxious_B_p':    'Parent Social Anxiety Symptoms (KSADS)',

    # -- KSADS Parent -- Other ---
    'parent_concentration_issues_B_p':'Parent Concentration Problems (KSADS)',
    'parent_easily_annoyed_B_p':      'Parent Irritability (KSADS)',
    'parent_sleep_problem_B_p':       'Parent Sleep Problems (KSADS)',
    'ptsd_diagnosis_present_B_p':     'Parent PTSD (KSADS)',
}

# ---
# Helper: resolve target display name
# ---
def get_target_display_name(raw_name):
    """Return the publication-ready display name for a target variable.
    Falls back to the raw name if no mapping exists."""
    return TARGET_DISPLAY_NAMES.get(raw_name, raw_name)

print(f"TARGET_DISPLAY_NAMES contains {len(TARGET_DISPLAY_NAMES)} mappings")

In [ ]:
#@title Stacking Ensemble Architecture (CatBoost-XGBoost-RF-TabPFN-Meta-Learner)

# Two-stage stacking ensemble. Stage 1: base learners (CatBoost, XGBoost,
# RF, TabPFN) generate out-of-fold predictions via inner CV. Stage 2:
# linear (regression) or logistic (classification) meta-learner trains
# on those held-out predictions. Tree-based models capture nonlinear
# interactions across mixed variable types; TabPFN approximates Bayesian
# posteriors for uncertainty quantification. TabPFN auto-skipped for
# across-category runs because feature counts typically exceed TabPFN's
# recommended d <= 500 ceiling.
# ElasticNet retained for comparison only, not part of the reported ensemble.

# ---
# TabPFN (Hollmann et al., 2025 Nature) is a transformer-based prior-data
# fitted network that approximates Bayesian posterior predictive
# distributions over tabular data, providing built-in uncertainty
# quantification without manual hyperparameter tuning. The stacking
# procedure is verified for data-leakage integrity in the Stacking Audit
# cell.
# ---

# ---
# MODEL HYPERPARAMETERS
# ---
# Every call site (run_catboost_analysis, validation cells, cross-
# category refit, stacking auditor) MUST reference this dict rather
# than hard-coding values. To change a hyperparameter, edit it here
# once.
#
# CatBoost 'best_params' may be overridden per-run by Optuna
# when hyperparameter_tuning=True; the values below are the
# non-Optuna defaults used in the manuscript.
# ---

# ---
# TabPFN Device Resolution -- detect GPU once, reuse everywhere
# ---
import torch as _torch
_TABPFN_DEVICE = 'cuda' if _torch.cuda.is_available() else 'cpu'
if _TABPFN_DEVICE == 'cuda':
    _gpu_name = _torch.cuda.get_device_name(0)
    _gpu_mem  = round(getattr(_torch.cuda.get_device_properties(0), 'total_memory', 0) / 1024**3, 1)
    print(f"[TabPFN] GPU detected: {_gpu_name} ({_gpu_mem} GB) -- using CUDA")
else:
    print("[TabPFN] No GPU detected -- using CPU (≤1000 samples only)")
del _torch  # avoid polluting namespace

PIPELINE_MODEL_CONFIGS = {
    'catboost': {
        'iterations': 1000,
        'learning_rate': 0.05,
        'depth': 6,
        'l2_leaf_reg': 3,
        'border_count': 128,
        'thread_count': -1,
        'verbose': False,
    },
    'random_forest': {
        'n_estimators': 200,
        'max_depth': 10,
    },
    'xgboost': {
        'n_estimators': 500,
        'learning_rate': 0.1,
        'max_depth': 6,
    },
    'tabpfn': {
        'device': _TABPFN_DEVICE,
        'version': 'v2',
    },
}

# Convenience accessors for code that instantiates models directly
# (e.g. cross-category refit, stacking auditor base_models dicts).
_RF_PARAMS  = PIPELINE_MODEL_CONFIGS['random_forest']
_XGB_PARAMS = PIPELINE_MODEL_CONFIGS['xgboost']
_CB_PARAMS  = PIPELINE_MODEL_CONFIGS['catboost']

# ---
# Target Display Name Resolution
# ---
def _resolve_target_name(raw_name):
    """Resolve raw target variable name to publication-ready display name.
    Uses TARGET_DISPLAY_NAMES from Cell 4 if available; falls back to raw name.
    Called at runtime (after this cell has executed)."""
    _tdn = globals().get('TARGET_DISPLAY_NAMES')
    return _tdn.get(raw_name, raw_name) if isinstance(_tdn, dict) else raw_name

# ---
# Ensemble Visualization Settings (Colab UI)
# ---
#@markdown ### Colour & Palette
#@markdown ---
#@markdown **ensemble_barplot_palette**: Colormap for FI bars. Set to *None* to use the single-color gradient below instead.
ensemble_barplot_palette = "rocket" #@param ["Purples_r", "Blues_r", "Reds_r", "YlOrBr_r", "Greens_r", "None", "YlOrRd", "YlGnBu", "RdYlBu", "RdYlGn", "coolwarm", "Spectral", "viridis", "plasma", "inferno", "magma", "cividis", "Set1", "Set2", "Set3", "Paired", "Dark2", "tab10", "tab20", "mako", "rocket", "Wistia", "cubehelix", "flare", "crest", "turbo", "rainbow"]
#@markdown **ensemble_barplot_color**: Single color → gradient (only active when palette = None; auto-set per-target in batch mode).
ensemble_barplot_color = "sienna" #@param ["crimson", "firebrick", "tomato", "darkorange", "orange", "gold", "yellow", "greenyellow", "limegreen", "forestgreen", "darkgreen", "teal", "darkcyan", "deepskyblue", "royalblue", "navy", "darkblue", "blueviolet", "purple", "mediumpurple", "orchid", "deeppink", "hotpink", "saddlebrown", "sienna", "peru", "black", "dimgray", "gray", "slategray"]
#@markdown **ensemble_heatmap_cmap**: Colormap for model-comparison heatmaps.
ensemble_heatmap_cmap = "Spectral" #@param ["YlOrRd", "coolwarm", "RdYlBu", "viridis", "plasma", "inferno", "magma", "Blues", "Reds", "Greens", "Spectral", "RdBu", "PiYG", "BrBG", "mako", "rocket", "flare", "crest"]

#@markdown ### Figure Settings
#@markdown ---
ensemble_top_n = 15 #@param {type: "integer"}
ensemble_ylabel_fontsize = 10 #@param {type: "integer"}

# -- Fixed defaults (removed from UI) ---
ensemble_fig_width  = 8
ensemble_fig_height = 6
ensemble_dpi        = 500

#@markdown ### Export
#@markdown ---
ensemble_save_plots = True #@param {"type": "boolean"}
ensemble_save_prefix = "ensemble" #@param {type: "string"}

def _resolve_ensemble_palette():
    """Resolve the effective palette/color for ensemble barplots.
    If a named palette is chosen (not 'None'), use it.
    Otherwise, build a dark-to-light gradient from the single color pick,
    so the top bar (rank 1) is the full colour and the bottom bar fades
    to a pale tint -- matching manuscript figure aesthetics.
    """
    if ensemble_barplot_palette != "None":
        return ensemble_barplot_palette
    # -- Gradient mode: dark → light tint of the target colour ---
    import matplotlib.colors as mcolors
    import numpy as np
    n = ENSEMBLE_VIZ_CONFIG.get('top_n', 10)
    try:
        base_rgb = np.array(mcolors.to_rgb(ensemble_barplot_color))
    except ValueError:
        base_rgb = np.array(mcolors.to_rgb('steelblue'))
    white = np.array([1.0, 1.0, 1.0])
    # Blend from 100% base (rank 1) → 30% base + 70% white (rank n)
    fracs = np.linspace(0.0, 0.92, n)
    return [mcolors.to_hex(base_rgb * (1 - f) + white * f) for f in fracs]

ENSEMBLE_VIZ_CONFIG = {
    'palette_fn':    _resolve_ensemble_palette,
    'heatmap_cmap':  ensemble_heatmap_cmap,
    'top_n':         ensemble_top_n,
    'fig_width':     ensemble_fig_width,
    'fig_height':    ensemble_fig_height,
    'ylabel_fs':     ensemble_ylabel_fontsize,
    'dpi':           ensemble_dpi,
    'save':          ensemble_save_plots,
    'save_prefix':   ensemble_save_prefix,
}
print(f"[Ensemble Viz] palette={ensemble_barplot_palette}, color={ensemble_barplot_color}, "
      f"heatmap={ensemble_heatmap_cmap}, top_n={ensemble_top_n}")

import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor, XGBClassifier
try:
    from catboost import CatBoostRegressor, CatBoostClassifier
except Exception:
    from catboost import CatBoostRegressor, CatBoostClassifier

from sklearn.base import BaseEstimator, RegressorMixin, ClassifierMixin
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from typing import List, Union, Dict, Any
from joblib import Parallel, delayed

# ---
# Helper functions
# ---

def adjusted_r2_score(y_true, y_pred, n_features):
    """Penalise R² for the number of predictors: adj-R² = 1 - (1 - R²)(n-1)/(n-p-1).

    """
    r2 = r2_score(y_true, y_pred)
    n_samples = len(y_true)
    adjusted_r2 = 1 - ((1 - r2) * (n_samples - 1) / (n_samples - n_features - 1))
    return adjusted_r2

def preprocess_boolean_target(y):
    """Coerce boolean, string-boolean, or two-valued numeric targets to {0, 1}.
    Required for learners that do not accept bool dtype labels.
    """
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]
    elif isinstance(y, pd.Series):
        y = y.copy()
    else:
        y = pd.Series(y)

    # Handle boolean values
    if y.dtype == bool:
        return y.astype(int)

    # Handle string 'True'/'False' values
    if y.dtype == object:
        try:
            return y.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)
        except Exception as e:
            print(f"[WARNING] {e}")
            pass

    # Handle numeric values that should be boolean
    unique_vals = np.unique(y)
    if len(unique_vals) == 2:
        # Map the smaller value to 0 and larger value to 1
        val_map = {val: idx for idx, val in enumerate(sorted(unique_vals))}
        return y.map(val_map).astype(int)

    return y

# Utility wrapper used when the target dtype is bool rather than int.
def modify_ensemble_for_boolean(model, X_train, y_train):
    """Inspect the target variable and force classification mode on an
    StackingEnsembleModel instance if the target is binary-coded boolean.
    """
    # Process the target variable
    y_processed = preprocess_boolean_target(y_train)

    # Check if we should force classification mode
    unique_vals = np.unique(y_processed)
    if len(unique_vals) == 2 and set(unique_vals) <= {0, 1}:
        model.task_type = 'classification'

        # Reinitialize models for classification
        if hasattr(model, '_init_classification_models'):
            model._init_classification_models()

    return model, y_processed

# ---

# ---

def _safe_clone(estimator):
    """Clone that handles CatBoost's numpy-typed class_weights."""
    try:
        return clone(estimator)
    except RuntimeError:
        params = {}
        for k, v in estimator.get_params().items():
            if isinstance(v, dict):
                # Convert numpy keys/values to native Python types
                params[k] = {int(dk) if hasattr(dk, 'item') else dk:
                             float(dv) if hasattr(dv, 'item') else dv
                             for dk, dv in v.items()}
            else:
                params[k] = v
        return estimator.__class__(**params)

class StackingEnsembleModel(BaseEstimator):
    # Trains base learners in parallel; uses their OOF predictions as inputs
    # to a linear meta-learner.
    def __init__(
        self,
        task_type: str = 'regression',
        model_configs: Dict[str, Dict[str, Any]] = None,
        meta_model=None,
        cv: int = 5,
        random_state: int = 42
    ):
        self.task_type = task_type
        self.model_configs = model_configs or {}
        self.cv = cv
        self.random_state = random_state
        self.base_models = {}
        self.meta_model = meta_model
        self.class_weights = None
        self.tabpfn_feature_reduction = None

        # Initialize base models
        if self.task_type == 'regression':
            self._init_regression_models()
            if self.meta_model is None:
                self.meta_model = LinearRegression()
        else:
            self._init_classification_models()
            if self.meta_model is None:
                # For classification, use LogisticRegression with random_state
                self.meta_model = LogisticRegression(random_state=self.random_state)

    def _init_regression_models(self):
        """Instantiate base regression learners.  TabPFN is opt-in: include
        'tabpfn' as a key in model_configs to enable it.
        """
        from sklearn.svm import SVR
        from sklearn.neural_network import MLPRegressor
        from sklearn.preprocessing import RobustScaler
        from sklearn.pipeline import Pipeline
        from tabpfn import TabPFNRegressor

        default_configs = {
            'catboost': {
                'iterations': 1000,
                'learning_rate': 0.05,
                'depth': 6,
                'l2_leaf_reg': 3,
                'border_count': 128,
                'thread_count': -1,
                'verbose': False,
            },
            'random_forest': {
                'n_estimators': 200,
                'max_depth': 10,
                'min_samples_split': 5,
                'min_samples_leaf': 2,
                'bootstrap': True
            },
            'xgboost': {
                'n_estimators': 500,
                'learning_rate': 0.1,
                'max_depth': 6,
                'min_child_weight': 1,
                'subsample': 0.8,
                'colsample_bytree': 0.8
            },
        }

        # TabPFN: opt-in only (include 'tabpfn' in model_configs to enable)
        if 'tabpfn' in self.model_configs:
            from tabpfn import TabPFNRegressor
            from tabpfn.constants import ModelVersion
            tabpfn_cfg = self.model_configs['tabpfn']
            self.base_models['tabpfn'] = TabPFNRegressor.create_default_for_version(
                ModelVersion.V2, device=tabpfn_cfg.get('device', _TABPFN_DEVICE)
            )

        # Merge default configs with user-provided configs
        for model_name in default_configs:
            config = {**default_configs[model_name], **self.model_configs.get(model_name, {})}
            config['random_state'] = self.random_state

            # Remove random_state from config if model doesn't support it
            if model_name == 'svm':
                config.pop('random_state', None)

            if model_name == 'catboost':
                self.base_models['catboost'] = CatBoostRegressor(**config)
            elif model_name == 'random_forest':
                self.base_models['random_forest'] = RandomForestRegressor(**config)
            elif model_name == 'xgboost':
                self.base_models['xgboost'] = XGBRegressor(**config)
            elif model_name == 'svm':
                self.base_models['svm'] = Pipeline([
                    ('scaler', RobustScaler()),
                    ('svr', SVR(**config))
                ])
            elif model_name == 'neural_net':
                self.base_models['neural_net'] = Pipeline([
                    ('scaler', RobustScaler()),
                    ('mlp', MLPRegressor(**config))
                ])

    def _init_classification_models(self):
        """Instantiate base classification learners including optional TabPFN."""
        from sklearn.svm import SVC
        from sklearn.neural_network import MLPClassifier
        from sklearn.preprocessing import RobustScaler
        from sklearn.pipeline import Pipeline
        from tabpfn import TabPFNClassifier

        default_configs = {
            'catboost': {
                'iterations': 1000,
                'learning_rate': 0.05,
                'depth': 6,
                'verbose': False,
                'class_weights': self.class_weights
            },
            'random_forest': {
                'n_estimators': 200,
                'max_depth': 10,
                'class_weight': self.class_weights
            },
            'xgboost': {
                'n_estimators': 500,
                'learning_rate': 0.1,
                'max_depth': 6,
                'min_child_weight': 1,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'scale_pos_weight': None
            },

        }

        # TabPFN: opt-in only (include 'tabpfn' in model_configs to enable)
        if 'tabpfn' in self.model_configs:
            from tabpfn import TabPFNClassifier
            from tabpfn.constants import ModelVersion
            tabpfn_cfg = self.model_configs['tabpfn']
            self.base_models['tabpfn'] = TabPFNClassifier.create_default_for_version(
                ModelVersion.V2, device=tabpfn_cfg.get('device', _TABPFN_DEVICE)
            )
        '''
            'svm': {
                'kernel': 'rbf',
                'C': 1.0,
                'probability': True,
                'class_weight': 'balanced'
            },'''

        # Merge default configs with user-provided configs
        for model_name in default_configs:
            config = {**default_configs[model_name], **self.model_configs.get(model_name, {})}
            config['random_state'] = self.random_state

            # Remove random_state from config if model doesn't support it
            if model_name == 'svm':
                config.pop('random_state', None)

            if model_name == 'catboost':
                self.base_models['catboost'] = CatBoostClassifier(**config)
            elif model_name == 'random_forest':
                self.base_models['random_forest'] = RandomForestClassifier(**config)
            elif model_name == 'xgboost':
                self.base_models['xgboost'] = XGBClassifier(**config)
            elif model_name == 'svm':
                self.base_models['svm'] = Pipeline([
                    ('scaler', RobustScaler()),
                    ('svc', SVC(**config))
                ])
            elif model_name == 'neural_net':
                self.base_models['neural_net'] = Pipeline([
                    ('scaler', RobustScaler()),
                    ('mlp', MLPClassifier(**config))
                ])

    # Set class weights for imbalanced classification targets
    def set_class_weights(self, y):
        """Set class weights for all base models and meta model"""
        if self.task_type == 'regression':
            return

        # Convert y to numpy array if it's a pandas Series/DataFrame
        if isinstance(y, (pd.Series, pd.DataFrame)):
            y = y.values.ravel()

        classes = np.unique(y)
        classes.sort()
        print("\nClass distribution in training data:")
        for c in classes:
            print(f"Class {c}: {np.sum(y == c)} samples")

        weights = compute_class_weight('balanced', classes=classes, y=y)
        self.class_weights = dict(zip(classes, weights))
        print("\nComputed class weights:")
        for c, w in self.class_weights.items():
            print(f"Class {c}: weight {w:.4f}")

        # Update and verify each model's class weights
        for name, model in self.base_models.items():
            if name == 'catboost':
                print(f"\nSetting class weights for {name}")
                model.set_params(class_weights=self.class_weights)
                # Verify weights were set
                params = model.get_params()
                print(f"CatBoost class weights after setting: {params.get('class_weights')}")

            elif name == 'random_forest':
                print(f"\nSetting class weights for {name}")
                model.set_params(class_weight=self.class_weights)
                # Verify weights were set
                params = model.get_params()
                print(f"RandomForest class weights after setting: {params.get('class_weight')}")

            elif name == 'xgboost':
                if len(classes) == 2:
                    print(f"\nSetting scale_pos_weight for {name}")
                    neg_count = np.sum(y == classes[0])
                    pos_count = np.sum(y == classes[1])
                    scale_weight = float(np.sqrt(neg_count / pos_count))
                    print(f"scale_pos_weight (sqrt): {scale_weight:.4f}  (full ratio would be {neg_count/pos_count:.4f})")
                    model.set_params(scale_pos_weight=scale_weight)
                    # Verify weight was set
                    params = model.get_params()
                    print(f"XGBoost scale_pos_weight after setting: {params.get('scale_pos_weight')}")

        # Meta-learner intentionally left WITHOUT class weights.
        # Base models already produce class-weight-adjusted OOF predictions;
        # weighting the meta-learner again would double-correct for imbalance.
        print("\nMeta model (LogisticRegression): no class weights (avoids double-weighting)")

    def _get_meta_features(self, X, y=None, is_train=True):
      """Generate out-of-fold meta-features for stacking ensemble.
      TabPFN: CatBoost-based feature selection (>100), fresh instance per fold,
      feature alignment on prediction.
      """
      if hasattr(X, 'get_features'):
          X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
      else:
          X_features = X

      if is_train:
          if self.task_type == 'classification' and y is not None:
              self.n_classes_ = len(np.unique(y))
          else:
              self.n_classes_ = 2
          meta_features = np.zeros((X_features.shape[0], len(self.base_models)))
          if self.task_type == 'classification':
              from sklearn.model_selection import StratifiedKFold
              kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)
          else:
              kf = KFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)

          # Feature selection for TabPFN (>100 features)
          tabpfn_features = None
          if X_features.shape[1] > 100 and 'tabpfn' in self.base_models and 'catboost' in self.base_models:
              print('\nSelecting top features for TabPFN using CatBoost importance...')
              if self.task_type == 'regression':
                  temp_cb = CatBoostRegressor(iterations=100, learning_rate=0.1, verbose=False, random_state=self.random_state)
              else:
                  temp_cb = CatBoostClassifier(iterations=100, learning_rate=0.1, verbose=False, random_state=self.random_state)
              temp_cb.fit(X_features, y)
              importance = pd.Series(temp_cb.feature_importances_, index=X_features.columns)
              selected_features = importance.nlargest(100).index
              tabpfn_features = X_features[selected_features]
              self.tabpfn_feature_reduction = selected_features
              print(f'Selected {len(selected_features)} top features for TabPFN')

          for i, (name, model) in enumerate(self.base_models.items()):
              print(f'\nGenerating meta-features for {name}...')
              temp_meta_features = np.zeros(X_features.shape[0])

              if name == 'tabpfn':
                  X_for_tabpfn = tabpfn_features if tabpfn_features is not None else X_features
                  for train_idx, val_idx in kf.split(X_for_tabpfn, y):
                      X_train_fold = X_for_tabpfn.iloc[train_idx]
                      y_train_fold = y.iloc[train_idx]
                      X_val_fold = X_for_tabpfn.iloc[val_idx]
                      from tabpfn.constants import ModelVersion
                      if self.task_type == 'regression':
                          from tabpfn import TabPFNRegressor
                          fold_model = TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                      else:
                          from tabpfn import TabPFNClassifier
                          fold_model = TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                      try:
                          fold_model.fit(X_train_fold, y_train_fold)
                          if self.task_type == 'classification':
                              fold_preds = fold_model.predict_proba(X_val_fold)[:, 1]
                          else:
                              fold_preds = fold_model.predict(X_val_fold)
                      except Exception as e:
                          print(f'TabPFN fold error: {e}, using fallback')
                          fold_preds = np.full(len(X_val_fold), np.mean(y_train_fold))
                      temp_meta_features[val_idx] = fold_preds
                  if tabpfn_features is not None:
                      model.selected_features_ = selected_features

              else:
                  for train_idx, val_idx in kf.split(X_features, y):
                      X_train_fold = X_features.iloc[train_idx]
                      y_train_fold = y.iloc[train_idx]
                      X_val_fold = X_features.iloc[val_idx]
                      model_clone = _safe_clone(model)
                      if self.task_type == 'classification':
                          if name == 'catboost':
                              model_clone.set_params(class_weights=self.class_weights)
                          elif name == 'random_forest':
                              model_clone.set_params(class_weight=self.class_weights)
                          elif name == 'xgboost' and len(np.unique(y)) == 2:
                              _y_spw = np.asarray(y).ravel()
                              _cls = np.unique(_y_spw)
                              model_clone.set_params(scale_pos_weight=float(np.sqrt(np.sum(_y_spw == _cls[0]) / np.sum(_y_spw == _cls[1]))))
                      model_clone.fit(X_train_fold, y_train_fold)
                      # Use predict_proba for classification (consistent scale with TabPFN)
                      if self.task_type == 'classification' and hasattr(model_clone, 'predict_proba'):
                          if getattr(self, 'n_classes_', 2) <= 2:
                              fold_preds = model_clone.predict_proba(X_val_fold)[:, 1]
                          else:
                              fold_preds = model_clone.predict(X_val_fold).astype(float)
                      else:
                          fold_preds = model_clone.predict(X_val_fold)
                      temp_meta_features[val_idx] = fold_preds

              meta_features[:, i] = temp_meta_features

              # Fit on full data
              if name == 'tabpfn':
                  X_for_tabpfn = tabpfn_features if tabpfn_features is not None else X_features
                  from tabpfn.constants import ModelVersion
                  if self.task_type == 'regression':
                      from tabpfn import TabPFNRegressor
                      self.base_models[name] = TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                  else:
                      from tabpfn import TabPFNClassifier
                      self.base_models[name] = TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                  _sw_full = None
                  if self.task_type == 'classification' and self.class_weights is not None:
                      _sw_full = np.array([float(self.class_weights.get(yi, 1.0)) for yi in y])
                  try:
                      try:
                          self.base_models[name].fit(X_for_tabpfn, y, sample_weight=_sw_full)
                      except TypeError:
                          self.base_models[name].fit(X_for_tabpfn, y)
                  except Exception as _tabpfn_err:
                      print(f"[WARNING] TabPFN full-data fit failed: {_tabpfn_err}. {'GPU OOM -- subsampling to ≤9999' if _TABPFN_DEVICE == 'cuda' else 'CPU limit -- subsampling to ≤999'}.")
                      _max_n = min(9999 if _TABPFN_DEVICE == 'cuda' else 999, len(X_for_tabpfn))
                      if _TABPFN_DEVICE == 'cuda':
                          import torch; torch.cuda.empty_cache()
                      _sub_idx = np.random.choice(len(X_for_tabpfn), _max_n, replace=False)
                      try:
                          self.base_models[name].fit(X_for_tabpfn.iloc[_sub_idx], y.iloc[_sub_idx])
                      except Exception as _e2:
                          print(f"[WARNING] TabPFN subsampled fit also failed: {_e2}. Predictions may be degraded.")
                  if tabpfn_features is not None:
                      self.base_models[name].selected_features_ = selected_features  # transfer to new instance
              else:
                  model.fit(X_features, y)

      else:
          meta_features = []
          for name, model in self.base_models.items():
              if name == 'tabpfn':
                  if hasattr(model, 'selected_features_'):
                      X_for_tabpfn = X_features[model.selected_features_]
                  else:
                      X_for_tabpfn = X_features
                  if hasattr(model, 'feature_names_in_'):
                      X_for_tabpfn = X_for_tabpfn.reindex(columns=list(model.feature_names_in_), fill_value=0)
                  try:
                      if self.task_type == 'classification':
                          if getattr(self, 'n_classes_', 2) <= 2:
                              preds = model.predict_proba(X_for_tabpfn)[:, 1]
                          else:
                              preds = model.predict(X_for_tabpfn).astype(float)
                      else:
                          preds = model.predict(X_for_tabpfn)
                  except Exception as e:
                      print(f'TabPFN prediction error: {e}, using fallback')
                      fallback = 0.5 if self.task_type == 'classification' else 0.0
                      preds = np.full(X_features.shape[0], fallback)
              else:
                  if self.task_type == 'classification' and hasattr(model, 'predict_proba'):
                      if getattr(self, 'n_classes_', 2) <= 2:
                          preds = model.predict_proba(X_features)[:, 1]
                      else:
                          preds = model.predict(X_features).astype(float)
                  else:
                      preds = model.predict(X_features)
              meta_features.append(preds)

          meta_features = np.column_stack(meta_features)

      return meta_features

    def explain_meta_model(self, X_train, y_train):

          import shap
          import numpy as np
          import networkx as nx
          import matplotlib.pyplot as plt
          from matplotlib.colors import LinearSegmentedColormap

          # Generate meta-features (base model predictions)
          meta_features = self._get_meta_features(X_train, y_train, is_train=False)

          # Get base model names for labels
          base_names = list(self.base_models.keys())

          print(f"Creating network visualization for meta-model: {type(self.meta_model).__name__}")

          # Calculate SHAP values
          try:
              background = shap.kmeans(meta_features, 10)
              explainer = shap.KernelExplainer(self.meta_model.predict, background)
              shap_values = explainer.shap_values(meta_features)

              # Convert to numpy array if needed
              if isinstance(shap_values, list):
                  shap_values = shap_values[0]  # Take first class for multi-class

              # Calculate feature importance (1st order effects)
              feature_importance = np.abs(shap_values).mean(axis=0)

              # Calculate pairwise interactions (2nd order effects)
              interactions = np.zeros((len(base_names), len(base_names)))

              # This is a simple approximation for interactions
              # Using shap.TreeExplainer.shap_interaction_values would be more accurate
              # but doesn't work for non-tree models
              for i in range(len(base_names)):
                  for j in range(i+1, len(base_names)):
                      # Calculate interaction strength through permutation analysis
                      interaction_strength = shap.utils.approximate_interactions(
                          i, j, shap_values, meta_features
                      )
                      interactions[i, j] = interaction_strength
                      interactions[j, i] = interaction_strength

              # Create network graph
              G = nx.Graph()

              # Add nodes (features)
              for i, name in enumerate(base_names):
                  G.add_node(name, size=feature_importance[i])

              # Add edges (interactions)
              for i in range(len(base_names)):
                  for j in range(i+1, len(base_names)):
                      if abs(interactions[i, j]) > 0.01:  # Threshold to reduce noise
                          G.add_edge(base_names[i], base_names[j],
                                    weight=abs(interactions[i, j]),
                                    color='red' if interactions[i, j] > 0 else 'blue')

              # Custom colormap similar to ShapIQ
              colors = [(0.0, 'darkblue'), (0.5, 'white'), (1.0, 'darkred')]
              custom_cmap = LinearSegmentedColormap.from_list('custom', colors)

              # Create plot
              plt.figure(figsize=(12, 10))

              # Compute position using spring layout
              pos = nx.spring_layout(G, k=1/np.sqrt(len(G.nodes())), seed=42)

              # Draw nodes with size based on importance
              node_sizes = [G.nodes[n]['size'] * 5000 + 100 for n in G.nodes()]
              nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightgray', alpha=0.8)

              # Draw edges with width and color based on interaction strength
              for u, v, data in G.edges(data=True):
                  nx.draw_networkx_edges(
                      G, pos, edgelist=[(u, v)], width=data['weight'] * 5,
                      edge_color=data['color'], alpha=min(0.8, data['weight'])
                  )

              # Draw labels
              nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')

              plt.title("Meta-model Feature Interaction Network", fontsize=16)
              plt.axis('off')
              plt.tight_layout()
              plt.show()

              # Create interaction matrix visualization
              plt.figure(figsize=(10, 8))
              mask = np.triu(np.ones_like(interactions, dtype=bool))  # Mask for upper triangle
              import seaborn as sns
              sns.heatmap(interactions, mask=mask, cmap=custom_cmap, center=0,
                          annot=True, fmt=".2f", linewidths=.5,
                          xticklabels=base_names, yticklabels=base_names)
              plt.title("Meta-model Pairwise Interaction Matrix", fontsize=16)
              plt.tight_layout()
              plt.show()

          except Exception as e:
              print(f"Error creating network visualization: {str(e)}")
              import traceback
              traceback.print_exc()

          return

    def fit(self, X, y, categorical_features=None):
        """Fit the stacking ensemble model"""
        # Convert Pool to features if needed
        if hasattr(X, 'get_features'):
            X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
        else:
            X_features = X

        # Set class weights before generating meta-features
        if self.task_type == 'classification':
            self.set_class_weights(y)

        # Generate meta-features for training
        meta_features = self._get_meta_features(X_features, y, is_train=True)

        # Fit the meta-model on the meta-features
        self.meta_model.fit(meta_features, y)

        return self

    def predict(self, X):
        # Convert Pool to features if needed
        if hasattr(X, 'get_features'):
            X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
        else:
            X_features = X

        # Generate meta-features for test data
        meta_features = self._get_meta_features(X_features, is_train=False)

        # Make final predictions using the meta-model
        predictions = self.meta_model.predict(meta_features)

        if self.task_type == 'classification':
            return np.round(predictions).astype(int)
        return predictions

    def predict_proba(self, X):
        if self.task_type != 'classification':
            raise ValueError("predict_proba is only available for classification tasks")

        # Convert Pool to features if needed
        if hasattr(X, 'get_features'):
            X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
        else:
            X_features = X

        # Generate meta-features for test data
        meta_features = self._get_meta_features(X_features, is_train=False)

        # Make probability predictions using the meta-model
        return self.meta_model.predict_proba(meta_features)

    def get_consensus_features(self, X_train, top_k=40):
        """Get consensus important features from tree-based models"""
        all_important_features = set()

        # Get feature importance from tree-based models
        for name, model in self.base_models.items():
            if name in ['catboost', 'random_forest', 'xgboost']:
                if hasattr(model, 'feature_importances_'):
                    # Get feature importance
                    importances = pd.Series(
                        model.feature_importances_,
                        index=X_train.columns
                    )
                    # Get top k features from this model
                    top_features = importances.nlargest(top_k).index
                    all_important_features.update(top_features)

        return list(all_important_features)

    def get_tabpfn_uncertainty(self, X):
        """Get uncertainty estimates from TabPFN"""
        if 'tabpfn' not in self.base_models:
            return None

        try:
            # Subset features if TabPFN trained on CatBoost-selected subset
            if hasattr(self.base_models['tabpfn'], 'selected_features_'):
                X = X[self.base_models['tabpfn'].selected_features_]
            if hasattr(self.base_models['tabpfn'], 'feature_names_in_'):
                X = X.reindex(columns=list(self.base_models['tabpfn'].feature_names_in_), fill_value=0)

            if self.task_type == 'classification':
                # Get probability predictions
                probas = self.base_models['tabpfn'].predict_proba(X)

                # Calculate confidence scores
                confidence = np.max(probas, axis=1)
                mean_confidence = np.mean(confidence)
                std_confidence = np.std(confidence)

                # Calculate entropy (uncertainty measure)
                entropy = -np.sum(probas * np.log2(probas + 1e-10), axis=1)
                mean_entropy = np.mean(entropy)

                uncertainty_stats = {
                    'mean_confidence': mean_confidence,
                    'std_confidence': std_confidence,
                    'confidence_interval': (mean_confidence - 1.96 * std_confidence,
                                        mean_confidence + 1.96 * std_confidence),
                    'mean_entropy': mean_entropy,
                    'min_confidence': np.min(confidence),
                    'max_confidence': np.max(confidence),
                    'uncertain_samples': np.sum(confidence < 0.7),  # samples with <70% confidence
                    'high_confidence_samples': np.sum(confidence > 0.9)  # samples with >90% confidence
                }

            else:  # regression
                # For regression, we can use the variation in predictions
                predictions = self.base_models['tabpfn'].predict(X)
                mean_pred = np.mean(predictions)
                std_pred = np.std(predictions)

                uncertainty_stats = {
                    'mean_prediction': mean_pred,
                    'std_prediction': std_pred,
                    'prediction_interval': (mean_pred - 1.96 * std_pred,
                                        mean_pred + 1.96 * std_pred)
                }

            return uncertainty_stats

        except Exception as e:
            print(f"Error calculating TabPFN uncertainty: {str(e)}")
            return None

    # TabPFN SHAP values
    def get_tabpfn_shap_values(self, model, X_test, feature_names=None, n_samples=50):
        """
        Get SHAP values specifically for TabPFN model with proper handling.
        """
        n_samples = min(n_samples, len(X_test))
        X_test_subset = X_test.iloc[:n_samples] if isinstance(X_test, pd.DataFrame) else X_test[:n_samples]

        if feature_names is None:
            feature_names = (X_test.columns if isinstance(X_test, pd.DataFrame)
                          else [f"feature_{i}" for i in range(X_test.shape[1])])

        def predict_function_for_shap(X):
            if hasattr(model, 'predict_proba'):
                return model.predict_proba(X)
            return model.predict(X)

        background = np.ones((1, X_test_subset.shape[1])) * float('nan')
        explainer = shap.Explainer(predict_function_for_shap, background)

        return explainer(X_test_subset)

    def plot_tabpfn_shap(self, shap_values, feature_names=None):
        """
        Create enhanced SHAP visualizations for TabPFN model.
        """
        if len(shap_values.shape) == 3:
            print("Computing SHAP values for the first class (index 0).")
            shap_values = shap_values[:, :, 0]

        plt.figure(figsize=(12, 8))
        shap.plots.bar(shap_values=shap_values, show=False)
        plt.title("Aggregate Feature Importances (TabPFN)")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values=shap_values, show=False)
        plt.title("Feature Importance Distribution (TabPFN)")
        plt.tight_layout()
        plt.show()

        most_important = np.abs(shap_values.values).mean(0).argsort()[-1]
        print(f'\nAnalyzing interactions for most important feature: "{feature_names[most_important]}"')

        if len(shap_values) > 1:
            inds = shap.utils.potential_interactions(
                shap_values[:, most_important],
                shap_values
            )

            for i in range(min(3, len(inds))):
                plt.figure(figsize=(10, 6))
                shap.plots.scatter(
                    shap_values[:, most_important],
                    color=shap_values[:, inds[i]],
                    show=False
                )
                plt.title(f"Interaction: {feature_names[most_important]} vs {feature_names[inds[i]]}")
                plt.tight_layout()
                plt.show()

    def explain_with_shap(self, X_train, X_test):
        """Enhanced SHAP analysis with special handling for TabPFN"""
        if hasattr(X_train, 'get_features'):
            X_train = pd.DataFrame(X_train.get_features(), columns=X_train.get_feature_names())
        if hasattr(X_test, 'get_features'):
            X_test = pd.DataFrame(X_test.get_features(), columns=X_test.get_feature_names())

        shap_values_dict = {}

        for name, model in self.base_models.items():
            print(f"\nGenerating SHAP values for base model: {name}...")

            try:
                if name == 'tabpfn':
                    print("\nPerforming specialized TabPFN SHAP analysis...")
                    shap_values = self.get_tabpfn_shap_values(
                        model,
                        X_test,
                        feature_names=X_test.columns,
                        n_samples=min(50, len(X_test))
                    )

                    shap_values_dict[name] = shap_values
                    self.plot_tabpfn_shap(shap_values, feature_names=X_test.columns)

                elif name in ['catboost', 'random_forest', 'xgboost']:
                    # Tree-based base learners: use TreeExplainer.
                    explainer = shap.TreeExplainer(model)
                    shap_values = explainer.shap_values(X_test)
                    shap_values_dict[name] = shap_values

                    plt.figure(figsize=(20, 16))
                    if isinstance(shap_values, list):
                        shap.summary_plot(
                            shap_values[1] if len(shap_values) > 1 else shap_values,
                            X_test,
                            plot_type="violin",
                            show=False,
                            max_display=ENSEMBLE_VIZ_CONFIG['top_n'],
                            title=f"SHAP Values - {name}"
                        )
                    else:
                        shap.summary_plot(
                            shap_values,
                            X_test,
                            plot_type="violin",
                            show=False,
                            max_display=ENSEMBLE_VIZ_CONFIG['top_n'],
                            title=f"SHAP Values - {name}"
                        )

                    plt.title(f"SHAP Feature Importance and Impact - {name}", fontsize=24)
                    plt.xlabel("SHAP value (impact on model output)", fontsize=18)
                    plt.ylabel("Features", fontsize=18)
                    plt.tick_params(axis='both', which='major', labelsize=14)

                    cbar = plt.gcf().axes[-1]
                    cbar.set_ylabel("Feature value", fontsize=18, labelpad=20)
                    cbar.tick_params(labelsize=14)

                    plt.tight_layout()
                    plt.show()

            except Exception as e:
                print(f"Could not generate SHAP values for {name}: {str(e)}")
                print(f"Error type: {type(e)}")
                print(f"Error details: {str(e)}")

        return shap_values_dict

    def get_feature_importance(self,  X_train, y_train):
        """Enhanced get_feature_importance with consensus features for SVM"""
        importance_dict = {}

        # Handle TabPFN separately with non-parallel permutation importance
        if 'tabpfn' in self.base_models:
            print("\nCalculating TabPFN feature importance...")
            try:
              ''' # Calculate feature importance without parallelization
                n_repeats = 2  # Reduced number of repeats for speed
                importances = []
                baseline_score = self.base_models['tabpfn'].score(X_train, y_train)

                for feature_idx in range(X_train.shape[1]):
                    X_permuted = X_train.copy()
                    X_permuted.iloc[:, feature_idx] = np.random.permutation(X_permuted.iloc[:, feature_idx])
                    permuted_score = self.base_models['tabpfn'].score(X_permuted, y_train)
                    importance = baseline_score - permuted_score
                    importances.append(importance)

                importance_dict['tabpfn'] = np.array(importances)
                print("Successfully calculated TabPFN importance")'''

              # Get uncertainty estimates
              uncertainty_stats = self.get_tabpfn_uncertainty(X_train)
              if uncertainty_stats:
                  print("\nTabPFN Uncertainty Statistics:")
                  for metric, value in uncertainty_stats.items():
                      if isinstance(value, tuple):
                          print(f"{metric}: ({value[0]:.4f}, {value[1]:.4f})")
                      else:
                          print(f"{metric}: {value:.4f}")

            except Exception as e:
                print(f"Could not calculate TabPFN importance: {str(e)}")

        # First get importance from tree-based models
        for name, model in self.base_models.items():
            if name in ['catboost', 'random_forest', 'xgboost']:
                if hasattr(model, 'feature_importances_'):
                    importance_dict[name] = model.feature_importances_
            elif name == 'elasticnet':
                if hasattr(model, 'coef_'):
                    importance_dict[name] = np.abs(model.coef_)

        # Get consensus features for SVM
        if 'svm' in self.base_models:
            try:
                from sklearn.inspection import permutation_importance

                # Get important features from tree models
                important_features = self.get_consensus_features(X_train)
                print(f"Number of consensus features for SVM: {len(important_features)}")

               # Before subsetting
                print("NaN values in X_train:", X_train.isnull().sum().sum())

                # After subsetting
                X_subset = X_train[important_features]
                print("NaN values in X_subset:", X_subset.isnull().sum().sum())

                # Check column continuity
                print("X_subset columns:", X_subset.columns)
                print("Important features:", important_features)

                # Handle NaN values before scaling if they exist
                if X_subset.isnull().any().any():  # Check if any NaN exists
                    print(f"Found NaN values in X_train subset crunch: {X_subset.isnull().sum().sum()}")
                    imputer = SimpleImputer(strategy='median')
                    X_subset = pd.DataFrame(
                        imputer.fit_transform(X_subset),
                        columns=X_subset.columns,
                        index=X_subset.index
                    )
                    print(f"Imputed {X_subset.isnull().sum().sum()} NaN values")
                else:
                    print("No NaN values found for X_train subset")
                # Get the model and scaler from pipeline
                model = self.base_models['svm']
                if hasattr(model, 'named_steps'):
                    scaler = model.named_steps['scaler']
                    X_scaled = scaler.transform(X_train_subset)
                    actual_model = model.named_steps['svr' if self.task_type == 'regression' else 'svc']

                    # Check if linear kernel for faster computation
                    if hasattr(actual_model, 'kernel') and actual_model.kernel == 'linear':
                        print("Using linear kernel coefficients for SVM importance")
                        # Create full-size importance array (all zeros except important features)
                        full_importance = np.zeros(X_train.shape[1])
                        for idx, feat in enumerate(important_features):
                            full_importance[X_train.columns.get_loc(feat)] = np.abs(actual_model.coef_[0][idx])
                        importance_dict['svm'] = full_importance
                    else:
                        print("Using permutation importance for RBF kernel SVM")
                        # Calculate permutation importance only for important features
                        result = permutation_importance(
                            actual_model, X_scaled, y_train,
                            n_repeats=10,
                            random_state=self.random_state,
                            n_jobs=-1
                        )

                        # Map back to full feature space
                        full_importance = np.zeros(X_train.shape[1])
                        for idx, feat in enumerate(important_features):
                            full_importance[X_train.columns.get_loc(feat)] = result.importances_mean[idx]
                        importance_dict['svm'] = full_importance

            except Exception as e:
                print(f"Could not calculate importance for SVM: {str(e)}")

        # Calculate permutation importance once for TabPFN
        if 'tabpfn' in self.base_models:
            print("\nCalculating TabPFN feature importance using permutation importance...")
            try:
                # Subset features to match what TabPFN was trained on
                X_for_perm = X_train
                if hasattr(self.base_models['tabpfn'], 'selected_features_'):
                    X_for_perm = X_train[self.base_models['tabpfn'].selected_features_]
                if hasattr(self.base_models['tabpfn'], 'feature_names_in_'):
                    X_for_perm = X_for_perm.reindex(columns=list(self.base_models['tabpfn'].feature_names_in_), fill_value=0)
                result = permutation_importance(
                    self.base_models['tabpfn'], X_for_perm, y_train,
                    n_repeats=10,
                    random_state=42,
                    n_jobs=-1
                )
                importance_dict['tabpfn'] = result.importances_mean
                print("Successfully calculated TabPFN importance")
            except Exception as e:
                print(f"Could not calculate TabPFN importance: {str(e)}")

        return importance_dict

    def evaluate_models(self, X_test, y_test, target_options=None, tp=None, category=None):
        """Evaluate all models including TabPFN"""
        metrics_dict = {}

        # Evaluate each base model
        for name, model in self.base_models.items():
            # TabPFN may use a feature subset (>100 cols → CatBoost selection)
            if name == 'tabpfn':
                X_eval = X_test
                if hasattr(model, 'selected_features_'):
                    X_eval = X_test[model.selected_features_]
                if hasattr(model, 'feature_names_in_'):
                    X_eval = X_eval.reindex(columns=list(model.feature_names_in_), fill_value=0)
                y_pred = model.predict(X_eval)
                if self.task_type == 'classification':
                    y_proba = model.predict_proba(X_eval)
            else:
                y_pred = model.predict(X_test)

            if self.task_type == 'regression':
                n_features = X_test.shape[1]  # Default to all features

                metrics = {
                    'mse': mean_squared_error(y_test, y_pred),
                    'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                    'mae': mean_absolute_error(y_test, y_pred),
                    'r2': r2_score(y_test, y_pred),
                    'adj_r2': adjusted_r2_score(y_test, y_pred, n_features)  # Add adjusted R²

                }
            else:
                metrics = {
                    'accuracy': accuracy_score(y_test, y_pred),
                    'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
                    'recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
                    'f1': f1_score(y_test, y_pred, average='macro', zero_division=0)
                }

                if len(np.unique(y_test)) == 2:
                    for avg in ['macro', 'weighted']:
                        metrics[f'precision_{avg}'] = precision_score(y_test, y_pred, average=avg, zero_division=0)
                        metrics[f'recall_{avg}'] = recall_score(y_test, y_pred, average=avg, zero_division=0)
                        metrics[f'f1_{avg}'] = f1_score(y_test, y_pred, average=avg, zero_division=0)

                    # Per-class metrics (fixes NaN for base models)
                    for cls_label in np.unique(y_test):
                        _cr = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
                        _k = cls_label if cls_label in _cr else str(cls_label)
                        if _k in _cr:
                            metrics[f'precision_class_{cls_label}'] = _cr[_k]['precision']
                            metrics[f'recall_class_{cls_label}'] = _cr[_k]['recall']
                            metrics[f'f1_class_{cls_label}'] = _cr[_k]['f1-score']

                    # AUC + AUC-PR for binary classification
                    _proba_base = None
                    if name == 'tabpfn' and self.task_type == 'classification':
                        _proba_base = y_proba[:, 1]
                    elif hasattr(model, 'predict_proba'):
                        try:
                            if name == 'tabpfn':
                                _proba_base = model.predict_proba(X_eval)[:, 1]
                            else:
                                _proba_base = model.predict_proba(X_test)[:, 1]
                        except Exception as e:
                            print(f"Could not get predict_proba for {name}: {e}")
                    if _proba_base is not None:
                        try:
                            metrics['auc'] = roc_auc_score(y_test, _proba_base)
                            _pr, _rc, _ = precision_recall_curve(y_test, _proba_base)
                            metrics['auc_pr'] = sk_auc(_rc, _pr)
                        except Exception as e:
                            print(f"Could not calculate AUC for {name}: {e}")

        metrics_dict[name] = metrics

        # Evaluate meta-model
        y_pred_meta = self.predict(X_test)
        if self.task_type == 'regression':
            metrics_dict['meta_model'] = {
                'mse': mean_squared_error(y_test, y_pred_meta),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred_meta)),
                'mae': mean_absolute_error(y_test, y_pred_meta),
                'r2': r2_score(y_test, y_pred_meta),
                'adj_r2': adjusted_r2_score(y_test, y_pred_meta, X_test.shape[1])  # Add adjusted R²

            }
        else:
            metrics_dict['meta_model'] = {
                'accuracy': accuracy_score(y_test, y_pred_meta),
                'precision': precision_score(y_test, y_pred_meta, average='macro', zero_division=0),
                'recall': recall_score(y_test, y_pred_meta, average='macro', zero_division=0),
                'f1': f1_score(y_test, y_pred_meta, average='macro', zero_division=0)
            }

            if len(np.unique(y_test)) == 2:
                # Add per-class metrics for meta-model
                for avg in ['macro', 'weighted']:
                    metrics_dict['meta_model'][f'precision_{avg}'] = precision_score(y_test, y_pred_meta, average=avg, zero_division=0)
                    metrics_dict['meta_model'][f'recall_{avg}'] = recall_score(y_test, y_pred_meta, average=avg, zero_division=0)
                    metrics_dict['meta_model'][f'f1_{avg}'] = f1_score(y_test, y_pred_meta, average=avg, zero_division=0)

                # Add per-class metrics
                class_report = classification_report(y_test, y_pred_meta, output_dict=True)
                for class_label in np.unique(y_test):
                    _key = class_label if class_label in class_report else str(class_label)
                    if _key in class_report:
                        metrics_dict['meta_model'][f'precision_class_{class_label}'] = class_report[_key]['precision']
                        metrics_dict['meta_model'][f'recall_class_{class_label}'] = class_report[_key]['recall']
                        metrics_dict['meta_model'][f'f1_class_{class_label}'] = class_report[_key]['f1-score']

                try:
                    proba = self.predict_proba(X_test)[:, 1]
                    metrics_dict['meta_model']['auc'] = roc_auc_score(y_test, proba)
                except Exception as e:
                    print("Could not calculate AUC for meta-model")

        # Create comparison visualizations with more detailed metrics
        self._plot_model_comparisons(metrics_dict, target_options, tp, category)

        return metrics_dict

    # TabPFN evaluation branch
    def evaluate_models(self, X_test, y_test, target_options=None, tp=None, category=None):
        """Evaluate all models including TabPFN"""
        from sklearn.metrics import precision_recall_curve, auc as sk_auc
        metrics_dict = {}

        # Evaluate each base model
        for name, model in self.base_models.items():


            # TabPFN may use a feature subset (>100 cols → CatBoost selection)
            if name == 'tabpfn':
                X_eval = X_test
                if hasattr(model, 'selected_features_'):
                    X_eval = X_test[model.selected_features_]
                if hasattr(model, 'feature_names_in_'):
                    X_eval = X_eval.reindex(columns=list(model.feature_names_in_), fill_value=0)
                y_pred = model.predict(X_eval)
                if self.task_type == 'classification':
                    y_proba = model.predict_proba(X_eval)
            else:
                y_pred = model.predict(X_test)
                print(f"Model: {name}, successfully fit to y_pred predict")

            if self.task_type == 'regression':
                n_features = X_test.shape[1]  # Default to all features

                metrics = {
                    'mse': mean_squared_error(y_test, y_pred),
                    'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                    'mae': mean_absolute_error(y_test, y_pred),
                    'r2': r2_score(y_test, y_pred),
                    'adj_r2': adjusted_r2_score(y_test, y_pred, n_features)  # Add adjusted R²

                }
            else:
                metrics = {
                    'accuracy': accuracy_score(y_test, y_pred),
                    'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
                    'recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
                    'f1': f1_score(y_test, y_pred, average='macro', zero_division=0)
                }

                if len(np.unique(y_test)) == 2:
                    for avg in ['macro', 'weighted']:
                        metrics[f'precision_{avg}'] = precision_score(y_test, y_pred, average=avg, zero_division=0)
                        metrics[f'recall_{avg}'] = recall_score(y_test, y_pred, average=avg, zero_division=0)
                        metrics[f'f1_{avg}'] = f1_score(y_test, y_pred, average=avg, zero_division=0)

                    # Per-class metrics (fixes NaN for base models)
                    for cls_label in np.unique(y_test):
                        _cr = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
                        _k = cls_label if cls_label in _cr else str(cls_label)
                        if _k in _cr:
                            metrics[f'precision_class_{cls_label}'] = _cr[_k]['precision']
                            metrics[f'recall_class_{cls_label}'] = _cr[_k]['recall']
                            metrics[f'f1_class_{cls_label}'] = _cr[_k]['f1-score']

                    # AUC + AUC-PR for binary classification
                    _proba_base = None
                    if name == 'tabpfn' and self.task_type == 'classification':
                        _proba_base = y_proba[:, 1]
                    elif hasattr(model, 'predict_proba'):
                        try:
                            if name == 'tabpfn':
                                _proba_base = model.predict_proba(X_eval)[:, 1]
                            else:
                                _proba_base = model.predict_proba(X_test)[:, 1]
                        except Exception as e:
                            print(f"Could not get predict_proba for {name}: {e}")
                    if _proba_base is not None:
                        try:
                            metrics['auc'] = roc_auc_score(y_test, _proba_base)
                            _pr, _rc, _ = precision_recall_curve(y_test, _proba_base)
                            metrics['auc_pr'] = sk_auc(_rc, _pr)
                        except Exception as e:
                            print(f"Could not calculate AUC for {name}: {e}")
                else:
                    try:
                        if hasattr(model, 'predict_proba'):
                            proba_mc = model.predict_proba(X_test)
                            metrics['auc_ovr'] = roc_auc_score(y_test, proba_mc, multi_class='ovr', average='macro')
                    except Exception as e:
                        print(f"Could not calculate multi-class AUC for {name}: {e}")

            metrics_dict[name] = metrics

        # Evaluate meta-model
        y_pred_meta = self.predict(X_test)
        if self.task_type == 'regression':
            metrics_dict['meta_model'] = {
                'mse': mean_squared_error(y_test, y_pred_meta),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred_meta)),
                'mae': mean_absolute_error(y_test, y_pred_meta),
                'r2': r2_score(y_test, y_pred_meta),
                'adj_r2': adjusted_r2_score(y_test, y_pred_meta, X_test.shape[1])  # Add adjusted R²

            }
        else:
            metrics_dict['meta_model'] = {
                'accuracy': accuracy_score(y_test, y_pred_meta),
                'precision': precision_score(y_test, y_pred_meta, average='macro', zero_division=0),
                'recall': recall_score(y_test, y_pred_meta, average='macro', zero_division=0),
                'f1': f1_score(y_test, y_pred_meta, average='macro', zero_division=0)
            }

            if len(np.unique(y_test)) == 2:
                # Add per-class metrics for meta-model
                for avg in ['macro', 'weighted']:
                    metrics_dict['meta_model'][f'precision_{avg}'] = precision_score(y_test, y_pred_meta, average=avg, zero_division=0)
                    metrics_dict['meta_model'][f'recall_{avg}'] = recall_score(y_test, y_pred_meta, average=avg, zero_division=0)
                    metrics_dict['meta_model'][f'f1_{avg}'] = f1_score(y_test, y_pred_meta, average=avg, zero_division=0)

                # Add per-class metrics
                class_report = classification_report(y_test, y_pred_meta, output_dict=True)
                for class_label in np.unique(y_test):
                    _key = class_label if class_label in class_report else str(class_label)
                    if _key in class_report:
                        metrics_dict['meta_model'][f'precision_class_{class_label}'] = class_report[_key]['precision']
                        metrics_dict['meta_model'][f'recall_class_{class_label}'] = class_report[_key]['recall']
                        metrics_dict['meta_model'][f'f1_class_{class_label}'] = class_report[_key]['f1-score']

                try:
                    print("Fallback method to do AUC for meta model...")
                    # First check if predict_proba exists and works properly
                    if hasattr(self, 'predict_proba'):
                        proba = self.predict_proba(X_test)
                        if proba is not None and proba.shape[1] >= 2:
                            # Calculate ROC-AUC
                            metrics_dict['meta_model']['auc'] = roc_auc_score(y_test, proba[:, 1])

                            # Calculate PR-AUC (Precision-Recall AUC)
                            from sklearn.metrics import precision_recall_curve, auc
                            precision, recall, _ = precision_recall_curve(y_test, proba[:, 1])
                            metrics_dict['meta_model']['auc_pr'] = auc(recall, precision)
                        else:
                            # Fallback: try to get probabilities directly from meta_model
                            meta_features = self._get_meta_features(X_test, is_train=False)
                            if hasattr(self.meta_model, 'predict_proba'):
                                meta_proba = self.meta_model.predict_proba(meta_features)

                                # Calculate ROC-AUC
                                metrics_dict['meta_model']['auc'] = roc_auc_score(y_test, meta_proba[:, 1])

                                # Calculate PR-AUC
                                from sklearn.metrics import precision_recall_curve
                                precision, recall, _ = precision_recall_curve(y_test, meta_proba[:, 1])
                                metrics_dict['meta_model']['auc_pr'] = auc(recall, precision)
                    else:
                        print("predict_proba not available for meta-model")
                except Exception as e:
                    print(f"Could not calculate AUC metrics for meta-model: {str(e)}")

                # -- THRESHOLD OPTIMIZATION ---
                # Find optimal threshold from precision-recall curve
                try:
                    _meta_proba = self.predict_proba(X_test)[:, 1]
                    _pr_curve, _rc_curve, _thresholds = precision_recall_curve(y_test, _meta_proba)
                    # F1 at each threshold
                    _f1_curve = 2 * (_pr_curve[:-1] * _rc_curve[:-1]) / (_pr_curve[:-1] + _rc_curve[:-1] + 1e-10)
                    _best_idx = np.argmax(_f1_curve)
                    _opt_thresh = float(_thresholds[_best_idx])
                    _y_pred_opt = (_meta_proba >= _opt_thresh).astype(int)

                    metrics_dict['meta_model']['optimal_threshold'] = _opt_thresh
                    metrics_dict['meta_model']['recall_class_1_at_opt'] = recall_score(y_test, _y_pred_opt, pos_label=1, zero_division=0)
                    metrics_dict['meta_model']['precision_class_1_at_opt'] = precision_score(y_test, _y_pred_opt, pos_label=1, zero_division=0)
                    metrics_dict['meta_model']['f1_class_1_at_opt'] = f1_score(y_test, _y_pred_opt, pos_label=1, zero_division=0)
                    metrics_dict['meta_model']['accuracy_at_opt'] = accuracy_score(y_test, _y_pred_opt)

                    # Store arrays for plotting (attach to self for _plot_model_comparisons)
                    self._threshold_results = {
                        'proba': _meta_proba,
                        'pr_curve': _pr_curve, 'rc_curve': _rc_curve,
                        'thresholds': _thresholds, 'f1_curve': _f1_curve,
                        'opt_threshold': _opt_thresh, 'opt_idx': _best_idx,
                        'y_pred_default': y_pred_meta, 'y_pred_opt': _y_pred_opt,
                        'y_test': y_test
                    }
                    print(f"\nOptimal threshold: {_opt_thresh:.4f}  "
                          f"(Recall@opt: {metrics_dict['meta_model']['recall_class_1_at_opt']:.3f}, "
                          f"Precision@opt: {metrics_dict['meta_model']['precision_class_1_at_opt']:.3f}, "
                          f"F1@opt: {metrics_dict['meta_model']['f1_class_1_at_opt']:.3f})")
                except Exception as e:
                    print(f"Could not compute optimal threshold: {e}")
                    self._threshold_results = None

                if 'auc' not in metrics_dict.get('meta_model', {}):
                  try:
                    print("Calculating AUC metrics directly from predictions... (Alternative method #2)")
                    # Since we're having dimensionality issues with the meta-model,
                    # Direct approach using the predicted classes

                    # For binary classification, convert predictions to probability-like scores
                    if len(np.unique(y_test)) == 2:
                        # Method 1: Use decision function if available (for LogisticRegression)
                        if hasattr(self.meta_model, 'decision_function'):
                            try:
                                # Generate meta-features properly
                                meta_features = self._get_meta_features(X_test, is_train=False)

                                # If there's a dimension mismatch, use simpler approach
                                if meta_features.shape[1] != self.meta_model.coef_.shape[1]:
                                    raise ValueError("Dimension mismatch")

                                decision_scores = self.meta_model.decision_function(meta_features)
                                metrics_dict['meta_model']['auc'] = roc_auc_score(y_test, decision_scores)

                                # Calculate PR-AUC
                                from sklearn.metrics import precision_recall_curve, auc
                                precision, recall, _ = precision_recall_curve(y_test, decision_scores)
                                metrics_dict['meta_model']['auc_pr'] = auc(recall, precision)
                                print("Successfully calculated AUC using decision_function")
                            except Exception as e:
                                print(f"[WARNING] {e}")
                                # Method 2: Use a simpler approach - binary prediction as score
                                # This isn't ideal but can serve as a fallback
                                metrics_dict['meta_model']['auc'] = roc_auc_score(y_test, y_pred_meta)

                                # PR-AUC
                                precision, recall, _ = precision_recall_curve(y_test, y_pred_meta)
                                metrics_dict['meta_model']['auc_pr'] = auc(recall, precision)
                                print("Used binary predictions as scores for AUC calculation")
                        else:
                            # For other models without decision_function
                            metrics_dict['meta_model']['auc'] = roc_auc_score(y_test, y_pred_meta)

                            # PR-AUC
                            precision, recall, _ = precision_recall_curve(y_test, y_pred_meta)
                            metrics_dict['meta_model']['auc_pr'] = auc(recall, precision)
                            print("Used binary predictions as scores for AUC calculation")
                  except Exception as e:
                    print(f"Could not calculate AUC metrics for meta-model: {str(e)}")
                    # Don't add the metrics if calculation fails

        # Create comparison visualizations with more detailed metrics
        self._plot_model_comparisons(metrics_dict, target_options, tp, category)

        return metrics_dict

    def _plot_model_comparisons(self, metrics_dict, target_options=None, tp=None, category=None):
        """Enhanced model comparison with clinical focus for classification."""
        from sklearn.metrics import confusion_matrix as _cm_func
        from IPython.display import display, HTML

        _subtitle = f"{_resolve_target_name(target_options) if target_options else ''} | TP {tp or ''}" + (f" | {category}" if category else "")
        is_clf = self.task_type == 'classification'
        models = list(metrics_dict.keys())

        # -- 1a. KEY METRICS BAR CHART (r² / Adj-r² excluded for regression)
        if is_clf:
            key_metrics     = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'auc_pr']
            r2_metrics      = []
        else:
            key_metrics     = ['mse', 'rmse', 'mae']
            r2_metrics      = ['r2', 'adj_r2']

        comp = []
        for m in models:
            for k in key_metrics:
                if k in metrics_dict[m] and metrics_dict[m][k] is not None and not (isinstance(metrics_dict[m][k], float) and np.isnan(metrics_dict[m][k])):
                    comp.append({'Model': m, 'Metric': k.upper().replace('_', '-'), 'Value': metrics_dict[m][k]})

        # -- Regression: MSE/RMSE/MAE + R²/Adj-R² side by side ---
        if not is_clf and r2_metrics:
            comp_r2 = []
            for m in models:
                for k in r2_metrics:
                    v = metrics_dict[m].get(k)
                    if v is not None and not (isinstance(v, float) and np.isnan(v)):
                        comp_r2.append({'Model': m, 'Metric': k.upper().replace('_', '-'), 'Value': v})

            if comp and comp_r2:
                cdf    = pd.DataFrame(comp)
                cdf_r2 = pd.DataFrame(comp_r2)
                fig, (ax_err, ax_r2) = plt.subplots(
                    1, 2, figsize=(max(14, len(models) * 3.5), 5))

                sns.barplot(data=cdf, x='Metric', y='Value', hue='Model',
                            ax=ax_err, palette='Set2')
                ax_err.set_title('MSE / RMSE / MAE', fontsize=13, fontweight='bold')
                ax_err.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

                sns.barplot(data=cdf_r2, x='Metric', y='Value', hue='Model',
                            ax=ax_r2, palette='Set2')
                ax_r2.set_title('R\u00b2 / Adjusted R\u00b2', fontsize=13, fontweight='bold')
                ax_r2.axhline(0, color='#90A4AE', lw=0.8, linestyle='--')
                ax_r2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

                fig.suptitle(f"Model Comparison -- {_subtitle}",
                             fontsize=14, fontweight='bold', y=1.02)
                plt.tight_layout()
                plt.show()

            elif comp:
                # Fallback: only error metrics available
                cdf = pd.DataFrame(comp)
                fig, ax = plt.subplots(figsize=(max(8, len(models)*2), 5))
                sns.barplot(data=cdf, x='Metric', y='Value', hue='Model',
                            ax=ax, palette='Set2')
                ax.set_title(f"Model Comparison -- {_subtitle}",
                             fontsize=14, fontweight='bold')
                ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
                plt.tight_layout()
                plt.show()

        # -- Classification: single figure (unchanged) ---
        elif comp:
            cdf = pd.DataFrame(comp)
            fig, ax = plt.subplots(figsize=(max(8, len(models)*2), 5))
            sns.barplot(data=cdf, x='Metric', y='Value', hue='Model',
                        ax=ax, palette='Set2')
            ax.set_title(f"Model Comparison -- {_subtitle}",
                         fontsize=14, fontweight='bold')
            ax.set_ylim(0, 1.05)
            import matplotlib.ticker as mticker
            ax.yaxis.set_major_locator(mticker.MultipleLocator(0.10))
            ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
            plt.tight_layout()
            plt.show()

        # -- OVERVIEW TABLE ---
        overview_cols = ['accuracy', 'auc', 'auc_pr', 'precision', 'recall', 'f1']
        if is_clf:
            overview = pd.DataFrame({m: {k: metrics_dict[m].get(k) for k in overview_cols} for m in models}).T
            overview.columns = ['Accuracy', 'AUC-ROC', 'AUC-PR', 'Precision(macro)', 'Recall(macro)', 'F1(macro)']
            print("\n" + "="*70)
            print("  OVERVIEW -- Macro-averaged Metrics")
            print("="*70)
            try:
                display(overview.style.format('{:.4f}', na_rep='--')
                    .background_gradient(cmap='YlGn', subset=['AUC-ROC', 'AUC-PR'], vmin=0, vmax=1)
                    .highlight_max(axis=0, color='#c6efce'))
            except Exception:
                display(overview)

        # -- PER-CLASS TABLE (clinical focus) ---
        if is_clf and len([k for k in metrics_dict[models[0]] if 'class_1' in k]) > 0:
            class_cols = ['precision_class_0', 'recall_class_0', 'f1_class_0',
                          'precision_class_1', 'recall_class_1', 'f1_class_1']
            class_data = {}
            for m in models:
                class_data[m] = {c: metrics_dict[m].get(c) for c in class_cols}
            class_df = pd.DataFrame(class_data).T
            class_df.columns = ['Prec(0)', 'Recall(0)', 'F1(0)', 'Prec(1)', 'Recall(1)', 'F1(1)']
            print("\n" + "="*70)
            print("  PER-CLASS PERFORMANCE  (Class 1 = target/positive)")
            print("="*70)
            try:
                display(class_df.style.format('{:.4f}', na_rep='--')
                    .background_gradient(cmap='YlOrRd', subset=['Recall(1)', 'F1(1)'], vmin=0, vmax=1))
            except Exception:
                display(class_df)

        # -- THRESHOLD OPTIMIZATION RESULTS ---
        tr = getattr(self, '_threshold_results', None)
        if tr is not None and is_clf:
            print("\n" + "="*70)
            print("  THRESHOLD OPTIMIZATION  (Meta-model)")
            print("="*70)
            _mm = metrics_dict.get('meta_model', {})
            thresh_compare = pd.DataFrame({
                'Default (0.50)': {
                    'Threshold': 0.50,
                    'Accuracy': _mm.get('accuracy'),
                    'Precision(1)': _mm.get('precision_class_1'),
                    'Recall(1)': _mm.get('recall_class_1'),
                    'F1(1)': _mm.get('f1_class_1'),
                },
                f"Optimal ({tr['opt_threshold']:.3f})": {
                    'Threshold': tr['opt_threshold'],
                    'Accuracy': _mm.get('accuracy_at_opt'),
                    'Precision(1)': _mm.get('precision_class_1_at_opt'),
                    'Recall(1)': _mm.get('recall_class_1_at_opt'),
                    'F1(1)': _mm.get('f1_class_1_at_opt'),
                }
            }).T
            try:
                display(thresh_compare.style.format('{:.4f}', na_rep='--')
                    .highlight_max(axis=0, subset=['Recall(1)', 'F1(1)'], color='#c6efce'))
            except Exception:
                display(thresh_compare)

            # -- PR CURVE + CONFUSION MATRICES ---
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))

            # PR Curve
            ax = axes[0]
            ax.plot(tr['rc_curve'], tr['pr_curve'], color='#2196F3', lw=2,
                    label=f"AUC-PR = {_mm.get('auc_pr', 0):.3f}")
            ax.scatter(tr['rc_curve'][tr['opt_idx']], tr['pr_curve'][tr['opt_idx']],
                       color='red', s=120, zorder=5, label=f"Optimal (t={tr['opt_threshold']:.3f})")
            base_rate = np.mean(tr['y_test'])
            ax.axhline(y=base_rate, color='gray', ls='--', alpha=0.5, label=f"Base rate ({base_rate:.3f})")
            ax.set_xlabel('Recall (Sensitivity)', fontsize=11)
            ax.set_ylabel('Precision (PPV)', fontsize=11)
            ax.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
            ax.legend(fontsize=9)
            ax.set_xlim(-0.02, 1.02)
            ax.set_ylim(-0.02, 1.02)
            ax.grid(True, alpha=0.3)

            # Confusion Matrix -- Default threshold
            ax = axes[1]
            cm_def = _cm_func(tr['y_test'], tr['y_pred_default'])
            sns.heatmap(cm_def, annot=True, fmt='d', cmap='Blues', ax=ax,
                        xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
            ax.set_title('Confusion Matrix\n(threshold = 0.50)', fontsize=13, fontweight='bold')

            # Confusion Matrix -- Optimal threshold
            ax = axes[2]
            cm_opt = _cm_func(tr['y_test'], tr['y_pred_opt'])
            sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', ax=ax,
                        xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
            ax.set_title(f'Confusion Matrix\n(threshold = {tr["opt_threshold"]:.3f})', fontsize=13, fontweight='bold')

            plt.suptitle(f'Classification Analysis -- {_subtitle}', fontsize=14, fontweight='bold', y=1.02)
            plt.tight_layout()
            plt.show()

        elif is_clf:
            # Fallback: just show basic table
            print("\nDetailed Metrics:")
            metrics_table = pd.DataFrame([metrics_dict[m] for m in models], index=models)
            display(metrics_table)

        if not is_clf:
            print("\nDetailed Metrics:")
            metrics_table = pd.DataFrame([metrics_dict[m] for m in models], index=models)
            display(metrics_table)

    def plot_feature_importance(self, X_train, y_train, target_options, tp, category=None):
        """Plot feature importance for each base model and meta-model coefficients"""
        feature_importances = self.get_feature_importance(X_train, y_train)

        # Plot feature importance for each base model
        for model_name, importance in feature_importances.items():
            # Plot ElasticNet coefficients
            if model_name == 'elasticnet':
                coefficients = pd.Series(importance, index=X_train.columns)
                # Apply display names if available
                if isinstance(globals().get('DISPLAY_NAMES'), dict) and len(globals()['DISPLAY_NAMES']) > 0:
                    coefficients.index = coefficients.index.map(lambda x: globals()['DISPLAY_NAMES'].get(x, x))
                top_20_coefs = coefficients.sort_values(key=abs, ascending=False).head(ENSEMBLE_VIZ_CONFIG['top_n'])

                plt.figure(figsize=(8, 8))
                sns.barplot(x=top_20_coefs.values, y=top_20_coefs.index, palette=ENSEMBLE_VIZ_CONFIG['palette_fn']())
                title = f"Top {ENSEMBLE_VIZ_CONFIG['top_n']} ElasticNet Coefficients for {'Regression' if self.task_type == 'regression' else 'Classification'}"
                title += f" ({_resolve_target_name(target_options)}, Timepoint {tp}"
                title += f", Category {category})" if category else ")"
                plt.title(title, fontsize=16)
                plt.xlabel("Coefficient Value", fontsize=12)
                plt.ylabel("Features", fontsize=12)
                for i, v in enumerate(top_20_coefs.values):
                    plt.text(v, i, f' {v:.4f}', va='center', fontsize=10)
                plt.tight_layout()
                plt.show()

            elif model_name == 'tabpfn':
              # Plot TabPFN importance from permutation importance
              feature_importance = pd.Series(importance, index=X_train.columns)
              top_20_features = feature_importance.sort_values(ascending=False).head(ENSEMBLE_VIZ_CONFIG['top_n'])

              plt.figure(figsize=(12, 8))
              sns.barplot(x=top_20_features.values, y=top_20_features.index, palette=ENSEMBLE_VIZ_CONFIG['palette_fn']())
              title = f"Top {ENSEMBLE_VIZ_CONFIG['top_n']} Important Features - TabPFN (Permutation Importance)"
              title += f" ({_resolve_target_name(target_options)}, Timepoint {tp}"
              title += f", Category {category})" if category else ")"
              plt.title(title, fontsize=16)
              plt.xlabel("Permutation Importance Score", fontsize=12)
              plt.ylabel("Features", fontsize=12)
              for i, v in enumerate(top_20_features.values):
                  plt.text(v, i, f' {v:.4f}', va='center', fontsize=10)
              plt.tight_layout()
              plt.show()
            else:
                # Plot tree-based model
                feature_importance = pd.Series(importance, index=X_train.columns)
                # Apply display names if available
                if isinstance(globals().get('DISPLAY_NAMES'), dict) and len(globals()['DISPLAY_NAMES']) > 0:
                    feature_importance.index = feature_importance.index.map(lambda x: globals()['DISPLAY_NAMES'].get(x, x))
                top_20_features = feature_importance.sort_values(ascending=False).head(ENSEMBLE_VIZ_CONFIG['top_n'])

                plt.figure(figsize=(12, 8))
                sns.barplot(x=top_20_features.values, y=top_20_features.index, palette=ENSEMBLE_VIZ_CONFIG['palette_fn']())
                title = f"Top {ENSEMBLE_VIZ_CONFIG['top_n']} Important Features - {model_name}"
                title += f" ({_resolve_target_name(target_options)}, Timepoint {tp}"
                title += f", Category {category})" if category else ")"
                plt.title(title, fontsize=16)
                plt.xlabel("Feature Importance", fontsize=12)
                plt.ylabel("Features", fontsize=12)
                plt.tight_layout()
                plt.show()

        # Plot meta-model coefficients if it's a linear model
        if hasattr(self.meta_model, 'coef_'):
            meta_coef = self.meta_model.coef_
            if meta_coef.ndim > 1:  # For multi-class
                meta_coef = np.mean(np.abs(meta_coef), axis=0)

            try:
                # Create feature names for meta-model (base model names)
                meta_feature_names = list(self.base_models.keys())
                meta_coefficients = pd.Series(meta_coef, index=meta_feature_names)

                plt.figure(figsize=(8, 6))
                sns.barplot(x=meta_coefficients.values, y=meta_coefficients.index, palette=ENSEMBLE_VIZ_CONFIG['palette_fn']())
                title = "Meta-Model Coefficients (Model Importance in Ensemble)"
                title += f" ({_resolve_target_name(target_options)}, Timepoint {tp}"
                title += f", Category {category})" if category else ")"
                plt.title(title, fontsize=16)
                plt.xlabel("Coefficient Value", fontsize=12)
                plt.ylabel("Base Model", fontsize=12)
                for i, v in enumerate(meta_coefficients.values):
                    plt.text(v, i, f' {v:.4f}', va='center', fontsize=10)
                plt.tight_layout()
                plt.show()

            except: # When it is enhanced, you will go here
                print("Enhanced feature importance visualization..")
                # Create feature names for meta-model including interactions
                base_names = list(self.base_models.keys())

                # Create names for interaction terms
                meta_feature_names = base_names.copy()
                for i, name1 in enumerate(base_names):
                    for j, name2 in enumerate(base_names[i+1:], i+1):
                        meta_feature_names.append(f"{name1} × {name2}")
                meta_feature_names.append("Weighted Average")

                # lengths match
                if len(meta_coef) > len(meta_feature_names):
                    meta_feature_names.extend([f"Feature {i}" for i in range(len(meta_feature_names), len(meta_coef))])
                meta_feature_names = meta_feature_names[:len(meta_coef)]

                meta_coefficients = pd.Series(meta_coef, index=meta_feature_names)

                # Sort by absolute value
                meta_coefficients = meta_coefficients.reindex(
                    meta_coefficients.abs().sort_values(ascending=False).index
                )

                plt.figure(figsize=(8, 6))
                sns.barplot(x=meta_coefficients.values, y=meta_coefficients.index, palette=ENSEMBLE_VIZ_CONFIG['palette_fn']())
                title = "Meta-Model Model Importance"
                title += f" ({_resolve_target_name(target_options)}, Timepoint {tp}"
                title += f", Category {category})" if category else ")"
                plt.title(title, fontsize=16)
                plt.xlabel("Coefficient Value", fontsize=12)
                plt.ylabel("Features", fontsize=12)
                for i, v in enumerate(meta_coefficients.values):
                    plt.text(v, i, f' {v:.4f}', va='center', fontsize=10)
                plt.tight_layout()
                plt.show()

    def explain_with_shap(self, X_train, X_test):
        """Enhanced SHAP analysis with special handling for TabPFN"""
        if hasattr(X_train, 'get_features'):
            X_train = pd.DataFrame(X_train.get_features(), columns=X_train.get_feature_names())
        if hasattr(X_test, 'get_features'):
            X_test = pd.DataFrame(X_test.get_features(), columns=X_test.get_feature_names())

        shap_values_dict = {}

        for name, model in self.base_models.items():
            print(f"\nGenerating SHAP values for base model: {name}...")

            try:
                if name == 'tabpfn':
                  try:
                    # Use specialized TabPFN SHAP analysis
                    print("\nPerforming specialized TabPFN SHAP analysis...")
                  except Exception as e:
                    print(f"Exception: {e}, alternating..")
                    print("\nPerforming specialized TabPFN SHAP analysis...")
                    # Prepare data for TabPFN SHAP
                    n_samples = min(50, len(X_test))
                    X_test_subset = X_test.iloc[:n_samples] if isinstance(X_test, pd.DataFrame) else X_test[:n_samples]
                    feature_names = X_test.columns

                    # Define prediction function
                    def predict_function_for_shap(X):
                        if hasattr(model, 'predict_proba'):
                            return model.predict_proba(X)
                        return model.predict(X)

                    # Create explainer with proper background
                    background = np.ones((1, X_test_subset.shape[1])) * float('nan')
                    explainer = shap.Explainer(predict_function_for_shap, background)

                    # Calculate SHAP values
                    shap_values = explainer(X_test_subset)
                    shap_values_dict[name] = shap_values

                    # Plot TabPFN SHAP visualizations
                    if len(shap_values.shape) == 3:
                        print("Computing SHAP values for the first class (index 0).")
                        shap_values = shap_values[:, :, 0]

                    # 1. Aggregate Feature Importances
                    plt.figure(figsize=(12, 8))
                    shap.plots.bar(shap_values=shap_values, show=False)
                    plt.title("Aggregate Feature Importances (TabPFN)")
                    plt.tight_layout()
                    plt.show()

                    # 2. Feature Importance Distribution
                    plt.figure(figsize=(12, 8))
                    shap.summary_plot(shap_values=shap_values, show=False)
                    plt.title("Feature Importance Distribution (TabPFN)")
                    plt.tight_layout()
                    plt.show()

                    # 3. Feature Interactions
                    most_important = np.abs(shap_values.values).mean(0).argsort()[-1]
                    print(f'\nAnalyzing interactions for most important feature: "{feature_names[most_important]}"')

                    if len(shap_values) > 1:
                        inds = shap.utils.potential_interactions(
                            shap_values[:, most_important],
                            shap_values
                        )

                        for i in range(min(3, len(inds))):
                            plt.figure(figsize=(10, 6))
                            shap.plots.scatter(
                                shap_values[:, most_important],
                                color=shap_values[:, inds[i]],
                                show=False
                            )
                            plt.title(f"Interaction: {feature_names[most_important]} vs {feature_names[inds[i]]}")
                            plt.tight_layout()
                            plt.show()

                    # Print top 50 SHAP values for TabPFN
                    print(f"\nTop 50 Feature SHAP Values for {name}:")
                    print("-" * 50)
                    mean_shap_values = np.abs(shap_values.values).mean(0)
                    feature_importance = pd.DataFrame({
                        'Feature': feature_names,
                        'SHAP Value': mean_shap_values
                    })
                    feature_importance = feature_importance.sort_values('SHAP Value', ascending=False)
                    for idx, row in feature_importance.head(50).iterrows():
                        print(f"{row['Feature']}: {row['SHAP Value']:.6f}")
                        print("-" * 50)

                elif name in ['catboost', 'random_forest', 'xgboost']:
                    # Tree-based models
                    explainer = shap.TreeExplainer(model)
                    shap_values = explainer.shap_values(X_test)
                    shap_values_dict[name] = shap_values

                    plt.figure(figsize=(20, 16))
                    if isinstance(shap_values, list):
                        shap.summary_plot(
                            shap_values[1] if len(shap_values) > 1 else shap_values,
                            X_test,
                            plot_type="violin",
                            show=False,
                            max_display=ENSEMBLE_VIZ_CONFIG['top_n'],
                            title=f"SHAP Values - {name}"
                        )
                    else:
                        shap.summary_plot(
                            shap_values,
                            X_test,
                            plot_type="violin",
                            show=False,
                            max_display=ENSEMBLE_VIZ_CONFIG['top_n'],
                            title=f"SHAP Values - {name}"
                        )

                    plt.title(f"SHAP Feature Importance and Impact - {name}", fontsize=24)
                    plt.xlabel("SHAP value (impact on model output)", fontsize=18)
                    plt.ylabel("Features", fontsize=18)
                    plt.tick_params(axis='both', which='major', labelsize=14)

                    cbar = plt.gcf().axes[-1]
                    cbar.set_ylabel("Feature value", fontsize=18, labelpad=20)
                    cbar.tick_params(labelsize=14)

                    plt.tight_layout()
                    plt.show()

                elif name in ['elasticnet']:
                    # Linear model
                    explainer = shap.LinearExplainer(model, X_train)
                    shap_values = explainer.shap_values(X_test)
                    shap_values_dict[name] = shap_values

            except Exception as e:
                print(f"Could not generate SHAP values for {name}: {str(e)}")

        return shap_values_dict

def create_stacking_ensemble(
    task_type: str = 'regression',
    model_configs: Dict[str, Dict[str, Any]] = None,
    meta_model=None,
    cv: int = 5,
    random_state: int = 42
) -> StackingEnsembleModel:
    """Factory function to create a stacking ensemble model"""
    return StackingEnsembleModel(
        task_type=task_type,
        model_configs=model_configs,
        meta_model=meta_model,
        cv=cv,
        random_state=random_state
    )

def create_meta_features_parallel(X, y, model, cv_splits, random_state=42):
    """Generate meta-features in parallel using joblib"""
    def _fit_predict_fold(train_idx, val_idx, X, y, model):
        if isinstance(X, pd.DataFrame):
            X_train_fold = X.iloc[train_idx]
            y_train_fold = y.iloc[train_idx]
            X_val_fold = X.iloc[val_idx]
        else:
            X_train_fold = X[train_idx]
            y_train_fold = y[train_idx]
            X_val_fold = X[val_idx]

        model_clone = _safe_clone(model)
        model_clone.fit(X_train_fold, y_train_fold)
        return val_idx, model_clone.predict(X_val_fold)

    # Create CV splits
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    splits = list(kf.split(X, y))

    # Generate predictions in parallel
    results = Parallel(n_jobs=-1)(
        delayed(_fit_predict_fold)(train_idx, val_idx, X, y, model)
        for train_idx, val_idx in splits
    )

    # Combine results
    meta_features = np.zeros(len(X))
    for val_idx, preds in results:
        meta_features[val_idx] = preds

    return meta_features

def enhance_stacking_ensemble_model():
    """
    Enhance the StackingEnsembleModel with additional functionality.
    Returns a class with enhanced methods.
    """
    # Store original methods as backup
    original_methods = {
        '_get_meta_features': StackingEnsembleModel._get_meta_features if hasattr(StackingEnsembleModel, '_get_meta_features') else None,
        'fit': StackingEnsembleModel.fit,
        'predict': StackingEnsembleModel.predict
    }

    def _enhanced_get_meta_features(self, X, y=None, is_train=True):
      """Enhanced meta-features generation with TabPFN optimization using CatBoost features"""
      if hasattr(X, 'get_features'):
          X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
      else:
          X_features = X

      if is_train:
          if self.task_type == 'classification' and y is not None:
              self.n_classes_ = len(np.unique(y))
          else:
              self.n_classes_ = 2
          meta_features = np.zeros((X_features.shape[0], len(self.base_models)))
          if self.task_type == 'classification':
              from sklearn.model_selection import StratifiedKFold
              kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)
          else:
              kf = KFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)

          # If number of features exceeds TabPFN limit, prepare selected features using CatBoost
          tabpfn_features = None
          if X_features.shape[1] > 100 and 'tabpfn' in self.base_models and 'catboost' in self.base_models:
              print("\nSelecting top features for TabPFN (selecting top 100 for optimal speed/performance)...")
              # Fit CatBoost on full data to get feature importance
              if self.task_type == 'regression':
                  temp_catboost = CatBoostRegressor(
                      iterations=100,
                      learning_rate=0.1,
                      verbose=False,
                      random_state=self.random_state
                  )
              else:
                  temp_catboost = CatBoostClassifier(
                      iterations=100,
                      learning_rate=0.1,
                      verbose=False,
                      random_state=self.random_state
                  )

              temp_catboost.fit(X_features, y)
              importance = pd.Series(temp_catboost.feature_importances_, index=X_features.columns)
              selected_features = importance.nlargest(100).index
              tabpfn_features = X_features[selected_features]
              self.tabpfn_feature_reduction = selected_features
              print(f"Selected {len(selected_features)} top features for TabPFN")

          for i, (name, model) in enumerate(self.base_models.items()):
              print(f"\nGenerating meta-features for {name}...")
              temp_meta_features = np.zeros(X_features.shape[0])

              if name == 'tabpfn':
                  # Use selected features for TabPFN if available
                  X_for_tabpfn = tabpfn_features if tabpfn_features is not None else X_features

                  for train_idx, val_idx in kf.split(X_for_tabpfn, y):
                      X_train_fold = X_for_tabpfn.iloc[train_idx]
                      y_train_fold = y.iloc[train_idx]
                      X_val_fold = X_for_tabpfn.iloc[val_idx]

                      if self.task_type == 'regression':
                          model_clone = TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                      else:
                          model_clone = TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)

                      try:
                          _sw = None
                          if self.task_type == 'classification' and self.class_weights is not None:
                              _sw = np.array([float(self.class_weights.get(yi, 1.0)) for yi in y_train_fold])
                          try:
                              model_clone.fit(X_train_fold, y_train_fold, sample_weight=_sw)
                          except TypeError:
                              model_clone.fit(X_train_fold, y_train_fold)
                          if self.task_type == 'classification':
                              if getattr(self, 'n_classes_', 2) <= 2:
                                  fold_preds = model_clone.predict_proba(X_val_fold)[:, 1]
                              else:
                                  fold_preds = model_clone.predict(X_val_fold).astype(float)
                          else:
                              fold_preds = model_clone.predict(X_val_fold)
                      except Exception as e:
                          print(f"TabPFN error: {e}, using fallback predictions")
                          fold_preds = np.full(len(X_val_fold), np.mean(y_train_fold))

                      temp_meta_features[val_idx] = fold_preds

                  # Store selected features for prediction time
                  if tabpfn_features is not None:
                      model.selected_features_ = selected_features

              else:
                  # Other models use all features
                  for train_idx, val_idx in kf.split(X_features, y):
                      X_train_fold = X_features.iloc[train_idx]
                      y_train_fold = y.iloc[train_idx]
                      X_val_fold = X_features.iloc[val_idx]

                      model_clone = _safe_clone(model)
                      if self.task_type == 'classification':
                          if name == 'catboost':
                              model_clone.set_params(class_weights=self.class_weights)
                          elif name == 'random_forest':
                              model_clone.set_params(class_weight=self.class_weights)
                          elif name == 'xgboost' and len(np.unique(y)) == 2:
                              _y_spw = np.asarray(y).ravel()
                              _cls = np.unique(_y_spw)
                              model_clone.set_params(scale_pos_weight=float(np.sqrt(np.sum(_y_spw == _cls[0]) / np.sum(_y_spw == _cls[1]))))

                      try:
                        model_clone.fit(X_train_fold, y_train_fold)
                      except Exception as e:
                        print(f"[WARNING] {e}")
                        model_clone.fit(X_train_fold, y_train_fold.iloc[:, 0])

                      # Use predict_proba for classification (consistent scale with TabPFN)
                      if self.task_type == 'classification' and hasattr(model_clone, 'predict_proba'):
                          if getattr(self, 'n_classes_', 2) <= 2:
                              fold_preds = model_clone.predict_proba(X_val_fold)[:, 1]
                          else:
                              fold_preds = model_clone.predict(X_val_fold).astype(float)
                      else:
                          fold_preds = model_clone.predict(X_val_fold)
                      temp_meta_features[val_idx] = fold_preds

              meta_features[:, i] = temp_meta_features

              # Fit on full data
              if name == 'tabpfn':
                  # Use selected features for final TabPFN model
                  X_for_tabpfn = tabpfn_features if tabpfn_features is not None else X_features
                  if self.task_type == 'regression':
                      self.base_models[name] = TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                  else:
                      self.base_models[name] = TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                  _sw_full = None
                  if self.task_type == 'classification' and self.class_weights is not None:
                      _sw_full = np.array([float(self.class_weights.get(yi, 1.0)) for yi in y])
                  try:
                      try:
                          self.base_models[name].fit(X_for_tabpfn, y, sample_weight=_sw_full)
                      except TypeError:
                          self.base_models[name].fit(X_for_tabpfn, y)
                  except Exception as _tabpfn_err:
                      print(f"[WARNING] TabPFN full-data fit failed: {_tabpfn_err}. {'GPU OOM -- subsampling to ≤9999' if _TABPFN_DEVICE == 'cuda' else 'CPU limit -- subsampling to ≤999'}.")
                      if _TABPFN_DEVICE == 'cuda':
                          import torch; torch.cuda.empty_cache()
                      _max_n = min(9999 if _TABPFN_DEVICE == 'cuda' else 999, len(X_for_tabpfn))
                      _sub_idx = np.random.choice(len(X_for_tabpfn), _max_n, replace=False)
                      try:
                          self.base_models[name].fit(X_for_tabpfn.iloc[_sub_idx], y.iloc[_sub_idx])
                      except Exception as _e2:
                          print(f"[WARNING] TabPFN subsampled fit also failed: {_e2}. Predictions may be degraded.")
                  if tabpfn_features is not None:
                      self.base_models[name].selected_features_ = selected_features  # transfer to new instance
              else:
                try:
                  model.fit(X_features, y)
                except Exception as e:
                  print(f"[WARNING] {e}")
                  model.fit(X_features, y.iloc[:, 0])

      else:
          # Held-out predictions
          meta_features = []
          for name, model in self.base_models.items():
              if name == 'tabpfn':
                  # Use selected features for prediction if available
                  if hasattr(model, 'selected_features_'):
                      X_for_tabpfn = X_features[model.selected_features_]
                  else:
                      X_for_tabpfn = X_features
                  # Align columns to match what TabPFN saw during fit
                  if hasattr(model, 'feature_names_in_'):
                      X_for_tabpfn = X_for_tabpfn.reindex(columns=list(model.feature_names_in_), fill_value=0)

                  try:
                      if self.task_type == 'classification':
                          if getattr(self, 'n_classes_', 2) <= 2:
                              preds = model.predict_proba(X_for_tabpfn)[:, 1]
                          else:
                              preds = model.predict(X_for_tabpfn).astype(float)
                      else:
                          preds = model.predict(X_for_tabpfn)
                  except Exception as e:
                      print(f"TabPFN prediction error: {e}, using fallback")
                      _fb = 0.5 if self.task_type == 'classification' else 0.0
                      preds = np.full(X_features.shape[0], _fb)
              else:
                  if self.task_type == 'classification' and hasattr(model, 'predict_proba'):
                      if getattr(self, 'n_classes_', 2) <= 2:
                          preds = model.predict_proba(X_features)[:, 1]
                      else:
                          preds = model.predict(X_features).astype(float)
                  else:
                      preds = model.predict(X_features)
              meta_features.append(preds)

          meta_features = np.column_stack(meta_features)

      return meta_features

    def _add_meta_interactions(self, meta_features):
        """Add interaction features between base model predictions"""
        if meta_features is None:
            raise ValueError("meta_features cannot be None")

        n_models = meta_features.shape[1]
        interactions = []

        # Add pairwise interactions
        for i in range(n_models):
            for j in range(i + 1, n_models):
                interactions.append(meta_features[:, i] * meta_features[:, j])

        # Add weighted averages
        weighted_avg = np.average(meta_features, axis=1, weights=np.ones(n_models)/n_models)
        interactions.append(weighted_avg)

        # Combine original features with interactions
        meta_features_enhanced = np.column_stack([meta_features] + interactions)

        return meta_features_enhanced

    def _enhanced_fit(self, X, y, categorical_features=None):
        """Enhanced fit method with improved error handling"""
        if hasattr(X, 'get_features'):
            X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
        else:
            X_features = X

        # Validate input data
        if isinstance(X_features, pd.DataFrame):
            has_null = X_features.isnull().any().any()
        else:
            has_null = np.isnan(X_features).any()

        if has_null:
            warnings.warn("Input contains null values. Handling with median imputation.")
            imputer = SimpleImputer(strategy='median')
            if isinstance(X_features, pd.DataFrame):
                X_features = pd.DataFrame(
                    imputer.fit_transform(X_features),
                    columns=X_features.columns,
                    index=X_features.index
                )
            else:
                X_features = imputer.fit_transform(X_features)

        # Set class weights for classification
        if self.task_type == 'classification':
            self.set_class_weights(y)

        # Generate meta-features
        meta_features = self._get_meta_features(X_features, y, is_train=True)
        meta_features_with_interactions = self._add_meta_interactions(meta_features)

        # Fit meta-model with cross-validation
        if isinstance(self.meta_model, (LogisticRegression, ElasticNet)):
            param_grid = {
                'C': np.logspace(-4, 4, 20) if isinstance(self.meta_model, LogisticRegression) else None,
                'alpha': np.logspace(-4, 4, 20) if isinstance(self.meta_model, ElasticNet) else None,
                'l1_ratio': np.linspace(0, 1, 11) if isinstance(self.meta_model, ElasticNet) else None
            }
            param_grid = {k: v for k, v in param_grid.items() if v is not None}

            cv = RandomizedSearchCV(
                self.meta_model,
                param_grid,
                n_iter=20,
                cv=5,
                random_state=self.random_state
            )
            cv.fit(meta_features_with_interactions, y)
            self.meta_model = cv.best_estimator_
        else:
            self.meta_model.fit(meta_features_with_interactions, y)

        return self

    def _enhanced_predict(self, X):
        """Enhanced prediction method"""
        if hasattr(X, 'get_features'):
            X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names())
        else:
            X_features = X

        # Generate meta-features
        meta_features = self._get_meta_features(X_features, is_train=False)
        meta_features_with_interactions = self._add_meta_interactions(meta_features)

        # Get predictions from meta-model
        predictions = self.meta_model.predict(meta_features_with_interactions)

        if self.task_type == 'classification':
            return np.round(predictions).astype(int)
        return predictions

    def get_tree_model_features(self, X_train, top_k=50):
        """Get important features from tree-based models"""
        all_important_features = set()

        # First get importance from tree-based models only
        for name, model in self.base_models.items():
            if name in ['catboost', 'random_forest', 'xgboost']:
                if hasattr(model, 'feature_importances_'):
                    # Get feature importance
                    importances = pd.Series(
                        model.feature_importances_,
                        index=X_train.columns
                    )
                    # Get top k features
                    top_features = importances.nlargest(top_k).index
                    all_important_features.update(top_features)

        return list(all_important_features)

    def plot_tabpfn_uncertainty(self, X_train, y_train, X_test=None):
        """Plot uncertainty visualization for TabPFN predictions"""
        if 'tabpfn' not in self.base_models:
            return None

        try:
            # Generate test points if not provided
            if X_test is None:
                X_test = np.linspace(np.min(X_train), np.max(X_train), 200).reshape(-1, 1)

            # Get TabPFN predictions with quantiles
            model = self.base_models['tabpfn']
            preds = model.predict(X_test, output_type="full")

            fig = plt.figure(figsize=(12,6))

            # Plot original data
            ax = fig.add_subplot(121)
            ax.scatter(X_train, y_train, label='Training data', color='red', s=10)
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_xlim(min(X_train) - 1, max(X_train) + 1)
            ax.set_ylim(np.min(y_train) - 1, np.max(y_train) + 1)
            ax.set_title('Training Data')
            ax.legend()

            # Plot uncertainty
            ax = fig.add_subplot(122)
            all_quantiles = preds["quantiles"]
            y = np.array(all_quantiles)

            # Calculate the maximum and minimum values
            y_max = np.max(y, axis=0)
            y_min = np.min(y, axis=0)

            # Calculate quantile bin widths
            quantile_bin_widths = np.diff(y, axis=0)

            # Normalize bin widths
            per_x_normalized_bin_widths = quantile_bin_widths / (y_max - y_min)

            # Plotting rectangles for uncertainty
            num_bins, num_data_points = per_x_normalized_bin_widths.shape
            rect_width = (X_test[1] - X_test[0]).squeeze()

            for i in range(num_data_points):
                for j in range(num_bins):
                    rect = plt.Rectangle(
                        xy=(X_test[i][0] - rect_width/2, y[j, i]),
                        width=rect_width,
                        height=quantile_bin_widths[j, i],
                        facecolor=plt.cm.viridis(per_x_normalized_bin_widths[j, i] * 5),
                        edgecolor='none'
                    )
                    ax.add_patch(rect)

            ax.scatter(X_train, y_train, label='Training data', color='red', s=10)
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            ax.set_title('TabPFN Prediction Uncertainty')
            plt.tight_layout()
            plt.show()

            return fig

        except Exception as e:
            print(f"Error plotting TabPFN uncertainty: {str(e)}")
            return None

    def analyze_feature_consensus(self, X_train, y_train, target_options, tp, category=None, top_n=None):
        if top_n is None:
            top_n = ENSEMBLE_VIZ_CONFIG['top_n']
        """Generate weighted consensus analysis using consensus features for SVM"""
        import pandas as pd
        import numpy as np
        import matplotlib.pyplot as plt
        import seaborn as sns
        from sklearn.inspection import permutation_importance

        # Get meta-model coefficients for weights
        if hasattr(self.meta_model, 'coef_'):
            meta_coef = self.meta_model.coef_
            if meta_coef.ndim > 1:
                meta_coef = np.mean(np.abs(meta_coef), axis=0)

            base_names = list(self.base_models.keys())
            model_weights = {}
            for i, name in enumerate(base_names):
                if i < len(meta_coef):
                    model_weights[name] = abs(meta_coef[i])
                else:
                    model_weights[name] = 0
        else:
            model_weights = {name: 1/len(self.base_models) for name in self.base_models.keys()}

        weight_sum = sum(model_weights.values())
        model_weights = {k: v/weight_sum for k, v in model_weights.items()}

        # Get feature importance from each model
        feature_ranks = {}
        feature_importances = {}
        all_features = set()

        # First get important features from tree models
        important_features = self.get_consensus_features(X_train)
        print(f"Number of consensus features from tree models: {len(important_features)}")

        # Store original column order
        original_columns = X_train.columns.tolist()

        for name, model in self.base_models.items():
            try:
                if name in ['svm', 'neural_net', 'tabpfn']:
                    print(f"Calculating permutation importance for {name} on consensus features...")
                    if name == 'tabpfn':
                        from sklearn.inspection import permutation_importance

                        try:
                            print("\nGenerating TabPFN uncertainty visualization...")
                            uncertainty_fig = self.plot_tabpfn_uncertainty(X_train, y_train)
                            if uncertainty_fig:
                                print("TabPFN uncertainty visualization generated successfully")
                        except Exception as e:
                            print(f"Warning: {e}")
                            continue

                    if hasattr(model, 'named_steps'):
                        # Preserve feature order for SVM
                        X_subset = X_train[important_features].reindex(columns=original_columns)

                        # Before subsetting
                        print("NaN values in X_train:", X_train.isnull().sum().sum())

                        # After subsetting
                        print("NaN values in X_subset:", X_subset.isnull().sum().sum())

                        # Check column continuity
                        print("X_subset columns:", X_subset.columns)
                        print("Important features:", important_features)

                        # Handle NaN values before scaling if they exist
                        if X_subset.isnull().any().any():  # Check if any NaN exists
                            print(f"Found NaN values in X_train subset crunch: {X_subset.isnull().sum().sum()}")
                            imputer = SimpleImputer(strategy='median')
                            X_subset = pd.DataFrame(
                                imputer.fit_transform(X_subset),
                                columns=X_subset.columns,
                                index=X_subset.index
                            )
                            print(f"Imputed {X_subset.isnull().sum().sum()} NaN values")
                        else:
                            print("No NaN values found for X_train subset")

                        scaler = model.named_steps['scaler']
                        X_scaled = scaler.transform(X_subset)
                        actual_model = model.named_steps['svr' if self.task_type == 'regression' else 'svc'] if name == 'svm' else model.named_steps['mlp']

                        if name == 'svm' and hasattr(actual_model, 'kernel') and actual_model.kernel == 'linear':
                            print("Using linear kernel coefficients for SVM importance")
                            full_importance = np.zeros(len(original_columns))
                            for idx, feat in enumerate(important_features):
                                full_importance[original_columns.index(feat)] = np.abs(actual_model.coef_[0][idx])
                            importances = full_importance
                        else:
                            from sklearn.inspection import permutation_importance

                            result = permutation_importance(
                                actual_model, X_scaled, y_train,
                                n_repeats=10,
                                random_state=42,
                                n_jobs=-1
                            )

                            full_importance = np.zeros(len(original_columns))
                            for idx, feat in enumerate(important_features):
                                full_importance[original_columns.index(feat)] = result.importances_mean[idx]
                            importances = full_importance
                    print(f"Successfully calculated importance for {name}")

                elif hasattr(model, 'feature_importances_'):
                    importances = model.feature_importances_
                elif hasattr(model, 'coef_'):
                    importances = np.abs(model.coef_)
                else:
                    continue

                # Create Series with feature names
                model_importance = pd.Series(importances, index=original_columns)
                feature_importances[name] = model_importance

                # Create rank Series (1 for most important)
                ranks = model_importance.rank(ascending=False)
                feature_ranks[name] = ranks

                all_features.update(model_importance.index)

            except Exception as e:
                print(f"Could not calculate importance for {name}: {str(e)}")
                print(f"Model type: {type(model)}")
                if hasattr(model, 'named_steps'):
                    print(f"Pipeline steps: {model.named_steps.keys()}")

        # Calculate consensus scores
        consensus_scores = {}
        for feature in all_features:
            weighted_score = 0
            feature_details = {'scores': [], 'ranks': [], 'weighted_scores': []}

            for model_name, weight in model_weights.items():
                if model_name in feature_importances:
                    importance = feature_importances[model_name].get(feature, 0)
                    rank = feature_ranks[model_name].get(feature, len(all_features))

                    norm_importance = importance / feature_importances[model_name].max()
                    weighted_importance = norm_importance * weight

                    feature_details['scores'].append(norm_importance)
                    feature_details['ranks'].append(rank)
                    feature_details['weighted_scores'].append(weighted_importance)
                    weighted_score += weighted_importance

            consensus_scores[feature] = {
                'weighted_score': weighted_score,
                'avg_rank': np.mean(feature_details['ranks']),
                'rank_std': np.std(feature_details['ranks']),
                'score_std': np.std(feature_details['scores']),
                'details': feature_details
            }

        # Convert to DataFrame and sort
        consensus_df = pd.DataFrame.from_dict(consensus_scores, orient='index')

        # -- Delta-Level Redundancy Filter ---
        # If a variable and its delta (T2−T0) both survive consensus,
        # keep only the higher-scored one to prevent construct inflation.
        # Runs BEFORE display-name mapping (index still has delta_ prefix).
        _raw_idx = consensus_df.index.tolist()
        _delta_drop = set()
        for _feat in _raw_idx:
            if _feat.startswith('delta_'):
                _base = _feat.replace('delta_', '', 1)
                if _base in _raw_idx and _base not in _delta_drop:
                    _s_delta = consensus_df.loc[_feat, 'weighted_score']
                    _s_base  = consensus_df.loc[_base, 'weighted_score']
                    _loser = _base if _s_delta >= _s_base else _feat
                    _delta_drop.add(_loser)
                    _winner = _base if _loser != _base else _feat
                    print(f"  [DELTA-REDUNDANCY] Dropping '{_loser}' "
                          f"(score={consensus_df.loc[_loser,'weighted_score']:.4f}) "
                          f"-- kept '{_winner}' "
                          f"(score={consensus_df.loc[_winner,'weighted_score']:.4f})")
        if _delta_drop:
            consensus_df = consensus_df.drop(index=list(_delta_drop))
            print(f"  [DELTA-REDUNDANCY] Removed {len(_delta_drop)} redundant level/delta feature(s)")

        # Apply display names if available
        _DN = globals().get('DISPLAY_NAMES')
        if isinstance(_DN, dict) and len(_DN) > 0:
            consensus_df.index = consensus_df.index.map(lambda x: _DN.get(x, x))
            # Deduplicate display-name collisions (keep highest weighted_score)
            if consensus_df.index.duplicated().any():
                _dup_names = consensus_df.index[consensus_df.index.duplicated(keep=False)].unique().tolist()
                print(f"  Warning: DISPLAY_NAMES collision on: {_dup_names} -- keeping highest-scored entry")
                consensus_df = consensus_df[~consensus_df.index.duplicated(keep='first')]

        top_features = consensus_df.sort_values('weighted_score', ascending=False).head(top_n)

        # Print publication-ready text and create visualizations
        print("\nFeature Importance Analysis for Publication")
        print("=" * 50)
        print(f"\nBased on our stacking ensemble model analysis")
        print(f"Time point: {tp}")
        if category:
            print(f"Category: {category}")
        print(f"\nModel Contributions to Meta-learner:")
        for model, weight in model_weights.items():
            print(f"- {model}: {weight:.2%}")

        print("\nTop Features by Weighted Consensus:")
        for i, (feature, row) in enumerate(top_features.iterrows(), 1):
            consistency = "High" if row['score_std'] < 0.1 else "Moderate" if row['score_std'] < 0.2 else "Variable"
            print(f"\n{i}. {feature}")
            print(f"   Weighted Importance: {row['weighted_score']:.4f}")
            print(f"   Average Rank: {row['avg_rank']:.1f}")
            print(f"   Consistency: {consistency}")

        # Visualizations -- uses ENSEMBLE_VIZ_CONFIG from cell UI
        _viz2 = ENSEMBLE_VIZ_CONFIG
        _palette2 = _viz2['palette_fn']()
        # Normalise: palette_fn may return a string name → convert to colour list
        if isinstance(_palette2, str):
            import seaborn as _sns_pal
            _palette2 = list(_sns_pal.color_palette(_palette2, n_colors=_viz2['top_n']))
        _scores2  = top_features['weighted_score'].values
        _labels2  = top_features.index.tolist()
        _consist2 = ['H' if s < 0.1 else 'M' if s < 0.2 else 'V'
                      for s in top_features['score_std'].values]

        fig2, ax2 = plt.subplots(figsize=(_viz2['fig_width'], _viz2['fig_height']))
        bars2 = ax2.barh(range(len(_labels2)), _scores2, color=_palette2[:len(_labels2)])
        ax2.set_yticks(range(len(_labels2)))
        ax2.set_yticklabels(_labels2, fontsize=_viz2['ylabel_fs'])
        ax2.invert_yaxis()
        ax2.set_xlabel('Weighted Ensemble Consensus Score', fontsize=9)
        ax2.set_title(f'{_resolve_target_name(target_options)}, Timepoint {tp}' +
                      (f' -- {category}' if category else ''), fontsize=14, fontweight='bold')

        # Score + consistency annotations at bar ends
        _x_max2 = max(_scores2) * 1.22
        for idx, (bar, sc, con) in enumerate(zip(bars2, _scores2, _consist2)):
            ax2.text(sc + _x_max2 * 0.015, idx, f'{sc:.3f}',
                     va='center', ha='left', fontsize=8, color='#333333')
        ax2.set_xlim(0, _x_max2)

        ax2.set_axisbelow(True)
        ax2.grid(False)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)

        plt.tight_layout()
        if _viz2['save']:
            _fn2 = f"{_viz2['save_prefix']}_consensus_plot"
            plt.savefig(f"{_fn2}.png", dpi=_viz2['dpi'], bbox_inches='tight')
        plt.show()

        # Enhanced heatmap with all models
        # Rename feature_importances index to match display names
        if isinstance(_DN, dict) and len(_DN) > 0:
            feature_importances = {
                model: imp.rename(index=lambda x: _DN.get(x, x))
                          .pipe(lambda s: s[~s.index.duplicated(keep='first')])
                for model, imp in feature_importances.items()
            }

        feature_comparison = pd.DataFrame(
            {model: importances.reindex(top_features.index, fill_value=0)
            for model, importances in feature_importances.items()}
        )
        feature_comparison = feature_comparison.div(feature_comparison.max())

        plt.figure(figsize=(12, 10))
        sns.heatmap(
            feature_comparison,
            cmap=_viz2['heatmap_cmap'],
            annot=True,
            fmt='.2f',
            cbar_kws={'label': 'Normalized Importance'},
            center=0.5
        )
        plt.title('Feature Importance Comparison Across All Models')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

        return consensus_df, model_weights, feature_comparison

    # -- Enhanced predict_proba (must match _enhanced_fit dimensionality) --
    def _enhanced_predict_proba(self, X):
        if self.task_type != 'classification':
            raise ValueError('predict_proba is only available for classification')
        X_features = pd.DataFrame(X.get_features(), columns=X.get_feature_names()) if hasattr(X, 'get_features') else X
        meta = self._get_meta_features(X_features, is_train=False)
        meta = self._add_meta_interactions(meta)
        return self.meta_model.predict_proba(meta)

    # Add all enhanced methods to the class
    StackingEnsembleModel._get_meta_features = _enhanced_get_meta_features
    StackingEnsembleModel._add_meta_interactions = _add_meta_interactions
    StackingEnsembleModel.fit = _enhanced_fit
    StackingEnsembleModel.predict = _enhanced_predict
    StackingEnsembleModel.predict_proba = _enhanced_predict_proba

    # Store enhanced methods for reference
    StackingEnsembleModel._enhanced_methods = {
        '_get_meta_features': _enhanced_get_meta_features,
        '_add_meta_interactions': _add_meta_interactions,
        'fit': _enhanced_fit,
        'predict': _enhanced_predict,
        'predict_proba': _enhanced_predict_proba,
    }

    # Store original methods for potential rollback
    StackingEnsembleModel._original_methods = original_methods

    StackingEnsembleModel.analyze_feature_consensus = analyze_feature_consensus

    StackingEnsembleModel.get_tree_model_features = get_tree_model_features
    StackingEnsembleModel.plot_tabpfn_uncertainty = plot_tabpfn_uncertainty

    return StackingEnsembleModel

# -- Apply enhanced methods (consensus, tree features, uncertainty) ---
enhance_stacking_ensemble_model()
print("[Ensemble] Enhanced methods attached to StackingEnsembleModel")

import catboost
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import PartialDependenceDisplay
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pygam import GAM, s
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

from collections import Counter
from smogn import smoter
import shap
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from itertools import combinations
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import warnings
from sklearn.exceptions import DataConversionWarning
from mapie.regression import SplitConformalRegressor
from mapie.classification import SplitConformalClassifier

from sklearn.model_selection import train_test_split
import optuna
from optuna.integration import CatBoostPruningCallback
from itertools import combinations
from sklearn.metrics import confusion_matrix, roc_curve, auc
from scipy import stats
from itertools import combinations
from collections import defaultdict
from tqdm import tqdm
from sklearn.linear_model import LinearRegression, LogisticRegression, ElasticNet
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn.constants import ModelVersion  # For TabPFN v2 (no auth required)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

category_results = defaultdict(dict)

# Ignore DataConversionWarning
warnings.filterwarnings(action='ignore', category=DataConversionWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# Set seaborn defaults for consistent figure styling.
sns.set(style="whitegrid")
sns.color_palette("cubehelix", as_cmap=True)

In [ ]:
#@title Main Analysis Runner

# Executes the ensemble pipeline for a selected target, timepoint,
# subgroup, and split type. Within-category mode runs each feature-set
# category independently to isolate domain-specific predictive power.
# Across-category mode uses the full predictor space minus circular
# exclusions. Cross-timepoint mode (feature_tp != target_tp) draws
# features from one wave to predict outcomes at another. Batch mode
# runs preconfigured multiple target sets which are compatible with downstream visualization cells
# Subgroup filters sex, ALE-stratified.
# isolate_category runs a single category if within-category is used.
# ---

# ---
# 1. TARGET & DATA
# ---
#@markdown ### 1 Target & Data
#@markdown ---
#@markdown **target_options**: Outcome variable to predict (ignored when batch_mode ≠ Single Target).
target_options = "adhd_D_p"  #@param ['GAD_present', 'MDD_KSADS_C', 'OCD_present', 'adhd_D_p', 'anhedonia_B_k', 'anorexia_B_k', 'anxdisord_D_p', 'area_deprivation_idx', 'asd_ssrs_sum', 'asdadhd_c', 'bad_grades', 'bdefs_distract_upset_p', 'bulimia_B_k', 'compulsive_p', 'delta_anxdisord_D_p', 'delta_depress_D_p', 'delta_parent_depress_D_p', 'delta_parent_depressed_D_p', 'delta_parent_depressed_p', 'delta_parent_worries_a_lot_p', 'delta_sbt_core_depression', 'dep_decrease_1.5sd', 'dep_decrease_1sd', 'dep_decrease_2sd', 'dep_increase_1.5sd', 'dep_increase_1sd', 'dep_increase_2sd', 'dep_onset_rci_1.7', 'dep_onset_rci_1.96', 'dep_onset_rci_2.3', 'dep_remission_rci_1.7', 'dep_remission_rci_1.96', 'dep_remission_rci_2.3', 'depadhd_c', 'depress_D_p', 'depress_D_p_rev', 'depress_update_p', 'depressed_mood_B_k', 'doesnt_get_along_p', 'elevated_mood_B_k', 'enjoys_little_p', 'excessive_worry_B_k', 'external_D_p', 'fatigue_present_B_k', 'gd_riskybets', 'hopeless_B_k', 'hopeless_B_p', 'impulsive_p', 'internal_D_p', 'ksads_1_840_t', 'latent_class_depression', 'lying_B_k', 'nb_correct_nt_2back', 'not_liked_p', 'obsessions_p', 'ocd_D_p', 'pa_sum_k', 'parent_adhd_D_p', 'parent_anhedonia_B_p', 'parent_anxdisord_D_p', 'parent_avoidant_person_D_p', 'parent_bad_family_relationship_p', 'parent_bad_relationships_p', 'parent_concentration_issues_B_p', 'parent_concentration_trouble_p', 'parent_dep_decrease_1.5sd', 'parent_dep_decrease_2sd', 'parent_dep_increase_1.5sd', 'parent_dep_increase_2sd', 'parent_dep_onset_rci_1.96', 'parent_dep_onset_rci_2.3', 'parent_dep_remission_rci_1.96', 'parent_dep_remission_rci_2.3', 'parent_depress_D_p', 'parent_depressed_mood_B_p', 'parent_depressed_p', 'parent_distress_due_to_social_anxiety_B_p', 'parent_drinks_frequency_p', 'parent_drunk_days_p', 'parent_dysthymia_B_p', 'parent_easily_annoyed_B_p', 'parent_excessive_worry_B_p', 'parent_external_D_p', 'parent_friendship_trouble_p', 'parent_happy_person_p', 'parent_impulsive_p', 'parent_income', 'parent_internal_D_p', 'parent_low_self_esteem_B_p', 'parent_not_liked_by_others_p', 'parent_obsessions_present_B_p', 'parent_obsessive_thoughts_p', 'parent_repeats_acts_p', 'parent_sleep_problem_B_p', 'parent_social_anxiety_disorder_B_p', 'parent_socially_anxious_B_p', 'parent_suicidal_thoughts_p', 'parent_tobacco_use_frequency_p', 'parent_two_more_depression_B_p', 'parent_wish_dead_present_B_p', 'parent_worries_a_lot_p', 'perfectionist_p', 'ptsd_diagnosis_present_B_p', 'sbt_academic_cognitive', 'sbt_aggression_irritability', 'sbt_anxiety_depression', 'sbt_avoidance_fear', 'sbt_core_dep_decrease_1.5sd', 'sbt_core_dep_decrease_2sd', 'sbt_core_dep_increase_1.5sd', 'sbt_core_dep_increase_2sd', 'sbt_core_dep_onset_rci_1.7', 'sbt_core_dep_onset_rci_1.96', 'sbt_core_dep_onset_rci_2.3', 'sbt_core_dep_remission_rci_1.7', 'sbt_core_dep_remission_rci_1.96', 'sbt_core_dep_remission_rci_2.3', 'sbt_core_depression', 'sbt_core_depression_delta_t0_t3', 'sbt_core_depression_raw', 'sbt_emotional_dysregulation', 'sbt_fatigue_sleep', 'sbt_guilt_hopelessness', 'sbt_perfectionism_achievement', 'sbt_social_withdrawal', 'sbt_somatic_depression', 'sbt_well_being', 'sleep_problem_B_k', 'social_anxiety_disorder_B_k', 'social_problems_D_p', 'socially_anxious_B_k', 'suicidal_ideation_present_B_k', 'suicidal_p', 'suicide_attempt_past_B_k', 'suicide_attempt_present_B_k', 'tb_cryst', 'tb_flanker', 'tb_fluid', 'tb_reading', 'thought_disorder_D_p', 'top_10_depression', 'top_10_sbt_core_depression', 'top_5_depression', 'top_5_sbt_core_depression', 'two_more_depression_B_k', 'weight', 'wish_dead_present_B_k']
#@markdown **split**: *across_categories* = full predictor space; *within_categories* = loop over feature-set categories separately.
split = "within_categories" #@param ["across_categories", "within_categories"]
#@markdown **tp_option**: ABCD assessment wave -- 0 (baseline, ages 9-10), 1 (ages 10-11), 2 (ages 11-12), 3 (ages 12-13), 4 (ages 13-14).
tp_option = '0' #@param ["0", "1", "2", "3", "4", "All"]
#@markdown **feature_tp_option**: Timepoint to draw features from. *Match target* = same wave as tp_option (default; preserves standard behaviour). Set to a different wave for cross-timepoint prediction (e.g. T0 features → T2 target).
feature_tp_option = "Match target" #@param ["Match target", "0", "1", "2", "3", "4"]
#@markdown **subgroup**: Subsample filter. *Full Sample* = no filter.
subgroup = "Full Sample" #@param ["Full Sample", "High ALE", "High ALE + Severe Parent MH", "Low ALE", "Female", "Male"]

# ---
# 2. MODEL ARCHITECTURE
# ---
#@markdown ### 2 Model Architecture
#@markdown ---
#@markdown **ensemble_version**: Use 4-model stacking ensemble (True) or single base-learner (False, legacy).
ensemble_version = True #@param {"type": "boolean"}
#@markdown **ensemble_type**: *stacking* (reported in manuscript) or *bagging* (legacy, not recommended).
ensemble_type = 'stacking' #@param ["bagging", "stacking"]
#@markdown **skip_tabpfn**: Drop TabPFN from ensemble (~20-30% faster; auto-skipped for across_categories).
skip_tabpfn = False #@param {"type": "boolean"}

# ---
# 3. BATCH MODE
# ---
#@markdown ### 3 Batch Mode
#@markdown ---
#@markdown **batch_mode**: Run a predefined bundle of targets. Overrides target_options above.
#@markdown *Single Target* uses the target_options dropdown instead.
batch_mode = "ADHD/Impulsivity/Compulsivity" #@param ["Single Target", "Academic/Cognitive", "ADHD/Impulsivity/Compulsivity", "All Classification", "All Manuscript Figures (F3-F6)", "Child Classification -- All", "Child Classification -- Anxiety", "Child Classification -- Depression", "Child Classification -- Other", "Child Classification -- Suicidality", "Child Dep Classification", "Child Mood & Anxiety (with Deltas)", "Child Only Deltas", "Child P-Factor", "Child Suicide Ideation", "Parent P-Factor", "Parent Suicide Ideation", "Exhaustive Target Union", "F3: Child Psychopathology & Cognition", "F4: Depression & Anxiety Developmental Sweep", "F4c: Subjective vs. Objective Outcomes", "F5: Parent Depression Trajectory + T2 Contrasts", "F6: Child and Parent M/D & Deltas", "Mood/Depression Subtypes Children", "OCD/Impulsivity & Parent Substance Use", "Parent Classification -- All", "Parent Classification -- Anxiety", "Parent Classification -- Depression", "Parent Classification -- Other", "Parent Dep Classification", "Parent Only Deltas", "Parent Psychopathology (Mood/ADHD/Social)", "Parent Social/Impulsivity/Compulsivity", "Primary Child & Parent ADHD/Anxiety/Mood Targets", "SBT Core Triad", "Suicidal Ideation (Child & Parent Dyad)", "Child Depression Worsening", "Parent Depression Worsening", "Parent Targets Fx", "Primary Targets Fin", "Formal DepAnx NeuroCog", "Social Problems", "Child Social Problems Developmental Sweep", "Parent Social Problems Developmental Sweep", "T0 Cognitive Task Battery"]
#@markdown **run_all_grid**: Warning: full grid -- all tp × subgroup × target × split (~840 runs). Overrides everything above.
run_all_grid = False #@param {"type": "boolean"}
#@markdown ---
#@markdown #### Within-Categories Options *(only when split = within_categories)*
#@markdown **single_category_mode**: Run only the selected category below (skip the full category loop).
single_category_mode = False #@param {type: "boolean"}
#@markdown **isolate_combinations**: Run all pairwise feature-set combinations anchored on the selected category.
isolate_combinations = False #@param {"type": "boolean"}
#@markdown **isolate_category**: Category to target (used by single_category_mode and isolate_combinations).
isolate_category = "Cognitive Task Parameters" #@param ["None", "Inter", "Cognitive Task Parameters", "Dynamic Cognitive Control Parameters", "Delta Other Psychopathology", "Other Personality Features", "Medical/Somatic Problems", "Sleep Problems", "Diet/Nutrition", "Physical Activity/Features", "Technology Use", "Social Relationship Quality", "Family Dynamics & Parenting", "Parent Anxiety", "Parent Anxiety and Fear-Related Issues", "Parent Mood Issues", "Parent Impulsivity and Behavior Regulation", "Parent Social Functioning", "Parent Interpersonal Relationships and Social Functioning", "Parent Cognitive and Attention Issues", "Parent Personality", "Parent Delta Psychopathology", "Parent Other Psychopathology", "Parent Psychopathology Composite", "Parent Drug Use", "SES & Mobility", "Residential Characteristics", "School Dynamics", "Anxiety", "ADHD", "Externalizing", "Family Psychopathology Aggregated", "Family Drug Use", "Ancestral-Genetic PCs/Ethnicity", "Task and Resting State", "Structural and DTI", "Adverse Life Events", "Familial Adverse Life Events", "Both Adverse Life Events", "Parent/Child Adverse Life Events", "Religion", "Religion Proxy", "Cognitive Style", "Emotional Regulation Strategies", "Ethnicity/Nationality", "Child ADHD", "Child Anxiety", "Child Externalizing", "Child Cognitive Task Outcomes", "Child Cognitive Style", "Child Delta", "Child Emotional Regulation Strategies", "Child Mood Issues", "Child Other Personality Features", "Child Personality Features", "Child Medical/Somatic Problems", "Child Sleep Problems", "Child Physical Activity/Features", "Child Technology Use", "Child Social Relationship Quality", "Child Social Personality Psychopathology Sum", "Child School Dynamics"]

# -- Map UI names → internal variable names ---
_SUBGROUP_MAP = {
    "Full Sample": None,
    "High ALE": "high_ale",
    "High ALE + Severe Parent MH": "high_ale_severe_p_mh",
    "Low ALE": "low_ale",
    "Female": "Female",
    "Male": "Male",
}
group = _SUBGROUP_MAP.get(subgroup, None)
# Resolve feature timepoint: None = match target (default), int = cross-tp mode
# When tp_option="All", _feat_tp must be None (match target) so each
# TP iteration uses same-wave features. Explicit cross-tp feature
# selection is incompatible with the All-TP sweep.
if tp_option == "All" and feature_tp_option != "Match target":
    print("Warning:  feature_tp_option is ignored when tp_option='All' (features always match target wave).")
_feat_tp = None if (feature_tp_option == "Match target" or tp_option == "All") else int(feature_tp_option)
_tp_all_mode = (tp_option == "All")  # flag used by dispatch logic below
run_all = run_all_grid
isolate = isolate_combinations
imputer = "knn"  # legacy param (unused but preserved for safety)

_TARGET_BUNDLES = {
    # -- All Manuscript Figures (F3-F6) (figure order: F3 → F4 → F4b → F5 → F6) --
    "All Manuscript Figures (F3-F6)": [
        # F3: Child Psychopathology + Cognitive (T2)
        "adhd_D_p",
        "external_D_p",
        "internal_D_p",
        "social_problems_D_p",
        "bad_grades",
        "tb_flanker",
        "tb_reading",
        # F4: Mood & Anxiety across development
        "sbt_core_depression",
        "anxdisord_D_p",
        # F4b: Mixed Anxiety/Depression subtype
        "sbt_anxiety_depression",
        # F5: Parent targets
        "parent_depress_D_p",
        "parent_anxdisord_D_p",
        "parent_bad_relationships_p",
        "parent_happy_person_p",
        "parent_adhd_D_p",
        # F6: Additional operationalizations (child & parent)
        "top_5_sbt_core_depression",
        "delta_sbt_core_depression",
        "sbt_core_dep_onset_rci_2.3",
        "delta_parent_depressed_p",
        "delta_parent_worries_a_lot_p",
    ],
    "F3: Child Psychopathology & Cognition": [
        "suicidal_p",
        "adhd_D_p",
        "external_D_p",
        "internal_D_p",
        "social_problems_D_p",
        "bad_grades",
        "tb_flanker",
    ],
    "Child Mood & Anxiety (with Deltas)": [
        "sbt_core_depression",
        "delta_sbt_core_depression",
        "anxdisord_D_p",
        "delta_anxdisord_D_p",
    ],
    "Child P-Factor": [
        "internal_D_p",
        "external_D_p",
    ],
    "Parent P-Factor": [
        "parent_internal_D_p",
        "parent_external_D_p",
    ],
    "Child Suicide Ideation": [
        "suicidal_p",
    ],
    "Parent Suicide Ideation": [
        "parent_suicidal_thoughts_p",
    ],
    "Parent (ADHD/Social/DSyndrome)": [
        "parent_adhd_D_p",
        "parent_bad_relationships_p",
        "parent_depressed_p",
    ],
    "F6: Child and Parent M/D & Deltas": [
        "sbt_core_depression",
        "delta_sbt_core_depression",
        "delta_anxdisord_D_p",
        "parent_depress_D_p",
        "parent_happy_person_p",
        "delta_parent_depressed_p",
        "delta_parent_worries_a_lot_p",
        "parent_adhd_D_p",
        "parent_bad_relationships_p",
        "parent_depressed_p",
    ],
    "OCD/Impulsivity & Parent Substance Use": [
        "compulsions_p",
        'obsessions_p',
        'perfectionist_p',
        "impulsive_p",
        "parent_impulsive_p",
        "parent_tobacco_use_frequency_p",
        "parent_drinks_frequency_p",
    ],
    "Suicidal Ideation (Child & Parent Dyad)": [
        "suicidal_p",
        "parent_suicidal_thoughts_p",
    ],
    # -- F4c: Subjective vs. Objective Outcomes --
    # Depression, Anxiety, ADHD, Educational Performance + additional
    # objective targets (crystallized intelligence, body weight).
    "F4c: Subjective vs. Objective Outcomes": [
        "sbt_core_depression",
        "anxdisord_D_p",
        "adhd_D_p",
        "bad_grades",
        "tb_cryst",
        "weight",
    ],
    # -- Mood/Depression Subtypes Children --
    # All 12 SBT symptom-cluster subscales capturing distinct
    # dimensions of child mood/depression phenomenology.
    "Mood/Depression Subtypes Children": [
        "sbt_core_depression",
        "sbt_anxiety_depression",
        "sbt_guilt_hopelessness",
        "sbt_fatigue_sleep",
        "sbt_somatic_depression",
        "sbt_emotional_dysregulation",
        "sbt_social_withdrawal",
        "sbt_avoidance_fear",
        "sbt_aggression_irritability",
        "sbt_academic_cognitive",
        "sbt_perfectionism_achievement",
    ],
    # -- SBT Core Triad (Core Depression + Academic/Cognitive + Perfectionism) --
    "SBT Core Triad": [
        "sbt_core_depression",
        "sbt_academic_cognitive",
        "sbt_perfectionism_achievement",
    ],
    # -- Primary Child & Parent ADHD/Anxiety/Mood Targets --
    # Core triad (ADHD, anxiety, depression) for both child and parent.
    "Primary Child & Parent ADHD/Anxiety/Mood Targets": [
        # Child
        "adhd_D_p",
        "anxdisord_D_p",
        "sbt_core_depression",
        # Parent
        "parent_adhd_D_p",
        "parent_anxdisord_D_p",
        "parent_depressed_p",
    ],

    # -- Classification Operationalization Comparisons ---
    "Child Dep Classification": [
        "dep_onset_rci_2.3",
        "dep_onset_rci_1.96",
        "dep_onset_rci_1.7",
        "dep_increase_2sd",
        "dep_increase_1.5sd",
        "dep_increase_1sd",
        "dep_remission_rci_2.3",
        "dep_remission_rci_1.96",
        "dep_remission_rci_1.7",
        "dep_decrease_2sd",
        "dep_decrease_1.5sd",
        "dep_decrease_1sd",
        "top_10_depression",
        "top_5_depression",
        "latent_class_depression",
        "MDD_KSADS_C",
    ],
    "Parent Dep Classification": [
        "parent_dep_onset_rci_2.3",
        "parent_dep_onset_rci_1.96",
        "parent_dep_increase_2sd",
        "parent_dep_increase_1.5sd",
        "parent_dep_remission_rci_2.3",
        "parent_dep_remission_rci_1.96",
        "parent_dep_decrease_2sd",
        "parent_dep_decrease_1.5sd",
        "top_10_depression_parent",
        "top_5_depression_parent",
    ],
    # -- Child Only Deltas ---
    "Child Only Deltas": [
        "delta_depress_D_p",
        "delta_sbt_core_depression",
        "delta_anxdisord_D_p",
        "delta_adhd_D_p",
        "delta_social_problems_D_p",
        "delta_somatic_problems_D_p",
        "delta_bad_grades",
        "delta_family_conflict_ss_k",
    ],
    # -- Academic/Cognitive ---
    # Concurrent academic performance, change score in academic performance,
    # and NIH Toolbox crystallized intelligence composite. Captures the
    # objective (test-based) and subjective (grade-based) axes of academic/
    # cognitive functioning. All three are regression targets.
    "Academic/Cognitive": [
        "bad_grades",
        "delta_bad_grades",
        "tb_cryst",
    ],
    # -- ADHD/Impulsivity/Compulsivity ---
    # Child CBCL DSM-oriented ADHD composite + individual impulsivity item
    # (cbcl_q41) + individual compulsivity item (cbcl_q66). Captures attention/
    # hyperactive dysregulation alongside two narrower dyscontrol dimensions
    # (stop-failure vs repetitive-action). All three are regression targets.
    "ADHD/Impulsivity/Compulsivity": [
        "adhd_D_p",
        "impulsive_p",
        "compulsions_p",
    ],
    # -- Parent Social/Impulsivity/Compulsivity ---
    # Parent ASR analog of the child ADHD/Impulsivity/Compulsivity batch:
    # Avoidant Personality DSM composite (closest analog of CBCL Social
    # Problems syndrome at the parent self-report level), plus four ASR
    # item-level mirrors of the corresponding child CBCL items: not-liked-
    # by-others, impulsivity, repeats acts (compulsions), obsessive thoughts.
    # All five are regression targets. Excludes parent_adhd_D_p and parent
    # relationship-trouble items by design (already covered in prior batches).
    "Parent Social/Impulsivity/Compulsivity": [
        "parent_avoidant_person_D_p",
        "parent_not_liked_by_others_p",
        "parent_impulsive_p",
        "parent_repeats_acts_p",
        "parent_obsessive_thoughts_p",
    ],
    # -- Parent Only Deltas ---
    "Parent Only Deltas": [
        "delta_parent_depress_D_p",
        "delta_parent_sleep_trouble_p",
        "delta_parent_worries_about_family_p",
        "delta_parent_not_liked_by_others_p",
        "delta_parent_feels_unloved_p",
        "delta_family_conflict_ss_p",
        "delta_parent_worries_about_future_p",
        "delta_parent_concentration_trouble_p",
        "delta_parent_bad_relationships_p",
        "delta_parent_financial_trouble_p",
        "delta_parent_somatic_problems_D_p",
    ],

    # -- Child Depression Worsening (Classification) ---
    # Three threshold operationalisations of child depression worsening:
    # 1.5 SD, 2 SD, and RCI ≥ 2.3 (all binary classification targets).
    # Top 10th and 5th percentile cross-sectional severity cutoffs also included.
    "Child Depression Worsening": [
        "dep_increase_1.5sd",
        "dep_increase_2sd",
        "top_10_depression",
        "top_5_depression",
    ],

    # -- Parent Depression Worsening (Classification) ---
    # Three threshold operationalisations of parent depression worsening:
    # 1.5 SD, 2 SD, and RCI ≥ 2.3 (all binary classification targets).
    # Top 10th and 5th percentile cross-sectional severity cutoffs also included.
    "Parent Depression Worsening": [
        "parent_dep_increase_1.5sd",
        "parent_dep_increase_2sd",
        "top_10_depression_parent",
        "top_5_depression_parent",
    ],

    # -- Parent Targets Fx ---
    "Parent Targets Fx": [
        "parent_suicidal_thoughts_p",
        "parent_bad_relationships_p",
        "parent_adhd_D_p",
        "parent_happy_person_p",
        "parent_external_D_p",
        "parent_internal_D_p",
    ],

    # -- Child KSADS Classification ---
    "Child Classification -- All": [
        "anhedonia_B_k", "anorexia_B_k", "bulimia_B_k", "depressed_mood_B_k",
        "elevated_mood_B_k", "excessive_worry_B_k", "fatigue_present_B_k",
        "GAD_present", "hopeless_B_k", "ksads_1_840_t", "latent_class_depression",
        "lying_B_k", "MDD_KSADS_C", "OCD_present", "sleep_problem_B_k",
        "social_anxiety_disorder_B_k", "socially_anxious_B_k",
        "suicidal_ideation_present_B_k", "suicide_attempt_past_B_k",
        "suicide_attempt_present_B_k", "top_10_depression", "top_5_depression",
        "two_more_depression_B_k",
    ],
    "Child Classification -- Anxiety": [
        "excessive_worry_B_k", "GAD_present", "OCD_present",
        "social_anxiety_disorder_B_k", "socially_anxious_B_k",
    ],
    "Child Classification -- Depression": [
        "anhedonia_B_k", "depressed_mood_B_k", "elevated_mood_B_k",
        "fatigue_present_B_k", "hopeless_B_k", "ksads_1_840_t",
        "latent_class_depression", "MDD_KSADS_C",
        "top_10_depression", "top_5_depression", "two_more_depression_B_k",
    ],
    "Child Classification -- Other": [
        "anorexia_B_k", "bulimia_B_k", "lying_B_k", "sleep_problem_B_k",
    ],
    "Child Classification -- Suicidality": [
        "suicidal_ideation_present_B_k", "suicide_attempt_past_B_k",
        "suicide_attempt_present_B_k",
    ],

    # -- Parent KSADS Classification ---
    "Parent Classification -- All": [
        "hopeless_B_p", "parent_anhedonia_B_p", "parent_concentration_issues_B_p",
        "parent_depressed_mood_B_p", "parent_distress_due_to_social_anxiety_B_p",
        "parent_dysthymia_B_p", "parent_easily_annoyed_B_p",
        "parent_excessive_worry_B_p", "parent_low_self_esteem_B_p",
        "parent_obsessions_present_B_p", "parent_sleep_problem_B_p",
        "parent_social_anxiety_disorder_B_p", "parent_socially_anxious_B_p",
        "parent_two_more_depression_B_p", "ptsd_diagnosis_present_B_p",
    ],
    "Parent Classification -- Anxiety": [
        "parent_distress_due_to_social_anxiety_B_p", "parent_excessive_worry_B_p",
        "parent_obsessions_present_B_p", "parent_social_anxiety_disorder_B_p",
        "parent_socially_anxious_B_p",
    ],
    "Parent Classification -- Depression": [
        "hopeless_B_p", "parent_anhedonia_B_p", "parent_depressed_mood_B_p",
        "parent_dysthymia_B_p", "parent_low_self_esteem_B_p",
        "parent_two_more_depression_B_p",
    ],
    "Parent Classification -- Other": [
        "parent_concentration_issues_B_p", "parent_easily_annoyed_B_p",
        "parent_sleep_problem_B_p", "ptsd_diagnosis_present_B_p",
    ],

    # -- Combined ---
    "All Classification": [
        "anhedonia_B_k", "anorexia_B_k", "bulimia_B_k", "depressed_mood_B_k",
        "elevated_mood_B_k", "excessive_worry_B_k", "fatigue_present_B_k",
        "GAD_present", "hopeless_B_k", "hopeless_B_p", "ksads_1_840_t",
        "latent_class_depression", "lying_B_k", "MDD_KSADS_C", "OCD_present",
        "parent_anhedonia_B_p", "parent_concentration_issues_B_p",
        "parent_depressed_mood_B_p", "parent_distress_due_to_social_anxiety_B_p",
        "parent_dysthymia_B_p", "parent_easily_annoyed_B_p",
        "parent_excessive_worry_B_p", "parent_low_self_esteem_B_p",
        "parent_obsessions_present_B_p", "parent_sleep_problem_B_p",
        "parent_social_anxiety_disorder_B_p", "parent_socially_anxious_B_p",
        "parent_two_more_depression_B_p", "ptsd_diagnosis_present_B_p",
        "sleep_problem_B_k", "social_anxiety_disorder_B_k", "socially_anxious_B_k",
        "suicidal_ideation_present_B_k", "suicide_attempt_past_B_k",
        "suicide_attempt_present_B_k", "top_10_depression", "top_5_depression",
        "two_more_depression_B_k",
    ],

    # -- Primary Targets (Final Validation) ---
    # Matched child CBCL → parent ASR DSM-oriented scales + social/relational.
    "Primary Targets Fin": [
        # Child (CBCL DSM-oriented)
        "depress_D_p",
        "anxdisord_D_p",
        "adhd_D_p",
        "social_problems_D_p",
        # Parent (ASR analogues)
        "parent_depress_D_p",
        "parent_anxdisord_D_p",
        "parent_adhd_D_p",
        "parent_bad_relationships_p",
    ],

    # -- Social Problems (Cross-Child/Parent) ---
    # Matched child CBCL Social Problems items + parent ASR social/relational
    # targets. Designed for joint analysis across informant and instrument.
    # Pairs with the child/parent Social Problems Developmental Sweep bundles
    # in _TP_SWEEP_BUNDLES for longitudinal (T0-T4) Sankey rendering.
    "Social Problems": [
        # Child (CBCL item + DSM domain composite)
        "not_liked_p",
        "social_problems_D_p",
        # Parent (ASR item + relational composite)
        "parent_not_liked_by_others_p",
        "parent_bad_relationships_p",
    ],

    # -- Formal Depression & Anxiety NeuroCog Classification ---
    # Mixed child depression severity/diagnosis (CBCL percentile, KSADS)
    # + anxiety classification (KSADS GAD, excessive worry, social anxiety).
    # All targets are binary classification.
    "Formal DepAnx NeuroCog": [
        # Depression severity / diagnosis
        "top_10_depression",
        "top_5_depression",
        "two_more_depression_B_k",
        "MDD_KSADS_C",
        "anhedonia_B_k",
        # Anxiety classification
        "GAD_present",
        "excessive_worry_B_k",
        "social_anxiety_disorder_B_k",
    ],

    # -- T0 Cognitive Task Battery (Regression) ---
    # Seven non-redundant cognitive task targets covering distinct constructs
    # at baseline (T0): crystallized intelligence (verbal knowledge), fluid
    # intelligence (composite EF/speed/WM/episodic memory), Flanker (NIH
    # inhibitory control), SST SSRT (motor response inhibition), 2-back number
    # correct (emotional working memory), RAVLT long-delay total (verbal
    # episodic memory), and LMT efficiency (visuospatial mental rotation).
    # All targets are continuous regression and have full CIRCULAR_EXCLUSIONS
    # entries. T0 baseline coverage verified against CogTimePointsExample.csv.
    "T0 Cognitive Task Battery": [
        # NIH Toolbox composites
        "tb_cryst",
        "tb_fluid",
        # NIH Toolbox subscale (executive control)
        "tb_flanker",
        # fMRI behavioural -- response inhibition (SST)
        "sst_mean_ssrt",
        # fMRI behavioural -- emotional working memory (EN-back)
        "nb_correct_nt_2back",
        # Verbal episodic memory (RAVLT)
        "ravlt_l_total",
        # Visuospatial mental rotation (LMT)
        "lmt_efficiency",
    ],
}

# "full Target Union" is the deduplicated union of all other bundles.
# Built programmatically so adding a target to any bundle above
# automatically includes it here.
_TARGET_BUNDLES["Exhaustive Target Union"] = list(dict.fromkeys(
    t for bundle in _TARGET_BUNDLES.values() for t in bundle
))

# -- Timepoint-sweep bundles: (target, timepoint) tuples ---
# Unlike _TARGET_BUNDLES which run all targets at a single tp_option,
# these bundles override BOTH target and timepoint per iteration.
# Archive keys use f"{target}__tp{tp}" to avoid overwrites.
_TP_SWEEP_BUNDLES = {
    # -- F4: Depression & Anxiety Developmental Sweep --
    "F4: Depression & Anxiety Developmental Sweep": [
        (target, str(tp))
        for target in [
            "sbt_core_depression",
            "anxdisord_D_p",
        ]
        for tp in range(5)  # T0 through T4
    ],

    # -- F5: Parent Depression Trajectory + T2 Contrasts --
    "F5: Parent Depression Trajectory + T2 Contrasts": [
        # Primary: parent depression at T0, T2, T4
        ("parent_depress_D_p", "0"),
        ("parent_depress_D_p", "2"),
        ("parent_depress_D_p", "4"),
        # Secondary: contrast targets at T2 only
        ("parent_adhd_D_p", "2"),
        ("parent_bad_relationships_p", "2"),
        ("parent_anxdisord_D_p", "2"),
        ("parent_happy_person_p", "2"),
    ],

    # -- Child Social Problems Developmental Sweep --
    # CBCL item (not_liked_p) + DSM domain composite (social_problems_D_p)
    # across T0-T4. Use with sankey_style = "developmental_sweep".
    "Child Social Problems Developmental Sweep": [
        (target, str(tp))
        for target in [
            "not_liked_p",
            "social_problems_D_p",
        ]
        for tp in range(5)  # T0 through T4
    ],

    # -- Parent Social Problems Developmental Sweep --
    # ASR item (parent_not_liked_by_others_p) + relational composite
    # (parent_bad_relationships_p) across T0-T4. Use with sankey_style =
    # "parent_developmental_sweep".
    "Parent Social Problems Developmental Sweep": [
        (target, str(tp))
        for target in [
            "parent_not_liked_by_others_p",
            "parent_bad_relationships_p",
        ]
        for tp in range(5)  # T0 through T4
    ],
}

# -- Per-target FI barplot colours for batch mode ---
# When batch_mode != "Single Target", the batch loop overrides
# ensemble_barplot_color before each run_analysis() call so every
# target gets a visually distinct feature-importance palette.
#
# Convention:
# Child Psychopathology+Cog  -- domain-coded
# Child Dep and Anx          -- depression = blue, anxiety = red
# Parent Psychopathology     -- analogous hues to child counterparts
_BATCH_TARGET_COLORS = {
    # -- F3: Child Psychopathology & Cognition ---
    "internal_D_p":              "mediumpurple",   # internalizing → purple
    "adhd_D_p":                  "gold",           # ADHD → yellow
    "external_D_p":              "forestgreen",    # externalizing → green
    "social_problems_D_p":       "darkorange",     # social problems → orange
    "not_liked_p":               "chocolate",      # child CBCL social item → warm brown-orange
    "bad_grades":                "slategray",      # cognitive/academic → grey
    "tb_flanker":                "turquoise",      # cognitive/academic → turquoise (match Sankey)
    "suicidal_p":              "#0A0A3C", # suicidality → very dark blue
    "tb_reading":                "silver",          # cognitive/academic → light grey
    # -- Child Mood & Anxiety (with Deltas) ---
    "sbt_core_depression":       "royalblue",      # depression → blue
    "delta_sbt_core_depression": "deepskyblue",    # delta depression → lighter blue
    "anxdisord_D_p":             "crimson",        # anxiety → red
    "delta_anxdisord_D_p":       "tomato",         # delta anxiety → lighter red
    "sbt_anxiety_depression":    "orchid",         # mixed anx/dep → purple-pink
    "top_5_sbt_core_depression": "steelblue",      # severe depression → steel blue
    "sbt_core_dep_onset_rci_2.3":"cornflowerblue", # onset RCI → lighter cornflower
    # -- Parent Psychopathology (Mood/ADHD/Social) ---
    "parent_depress_D_p":            "navy",       # parent depression → darker blue
    "parent_depressed_p":            "midnightblue", # parent depressed item → deep blue
    "parent_anxdisord_D_p":          "firebrick",  # parent anxiety → darker red
    "parent_adhd_D_p":               "orange",     # parent ADHD → warm yellow
    "parent_bad_relationships_p":    "peru",       # parent social → warm brown/orange
    "parent_not_liked_by_others_p":  "darkcyan",   # parent ASR social item → teal (complements peru)
    "parent_happy_person_p":         "salmon",     # parent happiness → warm salmon
    "delta_parent_depressed_p":      "deepskyblue",# delta depression → lighter blue
    "delta_parent_worries_a_lot_p":  "tomato",     # delta worry/anx → lighter red
    # -- Child Depression Worsening ---
    # Three thresholds: graded blue -- softer for looser criterion, richer for stricter
    "dep_increase_1.5sd":           "steelblue",      # child ≥1.5SD → mid blue
    "dep_increase_2sd":             "dodgerblue",     # child ≥2SD → vivid cobalt
    "dep_onset_rci_2.3":            "royalblue",      # child RCI 2.3 → deep royal blue
    # -- Parent Depression Worsening ---
    # Parallel graded scheme, shifted darker to distinguish parent from child
    "parent_dep_increase_1.5sd":    "slateblue",      # parent ≥1.5SD → slate blue
    "parent_dep_increase_2sd":      "darkslateblue",  # parent ≥2SD → deep navy
    "parent_dep_onset_rci_2.3":     "midnightblue",   # parent RCI 2.3 → midnight blue
    # -- OCD/Impulsivity & Parent Substance Use ---
    "compulsions_p":                     "darkviolet",  # OCD compulsions → deep violet
    "obsessions_p":                      "darkorchid",  # OCD obsessions → adjacent orchid
    "perfectionist_p":                   "plum",        # OCD perfectionism → lighter purple
    "impulsive_p":                       "hotpink",     # impulsivity → high-energy pink
    "parent_impulsive_p":                "deeppink",    # parent analogue → darker pink
    "parent_tobacco_use_frequency_p":    "saddlebrown", # tobacco → earthy brown
    "parent_drinks_frequency_p":         "teal",        # alcohol → cool teal
    "parent_drunk_days_p":               "darkcyan",    # alcohol variant → adjacent cyan
    # -- Parent Social/Impulsivity/Compulsivity ---
    # Colors mirror the sankey palette for this batch (cell 8).
    # parent_not_liked_by_others_p and parent_impulsive_p already defined above.
    "parent_avoidant_person_D_p":        "#4B6BBE",     # parent ASR avoidant personality → steel blue-violet (matches sankey)
    "parent_repeats_acts_p":             "darkturquoise", # parent ASR compulsions item → turquoise (matches sankey, mirrors child compulsions_p)
    "parent_obsessive_thoughts_p":       "#8E5DBF",     # parent ASR obsessions item → violet (matches sankey, distinct from child darkorchid)
    # -- Suicidal Ideation (Child & Parent Dyad) ---
    "suicidal_p":                        "#0A0A3C", # very dark blue
    "parent_suicidal_thoughts_p":        "#0A0A3C", # very dark blue (match child)
    # -- Child P-Factor & Parent P-Factor ---
    "parent_internal_D_p":               "rebeccapurple", # parent internalizing → deeper purple (child = mediumpurple)
    "parent_external_D_p":               "seagreen",      # parent externalizing → muted green (child = forestgreen)
    # -- F4c: Subjective vs. Objective Outcomes ---
    "tb_cryst":                          "cadetblue",   # crystallized intelligence → cool blue-green
    "weight":                            "sienna",      # body weight → earthy brown
    # -- Mood/Depression Subtypes Children (SBT subscales) ---
    # sbt_core_depression already defined above as "royalblue"
    # sbt_anxiety_depression already defined above as "orchid"
    "sbt_guilt_hopelessness":            "indigo",          # guilt/hopelessness → deep blue-purple
    "sbt_fatigue_sleep":                 "slateblue",       # fatigue/sleep → cool muted tone
    "sbt_somatic_depression":            "rosybrown",       # somatic/physical → earthy flesh-adjacent
    "sbt_emotional_dysregulation":       "orangered",       # volatile affect → hot, unstable
    "sbt_social_withdrawal":             "cadetblue",       # retreat/isolation → cool, receding
    "sbt_avoidance_fear":                "mediumpurple",    # anxiety-adjacent → lighter purple
    "sbt_aggression_irritability":       "firebrick",       # anger/hostility → deep aggressive red
    "sbt_academic_cognitive":            "#B8960B", # academic/cognitive → golden amber (match Sankey)
    "sbt_perfectionism_achievement":     "#C44A80", # perfectionism/achievement → rose magenta (match Sankey)
    # -- Child Only Deltas ---
    "delta_adhd_D_p":                "gold",           # ADHD → warm yellow (matches adhd_D_p)
    "delta_family_conflict_ss_k":    "sienna",         # family conflict → burnt orange
    "delta_bad_grades":              "slategray",       # academic → grey (matches bad_grades)
    "delta_social_problems_D_p":     "darkorange",     # social → orange (matches social_problems_D_p)
    "delta_somatic_problems_D_p":    "peru",           # somatic → warm brown
    "delta_depress_D_p":             "royalblue",      # depression → blue
    # -- Parent Only Deltas ---
    "delta_parent_depress_D_p":          "navy",           # parent depression syndrome → deep blue
    "delta_parent_sleep_trouble_p":      "darkslateblue",  # sleep/insomnia → cool indigo
    "delta_parent_worries_about_family_p":"indianred",     # anxiety domain → warm red
    "delta_parent_worries_about_future_p":"firebrick",     # anxiety domain → deeper red
    "delta_parent_not_liked_by_others_p":"darkcyan",       # social domain → teal
    "delta_parent_feels_unloved_p":      "steelblue",      # depression/mood → muted blue
    "delta_family_conflict_ss_p":        "sienna",         # family conflict → burnt orange
    "delta_parent_concentration_trouble_p":"mediumpurple", # ADHD/attention → purple
    "delta_parent_bad_relationships_p":  "teal",           # social domain → teal-green
    "delta_parent_financial_trouble_p":  "dimgray",        # SES/financial → stone grey
    "delta_parent_somatic_problems_D_p": "peru",           # somatic complaints → warm brown
    # -- Formal DepAnx NeuroCog ---
    "top_10_depression":                 "cornflowerblue",  # depression severity → soft blue
    "top_5_depression":                  "royalblue",       # severe depression → richer blue
    "two_more_depression_B_k":           "steelblue",       # ≥2 symptoms → cool steel blue
    "MDD_KSADS_C":                       "midnightblue",    # MDD diagnosis → deepest blue
    "anhedonia_B_k":                     "slateblue",       # anhedonia → violet-blue
    "GAD_present":                       "crimson",         # GAD → anxiety red
    "excessive_worry_B_k":               "tomato",          # excessive worry → lighter red
    "social_anxiety_disorder_B_k":       "indianred",       # social anxiety → muted red
    # -- T0 Cognitive Task Battery ---
    # Palette aligned with the child_cogtask_t0 Sankey style, which itself
    # tracks the Objective/Subjective-Report Spectrum heatmap column hues.
    # tb_cryst already defined above as "cadetblue" (shared with F4c batch).
    # tb_flanker already defined above as "turquoise" (shared with F3 / child_alts).
    "tb_fluid":                          "midnightblue",    # NIH fluid composite → deep indigo navy (heatmap "Comp")
    "sst_mean_ssrt":                     "firebrick",       # SST response inhibition → dark red (heatmap "SST")
    "nb_correct_nt_2back":               "teal",            # emotional WM → dark teal (heatmap "N-Back")
    "ravlt_l_total":                     "royalblue",       # verbal episodic memory → royal blue (heatmap "RAVLT")
    "lmt_efficiency":                    "saddlebrown",     # visuospatial mental rotation → brown (heatmap "LMT")
}

# ---
# 4. SPEED & VISUALIZATION
# ---
#@markdown ### 4 Speed & Visualization
#@markdown ---
#@markdown **visualization_mode**: Controls which plots are generated during analysis.
#@markdown - **All Visuals**: FI bar plots + SHAP for all models + consensus + PDP + GAM
#@markdown - **Ensemble Consensus + Best SHAP**: Ensemble comparison + consensus + SHAP (best model only)
#@markdown - **Ensemble Plots Only**: Model comparison charts only -- fastest option with visuals
#@markdown - **No Visuals**: Metrics only -- maximum speed for validation/batch runs
visualization_mode = "Ensemble Consensus + Best SHAP" #@param ["All Visuals", "Ensemble Consensus + Best SHAP", "Ensemble Plots Only", "No Visuals"]

# -- Derive internal flags from dropdown ---
run_visualizations            = (visualization_mode != "No Visuals")
show_feature_importance_plots = (visualization_mode == "All Visuals")
_run_consensus                = visualization_mode in ("All Visuals", "Ensemble Consensus + Best SHAP")
_run_pdp_gam                  = (visualization_mode == "All Visuals")

_VIZ_ENABLED_MODELS = set()
if visualization_mode == "All Visuals":
    _VIZ_ENABLED_MODELS = {'catboost', 'xgboost', 'random_forest', 'tabpfn', 'meta_model'}
elif visualization_mode == "Ensemble Consensus + Best SHAP":
    # Include base models so the best-model SHAP selector has candidates
    _VIZ_ENABLED_MODELS = {'catboost', 'xgboost', 'random_forest', 'meta_model'}

_mode_summary = {
    "All Visuals":                    "FI + SHAP (all models) + consensus + PDP + GAM",
    "Ensemble Consensus + Best SHAP": "Ensemble plots + consensus + SHAP (best model)",
    "Ensemble Plots Only":            "Model comparison only -- fastest with visuals",
    "No Visuals":                     "Metrics only -- maximum speed",
}
print(f"[VIZ] {visualization_mode}: {_mode_summary[visualization_mode]}")
if _VIZ_ENABLED_MODELS:
    print(f"      SHAP/FI candidates: {sorted(_VIZ_ENABLED_MODELS)}")

# ---
# 5. VALIDATION (Cells 12-16)
# ---
#@markdown ### 5 Validation (Cells 12-16)
#@markdown ---
#@markdown **run_validation_cells**: Turn off to skip Cells 12-16 (nested CV, demographics, site robustness, permutation, stacking audit).
run_validation_cells = True #@param {"type": "boolean"}

if not run_validation_cells:
    print("[VAL] Validation cells (10-15) will be skipped.")
else:
    print("[VAL] Validation cells (10-15) enabled.")

# ---
# 6. PREPROCESSING
# ---
#@markdown ### 6 Preprocessing
#@markdown ---
#@markdown **deal_with_na_values**: Strategy for missing values in train/test features.
deal_with_na_values = "impute_median" #@param ["fill_with_zero", "drop", "impute_mean", "impute_median"]

# ---
# 7. ADVANCED / EXPERIMENTAL
# ---
#@markdown ### 7 Advanced / Experimental
#@markdown ---
#@markdown **hyperparameter_tuning**: Enable Optuna-based HP search for base learners (slow).
hyperparameter_tuning = False #@param {"type": "boolean"}
#@markdown **with_optuna**: Use Optuna for meta-learner tuning (requires hyperparameter_tuning).
with_optuna = False #@param {"type": "boolean"}
num_of_optuna_trials = 5 #@param {type: "integer"}
#@markdown **selection_method**: Feature selection before model fitting. *None* = skip.
selection_method = 'None' #@param ['rfecv', 'interaction', 'h-stats', 'clustering', 'None']
#@markdown **conformal_prediction**: Add conformal prediction intervals to outputs.
conformal_prediction = False #@param {"type": "boolean"}
#@markdown **quantile_segment**: Segment predictions by target-variable quantile bins.
quantile_segment = False #@param {"type": "boolean"}
#@markdown **simple_test_model**: Replace ensemble with a single linear model (lasso or ridge).
simple_test_model = False #@param {"type": "boolean"}
model_type = "lasso" #@param ["lasso", "ridge"]

# -- Apply skip_tabpfn: remove TabPFN from any model_configs built below --
# TabPFN is auto-excluded for across-category analyses (d≈500-600,
# N×d ≈ 4.5M far exceeds TabPFN's d≤500 / N×d≤1M sweet spot).
# Within-category (d≈20-150) remains in TabPFN's optimal range.
_SKIP_TABPFN = skip_tabpfn or (split == 'across_categories')
if _SKIP_TABPFN:
    _tabpfn_reason = []
    if skip_tabpfn:
        _tabpfn_reason.append('skip_tabpfn=True')
    if split == 'across_categories':
        _tabpfn_reason.append(f'split={split!r} (d too high for TabPFN)')
    print(f"[MODEL] TabPFN excluded from ensemble ({'; '.join(_tabpfn_reason)}).")

from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV

# Function to create and tune Lasso or Ridge regression model
def create_tuned_linear_model(model_type, X_train, y_train, alpha_range=None):
    """
    Create and tune Lasso or Ridge regression model.

    Parameters:
    model_type : str
        'lasso' or 'ridge'
    X_train : DataFrame
        Training features
    y_train : Series/array
        Training target
    alpha_range : list, optional
        Range of alpha values to search. If None, uses default range.

    Returns:
    --------
    best_model : sklearn model
        Tuned Lasso or Ridge model
    """
    if alpha_range is None:
        alpha_range = np.logspace(-4, 1, 20)

    if model_type == 'lasso':
        model = Lasso(random_state=42, max_iter=10000)
    else:  # ridge
        model = Ridge( max_iter=10000)

    param_grid = {'alpha': alpha_range}

    grid_search = GridSearchCV(
        model, param_grid, cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    print(f"Best alpha for {model_type}: {grid_search.best_params_['alpha']}")
    print(f"Best score: {-grid_search.best_score_:.4f} (MSE)")

    return grid_search.best_estimator_

# Function to run linear model analysis with visualization
def run_linear_model_analysis(model_type, X_train, X_test, y_train, y_test, target_options, tp, category=None):
    """
    Run analysis with Lasso or Ridge regression

    Parameters:
    model_type : str
        'lasso' or 'ridge'
    X_train, X_test : DataFrame
        Training and test features
    y_train, y_test : Series/array
        Training and test targets
    target_options, tp, category : str/int
        Target variable, time point, and category for reporting
    """
    print(f"\nRunning {model_type.upper()} Regression Analysis")

    # Train model with cross-validation to select alpha
    model = create_tuned_linear_model(model_type, X_train, y_train)

    # Get coefficients
    coefficients = pd.Series(model.coef_, index=X_train.columns)

    # Plot top coefficients
    plt.figure(figsize=(12, 8))
    top_coefficients = coefficients.abs().sort_values(ascending=False).head(20)
    top_coefficients = coefficients[top_coefficients.index]

    sns.barplot(x=top_coefficients.values, y=top_coefficients.index, palette='coolwarm')
    title = f"Top 20 {model_type.capitalize()} Coefficients ({target_options}, Timepoint {tp}"
    title += f", Category {category})" if category else ")"
    plt.title(title, fontsize=16)
    plt.xlabel("Coefficient Value", fontsize=12)
    plt.ylabel("Features", fontsize=12)
    plt.tight_layout()
    plt.show()

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Display metrics
    evaluation_metrics = pd.DataFrame({
        'Metric': ['MSE', 'RMSE', 'MAE', 'R² Score'],
        'Value': [mse, rmse, mae, r2]
    })

    print("\nEvaluation Metrics:")
    display(evaluation_metrics)

    # Metrics visualization
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(evaluation_metrics.set_index('Metric').T, annot=True, fmt='.4f', cmap='coolwarm', linewidths=1.5, cbar=False)
    ax.set_title(f'Evaluation Metrics for {target_options} (Timepoint {tp})', fontsize=16)
    plt.tight_layout()
    plt.show()

    # Scatter plot of actual vs predicted
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.title(f'{model_type.capitalize()} Regression: Actual vs Predicted')
    plt.tight_layout()
    plt.show()

    # Add to category results for spider plot (for regression)
    if category is not None:
        if category not in category_results:
            category_results[category] = {}

        category_results[category]['score'] = r2
        # Calculate confidence intervals using bootstrap
        bootstrap_r2_scores = []
        n_bootstraps = 100

        for _ in range(n_bootstraps):
            indices = np.random.choice(len(y_test), len(y_test), replace=True)
            bootstrap_r2 = r2_score(y_test.iloc[indices], y_pred[indices])
            bootstrap_r2_scores.append(bootstrap_r2)

        ci = 1.96 * np.std(bootstrap_r2_scores)
        category_results[category]['score_lower'] = r2 - ci
        category_results[category]['score_upper'] = r2 + ci
        category_results[category]['r2'] = r2

    # Return model and metrics
    return {
        'model': model,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'coefficients': coefficients
    }

def ensure_unique_feature_names(df):
    """
    Ensure all feature names in the DataFrame are unique by appending a suffix to duplicates.
    """
    feature_names = list(df.columns)
    unique_names = []
    name_counts = {}

    for name in feature_names:
        if name in name_counts:
            name_counts[name] += 1
            unique_names.append(f"{name}_{name_counts[name]}")
        else:
            name_counts[name] = 0
            unique_names.append(name)

    df.columns = unique_names
    return df

# Winsorise numeric features at ±n_sigmas to limit the influence of
# extreme values on the scaler and downstream learners.
def handle_outliers(X, n_sigmas=5):
    for column in X.select_dtypes(include=[np.number]).columns:
        mean = X[column].mean()
        std = X[column].std()
        X[column] = X[column].clip(lower=mean - n_sigmas*std, upper=mean + n_sigmas*std)
    return X

# Resolve NaN and infinite values in both X and y.
# Methods: 'remove' (listwise deletion), 'fill_0' (zero-fill),
# 'impute' (median/mean via SimpleImputer).
# y can arrive as either a Series (from within_categories scale_y) or a
# single-column DataFrame; both are handled transparently.
def clean_data(X, y, method='fill_0', imputer_strategy=None):
    # Normalise y to DataFrame so axis=1 operations and .columns work
    # uniformly; restore original type before returning.
    _y_was_series = isinstance(y, pd.Series)
    _y_series_name = y.name if _y_was_series else None
    if _y_was_series:
        y = y.to_frame(name=_y_series_name or 'target')

    if method == 'remove':
        # Remove rows with infinite values
        mask_X = ~np.isinf(X).any(axis=1)
        mask_y = ~np.isinf(y).any(axis=1)
        mask = mask_X & mask_y

        # Remove rows with NaN values
        mask_X = ~np.isnan(X).any(axis=1)
        mask_y = ~np.isnan(y).any(axis=1)
        mask = mask & mask_X & mask_y

        X_clean = X[mask]
        y_clean = y[mask]

    elif method == 'fill_0':
        # Replace infinite values with 0
        X_clean = X.replace([np.inf, -np.inf], 0)
        y_clean = y.replace([np.inf, -np.inf], 0)

        # Replace NaN values with 0
        X_clean = X_clean.fillna(0)
        y_clean = y_clean.fillna(0)

    elif method == 'impute':
        # Replace infinite values with NaN
        X_clean = X.replace([np.inf, -np.inf], np.nan)
        y_clean = y.replace([np.inf, -np.inf], np.nan)

        # Impute NaN values
        if imputer_strategy in ['mean', 'median', 'most_frequent']:
            imputer = SimpleImputer(strategy=imputer_strategy)
            X_clean = pd.DataFrame(imputer.fit_transform(X_clean), columns=X_clean.columns)

            # Fit a separate imputer for y to avoid data leakage from X
            y_imputer = SimpleImputer(strategy=imputer_strategy)
            y_clean = pd.DataFrame(y_imputer.fit_transform(y_clean), columns=y_clean.columns)

        else:
            raise ValueError("Invalid imputer_strategy. Choose 'mean', 'median', or 'most_frequent'.")

    else:
        raise ValueError("Invalid method. Choose 'remove', 'fill_0', or 'impute'.")

    # Restore y to its original type
    if _y_was_series and isinstance(y_clean, pd.DataFrame):
        y_clean = y_clean.iloc[:, 0]
        y_clean.name = _y_series_name

    return X_clean, y_clean

class LightweightInteractionAnalyzer:
    def __init__(self, model, metric='rmse', n_jobs=-1):
        """
        Initialize the analyzer

        Parameters:
        model : fitted model object
            Should have predict method
        metric : str, default='rmse'
            Metric to use for interaction strength ('rmse' or 'r2')
        n_jobs : int, default=-1
            Number of jobs for parallel processing
        """
        self.model = model
        self.metric = metric
        self.n_jobs = n_jobs
        self.interaction_scores = {}

    def _calculate_feature_importance(self, X, y, feature_name):
        """Calculate single feature importance using permutation"""
        baseline_pred = self.model.predict(X)
        baseline_score = root_mean_squared_error(y, baseline_pred)

        X_permuted = X.copy()
        X_permuted[feature_name] = np.random.permutation(X_permuted[feature_name])
        permuted_pred = self.model.predict(X_permuted)
        permuted_score = root_mean_squared_error(y, permuted_pred)

        return abs(permuted_score - baseline_score)

    def _calculate_interaction_strength(self, X, y, feat1, feat2):
        """Calculate interaction strength between two features"""
        # Calculate individual importance
        imp1 = self._calculate_feature_importance(X, y, feat1)
        imp2 = self._calculate_feature_importance(X, y, feat2)

        # Calculate joint importance
        X_permuted = X.copy()
        X_permuted[feat1] = np.random.permutation(X_permuted[feat1])
        X_permuted[feat2] = np.random.permutation(X_permuted[feat2])

        baseline_pred = self.model.predict(X)
        baseline_score = root_mean_squared_error(y, baseline_pred)

        permuted_pred = self.model.predict(X_permuted)
        joint_score = root_mean_squared_error(y, permuted_pred)

        # Interaction strength is the difference between joint importance
        # and the sum of individual importances
        interaction = abs(joint_score - baseline_score) - (imp1 + imp2)
        return max(interaction, 0)  # non-negative

    def analyze_interactions(self, X, y, max_features=10):
        """
        Analyze feature interactions

        Parameters:
        X : pandas DataFrame
            Input features
        y : array-like
            Target variable
        max_features : int, default=10
            Maximum number of top features to consider for interactions
        """
        print("Calculating feature importances...")
        # First, get individual feature importances
        importances = {}
        for feature in tqdm(X.columns):
            importances[feature] = self._calculate_feature_importance(X, y, feature)

        # Select top features based on individual importance
        top_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)
        top_features = [f[0] for f in top_features[:max_features]]

        print("\nAnalyzing feature interactions...")
        # Calculate interactions between top features
        for feat1, feat2 in tqdm(list(combinations(top_features, 2))):
            interaction_strength = self._calculate_interaction_strength(X, y, feat1, feat2)
            self.interaction_scores[(feat1, feat2)] = interaction_strength

        return self.interaction_scores

    def plot_interactions(self, X, top_n=10):
        """Plot interaction heatmap"""
        # Create interaction matrix
        features = list(set([f for pair in self.interaction_scores.keys() for f in pair]))
        n_features = len(features)
        interaction_matrix = np.zeros((n_features, n_features))

        for (feat1, feat2), strength in self.interaction_scores.items():
            i = features.index(feat1)
            j = features.index(feat2)
            interaction_matrix[i, j] = strength
            interaction_matrix[j, i] = strength

        # Plot heatmap
        plt.figure(figsize=(12, 8))
        sns.heatmap(
            interaction_matrix,
            xticklabels=features,
            yticklabels=features,
            cmap='YlOrRd'
        )
        plt.title('Feature Interaction Strengths')
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

        # Plot top interactions as bar chart
        sorted_interactions = sorted(
            self.interaction_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:top_n]

        plt.figure(figsize=(12, 6))
        interaction_names = [' × '.join(k) for k, _ in sorted_interactions]
        interaction_strengths = [v for _, v in sorted_interactions]

        plt.bar(range(len(sorted_interactions)), interaction_strengths)
        plt.xticks(
            range(len(sorted_interactions)),
            interaction_names,
            rotation=45,
            ha='right'
        )
        plt.title('Top Feature Interactions')
        plt.ylabel('Interaction Strength')
        plt.tight_layout()
        plt.show()

def analyze_model_interactions(model, X_train, X_test, y_train, max_features=10):
    """Analyze feature interactions using lightweight methods"""
    analyzer = LightweightInteractionAnalyzer(model)
    interaction_scores = analyzer.analyze_interactions(X_train, y_train, max_features=max_features)
    analyzer.plot_interactions(X_train)
    return interaction_scores

# Optuna objective
def objective(trial, X, y, is_reg, num_classes):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 100, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 1e-7, 1, log=True),
        'random_state': 42
    }

    if is_reg:
        model = CatBoostRegressor(**params)
        eval_metric = 'RMSE'
    else:
        if num_classes > 2:
            params['loss_function'] = 'MultiClass'
            eval_metric = 'MultiClass'
        else:
            params['loss_function'] = 'Logloss'
            eval_metric = 'Logloss'
        model = CatBoostClassifier(**params)

    pruning_callback = CatBoostPruningCallback(trial, eval_metric)
    model.fit(X, y, eval_set=[(X, y)], early_stopping_rounds=100, verbose=False, callbacks=[pruning_callback])

    if is_reg:
        return model.best_score_['validation'][eval_metric]
    else:
        return model.best_score_['validation'][eval_metric]

# Interaction feature constructor
def create_interaction_features(X, max_interactions=1000):
    initial_features = X.columns
    n_features = len(initial_features)

    # Calculate total possible interactions
    total_interactions = n_features * (n_features - 1) // 2

    # Generate all combinations
    all_combinations = list(combinations(range(n_features), 2))

    # If total interactions exceed max_interactions, randomly select interactions
    if total_interactions > max_interactions:
        indices = np.random.RandomState(42).permutation(total_interactions)[:max_interactions]
        feature_pairs = [all_combinations[i] for i in indices]
    else:
        feature_pairs = all_combinations

    # Create interaction features
    interaction_features = {}
    for i, j in feature_pairs:
        feat_A, feat_B = initial_features[i], initial_features[j]
        interaction_name = f'{feat_A}_x_{feat_B}'
        interaction_features[interaction_name] = X.iloc[:, i] * X.iloc[:, j]

    return pd.DataFrame(interaction_features)

def visualize_feature_importance(X_train, y_train, top_n=20):
    """
    Create visualizations for feature importance based on correlation with target

    Parameters:
    X_train : pandas DataFrame
        Training features including interaction terms
    y_train : pandas Series
        Target variable
    top_n : int
        Number of top features to show in plots
    """
    # y_train is a Series with a name
    if isinstance(y_train, pd.DataFrame):
        y_train = y_train.iloc[:, 0]
    if not isinstance(y_train, pd.Series):
        y_train = pd.Series(y_train)
    if y_train.name is None:
        y_train.name = 'target'

    # Calculate correlations with target
    correlations = X_train.apply(lambda x: abs(x.corr(y_train)) if x.name != y_train.name else 0)

    # Sort correlations
    top_correlations = correlations.sort_values(ascending=False).head(top_n)

    # Create figure with subplots
    plt.figure(figsize=(15, 10))

    # 1. Bar plot of top feature correlations
    plt.subplot(2, 1, 1)
    sns.barplot(x=top_correlations.values, y=top_correlations.index)
    plt.title(f'Top {top_n} Feature Correlations with Target')
    plt.xlabel('Absolute Correlation')
    plt.ylabel('Features')

    # Separate interaction terms and original features
    interaction_features = [f for f in top_correlations.index if '_x_' in f]
    original_features = [f for f in top_correlations.index if '_x_' not in f]

    # Create color mapping
    colors = ['#ff9999' if '_x_' in f else '#66b3ff' for f in top_correlations.index]

    # 2. Horizontal bar plot with color-coded features
    plt.subplot(2, 1, 2)
    bars = plt.barh(range(len(top_correlations)), top_correlations.values, color=colors)
    plt.yticks(range(len(top_correlations)), top_correlations.index)
    plt.xlabel('Absolute Correlation')
    plt.title('Feature Importance (Blue: Original Features, Red: Interaction Terms)')

    plt.tight_layout()
    plt.show()

def visualize_interaction_matrix(X_train, y_train, top_n=20):
    """
    Create a heatmap of feature interactions

    Parameters:
    X_train : pandas DataFrame
        Training features including interaction terms
    y_train : pandas Series
        Target variable
    top_n : int
        Number of top features to include in heatmap
    """
    # Calculate correlations with target
    correlations = X_train.apply(lambda x: abs(x.corr(y_train)))

    # Get top features
    top_features = correlations.nlargest(top_n).index

    # Calculate correlation matrix for top features
    correlation_matrix = X_train[top_features].corr()

    # Create heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix,
                annot=True,
                cmap='coolwarm',
                center=0,
                fmt='.2f',
                square=True)
    plt.title('Feature Correlation Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def calculate_safe_correlations(X_train, y_train):
    # Convert y_train to 1D array if it's not already
    if isinstance(y_train, pd.DataFrame):
        y_train = y_train.iloc[:, 0].values
    elif isinstance(y_train, pd.Series):
        y_train = y_train.values
    y_train = y_train.ravel()

    correlations = {}

    # Calculate correlations one at a time
    for feature in X_train.columns:
        feature_values = X_train[feature].values.ravel()
        # Remove any NaN values
        mask = ~(np.isnan(feature_values) | np.isnan(y_train))
        if mask.sum() > 0:
            correlation = np.corrcoef(feature_values[mask], y_train[mask])[0, 1]
            correlations[feature] = abs(correlation) if not np.isnan(correlation) else 0
        else:
            correlations[feature] = 0

    return pd.Series(correlations)

def analyze_and_visualize_interactions(X_train, y_train, top_n=20):
    # 1. Feature importance visualization
    print("Plotting feature importance...")
    visualize_feature_importance(X_train, y_train, top_n)

    # 2. Interaction matrix visualization
    print("\nPlotting interaction matrix...")

    # 3. Print top interaction terms
    interaction_features = [f for f in X_train.columns if '_x_' in f]

    if interaction_features:

        X_interactions = X_train[interaction_features]
        correlations = calculate_safe_correlations(X_interactions, y_train)

        top_interactions = correlations.sort_values(ascending=False).head(top_n)

        print("\nTop 10 Interaction Features:")
        print("-" * 50)
        for feat, corr in top_interactions.items():
            # Split feature name for better readability
            feat1, feat2 = feat.split('_x_')
            print(f"{feat1} × {feat2}: {corr:.4f}")

        # Optional: Create visualization
        plt.figure(figsize=(10, 6))
        plt.bar(range(len(top_interactions)), top_interactions.values)
        plt.xticks(range(len(top_interactions)),
                  [f"{f.split('_x_')[0]}\n×\n{f.split('_x_')[1]}" for f in top_interactions.index],
                  rotation=45)
        plt.title('Top Feature Interactions by Correlation with Target')
        plt.ylabel('Absolute Correlation')
        plt.tight_layout()
        plt.show()


    return None

def plot_specific_interaction(model, X, feature1, feature2, n_samples=1000):
    """Plot detailed interaction between two specific features."""
    if len(X) > n_samples:
        X = X.sample(n_samples, random_state=42)

    # Create 2D grid of feature values
    x1_range = np.linspace(X[feature1].min(), X[feature1].max(), 20)
    x2_range = np.linspace(X[feature2].min(), X[feature2].max(), 20)
    xx1, xx2 = np.meshgrid(x1_range, x2_range)

    # Create predictions for grid
    grid_data = X.copy()
    predictions = np.zeros((20, 20))

    for i in range(20):
        for j in range(20):
            grid_data[feature1] = xx1[i, j]
            grid_data[feature2] = xx2[i, j]
            predictions[i, j] = model.predict(grid_data).mean()

    # Plot interaction
    plt.figure(figsize=(10, 8))
    plt.contourf(xx1, xx2, predictions, levels=20, cmap='viridis')
    plt.colorbar(label='Predicted Value')
    plt.xlabel(feature1)
    plt.ylabel(feature2)
    plt.title(f'Interaction Effect: {feature1} × {feature2}')
    plt.tight_layout()
    plt.show()

def calculate_h_statistics_batch(model, X, feature_pairs, n_samples=1000, batch_size=100):
    """Calculate H-statistics for multiple feature pairs in batches."""
    if len(X) > n_samples:
        X = X.sample(n_samples, random_state=42)

    results = []

    # Process feature pairs in batches
    for i in range(0, len(feature_pairs), batch_size):
        batch_pairs = feature_pairs[i:i + batch_size]
        batch_results = []

        # Get base predictions once for this batch (each batch one base)
        base_pred = model.predict(X)

        for feat1, feat2 in batch_pairs:
            # Create all permutations at once
            X_perm = X.copy()
            X_perm[feat1] = np.random.permutation(X_perm[feat1].values)
            pred1 = model.predict(X_perm)

            X_perm[feat2] = np.random.permutation(X_perm[feat2].values)
            pred_both = model.predict(X_perm)

            X_perm = X.copy()
            X_perm[feat2] = np.random.permutation(X_perm[feat2].values)
            pred2 = model.predict(X_perm)

            # Calculate H-statistic
            effect1 = np.mean(np.abs(base_pred - pred1))
            effect2 = np.mean(np.abs(base_pred - pred2))
            effect_both = np.mean(np.abs(base_pred - pred_both))

            h_stat = max(0, effect_both - (effect1 + effect2))
            batch_results.append(h_stat)

        results.extend(batch_results)

    return results

def select_features_with_h_statistics(X_train, X_test, y_train, is_reg=True, max_features=1000, top_interactions=50):
    """
    Select features based on H-statistics for interaction strength.

    Parameters:
    X_train : pandas DataFrame
        Training features
    X_test : pandas DataFrame
        Test features
    y_train : array-like
        Training target
    is_reg : bool
        Whether this is a regression (True) or classification (False) problem
    max_features : int
        Maximum number of features to return after selection
    top_interactions : int
        Number of top interactions to consider for feature creation

    Returns:
    --------
    X_train_selected : pandas DataFrame
        Training data with selected original and interaction features
    X_test_selected : pandas DataFrame
        Test data with selected original and interaction features
    """

    if is_reg:
        model = CatBoostRegressor(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            verbose=False,
            random_state=42
        )
    else:
        model = CatBoostClassifier(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            verbose=False,
            random_state=42
        )

    print("Training initial model...")
    model.fit(X_train, y_train)

    # Generate feature pairs
    features = list(X_train.columns)
    feature_pairs = [(features[i], features[j])
                    for i in range(len(features))
                    for j in range(i + 1, len(features))]

    # Calculate H-statistics with batching and sampling
    print("Calculating H-statistics for feature pairs...")
    h_stats = calculate_h_statistics_batch(model, X_train, feature_pairs,
                                         n_samples=500,  # Reduced sample size
                                         batch_size=100)  # Batch processing

    # Create interaction strengths list
    interaction_strengths = [(pair[0], pair[1], stat)
                           for pair, stat in zip(feature_pairs, h_stats)]

    # Sort by interaction strength
    interaction_strengths.sort(key=lambda x: x[2], reverse=True)

    # Create interaction features for top pairs
    print(f"\nCreating interaction features for top {top_interactions} pairs...")
    X_train_new = X_train.copy()
    X_test_new = X_test.copy()

    for feat1, feat2, strength in interaction_strengths[:top_interactions]:
        feat_name = f"{feat1}_x_{feat2}"
        X_train_new[feat_name] = X_train[feat1] * X_train[feat2]
        X_test_new[feat_name] = X_test[feat1] * X_test[feat2]

    # Calculate feature importance including new interaction features
    print("Calculating final feature importance...")
    model.fit(X_train_new, y_train)
    feature_importance = model.get_feature_importance()

    # Select top features based on importance
    feature_ranks = pd.Series(feature_importance, index=X_train_new.columns)
    selected_features = feature_ranks.nlargest(max_features).index

    # Visualize feature importance and interactions
    plt.figure(figsize=(15, 10))

    # Plot feature importance
    plt.subplot(2, 1, 1)
    top_20_features = feature_ranks.nlargest(20)
    sns.barplot(x=top_20_features.values, y=top_20_features.index)
    plt.title('Top 20 Features (Including Interactions)')
    plt.xlabel('Feature Importance')

    # Plot top interactions
    plt.subplot(2, 1, 2)
    top_interactions_df = pd.DataFrame(
        interaction_strengths[:10],
        columns=['Feature 1', 'Feature 2', 'H-Statistic']
    )
    sns.barplot(
        data=top_interactions_df,
        x='H-Statistic',
        y=top_interactions_df.apply(lambda x: f"{x['Feature 1']}\n×\n{x['Feature 2']}", axis=1)
    )
    plt.title('Top 10 Feature Interactions (H-Statistics)')
    plt.tight_layout()
    plt.show()

    # Create heatmap of top interactions
    n_top = 10  # Number of top features to show in heatmap
    top_features = list(set([f for pair in interaction_strengths[:n_top] for f in pair[:2]]))
    n_features = len(top_features)
    interaction_matrix = np.zeros((n_features, n_features))

    # Fill interaction matrix
    for i, feat1 in enumerate(top_features):
        for j, feat2 in enumerate(top_features):
            if i < j:
                # Find this pair in interaction_strengths if it exists
                strength = next((s for f1, f2, s in interaction_strengths
                               if (f1 == feat1 and f2 == feat2) or
                                  (f1 == feat2 and f2 == feat1)), 0)
                interaction_matrix[i, j] = strength
                interaction_matrix[j, i] = strength

    # Plot heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(interaction_matrix,
              xticklabels=top_features,
              yticklabels=top_features,
              cmap='YlOrRd',
              annot=True,
              fmt='.2e',
              annot_kws={'size': 6})  # Smaller font for in-cell heatmap annotations

    plt.title('Feature Interaction Strength Heatmap (H-Statistics)')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Print top interaction details
    print("\nTop 10 Feature Interactions (H-Statistics):")
    print("-" * 50)
    for feat1, feat2, strength in interaction_strengths[:10]:
        print(f"{feat1} × {feat2}: {strength:.4e}")

    # Return selected features
    X_train_selected = X_train_new[selected_features]
    X_test_selected = X_test_new[selected_features]

    print(f"\nFinal number of features: {len(selected_features)}")
    print(f"Number of interaction features: {sum('_x_' in f for f in selected_features)}")

    return X_train_selected, X_test_selected

def select_features_with_shap_interactions(X_train, X_test, y_train, split, is_reg=True, max_features=1000, top_interactions=50):
    """
    Select features based on SHAP interaction values.

    Parameters:
    X_train : pandas DataFrame
        Training features
    X_test : pandas DataFrame
        Test features
    y_train : array-like
        Training target
    is_reg : bool
        Whether this is a regression (True) or classification (False) problem
    max_features : int
        Maximum number of features to return after selection
    top_interactions : int
        Number of top interactions to consider for feature creation

    Returns:
    --------
    X_train_selected : pandas DataFrame
        Training data with selected original and interaction features
    X_test_selected : pandas DataFrame
        Test data with selected original and interaction features
    """

    import matplotlib.pyplot as plt
    import networkx as nx

    # Train a simple CatBoost model for SHAP analysis
    if is_reg:
        model = CatBoostRegressor(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            verbose=False,
            random_state=42
        )
    else:
        model = CatBoostClassifier(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            verbose=False,
            random_state=42
        )

    # Fit the model
    print("Training initial model for SHAP analysis...")
    model.fit(X_train, y_train)

    # Calculate SHAP interaction values
    print("Calculating SHAP interaction values...")
    explainer = shap.TreeExplainer(model)

    # Use a sample of the training data for interaction calculations
    sample_size = min(1000, len(X_train))
    X_train_sample = X_train.iloc[:sample_size]
    shap_interaction_values = explainer.shap_interaction_values(X_train_sample)

    if isinstance(shap_interaction_values, list):
        shap_interaction_values = shap_interaction_values[0]

    # Calculate mean absolute SHAP interaction values
    mean_interactions = np.abs(shap_interaction_values).mean(axis=0)

    # Get feature pairs with strongest interactions
    interaction_strengths = []
    features = X_train.columns

    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            strength = (mean_interactions[i, j] + mean_interactions[j, i]) / 2
            interaction_strengths.append((features[i], features[j], strength))

    # Sort interactions by strength
    interaction_strengths.sort(key=lambda x: x[2], reverse=True)

    # Create new interaction features for top pairs
    print(f"Creating interaction features for top {top_interactions} pairs...")
    X_train_new = X_train.copy()
    X_test_new = X_test.copy()

    for feat1, feat2, strength in interaction_strengths[:top_interactions]:
        feat_name = f"{feat1}_x_{feat2}"
        X_train_new[feat_name] = X_train[feat1] * X_train[feat2]
        X_test_new[feat_name] = X_test[feat1] * X_test[feat2]

    # Calculate feature importance including new interaction features
    print("Calculating final feature importance...")
    model.fit(X_train_new, y_train)
    feature_importance = model.get_feature_importance()

    # Select top features based on importance
    feature_ranks = pd.Series(feature_importance, index=X_train_new.columns)
    selected_features = feature_ranks.nlargest(max_features).index

    # Visualize feature importance and interactions
    plt.figure(figsize=(15, 10))

    # Plot feature importance
    plt.subplot(2, 1, 1)
    top_20_features = feature_ranks.nlargest(20)
    sns.barplot(x=top_20_features.values, y=top_20_features.index)
    plt.title('Top 20 Features (Including Interactions)')
    plt.xlabel('Feature Importance')

    # Plot top interactions
    plt.subplot(2, 1, 2)
    top_interactions_df = pd.DataFrame(
        interaction_strengths[:10],
        columns=['Feature 1', 'Feature 2', 'Interaction Strength']
    )
    sns.barplot(
        data=top_interactions_df,
        x='Interaction Strength',
        y=top_interactions_df.apply(lambda x: f"{x['Feature 1']}\n×\n{x['Feature 2']}", axis=1)
    )
    plt.title('Top 10 Feature Interactions')

    plt.tight_layout()
    plt.show()

    # Calculate feature importance including new interaction features
    print("Calculating final feature importance...")
    model.fit(X_train_new, y_train)
    feature_importance = model.get_feature_importance()

    # Select top features based on importance
    feature_ranks = pd.Series(feature_importance, index=X_train_new.columns)
    selected_features = feature_ranks.nlargest(max_features).index

    # Create heatmap of top interactions

    n_top = 10  # Number of top features to show in heatmap
    top_features = list(set([f for pair in interaction_strengths[:n_top] for f in pair[:2]]))
    n_features = len(top_features)
    interaction_matrix = np.zeros((n_features, n_features))

    # Fill interaction matrix
    for i, feat1 in enumerate(top_features):
        for j, feat2 in enumerate(top_features):
            if i < j:
                # Find this pair in interaction_strengths if it exists
                strength = next((s for f1, f2, s in interaction_strengths
                               if (f1 == feat1 and f2 == feat2) or
                                  (f1 == feat2 and f2 == feat1)), 0)
                interaction_matrix[i, j] = strength
                interaction_matrix[j, i] = strength

    # Plot heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(interaction_matrix,
                xticklabels=top_features,
                yticklabels=top_features,
                cmap='YlOrRd',
                annot=True,
                fmt='.2e')
    plt.title('Feature Interaction Strength Heatmap')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Print top interaction details
    if split == 'within_categories':
      print("\nTop 10 Feature Interactions:")
      print("-" * 50)
      for feat1, feat2, strength in interaction_strengths[:10]:
          print(f"{feat1} × {feat2}: {strength:.4e}")
    else:
      print("\nTop 50 Feature Interactions:")
      print("-" * 50)
      for feat1, feat2, strength in interaction_strengths[:50]:
          print(f"{feat1} × {feat2}: {strength:.4e}")

    # Return selected features
    X_train_selected = X_train_new[selected_features]
    X_test_selected = X_test_new[selected_features]

    print(f"\nFinal number of features: {len(selected_features)}")
    print(f"Number of interaction features: {sum('_x_' in f for f in selected_features)}")

    import networkx as nx
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    # Create network visualization of top interactions
    print("Creating interaction network graph...")

    # Create graph
    G = nx.Graph()

    # Get top N interactions for the graph
    top_n = min(20, len(interaction_strengths))  # Adjust based on readability
    top_interactions_for_graph = sorted(interaction_strengths[:top_n], key=lambda x: x[2], reverse=True)

    # Add nodes and edges
    for feat1, feat2, strength in top_interactions_for_graph:
        # Add nodes if they don't exist
        if feat1 not in G:
            G.add_node(feat1)
        if feat2 not in G:
            G.add_node(feat2)

        # Add edge with weight based on interaction strength
        G.add_edge(feat1, feat2, weight=strength)

    # Calculate node size based on individual feature importance
    node_importance = {}
    feature_importance = model.get_feature_importance()
    feature_dict = dict(zip(X_train.columns, feature_importance))

    for node in G.nodes():
        node_importance[node] = feature_dict.get(node, 0)

    # Normalize node sizes for visualization
    max_size = 800
    min_size = 300
    if max(node_importance.values()) > 0:  # Avoid division by zero
        node_sizes = [min_size + (max_size - min_size) * node_importance[node] / max(node_importance.values())
                    for node in G.nodes()]
    else:
        node_sizes = [min_size for node in G.nodes()]

    # Create figure
    plt.figure(figsize=(16, 12), facecolor='white')

    # Use a shell layout for more square-like appearance
    pos = nx.kamada_kawai_layout(G)  # This often gives more evenly spaced nodes

    # Create custom colormap for edges - from light blue to pink
    colors = ['#a6cffe', '#ff9dc0'] # Light blue to pink
    edge_cmap = LinearSegmentedColormap.from_list('edge_colormap', colors)

    # Get max weight for normalization
    max_weight = max([G[u][v]['weight'] for u, v in G.edges()])

    # Draw edges with color based on weight and width proportional to weight
    for u, v, data in G.edges(data=True):
        weight = data['weight']
        normalized_weight = weight / max_weight
        edge_width = 1 + 8 * normalized_weight  # Scale width between 1 and 9
        edge_color = edge_cmap(normalized_weight)

        # Draw each edge individually with its own color and width
        nx.draw_networkx_edges(G, pos, edgelist=[(u, v)],
                              width=edge_width,
                              edge_color=[edge_color],
                              alpha=0.8)

    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                         node_color='#4a90e2', # Medium blue
                         alpha=0.8,
                         linewidths=1,
                         edgecolors='navy')

    # Add labels
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold', font_family='sans-serif')

    # Add edge labels with weights
    edge_labels = {(u, v): f"{G[u][v]['weight']:.2e}" for u, v in G.edges()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9)

    plt.title("Feature Interaction Network Graph", fontsize=18, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    from matplotlib import cm

    return X_train_selected, X_test_selected

def plot_shap_interaction_violin_grid(model, X, top_n_features=4):
    """
    Create a grid of violin plots showing SHAP interaction values between top features.

    Parameters:
    model : fitted CatBoost model
        The trained model to analyze
    X : pandas DataFrame
        Feature data to analyze
    top_n_features : int
        Number of top features to include in the grid
    """
    # Calculate SHAP interaction values
    explainer = shap.TreeExplainer(model)

    # Use a sample of data if the dataset is large
    sample_size = min(1000, len(X))
    X_sample = X.iloc[:sample_size]

    print("Calculating SHAP interaction values...")
    shap_interaction_values = explainer.shap_interaction_values(X_sample)

    if isinstance(shap_interaction_values, list):
        shap_interaction_values = shap_interaction_values[0]

    # Get feature importance based on mean absolute SHAP values
    feature_importance = np.abs(shap_interaction_values).mean(axis=0).sum(axis=1)
    top_indices = np.argsort(-feature_importance)[:top_n_features]
    top_features = X.columns[top_indices]

    # Create figure
    fig, axes = plt.subplots(top_n_features, top_n_features,
                            figsize=(3*top_n_features, 3*top_n_features))

    # Create violin plots for each feature pair
    for i, feat1 in enumerate(top_features):
        for j, feat2 in enumerate(top_features):
            ax = axes[i, j]

            # Get indices for the features
            idx1 = list(X.columns).index(feat1)
            idx2 = list(X.columns).index(feat2)

            # Get interaction values
            interaction_vals = shap_interaction_values[:, idx1, idx2]

            # Create violin plot
            if i != j:  # Off-diagonal plots
                # Split data based on feature value being above/below median
                median_val = np.median(X_sample[feat2])
                high_mask = X_sample[feat2] > median_val

                # Create violin plot with two sides
                parts = ax.violinplot([interaction_vals[~high_mask], interaction_vals[high_mask]],
                                    positions=[0, 0],
                                    vert=True,
                                    widths=0.7,
                                    showmeans=True,
                                    showextrema=True)

                # Color the different sides
                for pc in parts['bodies']:
                    if parts['bodies'].index(pc) == 0:
                        pc.set_facecolor('red')
                        pc.set_alpha(0.4)
                    else:
                        pc.set_facecolor('blue')
                        pc.set_alpha(0.4)
            else:  # Diagonal plots
                # Single violin for main effects
                parts = ax.violinplot(interaction_vals,
                                    positions=[0],
                                    vert=True,
                                    widths=0.7,
                                    showmeans=True,
                                    showextrema=True)
                for pc in parts['bodies']:
                    pc.set_facecolor('blue')
                    pc.set_alpha(0.4)

            # Set plot limits and labels
            ax.set_ylim(-np.abs(interaction_vals).max(), np.abs(interaction_vals).max())
            ax.set_xlim(-1, 1)

            # Remove x-axis ticks
            ax.set_xticks([])

            # Add feature labels
            if i == 0:
                ax.set_title(feat2, fontsize=10)
            if j == 0:
                ax.set_ylabel(feat1, fontsize=10)

            # Add grid
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.suptitle("SHAP Interaction Values Distribution", y=1.02, fontsize=14)
    plt.show()

    # Print feature importance summary
    print("\nFeature Importance Based on Interactions:")
    print("-" * 50)
    for feat, importance in zip(top_features, feature_importance[top_indices]):
        print(f"{feat}: {importance:.4e}")

def analyze_interactions_with_violin(model, X_train, X_test, y_train, is_reg=True, top_n_features=25):
    """
    Wrapper function to analyze interactions and create violin plot grid.
    """
    # Train model if not already fitted
    if not hasattr(model, 'feature_importances_'):
        print("Training model...")
        model.fit(X_train, y_train)

    # Create violin plot grid
    print("Creating interaction violin plot grid...")
    plot_shap_interaction_violin_grid(model, X_test, top_n_features)

    return model

# Use beeswarm now instead of violin
def plot_shap_interaction_grid(model, X, top_n_features=25):
    """
    Create a grid of beeswarm plots showing SHAP interaction values between top features.
    """
    try:
        # Calculate SHAP interaction values
        explainer = shap.TreeExplainer(model)

        # Use a sample of data if the dataset is large
        sample_size = min(1000, len(X))
        X_sample = X.iloc[:sample_size]

        print("Calculating SHAP interaction values...")
        shap_interaction_values = explainer.shap_interaction_values(X_sample)

        if isinstance(shap_interaction_values, list):
            shap_interaction_values = shap_interaction_values[0]

        # Get feature importance based on mean absolute SHAP values
        feature_importance = np.abs(shap_interaction_values).mean(axis=0).sum(axis=1)
        top_indices = np.argsort(-feature_importance)[:top_n_features]
        top_features = X.columns[top_indices]

        # Create figure
        fig, axes = plt.subplots(top_n_features, top_n_features,
                                figsize=(2*top_n_features, 2*top_n_features))

        # Create beeswarm plots for each feature pair
        for i, feat1 in enumerate(top_features):
            for j, feat2 in enumerate(top_features):
                ax = axes[i, j]

                # Get indices for the features
                idx1 = list(X.columns).index(feat1)
                idx2 = list(X.columns).index(feat2)

                # Get interaction values and feature values
                interaction_vals = shap_interaction_values[:, idx1, idx2]
                feature_vals = X_sample[feat2].values

                if i != j:  # Off-diagonal plots
                    # Create scatter plot with jitter
                    if len(interaction_vals) > 0:
                        # Add jitter to x-axis
                        x_jitter = np.random.normal(0, 0.1, size=len(interaction_vals))

                        # Color points based on feature value
                        colors = feature_vals
                        scatter = ax.scatter(x_jitter, interaction_vals,
                                          c=colors, cmap='coolwarm',
                                          alpha=0.6, s=20)

                else:  # Diagonal plots
                    if len(interaction_vals) > 0:
                        x_jitter = np.random.normal(0, 0.1, size=len(interaction_vals))
                        ax.scatter(x_jitter, interaction_vals,
                                 color='blue', alpha=0.6, s=20)

                # Set plot limits and labels
                if len(interaction_vals) > 0:
                    ax.set_ylim(-np.abs(interaction_vals).max(), np.abs(interaction_vals).max())
                ax.set_xlim(-1, 1)

                # Remove x-axis ticks
                ax.set_xticks([])

                # Add feature labels
                if i == 0:
                    ax.set_title(feat2, fontsize=8, rotation=45, ha='right')
                if j == 0:
                    ax.set_ylabel(feat1, fontsize=8)

                # Add grid
                ax.grid(True, alpha=0.3)

                # Add colorbar for off-diagonal plots
                if i != j and i == top_n_features-1 and j == top_n_features-1:
                    plt.colorbar(scatter, ax=ax, label=feat2)

        plt.tight_layout()
        plt.suptitle("SHAP Interaction Values Distribution", y=1.02, fontsize=14)
        plt.show()

        # Print feature importance summary
        print("\nFeature Importance Based on Interactions:")
        print("-" * 50)
        for feat, importance in zip(top_features, feature_importance[top_indices]):
            print(f"{feat}: {importance:.4e}")

    except Exception as e:
        print(f"Error in plotting SHAP interactions: {str(e)}")
        print("Continuing with the rest of the analysis...")

def analyze_interactions_with_beeswarm(model, X_train, X_test, y_train, is_reg=True, top_n_features=25):
    """
    Wrapper function to analyze interactions and create beeswarm plot grid.
    Parameters:
    model : CatBoost model
        The model to analyze
    X_train : DataFrame
        Training features
    X_test : DataFrame
        Test features
    y_train : Series/array
        Training target
    is_reg : bool
        Whether this is a regression task
    top_n_features : int
        Number of top features to show in the interaction grid
    """
    print("Training model...")
    # Fit the model first
    model.fit(X_train, y_train)

    # Create beeswarm plot grid
    print("Creating interaction beeswarm plot grid...")
    plot_shap_interaction_grid(model, X_test, top_n_features)

    return model

def compare_metrics_across_categories(metrics_dict):
    """Compare and visualize metrics across different categories."""
    # Create DataFrame for comparison
    categories = []
    accuracies = []
    precisions = []
    recalls = []
    f1s = []
    aucs = []

    for cat, metrics in metrics_dict.items():
        categories.append(cat)
        accuracies.append(metrics['accuracy'])
        precisions.append(metrics['precision'])
        recalls.append(metrics['recall'])
        f1s.append(metrics['f1'])
        if 'auc' in metrics:
            aucs.append(metrics['auc'])
        else:
            aucs.append(None)

    comparison_df = pd.DataFrame({
        'Category': categories,
        'Accuracy': accuracies,
        'Precision': precisions,
        'Recall': recalls,
        'F1': f1s,
        'AUC-ROC': aucs
    })

    # Visualize comparison
    plt.figure(figsize=(15, 8))
    metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1']
    if not all(auc is None for auc in aucs):
        metrics_to_plot.append('AUC-ROC')

    comparison_df_melted = comparison_df.melt(
        id_vars=['Category'],
        value_vars=metrics_to_plot,
        var_name='Metric',
        value_name='Score'
    )

    sns.barplot(data=comparison_df_melted, x='Category', y='Score', hue='Metric')
    plt.title('Comparison of Metrics Across Categories')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Print comparison table
    print("\nMetrics Comparison Across Categories:")
    print(comparison_df.to_string(index=False))

# Flatten the nested metrics_dict_full structure into a single DataFrame
# with one row per (category × model), sorted by R² descending.
def create_metrics_dataframe(metrics_dict):
    rows = []

    # Iterate through each category and its corresponding metrics
    for category, metrics in zip(metrics_dict['category'], metrics_dict['metrics']):
        # For each model and its scores in the metrics
        for model, scores in metrics.items():
            # Create a row with category, model, and all metrics
            row = {
                'category': category,
                'model': model,
                **scores  # Unpack all metrics (mse, rmse, mae, r2)
            }
            rows.append(row)

    # Create DataFrame from the rows
    df = pd.DataFrame(rows)

    # Sort by R² score in descending order
    try:
      df_sorted = df.sort_values('r2', ascending=False)
    except Exception as e:
      print(f"[WARNING] {e}")
      df_sorted = df

    return df_sorted

# ---
def run_analysis(tp_option, group, target_options, split, isolate=False, isolate_category=None, quantile_segment=False, feature_tp=None):
    """
    Run analysis with option to isolate a category and analyze all its combinations.

    Parameters:
    tp_option : str
        Time point option
    group : str
        Group filter
    target_options : str
        Target variable
    split : str
        Split type ('across_categories' or 'within_categories')
    isolate : bool, default=False
        Whether to run isolation analysis
    isolate_category : str, default=None
        Category to combine with all others when isolate=True
    """
    tp = tp_option if tp_option else 2
    # group is mapped from subgroup dropdown in UI section above

    global results_summary
    global metrics_dict_full
    global consensus_df
    global model_weights
    global X_train, X_test, y_train, y_test
    global category_results, exported_within_dict

    results_summary = []
    consensus_df    = None
    model_weights   = None
    category_results = {}  # Score accumulator for spider plot; reset each run_analysis call.

    # Stores evaluation metrics for each category and time point
    metrics_dict_full = {'category': [],
                          'metrics': []}

    if tp_option == 0 and group == 'high_ale':
        print("No TP = 0 with high_ale=True")
        return

    if split == "across_categories":
        X_train, X_test, y_train, y_test = across_categories_data(int(tp_option), [f"{target_options}"], group=group, scale_y=False, verbose=True, feature_tp=feature_tp)
        _, consensus_df, model_weights = run_catboost_analysis(X_train, X_test, y_train, y_test, target_options, tp, hyperparameter_tuning=hyperparameter_tuning, conformal_prediction=conformal_prediction)

    elif split == "within_categories":

        if single_category_mode and isolate_category and isolate_category != "None":
            # -- Quick single-category mode ---
            # Runs only the selected category instead of looping all.
            print(f"\n Single-category mode: running only '{isolate_category}'")
            within_categories_results = within_categories_data(
                int(tp_option), [f"{target_options}"],
                group=group, scale_y=False, verbose=True,
                feature_tp=feature_tp
            )

            if isolate_category in within_categories_results:
                X_train, X_test, y_train, y_test = within_categories_results[isolate_category]
                print(f"\nAnalyzing category: {isolate_category}")
                print(f"Features shape: {X_train.shape}")

                _, _cdf, _mw = run_catboost_analysis(
                    X_train, X_test, y_train, y_test,
                    target_options, tp,
                    category=isolate_category,
                    hyperparameter_tuning=hyperparameter_tuning,
                    conformal_prediction=conformal_prediction
                )
                if _cdf is not None:
                    consensus_df  = _cdf
                    model_weights = _mw

                if ensemble_version:
                    results_ensemble = create_metrics_dataframe(metrics_dict_full)
                    try:
                        results_meta = results_ensemble[results_ensemble['model']=='meta_model']
                        results_meta.to_csv('results_ensemble_meta_model.csv', index=False)
                        display(results_meta)
                    except Exception as e:
                        print(f"[WARNING] {e}")
                        display(results_ensemble)
                        results_ensemble.to_csv('results_ensemble.csv', index=False)
                    print("Results ensemble saved.")
                else:
                    try:
                        results_summary_df = pd.DataFrame(results_summary)
                        display(results_summary_df)
                        results_summary_df.to_csv('results_summary.csv', index=False)
                        print("Results summary saved.")
                    except Exception as e:
                        print("No results summary to save.")
            else:
                _available = list(within_categories_results.keys())
                print(f"Warning:  Category '{isolate_category}' not found at T{tp_option}.")
                print(f"   Available categories: {_available}")

        elif isolate:
            # Get combinations for isolated category
            print(f"\nAnalyzing all combinations for {isolate_category}:")
            combinations_results = within_categories_data_combinations(
                int(tp_option),
                [f"{target_options}"],
                base_category=isolate_category,
                group=group,
                scale_y=False,
                verbose=True
            )

            # quantile_results = analyze_category_quantiles(combinations_data)
            # print_quantile_analysis(quantile_results)

            # Process each combination
            for combination_name, (X_train, X_test, y_train, y_test) in combinations_results.items():
                print(f"\nAnalyzing combination: {combination_name}")
                print(f"Features shape: {X_train.shape}")

                _, _cdf, _mw = run_catboost_analysis(
                    X_train, X_test, y_train, y_test,
                    target_options, tp,
                    category=combination_name,
                    hyperparameter_tuning=hyperparameter_tuning,
                    conformal_prediction=conformal_prediction
                )
                if _cdf is not None:
                    consensus_df   = _cdf
                    model_weights  = _mw

            try:
              results_summary_df = pd.DataFrame(results_summary)
              display(results_summary_df)
              results_summary_df.to_csv('results_summary.csv', index=False)
              print("Results summary saved.")
            except Exception as e:
              print("No results summary to save.")

        else:
            within_categories_results = within_categories_data(int(tp_option), [f"{target_options}"], group=group, scale_y=False, verbose=True, feature_tp=feature_tp)

            # Quantile analysis
            quantile_results = analyze_category_quantiles(within_categories_results)

            for category, data in within_categories_results.items():
                print(f"\nAnalyzing category: {category}")
                X_train, X_test, y_train, y_test = data

                if category == 'social_relationships' or category == 'other_psychopathology':
                  X_train = X_train.loc[:, ~X_train.columns.duplicated()]
                  X_test = X_test.loc[:, ~X_test.columns.duplicated()]

                _, _cdf, _mw = run_catboost_analysis(X_train, X_test, y_train, y_test, target_options, tp, category, hyperparameter_tuning=hyperparameter_tuning,
                                      conformal_prediction=conformal_prediction, quantile_segment=quantile_segment)
                if _cdf is not None:
                    consensus_df   = _cdf
                    model_weights  = _mw

            if not ensemble_version:
              try:
                results_summary_df = pd.DataFrame(results_summary)
                display(results_summary_df)
                results_summary_df.to_csv('results_summary.csv', index=False)
                print("Results summary saved.")
              except Exception as e:
                print("No results summary to save.")
            else:

              results_ensemble = create_metrics_dataframe(metrics_dict_full)
              try:
                results_meta = results_ensemble[results_ensemble['model']=='meta_model']
                results_meta.to_csv('results_ensemble_meta_model.csv', index=False)
                display(results_meta)
              except Exception as e:
                print(f"[WARNING] {e}")
                display(results_ensemble)
                results_ensemble.to_csv('results_ensemble.csv', index=False)

              print("Results ensemble saved.")

# Main per-category modelling loop. Preprocesses the category slice
# (encoding, outlier handling, scaling), fits either the stacking
# ensemble or a standalone CatBoost model, and collects performance
# metrics, SHAP importance, partial dependence, and optional
# conformal prediction intervals.
def run_catboost_analysis(X_train, X_test, y_train, y_test, target_options, tp, category=None,
                          hyperparameter_tuning=False, conformal_prediction=False, quantile_segment=False):
    global last_fitted_model  # Single declaration covers all paths

    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.impute import SimpleImputer
    from sklearn.model_selection import RandomizedSearchCV, cross_val_score
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    try:
        from sklearn.metrics import root_mean_squared_error  # sklearn ≥ 1.4
    except ImportError:
        from sklearn.metrics import mean_squared_error as _mse
        def root_mean_squared_error(y_true, y_pred): return _mse(y_true, y_pred) ** 0.5
    from scipy.stats import uniform, randint
    import matplotlib.pyplot as plt
    import shap
    from imblearn.over_sampling import SMOTE, SMOTENC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.preprocessing import OneHotEncoder

    print(f"Original shape: {X_train.shape}")

    # One-hot encode categorical features before modelling.
    one_hot_features = ['marital_status', 'child_religion', 'sex_orient_y', 'trans_id_y', 'crysflu_c']

    encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

    # Step 1: Encode categorical variables
    le = LabelEncoder()
    categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns
    for column in categorical_columns:
        X_train[column] = le.fit_transform(X_train[column].astype(str))
        X_test[column] = le.transform(X_test[column].astype(str))
    print("Categorical variables encoded")

    # Step 2: Remove constant features
    try:
      constant_columns = [col for col in X_train.columns if X_train[col].nunique() == 1]
      X_train = X_train.drop(columns=constant_columns)
      X_test = X_test.drop(columns=constant_columns)
      print(f"Removed {len(constant_columns)} constant columns")
    except Exception as e:
      print(f"No constant features to remove.")

    # Step 3: Handle outliers
    try:
      X_train = handle_outliers(X_train)
      X_test = handle_outliers(X_test)
      print("Outliers handled")
    except Exception as e:
      print("No outliers needed to be handled.")

    is_reg = is_regression(target_options)

    if not is_reg and (target_options == 'latent_class_depression' or target_options == 'depadhd_c' or target_options == 'asdadhd_c'):
        y_train = map_classes(y_train)
        y_test = map_classes(y_test)

    # Step 4a: Clean NaN / inf BEFORE fitting the scaler (NaN in fit → NaN-scaled columns).
    na_method_mapping = {
        "fill_with_zero": ("fill_0", None),
        "drop": ("remove", None),
        "impute_mean": ("impute", "mean"),
        "impute_median": ("impute", "median")
    }
    method, imputer_strategy = na_method_mapping[deal_with_na_values]
    X_train, y_train = clean_data(X_train, y_train, method=method, imputer_strategy=imputer_strategy)
    X_test, y_test = clean_data(X_test, y_test, method=method, imputer_strategy=imputer_strategy)
    X_train = X_train.loc[:, ~X_train.columns.duplicated(keep='first')]
    X_test = X_test.loc[:, ~X_test.columns.duplicated(keep='first')]

    # Step 4b: Feature scaling -- fit exclusively on clean training data.
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    print("Features scaled")

    print(f"Shape after preprocessing: {X_train.shape}")

    # Feature selection (if enabled)
    if selection_method != 'None' and selection_method is not None:
        if selection_method == 'rfecv':
            # Recursive Feature Elimination with Cross-Validation
            if is_reg:
                estimator = RandomForestRegressor(random_state=42)
                print("Using RandomForestRegressor for feature selection.")
            else:
                estimator = RandomForestClassifier(random_state=42)
                print("Using RandomForestClassifier for feature selection.")
            selector = RFECV(estimator, step=1, cv=5)
            selector = selector.fit(X_train,  y_train) # Use y.ravel to make it to 1d from 2d

            # Transform X_train and X_test using the selector
            X_train_transformed = selector.transform(X_train)
            X_test_transformed = selector.transform(X_test)

            # Convert numpy arrays back to DataFrame and update column names
            selected_features = [X_train.columns[i] for i in selector.get_support(indices=True)]
            X_train = pd.DataFrame(X_train_transformed, columns=selected_features)
            X_test = pd.DataFrame(X_test_transformed, columns=selected_features)
            print(f"Features reduced to {selector.n_features_} using RFECV.")

        elif selection_method == 'interaction':
            # Feature Interaction: Add new features that are interactions of existing features

            if is_reg:
                initial_model = CatBoostRegressor(
                    iterations=100,
                    depth=6,
                    learning_rate=0.1,
                    verbose=False,
                    random_state=42
                )
            else:
                initial_model = CatBoostClassifier(
                    iterations=100,
                    depth=6,
                    learning_rate=0.1,
                    verbose=False,
                    random_state=42
                )


            print("Performing SHAP interaction-based feature selection...")
            X_train, X_test = select_features_with_shap_interactions(
                X_train,
                X_test,
                y_train,
                split=str(split),
                is_reg=is_reg,
                max_features=1000,
                top_interactions=50
            )

        elif selection_method == 'h-stats':
          print("Performing H-Stats interaction-based feature selection...")

          X_train, X_test = select_features_with_h_statistics(
              X_train,
              X_test,
              y_train,
              is_reg=is_reg,
              max_features=1000,
              top_interactions=50
          )

        elif selection_method == 'clustering':
            # Feature Clustering: Cluster features and select cluster representatives
            n_clusters = 5
            kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(X_train.T)
            cluster_labels = kmeans.labels_
            selected_features = [X_train.columns[cluster_labels == i][0] for i in range(n_clusters)]
            X_train = X_train[selected_features]
            X_test = X_test[selected_features]
            print(f"Features reduced to {len(selected_features)} by clustering.")

    num_classes = len(np.unique(y_train)) if not is_reg else 0

    if quantile_segment:
        print("Calculating quantile-based sample weights...")
        #This one is for the dynamic 4 quantile segmentation
        #sample_weights = calculate_quantile_weights(y_train)

        # This one only for top 25%
        sample_weights = calculate_top_quartile_weights(y_train)

    else:
        sample_weights = None

    if hyperparameter_tuning:
        print("Tuning the hyperparameters")

        if with_optuna:
          print("Tuning hyperparameters with Optuna")

          # unique feature names
          X_train = ensure_unique_feature_names(X_train)
          X_test = ensure_unique_feature_names(X_test)

          study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler())
          study.optimize(lambda trial: objective(trial, X_train, y_train, is_reg, num_classes), n_trials=num_of_optuna_trials)

          best_params = study.best_params
          best_params['random_state'] = 42
          print("Best hyperparameters:", best_params)

          if ensemble_version:
            if ensemble_type == 'bagging':
                # Create ensemble model (existing bagging implementation)
                model_configs = {
                    'catboost': best_params,
                    'random_forest': dict(_RF_PARAMS),
                    'xgboost': dict(_XGB_PARAMS),
                    'elasticnet': {'alpha': 1.0, 'l1_ratio': 0.5}  # Only for regression
                }

                model = create_ensemble_model(
                    task_type='regression' if is_reg else 'classification',
                    model_configs=model_configs
                )
            else:  # ensemble_type == 'stacking'
                # Create stacking ensemble model

                model_configs = {
                    'catboost': best_params,
                    'random_forest': dict(_RF_PARAMS),
                    'xgboost': dict(_XGB_PARAMS),
                    #'svm': {'kernel': 'rbf', 'C': 1.0},
                    'tabpfn': dict(PIPELINE_MODEL_CONFIGS['tabpfn'])
                }
                if _SKIP_TABPFN: model_configs.pop('tabpfn', None)

                if split == 'within_categories' and category == 'Multimodal Neuroimaging':
                    print("Change to CPU for Neuroimaging.")
                    model_configs['tabpfn'] = {'device': 'cpu', 'max_features': 50, 'version': 'v2'}
                    if _SKIP_TABPFN: model_configs.pop('tabpfn', None)

                # Stacking uses Linear Regression as the meta-learner for regression
                # and LogisticRegression for classification
                if is_reg:
                    meta_model = LinearRegression()
                else:
                    meta_model = LogisticRegression(random_state=42)


                model = create_stacking_ensemble(
                    task_type='regression' if is_reg else 'classification',
                    model_configs=model_configs,
                    meta_model=meta_model,
                    cv=5,
                    random_state=42
                )

                model.explain_meta_model(X_train, y_train)
                # Expose fitted model to downstream cells
                last_fitted_model = model

          else:
            if is_reg:
                model = CatBoostRegressor(**best_params)
            else:
                model = CatBoostClassifier(**best_params)

          # Optuna provides feature importance for hyperparameters
          importance = optuna.importance.get_param_importances(study)

          print("\nHyperparameter importance:")
          for param, score in importance.items():
              print(f"{param}: {score:.4f}")
        else:
          if ensemble_version:
                if ensemble_type == 'bagging':
                    raise ValueError("Bagging ensemble path removed — use ensemble_type='stacking'")
                else:  # ensemble_type == 'stacking'
                    # Create and tune stacking ensemble
                    if is_reg:
                        meta_model = LinearRegression()
                    else:
                        meta_model = LogisticRegression(random_state=42)


                    model = create_stacking_ensemble(
                        task_type='regression' if is_reg else 'classification',
                        meta_model=meta_model,
                        cv=5,
                        random_state=42
                    )

                # Fit the model (stacking uses .fit; tuned bagging uses .tune_and_fit)
                if hasattr(model, "tune_and_fit"):
                    categorical_features = list(range(len(categorical_columns))) if categorical_columns else None
                    model = model.tune_and_fit(X_train, y_train, categorical_features=categorical_features)
                else:
                    model.fit(X_train, y_train)
                    model.explain_meta_model(X_train, y_train)
                # Expose fitted model to Cell 9
                if ensemble_type == 'stacking':
                    last_fitted_model = model
          else:
            # Hyperparameter tuning for CatBoost
            if is_reg:
                param_dist = {
                    'iterations': randint(100, 1000),
                    'learning_rate': uniform(0.01, 0.3),
                    'depth': randint(4, 10),
                    'l2_leaf_reg': uniform(1, 10),
                    'border_count': randint(32, 255)
                }
                model = CatBoostRegressor(random_state=42)
                scoring = 'neg_mean_squared_error'
            else:
                param_dist = {
                    'iterations': randint(100, 1000),
                    'learning_rate': uniform(0.01, 0.3),
                    'depth': randint(4, 10),
                    'l2_leaf_reg': uniform(1, 10),
                    'border_count': randint(32, 255)
                }
                model = CatBoostClassifier(random_state=42)
                scoring = 'accuracy'

            random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=100, cv=5, scoring=scoring, random_state=42, n_jobs=-1)
            random_search.fit(X_train, y_train)

            print("Best hyperparameters:", random_search.best_params_)
            model = random_search.best_estimator_
    else:
        print("Not tuning the hyperparameters")

        # Use default or predefined parameters
        best_params = {
              'iterations': 1000,
              'learning_rate': 0.05,
              'depth': 6,
              'l2_leaf_reg': 3,
              'border_count': 128,
              'thread_count': -1,
              'verbose': False
          }

        # Use optimal parameters
        if ensemble_version:
            if ensemble_type == 'bagging':
                # Create bagging ensemble model
                model_configs = {
                    'catboost': best_params,
                    'random_forest': dict(_RF_PARAMS),
                    'xgboost': dict(_XGB_PARAMS),
                    'elasticnet': {'alpha': 1.0, 'l1_ratio': 0.5}
                }

                model = create_ensemble_model(
                    task_type='regression' if is_reg else 'classification',
                    model_configs=model_configs
                )
            else:  # ensemble_type == 'stacking'
                # Create stacking ensemble model
                model_configs = {
                    'catboost': best_params,
                    'random_forest': dict(_RF_PARAMS),
                    'xgboost': dict(_XGB_PARAMS),
                    #'svm': {'kernel': 'rbf', 'C': 1.0},
                    'tabpfn': dict(PIPELINE_MODEL_CONFIGS['tabpfn'])
                    }
                if _SKIP_TABPFN: model_configs.pop('tabpfn', None)

                if split == 'within_categories' and category == 'Multimodal Neuroimaging':
                    print("Switch to CPU for neuroimaging.")
                    model_configs['tabpfn'] = {'device': 'cpu', 'max_features': 50, 'version': 'v2'}
                    if _SKIP_TABPFN: model_configs.pop('tabpfn', None)

                if is_reg:
                    meta_model = LinearRegression()
                else:
                    meta_model = LogisticRegression(random_state=42)


                model = create_stacking_ensemble(
                    task_type='regression' if is_reg else 'classification',
                    model_configs=model_configs,
                    meta_model=meta_model,
                    cv=5,
                    random_state=42
                )

        else:
          if is_reg:
              model = CatBoostRegressor(**best_params, random_state=42)
          else:
              model = CatBoostClassifier(**best_params, random_state=42)
    if is_reg:
        scoring = 'r2'
    else:
        scoring = 'roc_auc'

    # Add boolean target processing here
    if target_options == 'dep_onset_rci_2.3' or (len(np.unique(y_train)) == 2 and not is_reg):
        y_train = preprocess_boolean_target(y_train)
        y_test = preprocess_boolean_target(y_test)
        model, y_train = modify_ensemble_for_boolean(model, X_train, y_train)

    # Inner-loop CV handled by stacking OOF; standalone CV skipped.
    cv_scores = []

    # Create CatBoost Pool objects
    # Identify categorical columns
    categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

    # Single authoritative Pool construction.
    try:
        train_pool = Pool(X_train, y_train, cat_features=categorical_columns, weight=sample_weights)
        test_pool  = Pool(X_test,  y_test,  cat_features=categorical_columns)
    except Exception as e:
        print(f"[WARNING] Pool with cat_features failed ({e}); retrying without.")
        X_train = X_train.loc[:, ~X_train.columns.duplicated()]
        X_test  = X_test.loc[:,  ~X_test.columns.duplicated()]
        train_pool = Pool(X_train, y_train, weight=sample_weights)
        test_pool  = Pool(X_test,  y_test)

    if simple_test_model:
      if model_type == "lasso" or model_type == "ridge":
          if not is_regression(target_options):
              print(f"Warning: {model_type} regression is only for regression tasks. Using CatBoost for classification.")
          else:
              return run_linear_model_analysis(model_type, X_train, X_test, y_train, y_test, target_options, tp, category), None, None

    # Only for classification
    if not is_reg and not ensemble_version:
        from sklearn.utils.class_weight import compute_class_weight
        # Calculate class weights for this category's data
        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train.values.ravel())
        class_weights_dict = dict(zip(np.unique(y_train), class_weights))
        model.set_params(class_weights=class_weights_dict)

    # Fit the model
    if ensemble_version:
        # Ensemble path: fit on raw arrays rather than CatBoost Pool objects
        # Check if we need to reduce features for TabPFN
        if X_train.shape[1] >= 500 and 'model_configs' in dir() and 'tabpfn' in model_configs:
            print("\nFeature count exceeds TabPFN limit. Selecting top features to ensure it's under 500...")
            # Use CatBoost to select features
            if is_reg:
                feature_selector = CatBoostRegressor(
                    iterations=100,
                    learning_rate=0.1,
                    verbose=False,
                    random_state=42
                )
            else:
                feature_selector = CatBoostClassifier(
                    iterations=100,
                    learning_rate=0.1,
                    verbose=False,
                    random_state=42
                )

            # Fit CatBoost and get feature importance
            try:
              feature_selector.fit(X_train, y_train)
            except Exception as e:
              print(f"For TabPFN feature selection, y_train needs to not be multidimentional")
              feature_selector.fit(X_train, y_train.iloc[:, 0])

            importance = pd.Series(feature_selector.feature_importances_, index=X_train.columns)
            selected_features = importance.nlargest(490).index

            # Reduce features for all models
            X_train = X_train[selected_features]
            if isinstance(X_test, pd.DataFrame):
                X_test = X_test[selected_features]

            print(f"Reduced feature set from {X_train.shape[1]} to {len(selected_features)} features")

            categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

            # Create Pool objects with potentially reduced features
            if categorical_columns:
                train_pool = Pool(X_train, y_train, cat_features=categorical_columns, weight=sample_weights)
                test_pool = Pool(X_test, y_test, cat_features=categorical_columns)
            else:
                try:
                    train_pool = Pool(X_train, y_train, weight=sample_weights)
                    test_pool = Pool(X_test, y_test)
                except Exception as e:
                    print(f"[WARNING] {e}")
                    X_train = X_train.loc[:, ~X_train.columns.duplicated()]
                    X_test = X_test.loc[:, ~X_test.columns.duplicated()]
                    train_pool = Pool(X_train, y_train, weight=sample_weights)
                    test_pool = Pool(X_test, y_test)

        # Fit the model on the (possibly reduced) feature set
        if categorical_columns:
            model.fit(X_train, y_train, categorical_features=categorical_columns)
        else:
            model.fit(X_train, y_train)
        # Expose fitted model to Cell 9
        if ensemble_type == 'stacking':
            last_fitted_model = model
    else:
        # CatBoost fitting with Pool objects
        model.fit(train_pool, eval_set=test_pool,
                  use_best_model=True, early_stopping_rounds=50)

    if _run_consensus:
      # Local vars -- will be returned to caller for global assignment
      _local_consensus_df    = None
      _local_model_weights   = None
      try:

        consensus_df, model_weights, feature_comparison = model.analyze_feature_consensus(
            X_train, y_train,
            target_options,
            tp,
            category,
            top_n=ENSEMBLE_VIZ_CONFIG['top_n']
        )
        _local_consensus_df  = consensus_df
        _local_model_weights = model_weights

        # Access specific results if needed
        print("\nTop features with their consensus scores:")
        print(consensus_df.sort_values('weighted_score', ascending=False))
      except Exception as e:
        print("Failed to do consensus_df, model_weights, and feature comparison, because of: ", e)
        print("Only works for Ensemble: Stacking - Consensus Analysis")
    else:
      _local_consensus_df  = None
      _local_model_weights = None
      print("    Consensus analysis skipped (visualization_mode does not include consensus).")

    if sample_weights is not None and len(sample_weights) > 0:
      print(f"Sample weights included: n={len(sample_weights)}, unique={len(set(sample_weights))}")

    # Convert y_test to numpy array if it's a DataFrame or Series
    y_test_np = y_test.values if isinstance(y_test, (pd.DataFrame, pd.Series)) else np.array(y_test)
    y_test_np = y_test_np.ravel()  # it's 1-dimensional

    # Visualize selection_method interaction with SHAP

    # After model fitting, add conformal prediction if enabled
    if conformal_prediction:
        print("\nPerforming Conformal Prediction Analysis...")

        # Split validation set for conformal prediction
        X_train_conf, X_val_conf, y_train_conf, y_val_conf = train_test_split(
            X_train, y_train, test_size=0.2, random_state=42
        )

        if is_reg:
            # Regression conformal prediction
            mapie = SplitConformalRegressor(
                estimator=model,
                random_state=42
            )

            # Fit MAPIE
            mapie.fit(X_train_conf, y_train_conf)

            # Get prediction intervals
            y_pred_conf, y_pis = mapie.predict(X_test, alpha=[0.05, 0.32])

            # Visualize prediction intervals
            plt.figure(figsize=(15, 8))
            plt.fill_between(
                range(len(y_test)),
                y_pis[:, 0, 0],  # Lower bound for alpha=0.05
                y_pis[:, 1, 0],  # Upper bound for alpha=0.05
                alpha=0.3,
                label="95% Prediction Interval"
            )
            plt.fill_between(
                range(len(y_test)),
                y_pis[:, 0, 1],  # Lower bound for alpha=0.32
                y_pis[:, 1, 1],  # Upper bound for alpha=0.32
                alpha=0.3,
                label="68% Prediction Interval"
            )
            plt.plot(range(len(y_test)), y_test, 'ko', label='Actual', markersize=4)
            plt.plot(range(len(y_test)), y_pred_conf, 'r-', label='Predicted')
            if split == 'within_categories':
              plt.title(f"Conformal Prediction Intervals\n({target_options}, Timepoint {tp}, for {category})")
            else:
              plt.title(f"Conformal Prediction Intervals\n({target_options}, Timepoint {tp})")

            plt.xlabel("Sample Index")
            plt.ylabel("Target Value")
            plt.legend()
            plt.tight_layout()
            plt.show()

        else:
          # Classification conformal prediction
          mapie = SplitConformalClassifier(
              estimator=model,
              random_state=42
          )

          # Fit MAPIE
          mapie.fit(X_train_conf, y_train_conf)

          # Get prediction sets for different alpha values
          alpha_values = [0.05, 0.32, 0.5]  # 0.5 included to approximate predicted probabilities
          y_pred_conf, y_ps = mapie.predict(X_test, alpha=alpha_values)

          # Determine if it's binary or multiclass
          n_classes = len(np.unique(y_test))
          is_binary = n_classes == 2

          # Calculate prediction set sizes
          set_sizes = np.sum(y_ps, axis=2)

          # Calculate coverage using numpy arrays
          coverage_95 = np.mean([y_test_np[i] in np.where(y_ps[i, :, 0])[0] for i in range(len(y_test_np))])
          coverage_68 = np.mean([y_test_np[i] in np.where(y_ps[i, :, 1])[0] for i in range(len(y_test_np))])

          # Approximate class probabilities
          if is_binary:
              class_probs = 1 - y_ps[:, 0, :]  # Probability of class 1
              class_probs = np.column_stack((1 - class_probs, class_probs))  # Add probability of class 0
          else:
              class_probs = y_ps[:, :, -1]  # Use the last alpha value for multiclass

          # 1. Class Probability Distributions
          plt.figure(figsize=(15, 8))
          for i in range(class_probs.shape[1]):
              sns.kdeplot(class_probs[:, i], label=f'Class {i}')

          if split == 'within_categories':
              plt.title(f"Class Probability Distributions\n({target_options}, Timepoint {tp}, for {category})")
          else:
              plt.title(f"Class Probability Distributions\n({target_options}, Timepoint {tp})")
          plt.xlabel("Probability")
          plt.ylabel("Density")
          plt.legend()
          plt.tight_layout()
          plt.show()

          # 2. Prediction Set Sizes Distribution

          try:
            plt.figure(figsize=(15, 8))
            for i, alpha in enumerate(alpha_values):
                sns.histplot(set_sizes[:, i], kde=True, label=f'α = {alpha}', stat='density', alpha=0.6)
          except Exception as e:
            print("Full Alpha Distribution of Prediction Sets not available for ", target_options)

          if split == 'within_categories':
              plt.title(f"Distribution of Prediction Set Sizes\n({target_options}, Timepoint {tp}, for {category})")
          else:
              plt.title(f"Distribution of Prediction Set Sizes\n({target_options}, Timepoint {tp})")
          plt.xlabel("Number of Classes in Prediction Set")
          plt.ylabel("Density")
          plt.legend()
          plt.tight_layout()
          plt.show()

          # 3. Coverage vs Alpha
          coverages = [np.mean([y_test_np[i] in np.where(y_ps[i, :, j])[0] for i in range(len(y_test_np))]) for j in range(len(alpha_values))]
          plt.figure(figsize=(10, 6))
          plt.plot(alpha_values, coverages, 'bo-')
          plt.plot(alpha_values, alpha_values, 'r--', label='Ideal coverage')

          if split == 'within_categories':
              plt.title(f"Coverage vs. Alpha\n({target_options}, Timepoint {tp}, for {category})")
          else:
              plt.title(f"Coverage vs. Alpha\n({target_options}, Timepoint {tp})")
          plt.xlabel("Alpha")
          plt.ylabel("Empirical coverage")
          plt.legend()
          plt.tight_layout()
          plt.show()

          # 4. Confusion Matrix (for binary classification)
          if is_binary:
              y_pred = (class_probs[:, 1] > 0.5).astype(int)
              cm = confusion_matrix(y_test_np, y_pred)
              plt.figure(figsize=(10, 8))
              sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
              plt.title(f"Confusion Matrix\n({target_options}, Timepoint {tp})")
              plt.xlabel("Predicted Class")
              plt.ylabel("True Class")
              plt.tight_layout()
              plt.show()

          # 5. ROC Curve (for binary classification)
          try:
            if is_binary:
                fpr, tpr, _ = roc_curve(y_test_np, class_probs[:, 1])
                roc_auc = auc(fpr, tpr)
                plt.figure(figsize=(10, 8))
                plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
                plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
                plt.xlim([0.0, 1.0])
                plt.ylim([0.0, 1.05])
                plt.xlabel('False Positive Rate')
                plt.ylabel('True Positive Rate')
                plt.title(f'Receiver Operating Characteristic (ROC) Curve\n({target_options}, Timepoint {tp})')
                plt.legend(loc="lower right")
                plt.tight_layout()
                plt.show()
          except Exception as e:
            print("Cannot convert AUC variables because len y unique != 2: ", e)

          print(f"Coverage at 95% confidence: {coverage_95:.4f}")
          print(f"Coverage at 68% confidence: {coverage_68:.4f}")

    # Get feature importance
    if ensemble_version:
      # Get and plot feature importance from all models
      if not run_visualizations:
          print("    run_visualizations=False -- all FI/SHAP/PDP plots skipped.")
      elif show_feature_importance_plots:
          _fi_models = {k: v for k, v in model.base_models.items() if k in _VIZ_ENABLED_MODELS}
          if _fi_models:
              _orig_base = model.base_models
              model.base_models = _fi_models
              print(f"  FI plots for: {sorted(_fi_models.keys())}")
              try:
                  # -- Apply DISPLAY_NAMES to ensemble FI --
                if "DISPLAY_NAMES" in globals() and DISPLAY_NAMES:
                    model._display_names = DISPLAY_NAMES  # Pass to plotting
                model.plot_feature_importance(X_train, y_train, target_options, tp, category)
              finally:
                  model.base_models = _orig_base
          else:
              print("    No models enabled for FI plots.")
      else:
          print("    show_feature_importance_plots=False -- FI plots suppressed.")
          print("     SHAP will be shown for the best enabled model only.")
    else:
      feature_importance = model.get_feature_importance()
      feature_names = X_train.columns
      feature_importance = pd.Series(feature_importance, index=feature_names).sort_values(ascending=False)

      # Apply display names if available
      if 'DISPLAY_NAMES' in dir() and isinstance(DISPLAY_NAMES, dict) and len(DISPLAY_NAMES) > 0:
          feature_importance.index = feature_importance.index.map(lambda x: DISPLAY_NAMES.get(x, x))

      # Plot top 20 OR 30 feature importances
      if split == "across_categories":
        top_20_features = feature_importance.head(30).sort_values(ascending=True)
        top_str = 30
      else:
        top_20_features = feature_importance.head(20).sort_values(ascending=True)
        top_str = 20

      plt.figure(figsize=(8, 8))
      sns.barplot(x=top_20_features.values, y=top_20_features.index, palette='coolwarm')
      title = f"Top {top_str} CatBoost Important Features for {'Regression' if is_reg else 'Classification'} ({target_options}, Timepoint {tp}"
      title += f", Category {category})" if category else ")"
      plt.title(title, fontsize=16)
      plt.xlabel("Feature Importance", fontsize=12)
      plt.ylabel("Features", fontsize=12)
      for i, v in enumerate(top_20_features.values):
          plt.text(v, i, f' {v:.4f}', va='center', fontsize=10)
      plt.tight_layout()
      plt.show()

      # Partial Dependence Plots
      top_10_features = feature_importance.head(10).index
      if is_reg and _run_pdp_gam:
          fig, ax = plt.subplots(2, 5, figsize=(24, 12))
          for i, feature in enumerate(top_10_features):
            try:
              PartialDependenceDisplay.from_estimator(model, X_train, [feature], ax=ax[i // 5, i % 5])
            except Exception as e:
              print("Percentiles too close together, skipping this one as it is un-map-able.")
          plt.suptitle(f'Partial Dependence Plots for Top 10 Features ({target_options}, Timepoint {tp})', fontsize=20)
          plt.tight_layout(rect=[0, 0, 1, 0.96])
          plt.show()
      else:
          classes = model.classes_
          for target_class in classes:
              fig, ax = plt.subplots(2, 5, figsize=(24, 12))
              for i, feature in enumerate(top_10_features):
                try:
                  PartialDependenceDisplay.from_estimator(model, X_train, [feature], target=target_class, ax=ax[i // 5, i % 5])
                except ValueError:
                  print("Percentiles too close together, skipping this one as it is un-map-able.")
              plt.suptitle(f'Partial Dependence Plots for Top 10 Features (Class: {target_class}, {target_options}, Timepoint {tp})', fontsize=20)
              plt.tight_layout(rect=[0, 0, 1, 0.95])
              plt.show()

    # Calculate test predictions and metrics

    y_pred = model.predict(X_test)

    if ensemble_version:
        # Get metrics for all models and ensemble
        if not run_visualizations:
            import matplotlib; _was_interactive = matplotlib.is_interactive(); matplotlib.interactive(False)
        metrics_dict = model.evaluate_models(X_test, y_test, target_options, tp, category)
        if not run_visualizations:
            plt.close('all'); matplotlib.interactive(_was_interactive)
            print("    run_visualizations=False -- model comparison plots suppressed.")
        metrics_dict_full['metrics'].append(metrics_dict)

        try:
          metrics_dict_full['category'].append(category)
        except Exception as e:
          print(f"[WARNING] {e}")
          metrics_dict_full['category'].append('All')

        # Use ensemble metrics for overall evaluation
        if is_reg:
          try:
            evaluation_metrics = pd.DataFrame({
                'Metric': ['MSE', 'RMSE', 'MAE', 'R² Score'],
                'Value': [metrics_dict['ensemble']['mse'],
                        metrics_dict['ensemble']['rmse'],
                        metrics_dict['ensemble']['mae'],
                        metrics_dict['ensemble']['r2']]
            })
            score = metrics_dict['ensemble']['r2']
          except Exception as e:
            print(f"[WARNING] {e}")
            evaluation_metrics = pd.DataFrame({
                'Metric': ['MSE', 'RMSE', 'MAE', 'R² Score'],
                'Value': [metrics_dict['meta_model']['mse'],
                        metrics_dict['meta_model']['rmse'],
                        metrics_dict['meta_model']['mae'],
                        metrics_dict['meta_model']['r2']]
            })
            score = metrics_dict['meta_model']['r2']
        else:
          try:
            metrics = metrics_dict['ensemble']
            evaluation_metrics = pd.DataFrame({
                'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'] +
                        (['AUC-ROC'] if 'auc' in metrics else []),
                'Value': [metrics['accuracy'], metrics['precision'],
                        metrics['recall'], metrics['f1']] +
                        ([metrics['auc']] if 'auc' in metrics else [])
            })
            score = metrics['auc'] if 'auc' in metrics else metrics['accuracy']
          except Exception as e:
            print(f"[WARNING] {e}")

            metrics = metrics_dict['meta_model']
            evaluation_metrics = pd.DataFrame({
                'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'] +
                        (['AUC-ROC'] if 'auc' in metrics else []),
                'Value': [metrics['accuracy'], metrics['precision'],
                        metrics['recall'], metrics['f1']] +
                        ([metrics['auc']] if 'auc' in metrics else [])
            })
            score = metrics['auc'] if 'auc' in metrics else metrics['accuracy']

    else:
      if is_reg:
          mse = mean_squared_error(y_test, y_pred)
          rmse = np.sqrt(mse)
          mae = mean_absolute_error(y_test, y_pred)
          r2 = r2_score(y_test, y_pred)
          score = r2

          evaluation_metrics = pd.DataFrame({
              'Metric': ['MSE', 'RMSE', 'MAE', 'R² Score'],
              'Value': [mse, rmse, mae, r2]
          })
      else:
          # Get predictions for just this category's test data
          y_pred_category = model.predict(X_test)
          proba_category = model.predict_proba(test_pool)

          # Calculate metrics using only this category's predictions
          accuracy = accuracy_score(y_test, y_pred_category)
          precision = precision_score(y_test, y_pred_category, average='weighted')
          recall = recall_score(y_test, y_pred_category, average='weighted')
          f1 = f1_score(y_test, y_pred_category, average='weighted')

          # Store metrics for this category in a dictionary
          category_metrics = {
              'accuracy': accuracy,
              'precision': precision,
              'recall': recall,
              'f1': f1
          }

          if len(np.unique(y_test)) == 2:
              auc = roc_auc_score(y_test, model.predict_proba(test_pool)[:, 1])
              category_metrics['auc'] = auc
              evaluation_metrics = pd.DataFrame({
                  'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC-ROC'],
                  'Value': [accuracy, precision, recall, f1, auc]
              })
              score = auc
          else:
              try:
                  auc = roc_auc_score(y_test, proba_category, multi_class='ovr', average='macro')
                  score = auc
                  category_metrics['auc'] = auc
              except Exception as e:
                  print(f"Cannot define AUC for category {category} - {len(np.unique(y_test))} unique classes")
                  score = accuracy  # Fallback to accuracy as the score

              evaluation_metrics = pd.DataFrame({
                  'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
                  'Value': [accuracy, precision, recall, f1]
              })

          # Store metrics in global dictionary for tracking across categories
          if 'category_metrics' not in globals():
              global category_metrics_dict
              category_metrics_dict = {}

          category_metrics_dict[category] = category_metrics

          print(f"\nClassification Report for category {category}:")
          print(classification_report(y_test, y_pred))

          # Confusion Matrix with category label
          cm = confusion_matrix(y_test, y_pred)
          plt.figure(figsize=(10, 8))
          sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
          plt.title(f'Confusion Matrix - {category}')
          plt.ylabel('Actual')
          plt.xlabel('Predicted')
          plt.show()

          # Compare metrics across categories if we have multiple categories
          if category and len(category_metrics_dict) > 1:
              compare_metrics_across_categories(category_metrics_dict)

      if target_options != 'latent_class_depression':
        ci = stats.sem(cv_scores) * stats.t.ppf((1 + 0.95) / 2., len(cv_scores)-1)
      # CI assumes approximate normality of fold scores; may be less
      # precise for AUC with few folds but is standard practice.

      # The spider-plot score variable may be undefined for multiclass
      # targets where per-fold CV scoring is not applicable.
      try:
        category_results[category]['score'] = score
        category_results[category]['score_lower'] = score - ci
        category_results[category]['score_upper'] = score + ci
      except Exception as e:
        print(f"Warning: {e}")

      # Print metrics as a table
      print("\nEvaluation Metrics:")
      display(evaluation_metrics)

      display(
          evaluation_metrics.style
              .format({'Value': '{:.4f}'})
              .background_gradient(cmap='coolwarm', subset=['Value'])
              .set_caption(f'Evaluation Metrics -- {target_options} (Timepoint {tp})')
              .set_table_styles([{'selector': 'caption',
                                  'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
      )

      # After calculating metrics for spider plot (regression only)
      if is_reg:
          category_results[category]['r2'] = r2

    # (Calibration is handled separately in the nested CV cell.)

    # GAM Analysis -- uses CatBoost feature_importance (non-ensemble path only).
    _fi_available = isinstance(globals().get('feature_importance', None), pd.Series)
    if not _run_pdp_gam:
        print("    GAM/PDP analysis skipped (visualization_mode does not include PDP/GAM).")
    else:
      try:
        if not _fi_available:
            raise RuntimeError("feature_importance not available in ensemble mode; skipping GAM.")
        top_10_features = feature_importance.head(10).index
        X_train_top10 = X_train[top_10_features]
        X_test_top10 = X_test[top_10_features]

        # Create GAM formula using feature names
        gam_formula = " + ".join([f"s({i}, label='{name}')" for i, name in enumerate(top_10_features)])

        # Create GAM formula with just the indices
        if is_reg:
            gam = GAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9))
        else:
            gam = GAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6) + s(7) + s(8) + s(9),
                     distribution='binomial', link='logit')

        gam.fit(X_train_top10, y_train)

        # Assign term names after fitting
        feature_mapping = {i: name for i, name in enumerate(top_10_features)}
        gam.terms.feature_names = feature_mapping

        # GAM Visualization
        fig, axs = plt.subplots(2, 5, figsize=(25, 12))
        fig.suptitle(f'GAM Partial Dependence Plots for Top 10 Features\n({target_options}, Timepoint {tp}{", " + category if category else ""})',
                    fontsize=20, y=1.02)

        for i, (term, feature) in enumerate(zip(gam.terms, top_10_features)):
            if term.isintercept:
                continue

            XX = gam.generate_X_grid(term=i)
            pdep, confi = gam.partial_dependence(term=i, X=XX, width=0.95)

            ax = axs[i // 5, i % 5]
            ax.plot(XX[:, term.feature], pdep, color='#ff7f0e', label='Partial Effect')
            ax.fill_between(XX[:, term.feature], confi[:, 0], confi[:, 1], alpha=0.2, color='#ff7f0e')

            feature_values = X_train_top10.iloc[:, i]
            partial_dep = gam.partial_dependence(term=i, X=X_train_top10.values)
            ax.scatter(feature_values, partial_dep, alpha=0.1, color='#1f77b4', label='Data')

            ax.set_title(feature, fontsize=12)
            ax.set_xlabel('Value', fontsize=10)
            ax.set_ylabel('Partial Effect', fontsize=10)
            ax.grid(True, linestyle='--', alpha=0.7)
            ax.tick_params(axis='both', which='major', labelsize=8)

            importance = feature_importance[feature]
            ax.text(0.05, 0.95, f'CatBoost Importance: {importance:.4f}',
                   transform=ax.transAxes, fontsize=10,
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

            if i == 0:
                ax.legend(fontsize=8, loc='lower right')

        plt.tight_layout()
        plt.show()

        # Print GAM summary with feature names(? How)

        print("\nGAM Summary:")
        print(gam.summary())

        # Print feature mapping

        print("\nFeature Mapping:")
        print("-" * 50)
        for i, feature in enumerate(top_10_features):
            print(f"s({i}) = {feature}")

        # GAM Performance
        y_pred_gam = gam.predict(X_test_top10)
        if is_reg:
            mse_gam = mean_squared_error(y_test, y_pred_gam)
            r2_gam = r2_score(y_test, y_pred_gam)
            print(f"\nGAM Performance:\nMSE: {mse_gam:.4f}\nR2: {r2_gam:.4f}")
        else:
            accuracy_gam = accuracy_score(y_test, (y_pred_gam > 0.5).astype(int))
            print(f"\nGAM Performance:\nAccuracy: {accuracy_gam:.4f}")
            print("\nGAM Classification Report:")
            print(classification_report(y_test, (y_pred_gam > 0.5).astype(int)))

      except Exception as e:
        print(f"Error in GAM analysis: {str(e)}")
        print("Not enough dimensionality to do a GAM FI.")

    # SHAP Values Visualization
    if ensemble_version:
      # -- Per-model gating: only run SHAP for models enabled in _VIZ_ENABLED_MODELS --
      _shap_models = {k: v for k, v in model.base_models.items() if k in _VIZ_ENABLED_MODELS}
      if not run_visualizations or not _shap_models:
          print("    SHAP skipped (run_visualizations=False or no models enabled).")
      elif not show_feature_importance_plots and 'metrics_dict' in dir():
          # Pick the best enabled non-TabPFN model for a single SHAP run
          _metric_k = 'r2' if is_reg else 'auc'
          _best_shap_mn   = None
          _best_shap_score = -np.inf
          _skip_models = {'tabpfn', 'ensemble', 'meta_model'}
          for _mn, _mvals in metrics_dict.items():
              if _mn in _skip_models or _mn not in _shap_models:
                  continue
              _sc = _mvals.get(_metric_k, _mvals.get('accuracy', np.nan))
              if not (isinstance(_sc, float) and np.isnan(_sc)) and _sc > _best_shap_score:
                  _best_shap_score = _sc
                  _best_shap_mn    = _mn
          if _best_shap_mn and _best_shap_mn in model.base_models:
              print(f"\n  SHAP: best enabled model → {_best_shap_mn}"
                    f"  ({_metric_k.upper()} = {_best_shap_score:.4f})")
              _orig_base = model.base_models
              model.base_models = {_best_shap_mn: _orig_base[_best_shap_mn]}
              try:
                  shap_values_dict = model.explain_with_shap(X_train, X_test)
              finally:
                  model.base_models = _orig_base
          else:
              print("  Warning:  No enabled model found for SHAP -- skipping.")
      else:
          # Run SHAP only for enabled models
          _orig_base = model.base_models
          model.base_models = _shap_models
          print(f"  SHAP running for: {sorted(_shap_models.keys())}")
          try:
              shap_values_dict = model.explain_with_shap(X_train, X_test)
          finally:
              model.base_models = _orig_base
    elif not run_visualizations:
      print("    SHAP skipped (run_visualizations=False).")
    else:
      explainer = shap.TreeExplainer(model)
      shap_values = explainer.shap_values(X_test)

      shap_values = explainer.shap_values(X_test)

      if shap_values.ndim > 2:
          # individual x features x class

          # 2. SHAP values come as a list of arrays, one per class
          # For multiclass problems, shap_values is a list of arrays (one array per class)
          print(shap_values.shape)
          shap_values_abs = np.apply_along_axis(np.abs, 0, shap_values)

          # 3. Calculate mean SHAP values for each feature per class
          shap_values_mean = np.mean(shap_values_abs, axis=0)  # Average over instances for each class

          sorted_shap_values_mean_sum_idxs = np.argsort(np.sum(shap_values_mean, axis=1))

          idxs = sorted_shap_values_mean_sum_idxs[-10:]

          shap_values_mean = shap_values_mean[idxs, :]

          # 4. Prepare data for stacked bar plot
          n_classes = shap_values_mean.shape[1]
          n_features = 10
          class_names = inverse_map_classes(pd.Series(model.classes_)) # Replace with actual class names if available
          feature_names = X_test.columns[idxs]  # Feature names from your dataset

          # 5. Create the stacked bar plot
          plt.figure(figsize=(12, 8))
          colors = plt.get_cmap("tab10").colors[:n_classes]  # Colors for each class

          # Initialize bottom for stacking
          bottom = np.zeros(n_features)

          # Plot each class's contribution as a stacked bar
          for i, class_name in enumerate(class_names):
              print(feature_names.shape)
              print(shap_values_mean.shape)
              try:
                plt.barh(feature_names, shap_values_mean[:, i], left=bottom, color=colors[i], label=class_name)
              except Exception as e:
                print("Shape mismatch, going to the shap values mean for bottom variable name..")
                bottom = shap_values_mean.shape[1]
                plt.barh(feature_names, shap_values_mean[:, i], left=bottom, color=colors[i], label=class_name)

              bottom += shap_values_mean[:,i]  # Update bottom for stacking

          # 6. Add labels and title
          plt.xlabel('Features', fontsize=14)
          plt.ylabel('Mean SHAP Value (Absolute)', fontsize=14)
          plt.title("Stacked Global SHAP Feature Importance Across Classes", fontsize=16)
          plt.xticks(rotation=90)
          plt.legend(title="Classes", loc="upper right")
          plt.tight_layout()

          # 7. Display the plot
          plt.show()

          # For multi-class problems
          for i in range(0, 4):
            shap.summary_plot(shap_values[:,:,i], X_test, plot_type="bar", show=False)
            plt.title(f"SHAP Feature Importance Class {inverse_map_classes(int(model.classes_[i]))}", fontsize=20)
            plt.tight_layout()
            plt.show()

      if target_options == 'latent_class_depression' or target_options == 'depadhd_c' or target_options == 'asdadhd_c' or isinstance(shap_values, list):

        shap_values_raw = explainer.shap_values(X_test)
        # For multiclass problems
        n_classes = len(shap_values_raw)
        class_names = inverse_map_classes(pd.Series(range(n_classes))).tolist()

        # Global feature importance bar plot
        plt.figure(figsize=(12, 8))
        shap_values_all = np.abs(np.array(shap_values_raw)).mean(0)
        shap_values_exp = shap.Explanation(values=shap_values_all,
                                            data=X_test.values,
                                            feature_names=X_test.columns)
        shap.plots.bar(shap_values_exp, show=False)
        plt.title("Global Feature Importance (All Classes)", fontsize=16)
        plt.tight_layout()
        plt.show()

        # Bar plots for each class
        for i, class_name in enumerate(class_names):
          if i >= 4:
            break
          else:
            plt.figure(figsize=(12, 8))
            shap_values_exp = shap.Explanation(values=shap_values_raw[i],
                                                data=X_test.values,
                                                feature_names=X_test.columns)
            shap.plots.bar(shap_values_exp, show=False)
            plt.title(f"Feature Importance for Class: {class_name}", fontsize=16)
            plt.tight_layout()
            plt.show()

            # Individual SHAP value plot for the first instance of each class
            if len(shap_values_exp) > 0:
              try:
                plt.figure(figsize=(12, 8))
                shap.plots.bar(shap_values_exp[0], show=False)
                plt.title(f"Individual SHAP Values for Class: {class_name} (First Instance)", fontsize=16)
                plt.tight_layout()
                plt.show()
              except Exception as e:
                print("SHAP Values Error:", e)

      else:
        try:
          # For regression or binary classification
          plt.figure(figsize=(20, 16))
          shap.summary_plot(shap_values, X_test, plot_type="violin", show=False, max_display=20)
          plt.title("SHAP Feature Importance and Impact", fontsize=24)
          plt.xlabel("SHAP value (impact on model output)", fontsize=18)
          plt.ylabel("Features", fontsize=18)
          plt.tick_params(axis='both', which='major', labelsize=14)

          # Adjust colorbar
          cbar = plt.gcf().axes[-1]
          cbar.set_ylabel("Feature value", fontsize=18, labelpad=20)
          cbar.tick_params(labelsize=14)

          plt.tight_layout()
          plt.show()
        except Exception as e:
          print("Not enough data to do SHAP feature importance on this class/group prediction.")

    # Compute metrics based on task type
    try:
      if not is_reg:
          y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
          metrics = {
              'Category': category,
              'Time Point': f"T{tp}",
              'Task': 'Classification',
              'Accuracy': accuracy_score(y_test, y_pred),
              'Precision': precision_score(y_test, y_pred, average='weighted'),
              'Recall': recall_score(y_test, y_pred, average='weighted'),
              'F1-Score': f1_score(y_test, y_pred, average='weighted'),
              'ROC-AUC': roc_auc_score(y_test, y_proba) if y_proba is not None else None
          }
      else:
          metrics = {
              'Category': category,
              'Time Point': f"{tp}",
              'Task': 'Regression',
              'MSE': mean_squared_error(y_test, y_pred),
              'MAE': mean_absolute_error(y_test, y_pred),
              'R2': r2_score(y_test, y_pred)
          }

      # Append per-category metrics to the running summary list.
      if split == "within_categories":
        results_summary.append(metrics)
        print(f"Results summary appended with {metrics}")
    except Exception as e:
      print("Not showing summary_metrics table in computing metrics for classifier ensembles.")

    run_shapiq = False
    if ensemble_version and ensemble_type == 'stacking' and run_shapiq:
        # Run ShapIQ analysis

        print("\nRunning advanced ShapIQ interaction analysis...")

        shapiq_results = integrate_shapiq_analysis(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            model=model,
            target_options=target_options,
            tp=tp,
            category=category
            )

    return model, _local_consensus_df, _local_model_weights

# ---
# CACHE SYSTEM -- Main Analysis Runner
# On cache hit: restores all results instantly, skips model training.
# ---
_KEY    = make_run_key()
_cached = cache_load('main_runner', _KEY)
if _cached:
    results_summary      = _cached['results_summary']
    metrics_dict_full    = _cached['metrics_dict_full']
    X_train              = _cached['X_train']
    X_test               = _cached['X_test']
    y_train              = _cached['y_train']
    y_test               = _cached['y_test']
    exported_within_dict = _cached.get('exported_within_dict', {})
    sample_valid         = _cached.get('sample_valid')
    sample_valid_train   = _cached.get('sample_valid_train')
    sample_valid_test    = _cached.get('sample_valid_test')
    _batch_archive       = _cached.get('_batch_archive', {})     # restore batch archive from cache
    batch_mode           = _cached.get('_batch_mode', 'Single Target')
    report_load_state(_KEY)
    print("  Main runner loaded from cache -- skipping model training.")
    _run_main = False
else:
    report_init(_KEY)
    _RS['section'] = 'main_runner'
    _run_main = True

if _run_main:

    # ---
    # _batch_archive stores per-target snapshots so later cells (11, 12,
    # 14-19) can display cross-target summaries. Cleared each run.
    _batch_archive = {}   # {target_name: {results_summary, metrics_dict_full}}

    # -- Helper: create sample_valid for current target ---
    def _create_sample_valid_for_target():
        """Reconstruct sample_valid aligned with current X_train/X_test."""
        import pandas as _pd, numpy as _np
        try:
            _tp = int(tp_option)
            _grp_val = group if 'group' in dir() else None
            _filt = sample[sample['time'] == _tp].copy()
            if _grp_val and _grp_val != "None":
                if _grp_val == 'Female':
                    _filt = _filt[_filt['sex'] == 2]
                elif _grp_val == 'Male':
                    _filt = _filt[_filt['sex'] == 1]
                elif _grp_val == 'high_ale':
                    _filt = _filt[_filt['high_ale'] == True]
                elif _grp_val == 'high_ale_severe_p_mh':
                    _filt = _filt[_filt['high_ale_severe_p_mh'] == True]
                elif _grp_val == 'low_ale':
                    _filt = _filt[_filt['low_ale_children_p'] == True]
            if target_options in _filt.columns:
                _tmask = _filt[target_options].notna()
                if _filt[target_options].dtype == object:
                    _tmask = _tmask & (_filt[target_options].astype(str).str.strip() != '')
                _filt = _filt[_tmask]
            _filt = _filt.reset_index(drop=True)
            _expected = len(X_train) + len(X_test)
            if len(_filt) != _expected:
                return None, None, None
            _sv_train = _filt.iloc[X_train.index.values].reset_index(drop=True)
            _sv_test  = _filt.iloc[X_test.index.values].reset_index(drop=True)
            _sv_full  = _pd.concat([_sv_train, _sv_test], ignore_index=True)
            return _sv_full, _sv_train, _sv_test
        except Exception:
            return None, None, None

    if run_all:
        for tp_option in [0, 1, 2, 3, 4]:
            for group in ["high_ale", "high_ale_severe_p_mh", "Female", "Male", "low_ale", None]:
                for target_options in ["depress_D_p_rev", "depress_D_p", "tb_fluid", "latent_class_depression", "dep_onset_rci_1.96",
                                       "dep_onset_rci_2.3", "dep_remission_rci_1.96", "dep_remission_rci_2.3",
                                       "dep_increase_2sd", "dep_increase_1.5sd", "dep_decrease_2sd", "dep_decrease_1.5sd",
                                       "top_10_depression", "top_5_depression"]:
                    for split in ['across_categories', 'within_categories']:
                        print(f"  [grid] tp={tp_option} group={group} target={target_options} split={split}")
                        run_analysis(tp_option, group, target_options, split, isolate, isolate_category, feature_tp=_feat_tp)
                        _pipeline_cleanup(label=f'run_all tp{tp_option} {target_options}')

    elif not _tp_all_mode and batch_mode != "Single Target" and batch_mode in _TARGET_BUNDLES:
        # -- Batch mode: run each target in the selected bundle ---
        _batch_targets = _TARGET_BUNDLES[batch_mode]
        _batch_summary = []
        _n_batch = len(_batch_targets)

        print(f"\n{'\u2550'*70}")
        print(f"  BATCH MODE: {batch_mode}")
        print(f"  Targets ({_n_batch}): {_batch_targets}")
        print(f"{'\u2550'*70}")

        for _bi, _bt in enumerate(_batch_targets, 1):
            print(f"\n{'\u2501'*70}")
            print(f"  [{_bi}/{_n_batch}]  TARGET: {_bt}")
            print(f"{'\u2501'*70}")

            # Override the global target_options for this iteration
            target_options = _bt

            # -- Set target-specific FI barplot colour ---
            if _bt in _BATCH_TARGET_COLORS:
                ensemble_barplot_color   = _BATCH_TARGET_COLORS[_bt]
                ensemble_barplot_palette = "None"   # force monochrome mode
                ENSEMBLE_VIZ_CONFIG['palette_fn'] = _resolve_ensemble_palette
                print(f"  FI colour: {ensemble_barplot_color}")

            try:
                run_analysis(tp_option, group, target_options, split,
                             isolate, isolate_category, quantile_segment,
                             feature_tp=_feat_tp)

                # -- Create sample_valid for this target --
                sample_valid, sample_valid_train, sample_valid_test = _create_sample_valid_for_target()

                # -- Archive this target's results for later cells --
                import copy
                _batch_archive[_bt] = {
                    'results_summary':   copy.deepcopy(results_summary),
                    'metrics_dict_full': copy.deepcopy(metrics_dict_full),
                    'consensus_df':      copy.deepcopy(consensus_df) if ('consensus_df' in dir() and consensus_df is not None) else None,
                    'model_weights':     copy.deepcopy(model_weights) if ('model_weights' in dir() and model_weights is not None) else None,
                    'X_train':           X_train.copy()  if ('X_train' in dir() and X_train is not None) else None,
                    'y_train':           y_train.copy()  if ('y_train' in dir() and y_train is not None) else None,
                    'X_test':            X_test.copy()   if ('X_test'  in dir() and X_test  is not None) else None,
                    'y_test':            y_test.copy()   if ('y_test'  in dir() and y_test  is not None) else None,
                    'exported_within_dict': copy.deepcopy(exported_within_dict) if ('exported_within_dict' in dir() and isinstance(exported_within_dict, dict)) else {},
                    'split':             split if 'split' in dir() else 'across_categories',
                    'target':            _bt,
                    'category_results':  copy.deepcopy(category_results) if ('category_results' in dir() and isinstance(category_results, dict)) else {},
                    'sample_valid':      sample_valid.copy()       if ('sample_valid'      in dir() and sample_valid      is not None) else None,
                    'sample_valid_test': sample_valid_test.copy()  if ('sample_valid_test'  in dir() and sample_valid_test  is not None) else None,
                    'sample_valid_train':sample_valid_train.copy() if ('sample_valid_train' in dir() and sample_valid_train is not None) else None,
                    'last_fitted_model': copy.deepcopy(last_fitted_model) if ('last_fitted_model' in dir() and last_fitted_model is not None) else None,
                }

                # Harvest the best meta-model result for the summary table
                if metrics_dict_full.get('category'):
                    _best_score = -1e9
                    _best_entry = None
                    for _cat, _mets in zip(metrics_dict_full['category'],
                                           metrics_dict_full['metrics']):
                        if isinstance(_mets, dict) and 'meta_model' in _mets:
                            _mm = _mets['meta_model']
                            if isinstance(_mm, dict):
                                _sc = _mm.get('r2', _mm.get('auc', -1e9))
                                if _sc > _best_score:
                                    _best_score = _sc
                                    _best_entry = dict(_mm)
                                    _best_entry['best_category'] = _cat

                    if _best_entry:
                        _best_entry['target'] = _bt
                        _batch_summary.append(_best_entry)

                _pipeline_cleanup(label=f'batch {_bi}/{_n_batch}')

            except Exception as _be:
                print(f"  Warning: Failed for {_bt}: {_be}")
                import traceback; traceback.print_exc()
                _pipeline_cleanup(label='post-error')  # Cleanup even on failure

        # -- Print cross-target summary table ---
        print(f"\n{'\u2550'*70}")
        print(f"  BATCH COMPLETE: {batch_mode}")
        print(f"{'\u2550'*70}")

        if _batch_summary:
            _batch_df = pd.DataFrame(_batch_summary)
            _priority_cols = ['target', 'best_category']
            _other_cols = [c for c in _batch_df.columns if c not in _priority_cols]
            _batch_df = _batch_df[[c for c in _priority_cols if c in _batch_df.columns] + _other_cols]
            display(_batch_df)

            try:
                _batch_csv = f"batch_{batch_mode.replace(' ', '_').replace('+', '_')}.csv"
                _batch_df.to_csv(_batch_csv, index=False)
                print(f"\n  Saved: {_batch_csv}")
            except Exception as _e:
                print(f"Warning: {_e}")

            batch_results_df = _batch_df
        else:
            print("  No results collected.")
            batch_results_df = pd.DataFrame()

        # After batch, target_options / results_summary / metrics_dict_full
        # all reflect the LAST target. Cells 7-16 will show detail for that
        # target. _batch_archive holds ALL targets for the Batch Metrics panel.

    elif batch_mode in _TP_SWEEP_BUNDLES:
        # -- Timepoint-sweep batch: each entry is a (target, tp) tuple --
        _sweep_targets = _TP_SWEEP_BUNDLES[batch_mode]
        _batch_summary = []
        _n_batch = len(_sweep_targets)

        print(f"\n{'\u2550'*70}")
        print(f"  TP-SWEEP BATCH: {batch_mode}")
        print(f"  Runs ({_n_batch}): {_sweep_targets}")
        print(f"{'\u2550'*70}")

        for _bi, (_bt, _btp) in enumerate(_sweep_targets, 1):
            print(f"\n{'\u2501'*70}")
            print(f"  [{_bi}/{_n_batch}]  TARGET: {_bt}  |  TP: {_btp}")
            print(f"{'\u2501'*70}")

            # Override both global target_options and tp_option
            target_options = _bt
            tp_option = _btp

            # -- Set target-specific FI barplot colour ---
            if _bt in _BATCH_TARGET_COLORS:
                ensemble_barplot_color   = _BATCH_TARGET_COLORS[_bt]
                ensemble_barplot_palette = "None"
                ENSEMBLE_VIZ_CONFIG['palette_fn'] = _resolve_ensemble_palette
                print(f"  FI colour: {ensemble_barplot_color}")

            try:
                run_analysis(tp_option, group, target_options, split,
                             isolate, isolate_category, quantile_segment,
                             feature_tp=_feat_tp)

                # -- Create sample_valid for this target --
                sample_valid, sample_valid_train, sample_valid_test = _create_sample_valid_for_target()

                # Archive with target__tp key to avoid overwrites
                import copy
                _archive_key = f"{_bt}__tp{_btp}"
                _batch_archive[_archive_key] = {
                    'results_summary':   copy.deepcopy(results_summary),
                    'metrics_dict_full': copy.deepcopy(metrics_dict_full),
                    'consensus_df':      copy.deepcopy(consensus_df) if ('consensus_df' in dir() and consensus_df is not None) else None,
                    'model_weights':     copy.deepcopy(model_weights) if ('model_weights' in dir() and model_weights is not None) else None,
                    'X_train':           X_train.copy()  if ('X_train' in dir() and X_train is not None) else None,
                    'y_train':           y_train.copy()  if ('y_train' in dir() and y_train is not None) else None,
                    'X_test':            X_test.copy()   if ('X_test'  in dir() and X_test  is not None) else None,
                    'y_test':            y_test.copy()   if ('y_test'  in dir() and y_test  is not None) else None,
                    'exported_within_dict': copy.deepcopy(exported_within_dict) if ('exported_within_dict' in dir() and isinstance(exported_within_dict, dict)) else {},
                    'split':             split if 'split' in dir() else 'across_categories',
                    'target': _bt,
                    'timepoint': _btp,
                    'category_results':  copy.deepcopy(category_results) if ('category_results' in dir() and isinstance(category_results, dict)) else {},
                    'sample_valid':      sample_valid.copy()       if ('sample_valid'      in dir() and sample_valid      is not None) else None,
                    'sample_valid_test': sample_valid_test.copy()  if ('sample_valid_test'  in dir() and sample_valid_test  is not None) else None,
                    'sample_valid_train':sample_valid_train.copy() if ('sample_valid_train' in dir() and sample_valid_train is not None) else None,
                    'last_fitted_model': copy.deepcopy(last_fitted_model) if ('last_fitted_model' in dir() and last_fitted_model is not None) else None,
                }

                # Harvest the best meta-model result for the summary table
                if metrics_dict_full.get('category'):
                    _best_score = -1e9
                    _best_entry = None
                    for _cat, _mets in zip(metrics_dict_full['category'],
                                           metrics_dict_full['metrics']):
                        if isinstance(_mets, dict) and 'meta_model' in _mets:
                            _mm = _mets['meta_model']
                            if isinstance(_mm, dict):
                                _sc = _mm.get('r2', _mm.get('auc', -1e9))
                                if _sc > _best_score:
                                    _best_score = _sc
                                    _best_entry = dict(_mm)
                                    _best_entry['best_category'] = _cat

                    if _best_entry:
                        _best_entry['target'] = _bt
                        _best_entry['timepoint'] = _btp
                        _batch_summary.append(_best_entry)

                _pipeline_cleanup(label=f'tp-sweep {_bt} tp{_btp}')

            except Exception as _be:
                print(f"  Warning: Failed for {_bt} at tp{_btp}: {_be}")
                import traceback; traceback.print_exc()
                _pipeline_cleanup(label='post-error')

        # -- Print cross-target summary table ---
        print(f"\n{'\u2550'*70}")
        print(f"  TP-SWEEP BATCH COMPLETE: {batch_mode}")
        print(f"{'\u2550'*70}")

        if _batch_summary:
            _batch_df = pd.DataFrame(_batch_summary)
            _priority_cols = ['target', 'timepoint', 'best_category']
            _other_cols = [c for c in _batch_df.columns if c not in _priority_cols]
            _batch_df = _batch_df[[c for c in _priority_cols if c in _batch_df.columns] + _other_cols]
            display(_batch_df)

            try:
                _batch_csv = f"batch_{batch_mode.replace(' ', '_').replace('(', '').replace(')', '')}.csv"
                _batch_df.to_csv(_batch_csv, index=False)
                print(f"\n  Saved: {_batch_csv}")
            except Exception as _e:
                print(f"Warning: {_e}")

            batch_results_df = _batch_df
        else:
            print("  No results collected.")
            batch_results_df = pd.DataFrame()

    elif _tp_all_mode and batch_mode == "Single Target":
        # -- All-TP sweep for a single target ---
        # Iterates T0→T4, running the selected target at every available
        # timepoint. Archives results with target__tpN keys so downstream
        # cells (8, 9, 23) render cross-TP batch panels automatically.
        _all_tps = ["0", "1", "2", "3", "4"]
        _batch_summary = []
        _n_batch = len(_all_tps)

        print(f"\n{'='*70}")
        print(f"  ALL-TP SWEEP (Single Target): {target_options}")
        print(f"  Timepoints: {_all_tps}")
        print(f"{'='*70}")

        for _bi, _btp in enumerate(_all_tps, 1):
            print(f"\n{'-'*70}")
            print(f"  [{_bi}/{_n_batch}]  TARGET: {target_options}  |  TP: {_btp}")
            print(f"{'-'*70}")

            tp_option = _btp   # override global for this iteration

            # -- Set target-specific FI barplot colour ---
            if target_options in _BATCH_TARGET_COLORS:
                ensemble_barplot_color   = _BATCH_TARGET_COLORS[target_options]
                ensemble_barplot_palette = "None"
                ENSEMBLE_VIZ_CONFIG['palette_fn'] = _resolve_ensemble_palette
                print(f"    FI colour: {ensemble_barplot_color}")

            try:
                run_analysis(tp_option, group, target_options, split,
                             isolate, isolate_category, quantile_segment,
                             feature_tp=_feat_tp)

                sample_valid, sample_valid_train, sample_valid_test = _create_sample_valid_for_target()

                import copy
                _archive_key = f"{target_options}__tp{_btp}"
                _batch_archive[_archive_key] = {
                    'results_summary':   copy.deepcopy(results_summary),
                    'metrics_dict_full': copy.deepcopy(metrics_dict_full),
                    'consensus_df':      copy.deepcopy(consensus_df) if ('consensus_df' in dir() and consensus_df is not None) else None,
                    'model_weights':     copy.deepcopy(model_weights) if ('model_weights' in dir() and model_weights is not None) else None,
                    'X_train':           X_train.copy()  if ('X_train' in dir() and X_train is not None) else None,
                    'y_train':           y_train.copy()  if ('y_train' in dir() and y_train is not None) else None,
                    'X_test':            X_test.copy()   if ('X_test'  in dir() and X_test  is not None) else None,
                    'y_test':            y_test.copy()   if ('y_test'  in dir() and y_test  is not None) else None,
                    'exported_within_dict': copy.deepcopy(exported_within_dict) if ('exported_within_dict' in dir() and isinstance(exported_within_dict, dict)) else {},
                    'split':             split if 'split' in dir() else 'across_categories',
                    'target':            target_options,
                    'timepoint':         _btp,
                    'category_results':  copy.deepcopy(category_results) if ('category_results' in dir() and isinstance(category_results, dict)) else {},
                    'sample_valid':      sample_valid.copy()       if (sample_valid      is not None) else None,
                    'sample_valid_test': sample_valid_test.copy()  if (sample_valid_test  is not None) else None,
                    'sample_valid_train':sample_valid_train.copy() if (sample_valid_train is not None) else None,
                    'last_fitted_model': copy.deepcopy(last_fitted_model) if ('last_fitted_model' in dir() and last_fitted_model is not None) else None,
                }

                if metrics_dict_full.get('category'):
                    _best_score = -1e9
                    _best_entry = None
                    for _cat, _mets in zip(metrics_dict_full['category'],
                                           metrics_dict_full['metrics']):
                        if isinstance(_mets, dict) and 'meta_model' in _mets:
                            _mm = _mets['meta_model']
                            if isinstance(_mm, dict):
                                _sc = _mm.get('r2', _mm.get('auc', -1e9))
                                if _sc > _best_score:
                                    _best_score = _sc
                                    _best_entry = dict(_mm)
                                    _best_entry['best_category'] = _cat
                    if _best_entry:
                        _best_entry['target'] = target_options
                        _best_entry['timepoint'] = _btp
                        _batch_summary.append(_best_entry)

                _pipeline_cleanup(label=f'all-tp {target_options} tp{_btp}')

            except Exception as _be:
                print(f"  Warning:  Failed for {target_options} at tp{_btp}: {_be}")
                import traceback; traceback.print_exc()
                _pipeline_cleanup(label='post-error')

        # -- Summary table ---
        print(f"\n{'='*70}")
        print(f"  ALL-TP SWEEP COMPLETE: {target_options}")
        print(f"{'='*70}")

        if _batch_summary:
            _batch_df = pd.DataFrame(_batch_summary)
            _priority_cols = ['target', 'timepoint', 'best_category']
            _other_cols = [c for c in _batch_df.columns if c not in _priority_cols]
            _batch_df = _batch_df[[c for c in _priority_cols if c in _batch_df.columns] + _other_cols]
            display(_batch_df)
            try:
                _batch_csv = f"alltp_{target_options.replace(' ', '_')}.csv"
                _batch_df.to_csv(_batch_csv, index=False)
                print(f"\n  Saved: {_batch_csv}")
            except Exception as _e:
                print(f"Warning: {_e}")
            batch_results_df = _batch_df
        else:
            print("  No results collected.")
            batch_results_df = pd.DataFrame()

    elif _tp_all_mode and batch_mode != "Single Target" and batch_mode in _TARGET_BUNDLES:
        # -- All-TP sweep for a TARGET BUNDLE ---
        # Runs every target in the bundle × every timepoint (T0→T4).
        _all_tps = ["0", "1", "2", "3", "4"]
        _batch_targets = _TARGET_BUNDLES[batch_mode]
        _sweep_pairs = [(t, tp) for t in _batch_targets for tp in _all_tps]
        _batch_summary = []
        _n_batch = len(_sweep_pairs)

        print(f"\n{'='*70}")
        print(f"  ALL-TP × BATCH SWEEP: {batch_mode}")
        print(f"  Targets ({len(_batch_targets)}) × TPs ({len(_all_tps)}) = {_n_batch} runs")
        print(f"{'='*70}")

        for _bi, (_bt, _btp) in enumerate(_sweep_pairs, 1):
            print(f"\n{'-'*70}")
            print(f"  [{_bi}/{_n_batch}]  TARGET: {_bt}  |  TP: {_btp}")
            print(f"{'-'*70}")

            target_options = _bt
            tp_option = _btp

            if _bt in _BATCH_TARGET_COLORS:
                ensemble_barplot_color   = _BATCH_TARGET_COLORS[_bt]
                ensemble_barplot_palette = "None"
                ENSEMBLE_VIZ_CONFIG['palette_fn'] = _resolve_ensemble_palette
                print(f"    FI colour: {ensemble_barplot_color}")

            try:
                run_analysis(tp_option, group, target_options, split,
                             isolate, isolate_category, quantile_segment,
                             feature_tp=_feat_tp)

                sample_valid, sample_valid_train, sample_valid_test = _create_sample_valid_for_target()

                import copy
                _archive_key = f"{_bt}__tp{_btp}"
                _batch_archive[_archive_key] = {
                    'results_summary':   copy.deepcopy(results_summary),
                    'metrics_dict_full': copy.deepcopy(metrics_dict_full),
                    'consensus_df':      copy.deepcopy(consensus_df) if ('consensus_df' in dir() and consensus_df is not None) else None,
                    'model_weights':     copy.deepcopy(model_weights) if ('model_weights' in dir() and model_weights is not None) else None,
                    'X_train':           X_train.copy()  if ('X_train' in dir() and X_train is not None) else None,
                    'y_train':           y_train.copy()  if ('y_train' in dir() and y_train is not None) else None,
                    'X_test':            X_test.copy()   if ('X_test'  in dir() and X_test  is not None) else None,
                    'y_test':            y_test.copy()   if ('y_test'  in dir() and y_test  is not None) else None,
                    'exported_within_dict': copy.deepcopy(exported_within_dict) if ('exported_within_dict' in dir() and isinstance(exported_within_dict, dict)) else {},
                    'split':             split if 'split' in dir() else 'across_categories',
                    'target':            _bt,
                    'timepoint':         _btp,
                    'category_results':  copy.deepcopy(category_results) if ('category_results' in dir() and isinstance(category_results, dict)) else {},
                    'sample_valid':      sample_valid.copy()       if (sample_valid      is not None) else None,
                    'sample_valid_test': sample_valid_test.copy()  if (sample_valid_test  is not None) else None,
                    'sample_valid_train':sample_valid_train.copy() if (sample_valid_train is not None) else None,
                    'last_fitted_model': copy.deepcopy(last_fitted_model) if ('last_fitted_model' in dir() and last_fitted_model is not None) else None,
                }

                if metrics_dict_full.get('category'):
                    _best_score = -1e9
                    _best_entry = None
                    for _cat, _mets in zip(metrics_dict_full['category'],
                                           metrics_dict_full['metrics']):
                        if isinstance(_mets, dict) and 'meta_model' in _mets:
                            _mm = _mets['meta_model']
                            if isinstance(_mm, dict):
                                _sc = _mm.get('r2', _mm.get('auc', -1e9))
                                if _sc > _best_score:
                                    _best_score = _sc
                                    _best_entry = dict(_mm)
                                    _best_entry['best_category'] = _cat
                    if _best_entry:
                        _best_entry['target'] = _bt
                        _best_entry['timepoint'] = _btp
                        _batch_summary.append(_best_entry)

                _pipeline_cleanup(label=f'alltp-batch {_bt} tp{_btp}')

            except Exception as _be:
                print(f"  Warning:  Failed for {_bt} at tp{_btp}: {_be}")
                import traceback; traceback.print_exc()
                _pipeline_cleanup(label='post-error')

        # -- Summary table ---
        print(f"\n{'='*70}")
        print(f"  ALL-TP × BATCH SWEEP COMPLETE: {batch_mode}")
        print(f"{'='*70}")

        if _batch_summary:
            _batch_df = pd.DataFrame(_batch_summary)
            _priority_cols = ['target', 'timepoint', 'best_category']
            _other_cols = [c for c in _batch_df.columns if c not in _priority_cols]
            _batch_df = _batch_df[[c for c in _priority_cols if c in _batch_df.columns] + _other_cols]
            display(_batch_df)
            try:
                _batch_csv = f"batch_alltp_{batch_mode.replace(' ', '_').replace('(', '').replace(')', '')}.csv"
                _batch_df.to_csv(_batch_csv, index=False)
                print(f"\n  Saved: {_batch_csv}")
            except Exception as _e:
                print(f"Warning: {_e}")
            batch_results_df = _batch_df
        else:
            print("  No results collected.")
            batch_results_df = pd.DataFrame()

    else:
        run_analysis(tp_option, group, target_options, split, isolate, isolate_category, quantile_segment, feature_tp=_feat_tp)

    # After an All-TP sweep, tp_option holds the last iteration's value
    # (e.g. "4"). Restore a valid numeric string for the export section.
    if _tp_all_mode:
        if not isinstance(tp_option, str) or tp_option == "All":
            tp_option = "4"  # fallback to last wave

    print("\n" + "=" * 80)
    print("EXPORTING PIPELINE VARIABLES FOR VALIDATION")
    print("=" * 80)

    # Safety init -- ensure sample_valid vars exist even if export fails
    if 'sample_valid_test' not in dir():
        sample_valid_test = None
        sample_valid_train = None
        sample_valid = None

    try:
        t = int(tp_option)
        targets = [target_options]

        if split == 'across_categories':
            exported_data = across_categories_data(
                t, targets, group=group if group else None,
                scale_x=True, scale_y=False, verbose=True
            )
            X_train, X_test, y_train, y_test = exported_data
        else:
            exported_data = within_categories_data(
                t, targets, group=group if group else None,
                scale_x=True, scale_y=False, verbose=True
            )
            # Category selector for within_categories runs.
            _available_cats = list(exported_data.keys())
            print(f"   Available categories ({len(_available_cats)}): {_available_cats[:5]}{'...' if len(_available_cats)>5 else ''}")

            # Use isolate_category if valid, otherwise first key
            _ic = isolate_category if ('isolate_category' in dir() and isolate_category and isolate_category != "None") else None
            if _ic and _ic in exported_data:
                _export_cat = _ic
            else:
                _export_cat = _available_cats[0]
                if _ic:
                    print(f"   Warning:  isolate_category='{_ic}' not found in categories, using first")

            print(f"   Exporting category: '{_export_cat}' for Points 1-5")
            X_train, X_test, y_train, y_test = exported_data[_export_cat]
            exported_within_dict = exported_data  # Store full dict for optional looped validation

        # ================================================================
        # CREATE sample_valid FOR POINTS 2 & 3
        # Must happen BEFORE clean_data (which resets indices)
        # ================================================================
        # X_train.index currently contains scattered positions from
        # RangeIndex(0, N-1) assigned during across_categories_data().
        # These positions map to rows in the filtered data.
        _train_positions = X_train.index.values.copy()
        _test_positions = X_test.index.values.copy()

        # Reconstruct the same row filtering as across_categories_data
        # to get a demographic lookup table with matching row order
        _t_export = int(tp_option)
        _grp = group if group else None

        _filtered_rows = sample[sample['time'] == _t_export].copy()

        if _grp and _grp != "None":
            if _grp == 'Female':
                _filtered_rows = _filtered_rows[_filtered_rows['sex'] == 2]
            elif _grp == 'Male':
                _filtered_rows = _filtered_rows[_filtered_rows['sex'] == 1]
            elif _grp == 'high_ale':
                _filtered_rows = _filtered_rows[_filtered_rows['high_ale'] == True]
            elif _grp == 'high_ale_severe_p_mh':
                _filtered_rows = _filtered_rows[_filtered_rows['high_ale_severe_p_mh'] == True]
            elif _grp == 'low_ale':
                _filtered_rows = _filtered_rows[_filtered_rows['low_ale_children_p'] == True]
            # else: unknown group string, no additional filter

        # Mirror the row-level filtering applied inside the data functions.
        if 'depress_D_p' not in targets and 'depress_D_p_rev' not in targets \
           and "external_D_p" not in targets and "adhd_D_p" not in targets:
            if "latent_class_depression" in targets and "latent_class_depression" in _filtered_rows.columns:
                _before = len(_filtered_rows)
                _filtered_rows = _filtered_rows[_filtered_rows["latent_class_depression"] != ""]
                print(f"   Applied latent_class filter: {_before} → {len(_filtered_rows)} rows")

        # Drop rows with missing target values.
        # The data functions (across/within_categories_data) drop target-NaN
        # rows before train_test_split. We must replicate that here so the
        # row count of _filtered_rows matches len(X_train) + len(X_test).
        if target_options in _filtered_rows.columns:
            _target_mask = _filtered_rows[target_options].notna()
            # Also handle empty-string entries (e.g. categorical targets)
            if _filtered_rows[target_options].dtype == object:
                _target_mask = _target_mask & (_filtered_rows[target_options].astype(str).str.strip() != '')
            if (~_target_mask).any():
                _n_target_na = (~_target_mask).sum()
                print(f"   Dropped {_n_target_na} rows with missing/empty target '{target_options}'")
                _filtered_rows = _filtered_rows[_target_mask]

        _filtered_rows = _filtered_rows.reset_index(drop=True)

        # Verify alignment: row count must match
        _expected_n = len(X_train) + len(X_test)
        if len(_filtered_rows) != _expected_n:
            print(f"   Warning:  sample_valid row mismatch: filtered={len(_filtered_rows)}, "
                  f"data={_expected_n}. Points 2 & 3 may not work.")
            sample_valid = None
            sample_valid_train = None
            sample_valid_test = None
        else:
            sample_valid_train = _filtered_rows.iloc[_train_positions].reset_index(drop=True)
            sample_valid_test = _filtered_rows.iloc[_test_positions].reset_index(drop=True)
            sample_valid = pd.concat([sample_valid_train, sample_valid_test],
                                     ignore_index=True)
            print(f"   sample_valid created: {sample_valid.shape}")
            print(f"      train: {sample_valid_train.shape}, test: {sample_valid_test.shape}")

            # Quick sanity check: sex distribution should be reasonable
            if 'sex' in sample_valid.columns:
                _sex_counts = sample_valid_train['sex'].value_counts().to_dict()
                print(f"      Sex distribution (train): {_sex_counts}")

        # ================================================================
        # Apply same NaN/inf cleaning as main pipeline (BEFORE flattening y)
        # WARNING: This resets X_train/X_test to RangeIndex(0, N-1)
        # ================================================================
        na_method_mapping = {
            "fill_with_zero": ("fill_0", None),
            "drop": ("remove", None),
            "impute_mean": ("impute", "mean"),
            "impute_median": ("impute", "median")
        }
        method, imputer_strategy = na_method_mapping[deal_with_na_values]
        X_train, y_train = clean_data(X_train, y_train, method=method, imputer_strategy=imputer_strategy)
        X_test, y_test = clean_data(X_test, y_test, method=method, imputer_strategy=imputer_strategy)
        X_train = X_train.loc[:, ~X_train.columns.duplicated(keep='first')]
        X_test = X_test.loc[:, ~X_test.columns.duplicated(keep='first')]

        # Post-clean_data safety check: if clean_data dropped rows (method='remove'),
        # sample_valid is now misaligned. Detect and warn.
        if sample_valid_train is not None and len(sample_valid_train) != len(X_train):
            print(f"   Warning:  clean_data changed row count: sample_valid_train={len(sample_valid_train)}, "
                  f"X_train={len(X_train)}. Resetting sample_valid to None.")
            sample_valid = None
            sample_valid_train = None
            sample_valid_test = None

        # Flatten y if it's a DataFrame with one column (AFTER cleaning)
        if hasattr(y_train, 'columns') and len(y_train.columns) == 1:
            y_train = y_train.iloc[:, 0]
        if hasattr(y_test, 'columns') and len(y_test.columns) == 1:
            y_test = y_test.iloc[:, 0]

        print(f"\nExported pipeline variables:")
        print(f"   Target: {target_options}")
        print(f"   Timepoint: {tp_option}")
        print(f"   Split: {split}")
        print(f"   X_train: {X_train.shape}")
        print(f"   X_test:  {X_test.shape}")
        print(f"   y_train: {y_train.shape}, unique values: {len(y_train.unique()) if hasattr(y_train, 'unique') else len(set(y_train))}")
        print(f"   y_test:  {y_test.shape}")

    except Exception as e:
        print(f"\nWarning:  Could not export pipeline variables: {e}")
        print("   Points 1-5 will not have real pipeline data.")
        import traceback
        traceback.print_exc()
    # -- [CACHE SAVE] ---
    _RS['section'] = 'uncategorized'
    try:
        if 'X_train' in dir() and X_train is not None:
            report_log_data_info(X_train, X_test, y_train, y_test)
        if 'results_summary' in dir() and isinstance(results_summary, list) and results_summary:
            try:
                report_log_table(
                    pd.DataFrame(results_summary),
                    'Per-Category Performance', section='main_runner')
            except Exception:
                pass  # non-critical; logged upstream or optional visualization
        cache_save(
            {
                'results_summary':      globals().get('results_summary', []),
                'metrics_dict_full':    globals().get('metrics_dict_full', {}),
                'X_train':              globals().get('X_train'),
                'X_test':               globals().get('X_test'),
                'y_train':              globals().get('y_train'),
                'y_test':               globals().get('y_test'),
                'exported_within_dict': globals().get('exported_within_dict', {}),
                'sample_valid':         globals().get('sample_valid'),
                'sample_valid_train':   globals().get('sample_valid_train'),
                'sample_valid_test':    globals().get('sample_valid_test'),
                '_batch_archive':       _batch_archive if '_batch_archive' in dir() else {},
                '_batch_mode':          batch_mode if 'batch_mode' in dir() else 'Single Target',
            },
            cell_tag='main_runner', key=_KEY,
            label=f"{target_options}_tp{tp_option}_{split}",
        )
    except Exception as _e:
        print(f"[Cache] Could not save main_runner: {_e}")

In [ ]:
#@title Batch Summary Classification

# Summary visualization for batched Classification runs only. Reads _batch_archive
# and produces formatted AUC/F1/precision/recall tables with colormapped
# cells across all targets in the most recent batch. Designed for
# multi-target comparisons (e.g., depression worsening
# thresholds, severity percentiles).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker

# ---
# CONFIG
# ---
_SAVE_FIG    = True
_FIG_DPI     = 300
_RESULTS_DIR = globals().get('RESULTS_DIR', 'results')

_BG      = '#F8F7F3' # exact Sankey background (_DEP_WORSENING_BG)
_INK     = '#0F172A'
_INK_MID = '#475569'
_INK_LT  = '#94A3B8'
_BORDER  = '#E8E6E0' # warm-tinted border
_ROW_ALT = '#EDECEA' # slightly darker for zebra rows

_M_CFG = {
    'accuracy':  ('Accuracy',  0.50, 1.00,
                  LinearSegmentedColormap.from_list('a', ['#FEF2F2','#166534'])),
    'f1':        ('F1',        0.50, 1.00,
                  LinearSegmentedColormap.from_list('f', ['#FFFBEB','#1E3A8A'])),
    'precision': ('Precision', 0.50, 1.00,
                  LinearSegmentedColormap.from_list('p', ['#FAF5FF','#581C87'])),
    'recall':    ('Recall',    0.50, 1.00,
                  LinearSegmentedColormap.from_list('r', ['#ECFDF5','#92400E'])),
    'rmse':      ('RMSE',      None, None,
                  LinearSegmentedColormap.from_list('rm', ['#166534','#991B1B'])),
    'mae':       ('MAE',       None, None,
                  LinearSegmentedColormap.from_list('ma', ['#166634','#991B1B'])),
}

_TIERS = [
    (0.00, 0.65, '#FEE2E2'),
    (0.65, 0.75, '#FEF9C3'),
    (0.75, 0.85, '#DCFCE7'),
    (0.85, 1.01, '#DBEAFE'),
]

_MC = {
    'catboost':      '#0072B2',
    'xgboost':       '#D55E00',
    'random_forest': '#009E73',
    'tabpfn':        '#CC79A7',
    'elastic_net':   '#E69F00',
    'meta_model':    _INK,
    'ensemble':      _INK,
}

# ---
# GUARDS
# ---
_has_batch = ('_batch_archive' in dir() and isinstance(_batch_archive, dict)
              and len(_batch_archive) > 1)
if not _has_batch:
    try:
        _cached = cache_load('main_runner')
        if _cached and isinstance(_cached, dict):
            if _cached.get('_batch_archive') and len(_cached['_batch_archive']) > 1:
                _batch_archive = _cached['_batch_archive']
                _has_batch = True
    except Exception:
        pass

if not _has_batch:
    print("Warning:  Batch Metric Dashboard requires batch mode results.")
else:

    _TDN = globals().get('TARGET_DISPLAY_NAMES', {})

    # ---
    # DATA EXTRACTION
    # ---
    _rows = []
    for _tkey, _tdata in _batch_archive.items():
        _mdf  = _tdata.get('metrics_dict_full', {})
        _tr   = _tdata.get('target',
                            _tkey.split('__tp')[0] if '__tp' in _tkey else _tkey)
        _disp = _TDN.get(_tr, _TDN.get(_tkey, _tkey))

        _yt = _tdata.get('y_train')
        _clf = False
        if _yt is not None:
            try:    _clf = len(np.unique(_yt)) <= 10
            except Exception: pass
        if not _clf:
            try:    _clf = not globals().get('is_regression', lambda t: True)(_tr)
            except Exception: pass

        _best, _mscores = {}, {}
        if _mdf.get('category'):
            for _cat, _mets in zip(_mdf['category'], _mdf['metrics']):
                if isinstance(_mets, dict):
                    for _mn, _mv in _mets.items():
                        if isinstance(_mv, dict):
                            _pk0 = 'auc' if _clf else 'r2'
                            _sc  = _mv.get(_pk0)
                            if _sc is not None:
                                _mscores.setdefault(_mn, []).append(_sc)
                    _mm = _mets.get('meta_model', _mets.get('ensemble', {}))
                    if isinstance(_mm, dict):
                        _pk0 = 'auc' if _clf else 'r2'
                        _sc  = _mm.get(_pk0)
                        if _sc is not None and _sc > _best.get(_pk0, -999):
                            _best = dict(_mm)

        _np2, _nt2 = None, None
        if _yt is not None and _clf:
            try:
                _ya  = np.array(_yt)
                _nt2 = len(_ya)
                _np2 = int(np.sum(_ya == 1))
            except Exception: pass

        _bscores = {k: float(np.mean(v)) for k, v in _mscores.items()
                    if k not in ('meta_model', 'ensemble') and v}

        _rows.append({
            'key': _tkey, 'display': _disp, 'is_clf': _clf,
            'auc':       _best.get('auc'),
            'accuracy':  _best.get('accuracy'),
            'f1':        _best.get('f1', _best.get('f1_weighted')),
            'precision': _best.get('precision'),
            'recall':    _best.get('recall'),
            'r2':        _best.get('r2'),
            'rmse':      _best.get('rmse'),
            'mae':       _best.get('mae'),
            'n_pos': _np2, 'n_total': _nt2,
            'base_scores': _bscores,
        })

    _df = pd.DataFrame(_rows)
    _all_clf = _df['is_clf'].all()
    _all_reg = (~_df['is_clf']).all()

    _pk  = 'auc' if _all_clf else ('r2' if _all_reg else 'auc')
    _pkl = 'AUC' if _pk == 'auc' else 'R²'
    _df  = _df.sort_values(_pk, ascending=True,
                           na_position='first').reset_index(drop=True)
    _n   = len(_df)

    _sec_keys = (['accuracy', 'f1', 'precision', 'recall'] if _all_clf
                 else ['rmse', 'mae']                        if _all_reg
                 else ['accuracy', 'f1', 'rmse', 'mae'])
    _n_sec = len(_sec_keys)

    # ---
    # FIGURE -- column headers only, no figure-level text at all
    # ---
    _rh  = 0.42
    _fh  = _n * _rh + 0.90
    _fw  = 13.5

    fig = plt.figure(figsize=(_fw, _fh), facecolor=_BG)

    # widened panel 0 from 3.0 → 3.4 to accommodate long target names
    _wr = [3.4, 3.4, _n_sec * 1.15, 1.20]
    gs  = gridspec.GridSpec(
        1, 4, width_ratios=_wr,
        wspace=0.018,
        left=0.010, right=0.988,
        top=0.920, bottom=0.110)

    _ylim = (-0.5, _n - 0.5)

    def _zebra(ax):
        for i in range(_n):
            if i % 2 == 0:
                ax.axhspan(i - 0.5, i + 0.5,
                           facecolor=_ROW_ALT, alpha=0.55, zorder=0)

    def _spine(ax, bottom=True):
        for s in ax.spines.values():
            s.set_visible(False)
        if bottom:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_color(_BORDER)
            ax.spines['bottom'].set_linewidth(0.4)

    # ---
    # PANEL 0 -- Target labels
    # ---
    ax_n = fig.add_subplot(gs[0, 0], facecolor=_BG)
    _zebra(ax_n)

    for i, row in _df.iterrows():
        ax_n.text(0.02, i, row['display'],
                  va='center', ha='left',
                  fontsize=11, fontweight='600', color=_INK,
                  transform=ax_n.get_yaxis_transform(),
                  clip_on=False)

    ax_n.set_xlim(0, 1); ax_n.set_ylim(*_ylim)
    ax_n.set_yticks([]);  ax_n.set_xticks([])
    ax_n.set_title('Target', fontsize=10, fontweight='600',
                   color=_INK, pad=5, loc='left')
    _spine(ax_n, bottom=False)

    # ---
    # PANEL 1 -- AUC bar track
    # ---
    ax_a = fig.add_subplot(gs[0, 1], facecolor=_BG)
    _zebra(ax_a)

    _vals = _df[_pk].fillna(0).astype(float).values
    _dmin = float(_df[_pk].dropna().min())
    _dmax = float(_df[_pk].dropna().max())
    _ref  = 0.5 if _pk == 'auc' else 0.0
    _xlo  = max(_ref, np.floor((_dmin - 0.03) * 20) / 20)
    _xhi  = min(1.00, np.ceil((_dmax + 0.02)  * 20) / 20)

    for _t0, _t1, _tc in _TIERS:
        _a = max(_t0, _xlo); _b = min(_t1, _xhi + 0.04)
        if _b > _a:
            ax_a.axvspan(_a, _b, facecolor=_tc, alpha=0.35, zorder=0)

    if _ref >= _xlo:
        ax_a.axvline(_ref, color=_INK_LT, linewidth=0.65,
                     linestyle='--', alpha=0.45, zorder=1)

    for i in range(_n):
        row = _df.iloc[i]
        v   = float(_vals[i])
        bs  = row['base_scores'] if isinstance(row['base_scores'], dict) else {}

        ax_a.add_patch(FancyBboxPatch(
            (_xlo, i - 0.155), _xhi - _xlo, 0.31,
            boxstyle='round,pad=0.005',
            facecolor='#E2E8F0', edgecolor='none', zorder=1))

        _bw = max(0.0, v - _ref)
        if _bw > 0:
            ax_a.add_patch(FancyBboxPatch(
                (_ref, i - 0.155), _bw, 0.31,
                boxstyle='round,pad=0.005',
                facecolor='#334155', edgecolor='none', alpha=0.80, zorder=2))

        ax_a.text(v + 0.003, i, f'{v:.3f}',
                  va='center', ha='left',
                  fontsize=9.5, fontweight='600', color=_INK,
                  path_effects=[pe.withStroke(linewidth=2.0, foreground=_BG)],
                  zorder=5)

        if bs:
            _sv = list(bs.values())
            ax_a.plot([min(_sv), max(_sv)], [i - 0.30, i - 0.30],
                      color=_INK_LT, linewidth=0.8, alpha=0.4,
                      zorder=3, solid_capstyle='round')
            for mn, ms in bs.items():
                ax_a.scatter(ms, i - 0.30,
                             color=_MC.get(mn, _INK_LT),
                             s=16, zorder=4, edgecolors=_BG, linewidth=0.4)

    ax_a.set_xlim(_xlo - 0.005, _xhi + 0.070)
    ax_a.set_ylim(*_ylim)
    ax_a.set_yticks([])
    ax_a.tick_params(axis='x', labelsize=8, colors=_INK_MID, pad=2)
    ax_a.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax_a.set_title(f'Meta-model {_pkl}',
                   fontsize=10, fontweight='600', color=_INK, pad=5, loc='left')
    _spine(ax_a, bottom=True)
    ax_a.grid(axis='x', alpha=0.12, color=_BORDER, linewidth=0.35, zorder=0)

    _all_mn = set()
    for _r in _rows:
        if isinstance(_r.get('base_scores'), dict):
            _all_mn.update(_r['base_scores'].keys())
    _mlh = [Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=_MC.get(m, '#888'),
                   markersize=4.5,
                   label=m.replace('_', ' ').title())
            for m in _MC if m in _all_mn]
    if _mlh:
        _leg = ax_a.legend(handles=_mlh, fontsize=7, loc='lower right',
                           framealpha=0.95, edgecolor=_BORDER, fancybox=False,
                           ncol=1, handletextpad=0.3,
                           borderpad=0.4, labelspacing=0.18)
        _leg.set_title('Base learners', prop={'size': 7})

    # ---
    # PANEL 2 -- Secondary Metrics (column headers only, no panel title)
    # ---
    ax_m = fig.add_subplot(gs[0, 2], facecolor=_BG)
    _zebra(ax_m)

    for _jd in range(1, _n_sec):
        ax_m.axvline(_jd - 0.5, color=_BORDER, linewidth=0.35, alpha=0.65, zorder=1)

    _mat = np.full((_n, _n_sec), np.nan)
    for _i, _row in _df.iterrows():
        for _j, _mk in enumerate(_sec_keys):
            _v = _row.get(_mk)
            if _v is not None: _mat[_i, _j] = float(_v)

    for _j, _mk in enumerate(_sec_keys):
        _lbl, _vmin0, _vmax0, _cm0 = _M_CFG[_mk]
        _col   = _mat[:, _j]
        _valid = _col[~np.isnan(_col)]
        if not len(_valid): continue

        _vn = _vmin0 if _vmin0 is not None else float(_valid.min())
        _vx = _vmax0 if _vmax0 is not None else float(_valid.max())
        if _vn >= _vx: _vx = _vn + 0.01
        _norm = Normalize(vmin=_vn, vmax=_vx)

        for _i in range(_n):
            _v = _mat[_i, _j]
            if np.isnan(_v): continue
            _fc = _cm0(_norm(_v))

            ax_m.add_patch(FancyBboxPatch(
                (_j - 0.43, _i - 0.34), 0.86, 0.68,
                boxstyle='round,pad=0.030',
                facecolor=_fc, edgecolor=_BG, linewidth=1.2, zorder=2))

            _lum = 0.299 * _fc[0] + 0.587 * _fc[1] + 0.114 * _fc[2]
            _tc  = 'white' if _lum < 0.46 else _INK
            ax_m.text(_j, _i, f'{_v:.3f}',
                      ha='center', va='center',
                      fontsize=9.5, fontweight='600',
                      color=_tc, zorder=3)

    ax_m.set_xlim(-0.58, _n_sec - 0.42)
    ax_m.set_ylim(*_ylim)
    ax_m.set_yticks([])
    ax_m.set_xticks(range(_n_sec))
    # Metric names as column headers only -- no panel title above them
    ax_m.set_xticklabels([_M_CFG[k][0] for k in _sec_keys],
                          fontsize=10, fontweight='600', color=_INK)
    ax_m.xaxis.set_ticks_position('top')
    ax_m.xaxis.set_tick_params(length=0, pad=5)
    ax_m.set_title('', pad=0)   # explicitly empty
    for _sp in ax_m.spines.values(): _sp.set_visible(False)

    # ---
    # PANEL 3 -- Positive class %
    # ---
    ax_p = fig.add_subplot(gs[0, 3], facecolor=_BG)
    _zebra(ax_p)

    if _all_clf:
        for _i, _row in _df.iterrows():
            _np3 = _row.get('n_pos')
            _nt3 = _row.get('n_total')
            if _np3 is not None and _nt3 and _nt3 > 0:
                _pct = _np3 / _nt3
                _vis = max(_pct, 0.022)

                ax_p.add_patch(FancyBboxPatch(
                    (0.0, _i - 0.145), 1.0, 0.29,
                    boxstyle='round,pad=0.004',
                    facecolor='#E2E8F0', edgecolor='none', zorder=1))
                ax_p.add_patch(FancyBboxPatch(
                    (1.0 - _vis, _i - 0.145), _vis, 0.29,
                    boxstyle='round,pad=0.004',
                    facecolor='#2563EB', edgecolor='none', alpha=0.78, zorder=2))

                _lx = 1.0 - _vis / 2 if _pct > 0.12 else 1.0 - _vis - 0.04
                _ha = 'center' if _pct > 0.12 else 'right'
                _tc = 'white'  if _pct > 0.12 else _INK_MID
                ax_p.text(_lx, _i, f'{_pct:.1%}',
                           ha=_ha, va='center',
                           fontsize=8.5, fontweight='600',
                           color=_tc, zorder=3)

    ax_p.set_xlim(0, 1); ax_p.set_ylim(*_ylim)
    ax_p.set_yticks([])
    ax_p.set_xticks([0, 0.5, 1.0])
    ax_p.set_xticklabels(['0%', '50%', '100%'], fontsize=7.5, color=_INK_MID)
    ax_p.set_title('Pos. class %', fontsize=10, fontweight='600',
                   color=_INK, pad=5)
    _spine(ax_p, bottom=True)

    # ---
    # SAVE & SHOW
    # ---
    try:
        import os
        os.makedirs(_RESULTS_DIR, exist_ok=True)
        _out = f'{_RESULTS_DIR}/batch_metric_dashboard.png'
        fig.savefig(_out, dpi=_FIG_DPI, bbox_inches='tight', facecolor=_BG)
        print(f"  Saved: {_out}")
    except Exception as _e:
        print(f"  Warning:  Save error: {_e}")

    plt.show()
    print(f"\n   Batch Metric Dashboard -- {_n} targets rendered.")

In [ ]:
#@title Sankey Batch Visualization

# Sankey diagrams visualizing across-category consensus feature importance
# across configurable target/timepoint batches. Reads from _batch_archive.
# Shared features annotated (nT) across targets; ribbon opacity encodes
# weighted ensemble importance. Multiple style presets control target
# grouping, color scheme, layout, and which batch results to pull:
#   developmental_sweep / parent_developmental_sweep  -- longitudinal
#   pfactor_spectra / child_pfactor / parent_pfactor   -- int/ext spectra
#   child_depression / parent_depression               -- clf severity/change
#   sbt_long                                           -- subtype sweeps
#   primary_targets                                    -- adult 6-target panel
#   child_suicide / parent_suicide                     -- suicidal ideation
#   sbt_core_triad_female / _male                      -- sex-stratified
#   child_deltas / parent_deltas                       -- change-score batches
#   child_academic_cognitive                           -- academic/cognitive batch
#   child_adhd_impcomp                                 -- ADHD/Imp/Comp batch
#   parent_social_impcomp                              -- Parent Social/Imp/Comp batch
#@markdown Full Sankey visualization with multiple heatflow-style variants and batch execution.

# ---
# CONFIGURATION PARAMETERS
# ---

show_sankey = True  #@param {type: "boolean"}
sankey_top_features = 2  #@param {type: "integer"}
sankey_style = "child_adhd_impcomp"  #@param ["developmental_sweep", "parent_developmental_sweep", "pfactor_spectra", "child_pfactor", "parent_pfactor", "child_suicide", "parent_suicide", "child_depression", "parent_depression", "parent_deltas", "child_deltas", "child_academic_cognitive", "child_adhd_impcomp", "parent_social_impcomp", "child_alts", "parent_alts", "sbt_female", "sbt_long", "primary_targets", "sbt_core_triad", "sbt_core_triad_female", "sbt_core_triad_male", "heatflow_magenta", "heatflow_fire", "heatflow_ice", "heatflow_dark", "child_cogtask_t0", "run_all_styles"] {type: "string"}
sankey_landscape = False  #@param {type: "boolean"}

# ---
# IMPORTS AND SETUP
# ---

import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
import pathlib as _pathlib
from collections import defaultdict
from matplotlib.patches import FancyBboxPatch, Circle, Ellipse, Polygon, Rectangle
from matplotlib.colors import Normalize, LinearSegmentedColormap, to_rgba, to_hex, rgb_to_hsv, hsv_to_rgb
from matplotlib.path import Path
import matplotlib.patheffects as path_effects
import matplotlib.cm as mcm
from matplotlib.collections import LineCollection

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

try:
    _RS['section'] = 'enhanced_sankey'
except Exception:
    pass

# ---
# ENHANCED STYLE CONSTANTS & COLOR PALETTES
# ---

_NAVY = '#1B2A4A'
_STEEL = '#2C3E50'
_ACCENT = '#2980B9'
_LIGHT = '#ECF0F1'

# ---
# UNIFIED TITLE / SUBTITLE -- applied identically to every style
# ---

_SANKEY_TITLE    = 'Shared Predictive Structure Across Targets'
_SANKEY_SUBTITLE = ('Shared Top 10 Ensemble Consensus Predictive Features  '
                    '|  # Shared Across Targets (nT)')
_TITLE_COLOR     = '#000000'
_SUBTITLE_COLOR  = '#333333'
_TITLE_FONT      = 'DejaVu Sans'

# ---
# WIDER HORIZONTAL GAP -- push right column further right for more ribbon sweep
# ---

_LEFT_X  = 0.0
_RIGHT_X = 1.30

# ---
# HEATFLOW COLOR CONFIGURATIONS
# ---

HEATFLOW_THEMES = {
    'heatflow_magenta': {
        'name': 'Magenta Flow',
        'cmap': 'PuRd',
        'bg_color': '#FFFFFF',
        'zone_colors': ['#F3E5F5', '#E1BEE7', '#CE93D8', '#BA68C8'],
        'target_node_edge': '#6A1B9A',
        'feature_glow': '#E91E63',
        'text_color': '#4A148C',
        'header_color': '#8E24AA',
        'accent_color': '#C2185B',
    },
    'heatflow_fire': {
        'name': 'Inferno Flow',
        'cmap': 'inferno',
        'bg_color': '#FFFFFF',
        'zone_colors': ['#FFECB3', '#FFD54F', '#FFCA28', '#FFC107'],
        'target_node_edge': '#BF360C',
        'feature_glow': '#FF3D00',
        'text_color': '#3E2723',
        'header_color': '#DD2C00',
        'accent_color': '#FF6D00',
    },
    'heatflow_ice': {
        'name': 'Arctic Flow',
        'cmap': 'winter',
        'bg_color': '#FFFFFF',
        'zone_colors': ['#E1F5FE', '#B3E5FC', '#81D4FA', '#4FC3F7'],
        'target_node_edge': '#01579B',
        'feature_glow': '#03A9F4',
        'text_color': '#0D47A1',
        'header_color': '#0277BD',
        'accent_color': '#0288D1',
    },
    'heatflow_dark': {
        'name': 'Midnight Flow',
        'cmap': 'copper',
        'bg_color': '#FFFFFF',
        'zone_colors': ['#16213E', '#0F3460', '#1A1A2E', '#16213E'],
        'target_node_edge': '#E94560',
        'feature_glow': '#E94560',
        'text_color': '#EAEAEA',
        'header_color': '#E94560',
        'accent_color': '#FF6B6B',
        'dark_mode': True,
    },
}

def get_target_colors_for_theme(theme_key, n_targets):
    """Generate visually distinct target colors based on theme."""
    theme = HEATFLOW_THEMES[theme_key]

    base_hues = {
        'heatflow_magenta': [0.85, 0.88, 0.82, 0.9, 0.8, 0.92],
        'heatflow_fire': [0.05, 0.08, 0.02, 0.1, 0.12, 0.15],
        'heatflow_ice': [0.55, 0.58, 0.52, 0.6, 0.5, 0.62],
        'heatflow_dark': [0.05, 0.08, 0.12, 0.15, 0.18, 0.2],
    }

    hues = base_hues.get(theme_key, [0.5 + i * 0.1 for i in range(n_targets)])
    colors = []

    for i in range(n_targets):
        hue = hues[i % len(hues)]
        saturation = 0.7 if 'pastel' not in theme_key else 0.4
        value = 0.85 if 'dark' not in theme_key and 'neon' not in theme_key else 0.9
        rgb = hsv_to_rgb((hue, saturation, value))
        colors.append(rgb)

    return colors

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Helvetica Neue', 'Helvetica', 'Arial'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.pad_inches': 0.15,
    'axes.linewidth': 0,
})

# ---
# WIDE-LAYOUT
# ---

def _wide_figure_dims(n_targets, n_features, landscape=None):
    """
    Width-first figure sizing.

    Width ≥ 18″ for generous ribbon sweep.  Height is driven by the
    *feature* column (which almost always has more items), with a
    comfortable per-feature multiplier so labels and ribbons stay
    readable.  Targets get extra room via equalize_vertical_envelopes.
    """
    if landscape is None:
        landscape = globals().get('sankey_landscape', False)

    fig_w = max(18, 16 + max(n_targets - 4, 0) * 0.5)
    # Features drive height -- generous spacing so ribbon connections
    # remain individually traceable even with 40-60 features.
    min_h  = max(n_features * 0.42,              # readable per-feature
                 n_targets  * 0.75,              # readable per-target
                 10.0)
    fig_h  = min_h                               # let content drive it

    if landscape:
        fig_w = max(fig_w + 2, 20)
        fig_h = max(n_features * 0.34, n_targets * 0.55, 8.0)

    return fig_w, fig_h

def _equalize_vertical_envelopes(target_positions, feature_positions,
                                  y_top=0.95, y_bot_margin=0.05,
                                  target_span_frac=0.60):
    """
    Features keep their natural spacing.  Targets are shifted so their
    top aligns with the feature-column top, and spread to occupy
    target_span_frac of the feature envelope -- enough to look balanced
    without the extreme gaps that come from filling the entire height.
    """
    if not target_positions or not feature_positions:
        return target_positions, feature_positions

    def _extent(pos):
        return (max(p['y_top'] for p in pos.values()),
                min(p['y_bot'] for p in pos.values()))

    t_top, t_bot = _extent(target_positions)
    f_top, f_bot = _extent(feature_positions)
    t_span = t_top - t_bot
    f_span = f_top - f_bot

    if t_span <= 0 or f_span <= 0:
        return target_positions, feature_positions

    # Targets occupy target_span_frac of the feature span, top-aligned
    target_span = f_span * target_span_frac
    # But never shrink targets below their natural span
    target_span = max(target_span, t_span)
    # And never exceed the full feature span
    target_span = min(target_span, f_span)

    ref_top = f_top                  # top-align with features
    ref_bot = ref_top - target_span

    def _rescale_targets(positions, old_top, old_bot):
        old_span = old_top - old_bot
        if old_span <= 0:
            return positions
        new_span = ref_top - ref_bot
        new = {}
        for key, p in positions.items():
            frac  = (p['y_centre'] - old_bot) / old_span
            new_c = ref_bot + frac * new_span
            h     = p['h']
            new[key] = {
                'y_centre': new_c,
                'y_top':    new_c + h / 2,
                'y_bot':    new_c - h / 2,
                'h':        h,
            }
        return new

    target_positions = _rescale_targets(target_positions, t_top, t_bot)
    return target_positions, feature_positions

# ---
# LANDSCAPE ADJUSTMENT HELPER (unchanged logic, adapts on top of wide defaults)
# ---

def _landscape_adjust(fig_w, fig_h, spread_target, spread_feature,
                      n_targets, n_features, landscape=None):
    """
    Adjust figure dimensions and spread factors for landscape orientation.

    When landscape is True:
      - Width is increased by ~30% (min 18")
      - Height is compressed: uses tighter per-feature / per-target multipliers
        but still enforced at a sane minimum so labels never overlap
      - Target and feature spread_factors are reduced proportionally

    Returns (fig_w, fig_h, spread_target, spread_feature).
    """
    if landscape is None:
        landscape = globals().get('sankey_landscape', False)
    if not landscape:
        return fig_w, fig_h, spread_target, spread_feature

    # -- Width: widen for more ribbon sweep --
    fig_w = max(fig_w + 4, 20)

    # -- Height: compress but keep labels legible --
    fig_h_feat = n_features * 0.32
    fig_h_tgt  = n_targets  * 0.55
    fig_h = max(7.5, fig_h_feat, fig_h_tgt)

    # -- Spread factors: compress vertical gaps proportionally --
    spread_target  = spread_target  * 0.55
    spread_feature = spread_feature * 0.72

    return fig_w, fig_h, spread_target, spread_feature

def _landscape_fonts(landscape=None):
    """
    Return (target_fs, feature_fs_max, metric_fs, subtitle_fs) adjusted
    for landscape mode.  Portrait values are the existing defaults.
    """
    if landscape is None:
        landscape = globals().get('sankey_landscape', False)
    if landscape:
        return 12, 10, 9.5, 8.5      # target, feat_cap, metric, subtitle
    return 14, 12, 11.5, 9.5          # portrait defaults

# ---
# SHARED HELPERS
# ---

def _get_tdn(t):
    """Resolve display name, handling __tp archive keys gracefully."""
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    if t in _tdn:
        return _tdn[t]
    # Handle __tpN archive keys: 'sbt_core_depression__tp2' -> 'Mood/Depression (T2)'
    if '__tp' in t:
        base, tp_suffix = t.split('__tp', 1)
        base_display = _tdn.get(base, base)
        return f'{base_display} (T{tp_suffix})'
    return t

_TDN = globals().get('TARGET_DISPLAY_NAMES', {})  # legacy alias -- prefer _get_tdn()

# ---
# Timepoint-to-age relabeler (ABCD canonical: T0=~9y, T1=~10y, T2=~11y, T3=~12y, T4=~13y).
# Substitutes any trailing '(TN)' suffix with '(Age {9+N})'. Returns the label unchanged
# if no TP suffix is present. Callable from any Sankey draw function that wants
# age-based x-axis labels instead of ABCD wave indices.
# ---
_ABCD_TP_AGE = {'0': 9, '1': 10, '2': 11, '3': 12, '4': 13}
_ABCD_AGE_TP = {v: k for k, v in _ABCD_TP_AGE.items()}

def _tp_suffix_to_age(label):
    """Replace trailing '(TN)' with '(Age {9+N})' using the ABCD convention.

    Examples
    --------
    'Crystallized Intelligence (T2)' -> 'Crystallized Intelligence (Age 11)'
    'Poor Academic Grades (T4)'      -> 'Poor Academic Grades (Age 13)'
    'Some Target'                     -> 'Some Target'  (no change)
    """
    import re
    m = re.search(r'\(T([0-4])\)$', str(label))
    if not m:
        return label
    tp = m.group(1)
    return str(label)[:m.start()] + f'(Age {_ABCD_TP_AGE[tp]})'

def _age_label_to_tp(label):
    """Inverse of _tp_suffix_to_age -- reverse '(Age N)' back to '(TN-9)'.

    Used by metric lookups where _target_metrics was keyed on the original
    TP-format strings before relabeling.  Returns the input unchanged if
    no '(Age N)' suffix is present or the age is outside the ABCD range.
    """
    import re
    m = re.search(r'\(Age (\d+)\)$', str(label))
    if not m:
        return label
    age = int(m.group(1))
    if age not in _ABCD_AGE_TP:
        return label
    return str(label)[:m.start()] + f'(T{_ABCD_AGE_TP[age]})'

def _tp_suffix_to_year(label):
    """Replace trailing '(TN)' with '(Year N)' -- years-from-baseline framing.

    Use for parent self-report targets where the child-age framing of
    _tp_suffix_to_age is misleading (the ABCD age suffix refers to the child,
    not the parent rater).
    """
    import re
    m = re.search(r'\(T([0-4])\)$', str(label))
    if not m:
        return label
    return str(label)[:m.start()] + f'(Year {m.group(1)})'


def _year_label_to_tp(label):
    """Inverse of _tp_suffix_to_year -- reverse '(Year N)' back to '(TN)'."""
    import re
    m = re.search(r'\(Year ([0-4])\)$', str(label))
    if not m:
        return label
    return str(label)[:m.start()] + f'(T{m.group(1)})'

def _safe_save_fig(fig, directory, basename, dpi=600, formats=('png', 'pdf')):
    """Save figure to directory in multiple formats with error handling."""
    os.makedirs(directory, exist_ok=True)
    saved = []
    for fmt in formats:
        try:
            path = os.path.join(directory, f'{basename}.{fmt}')
            fig.savefig(path, dpi=dpi if fmt == 'png' else 300,
                       bbox_inches='tight', facecolor=fig.get_facecolor(),
                       backend='Agg' if fmt == 'png' else None,
                       metadata={'Creator': 'matplotlib'} if fmt == 'pdf' else {})
            saved.append(path)
        except Exception as e:
            print(f"  Warning:  Could not save {fmt}: {e}")
    return saved

def _get_cmap_safe(name='Set2'):
    """Get colormap compatible with old and new matplotlib."""
    try:
        return plt.colormaps[name]
    except (AttributeError, KeyError):
        return plt.cm.get_cmap(name)

def _create_custom_cmap(colors, name='custom'):
    """Create a custom colormap from a list of colors."""
    return LinearSegmentedColormap.from_list(name, colors)

def _luminance(color):
    """Relative luminance per WCAG 2.1."""
    rgba = to_rgba(color)
    rgb = np.array(rgba[:3])
    rgb = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    return 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]

def _contrast_ratio(c1, c2):
    """WCAG 2.1 contrast ratio between two colors."""
    l1, l2 = _luminance(c1), _luminance(c2)
    return (max(l1, l2) + 0.05) / (min(l1, l2) + 0.05)

def _ensure_contrast(bg_color, text_color, min_ratio=4.5):
    """Return text_color if contrast is sufficient, else flip to black or white."""
    if _contrast_ratio(bg_color, text_color) >= min_ratio:
        return text_color
    return '#000000' if _luminance(bg_color) > 0.5 else '#FFFFFF'

def _draw_sankey_title(ax, sankey_top_features):
    """Draw subtitle at bottom. Top margin preserved for manual title overlay."""
    # Invisible spacer so bbox_inches='tight' keeps the top margin
    ax.text(0.5, 1.06, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)
    # Subtitle moved to bottom of figure
    ax.text(0.5, -0.04, _SANKEY_SUBTITLE,
            ha='center', va='top', fontsize=10, fontweight='normal',
            color=_SUBTITLE_COLOR, fontfamily=_TITLE_FONT,
            transform=ax.transAxes)

def _draw_sankey_title_abs(ax, x_centre, y_title, y_subtitle):
    """Subtitle at bottom. Top margin preserved for manual title overlay."""
    # Invisible spacer so bbox_inches='tight' keeps the top margin
    ax.text(x_centre, y_title, ' ', ha='center', va='bottom', fontsize=17, alpha=0.0)
    # Subtitle moved to bottom of figure
    ax.text(0.5, -0.04, _SANKEY_SUBTITLE,
            ha='center', va='top', fontsize=10, fontweight='normal',
            color=_SUBTITLE_COLOR, fontfamily=_TITLE_FONT,
            transform=ax.transAxes)

# ---
# SANKEY DATA EXTRACTION
# ---

def _extract_sankey_data(batch_archive, top_n=10, normalize=False):
    """Extract Sankey flow data from batch archive consensus DataFrames."""
    rows = []
    for target, archive in batch_archive.items():
        cdf = archive.get('consensus_df')
        if cdf is None:
            continue
        if hasattr(cdf, 'empty') and cdf.empty:
            continue
        if not hasattr(cdf, 'columns'):
            continue
        if 'weighted_score' not in cdf.columns:
            print(f"  Warning:  '{target}' consensus_df missing 'weighted_score' column.")
            continue
        _cdf = cdf.dropna(subset=['weighted_score'])
        if _cdf.empty:
            continue
        _cdf_sorted = _cdf.sort_values('weighted_score', ascending=False).head(top_n)
        print(f"    {_get_tdn(target)}: consensus_df has {len(_cdf)} rows → took top {len(_cdf_sorted)}")

        _values = _cdf_sorted['weighted_score'].values.astype(float)
        if normalize:
            _vmin = _values.min()
            _vmax = _values.max()
            if _vmax > _vmin:
                _values = (_values - _vmin) / (_vmax - _vmin)
            else:
                _values = np.ones_like(_values)

        for (_feat_name, _row), _imp in zip(_cdf_sorted.iterrows(), _values):
            rows.append({
                'Target': _get_tdn(target),
                'Feature': str(_feat_name),
                'Importance': float(_imp),
            })
    return rows

def _sankey_common_data(df_sankey):
    """Compute common Sankey layout data structures."""
    targets_list = df_sankey['Target'].unique().tolist()
    feat_target_count = df_sankey.groupby('Feature')['Target'].nunique()
    feat_total_imp = df_sankey.groupby('Feature')['Importance'].sum()

    feat_order = (
        pd.DataFrame({'n_targets': feat_target_count, 'total_imp': feat_total_imp})
        .sort_values(['n_targets', 'total_imp'], ascending=[False, False])
        .index.tolist()
    )
    features_list = feat_order
    target_totals = df_sankey.groupby('Target')['Importance'].sum()
    feature_totals = df_sankey.groupby('Feature')['Importance'].sum()

    return {
        'targets_list': targets_list,
        'features_list': features_list,
        'feat_target_count': feat_target_count,
        'feat_total_imp': feat_total_imp,
        'target_totals': target_totals,
        'feature_totals': feature_totals,
        'n_targets': len(targets_list),
        'n_features': len(features_list),
    }

def _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features):
    """Print summary stats and feature overlap matrix."""
    n_transdiag = int((feat_target_count >= 2).sum())
    n_unique = int((feat_target_count == 1).sum())
    print(f"  Sankey: {n_targets} targets, {n_features} unique features")
    print(f"     • {n_transdiag} transdiagnostic (≥2 targets)")
    print(f"     • {n_unique} target-specific")

    feat_sets = {}
    for t in df_sankey['Target'].unique():
        feat_sets[t] = set(df_sankey[df_sankey['Target'] == t]['Feature'])
    print("\n  Feature Overlap Matrix:")
    targs = list(feat_sets.keys())
    for i, t1 in enumerate(targs):
        for t2 in targs[i + 1:]:
            overlap = feat_sets[t1] & feat_sets[t2]
            if overlap:
                print(f"    {t1} ∩ {t2}: {len(overlap)} shared -- {', '.join(list(overlap)[:5])}")

def _compute_node_positions_enhanced(items_list, totals, global_max, pad,
                                      min_node_h=0.025, max_node_h=0.12,
                                      spread_factor=1.5):
    """Compute vertical positions for a column of Sankey nodes."""
    def _node_height(total):
        if global_max <= 0:
            return min_node_h
        return min_node_h + (max_node_h - min_node_h) * (total / global_max)

    positions = {}
    y = 1.0
    for item in items_list:
        h = _node_height(totals[item])
        y -= h
        positions[item] = {
            'y_centre': y + h / 2,
            'h': h,
            'y_top': y + h,
            'y_bot': y,
        }
        y -= pad * spread_factor
    return positions

def _draw_bezier_ribbon_enhanced(ax, left_x, node_w, right_x,
                                  y_l_top, y_l_bot, y_r_top, y_r_bot,
                                  color_rgba, zorder=1, edge_color='none', edge_lw=0,
                                  gradient_effect=False, glow_layers=0,
                                  right_node_w=None):
    """Draw a single Bézier ribbon between left and right ports."""
    _rnw = right_node_w if right_node_w is not None else node_w
    cx = (left_x + node_w + right_x - _rnw) / 2
    verts = [
        (left_x + node_w, y_l_top),
        (cx, y_l_top), (cx, y_r_top),
        (right_x - _rnw, y_r_top),
        (right_x - _rnw, y_r_bot),
        (cx, y_r_bot), (cx, y_l_bot),
        (left_x + node_w, y_l_bot),
        (left_x + node_w, y_l_top),
    ]
    codes = [
        Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
        Path.CLOSEPOLY,
    ]

    if glow_layers > 0:
        for i in range(glow_layers, 0, -1):
            glow_alpha = color_rgba[3] * (0.22 / i)
            glow_patch = matplotlib.patches.PathPatch(
                Path(verts, codes),
                facecolor=color_rgba[:3] + (glow_alpha,),
                edgecolor='none', lw=0, zorder=zorder - 1)
            ax.add_patch(glow_patch)

    patch = matplotlib.patches.PathPatch(
        Path(verts, codes), facecolor=color_rgba,
        edgecolor=edge_color, lw=edge_lw, zorder=zorder)
    ax.add_patch(patch)
    return patch

def _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                          target_positions, feature_positions,
                          target_totals, feature_totals):
    """Yield ribbon geometry for each flow row."""
    flows = df_sankey.sort_values('Importance', ascending=False)
    target_port_offset = {t: 0.0 for t in targets_list}
    feature_port_offset = {f: 0.0 for f in features_list}

    for t in targets_list:
        sub = flows[flows['Target'] == t].sort_values('Importance', ascending=False)
        total = sub['Importance'].sum()
        for _, row in sub.iterrows():
            frac = row['Importance'] / total if total > 0 else 0
            ribbon_h_left = target_positions[t]['h'] * frac
            f = row['Feature']
            f_total = feature_totals[f]
            ribbon_h_right = (feature_positions[f]['h'] * (row['Importance'] / f_total)
                              if f_total > 0 else 0)

            y_l_top = target_positions[t]['y_top'] - target_port_offset[t]
            y_l_bot = y_l_top - ribbon_h_left
            y_r_top = feature_positions[f]['y_top'] - feature_port_offset[f]
            y_r_bot = y_r_top - ribbon_h_right
            target_port_offset[t] += ribbon_h_left
            feature_port_offset[f] += ribbon_h_right

            yield {
                'target': t, 'feature': f,
                'importance': row['Importance'],
                'y_l_top': y_l_top, 'y_l_bot': y_l_bot,
                'y_r_top': y_r_top, 'y_r_bot': y_r_bot,
                'ribbon_h_left': ribbon_h_left,
                'ribbon_h_right': ribbon_h_right,
            }

# ---
# ENHANCED HEATFLOW STYLE
# ---

def _draw_sankey_heatflow_enhanced(df_sankey, sankey_top_features, save_dir=None,
                                    theme_key='heatflow_fire', show_colorbar=True):
    """Enhanced heatflow Sankey -- background zones & target totals removed."""
    theme = HEATFLOW_THEMES[theme_key]
    is_dark = theme.get('dark_mode', False)

    _cd = _sankey_common_data(df_sankey)
    targets_list = _cd['targets_list']
    features_list = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals = _cd['target_totals']
    feature_totals = _cd['feature_totals']
    n_targets = _cd['n_targets']
    n_features = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    cmap_heat = _get_cmap_safe(theme['cmap'])
    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm = Normalize(vmin=_imp_min, vmax=_imp_max)
    _sm = mcm.ScalarMappable(norm=_norm, cmap=cmap_heat)

    target_colors = {}
    target_rgb_list = get_target_colors_for_theme(theme_key, n_targets)
    for i, t in enumerate(targets_list):
        target_colors[t] = target_rgb_list[i]

    _pad = max(0.015, min(0.035, 1.2 / max(n_features, 1)))
    left_x  = _LEFT_X
    right_x = _RIGHT_X
    node_w = 0.028

    fig_w, fig_h = _wide_figure_dims(n_targets, n_features)
    _spread_t = 1.6
    _spread_f = 1.5
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()

    global_max = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)

    # -- Equalize vertical envelopes so both columns span the same range --
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(theme['bg_color'])
    ax.set_facecolor(theme['bg_color'])

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04

    ax.set_xlim(-0.50, right_x + 0.65)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _ribbon_c = _sm.to_rgba(rb['importance'])[:3]
        _alpha = 0.30 + 0.60 * _norm(rb['importance'])

        if _norm(rb['importance']) > 0.7:
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                rb['y_l_top'] + 0.003, rb['y_l_bot'] - 0.003,
                rb['y_r_top'] + 0.003, rb['y_r_bot'] - 0.003,
                color_rgba=_ribbon_c + (_alpha * 0.3,),
                zorder=1)

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            rb['y_l_top'], rb['y_l_bot'],
            rb['y_r_top'], rb['y_r_bot'],
            color_rgba=_ribbon_c + (_alpha,),
            zorder=2, edge_color=_ribbon_c + (min(_alpha + 0.08, 0.8),),
            edge_lw=0.3)

    for t, pos in target_positions.items():
        shadow_offset = 0.004
        ax.add_patch(FancyBboxPatch(
            (left_x - node_w + shadow_offset, pos['y_bot'] - shadow_offset),
            node_w * 2, pos['h'],
            boxstyle="round,pad=0.006",
            facecolor='#000000' if not is_dark else '#333333',
            edgecolor='none', linewidth=0,
            zorder=2, alpha=0.2))

        ax.add_patch(FancyBboxPatch(
            (left_x - node_w, pos['y_bot']),
            node_w * 2, pos['h'],
            boxstyle="round,pad=0.006",
            facecolor=target_colors[t],
            edgecolor=theme['target_node_edge'],
            linewidth=1.5, zorder=3))

        _tm = _target_metrics.get(t) or _target_metrics.get(_get_tdn(t))
        _label_x = left_x - node_w - 0.020

        if _tm is not None:
            _val, _lbl = _tm
            txt = ax.text(_label_x, pos['y_centre'] + 0.013,
                    t, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=theme['text_color'], fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=theme['bg_color'])])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.015,
                    f'{_lbl} = {_val:.3f}',
                    ha='right', va='center', fontsize=_lf[2], fontweight='bold',
                    color=theme['accent_color'],
                    fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=theme['bg_color'])])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                    t, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=theme['text_color'], fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=theme['bg_color'])])

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for f, pos in feature_positions.items():
        n_tgt = feat_target_count[f]
        _avg_imp = feature_totals[f] / max(n_tgt, 1)
        _nc = _sm.to_rgba(_avg_imp)[:3]

        if n_tgt >= 2:
            glow_padding = 0.005
            ax.add_patch(FancyBboxPatch(
                (right_x - node_w - glow_padding, pos['y_bot'] - glow_padding),
                node_w * 2 + glow_padding * 2, pos['h'] + glow_padding * 2,
                boxstyle="round,pad=0.008",
                facecolor='none', edgecolor=theme['feature_glow'],
                linewidth=2.5, zorder=2, alpha=0.6))

        ax.add_patch(FancyBboxPatch(
            (right_x - node_w, pos['y_bot']),
            node_w * 2, pos['h'],
            boxstyle="round,pad=0.006",
            facecolor=_nc,
            edgecolor='#444444' if not is_dark else '#888888',
            linewidth=1.0, zorder=3))

        badge = f" ({n_tgt}T)" if n_tgt >= 2 else ""
        fw = 'bold' if n_tgt >= 2 else 'normal'
        label_text = f"{f}{badge}"

        txt = ax.text(right_x + node_w + 0.020, pos['y_centre'],
                label_text,
                ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                color=theme['text_color'], fontfamily=_TITLE_FONT,
                zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=theme['bg_color'])])

    _title_c = theme['text_color'] if not is_dark else '#EAEAEA'

    _title_y = 1.04
    ax.text(0.50, _title_y, ' ', ha='center', va='bottom', fontsize=15,
            transform=ax.transAxes, alpha=0.0)
    # Subtitle moved to bottom
    ax.text(0.50, -0.015,
            f'Top {sankey_top_features} Consensus Features per Target  \u00b7  '
            'Shared Features Annotated (nT)  \u00b7  '
            'R\u00b2 from Across-Category Analyses\n'
            'Ribbon Opacity \u221d Weighted Ensemble Importance  \u00b7  '
            'Circular Features were Removed and Not Visible',
            ha='center', va='top', fontsize=_lf[3], fontweight='bold',
            color=_title_c, fontfamily=_TITLE_FONT, linespacing=1.4,
            transform=ax.transAxes)

    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    _hdr_text_c = _ensure_contrast(theme['zone_colors'][0], theme['header_color'])
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=10, fontweight='bold', color=_hdr_text_c,
            fontfamily=_TITLE_FONT,
            bbox=dict(boxstyle='round,pad=0.20', facecolor=theme['zone_colors'][0],
                     edgecolor=theme['target_node_edge'], alpha=0.75, linewidth=0.7))
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=10, fontweight='bold', color=_hdr_text_c,
            fontfamily=_TITLE_FONT,
            bbox=dict(boxstyle='round,pad=0.20', facecolor=theme['zone_colors'][0],
                     edgecolor=theme['feature_glow'], alpha=0.75, linewidth=0.7))

    if show_colorbar:
        _cbar_ax = fig.add_axes([0.90, 0.20, 0.015, 0.50])
        _cb = plt.colorbar(_sm, cax=_cbar_ax)
        _cb.set_label('Consensus Importance', fontsize=10, color=theme['text_color'])
        _cb.ax.tick_params(labelsize=8, colors=theme['text_color'])
        if is_dark:
            _cbar_ax.set_facecolor('#2A2A2A')

    plt.tight_layout(rect=[0, 0, 0.88, 0.98])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')

    saved = _safe_save_fig(fig, save_dir, f'sankey_{theme_key}', dpi=600)
    plt.show()
    for p in saved:
        print(f"  Saved: {p}")

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    return fig

# ---
# ADDITIONAL UNIQUE VISUAL STYLES
# ---

def _extract_target_metrics(batch_archive):
    """
    Extract best ensemble metric (R² or AUC) per target from _batch_archive.
    Returns dict {display_name: (metric_value, metric_label)}.
    """
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    result = {}
    for raw_key, archive in batch_archive.items():
        mdf = archive.get('metrics_dict_full', {})
        cats = mdf.get('category', [])
        mets = mdf.get('metrics', [])
        if not cats or not mets:
            continue
        best_score, best_label = None, 'R\u00b2'
        for _cat, _mdict in zip(cats, mets):
            if not isinstance(_mdict, dict):
                continue
            _mm = _mdict.get('meta_model', _mdict.get('ensemble', {}))
            if not isinstance(_mm, dict):
                continue
            _r2  = _mm.get('r2', None)
            _auc = _mm.get('auc', _mm.get('roc_auc', None))
            if _r2 is not None:
                if best_score is None or _r2 > best_score:
                    best_score, best_label = _r2, 'R\u00b2'
            elif _auc is not None:
                if best_score is None or _auc > best_score:
                    best_score, best_label = _auc, 'AUC'
        if best_score is not None:
            display_name = _tdn.get(raw_key, raw_key)
            result[display_name] = (best_score, best_label)
            # Also store under _get_tdn for __tp archive keys
            _tdn_name = _get_tdn(raw_key)
            if _tdn_name != display_name:
                result[_tdn_name] = (best_score, best_label)
    return result

_PFACTOR_DOMAIN_COLORS = {
    'internal_D_p':          (0.55, 0.36, 0.95),
    'external_D_p':          (0.45, 0.75, 0.20),
    'parent_internal_D_p':   (0.48, 0.10, 0.48),
    'parent_external_D_p':   (0.08, 0.48, 0.38),
    'dep_increase_2sd':              (0.12, 0.56, 1.00),
    'delta_depress_D_p':             (0.69, 0.77, 0.87),
    'parent_dep_increase_2sd':       (0.28, 0.24, 0.55),
    'delta_parent_depress_D_p':      (0.39, 0.58, 0.93),
    'Recent Clinical Change Child Depression Syndrome \u2265 2 SD':  (0.12, 0.56, 1.00),
    'Change in Child Dimensional Depression Syndrome':               (0.69, 0.77, 0.87),
    'Recent Clinical Change Parent Depression Syndrome \u2265 2 SD': (0.28, 0.24, 0.55),
    'Change in Parent Dimensional Depression Syndrome':              (0.39, 0.58, 0.93),
    'Internalizing':         (0.55, 0.36, 0.95),
    'Externalizing':         (0.45, 0.75, 0.20),
    'Internalizing Spectra': (0.55, 0.36, 0.95),
    'Externalizing Spectra': (0.45, 0.75, 0.20),
    'Child Internalizing Spectra': (0.55, 0.36, 0.95),
    'Child Externalizing Spectra': (0.45, 0.75, 0.20),
    'Par. Internalizing':    (0.48, 0.10, 0.48),
    'Par. Externalizing':    (0.08, 0.48, 0.38),
    'Parent Internalizing Spectra': (0.48, 0.10, 0.48),
    'Parent Externalizing Spectra': (0.08, 0.48, 0.38),
    # -- P-Factor Spectra: Suicide & Academic --
    'suicidal_p':                   (0.04, 0.04, 0.24),   # very dark blue
    'parent_suicidal_thoughts_p':   (0.04, 0.04, 0.24),   # very dark blue (match child)
    'bad_grades':                   (0.45, 0.45, 0.45),   # grey
    'Suicidal Ideation':            (0.04, 0.04, 0.24),
    'Parent Suicidal Thoughts':     (0.04, 0.04, 0.24),
    'Academic Performance':         (0.45, 0.45, 0.45),
}

def _pfactor_target_color(target_name):
    if target_name in _PFACTOR_DOMAIN_COLORS:
        return _PFACTOR_DOMAIN_COLORS[target_name]
    tl = target_name.lower()
    if 'par' in tl and 'internal' in tl:
        return _PFACTOR_DOMAIN_COLORS['parent_internal_D_p']
    if 'par' in tl and 'external' in tl:
        return _PFACTOR_DOMAIN_COLORS['parent_external_D_p']
    if 'internal' in tl:
        return _PFACTOR_DOMAIN_COLORS['internal_D_p']
    if 'external' in tl:
        return _PFACTOR_DOMAIN_COLORS['external_D_p']
    if 'suicid' in tl:
        return _PFACTOR_DOMAIN_COLORS['suicidal_p']
    if 'bad_grade' in tl or 'academic' in tl:
        return _PFACTOR_DOMAIN_COLORS['bad_grades']
    _is_parent = 'parent' in tl or 'par.' in tl
    _is_clf    = '\u2265 2 sd' in tl or 'clinical change' in tl or 'increase' in tl
    _is_dep    = ('depress' in tl or 'dimensional' in tl)
    if _is_dep:
        if _is_parent and _is_clf:
            return _PFACTOR_DOMAIN_COLORS['parent_dep_increase_2sd']
        if _is_parent:
            return _PFACTOR_DOMAIN_COLORS['delta_parent_depress_D_p']
        if _is_clf:
            return _PFACTOR_DOMAIN_COLORS['dep_increase_2sd']
        return _PFACTOR_DOMAIN_COLORS['delta_depress_D_p']
    return (0.5, 0.5, 0.5)

_PFACTOR_TARGET_ORDER = [
    'parent_internal_D_p', 'Par. Internalizing', 'Par. Internal',
    'internal_D_p', 'Internalizing', 'Internalizing Spectra',
    'parent_external_D_p', 'Par. Externalizing', 'Par. External',
    'external_D_p', 'Externalizing', 'Externalizing Spectra',
    'suicidal_p', 'Suicidal Ideation',
    'parent_suicidal_thoughts_p', 'Parent Suicidal Thoughts',
    'bad_grades', 'Academic Performance',
]

def _pfactor_sort_targets(targets_list):
    return sorted(targets_list)
def _pfactor_sort_features(features_list, df_sankey, target_order):
    feat_info = {}
    for f in features_list:
        rows = df_sankey[df_sankey['Feature'] == f]
        n_tgt = rows['Target'].nunique()
        total_imp = rows['Importance'].sum()
        dom_target = rows.loc[rows['Importance'].idxmax(), 'Target']
        try:
            dom_idx = target_order.index(dom_target)
        except ValueError:
            dom_idx = 99
        feat_info[f] = {
            'n_tgt': n_tgt, 'total_imp': total_imp,
            'dom_target': dom_target, 'dom_idx': dom_idx,
        }
def _infer_feature_domain(feat_name):
    fl = feat_name.lower()
    if any(k in fl for k in ('(2t)', '2t)', 'change', 'delta')):
        return 'Change Scores', _FEATURE_DOMAIN_ACCENTS['Change Scores']
    if fl.startswith('par.') or fl.startswith('par ') or fl.endswith('_p'):
        return 'Parent Report', _FEATURE_DOMAIN_ACCENTS['Parent Report']
    if any(k in fl for k in ('fam.', 'fam ', 'family', 'friend', 'social',
                              'relationship', 'conflict', 'unliked', 'not liked')):
        return 'Family/Social', _FEATURE_DOMAIN_ACCENTS['Family/Social']
    if any(k in fl for k in ('cognit', 'task', 'inhibit', 'goal persist',
                              'concentration', 'attention', 'memory')):
        return 'Cognitive/Academic', _FEATURE_DOMAIN_ACCENTS['Cognitive/Academic']
    if any(k in fl for k in ('somatic', 'sleep', 'eating', 'drug', 'bmi',
                              'physical', 'medical')):
        return 'Physical/Medical', _FEATURE_DOMAIN_ACCENTS['Physical/Medical']
    if any(k in fl for k in ('age', 'sex', 'income', 'ses', 'education')):
        return 'Demographic', _FEATURE_DOMAIN_ACCENTS['Demographic']
    return 'Youth Behavioral', _FEATURE_DOMAIN_ACCENTS['Youth Behavioral']

def _draw_sankey_pfactor_spectra(df_sankey, sankey_top_features, save_dir=None):
    """P-Factor Spectra Sankey -- domain-coloured heatflow variant."""
    _cd = _sankey_common_data(df_sankey)
    targets_list   = _cd['targets_list']
    features_list  = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals  = _cd['target_totals']
    feature_totals = _cd['feature_totals']
    n_targets  = _cd['n_targets']
    n_features = _cd['n_features']

    targets_list = _pfactor_sort_targets(targets_list)

    _SANKEY_DEP_RENAME = {
        'dep_increase_2sd':          'Recent Clinical Change Child Depression Syndrome \u2265 2 SD',
        'delta_depress_D_p':         'Change in Child Dimensional Depression Syndrome',
        'parent_dep_increase_2sd':   'Recent Clinical Change Parent Depression Syndrome \u2265 2 SD',
        'delta_parent_depress_D_p':  'Change in Parent Dimensional Depression Syndrome',
    }
    _PFACTOR_DISPLAY_RENAME = {
        'Internalizing Spectra':  'Child Internalizing Spectra',
        'Externalizing Spectra':  'Child Externalizing Spectra',
        'Internalizing':          'Child Internalizing Spectra',
        'Externalizing':          'Child Externalizing Spectra',
        'internal_D_p':           'Child Internalizing Spectra',
        'external_D_p':           'Child Externalizing Spectra',
        'parent_internal_D_p':    'Parent Internalizing Spectra',
        'parent_external_D_p':    'Parent Externalizing Spectra',
        'suicidal_p':             'Suicidal Ideation',
        'parent_suicidal_thoughts_p': 'Parent Suicidal Thoughts',
        'bad_grades':             'Academic Performance',
    }
    def _edge_for(t):
        tl = t.lower()
        if 'internal' in tl:   return node_edge_int
        if 'external' in tl:   return node_edge_ext
        if 'suicid' in tl:     return '#060630' # very dark blue edge
        if 'bad_grade' in tl or 'academic' in tl:  return '#4A4A4A' # dark grey edge
        return '#444444'

    _feat_targets_ranked = {}
    _feat_dom_target = {}
    for f in features_list:
        f_rows = df_sankey[df_sankey['Feature'] == f].sort_values('Importance', ascending=False)
        _feat_targets_ranked[f] = list(zip(f_rows['Target'].values, f_rows['Importance'].values))
        if not f_rows.empty:
            _feat_dom_target[f] = f_rows.iloc[0]['Target']
        else:
            _feat_dom_target[f] = targets_list[0] if targets_list else ''

    _pad = max(0.015, min(0.035, 1.2 / max(n_features, 1)))
    left_x  = _LEFT_X
    right_x = _RIGHT_X
    node_w = 0.028

    # -- Width-first figure sizing --
    fig_w, fig_h = _wide_figure_dims(n_targets, n_features)
    _spread_t = 1.8
    _spread_f = 1.5
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()

    global_max = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.14, 3.5 / max(n_features, 1))

    _target_node_w = 0.036

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)

    # -- Equalize vertical envelopes --
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04

    ax.set_xlim(-0.65, right_x + 0.65)
    ax.set_ylim(y_min, 1.08)
    ax.axis('off')

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb = target_colors.get(rb['target'], (0.5, 0.5, 0.5))
        rel = _norm(rb['importance'])
        _alpha = 0.18 + 0.67 * rel

        if rel > 0.60:
            _draw_bezier_ribbon_enhanced(
                ax, left_x, _target_node_w, right_x,
                rb['y_l_top'] + 0.005, rb['y_l_bot'] - 0.005,
                rb['y_r_top'] + 0.005, rb['y_r_bot'] - 0.005,
                color_rgba=_rgb + (_alpha * 0.35,),
                zorder=1, right_node_w=node_w)

        if rel < 0.35:
            _edge_c = (1.0, 1.0, 1.0, 0.30)
            _edge_lw = 0.5
        else:
            _edge_c = _rgb + (min(_alpha + 0.10, 0.85),)
            _edge_lw = 0.4

        _draw_bezier_ribbon_enhanced(
            ax, left_x, _target_node_w, right_x,
            rb['y_l_top'], rb['y_l_bot'],
            rb['y_r_top'], rb['y_r_bot'],
            color_rgba=_rgb + (_alpha,),
            zorder=2, edge_color=_edge_c, edge_lw=_edge_lw, right_node_w=node_w)

    for t, pos in target_positions.items():
        rgb = target_colors[t]
        edge_c = _edge_for(t)

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.006, pos['y_bot'] - 0.006),
            _target_node_w * 2, pos['h'],
            boxstyle="round,pad=0.007",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.22))

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle="round,pad=0.007",
            facecolor=rgb, edgecolor=edge_c,
            linewidth=2.2, zorder=3))

        _disp_t = _display_target(t)
        _tm = _target_metrics.get(t) or _target_metrics.get(_disp_t)
        _label_x = left_x - _target_node_w - 0.025

        if _tm is not None:
            _val, _lbl = _tm
            txt = ax.text(_label_x, pos['y_centre'] + 0.018,
                    _disp_t, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=text_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=bg_color)])
            _metric_color = _edge_for(t)
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.022,
                    f'{_lbl} = {_val:.3f}',
                    ha='right', va='center', fontsize=_lf[2], fontweight='bold',
                    color=_metric_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=bg_color)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                    _disp_t, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=text_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=bg_color)])

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))

    for f, pos in feature_positions.items():
        n_tgt = feat_target_count[f]
        dom_t = _feat_dom_target[f]
        _t_overlap = min((n_tgt - 2) / 3.0, 1.0) if n_tgt >= 2 else 0.0
        fc = target_colors.get(dom_t, (0.5, 0.5, 0.5)) if n_tgt == 1 else (0.55 - 0.45 * _t_overlap, 0.55 - 0.45 * _t_overlap, 0.55 - 0.45 * _t_overlap)

        _nx = right_x - node_w
        _nw = node_w * 2
        _ny = pos['y_bot']
        _nh = pos['h']

        ax.add_patch(FancyBboxPatch(
            (_nx + 0.004, _ny - 0.004), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.18))

        if n_tgt >= 2:
            # Dual-glow border showing the two strongest target domains
            _dom_targets = sorted(
                _feat_targets_ranked.get(f, []),
                key=lambda x: x[1], reverse=True)[:2]
            _glow_colors = [target_colors.get(dt[0], (0.5, 0.5, 0.5)) for dt in _dom_targets]
            if len(_glow_colors) >= 2:
                ax.add_patch(FancyBboxPatch(
                    (_nx - 0.007, _ny - 0.007), _nw + 0.014, _nh + 0.014,
                    boxstyle="round,pad=0.008",
                    facecolor='none',
                    edgecolor=_glow_colors[1],
                    linewidth=2.0, zorder=2, alpha=0.45))
                ax.add_patch(FancyBboxPatch(
                    (_nx - 0.004, _ny - 0.004), _nw + 0.008, _nh + 0.008,
                    boxstyle="round,pad=0.006",
                    facecolor='none',
                    edgecolor=_glow_colors[0],
                    linewidth=2.0, zorder=2, alpha=0.45))
            elif len(_glow_colors) == 1:
                ax.add_patch(FancyBboxPatch(
                    (_nx - 0.005, _ny - 0.005), _nw + 0.010, _nh + 0.010,
                    boxstyle="round,pad=0.007",
                    facecolor='none',
                    edgecolor=_glow_colors[0],
                    linewidth=2.0, zorder=2, alpha=0.45))

        ax.add_patch(FancyBboxPatch(
            (_nx, _ny), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor=fc, edgecolor='#444444', linewidth=1.0, zorder=3))

        badge = f" ({n_tgt}T)" if n_tgt >= 2 else ""
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + node_w + 0.020, pos['y_centre'],
                f"{f}{badge}",
                ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                color=text_color, fontfamily=_TITLE_FONT, zorder=5, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=bg_color)])

    _title_y = 1.04
    ax.text(0.50, _title_y, ' ', ha='center', va='bottom', fontsize=15,
            transform=ax.transAxes, alpha=0.0)
    # Subtitle moved to bottom
    ax.text(0.50, -0.015,
            f'Top {sankey_top_features} Consensus Features per Target  \u00b7  '
            'Shared Features Annotated (nT)  \u00b7  '
            'R\u00b2 from Across-Category Analyses\n'
            'Ribbon Opacity \u221d Weighted Ensemble Importance  \u00b7  '
            'Circular Features were Removed and Not Visible',
            ha='center', va='top', fontsize=_lf[3], fontweight='bold',
            color='#1a1a1a', fontfamily=_TITLE_FONT, linespacing=1.4,
            transform=ax.transAxes)

    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.045
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=node_edge_int, fontfamily=_TITLE_FONT)
    ax.plot([left_x - _target_node_w * 1.5, left_x + _target_node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=node_edge_int, lw=1.0, alpha=0.5, clip_on=False)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=node_edge_ext, fontfamily=_TITLE_FONT)
    ax.plot([right_x - node_w * 1.5, right_x + node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=node_edge_ext, lw=1.0, alpha=0.5, clip_on=False)

    plt.tight_layout(rect=[0, 0.01, 0.90, 0.98])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_pfactor_spectra', dpi=600)
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    return fig

# ---
# CHILD ALTs -- 6-Target Domain-Coloured Heatflow
# ---

_CHILD_ALTS_COLORS = {
    'adhd_D_p':              (0.82, 0.68, 0.15),
    'external_D_p':          (0.45, 0.75, 0.20),
    'internal_D_p':          (0.55, 0.36, 0.95),
    'social_problems_D_p':   (0.90, 0.50, 0.13),
    'bad_grades':            (0.28, 0.28, 0.32),
    'tb_flanker':            (0.00, 0.81, 0.82),
    'suicidal_p':          (0.00, 0.00, 0.55),
}

_CHILD_ALTS_DISPLAY_ALIASES = {
    'ADHD':                     'adhd_D_p',
    'ADHD Symptoms':            'adhd_D_p',
    'Externalizing':            'external_D_p',
    'Externalizing Spectra':    'external_D_p',
    'Child Externalizing Spectra': 'external_D_p',
    'Internalizing':            'internal_D_p',
    'Internalizing Spectra':    'internal_D_p',
    'Child Internalizing Spectra': 'internal_D_p',
    'Social Problems':          'social_problems_D_p',
    'Bad Grades':               'bad_grades',
    'Poor Academic Performance': 'bad_grades',
    'Flanker':                  'tb_flanker',
    'Flanker Task':             'tb_flanker',
    'Flanker (NIH Toolbox)':    'tb_flanker',
    'Suicidality':              'suicidal_p',
    'Suicidal Ideation':        'suicidal_p',
}

_CHILD_ALTS_ORDER = [
    'internal_D_p',
    'external_D_p',
    'adhd_D_p',
    'social_problems_D_p',
    'bad_grades',
    'tb_flanker',
    'suicidal_p',
]

_CHILD_ALTS_EDGE    = '#2C3440'
_CHILD_ALTS_BG      = '#FFFFFF'

def _child_alts_resolve_color(target_name):
    if target_name in _CHILD_ALTS_COLORS:
        return _CHILD_ALTS_COLORS[target_name]
    raw = _CHILD_ALTS_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_ALTS_COLORS:
        return _CHILD_ALTS_COLORS[raw]
    tl = target_name.lower()
    if 'adhd' in tl:                                   return _CHILD_ALTS_COLORS['adhd_D_p']
    if 'external' in tl:                               return _CHILD_ALTS_COLORS['external_D_p']
    if 'internal' in tl:                               return _CHILD_ALTS_COLORS['internal_D_p']
    if 'social' in tl:                                 return _CHILD_ALTS_COLORS['social_problems_D_p']
    if 'grade' in tl or 'academic' in tl:              return _CHILD_ALTS_COLORS['bad_grades']
    if 'flanker' in tl:                                return _CHILD_ALTS_COLORS['tb_flanker']
    if 'suicid' in tl:                                 return _CHILD_ALTS_COLORS['suicidal_p']
    return (0.5, 0.5, 0.5)

def _child_alts_sort_targets(targets_list):
    return sorted(targets_list)
def _child_alts_edge_for(target_name):
    rgb = _child_alts_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

_PARENT_ALTS_COLORS = {
    'parent_suicidal_thoughts_p':  (0.06, 0.08, 0.35),
    'parent_bad_relationships_p':  (0.80, 0.35, 0.05),
    'parent_adhd_D_p':             (0.82, 0.72, 0.22),
    'parent_happy_person_p':       (0.90, 0.35, 0.55),
    'parent_external_D_p':         (0.10, 0.42, 0.22),
    'parent_internal_D_p':         (0.30, 0.05, 0.45),
}

_PARENT_ALTS_DISPLAY_ALIASES = {
    'Parent Suicidal Thoughts':             'parent_suicidal_thoughts_p',
    'Parent Suicidal Ideation':             'parent_suicidal_thoughts_p',
    'Parent Suicidality':                   'parent_suicidal_thoughts_p',
    'Parent Social Quality':                'parent_bad_relationships_p',
    'Parent Bad Relationships':             'parent_bad_relationships_p',
    'Parent Interpersonal Difficulties':    'parent_bad_relationships_p',
    'Parent Internalizing':                 'parent_internal_D_p',
    'Parent Internalizing Spectra':         'parent_internal_D_p',
    'Parent ADHD':                          'parent_adhd_D_p',
    'Parent ADHD Symptoms':                 'parent_adhd_D_p',
    'Parent Happiness':                     'parent_happy_person_p',
    'Parent Happiness (Item)':              'parent_happy_person_p',
    'Parent Wellbeing':                     'parent_happy_person_p',
    'Parent Happy Person':                  'parent_happy_person_p',
    'Parent Externalizing':                 'parent_external_D_p',
    'Parent Externalizing Spectra':         'parent_external_D_p',
}

_PARENT_ALTS_ORDER = [
    'parent_suicidal_thoughts_p',
    'parent_bad_relationships_p',
    'parent_adhd_D_p',
    'parent_happy_person_p',
    'parent_external_D_p',
    'parent_internal_D_p',
]

_PARENT_ALTS_EDGE = '#2C3440'
_PARENT_ALTS_BG   = '#FFFFFF'

def _parent_alts_resolve_color(target_name):
    if target_name in _PARENT_ALTS_COLORS:
        return _PARENT_ALTS_COLORS[target_name]
    raw = _PARENT_ALTS_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _PARENT_ALTS_COLORS:
        return _PARENT_ALTS_COLORS[raw]
    tl = target_name.lower()
    if 'suicid' in tl:                                  return _PARENT_ALTS_COLORS['parent_suicidal_thoughts_p']
    if 'relationship' in tl or 'social' in tl:          return _PARENT_ALTS_COLORS['parent_bad_relationships_p']
    if 'adhd' in tl:                                    return _PARENT_ALTS_COLORS['parent_adhd_D_p']
    if 'happi' in tl or 'wellbeing' in tl:              return _PARENT_ALTS_COLORS['parent_happy_person_p']
    if 'external' in tl:                                return _PARENT_ALTS_COLORS['parent_external_D_p']
    if 'internal' in tl:                                return _PARENT_ALTS_COLORS['parent_internal_D_p']
    return (0.5, 0.5, 0.5)

def _parent_alts_sort_targets(targets_list):
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

def _parent_alts_edge_for(target_name):
    rgb = _parent_alts_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _draw_sankey_domain_style(df_sankey, sankey_top_features, save_dir=None,
                               sort_fn=None, color_fn=None, edge_fn=None,
                               edge_col='#2C3440', bg_color='#FFFFFF',
                               save_name='sankey_domain', label_override=None,
                               spread_target_override=None,
                               spread_feature_override=None,
                               column_gap_override=None,
                               feature_share_darkening=False):
    """
    Shared draw function for child_alts, parent_alts, and any future
    domain-coloured heatflow Sankeys.  Avoids duplicating ~200 lines.
    """
    _BG       = bg_color
    _EDGE_COL = edge_col

    _cd = _sankey_common_data(df_sankey)
    targets_list      = sort_fn(_cd['targets_list']) if sort_fn else _cd['targets_list']
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: color_fn(t) for t in targets_list}

    _pad      = max(0.015, min(0.035, 1.2 / max(n_features, 1)))
    left_x  = _LEFT_X
    right_x = _RIGHT_X
    # -- Per-style column gap override (push features column further right) --
    if column_gap_override is not None:
        right_x = left_x + column_gap_override
    node_w    = 0.028
    _target_node_w = 0.036

    # -- Width-first figure sizing --
    fig_w, fig_h = _wide_figure_dims(n_targets, n_features)
    _spread_t = 1.8
    _spread_f = 1.5
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    # -- Per-style spread overrides (applied after landscape adjust) --
    if spread_target_override is not None:
        _spread_t = spread_target_override
    if spread_feature_override is not None:
        _spread_f = spread_feature_override
    _lf = _landscape_fonts()

    global_max   = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h  = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)

    # -- Equalize vertical envelopes --
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04

    ax.set_xlim(-0.65, right_x + 0.65)
    ax.set_ylim(y_min, 1.08)
    ax.axis('off')

    # -- Ribbons --
    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel  = _norm(rb['importance'])
        _alpha = 0.18 + 0.67 * rel

        if rel > 0.60:
            _draw_bezier_ribbon_enhanced(
                ax, left_x, _target_node_w, right_x,
                rb['y_l_top'] + 0.005, rb['y_l_bot'] - 0.005,
                rb['y_r_top'] + 0.005, rb['y_r_bot'] - 0.005,
                color_rgba=_rgb + (_alpha * 0.35,),
                zorder=1, right_node_w=node_w)

        if rel < 0.35:
            _edge_c = (1.0, 1.0, 1.0, 0.30)
            _edge_lw = 0.5
        else:
            _edge_c = _rgb + (min(_alpha + 0.10, 0.85),)
            _edge_lw = 0.4

        _draw_bezier_ribbon_enhanced(
            ax, left_x, _target_node_w, right_x,
            rb['y_l_top'], rb['y_l_bot'],
            rb['y_r_top'], rb['y_r_bot'],
            color_rgba=_rgb + (_alpha,),
            zorder=2, edge_color=_edge_c, edge_lw=_edge_lw, right_node_w=node_w)

    # -- Target nodes --
    text_color = '#1a1a1a'
    for t, pos in target_positions.items():
        rgb = _tc[t]

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.006, pos['y_bot'] - 0.006),
            _target_node_w * 2, pos['h'],
            boxstyle="round,pad=0.007",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.22))

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle="round,pad=0.007",
            facecolor=rgb, edgecolor=_EDGE_COL,
            linewidth=2.2, zorder=3))

        display = _tdn.get(t, t)
        if label_override:
            display = label_override.get(display, label_override.get(t, display))
        _tm = _target_metrics.get(t) or _target_metrics.get(display)
        _label_x = left_x - _target_node_w - 0.025

        if _tm is not None:
            _val, _lbl = _tm
            txt = ax.text(_label_x, pos['y_centre'] + 0.018,
                    display, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=text_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])
            _metric_color = edge_fn(t) if edge_fn else _EDGE_COL
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.022,
                    f'{_lbl} = {_val:.3f}',
                    ha='right', va='center', fontsize=_lf[2], fontweight='bold',
                    color=_metric_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                    display, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=text_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])

    # -- Feature nodes --
    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))

    _feat_dom_target = {}
    for f in features_list:
        f_rows = df_sankey[df_sankey['Feature'] == f].sort_values('Importance', ascending=False)
        _feat_dom_target[f] = f_rows.iloc[0]['Target'] if not f_rows.empty else (
            targets_list[0] if targets_list else '')

    for f, pos in feature_positions.items():
        n_tgt = feat_target_count[f]
        dom_t = _feat_dom_target[f]
        if n_tgt == 1:
            fc = _tc.get(dom_t, (0.5, 0.5, 0.5))
        elif feature_share_darkening:
            # Graded darkening: bluish-grey at n_tgt=2 -> near-black at n_tgt=n_targets.
            # Visual: features shared by many targets stand out as the darkest spine.
            _share_base = (0.45, 0.52, 0.62)   # current default multi-target colour
            _share_dark = (0.05, 0.07, 0.10)   # near-black floor
            _span = max(1, n_targets - 1)
            _t = min(1.0, max(0.0, (n_tgt - 1) / _span))
            fc = tuple(_share_base[i] * (1.0 - _t) + _share_dark[i] * _t for i in range(3))
        else:
            fc = (0.45, 0.52, 0.62)

        _nx = right_x - node_w
        _nw = node_w * 2
        _ny = pos['y_bot']
        _nh = pos['h']

        ax.add_patch(FancyBboxPatch(
            (_nx + 0.004, _ny - 0.004), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.18))

        if n_tgt >= 2:
            f_rows = df_sankey[df_sankey['Feature'] == f].sort_values('Importance', ascending=False)
            top2_targets = f_rows['Target'].head(2).tolist()
            glow_outer = _tc.get(top2_targets[0], (0.5, 0.5, 0.5))
            glow_inner = _tc.get(top2_targets[1], (0.5, 0.5, 0.5)) if len(top2_targets) > 1 else glow_outer
            ax.add_patch(FancyBboxPatch(
                (_nx - 0.007, _ny - 0.007), _nw + 0.014, _nh + 0.014,
                boxstyle="round,pad=0.008",
                facecolor='none', edgecolor=glow_outer + (0.50,),
                linewidth=2.0, zorder=2))
            ax.add_patch(FancyBboxPatch(
                (_nx - 0.004, _ny - 0.004), _nw + 0.008, _nh + 0.008,
                boxstyle="round,pad=0.006",
                facecolor='none', edgecolor=glow_inner + (0.45,),
                linewidth=2.0, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (_nx, _ny), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor=fc, edgecolor='#444444', linewidth=1.0, zorder=3))

        badge = f" ({n_tgt}T)" if n_tgt >= 2 else ""
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + node_w + 0.020, pos['y_centre'],
                f"{f}{badge}",
                ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                color=text_color, fontfamily=_TITLE_FONT, zorder=5, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=_BG)])

    # -- Title / subtitle --
    _title_y = 1.04
    ax.text(0.50, _title_y, ' ', ha='center', va='bottom', fontsize=15,
            transform=ax.transAxes, alpha=0.0)
    ax.text(0.50, -0.015,
            f'Top {sankey_top_features} Consensus Features per Target  \u00b7  '
            'Shared Features Annotated (nT)  \u00b7  '
            'R\u00b2 from Across-Category Analyses\n'
            'Ribbon Opacity \u221d Weighted Ensemble Importance  \u00b7  '
            'Circular Features were Removed and Not Visible',
            ha='center', va='top', fontsize=_lf[3], fontweight='bold',
            color='#1a1a1a', fontfamily=_TITLE_FONT, linespacing=1.4,
            transform=ax.transAxes)

    # -- Column headers -- parallel, above whichever column is taller --
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.045
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.plot([left_x - _target_node_w * 1.5, left_x + _target_node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=_EDGE_COL, lw=1.0, alpha=0.5, clip_on=False)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.plot([right_x - node_w * 1.5, right_x + node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=_EDGE_COL, lw=1.0, alpha=0.5, clip_on=False)

    plt.tight_layout(rect=[0, 0.01, 0.90, 0.98])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, save_name, dpi=600)
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    return fig

# ---
# SBT CORE TRIAD -- Core Depression + Academic/Cognitive + Perfectionism
# ---

_SBT_CORE_TRIAD_COLORS = {
    'sbt_core_depression':           (0.102, 0.227, 0.420),   # deep navy
    'sbt_academic_cognitive':        (0.722, 0.588, 0.043),   # golden amber
    'sbt_perfectionism_achievement': (0.769, 0.290, 0.502),   # rose magenta
}

_SBT_CORE_TRIAD_DISPLAY_ALIASES = {
    'Core Depression':                      'sbt_core_depression',
    'Mood/Depression (Primary Features)':   'sbt_core_depression',
    'Academic/Cognitive':                   'sbt_academic_cognitive',
    'Depression Subtype (Poor Academic/Cognitive)': 'sbt_academic_cognitive',
    'Poor Academic/Cognitive':              'sbt_academic_cognitive',
    'Perfectionism/Achievement':            'sbt_perfectionism_achievement',
    'Depression Subtype (Perfectionism/High-Achievement)': 'sbt_perfectionism_achievement',
    'High-Achievement':                     'sbt_perfectionism_achievement',
}

_SBT_CORE_TRIAD_ORDER = [
    'sbt_core_depression',
    'sbt_academic_cognitive',
    'sbt_perfectionism_achievement',
]

_SBT_CORE_TRIAD_EDGE = '#2C3440'
_SBT_CORE_TRIAD_BG   = '#FFFFFF'

def _sbt_core_triad_resolve_color(target_name):
    if target_name in _SBT_CORE_TRIAD_COLORS:
        return _SBT_CORE_TRIAD_COLORS[target_name]
    raw = _SBT_CORE_TRIAD_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _SBT_CORE_TRIAD_COLORS:
        return _SBT_CORE_TRIAD_COLORS[raw]
    tl = target_name.lower()
    if 'core' in tl or 'primary' in tl:       return _SBT_CORE_TRIAD_COLORS['sbt_core_depression']
    if 'academic' in tl or 'cognitive' in tl:  return _SBT_CORE_TRIAD_COLORS['sbt_academic_cognitive']
    if 'perfect' in tl or 'achiev' in tl:      return _SBT_CORE_TRIAD_COLORS['sbt_perfectionism_achievement']
    return (0.5, 0.5, 0.5)

def _sbt_core_triad_sort_targets(targets_list):
    return sorted(targets_list)
def _sbt_core_triad_edge_for(target_name):
    rgb = _sbt_core_triad_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

# ---
# SBT CORE TRIAD -- FEMALE VARIANT (warmer tones: plum navy / warm gold / rose)
# ---

_SBT_CORE_TRIAD_FEMALE_COLORS = {
    'sbt_core_depression':           (0.145, 0.200, 0.440),   # plum-shifted navy
    'sbt_academic_cognitive':        (0.745, 0.555, 0.110),   # warm amber-gold
    'sbt_perfectionism_achievement': (0.800, 0.245, 0.460),   # deeper rose
}

_SBT_CORE_TRIAD_FEMALE_EDGE = '#4A2040' # warm plum edge
_SBT_CORE_TRIAD_FEMALE_BG   = '#FFFFFF'

def _sbt_core_triad_female_resolve_color(target_name):
    if target_name in _SBT_CORE_TRIAD_FEMALE_COLORS:
        return _SBT_CORE_TRIAD_FEMALE_COLORS[target_name]
    raw = _SBT_CORE_TRIAD_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _SBT_CORE_TRIAD_FEMALE_COLORS:
        return _SBT_CORE_TRIAD_FEMALE_COLORS[raw]
    tl = target_name.lower()
    if 'core' in tl or 'primary' in tl:       return _SBT_CORE_TRIAD_FEMALE_COLORS['sbt_core_depression']
    if 'academic' in tl or 'cognitive' in tl:  return _SBT_CORE_TRIAD_FEMALE_COLORS['sbt_academic_cognitive']
    if 'perfect' in tl or 'achiev' in tl:      return _SBT_CORE_TRIAD_FEMALE_COLORS['sbt_perfectionism_achievement']
    return (0.5, 0.5, 0.5)

def _sbt_core_triad_female_edge_for(target_name):
    rgb = _sbt_core_triad_female_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

# ---
# SBT CORE TRIAD -- MALE VARIANT (cooler tones: steel navy / olive gold / slate mauve)
# ---

_SBT_CORE_TRIAD_MALE_COLORS = {
    'sbt_core_depression':           (0.075, 0.250, 0.400),   # steel-blue navy
    'sbt_academic_cognitive':        (0.680, 0.620, 0.020),   # cool olive-gold
    'sbt_perfectionism_achievement': (0.710, 0.330, 0.545),   # muted slate-mauve
}

_SBT_CORE_TRIAD_MALE_EDGE = '#1E2E48' # cool steel edge
_SBT_CORE_TRIAD_MALE_BG   = '#FFFFFF'

def _sbt_core_triad_male_resolve_color(target_name):
    if target_name in _SBT_CORE_TRIAD_MALE_COLORS:
        return _SBT_CORE_TRIAD_MALE_COLORS[target_name]
    raw = _SBT_CORE_TRIAD_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _SBT_CORE_TRIAD_MALE_COLORS:
        return _SBT_CORE_TRIAD_MALE_COLORS[raw]
    tl = target_name.lower()
    if 'core' in tl or 'primary' in tl:       return _SBT_CORE_TRIAD_MALE_COLORS['sbt_core_depression']
    if 'academic' in tl or 'cognitive' in tl:  return _SBT_CORE_TRIAD_MALE_COLORS['sbt_academic_cognitive']
    if 'perfect' in tl or 'achiev' in tl:      return _SBT_CORE_TRIAD_MALE_COLORS['sbt_perfectionism_achievement']
    return (0.5, 0.5, 0.5)

def _sbt_core_triad_male_edge_for(target_name):
    rgb = _sbt_core_triad_male_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

# -- Draw wrappers for all three core-triad variants --

def _draw_sankey_sbt_core_triad(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_sbt_core_triad_sort_targets,
        color_fn=_sbt_core_triad_resolve_color,
        edge_fn=_sbt_core_triad_edge_for,
        edge_col=_SBT_CORE_TRIAD_EDGE, bg_color=_SBT_CORE_TRIAD_BG,
        save_name='sankey_sbt_core_triad',
        label_override={
            'sbt_core_depression':           'Core Depression',
            'sbt_academic_cognitive':        'Academic/Cognitive',
            'sbt_perfectionism_achievement': 'Perfectionism/Achievement',
        })

def _draw_sankey_sbt_core_triad_female(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_sbt_core_triad_sort_targets,
        color_fn=_sbt_core_triad_female_resolve_color,
        edge_fn=_sbt_core_triad_female_edge_for,
        edge_col=_SBT_CORE_TRIAD_FEMALE_EDGE, bg_color=_SBT_CORE_TRIAD_FEMALE_BG,
        save_name='sankey_sbt_core_triad_female',
        label_override={
            'sbt_core_depression':           'Core Depression',
            'sbt_academic_cognitive':        'Academic/Cognitive',
            'sbt_perfectionism_achievement': 'Perfectionism/Achievement',
        })

def _draw_sankey_sbt_core_triad_male(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_sbt_core_triad_sort_targets,
        color_fn=_sbt_core_triad_male_resolve_color,
        edge_fn=_sbt_core_triad_male_edge_for,
        edge_col=_SBT_CORE_TRIAD_MALE_EDGE, bg_color=_SBT_CORE_TRIAD_MALE_BG,
        save_name='sankey_sbt_core_triad_male',
        label_override={
            'sbt_core_depression':           'Core Depression',
            'sbt_academic_cognitive':        'Academic/Cognitive',
            'sbt_perfectionism_achievement': 'Perfectionism/Achievement',
        })

def _draw_sankey_child_alts(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_child_alts_sort_targets,
        color_fn=_child_alts_resolve_color,
        edge_fn=_child_alts_edge_for,
        edge_col=_CHILD_ALTS_EDGE, bg_color=_CHILD_ALTS_BG,
        save_name='sankey_child_alts',
        label_override={
            'Suicidal Ideation (Child)': 'Suicidal Ideation',
        })

def _draw_sankey_parent_alts(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_parent_alts_sort_targets,
        color_fn=_parent_alts_resolve_color,
        edge_fn=_parent_alts_edge_for,
        edge_col=_PARENT_ALTS_EDGE, bg_color=_PARENT_ALTS_BG,
        save_name='sankey_parent_alts',
        label_override={
            'Parent Happiness (Item)': 'Parent Happiness',
            'Parent Suicidal Ideation (Child)': 'Parent Suicidal Ideation',
        })

# ---
# CHILD COGNITIVE TASK BATTERY (T0) -- domain-coloured Sankey
# ---
# 7 targets covering distinct cognitive constructs at baseline (T0):
# crystallized intelligence (verbal knowledge), fluid composite (EF/speed/WM/
# episodic), Flanker (NIH inhibitory control), SST SSRT (motor inhibition),
# 2-back (emotional working memory), RAVLT long-delay (verbal episodic
# memory), LMT efficiency (visuospatial mental rotation). Palette is CUD-safe
# (Wong 2011): NIH Toolbox composites in cool blue-greens, paradigm-specific
# tasks in distinct hues across the warm-cool axis. Verified under
# deuteranopia, protanopia, tritanopia.
# ---

_CHILD_COGTASK_T0_COLORS = {
    # Palette tuned to match the Objective/Subjective-Report Spectrum heatmap.
    # Targets sharing a column with the heatmap inherit that column's hue;
    # the two NIH composites without dedicated columns (tb_cryst, tb_flanker)
    # take complementary cool tones that avoid green and grey for clean
    # external pairing with green/grey-heavy figures.
    #
    # NIH Toolbox composite -- matches "Comp / Fluid Intell." in heatmap
    'tb_fluid':                     (0.122, 0.176, 0.502),   # #1F2D80 deep indigo navy
    # Crystallized sibling (not in heatmap) -- cool steel-blue, no green/grey
    'tb_cryst':                     (0.290, 0.435, 0.647),   # #4A6FA5 steel-blue
    # Flanker (not in heatmap) -- lighter NIH family member, fluid constituent
    'tb_flanker':                   (0.337, 0.706, 0.914),   # #56B4E9 CUD sky blue
    # SST -- matches "SST" column hue in heatmap
    'sst_mean_ssrt':                (0.722, 0.204, 0.204),   # #B83434 firebrick red
    # N-back -- matches "N-Back" column hue in heatmap
    'nb_correct_nt_2back':          (0.055, 0.494, 0.537),   # #0E7E89 dark teal
    # RAVLT -- matches "RAVLT" column hue in heatmap
    'ravlt_l_total':                (0.165, 0.310, 0.843),   # #2A4FD7 royal blue
    # LMT -- matches "LMT" column hue in heatmap
    'lmt_efficiency':               (0.420, 0.306, 0.200),   # #6B4E33 saddle brown
}

_CHILD_COGTASK_T0_TARGET_EDGE = {
    'tb_fluid':                     '#0F1850',   # deeper indigo for navy fill
    'tb_cryst':                     '#2D456B',   # deeper steel-blue
    'tb_flanker':                   '#1F6F9F',   # deeper sky-blue
    'sst_mean_ssrt':                '#7A2020',   # deeper firebrick
    'nb_correct_nt_2back':          '#095157',   # deeper teal
    'ravlt_l_total':                '#1A3387',   # deeper royal blue
    'lmt_efficiency':               '#443120',   # deeper saddle brown
}

_CHILD_COGTASK_T0_DOMAIN_CLEAN_LABELS = {
    'tb_cryst':                     'Crystallized Intelligence',
    'tb_fluid':                     'Fluid Intelligence',
    'tb_flanker':                   'Cognitive Control (Flanker)',
    'sst_mean_ssrt':                'Response Inhibition (SSRT)',
    'nb_correct_nt_2back':          'Emotional Working Memory (2-Back)',
    'ravlt_l_total':                'Verbal Episodic Memory (RAVLT)',
    'lmt_efficiency':               'Visuospatial Mental Rotation (LMT)',
}

_CHILD_COGTASK_T0_DISPLAY_ALIASES = {
    # tb_cryst aliases (overlap with F4c style is intentional)
    'Crystallized Intelligence':            'tb_cryst',
    'Crystallized Composite (NIH)':         'tb_cryst',
    'Cryst./Fluid Comp.':                   'tb_cryst',
    # tb_fluid aliases
    'Fluid Intelligence':                   'tb_fluid',
    'Fluid Composite (NIH)':                'tb_fluid',
    # tb_flanker aliases (overlap with child_alts style is intentional)
    'Cognitive Control (Flanker)':          'tb_flanker',
    'Flanker':                              'tb_flanker',
    'Flanker Task':                         'tb_flanker',
    'Flanker (NIH Toolbox)':                'tb_flanker',
    'Flanker Inhibitory Control Task (NIH)':'tb_flanker',
    # sst_mean_ssrt aliases
    'Response Inhibition (SSRT)':           'sst_mean_ssrt',
    'Response Inhibition (SST SSRT)':       'sst_mean_ssrt',
    'SST Task Mean SSRT':                   'sst_mean_ssrt',
    # nb_correct_nt_2back aliases
    'Emotional Working Memory (2-Back)':    'nb_correct_nt_2back',
    '2-Back Acc.':                          'nb_correct_nt_2back',
    # ravlt_l_total aliases
    'Verbal Episodic Memory (RAVLT)':       'ravlt_l_total',
    'RAVLT Long Tot.':                      'ravlt_l_total',
    # lmt_efficiency aliases
    'Visuospatial Mental Rotation (LMT)':   'lmt_efficiency',
    'LMT Efficiency':                       'lmt_efficiency',
}

# Display order: NIH Toolbox composites first, then NIH subscale, then
# fMRI-paradigm behavioural tasks, then off-NIH off-fMRI paradigms.
# Mirrors the construct-coverage narrative (broad -> narrow -> orthogonal).
_CHILD_COGTASK_T0_ORDER = [
    'tb_cryst',
    'tb_fluid',
    'tb_flanker',
    'sst_mean_ssrt',
    'nb_correct_nt_2back',
    'ravlt_l_total',
    'lmt_efficiency',
]

_CHILD_COGTASK_T0_EDGE = '#1A2332'   # dark slate axis edge -- matches academic_cognitive
_CHILD_COGTASK_T0_BG   = '#FFFFFF'

def _child_cogtask_t0_resolve_color(target_name):
    """Resolve target colour for the T0 cognitive-task Sankey style.

    Tries (1) raw target key, (2) display-alias lookup, (3) substring
    fallback for archive keys like 'tb_cryst__tp0' or display names with
    suffixes. Returns mid-grey if no rule matches.
    """
    if target_name in _CHILD_COGTASK_T0_COLORS:
        return _CHILD_COGTASK_T0_COLORS[target_name]
    raw = _CHILD_COGTASK_T0_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_COGTASK_T0_COLORS:
        return _CHILD_COGTASK_T0_COLORS[raw]
    tl = target_name.lower()
    if 'cryst' in tl:                                            return _CHILD_COGTASK_T0_COLORS['tb_cryst']
    if 'fluid' in tl:                                            return _CHILD_COGTASK_T0_COLORS['tb_fluid']
    if 'flanker' in tl:                                          return _CHILD_COGTASK_T0_COLORS['tb_flanker']
    if 'ssrt' in tl or ('sst' in tl and 'inhib' in tl):          return _CHILD_COGTASK_T0_COLORS['sst_mean_ssrt']
    if '2-back' in tl or '2back' in tl or 'working memory' in tl: return _CHILD_COGTASK_T0_COLORS['nb_correct_nt_2back']
    if 'ravlt' in tl or 'verbal episodic' in tl:                 return _CHILD_COGTASK_T0_COLORS['ravlt_l_total']
    if 'lmt' in tl or 'mental rotation' in tl or 'visuospatial' in tl: return _CHILD_COGTASK_T0_COLORS['lmt_efficiency']
    return (0.5, 0.5, 0.5)

def _child_cogtask_t0_domain(target_name):
    """Return a short domain key for a T0 cognitive-task target."""
    if target_name in _CHILD_COGTASK_T0_COLORS:
        raw = target_name
    else:
        raw = _CHILD_COGTASK_T0_DISPLAY_ALIASES.get(target_name, target_name)
    tl = raw.lower()
    if 'cryst' in tl:                                            return 'crystallized'
    if 'fluid' in tl:                                            return 'fluid'
    if 'flanker' in tl:                                          return 'inhibition_flanker'
    if 'ssrt' in tl or ('sst' in tl and 'inhib' in tl):          return 'inhibition_sst'
    if '2-back' in tl or '2back' in tl:                          return 'wm_emotional'
    if 'ravlt' in tl:                                            return 'memory_verbal'
    if 'lmt' in tl or 'mental rotation' in tl:                   return 'visuospatial'
    return 'other'

def _child_cogtask_t0_edge_for(target_name):
    """Resolve target node edge colour."""
    if target_name in _CHILD_COGTASK_T0_TARGET_EDGE:
        return _CHILD_COGTASK_T0_TARGET_EDGE[target_name]
    raw = _CHILD_COGTASK_T0_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_COGTASK_T0_TARGET_EDGE:
        return _CHILD_COGTASK_T0_TARGET_EDGE[raw]
    rgb = _child_cogtask_t0_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _child_cogtask_t0_sort_targets(targets_list):
    """Order targets by construct narrative; unrecognised targets append in
    their incoming order to keep the drawer tolerant of new additions."""
    ranked = []
    leftover = []
    for t in targets_list:
        # Resolve archive keys / display names back to canonical
        canonical = t if t in _CHILD_COGTASK_T0_ORDER else _CHILD_COGTASK_T0_DISPLAY_ALIASES.get(t, t)
        if canonical in _CHILD_COGTASK_T0_ORDER:
            ranked.append((t, _CHILD_COGTASK_T0_ORDER.index(canonical)))
        else:
            leftover.append(t)
    ranked.sort(key=lambda x: x[1])
    return [t for t, _ in ranked] + leftover

def _draw_sankey_child_cogtask_t0(df_sankey, sankey_top_features, save_dir=None):
    """T0 Cognitive Task Battery Sankey.

    Routes through `_draw_sankey_domain_style` (the modern shared drawer
    used by `child_alts` / `parent_alts`), so this style automatically picks
    up the latest visual refinements (width-first sizing, landscape adjust,
    figure padding, target/feature node geometry) without duplicating
    drawer code.

    Two style-specific behaviours:
      1. Targets are sorted by R\u00b2 (best at top, worst at bottom) using
         `_batch_archive` metrics -- mirroring the SBT-by-perf pattern.
         Falls back to the construct-narrative order if metrics unavailable.
      2. Target spread is increased to 2.6 (vs 1.8 default) to give ribbons
         more breathing room on the left side for clearer connection traces.
    """
    # Build a perf-aware sort closure using the same machinery as SBT
    _ba = globals().get('_batch_archive', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _sort_by_r2(targets_list):
        if not _tm:
            # No metrics yet -- fall back to construct-narrative order
            return _child_cogtask_t0_sort_targets(targets_list)
        _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
        def _score(t):
            tm = _tm.get(t)
            if tm is None:
                tm = _tm.get(_tdn.get(t, ''))
            if tm is None:
                # try canonical key form (display alias -> raw)
                canonical = _CHILD_COGTASK_T0_DISPLAY_ALIASES.get(t, t)
                tm = _tm.get(canonical) or _tm.get(_tdn.get(canonical, ''))
            return tm[0] if tm is not None else float('-inf')
        # reverse=True puts highest R\u00b2 first; first element renders at TOP
        return sorted(targets_list, key=_score, reverse=True)

    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_sort_by_r2,
        color_fn=_child_cogtask_t0_resolve_color,
        edge_fn=_child_cogtask_t0_edge_for,
        edge_col=_CHILD_COGTASK_T0_EDGE, bg_color=_CHILD_COGTASK_T0_BG,
        save_name='sankey_child_cogtask_t0',
        spread_target_override=2.6,    # vertical: 1.8 default -> 2.6 (45% more)
        column_gap_override=2.2,       # horizontal: 1.30 default -> 2.2 (69% wider)
        feature_share_darkening=True,  # darken multi-target features toward black
        label_override={
            'Crystallized Composite (NIH)':            'Crystallized Intelligence',
            'Cryst./Fluid Comp.':                      'Crystallized Intelligence',
            'Flanker Inhibitory Control Task (NIH)':   'Cognitive Control (Flanker)',
            'SST Task Mean SSRT':                      'Response Inhibition (SSRT)',
            '2-Back Acc.':                             'Emotional Working Memory (2-Back)',
            'RAVLT Long Tot.':                         'Verbal Episodic Memory (RAVLT)',
            'LMT Efficiency':                          'Visuospatial Mental Rotation (LMT)',
        })

# ---
# DEP WORSENING -- shared helpers + two dedicated Sankey styles
# ---

_CHILD_DEP_WORSENING_ORDER = [
    'top_5_depression',
    'top_10_depression',
    'dep_increase_2sd',
    'dep_increase_1.5sd',
    'dep_onset_rci_2.3',
    'Depression (Top 5th Percentile)',
    'Depression (Top 10th Percentile)',
    'Depression Change (\u2265 2 SD)',
    'Depression Change (\u2265 1.5 SD)',
    'Depression Onset (RCI \u2265 2.3)',
    'Depression Increase (\u2265 2 SD)',
    'Depression Increase (\u2265 1.5 SD)',
]

_CHILD_LABEL_OVERRIDE = {
    'dep_increase_2sd':              'Depression Change (\u2265 2 SD)',
    'dep_increase_1.5sd':            'Depression Change (\u2265 1.5 SD)',
    'Depression Increase (\u2265 2 SD)':   'Depression Change (\u2265 2 SD)',
    'Depression Increase (\u2265 1.5 SD)': 'Depression Change (\u2265 1.5 SD)',
}

_PARENT_DEP_WORSENING_ORDER = [
    'top_5_depression_parent',
    'top_10_depression_parent',
    'parent_dep_increase_2sd',
    'parent_dep_increase_1.5sd',
    'parent_dep_onset_rci_2.3',
    'Parent Depression (Top 5th Percentile)',
    'Parent Depression (Top 10th Percentile)',
    'Parent Depression Changes (\u2265 2 SD)',
    'Parent Depression Changes (\u2265 1.5 SD)',
    'Parent Depression Onset (RCI \u2265 2.3)',
]

_DEP_WORSENING_COLORS = {
    # -- Child: saturated cool blues → contrasting violets --
    'top_5_depression':          (0.035, 0.150, 0.500),   # deep navy
    'top_10_depression':         (0.180, 0.420, 0.750),   # vivid blue
    'dep_increase_2sd':          (0.200, 0.010, 0.420),   # deep indigo-violet
    'dep_increase_1.5sd':        (0.560, 0.240, 0.740),   # bright orchid
    'dep_onset_rci_2.3':         (0.580, 0.350, 0.680),   # soft lavender-violet
    # -- Parent: warmer teal-blues → contrasting plums --
    'top_5_depression_parent':   (0.050, 0.180, 0.440),   # dark steel-blue
    'top_10_depression_parent':  (0.160, 0.360, 0.620),   # medium steel-blue
    'parent_dep_increase_2sd':   (0.260, 0.020, 0.240),   # deep wine-plum
    'parent_dep_increase_1.5sd': (0.540, 0.220, 0.500),   # bright rose-plum
    'parent_dep_onset_rci_2.3':  (0.580, 0.320, 0.480),   # dusty mauve
}

_DEP_WORSENING_BG    = '#FFFFFF'
_CHILD_DEP_EDGE      = '#0A2470'
_PARENT_DEP_EDGE     = '#0C3058'

_CHILD_FEAT_TOP  = (0.340, 0.520, 0.780)
_CHILD_FEAT_BOT  = (0.035, 0.150, 0.500)
_PARENT_FEAT_TOP = (0.280, 0.480, 0.580)
_PARENT_FEAT_BOT = (0.060, 0.250, 0.380)

def _dep_feat_fill(pos_frac, top, bot):
    r = top[0] + (bot[0] - top[0]) * pos_frac
    g = top[1] + (bot[1] - top[1]) * pos_frac
    b = top[2] + (bot[2] - top[2]) * pos_frac
    return (r, g, b)

def _dep_worsening_sort_targets(targets_list, order, is_parent=False):
    return sorted(targets_list)
def _dep_worsening_color(target_name):
    if target_name in _DEP_WORSENING_COLORS:
        return _DEP_WORSENING_COLORS[target_name]
    tl = target_name.lower()
    _is_parent = 'parent' in tl
    _is_top5   = 'top_5' in tl or 'top 5' in tl
    _is_top10  = 'top_10' in tl or 'top 10' in tl
    _is_rci    = 'rci' in tl or 'onset' in tl
    _is_15sd   = '1.5' in tl
    _is_2sd    = not _is_rci and not _is_15sd and ('2 sd' in tl or '2sd' in tl or '\u2265 2' in tl)
    if _is_parent:
        if _is_top5:   return _DEP_WORSENING_COLORS['top_5_depression_parent']
        if _is_top10:  return _DEP_WORSENING_COLORS['top_10_depression_parent']
        if _is_rci:    return _DEP_WORSENING_COLORS['parent_dep_onset_rci_2.3']
        if _is_15sd:   return _DEP_WORSENING_COLORS['parent_dep_increase_1.5sd']
        return _DEP_WORSENING_COLORS['parent_dep_increase_2sd']
    if _is_top5:   return _DEP_WORSENING_COLORS['top_5_depression']
    if _is_top10:  return _DEP_WORSENING_COLORS['top_10_depression']
    if _is_rci:    return _DEP_WORSENING_COLORS['dep_onset_rci_2.3']
    if _is_15sd:   return _DEP_WORSENING_COLORS['dep_increase_1.5sd']
    return _DEP_WORSENING_COLORS['dep_increase_2sd']

_DEP_SANKEY_SUBTITLE = (
    'Across-Category Ensemble Classification Performance and Feature Importance Combined'
    ' \u00b7 Depression Syndrome Severity: Top 5th/10th Percentile'
    ' & Change from Prior Timepoint: \u2265 1.5 SD \u00b7 \u2265 2 SD'
)

def _draw_sankey_dep_worsening_shared(df_sankey, sankey_top_features, save_dir=None,
                                       is_parent=False, label_override=None):
    """Shared implementation for child and parent depression worsening Sankeys."""
    if is_parent:
        _TARGET_ORDER = _PARENT_DEP_WORSENING_ORDER
        _EDGE_COL     = _PARENT_DEP_EDGE
        _SAVE_NAME    = 'sankey_parent_clinical_depression'
        _FEAT_TOP, _FEAT_BOT = _PARENT_FEAT_BOT, _PARENT_FEAT_TOP
        _FEAT_EDGE    = '#081829'
    else:
        _TARGET_ORDER = _CHILD_DEP_WORSENING_ORDER
        _EDGE_COL     = _CHILD_DEP_EDGE
        _SAVE_NAME    = 'sankey_child_clinical_depression'
        _FEAT_TOP, _FEAT_BOT = _CHILD_FEAT_BOT, _CHILD_FEAT_TOP
        _FEAT_EDGE    = '#091F38'
    _BG = _DEP_WORSENING_BG

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _dep_worsening_sort_targets(_cd['targets_list'], _TARGET_ORDER)
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc  = {t: _dep_worsening_color(t) for t in targets_list}

    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x  = _LEFT_X
    right_x = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028           # feature node half-width (square: 0.056 wide = _FEAT_H)

    # -- Width-first figure sizing --
    fig_w, fig_h = _wide_figure_dims(n_targets, n_features)
    fig_w, fig_h = fig_w * 1.5, fig_h * 1.5   # scale up for higher resolution
    _spread_t = 4.5
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    _target_fs = 19                # override target label size

    global_max   = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h  = min(0.14, 3.5 / max(n_features, 1))

    target_positions  = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)

    # -- Equalize vertical envelopes --
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    # -- Normalize all target node heights to uniform size --
    # Rect pad=0.010; pill pad=0.022 → shrink pill internal h to compensate.
    _RECT_H = 0.19
    _PILL_H = 0.16        # pill pad=0.016 compensated for ~equal rendered height
    if target_positions:
        for t in targets_list:
            _tl = t.lower()
            _is_sd = ('sd' in _tl or '≥' in t) and 'top' not in _tl and 'rci' not in _tl and 'onset' not in _tl
            _new_h = _PILL_H if _is_sd else _RECT_H
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c,
                'y_top':    _c + _new_h / 2,
                'y_bot':    _c - _new_h / 2,
                'h':        _new_h,
            }

    # -- Normalize feature node heights to uniform --
    _FEAT_H = 0.056
    if feature_positions:
        for f_key in features_list:
            if f_key in feature_positions:
                _c = feature_positions[f_key]['y_centre']
                feature_positions[f_key] = {
                    'y_centre': _c,
                    'y_top':    _c + _FEAT_H / 2,
                    'y_bot':    _c - _FEAT_H / 2,
                    'h':        _FEAT_H,
                }

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.28
    ax.set_xlim(-0.65, right_x + 0.78)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    # -- Port gap: shrink each ribbon port slightly for visible separation --
    _PORT_GAP = 0.002   # data-coord gap between adjacent ribbon ports

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        # Inset ports by half-gap on each side for visible separation
        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        # -- Saturation + luminance ramp --
        _desat = 0.55 + 0.45 * rel
        _white = (0.88, 0.90, 0.94)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        # -- Glow halo on high-importance ribbons --
        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.005, _yr_b - 0.005,
                color_rgba=_fill + (_alpha * 0.15,), zorder=1,
                glow_layers=_glow_n)

        # -- Edge darkening on top ~30% (subtle) --
        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

    for t, pos in target_positions.items():
        col = _tc.get(t, (0.5, 0.5, 0.5))
        _tl = t.lower()
        _is_sd = ('sd' in _tl or '\u2265' in t) and 'top' not in _tl and 'rci' not in _tl and 'onset' not in _tl

        # SD-change targets → pill/capsule; top-percentile → sharp rect
        _bstyle = 'round,pad=0.016' if _is_sd else 'round,pad=0.008'

        ax.add_patch(FancyBboxPatch(
            (left_x - node_w + 0.004, pos['y_bot'] - 0.003),
            node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))
        ax.add_patch(FancyBboxPatch(
            (left_x - node_w, pos['y_bot']),
            node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=col, edgecolor=_EDGE_COL,
            linewidth=1.8, zorder=3))

        display = _tdn.get(t, t)
        if label_override:
            display = label_override.get(t, label_override.get(display, display))
        _tm = _target_metrics.get(t) or _target_metrics.get(display)
        _label_x = left_x - node_w - 0.04

        if _tm is not None:
            _val, _lbl = _tm
            txt = ax.text(_label_x, pos['y_centre'] + 0.015,
                    display, ha='right', va='center', fontsize=_target_fs, fontweight='bold',
                    color='#1a1a1a', fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.018,
                    f'{_lbl} = {_val:.3f}',
                    ha='right', va='center', fontsize=_lf[2], fontweight='bold',
                    color=_EDGE_COL, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                    display, ha='right', va='center', fontsize=_target_fs, fontweight='bold',
                    color='#1a1a1a', fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground=_BG)])

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count[f]
        dom_rows = df_sankey[df_sankey['Feature'] == f]
        dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                 if not dom_rows.empty else targets_list[0])
        glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))

        if n_tgt >= 2:
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        # -- Single-target features inherit their target colour --
        if n_tgt == 1:
            _fc = _tc.get(dom_t, (0.5, 0.5, 0.5))
        else:
            # Dark greyish-blue → black across 2T/3T/4T
            _t = min((n_tgt - 2) / 2.0, 1.0)           # 0 at 2T, 0.5 at 3T, 1.0 at 4T+
            _fc = (0.32 - 0.27 * _t,                   # R: 0.32 → 0.05
                   0.36 - 0.29 * _t,                   # G: 0.36 → 0.07
                   0.46 - 0.33 * _t)                   # B: 0.46 → 0.13
        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))
        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc,
            edgecolor=_FEAT_EDGE, linewidth=0.7, zorder=3))

        badge = f" ({n_tgt}T)" if n_tgt >= 2 else ""
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                f"{f}{badge}",
                ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                color='#1a1a1a', fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(linewidth=2.5, foreground=_BG)])

    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw], [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_EDGE_COL, lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw], [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_EDGE_COL, lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    plt.tight_layout(rect=[0.01, 0.06, 0.86, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, _SAVE_NAME, dpi=1200)
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    return fig

def _draw_sankey_child_dep_worsening(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_dep_worsening_shared(
        df_sankey, sankey_top_features, save_dir=save_dir,
        is_parent=False, label_override=_CHILD_LABEL_OVERRIDE)

def _draw_sankey_parent_dep_worsening(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_dep_worsening_shared(
        df_sankey, sankey_top_features, save_dir=save_dir,
        is_parent=True, label_override=None)

# ---
# DEPRESSION SUBTYPE SANKEYS -- Female & Male Stratified
# ---

_SBT_BLUE_TOP = (0.114, 0.302, 0.580)

_SBT_TARGET_COLORS = {
    'sbt_core_depression':             (0.114, 0.302, 0.580),
    'sbt_anxiety_depression':          (0.690, 0.125, 0.188),
    'sbt_guilt_hopelessness':          (0.227, 0.227, 0.290),
    'sbt_fatigue_sleep':               (0.165, 0.478, 0.478),
    'sbt_somatic_depression':          (0.545, 0.353, 0.169),
    'sbt_emotional_dysregulation':     (0.753, 0.416, 0.063),
    'sbt_social_withdrawal':           (0.353, 0.439, 0.533),
    'sbt_avoidance_fear':              (0.478, 0.227, 0.541),
    'sbt_aggression_irritability':     (0.176, 0.416, 0.180),
    'sbt_academic_cognitive':          (0.722, 0.588, 0.043),
    'sbt_perfectionism_achievement':   (0.769, 0.290, 0.502),
}

_SBT_TARGET_ORDER = [
    'sbt_core_depression',
    'sbt_anxiety_depression',
    'sbt_guilt_hopelessness',
    'sbt_fatigue_sleep',
    'sbt_somatic_depression',
    'sbt_emotional_dysregulation',
    'sbt_social_withdrawal',
    'sbt_avoidance_fear',
    'sbt_aggression_irritability',
    'sbt_academic_cognitive',
    'sbt_perfectionism_achievement',
]

_SBT_DISPLAY_NAMES = {
    'sbt_core_depression':           'Core Depression',
    'sbt_anxiety_depression':        'Anxiety\u2013Depression',
    'sbt_guilt_hopelessness':        'Guilt & Hopelessness',
    'sbt_fatigue_sleep':             'Fatigue & Sleep',
    'sbt_somatic_depression':        'Somatic Depression',
    'sbt_emotional_dysregulation':   'Emotional Dysregulation',
    'sbt_social_withdrawal':         'Social Withdrawal',
    'sbt_avoidance_fear':            'Avoidance & Fear',
    'sbt_aggression_irritability':   'Aggression & Irritability',
    'sbt_academic_cognitive':        'Academic & Cognitive',
    'sbt_perfectionism_achievement': 'Perfectionism & Achievement',
}

_SBT_THEMES = {
    'female': {
        'bg_color':   '#FFFFFF',
        'edge_color': '#6B2040',
        'text_color': '#1a1a1a',
        'hdr_color':  '#6B2040',
        'save_name':  'sankey_sbt_female',
        'title':      'Depression Subtype Predictive Feature Architecture (Female)',
    },
    'long': {
        'bg_color':   '#FFFFFF',
        'edge_color': '#2A3A5A',
        'text_color': '#1a1a1a',
        'hdr_color':  '#2A3A5A',
        'save_name':  'sankey_sbt_long',
        'title':      'Depression Subtype Predictive Feature Architecture (Longitudinal)',
    },
}

def _sbt_blend_color(pos_frac, subtype_rgb):
    r = _SBT_BLUE_TOP[0] + (subtype_rgb[0] - _SBT_BLUE_TOP[0]) * pos_frac
    g = _SBT_BLUE_TOP[1] + (subtype_rgb[1] - _SBT_BLUE_TOP[1]) * pos_frac
    b = _SBT_BLUE_TOP[2] + (subtype_rgb[2] - _SBT_BLUE_TOP[2]) * pos_frac
    return (r, g, b)

def _sbt_sort_targets(targets_list):
    return sorted(targets_list)
def _sbt_sort_targets_by_perf(targets_list, target_metrics):
    if not target_metrics:
        return _sbt_sort_targets(targets_list)

    def _get_score(t):
        tm = target_metrics.get(t)
        if tm is None:
            tm = target_metrics.get(_sbt_display(t))
        if tm is None:
            _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
            tm = target_metrics.get(_tdn.get(t, ''))
        return tm[0] if tm is not None else -1.0

    return sorted(targets_list, key=_get_score, reverse=True)

def _sbt_resolve_color(target_name):
    if target_name in _SBT_TARGET_COLORS:
        return _SBT_TARGET_COLORS[target_name]
    for raw, display in _SBT_DISPLAY_NAMES.items():
        if target_name == display:
            return _SBT_TARGET_COLORS[raw]
    tl = target_name.lower()
    # Explicit substring fallbacks for display-name variants
    if 'core' in tl or 'primary' in tl or ('mood' in tl and 'depress' in tl):
        return _SBT_TARGET_COLORS['sbt_core_depression']
    if 'anxiety' in tl:
        return _SBT_TARGET_COLORS['sbt_anxiety_depression']
    if 'guilt' in tl or 'hopeless' in tl:
        return _SBT_TARGET_COLORS['sbt_guilt_hopelessness']
    if 'fatigue' in tl or 'sleep' in tl:
        return _SBT_TARGET_COLORS['sbt_fatigue_sleep']
    if 'somatic' in tl:
        return _SBT_TARGET_COLORS['sbt_somatic_depression']
    if 'emotional' in tl or 'dysregul' in tl:
        return _SBT_TARGET_COLORS['sbt_emotional_dysregulation']
    if 'social' in tl or 'withdrawal' in tl:
        return _SBT_TARGET_COLORS['sbt_social_withdrawal']
    if 'avoidance' in tl or 'fear' in tl:
        return _SBT_TARGET_COLORS['sbt_avoidance_fear']
    if 'aggress' in tl or 'irritab' in tl:
        return _SBT_TARGET_COLORS['sbt_aggression_irritability']
    if 'academic' in tl or 'cognitive' in tl:
        return _SBT_TARGET_COLORS['sbt_academic_cognitive']
    if 'perfect' in tl or 'achiev' in tl:
        return _SBT_TARGET_COLORS['sbt_perfectionism_achievement']
    return (0.4, 0.4, 0.4)

def _sbt_display(target_name):
    if target_name in _SBT_DISPLAY_NAMES:
        return _SBT_DISPLAY_NAMES[target_name]
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    return _tdn.get(target_name, target_name)

def _draw_sankey_sbt(df_sankey, sankey_top_features, save_dir=None, sex='female'):
    """Depression Subtype Sankey -- sex-stratified."""
    theme = _SBT_THEMES[sex]
    _BG        = theme['bg_color']
    _EDGE_COL  = theme['edge_color']
    _SAVE_NAME = theme['save_name']

    _cd = _sankey_common_data(df_sankey)
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})

    targets_list = _sbt_sort_targets_by_perf(_cd['targets_list'], _target_metrics)

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {}
    _t_imp_range = {}
    for t in targets_list:
        _tc[t] = _sbt_resolve_color(t)
        t_rows = df_sankey[df_sankey['Target'] == t]
        if not t_rows.empty:
            _t_imp_range[t] = (t_rows['Importance'].min(), t_rows['Importance'].max())
        else:
            _t_imp_range[t] = (0.0, 1.0)

    _pad      = max(0.015, min(0.035, 1.2 / max(n_features, 1)))
    left_x  = _LEFT_X
    right_x = _RIGHT_X
    node_w    = 0.028
    _target_node_w = 0.036

    # -- Reverse layout: targets on right, features on left for 'long' --
    _reverse = (sex == 'long')
    if _reverse:
        _tgt_x  = right_x      # targets on the right
        _feat_x = left_x       # features on the left
        _tgt_label_ha  = 'left'
        _feat_label_ha = 'right'
        _tgt_label_dx  = _target_node_w + 0.022   # label offset from tgt_x
        _feat_label_dx = -(node_w + 0.020)         # label offset from feat_x
    else:
        _tgt_x  = left_x       # targets on the left (default)
        _feat_x = right_x      # features on the right
        _tgt_label_ha  = 'right'
        _feat_label_ha = 'left'
        _tgt_label_dx  = -(_target_node_w + 0.022)
        _feat_label_dx = node_w + 0.020

    # -- Width-first figure sizing --
    fig_w, fig_h = _wide_figure_dims(n_targets, n_features)
    _spread_t = 2.5
    _spread_f = 1.5
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()

    global_max   = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h  = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)

    # -- Equalize vertical envelopes -- 80 % for 11 subtypes --
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions, target_span_frac=0.80)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04
    ax.set_xlim(-0.65, right_x + 0.65)
    ax.set_ylim(y_min, 1.08)
    ax.axis('off')

    # -- Ribbons --
    # Uses the full subtype color directly (no navy→color blend), with alpha
    # encoding importance -- matching the pfactor_spectra visual style.
    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        t = rb['target']
        _rgb = _tc.get(t, (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])
        _alpha = 0.18 + 0.67 * rel

        if _reverse:
            _rib_left_nw  = node_w
            _rib_right_nw = _target_node_w
            _yl_top, _yl_bot = rb['y_r_top'], rb['y_r_bot']
            _yr_top, _yr_bot = rb['y_l_top'], rb['y_l_bot']
        else:
            _rib_left_nw  = _target_node_w
            _rib_right_nw = node_w
            _yl_top, _yl_bot = rb['y_l_top'], rb['y_l_bot']
            _yr_top, _yr_bot = rb['y_r_top'], rb['y_r_bot']

        # Glow halo for high-importance ribbons
        if rel > 0.60:
            _draw_bezier_ribbon_enhanced(
                ax, left_x, _rib_left_nw, right_x,
                _yl_top + 0.005, _yl_bot - 0.005,
                _yr_top + 0.005, _yr_bot - 0.005,
                color_rgba=_rgb + (_alpha * 0.35,),
                zorder=1, right_node_w=_rib_right_nw)

        # Low-importance ribbons: white edges for visual separation
        if rel < 0.35:
            _edge_c = (1.0, 1.0, 1.0, 0.30)
            _edge_lw = 0.5
        else:
            _edge_c = _rgb + (min(_alpha + 0.10, 0.85),)
            _edge_lw = 0.4

        _draw_bezier_ribbon_enhanced(
            ax, left_x, _rib_left_nw, right_x,
            _yl_top, _yl_bot,
            _yr_top, _yr_bot,
            color_rgba=_rgb + (_alpha,),
            zorder=2, edge_color=_edge_c, edge_lw=_edge_lw,
            right_node_w=_rib_right_nw)

    # -- Target nodes --
    for i, (t, pos) in enumerate(zip(targets_list,
                                      [target_positions[t] for t in targets_list])):
        col = _tc[t]

        ax.add_patch(FancyBboxPatch(
            (_tgt_x - _target_node_w + 0.005, pos['y_bot'] - 0.005),
            _target_node_w * 2, pos['h'],
            boxstyle='round,pad=0.008',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.18))

        ax.add_patch(FancyBboxPatch(
            (_tgt_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle='round,pad=0.008',
            facecolor=col, edgecolor=_EDGE_COL,
            linewidth=2.2, zorder=3))

        display = _sbt_display(t)
        _tm = _target_metrics.get(t) or _target_metrics.get(display)
        _label_x = _tgt_x + _tgt_label_dx

        if _tm is not None:
            _val, _lbl = _tm
            txt = ax.text(_label_x, pos['y_centre'] + 0.018,
                    display, ha=_tgt_label_ha, va='center', fontsize=_lf[0], fontweight='bold',
                    color=theme['text_color'], fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.025,
                    f'{_lbl} = {_val:.3f}',
                    ha=_tgt_label_ha, va='center', fontsize=_lf[2], fontweight='bold',
                    color=_EDGE_COL, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                    display, ha=_tgt_label_ha, va='center', fontsize=_lf[0], fontweight='bold',
                    color=theme['text_color'], fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])

    # -- Feature nodes -- target-inherited color for single-target, dark blue for multi --
    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count[f]
        dom_rows = df_sankey[df_sankey['Feature'] == f].sort_values('Importance', ascending=False)
        dom_t = (dom_rows.iloc[0]['Target']
                 if not dom_rows.empty else targets_list[0])

        _nx = _feat_x - node_w
        _nw = node_w * 2
        _ny = pos['y_bot']
        _nh = pos['h']

        ax.add_patch(FancyBboxPatch(
            (_nx + 0.004, _ny - 0.004), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.18))

        if n_tgt >= 2:
            top2_targets = dom_rows['Target'].head(2).tolist()
            glow_outer = _tc.get(top2_targets[0], (0.5, 0.5, 0.5))
            glow_inner = _tc.get(top2_targets[1], (0.5, 0.5, 0.5)) if len(top2_targets) > 1 else glow_outer
            ax.add_patch(FancyBboxPatch(
                (_nx - 0.007, _ny - 0.007), _nw + 0.014, _nh + 0.014,
                boxstyle="round,pad=0.008",
                facecolor='none', edgecolor=glow_outer + (0.50,),
                linewidth=2.0, zorder=2))
            ax.add_patch(FancyBboxPatch(
                (_nx - 0.004, _ny - 0.004), _nw + 0.008, _nh + 0.008,
                boxstyle="round,pad=0.006",
                facecolor='none', edgecolor=glow_inner + (0.45,),
                linewidth=2.0, zorder=2))

        # Single-target features inherit that target's color;
        # multi-target features get very dark blue.
        if n_tgt == 1:
            _fc = _tc.get(dom_t, (0.5, 0.5, 0.5))
        else:
            _fc = (0.06, 0.10, 0.22)

        ax.add_patch(FancyBboxPatch(
            (_nx, _ny), _nw, _nh,
            boxstyle='round,pad=0.008',
            facecolor=_fc,
            edgecolor='#444444', linewidth=1.0, zorder=3))

        badge = f" ({n_tgt}T)" if n_tgt >= 2 else ""
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(_feat_x + _feat_label_dx, pos['y_centre'],
                f"{f}{badge}",
                ha=_feat_label_ha, va='center', fontsize=_fs_feat, fontweight=fw,
                color=theme['text_color'], fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])

    # -- Column headers --
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    ax.text(_tgt_x, _hdr_y, 'SUBTYPES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.text(_feat_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.plot([_tgt_x - _target_node_w * 1.5, _tgt_x + _target_node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=_EDGE_COL, lw=1.0, alpha=0.5, clip_on=False, zorder=5)
    ax.plot([_feat_x - node_w * 1.5, _feat_x + node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=_EDGE_COL, lw=1.0, alpha=0.5, clip_on=False, zorder=5)

    ax.text(0.50, 1.06, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)
    # Subtitle at bottom with methodology notes
    ax.text(0.50, -0.015,
            f'Top {sankey_top_features} Consensus Features per Subtype  \u00b7  '
            'Shared Features Annotated (nT)  \u00b7  '
            'R\u00b2 from Across-Category Analyses\n'
            'Ribbon Opacity \u221d Weighted Ensemble Importance  \u00b7  '
            'Ribbon Color = Subtype Identity  \u00b7  '
            'Circular Features Removed',
            ha='center', va='top', fontsize=_lf[3], fontweight='bold',
            color='#1a1a1a', fontfamily=_TITLE_FONT, linespacing=1.4,
            transform=ax.transAxes)

    plt.tight_layout(rect=[0.01, 0.015, 0.88, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, _SAVE_NAME, dpi=600)
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    return fig

def _draw_sankey_sbt_female(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_sbt(df_sankey, sankey_top_features, save_dir=save_dir, sex='female')

def _draw_sankey_sbt_long(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_sbt(df_sankey, sankey_top_features, save_dir=save_dir, sex='long')

# ---
# PRIMARY TARGETS -- Matched Child CBCL ↔ Parent ASR DSM-Oriented Scales
# ---
# 8 targets: 4 child (brighter) + 4 parent (deeper/muted) paired by construct.
# Visual pairing: same hue family, child=saturated, parent=desaturated.

_PRIMARY_TARGETS_COLORS = {
    # Child (CBCL DSM-oriented) -- saturated
    'depress_D_p':              (0.15, 0.39, 0.91),   # vivid blue
    'anxdisord_D_p':            (0.85, 0.18, 0.18),   # red
    'adhd_D_p':                 (0.82, 0.68, 0.15),   # gold
    'social_problems_D_p':      (0.90, 0.50, 0.13),   # amber-orange
    # Parent (ASR analogues) -- deeper/muted
    'parent_depress_D_p':       (0.20, 0.28, 0.62),   # deep blue
    'parent_anxdisord_D_p':     (0.62, 0.12, 0.12),   # dark red
    'parent_adhd_D_p':          (0.65, 0.55, 0.10),   # dark gold
    'parent_bad_relationships_p': (0.70, 0.30, 0.05), # burnt sienna
}

_PRIMARY_TARGETS_ORDER = [
    # Child block
    'depress_D_p', 'anxdisord_D_p', 'adhd_D_p', 'social_problems_D_p',
    # Parent block
    'parent_depress_D_p', 'parent_anxdisord_D_p', 'parent_adhd_D_p', 'parent_bad_relationships_p',
]

_PRIMARY_TARGETS_DISPLAY = {
    'depress_D_p':                'Depression Syndrome',
    'anxdisord_D_p':              'Anxiety Disorder',
    'adhd_D_p':                   'ADHD',
    'social_problems_D_p':        'Social Problems',
    'parent_depress_D_p':         'Parent Depression Syndrome',
    'parent_anxdisord_D_p':       'Parent Anxiety Disorder',
    'parent_adhd_D_p':            'Parent ADHD',
    'parent_bad_relationships_p': 'Parent Interpersonal Difficulties',
}

_PRIMARY_TARGETS_EDGE = '#1E293B'
_PRIMARY_TARGETS_BG   = '#FFFFFF'

def _primary_targets_resolve_color(target_name):
    if target_name in _PRIMARY_TARGETS_COLORS:
        return _PRIMARY_TARGETS_COLORS[target_name]
    # Reverse lookup by display name
    for raw, display in _PRIMARY_TARGETS_DISPLAY.items():
        if target_name == display:
            return _PRIMARY_TARGETS_COLORS[raw]
    # Substring fallback
    tl = target_name.lower()
    if 'parent' in tl:
        if 'depress' in tl:   return _PRIMARY_TARGETS_COLORS['parent_depress_D_p']
        if 'anx' in tl:       return _PRIMARY_TARGETS_COLORS['parent_anxdisord_D_p']
        if 'adhd' in tl:      return _PRIMARY_TARGETS_COLORS['parent_adhd_D_p']
        if 'relation' in tl or 'social' in tl or 'interpers' in tl:
            return _PRIMARY_TARGETS_COLORS['parent_bad_relationships_p']
    else:
        if 'depress' in tl:   return _PRIMARY_TARGETS_COLORS['depress_D_p']
        if 'anx' in tl:       return _PRIMARY_TARGETS_COLORS['anxdisord_D_p']
        if 'adhd' in tl:      return _PRIMARY_TARGETS_COLORS['adhd_D_p']
        if 'social' in tl:    return _PRIMARY_TARGETS_COLORS['social_problems_D_p']
    return (0.45, 0.45, 0.50)

def _primary_targets_edge_for(target_name):
    rgb = _primary_targets_resolve_color(target_name)
    return tuple(max(0, c * 0.6) for c in rgb)

def _primary_targets_sort(targets_list):
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _score(t):
        entry = _tm.get(t)
        if entry is None:
            entry = _tm.get(_tdn.get(t, ''))
        if entry is None:
            entry = _tm.get(_PRIMARY_TARGETS_DISPLAY.get(t, ''))
        return entry[0] if entry is not None else -1.0

    scores = {t: _score(t) for t in targets_list}
    has_scores = any(v > 0 for v in scores.values())

    if has_scores:
        # Sort within child and parent blocks separately by performance
        child = [t for t in targets_list if 'parent' not in t.lower()]
        parent = [t for t in targets_list if 'parent' in t.lower()]
        child.sort(key=lambda t: scores.get(t, -1), reverse=True)
        parent.sort(key=lambda t: scores.get(t, -1), reverse=True)
        return child + parent

    # Fallback: canonical order
def _draw_sankey_primary_targets(df_sankey, sankey_top_features, save_dir=None):
    """Primary Targets Sankey -- matched child↔parent DSM-oriented scales.
    Standard layout (targets left, features right) with full visual polish."""

    _BG = _PRIMARY_TARGETS_BG
    _EDGE_COL = _PRIMARY_TARGETS_EDGE
    text_color = '#1a1a1a'

    _cd = _sankey_common_data(df_sankey)
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    # -- Metrics suppressed for manual Illustrator annotation --
    _target_metrics = {}
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})

    targets_list = _primary_targets_sort(_cd['targets_list'])

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm = Normalize(vmin=_imp_min, vmax=_imp_max)

    target_colors = {t: _primary_targets_resolve_color(t) for t in targets_list}

    # Display name resolution
    def _display_target(t):
        d = _PRIMARY_TARGETS_DISPLAY.get(t)
        if d:
            return d
        d = _tdn.get(t)
        if d:
            return d
        return t

    # Dominant target per feature (for single-target coloring)
    _feat_dom_target = {}
    for f in features_list:
        frows = df_sankey[df_sankey['Feature'] == f].sort_values('Importance', ascending=False)
        _feat_dom_target[f] = frows.iloc[0]['Target'] if not frows.empty else targets_list[0]

    _pad = max(0.015, min(0.035, 1.2 / max(n_features, 1)))
    left_x   = _LEFT_X
    right_x  = _RIGHT_X
    node_w   = 0.028
    _target_node_w = 0.038

    # -- Figure sizing --
    fig_w, fig_h = _wide_figure_dims(n_targets, n_features)
    _spread_t = 2.2
    _spread_f = 1.5
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)

    # Equalize vertical envelopes
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions, target_span_frac=0.82)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04
    ax.set_xlim(-0.65, right_x + 0.65)
    ax.set_ylim(y_min, 1.08)
    ax.axis('off')

    # -- Child/Parent divider line between target blocks --
    child_targets = [t for t in targets_list if 'parent' not in t.lower()]
    parent_targets = [t for t in targets_list if 'parent' in t.lower()]
    if child_targets and parent_targets:
        last_child_y = target_positions[child_targets[-1]]['y_bot']
        first_parent_y = target_positions[parent_targets[0]]['y_top']
        _div_y = (last_child_y + first_parent_y) / 2
        ax.plot([left_x - _target_node_w * 2.5, left_x + _target_node_w * 2.5],
                [_div_y, _div_y], color='#CBD5E1', lw=0.8, ls='--', alpha=0.6, zorder=1)
        ax.text(left_x - _target_node_w * 2.8, _div_y + 0.012, 'CHILD',
                fontsize=7, color='#94A3B8', fontweight='600', ha='right',
                fontfamily=_TITLE_FONT, alpha=0.7)
        ax.text(left_x - _target_node_w * 2.8, _div_y - 0.016, 'PARENT',
                fontsize=7, color='#94A3B8', fontweight='600', ha='right',
                fontfamily=_TITLE_FONT, alpha=0.7)

    # -- Ribbons (full-color + alpha, pfactor style) --
    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb = target_colors.get(rb['target'], (0.5, 0.5, 0.5))
        rel = _norm(rb['importance'])
        _alpha = 0.18 + 0.67 * rel

        # Glow halo for high-importance ribbons
        if rel > 0.60:
            _draw_bezier_ribbon_enhanced(
                ax, left_x, _target_node_w, right_x,
                rb['y_l_top'] + 0.005, rb['y_l_bot'] - 0.005,
                rb['y_r_top'] + 0.005, rb['y_r_bot'] - 0.005,
                color_rgba=_rgb + (_alpha * 0.35,),
                zorder=1, right_node_w=node_w)

        # Low-importance: white edges for separation
        if rel < 0.35:
            _edge_c = (1.0, 1.0, 1.0, 0.30)
            _edge_lw = 0.5
        else:
            _edge_c = _rgb + (min(_alpha + 0.10, 0.85),)
            _edge_lw = 0.4

        _draw_bezier_ribbon_enhanced(
            ax, left_x, _target_node_w, right_x,
            rb['y_l_top'], rb['y_l_bot'],
            rb['y_r_top'], rb['y_r_bot'],
            color_rgba=_rgb + (_alpha,),
            zorder=2, edge_color=_edge_c, edge_lw=_edge_lw, right_node_w=node_w)

    # -- Target nodes (left) --
    for t, pos in target_positions.items():
        rgb = target_colors[t]
        edge_c = _primary_targets_edge_for(t)

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.006, pos['y_bot'] - 0.006),
            _target_node_w * 2, pos['h'],
            boxstyle="round,pad=0.007",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.22))

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle="round,pad=0.007",
            facecolor=rgb, edgecolor=edge_c,
            linewidth=2.2, zorder=3))

        _disp_t = _display_target(t)
        _tm = _target_metrics.get(t) or _target_metrics.get(_disp_t)
        _label_x = left_x - _target_node_w - 0.025

        if _tm is not None:
            _val, _lbl = _tm
            txt = ax.text(_label_x, pos['y_centre'] + 0.018,
                    _disp_t, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=text_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])
            _metric_color = edge_c
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.022,
                    f'{_lbl} = {_val:.3f}',
                    ha='right', va='center', fontsize=_lf[2], fontweight='bold',
                    color=_metric_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                    _disp_t, ha='right', va='center', fontsize=_lf[0], fontweight='bold',
                    color=text_color, fontfamily=_TITLE_FONT, zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(linewidth=3.5, foreground=_BG)])

    # -- Feature nodes (right) --
    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))

    for f, pos in feature_positions.items():
        n_tgt = feat_target_count[f]
        dom_t = _feat_dom_target[f]
        fc = target_colors.get(dom_t, (0.5, 0.5, 0.5)) if n_tgt == 1 else (0.42, 0.48, 0.58)

        _nx = right_x - node_w
        _nw = node_w * 2
        _ny = pos['y_bot']
        _nh = pos['h']

        # Drop shadow
        ax.add_patch(FancyBboxPatch(
            (_nx + 0.004, _ny - 0.004), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.18))

        # Multi-target glow rings
        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f].sort_values('Importance', ascending=False)
            top2 = dom_rows['Target'].head(2).tolist()
            glow_outer = target_colors.get(top2[0], (0.5, 0.5, 0.5))
            glow_inner = target_colors.get(top2[1], (0.5, 0.5, 0.5)) if len(top2) > 1 else glow_outer
            ax.add_patch(FancyBboxPatch(
                (_nx - 0.007, _ny - 0.007), _nw + 0.014, _nh + 0.014,
                boxstyle="round,pad=0.008",
                facecolor='none', edgecolor=glow_outer + (0.45,),
                linewidth=2.0, zorder=2))
            ax.add_patch(FancyBboxPatch(
                (_nx - 0.004, _ny - 0.004), _nw + 0.008, _nh + 0.008,
                boxstyle="round,pad=0.006",
                facecolor='none', edgecolor=glow_inner + (0.40,),
                linewidth=2.0, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (_nx, _ny), _nw, _nh,
            boxstyle="round,pad=0.006",
            facecolor=fc, edgecolor='#444444', linewidth=1.0, zorder=3))

        badge = f" ({n_tgt}T)" if n_tgt >= 2 else ""
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + node_w + 0.020, pos['y_centre'],
                f"{f}{badge}",
                ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                color=text_color, fontfamily=_TITLE_FONT, zorder=5, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground=_BG)])

    # -- Column headers --
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.045
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.plot([left_x - _target_node_w * 1.5, left_x + _target_node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=_EDGE_COL, lw=1.0, alpha=0.5, clip_on=False)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_EDGE_COL, fontfamily=_TITLE_FONT)
    ax.plot([right_x - node_w * 1.5, right_x + node_w * 1.5],
            [_hdr_y - 0.003, _hdr_y - 0.003],
            color=_EDGE_COL, lw=1.0, alpha=0.5, clip_on=False)

    # -- Title placeholder + subtitle --
    ax.text(0.50, 1.06, ' ', ha='center', va='bottom', fontsize=15,
            transform=ax.transAxes, alpha=0.0)
    ax.text(0.50, -0.015,
            f'Top {sankey_top_features} Consensus Features per Target  \u00b7  '
            'Shared Features Annotated (nT)  \u00b7  '
            'Matched Child CBCL \u2194 Parent ASR DSM-Oriented Scales\n'
            'Ribbon Opacity \u221d Weighted Ensemble Importance  \u00b7  '
            'Ribbon Color = Target Identity  \u00b7  '
            'Circular Features Removed',
            ha='center', va='top', fontsize=_lf[3], fontweight='bold',
            color='#1a1a1a', fontfamily=_TITLE_FONT, linespacing=1.4,
            transform=ax.transAxes)

    plt.tight_layout(rect=[0, 0.015, 0.90, 0.98])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_primary_targets', dpi=600)
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    return fig

# ---
# CHILD DELTAS -- domain-coloured Sankey for Child Only Deltas batch
# ---

_CHILD_DELTAS_COLORS = {
    'delta_depress_D_p':            (0.10, 0.28, 0.62),   # royal blue
    'delta_sbt_core_depression':    (0.16, 0.42, 0.80),   # lighter blue
    'delta_anxdisord_D_p':          (0.72, 0.14, 0.14),   # crimson
    'delta_adhd_D_p':               (0.80, 0.68, 0.12),   # gold
    'delta_social_problems_D_p':    (0.85, 0.55, 0.10),   # dark orange
    'delta_somatic_problems_D_p':   (0.60, 0.45, 0.22),   # peru
    'delta_bad_grades':             (0.44, 0.50, 0.56),   # slate grey
    'delta_family_conflict_ss_k':   (0.62, 0.32, 0.12),   # sienna
}

_CHILD_DELTAS_DISPLAY_ALIASES = {
    'Depression Syndrome (Change Score)':       'delta_depress_D_p',
    'Change in Depression':                     'delta_depress_D_p',
    'Mood/Depression (Change Score)':           'delta_sbt_core_depression',
    'Change in Anxiety':                        'delta_anxdisord_D_p',
    'Anxiety (Change Score)':                   'delta_anxdisord_D_p',
    'Change in ADHD':                           'delta_adhd_D_p',
    'Change in Social Problems':                'delta_social_problems_D_p',
    'Change in Somatic Issues':                 'delta_somatic_problems_D_p',
    'Change in Bad Grades':                     'delta_bad_grades',
    'Change in Fam. Conflict (Y)':              'delta_family_conflict_ss_k',
}

_CHILD_DELTAS_ORDER = [
    'delta_depress_D_p',
    'delta_sbt_core_depression',
    'delta_anxdisord_D_p',
    'delta_adhd_D_p',
    'delta_social_problems_D_p',
    'delta_somatic_problems_D_p',
    'delta_bad_grades',
    'delta_family_conflict_ss_k',
]

_CHILD_DELTAS_EDGE = '#1A2332'
_CHILD_DELTAS_BG   = '#FFFFFF'

def _child_deltas_resolve_color(target_name):
    if target_name in _CHILD_DELTAS_COLORS:
        return _CHILD_DELTAS_COLORS[target_name]
    raw = _CHILD_DELTAS_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_DELTAS_COLORS:
        return _CHILD_DELTAS_COLORS[raw]
    tl = target_name.lower()
    if 'mood' in tl or 'sbt_core' in tl:       return _CHILD_DELTAS_COLORS['delta_sbt_core_depression']
    if 'depress' in tl or 'dep_' in tl:        return _CHILD_DELTAS_COLORS['delta_depress_D_p']
    if 'anxi' in tl:                            return _CHILD_DELTAS_COLORS['delta_anxdisord_D_p']
    if 'adhd' in tl or 'attention' in tl:       return _CHILD_DELTAS_COLORS['delta_adhd_D_p']
    if 'social' in tl:                           return _CHILD_DELTAS_COLORS['delta_social_problems_D_p']
    if 'somatic' in tl:                          return _CHILD_DELTAS_COLORS['delta_somatic_problems_D_p']
    if 'grade' in tl or 'academic' in tl:       return _CHILD_DELTAS_COLORS['delta_bad_grades']
    if 'conflict' in tl or 'family' in tl:      return _CHILD_DELTAS_COLORS['delta_family_conflict_ss_k']
    return (0.5, 0.5, 0.5)

def _child_deltas_sort_targets(targets_list):
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

def _child_deltas_edge_for(target_name):
    rgb = _child_deltas_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _draw_sankey_child_deltas(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_child_deltas_sort_targets,
        color_fn=_child_deltas_resolve_color,
        edge_fn=_child_deltas_edge_for,
        edge_col=_CHILD_DELTAS_EDGE, bg_color=_CHILD_DELTAS_BG,
        save_name='sankey_child_deltas')

# ---
# PARENT DELTAS -- domain-coloured Sankey for Parent Only Deltas batch
# ---

_PARENT_DELTAS_COLORS = {
    # Depression / Mood domain (blue family)
    'delta_parent_depress_D_p':          (0.08, 0.12, 0.42),   # deep navy
    'delta_parent_feels_unloved_p':      (0.22, 0.38, 0.62),   # steel blue
    'delta_parent_sleep_trouble_p':      (0.18, 0.22, 0.52),   # slate indigo
    # Anxiety domain (red family)
    'delta_parent_worries_about_family_p':(0.72, 0.22, 0.18),  # indian red
    'delta_parent_worries_about_future_p':(0.58, 0.12, 0.12),  # firebrick
    # ADHD / Attention domain (purple)
    'delta_parent_concentration_trouble_p':(0.50, 0.28, 0.68), # medium purple
    # Social domain (teal family)
    'delta_parent_not_liked_by_others_p': (0.08, 0.48, 0.48),  # dark cyan
    'delta_parent_bad_relationships_p':   (0.06, 0.42, 0.38),  # teal
    # Family conflict (warm orange)
    'delta_family_conflict_ss_p':         (0.62, 0.32, 0.12),  # sienna
    # SES / Financial (stone)
    'delta_parent_financial_trouble_p':   (0.40, 0.40, 0.40),  # dim grey
    # Somatic (earthy brown)
    'delta_parent_somatic_problems_D_p':  (0.60, 0.45, 0.22),  # peru
}

_PARENT_DELTAS_DISPLAY_ALIASES = {
    'Parent Depression Syndrome (Change Score)':   'delta_parent_depress_D_p',
    'Change in Par. Feels Unloved':                'delta_parent_feels_unloved_p',
    'Change in Par. Sleep':                        'delta_parent_sleep_trouble_p',
    'Change in Par. Fam. Worry':                   'delta_parent_worries_about_family_p',
    'Change in Par. Future Worry':                 'delta_parent_worries_about_future_p',
    'Change in Par. Concentration':                'delta_parent_concentration_trouble_p',
    'Change in Par. Unliked':                      'delta_parent_not_liked_by_others_p',
    'Change in Par. General Relationship Quality': 'delta_parent_bad_relationships_p',
    'Change in Fam. Conflict':                     'delta_family_conflict_ss_p',
    'Change in Par. Finances':                     'delta_parent_financial_trouble_p',
    'Change in Par. Somatic':                      'delta_parent_somatic_problems_D_p',
}

# Display order: depression → anxiety → ADHD → social → family → financial → somatic
_PARENT_DELTAS_ORDER = [
    'delta_parent_depress_D_p',
    'delta_parent_feels_unloved_p',
    'delta_parent_sleep_trouble_p',
    'delta_parent_worries_about_family_p',
    'delta_parent_worries_about_future_p',
    'delta_parent_concentration_trouble_p',
    'delta_parent_not_liked_by_others_p',
    'delta_parent_bad_relationships_p',
    'delta_family_conflict_ss_p',
    'delta_parent_financial_trouble_p',
    'delta_parent_somatic_problems_D_p',
]

_PARENT_DELTAS_EDGE = '#2C3440'
_PARENT_DELTAS_BG   = '#FFFFFF'

def _parent_deltas_resolve_color(target_name):
    if target_name in _PARENT_DELTAS_COLORS:
        return _PARENT_DELTAS_COLORS[target_name]
    raw = _PARENT_DELTAS_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _PARENT_DELTAS_COLORS:
        return _PARENT_DELTAS_COLORS[raw]
    tl = target_name.lower()
    # Substring fallbacks for any display-name variants
    if 'depress' in tl or 'dep_' in tl:            return _PARENT_DELTAS_COLORS['delta_parent_depress_D_p']
    if 'unloved' in tl:                             return _PARENT_DELTAS_COLORS['delta_parent_feels_unloved_p']
    if 'sleep' in tl:                               return _PARENT_DELTAS_COLORS['delta_parent_sleep_trouble_p']
    if 'fam. worry' in tl or 'family_p' in tl:     return _PARENT_DELTAS_COLORS['delta_parent_worries_about_family_p']
    if 'future' in tl or 'worry' in tl:            return _PARENT_DELTAS_COLORS['delta_parent_worries_about_future_p']
    if 'concentrat' in tl or 'attention' in tl:    return _PARENT_DELTAS_COLORS['delta_parent_concentration_trouble_p']
    if 'unliked' in tl or 'not_liked' in tl:       return _PARENT_DELTAS_COLORS['delta_parent_not_liked_by_others_p']
    if 'relationship' in tl:                        return _PARENT_DELTAS_COLORS['delta_parent_bad_relationships_p']
    if 'conflict' in tl:                            return _PARENT_DELTAS_COLORS['delta_family_conflict_ss_p']
    if 'financ' in tl:                              return _PARENT_DELTAS_COLORS['delta_parent_financial_trouble_p']
    if 'somatic' in tl:                             return _PARENT_DELTAS_COLORS['delta_parent_somatic_problems_D_p']
    return (0.5, 0.5, 0.5)

def _parent_deltas_sort_targets(targets_list):
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _r2_for(t):
        entry = _tm.get(t)
        if entry is None:
            entry = _tm.get(_tdn.get(t, ''))
        if entry is None:
            raw = _PARENT_DELTAS_DISPLAY_ALIASES.get(t)
            if raw:
                entry = _tm.get(raw) or _tm.get(_tdn.get(raw, ''))
        return entry[0] if entry is not None else None

    r2_vals = {t: _r2_for(t) for t in targets_list}
    has_metrics = any(v is not None for v in r2_vals.values())

    if has_metrics:
        return sorted(targets_list,
                       key=lambda t: r2_vals.get(t) if r2_vals.get(t) is not None else -999,
                       reverse=True)

    def _rank(t):
        if t in _PARENT_DELTAS_ORDER:
            return _PARENT_DELTAS_ORDER.index(t)
        raw = _PARENT_DELTAS_DISPLAY_ALIASES.get(t)
        if raw and raw in _PARENT_DELTAS_ORDER:
            return _PARENT_DELTAS_ORDER.index(raw)
        tl = t.lower()
        if 'depress' in tl:       return 0
        if 'unloved' in tl:       return 1
        if 'sleep' in tl:         return 2
        if 'fam. worry' in tl:    return 3
        if 'future' in tl:        return 4
        if 'concentrat' in tl:    return 5
        if 'unliked' in tl:       return 6
        if 'relationship' in tl:  return 7
        if 'conflict' in tl:      return 8
        if 'financ' in tl:        return 9
        if 'somatic' in tl:       return 10
        return 99
    return sorted(targets_list, key=_rank)

def _parent_deltas_edge_for(target_name):
    rgb = _parent_deltas_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _draw_sankey_parent_deltas(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_domain_style(
        df_sankey, sankey_top_features, save_dir=save_dir,
        sort_fn=_parent_deltas_sort_targets,
        color_fn=_parent_deltas_resolve_color,
        edge_fn=_parent_deltas_edge_for,
        edge_col=_PARENT_DELTAS_EDGE, bg_color=_PARENT_DELTAS_BG,
        save_name='sankey_parent_deltas')

# ---
# PARENT DEVELOPMENTAL SWEEP SANKEY
# Parent Depression Syndrome (blue), Parent Depressed (red),
# Parent ADHD (gold), Parent Happiness (pink) across T0->T4
# ---

_PARENT_DEV_SWEEP_COLORS = {
    # -- Parent Depression Syndrome -- blue gradient (T0 lightest → T4 darkest) --
    'parent_depress_D_p__tp0':    (0.62, 0.76, 0.92),   # pale sky
    'parent_depress_D_p__tp1':    (0.40, 0.58, 0.82),   # soft blue
    'parent_depress_D_p__tp2':    (0.22, 0.42, 0.72),   # mid blue
    'parent_depress_D_p__tp3':    (0.12, 0.28, 0.58),   # rich blue
    'parent_depress_D_p__tp4':    (0.04, 0.16, 0.42),   # deep navy

    # -- Parent Depressed (Item) -- red gradient (T0 lightest → T4 darkest) --
    'parent_depressed_p__tp0':    (0.94, 0.68, 0.66),   # pale rose
    'parent_depressed_p__tp1':    (0.86, 0.46, 0.42),   # soft red
    'parent_depressed_p__tp2':    (0.76, 0.26, 0.24),   # mid red
    'parent_depressed_p__tp3':    (0.60, 0.14, 0.14),   # rich red
    'parent_depressed_p__tp4':    (0.44, 0.06, 0.08),   # deep crimson

    # -- Parent ADHD -- yellow/gold gradient (T0 lightest → T4 darkest) --
    'parent_adhd_D_p__tp0':       (0.96, 0.90, 0.55),   # pale gold
    'parent_adhd_D_p__tp1':       (0.92, 0.82, 0.35),   # soft gold
    'parent_adhd_D_p__tp2':       (0.82, 0.72, 0.22),   # mid gold
    'parent_adhd_D_p__tp3':       (0.68, 0.58, 0.12),   # rich gold
    'parent_adhd_D_p__tp4':       (0.52, 0.42, 0.04),   # deep amber

    # -- Parent Happiness -- pink gradient (T0 lightest → T4 darkest) --
    'parent_happy_person_p__tp0': (0.98, 0.72, 0.80),   # pale blush
    'parent_happy_person_p__tp1': (0.94, 0.52, 0.64),   # soft pink
    'parent_happy_person_p__tp2': (0.90, 0.35, 0.55),   # mid rose
    'parent_happy_person_p__tp3': (0.78, 0.22, 0.42),   # rich magenta
    'parent_happy_person_p__tp4': (0.62, 0.10, 0.30),   # deep berry
}

_PARENT_DEV_SWEEP_ORDER = [
    # T4 (latest wave) at top → T0 (baseline) at bottom
    # Depression Syndrome
    'parent_depress_D_p__tp4', 'parent_depress_D_p__tp3',
    'parent_depress_D_p__tp2', 'parent_depress_D_p__tp1',
    'parent_depress_D_p__tp0',
    # Depressed Item
    'parent_depressed_p__tp4', 'parent_depressed_p__tp3',
    'parent_depressed_p__tp2', 'parent_depressed_p__tp1',
    'parent_depressed_p__tp0',
    # ADHD
    'parent_adhd_D_p__tp4', 'parent_adhd_D_p__tp3',
    'parent_adhd_D_p__tp2', 'parent_adhd_D_p__tp1',
    'parent_adhd_D_p__tp0',
    # Happiness
    'parent_happy_person_p__tp4', 'parent_happy_person_p__tp3',
    'parent_happy_person_p__tp2', 'parent_happy_person_p__tp1',
    'parent_happy_person_p__tp0',
]

# Parents don't have a single age -- use assessment wave labels
# Base constants used downstream: _TP_AGE_MAP (referenced by
# _dev_sweep_resolve_color and _extract_tp_from_name), _DEV_SWEEP_BG
# (referenced by _draw_sankey_developmental_sweep), and
# _FEATURE_DOMAIN_ACCENTS (referenced by helper functions).
# ABCD annual follow-up ages for each timepoint:
_TP_AGE_MAP = {
    '0': '9-10y',   # Baseline (T0)
    '1': '10-11y',  # 1-Year (T1)
    '2': '11-12y',  # 2-Year (T2)
    '3': '12-13y',  # 3-Year (T3)
    '4': '13-14y',  # 4-Year (T4)
}

# Background color for developmental-sweep Sankey (matches other Sankey BGs)
_DEV_SWEEP_BG = '#F8F7F3'   # soft off-white / cream

# Feature domain accent colors (Tol-muted-inspired, colorblind-safe)
# Used by _infer_feature_domain() helper. Keys enumerated from the function body.
_FEATURE_DOMAIN_ACCENTS = {
    'Change Scores':      '#44AA99',  # Tol muted teal (longitudinal delta)
    'Parent Report':      '#AA4499',  # Tol muted purple (external observer)
    'Family/Social':      '#CC6677',  # Tol muted rose (relational warmth)
    'Cognitive/Academic': '#332288',  # Tol muted indigo (academic gravitas)
    'Physical/Medical':   '#882255',  # Tol muted wine (body)
    'Demographic':        '#999933',  # Tol muted olive (neutral context)
    'Youth Behavioral':   '#117733',  # Tol muted green (child self-report default)
}

_PARENT_TP_WAVE_MAP = {
    '0': 'Baseline',
    '1': 'Year 1',
    '2': 'Year 2',
    '3': 'Year 3',
    '4': 'Year 4',
}

_PARENT_DEV_SWEEP_BG = '#FFFFFF'

# -- Edge colours per domain --
_PARENT_DEV_EDGE = {
    'depression_syndrome': '#0A2647', # navy
    'depressed_item':      '#4A0404', # dark red
    'adhd':                '#3B2E00', # dark brown-gold
    'happiness':           '#4A0028', # dark berry
}

def _parent_dev_sweep_feature_color(feature, df_sankey, feat_target_count, targets_list):
    """Feature node colour: single-target inherits target colour;
    multi-target goes from dark blueish-grey to near-black with overlap."""
    n_tgt = feat_target_count.get(feature, 1)
    rows = df_sankey[df_sankey['Feature'] == feature]
    if rows.empty:
        return (0.5, 0.5, 0.5)
    if n_tgt == 1:
        dom_t = rows.loc[rows['Importance'].idxmax(), 'Target']
        return _parent_dev_sweep_resolve_color(dom_t)
    # Multi-target: light steel → hard black across 2T-5T+
    _t = min((n_tgt - 2) / 8.0, 1.0)   # gentler ramp for 20 possible targets
    return (0.62 - 0.58 * _t,
            0.64 - 0.58 * _t,
            0.70 - 0.60 * _t)

def _dev_sweep_sort_targets(targets_list):
    """Sort dev-sweep targets TP-first (T4 top -> T0 bottom), then by target-order
    within each TP. TP-first ordering keeps each timepoint block vertically contiguous,
    which is required for the left-margin age labels to align properly when the batch
    contains more than one target (e.g., social_problems_D_p + not_liked_p).
    """
    order_map = {t: i for i, t in enumerate(_DEV_SWEEP_ORDER)}
    for raw_key, idx_val in list(order_map.items()):
        display = _get_tdn(raw_key)
        if display not in order_map:
            order_map[display] = idx_val

    def _sort_key(t):
        tp = _extract_tp_from_name(t)
        # T4 (tp=4) must come first -> use (-tp) so negatives sort ascending to top.
        # tp=-1 (unrecognized) goes to bottom (sorts last since -(-1) = 1).
        tp_priority = -tp if tp >= 0 else 1
        target_rank = order_map.get(t, 999)
        return (tp_priority, target_rank)

    return sorted(targets_list, key=_sort_key)

def _dev_sweep_feature_color(feature, df_sankey, feat_target_count, targets_list):
    """Feature node colour: single-target inherits target colour;
    multi-target goes from dark blueish-grey to near-black with overlap."""
    n_tgt = feat_target_count.get(feature, 1)
    rows = df_sankey[df_sankey['Feature'] == feature]
    if rows.empty:
        return (0.5, 0.5, 0.5)
    if n_tgt == 1:
        dom_t = rows.loc[rows['Importance'].idxmax(), 'Target']
        return _dev_sweep_resolve_color(dom_t)
    # Multi-target: light steel → hard black across 2T-5T+
    _t = min((n_tgt - 2) / 3.0, 1.0)   # 0 at 2T, 0.33 at 3T, 0.67 at 4T, 1.0 at 5T+
    return (0.62 - 0.58 * _t,           # R: 0.62 → 0.04
            0.64 - 0.58 * _t,           # G: 0.64 → 0.06
            0.70 - 0.60 * _t)           # B: 0.70 → 0.10

def _draw_sankey(df_sankey, sankey_top_features, save_dir=None, style='heatflow_fire'):
    """Dispatch to the appropriate Sankey style function."""
    if style in HEATFLOW_THEMES:
        return _draw_sankey_heatflow_enhanced(df_sankey, sankey_top_features,
                                               save_dir=save_dir, theme_key=style)
    _dispatchers = {
        'pfactor_spectra':          _draw_sankey_pfactor_spectra,
        'child_pfactor':            _draw_sankey_child_pfactor,
        'parent_pfactor':           _draw_sankey_parent_pfactor,
        'child_suicide':            _draw_sankey_child_suicide,
        'parent_suicide':           _draw_sankey_parent_suicide,
        'child_depression':         _draw_sankey_child_dep_worsening,
        'parent_depression':        _draw_sankey_parent_dep_worsening,
        'parent_deltas':            _draw_sankey_parent_deltas,
        'child_deltas':             _draw_sankey_child_deltas,
        'child_academic_cognitive': _draw_sankey_child_academic_cognitive,
        'child_adhd_impcomp':       _draw_sankey_child_adhd_impcomp,
        'parent_social_impcomp':    _draw_sankey_parent_social_impcomp,
        'child_alts':               _draw_sankey_child_alts,
        'parent_alts':              _draw_sankey_parent_alts,
        'sbt_female':               _draw_sankey_sbt_female,
        'sbt_long':                 _draw_sankey_sbt_long,
        'primary_targets':          _draw_sankey_primary_targets,
        'sbt_core_triad':           _draw_sankey_sbt_core_triad,
        'sbt_core_triad_female':    _draw_sankey_sbt_core_triad_female,
        'sbt_core_triad_male':      _draw_sankey_sbt_core_triad_male,
        'child_cogtask_t0':         _draw_sankey_child_cogtask_t0,





        'developmental_sweep':      _draw_sankey_developmental_sweep,
        'parent_developmental_sweep': _draw_sankey_parent_developmental_sweep,
    }
    _fn = _dispatchers.get(style)
    if _fn is None:
        print(f"  Warning:  Unknown sankey_style '{style}'. Falling back to 'heatflow_fire'.")
        return _draw_sankey_heatflow_enhanced(df_sankey, sankey_top_features,
                                               save_dir=save_dir, theme_key='heatflow_fire')
    return _fn(df_sankey, sankey_top_features, save_dir=save_dir)

def _run_all_styles(df_sankey, sankey_top_features, save_dir=None):
    """Run all available Sankey styles and save each one."""
    all_styles = list(HEATFLOW_THEMES.keys()) + [
        'pfactor_spectra', 'child_pfactor', 'parent_pfactor', 'child_suicide', 'parent_suicide', 'child_depression', 'parent_depression',
        'child_deltas', 'parent_deltas',
        'child_academic_cognitive', 'child_adhd_impcomp', 'parent_social_impcomp',
        'child_alts', 'parent_alts',
        'sbt_female', 'sbt_long', 'primary_targets', 'sbt_core_triad',
        'sbt_core_triad_female', 'sbt_core_triad_male',
        'child_cogtask_t0',
    ]

    print("\n" + "=" * 80)
    print("  RUNNING ALL SANKEY STYLES")
    print("=" * 80)
    print(f"  Total styles to render: {len(all_styles)}")
    print("=" * 80 + "\n")

    results = []
    for i, style in enumerate(all_styles, 1):
        print(f"\n{'-' * 60}")
        print(f"  [{i}/{len(all_styles)}] Rendering: {style}")
        print(f"{'-' * 60}")
        try:
            fig = _draw_sankey(df_sankey, sankey_top_features, save_dir=save_dir, style=style)
            results.append({'style': style, 'status': 'success', 'fig': fig})
            plt.close(fig)
        except Exception as e:
            print(f"  Error rendering {style}: {e}")
            results.append({'style': style, 'status': 'error', 'error': str(e)})

    print("\n" + "=" * 80)
    print("  ALL STYLES COMPLETE")
    print("=" * 80)
    success_count = sum(1 for r in results if r['status'] == 'success')
    print(f"  Successful: {success_count}/{len(all_styles)}")
    return results

# ---
# Color and dispatch tables for child/parent pfactor and suicide sweep styles.
# Late-bound after the heatflow definitions above so the developmental sweep
# helpers can use these palettes.
# ---

# Base _DEV_SWEEP_COLORS dict for the child developmental sweep. Seeded
# with F4 gradients (sbt_core_depression, anxdisord_D_p) plus int/ext/
# suicidal gradients and child social-problem gradients.
_DEV_SWEEP_COLORS = {
    # Child Core Depression (F4) -- royal-blue gradient (T0 lightest → T4 darkest)
    'sbt_core_depression__tp0':  (0.66, 0.78, 0.94),   # pale sky
    'sbt_core_depression__tp1':  (0.44, 0.60, 0.84),   # soft royal blue
    'sbt_core_depression__tp2':  (0.24, 0.42, 0.74),   # mid royal blue
    'sbt_core_depression__tp3':  (0.12, 0.26, 0.60),   # rich royal blue
    'sbt_core_depression__tp4':  (0.04, 0.12, 0.44),   # deep navy
    # Child Anxiety Disorder (F4) -- crimson gradient (T0 lightest → T4 darkest)
    'anxdisord_D_p__tp0':        (0.96, 0.70, 0.68),   # pale coral
    'anxdisord_D_p__tp1':        (0.90, 0.48, 0.46),   # soft crimson
    'anxdisord_D_p__tp2':        (0.80, 0.26, 0.26),   # mid crimson
    'anxdisord_D_p__tp3':        (0.64, 0.12, 0.14),   # rich crimson
    'anxdisord_D_p__tp4':        (0.46, 0.04, 0.08),   # deep garnet
    # Child Internalizing -- purple gradient (T0 lightest → T4 darkest)
    'internal_D_p__tp0':         (0.80, 0.68, 0.96),
    'internal_D_p__tp1':         (0.68, 0.48, 0.90),
    'internal_D_p__tp2':         (0.55, 0.36, 0.85),
    'internal_D_p__tp3':         (0.42, 0.22, 0.72),
    'internal_D_p__tp4':         (0.30, 0.10, 0.58),
    # Child Externalizing -- green gradient
    'external_D_p__tp0':         (0.72, 0.90, 0.62),
    'external_D_p__tp1':         (0.58, 0.82, 0.44),
    'external_D_p__tp2':         (0.45, 0.72, 0.30),
    'external_D_p__tp3':         (0.32, 0.60, 0.18),
    'external_D_p__tp4':         (0.20, 0.46, 0.08),
    # Child Suicidal Ideation -- very dark blue gradient
    'suicidal_p__tp0':           (0.48, 0.48, 0.66),
    'suicidal_p__tp1':           (0.34, 0.34, 0.54),
    'suicidal_p__tp2':           (0.22, 0.22, 0.42),
    'suicidal_p__tp3':           (0.12, 0.12, 0.32),
    'suicidal_p__tp4':           (0.04, 0.04, 0.24),
    # Child Social Problems (DSM domain) -- amber/orange gradient
    'social_problems_D_p__tp0': (1.00, 0.91, 0.74),   # pale peach
    'social_problems_D_p__tp1': (0.99, 0.76, 0.40),   # soft amber
    'social_problems_D_p__tp2': (0.93, 0.55, 0.12),   # warm orange
    'social_problems_D_p__tp3': (0.72, 0.37, 0.03),   # burnt orange
    'social_problems_D_p__tp4': (0.45, 0.21, 0.02),   # dark burnt
    # Child Not-Liked (CBCL item) -- chocolate gradient (warm brown-orange,
    # paired with but distinguishable from social_problems_D_p amber/orange)
    'not_liked_p__tp0': (0.74, 0.94, 0.94),          # pale cyan
    'not_liked_p__tp1': (0.40, 0.82, 0.88),          # light cyan
    'not_liked_p__tp2': (0.10, 0.58, 0.70),          # medium teal-blue
    'not_liked_p__tp3': (0.03, 0.35, 0.48),          # deep teal-blue
    'not_liked_p__tp4': (0.01, 0.15, 0.25),          # dark teal
}

# Base _DEV_SWEEP_ORDER list for the child developmental sweep. T4 (latest
# wave) at top -> T0 (baseline) at bottom within each target block. Order:
# F4 primary targets first, then int/ext/sui, then child social.
_DEV_SWEEP_ORDER = [
    # F4 primary: Core Depression + Anxiety Disorder
    'sbt_core_depression__tp4', 'sbt_core_depression__tp3',
    'sbt_core_depression__tp2', 'sbt_core_depression__tp1',
    'sbt_core_depression__tp0',
    'anxdisord_D_p__tp4', 'anxdisord_D_p__tp3',
    'anxdisord_D_p__tp2', 'anxdisord_D_p__tp1',
    'anxdisord_D_p__tp0',
    # Broadband psychopathology
    'internal_D_p__tp4', 'internal_D_p__tp3', 'internal_D_p__tp2', 'internal_D_p__tp1', 'internal_D_p__tp0',
    'external_D_p__tp4', 'external_D_p__tp3', 'external_D_p__tp2', 'external_D_p__tp1', 'external_D_p__tp0',
    'suicidal_p__tp4', 'suicidal_p__tp3', 'suicidal_p__tp2', 'suicidal_p__tp1', 'suicidal_p__tp0',
    # Child Social Problems (DSM domain + CBCL item)
    'social_problems_D_p__tp4', 'social_problems_D_p__tp3',
    'social_problems_D_p__tp2', 'social_problems_D_p__tp1',
    'social_problems_D_p__tp0',
    'not_liked_p__tp4', 'not_liked_p__tp3', 'not_liked_p__tp2',
    'not_liked_p__tp1', 'not_liked_p__tp0',
]

# Domain-aware _dev_sweep_resolve_color.
def _dev_sweep_resolve_color(target_name):
    if target_name in _DEV_SWEEP_COLORS:
        return _DEV_SWEEP_COLORS[target_name]
    tl = target_name.lower()
    def _child_prefix(tl):
        if 'internal' in tl:   return 'internal_D_p'
        if 'external' in tl:   return 'external_D_p'
        if 'suicid' in tl:    return 'suicidal_p'
        if 'anxi' in tl:      return 'anxdisord_D_p'
        if 'depress' in tl or 'mood' in tl: return 'sbt_core_depression'
        return None
    prefix = _child_prefix(tl)
    if prefix is None:
        return (0.5, 0.5, 0.5)
    for tp in range(5):
        age = _TP_AGE_MAP[str(tp)]
        if f'(t{tp})' in tl or age.lower() in tl:
            return _DEV_SWEEP_COLORS.get(f'{prefix}__tp{tp}', (0.5, 0.5, 0.5))
    return _DEV_SWEEP_COLORS.get(f'{prefix}__tp2', (0.5, 0.5, 0.5))

# Parent int/ext/sui gradients added to _PARENT_DEV_SWEEP_COLORS.
_PARENT_DEV_SWEEP_COLORS.update({
    # Parent Internalizing -- purple gradient
    'parent_internal_D_p__tp0':   (0.78, 0.60, 0.92),
    'parent_internal_D_p__tp1':   (0.65, 0.40, 0.82),
    'parent_internal_D_p__tp2':   (0.48, 0.20, 0.68),
    'parent_internal_D_p__tp3':   (0.36, 0.10, 0.52),
    'parent_internal_D_p__tp4':   (0.24, 0.04, 0.38),
    # Parent Externalizing -- green gradient
    'parent_external_D_p__tp0':   (0.62, 0.84, 0.70),
    'parent_external_D_p__tp1':   (0.40, 0.72, 0.55),
    'parent_external_D_p__tp2':   (0.22, 0.58, 0.42),
    'parent_external_D_p__tp3':   (0.10, 0.44, 0.30),
    'parent_external_D_p__tp4':   (0.04, 0.30, 0.20),
    # Parent Suicidal Thoughts -- very dark blue gradient
    'parent_suicidal_thoughts_p__tp0': (0.42, 0.42, 0.62),
    'parent_suicidal_thoughts_p__tp1': (0.30, 0.30, 0.50),
    'parent_suicidal_thoughts_p__tp2': (0.20, 0.20, 0.40),
    'parent_suicidal_thoughts_p__tp3': (0.12, 0.12, 0.32),
    'parent_suicidal_thoughts_p__tp4': (0.04, 0.04, 0.24),
})

# Extension of _PARENT_DEV_SWEEP_ORDER.
_PARENT_DEV_SWEEP_ORDER.extend([
    'parent_internal_D_p__tp4', 'parent_internal_D_p__tp3', 'parent_internal_D_p__tp2', 'parent_internal_D_p__tp1', 'parent_internal_D_p__tp0',
    'parent_external_D_p__tp4', 'parent_external_D_p__tp3', 'parent_external_D_p__tp2', 'parent_external_D_p__tp1', 'parent_external_D_p__tp0',
    'parent_suicidal_thoughts_p__tp4', 'parent_suicidal_thoughts_p__tp3', 'parent_suicidal_thoughts_p__tp2', 'parent_suicidal_thoughts_p__tp1', 'parent_suicidal_thoughts_p__tp0',
])

# Base definition of _parent_dev_sweep_domain. Required so the override
# chain below has a base function to wrap. Seeds the four core parent
# developmental-sweep domains (depression_syndrome, depressed_item, adhd,
# happiness); subsequent overrides add relationships, unliked,
# internalizing, externalizing, and suicidal.
def _parent_dev_sweep_domain(target_name):
    """Base: map a parent developmental-sweep target to its domain key.

    Successive overrides wrap this function and add additional domains
    (relationships, then unliked).
    """
    tl = target_name.lower()
    # Check __tpN raw-key prefix first
    if '__tp' in target_name:
        prefix = target_name.split('__tp')[0]
        if 'parent_depress_D_p' == prefix:       return 'depression_syndrome'
        if 'parent_depressed_p' == prefix:       return 'depressed_item'
        if 'parent_adhd_D_p' == prefix:          return 'adhd'
        if 'parent_happy_person_p' == prefix:    return 'happiness'
    # Keyword match on display name / raw name fallback
    if 'happy' in tl or 'happiness' in tl:
        return 'happiness'
    if 'adhd' in tl or 'attention' in tl:
        return 'adhd'
    if 'depressed_p' in tl or 'depressed (item)' in tl or 'depressed item' in tl:
        return 'depressed_item'
    if 'depress' in tl or 'mood' in tl or 'syndrome' in tl:
        return 'depression_syndrome'
    # Safe default
    return 'depression_syndrome'

# Extend _parent_dev_sweep_domain to include int/ext/sui domains.
_orig_parent_dev_sweep_domain = _parent_dev_sweep_domain

def _extract_tp_from_name(target_name):
    """Extract integer timepoint from a __tpN raw key or display name."""
    if '__tp' in target_name:
        try:
            return int(target_name.split('__tp')[1])
        except ValueError:
            pass
    tl = target_name.lower()
    for tp in range(5):
        age = _TP_AGE_MAP.get(str(tp), '')
        wave = _PARENT_TP_WAVE_MAP.get(str(tp), '')
        if (f'(t{tp})' in tl
                or f'({age.lower()})' in tl
                or f'({wave.lower()})' in tl):
            return tp
    return -1

_CHILD_DOMAIN_CLEAN_LABELS = {
    'sbt_core_depression':  'Mood / Depression',
    'anxdisord_D_p':        'Anxiety Disorder',
    'internal_D_p':         'Internalizing',
    'external_D_p':         'Externalizing',
    'suicidal_p':           'Suicidal Ideation',
    # Child Social Problems sweep targets:
    'social_problems_D_p':  'Social Problems',
    'not_liked_p':          'Not Liked by Others',
}

_PARENT_DOMAIN_CLEAN_LABELS = {
    'parent_depress_D_p':          'Depression Syndrome',
    'parent_depressed_p':          'Depressed (Item)',
    'parent_adhd_D_p':             'ADHD',
    'parent_happy_person_p':       'Happiness',
    'parent_internal_D_p':         'Internalizing',
    'parent_external_D_p':         'Externalizing',
    'parent_suicidal_thoughts_p':  'Suicidal Thoughts',
}

def _clean_domain_label(target_name, domain_labels):
    """Return a short domain-only display name (no timepoint).
    Falls back to _get_tdn with timepoint stripped."""
    # Check raw key base
    if '__tp' in target_name:
        base = target_name.split('__tp')[0]
        if base in domain_labels:
            return domain_labels[base]
    # Check if the full name matches a domain label value already
    for raw, label in domain_labels.items():
        if target_name == label:
            return label
    # Strip parenthetical suffix from display name
    disp = _get_tdn(target_name)
    import re
    disp_clean = re.sub(r'\s*\(T\d\)\s*$', '', disp)
    disp_clean = re.sub(r'\s*\(~\d+y\)\s*$', '', disp_clean)
    disp_clean = re.sub(r'\s*\((Baseline|Year \d)\)\s*$', '', disp_clean)
    disp_clean = disp_clean.replace(' (Primary Features)', '')
    disp_clean = disp_clean.replace(' (Syndrome)', '')
    disp_clean = disp_clean.replace(' (Item)', '')
    disp_clean = disp_clean.strip()
    # After cleanup, the stripped name may match a raw-key entry in
    # domain_labels (e.g., "not_liked_p" -> "Not Liked by Others"). Perform
    # a final lookup so targets without TARGET_DISPLAY_NAMES entries still
    # resolve to their clean domain label.
    if disp_clean in domain_labels:
        return domain_labels[disp_clean]
    return disp_clean

# ---
# 4. Domain-based shape differentiation
# ---

def _domain_boxstyle(target_name):
    """Return matplotlib boxstyle string based on domain.

    - Broadband spectra (int / ext):     pill  → round,pad=0.018
    - Specific syndromes (dep / anx):    rect  → round,pad=0.006
    - Suicidal ideation:                 mid   → round,pad=0.012
    - Everything else (adhd / happiness): rect
    """
    tl = target_name.lower()
    if 'internal' in tl or 'external' in tl:
        return 'round,pad=0.018'      # pill / capsule
    if 'suicid' in tl:
        return 'round,pad=0.012'      # intermediate rounding
    return 'round,pad=0.006'          # sharp rectangle

# ---
# 5. Timepoint-block dividers + margin age annotations (shared helper)
# ---

def _draw_tp_dividers_and_labels(ax, targets_list, target_positions,
                                  left_x, target_node_w,
                                  tp_label_map, bg_color,
                                  label_fontsize=18, divider_color='#B0BEC5',
                                  label_color='#546E7F'):
    """Draw dashed dividers between timepoint blocks and place age/wave
    annotations in the left margin.

    Parameters
    tp_label_map : dict  {int_tp: str_label}   e.g. {4: '~13 y', 3: '~12 y', …}
    """

    # -- Group targets by timepoint --
    tp_blocks = {}          # tp_int -> [target_name, ...]
    for t in targets_list:
        tp = _extract_tp_from_name(t)
        tp_blocks.setdefault(tp, []).append(t)

    tp_keys = sorted(tp_blocks.keys(), reverse=True)   # T4 first (top)

    # -- Divider lines between consecutive TP blocks --
    _div_hw = target_node_w * 3.5     # half-width of dashed line
    for i in range(len(tp_keys) - 1):
        tp_above = tp_keys[i]
        tp_below = tp_keys[i + 1]
        last_above  = tp_blocks[tp_above][-1]
        first_below = tp_blocks[tp_below][0]
        y_div = (target_positions[last_above]['y_bot']
                 + target_positions[first_below]['y_top']) / 2
        ax.plot([left_x - _div_hw, left_x + _div_hw],
                [y_div, y_div],
                color=divider_color, lw=0.9, ls='--', alpha=0.55, zorder=1)

    # -- Age / wave labels centred on each TP block --
    _margin_x = left_x - target_node_w - 0.12   # well to the left of labels
    for tp in tp_keys:
        block = tp_blocks[tp]
        y_centres = [target_positions[t]['y_centre'] for t in block
                     if t in target_positions]
        if not y_centres:
            continue
        y_mid = (max(y_centres) + min(y_centres)) / 2
        label = tp_label_map.get(tp, f'T{tp}')
        txt = ax.text(_margin_x, y_mid, label,
                      ha='center', va='center',
                      fontsize=label_fontsize, fontweight='bold',
                      color=label_color, fontfamily=_TITLE_FONT,
                      rotation=0, zorder=6, clip_on=False)
        txt.set_path_effects([
            path_effects.withStroke(linewidth=3.5, foreground=bg_color)])

# ---
# Developmental sweep Sankey (child targets)
# ---

def _draw_sankey_developmental_sweep(df_sankey, sankey_top_features, save_dir=None):
    """Developmental Sweep Sankey -- multi-domain across T0-T4.

    Implements:
      1. Timepoint-first ordering  (T4 block at top → T0 at bottom)
      2. Port-gap 0.003
      3. Saturation + luminance ramp
      4. Domain-based target node shape
      5. Age annotations in left margin + dashed TP dividers
      6. Figure ×1.5 + 1200 DPI
    """

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _dev_sweep_sort_targets(_cd['targets_list'])
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: _dev_sweep_resolve_color(t) for t in targets_list}
    _BG = _DEV_SWEEP_BG

    # -- Layout constants ---
    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x    = _LEFT_X
    right_x   = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028
    _target_node_w = 0.036

    # -- 6. Figure scaling ×1.5 ---
    _, _base_h = _wide_figure_dims(n_targets, n_features)
    fig_w = 15 * 1.5
    fig_h = max(_base_h * 1.3, n_features * 0.52, 22) * 1.5
    _spread_t = 5.5
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    _target_fs = 17

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    # -- Uniform target node heights --
    _RECT_H = 0.16
    for t in targets_list:
        if t in target_positions:
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c, 'y_top': _c + _RECT_H / 2,
                'y_bot': _c - _RECT_H / 2, 'h': _RECT_H}

    # -- Uniform feature node heights --
    _FEAT_H = 0.056
    for f_key in features_list:
        if f_key in feature_positions:
            _c = feature_positions[f_key]['y_centre']
            feature_positions[f_key] = {
                'y_centre': _c, 'y_top': _c + _FEAT_H / 2,
                'y_bot': _c - _FEAT_H / 2, 'h': _FEAT_H}

    # -- Figure --
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.28
    ax.set_xlim(-0.28, right_x + 0.22)       # extra left margin for age labels
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    # -- 5. Timepoint dividers + age annotations ---
    _child_tp_labels = {
        4: '~13 y', 3: '~12 y', 2: '~11 y', 1: '~10 y', 0: '~9 y',
    }
    _draw_tp_dividers_and_labels(
        ax, targets_list, target_positions,
        left_x, _target_node_w,
        tp_label_map=_child_tp_labels,
        bg_color=_BG,
        label_fontsize=18,
        divider_color='#B0BEC5',
        label_color='#546E7F')

# ---
    # RIBBONS -- 2. Port-gap = 0.003 &  3. Saturation + luminance ramp
# ---
    _PORT_GAP = 0.003

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        # 3. Saturation + luminance ramp (confirmed present)
        _desat = 0.55 + 0.45 * rel
        _white = (0.88, 0.90, 0.94)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.003, _yr_b - 0.003,
                color_rgba=_rgb + (0.08,), zorder=1,
                glow_layers=_glow_n)

        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

# ---
    # TARGET NODES -- 4. Shape differentiation &  5. Clean domain labels
# ---

    for t, pos in target_positions.items():
        rgb = _tc.get(t, (0.5, 0.5, 0.5))

        # Domain-aware edge colour
        tl = t.lower()
        if 'internal' in tl:     _edge_col = '#2A0845'
        elif 'external' in tl:   _edge_col = '#0A3320'
        elif 'suicid' in tl:     _edge_col = '#060630'
        elif 'anxi' in tl:       _edge_col = '#4A0404'
        else:                    _edge_col = '#0A2647'

        # 4. Domain-based boxstyle
        _bstyle = _domain_boxstyle(t)

        # Shadow
        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.004, pos['y_bot'] - 0.003),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        # Node
        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=rgb, edgecolor=_edge_col,
            linewidth=1.8, zorder=3))

        # 5. Clean domain-only label (age is in the margin now)
        _domain_label = _clean_domain_label(t, _CHILD_DOMAIN_CLEAN_LABELS)

        _metric_val = _target_metrics.get(t) or _target_metrics.get(_get_tdn(t))
        _label_x = left_x - _target_node_w - 0.04

        if _metric_val is not None:
            _v, _l = _metric_val
            txt = ax.text(_label_x, pos['y_centre'] + 0.025,
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.028,
                           f'{_l} = {_v:.3f}',
                           ha='right', va='center',
                           fontsize=_target_fs - 3, fontweight='bold',
                           color=_edge_col, fontfamily=_TITLE_FONT,
                           zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])

# ---
    # FEATURE NODES -- multi-domain edge logic
# ---

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count.get(f, 1)
        _fc = _dev_sweep_feature_color(f, df_sankey, feat_target_count, targets_list)

        _f_rows = df_sankey[df_sankey['Feature'] == f]
        _domain_counts = {}
        for _ft in _f_rows['Target']:
            _ftl = _ft.lower()
            if 'internal' in _ftl:                         _domain_counts['int'] = _domain_counts.get('int', 0) + 1
            elif 'external' in _ftl:                       _domain_counts['ext'] = _domain_counts.get('ext', 0) + 1
            elif 'suicid' in _ftl:                         _domain_counts['sui'] = _domain_counts.get('sui', 0) + 1
            elif 'depress' in _ftl or 'mood' in _ftl:     _domain_counts['dep'] = _domain_counts.get('dep', 0) + 1
            elif 'anxi' in _ftl:                           _domain_counts['anx'] = _domain_counts.get('anx', 0) + 1
        _dom_winner = max(_domain_counts, key=_domain_counts.get) if _domain_counts else 'dep'
        _edge_map = {
            'dep': '#091F38', 'anx': '#3A0808',
            'int': '#2A0845', 'ext': '#0A3320', 'sui': '#060630',
        }
        _feat_edge = _edge_map.get(_dom_winner, '#091F38')

        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f]
            dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                     if not dom_rows.empty else targets_list[0])
            glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))
        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc, edgecolor=_feat_edge,
            linewidth=0.7, zorder=3))

        _disp_name = DISPLAY_NAMES.get(f, f) if 'DISPLAY_NAMES' in dir() else f
        badge = f' ({n_tgt}T)' if n_tgt >= 2 else ''
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                      f'{_disp_name}{badge}',
                      ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                      color='#1a1a1a', fontfamily=_TITLE_FONT,
                      zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(
            linewidth=2.5, foreground=_BG)])

    # -- Column headers --
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    _hdr_dep_col = '#0A2647'
    _hdr_feat_col = '#1B5E20'
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_dep_col,
            fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_feat_col,
            fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_dep_col, lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_feat_col, lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    plt.tight_layout(rect=[0.01, 0.01, 0.99, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    # 6. Save at 1200 DPI
    saved = _safe_save_fig(fig, save_dir, 'sankey_developmental_sweep',
                           dpi=1200, formats=('png', 'pdf'))
    plt.show()
    for p in saved:
        print(f"  Saved: {p}")
    return fig

# ---
# Parent developmental sweep wrappers
# ---

def _draw_sankey_child_pfactor(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_developmental_sweep(df_sankey, sankey_top_features, save_dir=save_dir)

def _draw_sankey_child_suicide(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_developmental_sweep(df_sankey, sankey_top_features, save_dir=save_dir)

def _parent_dev_sweep_domain(target_name):
    """Return the domain key -- extended with relationships."""
    tl = target_name.lower()
    if 'relationship' in tl or 'bad_rel' in tl:
        return 'relationships'
    return _orig_parent_dev_sweep_domain_v2(target_name)

_PARENT_DEV_EDGE.update({
    'relationships': '#3D1800',
})

_PARENT_DOMAIN_CLEAN_LABELS['parent_bad_relationships_p'] = 'Interpersonal Difficulties'

def _parent_dev_sweep_resolve_color(target_name):
    """Resolve colour -- extended for relationships (orange)."""
    if target_name in _PARENT_DEV_SWEEP_COLORS:
        return _PARENT_DEV_SWEEP_COLORS[target_name]
    tl = target_name.lower()
    dom = _parent_dev_sweep_domain(target_name)
    _prefix_map = {
        'depression_syndrome': 'parent_depress_D_p',
        'depressed_item':      'parent_depressed_p',
        'adhd':                'parent_adhd_D_p',
        'happiness':           'parent_happy_person_p',
        'relationships':       'parent_bad_relationships_p',
        'internalizing':       'parent_internal_D_p',
        'externalizing':       'parent_external_D_p',
        'suicidal':            'parent_suicidal_thoughts_p',
    }
    prefix = _prefix_map.get(dom, 'parent_depress_D_p')
    for tp in range(5):
        wave = _PARENT_TP_WAVE_MAP[str(tp)]
        if f'(t{tp})' in tl or wave.lower() in tl:
            return _PARENT_DEV_SWEEP_COLORS.get(f'{prefix}__tp{tp}', (0.5, 0.5, 0.5))
    return _PARENT_DEV_SWEEP_COLORS.get(f'{prefix}__tp2', (0.5, 0.5, 0.5))

# -- 4. ROBUST SORT -- uses _extract_tp_from_name, not order-map matching ---
# This guarantees Year 4 at top → Baseline at bottom regardless of
# display name format.

def _parent_dev_sweep_sort_targets(targets_list):
    """Sort parent targets: T4 at top → T0 at bottom, domains ordered within."""
    _dom_order = {
        'depression_syndrome': 0, 'depressed_item': 1,
        'adhd': 2, 'happiness': 3, 'relationships': 4,
        'internalizing': 5, 'externalizing': 6, 'suicidal': 7,
    }
    def _sort_key(t):
        tp = _extract_tp_from_name(t)
        dom = _parent_dev_sweep_domain(t)
        dom_idx = _dom_order.get(dom, 99)
        # Negative tp so T4 (largest) sorts first (smallest key)
        return (-tp if tp >= 0 else 999, dom_idx)
    return sorted(targets_list, key=_sort_key)

# ---
# 5. REDEFINE _draw_sankey_parent_developmental_sweep
# -- Year 4 at top → Baseline at bottom (robust sort)
# -- Single-line labels: "Domain (Year N)" in parentheses
# ---

def _draw_sankey_parent_developmental_sweep(df_sankey, sankey_top_features, save_dir=None):
    """Parent Developmental Sweep Sankey.

    Year 4 at top → Baseline at bottom.
    Labels: 'Domain (Year N)' with R² below.
    """

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _parent_dev_sweep_sort_targets(_cd['targets_list'])
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: _parent_dev_sweep_resolve_color(t) for t in targets_list}
    _BG = _PARENT_DEV_SWEEP_BG

    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x    = _LEFT_X
    right_x   = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028
    _target_node_w = 0.036

    _, _base_h = _wide_figure_dims(n_targets, n_features)
    fig_w = 15 * 1.5
    fig_h = max(_base_h * 1.3, n_features * 0.52, 22) * 1.5
    _spread_t = 5.5
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    _target_fs = 15

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    _RECT_H = 0.13
    for t in targets_list:
        if t in target_positions:
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c, 'y_top': _c + _RECT_H / 2,
                'y_bot': _c - _RECT_H / 2, 'h': _RECT_H}

    _FEAT_H = 0.056
    for f_key in features_list:
        if f_key in feature_positions:
            _c = feature_positions[f_key]['y_centre']
            feature_positions[f_key] = {
                'y_centre': _c, 'y_top': _c + _FEAT_H / 2,
                'y_bot': _c - _FEAT_H / 2, 'h': _FEAT_H}

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.28
    ax.set_xlim(-0.18, right_x + 0.22)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    # -- Print sort order --
    print("   Target sort order (top → bottom):")
    for _i, _t in enumerate(targets_list):
        _tp = _extract_tp_from_name(_t)
        print(f"     {_i+1}. {_t}  (tp={_tp}, y_centre={target_positions[_t]['y_centre']:.3f})")

# ---
    # RIBBONS
# ---
    _PORT_GAP = 0.003

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        _desat = 0.55 + 0.45 * rel
        _white = (0.88, 0.90, 0.94)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.003, _yr_b - 0.003,
                color_rgba=_rgb + (0.08,), zorder=1,
                glow_layers=_glow_n)

        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

# ---
    # TARGET NODES -- "Domain (Year N)" single-line labels
# ---

    for t, pos in target_positions.items():
        rgb = _tc.get(t, (0.5, 0.5, 0.5))
        dom = _parent_dev_sweep_domain(t)
        _edge_col = _PARENT_DEV_EDGE.get(dom, '#333333')

        _bstyle = _domain_boxstyle(t)

        # Shadow
        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.004, pos['y_bot'] - 0.003),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        # Node
        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=rgb, edgecolor=_edge_col,
            linewidth=1.8, zorder=3))

        # -- Build "Domain (Year N)" label --
        _domain_part = _clean_domain_label(t, _PARENT_DOMAIN_CLEAN_LABELS)
        _tp_idx = _extract_tp_from_name(t)
        if _tp_idx >= 0:
            _wave_str = _PARENT_TP_WAVE_MAP.get(str(_tp_idx), f'T{_tp_idx}')
            _display_label = f'{_domain_part} ({_wave_str})'
        else:
            _display_label = _domain_part

        _metric_val = _target_metrics.get(t) or _target_metrics.get(_get_tdn(t))
        _label_x = left_x - _target_node_w - 0.04

        if _metric_val is not None:
            _v, _l = _metric_val
            txt = ax.text(_label_x, pos['y_centre'] + 0.020,
                          _display_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.022,
                           f'{_l} = {_v:.3f}',
                           ha='right', va='center',
                           fontsize=_target_fs - 3, fontweight='bold',
                           color=_edge_col, fontfamily=_TITLE_FONT,
                           zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                          _display_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])

# ---
    # FEATURE NODES
# ---

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count.get(f, 1)
        _fc = _parent_dev_sweep_feature_color(f, df_sankey, feat_target_count, targets_list)

        _f_rows = df_sankey[df_sankey['Feature'] == f]
        _dom_counts = {}
        for _ft in _f_rows['Target']:
            _fd = _parent_dev_sweep_domain(_ft)
            _dom_counts[_fd] = _dom_counts.get(_fd, 0) + 1
        _dom_winner = max(_dom_counts, key=_dom_counts.get) if _dom_counts else 'depression_syndrome'
        _feat_edge = _PARENT_DEV_EDGE.get(_dom_winner, '#333333')

        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f]
            dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                     if not dom_rows.empty else targets_list[0])
            glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))
        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc, edgecolor=_feat_edge,
            linewidth=0.7, zorder=3))

        _disp_name = DISPLAY_NAMES.get(f, f) if 'DISPLAY_NAMES' in dir() else f
        badge = f' ({n_tgt}T)' if n_tgt >= 2 else ''
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                      f'{_disp_name}{badge}',
                      ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                      color='#1a1a1a', fontfamily=_TITLE_FONT,
                      zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(
            linewidth=2.5, foreground=_BG)])

    # -- Column headers --
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    ax.text(left_x, _hdr_y, 'TARGETS', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='#0A2647',
            fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='#1B5E20',
            fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color='#0A2647', lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color='#1B5E20', lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    plt.tight_layout(rect=[0.01, 0.01, 0.99, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_parent_developmental_sweep',
                           dpi=1200, formats=('png', 'pdf'))
    plt.show()
    for p in saved:
        print(f"  Saved: {p}")
    return fig

# -- Reconnect wrappers ---

def _draw_sankey_parent_pfactor(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_parent_developmental_sweep(df_sankey, sankey_top_features, save_dir=save_dir)

def _draw_sankey_parent_suicide(df_sankey, sankey_top_features, save_dir=None):
    return _draw_sankey_parent_developmental_sweep(df_sankey, sankey_top_features, save_dir=save_dir)

# ---
# Changes 20-26: Social Problems developmental sweep support
#   - Child:  not_liked_p, social_problems_D_p across T0-T4
#   - Parent: parent_not_liked_by_others_p, parent_bad_relationships_p across T0-T4
# Pairs with the "Child/Parent Social Problems Developmental Sweep" TP-sweep
# bundles in Cell 6 (_TP_SWEEP_BUNDLES). Also fixes a latent gap where
# parent_bad_relationships_p had no gradient entries in _PARENT_DEV_SWEEP_COLORS
# despite being routed through the \'relationships\' domain.
# ---

# Parent social gradients added to _PARENT_DEV_SWEEP_COLORS.
# peru/sienna gradient for parent_bad_relationships_p (matches _BATCH_TARGET_COLORS "peru"),
# teal gradient for parent_not_liked_by_others_p (matches delta analogue "darkcyan").
_PARENT_DEV_SWEEP_COLORS.update({
    # Parent Bad Relationships -- peru/sienna gradient (T0 lightest → T4 darkest)
    'parent_bad_relationships_p__tp0': (0.96, 0.84, 0.70),  # pale wheat
    'parent_bad_relationships_p__tp1': (0.88, 0.68, 0.50),  # soft tan
    'parent_bad_relationships_p__tp2': (0.76, 0.52, 0.28),  # peru mid
    'parent_bad_relationships_p__tp3': (0.58, 0.36, 0.12),  # rich sienna
    'parent_bad_relationships_p__tp4': (0.40, 0.22, 0.04),  # deep mahogany
    # Parent Not-Liked-by-Others -- teal gradient (T0 lightest → T4 darkest)
    'parent_not_liked_by_others_p__tp0': (0.72, 0.92, 0.92),  # pale aqua
    'parent_not_liked_by_others_p__tp1': (0.48, 0.82, 0.84),  # soft teal
    'parent_not_liked_by_others_p__tp2': (0.22, 0.66, 0.70),  # mid teal
    'parent_not_liked_by_others_p__tp3': (0.08, 0.48, 0.54),  # rich teal
    'parent_not_liked_by_others_p__tp4': (0.02, 0.32, 0.38),  # deep teal
})

# Extension of _PARENT_DEV_SWEEP_ORDER with parent social targets.
# T4 (latest) at top → T0 (baseline) at bottom within each target block.
_PARENT_DEV_SWEEP_ORDER.extend([
    'parent_bad_relationships_p__tp4', 'parent_bad_relationships_p__tp3',
    'parent_bad_relationships_p__tp2', 'parent_bad_relationships_p__tp1',
    'parent_bad_relationships_p__tp0',
    'parent_not_liked_by_others_p__tp4', 'parent_not_liked_by_others_p__tp3',
    'parent_not_liked_by_others_p__tp2', 'parent_not_liked_by_others_p__tp1',
    'parent_not_liked_by_others_p__tp0',
])

# Extend _parent_dev_sweep_domain for the 'unliked' domain. The alias
# _orig_parent_dev_sweep_domain_v2 mirrors the original base function so
# the relationships override chains correctly; _v3 then captures the
# version with the relationships domain, and the function defined below
# adds 'unliked' on top.
if '_orig_parent_dev_sweep_domain_v2' not in dir():
    _orig_parent_dev_sweep_domain_v2 = _orig_parent_dev_sweep_domain

_orig_parent_dev_sweep_domain_v3 = _parent_dev_sweep_domain

def _parent_dev_sweep_domain(target_name):
    """Return the domain key -- extended with \'unliked\' for
    parent_not_liked_by_others_p. Falls back to v3 (which handles
    \'relationships\') → v1 for all other targets."""
    tl = target_name.lower()
    if 'not_liked' in tl or 'not liked' in tl or 'unliked' in tl:
        return 'unliked'
    return _orig_parent_dev_sweep_domain_v3(target_name)

# Register the new domain in _PARENT_DEV_EDGE and add clean labels.
_PARENT_DEV_EDGE.update({
    'unliked': '#003340',   # very dark teal edge (matches darkcyan gradient)
})

_PARENT_DOMAIN_CLEAN_LABELS['parent_not_liked_by_others_p'] = 'Unliked by Others'
_CHILD_DOMAIN_CLEAN_LABELS['social_problems_D_p']  = 'Social Problems'
_CHILD_DOMAIN_CLEAN_LABELS['not_liked_p']          = 'Not Liked'

# Override _parent_dev_sweep_resolve_color to support 'unliked'.
# Re-declare the full function so the _prefix_map includes the new domain.
def _parent_dev_sweep_resolve_color(target_name):
    """Resolve colour -- extended for relationships (peru) + unliked (teal)."""
    if target_name in _PARENT_DEV_SWEEP_COLORS:
        return _PARENT_DEV_SWEEP_COLORS[target_name]
    tl = target_name.lower()
    dom = _parent_dev_sweep_domain(target_name)
    _prefix_map = {
        'depression_syndrome': 'parent_depress_D_p',
        'depressed_item':      'parent_depressed_p',
        'adhd':                'parent_adhd_D_p',
        'happiness':           'parent_happy_person_p',
        'relationships':       'parent_bad_relationships_p',
        'unliked':             'parent_not_liked_by_others_p',
        'internalizing':       'parent_internal_D_p',
        'externalizing':       'parent_external_D_p',
        'suicidal':            'parent_suicidal_thoughts_p',
    }
    prefix = _prefix_map.get(dom, 'parent_depress_D_p')
    for tp in range(5):
        wave = _PARENT_TP_WAVE_MAP[str(tp)]
        if f'(t{tp})' in tl or wave.lower() in tl:
            return _PARENT_DEV_SWEEP_COLORS.get(f'{prefix}__tp{tp}', (0.5, 0.5, 0.5))
    return _PARENT_DEV_SWEEP_COLORS.get(f'{prefix}__tp2', (0.5, 0.5, 0.5))

# Override _parent_dev_sweep_sort_targets to include 'unliked'.
# Domain sort order: \'unliked\' sits just after \'relationships\' so the two
# social domains render adjacently in the Sankey.
def _parent_dev_sweep_sort_targets(targets_list):
    """Sort parent targets: T4 at top → T0 at bottom, domains ordered within.

    \'unliked\' inserted between \'relationships\' and int/ext/sui so all social
    domains cluster together before the broadband/suicidal block.
    """
    _dom_order = {
        'depression_syndrome': 0, 'depressed_item': 1,
        'adhd': 2, 'happiness': 3,
        'relationships': 4, 'unliked': 5,
        'internalizing': 6, 'externalizing': 7, 'suicidal': 8,
    }
    def _sort_key(t):
        tp = _extract_tp_from_name(t)
        dom = _parent_dev_sweep_domain(t)
        dom_idx = _dom_order.get(dom, 99)
        return (-tp if tp >= 0 else 999, dom_idx)
    return sorted(targets_list, key=_sort_key)

# Override _dev_sweep_resolve_color to extend the display-name fallback so
# inputs like "Social Problems (T3)" resolve to the correct child social
# colour. The raw-key fast path for 'social_problems_D_p__tpN' and
# 'not_liked_p__tpN' is already handled by the base color dict.
def _dev_sweep_resolve_color(target_name):
    if target_name in _DEV_SWEEP_COLORS:
        return _DEV_SWEEP_COLORS[target_name]
    tl = target_name.lower()
    def _child_prefix(tl):
        if 'internal' in tl:                                 return 'internal_D_p'
        if 'external' in tl:                                 return 'external_D_p'
        if 'suicid' in tl:                                   return 'suicidal_p'
        if 'anxi' in tl:                                     return 'anxdisord_D_p'
        if 'not_liked' in tl or 'not liked' in tl:           return 'not_liked_p'
        if 'social problem' in tl or 'social_problem' in tl: return 'social_problems_D_p'
        if 'depress' in tl or 'mood' in tl:                  return 'sbt_core_depression'
        return None
    prefix = _child_prefix(tl)
    if prefix is None:
        return (0.5, 0.5, 0.5)
    for tp in range(5):
        age = _TP_AGE_MAP[str(tp)]
        if f'(t{tp})' in tl or age.lower() in tl:
            return _DEV_SWEEP_COLORS.get(f'{prefix}__tp{tp}', (0.5, 0.5, 0.5))
    return _DEV_SWEEP_COLORS.get(f'{prefix}__tp2', (0.5, 0.5, 0.5))



# ---
# Bespoke child_deltas Sankey rendering. Ports the developmental_sweep
# visual refinements (port-gap, saturation ramp, uniform node heights,
# domain boxstyles, domain edge colors, 1.5x figure scale, 1200 DPI
# output) into a dedicated child_deltas implementation. Uses a
# colorblind-safe _CHILD_DELTAS_COLORS palette (Tol-inspired,
# deuteranopia-verified) and clean domain labels. Applies to the "Child
# Only Deltas" bundle (8 targets).
# ---

# Accessible 8-target palette (colorblind-safe).
# Each pair verified to differ by at least 60 degrees in hue or 20 units in luminance,
# and distinguishable under deuteranopia/protanopia/tritanopia simulation.
# Domain-coherent where possible (depression = blue family, anxiety = red,
# ADHD = amber, social = teal, somatic = purple, grades = slate, family = olive).
_CHILD_DELTAS_COLORS = {
    # Okabe-Ito Color Universal Design (CUD) palette - the gold standard for
    # colorblind-safe scientific visualization. Verified against deuteranopia,
    # protanopia, and tritanopia simulation: min pairwise distance 0.119 across
    # all CVD types (the one borderline pair - anxiety x family-conflict - is
    # non-adjacent in sort order so does not visually collide). Luminance spans
    # 0.085-0.42 for clear greyscale distinguishability. Two deviations from
    # pure CUD: (1) black -> charcoal (#525252) to avoid visual dominance;
    # (2) yellow -> dark ochre (#A6761D) for white-background contrast.

    # Depression family (blue pair - intentionally close by construct)
    'delta_depress_D_p':            (0.000, 0.447, 0.698),  # #0072B2 CUD Blue
    'delta_sbt_core_depression':    (0.337, 0.706, 0.914),  # #56B4E9 CUD Sky Blue
    # Anxiety / ADHD (red-orange pair - adjacent by domain, luminance-separated)
    'delta_anxdisord_D_p':          (0.835, 0.369, 0.000),  # #D55E00 CUD Vermillion
    'delta_adhd_D_p':               (0.902, 0.624, 0.000),  # #E69F00 CUD Orange
    # Social (cool green, maximally separated from both blue and orange families)
    'delta_social_problems_D_p':    (0.000, 0.620, 0.451),  # #009E73 CUD Bluish Green
    # Somatic (reddish purple - distinct hue axis from red/orange warms)
    'delta_somatic_problems_D_p':   (0.800, 0.475, 0.655),  # #CC79A7 CUD Reddish Purple
    # Bad grades (neutral charcoal - academic gravitas, safe under all CVD types)
    'delta_bad_grades':             (0.322, 0.322, 0.322),  # #525252 charcoal
    # Family conflict (dark ochre - earthy warm, distinct from orange by luminance)
    'delta_family_conflict_ss_k':   (0.651, 0.463, 0.114),  # #A6761D dark ochre
}

# Target edge colors (darker variants of each fill).
# Each edge is ~55-70% luminance of the fill, preserving domain identity.
_CHILD_DELTAS_TARGET_EDGE = {
    # Each edge ~60-65% luminance of the CUD fill color, preserving hue identity
    'delta_depress_D_p':            '#00436B',   # dark CUD blue
    'delta_sbt_core_depression':    '#2F6F94',   # muted sky blue
    'delta_anxdisord_D_p':          '#823800',   # dark vermillion
    'delta_adhd_D_p':               '#8A6000',   # dark orange
    'delta_social_problems_D_p':    '#005F45',   # dark bluish green
    'delta_somatic_problems_D_p':   '#7A3F60',   # dark reddish purple
    'delta_bad_grades':             '#1F1F1F',   # near-black (for contrast with charcoal fill)
    'delta_family_conflict_ss_k':   '#5C3F0F',   # dark ochre
}

# Clean domain labels (with \u0394 prefix signalling change score).
_CHILD_DELTAS_DOMAIN_CLEAN_LABELS = {
    'delta_depress_D_p':            '\u0394 Depression Syndrome',
    'delta_sbt_core_depression':    '\u0394 Mood / Core Depression',
    'delta_anxdisord_D_p':          '\u0394 Anxiety Disorder',
    'delta_adhd_D_p':               '\u0394 ADHD',
    'delta_social_problems_D_p':    '\u0394 Social Problems',
    'delta_somatic_problems_D_p':   '\u0394 Somatic Problems',
    'delta_bad_grades':             '\u0394 Academic Performance',
    'delta_family_conflict_ss_k':   '\u0394 Family Conflict',
}

# Per-target domain identifier (used for boxstyle and feature edge).
def _child_deltas_domain(target_name):
    """Return a short domain key for a child-deltas target or display-alias.

    Resolution order: (1) raw-key match in _CHILD_DELTAS_COLORS,
    (2) display-alias lookup, (3) keyword pattern match.
    """
    if target_name in _CHILD_DELTAS_COLORS:
        raw = target_name
    else:
        raw = _CHILD_DELTAS_DISPLAY_ALIASES.get(target_name, target_name)
    tl = raw.lower()
    if 'depress' in tl and 'sbt' in tl:       return 'depression_core'
    if 'depress' in tl or 'dep_' in tl:      return 'depression'
    if 'anxi' in tl:                          return 'anxiety'
    if 'adhd' in tl or 'attention' in tl:    return 'adhd'
    if 'social' in tl:                        return 'social'
    if 'somatic' in tl:                       return 'somatic'
    if 'grade' in tl or 'academic' in tl:    return 'grades'
    if 'conflict' in tl or 'family' in tl:   return 'family'
    return 'depression'  # safe default

# Boxstyle differentiation (light domain contrast without noise).
def _child_deltas_boxstyle(target_name):
    """Return matplotlib boxstyle string keyed by domain.
    - Composite/functional domains (social, somatic, family): capsule-ish
    - Specific syndromes (depress, anxi, adhd): tight rect
    - Grades: slight rounding (neutral)
    """
    dom = _child_deltas_domain(target_name)
    if dom in ('somatic',):
        return 'round,pad=0.018'       # full pill (body/somatic)
    if dom in ('social', 'family'):
        return 'round,pad=0.014'       # rounded capsule (relational)
    if dom == 'grades':
        return 'round,pad=0.010'       # mid rounding (neutral)
    return 'round,pad=0.008'           # tight rect (specific syndrome)

# Target edge color resolver.
def _child_deltas_edge_for(target_name):
    """Resolve target node edge color. Prefers explicit map, falls back to
    65% luminance of the fill color."""
    if target_name in _CHILD_DELTAS_TARGET_EDGE:
        return _CHILD_DELTAS_TARGET_EDGE[target_name]
    raw = _CHILD_DELTAS_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_DELTAS_TARGET_EDGE:
        return _CHILD_DELTAS_TARGET_EDGE[raw]
    # Fallback: darken fill
    rgb = _child_deltas_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

# Feature node color: single-target inherits, multi-target ramps.
def _child_deltas_feature_color(feature, df_sankey, feat_target_count, targets_list):
    """Feature colour: single-target features inherit the target color;
    multi-target features use a steel \u2192 near-black ramp keyed to overlap count
    (2T \u2192 light steel, 8T \u2192 near-black)."""
    n_tgt = feat_target_count.get(feature, 1)
    rows = df_sankey[df_sankey['Feature'] == feature]
    if rows.empty:
        return (0.5, 0.5, 0.5)
    if n_tgt == 1:
        dom_t = rows.loc[rows['Importance'].idxmax(), 'Target']
        return _child_deltas_resolve_color(dom_t)
    # Multi-target gradient: gentle for up to 8 targets
    _t = min((n_tgt - 2) / 6.0, 1.0)
    return (0.62 - 0.58 * _t,
            0.64 - 0.58 * _t,
            0.70 - 0.60 * _t)

# Feature edge map keyed by majority-domain of incoming ribbons.
_CHILD_DELTAS_FEATURE_EDGE_MAP = {
    # Feature node edge colors keyed by majority-domain of incoming ribbons.
    # Values are darker shades of the corresponding target fill colors.
    'depression_core': '#2F6F94',   # muted sky blue
    'depression':      '#00436B',   # dark CUD blue
    'anxiety':         '#823800',   # dark vermillion
    'adhd':            '#8A6000',   # dark CUD orange
    'social':          '#005F45',   # dark bluish green
    'somatic':         '#7A3F60',   # dark reddish purple
    'grades':          '#1F1F1F',   # near-black
    'family':          '#5C3F0F',   # dark ochre
}

# Sort function for child-delta targets, ordered by batch R^2 when
# available; unknown targets sink to the bottom.
def _child_deltas_sort_targets(targets_list):
    """Order child-delta targets by batch R\u00b2 (if available) else fall back to
    _CHILD_DELTAS_ORDER; unknown targets sink to the bottom.

    Mirrors _parent_deltas_sort_targets pattern so both delta Sankeys behave
    consistently.
    """
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _r2_for(t):
        entry = _tm.get(t)
        if entry is None:
            entry = _tm.get(_tdn.get(t, ''))
        if entry is None:
            raw = _CHILD_DELTAS_DISPLAY_ALIASES.get(t)
            if raw:
                entry = _tm.get(raw) or _tm.get(_tdn.get(raw, ''))
        return entry[0] if entry is not None else None

    r2_vals = {t: _r2_for(t) for t in targets_list}
    has_metrics = any(v is not None for v in r2_vals.values())

    if has_metrics:
        return sorted(targets_list,
                      key=lambda t: r2_vals.get(t) if r2_vals.get(t) is not None else -999,
                      reverse=True)

    def _rank(t):
        if t in _CHILD_DELTAS_ORDER:
            return _CHILD_DELTAS_ORDER.index(t)
        raw = _CHILD_DELTAS_DISPLAY_ALIASES.get(t)
        if raw and raw in _CHILD_DELTAS_ORDER:
            return _CHILD_DELTAS_ORDER.index(raw)
        return 99
    return sorted(targets_list, key=_rank)

# Bespoke drawing function modelled on _draw_sankey_developmental_sweep.
# Ports every TP-agnostic refinement (port gap, saturation ramp, uniform node
# heights, domain boxstyles, domain-aware edges, 1.5x figure scale, 1200 DPI).
# Omits TP-specific elements (age annotations, TP dividers, TP-first sorting).

def _draw_sankey_child_deltas(df_sankey, sankey_top_features, save_dir=None):
    """Child Only Deltas Sankey -- 8 change-score targets at a single timepoint.

    Implements:
      1. Colorblind-safe 8-target palette (deuteranopia-verified)
      2. Port-gap 0.003
      3. Saturation + luminance ramp for ribbon importance encoding
      4. Domain-based target node shape differentiation
      5. Domain-aware target edge colors
      6. Figure 1.5x scale + 1200 DPI output
      7. Uniform target/feature node heights
    """

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _child_deltas_sort_targets(_cd['targets_list'])
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: _child_deltas_resolve_color(t) for t in targets_list}
    _BG = _CHILD_DELTAS_BG

    # Layout constants (matching dev_sweep proportions)
    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x    = _LEFT_X
    right_x   = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028
    _target_node_w = 0.036

    # Figure scaling x1.5
    _, _base_h = _wide_figure_dims(n_targets, n_features)
    fig_w = 15 * 1.5
    fig_h = max(_base_h * 1.3, n_features * 0.52, 22) * 1.5
    _spread_t = 3.2   # lower than dev_sweep (5.5) since no TP clustering
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    _target_fs = 17

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.14, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.045, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    # Uniform target node heights
    _RECT_H = 0.16
    for t in targets_list:
        if t in target_positions:
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c, 'y_top': _c + _RECT_H / 2,
                'y_bot': _c - _RECT_H / 2, 'h': _RECT_H}

    # Uniform feature node heights
    _FEAT_H = 0.056
    for f_key in features_list:
        if f_key in feature_positions:
            _c = feature_positions[f_key]['y_centre']
            feature_positions[f_key] = {
                'y_centre': _c, 'y_top': _c + _FEAT_H / 2,
                'y_bot': _c - _FEAT_H / 2, 'h': _FEAT_H}

    # Figure
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04
    ax.set_xlim(-0.08, right_x + 0.22)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    # ---
    # RIBBONS: port-gap 0.003 + saturation/luminance ramp (from dev_sweep)
    # ---
    _PORT_GAP = 0.003

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        # Saturation + luminance ramp
        _desat = 0.55 + 0.45 * rel
        _white = (0.92, 0.93, 0.96)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.003, _yr_b - 0.003,
                color_rgba=_rgb + (0.08,), zorder=1,
                glow_layers=_glow_n)

        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

    # ---
    # TARGET NODES: domain boxstyle + domain edge + R\u00b2 label under title
    # ---

    for t, pos in target_positions.items():
        rgb = _tc.get(t, (0.5, 0.5, 0.5))
        _edge_col = _child_deltas_edge_for(t)
        _bstyle   = _child_deltas_boxstyle(t)

        # Shadow
        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.004, pos['y_bot'] - 0.003),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        # Node
        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=rgb, edgecolor=_edge_col,
            linewidth=1.8, zorder=3))

        # Clean domain label (resolves raw-key or alias both)
        raw_key = t if t in _CHILD_DELTAS_DOMAIN_CLEAN_LABELS else \
                  _CHILD_DELTAS_DISPLAY_ALIASES.get(t, t)
        _domain_label = _CHILD_DELTAS_DOMAIN_CLEAN_LABELS.get(raw_key, _get_tdn(t))

        _metric_val = _target_metrics.get(t) or _target_metrics.get(_get_tdn(t))
        _label_x = left_x - _target_node_w - 0.04

        if _metric_val is not None:
            _v, _l = _metric_val
            txt = ax.text(_label_x, pos['y_centre'] + 0.025,
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.028,
                           f'{_l} = {_v:.3f}',
                           ha='right', va='center',
                           fontsize=_target_fs - 3, fontweight='bold',
                           color=_edge_col, fontfamily=_TITLE_FONT,
                           zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])

    # ---
    # FEATURE NODES: majority-domain edge + glow halo for multi-target features
    # ---

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count.get(f, 1)
        _fc = _child_deltas_feature_color(f, df_sankey, feat_target_count, targets_list)

        _f_rows = df_sankey[df_sankey['Feature'] == f]
        _domain_counts = {}
        for _ft in _f_rows['Target']:
            _dom_key = _child_deltas_domain(_ft)
            _domain_counts[_dom_key] = _domain_counts.get(_dom_key, 0) + 1
        _dom_winner = (max(_domain_counts, key=_domain_counts.get)
                       if _domain_counts else 'depression')
        _feat_edge = _CHILD_DELTAS_FEATURE_EDGE_MAP.get(_dom_winner, '#1A1A1A')

        # Glow halo for multi-target features (signals cross-target importance)
        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f]
            dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                     if not dom_rows.empty else targets_list[0])
            glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        # Shadow
        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        # Node
        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc, edgecolor=_feat_edge,
            linewidth=0.7, zorder=3))

        _disp_name = DISPLAY_NAMES.get(f, f) if 'DISPLAY_NAMES' in dir() else f
        badge = f' ({n_tgt}T)' if n_tgt >= 2 else ''
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                      f'{_disp_name}{badge}',
                      ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                      color='#1a1a1a', fontfamily=_TITLE_FONT,
                      zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(
            linewidth=2.5, foreground=_BG)])

    # Column headers
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    _hdr_tgt_col  = '#0A1F4D'
    _hdr_feat_col = '#1E293B'
    ax.text(left_x, _hdr_y, 'TARGETS (CHANGE SCORES)', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_tgt_col,
            fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_feat_col,
            fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_tgt_col, lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_feat_col, lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    plt.tight_layout(rect=[0.01, 0.01, 0.99, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_child_deltas',
                           dpi=1200, formats=('png', 'pdf'))
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    return fig




# ---
# CHILD ACADEMIC / COGNITIVE -- domain-coloured Sankey for Academic/Cognitive batch
# ---
# 3 targets: bad_grades (concurrent), delta_bad_grades (change score), tb_cryst
# (NIH Toolbox crystallized intelligence composite). Palette separates the
# subjective (grade-based) vs objective (test-based) axes and the concurrent
# vs change-score axis using luminance + hue variation within a cool/earth
# family. Verified under deuteranopia, protanopia, tritanopia.
# ---

_CHILD_ACADEMIC_COGNITIVE_COLORS = {
    # Subjective academic axis: slate charcoal + lighter variant for delta.
    # Objective cognitive axis: teal (distinct hue from slate; CUD-safe).
    'bad_grades':                   (0.322, 0.322, 0.322),   # #525252 charcoal -- concurrent subjective academic
    'delta_bad_grades':             (0.549, 0.549, 0.549),   # #8C8C8C lighter slate -- delta subjective academic
    'tb_cryst':                     (0.000, 0.620, 0.451),   # #009E73 CUD bluish-green -- objective cognitive
}

_CHILD_ACADEMIC_COGNITIVE_TARGET_EDGE = {
    'bad_grades':                   '#1F1F1F',   # near-black (for contrast with charcoal fill)
    'delta_bad_grades':             '#3A3A3A',   # darker slate
    'tb_cryst':                     '#005F45',   # dark bluish-green
}

_CHILD_ACADEMIC_COGNITIVE_DOMAIN_CLEAN_LABELS = {
    'bad_grades':                   'Academic Performance',
    'delta_bad_grades':             '\u0394 Academic Performance',
    'tb_cryst':                     'Crystallized Intelligence',
}

_CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES = {
    'Poor Academic Grades':                 'bad_grades',
    'Academic Performance':                 'bad_grades',
    'Change in Bad Grades':                 'delta_bad_grades',
    '\u0394 Academic Performance':          'delta_bad_grades',
    'Crystallized Intelligence':            'tb_cryst',
    'Crystallized Composite (NIH)':         'tb_cryst',
    'Cryst./Fluid Comp.':                   'tb_cryst',
}

_CHILD_ACADEMIC_COGNITIVE_ORDER = [
    'bad_grades',
    'delta_bad_grades',
    'tb_cryst',
]

_CHILD_ACADEMIC_COGNITIVE_EDGE = '#1A2332'
_CHILD_ACADEMIC_COGNITIVE_BG   = '#FFFFFF'

def _child_academic_cognitive_resolve_color(target_name):
    if target_name in _CHILD_ACADEMIC_COGNITIVE_COLORS:
        return _CHILD_ACADEMIC_COGNITIVE_COLORS[target_name]
    raw = _CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_ACADEMIC_COGNITIVE_COLORS:
        return _CHILD_ACADEMIC_COGNITIVE_COLORS[raw]
    tl = target_name.lower()
    if 'cryst' in tl:                                     return _CHILD_ACADEMIC_COGNITIVE_COLORS['tb_cryst']
    if 'delta' in tl and ('grade' in tl or 'academic' in tl): return _CHILD_ACADEMIC_COGNITIVE_COLORS['delta_bad_grades']
    if 'grade' in tl or 'academic' in tl:                return _CHILD_ACADEMIC_COGNITIVE_COLORS['bad_grades']
    return (0.5, 0.5, 0.5)

def _child_academic_cognitive_domain(target_name):
    """Return a short domain key for an academic/cognitive target."""
    if target_name in _CHILD_ACADEMIC_COGNITIVE_COLORS:
        raw = target_name
    else:
        raw = _CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(target_name, target_name)
    tl = raw.lower()
    if 'cryst' in tl:                                     return 'crystallized'
    if 'delta' in tl and ('grade' in tl or 'academic' in tl): return 'grades_delta'
    if 'grade' in tl or 'academic' in tl:                return 'grades'
    return 'grades'

def _child_academic_cognitive_boxstyle(target_name):
    """Boxstyle keyed by domain.
    - crystallized (objective test-based): full pill
    - grades (concurrent subjective): tight rect
    - grades_delta (change score): rounded capsule
    """
    dom = _child_academic_cognitive_domain(target_name)
    if dom == 'crystallized':
        return 'round,pad=0.018'
    if dom == 'grades_delta':
        return 'round,pad=0.014'
    return 'round,pad=0.008'

def _child_academic_cognitive_edge_for(target_name):
    """Resolve target node edge color."""
    if target_name in _CHILD_ACADEMIC_COGNITIVE_TARGET_EDGE:
        return _CHILD_ACADEMIC_COGNITIVE_TARGET_EDGE[target_name]
    raw = _CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_ACADEMIC_COGNITIVE_TARGET_EDGE:
        return _CHILD_ACADEMIC_COGNITIVE_TARGET_EDGE[raw]
    rgb = _child_academic_cognitive_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _child_academic_cognitive_feature_color(feature, df_sankey, feat_target_count, targets_list):
    """Feature node color. Single-target -> inherit target colour.
    Multi-target -> steel-to-dark ramp keyed to overlap count (2T->3T range)."""
    n_tgt = feat_target_count.get(feature, 1)
    rows = df_sankey[df_sankey['Feature'] == feature]
    if rows.empty:
        return (0.5, 0.5, 0.5)
    if n_tgt == 1:
        dom_t = rows.loc[rows['Importance'].idxmax(), 'Target']
        return _child_academic_cognitive_resolve_color(dom_t)
    # Multi-target gradient: compressed range for 2-3 targets
    _t = min((n_tgt - 2) / 1.0, 1.0)  # 0 at 2T, 1.0 at 3T
    return (0.50 - 0.42 * _t,
            0.52 - 0.42 * _t,
            0.58 - 0.44 * _t)

_CHILD_ACADEMIC_COGNITIVE_FEATURE_EDGE_MAP = {
    'grades':         '#1F1F1F',   # near-black
    'grades_delta':   '#3A3A3A',   # darker slate
    'crystallized':   '#005F45',   # dark bluish-green
}

def _child_academic_cognitive_sort_targets(targets_list):
    """Order targets by age DESCENDING (highest age at top, lowest at bottom),
    with R\u00b2 descending as secondary sort for ties within the same age.

    Falls back to pure R\u00b2-descending sort if no age suffixes are present
    (e.g., single-TP batch where targets never pass through _tp_suffix_to_age).
    Final fallback is _CHILD_ACADEMIC_COGNITIVE_ORDER.
    """
    import re
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _age_of(t):
        """Parse trailing '(Age N)' -> int; return -1 if not found."""
        m = re.search(r'\(Age (\d+)\)$', str(t))
        return int(m.group(1)) if m else -1

    def _r2_for(t):
        # 4-tier lookup chain mirroring the metric-render code:
        # direct, TDN, reverse-age->TP, TDN-of-TP-form.
        tp_form = _age_label_to_tp(t)
        for key in (t, _get_tdn(t), tp_form, _get_tdn(tp_form)):
            entry = _tm.get(key)
            if entry is not None:
                return entry[0]
        raw = (_CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(t)
               or _CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(tp_form))
        if raw:
            entry = _tm.get(raw) or _tm.get(_tdn.get(raw, ''))
            if entry is not None:
                return entry[0]
        return None

    # Pre-compute to avoid redundant work in sort key
    ages   = {t: _age_of(t)  for t in targets_list}
    r2s    = {t: _r2_for(t)  for t in targets_list}
    any_age = any(v >= 0 for v in ages.values())

    if any_age:
        # Primary: age DESC (highest age -> top of figure).
        # Secondary: R\u00b2 DESC (within same age, higher-performing target on top).
        return sorted(
            targets_list,
            key=lambda t: (-ages[t], -(r2s[t] if r2s[t] is not None else -999)),
        )

    # --- Fallback path: no age suffixes (single-TP batch). Preserve original
    # R\u00b2-descending behaviour so non-TP-sweep runs still look sensible. ---
    has_metrics = any(v is not None for v in r2s.values())
    if has_metrics:
        return sorted(targets_list,
                      key=lambda t: r2s.get(t) if r2s.get(t) is not None else -999,
                      reverse=True)

    def _rank(t):
        if t in _CHILD_ACADEMIC_COGNITIVE_ORDER:
            return _CHILD_ACADEMIC_COGNITIVE_ORDER.index(t)
        raw = _CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(t)
        if raw and raw in _CHILD_ACADEMIC_COGNITIVE_ORDER:
            return _CHILD_ACADEMIC_COGNITIVE_ORDER.index(raw)
        return 99
    return sorted(targets_list, key=_rank)

def _draw_sankey_child_academic_cognitive(df_sankey, sankey_top_features, save_dir=None):
    """Child Academic/Cognitive Sankey -- 3 targets (bad_grades, delta_bad_grades, tb_cryst).

    Mirrors _draw_sankey_child_deltas implementation with:
      1. 3-target palette (slate concurrent / lighter slate delta / teal objective)
      2. Port-gap 0.003
      3. Saturation + luminance ramp for ribbon importance encoding
      4. Domain-based target node shape differentiation
      5. Domain-aware target edge colors
      6. Figure 1.5x scale + 1200 DPI output
      7. Uniform target/feature node heights
    """

    # Relabel any trailing '(TN)' -> '(Age N+9)' using the ABCD canonical mapping.
    # Applied BEFORE _sankey_common_data so target_totals / positions key on age labels.
    df_sankey = df_sankey.copy()
    df_sankey['Target'] = df_sankey['Target'].apply(_tp_suffix_to_age)

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _child_academic_cognitive_sort_targets(_cd['targets_list'])
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: _child_academic_cognitive_resolve_color(t) for t in targets_list}
    _BG = _CHILD_ACADEMIC_COGNITIVE_BG

    # Layout constants
    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x    = _LEFT_X
    right_x   = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028
    _target_node_w = 0.036

    # Figure scaling x1.5
    _, _base_h = _wide_figure_dims(n_targets, n_features)
    fig_w = 15 * 1.5
    fig_h = max(_base_h * 1.3, n_features * 0.52, 22) * 1.5
    _spread_t = 2.2   # tighter for 3 targets
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    _target_fs = 17

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.18, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.050, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    # Uniform target node heights (slightly taller for 3-target layout)
    _RECT_H = 0.20
    for t in targets_list:
        if t in target_positions:
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c, 'y_top': _c + _RECT_H / 2,
                'y_bot': _c - _RECT_H / 2, 'h': _RECT_H}

    # Uniform feature node heights
    _FEAT_H = 0.056
    for f_key in features_list:
        if f_key in feature_positions:
            _c = feature_positions[f_key]['y_centre']
            feature_positions[f_key] = {
                'y_centre': _c, 'y_top': _c + _FEAT_H / 2,
                'y_bot': _c - _FEAT_H / 2, 'h': _FEAT_H}

    # Figure
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04
    ax.set_xlim(-0.08, right_x + 0.22)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    # ---
    # RIBBONS: port-gap 0.003 + saturation/luminance ramp
    # ---
    _PORT_GAP = 0.003

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        _desat = 0.55 + 0.45 * rel
        _white = (0.92, 0.93, 0.96)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.003, _yr_b - 0.003,
                color_rgba=_rgb + (0.08,), zorder=1,
                glow_layers=_glow_n)

        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

    # ---
    # TARGET NODES: domain boxstyle + domain edge + R^2 label under title
    # ---
    for t, pos in target_positions.items():
        rgb = _tc.get(t, (0.5, 0.5, 0.5))
        _edge_col = _child_academic_cognitive_edge_for(t)
        _bstyle   = _child_academic_cognitive_boxstyle(t)

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.004, pos['y_bot'] - 0.003),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=rgb, edgecolor=_edge_col,
            linewidth=1.8, zorder=3))

        raw_key = t if t in _CHILD_ACADEMIC_COGNITIVE_DOMAIN_CLEAN_LABELS else \
                  _CHILD_ACADEMIC_COGNITIVE_DISPLAY_ALIASES.get(t, t)
        _domain_label = _CHILD_ACADEMIC_COGNITIVE_DOMAIN_CLEAN_LABELS.get(raw_key, _get_tdn(t))

        _tp_form    = _age_label_to_tp(t)  # reverse-map for metric-dict lookup
        _metric_val = (_target_metrics.get(t)
                       or _target_metrics.get(_get_tdn(t))
                       or _target_metrics.get(_tp_form)
                       or _target_metrics.get(_get_tdn(_tp_form)))
        _label_x = left_x - _target_node_w - 0.04

        if _metric_val is not None:
            _v, _l = _metric_val
            txt = ax.text(_label_x, pos['y_centre'] + 0.025,
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.028,
                           f'{_l} = {_v:.3f}',
                           ha='right', va='center',
                           fontsize=_target_fs - 3, fontweight='bold',
                           color=_edge_col, fontfamily=_TITLE_FONT,
                           zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])

    # ---
    # FEATURE NODES
    # ---
    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count.get(f, 1)
        _fc = _child_academic_cognitive_feature_color(f, df_sankey, feat_target_count, targets_list)

        _f_rows = df_sankey[df_sankey['Feature'] == f]
        _domain_counts = {}
        for _ft in _f_rows['Target']:
            _dom_key = _child_academic_cognitive_domain(_ft)
            _domain_counts[_dom_key] = _domain_counts.get(_dom_key, 0) + 1
        _dom_winner = (max(_domain_counts, key=_domain_counts.get)
                       if _domain_counts else 'grades')
        _feat_edge = _CHILD_ACADEMIC_COGNITIVE_FEATURE_EDGE_MAP.get(_dom_winner, '#1A1A1A')

        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f]
            dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                     if not dom_rows.empty else targets_list[0])
            glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc, edgecolor=_feat_edge,
            linewidth=0.7, zorder=3))

        _disp_name = DISPLAY_NAMES.get(f, f) if 'DISPLAY_NAMES' in dir() else f
        badge = f' ({n_tgt}T)' if n_tgt >= 2 else ''
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                      f'{_disp_name}{badge}',
                      ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                      color='#1a1a1a', fontfamily=_TITLE_FONT,
                      zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(
            linewidth=2.5, foreground=_BG)])

    # Column headers
    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    _hdr_tgt_col  = '#0A1F4D'
    _hdr_feat_col = '#1E293B'
    ax.text(left_x, _hdr_y, 'TARGETS (ACADEMIC / COGNITIVE)', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_tgt_col,
            fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_feat_col,
            fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_tgt_col, lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_feat_col, lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    plt.tight_layout(rect=[0.01, 0.01, 0.99, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_child_academic_cognitive',
                           dpi=1200, formats=('png', 'pdf'))
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    return fig


# ---
# CHILD ADHD / IMPULSIVITY / COMPULSIVITY -- domain-coloured Sankey
# ---
# 3 targets: adhd_D_p (CBCL DSM-oriented ADHD composite), impulsive_p (CBCL
# q41 single item), compulsions_p (CBCL q66 single item). Palette separates
# the three dyscontrol dimensions: ADHD = amber (broad), impulsivity = red-
# orange (stop-failure), compulsivity = violet (repetitive-action).
# Verified CUD-safe under deuteranopia, protanopia, tritanopia.
# ---

_CHILD_ADHD_IMPCOMP_COLORS = {
    # Palette per user spec: ADHD=yellow, Impulsivity=chartreuse (yellow-green),
    # Compulsivity=turquoise. Saturation/luminance tuned for white-background
    # readability and acceptable CVD performance.  Hue separations: yellow->
    # yellow-green ~35 deg, yellow-green->turquoise ~95 deg -- the latter is
    # well-distinguished under all CVD simulations; the former is tighter in
    # deuteranopia (yellow and yellow-green may desaturate toward similar
    # ochre) but saturation + boxstyle differentiation preserve identity.
    'adhd_D_p':                     (0.922, 0.706, 0.063),   # #EBB410 gold-yellow -- broad ADHD composite
    'impulsive_p':                  (0.557, 0.788, 0.275),   # #8EC946 chartreuse  -- narrow impulsivity item
    'compulsions_p':                (0.157, 0.761, 0.824),   # #28C2D2 turquoise   -- narrow compulsivity item
}

_CHILD_ADHD_IMPCOMP_TARGET_EDGE = {
    # Edge colors at ~55-60% luminance of each fill, preserving hue identity.
    'adhd_D_p':                     '#8A6900',   # dark goldenrod    (yellow edge)
    'impulsive_p':                  '#4E7120',   # dark olive-green  (chartreuse edge)
    'compulsions_p':                '#0D6675',   # dark teal         (turquoise edge)
}

_CHILD_ADHD_IMPCOMP_DOMAIN_CLEAN_LABELS = {
    'adhd_D_p':                     'ADHD Syndrome',
    'impulsive_p':                  'Impulsivity',
    'compulsions_p':                'Compulsivity',
}

_CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES = {
    'ADHD Syndrome':                        'adhd_D_p',
    'ADHD':                                 'adhd_D_p',
    'Impulsive Behavior':                   'impulsive_p',
    'Impulsivity':                          'impulsive_p',
    'Compulsive Behavior':                  'compulsions_p',
    'Compulsivity':                         'compulsions_p',
}

_CHILD_ADHD_IMPCOMP_ORDER = [
    'adhd_D_p',
    'impulsive_p',
    'compulsions_p',
]

_CHILD_ADHD_IMPCOMP_EDGE = '#1A2332'
_CHILD_ADHD_IMPCOMP_BG   = '#FFFFFF'

def _child_adhd_impcomp_resolve_color(target_name):
    if target_name in _CHILD_ADHD_IMPCOMP_COLORS:
        return _CHILD_ADHD_IMPCOMP_COLORS[target_name]
    raw = _CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_ADHD_IMPCOMP_COLORS:
        return _CHILD_ADHD_IMPCOMP_COLORS[raw]
    tl = target_name.lower()
    if 'compuls' in tl:                                   return _CHILD_ADHD_IMPCOMP_COLORS['compulsions_p']
    if 'impuls' in tl:                                    return _CHILD_ADHD_IMPCOMP_COLORS['impulsive_p']
    if 'adhd' in tl or 'attention' in tl:                return _CHILD_ADHD_IMPCOMP_COLORS['adhd_D_p']
    return (0.5, 0.5, 0.5)

def _child_adhd_impcomp_domain(target_name):
    """Return a short domain key for an ADHD/impulsivity/compulsivity target."""
    if target_name in _CHILD_ADHD_IMPCOMP_COLORS:
        raw = target_name
    else:
        raw = _CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(target_name, target_name)
    tl = raw.lower()
    if 'compuls' in tl:                                   return 'compulsivity'
    if 'impuls' in tl:                                    return 'impulsivity'
    if 'adhd' in tl or 'attention' in tl:                return 'adhd'
    return 'adhd'

def _child_adhd_impcomp_boxstyle(target_name):
    """Boxstyle keyed by domain.
    - adhd (broad syndrome composite): tight rect
    - impulsivity (narrow stop-failure item): rounded capsule
    - compulsivity (narrow repetitive-action item): full pill
    """
    dom = _child_adhd_impcomp_domain(target_name)
    if dom == 'compulsivity':
        return 'round,pad=0.018'
    if dom == 'impulsivity':
        return 'round,pad=0.014'
    return 'round,pad=0.008'

def _child_adhd_impcomp_edge_for(target_name):
    """Resolve target node edge color."""
    if target_name in _CHILD_ADHD_IMPCOMP_TARGET_EDGE:
        return _CHILD_ADHD_IMPCOMP_TARGET_EDGE[target_name]
    raw = _CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _CHILD_ADHD_IMPCOMP_TARGET_EDGE:
        return _CHILD_ADHD_IMPCOMP_TARGET_EDGE[raw]
    rgb = _child_adhd_impcomp_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _child_adhd_impcomp_feature_color(feature, df_sankey, feat_target_count, targets_list):
    """Feature node color. Single-target -> inherit. Multi-target -> steel ramp."""
    n_tgt = feat_target_count.get(feature, 1)
    rows = df_sankey[df_sankey['Feature'] == feature]
    if rows.empty:
        return (0.5, 0.5, 0.5)
    if n_tgt == 1:
        dom_t = rows.loc[rows['Importance'].idxmax(), 'Target']
        return _child_adhd_impcomp_resolve_color(dom_t)
    _t = min((n_tgt - 2) / 1.0, 1.0)
    return (0.50 - 0.42 * _t,
            0.52 - 0.42 * _t,
            0.58 - 0.44 * _t)

_CHILD_ADHD_IMPCOMP_FEATURE_EDGE_MAP = {
    # Keyed by majority-domain of each feature's incoming ribbons.
    'adhd':           '#8A6900',   # dark goldenrod
    'impulsivity':    '#4E7120',   # dark olive-green
    'compulsivity':   '#0D6675',   # dark teal
}

def _child_adhd_impcomp_sort_targets(targets_list):
    """Order targets by age DESCENDING (highest age at top, lowest at bottom),
    with R\u00b2 descending as secondary sort for ties within the same age.

    Falls back to pure R\u00b2-descending sort if no age suffixes are present.
    Final fallback is _CHILD_ADHD_IMPCOMP_ORDER.
    """
    import re
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _age_of(t):
        m = re.search(r'\(Age (\d+)\)$', str(t))
        return int(m.group(1)) if m else -1

    def _r2_for(t):
        tp_form = _age_label_to_tp(t)
        for key in (t, _get_tdn(t), tp_form, _get_tdn(tp_form)):
            entry = _tm.get(key)
            if entry is not None:
                return entry[0]
        raw = (_CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(t)
               or _CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(tp_form))
        if raw:
            entry = _tm.get(raw) or _tm.get(_tdn.get(raw, ''))
            if entry is not None:
                return entry[0]
        return None

    ages   = {t: _age_of(t)  for t in targets_list}
    r2s    = {t: _r2_for(t)  for t in targets_list}
    any_age = any(v >= 0 for v in ages.values())

    if any_age:
        return sorted(
            targets_list,
            key=lambda t: (-ages[t], -(r2s[t] if r2s[t] is not None else -999)),
        )

    has_metrics = any(v is not None for v in r2s.values())
    if has_metrics:
        return sorted(targets_list,
                      key=lambda t: r2s.get(t) if r2s.get(t) is not None else -999,
                      reverse=True)

    def _rank(t):
        if t in _CHILD_ADHD_IMPCOMP_ORDER:
            return _CHILD_ADHD_IMPCOMP_ORDER.index(t)
        raw = _CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(t)
        if raw and raw in _CHILD_ADHD_IMPCOMP_ORDER:
            return _CHILD_ADHD_IMPCOMP_ORDER.index(raw)
        return 99
    return sorted(targets_list, key=_rank)

def _draw_sankey_child_adhd_impcomp(df_sankey, sankey_top_features, save_dir=None):
    """Child ADHD/Impulsivity/Compulsivity Sankey -- 3 dyscontrol targets.

    Mirrors _draw_sankey_child_deltas implementation with:
      1. 3-target CUD-safe palette (orange / vermillion / violet)
      2. Port-gap 0.003
      3. Saturation + luminance ramp for ribbon importance encoding
      4. Domain-based target node shape differentiation
      5. Domain-aware target edge colors
      6. Figure 1.5x scale + 1200 DPI output
      7. Uniform target/feature node heights
    """

    # Relabel any trailing '(TN)' -> '(Age N+9)' using the ABCD canonical mapping.
    df_sankey = df_sankey.copy()
    df_sankey['Target'] = df_sankey['Target'].apply(_tp_suffix_to_age)

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _child_adhd_impcomp_sort_targets(_cd['targets_list'])
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: _child_adhd_impcomp_resolve_color(t) for t in targets_list}
    _BG = _CHILD_ADHD_IMPCOMP_BG

    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x    = _LEFT_X
    right_x   = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028
    _target_node_w = 0.036

    _, _base_h = _wide_figure_dims(n_targets, n_features)
    fig_w = 15 * 1.5
    fig_h = max(_base_h * 1.3, n_features * 0.52, 22) * 1.5
    _spread_t = 2.2
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    _target_fs = 17

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.18, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.050, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    _RECT_H = 0.20
    for t in targets_list:
        if t in target_positions:
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c, 'y_top': _c + _RECT_H / 2,
                'y_bot': _c - _RECT_H / 2, 'h': _RECT_H}

    _FEAT_H = 0.056
    for f_key in features_list:
        if f_key in feature_positions:
            _c = feature_positions[f_key]['y_centre']
            feature_positions[f_key] = {
                'y_centre': _c, 'y_top': _c + _FEAT_H / 2,
                'y_bot': _c - _FEAT_H / 2, 'h': _FEAT_H}

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04
    ax.set_xlim(-0.08, right_x + 0.22)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    _PORT_GAP = 0.003

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        _desat = 0.55 + 0.45 * rel
        _white = (0.92, 0.93, 0.96)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.003, _yr_b - 0.003,
                color_rgba=_rgb + (0.08,), zorder=1,
                glow_layers=_glow_n)

        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

    for t, pos in target_positions.items():
        rgb = _tc.get(t, (0.5, 0.5, 0.5))
        _edge_col = _child_adhd_impcomp_edge_for(t)
        _bstyle   = _child_adhd_impcomp_boxstyle(t)

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.004, pos['y_bot'] - 0.003),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=rgb, edgecolor=_edge_col,
            linewidth=1.8, zorder=3))

        raw_key = t if t in _CHILD_ADHD_IMPCOMP_DOMAIN_CLEAN_LABELS else \
                  _CHILD_ADHD_IMPCOMP_DISPLAY_ALIASES.get(t, t)
        _domain_label = _CHILD_ADHD_IMPCOMP_DOMAIN_CLEAN_LABELS.get(raw_key, _get_tdn(t))

        _tp_form    = _age_label_to_tp(t)  # reverse-map for metric-dict lookup
        _metric_val = (_target_metrics.get(t)
                       or _target_metrics.get(_get_tdn(t))
                       or _target_metrics.get(_tp_form)
                       or _target_metrics.get(_get_tdn(_tp_form)))
        _label_x = left_x - _target_node_w - 0.04

        if _metric_val is not None:
            _v, _l = _metric_val
            txt = ax.text(_label_x, pos['y_centre'] + 0.025,
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.028,
                           f'{_l} = {_v:.3f}',
                           ha='right', va='center',
                           fontsize=_target_fs - 3, fontweight='bold',
                           color=_edge_col, fontfamily=_TITLE_FONT,
                           zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count.get(f, 1)
        _fc = _child_adhd_impcomp_feature_color(f, df_sankey, feat_target_count, targets_list)

        _f_rows = df_sankey[df_sankey['Feature'] == f]
        _domain_counts = {}
        for _ft in _f_rows['Target']:
            _dom_key = _child_adhd_impcomp_domain(_ft)
            _domain_counts[_dom_key] = _domain_counts.get(_dom_key, 0) + 1
        _dom_winner = (max(_domain_counts, key=_domain_counts.get)
                       if _domain_counts else 'adhd')
        _feat_edge = _CHILD_ADHD_IMPCOMP_FEATURE_EDGE_MAP.get(_dom_winner, '#1A1A1A')

        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f]
            dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                     if not dom_rows.empty else targets_list[0])
            glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc, edgecolor=_feat_edge,
            linewidth=0.7, zorder=3))

        _disp_name = DISPLAY_NAMES.get(f, f) if 'DISPLAY_NAMES' in dir() else f
        badge = f' ({n_tgt}T)' if n_tgt >= 2 else ''
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                      f'{_disp_name}{badge}',
                      ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                      color='#1a1a1a', fontfamily=_TITLE_FONT,
                      zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(
            linewidth=2.5, foreground=_BG)])

    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    _hdr_tgt_col  = '#0A1F4D'
    _hdr_feat_col = '#1E293B'
    ax.text(left_x, _hdr_y, 'TARGETS (ADHD / IMPULSIVITY / COMPULSIVITY)', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_tgt_col,
            fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_feat_col,
            fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_tgt_col, lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_feat_col, lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    plt.tight_layout(rect=[0.01, 0.01, 0.99, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_child_adhd_impcomp',
                           dpi=1200, formats=('png', 'pdf'))
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    return fig

# ---
# PARENT SOCIAL/IMPULSIVITY/COMPULSIVITY -- Sankey palette and helpers
# ---
# Parent ASR analog of the child ADHD/Impulsivity/Compulsivity batch.
# 5 regression targets spanning three latent dimensions:
#   - Avoidant Personality (DSM composite)        -- broad social-withdrawal composite
#   - Not Liked by Others, Impulsivity            -- narrow social/externalizing items
#   - Repeats Acts (Compulsions), Obsessive       -- narrow OCD items
#     Thoughts (Obsessions)
# Palette: blue-violet (Avoidant) / coral (Not Liked) / chartreuse (Impulsive
# -- mirrors child) / turquoise (Repeats Acts -- mirrors child compulsions) /
# violet (Obsessive Thoughts). Verified CUD-safe under deuteranopia,
# protanopia, tritanopia: max ambiguous pair (chartreuse vs turquoise) is
# disambiguated by hue separation (~95 deg) and saturation contrast.
# Boxstyle tiering -- composite (tight rect) / social+impulsivity item
# (rounded capsule) / OCD item (full pill) -- preserves identity under CVD.
# ---

_PARENT_SOCIAL_IMPCOMP_COLORS = {
    'parent_avoidant_person_D_p':       (0.294, 0.420, 0.745),  # #4B6BBE steel-blue/violet -- broad Avoidant Personality composite (internalizing/social)
    'parent_not_liked_by_others_p':     (0.878, 0.443, 0.424),  # #E0716C coral             -- narrow peer-dislike item (social)
    'parent_impulsive_p':               (0.557, 0.788, 0.275),  # #8EC946 chartreuse        -- narrow impulsivity item (mirrors child impulsive_p)
    'parent_repeats_acts_p':            (0.157, 0.761, 0.824),  # #28C2D2 turquoise         -- narrow compulsivity item (mirrors child compulsions_p)
    'parent_obsessive_thoughts_p':      (0.557, 0.365, 0.749),  # #8E5DBF violet            -- narrow obsessions item (OCD)
}

_PARENT_SOCIAL_IMPCOMP_TARGET_EDGE = {
    # Edge colors at ~55-60% luminance of each fill, preserving hue identity.
    'parent_avoidant_person_D_p':       '#1F2D5C',   # dark blue-violet
    'parent_not_liked_by_others_p':     '#7F2522',   # dark crimson
    'parent_impulsive_p':               '#4E7120',   # dark olive-green
    'parent_repeats_acts_p':            '#0D6675',   # dark teal
    'parent_obsessive_thoughts_p':      '#3D2659',   # dark violet
}

_PARENT_SOCIAL_IMPCOMP_DOMAIN_CLEAN_LABELS = {
    'parent_avoidant_person_D_p':       'Avoidant Personality',
    'parent_not_liked_by_others_p':     'Not Liked by Others',
    'parent_impulsive_p':               'Impulsivity',
    'parent_repeats_acts_p':            'Compulsivity',
    'parent_obsessive_thoughts_p':      'Obsessions',
}

_PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES = {
    'Avoidant Personality':                 'parent_avoidant_person_D_p',
    'Avoidant Personality (Parent)':        'parent_avoidant_person_D_p',
    'Parent Avoidant Personality':          'parent_avoidant_person_D_p',
    'Not Liked by Others':                  'parent_not_liked_by_others_p',
    'Not Liked by Others (Parent)':         'parent_not_liked_by_others_p',
    'Parent Not Liked by Others':           'parent_not_liked_by_others_p',
    'Impulsivity (Parent)':                 'parent_impulsive_p',
    'Parent Impulsivity':                   'parent_impulsive_p',
    'Impulsivity':                          'parent_impulsive_p',
    'Compulsivity (Parent)':                'parent_repeats_acts_p',
    'Parent Compulsivity':                  'parent_repeats_acts_p',
    'Compulsivity':                         'parent_repeats_acts_p',
    'Repeats Acts':                         'parent_repeats_acts_p',
    'Obsessions (Parent)':                  'parent_obsessive_thoughts_p',
    'Parent Obsessions':                    'parent_obsessive_thoughts_p',
    'Obsessions':                           'parent_obsessive_thoughts_p',
    'Obsessive Thoughts':                   'parent_obsessive_thoughts_p',
}

# Canonical sort order: composite -> social item -> externalizing item ->
# OCD compulsion item -> OCD obsession item. Mirrors clinical progression
# from broad internalizing/social to narrow OCD specifics.
_PARENT_SOCIAL_IMPCOMP_ORDER = [
    'parent_avoidant_person_D_p',
    'parent_not_liked_by_others_p',
    'parent_impulsive_p',
    'parent_repeats_acts_p',
    'parent_obsessive_thoughts_p',
]

_PARENT_SOCIAL_IMPCOMP_EDGE = '#1A2332'
_PARENT_SOCIAL_IMPCOMP_BG   = '#FFFFFF'

def _parent_social_impcomp_resolve_color(target_name):
    if target_name in _PARENT_SOCIAL_IMPCOMP_COLORS:
        return _PARENT_SOCIAL_IMPCOMP_COLORS[target_name]
    raw = _PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _PARENT_SOCIAL_IMPCOMP_COLORS:
        return _PARENT_SOCIAL_IMPCOMP_COLORS[raw]
    tl = target_name.lower()
    if 'avoidant' in tl:                                  return _PARENT_SOCIAL_IMPCOMP_COLORS['parent_avoidant_person_D_p']
    if 'not liked' in tl or 'not_liked' in tl:            return _PARENT_SOCIAL_IMPCOMP_COLORS['parent_not_liked_by_others_p']
    if 'compuls' in tl or 'repeats' in tl:                return _PARENT_SOCIAL_IMPCOMP_COLORS['parent_repeats_acts_p']
    if 'obsess' in tl:                                    return _PARENT_SOCIAL_IMPCOMP_COLORS['parent_obsessive_thoughts_p']
    if 'impuls' in tl:                                    return _PARENT_SOCIAL_IMPCOMP_COLORS['parent_impulsive_p']
    return (0.5, 0.5, 0.5)

def _parent_social_impcomp_domain(target_name):
    """Return a short domain key for a parent social/impulsivity/compulsivity target."""
    if target_name in _PARENT_SOCIAL_IMPCOMP_COLORS:
        raw = target_name
    else:
        raw = _PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(target_name, target_name)
    tl = raw.lower()
    if 'avoidant' in tl:                                  return 'avoidant'
    if 'not_liked' in tl or 'not liked' in tl:            return 'social'
    if 'compuls' in tl or 'repeats' in tl:                return 'compulsivity'
    if 'obsess' in tl:                                    return 'obsession'
    if 'impuls' in tl:                                    return 'impulsivity'
    return 'avoidant'

def _parent_social_impcomp_boxstyle(target_name):
    """Boxstyle keyed by domain.
    - avoidant (broad DSM composite): tight rect
    - social / impulsivity (narrow externalising-side items): rounded capsule
    - compulsivity / obsession (narrow OCD items): full pill
    """
    dom = _parent_social_impcomp_domain(target_name)
    if dom in ('compulsivity', 'obsession'):
        return 'round,pad=0.018'
    if dom in ('social', 'impulsivity'):
        return 'round,pad=0.014'
    return 'round,pad=0.008'

def _parent_social_impcomp_edge_for(target_name):
    """Resolve target node edge color."""
    if target_name in _PARENT_SOCIAL_IMPCOMP_TARGET_EDGE:
        return _PARENT_SOCIAL_IMPCOMP_TARGET_EDGE[target_name]
    raw = _PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(target_name)
    if raw and raw in _PARENT_SOCIAL_IMPCOMP_TARGET_EDGE:
        return _PARENT_SOCIAL_IMPCOMP_TARGET_EDGE[raw]
    rgb = _parent_social_impcomp_resolve_color(target_name)
    return tuple(max(0, c * 0.65) for c in rgb)

def _parent_social_impcomp_feature_color(feature, df_sankey, feat_target_count, targets_list):
    """Feature node color. Single-target -> inherit. Multi-target -> steel ramp."""
    n_tgt = feat_target_count.get(feature, 1)
    rows = df_sankey[df_sankey['Feature'] == feature]
    if rows.empty:
        return (0.5, 0.5, 0.5)
    if n_tgt == 1:
        dom_t = rows.loc[rows['Importance'].idxmax(), 'Target']
        return _parent_social_impcomp_resolve_color(dom_t)
    _t = min((n_tgt - 2) / 3.0, 1.0)
    return (0.50 - 0.42 * _t,
            0.52 - 0.42 * _t,
            0.58 - 0.44 * _t)

_PARENT_SOCIAL_IMPCOMP_FEATURE_EDGE_MAP = {
    # Keyed by majority-domain of each feature's incoming ribbons.
    'avoidant':       '#1F2D5C',   # dark blue-violet
    'social':         '#7F2522',   # dark crimson
    'impulsivity':    '#4E7120',   # dark olive-green
    'compulsivity':   '#0D6675',   # dark teal
    'obsession':      '#3D2659',   # dark violet
}

def _parent_social_impcomp_sort_targets(targets_list):
    """Order targets by age DESCENDING (highest age at top, lowest at bottom),
    with R\u00b2 descending as secondary sort for ties within the same age.

    Falls back to pure R\u00b2-descending sort if no age suffixes are present.
    Final fallback is _PARENT_SOCIAL_IMPCOMP_ORDER.
    """
    import re
    _ba = globals().get('_batch_archive', {})
    _tdn = globals().get('TARGET_DISPLAY_NAMES', {})
    _tm = _extract_target_metrics(_ba) if _ba else {}

    def _age_of(t):
        # Parse "(Year N)" suffix (parent self-report, years-from-baseline).
        m = re.search(r'\(Year (\d+)\)$', str(t))
        return int(m.group(1)) if m else -1

    def _r2_for(t):
        tp_form = _year_label_to_tp(t)
        for key in (t, _get_tdn(t), tp_form, _get_tdn(tp_form)):
            entry = _tm.get(key)
            if entry is not None:
                return entry[0]
        raw = (_PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(t)
               or _PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(tp_form))
        if raw:
            entry = _tm.get(raw) or _tm.get(_tdn.get(raw, ''))
            if entry is not None:
                return entry[0]
        return None

    ages   = {t: _age_of(t)  for t in targets_list}
    r2s    = {t: _r2_for(t)  for t in targets_list}
    any_age = any(v >= 0 for v in ages.values())

    if any_age:
        return sorted(
            targets_list,
            key=lambda t: (-ages[t], -(r2s[t] if r2s[t] is not None else -999)),
        )

    has_metrics = any(v is not None for v in r2s.values())
    if has_metrics:
        return sorted(targets_list,
                      key=lambda t: r2s.get(t) if r2s.get(t) is not None else -999,
                      reverse=True)

    def _rank(t):
        if t in _PARENT_SOCIAL_IMPCOMP_ORDER:
            return _PARENT_SOCIAL_IMPCOMP_ORDER.index(t)
        raw = _PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(t)
        if raw and raw in _PARENT_SOCIAL_IMPCOMP_ORDER:
            return _PARENT_SOCIAL_IMPCOMP_ORDER.index(raw)
        return 99
    return sorted(targets_list, key=_rank)

def _draw_sankey_parent_social_impcomp(df_sankey, sankey_top_features, save_dir=None):
    """Parent Social/Impulsivity/Compulsivity Sankey -- 5 ASR-rated targets.

    ASR analog of the child ADHD/Impulsivity/Compulsivity Sankey. Mirrors
    _draw_sankey_child_adhd_impcomp implementation with:
      1. 5-target CUD-safe palette (blue-violet / coral / chartreuse /
         turquoise / violet)
      2. Port-gap 0.003
      3. Saturation + luminance ramp for ribbon importance encoding
      4. 3-tier domain-based target node shape (composite / social-impulsivity
         item / OCD item)
      5. Domain-aware target edge colors
      6. Figure 1.5x scale + 1200 DPI output
      7. Uniform target/feature node heights
    """

    # Relabel any trailing '(TN)' -> '(Year N)' -- years-from-baseline framing.
    # Parent targets are self-reports; the ABCD child-age mapping used by
    # _tp_suffix_to_age is misleading here (ages refer to the child rather
    # than the parent rater). Year-from-baseline preserves longitudinal
    # ordering without imposing a child-age frame.
    df_sankey = df_sankey.copy()
    df_sankey['Target'] = df_sankey['Target'].apply(_tp_suffix_to_year)

    _cd = _sankey_common_data(df_sankey)
    targets_list      = _parent_social_impcomp_sort_targets(_cd['targets_list'])
    features_list     = _cd['features_list']
    feat_target_count = _cd['feat_target_count']
    target_totals     = _cd['target_totals']
    feature_totals    = _cd['feature_totals']
    n_targets         = _cd['n_targets']
    n_features        = _cd['n_features']

    _ba = globals().get('_batch_archive', {})
    _target_metrics = _extract_target_metrics(_ba) if _ba else {}

    _imp_min = df_sankey['Importance'].min() if len(df_sankey) > 0 else 0
    _imp_max = df_sankey['Importance'].max() if len(df_sankey) > 0 else 1
    _norm    = Normalize(vmin=_imp_min, vmax=_imp_max)

    _tc = {t: _parent_social_impcomp_resolve_color(t) for t in targets_list}
    _BG = _PARENT_SOCIAL_IMPCOMP_BG

    _pad      = max(0.022, min(0.040, 1.2 / max(n_features, 1)))
    left_x    = _LEFT_X
    right_x   = _RIGHT_X
    node_w    = 0.052
    _feat_nw  = 0.028
    _target_node_w = 0.036

    _, _base_h = _wide_figure_dims(n_targets, n_features)
    fig_w = 15 * 1.5
    # -- Adaptive sizing (scales to n_targets so 5x3-year batches don't crowd) --
    # Extra inches of figure height for tall batches (n_targets > 3).
    _tall_bonus = max(0, (n_targets - 3) * 0.55)
    fig_h = (max(_base_h * 1.3, n_features * 0.52, 22) + _tall_bonus) * 1.5
    # More vertical breathing room as n_targets grows.
    _spread_t = 2.2 + 0.10 * max(0, n_targets - 3)
    _spread_f = 1.9
    fig_w, fig_h, _spread_t, _spread_f = _landscape_adjust(
        fig_w, fig_h, _spread_t, _spread_f, n_targets, n_features)
    _lf = _landscape_fonts()
    # Target font: 17 at n_targets <= 3 (matches child sankey); shrinks
    # smoothly to ~10 at n_targets = 15. R2 sub-label remains readable.
    _target_fs = max(10, min(17, int(round(51 / max(n_targets, 3)))))

    global_max  = max(target_totals.max(), feature_totals.max()) if len(target_totals) > 0 else 1.0
    _max_node_h = min(0.18, 3.5 / max(n_features, 1))

    target_positions = _compute_node_positions_enhanced(
        targets_list, target_totals, global_max, _pad,
        min_node_h=0.050, max_node_h=_max_node_h, spread_factor=_spread_t)
    feature_positions = _compute_node_positions_enhanced(
        features_list, feature_totals, global_max, _pad,
        min_node_h=0.025, max_node_h=_max_node_h, spread_factor=_spread_f)
    target_positions, feature_positions = _equalize_vertical_envelopes(
        target_positions, feature_positions)

    # Adaptive rect height: 0.20 at n_targets <= 3 (matches child sankey);
    # shrinks for tall batches so the column doesn't visually overflow.
    _RECT_H = min(0.20, 0.85 / max(n_targets, 1))
    for t in targets_list:
        if t in target_positions:
            _c = target_positions[t]['y_centre']
            target_positions[t] = {
                'y_centre': _c, 'y_top': _c + _RECT_H / 2,
                'y_bot': _c - _RECT_H / 2, 'h': _RECT_H}

    # -- Year-group dividers: extra spacing between year clusters --------------
    # When the same target set repeats across multiple years (e.g. 5 targets
    # x 3 years = 15 rows), inject an extra vertical gap between year groups
    # so the eye can parse the structure. Only triggers for n_targets >= 6.
    if n_targets >= 6:
        import re as _re_yg
        def _year_of_label_yg(t):
            m = _re_yg.search(r'\(Year (\d+)\)$', str(t))
            return int(m.group(1)) if m else None
        _gap_extra = _RECT_H * 0.55  # 55% of rect height as group separator
        _shift     = 0.0
        _prev_year = None
        for idx, t in enumerate(targets_list):
            yr = _year_of_label_yg(t)
            if t in target_positions:
                if (idx > 0 and _prev_year is not None and yr is not None
                        and yr != _prev_year):
                    _shift -= _gap_extra
                p = target_positions[t]
                target_positions[t] = {
                    'y_centre': p['y_centre'] + _shift,
                    'y_top':    p['y_top']    + _shift,
                    'y_bot':    p['y_bot']    + _shift,
                    'h':        p['h']}
                _prev_year = yr

    _FEAT_H = 0.056
    for f_key in features_list:
        if f_key in feature_positions:
            _c = feature_positions[f_key]['y_centre']
            feature_positions[f_key] = {
                'y_centre': _c, 'y_top': _c + _FEAT_H / 2,
                'y_bot': _c - _FEAT_H / 2, 'h': _FEAT_H}

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.set_facecolor(_BG)
    ax.set_facecolor(_BG)

    all_y = ([p['y_bot'] for p in target_positions.values()] +
             [p['y_bot'] for p in feature_positions.values()])
    y_min = min(all_y) - 0.04
    ax.set_xlim(-0.08, right_x + 0.22)
    ax.set_ylim(y_min, 1.06)
    ax.axis('off')

    _PORT_GAP = 0.003

    for rb in _iterate_ribbon_ports(df_sankey, targets_list, features_list,
                                    target_positions, feature_positions,
                                    target_totals, feature_totals):
        _rgb  = _tc.get(rb['target'], (0.5, 0.5, 0.5))
        rel   = _norm(rb['importance'])

        _yl_t = rb['y_l_top'] - _PORT_GAP
        _yl_b = rb['y_l_bot'] + _PORT_GAP
        _yr_t = rb['y_r_top'] - _PORT_GAP * 0.5
        _yr_b = rb['y_r_bot'] + _PORT_GAP * 0.5
        if _yl_t <= _yl_b or _yr_t <= _yr_b:
            _yl_t, _yl_b = rb['y_l_top'], rb['y_l_bot']
            _yr_t, _yr_b = rb['y_r_top'], rb['y_r_bot']

        _desat = 0.55 + 0.45 * rel
        _white = (0.92, 0.93, 0.96)
        _fill  = tuple(_white[i] + (_rgb[i] - _white[i]) * _desat for i in range(3))
        _alpha = 0.55 + 0.30 * rel

        if rel > 0.45:
            _glow_n = 2 if rel > 0.75 else 1
            _draw_bezier_ribbon_enhanced(
                ax, left_x, node_w, right_x,
                _yl_t + 0.005, _yl_b - 0.005,
                _yr_t + 0.003, _yr_b - 0.003,
                color_rgba=_rgb + (0.08,), zorder=1,
                glow_layers=_glow_n)

        if rel > 0.70:
            _edge_dark = tuple(max(0, c * 0.70) for c in _rgb)
            _e_lw    = 0.35 + 0.25 * (rel - 0.70) / 0.30
            _e_alpha = 0.22 + 0.18 * (rel - 0.70) / 0.30
            _edge_c  = _edge_dark + (_e_alpha,)
        else:
            _edge_c = _fill + (min(_alpha + 0.05, 0.65),)
            _e_lw   = 0.2

        _draw_bezier_ribbon_enhanced(
            ax, left_x, node_w, right_x,
            _yl_t, _yl_b, _yr_t, _yr_b,
            color_rgba=_fill + (_alpha,),
            zorder=2,
            edge_color=_edge_c, edge_lw=_e_lw,
            right_node_w=_feat_nw)

    for t, pos in target_positions.items():
        rgb = _tc.get(t, (0.5, 0.5, 0.5))
        _edge_col = _parent_social_impcomp_edge_for(t)
        _bstyle   = _parent_social_impcomp_boxstyle(t)

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w + 0.004, pos['y_bot'] - 0.003),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        ax.add_patch(FancyBboxPatch(
            (left_x - _target_node_w, pos['y_bot']),
            _target_node_w * 2, pos['h'],
            boxstyle=_bstyle,
            facecolor=rgb, edgecolor=_edge_col,
            linewidth=1.8, zorder=3))

        raw_key = t if t in _PARENT_SOCIAL_IMPCOMP_DOMAIN_CLEAN_LABELS else \
                  _PARENT_SOCIAL_IMPCOMP_DISPLAY_ALIASES.get(t, t)
        _domain_label = _PARENT_SOCIAL_IMPCOMP_DOMAIN_CLEAN_LABELS.get(raw_key, _get_tdn(t))

        _tp_form    = _year_label_to_tp(t)  # reverse-map for metric-dict lookup
        _metric_val = (_target_metrics.get(t)
                       or _target_metrics.get(_get_tdn(t))
                       or _target_metrics.get(_tp_form)
                       or _target_metrics.get(_get_tdn(_tp_form)))
        _label_x = left_x - _target_node_w - 0.04

        if _metric_val is not None:
            _v, _l = _metric_val
            txt = ax.text(_label_x, pos['y_centre'] + 0.025,
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
            mtxt = ax.text(_label_x, pos['y_centre'] - 0.028,
                           f'{_l} = {_v:.3f}',
                           ha='right', va='center',
                           fontsize=_target_fs - 3, fontweight='bold',
                           color=_edge_col, fontfamily=_TITLE_FONT,
                           zorder=4, clip_on=False)
            mtxt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])
        else:
            txt = ax.text(_label_x, pos['y_centre'],
                          _domain_label, ha='right', va='center',
                          fontsize=_target_fs, fontweight='bold',
                          color='#1a1a1a', fontfamily=_TITLE_FONT,
                          zorder=4, clip_on=False)
            txt.set_path_effects([path_effects.withStroke(
                linewidth=2.5, foreground=_BG)])

    _fs_feat = max(8, min(_lf[1], 400 / max(n_features, 1)))
    for _fi, (f, pos) in enumerate(feature_positions.items()):
        n_tgt = feat_target_count.get(f, 1)
        _fc = _parent_social_impcomp_feature_color(f, df_sankey, feat_target_count, targets_list)

        _f_rows = df_sankey[df_sankey['Feature'] == f]
        _domain_counts = {}
        for _ft in _f_rows['Target']:
            _dom_key = _parent_social_impcomp_domain(_ft)
            _domain_counts[_dom_key] = _domain_counts.get(_dom_key, 0) + 1
        _dom_winner = (max(_domain_counts, key=_domain_counts.get)
                       if _domain_counts else 'avoidant')
        _feat_edge = _PARENT_SOCIAL_IMPCOMP_FEATURE_EDGE_MAP.get(_dom_winner, '#1A1A1A')

        if n_tgt >= 2:
            dom_rows = df_sankey[df_sankey['Feature'] == f]
            dom_t = (dom_rows.loc[dom_rows['Importance'].idxmax(), 'Target']
                     if not dom_rows.empty else targets_list[0])
            glow_col = _tc.get(dom_t, (0.5, 0.5, 0.5))
            ax.add_patch(FancyBboxPatch(
                (right_x - _feat_nw - 0.006, pos['y_bot'] - 0.006),
                _feat_nw * 2 + 0.012, pos['h'] + 0.012,
                boxstyle='round,pad=0.008',
                facecolor='none', edgecolor=glow_col + (0.38,),
                linewidth=1.3, zorder=2))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw + 0.004, pos['y_bot'] - 0.003),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor='#000000', edgecolor='none', linewidth=0,
            zorder=2, alpha=0.14))

        ax.add_patch(FancyBboxPatch(
            (right_x - _feat_nw, pos['y_bot']),
            _feat_nw * 2, pos['h'],
            boxstyle='round,pad=0.010',
            facecolor=_fc, edgecolor=_feat_edge,
            linewidth=0.7, zorder=3))

        _disp_name = DISPLAY_NAMES.get(f, f) if 'DISPLAY_NAMES' in dir() else f
        badge = f' ({n_tgt}T)' if n_tgt >= 2 else ''
        fw = 'bold' if n_tgt >= 2 else 'normal'
        txt = ax.text(right_x + _feat_nw + 0.025, pos['y_centre'],
                      f'{_disp_name}{badge}',
                      ha='left', va='center', fontsize=_fs_feat, fontweight=fw,
                      color='#1a1a1a', fontfamily=_TITLE_FONT,
                      zorder=4, clip_on=False)
        txt.set_path_effects([path_effects.withStroke(
            linewidth=2.5, foreground=_BG)])

    _hdr_y = max(
        max(p['y_top'] for p in target_positions.values()),
        max(p['y_top'] for p in feature_positions.values()),
    ) + 0.035
    _hdr_tgt_col  = '#0A1F4D'
    _hdr_feat_col = '#1E293B'
    ax.text(left_x, _hdr_y, 'TARGETS (PARENT: AVOIDANT / SOCIAL / IMPULSIVITY / COMPULSIVITY / OBSESSIONS)',
            ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_tgt_col,
            fontfamily=_TITLE_FONT)
    ax.text(right_x, _hdr_y, 'FEATURES', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=_hdr_feat_col,
            fontfamily=_TITLE_FONT)
    _rule_hw = 0.10
    ax.plot([left_x - _rule_hw, left_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_tgt_col, lw=0.5, alpha=0.35, zorder=5)
    ax.plot([right_x - _rule_hw, right_x + _rule_hw],
            [_hdr_y - 0.004, _hdr_y - 0.004],
            color=_hdr_feat_col, lw=0.5, alpha=0.35, zorder=5)

    ax.text(0.50, 1.085, ' ', ha='center', va='bottom', fontsize=17,
            transform=ax.transAxes, alpha=0.0)

    _sankey_print_summary(df_sankey, feat_target_count, n_targets, n_features)
    plt.tight_layout(rect=[0.01, 0.01, 0.99, 0.97])

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), 'manuscript_figures')
    saved = _safe_save_fig(fig, save_dir, 'sankey_parent_social_impcomp',
                           dpi=1200, formats=('png', 'pdf'))
    plt.show()
    for p in saved:
        print(f"  Saved -> {p}")
    return fig


# ---
# SANKEY STYLE-SPECIFIC EXCLUSIONS
# ---
# Visualization-only exclusions: targets listed here are kept out of the
# Sankey figure for the given style, but are NOT removed from _batch_archive
# and continue to be part of the main analyses. Add both the raw key and the
# display name form (what appears in df_sankey['Target']) to catch either.
# Edit this dict to customize which targets get hidden per style.
_SANKEY_EXCLUDE_TARGETS_BY_STYLE = {
    'child_deltas': {
        # Remove delta_family_conflict_ss_k from the Child Only Deltas Sankey
        'delta_family_conflict_ss_k',
        'Change in Fam. Conflict (Y)',
    },
    'child_academic_cognitive': {
        # Remove delta_bad_grades (all TPs) from the Academic/Cognitive Sankey.
        # ML run still trains/evaluates these -- they simply don't render here.
        # Variants are listed because exclusion matches df_sankey['Target'] by
        # exact string, and _get_tdn appends '(TN)' for TP-sweep archive keys.
        'delta_bad_grades',
        'delta_bad_grades (T0)',
        'delta_bad_grades (T1)',
        'delta_bad_grades (T2)',
        'delta_bad_grades (T3)',
        'delta_bad_grades (T4)',
    },
    # Add other style -> set-of-targets entries here as needed.
}

# ---
# EXECUTE SANKEY
# ---

_has_batch = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) >= 1
)

if not _has_batch:
    print("Warning: No _batch_archive found. Run Cell 6 in batch mode first.")
else:
    print(f"_batch_archive: {len(_batch_archive)} entries")
    for _k in _batch_archive:
        print(f"   \u2022 {_k}")

    if show_sankey:
        print("\n" + "=" * 80)
        print("  ENHANCED SANKEY \u2014 Transdiagnostic Feature Flow")
        print(f"     Style: {sankey_style}")
        if sankey_landscape:
            print("     Mode:  LANDSCAPE")
        print("=" * 80)

        _sankey_rows = _extract_sankey_data(_batch_archive, top_n=sankey_top_features)

        if not _sankey_rows:
            print("  Warning: No consensus_df data found in _batch_archive.")
            print("     Ensure visualization_mode includes consensus in Cell 6.")
        else:
            df_sankey = pd.DataFrame(_sankey_rows)

            # -- Apply style-specific target exclusions (visualization-only) --
            _excludes = _SANKEY_EXCLUDE_TARGETS_BY_STYLE.get(sankey_style, set())
            if _excludes:
                _present = set(df_sankey['Target'].unique()) & _excludes
                if _present:
                    _before_n = len(df_sankey)
                    df_sankey = df_sankey[~df_sankey['Target'].isin(_excludes)].reset_index(drop=True)
                    _dropped = _before_n - len(df_sankey)
                    print(f"  Note: Style '{sankey_style}' excludes {sorted(_present)} "
                          f"({_dropped} feature rows filtered; targets remain in _batch_archive)")

            # -- Diagnostic: verify per-target feature counts --
            _per_target = df_sankey.groupby('Target')['Feature'].count()
            print(f"  sankey_top_features = {sankey_top_features}")
            print(f"  Total unique features on figure: {df_sankey['Feature'].nunique()}")
            for _tgt, _cnt in _per_target.items():
                _flag = '  WARN' if _cnt != sankey_top_features else '  ok'
                print(f"  {_flag} {_tgt}: {_cnt} features")

            if sankey_style == 'run_all_styles':
                _run_all_styles(df_sankey, sankey_top_features,
                               save_dir=os.path.join(os.getcwd(), 'manuscript_figures'))
            else:
                _draw_sankey(df_sankey, sankey_top_features,
                            save_dir=os.path.join(os.getcwd(), 'manuscript_figures'),
                            style=sankey_style)
                print(f"  Sankey rendered with style: '{sankey_style}'")
    else:
        print("Sankey display disabled (show_sankey = False).")

In [ ]:
#@title Extract Top Predictive Features from ML: Correlation Matrix, Partial Correlation Network, and Dendrogram

# EBICglasso partial correlation networks, Pearson correlation matrices,
# and hierarchical clustering dendrograms extracted from top ML-predictive features
# for active targets run above.
# Fiedler eigenvector layout positions nodes to reflect network community
# structure. Optional bootstrap CI (edge stability) and case-dropping
# centrality stability analyses.

# ---
# UI CONTROLS -- toggle expensive bootstrap analyses on/off
# ---
run_bootstrap_CI         = False  #@param {type:"boolean"}
run_centrality_stability = False  #@param {type:"boolean"}
n_bootstrap              = 1000   #@param {type:"slider", min:100, max:5000, step:100}

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
from matplotlib.patches import Wedge, Circle
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
# replaced GraphicalLassoCV with graphical_lasso for EBICglasso
from sklearn.covariance import graphical_lasso
from sklearn.manifold import MDS
from sklearn.linear_model import LinearRegression
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

import sklearn, networkx as _nx_ver
print(f"  Software: sklearn={sklearn.__version__}, "
      f"networkx={_nx_ver.__version__}, matplotlib={matplotlib.__version__}")

try:
    _RS['section'] = 'cross_category'
except Exception:
    pass  # non-critical; logged upstream or optional visualization

# ---
# 0. STYLE CONSTANTS
# ---

_CLR_SLATE    = '#1F2D3D'
_CLR_CHARCOAL = '#34495E'
_CLR_STEEL    = '#546E7A'
_CLR_SILVER   = '#BDC3C7'
_CLR_CLOUD    = '#EEF2F7'

_TXT_SECONDARY = '#4A5568'
_TXT_MUTED     = '#607D8B'
_TXT_HEADER    = '#546E7A'
_ARROW_CLR    = '#78909C'

_EDGE_POS = '#0072B2'
# Okabe-Ito vermillion
_EDGE_NEG = '#D55E00'

# module-level dash constant -- was (0,(5,3)) in draw, (0,(4,2)) in legend
_DASH_NEG = (0, (5, 3))

# border colours swapped for directional consistency with edges
# Positive EI → blue family (matches _EDGE_POS blue = positive partial r)
# Negative EI → warm/orange family
_BORDER_POS  = '#2E86AB'
_BORDER_NEG  = '#E67E22'
_BORDER_ZERO = '#BDC3C7'

_TARGET_CLR = '#C0392B'
_RING_UNFILLED = '#E0E0E0'

# _EDGE_MAXIMUM is now actively wired into magnitude-based rendering
_EDGE_MAXIMUM = 1.0

_SHADOW_ALPHA  = 0.09
_SHADOW_SCALE  = 1.04
_CURVE_STRENGTH = 0.12  # kept as fallback; _draw_fancy_edge uses adaptive version

# -- Network panel background ---
_NET_BG        = '#F8F7F3'
_GLOW_SPREAD_1 = 5.5
_GLOW_SPREAD_2 = 2.8
_GLOW_ALPHA_1  = 0.09
_GLOW_ALPHA_2  = 0.22
# reduced from 0.20 for better edge contrast on warm background
_EDGE_TINT_F   = 0.12

# magnitude-based edge width and opacity range
# Wider range than the old 2.0-2.6 rank-based band: Epskamp/qgraph default ≈ 0.5-5pt
_LW_MIN    = 0.6
_LW_MAX    = 4.0
_ALPHA_MIN = 0.12
_ALPHA_MAX = 0.85

# EBICglasso hyperparameter (Epskamp, Borsboom & Fried 2018)
# γ = 1.0 is the conservative alternative to the 0.5 default; produces sparser
# networks where retained edges are more robust. Well-cited in Foygel & Drton
# (2010) and routinely used by Fried's group for robustness checks.
_EBIC_GAMMA    = 1.0
_EBIC_N_ALPHA  = 100   # lambda grid size for main analysis
_EBIC_N_ALPHA_BOOT = 50  # reduced grid for bootstrap (speed)

# Visualization-only display threshold (qgraph `minimum` equivalent).
# Edges with |partial r| below this are omitted from the figure for visual
# clarity; the full EBICglasso adjacency matrix is preserved in the pcor_df
# and available in supplement. At n ≈ 1000, |pcor| < 0.10 has very modest
# practical effect size.
_DISPLAY_THRESH = 0.07

_DOMAIN_COLORS = {
    'Youth Behavioral':      '#4A90D9',
    'Parent Report':         '#E67E22',
    'Family/Social':         '#27AE60',
    'Cognitive/Academic':    '#8E44AD',
    'Neuroimaging':          '#E74C3C',
    'Physical/Medical':      '#16A085',
    'Demographic/Other':     '#7F8C8D',
    'Change Scores':         '#2C3E50',
    'Sleep/Screen':          '#F39C12',
    'SES/Environment':       '#1ABC9C',
}
_DOMAIN_FALLBACK = '#7F8C8D'

_CAT_COLORS = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2',
    '#D55E00', '#CC79A7', '#999999', '#44AA99', '#882255',
    '#DDCC77', '#117733', '#332288', '#AA4499', '#88CCEE',
    '#661100', '#6699CC', '#BBBBBB', '#275D8E', '#AE5F3C',
]

_CORR_CMAP = LinearSegmentedColormap.from_list(
    'bwr_accessible',
    ['#2166AC', '#4393C3', '#92C5DE', '#D1E5F0',
     '#F7F7F7',
     '#FDDBC7', '#F4A582', '#D6604D', '#B2182B'], N=256)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Inter', 'Arial', 'DejaVu Sans'],
    'font.size': 10,
    'font.weight': 'regular',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# ---
# 0B. HELPER FUNCTIONS
# ---

# ---
# EBICglasso -- replaces GraphicalLassoCV throughout
# ---

def _ebic_glasso(X_scaled, gamma=_EBIC_GAMMA, n_alpha=_EBIC_N_ALPHA,
                 alpha_min_ratio=0.01, max_iter=1000):
    """
    EBICglasso: select GLASSO regularization parameter by minimizing the
    Extended Bayesian Information Criterion (Foygel & Drton 2010).

    This is the standard estimation method in psychological network analysis
    (Epskamp & Fried 2018; Burger et al. 2022).

    Parameters
    X_scaled : ndarray (n, p) -- standardised data
    gamma    : float -- EBIC hyperparameter (0 = BIC, 0.5 = default, 1 = conservative)
    n_alpha  : int -- number of lambda values in the grid
    alpha_min_ratio : float -- ratio of min to max lambda
    max_iter : int -- GLASSO iterations per lambda

    Returns
    best_prec  : ndarray (p, p) or None
    best_cov   : ndarray (p, p) or None
    best_alpha : float or None
    gamma      : float -- the γ used (for reporting)
    """
    n, p = X_scaled.shape
    S = np.cov(X_scaled, rowvar=False, bias=False)  # sample covariance

    # Build lambda grid: from max (fully sparse) to min (dense)
    alpha_max = np.max(np.abs(S - np.diag(np.diag(S))))
    if alpha_max < 1e-10:
        alpha_max = 1.0
    alpha_min = alpha_max * alpha_min_ratio
    alphas = np.logspace(np.log10(alpha_max), np.log10(alpha_min), n_alpha)

    best_ebic  = np.inf
    best_prec  = None
    best_cov   = None
    best_alpha = None

    for alpha in alphas:
        try:
            cov_, prec_ = graphical_lasso(S, alpha=alpha, max_iter=max_iter,
                                          mode='cd')
            # Log-likelihood (Gaussian)
            sign, logdet = np.linalg.slogdet(prec_)
            if sign <= 0:
                continue
            ll = logdet - np.trace(S @ prec_)

            # Number of non-zero off-diagonal unique edges
            upper = np.abs(prec_[np.triu_indices(p, k=1)])
            E = int(np.sum(upper > 1e-10))

            # EBIC (Foygel & Drton 2010)
            ebic = -n * ll + E * np.log(n) + 4.0 * gamma * E * np.log(p)

            if ebic < best_ebic:
                best_ebic  = ebic
                best_prec  = prec_
                best_cov   = cov_
                best_alpha = alpha
        except Exception:
            continue

    return best_prec, best_cov, best_alpha, gamma

def _draw_bezier_edge(ax, p1, p2, color, lw, alpha, linestyle='-', edge_idx=0):
    """Original flat-colour Bézier edge -- kept for bootstrap CI figure."""
    mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
    dx, dy = p2[0]-p1[0], p2[1]-p1[1]
    length = max(np.sqrt(dx**2 + dy**2), 1e-6)
    sign = 1 if edge_idx % 2 == 0 else -1
    curve_offset = _CURVE_STRENGTH * length * sign
    ctrl_x = mx - (dy / length) * curve_offset
    ctrl_y = my + (dx / length) * curve_offset
    verts = [p1, (ctrl_x, ctrl_y), p2]
    codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3]
    path = Path(verts, codes)
    if linestyle == '-':
        patch = PathPatch(path, facecolor='none', edgecolor=color,
                          lw=lw, alpha=alpha, capstyle='round', zorder=1)
    else:
        patch = PathPatch(path, facecolor='none', edgecolor=color,
                          lw=lw, alpha=alpha, capstyle='round',
                          linestyle=linestyle, zorder=1)
    ax.add_patch(patch)
    return (ctrl_x, ctrl_y)

def _adaptive_curve_strength(p1, p2, data_span=12.0, min_cs=0.06, max_cs=0.22):
    """
    Returns a curve_strength value that scales with chord length relative to
    the overall layout data span.
      Short edges → low curvature  (clean, unobtrusive, avoids tight loops)
      Long edges  → higher curvature (visually separates arc from chord,
                                      reduces overlap with other edges)
    """
    dist = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
    norm = np.clip(dist / (data_span + 1e-9), 0.0, 1.0)
    return min_cs + (max_cs - min_cs) * norm

def _draw_fancy_edge(ax, p1, p2, color1, color2, base_color,
                     lw, base_alpha, linestyle='-',
                     edge_idx=0, is_strong=False,
                     data_span=12.0, curve_sign=None):
    """
    Tinted + selective-glow Bézier edge for the warm off-white network panel.
    Uses _adaptive_curve_strength() for per-edge curvature.

    curve_sign overrides edge_idx-based alternation when provided.
    For circle layouts, caller passes geometry-based sign so edges always curve
    outward from the layout center.
    """
    mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
    dx, dy = p2[0]-p1[0], p2[1]-p1[1]
    length  = max(np.sqrt(dx**2+dy**2), 1e-6)
    if curve_sign is not None:
        sign = curve_sign
    else:
        sign = 1 if edge_idx % 2 == 0 else -1

    cs_val  = _adaptive_curve_strength(p1, p2, data_span=data_span)
    ctrl_x  = mx - (dy/length) * cs_val * length * sign
    ctrl_y  = my + (dx/length) * cs_val * length * sign

    c_base  = np.array(matplotlib.colors.to_rgba(base_color)[:3])
    c_n1    = np.array(matplotlib.colors.to_rgba(color1)[:3])
    c_n2    = np.array(matplotlib.colors.to_rgba(color2)[:3])
    c_node  = (c_n1 + c_n2) / 2.0
    c_tint  = np.clip((1.0 - _EDGE_TINT_F) * c_base +
                       _EDGE_TINT_F * c_node, 0.0, 1.0)

    def _bpath():
        return Path([(p1[0],p1[1]),(ctrl_x,ctrl_y),(p2[0],p2[1])],
                    [Path.MOVETO, Path.CURVE3, Path.CURVE3])

    if is_strong:
        _glow_layers = [
            (7.0,  0.04),
            (5.0,  0.07),
            (3.5,  0.11),
            (2.2,  0.16),
            (1.4,  0.22),
        ]
        for _glw_mult, _galp in _glow_layers:
            ax.add_patch(PathPatch(_bpath(), facecolor='none',
                                   edgecolor=tuple(c_tint),
                                   lw=lw * _glw_mult, alpha=_galp,
                                   capstyle='round', zorder=1))

    ax.add_patch(PathPatch(_bpath(), facecolor='none',
                           edgecolor=tuple(c_tint),
                           lw=lw, alpha=base_alpha,
                           capstyle='round', linestyle=linestyle, zorder=2))

    return (ctrl_x, ctrl_y)

# bootstrap now uses _ebic_glasso instead of GraphicalLassoCV
def _bootstrap_edge_weights(X_raw, feat_names, n_boot=1000, seed=42,
                            ci_level=0.95, gamma=_EBIC_GAMMA):
    rng = np.random.RandomState(seed)
    n, p = X_raw.shape
    edge_samples = np.full((n_boot, p, p), np.nan)
    n_converged = 0
    for b in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        Xb = StandardScaler().fit_transform(X_raw[idx])
        try:
            prec, _, _, _ = _ebic_glasso(Xb, gamma=gamma,
                                          n_alpha=_EBIC_N_ALPHA_BOOT,
                                          max_iter=1000)
            if prec is None:
                continue
            d = np.sqrt(np.diag(prec))
            pcor = -prec / np.outer(d, d)
            np.fill_diagonal(pcor, 1.0)
            edge_samples[b] = pcor
            n_converged += 1
        except Exception:
            pass  # non-critical; logged upstream or optional visualization
    alpha = 1.0 - ci_level
    ci_lo = np.nanpercentile(edge_samples, 100 * alpha / 2, axis=0)
    ci_hi = np.nanpercentile(edge_samples, 100 * (1 - alpha / 2), axis=0)
    median = np.nanmedian(edge_samples, axis=0)
    # inclusion = proportion of bootstrap samples where
    # EBICglasso retains the edge (i.e. |pcor| > 1e-6 after regularisation).
    # With EBICglasso (γ=0.5), zero edges are truly zero from regularisation,
    # so this threshold captures the meaningful sparsity pattern.
    inclusion = np.nanmean(np.abs(edge_samples) > 1e-6, axis=0)
    return {
        'edge_samples': edge_samples,
        'ci_lower': pd.DataFrame(ci_lo, index=feat_names, columns=feat_names),
        'ci_upper': pd.DataFrame(ci_hi, index=feat_names, columns=feat_names),
        'inclusion_prob': pd.DataFrame(inclusion, index=feat_names, columns=feat_names),
        'median': pd.DataFrame(median, index=feat_names, columns=feat_names),
        'n_converged': n_converged,
    }

# centrality stability now uses _ebic_glasso
# centrality funcs use thresholded graph
def _centrality_stability(X_raw, feat_names, centrality_func,
                          n_boot=1000, seed=42, cor_threshold=0.7,
                          gamma=_EBIC_GAMMA):
    from scipy.stats import spearmanr
    rng = np.random.RandomState(seed)
    n, p = X_raw.shape
    try:
        X_full_sc = StandardScaler().fit_transform(X_raw)
        prec, _, _, _ = _ebic_glasso(X_full_sc, gamma=gamma,
                                      n_alpha=_EBIC_N_ALPHA,
                                      max_iter=1000)
        if prec is None:
            return {'CS': np.nan, 'proportions': [], 'avg_cors': [], 'sd_cors': []}
        d = np.sqrt(np.diag(prec))
        pcor = -prec / np.outer(d, d)
        np.fill_diagonal(pcor, 1.0)
        pcor_full = pd.DataFrame(pcor, index=feat_names, columns=feat_names)
    except Exception:
        return {'CS': np.nan, 'proportions': [], 'avg_cors': [], 'sd_cors': []}
    cent_full = centrality_func(pcor_full, feat_names)
    cent_full_arr = np.array([cent_full[f] for f in feat_names])
    proportions = np.arange(0.10, 0.80, 0.05)
    avg_cors = []
    sd_cors  = []
    _bpp = min(n_boot, 200)
    for prop in proportions:
        subset_n = max(int(n * (1 - prop)), p + 2)
        if subset_n <= p + 1:
            avg_cors.append(np.nan); sd_cors.append(np.nan); continue
        cors = []
        for _ in range(_bpp):
            idx = rng.choice(n, size=subset_n, replace=False)
            Xsub = StandardScaler().fit_transform(X_raw[idx])
            try:
                prec_s, _, _, _ = _ebic_glasso(Xsub, gamma=gamma,
                                                n_alpha=_EBIC_N_ALPHA_BOOT,
                                                max_iter=1000)
                if prec_s is None:
                    continue
                d_s = np.sqrt(np.diag(prec_s))
                pcor_s = -prec_s / np.outer(d_s, d_s)
                np.fill_diagonal(pcor_s, 1.0)
                pcor_sub = pd.DataFrame(pcor_s, index=feat_names, columns=feat_names)
                cent_sub = centrality_func(pcor_sub, feat_names)
                cent_sub_arr = np.array([cent_sub[f] for f in feat_names])
                r, _ = spearmanr(cent_full_arr, cent_sub_arr)
                cors.append(r)
            except Exception:
                pass  # non-critical; logged upstream or optional visualization
        avg_cors.append(np.mean(cors) if cors else np.nan)
        sd_cors.append(np.std(cors) if len(cors) > 1 else np.nan)
    cs_val = 0.0
    for prop, avg_r in zip(proportions, avg_cors):
        if not np.isnan(avg_r) and avg_r >= cor_threshold:
            cs_val = prop
    return {'CS': cs_val, 'proportions': list(proportions),
            'avg_cors': avg_cors, 'sd_cors': sd_cors}

# ---
# EI and Strength computed from thresholded graph G
# (Robinaugh et al. 2016 definition: sum of edge weights in estimated network)
# ---

def _compute_EI_from_graph(G, feat_names):
    """Expected Influence from thresholded network G (Robinaugh et al. 2016).
    Sums signed edge weights of retained (non-zero) edges only."""
    import networkx as nx
    return {f: float(sum(G[f][nb]['weight'] for nb in G.neighbors(f)))
            if f in G else 0.0 for f in feat_names}

def _compute_EI_thresholded(pcor_df, feat_names, thresh=1e-6):
    """Build a temporary thresholded graph from pcor_df, then compute EI.
    Used by centrality stability where the full graph object is not available."""
    import networkx as nx
    G_tmp = nx.Graph()
    for f in feat_names:
        G_tmp.add_node(f)
    n = len(feat_names)
    for i in range(n):
        for j in range(i + 1, n):
            pc = pcor_df.iloc[i, j]
            if abs(pc) > thresh:
                G_tmp.add_edge(feat_names[i], feat_names[j], weight=pc)
    return _compute_EI_from_graph(G_tmp, feat_names)

def _compute_strength_from_graph(G, feat_names):
    """Node strength from thresholded network G (sum of |edge weights|)."""
    import networkx as nx
    return {f: float(sum(abs(G[f][nb]['weight']) for nb in G.neighbors(f)))
            if f in G else 0.0 for f in feat_names}

def _compute_strength_thresholded(pcor_df, feat_names, thresh=1e-6):
    """Build a temporary thresholded graph from pcor_df, then compute strength."""
    import networkx as nx
    G_tmp = nx.Graph()
    for f in feat_names:
        G_tmp.add_node(f)
    n = len(feat_names)
    for i in range(n):
        for j in range(i + 1, n):
            pc = pcor_df.iloc[i, j]
            if abs(pc) > thresh:
                G_tmp.add_edge(feat_names[i], feat_names[j], weight=pc)
    return _compute_strength_from_graph(G_tmp, feat_names)

def _fiedler_circle_layout(G, scale=6.5):
    """
    Place nodes on a circle ordered by the Fiedler vector (second eigenvector
    of the normalised graph Laplacian).  Structurally similar nodes end up
    adjacent, reducing edge crossings and exposing community structure.

    Falls back to plain circular layout if the graph has < 3 nodes or if
    the eigensolver fails.

    A lightweight 2-opt pass is then run to further reduce edge crossings.
    """
    import numpy as np
    import networkx as nx
    nodes = list(G.nodes())
    n = len(nodes)
    if n < 3:
        return nx.circular_layout(G, scale=scale)

    try:
        idx = {nd: i for i, nd in enumerate(nodes)}
        A = np.zeros((n, n))
        for u, v, d in G.edges(data=True):
            w = abs(d.get('weight', 1.0))
            A[idx[u], idx[v]] = w
            A[idx[v], idx[u]] = w

        deg = A.sum(axis=1)
        deg_safe = np.where(deg > 0, deg, 1.0)
        D_inv_sqrt = np.diag(1.0 / np.sqrt(deg_safe))
        L_norm = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt

        eigvals, eigvecs = np.linalg.eigh(L_norm)
        fiedler = eigvecs[:, 1]

        order = np.argsort(fiedler)
        ordered_nodes = [nodes[i] for i in order]

        def _count_crossings(seq):
            pos_map = {nd: i for i, nd in enumerate(seq)}
            edges = list(G.edges())
            crossings = 0
            for k in range(len(edges)):
                a, b = pos_map[edges[k][0]], pos_map[edges[k][1]]
                if a > b: a, b = b, a
                for l in range(k + 1, len(edges)):
                    c, d = pos_map[edges[l][0]], pos_map[edges[l][1]]
                    if c > d: c, d = d, c
                    if a < c < b < d or c < a < d < b:
                        crossings += 1
            return crossings

        improved = True
        best_seq = ordered_nodes[:]
        best_cross = _count_crossings(best_seq)
        iterations = 0
        while improved and iterations < 60:
            improved = False
            iterations += 1
            for i in range(n - 1):
                for j in range(i + 1, n):
                    candidate = best_seq[:i] + best_seq[i:j+1][::-1] + best_seq[j+1:]
                    c = _count_crossings(candidate)
                    if c < best_cross:
                        best_seq, best_cross = candidate, c
                        improved = True

        ordered_nodes = best_seq

        ei_vals = nx.get_node_attributes(G, 'ei') if nx.get_node_attributes(G, 'ei') else {}
        if ei_vals:
            top_ei_node = max(ei_vals, key=lambda nd: ei_vals[nd])
        else:
            top_ei_node = best_seq[0]
        if top_ei_node in best_seq:
            rot = best_seq.index(top_ei_node)
            best_seq = best_seq[rot:] + best_seq[:rot]
        ordered_nodes = best_seq

        angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n, endpoint=False)
        pos = {nd: (scale * np.cos(angles[i]), scale * np.sin(angles[i]))
               for i, nd in enumerate(ordered_nodes)}
        return pos

    except Exception:
        return nx.circular_layout(G, scale=scale)

# precision-implied R² (Haslbeck & Waldorp 2018 canonical method)
# Uses **sample** covariance (not regularised) when prec/cov provided.
def _compute_predictability(feat_list, prec=None, cov=None,
                            cross_df=None, pcor_df=None, thresh=0.05):
    """
    Node predictability (R²).

    If prec and cov matrices are provided, uses precision-implied coefficients:
        R²_j = 1 − 1 / (Θ_jj · Σ_jj)
    (Haslbeck & Waldorp 2018, as implemented in mgm::predict / EGAnet).

    cov should be the **sample** covariance of the standardised
    data, not the regularised covariance from GLASSO, to match the Haslbeck &
    Waldorp derivation.

    Otherwise, falls back to OLS regression on network neighbors.
    """
    r2 = {}
    if prec is not None and cov is not None:
        for i, f in enumerate(feat_list):
            r2[f] = max(0.0, 1.0 - 1.0 / (prec[i, i] * cov[i, i] + 1e-12))
        return r2
    # Fallback: OLS (used by consensus network where no single precision matrix exists)
    if cross_df is None or pcor_df is None:
        return {f: 0.0 for f in feat_list}
    for f in feat_list:
        neighbors = [o for o in feat_list
                     if o != f and abs(pcor_df.loc[f, o]) > thresh]
        if len(neighbors) == 0:
            r2[f] = 0.0; continue
        subset = cross_df[neighbors + [f]].dropna()
        if len(subset) < 10:
            r2[f] = 0.0; continue
        reg = LinearRegression()
        reg.fit(subset[neighbors].values, subset[f].values)
        r2[f] = max(0.0, reg.score(subset[neighbors].values, subset[f].values))
    return r2

def _network_stats(G):
    nn = G.number_of_nodes(); ne = G.number_of_edges()
    mx = nn * (nn - 1) / 2
    density = ne / mx if mx > 0 else 0.0
    if ne > 0:
        ws = [abs(G.edges[e]['weight']) for e in G.edges()]
        return {'n_nodes': nn, 'n_edges': ne, 'density': density,
                'avg_abs_weight': np.mean(ws), 'median_abs_weight': np.median(ws),
                'max_abs_weight': np.max(ws)}
    return {'n_nodes': nn, 'n_edges': ne, 'density': density,
            'avg_abs_weight': 0, 'median_abs_weight': 0, 'max_abs_weight': 0}

# MDS stress fallback threshold lowered 0.15 → 0.10
# (Kruskal: ≤0.10 "fair", ≤0.05 "good"; 0.15 was too lax)
def _mds_layout(G, pcor_df, feat_list, scale=5.0, seed=42):
    feat_in_g = [nd for nd in feat_list if nd in G.nodes()]
    sub = pcor_df.loc[feat_in_g, feat_in_g].values
    dist = 1 - np.abs(sub)
    np.fill_diagonal(dist, 0)
    dist = np.clip(dist, 0, None)
    mds = MDS(n_components=2, dissimilarity='precomputed',
              random_state=seed, normalized_stress='auto', max_iter=500)
    coords = mds.fit_transform(dist)
    stress = mds.stress_
    if stress > 0.10:
        print(f"  Warning:  MDS stress = {stress:.4f} > 0.10 -- falling back to Fruchterman-Reingold")
        import networkx as nx
        pos = nx.spring_layout(G, weight='aw', k=2.5/len(G)**0.5,
                               iterations=200, seed=seed)
        pa = np.array([list(pos[nd]) for nd in feat_in_g]) * scale
        pos = {feat_in_g[i]: (pa[i, 0], pa[i, 1]) for i in range(len(feat_in_g))}
        for nd in G.nodes():
            if nd not in pos:
                pos[nd] = nx.spring_layout(G, weight='aw', seed=seed)[nd]
                pos[nd] = (pos[nd][0] * scale, pos[nd][1] * scale)
        return pos, f'Fruchterman-Reingold (MDS stress too high: {stress:.3f})', stress
    coords = coords / (np.abs(coords).max() + 1e-9) * scale
    pos = {feat_in_g[i]: (coords[i, 0], coords[i, 1]) for i in range(len(feat_in_g))}
    for nd in G.nodes():
        if nd not in pos:
            import networkx as nx
            extra = nx.spring_layout(G, weight='aw', seed=seed)
            pos[nd] = (extra[nd][0] * scale, extra[nd][1] * scale)
    return pos, f'MDS (stress-1 = {stress:.4f})', stress

def _draw_predictability_node(ax, x, y, radius, r2, node_color,
                               border_color, border_width):
    """
    Predictability-ring node for warm off-white background.
    Drop shadow provides depth. Ring fill = R² in darkened node colour.

    Inner circle radius 0.07→0.30 so donut hole is visible.
    Node z-orders bumped (shadow→3, bg→4, fill→5, inner→6) so
      edges (zorder 1-2) never bleed over nodes.
    Shadow offset is radius-relative, not hardcoded data coords.
    """
    _shadow_off = radius * 0.25
    shadow = Circle((x + _shadow_off, y - _shadow_off),
                    radius * _SHADOW_SCALE,
                    facecolor='#000000', alpha=_SHADOW_ALPHA,
                    edgecolor='none', zorder=3)
    ax.add_patch(shadow)
    bg = Circle((x, y), radius, facecolor=_RING_UNFILLED,
                edgecolor=border_color, linewidth=border_width, zorder=4)
    ax.add_patch(bg)
    if r2 > 0.01:
        _nc_rgb = np.array(matplotlib.colors.to_rgb(node_color))
        _nc_dark = tuple(np.clip(_nc_rgb * 0.65, 0, 1))
        theta_start = 90
        theta_end   = 90 - (r2 * 360)
        fill = Wedge((x, y), radius, theta_end, theta_start,
                     facecolor=_nc_dark, edgecolor='none', alpha=1.0, zorder=5)
        ax.add_patch(fill)
    inner = Circle((x, y), radius * 0.30, facecolor='white',
                   edgecolor='none', alpha=1.0, zorder=6)
    ax.add_patch(inner)
    # tick mark at 12 o'clock -- R² reference line
    # (Haslbeck & Waldorp 2018 convention; makes ring fill readable)
    _tick_inner = radius * 0.28
    _tick_outer = radius * 1.02
    ax.plot([x, x], [y + _tick_inner, y + _tick_outer],
            color=border_color, lw=1.5, solid_capstyle='round',
            alpha=0.7, zorder=6, clip_on=False)

def _get_domain_colors(feat_list, cat_of_feat, is_within):
    _f2c = None
    try:
        if 'FEATURE_TO_CATEGORY' in globals() and isinstance(FEATURE_TO_CATEGORY, dict):
            _f2c = FEATURE_TO_CATEGORY
    except Exception:
        pass  # non-critical; logged upstream or optional visualization
    if _f2c and len(_f2c) > 0:
        domains_used = {}
        colors = {}
        for f in feat_list:
            domain = _f2c.get(f, 'Demographic/Other')
            domains_used[domain] = True
            colors[f] = _DOMAIN_COLORS.get(domain, _DOMAIN_FALLBACK)
        handles = [mpatches.Patch(fc=_DOMAIN_COLORS.get(d, _DOMAIN_FALLBACK),
                                   ec='none', alpha=0.85, label=d)
                   for d in sorted(domains_used.keys())]
        return colors, handles
    if is_within:
        unique_cats = list(dict.fromkeys(cat_of_feat[f] for f in feat_list))
        cat_cmap = {c: _CAT_COLORS[i % len(_CAT_COLORS)]
                    for i, c in enumerate(unique_cats)}
        colors = {f: cat_cmap[cat_of_feat[f]] for f in feat_list}
        handles = [mpatches.Patch(fc=cat_cmap[c], ec='none', alpha=0.85,
                                   label=c[:25]) for c in unique_cats]
        return colors, handles
    colors = {f: _CAT_COLORS[i % len(_CAT_COLORS)]
              for i, f in enumerate(feat_list)}
    return colors, None

def _save_fig(fig, basename, facecolor='white'):
    fig.savefig(f'{basename}.png', dpi=300, bbox_inches='tight', facecolor=facecolor)
    fig.savefig(f'{basename}.pdf', bbox_inches='tight', facecolor=facecolor)
    print(f"  Saved: {basename}.png  +  {basename}.pdf")

# caption updated for EBICglasso (γ reported, Burger et al. checklist now accurate)
# edge selection described as EBICglasso regularisation, not arbitrary threshold
# n/p ratio reported
def _print_network_caption(analysis_title, n_subj, n_feat, gamma_str,
                           layout_lbl, nstats, pred_r2,
                           target_lbl=None, nstats_full=None):
    avg_r2 = np.mean(list(pred_r2.values())) if pred_r2 else 0
    _np_ratio = n_subj / max(n_feat, 1)
    cap = (
        f"\n  +- FIGURE CAPTION (per Burger et al. 2022 reporting standards) ---\n"
        f"  | {analysis_title}\n"
        f"  | Partial correlation network of top-{n_feat} ML ensemble features"
    )
    if target_lbl:
        cap += f"\n  | for {target_lbl}"
    cap += (
        f".\n"
        f"  | Edges: regularised partial correlations (EBICglasso, {gamma_str}).\n"
        f"  | Edge selection: edges shrunk to zero by EBICglasso are excluded.\n"
        f"  | Display threshold: |partial r| < {_DISPLAY_THRESH} omitted for visual\n"
        f"  |   clarity; full EBICglasso adjacency available in supplement.\n"
    )
    if nstats_full:
        cap += (
            f"  | Full model: {nstats_full['n_edges']} edges "
            f"(density = {nstats_full['density']:.3f});  "
            f"displayed: {nstats['n_edges']} edges.\n"
        )
    cap += (
        f"  | Edge colour: blue = positive, vermillion = negative (Okabe-Ito).\n"
        f"  | Negative edges rendered as dashed lines for grayscale accessibility.\n"
        f"  | Edge width/opacity ∝ |partial r|, fixed maximum = {_EDGE_MAXIMUM}\n"
        f"  | (enables cross-figure comparison).\n"
        f"  | Node size ∝ ensemble consensus importance.\n"
        f"  | Colored fill within each node ring = predictability\n"
        f"  | (R²: precision-implied variance explained; avg = {avg_r2:.2f}).\n"
        f"  | R² computed via Haslbeck & Waldorp (2018) precision-implied method\n"
        f"  | using sample covariance of standardised data.\n"
        f"  | Border colour: blue = positive EI, orange = negative EI.\n"
        f"  | EI computed from full EBICglasso model (Robinaugh et al. 2016).\n"
        f"  | Layout: {layout_lbl}.\n"
        f"  | Network (displayed): {nstats['n_nodes']} nodes, {nstats['n_edges']} edges, "
        f"density = {nstats['density']:.3f},\n"
        f"  |   avg |weight| = {nstats['avg_abs_weight']:.4f}.\n"
        f"  | N = {n_subj:,}  ·  n/p = {_np_ratio:.0f}.\n"
        f"  | NOTE: Network constructed from top ML-selected features;\n"
        f"  |   conditional dependencies reflect inter-feature structure among\n"
        f"  |   top predictors, not a theory-driven psychometric network.\n"
        f"  | Cross-sectional structure; not interpretable as within-person\n"
        f"  | causal dynamics (Borsboom et al. 2021).\n"
        f"  +---"
    )
    print(cap)

# ---
# SIDEBAR RENDERER -- shared by per-target and consensus network figures
# ---

def _render_centrality_sidebar(ax_ei, ei_sorted, label_of_feat, feat_colors,
                                biv_r, imp_of_feat=None):
    """
    Chip-table sidebar: two metric columns (EI, r(target)).

    Each metric cell = rounded chip with proportional fill + bold centred number.
      • EI, r(target) -- diverging: blue fill (+), red fill (−), fills from centre
    Feature column: small coloured dot (node colour) + rank number + name.
    Rows sorted by |EI| descending at call site.
    """
    _tr = ax_ei.transAxes
    ax_ei.axis('off')
    _n_rows = len(ei_sorted)

    _COL_HDR_BG   = '#E4EBF5'
    _COL_HDR_CLR  = '#263238'
    _SEP_CLR      = '#BED0E8'
    _ROW_GRAD_TOP = np.array([0.82, 0.90, 0.97])
    _ROW_GRAD_BOT = np.array([0.97, 0.99, 1.00])
    _FEAT_CLR     = '#1A2533'
    _RANK_CLR     = '#546E7A'
    _FOOT_CLR     = '#78909C'
    _CHIP_TRK     = '#DDE6F0'
    _CHIP_POS     = '#1565C0'
    _CHIP_NEG     = '#C62828'
    _NUM_POS      = '#0A0A0A'
    _NUM_NEG      = '#0A0A0A'
    _NUM_ZERO     = '#444444'

    _col_h    = 0.038
    _usable   = 1.0 - _col_h - 0.004
    _row_h    = min(_usable / max(_n_rows, 1), 0.074)
    _chip_h_f = 0.68

    def _row_cy(i):
        return 1.0 - _col_h - _row_h * (i + 0.5)

    _ei_vals  = [v for _, v in ei_sorted]
    _ei_max   = max(abs(v) for v in _ei_vals) if _ei_vals else 1.0
    _ei_max   = _ei_max or 1.0

    # when biv_r is None (consensus: multi-target), skip r(target) chips
    _show_biv_r = biv_r is not None

    if _show_biv_r:
        _ry_vals  = [biv_r.get(f, 0.0) for f, _ in ei_sorted]
        _ry_abs   = [abs(v) for v in _ry_vals]
        _ry_max   = max(_ry_abs) if _ry_abs else 1.0
        _ry_max   = _ry_max or 1.0
    else:
        _ry_max = 1.0

    _dot_x    = 0.022
    _dot_r    = 0.009
    _rank_x   = 0.048
    _name_x0  = 0.068
    _name_x1  = 0.480
    _ei_cl, _ei_cr   = 0.492, 0.720
    _ry_cl,  _ry_cr  = 0.734, 0.988

    _col_bot = 1.0 - _col_h
    _col_cy  = 1.0 - _col_h * 0.50
    ax_ei.add_patch(mpatches.Rectangle(
        (0.0, _col_bot), 1.0, _col_h,
        fc=_COL_HDR_BG, ec='none', transform=_tr, zorder=1))
    ax_ei.plot([0.0, 1.0], [_col_bot, _col_bot],
               color='#8FAEC8', lw=1.4, transform=_tr, zorder=2, clip_on=False)

    _hkw = dict(va='center', transform=_tr, zorder=3, family='sans-serif',
                fontsize=7.8, fontweight='800', color=_COL_HDR_CLR)
    ax_ei.text(_name_x0,                    _col_cy, 'FEATURE',   ha='left',   **_hkw)
    ax_ei.text((_ei_cl  + _ei_cr)  / 2,    _col_cy, 'EI (|EI|↓)', ha='center', **_hkw)
    _ry_hdr = 'r(target)' if _show_biv_r else '(multi-target)'
    ax_ei.text((_ry_cl  + _ry_cr)  / 2,    _col_cy, _ry_hdr,  ha='center', **_hkw)

    for _ri, (_f, _eiv) in enumerate(ei_sorted):
        _yc   = _row_cy(_ri)
        _rbot = _yc - _row_h * 0.50
        _nc   = feat_colors.get(_f, '#888')
        _lbl  = label_of_feat.get(_f, _f)
        _ryv  = biv_r.get(_f, 0.0) if _show_biv_r else None
        _ch   = _chip_h_f * _row_h

        _t = _ri / max(_n_rows - 1, 1)
        _row_rgb = (1 - _t) * _ROW_GRAD_TOP + _t * _ROW_GRAD_BOT
        ax_ei.add_patch(mpatches.Rectangle(
            (0.0, _rbot), 1.0, _row_h,
            fc=tuple(_row_rgb), ec='none', transform=_tr, zorder=0))

        ax_ei.plot([0.0, 1.0], [_rbot, _rbot],
                   color=_SEP_CLR, lw=0.4, transform=_tr, zorder=1, clip_on=False)

        ax_ei.add_patch(Circle((_dot_x, _yc), _dot_r,
                               facecolor=_nc, edgecolor='white',
                               linewidth=0.5, alpha=0.90,
                               transform=_tr, zorder=3))

        ax_ei.text(_rank_x, _yc, str(_ri + 1),
                   ha='center', va='center', transform=_tr,
                   fontsize=6.8, fontweight='700', color=_RANK_CLR,
                   zorder=4, family='sans-serif')

        import textwrap as _tw
        if len(_lbl) <= 22:
            _name_lines = [_lbl]
            _name_fs    = 8.8
        else:
            _wrapped    = _tw.wrap(_lbl, width=22)[:2]
            _name_lines = _wrapped
            _name_fs    = 7.8 if max(len(l) for l in _wrapped) > 18 else 8.2
        _n_lines  = len(_name_lines)
        _line_gap = _row_h * 0.44
        _name_y_start = _yc + (_line_gap * (_n_lines - 1) / 2.0)
        for _li, _line in enumerate(_name_lines):
            ax_ei.text(_name_x0, _name_y_start - _li * _line_gap,
                       _line, fontsize=_name_fs, color=_FEAT_CLR,
                       va='center', fontweight='600',
                       transform=_tr, zorder=3, family='sans-serif',
                       clip_on=True)

        def _chip(x0, x1, value, vmax, diverging, fill_clr, num_clr):
            _cw  = x1 - x0
            _cx  = (x0 + x1) / 2.0
            _cbot = _yc - _ch * 0.50
            ax_ei.add_patch(mpatches.FancyBboxPatch(
                (x0, _cbot), _cw, _ch,
                boxstyle='round,pad=0.005',
                fc=_CHIP_TRK, ec='#B0C4D8', lw=0.5,
                transform=_tr, zorder=2))
            _frac = min(abs(value) / vmax, 1.0) if vmax > 0 else 0.0
            if _frac > 0.02:
                _pad_v = _ch * 0.12
                if diverging:
                    _hw   = _cw / 2.0
                    _fw   = _frac * _hw
                    _fx0  = _cx if value >= 0 else (_cx - _fw)
                    ax_ei.add_patch(mpatches.FancyBboxPatch(
                        (_fx0, _cbot + _pad_v), _fw, _ch - 2 * _pad_v,
                        boxstyle='round,pad=0.003',
                        fc=fill_clr, ec='none', alpha=0.78,
                        transform=_tr, zorder=3))
                    ax_ei.plot([_cx, _cx],
                               [_cbot + _pad_v, _cbot + _ch - _pad_v],
                               color='#8FAEC8', lw=0.7,
                               transform=_tr, zorder=4)
                else:
                    _fw  = _frac * _cw
                    ax_ei.add_patch(mpatches.FancyBboxPatch(
                        (x0, _cbot + _ch * 0.12), _fw, _ch * 0.76,
                        boxstyle='round,pad=0.003',
                        fc=fill_clr, ec='none', alpha=0.78,
                        transform=_tr, zorder=3))
            _sgn  = '+' if (diverging and value > 0.005) else ''
            ax_ei.text(_cx, _yc, f'{_sgn}{value:.2f}',
                       ha='center', va='center', transform=_tr,
                       fontsize=8.5, fontweight='800', color=num_clr,
                       zorder=5, family='sans-serif')

        _ei_nc = _NUM_POS if _eiv > 0.005 else (_NUM_NEG if _eiv < -0.005 else _NUM_ZERO)
        _chip(_ei_cl,  _ei_cr,  _eiv,  _ei_max,  True,  _CHIP_POS if _eiv >= 0 else _CHIP_NEG, _ei_nc)

        if _show_biv_r and _ryv is not None:
            _ry_nc = _NUM_POS if _ryv > 0.005 else (_NUM_NEG if _ryv < -0.005 else _NUM_ZERO)
            _chip(_ry_cl,  _ry_cr,  _ryv,  _ry_max,  True,  _CHIP_POS if _ryv >= 0 else _CHIP_NEG, _ry_nc)
        else:
            # Multi-target consensus: show "--" instead of misleading 0.00
            ax_ei.text((_ry_cl + _ry_cr) / 2, _yc, '--',
                       ha='center', va='center', transform=_tr,
                       fontsize=8.5, fontweight='600', color=_NUM_ZERO,
                       zorder=5, family='sans-serif')

# ---
# SHARED EDGE-RENDERING HELPER
# ---

def _render_edges(ax, G, pos, node_list, data_span):
    """
    Draw magnitude-based edges with statistical glow threshold.

    Returns dict of {(u,v): (mid_x, mid_y, weight, p1, p2)} for the top-N
    edges (used for on-figure weight labels with perpendicular offset).

    Geometry-based curve direction -- edges always curve outward
      from the layout center, avoiding arcs that cross through node interiors.
    """
    if G.number_of_edges() == 0:
        return {}

    _edge_sorted = sorted(G.edges(data=True),
                          key=lambda x: abs(x[2]['weight']), reverse=True)

    _weights = [abs(d['weight']) for _, _, d in _edge_sorted]
    _glow_thresh = np.mean(_weights) + 1.0 * np.std(_weights)
    _strong_edges = set()
    for _u, _v, _d in _edge_sorted:
        if abs(_d['weight']) >= _glow_thresh:
            _strong_edges.add((_u, _v))
            _strong_edges.add((_v, _u))

    # compute layout center for outward-curve sign
    _all_x = [pos[nd][0] for nd in node_list]
    _all_y = [pos[nd][1] for nd in node_list]
    _cx = np.mean(_all_x)
    _cy = np.mean(_all_y)

    _edge_midpoints = {}

    for _eidx, (_u, _v) in enumerate(G.edges()):
        _pc  = G.edges[(_u, _v)]['weight']
        _aw  = abs(_pc)

        # Power-law mapping (exponent 0.6) concentrates visual contrast in
        # the mid-to-strong range where interesting edges live, while
        # compressing near-zero edges into ghostlike hairlines.
        _t   = min(_aw / _EDGE_MAXIMUM, 1.0) ** 0.6
        _lw  = _LW_MIN + (_LW_MAX - _LW_MIN) * _t
        _alp = _ALPHA_MIN + (_ALPHA_MAX - _ALPHA_MIN) * _t

        _base_ec = _EDGE_POS if _pc > 0 else _EDGE_NEG
        _ls      = '-' if _pc > 0 else _DASH_NEG
        _nc_u    = G.nodes[_u]['color']
        _nc_v    = G.nodes[_v]['color']
        _is_str  = ((_u, _v) in _strong_edges or (_v, _u) in _strong_edges)

        # compute outward curve sign from geometry
        # Chord midpoint relative to layout center; perpendicular direction
        # that points outward gets positive sign.
        _p1, _p2 = pos[_u], pos[_v]
        _emx = (_p1[0] + _p2[0]) / 2.0 - _cx
        _emy = (_p1[1] + _p2[1]) / 2.0 - _cy
        _edx = _p2[0] - _p1[0]
        _edy = _p2[1] - _p1[1]
        # Cross product of chord direction with radial direction determines
        # which perpendicular side is "outward"
        _cross = (-_edy) * _emx + _edx * _emy
        _csign = 1 if _cross >= 0 else -1

        ctrl = _draw_fancy_edge(ax, _p1, _p2,
                                _nc_u, _nc_v, _base_ec,
                                _lw, _alp, _ls, _eidx, _is_str,
                                data_span=data_span, curve_sign=_csign)

        if (_u, _v) in _strong_edges and (_u, _v) not in _edge_midpoints:
            _bmx = 0.25*_p1[0] + 0.5*ctrl[0] + 0.25*_p2[0]
            _bmy = 0.25*_p1[1] + 0.5*ctrl[1] + 0.25*_p2[1]
            _edge_midpoints[(_u, _v)] = (_bmx, _bmy, _pc, _p1, _p2)

    return _edge_midpoints

def _render_edge_labels(ax, edge_midpoints, max_labels=3):
    """On-figure edge weight annotations for strongest edges.

    Labels are offset perpendicular to the chord so they don't
    overlap the arc or sit on top of nodes for short edges.
    """
    _sorted = sorted(edge_midpoints.items(),
                     key=lambda kv: abs(kv[1][2]), reverse=True)[:max_labels]
    for (_u, _v), vals in _sorted:
        if len(vals) == 5:
            _mx, _my, _wt, _p1, _p2 = vals
            # Perpendicular to chord, nudge outward from arc midpoint
            _pdx = -(_p2[1] - _p1[1])
            _pdy =  (_p2[0] - _p1[0])
            _plen = max(np.sqrt(_pdx**2 + _pdy**2), 1e-6)
            _nudge = 0.25  # data units
            _lx = _mx + _nudge * _pdx / _plen
            _ly = _my + _nudge * _pdy / _plen
        else:
            _mx, _my, _wt = vals[:3]
            _lx, _ly = _mx, _my
        ax.text(_lx, _ly, f'{_wt:+.2f}', fontsize=7, color=_CLR_SLATE,
                ha='center', va='center', zorder=8,
                bbox=dict(boxstyle='round,pad=0.15', fc='white',
                          alpha=0.88, ec='#CFD8DC', lw=0.5))

# ---
# LABEL REPULSION HELPER -- resolves collision on circle layouts
# ---

def _repel_labels(label_positions, node_positions, min_sep=1.5,
                  node_pull=0.15, iterations=80):
    """
    Spring-repulsion pass to separate overlapping node labels.

    label_positions : list of [lx, ly] -- MUTATED in place
    node_positions  : list of (nx, ny) -- label anchor points (attract labels)
    min_sep         : minimum distance between label centres before push
    node_pull       : fraction of each step that pulls labels toward anchor
    iterations      : number of relaxation passes

    Returns the mutated label_positions.
    """
    lp = [list(p) for p in label_positions]
    n = len(lp)
    for _ in range(iterations):
        for i in range(n):
            fx, fy = 0.0, 0.0
            for j in range(n):
                if i == j:
                    continue
                dx = lp[i][0] - lp[j][0]
                dy = lp[i][1] - lp[j][1]
                dist = max(np.sqrt(dx**2 + dy**2), 0.01)
                if dist < min_sep:
                    push = (min_sep - dist) / dist * 0.30
                    fx += dx * push
                    fy += dy * push
            # Light pull back toward own node to prevent drift
            ax, ay = node_positions[i]
            fx += (ax - lp[i][0]) * node_pull
            fy += (ay - lp[i][1]) * node_pull
            lp[i][0] += fx
            lp[i][1] += fy
    return lp

# ---
# MAIN BODY FUNCTION
# ---

def _run_val_11():

    # -- 1. PREREQUISITES ---
    _has_train   = 'X_train' in globals() and 'y_train' in globals() and X_train is not None
    _has_within  = ('exported_within_dict' in globals()
                    and isinstance(exported_within_dict, dict)
                    and len(exported_within_dict) > 0)
    _has_metrics = ('metrics_dict_full' in globals()
                    and isinstance(metrics_dict_full, dict)
                    and len(metrics_dict_full.get('category', [])) > 0)
    _has_model   = ('last_fitted_model' in globals()
                    and last_fitted_model is not None
                    and hasattr(last_fitted_model, 'base_models')
                    and hasattr(last_fitted_model, 'meta_model'))
    _split       = split if 'split' in globals() else 'across_categories'
    _is_within   = _split == 'within_categories'
    _is_across   = _split == 'across_categories'
    _is_reg      = (len(np.unique(y_train)) > 10) if _has_train else False
    _dn          = DISPLAY_NAMES if ('DISPLAY_NAMES' in globals() and isinstance(DISPLAY_NAMES, dict)) else {}
    _target_lbl  = target_options if 'target_options' in globals() else '?'
    _tp_lbl      = tp_option      if 'tp_option'      in globals() else '?'

    if not _has_train:
        print("Error:  X_train / y_train not found -- run the Main Analysis Runner first.")
        return

    print("-" * 72)
    print(f"  Partial Correlation Network (EBICglasso, γ = {_EBIC_GAMMA})")
    print(f"  Mode   : {'Within' if _is_within else 'Across'}-categories")
    print(f"  Target : {_target_lbl}  ·  T{_tp_lbl}")
    print(f"  Train  : {X_train.shape[0]:,} × {X_train.shape[1]:,}")
    print("-" * 72)

    # -- 2. CONSENSUS IMPORTANCE ---
    def _abbrev_cat(cat, max_len=22):
        mapping = {
            'Interpersonal Relationships & Social Functioning': 'Social Functioning',
            'Cognitive Task Outcomes': 'Cognitive Tasks',
            'Multimodal Neuroimaging': 'Neuroimaging',
            'Residential Characteristics': 'Residential',
            'Family Dynamics & Parenting': 'Family Dynamics',
            'Other Personality Features': 'Personality',
            'Medical/Somatic Problems': 'Medical',
            'Physical Activity/Features': 'Physical Activity',
            'Task and Resting State': 'Task/Rest fMRI',
            'School Dynamics': 'School',
            'Social Relationship Quality': 'Social Rel.',
            'Technology Use': 'Tech Use',
            'Sleep Problems': 'Sleep', 'SES & Mobility': 'SES',
        }
        for long, short in mapping.items():
            cat = cat.replace(long, short)
        cat = cat.replace('and ', '& ').replace('_', ' ')
        return cat[:max_len] + ('…' if len(cat) > max_len else '')

    def _consensus_from_pretrained(model, Xtr, ytr):
        original_columns = Xtr.columns.tolist()
        feature_importances = {}; all_features = set()
        if hasattr(model.meta_model, 'coef_'):
            mc = model.meta_model.coef_
            if mc.ndim > 1: mc = np.mean(np.abs(mc), axis=0)
            bn = list(model.base_models.keys())
            mw = {bn[i]: abs(mc[i]) if i < len(mc) else 0 for i in range(len(bn))}
        else:
            mw = {n: 1.0 for n in model.base_models}
        ws = sum(mw.values()) or 1
        mw = {k: v/ws for k,v in mw.items()}
        for name, bm in model.base_models.items():
            try:
                if hasattr(bm, 'feature_importances_'): imp = bm.feature_importances_
                elif hasattr(bm, 'coef_'):
                    imp = np.abs(bm.coef_)
                    if imp.ndim > 1: imp = imp.mean(axis=0)
                else: continue
                if len(imp) != len(original_columns): continue
                mi = pd.Series(imp, index=original_columns)
                feature_importances[name] = mi
                all_features.update(mi.index)
            except Exception: pass
        if not feature_importances: return None
        cs = {}
        for feat in all_features:
            ws2 = 0.0
            for mn, w in mw.items():
                if mn in feature_importances:
                    iv = feature_importances[mn].get(feat, 0)
                    ws2 += (iv / (feature_importances[mn].max() + 1e-12)) * w
            cs[feat] = ws2
        return pd.Series(cs).sort_values(ascending=False)

    def _consensus_importance_refit(Xtr, ytr):
        imp_dict = {}
        _imp = SimpleImputer(strategy='median')
        Xcln = pd.DataFrame(_imp.fit_transform(Xtr.replace([np.inf,-np.inf],np.nan)), columns=Xtr.columns)
        Xcln = Xcln.loc[:, Xcln.nunique() > 1]
        ycln = np.array(ytr).ravel()
        try:
            from catboost import CatBoostRegressor, CatBoostClassifier
            m = (CatBoostRegressor if _is_reg else CatBoostClassifier)(iterations=1000,learning_rate=0.05,verbose=0,random_state=42)
            m.fit(Xcln,ycln); r=m.feature_importances_; imp_dict['catboost']=r/(r.sum()+1e-12)
        except Exception: pass
        try:
            from xgboost import XGBRegressor, XGBClassifier
            m = (XGBRegressor if _is_reg else XGBClassifier)(n_estimators=_XGB_PARAMS['n_estimators'],max_depth=_XGB_PARAMS['max_depth'],learning_rate=_XGB_PARAMS['learning_rate'],random_state=42,verbosity=0,**({} if _is_reg else {'eval_metric':'logloss'}))
            m.fit(Xcln,ycln); r=m.feature_importances_; imp_dict['xgboost']=r/(r.sum()+1e-12)
        except Exception: pass
        try:
            from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
            m = (RandomForestRegressor if _is_reg else RandomForestClassifier)(n_estimators=_RF_PARAMS['n_estimators'],max_depth=_RF_PARAMS['max_depth'],random_state=42,n_jobs=-1)
            m.fit(Xcln,ycln); r=m.feature_importances_; imp_dict['random_forest']=r/(r.sum()+1e-12)
        except Exception: pass
        if not imp_dict: return None, Xcln
        return pd.DataFrame(imp_dict, index=Xcln.columns).mean(axis=1).sort_values(ascending=False), Xcln

    def _consensus_from_globals():
        try: _gdf = consensus_df if 'consensus_df' in globals() else None
        except Exception: _gdf = None
        if _gdf is None or not isinstance(_gdf, pd.DataFrame) or 'weighted_score' not in _gdf.columns: return None, None
        _rev = {v:k for k,v in _dn.items()}
        _s = _gdf.sort_values('weighted_score', ascending=False)
        _c = pd.Series(_s['weighted_score'].values, index=[_rev.get(f,f) for f in _s.index])
        _c = _c[~_c.index.duplicated(keep='first')]
        try: _mw = model_weights if 'model_weights' in globals() and isinstance(model_weights, dict) else {}
        except Exception: _mw = {}
        return _c, _mw

    def _get_top10_consensus(Xtr, ytr):
        _cg, _mg = _consensus_from_globals()
        if _cg is not None and len(_cg) > 0:
            print("\n>  Using consensus_df (exact match -- incl. TabPFN)")
            return _cg, 'main_runner_consensus_df'
        if _has_model:
            c = _consensus_from_pretrained(last_fitted_model, Xtr, ytr)
            if c is not None and len(c) > 0: return c, 'pretrained_ensemble'
        c, _ = _consensus_importance_refit(Xtr, ytr)
        return c, 'matched_refit'

    def _get_top1_consensus_within(Xtr, ytr):
        c, _ = _consensus_importance_refit(Xtr, ytr)
        return c

    # -- 3. FEATURE SELECTION ---
    _feat_list = []; _cat_of_feat = {}; _imp_of_feat = {}; _label_of_feat = {}

    if _is_across:
        _consensus, _source = _get_top10_consensus(X_train, y_train)
        if _consensus is None: print("Error:  No models converged."); return
        _top10 = _consensus.head(10)
        for _rk, (_f, _v) in enumerate(_top10.items(), 1):
            _feat_list.append(_f)
            _cat_of_feat[_f] = f'#{_rk}'
            _imp_of_feat[_f] = float(_v)
            _label_of_feat[_f] = _display_name(_f)
        _analysis_title = f'Top-10 Ensemble Features  ·  {_target_lbl}  (T{_tp_lbl})'

    elif _is_within:
        if not _has_within: print("Error:  exported_within_dict not found."); return
        _perf_col = 'r2' if _is_reg else 'auc'
        _cat_perf = {}
        if _has_metrics:
            for _cat, _mdict in zip(metrics_dict_full['category'], metrics_dict_full['metrics']):
                _m = _mdict.get('ensemble', _mdict.get('meta_model', {}))
                _v = _m.get(_perf_col, _m.get('accuracy', np.nan))
                if _cat is not None and not np.isnan(_v): _cat_perf[_cat] = float(_v)
        _ranked = sorted(exported_within_dict.keys(), key=lambda c: _cat_perf.get(c, -np.inf), reverse=True)[:10]
        for _cat in _ranked:
            _Xtr,_Xte,_ytr,_yte = exported_within_dict[_cat]
            _c = _get_top1_consensus_within(_Xtr, _ytr)
            if _c is None or len(_c)==0: continue
            _f = _c.index[0]; _v = float(_c.iloc[0])
            _feat_list.append(_f); _cat_of_feat[_f] = _cat
            _imp_of_feat[_f] = _v; _label_of_feat[_f] = _display_name(_f)
        _analysis_title = f'Top-10 Category Representatives  ·  {_target_lbl}  (T{_tp_lbl})'
    else:
        return

    # -- 4. BUILD CROSS-FEATURE MATRIX ---
    _feat_list = [f for f in _feat_list if f in X_train.columns]
    if len(_feat_list) < 3: print(f"Warning:  < 3 features available."); return

    _cross_df = X_train[_feat_list].copy().replace([np.inf,-np.inf], np.nan).dropna()
    _scaler = StandardScaler()
    _cross_scl = pd.DataFrame(_scaler.fit_transform(_cross_df), columns=_cross_df.columns)
    _labels_plot = [_label_of_feat.get(f, f) for f in _feat_list]
    n = len(_feat_list)

    _feat_colors, _domain_handles = _get_domain_colors(_feat_list, _cat_of_feat, _is_within)

    _corr = _cross_df.corr(method='pearson')
    _dist = (1 - _corr.abs()).clip(lower=0)
    np.fill_diagonal(_dist.values, 0)
    _linkage = linkage(squareform(_dist, checks=False), method='average')
    _order = dendrogram(_linkage, no_plot=True)['leaves']
    _corr_ord = _corr.iloc[_order, _order]
    _lbl_ord = [_labels_plot[i] for i in _order]

    try:
        _batch_name = _target_lbl
        if 'batch_corr_dict' not in globals():
            globals()['batch_corr_dict'] = {}
            globals()['batch_feat_list_corr'] = list(_feat_list)
        _common = globals()['batch_feat_list_corr']
        _corr_sub = _corr.loc[_common, _common].copy()
        globals()['batch_corr_dict'][_batch_name] = _corr_sub
    except Exception:
        pass  # non-critical; logged upstream or optional visualization

    # report n/p ratio
    _np_ratio = len(_cross_df) / max(n, 1)
    print(f"\n  Feature matrix: {_cross_df.shape[0]:,} × {n}  (n/p = {_np_ratio:.0f})")

    # -- 5. FIGURE 1 -- Correlation Heatmap ---
    print("\n" + "=" * 72)
    print("  Figure 1 -- Clustered Pearson Correlation Matrix")
    print("=" * 72)

    _cell_px = 72
    _max_llen = max(len(l) for l in _lbl_ord)
    _lbl_inch = max(_max_llen * 0.078, 1.8)
    _fw1 = n*(_cell_px/100) + _lbl_inch + 2.8
    _fh1 = n*(_cell_px/100) + 3.5

    fig1, ax1 = plt.subplots(figsize=(_fw1, _fh1))
    fig1.patch.set_facecolor('white')
    _im1 = ax1.imshow(_corr_ord.values, cmap=_CORR_CMAP, vmin=-1, vmax=1, aspect='auto', interpolation='nearest')
    for i in range(n):
        for j in range(i+1, n):
            ax1.add_patch(plt.Rectangle((j-.5,i-.5),1,1, fc='white',ec='white',lw=0,zorder=2))
    _fs_ann = max(7, min(11, 14 - n*0.35))
    for i in range(n):
        for j in range(i+1):
            v = _corr_ord.values[i,j]
            tc = 'white' if abs(v)>0.50 else _CLR_CHARCOAL
            fw = 'bold' if abs(v)>0.28 else 'normal'
            ax1.text(j,i,f'{v:.2f}', ha='center',va='center', color=tc, fontsize=_fs_ann, fontweight=fw, zorder=3)
    _tf = max(8.5, min(11, 130/max(n,1)))
    ax1.set_xticks(range(n)); ax1.set_yticks(range(n))
    ax1.set_xticklabels(_lbl_ord, rotation=45, ha='right', fontsize=_tf, color=_CLR_CHARCOAL)
    ax1.set_yticklabels(_lbl_ord, fontsize=_tf, color=_CLR_CHARCOAL)
    ax1.tick_params(axis='both', length=0, pad=8)
    cb1 = fig1.colorbar(_im1, ax=ax1, shrink=0.52, aspect=20, pad=0.03, ticks=[-1,-0.5,0,0.5,1])
    cb1.set_label('Pearson r', fontsize=10, color=_CLR_CHARCOAL, labelpad=8)
    cb1.ax.tick_params(labelsize=9, colors=_CLR_CHARCOAL); cb1.outline.set_visible(False)
    for sp in ax1.spines.values(): sp.set_visible(False)
    ax1.grid(False)
    ax1.set_title(f'{_analysis_title}\nClustered Pearson Correlation Matrix  (N = {len(_cross_df):,})',
                  fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=16)
    plt.tight_layout()
    _save_fig(fig1, 'cross_feature_correlation_matrix')
    plt.show()

    # -- 6. FIGURE 2 -- Partial Correlation Network ---
    print("\n" + "=" * 72)
    print("  Figure 2 -- Partial Correlation Network  (EBICglasso + Circular)")
    print("=" * 72)

    _pcor_df = None; _prec = None; _best_alpha = None; _sample_cov = None
    try:
        # EBICglasso replaces GraphicalLassoCV
        _prec, _ebic_cov, _best_alpha, _gamma_used = _ebic_glasso(
            _cross_scl.values, gamma=_EBIC_GAMMA,
            n_alpha=_EBIC_N_ALPHA, max_iter=1000)
        if _prec is not None:
            _dsqrt = np.sqrt(np.diag(_prec))
            _pcor = -_prec / np.outer(_dsqrt, _dsqrt)
            np.fill_diagonal(_pcor, 1.0)
            _pcor_df = pd.DataFrame(_pcor, index=_feat_list, columns=_feat_list)
            # use sample covariance (not regularised) for R²
            _sample_cov = np.cov(_cross_scl.values, rowvar=False, bias=False)
            print(f"  EBICglasso: γ = {_gamma_used}, λ (selected) = {_best_alpha:.6f}")
        else:
            raise ValueError("EBICglasso returned no valid solution")
    except Exception as _eg:
        print(f"  Warning:  EBICglasso failed ({_eg}) -- pseudoinverse fallback.")
        try:
            _sample_cov = np.cov(_cross_scl.values, rowvar=False, bias=False)
            _prec = np.linalg.pinv(_sample_cov)
            _dsqrt = np.sqrt(np.clip(np.diag(_prec), 1e-12, None))
            _pcor = -_prec / np.outer(_dsqrt, _dsqrt)
            np.fill_diagonal(_pcor, 1.0)
            _pcor_df = pd.DataFrame(_pcor, index=_feat_list, columns=_feat_list)
        except Exception as _eg2:
            print(f"  Error:  Fallback failed: {_eg2}")

    if _pcor_df is not None:
        import networkx as nx
        import textwrap

        try:
            _batch_name = _target_lbl
            if 'batch_pcor_dict' not in globals():
                globals()['batch_pcor_dict'] = {}
                globals()['batch_feat_list'] = list(_feat_list)
            _common = globals()['batch_feat_list']
            _pcor_sub = _pcor_df.loc[_common, _common].copy()
            globals()['batch_pcor_dict'][_batch_name] = _pcor_sub
        except Exception:
            pass  # non-critical; logged upstream or optional visualization

        # Build FULL graph (all EBICglasso-retained edges, 1e-6 noise filter)
        # for centrality metrics (EI, Strength) per Robinaugh et al. (2016).
        G_full = nx.Graph()
        for _f in _feat_list:
            G_full.add_node(_f, label=_label_of_feat.get(_f,_f),
                            color=_feat_colors.get(_f,'#888'), imp=_imp_of_feat.get(_f,0.01))
        _n_edges_full = 0
        for _i in range(n):
            for _j in range(_i+1, n):
                _pc = _pcor_df.iloc[_i, _j]
                if abs(_pc) > 1e-6:
                    G_full.add_edge(_feat_list[_i], _feat_list[_j], weight=_pc, aw=abs(_pc))
                    _n_edges_full += 1

        # EI and Strength from full EBICglasso model (not display-filtered)
        _EI = _compute_EI_from_graph(G_full, _feat_list)
        _strength = _compute_strength_from_graph(G_full, _feat_list)

        # Build DISPLAY graph (|pcor| > _DISPLAY_THRESH for visual clarity)
        G = nx.Graph()
        for _f in _feat_list:
            G.add_node(_f, label=_label_of_feat.get(_f,_f),
                       color=_feat_colors.get(_f,'#888'), imp=_imp_of_feat.get(_f,0.01))
        for _i in range(n):
            for _j in range(_i+1, n):
                _pc = _pcor_df.iloc[_i, _j]
                if abs(_pc) > _DISPLAY_THRESH:
                    G.add_edge(_feat_list[_i], _feat_list[_j], weight=_pc, aw=abs(_pc))

        _nstats = _network_stats(G)
        _nstats_full = _network_stats(G_full)
        print(f"  EBICglasso model: {_nstats_full['n_edges']} edges retained (density = {_nstats_full['density']:.3f})")
        print(f"  Displayed (|pcor| > {_DISPLAY_THRESH}): {_nstats['n_nodes']} nodes, {_nstats['n_edges']} edges")
        print(f"  Avg |w| (displayed): {_nstats['avg_abs_weight']:.4f}")

        for _f in _feat_list:
            if _f in G.nodes:
                G.nodes[_f]['ei'] = _EI.get(_f, 0.0)

        # precision-implied R² using sample covariance
        if _prec is not None and _sample_cov is not None:
            _pred_r2 = _compute_predictability(_feat_list, prec=_prec,
                                               cov=_sample_cov)
        else:
            _pred_r2 = _compute_predictability(_feat_list, cross_df=_cross_df,
                                               pcor_df=_pcor_df, thresh=1e-6)
        print(f"  Predictability (precision-implied, sample cov): avg R² = {np.mean(list(_pred_r2.values())):.3f}")
        for _f in sorted(_pred_r2, key=_pred_r2.get, reverse=True):
            print(f"    {_label_of_feat.get(_f,_f):<45} R² = {_pred_r2[_f]:.3f}")

        _biv_r = {}
        if _has_train:
            _ya_series = pd.Series(np.array(y_train).ravel(), index=X_train.index)
            _ya_cross = _ya_series.reindex(_cross_df.index).values
            for _f in _feat_list:
                _x = _cross_df[_f].values
                _valid = ~(np.isnan(_x) | np.isnan(_ya_cross))
                if _valid.sum() > 10:
                    from scipy.stats import pearsonr
                    _r, _ = pearsonr(_x[_valid], _ya_cross[_valid])
                    _biv_r[_f] = _r
                else:
                    _biv_r[_f] = 0.0

        # dynamic circle scale -- spreads nodes for dense networks
        _circle_scale = 5.5 + n * 0.25
        _pos = _fiedler_circle_layout(G, scale=_circle_scale)
        _layout_lbl = 'Spectral (Fiedler)'
        print(f"  Layout: {_layout_lbl} (nodes ordered by Fiedler vector + 2-opt)")

        _node_list = list(G.nodes())
        _imp_arr = np.array([G.nodes[nd]['imp'] for nd in _node_list])
        _imp_rng = max(_imp_arr.max() - _imp_arr.min(), 1e-9)

        _xs = [_pos[nd][0] for nd in _node_list]
        _ys = [_pos[nd][1] for nd in _node_list]
        _data_span = max(max(_xs)-min(_xs), max(_ys)-min(_ys), 0.01)
        _base_r = _data_span * 0.032
        _radii = [_base_r * (0.4 + 2.6 * ((_imp_arr[i]-_imp_arr.min())/_imp_rng)**0.5)
                  for i in range(len(_node_list))]

        _border_colors = []
        _border_widths = []
        _ei_arr = np.array([abs(_EI[nd]) for nd in _node_list])
        _ei_rng = max(_ei_arr.max(), 1e-9)
        for _i, _nd in enumerate(_node_list):
            _eiv = _EI[_nd]
            if abs(_eiv) < 0.01: _border_colors.append(_BORDER_ZERO)
            elif _eiv > 0: _border_colors.append(_BORDER_POS)
            else: _border_colors.append(_BORDER_NEG)
            # floor raised 1.2→2.0pt so low-EI borders are visible
            _border_widths.append(2.0 + 8.0 * (_ei_arr[_i] / _ei_rng))

        _net_w = max(11, n*1.10)
        _ei_w  = max(5.5, n*0.46)
        _fig_h = max(10, n*1.05)

        fig2 = plt.figure(figsize=(_net_w + _ei_w + 1.0, _fig_h))
        fig2.patch.set_facecolor('white')
        _gs = gridspec.GridSpec(1, 2, figure=fig2, width_ratios=[_net_w, _ei_w],
                                wspace=0.14, left=0.01, right=0.99, top=0.91, bottom=0.16)
        ax2 = fig2.add_subplot(_gs[0, 0])
        ax_ei = fig2.add_subplot(_gs[0, 1])
        ax2.set_facecolor(_NET_BG); ax2.grid(False); ax2.axis('off')

        # -- Edges (shared helper) ---
        _edge_midpoints = _render_edges(ax2, G, _pos, _node_list, _data_span)

        # -- Nodes ---
        _node_idx = {nd: i for i, nd in enumerate(_node_list)}
        for _i, _nd in enumerate(_node_list):
            _x, _y = _pos[_nd]
            _r2v = _pred_r2.get(_nd, 0.0)
            _nc  = G.nodes[_nd]['color']
            _draw_predictability_node(ax2, _x, _y, _radii[_i], _r2v, _nc,
                                      _border_colors[_i], _border_widths[_i])

        # -- Labels (with repulsion to prevent collision) ---
        _cx = np.mean(_xs); _cy = np.mean(_ys)
        _off = _data_span * 0.40
        _fs_lbl = max(10, min(13, 16 - n*0.14))
        _ww = 22
        _max_radius = max(_radii) if _radii else 1.0

        # collect initial radial positions, then repel
        _init_lbl_pos = []
        _node_anchors = []
        _lbl_texts = []
        for _nd in _node_list:
            _x, _y = _pos[_nd]
            _raw = G.nodes[_nd]['label']
            _lbl_texts.append('\n'.join(textwrap.wrap(_raw, width=_ww)))
            _dx = _x - _cx; _dy = _y - _cy
            _d  = max(np.sqrt(_dx**2+_dy**2), 0.001)
            _init_lbl_pos.append([_x + _off * _dx/_d, _y + _off * _dy/_d])
            _node_anchors.append((_x + _off * 0.5 * _dx/_d,
                                  _y + _off * 0.5 * _dy/_d))

        _repelled = _repel_labels(_init_lbl_pos, _node_anchors,
                                  min_sep=_data_span * 0.12,
                                  node_pull=0.12, iterations=80)

        for _li, _nd in enumerate(_node_list):
            _x, _y = _pos[_nd]
            _lx, _ly = _repelled[_li]
            _ni = _node_idx[_nd]
            _shrinkA = 10 + 8 * (_radii[_ni] / _max_radius)
            ax2.annotate(_lbl_texts[_li], xy=(_x,_y), xytext=(_lx,_ly),
                fontsize=_fs_lbl, color=_CLR_SLATE, fontweight=600,
                ha='center', va='center', clip_on=False,
                arrowprops=dict(arrowstyle='-', color=_ARROW_CLR,
                                lw=0.9, shrinkA=_shrinkA, shrinkB=4),
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.93,
                          ec=_feat_colors.get(_nd, _CLR_SILVER), lw=1.5),
                path_effects=[pe.withStroke(linewidth=3.0, foreground='white'),
                              pe.Normal()],
                zorder=7)

        _pad = _data_span * 0.30 + 2.2
        ax2.set_xlim(min(_xs)-_pad, max(_xs)+_pad)
        ax2.set_ylim(min(_ys)-_pad, max(_ys)+_pad)

        # -- Network key + edge scale (bottom-left) ---
        _legend_items = [
            plt.Line2D([0],[0], color=_EDGE_POS, lw=2.0, linestyle='-',
                       label='Positive pcor (solid)'),
            plt.Line2D([0],[0], color=_EDGE_NEG, lw=2.0, linestyle=_DASH_NEG,
                       label='Negative pcor (dashed)'),
            plt.Line2D([0],[0], marker='o', color='none',
                       markerfacecolor='#111111',
                       markerfacecoloralt=_RING_UNFILLED,
                       fillstyle='left',
                       markeredgecolor=_RING_UNFILLED, markeredgewidth=1.2,
                       markersize=9,
                       label='Ring fill = R² predictability'),
            plt.Line2D([0],[0], marker='o', color='none', markerfacecolor='none',
                       markeredgecolor=_BORDER_POS, markeredgewidth=3.5, markersize=9,
                       label='Blue border = +EI'),
            plt.Line2D([0],[0], marker='o', color='none', markerfacecolor='none',
                       markeredgecolor=_BORDER_NEG, markeredgewidth=3.5, markersize=9,
                       label='Orange border = −EI'),
        ]
        for _sr in [0.10, 0.20, 0.30]:
            _t_s = min(_sr / _EDGE_MAXIMUM, 1.0) ** 0.6
            _sw  = max(_LW_MIN + (_LW_MAX - _LW_MIN) * _t_s, 0.4)
            _sa  = max(_ALPHA_MIN + (_ALPHA_MAX - _ALPHA_MIN) * _t_s, 0.10)
            _sl  = f'|r| = {_sr:.2f}+' if _sr == 0.30 else f'|r| = {_sr:.2f}'
            _legend_items.append(
                plt.Line2D([0],[0], color=_EDGE_POS, lw=_sw, alpha=_sa,
                           solid_capstyle='round', label=_sl))
        if _domain_handles:
            _legend_items = _domain_handles + _legend_items
        fig2.legend(handles=_legend_items, fontsize=7.5,
                    framealpha=0.92, edgecolor='#CFD8DC', fancybox=True,
                    title='Key & Edge Scale', title_fontsize=8.5, borderpad=0.7,
                    labelspacing=0.45, handlelength=1.8, handletextpad=0.5,
                    columnspacing=1.2, ncol=4,
                    loc='lower left', bbox_to_anchor=(0.01, 0.04))

        # report γ instead of CV-selected α
        _gamma_str = f"γ = {_EBIC_GAMMA}, λ = {_best_alpha:.6f}" if _best_alpha else "Pseudoinverse fallback -- EBICglasso failed"

        _ei_sorted = sorted(_EI.items(), key=lambda kv: abs(kv[1]), reverse=True)
        _render_centrality_sidebar(ax_ei, _ei_sorted, _label_of_feat,
                                   _feat_colors, _biv_r, _imp_of_feat)

        _suptitle_method = f"EBICglasso, {_gamma_str}" if _best_alpha else "Pseudoinverse fallback"
        fig2.suptitle(
            f'{_analysis_title}\n'
            f'Partial Correlation Network  ({_suptitle_method})',
            fontsize=12, fontweight='bold', color=_CLR_SLATE, x=0.50, y=0.97, ha='center')

        _save_fig(fig2, 'cross_feature_partial_network', facecolor='white')
        plt.show()

        _print_network_caption(
            _analysis_title, len(_cross_df), n, _gamma_str, _layout_lbl,
            _nstats, _pred_r2, target_lbl=_target_lbl,
            nstats_full=_nstats_full)

        # Partial Correlation Heatmap
        _pcor_ord = _pcor_df.iloc[_order, _order]
        _fw2 = n*(_cell_px/100)+_lbl_inch+2.8; _fh2 = n*(_cell_px/100)+3.5
        fig2b, ax2b = plt.subplots(figsize=(_fw2, _fh2))
        fig2b.patch.set_facecolor('white')
        _im2b = ax2b.imshow(_pcor_ord.values, cmap=_CORR_CMAP, vmin=-0.5, vmax=0.5, aspect='auto')
        for i in range(n):
            for j in range(i+1,n):
                ax2b.add_patch(plt.Rectangle((j-.5,i-.5),1,1,fc='white',ec='white',lw=0,zorder=2))
        for i in range(n):
            for j in range(i+1):
                if i==j: continue
                v = _pcor_ord.values[i,j]
                if abs(v)>0.01:
                    tc = 'white' if abs(v)>0.22 else _CLR_CHARCOAL
                    ax2b.text(j,i,f'{v:.2f}',ha='center',va='center',color=tc,fontsize=_fs_ann,
                              fontweight='bold' if abs(v)>0.10 else 'normal',zorder=3)
        ax2b.set_xticks(range(n)); ax2b.set_yticks(range(n))
        ax2b.set_xticklabels(_lbl_ord, rotation=45, ha='right', fontsize=_tf, color=_CLR_CHARCOAL)
        ax2b.set_yticklabels(_lbl_ord, fontsize=_tf, color=_CLR_CHARCOAL)
        ax2b.tick_params(axis='both', length=0, pad=8)
        cb2 = fig2b.colorbar(_im2b, ax=ax2b, shrink=0.52, aspect=20, pad=0.03, ticks=[-0.5,-0.25,0,0.25,0.5])
        cb2.set_label('Partial r (EBICglasso)', fontsize=10, color=_CLR_CHARCOAL, labelpad=8)
        cb2.ax.tick_params(labelsize=9, colors=_CLR_CHARCOAL); cb2.outline.set_visible(False)
        for sp in ax2b.spines.values(): sp.set_visible(False)
        ax2b.grid(False)
        ax2b.set_title(f'{_analysis_title}\nClustered Partial Correlation Matrix  (EBICglasso, γ={_EBIC_GAMMA})  N = {len(_cross_df):,}',
                        fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=16)
        plt.tight_layout()
        _save_fig(fig2b, 'cross_feature_partial_heatmap')
        plt.show()

    # -- 6B. BOOTSTRAP CIs (optional) ---
    _boot_results = None
    if _pcor_df is not None and run_bootstrap_CI:
        print("\n" + "=" * 72)
        print(f"  Bootstrap Edge-Weight Stability  ({n_bootstrap:,} resamples, EBICglasso)")
        print("=" * 72)
        try:
            _boot_results = _bootstrap_edge_weights(_cross_df.values, _feat_list,
                                                     n_boot=n_bootstrap, seed=42,
                                                     gamma=_EBIC_GAMMA)
            print(f"  Converged: {_boot_results['n_converged']}/{n_bootstrap}")
            _edge_data = []
            for _i in range(n):
                for _j in range(_i+1, n):
                    _w = _pcor_df.iloc[_i,_j]; _lo = _boot_results['ci_lower'].iloc[_i,_j]
                    _hi = _boot_results['ci_upper'].iloc[_i,_j]; _ip = _boot_results['inclusion_prob'].iloc[_i,_j]
                    if abs(_w)>0.001 or _ip>0.10:
                        _li = _label_of_feat.get(_feat_list[_i],_feat_list[_i])[:15]
                        _lj = _label_of_feat.get(_feat_list[_j],_feat_list[_j])[:15]
                        _edge_data.append({'label':f'{_li}--{_lj}','weight':_w,'ci_lo':_lo,'ci_hi':_hi,'incl_prob':_ip})
            if _edge_data:
                _edge_data.sort(key=lambda x: x['weight'])
                fig_b, ax_b = plt.subplots(figsize=(10, max(6, len(_edge_data)*0.35+2)))
                fig_b.patch.set_facecolor('white')
                for _ei, _ed in enumerate(_edge_data):
                    _c = _EDGE_POS if _ed['weight']>0 else _EDGE_NEG
                    ax_b.errorbar(_ed['weight'], _ei,
                        xerr=[[_ed['weight']-_ed['ci_lo']],[_ed['ci_hi']-_ed['weight']]],
                        fmt='o', color=_c, alpha=0.4+0.5*_ed['incl_prob'],
                        markersize=6, capsize=3, capthick=1.2, elinewidth=1.5, zorder=2)
                ax_b.axvline(0, color=_CLR_SILVER, lw=1, ls='--', zorder=1)
                ax_b.set_yticks(range(len(_edge_data)))
                ax_b.set_yticklabels([d['label'] for d in _edge_data], fontsize=8.5, color=_CLR_CHARCOAL)
                ax_b.set_xlabel('Partial Correlation (95% Bootstrap CI)', fontsize=11, color=_CLR_CHARCOAL)
                ax_b.set_title(f'{_analysis_title}\nEdge-Weight Bootstrap ({_boot_results["n_converged"]}/{n_bootstrap:,}, EBICglasso γ={_EBIC_GAMMA})',
                               fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=14)
                ax_b.spines['top'].set_visible(False); ax_b.spines['right'].set_visible(False)
                ax_b.grid(axis='x', linestyle=':', alpha=0.4, color=_CLR_SILVER)
                plt.tight_layout()
                _save_fig(fig_b, 'cross_feature_bootstrap_CI')
                plt.show()
        except Exception as _e: print(f"  Warning:  Bootstrap failed: {_e}")

    # -- 6C. CENTRALITY STABILITY (optional) ---
    _cs_ei = _cs_str = None
    if _pcor_df is not None and run_centrality_stability:
        print("\n" + "=" * 72)
        print(f"  Centrality Stability  (CS coefficient, {n_bootstrap:,} resamples, EBICglasso)")
        print("=" * 72)
        try:
            # centrality computed from thresholded graph
            _cs_ei = _centrality_stability(_cross_df.values, _feat_list,
                lambda pdf, fnames: _compute_EI_thresholded(pdf, fnames, thresh=1e-6),
                n_boot=n_bootstrap, seed=42, gamma=_EBIC_GAMMA)
            _cs_str = _centrality_stability(_cross_df.values, _feat_list,
                lambda pdf, fnames: _compute_strength_thresholded(pdf, fnames, thresh=1e-6),
                n_boot=n_bootstrap, seed=42, gamma=_EBIC_GAMMA)
            print(f"  CS(EI)       = {_cs_ei['CS']:.2f}  {'stable' if _cs_ei['CS']>=0.50 else 'Warning: marginal' if _cs_ei['CS']>=0.25 else 'Error: unstable'}")
            print(f"  CS(Strength) = {_cs_str['CS']:.2f}  {'stable' if _cs_str['CS']>=0.50 else 'Warning: marginal' if _cs_str['CS']>=0.25 else 'Error: unstable'}")

            if _cs_ei['proportions'] and _cs_str['proportions']:
                fig_cs, ax_cs = plt.subplots(figsize=(8, 5))
                fig_cs.patch.set_facecolor('white')

                _ei_avg = np.array(_cs_ei['avg_cors'], dtype=float)
                _ei_sd  = np.array(_cs_ei['sd_cors'],  dtype=float)
                _str_avg = np.array(_cs_str['avg_cors'], dtype=float)
                _str_sd  = np.array(_cs_str['sd_cors'],  dtype=float)
                _props   = np.array(_cs_ei['proportions'])

                ax_cs.plot(_props, _ei_avg, 'o-', color=_EDGE_POS, lw=2, markersize=5,
                           label=f"EI  (CS = {_cs_ei['CS']:.2f})")
                _ei_valid = ~np.isnan(_ei_avg) & ~np.isnan(_ei_sd)
                ax_cs.fill_between(_props[_ei_valid],
                                   (_ei_avg - _ei_sd)[_ei_valid],
                                   (_ei_avg + _ei_sd)[_ei_valid],
                                   alpha=0.15, color=_EDGE_POS)

                ax_cs.plot(_props, _str_avg, 's-', color=_EDGE_NEG, lw=2, markersize=5,
                           label=f"Strength  (CS = {_cs_str['CS']:.2f})")
                _str_valid = ~np.isnan(_str_avg) & ~np.isnan(_str_sd)
                ax_cs.fill_between(_props[_str_valid],
                                   (_str_avg - _str_sd)[_str_valid],
                                   (_str_avg + _str_sd)[_str_valid],
                                   alpha=0.15, color=_EDGE_NEG)

                ax_cs.axhline(0.7, color=_CLR_SILVER, ls='--', lw=1, label='cor = 0.70')
                ax_cs.set_xlabel('Proportion Dropped', fontsize=11, color=_CLR_CHARCOAL)
                ax_cs.set_ylabel('Avg Correlation w/ Original', fontsize=11, color=_CLR_CHARCOAL)
                ax_cs.set_ylim(0, 1.05)
                ax_cs.set_xlim(_props[0] - 0.02, _props[-1] + 0.02)
                ax_cs.legend(fontsize=9.5, framealpha=0.95, edgecolor='#CFD8DC')
                ax_cs.set_title(f'{_analysis_title}\nCentrality Stability (case-dropping bootstrap, EBICglasso)',
                                fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=14)
                ax_cs.spines['top'].set_visible(False); ax_cs.spines['right'].set_visible(False)
                ax_cs.grid(axis='y', linestyle=':', alpha=0.4, color=_CLR_SILVER)
                plt.tight_layout()
                _save_fig(fig_cs, 'cross_feature_centrality_stability')
                plt.show()
        except Exception as _e: print(f"  Warning:  CS failed: {_e}")

    # -- 7. DENDROGRAM ---
    print("\n" + "=" * 72)
    print("  Figure 3 -- Dendrogram")
    print("=" * 72)

    _dend_labels = [_label_of_feat.get(_feat_list[i], _feat_list[i]) for i in range(n)]
    fig3, ax3 = plt.subplots(figsize=(max(13, n*1.1+2), 7))
    fig3.patch.set_facecolor('white'); ax3.set_facecolor('white')
    _dend = dendrogram(_linkage, labels=_dend_labels, leaf_rotation=45,
                       leaf_font_size=max(8.5, min(11, 130/max(n,1))),
                       color_threshold=0.65*_linkage[:,2].max(),
                       above_threshold_color=_CLR_SILVER, ax=ax3)
    for tl in ax3.get_xticklabels():
        for fi in _feat_list:
            if _label_of_feat.get(fi,fi)==tl.get_text():
                tl.set_color(_feat_colors.get(fi, _CLR_CHARCOAL)); tl.set_fontweight(600); break
    ax3.set_ylabel('Correlation Distance (1−|r|)', fontsize=11, color=_CLR_CHARCOAL, labelpad=8)
    ax3.set_title(f'{_analysis_title}\nFeature Clustering by Correlation Distance',
                  fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=14)
    ax3.spines['left'].set_color(_CLR_SILVER); ax3.spines['bottom'].set_color(_CLR_SILVER)
    ax3.yaxis.grid(True, linestyle=':', linewidth=0.5, color=_CLR_SILVER, alpha=0.6)
    ax3.set_axisbelow(True)
    _yb = -0.03*_linkage[:,2].max(); _lxs = sorted(ax3.get_xticks())
    for xi, li in enumerate(_dend['leaves']):
        fi = _feat_list[li]; c = _feat_colors.get(fi, '#999')
        if xi < len(_lxs): ax3.plot(_lxs[xi], _yb, 's', color=c, markersize=9, clip_on=False, zorder=5)
    plt.tight_layout()
    _save_fig(fig3, 'cross_feature_dendrogram')
    plt.show()

    # -- 8. SUMMARY TABLE ---
    print("\n" + "=" * 72)
    print("  Summary Table")
    print("=" * 72)

    _biv_r_target = {}
    if _has_train:
        _ya_series = pd.Series(np.array(y_train).ravel(), index=X_train.index)
        _ya_cross = _ya_series.reindex(_cross_df.index).values
        for fi in _feat_list:
            _x = _cross_df[fi].values
            _valid = ~(np.isnan(_x) | np.isnan(_ya_cross))
            if _valid.sum() > 10:
                from scipy.stats import pearsonr
                r, p = pearsonr(_x[_valid], _ya_cross[_valid])
                _biv_r_target[fi] = (r, p)
            else:
                _biv_r_target[fi] = (np.nan, np.nan)

    _rows = []
    for fi in _feat_list:
        cat = _cat_of_feat.get(fi,'--'); imp = _imp_of_feat.get(fi, np.nan)
        if fi in _corr.index:
            oth = _corr.loc[fi].drop(fi, errors='ignore').abs()
            mr = float(oth.mean()); mxr = float(oth.max())
            mxp = _label_of_feat.get(oth.idxmax(), oth.idxmax())
        else: mr=mxr=np.nan; mxp='--'
        pcmx = np.nan; pcpt = '--'
        if _pcor_df is not None and fi in _pcor_df.index:
            rpc = _pcor_df.loc[fi].drop(fi, errors='ignore').abs()
            if len(rpc)>0: pcmx=float(rpc.max()); pcpt=_label_of_feat.get(rpc.idxmax(),rpc.idxmax())
        ei = _EI.get(fi, np.nan) if _pcor_df is not None else np.nan
        st = _strength.get(fi, np.nan) if _pcor_df is not None else np.nan
        r2 = _pred_r2.get(fi, np.nan) if _pcor_df is not None else np.nan
        br, _bp = _biv_r_target.get(fi, (np.nan,np.nan))
        _rows.append({
            'Category/Rank': _abbrev_cat(cat) if _is_within else cat,
            'Feature': _label_of_feat.get(fi,fi),
            'Imp.': imp, 'r(target)': br,
            'EI': ei, 'Str': st, 'R²': r2,
            'Mean|r|': mr, 'Max|r|': mxr, 'Strongest Biv.': mxp,
            'Max|pcor|': pcmx, 'Strongest Cond.': pcpt,
        })

    _sdf = pd.DataFrame(_rows)
    _nc = ['Imp.','r(target)','EI','Str','R²','Mean|r|','Max|r|','Max|pcor|']
    _sty = (_sdf.style
        .format({c:'{:.4f}' for c in _nc})
        .background_gradient(cmap='YlOrRd', subset=['Imp.'], vmin=0)
        .background_gradient(cmap='RdBu_r', subset=['r(target)'], vmin=-0.3, vmax=0.3)
        .background_gradient(cmap='RdBu_r', subset=['EI'], vmin=-0.5, vmax=0.5)
        .background_gradient(cmap='Blues', subset=['Str'], vmin=0, vmax=1)
        .background_gradient(cmap='Greens', subset=['R²'], vmin=0, vmax=0.5)
        .background_gradient(cmap='Blues', subset=['Mean|r|','Max|r|'], vmin=0, vmax=0.5)
        .background_gradient(cmap='Purples', subset=['Max|pcor|'], vmin=0, vmax=0.3)
        .set_caption(f'Cross-Feature Summary -- {_target_lbl} (T{_tp_lbl})')
        .set_table_styles([
            {'selector':'caption','props':[('font-size','14px'),('font-weight','700'),('color',_CLR_SLATE)]},
            {'selector':'th','props':[('background-color','#EEF2F7'),('color',_CLR_CHARCOAL),('font-size','10px'),('font-weight','600'),('text-transform','uppercase'),('padding','6px 8px')]},
            {'selector':'td','props':[('padding','5px 8px'),('font-size','11px')]},
        ]).hide(axis='index'))
    display(_sty)

    _gamma_str = f"γ = {_EBIC_GAMMA}, λ = {_best_alpha:.6f}" if _best_alpha else "Pseudoinverse fallback -- EBICglasso failed"
    print(f"\n Cross-feature analysis complete.")
    print(f"   • {n} features · Layout: {_layout_lbl}" if _pcor_df is not None else f"   • {n} features")
    if _pcor_df is not None:
        print(f"   • EBICglasso ({_gamma_str})")
        print(f"   • Full model: {_nstats_full['n_edges']} edges (density = {_nstats_full['density']:.3f})")
        print(f"   • Displayed (|pcor| > {_DISPLAY_THRESH}): {_nstats['n_edges']} edges (density = {_nstats['density']:.3f})")
        print(f"   • Centrality: EI (from full EBICglasso model, sorted by |EI|)")
        print(f"   • r(target): bivariate Pearson r between each feature and target (same N as network)")
        print(f"   • Predictability (precision-implied, sample cov): avg R² = {np.mean(list(_pred_r2.values())):.3f}")
        print(f"   • Edge scaling: magnitude-based, fixed maximum = {_EDGE_MAXIMUM} (cross-figure comparable)")
        print(f"   • n/p ratio: {_np_ratio:.0f}")
    if _boot_results: print(f"   • Bootstrap: {_boot_results['n_converged']}/{n_bootstrap}, 95% CIs (EBICglasso)")
    elif not run_bootstrap_CI: print(f"   • Bootstrap: disabled (enable in UI)")
    if _cs_ei: print(f"   • CS(EI) = {_cs_ei['CS']:.2f}, CS(Str) = {_cs_str['CS']:.2f}")
    elif not run_centrality_stability: print(f"   • Centrality stability: disabled (enable in UI)")
    print(f"   • Exports: PNG (300 DPI) + PDF (vector)")

    _run_tgt = (_pcor_df is not None and _has_train and len(_feat_list) >= 3)
    if _run_tgt:
        print("\n\n" + "=" * 72)
        print("  TARGET-INCLUSIVE NETWORK")
        print("  (Same methodology -- MDS, R² rings, fixed max)")
        print("=" * 72)
        print("    Target-inclusive network uses same upgraded methodology.")
        print("     Append target-inclusive section from v1 with these substitutions:")
        print("     • nx.circular_layout → _mds_layout()")
        print("     • nx.draw_networkx_nodes → _draw_predictability_node()")
        print("     • aw/ew_max → aw/_EDGE_MAXIMUM")
        print("     • solid negative → dashed (ls = _DASH_NEG)")

# ---
# CONSENSUS HELPERS -- called after all batch runs complete
# ---

def _plot_consensus_pcor_network(batch_pcor_dict, feat_list, label_of_feat,
                                  imp_of_feat, feat_colors, thresh=None,
                                  batch_corr_dict=None):
    """Consensus partial-correlation network across batches via Fisher-z mean.

    R² now computed via precision-implied method by inverting
    the Fisher-z averaged correlation matrix, making R² comparable with
    per-target networks.
    """
    import networkx as nx
    if not batch_pcor_dict:
        print("No batch-specific partial-correlation matrices found."); return

    mats = []
    for _, pcor in batch_pcor_dict.items():
        p = pcor.values.copy()
        p = np.clip(p, -0.9999, 0.9999)
        mats.append(np.arctanh(p))
    z_mean = np.nanmean(np.stack(mats, axis=0), axis=0)
    r_mean = np.tanh(z_mean)
    np.fill_diagonal(r_mean, 1.0)
    pcor_cons = pd.DataFrame(r_mean, index=feat_list, columns=feat_list)

    n = len(feat_list)

    # Full graph (all nonzero consensus edges) for EI
    G_full = nx.Graph()
    for f in feat_list:
        G_full.add_node(f, label=label_of_feat.get(f, f),
                        color=feat_colors.get(f, '#888'),
                        imp=imp_of_feat.get(f, 1.0))
    for i in range(n):
        for j in range(i+1, n):
            fi, fj = feat_list[i], feat_list[j]
            pc = pcor_cons.loc[fi, fj]
            if abs(pc) > 1e-6:
                G_full.add_edge(fi, fj, weight=pc, aw=abs(pc))

    EI = _compute_EI_from_graph(G_full, feat_list)
    nstats_full = _network_stats(G_full)

    # Display graph (|pcor| > _DISPLAY_THRESH)
    G = nx.Graph()
    for f in feat_list:
        G.add_node(f, label=label_of_feat.get(f, f),
                   color=feat_colors.get(f, '#888'),
                   imp=imp_of_feat.get(f, 1.0))
    for i in range(n):
        for j in range(i+1, n):
            fi, fj = feat_list[i], feat_list[j]
            pc = pcor_cons.loc[fi, fj]
            if abs(pc) > _DISPLAY_THRESH:
                G.add_edge(fi, fj, weight=pc, aw=abs(pc))

    if G.number_of_edges() == 0:
        print(f"No edges above |pcor| > {_DISPLAY_THRESH} in consensus network."); return

    nstats = _network_stats(G)
    print(f"  Consensus model: {nstats_full['n_edges']} edges (density = {nstats_full['density']:.3f})")
    print(f"  Displayed (|pcor| > {_DISPLAY_THRESH}): {nstats['n_nodes']} nodes, {nstats['n_edges']} edges")

    for f in feat_list:
        if f in G.nodes:
            G.nodes[f]['ei'] = EI.get(f, 0.0)

    _circle_scale = 5.5 + n * 0.25
    pos = _fiedler_circle_layout(G, scale=_circle_scale)
    node_list = list(G.nodes())
    imp_arr = np.array([G.nodes[nd]['imp'] for nd in node_list])
    imp_rng = max(imp_arr.max() - imp_arr.min(), 1e-9)

    xs = [pos[nd][0] for nd in node_list]
    ys = [pos[nd][1] for nd in node_list]
    data_span = max(max(xs) - min(xs), max(ys) - min(ys), 0.01)
    base_r = data_span * 0.032
    radii = [base_r * (0.4 + 2.6 * ((imp_arr[i] - imp_arr.min()) / imp_rng) ** 0.5)
             for i in range(len(node_list))]

    # precision-implied R² for consensus network
    # Invert the Fisher-z averaged correlation matrix to get an approximate
    # precision matrix, making R² comparable with per-target networks
    # (Haslbeck & Waldorp 2018).
    pred_r2 = None
    if batch_corr_dict and len(batch_corr_dict) > 0:
        try:
            corr_mats = []
            for _, corr in batch_corr_dict.items():
                c = corr.values.copy()
                c = np.clip(c, -0.9999, 0.9999)
                corr_mats.append(np.arctanh(c))
            z_corr_mean = np.nanmean(np.stack(corr_mats, axis=0), axis=0)
            r_corr_mean = np.tanh(z_corr_mean)
            np.fill_diagonal(r_corr_mean, 1.0)
            # For standardised data, Θ = R⁻¹.
            # Ridge regularisation (1e-4) stabilises inversion of a potentially
            # near-singular averaged correlation matrix, preventing artifactual
            # R² ≈ 1.0 from inflated precision diagonals (pinv can "succeed"
            # numerically while producing garbage values).
            _n_feat = len(feat_list)
            r_corr_mean_reg = r_corr_mean + 1e-4 * np.eye(_n_feat)
            prec_cons = np.linalg.inv(r_corr_mean_reg)
            cov_cons = r_corr_mean  # correlation = covariance for standardised data
            pred_r2 = _compute_predictability(feat_list, prec=prec_cons,
                                              cov=cov_cons)
            # Sanity check: if ridge-regularised inversion still produces
            # implausible R² (e.g. > 0.99 for any node), fall back to OLS
            _max_r2 = max(pred_r2.values()) if pred_r2 else 0
            if _max_r2 > 0.99:
                print(f"  Warning:  Consensus precision inversion unstable (max R² = {_max_r2:.3f}), falling back to OLS")
                pred_r2 = None
            else:
                print(f"  Predictability: precision-implied from consensus correlation (comparable w/ per-target)")
        except Exception as _e:
            print(f"  Warning:  Precision-implied R² failed for consensus ({_e}), falling back to OLS")
            pred_r2 = None

    if pred_r2 is None:
        # Fallback: OLS (only if correlation-based method fails)
        cross_df = X_train[feat_list].replace([np.inf, -np.inf], np.nan).dropna()
        pred_r2 = _compute_predictability(feat_list, cross_df=cross_df,
                                          pcor_df=pcor_cons, thresh=1e-6)

    border_colors = []
    border_widths = []
    ei_arr = np.array([abs(EI[nd]) for nd in node_list])
    ei_rng = max(ei_arr.max(), 1e-9)
    for i, nd in enumerate(node_list):
        eiv = EI[nd]
        if abs(eiv) < 0.01:   border_colors.append(_BORDER_ZERO)
        elif eiv > 0:         border_colors.append(_BORDER_POS)
        else:                  border_colors.append(_BORDER_NEG)
        # floor raised 1.2→2.0pt
        border_widths.append(2.0 + 8.0 * (ei_arr[i] / ei_rng))

    net_w = max(11, n * 1.10)
    ei_w  = max(5.5, n * 0.46)
    fig_h = max(10, n * 1.05)

    fig = plt.figure(figsize=(net_w + ei_w + 1.0, fig_h))
    fig.patch.set_facecolor('white')
    gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[net_w, ei_w],
                           wspace=0.14, left=0.01, right=0.99, top=0.91, bottom=0.16)
    ax_net = fig.add_subplot(gs[0, 0])
    ax_ei  = fig.add_subplot(gs[0, 1])

    ax_net.set_facecolor(_NET_BG); ax_net.grid(False); ax_net.axis('off')

    # -- Edges (shared helper) ---
    _edge_midpoints = _render_edges(ax_net, G, pos, node_list, data_span)

    import textwrap as _tw
    _cx = np.mean(xs); _cy = np.mean(ys)
    _off = data_span * 0.40
    _fs_lbl = max(10, min(13, 16 - n * 0.14))
    _ww = 22
    _max_radius = max(radii) if radii else 1.0
    _node_idx = {nd: i for i, nd in enumerate(node_list)}

    # Draw nodes first
    for i, nd in enumerate(node_list):
        x, y = pos[nd]
        _draw_predictability_node(ax_net, x, y, radii[i],
                                  pred_r2.get(nd, 0.0),
                                  G.nodes[nd]['color'],
                                  border_colors[i], border_widths[i])

    # collect initial radial positions, then repel
    _init_lbl_pos = []
    _node_anchors = []
    _lbl_texts = []
    for i, nd in enumerate(node_list):
        x, y = pos[nd]
        _raw = G.nodes[nd]['label']
        _lbl_texts.append('\n'.join(_tw.wrap(_raw, width=_ww)))
        _dx = x - _cx; _dy = y - _cy
        _d  = max(np.sqrt(_dx**2 + _dy**2), 0.001)
        _init_lbl_pos.append([x + _off * _dx / _d, y + _off * _dy / _d])
        _node_anchors.append((x + _off * 0.5 * _dx / _d,
                              y + _off * 0.5 * _dy / _d))

    _repelled = _repel_labels(_init_lbl_pos, _node_anchors,
                              min_sep=data_span * 0.12,
                              node_pull=0.12, iterations=80)

    for i, nd in enumerate(node_list):
        x, y = pos[nd]
        _lx, _ly = _repelled[i]
        _shrinkA = 10 + 8 * (radii[i] / _max_radius)
        ax_net.annotate(_lbl_texts[i], xy=(x, y), xytext=(_lx, _ly),
            fontsize=_fs_lbl, color=_CLR_SLATE, fontweight=600,
            ha='center', va='center', clip_on=False,
            arrowprops=dict(arrowstyle='-', color=_ARROW_CLR,
                            lw=0.9, shrinkA=_shrinkA, shrinkB=4),
            bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.93,
                      ec=feat_colors.get(nd, _CLR_SILVER), lw=1.5),
            path_effects=[pe.withStroke(linewidth=3.0, foreground='white'),
                          pe.Normal()],
            zorder=7)

    _pad = data_span * 0.30 + 2.2
    ax_net.set_xlim(min(xs) - _pad, max(xs) + _pad)
    ax_net.set_ylim(min(ys) - _pad, max(ys) + _pad)

    # -- Network key + edge scale (bottom-left) ---
    _legend_items = [
        plt.Line2D([0],[0], color=_EDGE_POS, lw=2.0, linestyle='-',
                   label='Positive pcor (solid)'),
        plt.Line2D([0],[0], color=_EDGE_NEG, lw=2.0, linestyle=_DASH_NEG,
                   label='Negative pcor (dashed)'),
        plt.Line2D([0],[0], marker='o', color='none', markerfacecolor=_RING_UNFILLED,
                   markeredgecolor=_BORDER_POS, markeredgewidth=2.5, markersize=10,
                   label='Ring fill = R² predictability'),
        plt.Line2D([0],[0], marker='o', color='none', markerfacecolor='none',
                   markeredgecolor=_BORDER_POS, markeredgewidth=2.5, markersize=10,
                   label='Blue border = +EI, orange = −EI'),
    ]
    for _sr in [0.10, 0.20, 0.30]:
        _t_s = min(_sr / _EDGE_MAXIMUM, 1.0) ** 0.6
        _sw  = max(_LW_MIN + (_LW_MAX - _LW_MIN) * _t_s, 0.4)
        _sa  = max(_ALPHA_MIN + (_ALPHA_MAX - _ALPHA_MIN) * _t_s, 0.10)
        _sl  = f'|r| = {_sr:.2f}+' if _sr == 0.30 else f'|r| = {_sr:.2f}'
        _legend_items.append(
            plt.Line2D([0],[0], color=_EDGE_POS, lw=_sw, alpha=_sa,
                       solid_capstyle='round', label=_sl))
    fig.legend(handles=_legend_items, fontsize=8.5,
               framealpha=0.92, edgecolor='#CFD8DC', fancybox=True,
               title='Key & Edge Scale', title_fontsize=10, borderpad=0.7,
               labelspacing=0.45, handlelength=1.8, handletextpad=0.5,
               columnspacing=1.2, ncol=4,
               loc='lower left', bbox_to_anchor=(0.01, 0.04))

    n_batch = len(batch_pcor_dict)
    fig.suptitle(
        f'Consensus Partial Correlation Network\n'
        f'(Fisher-z averaged across {n_batch} targets, EBICglasso γ={_EBIC_GAMMA})',
        fontsize=12, fontweight='bold', color=_CLR_SLATE, x=0.50, y=0.97, ha='center')

    ei_sorted_cons = sorted(EI.items(), key=lambda kv: abs(kv[1]), reverse=True)
    # pass None -- consensus has no single target for bivariate r
    _render_centrality_sidebar(ax_ei, ei_sorted_cons, label_of_feat,
                               feat_colors, None, imp_of_feat)

    _save_fig(fig, 'consensus_partial_correlation_network')
    plt.show()
    print(f"  Avg R² (predictability) = {np.mean(list(pred_r2.values())):.3f}")

    # consensus network was missing Burger et al. caption
    _cons_n_subj = X_train[feat_list].replace([np.inf, -np.inf], np.nan).dropna().shape[0]
    _cons_gamma_str = f"γ = {_EBIC_GAMMA} (consensus Fisher-z averaged, {n_batch} targets)"
    _print_network_caption(
        f'Consensus Network ({n_batch} targets)',
        _cons_n_subj, n, _cons_gamma_str,
        'Spectral (Fiedler)', nstats, pred_r2,
        target_lbl='All targets (consensus)',
        nstats_full=nstats_full)
    # disclose that display threshold acts as sparsity mechanism
    # on the Fisher-z averaged matrix (individual EBICglasso models are sparse,
    # but averaging fills in zeros → dense consensus matrix)
    print(f"  | NOTE (consensus-specific): Individual per-target partial correlation")
    print(f"  |   matrices are EBICglasso-sparse, but Fisher-z averaging across targets")
    print(f"  |   produces non-zero entries where not all targets zero the same edges.")
    print(f"  |   Display threshold |pcor| > {_DISPLAY_THRESH} therefore acts as the")
    print(f"  |   effective sparsity mechanism for this consensus network.")

def _plot_consensus_dendrogram(batch_corr_dict, feat_list, label_of_feat, feat_colors):
    """Consensus dendrogram from Fisher-z averaged Pearson correlations."""
    if not batch_corr_dict:
        print("No batch-specific correlation matrices found."); return

    mats = []
    for _, corr in batch_corr_dict.items():
        c = corr.values.copy()
        c = np.clip(c, -0.9999, 0.9999)
        mats.append(np.arctanh(c))
    z_mean = np.nanmean(np.stack(mats, axis=0), axis=0)
    r_mean = np.tanh(z_mean)
    np.fill_diagonal(r_mean, 1.0)

    corr_cons = pd.DataFrame(r_mean, index=feat_list, columns=feat_list)
    dist = (1 - corr_cons.abs()).clip(lower=0)
    np.fill_diagonal(dist.values, 0)

    _link = linkage(squareform(dist.values, checks=False), method='average')
    leaves = dendrogram(_link, no_plot=True)['leaves']
    feat_ord = [feat_list[i] for i in leaves]
    lbl_ord  = [label_of_feat.get(f, f) for f in feat_ord]
    corr_ord = corr_cons.loc[feat_ord, feat_ord]

    n = len(feat_ord)
    n_batch = len(batch_corr_dict)

    cell_px = 72
    max_llen = max(len(l) for l in lbl_ord)
    lbl_inch = max(max_llen * 0.078, 1.8)
    fw = n * (cell_px / 100) + lbl_inch + 2.8
    fh = n * (cell_px / 100) + 3.5

    fig, ax = plt.subplots(figsize=(fw, fh))
    fig.patch.set_facecolor('white')
    im = ax.imshow(corr_ord.values, cmap=_CORR_CMAP, vmin=-1, vmax=1,
                   aspect='auto', interpolation='nearest')
    for i in range(n):
        for j in range(i+1, n):
            ax.add_patch(plt.Rectangle((j-.5, i-.5), 1, 1,
                                       fc='white', ec='white', lw=0, zorder=2))
    fs_ann = max(7, min(11, 14 - n * 0.35))
    for i in range(n):
        for j in range(i+1):
            v = corr_ord.values[i, j]
            tc = 'white' if abs(v) > 0.50 else _CLR_CHARCOAL
            fwgt = 'bold' if abs(v) > 0.28 else 'normal'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color=tc, fontsize=fs_ann, fontweight=fwgt, zorder=3)
    tf = max(8.5, min(11, 130 / max(n, 1)))
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(lbl_ord, rotation=45, ha='right', fontsize=tf, color=_CLR_CHARCOAL)
    ax.set_yticklabels(lbl_ord, fontsize=tf, color=_CLR_CHARCOAL)
    ax.tick_params(axis='both', length=0, pad=8)
    cb = fig.colorbar(im, ax=ax, shrink=0.52, aspect=20, pad=0.03,
                      ticks=[-1, -0.5, 0, 0.5, 1])
    cb.set_label(f'Pearson r (consensus across {n_batch} targets)', fontsize=10,
                 color=_CLR_CHARCOAL, labelpad=8)
    cb.ax.tick_params(labelsize=9, colors=_CLR_CHARCOAL); cb.outline.set_visible(False)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.grid(False)
    ax.set_title(f'Consensus Clustered Pearson Correlation Matrix\n'
                 f'(Fisher-z averaged across {n_batch} targets)',
                 fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=16)
    plt.tight_layout()
    _save_fig(fig, 'consensus_correlation_matrix_heatmap')
    plt.show()

    _dend_labels = [label_of_feat.get(feat_list[i], feat_list[i]) for i in range(n)]
    fig3, ax3 = plt.subplots(figsize=(max(13, n * 1.1 + 2), 7))
    fig3.patch.set_facecolor('white'); ax3.set_facecolor('white')
    _dend = dendrogram(_link, labels=_dend_labels, leaf_rotation=45,
                       leaf_font_size=max(8.5, min(11, 130 / max(n, 1))),
                       color_threshold=0.65 * _link[:, 2].max(),
                       above_threshold_color=_CLR_SILVER, ax=ax3)
    for tl in ax3.get_xticklabels():
        for fi in feat_list:
            if label_of_feat.get(fi, fi) == tl.get_text():
                tl.set_color(feat_colors.get(fi, _CLR_CHARCOAL))
                tl.set_fontweight(600); break
    ax3.set_ylabel('Correlation Distance (1−|r|)', fontsize=11, color=_CLR_CHARCOAL, labelpad=8)
    ax3.set_title(f'Consensus Feature Dendrogram\n'
                  f'(Fisher-z averaged across {n_batch} targets)',
                  fontsize=12, fontweight='bold', color=_CLR_SLATE, pad=14)
    ax3.spines['left'].set_color(_CLR_SILVER); ax3.spines['bottom'].set_color(_CLR_SILVER)
    ax3.yaxis.grid(True, linestyle=':', linewidth=0.5, color=_CLR_SILVER, alpha=0.6)
    ax3.set_axisbelow(True)
    _yb = -0.03 * _link[:, 2].max()
    _lxs = sorted(ax3.get_xticks())
    for xi, li in enumerate(_dend['leaves']):
        fi = feat_list[li]; c = feat_colors.get(fi, '#999')
        if xi < len(_lxs):
            ax3.plot(_lxs[xi], _yb, 's', color=c, markersize=9, clip_on=False, zorder=5)
    plt.tight_layout()
    _save_fig(fig3, 'consensus_feature_dendrogram')
    plt.show()

# ---
# MODULE-LEVEL _display_name
# ---

def _display_name(feat):
    _dn = DISPLAY_NAMES if ('DISPLAY_NAMES' in globals() and isinstance(DISPLAY_NAMES, dict)) else {}
    return _dn.get(feat, feat)

# -- Batch dispatch ---
_is_bv_11 = (
    '_batch_archive' in dir() and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_11:
    try:
        _cbv = cache_load('main_runner')
        if (_cbv and isinstance(_cbv, dict) and _cbv.get('_batch_archive')
                and len(_cbv['_batch_archive']) > 1
                and any(v.get('X_train') is not None for v in _cbv['_batch_archive'].values())):
            _batch_archive = _cbv['_batch_archive']; _is_bv_11 = True
    except Exception: pass

if _is_bv_11:
    _vgs = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split','last_fitted_model',
        'sample_valid','sample_valid_test','sample_valid_train',
    ]}
    print(f"\n{'='*72}\n  BATCH -- {len(_batch_archive)} targets\n{'='*72}")
    for _vi, (_vk, _vd) in enumerate(_batch_archive.items()):
        _vl = _vd.get('target', _vk)
        print(f"\n{'-'*68}\n  [{_vi+1}/{len(_batch_archive)}] {_vl}\n{'-'*68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split',
                    'last_fitted_model','sample_valid','sample_valid_test','sample_valid_train']:
            if _vd.get(_gk) is not None: globals()[_gk] = _vd[_gk]
        globals()['target_options'] = _vl
        if 'timepoint' in _vd: globals()['tp_option'] = _vd['timepoint']
        try: _run_val_11()
        except Exception as _e: print(f"  Warning:  {_vl}: {_e}")
    for _k,_v in _vgs.items(): globals()[_k] = _v
    print(f"\n{'='*72}\n  BATCH COMPLETE -- {len(_batch_archive)} targets\n{'='*72}")

    # -- Consensus network & dendrogram across all batch runs ---
    try:
        if 'batch_pcor_dict' in globals() and globals()['batch_pcor_dict']:
            print(f"\n{'='*72}")
            print("  Consensus Partial Correlation Network  (across all targets)")
            print(f"{'='*72}")
            _dn = DISPLAY_NAMES if ('DISPLAY_NAMES' in globals() and isinstance(DISPLAY_NAMES, dict)) else {}
            _fl = globals().get('batch_feat_list', [])
            _lof = {f: _dn.get(f, f) for f in _fl}
            _iof = {}; _fc = {}
            for _bv in _batch_archive.values():
                _Xt = _bv.get('X_train')
                if _Xt is not None:
                    _fc2, _ = _get_domain_colors(_fl, {f: '#1' for f in _fl}, False)
                    _fc.update(_fc2); break
            # pass batch_corr_dict for precision-implied R²
            _plot_consensus_pcor_network(globals()['batch_pcor_dict'], _fl, _lof,
                                         _iof, _fc,
                                         batch_corr_dict=globals().get('batch_corr_dict'))
    except Exception as _e:
        print(f"  Warning:  Consensus network error: {_e}")
    try:
        if 'batch_corr_dict' in globals() and globals()['batch_corr_dict']:
            print(f"\n{'='*72}")
            print("  Consensus Dendrogram  (across all targets)")
            print(f"{'='*72}")
            _dn = DISPLAY_NAMES if ('DISPLAY_NAMES' in globals() and isinstance(DISPLAY_NAMES, dict)) else {}
            _fl = globals().get('batch_feat_list_corr', [])
            _lof = {f: _dn.get(f, f) for f in _fl}
            _fc = {}
            for _bv in _batch_archive.values():
                _Xt = _bv.get('X_train')
                if _Xt is not None:
                    _fc2, _ = _get_domain_colors(_fl, {f: '#1' for f in _fl}, False)
                    _fc.update(_fc2); break
            _plot_consensus_dendrogram(globals()['batch_corr_dict'], _fl, _lof, _fc)
    except Exception as _e:
        print(f"  Warning:  Consensus dendrogram error: {_e}")

else:
    _run_val_11()

# ---
# BATCH CROSS-TARGET FEATURE IMPORTANCE HEATMAP
# ---

try:
    if _is_bv_11 and len(_batch_archive) >= 2:
        import pandas as pd, numpy as np
        from matplotlib.colors import LinearSegmentedColormap

        _TOP_N_PER_TARGET = 15
        _MAX_COLS = 35

        _feat_records = {}
        _target_labels = {}
        _target_metrics = {}

        for _fk, _fv in _batch_archive.items():
            _tname = _fv.get('target', _fk)
            _tlabel = TARGET_DISPLAY_NAMES.get(_tname, _tname) if 'TARGET_DISPLAY_NAMES' in dir() else _tname
            _target_labels[_tname] = _tlabel

            _cdf = _fv.get('consensus_df')
            if _cdf is not None and hasattr(_cdf, 'columns') and len(_cdf) > 0:
                _imp_col = None
                for _cc in ['consensus_importance', 'mean_importance', 'importance']:
                    if _cc in _cdf.columns:
                        _imp_col = _cc; break
                if _imp_col is None:
                    _num_cols = _cdf.select_dtypes(include='number').columns.tolist()
                    if _num_cols:
                        _imp_col = _num_cols[0]

                if _imp_col and 'feature' in _cdf.columns:
                    _sorted_feats = _cdf.nlargest(_TOP_N_PER_TARGET, _imp_col)
                    _imp_dict = dict(zip(_sorted_feats['feature'], _sorted_feats[_imp_col]))
                    _max_imp = max(_imp_dict.values()) if _imp_dict else 1
                    if _max_imp > 0:
                        _imp_dict = {k: v / _max_imp for k, v in _imp_dict.items()}
                    _feat_records[_tname] = _imp_dict

            _mm = _fv.get('metrics_dict_full', {})
            if _mm.get('category') and _mm.get('metrics'):
                for _ci, _mi in zip(_mm['category'], _mm['metrics']):
                    if isinstance(_mi, dict) and 'meta_model' in _mi:
                        _meta = _mi['meta_model']
                        if isinstance(_meta, dict):
                            _target_metrics[_tname] = _meta.get('auc', _meta.get('r2', 0))

        if len(_feat_records) >= 2:
            from collections import Counter
            _all_feats = Counter()
            for _imp_dict in _feat_records.values():
                for _f in _imp_dict:
                    _all_feats[_f] += 1
            _top_feats = [f for f, _ in _all_feats.most_common(_MAX_COLS)]

            _sorted_targets = sorted(_feat_records.keys(),
                                     key=lambda t: _target_metrics.get(t, 0), reverse=True)

            _matrix = pd.DataFrame(index=[_target_labels[t] for t in _sorted_targets],
                                    columns=_top_feats, dtype=float)
            for _tname in _sorted_targets:
                _tlabel = _target_labels[_tname]
                for _f in _top_feats:
                    _matrix.loc[_tlabel, _f] = _feat_records[_tname].get(_f, 0.0)

            _n_tgts = len(_sorted_targets)
            _n_feats = len(_top_feats)
            _fig_w = max(14, _n_feats * 0.45 + 4)
            _fig_h = max(6, _n_tgts * 0.5 + 2)

            fig_fi, ax_fi = plt.subplots(figsize=(_fig_w, _fig_h))

            _cmap_fi = LinearSegmentedColormap.from_list(
                'imp_cmap', ['#F8F9FB', '#BFDBFE', '#3B82F6', '#1E3A8A'], N=256)

            _im_fi = ax_fi.imshow(_matrix.values.astype(float), aspect='auto',
                                   cmap=_cmap_fi, vmin=0, vmax=1)

            ax_fi.set_xticks(range(_n_feats))
            ax_fi.set_xticklabels(_top_feats, rotation=55, ha='right', fontsize=7.5)
            ax_fi.set_yticks(range(_n_tgts))
            ax_fi.set_yticklabels(_matrix.index, fontsize=9)

            for _ri in range(_n_tgts):
                for _ci in range(_n_feats):
                    _v = _matrix.values[_ri, _ci]
                    if _v > 0.15:
                        _tc = 'white' if _v > 0.55 else '#1E3A8A'
                        ax_fi.text(_ci, _ri, f'{_v:.2f}', ha='center', va='center',
                                   fontsize=6.5, color=_tc, fontweight='bold' if _v > 0.7 else 'normal')

            ax_fi.set_title(f'Cross-Target Feature Importance -- Top {_n_feats} Features\n'
                            f'({_n_tgts} targets, normalized 0-1 within each target)',
                            fontsize=13, fontweight='bold', pad=14)

            _cbar = plt.colorbar(_im_fi, ax=ax_fi, shrink=0.6, pad=0.02)
            _cbar.set_label('Normalized Importance', fontsize=10)

            plt.tight_layout()
            try:
                fig_fi.savefig(f"{results_dir_summary}/batch_feature_importance_heatmap.png",
                               dpi=250, bbox_inches='tight', facecolor='white')
                fig_fi.savefig(f"{results_dir_summary}/batch_feature_importance_heatmap.pdf",
                               bbox_inches='tight', facecolor='white')
                print(f"   Saved: batch_feature_importance_heatmap.png / .pdf")
            except Exception as _e:
                print(f"Warning: {_e}")
            plt.show()
        else:
            print("  Skipped:  Cross-target feature heatmap: <2 targets with importance data")

except Exception as _e_fi:
    print(f"  Warning:  Cross-target feature heatmap error: {_e_fi}")

In [ ]:
#@title Model Diagnostics

# Advanced diagnostics for the most recent analysis run. Togglable:
# base-learner diversity analysis (prediction scatter matrices), bootstrap
# feature importance stability (resampled rankings with violin plots),
# learning curves (performance vs training set size), SHAP beeswarm plots
# (per-model and consensus), and supplementary effect size / feature
# overlap / prediction diagnostic analyses.

# ---
# UI PANEL -- Toggle each analysis and configure sub-parameters
# ---

# -- Master Toggles ---
run_diversity_analysis = True  #@param {type: "boolean"}
run_bootstrap_stability = True  #@param {type: "boolean"}
run_learning_curves = True  #@param {type: "boolean"}
run_shap_beeswarm = True  #@param {type: "boolean"}
run_supplementary = True  #@param {type: "boolean"}

# -- Diversity Analysis Sub-Params ---
show_scatter_matrix = True  #@param {type: "boolean"}
top_n_scatter_pairs = 3  #@param {type: "integer"}

# -- Bootstrap Stability Sub-Params ---
n_bootstraps = 100  #@param [50, 100, 200, 500] {type: "raw"}
top_k_features = 20  #@param [10, 15, 20, 30] {type: "raw"}
show_violin_plot = True  #@param {type: "boolean"}

# -- Learning Curves Sub-Params ---
n_fractions = 10  #@param [5, 8, 10, 15, 20] {type: "raw"}
n_repeats = 5  #@param [3, 5, 10] {type: "raw"}
show_marginal_gains = True  #@param {type: "boolean"}

# -- SHAP Beeswarm Sub-Params ---
top_n_shap = 25  #@param [15, 20, 25, 30, 40] {type: "raw"}
show_weight_chart = True  #@param {type: "boolean"}
use_display_names = True  #@param {type: "boolean"}

# -- Supplementary Sub-Params ---
run_effect_sizes = True  #@param {type: "boolean"}
run_feature_overlap = True  #@param {type: "boolean"}
run_prediction_diagnostics = True  #@param {type: "boolean"}
top_k_overlap = 20  #@param [10, 15, 20, 30] {type: "raw"}

# -- Batch Target Selection ---
#@markdown ### Batch Target Selection
diagnostics_target = "last" #@param {type: "string"}
#@markdown Set to `"last"` for current globals, or a target key from `_batch_archive`.

if (diagnostics_target != "last"
    and '_batch_archive' in dir() and isinstance(_batch_archive, dict)
    and diagnostics_target in _batch_archive):
    _diag_data = _batch_archive[diagnostics_target]
    import copy
    for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                'consensus_df','model_weights','split','exported_within_dict',
                'last_fitted_model','category_results',
                'sample_valid','sample_valid_test','sample_valid_train']:
        if _diag_data.get(_gk) is not None:
            globals()[_gk] = _diag_data[_gk]
    globals()['target_options'] = _diag_data.get('target', diagnostics_target)
    if 'timepoint' in _diag_data:
        globals()['tp_option'] = _diag_data['timepoint']
    print(f" Loaded batch target: {diagnostics_target}")
elif diagnostics_target != "last":
    if '_batch_archive' in dir() and isinstance(_batch_archive, dict):
        print(f"Warning:  Target '{diagnostics_target}' not found in _batch_archive.")
        print(f"   Available: {list(_batch_archive.keys())}")
    else:
        print(f"Warning:  No _batch_archive available. Using current globals.")

# ---
# SHARED IMPORTS
# ---
import time as _time
import warnings as _warnings
import traceback as _traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import squareform
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    roc_auc_score, accuracy_score, f1_score, log_loss
)
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.inspection import permutation_importance
from IPython.display import display, HTML
import copy

_warnings.filterwarnings('ignore', category=FutureWarning)
_warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# ---
# SHARED STYLE
# ---
plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

_SECTION_COLORS = {
    'diversity': '#4C72B0',
    'stability': '#DD8452',
    'learning': '#55A868',
    'shap': '#C44E52',
    'supplementary': '#8172B3',
}

# ---
# SHARED UTILITY FUNCTIONS
# ---

def _detect_task_type(y):
    try:
        _global_fn = globals().get('detect_task_type', None)
        if _global_fn is not None:
            return _global_fn(y)
    except Exception:
        pass
    # Local fallback
    _y_arr = np.asarray(y).ravel()
    _unique = np.unique(_y_arr[~np.isnan(_y_arr)] if np.issubdtype(_y_arr.dtype, np.floating) else _y_arr)
    if len(_unique) <= 20 and np.all(_unique == _unique.astype(int)):
        return 'classification'
    return 'regression'

def _create_lightweight_model(is_classification, n_features):
    """Create a lightweight CatBoost model for bootstrap/learning curve analyses.
    Uses global factory if available, otherwise builds locally."""
    try:
        _global_fn = globals().get('create_lightweight_model', None)
        if _global_fn is not None:
            return _global_fn(is_classification, n_features)
    except Exception:
        pass
    # Local fallback -- CatBoost with conservative settings
    try:
        if is_classification:
            from catboost import CatBoostClassifier
            return CatBoostClassifier(
                iterations=200,
                depth=4,
                learning_rate=0.05,
                l2_leaf_reg=5,
                verbose=0,
                random_seed=42,
                allow_writing_files=False,
                thread_count=1
            )
        else:
            from catboost import CatBoostRegressor
            return CatBoostRegressor(
                iterations=200,
                depth=4,
                learning_rate=0.05,
                l2_leaf_reg=5,
                verbose=0,
                random_seed=42,
                allow_writing_files=False,
                thread_count=1
            )
    except ImportError:
        from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
        if is_classification:
            return GradientBoostingClassifier(
                n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
            )
        else:
            return GradientBoostingRegressor(
                n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
            )

def _get_display_name(name):
    _dn = globals().get('DISPLAY_NAMES', None)
    if _dn is not None and isinstance(_dn, dict):
        return _dn.get(name, name)
    return name

def _feature_names(X):
    """Extract feature names from DataFrame or generate generic names."""
    if hasattr(X, 'columns'):
        return list(X.columns)
    return [f'feature_{i}' for i in range(X.shape[1])]

def _to_numpy(X):
    """Convert to dense numpy array."""
    if hasattr(X, 'values'):
        return X.values
    if hasattr(X, 'toarray'):
        return X.toarray()
    return np.asarray(X)

def _section_header(title, color='#333333'):
    """Print a styled section header."""
    display(HTML(
        f'<div style="background: linear-gradient(135deg, {color}22, {color}11); '
        f'border-left: 4px solid {color}; padding: 12px 16px; margin: 16px 0 12px 0; '
        f'border-radius: 0 8px 8px 0;">'
        f'<h3 style="margin:0; color: {color};"> {title}</h3></div>'
    ))

def _subsection_header(title):
    """Print a styled subsection header."""
    display(HTML(
        f'<div style="border-bottom: 2px solid #ddd; padding-bottom: 4px; '
        f'margin: 12px 0 8px 0;">'
        f'<h4 style="margin:0; color: #555;">▸ {title}</h4></div>'
    ))

def _safe_html_table(df, caption='', max_rows=50):
    """Display a DataFrame as a styled HTML table."""
    _style = df.head(max_rows).style.set_caption(caption).set_table_styles([
        {'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '13px'),
                                            ('text-align', 'left'), ('padding', '6px 0')]},
        {'selector': 'th', 'props': [('background-color', '#f7f7f7'), ('padding', '6px 10px'),
                                      ('border-bottom', '2px solid #ddd'), ('font-size', '11px')]},
        {'selector': 'td', 'props': [('padding', '5px 10px'), ('font-size', '11px')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#fafafa')]},
    ])
    display(_style)

# ---
# UNIFIED GATE -- Check required globals from Cell 6 (Main Analysis Runner)
# ---
_SKIP = False
_GATE_MESSAGES = []

for _req in ['X_train', 'X_test', 'y_train', 'y_test']:
    if _req not in globals():
        _GATE_MESSAGES.append(f"Missing required variable: `{_req}` -- run Cell 6 (Main Analysis Runner) first.")
        _SKIP = True

if _SKIP:
    display(HTML(
        '<div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px; '
        'padding:16px; margin:12px 0;">'
        '<strong>Warning: Diagnostics Hub -- Skipped</strong><br>'
        + '<br>'.join(_GATE_MESSAGES) +
        '</div>'
    ))

# ---
# TIMING INFRASTRUCTURE
# ---
_section_times = {}
_hub_start = _time.time()

if not _SKIP:

    # Resolve shared references
    _X_train_np = _to_numpy(X_train)
    _X_test_np = _to_numpy(X_test)
    _y_train_np = np.asarray(y_train).ravel()
    _y_test_np = np.asarray(y_test).ravel()
    _feat_names = _feature_names(X_train)
    _n_features = _X_train_np.shape[1]

    _task_type = None
    if 'last_fitted_model' in globals() and last_fitted_model is not None and hasattr(last_fitted_model, 'task_type'):
        _task_type = last_fitted_model.task_type
    if _task_type is None:
        _task_type = _detect_task_type(_y_train_np)

    _is_classification = (_task_type == 'classification')

    display(HTML(
        '<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); color: white; '
        'padding: 20px 24px; border-radius: 12px; margin-bottom: 16px;">'
        '<h2 style="margin:0 0 8px 0; color: #e94560;"> Advanced Model Diagnostics Hub</h2>'
        '<p style="margin:0; opacity:0.85;">Consolidated analyses: Diversity · Stability · '
        'Learning Curves · SHAP · Supplementary</p>'
        f'<p style="margin:4px 0 0 0; opacity:0.65; font-size:12px;">'
        f'Task type: <b>{_task_type}</b> | Features: <b>{_n_features}</b> | '
        f'Train samples: <b>{_X_train_np.shape[0]}</b> | Test samples: <b>{_X_test_np.shape[0]}</b></p>'
        '</div>'
    ))

# ---
    # SECTION 1 -- Base Learner Prediction Diversity
# ---
    if run_diversity_analysis:
        _t0 = _time.time()
        _section_header('Base Learner Prediction Diversity', _SECTION_COLORS['diversity'])

        try:
            # --- Extract base learner predictions ---
            _has_model = ('last_fitted_model' in globals() and last_fitted_model is not None
                          and hasattr(last_fitted_model, 'models'))
            if not _has_model:
                display(HTML(
                    '<div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px; '
                    'padding:12px; margin:8px 0;">'
                    '<strong>Warning: Skipped:</strong> <code>last_fitted_model</code> not found or has no '
                    '<code>.models</code> attribute.</div>'
                ))
            else:
                _base_models = last_fitted_model.models  # dict: name -> fitted model
                _model_names = list(_base_models.keys())
                _n_models = len(_model_names)

                print(f"Found {_n_models} base learners: {', '.join(_model_names)}")

                # Collect predictions on test set
                _pred_dict = {}
                for _mname, _mobj in _base_models.items():
                    try:
                        # Determine feature set (TabPFN may use a reduced feature set)
                        _X_input = X_test
                        if _mname.lower() == 'tabpfn' or (hasattr(_mobj, 'selected_features_') and
                                                           _mobj.selected_features_ is not None):
                            try:
                                _sel_feats = _mobj.selected_features_
                                if hasattr(X_test, 'columns'):
                                    _X_input = X_test[_sel_feats]
                                else:
                                    _X_input = _X_test_np[:, _sel_feats]
                            except Exception:
                                _X_input = X_test

                        if _is_classification and hasattr(_mobj, 'predict_proba'):
                            _preds = _mobj.predict_proba(_X_input)
                            if _preds.ndim == 2:
                                _preds = _preds[:, 1] if _preds.shape[1] == 2 else _preds[:, 0]
                        else:
                            try:
                                _preds = _mobj.predict(_X_input)
                            except Exception:
                                # CatBoost fallback: try with Pool
                                try:
                                    from catboost import Pool as _CatPool
                                    _pool = _CatPool(_X_input)
                                    _preds = _mobj.predict(_pool)
                                except Exception as _pool_err:
                                    raise _pool_err

                        _pred_dict[_mname] = np.asarray(_preds, dtype=float).ravel()
                    except Exception as _e:
                        print(f"  Warning: Could not get predictions from {_mname}: {_e}")

                if len(_pred_dict) < 2:
                    display(HTML(
                        '<div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px; '
                        'padding:12px; margin:8px 0;">'
                        '<strong>Warning: Skipped:</strong> Need at least 2 base learners with valid '
                        'predictions for diversity analysis.</div>'
                    ))
                else:
                    _pred_df = pd.DataFrame(_pred_dict)
                    _active_names = list(_pred_dict.keys())
                    _n_active = len(_active_names)

                    # ----- 1a. Pairwise Correlation Heatmap -----
                    _subsection_header('Pairwise Prediction Correlations')
                    _corr_matrix = _pred_df.corr(method='pearson')

                    # Display names for axes
                    _disp_names = [_get_display_name(n) if use_display_names else n for n in _active_names]
                    _corr_display = _corr_matrix.copy()
                    _corr_display.index = _disp_names
                    _corr_display.columns = _disp_names

                    fig, ax = plt.subplots(figsize=(max(8, _n_active * 1.2), max(6, _n_active * 1.0)))
                    _mask = np.triu(np.ones_like(_corr_matrix, dtype=bool), k=1)
                    sns.heatmap(
                        _corr_display, mask=_mask, annot=True, fmt='.3f',
                        cmap='RdYlBu_r', center=0.5, vmin=0, vmax=1,
                        linewidths=0.5, linecolor='white',
                        cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
                        ax=ax, square=True
                    )
                    ax.set_title('Base Learner Prediction Correlation Matrix', fontweight='bold', pad=12)
                    plt.xticks(rotation=45, ha='right')
                    plt.yticks(rotation=0)
                    plt.tight_layout()
                    plt.show()

                    # Compute mean pairwise correlation (lower triangle only)
                    _tril_idx = np.tril_indices(_n_active, k=-1)
                    _corr_vals = _corr_matrix.values[_tril_idx]
                    _mean_corr = np.mean(_corr_vals)
                    _min_corr = np.min(_corr_vals)
                    _max_corr = np.max(_corr_vals)

                    print(f"\nPairwise correlation summary:")
                    print(f"  Mean:  {_mean_corr:.4f}")
                    print(f"  Min:   {_min_corr:.4f}")
                    print(f"  Max:   {_max_corr:.4f}")
                    print(f"  Range: {_max_corr - _min_corr:.4f}")

                    # Diversity interpretation
                    if _mean_corr < 0.5:
                        _div_verdict = "HIGH diversity -- base learners capture substantially different patterns"
                    elif _mean_corr < 0.75:
                        _div_verdict = "MODERATE diversity -- reasonable variation among base learners"
                    elif _mean_corr < 0.9:
                        _div_verdict = "LOW diversity -- base learners largely agree; ensemble gains may be limited"
                    else:
                        _div_verdict = "VERY LOW diversity -- near-redundant predictions; consider replacing similar models"
                    print(f"  Interpretation: {_div_verdict}")

                    # ----- 1b. Individual Model Performance -----
                    _subsection_header('Individual Model Performance')
                    _perf_records = []
                    for _mname in _active_names:
                        _p = _pred_dict[_mname]
                        if _is_classification:
                            try:
                                _auc = roc_auc_score(_y_test_np, _p)
                            except Exception:
                                _auc = np.nan
                            _p_binary = (_p >= 0.5).astype(int)
                            _acc = accuracy_score(_y_test_np, _p_binary)
                            _perf_records.append({
                                'Model': _get_display_name(_mname) if use_display_names else _mname,
                                'AUC': _auc,
                                'Accuracy': _acc,
                            })
                        else:
                            _r2 = r2_score(_y_test_np, _p)
                            _rmse = np.sqrt(mean_squared_error(_y_test_np, _p))
                            _mae = mean_absolute_error(_y_test_np, _p)
                            _perf_records.append({
                                'Model': _get_display_name(_mname) if use_display_names else _mname,
                                'R²': _r2,
                                'RMSE': _rmse,
                                'MAE': _mae,
                            })

                    _perf_df = pd.DataFrame(_perf_records)
                    _safe_html_table(_perf_df, caption='Individual Base Learner Performance on Test Set')

                    # Performance bar chart
                    _metric_col = 'AUC' if _is_classification else 'R²'
                    _perf_sorted = _perf_df.sort_values(_metric_col, ascending=True)

                    fig, ax = plt.subplots(figsize=(10, max(4, _n_active * 0.6)))
                    _bar_colors = plt.cm.viridis(np.linspace(0.3, 0.85, _n_active))
                    bars = ax.barh(
                        _perf_sorted['Model'], _perf_sorted[_metric_col],
                        color=_bar_colors, edgecolor='white', linewidth=0.5
                    )
                    for _bar in bars:
                        _w = _bar.get_width()
                        ax.text(_w + 0.005, _bar.get_y() + _bar.get_height() / 2,
                                f'{_w:.4f}', ha='left', va='center', fontsize=10)
                    ax.set_xlabel(_metric_col)
                    ax.set_title(f'Base Learner {_metric_col} on Test Set', fontweight='bold')
                    ax.set_xlim(left=min(0, _perf_sorted[_metric_col].min() - 0.05))
                    plt.tight_layout()
                    plt.show()

                    # ----- 1c. Meta-Learner Weights -----
                    _subsection_header('Meta-Learner Weights')
                    _meta_weights = None
                    if hasattr(last_fitted_model, 'meta_model') and hasattr(last_fitted_model.meta_model, 'coef_'):
                        _coefs = np.asarray(last_fitted_model.meta_model.coef_).ravel()
                        # Map coefficients to model names
                        # Meta-model input order matches _base_models key order
                        _meta_model_names = list(_base_models.keys())
                        if len(_coefs) == len(_meta_model_names):
                            _meta_weights = dict(zip(_meta_model_names, _coefs))
                        elif len(_coefs) > len(_meta_model_names):
                            # May have extra columns (e.g., original features passed through)
                            _meta_weights = dict(zip(_meta_model_names, _coefs[:len(_meta_model_names)]))
                    elif 'model_weights' in globals() and model_weights is not None:
                        _meta_weights = model_weights

                    if _meta_weights is not None:
                        _weight_df = pd.DataFrame([
                            {'Model': _get_display_name(k) if use_display_names else k,
                             'Weight': v,
                             'Abs Weight': abs(v),
                             'Weight %': 0.0}
                            for k, v in _meta_weights.items() if k in _active_names
                        ])
                        if len(_weight_df) > 0:
                            _total_abs = _weight_df['Abs Weight'].sum()
                            if _total_abs > 0:
                                _weight_df['Weight %'] = (_weight_df['Abs Weight'] / _total_abs * 100)
                            _weight_df = _weight_df.sort_values('Abs Weight', ascending=True)

                            _safe_html_table(
                                _weight_df[['Model', 'Weight', 'Weight %']].round(4),
                                caption='Meta-Learner Coefficients (Weights)'
                            )

                            fig, ax = plt.subplots(figsize=(10, max(4, len(_weight_df) * 0.5)))
                            _wcolors = ['#e74c3c' if w < 0 else '#2ecc71' for w in _weight_df['Weight']]
                            ax.barh(_weight_df['Model'], _weight_df['Weight'], color=_wcolors,
                                    edgecolor='white', linewidth=0.5)
                            ax.axvline(0, color='black', linewidth=0.8, linestyle='-')
                            for i, (_, row) in enumerate(_weight_df.iterrows()):
                                _w = row['Weight']
                                _ha = 'left' if _w >= 0 else 'right'
                                _offset = 0.005 if _w >= 0 else -0.005
                                ax.text(_w + _offset, i, f'{_w:.4f}', ha=_ha, va='center', fontsize=10)
                            ax.set_xlabel('Meta-Learner Coefficient')
                            ax.set_title('Meta-Learner Weights by Base Model', fontweight='bold')
                            plt.tight_layout()
                            plt.show()
                    else:
                        print("Meta-learner weights not available.")

                    # ----- 1d. Scatter Matrix of Most Diverse Pairs -----
                    if show_scatter_matrix and _n_active >= 2:
                        _subsection_header(f'Scatter Plots -- Top {top_n_scatter_pairs} Most Diverse Pairs')

                        # Find most diverse (lowest corr) pairs
                        _pair_corrs = []
                        for i in range(len(_active_names)):
                            for j in range(i + 1, len(_active_names)):
                                _pair_corrs.append((
                                    _active_names[i], _active_names[j],
                                    _corr_matrix.iloc[i, j]
                                ))
                        _pair_corrs.sort(key=lambda x: x[2])
                        _top_pairs = _pair_corrs[:top_n_scatter_pairs]

                        _n_pairs = len(_top_pairs)
                        fig, axes = plt.subplots(1, _n_pairs, figsize=(6 * _n_pairs, 5))
                        if _n_pairs == 1:
                            axes = [axes]

                        for _idx, (_m1, _m2, _r) in enumerate(_top_pairs):
                            ax = axes[_idx]
                            _d1 = _get_display_name(_m1) if use_display_names else _m1
                            _d2 = _get_display_name(_m2) if use_display_names else _m2
                            ax.scatter(_pred_dict[_m1], _pred_dict[_m2],
                                       alpha=0.4, s=20, edgecolors='none', c=_SECTION_COLORS['diversity'])
                            # Identity line
                            _lims = [
                                min(ax.get_xlim()[0], ax.get_ylim()[0]),
                                max(ax.get_xlim()[1], ax.get_ylim()[1])
                            ]
                            ax.plot(_lims, _lims, 'k--', alpha=0.3, linewidth=1)
                            ax.set_xlim(_lims)
                            ax.set_ylim(_lims)
                            ax.set_xlabel(_d1, fontsize=10)
                            ax.set_ylabel(_d2, fontsize=10)
                            ax.set_title(f'r = {_r:.3f}', fontsize=11, fontweight='bold')
                            ax.set_aspect('equal', adjustable='box')

                        fig.suptitle('Most Diverse Base Learner Pairs (Predictions)', fontweight='bold', y=1.02)
                        plt.tight_layout()
                        plt.show()

                    # ----- 1e. Diversity Summary Box -----
                    _subsection_header('Diversity Summary')
                    _ensemble_preds = None
                    if hasattr(last_fitted_model, 'predict'):
                        try:
                            if _is_classification and hasattr(last_fitted_model, 'predict_proba'):
                                _ensemble_preds = last_fitted_model.predict_proba(_X_test_np)
                                if _ensemble_preds.ndim == 2:
                                    _ensemble_preds = _ensemble_preds[:, 1]
                            else:
                                _ensemble_preds = last_fitted_model.predict(_X_test_np)
                            _ensemble_preds = np.asarray(_ensemble_preds).ravel()
                        except Exception:
                            _ensemble_preds = None

                    _summary_rows = []
                    _summary_rows.append({'Metric': 'Number of Base Learners', 'Value': str(_n_active)})
                    _summary_rows.append({'Metric': 'Mean Pairwise Correlation', 'Value': f'{_mean_corr:.4f}'})
                    _summary_rows.append({'Metric': 'Min Pairwise Correlation', 'Value': f'{_min_corr:.4f}'})
                    _summary_rows.append({'Metric': 'Max Pairwise Correlation', 'Value': f'{_max_corr:.4f}'})
                    _summary_rows.append({'Metric': 'Diversity Assessment', 'Value': _div_verdict})

                    if _ensemble_preds is not None:
                        if _is_classification:
                            try:
                                _ens_auc = roc_auc_score(_y_test_np, _ensemble_preds)
                                _best_base = _perf_df['AUC'].max()
                                _summary_rows.append({'Metric': 'Ensemble AUC', 'Value': f'{_ens_auc:.4f}'})
                                _summary_rows.append({'Metric': 'Best Base AUC', 'Value': f'{_best_base:.4f}'})
                                _summary_rows.append({'Metric': 'Ensemble Gain', 'Value': f'{_ens_auc - _best_base:+.4f}'})
                            except Exception:
                                pass
                        else:
                            _ens_r2 = r2_score(_y_test_np, _ensemble_preds)
                            _best_base = _perf_df['R²'].max()
                            _summary_rows.append({'Metric': 'Ensemble R²', 'Value': f'{_ens_r2:.4f}'})
                            _summary_rows.append({'Metric': 'Best Base R²', 'Value': f'{_best_base:.4f}'})
                            _summary_rows.append({'Metric': 'Ensemble Gain', 'Value': f'{_ens_r2 - _best_base:+.4f}'})

                    _safe_html_table(pd.DataFrame(_summary_rows), caption='Prediction Diversity Summary')

        except Exception as _e:
            print(f"Error: Diversity analysis error: {_e}")
            _traceback.print_exc()

        _section_times['Diversity Analysis'] = _time.time() - _t0
        print(f"\n Diversity Analysis completed in {_section_times['Diversity Analysis']:.1f}s")

# ---
    # SECTION 2 -- Feature Importance Stability (Bootstrap)
# ---
    if run_bootstrap_stability:
        _t0 = _time.time()
        _section_header('Feature Importance Stability (Bootstrap)', _SECTION_COLORS['stability'])

        try:
            _is_clf_local = _is_classification
            _model_template = _create_lightweight_model(_is_clf_local, _n_features)

            print(f"Running {n_bootstraps} bootstrap iterations with {type(_model_template).__name__}...")
            print(f"Tracking top {top_k_features} features by importance")

            _n_train = _X_train_np.shape[0]
            _importance_matrix = np.zeros((n_bootstraps, _n_features))
            _rank_matrix = np.zeros((n_bootstraps, _n_features), dtype=int)
            _oob_scores = []

            for _b in range(n_bootstraps):
                _rng = np.random.RandomState(_b)
                _boot_idx = _rng.choice(_n_train, size=_n_train, replace=True)
                _oob_idx = np.setdiff1d(np.arange(_n_train), _boot_idx)

                _X_boot = _X_train_np[_boot_idx]
                _y_boot = _y_train_np[_boot_idx]

                _m = copy.deepcopy(_model_template)
                try:
                    _m.fit(_X_boot, _y_boot)
                except Exception:
                    # Retry without deepcopy
                    _m = _create_lightweight_model(_is_clf_local, _n_features)
                    _m.fit(_X_boot, _y_boot)

                # Extract feature importance
                if hasattr(_m, 'feature_importances_'):
                    _imp = np.asarray(_m.feature_importances_).ravel()
                elif hasattr(_m, 'get_feature_importance'):
                    _imp = np.asarray(_m.get_feature_importance()).ravel()
                elif hasattr(_m, 'coef_'):
                    _imp = np.abs(np.asarray(_m.coef_).ravel())
                else:
                    _imp = np.zeros(_n_features)

                _importance_matrix[_b] = _imp
                # Ranks: rank 1 = most important
                _rank_matrix[_b] = stats.rankdata(-_imp, method='ordinal').astype(int)

                # OOB score
                if len(_oob_idx) > 0:
                    try:
                        _X_oob = _X_train_np[_oob_idx]
                        _y_oob = _y_train_np[_oob_idx]
                        if _is_clf_local:
                            _oob_pred = _m.predict(_X_oob)
                            _oob_scores.append(accuracy_score(_y_oob, _oob_pred))
                        else:
                            _oob_pred = _m.predict(_X_oob)
                            _oob_scores.append(r2_score(_y_oob, _oob_pred))
                    except Exception:
                        pass

                if (_b + 1) % max(1, n_bootstraps // 5) == 0:
                    print(f"  Bootstrap {_b + 1}/{n_bootstraps} complete")

            print(f"Bootstrap complete. OOB scores available: {len(_oob_scores)}")
            if _oob_scores:
                print(f"  OOB {'Accuracy' if _is_clf_local else 'R²'}: "
                      f"mean={np.mean(_oob_scores):.4f}, std={np.std(_oob_scores):.4f}")

            # ---- Compute stability metrics ----
            _mean_importance = _importance_matrix.mean(axis=0)
            _std_importance = _importance_matrix.std(axis=0)
            _mean_rank = _rank_matrix.mean(axis=0)
            _std_rank = _rank_matrix.std(axis=0)
            _median_rank = np.median(_rank_matrix, axis=0)

            # Stability score: lower std_rank relative to mean_rank = more stable
            _cv_rank = np.where(_mean_rank > 0, _std_rank / _mean_rank, 0)
            _stability_score = 1.0 - np.clip(_cv_rank / _cv_rank.max() if _cv_rank.max() > 0 else _cv_rank, 0, 1)

            _stability_df = pd.DataFrame({
                'Feature': _feat_names,
                'Mean Importance': _mean_importance,
                'Std Importance': _std_importance,
                'Mean Rank': _mean_rank,
                'Std Rank': _std_rank,
                'Median Rank': _median_rank,
                'Stability Score': _stability_score,
            }).sort_values('Mean Importance', ascending=False)

            _top_feat_idx = _stability_df.head(top_k_features).index.tolist()
            _top_feat_names = _stability_df.head(top_k_features)['Feature'].tolist()

            # ----- 2a. Rank-Frequency Heatmap -----
            _subsection_header(f'Feature Rank Distribution (Top {top_k_features})')

            # Build rank frequency matrix for top features
            _top_k_actual = min(top_k_features, _n_features)
            _top_indices = [_feat_names.index(fn) for fn in _top_feat_names[:_top_k_actual]]
            _max_rank_show = min(_n_features, 30)
            _rank_freq = np.zeros((_top_k_actual, _max_rank_show))

            for _fi, _global_idx in enumerate(_top_indices):
                for _r in range(_max_rank_show):
                    _rank_freq[_fi, _r] = np.mean(_rank_matrix[:, _global_idx] == (_r + 1))

            _rank_freq_df = pd.DataFrame(
                _rank_freq,
                index=[_get_display_name(fn) if use_display_names else fn for fn in _top_feat_names[:_top_k_actual]],
                columns=[str(r + 1) for r in range(_max_rank_show)]
            )

            fig, ax = plt.subplots(figsize=(max(12, _max_rank_show * 0.5), max(6, _top_k_actual * 0.4)))
            sns.heatmap(
                _rank_freq_df, cmap='YlOrRd', ax=ax, linewidths=0.3,
                cbar_kws={'label': 'Frequency', 'shrink': 0.8},
                xticklabels=True, yticklabels=True
            )
            ax.set_xlabel('Rank Position')
            ax.set_ylabel('Feature')
            ax.set_title('Bootstrap Rank Distribution Heatmap', fontweight='bold')
            plt.xticks(fontsize=8)
            plt.yticks(fontsize=9)
            plt.tight_layout()
            plt.show()

            # ----- 2b. Stability Score Bar Chart -----
            _subsection_header('Feature Stability Scores')
            _top_stab = _stability_df.head(top_k_features).sort_values('Stability Score', ascending=True)
            _disp_labels = [_get_display_name(fn) if use_display_names else fn
                            for fn in _top_stab['Feature']]

            fig, ax = plt.subplots(figsize=(10, max(5, top_k_features * 0.35)))
            _cmap = plt.cm.RdYlGn
            _norm = plt.Normalize(vmin=0, vmax=1)
            _colors = [_cmap(_norm(s)) for s in _top_stab['Stability Score']]
            bars = ax.barh(_disp_labels, _top_stab['Stability Score'], color=_colors,
                           edgecolor='white', linewidth=0.5)
            for _bar in bars:
                _w = _bar.get_width()
                ax.text(_w + 0.01, _bar.get_y() + _bar.get_height() / 2,
                        f'{_w:.3f}', ha='left', va='center', fontsize=9)
            ax.set_xlabel('Stability Score (1 = perfectly stable)')
            ax.set_title(f'Top {top_k_features} Feature Stability Scores ({n_bootstraps} bootstraps)',
                         fontweight='bold')
            ax.set_xlim(0, 1.15)
            plt.tight_layout()
            plt.show()

            # ----- 2c. Violin / Box Plot -----
            if show_violin_plot and _top_k_actual >= 2:
                _subsection_header('Rank Distribution (Violin + Box)')
                _violin_data = []
                for _fn in _top_feat_names[:min(_top_k_actual, 15)]:
                    _gi = _feat_names.index(_fn)
                    _dn = _get_display_name(_fn) if use_display_names else _fn
                    for _r in _rank_matrix[:, _gi]:
                        _violin_data.append({'Feature': _dn, 'Rank': _r})
                _violin_df = pd.DataFrame(_violin_data)

                _n_violin = _violin_df['Feature'].nunique()
                fig, ax = plt.subplots(figsize=(max(10, _n_violin * 0.8), 6))
                _order = [_get_display_name(fn) if use_display_names else fn
                          for fn in _top_feat_names[:min(_top_k_actual, 15)]]
                sns.violinplot(data=_violin_df, x='Feature', y='Rank', order=_order,
                               inner='box', palette='Set2', ax=ax, cut=0)
                ax.set_xlabel('')
                ax.set_ylabel('Rank (1 = most important)')
                ax.set_title('Feature Rank Distributions Across Bootstraps', fontweight='bold')
                ax.invert_yaxis()
                plt.xticks(rotation=45, ha='right', fontsize=9)
                plt.tight_layout()
                plt.show()

            # ----- 2d. Summary Table -----
            _subsection_header('Stability Summary Table')
            _display_stab = _stability_df.head(top_k_features).copy()
            _display_stab['Feature'] = [_get_display_name(fn) if use_display_names else fn
                                        for fn in _display_stab['Feature']]
            _display_stab = _display_stab.round(4)
            _safe_html_table(_display_stab, caption=f'Top {top_k_features} Features -- Bootstrap Stability')

            # ----- 2e. Comparison with Consensus -----
            if 'consensus_df' in dir() and consensus_df is not None:
                _subsection_header('Comparison with Consensus Rankings')
                try:
                    _con = consensus_df.copy()
                    _con_feat_col = None
                    for _c in ['feature', 'Feature', 'feature_name', 'Feature_Name']:
                        if _c in _con.columns:
                            _con_feat_col = _c
                            break
                    if _con_feat_col is not None:
                        _con_top = _con.head(top_k_features)[_con_feat_col].tolist()
                        _boot_top = _top_feat_names[:top_k_features]
                        _overlap = set(_con_top) & set(_boot_top)
                        _jaccard = len(_overlap) / len(set(_con_top) | set(_boot_top)) if len(set(_con_top) | set(_boot_top)) > 0 else 0

                        print(f"Top-{top_k_features} overlap with consensus: {len(_overlap)}/{top_k_features}")
                        print(f"Jaccard similarity: {_jaccard:.4f}")
                        print(f"Overlapping features: {sorted(_overlap)}")

                        _only_boot = set(_boot_top) - set(_con_top)
                        _only_con = set(_con_top) - set(_boot_top)
                        if _only_boot:
                            print(f"In bootstrap only: {sorted(_only_boot)}")
                        if _only_con:
                            print(f"In consensus only: {sorted(_only_con)}")
                except Exception as _e:
                    print(f"  Could not compare with consensus: {_e}")

        except Exception as _e:
            print(f"Error: Bootstrap stability error: {_e}")
            _traceback.print_exc()

        _section_times['Bootstrap Stability'] = _time.time() - _t0
        print(f"\n Bootstrap Stability completed in {_section_times['Bootstrap Stability']:.1f}s")

# ---
    # SECTION 3 -- Learning Curves
# ---
    if run_learning_curves:
        _t0 = _time.time()
        _section_header('Learning Curves', _SECTION_COLORS['learning'])

        try:
            _is_clf_lc = _is_classification
            _model_lc = _create_lightweight_model(_is_clf_lc, _n_features)

            _fractions = np.linspace(0.1, 1.0, n_fractions)
            _n_train_total = _X_train_np.shape[0]
            _metric_name = 'AUC' if _is_clf_lc else 'R²'

            print(f"Training {type(_model_lc).__name__} at {n_fractions} data fractions × {n_repeats} repeats")
            print(f"Training set size: {_n_train_total}")

            _train_scores_all = np.zeros((n_fractions, n_repeats))
            _test_scores_all = np.zeros((n_fractions, n_repeats))

            for _fi, _frac in enumerate(_fractions):
                _n_samples = max(10, int(_frac * _n_train_total))
                for _rep in range(n_repeats):
                    _rng = np.random.RandomState(_rep * 1000 + _fi)
                    _idx = _rng.choice(_n_train_total, size=_n_samples, replace=False)
                    _X_sub = _X_train_np[_idx]
                    _y_sub = _y_train_np[_idx]

                    _m = copy.deepcopy(_model_lc)
                    try:
                        _m.fit(_X_sub, _y_sub)
                    except Exception:
                        _m = _create_lightweight_model(_is_clf_lc, _n_features)
                        _m.fit(_X_sub, _y_sub)

                    # Train score
                    if _is_clf_lc:
                        try:
                            _tr_pred = _m.predict_proba(_X_sub)
                            if _tr_pred.ndim == 2:
                                _tr_pred = _tr_pred[:, 1]
                            _train_scores_all[_fi, _rep] = roc_auc_score(_y_sub, _tr_pred)
                        except Exception:
                            _train_scores_all[_fi, _rep] = accuracy_score(_y_sub, _m.predict(_X_sub))

                        try:
                            _te_pred = _m.predict_proba(_X_test_np)
                            if _te_pred.ndim == 2:
                                _te_pred = _te_pred[:, 1]
                            _test_scores_all[_fi, _rep] = roc_auc_score(_y_test_np, _te_pred)
                        except Exception:
                            _test_scores_all[_fi, _rep] = accuracy_score(_y_test_np, _m.predict(_X_test_np))
                    else:
                        _train_scores_all[_fi, _rep] = r2_score(_y_sub, _m.predict(_X_sub))
                        _test_scores_all[_fi, _rep] = r2_score(_y_test_np, _m.predict(_X_test_np))

                if (_fi + 1) % max(1, n_fractions // 5) == 0:
                    print(f"  Fraction {_fi + 1}/{n_fractions} ({_frac:.1%}, n={_n_samples}) complete")

            # Compute statistics
            _train_mean = _train_scores_all.mean(axis=1)
            _train_std = _train_scores_all.std(axis=1)
            _test_mean = _test_scores_all.mean(axis=1)
            _test_std = _test_scores_all.std(axis=1)

            # 95% CI
            _ci_mult = 1.96
            _train_ci_lo = _train_mean - _ci_mult * _train_std / np.sqrt(n_repeats)
            _train_ci_hi = _train_mean + _ci_mult * _train_std / np.sqrt(n_repeats)
            _test_ci_lo = _test_mean - _ci_mult * _test_std / np.sqrt(n_repeats)
            _test_ci_hi = _test_mean + _ci_mult * _test_std / np.sqrt(n_repeats)

            _sample_sizes = (np.array(_fractions) * _n_train_total).astype(int)
            _sample_sizes = np.maximum(_sample_sizes, 10)

            # ----- Convergence Detection -----
            _test_diffs = np.diff(_test_mean)
            _converged = False
            _convergence_idx = None
            _convergence_threshold = 0.005  # 0.5% improvement

            for _ci in range(len(_test_diffs) - 2):
                if all(abs(_test_diffs[_ci + j]) < _convergence_threshold for j in range(3)):
                    _converged = True
                    _convergence_idx = _ci
                    break

            # Elbow detection via max curvature
            _elbow_idx = None
            if len(_test_mean) >= 4:
                _second_diff = np.diff(_test_mean, n=2)
                if len(_second_diff) > 0:
                    _elbow_idx = np.argmax(np.abs(_second_diff)) + 1

            # ----- 3a. Learning Curve Plot -----
            _subsection_header('Learning Curve with 95% CI')

            fig, ax = plt.subplots(figsize=(12, 7))

            # CI bands
            ax.fill_between(_sample_sizes, _train_ci_lo, _train_ci_hi,
                            alpha=0.15, color='#2ecc71', label='Train 95% CI')
            ax.fill_between(_sample_sizes, _test_ci_lo, _test_ci_hi,
                            alpha=0.15, color='#e74c3c', label='Test 95% CI')

            # Mean lines
            ax.plot(_sample_sizes, _train_mean, 'o-', color='#2ecc71', linewidth=2,
                    markersize=5, label=f'Train {_metric_name} (mean)')
            ax.plot(_sample_sizes, _test_mean, 's-', color='#e74c3c', linewidth=2,
                    markersize=5, label=f'Test {_metric_name} (mean)')

            # Gap annotation
            _gap = _train_mean - _test_mean
            _final_gap = _gap[-1]
            ax.annotate(
                f'Gap = {_final_gap:.4f}',
                xy=(_sample_sizes[-1], (_train_mean[-1] + _test_mean[-1]) / 2),
                xytext=(15, 0), textcoords='offset points',
                fontsize=10, ha='left', va='center',
                arrowprops=dict(arrowstyle='->', color='gray'),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
            )

            # Convergence marker
            if _converged and _convergence_idx is not None:
                ax.axvline(_sample_sizes[_convergence_idx], color='purple', linestyle='--',
                           alpha=0.5, label=f'Convergence (n≈{_sample_sizes[_convergence_idx]})')

            # Elbow marker
            if _elbow_idx is not None and _elbow_idx < len(_sample_sizes):
                ax.axvline(_sample_sizes[_elbow_idx], color='orange', linestyle=':',
                           alpha=0.5, label=f'Elbow (n≈{_sample_sizes[_elbow_idx]})')

            ax.set_xlabel('Training Set Size', fontsize=12)
            ax.set_ylabel(_metric_name, fontsize=12)
            ax.set_title(f'Learning Curve -- {type(_model_lc).__name__} ({n_repeats} repeats)',
                         fontweight='bold', fontsize=14)
            ax.legend(loc='lower right', frameon=True, framealpha=0.9)
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

            # ----- 3b. Marginal Gains -----
            if show_marginal_gains and len(_test_mean) >= 3:
                _subsection_header('Marginal Gains per Data Increment')
                _marginal = np.diff(_test_mean)
                _midpoints = (_sample_sizes[:-1] + _sample_sizes[1:]) / 2

                fig, ax = plt.subplots(figsize=(10, 4))
                _bar_colors = ['#2ecc71' if m > 0 else '#e74c3c' for m in _marginal]
                ax.bar(_midpoints, _marginal, width=(_sample_sizes[1] - _sample_sizes[0]) * 0.6,
                       color=_bar_colors, edgecolor='white', linewidth=0.5, alpha=0.8)
                ax.axhline(0, color='black', linewidth=0.8)
                ax.axhline(_convergence_threshold, color='purple', linestyle='--', alpha=0.4,
                           label=f'Convergence threshold ({_convergence_threshold})')
                ax.axhline(-_convergence_threshold, color='purple', linestyle='--', alpha=0.4)
                ax.set_xlabel('Training Set Size (midpoint)')
                ax.set_ylabel(f'Δ{_metric_name}')
                ax.set_title('Marginal Test Performance Gain per Data Increment', fontweight='bold')
                ax.legend()
                plt.tight_layout()
                plt.show()

            # ----- 3c. Convergence Assessment Box -----
            _subsection_header('Convergence Assessment')
            _assess_rows = [
                {'Metric': 'Final Train Score', 'Value': f'{_train_mean[-1]:.4f}'},
                {'Metric': 'Final Test Score', 'Value': f'{_test_mean[-1]:.4f}'},
                {'Metric': 'Train-Test Gap', 'Value': f'{_final_gap:.4f}'},
                {'Metric': 'Converged?', 'Value': 'Yes' if _converged else 'No'},
            ]
            if _converged and _convergence_idx is not None:
                _assess_rows.append({
                    'Metric': 'Convergence Point',
                    'Value': f'n ≈ {_sample_sizes[_convergence_idx]} ({_fractions[_convergence_idx]:.0%} of data)'
                })
            if _elbow_idx is not None and _elbow_idx < len(_sample_sizes):
                _assess_rows.append({
                    'Metric': 'Elbow Point',
                    'Value': f'n ≈ {_sample_sizes[_elbow_idx]} ({_fractions[_elbow_idx]:.0%} of data)'
                })

            # Bias-variance diagnosis
            if _final_gap > 0.15:
                _bv_diagnosis = "HIGH VARIANCE -- model overfits; more data or regularization may help"
            elif _final_gap > 0.05:
                _bv_diagnosis = "MODERATE VARIANCE -- some overfitting present"
            elif _test_mean[-1] < 0.3:
                _bv_diagnosis = "HIGH BIAS -- model underfits; consider more complexity or features"
            else:
                _bv_diagnosis = "GOOD FIT -- balanced bias-variance tradeoff"
            _assess_rows.append({'Metric': 'Bias-Variance Diagnosis', 'Value': _bv_diagnosis})

            # Score improvement from 50% to 100% data
            if n_fractions >= 5:
                _half_idx = n_fractions // 2
                _improvement = _test_mean[-1] - _test_mean[_half_idx]
                _assess_rows.append({
                    'Metric': f'Gain from {_fractions[_half_idx]:.0%} → 100% data',
                    'Value': f'{_improvement:+.4f}'
                })

            _safe_html_table(pd.DataFrame(_assess_rows), caption='Learning Curve Summary')

        except Exception as _e:
            print(f"Error: Learning curves error: {_e}")
            _traceback.print_exc()

        _section_times['Learning Curves'] = _time.time() - _t0
        print(f"\n Learning Curves completed in {_section_times['Learning Curves']:.1f}s")

# ---
    # SECTION 4 -- Ensemble-Weighted SHAP Beeswarm
# ---
    if run_shap_beeswarm:
        _t0 = _time.time()
        _section_header('Ensemble-Weighted SHAP Beeswarm', _SECTION_COLORS['shap'])

        try:
            _shap_available = True
            try:
                import shap
            except ImportError:
                _shap_available = False
                display(HTML(
                    '<div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px; '
                    'padding:12px; margin:8px 0;">'
                    '<strong>Warning: SHAP library not available.</strong> '
                    'Install with <code>pip install shap</code>.</div>'
                ))

            _has_model_shap = ('last_fitted_model' in globals() and last_fitted_model is not None
                               and hasattr(last_fitted_model, 'models'))

            if _shap_available and _has_model_shap:
                _base_models_shap = last_fitted_model.models
                _shap_model_names = list(_base_models_shap.keys())

                # Get meta-learner weights for weighting SHAP values
                _shap_weights = {}
                if hasattr(last_fitted_model, 'meta_model') and hasattr(last_fitted_model.meta_model, 'coef_'):
                    _coefs = np.asarray(last_fitted_model.meta_model.coef_).ravel()
                    for _i, _mn in enumerate(_shap_model_names):
                        if _i < len(_coefs):
                            _shap_weights[_mn] = _coefs[_i]
                elif 'model_weights' in dir() and model_weights is not None:
                    _shap_weights = dict(model_weights)

                # Normalize weights to sum to 1 (absolute values)
                if _shap_weights:
                    _total_w = sum(abs(v) for v in _shap_weights.values())
                    if _total_w > 0:
                        _shap_weights = {k: abs(v) / _total_w for k, v in _shap_weights.items()}
                else:
                    # Equal weights fallback
                    _n_shap_models = len(_shap_model_names)
                    _shap_weights = {mn: 1.0 / _n_shap_models for mn in _shap_model_names}

                print(f"Computing SHAP values for {len(_shap_model_names)} base models...")
                print(f"Weights: {', '.join(f'{k}: {v:.3f}' for k, v in _shap_weights.items())}")

                # Use a subsample for SHAP computation efficiency
                _shap_max_samples = min(200, _X_test_np.shape[0])
                _shap_idx = np.random.RandomState(42).choice(
                    _X_test_np.shape[0], size=_shap_max_samples, replace=False
                )
                _X_shap = _X_test_np[_shap_idx]
                _X_shap_bg = _X_train_np[:min(100, _X_train_np.shape[0])]

                _per_model_shap = {}
                _excluded_models = []

                for _mn in _shap_model_names:
                    _model_obj = _base_models_shap[_mn]

                    # Skip TabPFN (not supported by standard SHAP explainers)
                    _model_class_name = type(_model_obj).__name__
                    if 'tabpfn' in _model_class_name.lower() or 'TabPFN' in _model_class_name:
                        print(f"  Skipping {_mn} (TabPFN not supported by SHAP)")
                        _excluded_models.append(_mn)
                        continue

                    try:
                        # Try TreeExplainer first (fast for tree-based models)
                        _is_tree = any(
                            hasattr(_model_obj, attr) for attr in
                            ['feature_importances_', 'get_booster', 'get_feature_importance']
                        ) and hasattr(_model_obj, 'predict')

                        _tree_types = (
                            'catboost', 'xgboost', 'lgbm', 'lightgbm', 'randomforest',
                            'gradientboosting', 'extratrees', 'decisiontree'
                        )
                        _is_tree = _is_tree or any(
                            t in _model_class_name.lower() for t in _tree_types
                        )

                        if _is_tree:
                            try:
                                _explainer = shap.TreeExplainer(_model_obj)
                                _sv = _explainer.shap_values(_X_shap)
                                if isinstance(_sv, list):
                                    _sv = _sv[1] if len(_sv) == 2 else _sv[0]
                                _per_model_shap[_mn] = np.asarray(_sv)
                                print(f"  {_mn}: TreeExplainer OK, shape {_per_model_shap[_mn].shape}")
                                continue
                            except Exception as _tree_err:
                                print(f"  {_mn}: TreeExplainer failed ({_tree_err}), trying LinearExplainer...")

                        # Try LinearExplainer
                        _is_linear = hasattr(_model_obj, 'coef_')
                        if _is_linear:
                            try:
                                _explainer = shap.LinearExplainer(_model_obj, _X_shap_bg)
                                _sv = _explainer.shap_values(_X_shap)
                                if isinstance(_sv, list):
                                    _sv = _sv[1] if len(_sv) == 2 else _sv[0]
                                _per_model_shap[_mn] = np.asarray(_sv)
                                print(f"  {_mn}: LinearExplainer OK, shape {_per_model_shap[_mn].shape}")
                                continue
                            except Exception as _lin_err:
                                print(f"  {_mn}: LinearExplainer failed ({_lin_err}), trying KernelExplainer...")

                        # Fallback: KernelExplainer (slow but universal)
                        _predict_fn = _model_obj.predict
                        if _is_classification and hasattr(_model_obj, 'predict_proba'):
                            _predict_fn = lambda x, _m=_model_obj: _m.predict_proba(x)[:, 1]

                        _bg = shap.kmeans(_X_shap_bg, min(10, _X_shap_bg.shape[0]))
                        _explainer = shap.KernelExplainer(_predict_fn, _bg)
                        _sv = _explainer.shap_values(_X_shap, nsamples=100)
                        if isinstance(_sv, list):
                            _sv = _sv[1] if len(_sv) == 2 else _sv[0]
                        _per_model_shap[_mn] = np.asarray(_sv)
                        print(f"  {_mn}: KernelExplainer OK, shape {_per_model_shap[_mn].shape}")

                    except Exception as _e:
                        print(f"  {_mn}: All SHAP methods failed: {_e}")
                        _excluded_models.append(_mn)

                if len(_per_model_shap) == 0:
                    display(HTML(
                        '<div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px; '
                        'padding:12px; margin:8px 0;">'
                        '<strong>Warning: Could not compute SHAP values for any model.</strong></div>'
                    ))
                else:
                    # ----- Compute weighted SHAP -----
                    _weighted_shap = np.zeros((_shap_max_samples, _n_features))
                    _total_weight_used = 0
                    for _mn, _sv in _per_model_shap.items():
                        _w = _shap_weights.get(_mn, 0)
                        if _sv.shape == _weighted_shap.shape:
                            _weighted_shap += _w * _sv
                            _total_weight_used += _w

                    if _total_weight_used > 0:
                        _weighted_shap /= _total_weight_used

                    # Feature importance ranking by mean |SHAP|
                    _mean_abs_shap = np.mean(np.abs(_weighted_shap), axis=0)
                    _shap_order = np.argsort(-_mean_abs_shap)
                    _top_shap_idx = _shap_order[:top_n_shap]

                    # ----- 4a. Manual Beeswarm Plot -----
                    _subsection_header(f'Ensemble-Weighted SHAP Beeswarm (Top {top_n_shap})')

                    _n_show = min(top_n_shap, len(_top_shap_idx))
                    _shap_display = _weighted_shap[:, _top_shap_idx[:_n_show]]
                    _feat_display = [_feat_names[i] for i in _top_shap_idx[:_n_show]]
                    _feat_display_names = [_get_display_name(fn) if use_display_names else fn
                                           for fn in _feat_display]
                    _X_shap_display = _X_shap[:, _top_shap_idx[:_n_show]]

                    # Beeswarm: each feature on y-axis, SHAP value on x-axis, color by feature value
                    fig_width = 12 if not show_weight_chart else 15
                    if show_weight_chart:
                        fig, (ax_bee, ax_w) = plt.subplots(
                            1, 2, figsize=(fig_width, max(7, _n_show * 0.35)),
                            gridspec_kw={'width_ratios': [4, 1], 'wspace': 0.05}
                        )
                    else:
                        fig, ax_bee = plt.subplots(figsize=(12, max(7, _n_show * 0.35)))

                    _cmap_beeswarm = plt.cm.RdBu_r
                    for _fi in range(_n_show):
                        _y_pos = _n_show - 1 - _fi
                        _shap_vals = _shap_display[:, _fi]
                        _feat_vals = _X_shap_display[:, _fi]

                        # Normalize feature values to [0, 1] for coloring
                        _fmin, _fmax = np.nanmin(_feat_vals), np.nanmax(_feat_vals)
                        if _fmax > _fmin:
                            _normed = (_feat_vals - _fmin) / (_fmax - _fmin)
                        else:
                            _normed = np.full_like(_feat_vals, 0.5)

                        # Add jitter to y position
                        _jitter = np.random.RandomState(42 + _fi).uniform(-0.25, 0.25, size=len(_shap_vals))
                        _y_jittered = _y_pos + _jitter

                        # Sort by SHAP value so high-value points are drawn on top
                        _sort_idx = np.argsort(np.abs(_shap_vals))
                        ax_bee.scatter(
                            _shap_vals[_sort_idx], _y_jittered[_sort_idx],
                            c=_normed[_sort_idx], cmap=_cmap_beeswarm,
                            s=8, alpha=0.6, edgecolors='none', rasterized=True
                        )

                    ax_bee.set_yticks(range(_n_show))
                    ax_bee.set_yticklabels(list(reversed(_feat_display_names)), fontsize=9)
                    ax_bee.set_xlabel('SHAP Value (ensemble-weighted)', fontsize=11)
                    ax_bee.axvline(0, color='gray', linewidth=0.8, linestyle='-')
                    ax_bee.set_title(f'Ensemble-Weighted SHAP Beeswarm (Top {_n_show})',
                                     fontweight='bold', fontsize=13)

                    # Colorbar
                    _sm = plt.cm.ScalarMappable(cmap=_cmap_beeswarm, norm=plt.Normalize(0, 1))
                    _sm.set_array([])
                    _cbar = plt.colorbar(_sm, ax=ax_bee, shrink=0.5, aspect=30, pad=0.02)
                    _cbar.set_label('Feature Value (normalized)', fontsize=10)
                    _cbar.set_ticks([0, 0.5, 1])
                    _cbar.set_ticklabels(['Low', 'Mid', 'High'])

                    # ----- 4b. Weight sidebar -----
                    if show_weight_chart:
                        _shap_importance = _mean_abs_shap[_top_shap_idx[:_n_show]]
                        _shap_imp_reversed = list(reversed(_shap_importance))

                        ax_w.barh(range(_n_show), _shap_imp_reversed,
                                  color=_SECTION_COLORS['shap'], alpha=0.7, edgecolor='white')
                        ax_w.set_yticks([])
                        ax_w.set_xlabel('Mean |SHAP|', fontsize=10)
                        ax_w.set_title('Importance', fontweight='bold', fontsize=10)

                    plt.tight_layout()
                    plt.show()

                    # ----- 4c. SHAP Summary Table -----
                    _subsection_header('SHAP Feature Importance Summary')
                    _shap_table_rows = []
                    for _rank, _gi in enumerate(_top_shap_idx[:_n_show]):
                        _fn = _feat_names[_gi]
                        _dn = _get_display_name(_fn) if use_display_names else _fn
                        _shap_table_rows.append({
                            'Rank': _rank + 1,
                            'Feature': _dn,
                            'Mean |SHAP|': _mean_abs_shap[_gi],
                            'Max |SHAP|': np.max(np.abs(_weighted_shap[:, _gi])),
                            'Std SHAP': np.std(_weighted_shap[:, _gi]),
                        })
                    _shap_table_df = pd.DataFrame(_shap_table_rows).round(4)
                    _safe_html_table(_shap_table_df, caption=f'Top {_n_show} Ensemble-Weighted SHAP Features')

                    # Report excluded models
                    if _excluded_models:
                        print(f"\nModels excluded from SHAP: {', '.join(_excluded_models)}")
                    print(f"Models included: {', '.join(_per_model_shap.keys())}")
                    print(f"SHAP computed on {_shap_max_samples} test samples")

        except Exception as _e:
            print(f"Error: SHAP Beeswarm error: {_e}")
            _traceback.print_exc()

        _section_times['SHAP Beeswarm'] = _time.time() - _t0
        print(f"\n SHAP Beeswarm completed in {_section_times['SHAP Beeswarm']:.1f}s")

# ---
    # SECTION 5 -- Supplementary Quick Analyses
# ---
    if run_supplementary:
        _t0 = _time.time()
        _section_header('Supplementary Quick Analyses', _SECTION_COLORS['supplementary'])

# ---
        # 5A -- Effect Size Summary
# ---
        if run_effect_sizes:
            _subsection_header('5A. Effect Size Summary')

            try:
                # Get current target info
                _current_target = None
                if 'tp_option' in globals() and tp_option is not None:
                    _current_target = tp_option
                elif 'target_options' in globals() and target_options is not None and len(target_options) > 0:
                    _current_target = target_options[0]

                print(f"Current target: {_current_target}")

                # Compute effect sizes for current target
                _effect_rows = []

                if not _is_classification:
                    # Cohen's f² from R²
                    _model_for_pred = None
                    if 'last_fitted_model' in globals() and last_fitted_model is not None:
                        _model_for_pred = last_fitted_model

                    if _model_for_pred is not None:
                        try:
                            _y_pred_eff = _model_for_pred.predict(_X_test_np).ravel()
                            _r2_eff = r2_score(_y_test_np, _y_pred_eff)
                            _f2 = _r2_eff / (1 - _r2_eff) if _r2_eff < 1.0 else float('inf')

                            # Cohen's f² benchmarks: 0.02 small, 0.15 medium, 0.35 large
                            if _f2 >= 0.35:
                                _f2_label = 'Large'
                            elif _f2 >= 0.15:
                                _f2_label = 'Medium'
                            elif _f2 >= 0.02:
                                _f2_label = 'Small'
                            else:
                                _f2_label = 'Negligible'

                            _effect_rows.append({
                                'Target': _current_target or 'Current',
                                'R²': round(_r2_eff, 4),
                                "Cohen's f²": round(_f2, 4),
                                'Effect Size': _f2_label,
                            })
                        except Exception as _e_eff:
                            print(f"  Could not compute effect size: {_e_eff}")

                    # Also compute Cohen's d between predicted and actual
                    if _model_for_pred is not None:
                        try:
                            _y_pred_d = _model_for_pred.predict(_X_test_np).ravel()
                            _pooled_std = np.sqrt(
                                (np.std(_y_test_np) ** 2 + np.std(_y_pred_d) ** 2) / 2
                            )
                            if _pooled_std > 0:
                                _cohens_d = (np.mean(_y_pred_d) - np.mean(_y_test_np)) / _pooled_std
                            else:
                                _cohens_d = 0.0
                            print(f"  Cohen's d (predicted vs actual means): {_cohens_d:.4f}")
                        except Exception:
                            pass

                else:
                    # Classification: compute effect size from AUC
                    if 'last_fitted_model' in globals() and last_fitted_model is not None:
                        try:
                            if hasattr(last_fitted_model, 'predict_proba'):
                                _y_prob = last_fitted_model.predict_proba(_X_test_np)
                                if _y_prob.ndim == 2:
                                    _y_prob = _y_prob[:, 1]
                            else:
                                _y_prob = last_fitted_model.predict(_X_test_np)
                            _auc_eff = roc_auc_score(_y_test_np, _y_prob)
                            # Convert AUC to Cohen's d (approximate: d ≈ sqrt(2) * Φ⁻¹(AUC))
                            _cohens_d_from_auc = np.sqrt(2) * stats.norm.ppf(_auc_eff)

                            if abs(_cohens_d_from_auc) >= 0.8:
                                _d_label = 'Large'
                            elif abs(_cohens_d_from_auc) >= 0.5:
                                _d_label = 'Medium'
                            elif abs(_cohens_d_from_auc) >= 0.2:
                                _d_label = 'Small'
                            else:
                                _d_label = 'Negligible'

                            _effect_rows.append({
                                'Target': _current_target or 'Current',
                                'AUC': round(_auc_eff, 4),
                                "Cohen's d (from AUC)": round(_cohens_d_from_auc, 4),
                                'Effect Size': _d_label,
                            })
                        except Exception as _e_eff:
                            print(f"  Could not compute classification effect size: {_e_eff}")

                # Batch mode: process other targets from _batch_archive
                if '_batch_archive' in globals() and _batch_archive is not None and isinstance(_batch_archive, dict):
                    print(f"  Processing batch archive ({len(_batch_archive)} targets)...")
                    for _bt, _bdata in _batch_archive.items():
                        if _bt == _current_target:
                            continue
                        try:
                            _b_metrics = None
                            if isinstance(_bdata, dict):
                                _b_metrics = _bdata.get('metrics', _bdata.get('metrics_dict_full', None))
                                if _b_metrics is None and 'r2' in _bdata:
                                    _b_metrics = _bdata

                            if _b_metrics is not None:
                                _b_r2 = None
                                _b_auc = None
                                for _key in ['r2', 'R²', 'test_r2', 'R2', 'r2_score']:
                                    if _key in _b_metrics:
                                        _b_r2 = _b_metrics[_key]
                                        break
                                for _key in ['auc', 'AUC', 'test_auc', 'roc_auc']:
                                    if _key in _b_metrics:
                                        _b_auc = _b_metrics[_key]
                                        break

                                if _b_r2 is not None:
                                    _b_f2 = _b_r2 / (1 - _b_r2) if _b_r2 < 1.0 else float('inf')
                                    if _b_f2 >= 0.35:
                                        _b_f2_label = 'Large'
                                    elif _b_f2 >= 0.15:
                                        _b_f2_label = 'Medium'
                                    elif _b_f2 >= 0.02:
                                        _b_f2_label = 'Small'
                                    else:
                                        _b_f2_label = 'Negligible'
                                    _effect_rows.append({
                                        'Target': _bt,
                                        'R²': round(_b_r2, 4),
                                        "Cohen's f²": round(_b_f2, 4),
                                        'Effect Size': _b_f2_label,
                                    })
                                elif _b_auc is not None:
                                    _b_d = np.sqrt(2) * stats.norm.ppf(max(0.501, min(0.999, _b_auc)))
                                    if abs(_b_d) >= 0.8:
                                        _b_d_label = 'Large'
                                    elif abs(_b_d) >= 0.5:
                                        _b_d_label = 'Medium'
                                    elif abs(_b_d) >= 0.2:
                                        _b_d_label = 'Small'
                                    else:
                                        _b_d_label = 'Negligible'
                                    _effect_rows.append({
                                        'Target': _bt,
                                        'AUC': round(_b_auc, 4),
                                        "Cohen's d (from AUC)": round(_b_d, 4),
                                        'Effect Size': _b_d_label,
                                    })
                        except Exception as _e_batch:
                            print(f"  Batch target {_bt}: {_e_batch}")

                if _effect_rows:
                    _effect_df = pd.DataFrame(_effect_rows)
                    _safe_html_table(_effect_df, caption='Effect Size Summary Across Targets')

                    # Effect size bar chart
                    _metric_col_eff = "Cohen's f²" if "Cohen's f²" in _effect_df.columns else "Cohen's d (from AUC)"
                    if _metric_col_eff in _effect_df.columns:
                        _eff_sorted = _effect_df.dropna(subset=[_metric_col_eff]).sort_values(
                            _metric_col_eff, ascending=True
                        )
                        if len(_eff_sorted) > 0:
                            fig, ax = plt.subplots(figsize=(10, max(3, len(_eff_sorted) * 0.5)))
                            _eff_colors = []
                            for _es in _eff_sorted['Effect Size']:
                                if _es == 'Large':
                                    _eff_colors.append('#27ae60')
                                elif _es == 'Medium':
                                    _eff_colors.append('#f39c12')
                                elif _es == 'Small':
                                    _eff_colors.append('#e67e22')
                                else:
                                    _eff_colors.append('#e74c3c')

                            ax.barh(_eff_sorted['Target'], _eff_sorted[_metric_col_eff],
                                    color=_eff_colors, edgecolor='white', linewidth=0.5)

                            # Reference lines for Cohen's f² thresholds
                            if "f²" in _metric_col_eff:
                                for _thresh, _lbl in [(0.02, 'Small'), (0.15, 'Medium'), (0.35, 'Large')]:
                                    ax.axvline(_thresh, color='gray', linestyle=':', alpha=0.4)
                                    ax.text(_thresh, len(_eff_sorted) - 0.5, _lbl,
                                            fontsize=8, color='gray', ha='center')

                            ax.set_xlabel(_metric_col_eff)
                            ax.set_title('Effect Sizes Across Targets', fontweight='bold')
                            plt.tight_layout()
                            plt.show()
                else:
                    print("  No effect size data available.")

            except Exception as _e:
                print(f"Error: Effect size error: {_e}")
                _traceback.print_exc()

# ---
        # 5B -- Feature Overlap Across Targets
# ---
        if run_feature_overlap:
            _subsection_header('5B. Feature Overlap Across Targets')

            try:
                _target_features = {}

                # Current target features from consensus_df
                if 'consensus_df' in globals() and consensus_df is not None:
                    _con = consensus_df.copy()
                    _con_feat_col = None
                    for _c in ['feature', 'Feature', 'feature_name', 'Feature_Name']:
                        if _c in _con.columns:
                            _con_feat_col = _c
                            break
                    if _con_feat_col is not None:
                        _current_target_fo = None
                        if 'tp_option' in globals() and tp_option is not None:
                            _current_target_fo = tp_option
                        elif 'target_options' in globals() and target_options is not None and len(target_options) > 0:
                            _current_target_fo = target_options[0]
                        if _current_target_fo:
                            _target_features[_current_target_fo] = _con.head(top_k_overlap)[_con_feat_col].tolist()

                # Batch targets from _batch_archive
                if '_batch_archive' in globals() and _batch_archive is not None and isinstance(_batch_archive, dict):
                    for _bt, _bdata in _batch_archive.items():
                        if _bt in _target_features:
                            continue
                        try:
                            _b_con = None
                            if isinstance(_bdata, dict):
                                _b_con = _bdata.get('consensus_df', None)
                                if _b_con is None:
                                    _b_con = _bdata.get('consensus', None)
                                if _b_con is None and 'results_summary' in _bdata:
                                    _rs = _bdata['results_summary']
                                    if isinstance(_rs, pd.DataFrame):
                                        _b_con = _rs

                            if _b_con is not None and isinstance(_b_con, pd.DataFrame):
                                _b_feat_col = None
                                for _c in ['feature', 'Feature', 'feature_name', 'Feature_Name']:
                                    if _c in _b_con.columns:
                                        _b_feat_col = _c
                                        break
                                if _b_feat_col is not None:
                                    _target_features[_bt] = _b_con.head(top_k_overlap)[_b_feat_col].tolist()
                        except Exception:
                            pass

                if len(_target_features) < 2:
                    print(f"  Only {len(_target_features)} target(s) with feature lists found. "
                          "Need ≥ 2 for overlap analysis.")
                    if _target_features:
                        print(f"  Available: {list(_target_features.keys())}")
                else:
                    _overlap_targets = list(_target_features.keys())
                    _n_targets = len(_overlap_targets)
                    print(f"  Comparing top-{top_k_overlap} features across {_n_targets} targets: "
                          f"{', '.join(_overlap_targets)}")

                    # Compute Jaccard similarity matrix
                    _jaccard_matrix = np.zeros((_n_targets, _n_targets))
                    for _i in range(_n_targets):
                        for _j in range(_n_targets):
                            _set_i = set(_target_features[_overlap_targets[_i]])
                            _set_j = set(_target_features[_overlap_targets[_j]])
                            _union = _set_i | _set_j
                            _jaccard_matrix[_i, _j] = len(_set_i & _set_j) / len(_union) if len(_union) > 0 else 0

                    _jaccard_df = pd.DataFrame(
                        _jaccard_matrix,
                        index=_overlap_targets,
                        columns=_overlap_targets
                    ).round(3)

                    # Heatmap
                    fig, ax = plt.subplots(figsize=(max(6, _n_targets * 1.2), max(5, _n_targets * 1.0)))
                    sns.heatmap(
                        _jaccard_df, annot=True, fmt='.3f', cmap='YlGnBu',
                        vmin=0, vmax=1, linewidths=0.5, linecolor='white',
                        cbar_kws={'label': 'Jaccard Similarity', 'shrink': 0.8},
                        ax=ax, square=True
                    )
                    ax.set_title(f'Feature Overlap (Top {top_k_overlap}) -- Jaccard Similarity',
                                 fontweight='bold')
                    plt.xticks(rotation=45, ha='right')
                    plt.tight_layout()
                    plt.show()

                    _safe_html_table(_jaccard_df, caption='Pairwise Jaccard Similarity')

                    # Find universal features (in all targets)
                    _all_feat_sets = [set(_target_features[t]) for t in _overlap_targets]
                    _universal = _all_feat_sets[0]
                    for _s in _all_feat_sets[1:]:
                        _universal = _universal & _s

                    if _universal:
                        print(f"\n  Universal features (in all {_n_targets} targets): "
                              f"{len(_universal)}/{top_k_overlap}")
                        for _uf in sorted(_universal):
                            _dn = _get_display_name(_uf) if use_display_names else _uf
                            print(f"    • {_dn}")
                    else:
                        print(f"\n  No features appear in all {_n_targets} targets' top-{top_k_overlap}.")

                    # Unique features (only in one target)
                    _unique_per_target = {}
                    for _ti, _tn in enumerate(_overlap_targets):
                        _others = set()
                        for _tj in range(_n_targets):
                            if _ti != _tj:
                                _others |= _all_feat_sets[_tj]
                        _unique_per_target[_tn] = _all_feat_sets[_ti] - _others

                    _has_unique = any(len(v) > 0 for v in _unique_per_target.values())
                    if _has_unique:
                        print(f"\n  Target-unique features:")
                        for _tn, _ufs in _unique_per_target.items():
                            if _ufs:
                                _dn_list = [_get_display_name(f) if use_display_names else f for f in sorted(_ufs)]
                                print(f"    {_tn}: {', '.join(_dn_list)}")

            except Exception as _e:
                print(f"Error: Feature overlap error: {_e}")
                _traceback.print_exc()

# ---
        # 5C -- Prediction Distribution Diagnostics
# ---
        if run_prediction_diagnostics:
            _subsection_header('5C. Prediction Distribution Diagnostics')

            try:
                _has_ensemble = ('last_fitted_model' in globals() and last_fitted_model is not None
                                 and hasattr(last_fitted_model, 'predict'))

                if not _has_ensemble:
                    print("  last_fitted_model not available for prediction diagnostics.")
                else:
                    try:
                        _y_pred_diag = last_fitted_model.predict(_X_test_np).ravel()
                    except Exception:
                        _y_pred_diag = None

                    if _y_pred_diag is not None:
                        _residuals = _y_test_np - _y_pred_diag

                        if not _is_classification:
                            # 4-panel diagnostic plot
                            fig, axes = plt.subplots(2, 2, figsize=(14, 11))

                            # Panel 1: Predicted vs Actual
                            ax = axes[0, 0]
                            ax.scatter(_y_test_np, _y_pred_diag, alpha=0.5, s=25,
                                       edgecolors='none', c=_SECTION_COLORS['supplementary'])
                            _lims = [
                                min(_y_test_np.min(), _y_pred_diag.min()),
                                max(_y_test_np.max(), _y_pred_diag.max())
                            ]
                            _pad = (_lims[1] - _lims[0]) * 0.05
                            _lims = [_lims[0] - _pad, _lims[1] + _pad]
                            ax.plot(_lims, _lims, 'k--', alpha=0.5, linewidth=1, label='Identity')
                            ax.set_xlim(_lims)
                            ax.set_ylim(_lims)
                            ax.set_xlabel('Actual', fontsize=11)
                            ax.set_ylabel('Predicted', fontsize=11)
                            ax.set_title('Predicted vs Actual', fontweight='bold')
                            ax.legend()

                            # Add R² annotation
                            _r2_diag = r2_score(_y_test_np, _y_pred_diag)
                            ax.text(0.05, 0.95, f'R² = {_r2_diag:.4f}',
                                    transform=ax.transAxes, fontsize=11,
                                    verticalalignment='top',
                                    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

                            # Panel 2: Residual Distribution
                            ax = axes[0, 1]
                            ax.hist(_residuals, bins=30, color=_SECTION_COLORS['supplementary'],
                                    edgecolor='white', alpha=0.8, density=True)
                            # Overlay normal fit
                            _res_mean = np.mean(_residuals)
                            _res_std = np.std(_residuals)
                            _x_norm = np.linspace(_residuals.min(), _residuals.max(), 100)
                            _y_norm = stats.norm.pdf(_x_norm, _res_mean, _res_std)
                            ax.plot(_x_norm, _y_norm, 'r-', linewidth=2, label='Normal fit')
                            ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
                            ax.set_xlabel('Residual', fontsize=11)
                            ax.set_ylabel('Density', fontsize=11)
                            ax.set_title('Residual Distribution', fontweight='bold')
                            ax.legend()

                            # Shapiro-Wilk test (subsample if too large)
                            _sw_sample = _residuals[:min(5000, len(_residuals))]
                            try:
                                _sw_stat, _sw_p = stats.shapiro(_sw_sample)
                                ax.text(0.05, 0.95,
                                        f'Shapiro-Wilk p={_sw_p:.4f}\n'
                                        f'{"Normal" if _sw_p > 0.05 else "Non-normal"}',
                                        transform=ax.transAxes, fontsize=9,
                                        verticalalignment='top',
                                        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
                            except Exception:
                                pass

                            # Panel 3: Q-Q Plot
                            ax = axes[1, 0]
                            stats.probplot(_residuals, dist='norm', plot=ax)
                            ax.set_title('Q-Q Plot (Residuals)', fontweight='bold')
                            ax.get_lines()[0].set_markerfacecolor(_SECTION_COLORS['supplementary'])
                            ax.get_lines()[0].set_alpha(0.5)
                            ax.get_lines()[0].set_markersize(4)

                            # Panel 4: Residuals vs Predicted
                            ax = axes[1, 1]
                            ax.scatter(_y_pred_diag, _residuals, alpha=0.5, s=25,
                                       edgecolors='none', c=_SECTION_COLORS['supplementary'])
                            ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
                            # Add LOWESS trend line
                            try:
                                from statsmodels.nonparametric.smoothers_lowess import lowess
                                _lowess_result = lowess(_residuals, _y_pred_diag, frac=0.3)
                                ax.plot(_lowess_result[:, 0], _lowess_result[:, 1],
                                        'r-', linewidth=2, label='LOWESS')
                                ax.legend()
                            except ImportError:
                                # Simple moving average fallback
                                _sort_idx = np.argsort(_y_pred_diag)
                                _window = max(5, len(_residuals) // 20)
                                _moving_avg = pd.Series(_residuals[_sort_idx]).rolling(
                                    _window, center=True).mean()
                                ax.plot(_y_pred_diag[_sort_idx], _moving_avg,
                                        'r-', linewidth=2, label='Trend')
                                ax.legend()
                            except Exception:
                                pass

                            ax.set_xlabel('Predicted', fontsize=11)
                            ax.set_ylabel('Residual', fontsize=11)
                            ax.set_title('Residuals vs Predicted', fontweight='bold')

                            fig.suptitle('Prediction Distribution Diagnostics', fontweight='bold',
                                         fontsize=14, y=1.01)
                            plt.tight_layout()
                            plt.show()

                            # Summary statistics
                            _diag_stats = [
                                {'Statistic': 'Mean Residual', 'Value': f'{_res_mean:.6f}'},
                                {'Statistic': 'Std Residual', 'Value': f'{_res_std:.4f}'},
                                {'Statistic': 'Skewness', 'Value': f'{stats.skew(_residuals):.4f}'},
                                {'Statistic': 'Kurtosis', 'Value': f'{stats.kurtosis(_residuals):.4f}'},
                                {'Statistic': 'Max |Residual|', 'Value': f'{np.max(np.abs(_residuals)):.4f}'},
                                {'Statistic': 'RMSE', 'Value': f'{np.sqrt(np.mean(_residuals**2)):.4f}'},
                                {'Statistic': 'MAE', 'Value': f'{np.mean(np.abs(_residuals)):.4f}'},
                            ]
                            _safe_html_table(pd.DataFrame(_diag_stats),
                                             caption='Residual Diagnostics Summary')

                        else:
                            # Classification diagnostics
                            _y_pred_proba = None
                            if hasattr(last_fitted_model, 'predict_proba'):
                                try:
                                    _y_pred_proba = last_fitted_model.predict_proba(_X_test_np)
                                    if _y_pred_proba.ndim == 2:
                                        _y_pred_proba = _y_pred_proba[:, 1]
                                except Exception:
                                    pass

                            if _y_pred_proba is not None:
                                fig, axes = plt.subplots(1, 3, figsize=(16, 5))

                                # Probability distribution by class
                                ax = axes[0]
                                for _cls in sorted(np.unique(_y_test_np)):
                                    _mask = _y_test_np == _cls
                                    ax.hist(_y_pred_proba[_mask], bins=25, alpha=0.6,
                                            label=f'Class {int(_cls)}', density=True)
                                ax.set_xlabel('Predicted Probability')
                                ax.set_ylabel('Density')
                                ax.set_title('Prediction Distributions by Class', fontweight='bold')
                                ax.legend()

                                # Calibration-like scatter
                                ax = axes[1]
                                _n_bins_cal = 10
                                _bin_edges = np.linspace(0, 1, _n_bins_cal + 1)
                                _bin_true = []
                                _bin_pred = []
                                for _bi in range(_n_bins_cal):
                                    _mask = (_y_pred_proba >= _bin_edges[_bi]) & (_y_pred_proba < _bin_edges[_bi + 1])
                                    if _mask.sum() > 0:
                                        _bin_true.append(_y_test_np[_mask].mean())
                                        _bin_pred.append(_y_pred_proba[_mask].mean())
                                ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
                                ax.scatter(_bin_pred, _bin_true, s=60, zorder=5,
                                           c=_SECTION_COLORS['supplementary'])
                                ax.plot(_bin_pred, _bin_true, alpha=0.5, c=_SECTION_COLORS['supplementary'])
                                ax.set_xlabel('Mean Predicted Probability')
                                ax.set_ylabel('Fraction of Positives')
                                ax.set_title('Calibration Plot', fontweight='bold')

                                # Confusion-like: predicted vs actual counts
                                ax = axes[2]
                                _y_pred_cls = (_y_pred_proba >= 0.5).astype(int)
                                _conf_data = pd.crosstab(
                                    pd.Series(_y_test_np, name='Actual'),
                                    pd.Series(_y_pred_cls, name='Predicted')
                                )
                                sns.heatmap(_conf_data, annot=True, fmt='d', cmap='Blues', ax=ax)
                                ax.set_title('Confusion Matrix', fontweight='bold')

                                fig.suptitle('Classification Diagnostics', fontweight='bold',
                                             fontsize=14, y=1.02)
                                plt.tight_layout()
                                plt.show()
                            else:
                                print("  Could not obtain predicted probabilities for diagnostics.")

            except Exception as _e:
                print(f"Error: Prediction diagnostics error: {_e}")
                _traceback.print_exc()

# ---
        # 5D -- Cross-Validated R² Distribution
# ---
        _subsection_header('5D. Cross-Validated Performance Distribution')

        try:
            _cv_model = _create_lightweight_model(_is_classification, _n_features)
            _cv_metric = 'roc_auc' if _is_classification else 'r2'
            _cv_display = 'AUC' if _is_classification else 'R²'

            # Combine train + test for CV
            _X_full = np.vstack([_X_train_np, _X_test_np])
            _y_full = np.concatenate([_y_train_np, _y_test_np])

            if _is_classification:
                _cv_splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
            else:
                _cv_splitter = KFold(n_splits=10, shuffle=True, random_state=42)

            print(f"Running 10-fold CV with {type(_cv_model).__name__}...")
            _cv_scores = cross_val_score(
                _cv_model, _X_full, _y_full,
                cv=_cv_splitter, scoring=_cv_metric, n_jobs=-1
            )
            print(f"  CV {_cv_display}: {_cv_scores.mean():.4f} ± {_cv_scores.std():.4f}")
            print(f"  Fold scores: {', '.join(f'{s:.4f}' for s in _cv_scores)}")

            # Violin + box + strip plot
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # Panel 1: Violin + box
            ax = axes[0]
            parts = ax.violinplot(_cv_scores, positions=[0], showmeans=True, showmedians=True)
            for _pc in parts['bodies']:
                _pc.set_facecolor(_SECTION_COLORS['supplementary'])
                _pc.set_alpha(0.5)
            parts['cmeans'].set_color('red')
            parts['cmedians'].set_color('blue')
            ax.scatter(np.zeros_like(_cv_scores), _cv_scores, c='black', s=40, zorder=5, alpha=0.6)
            ax.set_xticks([0])
            ax.set_xticklabels([f'10-Fold CV'])
            ax.set_ylabel(_cv_display)
            ax.set_title(f'CV {_cv_display} Distribution', fontweight='bold')

            # Add mean ± CI annotation
            _cv_mean = _cv_scores.mean()
            _cv_ci = 1.96 * _cv_scores.std() / np.sqrt(len(_cv_scores))
            ax.text(0.3, _cv_mean, f'Mean = {_cv_mean:.4f}\n95% CI: ±{_cv_ci:.4f}',
                    fontsize=10, va='center',
                    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

            # Panel 2: Histogram
            ax = axes[1]
            ax.hist(_cv_scores, bins=max(5, len(_cv_scores) // 2),
                    color=_SECTION_COLORS['supplementary'], edgecolor='white', alpha=0.8)
            ax.axvline(_cv_mean, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {_cv_mean:.4f}')
            ax.axvline(np.median(_cv_scores), color='blue', linestyle='--', linewidth=1.5,
                       label=f'Median: {np.median(_cv_scores):.4f}')
            ax.set_xlabel(_cv_display)
            ax.set_ylabel('Count')
            ax.set_title(f'CV {_cv_display} Histogram', fontweight='bold')
            ax.legend()

            fig.suptitle(f'Cross-Validated {_cv_display} Distribution (10-fold)', fontweight='bold',
                         fontsize=14, y=1.02)
            plt.tight_layout()
            plt.show()

            # Compare CV score with holdout test score
            if 'last_fitted_model' in globals() and last_fitted_model is not None:
                try:
                    if _is_classification:
                        if hasattr(last_fitted_model, 'predict_proba'):
                            _holdout_pred = last_fitted_model.predict_proba(_X_test_np)
                            if _holdout_pred.ndim == 2:
                                _holdout_pred = _holdout_pred[:, 1]
                        else:
                            _holdout_pred = last_fitted_model.predict(_X_test_np)
                        _holdout_score = roc_auc_score(_y_test_np, _holdout_pred)
                    else:
                        _holdout_pred = last_fitted_model.predict(_X_test_np).ravel()
                        _holdout_score = r2_score(_y_test_np, _holdout_pred)

                    _cv_vs_holdout = [
                        {'Metric': f'CV {_cv_display} (mean)', 'Value': f'{_cv_mean:.4f}'},
                        {'Metric': f'CV {_cv_display} (std)', 'Value': f'{_cv_scores.std():.4f}'},
                        {'Metric': f'CV 95% CI', 'Value': f'[{_cv_mean - _cv_ci:.4f}, {_cv_mean + _cv_ci:.4f}]'},
                        {'Metric': f'Holdout {_cv_display}', 'Value': f'{_holdout_score:.4f}'},
                        {'Metric': 'Difference (CV - Holdout)', 'Value': f'{_cv_mean - _holdout_score:+.4f}'},
                    ]

                    _diff = abs(_cv_mean - _holdout_score)
                    if _diff < _cv_scores.std():
                        _cv_verdict = "Holdout score is within 1 SD of CV mean -- consistent"
                    elif _diff < 2 * _cv_scores.std():
                        _cv_verdict = "Holdout score is 1-2 SD from CV mean -- minor discrepancy"
                    else:
                        _cv_verdict = "Holdout score is > 2 SD from CV mean -- potential data leakage or distribution shift"
                    _cv_vs_holdout.append({'Metric': 'Assessment', 'Value': _cv_verdict})

                    _safe_html_table(pd.DataFrame(_cv_vs_holdout), caption='CV vs Holdout Comparison')
                except Exception as _e_cv:
                    print(f"  Could not compare CV vs holdout: {_e_cv}")

        except Exception as _e:
            print(f"Error: Cross-validation error: {_e}")
            _traceback.print_exc()

        _section_times['Supplementary'] = _time.time() - _t0
        print(f"\n Supplementary Analyses completed in {_section_times['Supplementary']:.1f}s")

# ---
    # SUMMARY FOOTER
# ---
    _hub_elapsed = _time.time() - _hub_start

    _footer_rows = []
    for _sname, _stime in _section_times.items():
        _footer_rows.append(f'<tr><td style="padding:4px 12px;">{_sname}</td>'
                            f'<td style="padding:4px 12px; text-align:right;">{_stime:.1f}s</td></tr>')

    _skipped = []
    if not run_diversity_analysis:
        _skipped.append('Diversity Analysis')
    if not run_bootstrap_stability:
        _skipped.append('Bootstrap Stability')
    if not run_learning_curves:
        _skipped.append('Learning Curves')
    if not run_shap_beeswarm:
        _skipped.append('SHAP Beeswarm')
    if not run_supplementary:
        _skipped.append('Supplementary')

    _skip_html = ''
    if _skipped:
        _skip_html = (
            f'<p style="margin:8px 0 0 0; opacity:0.7; font-size:12px;">'
            f'Skipped: {", ".join(_skipped)}</p>'
        )

    display(HTML(
        '<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); color: white; '
        'padding: 16px 20px; border-radius: 12px; margin-top: 20px;">'
        '<h3 style="margin:0 0 8px 0; color: #e94560;"> Diagnostics Hub -- Complete</h3>'
        '<table style="color: white; border-collapse: collapse;">'
        '<tr style="border-bottom: 1px solid rgba(255,255,255,0.2);">'
        '<th style="padding:4px 12px; text-align:left;">Section</th>'
        '<th style="padding:4px 12px; text-align:right;">Time</th></tr>'
        + ''.join(_footer_rows) +
        f'<tr style="border-top: 2px solid rgba(255,255,255,0.3);">'
        f'<td style="padding:6px 12px; font-weight:bold;">Total</td>'
        f'<td style="padding:6px 12px; text-align:right; font-weight:bold;">{_hub_elapsed:.1f}s</td></tr>'
        '</table>'
        + _skip_html +
        '</div>'
    ))

In [ ]:
#@title Pipeline Validation

# Smoke test on a 200-sample subsample. Confirms the stacking model
# initializes, fits, and predicts without error. Verifies the hasattr()
# guard correctly distinguishes stacking from legacy bagging paths.
# ---

# Fallback if create_validation_model not yet defined (defined below)
if 'create_validation_model' not in dir():
    def create_validation_model(is_classification=True):
        if 'create_stacking_ensemble' in dir():
            from sklearn.linear_model import LinearRegression, LogisticRegression
            meta = LogisticRegression(random_state=42) if is_classification else LinearRegression()
            return create_stacking_ensemble(
                task_type='classification' if is_classification else 'regression',
                meta_model=meta, cv=5, random_state=42)
        from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
        return GradientBoostingClassifier(n_estimators=50, max_depth=4, random_state=42) if is_classification \
            else GradientBoostingRegressor(n_estimators=50, max_depth=4, random_state=42)

# -- Skip if run_validation_cells=False --
_SKIP_SMOKE = ('run_validation_cells' in globals() and not run_validation_cells)
if _SKIP_SMOKE:
    print('Skipped:  Pipeline smoke test skipped (run_validation_cells=False)')

if not _SKIP_SMOKE:
    try:
        print("Running pipeline smoke test...")

        X_smoke = X_train.iloc[:200].copy()
        y_smoke = y_train.iloc[:200].copy() if hasattr(y_train, "iloc") else y_train[:200]

        # 1. Test stacking model creation and fit
        task_type_smoke = detect_task_type(y_smoke)
        is_clf = (task_type_smoke == 'classification')
        m = create_validation_model(is_classification=is_clf)
        m.fit(X_smoke, y_smoke)
        print("  Stacking model .fit() OK")

        # 2. Test predict / predict_proba
        preds = m.predict(X_smoke)
        print(f"  .predict() OK -- shape: {preds.shape}")
        if is_clf and hasattr(m, 'predict_proba'):
            proba = m.predict_proba(X_smoke)
            print(f"  .predict_proba() OK -- shape: {proba.shape}")

        # 3. Test hasattr guard
        if hasattr(m, "tune_and_fit"):
            print("  Warning:  Model has tune_and_fit (bagging/tuned path)")
        else:
            print("  No tune_and_fit (stacking path) -- hasattr guard works")

        # 4. Test X_full construction
        X_full = pd.concat([X_train, X_test], axis=0) if 'X_test' in locals() else X_train
        y_full = pd.concat([y_train, y_test], axis=0) if 'y_test' in locals() else y_train
        print(f"  X_full constructed -- shape: {X_full.shape} (X_train: {X_train.shape})")

        print("\nAll smoke tests passed.")

    except Exception as e:
        print(f"\nSmoke test failed: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
#@title V | Outer-Fold Nested CV Metrics & Calibration

# Outer-fold nested CV producing unbiased performance estimates with 95%
# CIs (t-distribution). Regression outputs include predicted-vs-actual
# scatter and fold-wise R2 distributions. Classification outputs include
# calibration curves with Wilson-score CIs, decision curve analysis, and
# fold-wise AUC distributions. Configurable number of outer folds.
#@markdown ---
n_outer_folds = 10 #@param {type: "integer"}

"""
Outer-Fold Nested Cross-Validation Analysis
--------------------------------------------
True outer-fold evaluation (configurable n held-out folds) producing
unbiased per-target performance estimates with 95% CIs.

Outputs
  Regression    : Predicted-vs-actual scatter, fold-wise R² distribution.
  Classification: Calibration curves with Wilson-score CI, decision curve
                  analysis, fold-wise AUC distribution.

Integration
-----------
Expects X_train, X_test, y_train, y_test, and model from a preceding
analysis run. Pass model_factory to retrain fresh estimators per fold.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score, confusion_matrix, accuracy_score, f1_score,
    r2_score, mean_squared_error, mean_absolute_error,
    roc_curve
)
from sklearn.model_selection import StratifiedKFold, KFold


# ---
# NestedCVAnalysis Class (Handles both task types)
# ---

class NestedCVAnalysis:
    """
    Explicit nested CV reporter for classification AND regression.
    TASK TYPE SUPPORT:
    - Classification: AUC, sensitivity, specificity, PPV, NPV, accuracy, F1
    - Regression: R², RMSE, MAE, explained variance
    """

    def __init__(self, X, y, task_type='auto', n_outer=5, random_state=42):
        """Initialize the nested-CV reporter.

        Parameters
        ----------
        X : array-like or DataFrame
            Feature matrix.
        y : array-like or Series
            Target.
        task_type : str
            'auto', 'classification', or 'regression'.
        n_outer : int
            Number of outer CV folds (default 5).
        random_state : int
            Random seed.
        """
        self.X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        # Ensure y is always 1-dimensional (batch archive may produce (N,1) arrays)
        if isinstance(y, pd.Series):
            self.y = y
        elif isinstance(y, pd.DataFrame):
            self.y = y.iloc[:, 0]
        else:
            _y = np.asarray(y)
            if _y.ndim > 1:
                _y = _y.ravel()
            self.y = pd.Series(_y)
        self.n_outer = n_outer
        self.random_state = random_state
        if task_type == 'auto':
            self.task_type = detect_task_type(self.y.values)
        else:
            self.task_type = task_type

        self.is_classification = self.task_type == 'classification'
        self.is_regression = self.task_type == 'regression'

        self.outer_fold_results = {}
        self.clinical_thresholds = [0.3, 0.5, 0.7]

        print(f"Task type detected: {self.task_type.upper()}")
        if self.is_classification:
            print(f"   Unique classes: {len(np.unique(self.y))}")
            print(f"   Class balance: {dict(pd.Series(self.y).value_counts())}")
        else:
            print(f"   Value range: [{self.y.min():.3f}, {self.y.max():.3f}]")
            print(f"   Mean: {self.y.mean():.3f}, SD: {self.y.std():.3f}")

    def run_outer_cv(self, model_factory=None, base_model=None, results_dir='nestedcv_results'):
        """
        Run explicit outer CV loop.

        USAGE OPTIONS:
        1. Pipeline mode: pass base_model (trained model from notebook)
           reporter.run_outer_cv(base_model=model)

        2. Retrain mode: pass model_factory (function returning fresh model)
           def model_factory():
               return LogisticRegression(max_iter=1000)
           reporter.run_outer_cv(model_factory=model_factory)

        Parameters:
        model_factory : callable, optional
            Function that returns a fresh model instance (for retraining)
        base_model : estimator, optional
            Pre-trained model to use for predictions (from pipeline)
        results_dir : str
            Directory to save results
        """
        if base_model is not None:
            return self._analyze_pretrained_model(base_model, results_dir)
        elif model_factory is not None:
            return self._run_explicit_outer_cv(model_factory, results_dir)
        else:
            raise ValueError("Either base_model (trained model) or model_factory (callable) must be provided")

    # =================================================================
    # PIPELINE MODE: Analyze pretrained model
    # =================================================================

    def _analyze_pretrained_model(self, model, results_dir):
        """Analyze a pretrained model using stratified CV."""
        print(f"\n PIPELINE MODE: Analyzing pretrained {model.__class__.__name__}")
        print(f"   (Using trained model with {self.n_outer}-fold outer CV)")

        os.makedirs(results_dir, exist_ok=True)

        if self.is_classification:
            return self._analyze_pretrained_classification(model, results_dir)
        else:
            return self._analyze_pretrained_regression(model, results_dir)

    def _analyze_pretrained_classification(self, model, results_dir):
        """Analyze pretrained classification model."""
        skf = StratifiedKFold(n_splits=self.n_outer, shuffle=True, random_state=self.random_state)

        fold_results = {
            'fold': [], 'auc': [], 'sensitivity': [], 'specificity': [],
            'ppv': [], 'npv': [], 'accuracy': [], 'f1': [], 'brier': [],
            'y_true': [], 'y_proba': [], 'y_pred': []
        }

        oof_df_list = []

        print(f"   Running {self.n_outer} outer folds...")

        for fold_num, (train_idx, test_idx) in enumerate(skf.split(self.X, self.y)):
            X_test_fold = self.X.iloc[test_idx]
            y_test_fold = self.y.iloc[test_idx]
            X_train_fold = self.X.iloc[train_idx]
            y_train_fold = self.y.iloc[train_idx]

            # Get predictions from pretrained model
            # -- Retrain per fold to prevent leakage --
            try:
                _fold_model = create_validation_model(is_classification=True)
                _fold_model.fit(X_train_fold, y_train_fold)
                y_proba = _fold_model.predict_proba(X_test_fold)[:, 1]
            except Exception as _e:
                print(f"  Warning: Fold {fold_num} retrain failed ({_e}), using original model")
                y_proba = model.predict_proba(X_test_fold)[:, 1]

            # Optimal threshold (Youden's J)
            fpr, tpr, thresholds = roc_curve(y_test_fold, y_proba)
            optimal_idx = np.argmax(tpr - fpr)
            optimal_threshold = thresholds[optimal_idx]
            y_pred = (y_proba >= optimal_threshold).astype(int)

            # Metrics for this fold
            _cm = confusion_matrix(y_test_fold, y_pred)
            if _cm.shape == (2, 2):
                tn, fp, fn, tp = _cm.ravel()
            else:
                tn, fp, fn, tp = 0, 0, 0, 0

            fold_results['fold'].append(fold_num)
            fold_results['auc'].append(roc_auc_score(y_test_fold, y_proba))
            fold_results['sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            fold_results['specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
            fold_results['ppv'].append(tp / (tp + fp) if (tp + fp) > 0 else 0)
            fold_results['npv'].append(tn / (tn + fn) if (tn + fn) > 0 else 0)
            fold_results['accuracy'].append(accuracy_score(y_test_fold, y_pred))
            fold_results['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
            fold_results['brier'].append(brier_score_loss(y_test_fold, y_proba))
            fold_results['y_true'].append(y_test_fold.values)
            fold_results['y_proba'].append(y_proba)
            fold_results['y_pred'].append(y_pred)
            oof_df = pd.DataFrame({
                'fold': fold_num,
                'y_true': y_test_fold.values,
                'y_proba': y_proba,
                'y_pred': y_pred
            })
            oof_df_list.append(oof_df)

            print(f"      Fold {fold_num+1}: AUC={fold_results['auc'][-1]:.4f}, Acc={fold_results['accuracy'][-1]:.4f}")

        self.outer_fold_results = fold_results

        # Save results
        oof_all = pd.concat(oof_df_list, ignore_index=True)
        oof_all.to_csv(f"{results_dir}/oof_predictions_classification.csv", index=False)

        metrics_df = pd.DataFrame({k: v for k, v in fold_results.items()
                                   if k not in ['y_true', 'y_proba', 'y_pred']})
        metrics_df.to_csv(f"{results_dir}/fold_metrics_classification.csv", index=False)

        print(f"Saved to {results_dir}/")
        return fold_results

    def _analyze_pretrained_regression(self, model, results_dir):
        """Analyze regression model using per-fold retraining ( FIX).
        NOTE: The original code used the pretrained model directly on test folds,
        causing leakage. Now retrains a fresh model per outer fold."""
        kf = KFold(n_splits=self.n_outer, shuffle=True, random_state=self.random_state)

        fold_results = {
            'fold': [], 'r2': [], 'rmse': [], 'mae': [], 'explained_var': [],
            'y_true': [], 'y_pred': []
        }

        oof_df_list = []

        print(f"   Running {self.n_outer} outer folds...")

        for fold_num, (train_idx, test_idx) in enumerate(kf.split(self.X)):
            X_test_fold = self.X.iloc[test_idx]
            y_test_fold = self.y.iloc[test_idx]
            X_train_fold = self.X.iloc[train_idx]
            y_train_fold = self.y.iloc[train_idx]

            # Get predictions
            # -- Retrain per fold to prevent leakage --
            try:
                _fold_model = create_validation_model(is_classification=False)
                _fold_model.fit(X_train_fold, y_train_fold)
                y_pred = _fold_model.predict(X_test_fold)
            except Exception as _e:
                print(f"  Warning: Fold {fold_num} retrain failed ({_e}), using original model")
                y_pred = model.predict(X_test_fold)
            r2 = r2_score(y_test_fold, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test_fold, y_pred))
            mae = mean_absolute_error(y_test_fold, y_pred)
            exp_var = 1 - (np.var(y_test_fold - y_pred) / np.var(y_test_fold))

            fold_results['fold'].append(fold_num)
            fold_results['r2'].append(r2)
            fold_results['rmse'].append(rmse)
            fold_results['mae'].append(mae)
            fold_results['explained_var'].append(exp_var)
            fold_results['y_true'].append(y_test_fold.values)
            fold_results['y_pred'].append(y_pred)
            oof_df = pd.DataFrame({
                'fold': fold_num,
                'y_true': y_test_fold.values,
                'y_pred': y_pred,
                'residual': y_test_fold.values - y_pred
            })
            oof_df_list.append(oof_df)

            print(f"      Fold {fold_num+1}: R²={r2:.4f}, RMSE={rmse:.4f}")

        self.outer_fold_results = fold_results

        # Save results
        oof_all = pd.concat(oof_df_list, ignore_index=True)
        oof_all.to_csv(f"{results_dir}/oof_predictions_regression.csv", index=False)

        metrics_df = pd.DataFrame({k: v for k, v in fold_results.items()
                                   if k not in ['y_true', 'y_pred']})
        metrics_df.to_csv(f"{results_dir}/fold_metrics_regression.csv", index=False)

        print(f"Saved to {results_dir}/")
        return fold_results

    # =================================================================
    # RETRAIN MODE: Explicit outer CV with fresh model training
    # =================================================================

    def _run_explicit_outer_cv(self, model_factory, results_dir):
        """Run explicit outer CV (retraining fresh models each fold)."""
        print(f"\n RETRAIN MODE: Running explicit outer CV ({self.n_outer} folds)")

        os.makedirs(results_dir, exist_ok=True)

        if self.is_classification:
            return self._explicit_cv_classification(model_factory, results_dir)
        else:
            return self._explicit_cv_regression(model_factory, results_dir)

    def _explicit_cv_classification(self, model_factory, results_dir):
        """Explicit outer CV for classification."""
        skf = StratifiedKFold(n_splits=self.n_outer, shuffle=True, random_state=self.random_state)

        fold_results = {
            'fold': [], 'auc': [], 'sensitivity': [], 'specificity': [],
            'ppv': [], 'npv': [], 'accuracy': [], 'f1': [], 'brier': [],
            'y_true': [], 'y_proba': [], 'y_pred': []
        }

        oof_df_list = []

        for fold_num, (train_idx, test_idx) in enumerate(skf.split(self.X, self.y)):
            X_train_fold = self.X.iloc[train_idx]
            X_test_fold = self.X.iloc[test_idx]
            y_train_fold = self.y.iloc[train_idx]
            y_test_fold = self.y.iloc[test_idx]

            # Train fresh model
            model = model_factory()
            model.fit(X_train_fold, y_train_fold)
            y_proba = model.predict_proba(X_test_fold)[:, 1]
            fpr, tpr, thresholds = roc_curve(y_test_fold, y_proba)
            optimal_idx = np.argmax(tpr - fpr)
            optimal_threshold = thresholds[optimal_idx]
            y_pred = (y_proba >= optimal_threshold).astype(int)
            _cm = confusion_matrix(y_test_fold, y_pred)
            if _cm.shape == (2, 2):
                tn, fp, fn, tp = _cm.ravel()
            else:
                tn, fp, fn, tp = 0, 0, 0, 0

            fold_results['fold'].append(fold_num)
            fold_results['auc'].append(roc_auc_score(y_test_fold, y_proba))
            fold_results['sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            fold_results['specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
            fold_results['ppv'].append(tp / (tp + fp) if (tp + fp) > 0 else 0)
            fold_results['npv'].append(tn / (tn + fn) if (tn + fn) > 0 else 0)
            fold_results['accuracy'].append(accuracy_score(y_test_fold, y_pred))
            fold_results['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
            fold_results['brier'].append(brier_score_loss(y_test_fold, y_proba))

            fold_results['y_true'].append(y_test_fold.values)
            fold_results['y_proba'].append(y_proba)
            fold_results['y_pred'].append(y_pred)

            oof_df = pd.DataFrame({
                'fold': fold_num,
                'y_true': y_test_fold.values,
                'y_proba': y_proba,
                'y_pred': y_pred
            })
            oof_df_list.append(oof_df)

            print(f"   Fold {fold_num+1}: AUC={fold_results['auc'][-1]:.4f}, Acc={fold_results['accuracy'][-1]:.4f}")

        self.outer_fold_results = fold_results

        oof_all = pd.concat(oof_df_list, ignore_index=True)
        oof_all.to_csv(f"{results_dir}/oof_predictions_classification.csv", index=False)

        metrics_df = pd.DataFrame({k: v for k, v in fold_results.items()
                                   if k not in ['y_true', 'y_proba', 'y_pred']})
        metrics_df.to_csv(f"{results_dir}/fold_metrics_classification.csv", index=False)

        print(f"Saved to {results_dir}/")
        return fold_results

    def _explicit_cv_regression(self, model_factory, results_dir):
        """Explicit outer CV for regression."""
        kf = KFold(n_splits=self.n_outer, shuffle=True, random_state=self.random_state)

        fold_results = {
            'fold': [], 'r2': [], 'rmse': [], 'mae': [], 'explained_var': [],
            'y_true': [], 'y_pred': []
        }

        oof_df_list = []

        for fold_num, (train_idx, test_idx) in enumerate(kf.split(self.X)):
            X_train_fold = self.X.iloc[train_idx]
            X_test_fold = self.X.iloc[test_idx]
            y_train_fold = self.y.iloc[train_idx]
            y_test_fold = self.y.iloc[test_idx]

            # Train fresh model
            model = model_factory()
            model.fit(X_train_fold, y_train_fold)
            y_pred = model.predict(X_test_fold)
            r2 = r2_score(y_test_fold, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test_fold, y_pred))
            mae = mean_absolute_error(y_test_fold, y_pred)
            exp_var = 1 - (np.var(y_test_fold - y_pred) / np.var(y_test_fold))

            fold_results['fold'].append(fold_num)
            fold_results['r2'].append(r2)
            fold_results['rmse'].append(rmse)
            fold_results['mae'].append(mae)
            fold_results['explained_var'].append(exp_var)
            fold_results['y_true'].append(y_test_fold.values)
            fold_results['y_pred'].append(y_pred)

            oof_df = pd.DataFrame({
                'fold': fold_num,
                'y_true': y_test_fold.values,
                'y_pred': y_pred,
                'residual': y_test_fold.values - y_pred
            })
            oof_df_list.append(oof_df)

            print(f"   Fold {fold_num+1}: R²={r2:.4f}, RMSE={rmse:.4f}")

        self.outer_fold_results = fold_results

        oof_all = pd.concat(oof_df_list, ignore_index=True)
        oof_all.to_csv(f"{results_dir}/oof_predictions_regression.csv", index=False)

        metrics_df = pd.DataFrame({k: v for k, v in fold_results.items()
                                   if k not in ['y_true', 'y_pred']})
        metrics_df.to_csv(f"{results_dir}/fold_metrics_regression.csv", index=False)

        print(f"Saved to {results_dir}/")
        return fold_results

    # =================================================================
    # REPORTING METHODS
    # =================================================================

    def summarize_outer_folds(self):
        """Aggregate outer-fold metrics with 95% t-CI."""
        if not self.outer_fold_results:
            raise ValueError("Run run_outer_cv() first.")

        if self.is_classification:
            return self._summarize_classification()
        else:
            return self._summarize_regression()

    def _summarize_classification(self):
        """Summarize classification metrics."""
        metrics_list = ['auc', 'sensitivity', 'specificity', 'ppv', 'npv', 'accuracy', 'f1', 'brier']
        summary = {}

        for metric in metrics_list:
            values = np.array(self.outer_fold_results[metric])
            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1)
            n = len(values)
            t_crit = stats.t.ppf(0.975, df=n-1)
            se = std_val / np.sqrt(n)
            ci_margin = t_crit * se

            summary[metric] = {
                'mean': mean_val,
                'std': std_val,
                'ci_lower': mean_val - ci_margin,
                'ci_upper': mean_val + ci_margin,
                'fold_values': list(values)
            }

        df = pd.DataFrame({
            'Metric': list(summary.keys()),
            'Mean': [summary[m]['mean'] for m in summary],
            'Std': [summary[m]['std'] for m in summary],
            '95% CI Lower': [summary[m]['ci_lower'] for m in summary],
            '95% CI Upper': [summary[m]['ci_upper'] for m in summary]
        })

        return df, summary

    def _summarize_regression(self):
        """Summarize regression metrics."""
        metrics_list = ['r2', 'rmse', 'mae', 'explained_var']
        summary = {}

        for metric in metrics_list:
            if metric in self.outer_fold_results:
                values = np.array(self.outer_fold_results[metric])
                mean_val = np.mean(values)
                std_val = np.std(values, ddof=1)
                n = len(values)
                t_crit = stats.t.ppf(0.975, df=n-1)
                se = std_val / np.sqrt(n)
                ci_margin = t_crit * se

                summary[metric] = {
                    'mean': mean_val,
                    'std': std_val,
                    'ci_lower': mean_val - ci_margin,
                    'ci_upper': mean_val + ci_margin,
                    'fold_values': list(values)
                }

        df = pd.DataFrame({
            'Metric': list(summary.keys()),
            'Mean': [summary[m]['mean'] for m in summary],
            'Std': [summary[m]['std'] for m in summary],
            '95% CI Lower': [summary[m]['ci_lower'] for m in summary],
            '95% CI Upper': [summary[m]['ci_upper'] for m in summary]
        })

        return df, summary

    def plot_calibration_curve(self, n_bins=10, figsize=(10, 6), save_path=None):
        """Calibration curve - classification only."""
        if not self.is_classification:
            print("Warning:  Calibration curve only for classification. Skipping.")
            return None

        if not self.outer_fold_results:
            raise ValueError("Run run_outer_cv() first.")

        y_true_all = np.concatenate(self.outer_fold_results['y_true'])
        y_proba_all = np.concatenate(self.outer_fold_results['y_proba'])

        bins = np.linspace(0, 1, n_bins + 1)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_true = np.zeros(n_bins)
        bin_ci_lower = np.zeros(n_bins)
        bin_ci_upper = np.zeros(n_bins)

        for i in range(n_bins):
            mask = (y_proba_all >= bins[i]) & (y_proba_all < bins[i + 1])
            if mask.sum() > 0:
                bin_true[i] = y_true_all[mask].mean()
                n = mask.sum()
                p = bin_true[i]
                # Wilson score CI
                z = 1.96
                denom = 1 + z**2 / n
                center = (p + z**2 / (2*n)) / denom
                margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
                bin_ci_lower[i] = max(0, center - margin)
                bin_ci_upper[i] = min(1, center + margin)

        ece = np.mean(np.abs(bin_true - bin_centers))

        fig, ax = plt.subplots(figsize=figsize)
        ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Perfectly calibrated')
        ax.errorbar(bin_centers, bin_true,
                   # Wilson CI center != bin_true, so deltas can be negative.
                   # np.clip ensures yerr values are always >= 0, preventing the
                   # matplotlib ValueError on extreme class imbalance targets.
                   yerr=[np.clip(bin_true - bin_ci_lower, 0, None),
                         np.clip(bin_ci_upper - bin_true, 0, None)],
                   fmt='o-', capsize=4, capthick=2, lw=2, markersize=8,
                   label=f'Model (ECE = {ece:.4f})')
        ax.fill_between(bin_centers, bin_ci_lower, bin_ci_upper, alpha=0.2)
        ax.set_xlabel('Mean Predicted Probability', fontsize=12)
        ax.set_ylabel('Fraction of Positives', fontsize=12)
        ax.set_title('[POINT 1] Calibration Curve\n(Outer Fold Aggregated, Wilson 95% CI)',
                    fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"   Saved: {save_path}")
        plt.show()

        return {'bin_centers': bin_centers, 'bin_true': bin_true, 'ece': ece}

    def plot_roc_curve(self, figsize=(10, 8), save_path=None):
        """ROC curve with per-fold curves and mean +/- SD shading (classification only)."""
        if not self.is_classification:
            print("Note: ROC curve only for classification. Skipping.")
            return None

        if not self.outer_fold_results:
            raise ValueError("Run run_outer_cv() first.")

        from sklearn.metrics import roc_curve as sk_roc_curve, auc

        fig, ax = plt.subplots(figsize=figsize)

        mean_fpr = np.linspace(0, 1, 100)
        tprs = []

        for fold_num in range(len(self.outer_fold_results["y_true"])):
            y_true_fold = self.outer_fold_results["y_true"][fold_num]
            y_proba_fold = self.outer_fold_results["y_proba"][fold_num]

            fpr, tpr, _ = sk_roc_curve(y_true_fold, y_proba_fold)
            fold_auc = auc(fpr, tpr)

            # Interpolate to common FPR grid
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)

            ax.plot(fpr, tpr, alpha=0.3, lw=1,
                    label=f'Fold {fold_num+1} (AUC = {fold_auc:.3f})')

        # Mean ROC
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = auc(mean_fpr, mean_tpr)
        std_auc = np.std([auc(mean_fpr, t) for t in tprs])

        ax.plot(mean_fpr, mean_tpr, color="b", lw=3,
                label=f'Mean ROC (AUC = {mean_auc:.3f} \u00b1 {std_auc:.3f})')

        # +/- 1 SD shading
        std_tpr = np.std(tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        ax.fill_between(mean_fpr, tprs_lower, tprs_upper, color="blue", alpha=0.15,
                        label='\u00b1 1 SD')

        ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
        ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
        ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
        ax.set_title('[POINT 1] ROC Curve\n(Per-Fold with Mean \u00b1 SD)',
                     fontsize=14, fontweight='bold')
        ax.legend(fontsize=9, loc="lower right")
        ax.grid(True, alpha=0.3)
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"  Saved: {save_path}")
        plt.show()

        return {'mean_auc': mean_auc, 'std_auc': std_auc}

    def plot_decision_curve(self, figsize=(10, 6), save_path=None):
        """Decision curve - classification only."""
        if not self.is_classification:
            print("Warning:  Decision curve only for classification. Skipping.")
            return None

        if not self.outer_fold_results:
            raise ValueError("Run run_outer_cv() first.")

        y_true_all = np.concatenate(self.outer_fold_results['y_true'])
        y_proba_all = np.concatenate(self.outer_fold_results['y_proba'])

        thresholds = np.linspace(0.05, 0.95, 50)
        net_benefits = []

        for threshold in thresholds:
            y_pred_thresh = (y_proba_all >= threshold).astype(int)
            tp = np.sum((y_pred_thresh == 1) & (y_true_all == 1))
            fp = np.sum((y_pred_thresh == 1) & (y_true_all == 0))
            n = len(y_true_all)
            nb = (tp / n) - (fp / n) * (threshold / (1 - threshold + 1e-10))
            net_benefits.append(nb)

        net_benefits = np.array(net_benefits)
        prevalence = np.mean(y_true_all)
        treat_all_nb = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds + 1e-10))

        fig, ax = plt.subplots(figsize=figsize)
        ax.plot(thresholds, net_benefits, label='Model', lw=3, color='#1f77b4')
        ax.plot(thresholds, treat_all_nb, label='Treat All', lw=2, linestyle='--', color='red')
        ax.axhline(y=0, label='Treat None', lw=2, linestyle='--', color='gray')
        ax.fill_between(thresholds, 0, net_benefits, alpha=0.2, color='#1f77b4')
        ax.axvspan(0.2, 0.8, alpha=0.1, color='green', label='Clinical range')
        ax.set_xlabel('Probability Threshold', fontsize=12)
        ax.set_ylabel('Net Benefit', fontsize=12)
        ax.set_title('[POINT 1] Decision Curve Analysis\n(Outer Fold Aggregated)',
                    fontsize=14, fontweight='bold')
        ax.legend(fontsize=11, loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"   Saved: {save_path}")
        plt.show()

        return {'thresholds': thresholds, 'net_benefits': net_benefits}

    def plot_fold_distributions(self, figsize=(14, 8), save_path=None):
        """Plot fold-wise distributions (all metrics)."""
        if not self.outer_fold_results:
            raise ValueError("Run run_outer_cv() first.")

        if self.is_classification:
            metrics = ['auc', 'sensitivity', 'specificity', 'ppv', 'accuracy']
        else:
            metrics = ['r2', 'rmse', 'mae', 'explained_var']

        metrics = [m for m in metrics if m in self.outer_fold_results]

        n_cols = 3
        n_rows = (len(metrics) + 2) // 3

        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = axes.flatten()

        for idx, metric in enumerate(metrics):
            values = self.outer_fold_results[metric]

            axes[idx].violinplot([values], positions=[0], vert=True, widths=0.5,
                                showmeans=True, showextrema=True)
            axes[idx].boxplot([values], positions=[0], widths=0.15, patch_artist=True,
                             boxprops=dict(facecolor='lightblue', alpha=0.7))

            y_scatter = np.random.normal(0, 0.02, size=len(values))
            axes[idx].scatter(y_scatter, values, alpha=0.7, s=80, color='red',
                            edgecolors='black', label='Outer folds')

            mean_val = np.mean(values)
            std_val = np.std(values, ddof=1)
            n = len(values)
            t_crit = stats.t.ppf(0.975, df=n-1)
            ci_margin = t_crit * (std_val / np.sqrt(n))

            axes[idx].hlines(mean_val, -0.3, 0.3, colors='green', linewidth=2.5, label='Mean')
            axes[idx].set_ylabel(metric.upper(), fontsize=11, fontweight='bold')
            axes[idx].set_title(f'{metric.upper()}\n{mean_val:.3f} [{mean_val-ci_margin:.3f}-{mean_val+ci_margin:.3f}]',
                               fontsize=11)
            axes[idx].set_xlim([-0.5, 0.5])
            axes[idx].set_xticks([])
            axes[idx].grid(True, alpha=0.3, axis='y')

        for idx in range(len(metrics), len(axes)):
            fig.delaxes(axes[idx])

        plt.suptitle(f'[POINT 1] Outer-Fold Distributions (n={self.n_outer} folds, {self.task_type.upper()})',
                    fontsize=13, fontweight='bold')
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"   Saved: {save_path}")
        plt.show()

    def threshold_metrics_table(self, thresholds=None):
        """Metrics at thresholds (classification only)."""
        if not self.is_classification:
            print("Warning:  Threshold metrics only for classification. Skipping.")
            return None

        if thresholds is None:
            thresholds = self.clinical_thresholds

        if not self.outer_fold_results:
            raise ValueError("Run run_outer_cv() first.")

        y_true_all = np.concatenate(self.outer_fold_results['y_true'])
        y_proba_all = np.concatenate(self.outer_fold_results['y_proba'])

        results = []

        for threshold in thresholds:
            y_pred = (y_proba_all >= threshold).astype(int)
            _cm = confusion_matrix(y_true_all, y_pred)
            if _cm.shape == (2, 2):
                tn, fp, fn, tp = _cm.ravel()
            else:
                tn, fp, fn, tp = 0, 0, 0, 0

            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
            npv = tn / (tn + fn) if (tn + fn) > 0 else 0

            # Bootstrap CI
            n_boot = 1000
            boot_sens, boot_spec, boot_ppv = [], [], []

            np.random.seed(self.random_state)
            for _ in range(n_boot):
                idx = np.random.choice(len(y_proba_all), len(y_proba_all), replace=True)
                y_true_b = y_true_all[idx]
                y_pred_b = y_pred[idx]
                try:
                    tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_true_b, y_pred_b).ravel()
                    boot_sens.append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
                    boot_spec.append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)
                    boot_ppv.append(tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0)
                except Exception as e:
                    print(f"[WARNING] {e}")
                    pass

            results.append({
                'Threshold': f'{threshold:.2f}',
                'Sensitivity': f"{sensitivity:.3f} [{np.percentile(boot_sens, 2.5):.3f}-{np.percentile(boot_sens, 97.5):.3f}]",
                'Specificity': f"{specificity:.3f} [{np.percentile(boot_spec, 2.5):.3f}-{np.percentile(boot_spec, 97.5):.3f}]",
                'PPV': f"{ppv:.3f} [{np.percentile(boot_ppv, 2.5):.3f}-{np.percentile(boot_ppv, 97.5):.3f}]",
                'NPV': f'{npv:.3f}',
                'N': len(y_true_all)
            })

        return pd.DataFrame(results)

    def generate_report(self, title="NESTED CV PERFORMANCE REPORT", save_plots=False):
        """Generate complete report with all outputs."""
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)
        print(f"\nTask Type: {self.task_type.upper()}")
        print(f"Outer-fold CV: {self.n_outer} folds")
        print("Metrics aggregated with t-distribution 95% CI\n")

        # Summary statistics
        df_summary, _ = self.summarize_outer_folds()
        print("1. OUTER-FOLD METRICS (Mean ± 95% CI)")
        print("-" * 80)
        try:
            _num_cols_s = df_summary.select_dtypes(include=[np.number]).columns.tolist()
            display(df_summary.style
                .format({c: '{:.4f}' for c in _num_cols_s})
                .set_table_styles([
                    {'selector': 'th', 'props': [('background-color', '#ECEFF1'), ('font-size', '12px'),
                                                  ('font-weight', '600'), ('padding', '6px 10px')]},
                    {'selector': 'td', 'props': [('padding', '5px 10px'), ('font-size', '12px')]},
                ])
                .hide(axis='index'))
        except Exception:
            print(df_summary.to_string(index=False))

        # Threshold metrics (classification only)
        if self.is_classification:
            print("\n2. METRICS AT CLINICAL THRESHOLDS (bootstrap 95% CI)")
            print("-" * 80)
            threshold_df = self.threshold_metrics_table()
            if threshold_df is not None:
                print(threshold_df.to_string(index=False))

        # Visualizations
        print("\n3. VISUALIZATIONS")
        print("-" * 80)

        self.plot_fold_distributions(save_path='ncv_fold_dist.png' if save_plots else None)

        if self.is_classification:
            self.plot_roc_curve(save_path='ncv_roc_curve.png' if save_plots else None)
            self.plot_calibration_curve(save_path='ncv_calibration.png' if save_plots else None)
            self.plot_decision_curve(save_path='ncv_decision_curve.png' if save_plots else None)

        print("\n" + "=" * 80)
        print("[POINT 1] COMPLETE")
        print("=" * 80)

# ---
# HELPER FUNCTION: Create Validation Model (Same as Main Pipeline)
# ---

def create_validation_model(is_classification=True):
    """
    Creates the EXACT same ensemble model used in main pipeline.
    This matches run_catboost_analysis() with ensemble_version=True, ensemble_type='stacking'

    CRITICAL: This ensures validation uses identical model architecture to training
    """
    from sklearn.ensemble import StackingClassifier, StackingRegressor
    from sklearn.linear_model import LogisticRegression, LinearRegression
    # Best params for CatBoost (from PIPELINE_MODEL_CONFIGS )
    best_params = dict(_CB_PARAMS)

    # Model configs (from PIPELINE_MODEL_CONFIGS)
    model_configs = {
        'catboost': best_params,
        'random_forest': dict(_RF_PARAMS),
        'xgboost': dict(_XGB_PARAMS),
        'tabpfn': dict(PIPELINE_MODEL_CONFIGS['tabpfn'])
    }
    # -- Respect _SKIP_TABPFN gate (across-category / manual skip) --
    if globals().get('_SKIP_TABPFN', False):
        model_configs.pop('tabpfn', None)

    if is_classification:
        meta_model = LogisticRegression(random_state=42)
    else:
        meta_model = LinearRegression()

    # Create stacking ensemble using the same factory as main pipeline
    model = create_stacking_ensemble(
        task_type='regression' if not is_classification else 'classification',
        model_configs=model_configs,
        meta_model=meta_model,
        cv=5,
        random_state=42
    )

    # TabPFN now opt-in via model_configs (no deletion needed)

    return model

# ---


# ---
# CACHE SYSTEM -- nested_cv

# ---
# BATCH-AWARE WRAPPER -- Outer-Fold Nested CV Metrics & Calibration
# ---
# When _batch_archive holds multiple targets with archived state, this
# wrapper iterates over each, restoring globals before each pass.
# In single-target mode the validation runs exactly once (no change).
# ---

def _run_val_15():
    """Run this cell's validation for the CURRENT globals."""
    global reporter_point1  # expose for downstream consumption
    global _within_ncv_results  # expose for downstream consumption
    # ---
    # -- Validation cell gate (reads Cell 6 params) ---
    _skip_nested = not globals().get('run_validation_cells', True)
    _VAL_N_OUTER = n_outer_folds

    # -- Dependency guard: ensure Cell 0 infrastructure is loaded ---
    if 'make_run_key' not in globals():
        print('\nWarning:  Cell 0 (Report State) has not been run in this session.')
        print('   Please run Cells 1 → 12 first (in order), then re-run this cell.')
        print('   (After a runtime restart, all cells must be re-executed in order.)')
        _RUN_12 = False
    elif _skip_nested:
        print('Skipped:  Cell 12 -- Nested CV skipped (run_validation_cells=False)')
        _RUN_12 = False
    else:
        _CTAG_12  = 'nested_cv'
        _CKEY_12  = make_run_key({'cell': _CTAG_12})
        _CACHED_12 = cache_load(_CTAG_12, _CKEY_12)
        if _CACHED_12:
            print(f"  {_CTAG_12} loaded from cache -- skipping execution.")
            _RUN_12 = False
        else:
            _RS['section'] = 'nested_cv'
            _done_folds_12 = fold_load_all(_CKEY_12, _CTAG_12)
            _RUN_12 = True

    if _RUN_12:

        # EXECUTION: Run reporter with pipeline or standalone
        # ======================================================================

        try:
            # -- Task-type helpers: use Cell 4 definitions if available, else derive from y_train --
            if 'is_regression' not in globals() or not callable(globals().get('is_regression')):
                globals()['is_regression'] = lambda target: target in (globals().get('continuous_outcome_variables', []))
            if 'is_classification' not in globals() or not callable(globals().get('is_classification')):
                globals()['is_classification'] = lambda target: target in (globals().get('binary_or_categorical_outcome_variables', []))

# ---
            # DETECT MODE: within_categories vs across_categories
# ---

            _is_within_mode = (
                'exported_within_dict' in globals()
                and isinstance(exported_within_dict, dict)
                and len(exported_within_dict) > 0
                and 'split' in globals()
                and split == 'within_categories'
            )

            if _is_within_mode:
# ---
                # WITHIN-CATEGORIES MODE: Per-category nested CV
# ---

                print("\n" + "=" * 80)
                print(" Running [POINT 1] -- WITHIN-CATEGORIES Nested CV")
                print("=" * 80)

                _n_cats = len(exported_within_dict)
                task_type = detect_task_type(list(exported_within_dict.values())[0][2])
                _is_cls = (task_type == 'classification')
                _metric_name = 'AUC' if _is_cls else 'R²'
                _metric_key = 'auc' if _is_cls else 'r2'

                print(f"   Task type: {task_type}")
                print(f"   Categories: {_n_cats}")
                print(f"   Folds per category: {_VAL_N_OUTER}")
                print(f"   Total model fits: ~{_n_cats * 5}")
                print(f"   Using STACKING ENSEMBLE model factory (same as main pipeline)")
                print()

                # Collect per-category results
                _within_ncv_results = {}  # {cat: {'cv_r2_mean', 'cv_r2_std', 'cv_r2_folds', 'holdout_r2', ...}}

                results_dir_point1 = os.path.join(os.getcwd(), "nestedcv_results_point1")
                os.makedirs(results_dir_point1, exist_ok=True)

                for _cat_idx, (_cat_name, _cat_data) in enumerate(exported_within_dict.items()):
                    _Xtr, _Xte, _ytr, _yte = _cat_data
                    _n_feat = _Xtr.shape[1]

                    print(f"  [{_cat_idx+1}/{_n_cats}] {_cat_name} ({_n_feat} features, "
                          f"N_train={_Xtr.shape[0]:,}, N_test={_Xte.shape[0]:,})")

                    # -- 5-fold nested CV on training data --
                    _Xtr_cv = _Xtr.copy() if isinstance(_Xtr, pd.DataFrame) else pd.DataFrame(_Xtr)
                    _ytr_cv = _ytr.copy() if isinstance(_ytr, pd.Series) else pd.Series(np.asarray(_ytr).ravel())

                    # Reset indices to prevent iloc issues
                    _Xtr_cv = _Xtr_cv.reset_index(drop=True)
                    _ytr_cv = _ytr_cv.reset_index(drop=True)

                    kf = KFold(n_splits=_VAL_N_OUTER, shuffle=True, random_state=42)
                    _fold_metrics = {'r2': [], 'rmse': [], 'mae': [], 'explained_var': []}
                    if _is_cls:
                        _fold_metrics = {'auc': [], 'accuracy': [], 'f1': [], 'brier': []}

                    _fold_ok = True
                    for _fold_num, (_train_idx, _test_idx) in enumerate(
                        kf.split(_Xtr_cv) if not _is_cls else
                        StratifiedKFold(n_splits=_VAL_N_OUTER, shuffle=True, random_state=42).split(_Xtr_cv, _ytr_cv)
                    ):
                        try:
                            _Xf_train = _Xtr_cv.iloc[_train_idx]
                            _Xf_test = _Xtr_cv.iloc[_test_idx]
                            _yf_train = _ytr_cv.iloc[_train_idx]
                            _yf_test = _ytr_cv.iloc[_test_idx]

                            _model = create_validation_model(is_classification=_is_cls)
                            _model.fit(_Xf_train, _yf_train)

                            if _is_cls:
                                _y_pred = _model.predict(_Xf_test)
                                _y_proba = (_model.predict_proba(_Xf_test)[:, 1]
                                            if hasattr(_model, 'predict_proba') else None)
                                _fold_metrics['auc'].append(
                                    roc_auc_score(_yf_test, _y_proba) if _y_proba is not None else np.nan)
                                _fold_metrics['accuracy'].append(accuracy_score(_yf_test, _y_pred))
                                _fold_metrics['f1'].append(f1_score(_yf_test, _y_pred, zero_division=0))
                                if _y_proba is not None:
                                    _fold_metrics['brier'].append(brier_score_loss(_yf_test, _y_proba))
                            else:
                                _y_pred = _model.predict(_Xf_test)
                                from sklearn.metrics import explained_variance_score
                                _fold_metrics['r2'].append(r2_score(_yf_test, _y_pred))
                                _fold_metrics['rmse'].append(np.sqrt(mean_squared_error(_yf_test, _y_pred)))
                                _fold_metrics['mae'].append(mean_absolute_error(_yf_test, _y_pred))
                                _fold_metrics['explained_var'].append(explained_variance_score(_yf_test, _y_pred))

                        except Exception as _fe:
                            print(f"      Warning: Fold {_fold_num+1} failed: {_fe}")
                            _fold_ok = False
                            break

                    if not _fold_ok or len(_fold_metrics.get(_metric_key, [])) == 0:
                        print(f"      Warning: Skipped (fold failure)")
                        continue

                    # -- Holdout evaluation --
                    try:
                        _ho_model = create_validation_model(is_classification=_is_cls)
                        _ho_model.fit(_Xtr_cv, _ytr_cv)

                        _Xte_ho = _Xte.copy() if isinstance(_Xte, pd.DataFrame) else pd.DataFrame(_Xte)
                        _yte_ho = _yte.copy() if isinstance(_yte, pd.Series) else pd.Series(np.asarray(_yte).ravel())

                        if _is_cls:
                            _ho_pred = _ho_model.predict(_Xte_ho)
                            _ho_proba = (_ho_model.predict_proba(_Xte_ho)[:, 1]
                                         if hasattr(_ho_model, 'predict_proba') else None)
                            _ho_primary = roc_auc_score(_yte_ho, _ho_proba) if _ho_proba is not None else np.nan
                            _ho_rmse = np.nan
                            _ho_mae = np.nan
                        else:
                            _ho_pred = _ho_model.predict(_Xte_ho)
                            _ho_primary = r2_score(_yte_ho, _ho_pred)
                            _ho_rmse = np.sqrt(mean_squared_error(_yte_ho, _ho_pred))
                            _ho_mae = mean_absolute_error(_yte_ho, _ho_pred)
                    except Exception as _he:
                        print(f"      Warning: Holdout failed: {_he}")
                        _ho_primary = np.nan
                        _ho_rmse = np.nan
                        _ho_mae = np.nan

                    _cv_mean = np.mean(_fold_metrics[_metric_key])
                    _cv_std = np.std(_fold_metrics[_metric_key])

                    _within_ncv_results[_cat_name] = {
                        'n_features':     _n_feat,
                        'n_train':        _Xtr.shape[0],
                        'n_test':         _Xte.shape[0],
                        'cv_folds':       _fold_metrics[_metric_key],
                        'cv_mean':        _cv_mean,
                        'cv_std':         _cv_std,
                        'cv_rmse_mean':   np.mean(_fold_metrics.get('rmse', [np.nan])),
                        'cv_mae_mean':    np.mean(_fold_metrics.get('mae', [np.nan])),
                        'holdout_primary': _ho_primary,
                        'holdout_rmse':    _ho_rmse,
                        'holdout_mae':     _ho_mae,
                    }

                    print(f"      CV {_metric_name}: {_cv_mean:.4f} ± {_cv_std:.4f}  |  "
                          f"Holdout {_metric_name}: {_ho_primary:.4f}")

                    # Free fold models between categories
                    try:
                        del _model, _ho_model
                    except NameError:
                        pass
                    _pipeline_cleanup(label=f'P1 cat {_cat_idx+1}/{_n_cats}', verbose=(_cat_idx % 5 == 4))

# ---
                # AGGREGATE REPORT
# ---

                if len(_within_ncv_results) > 0:
                    print("\n" + "=" * 80)
                    print("[POINT 1] Within-Categories Nested CV -- Aggregate Report")
                    print("=" * 80)

                    # -- Per-category results table --
                    _ncv_rows = []
                    for _cat, _res in _within_ncv_results.items():
                        _row = {
                            'Category': _cat,
                            'N Feat.': _res['n_features'],
                        }
                        _row[f'CV {_metric_name} (mean)'] = _res['cv_mean']
                        _row[f'CV {_metric_name} (SD)'] = _res['cv_std']
                        _row[f'Holdout {_metric_name}'] = _res['holdout_primary']
                        if not _is_cls:
                            _row['CV RMSE'] = _res['cv_rmse_mean']
                            _row['Holdout RMSE'] = _res['holdout_rmse']
                        _ncv_rows.append(_row)

                    _ncv_df = pd.DataFrame(_ncv_rows).sort_values(
                        f'CV {_metric_name} (mean)', ascending=False
                    ).reset_index(drop=True)

                    # -- Merge with original pipeline metrics if available --
                    if 'metrics_dict_full' in globals() and isinstance(metrics_dict_full, dict):
                        _orig_lookup = {}
                        for _cat, _mdict in zip(metrics_dict_full.get('category', []),
                                                 metrics_dict_full.get('metrics', [])):
                            _m = _mdict.get('ensemble', _mdict.get('meta_model', {}))
                            if _m:
                                _orig_lookup[_cat] = _m.get('r2' if not _is_cls else 'auc', np.nan)
                        _ncv_df[f'Original {_metric_name}'] = _ncv_df['Category'].map(_orig_lookup)
                        _ncv_df[f'Δ (CV − Orig)'] = _ncv_df[f'CV {_metric_name} (mean)'] - _ncv_df[f'Original {_metric_name}']

                    # Style and display
                    _num_cols = _ncv_df.select_dtypes(include=[np.number]).columns.tolist()
                    _gradient_col = f'CV {_metric_name} (mean)'

                    _ncv_styler = (_ncv_df.style
                        .format({c: '{:.4f}' for c in _num_cols if c != 'N Feat.'})
                        .format({'N Feat.': '{:.0f}'})
                        .background_gradient(cmap='RdYlGn', subset=[_gradient_col], vmin=0.5 if _is_cls else 0)
                        .set_caption(f'[POINT 1] Per-Category Nested CV -- {target_options} (T{tp_option})')
                        .set_table_styles([
                            {'selector': 'caption',
                             'props': [('font-size', '14px'), ('font-weight', '700'),
                                       ('color', '#455A64'), ('padding-bottom', '8px')]},
                            {'selector': 'th',
                             'props': [('background-color', '#ECEFF1'), ('color', '#455A64'),
                                       ('font-size', '11px'), ('font-weight', '600'),
                                       ('text-transform', 'uppercase'), ('padding', '6px 10px')]},
                            {'selector': 'td',
                             'props': [('padding', '5px 10px'), ('font-size', '12px')]},
                        ])
                        .hide(axis='index')
                    )

                    # Highlight Δ column if present
                    if f'Δ (CV − Orig)' in _ncv_df.columns:
                        _ncv_styler = _ncv_styler.background_gradient(
                            cmap='RdYlBu', subset=[f'Δ (CV − Orig)'],
                            vmin=-0.05, vmax=0.05
                        )

                    display(_ncv_styler)
                    print()

                    # -- Aggregate metrics --
                    _all_cv_means = [r['cv_mean'] for r in _within_ncv_results.values()]
                    _all_ho = [r['holdout_primary'] for r in _within_ncv_results.values()
                               if not np.isnan(r['holdout_primary'])]
                    _weights = np.array([r['n_test'] for r in _within_ncv_results.values()])
                    _weights_norm = _weights / _weights.sum()

                    _weighted_cv = np.average(_all_cv_means, weights=_weights_norm)
                    _unweighted_cv = np.mean(_all_cv_means)
                    _weighted_ho = np.average(_all_ho, weights=_weights_norm[:len(_all_ho)]) if _all_ho else np.nan

                    print(f"  AGGREGATE {_metric_name}:")
                    print(f"    Weighted mean (by N_test):  CV = {_weighted_cv:.4f}  |  Holdout = {_weighted_ho:.4f}")
                    print(f"    Unweighted mean:            CV = {_unweighted_cv:.4f}")
                    print(f"    Categories validated: {len(_within_ncv_results)}/{_n_cats}")

                    # -- Comparison figure: Original vs Nested CV R² --
                    try:
                        _BLUE = '#1565C0'
                        _TEAL = '#00838F'
                        _RED = '#C62828'
                        _GREY = '#455A64'

                        # Sort by CV performance
                        _plot_cats = _ncv_df.sort_values(f'CV {_metric_name} (mean)', ascending=True)

                        fig, axes = plt.subplots(1, 2, figsize=(18, max(8, len(_plot_cats) * 0.35)),
                                                  gridspec_kw={'width_ratios': [3, 2]})

                        # Left panel: Original vs Nested CV per category
                        _y_pos = np.arange(len(_plot_cats))
                        _bar_h = 0.35

                        _cv_vals = _plot_cats[f'CV {_metric_name} (mean)'].values
                        _ho_vals = _plot_cats[f'Holdout {_metric_name}'].values

                        axes[0].barh(_y_pos - _bar_h/2, _cv_vals, height=_bar_h,
                                     color=_BLUE, alpha=0.85, label=f'Nested CV {_metric_name}',
                                     edgecolor='white', linewidth=0.3)
                        axes[0].barh(_y_pos + _bar_h/2, _ho_vals, height=_bar_h,
                                     color=_TEAL, alpha=0.85, label=f'Holdout {_metric_name}',
                                     edgecolor='white', linewidth=0.3)

                        if f'Original {_metric_name}' in _plot_cats.columns:
                            _orig_vals = _plot_cats[f'Original {_metric_name}'].values
                            axes[0].scatter(_orig_vals, _y_pos, color=_RED, s=40, zorder=5,
                                            marker='D', label=f'Original {_metric_name} (train/test)')

                        axes[0].set_yticks(_y_pos)
                        _short_names = []
                        for c in _plot_cats['Category'].values:
                            c_short = (str(c).replace('Interpersonal Relationships & Social Functioning', 'Social Functioning')
                                        .replace('Parent Cognitive and Attention Issues', 'Parent Cog. & Attn')
                                        .replace('Parent Impulsivity and Behavior Regulation', 'Parent Impuls. & Behav.')
                                        .replace('Family Psychopathology & Well-Being', 'Family Psychopath.')
                                        .replace('Dynamic Cognitive Control Parameters', 'Dyn. Cog. Control'))
                            if len(c_short) > 35:
                                c_short = c_short[:33] + '…'
                            _short_names.append(c_short)
                        axes[0].set_yticklabels(_short_names, fontsize=9)
                        axes[0].set_xlabel(f'{_metric_name}', fontsize=11)
                        axes[0].set_title(f'[POINT 1] Per-Category Validation\n'
                                          f'{target_options} (T{tp_option})',
                                          fontsize=13, fontweight='bold', color=_GREY)
                        axes[0].legend(fontsize=9, loc='lower right', framealpha=0.9)
                        axes[0].axvline(0, color='#CFD8DC', linewidth=0.8, zorder=0)
                        axes[0].spines['top'].set_visible(False)
                        axes[0].spines['right'].set_visible(False)

                        # Right panel: CV fold distributions for top-8 categories
                        _top8 = _ncv_df.nlargest(8, f'CV {_metric_name} (mean)')
                        _fold_data = []
                        _fold_labels = []
                        for _, _row in _top8.iterrows():
                            _cat = _row['Category']
                            if _cat in _within_ncv_results:
                                _folds = _within_ncv_results[_cat]['cv_folds']
                                _fold_data.append(_folds)
                                _label = str(_cat)
                                if len(_label) > 25:
                                    _label = _label[:23] + '…'
                                _fold_labels.append(_label)

                        if _fold_data:
                            bp = axes[1].boxplot(_fold_data, vert=False, patch_artist=True,
                                                 labels=_fold_labels,
                                                 boxprops=dict(facecolor=_BLUE, alpha=0.3),
                                                 medianprops=dict(color=_RED, linewidth=2),
                                                 whiskerprops=dict(color=_GREY),
                                                 capprops=dict(color=_GREY),
                                                 flierprops=dict(markerfacecolor=_GREY, markersize=4))
                            axes[1].set_xlabel(f'{_metric_name}', fontsize=11)
                            axes[1].set_title(f'Fold Distributions (Top 8)',
                                              fontsize=13, fontweight='bold', color=_GREY)
                            axes[1].axvline(0, color='#CFD8DC', linewidth=0.8, zorder=0)
                            axes[1].spines['top'].set_visible(False)
                            axes[1].spines['right'].set_visible(False)
                            axes[1].tick_params(axis='y', labelsize=9)

                        plt.tight_layout()
                        plt.savefig(f'{results_dir_point1}/within_categories_ncv_comparison.png',
                                    dpi=300, bbox_inches='tight', facecolor='white')
                        plt.show()
                        print(f"\n   Saved: {results_dir_point1}/within_categories_ncv_comparison.png")

                    except Exception as _pe:
                        print(f"  Warning: Comparison plot failed: {_pe}")

                    # -- Overfitting diagnostic: paired dot plot --
                    if f'Δ (CV − Orig)' in _ncv_df.columns and f'Original {_metric_name}' in _ncv_df.columns:
                        try:
                            _diag_df = _ncv_df.dropna(subset=[f'Original {_metric_name}', f'CV {_metric_name} (mean)'])
                            _diag_df = _diag_df.sort_values(f'Original {_metric_name}', ascending=True).reset_index(drop=True)

                            fig, ax = plt.subplots(figsize=(10, max(6, len(_diag_df) * 0.3)))
                            _y = np.arange(len(_diag_df))
                            _orig = _diag_df[f'Original {_metric_name}'].values
                            _cv = _diag_df[f'CV {_metric_name} (mean)'].values
                            _cats = _diag_df['Category'].values

                            # Connecting lines colored by direction
                            for _j in range(len(_diag_df)):
                                _lc = '#C62828' if _cv[_j] < _orig[_j] - 0.005 else '#2E7D32' if _cv[_j] > _orig[_j] + 0.005 else '#90A4AE'
                                ax.plot([_orig[_j], _cv[_j]], [_y[_j], _y[_j]], color=_lc, lw=1.5, alpha=0.7)

                            ax.scatter(_orig, _y, color=_RED, s=50, zorder=5, marker='D', label=f'Original {_metric_name}')
                            ax.scatter(_cv, _y, color=_BLUE, s=50, zorder=5, marker='o', label=f'Nested CV {_metric_name}')

                            _short = [str(c)[:30] + '…' if len(str(c)) > 30 else str(c) for c in _cats]
                            ax.set_yticks(_y)
                            ax.set_yticklabels(_short, fontsize=8)
                            ax.set_xlabel(f'{_metric_name}', fontsize=11)
                            ax.set_title(f'Overfitting Diagnostic: Original vs Nested CV {_metric_name}\n'
                                         f'Red line = performance drop (potential overfit)',
                                         fontsize=12, fontweight='bold', color=_GREY)
                            ax.legend(fontsize=9, loc='lower right')
                            ax.spines['top'].set_visible(False)
                            ax.spines['right'].set_visible(False)
                            ax.grid(axis='x', alpha=0.2)
                            plt.tight_layout()
                            plt.savefig(f'{results_dir_point1}/overfitting_diagnostic.png',
                                        dpi=300, bbox_inches='tight', facecolor='white')
                            plt.show()
                            print(f"   Saved: {results_dir_point1}/overfitting_diagnostic.png")

                            _deltas = _diag_df[f'Δ (CV − Orig)'].values
                            _n_overfit = np.sum(_deltas < -0.01)
                            _n_stable = np.sum(np.abs(_deltas) <= 0.01)
                            _n_gain = np.sum(_deltas > 0.01)
                            print(f"\n  Overfitting summary:")
                            print(f"    Overfit (Δ < -0.01): {_n_overfit}/{len(_deltas)} categories")
                            print(f"    Stable (|Δ| ≤ 0.01): {_n_stable}/{len(_deltas)} categories")
                            print(f"    Gain   (Δ > +0.01):  {_n_gain}/{len(_deltas)} categories")

                        except Exception as _oe:
                            print(f"  Warning: Overfitting diagnostic failed: {_oe}")

                    # Save results
                    _ncv_df.to_csv(f'{results_dir_point1}/within_categories_ncv_results.csv', index=False)
                    print(f"\n   Saved: {results_dir_point1}/within_categories_ncv_results.csv")

                else:
                    print("\n  Warning: No categories completed successfully.")

            elif 'X_train' in globals() and 'y_train' in globals():
# ---
                # ACROSS-CATEGORIES MODE: Original single-pool nested CV
# ---

                print("\n Running [POINT 1] with pipeline variables")
                print("   Using: X_train, y_train, model from notebook")

                # Detect task type
                task_type = detect_task_type(y_train)
                _task_is_classification = (task_type == "classification")

                is_classification = _task_is_classification

                print(f"   Task type detected: {task_type}")
                print("   Using STACKING ENSEMBLE model factory (same as main pipeline)")

                # ============================================================
                # OUTER 5-FOLD CV (TRAINING DATA ONLY)
                # ============================================================

                X_cv = X_train.copy()
                y_cv = y_train.copy()

                reporter_point1 = NestedCVAnalysis(
                    X_cv,
                    y_cv,
                    task_type='auto',
                    n_outer=_VAL_N_OUTER
                )

                results_dir_point1 = os.path.join(os.getcwd(), "nestedcv_results_point1")
                os.makedirs(results_dir_point1, exist_ok=True)

                reporter_point1.run_outer_cv(
                    model_factory=lambda: create_validation_model(
                        is_classification=is_classification
                    ),
                    results_dir=results_dir_point1
                )

                reporter_point1.generate_report(
                    title="[POINT 1] Nested CV Performance - RP",
                    save_plots=True
                )

                # ============================================================
                # OOF PRED-VS-ACTUAL SCATTER (the key regression figure)
                # ============================================================
                if not is_classification and hasattr(reporter_point1, 'outer_fold_results'):
                    try:
                        y_true_oof = np.concatenate(reporter_point1.outer_fold_results['y_true'])
                        y_pred_oof = np.concatenate(reporter_point1.outer_fold_results['y_pred'])

                        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

                        # Panel 1: Hexbin density plot (N≈8000 makes scatter unreadable)
                        from matplotlib.colors import LogNorm
                        lims = [min(y_true_oof.min(), y_pred_oof.min()),
                                max(y_true_oof.max(), y_pred_oof.max())]

                        hb = axes[0].hexbin(y_true_oof, y_pred_oof, gridsize=40,
                                            cmap='YlOrRd', mincnt=1, norm=LogNorm())
                        cb = fig.colorbar(hb, ax=axes[0], shrink=0.8, label='Count (log)')

                        # Identity line + fit line
                        axes[0].plot(lims, lims, 'k--', lw=1.5, label='Perfect prediction')
                        z = np.polyfit(y_true_oof, y_pred_oof, 1)
                        p = np.poly1d(z)
                        x_line = np.linspace(lims[0], lims[1], 100)
                        axes[0].plot(x_line, p(x_line), color='#1565C0', lw=2, alpha=0.9,
                                    label=f'Fit: y={z[0]:.2f}x+{z[1]:.2f}')

                        oof_r2 = r2_score(y_true_oof, y_pred_oof)
                        oof_rmse = np.sqrt(mean_squared_error(y_true_oof, y_pred_oof))
                        oof_mae = mean_absolute_error(y_true_oof, y_pred_oof)
                        axes[0].set_xlabel('Actual', fontsize=12)
                        axes[0].set_ylabel('Predicted (OOF)', fontsize=12)
                        axes[0].set_title(f'OOF Predictions (5-Fold CV)\n'
                                         f'R²={oof_r2:.3f}, RMSE={oof_rmse:.3f}, MAE={oof_mae:.3f}, N={len(y_true_oof)}',
                                         fontsize=11, fontweight='bold')
                        axes[0].legend(fontsize=9, loc='upper left')
                        axes[0].spines['top'].set_visible(False)
                        axes[0].spines['right'].set_visible(False)

                        # Panel 2: Residual distribution with ±1 SD shading
                        residuals_oof = y_true_oof - y_pred_oof
                        res_mean = np.mean(residuals_oof)
                        res_sd = np.std(residuals_oof)
                        res_skew = pd.Series(residuals_oof).skew()

                        axes[1].hist(residuals_oof, bins=60, color='#42A5F5', edgecolor='black',
                                    alpha=0.7, density=True, linewidth=0.3)
                        axes[1].axvline(0, color='#C62828', linestyle='--', lw=2, label='Zero')
                        axes[1].axvline(res_mean, color='#2E7D32', linestyle=':',
                                       lw=2, label=f'Mean={res_mean:.4f}')
                        axes[1].axvspan(res_mean - res_sd, res_mean + res_sd,
                                       alpha=0.12, color='#2E7D32', label='±1 SD')
                        axes[1].set_xlabel('Residual (Actual − Predicted)', fontsize=12)
                        axes[1].set_ylabel('Density', fontsize=12)
                        axes[1].set_title(f'Residual Distribution\n'
                                         f'Mean={res_mean:.4f}, SD={res_sd:.3f}, Skew={res_skew:.3f}',
                                         fontsize=11, fontweight='bold')
                        axes[1].legend(fontsize=9)
                        axes[1].spines['top'].set_visible(False)
                        axes[1].spines['right'].set_visible(False)

                        plt.suptitle('[POINT 1] Out-of-Fold Prediction Quality',
                                    fontsize=14, fontweight='bold')
                        plt.tight_layout()
                        plt.savefig(f"{results_dir_point1}/oof_pred_vs_actual.png",
                                   dpi=300, bbox_inches='tight')
                        plt.show()
                        print(f"   Saved: {results_dir_point1}/oof_pred_vs_actual.png")

                    except Exception as e:
                        print(f"   Warning:  Could not plot OOF predictions: {e}")

                # ============================================================
                # OOF CLASSIFICATION FIGURE (ROC + Calibration 2-panel)
                # ============================================================
                if is_classification and hasattr(reporter_point1, 'outer_fold_results'):
                    try:
                        from sklearn.metrics import roc_curve as sk_roc_curve, auc as sk_auc
                        from sklearn.calibration import calibration_curve

                        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

                        # -- Panel A: Pooled ROC with per-fold curves --
                        mean_fpr = np.linspace(0, 1, 100)
                        tprs = []
                        for _fi in range(len(reporter_point1.outer_fold_results['y_true'])):
                            _yt = reporter_point1.outer_fold_results['y_true'][_fi]
                            _yp = reporter_point1.outer_fold_results['y_proba'][_fi]
                            fpr, tpr, _ = sk_roc_curve(_yt, _yp)
                            _fauc = sk_auc(fpr, tpr)
                            interp_tpr = np.interp(mean_fpr, fpr, tpr)
                            interp_tpr[0] = 0.0
                            tprs.append(interp_tpr)
                            axes[0].plot(fpr, tpr, alpha=0.25, lw=1,
                                         label=f'Fold {_fi+1} (AUC={_fauc:.3f})')
                        mean_tpr = np.mean(tprs, axis=0)
                        mean_tpr[-1] = 1.0
                        mean_auc = sk_auc(mean_fpr, mean_tpr)
                        std_auc = np.std([sk_auc(mean_fpr, t) for t in tprs])
                        axes[0].plot(mean_fpr, mean_tpr, color='#1565C0', lw=3,
                                     label=f'Mean (AUC={mean_auc:.3f} \u00b1 {std_auc:.3f})')
                        std_tpr = np.std(tprs, axis=0)
                        axes[0].fill_between(mean_fpr,
                                             np.maximum(mean_tpr - std_tpr, 0),
                                             np.minimum(mean_tpr + std_tpr, 1),
                                             color='#1565C0', alpha=0.12, label='\u00b1 1 SD')
                        axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance')
                        axes[0].set_xlabel('False Positive Rate', fontsize=12)
                        axes[0].set_ylabel('True Positive Rate', fontsize=12)
                        axes[0].set_title(f'OOF ROC ({_VAL_N_OUTER}-Fold CV)\n'
                                          f'Mean AUC={mean_auc:.3f}, N={sum(len(y) for y in reporter_point1.outer_fold_results["y_true"])}',
                                          fontsize=11, fontweight='bold')
                        axes[0].legend(fontsize=8, loc='lower right')
                        axes[0].set_xlim([-0.02, 1.02])
                        axes[0].set_ylim([-0.02, 1.02])
                        axes[0].spines['top'].set_visible(False)
                        axes[0].spines['right'].set_visible(False)

                        # -- Panel B: Calibration curve with Wilson CIs --
                        y_true_all = np.concatenate(reporter_point1.outer_fold_results['y_true'])
                        y_proba_all = np.concatenate(reporter_point1.outer_fold_results['y_proba'])
                        n_bins = 10
                        bins = np.linspace(0, 1, n_bins + 1)
                        bin_centers = (bins[:-1] + bins[1:]) / 2
                        bin_true = np.zeros(n_bins)
                        bin_ci_lo = np.zeros(n_bins)
                        bin_ci_hi = np.zeros(n_bins)
                        for _bi in range(n_bins):
                            mask = (y_proba_all >= bins[_bi]) & (y_proba_all < bins[_bi + 1])
                            if mask.sum() > 0:
                                p = y_true_all[mask].mean()
                                n = mask.sum()
                                bin_true[_bi] = p
                                z = 1.96
                                denom = 1 + z**2 / n
                                center = (p + z**2 / (2*n)) / denom
                                margin = z * np.sqrt((p*(1-p) + z**2/(4*n)) / n) / denom
                                bin_ci_lo[_bi] = max(0, center - margin)
                                bin_ci_hi[_bi] = min(1, center + margin)
                        ece = np.mean(np.abs(bin_true - bin_centers))
                        axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect')
                        axes[1].errorbar(bin_centers, bin_true,
                                         yerr=[np.clip(bin_true - bin_ci_lo, 0, None),
                                               np.clip(bin_ci_hi - bin_true, 0, None)],
                                         fmt='o-', capsize=4, capthick=2, lw=2, markersize=8,
                                         color='#1565C0', label=f'Model (ECE={ece:.4f})')
                        axes[1].fill_between(bin_centers, bin_ci_lo, bin_ci_hi,
                                             alpha=0.15, color='#1565C0')
                        axes[1].set_xlabel('Mean Predicted Probability', fontsize=12)
                        axes[1].set_ylabel('Observed Fraction Positive', fontsize=12)
                        axes[1].set_title(f'OOF Calibration (Wilson 95% CI)\nECE={ece:.4f}',
                                          fontsize=11, fontweight='bold')
                        axes[1].legend(fontsize=9)
                        axes[1].set_xlim([0, 1])
                        axes[1].set_ylim([0, 1])
                        axes[1].spines['top'].set_visible(False)
                        axes[1].spines['right'].set_visible(False)

                        plt.suptitle('[POINT 1] Out-of-Fold Classification Quality',
                                     fontsize=14, fontweight='bold')
                        plt.tight_layout()
                        plt.savefig(f"{results_dir_point1}/oof_clf_roc_calibration.png",
                                    dpi=300, bbox_inches='tight')
                        plt.show()
                        print(f"  Saved: {results_dir_point1}/oof_clf_roc_calibration.png")

                    except Exception as e:
                        print(f"   Warning:  Could not plot classification OOF figure: {e}")

                # ============================================================
                # FINAL INDEPENDENT HOLDOUT EVALUATION
                # ============================================================

                if ('X_test' in globals()) and ('y_test' in globals()):

                    print("\n Running FINAL UNTOUCHED HOLDOUT evaluation")

                    final_model = create_validation_model(
                        is_classification=is_classification
                    )

                    final_model.fit(X_train, y_train)

                    if is_classification:

                        y_pred = final_model.predict(X_test)
                        y_proba = (
                            final_model.predict_proba(X_test)[:, 1]
                            if hasattr(final_model, "predict_proba")
                            else None
                        )

                        print("\n[HOLDOUT RESULTS]")
                        print("Accuracy:", accuracy_score(y_test, y_pred))
                        print("F1:", f1_score(y_test, y_pred))

                        if y_proba is not None:
                            print("ROC-AUC:", roc_auc_score(y_test, y_proba))
                            print("Brier:", brier_score_loss(y_test, y_proba))

                    else:

                        y_pred = final_model.predict(X_test)

                        print("\n[HOLDOUT RESULTS]")
                        print("R2:", r2_score(y_test, y_pred))
                        print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
                        print("MAE:", mean_absolute_error(y_test, y_pred))

            else:
                print("\nError: X_train or y_train not found")
                print("   Make sure to run the data preparation cells first")

        except Exception as e:
            print(f"\nError: Error in [POINT 1]: {e}")
            import traceback
            traceback.print_exc()

        print("\n[POINT 1] Complete!")

        # -- [CACHE + REPORT SAVE] ---
        _RS['section'] = 'uncategorized'
        try:
            _pipeline_cleanup(label='P1 complete')

            cache_save(
                {'reporter_point1': globals().get('reporter_point1')},
                cell_tag=_CTAG_12, key=_CKEY_12,
                label=f"{target_options}_tp{tp_option}_[POINT 1] Nested CV",
            )
            report_save_state(make_run_key())
        except Exception as _ce_12:
            print(f"[Cache] Could not save nested_cv: {_ce_12}")

# -- Batch dispatch ---
_is_bv_15 = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_15:
    try:
        _cbv_15 = cache_load('main_runner')
        if (_cbv_15 and isinstance(_cbv_15, dict)
                and _cbv_15.get('_batch_archive')
                and len(_cbv_15['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_15['_batch_archive'].values())):
            _batch_archive = _cbv_15['_batch_archive']
            _is_bv_15 = True
    except Exception:
        pass

if _is_bv_15:
    _vgs_15 = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Outer-Fold Nested CV Metrics & Calibration: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_15, (_vk_15, _vd_15) in enumerate(_batch_archive.items()):
        _vl_15 = _vd_15.get('target', _vk_15)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_15+1}/{len(_batch_archive)}] Target: {_vl_15}")
        print(f"{'-' * 68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split']:
            if _vd_15.get(_gk) is not None:
                globals()[_gk] = _vd_15[_gk]
        globals()['target_options'] = _vl_15
        if 'timepoint' in _vd_15:
            globals()['tp_option'] = _vd_15['timepoint']
        _run_val_15()
        # -- Archive validation reporters back into _batch_archive --
        import copy
        if _vk_15 in _batch_archive:
            _batch_archive[_vk_15]['reporter_point1'] = globals().get('reporter_point1')
            _batch_archive[_vk_15]['_within_ncv_results'] = copy.deepcopy(globals().get('_within_ncv_results'))

    for _k, _v in _vgs_15.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Outer-Fold Nested CV Metrics & Calibration: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_val_15()

In [ ]:
#@title V | Extended Demographic Stratified Performance (Sex-SES-ALE-Education)

# Per-subgroup performance evaluation stratified by child sex, SES tertile,
# ALE exposure, parent sex, and parent education. Produces per-subgroup
# performance tables with bootstrap 95% CIs, ROC curve overlays,
# calibration curves, forest plots, and formal fairness gap analysis
# quantifying performance differentials across strata.
#@markdown ---
n_bootstrap = 1000 #@param {type: "integer"}

"""
Stratified Performance & Fairness Analysis
------------------------------------------
Per-subgroup evaluation of predictive performance to assess fairness
and generalisability regarding differential
performance across demographic strata (sex, SES tertile, ALE exposure).

Stratification variables
------------------------
  sex             : Male (1) / Female (2)
  SES             : Tertile-based Low / Mid / High
  high_ALE / low_ALE : Adverse life-event exposure groups
  sex_P           : Parent sex (1 / 2)
  parent_education: Low / Mid / High (tertiles)

Outputs
  Per-subgroup performance tables with bootstrap 95% CIs,
  ROC curve overlay, calibration curves, forest plots,
  and formal fairness gap analysis.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import os
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, accuracy_score, f1_score,
    r2_score, mean_squared_error, mean_absolute_error, roc_curve,
    precision_recall_curve, average_precision_score,
    brier_score_loss, classification_report
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold, KFold
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# -- Read bootstrap count from UI params --
_VAL_N_BOOTSTRAP = n_bootstrap

# ---
# STYLE CONSTANTS
# ---
_SLATE    = '#1F2D3D'
_CHARCOAL = '#34495E'
_STEEL    = '#5D6D7E'
_SILVER   = '#BDC3C7'
_SNOW     = '#FAFBFC'

_SUBGROUP_COLORS = {
    'Overall':          '#455A64',
    'Sex: Male':        '#1565C0',
    'Sex: Female':      '#E91E63',
    'SES: Low':         '#E65100',
    'SES: Mid':         '#F9A825',
    'SES: High':        '#2E7D32',
    'ALE: Low ALE':     '#7B1FA2',
    'ALE: Not Low ALE': '#CE93D8',
    'ALE: High ALE':    '#4A148C',
    'ALE: Not High ALE':'#BA68C8',
    'ParSex: Parent Male':   '#00838F',
    'ParSex: Parent Female': '#4DD0E1',
    'Edu: Low Edu':     '#795548',
    'Edu: Mid Edu':     '#A1887F',
    'Edu: High Edu':    '#3E2723',
}

def _sg_color(label):
    if label in _SUBGROUP_COLORS:
        return _SUBGROUP_COLORS[label]
    for prefix, col in [('Sex:', '#1565C0'), ('SES:', '#E65100'),
                        ('ALE:', '#7B1FA2'), ('ParSex:', '#00838F'),
                        ('Edu:', '#2E7D32')]:
        if label.startswith(prefix):
            return col
    return '#78909C'

plt.rcParams.update({
    'font.family':        'sans-serif',
    'font.sans-serif':   ['DejaVu Sans', 'Helvetica Neue', 'Helvetica', 'Arial'],
    'font.size':          10,
    'axes.titlesize':     12,
    'axes.titleweight':   'bold',
    'axes.labelsize':     10,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'figure.facecolor':   'white',
    'savefig.facecolor':  'white',
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
})


# ---
# CLASSIFICATION METRICS WITH BOOTSTRAP CIs
# ---

def _compute_subgroup_metrics_classification(y_true, y_proba, label,
                                              threshold=0.5, n_boot=_VAL_N_BOOTSTRAP):
    """
    Comprehensive classification metrics with bootstrap 95% CIs.
    Returns both display strings and raw values for plotting.
    """
    y_true = np.asarray(y_true).ravel()
    y_proba = np.asarray(y_proba).ravel()
    n = len(y_true)

    # -- Point estimates ---
    y_pred = (y_proba >= threshold).astype(int)
    n_pos = int(y_true.sum())
    n_neg = n - n_pos
    base_rate = n_pos / n if n > 0 else 0

    auc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan
    auc_pr = average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan
    brier = brier_score_loss(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan

    _cm = confusion_matrix(y_true, y_pred)
    if _cm.shape == (2, 2):
        tn, fp, fn, tp = _cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1   = 2 * ppv * sens / (ppv + sens) if (ppv + sens) > 0 else 0
    acc  = accuracy_score(y_true, y_pred)

    # -- Optimal threshold (F1-maximising on PR curve) ---
    try:
        _pr, _rc, _thr = precision_recall_curve(y_true, y_proba)
        _f1c = 2 * (_pr[:-1] * _rc[:-1]) / (_pr[:-1] + _rc[:-1] + 1e-10)
        _best_idx = np.argmax(_f1c)
        opt_thresh = float(_thr[_best_idx])
        y_pred_opt = (y_proba >= opt_thresh).astype(int)
        _cm_opt = confusion_matrix(y_true, y_pred_opt)
        if _cm_opt.shape == (2, 2):
            tn_o, fp_o, fn_o, tp_o = _cm_opt.ravel()
        else:
            tn_o, fp_o, fn_o, tp_o = 0, 0, 0, 0
        sens_opt = tp_o / (tp_o + fn_o) if (tp_o + fn_o) > 0 else 0
        ppv_opt  = tp_o / (tp_o + fp_o) if (tp_o + fp_o) > 0 else 0
        f1_opt   = 2 * ppv_opt * sens_opt / (ppv_opt + sens_opt) if (ppv_opt + sens_opt) > 0 else 0
    except Exception:
        opt_thresh, sens_opt, ppv_opt, f1_opt = 0.5, sens, ppv, f1

    # -- Bootstrap CIs ---
    rng = np.random.RandomState(42)
    boot = {'auc': [], 'auc_pr': [], 'sens': [], 'spec': [],
            'ppv': [], 'npv': [], 'f1': [], 'brier': []}
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        yb_t = y_true[idx]
        yb_p = y_proba[idx]
        if len(np.unique(yb_t)) < 2:
            continue
        yb_pred = (yb_p >= threshold).astype(int)
        _cmb = confusion_matrix(yb_t, yb_pred)
        if _cmb.shape != (2, 2):
            continue
        tn_b, fp_b, fn_b, tp_b = _cmb.ravel()
        boot['auc'].append(roc_auc_score(yb_t, yb_p))
        boot['auc_pr'].append(average_precision_score(yb_t, yb_p))
        boot['brier'].append(brier_score_loss(yb_t, yb_p))
        boot['sens'].append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
        boot['spec'].append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)
        boot['ppv'].append(tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0)
        boot['npv'].append(tn_b / (tn_b + fn_b) if (tn_b + fn_b) > 0 else 0)
        _s = boot['sens'][-1]; _p = boot['ppv'][-1]
        boot['f1'].append(2 * _p * _s / (_p + _s) if (_p + _s) > 0 else 0)

    def _ci(vals):
        if len(vals) < 10:
            return (np.nan, np.nan)
        return (np.percentile(vals, 2.5), np.percentile(vals, 97.5))

    ci = {k: _ci(v) for k, v in boot.items()}

    return {
        'Subgroup': label, 'N': n, 'Pos': n_pos, 'Neg': n_neg,
        'Base Rate': base_rate,
        # Raw values for plotting
        'auc_val': auc, 'auc_lo': ci['auc'][0], 'auc_hi': ci['auc'][1],
        'auc_pr_val': auc_pr, 'auc_pr_lo': ci['auc_pr'][0], 'auc_pr_hi': ci['auc_pr'][1],
        'sens_val': sens, 'sens_lo': ci['sens'][0], 'sens_hi': ci['sens'][1],
        'spec_val': spec, 'spec_lo': ci['spec'][0], 'spec_hi': ci['spec'][1],
        'ppv_val': ppv, 'ppv_lo': ci['ppv'][0], 'ppv_hi': ci['ppv'][1],
        'npv_val': npv, 'npv_lo': ci['npv'][0], 'npv_hi': ci['npv'][1],
        'f1_val': f1, 'f1_lo': ci['f1'][0], 'f1_hi': ci['f1'][1],
        'brier_val': brier, 'brier_lo': ci['brier'][0], 'brier_hi': ci['brier'][1],
        # Optimal threshold
        'opt_thresh': opt_thresh, 'sens_opt': sens_opt, 'ppv_opt': ppv_opt, 'f1_opt': f1_opt,
        # For ROC/PR curves
        '_y_true': y_true, '_y_proba': y_proba,
    }

# ---
# REGRESSION METRICS WITH BOOTSTRAP CIs
# ---

def _compute_subgroup_metrics_regression(y_true, y_pred, label, n_boot=_VAL_N_BOOTSTRAP):
    """Regression metrics with bootstrap 95% CIs."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    rng = np.random.RandomState(42)
    boot_r2, boot_rmse, boot_mae = [], [], []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        y_t = y_true[idx]
        y_p = y_pred[idx]
        if np.var(y_t) > 0:
            boot_r2.append(r2_score(y_t, y_p))
            boot_rmse.append(np.sqrt(mean_squared_error(y_t, y_p)))
            boot_mae.append(mean_absolute_error(y_t, y_p))
    r2_lo = np.percentile(boot_r2, 2.5) if boot_r2 else np.nan
    r2_hi = np.percentile(boot_r2, 97.5) if boot_r2 else np.nan
    rmse_lo = np.percentile(boot_rmse, 2.5) if boot_rmse else np.nan
    rmse_hi = np.percentile(boot_rmse, 97.5) if boot_rmse else np.nan
    mae_lo = np.percentile(boot_mae, 2.5) if boot_mae else np.nan
    mae_hi = np.percentile(boot_mae, 97.5) if boot_mae else np.nan
    return {
        'Subgroup': label, 'N': len(y_true),
        'R²': f"{r2:.3f} [{r2_lo:.3f}-{r2_hi:.3f}]",
        'RMSE': f"{rmse:.3f} [{rmse_lo:.3f}-{rmse_hi:.3f}]",
        'MAE': f"{mae:.3f} [{mae_lo:.3f}-{mae_hi:.3f}]",
        'r2_val': r2, 'r2_lo': r2_lo, 'r2_hi': r2_hi,
        'rmse_val': rmse, 'rmse_lo': rmse_lo, 'rmse_hi': rmse_hi,
        'mae_val': mae, 'mae_lo': mae_lo, 'mae_hi': mae_hi,
        '_residuals': y_true - y_pred,
    }

# ---
# VISUALIZATION: Classification -- 4-panel figure
# ---

def _plot_classification_stratified(all_rows, results_dir, target_lbl, tp_lbl):
    """
    Publication-quality 4-panel figure for classification stratification:
      Panel A -- Forest plot: AUC-ROC with bootstrap 95% CIs
      Panel B -- Forest plot: AUC-PR with bootstrap 95% CIs
      Panel C -- ROC curves overlaid per subgroup
      Panel D -- Calibration curves per subgroup
    """
    # Filter to rows with valid AUC
    rows = [r for r in all_rows if not np.isnan(r.get('auc_val', np.nan))]
    if len(rows) < 2:
        print("  Warning:  Insufficient subgroups for classification plots.")
        return

    n_sg = len(rows)
    fig = plt.figure(figsize=(22, max(8, n_sg * 0.55 + 3)))
    fig.patch.set_facecolor('white')

    # GridSpec: top row = forest plots, bottom row = curves
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.30,
                           height_ratios=[max(1.0, n_sg * 0.12), 1.0])
    ax_forest_roc = fig.add_subplot(gs[0, 0])
    ax_forest_pr  = fig.add_subplot(gs[0, 1])
    ax_roc        = fig.add_subplot(gs[1, 0])
    ax_cal        = fig.add_subplot(gs[1, 1])

    # -- Panel A: AUC-ROC Forest Plot ---
    plot_rows = list(reversed(rows))
    labels = [r['Subgroup'] for r in plot_rows]
    y_pos = np.arange(len(labels))

    overall_auc = next((r['auc_val'] for r in rows if r['Subgroup'] == 'Overall'), None)

    for i, r in enumerate(plot_rows):
        col = _sg_color(r['Subgroup'])
        val, lo, hi = r['auc_val'], r['auc_lo'], r['auc_hi']
        ax_forest_roc.plot([lo, hi], [i, i], color=col, lw=2.5, solid_capstyle='round')
        ax_forest_roc.plot(val, i, 'o', color=col, ms=9, mec='white', mew=1.5, zorder=5)
        ax_forest_roc.text(hi + 0.008, i,
                           f'{val:.3f} [{lo:.3f}-{hi:.3f}]',
                           va='center', fontsize=8, fontweight='bold', color=col)
        # N annotation on left
        ax_forest_roc.text(lo - 0.008, i,
                           f'n={r["N"]:,}',
                           va='center', ha='right', fontsize=9.5, color=_STEEL)

    if overall_auc is not None:
        ax_forest_roc.axvline(overall_auc, color=_SILVER, ls=':', lw=1, alpha=0.6)

    ax_forest_roc.set_yticks(y_pos)
    ax_forest_roc.set_yticklabels(labels, fontsize=9)
    ax_forest_roc.set_xlabel('AUC-ROC (bootstrap 95% CI)', fontsize=10)
    ax_forest_roc.set_title('A.  AUC-ROC by Subgroup', fontsize=12,
                            fontweight='bold', color=_SLATE, loc='left')
    ax_forest_roc.grid(axis='x', alpha=0.15, ls='--')

    # -- Panel B: AUC-PR Forest Plot ---
    overall_pr = next((r['auc_pr_val'] for r in rows if r['Subgroup'] == 'Overall'), None)

    for i, r in enumerate(plot_rows):
        col = _sg_color(r['Subgroup'])
        val, lo, hi = r['auc_pr_val'], r['auc_pr_lo'], r['auc_pr_hi']
        ax_forest_pr.plot([lo, hi], [i, i], color=col, lw=2.5, solid_capstyle='round')
        ax_forest_pr.plot(val, i, 's', color=col, ms=8, mec='white', mew=1.5, zorder=5)
        ax_forest_pr.text(hi + 0.008, i,
                          f'{val:.3f} [{lo:.3f}-{hi:.3f}]',
                          va='center', fontsize=8, fontweight='bold', color=col)

    # Base rate reference line
    base_rate = next((r['Base Rate'] for r in rows if r['Subgroup'] == 'Overall'), None)
    if base_rate is not None:
        ax_forest_pr.axvline(base_rate, color='#C62828', ls='--', lw=1, alpha=0.5)
        ax_forest_pr.text(base_rate + 0.003, len(plot_rows) - 0.3,
                          f'chance ({base_rate:.3f})', fontsize=9.5,
                          color='#C62828', style='italic')
    if overall_pr is not None:
        ax_forest_pr.axvline(overall_pr, color=_SILVER, ls=':', lw=1, alpha=0.6)

    ax_forest_pr.set_yticks(y_pos)
    ax_forest_pr.set_yticklabels(labels, fontsize=9)
    ax_forest_pr.set_xlabel('AUC-PR (bootstrap 95% CI)', fontsize=10)
    ax_forest_pr.set_title('B.  AUC-PR by Subgroup', fontsize=12,
                           fontweight='bold', color=_SLATE, loc='left')
    ax_forest_pr.grid(axis='x', alpha=0.15, ls='--')

    # -- Panel C: ROC Curves Overlay ---
    for r in rows:
        col = _sg_color(r['Subgroup'])
        lw = 2.5 if r['Subgroup'] == 'Overall' else 1.8
        ls = '-' if r['Subgroup'] == 'Overall' else '-'
        alpha = 1.0 if r['Subgroup'] == 'Overall' else 0.75
        fpr, tpr, _ = roc_curve(r['_y_true'], r['_y_proba'])
        ax_roc.plot(fpr, tpr, color=col, lw=lw, ls=ls, alpha=alpha,
                    label=f"{r['Subgroup']}  (AUC={r['auc_val']:.3f})")

    ax_roc.plot([0, 1], [0, 1], '--', color=_SILVER, lw=1)
    ax_roc.set_xlabel('False Positive Rate', fontsize=10)
    ax_roc.set_ylabel('True Positive Rate', fontsize=10)
    ax_roc.set_title('C.  ROC Curves by Subgroup', fontsize=12,
                     fontweight='bold', color=_SLATE, loc='left')
    ax_roc.legend(fontsize=8, loc='lower right', framealpha=0.95,
                  edgecolor=_SILVER)
    ax_roc.set_xlim([-0.02, 1.02])
    ax_roc.set_ylim([-0.02, 1.02])
    ax_roc.grid(alpha=0.12, ls='--')

    # -- Panel D: Calibration Curves ---
    for r in rows:
        col = _sg_color(r['Subgroup'])
        lw = 2.5 if r['Subgroup'] == 'Overall' else 1.8
        alpha = 1.0 if r['Subgroup'] == 'Overall' else 0.75
        try:
            prob_true, prob_pred = calibration_curve(
                r['_y_true'], r['_y_proba'],
                n_bins=8, strategy='quantile')
            ax_cal.plot(prob_pred, prob_true, 'o-', color=col, lw=lw,
                        alpha=alpha, ms=5, mec='white', mew=0.8,
                        label=f"{r['Subgroup']}  (Brier={r['brier_val']:.3f})")
        except Exception:
            pass

    ax_cal.plot([0, 1], [0, 1], '--', color=_SILVER, lw=1, label='Perfect')
    ax_cal.set_xlabel('Mean Predicted Probability', fontsize=10)
    ax_cal.set_ylabel('Fraction of Positives', fontsize=10)
    ax_cal.set_title('D.  Calibration by Subgroup', fontsize=12,
                     fontweight='bold', color=_SLATE, loc='left')
    ax_cal.legend(fontsize=8, loc='lower right', framealpha=0.95,
                  edgecolor=_SILVER)
    ax_cal.set_xlim([-0.02, 1.02])
    ax_cal.set_ylim([-0.02, 1.02])
    ax_cal.grid(alpha=0.12, ls='--')

    fig.suptitle(
        f'Stratified Classification Performance  ·  {target_lbl}  (T{tp_lbl})',
        fontsize=14, fontweight='bold', color=_SLATE, y=1.01)

    out_path = f"{results_dir}/stratified_classification_4panel.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"   Saved: {out_path}")

# ---
# VISUALIZATION: Classification -- Threshold-optimised comparison
# ---

def _plot_threshold_comparison(all_rows, results_dir, target_lbl, tp_lbl):
    """
    Bar chart comparing default (0.50) vs optimal threshold metrics
    per subgroup: Recall(1), Precision(1), F1(1).
    """
    rows = [r for r in all_rows
            if not np.isnan(r.get('auc_val', np.nan)) and r['Subgroup'] != 'Overall']
    if len(rows) < 1:
        return

    labels = [r['Subgroup'] for r in rows]
    n = len(labels)
    x = np.arange(n)
    w = 0.25

    fig, axes = plt.subplots(1, 3, figsize=(min(20, 6 + n * 1.2), 5))
    fig.patch.set_facecolor('white')

    metrics = [
        ('Recall (Class 1)',    'sens_val', 'sens_opt'),
        ('Precision (Class 1)', 'ppv_val',  'ppv_opt'),
        ('F1 (Class 1)',        'f1_val',   'f1_opt'),
    ]

    for ax, (title, key_def, key_opt) in zip(axes, metrics):
        vals_def = [r[key_def] for r in rows]
        vals_opt = [r[key_opt] for r in rows]

        bars1 = ax.bar(x - w/2, vals_def, w, label='Default (0.50)',
                       color='#90A4AE', edgecolor='white', lw=0.5)
        bars2 = ax.bar(x + w/2, vals_opt, w, label='Optimised',
                       color='#1565C0', edgecolor='white', lw=0.5)

        for b in bars1:
            h = b.get_height()
            if h > 0.01:
                ax.text(b.get_x() + b.get_width()/2, h + 0.01,
                        f'{h:.2f}', ha='center', fontsize=9.5, color='#546E7A')
        for b in bars2:
            h = b.get_height()
            if h > 0.01:
                ax.text(b.get_x() + b.get_width()/2, h + 0.01,
                        f'{h:.2f}', ha='center', fontsize=9.5, color='#1565C0',
                        fontweight='bold')

        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8.5)
        ax.set_title(title, fontsize=11, fontweight='bold', color=_SLATE)
        ax.set_ylim(0, max(max(vals_def + vals_opt) * 1.25, 0.1))
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(axis='y', alpha=0.15, ls='--')

    fig.suptitle(
        f'Default vs Optimised Threshold  ·  {target_lbl}  (T{tp_lbl})',
        fontsize=13, fontweight='bold', color=_SLATE, y=1.02)
    plt.tight_layout()
    out_path = f"{results_dir}/stratified_threshold_comparison.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"   Saved: {out_path}")

# ---
# TABLE: Styled HTML table for classification
# ---

def _display_classification_table(all_rows, target_lbl, tp_lbl):
    """Display a publication-quality styled table with all classification metrics."""
    rows_display = []
    for r in all_rows:
        def _fmt(val, lo, hi):
            if np.isnan(val):
                return '--'
            return f"{val:.3f} [{lo:.3f}-{hi:.3f}]"

        rows_display.append({
            'Subgroup':    r['Subgroup'],
            'N':           r['N'],
            'Pos / Neg':   f"{r['Pos']} / {r['Neg']}",
            'Base Rate':   f"{r['Base Rate']:.3f}",
            'AUC-ROC':     _fmt(r['auc_val'], r['auc_lo'], r['auc_hi']),
            'AUC-PR':      _fmt(r['auc_pr_val'], r['auc_pr_lo'], r['auc_pr_hi']),
            'Brier':       f"{r['brier_val']:.4f}" if not np.isnan(r['brier_val']) else '--',
            'Sens (0.50)': f"{r['sens_val']:.3f}",
            'Spec (0.50)': f"{r['spec_val']:.3f}",
            'PPV (0.50)':  f"{r['ppv_val']:.3f}",
            'F1 (0.50)':   f"{r['f1_val']:.3f}",
            'Opt Thresh':  f"{r['opt_thresh']:.3f}",
            'Sens (opt)':  f"{r['sens_opt']:.3f}",
            'PPV (opt)':   f"{r['ppv_opt']:.3f}",
            'F1 (opt)':    f"{r['f1_opt']:.3f}",
        })

    df = pd.DataFrame(rows_display)
    # Numeric columns for gradient
    _num_cols = ['AUC-ROC', 'AUC-PR', 'Brier', 'Sens (0.50)', 'Spec (0.50)',
                 'PPV (0.50)', 'F1 (0.50)', 'Sens (opt)', 'PPV (opt)', 'F1 (opt)']

    styler = (df.style
        .set_caption(f'Stratified Classification Performance  ·  '
                     f'{target_lbl}  (T{tp_lbl})')
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '14px'), ('font-weight', '700'),
                       ('color', _SLATE), ('padding-bottom', '10px')]},
            {'selector': 'th',
             'props': [('background-color', '#ECEFF1'), ('color', _CHARCOAL),
                       ('font-size', '9.5px'), ('font-weight', '600'),
                       ('text-transform', 'uppercase'), ('padding', '7px 8px')]},
            {'selector': 'td',
             'props': [('padding', '5px 8px'), ('font-size', '10.5px')]},
            {'selector': 'tr:first-child td',
             'props': [('background-color', '#E8EAF6'), ('font-weight', '600')]},
        ])
        .hide(axis='index')
    )
    display(styler)
    return df

# ---
# FAIRNESS SUMMARY
# ---

def _print_fairness_summary(all_rows):
    """Print formal fairness gap analysis."""
    subgroup_rows = [r for r in all_rows if r['Subgroup'] != 'Overall']
    if len(subgroup_rows) < 2:
        return

    print("\n" + "=" * 72)
    print("  FAIRNESS GAP ANALYSIS")
    print("=" * 72)

    # Group subgroups by demographic dimension
    dims = {}
    for r in subgroup_rows:
        prefix = r['Subgroup'].split(':')[0].strip()
        dims.setdefault(prefix, []).append(r)

    for dim, dim_rows in dims.items():
        if len(dim_rows) < 2:
            continue
        aucs = [r['auc_val'] for r in dim_rows if not np.isnan(r['auc_val'])]
        auc_prs = [r['auc_pr_val'] for r in dim_rows if not np.isnan(r['auc_pr_val'])]
        senses = [r['sens_val'] for r in dim_rows]
        f1_opts = [r['f1_opt'] for r in dim_rows]

        print(f"\n  {dim}")
        print(f"  {'-' * 60}")

        if len(aucs) >= 2:
            gap_auc = max(aucs) - min(aucs)
            best = [r['Subgroup'] for r in dim_rows if r['auc_val'] == max(aucs)][0]
            worst = [r['Subgroup'] for r in dim_rows if r['auc_val'] == min(aucs)][0]
            flag = '  WARN' if gap_auc > 0.05 else '  ok'
            print(f"    AUC-ROC gap:  {gap_auc:.4f}  "
                  f"(best: {best}, worst: {worst}){flag}")

        if len(auc_prs) >= 2:
            gap_pr = max(auc_prs) - min(auc_prs)
            flag = '  WARN' if gap_pr > 0.05 else '  ok'
            print(f"    AUC-PR gap:   {gap_pr:.4f}{flag}")

        if len(senses) >= 2:
            gap_sens = max(senses) - min(senses)
            flag = '  WARN' if gap_sens > 0.10 else '  ok'
            print(f"    Sens gap:     {gap_sens:.4f}  (equalized opportunity){flag}")

        if len(f1_opts) >= 2:
            gap_f1 = max(f1_opts) - min(f1_opts)
            flag = '  WARN' if gap_f1 > 0.10 else '  ok'
            print(f"    F1(opt) gap:  {gap_f1:.4f}{flag}")

    print(f"\n  Thresholds: AUC/AUC-PR gap > 0.05 Warning: , Sens/F1 gap > 0.10 WARN")

# ---
# VISUALIZATION: Regression (preserved from original)
# ---

def _plot_regression_stratified(all_rows, results_dir, target_lbl, tp_lbl):
    """Forest plot + residual boxplot for regression."""
    rows = [r for r in all_rows if not np.isnan(r.get('r2_val', np.nan))]
    if len(rows) < 2:
        return

    n_sg = len(rows)
    fig_height = max(6, 0.55 * n_sg + 1.5)
    fig, axes = plt.subplots(1, 2, figsize=(18, fig_height))

    plot_rows = list(reversed(rows))
    labels = [r['Subgroup'] for r in plot_rows]
    y_pos = np.arange(len(labels))

    overall_r2 = next((r['r2_val'] for r in rows if r['Subgroup'] == 'Overall'), None)

    for i, r in enumerate(plot_rows):
        col = _sg_color(r['Subgroup'])
        val, lo, hi = r['r2_val'], r['r2_lo'], r['r2_hi']
        axes[0].plot([lo, hi], [i, i], color=col, lw=2.5, solid_capstyle='round')
        axes[0].plot(val, i, 'o', color=col, ms=9, mec='white', mew=1.5, zorder=5)
        axes[0].text(hi + 0.003, i, f'{val:.3f}', va='center', fontsize=9,
                     fontweight='bold', color=col)

    if overall_r2 is not None:
        axes[0].axvline(overall_r2, color=_SILVER, ls=':', lw=1, alpha=0.6)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(labels, fontsize=9)
    axes[0].set_xlabel('R² (bootstrap 95% CI)', fontsize=11)
    axes[0].set_title('R² by Demographic Subgroup', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.2)

    # Residual boxplot
    _box_data = [r['_residuals'] for r in plot_rows if '_residuals' in r]
    _box_labels = [r['Subgroup'] for r in plot_rows if '_residuals' in r]
    if _box_data:
        bp = axes[1].boxplot(_box_data, vert=False, patch_artist=True,
                             labels=_box_labels,
                             boxprops=dict(facecolor='#E3F2FD', edgecolor='#1565C0'),
                             medianprops=dict(color='#C62828', linewidth=2),
                             whiskerprops=dict(color='#455A64'),
                             capprops=dict(color='#455A64'),
                             flierprops=dict(markerfacecolor='#90A4AE', markersize=3))
        axes[1].axvline(0, color='#C62828', ls='--', lw=1.5, alpha=0.5)
        axes[1].set_xlabel('Residual (Actual − Predicted)', fontsize=11)
        axes[1].set_title('Residual Distribution by Subgroup', fontsize=12, fontweight='bold')

    plt.suptitle(f'Stratified Holdout Performance  ·  {target_lbl}  (T{tp_lbl})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    out_path = f"{results_dir}/stratified_forest_residual.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"   Saved: {out_path}")

# ---
# HELPER: Create Validation Model (independent of Point 1)
# ---

def create_validation_model_p2(is_classification=True):
    """
    Creates the same ensemble model used in main pipeline.
    Matches run_catboost_analysis() with stacking ensemble.
    """
    from sklearn.linear_model import LogisticRegression, LinearRegression

    best_params = dict(_CB_PARAMS)

    model_configs = {
        'catboost': best_params,
        'random_forest': dict(_RF_PARAMS),
        'xgboost': dict(_XGB_PARAMS),
        'tabpfn': dict(PIPELINE_MODEL_CONFIGS['tabpfn']),
    }
    # -- Respect _SKIP_TABPFN gate (across-category / manual skip) --
    if globals().get('_SKIP_TABPFN', False):
        model_configs.pop('tabpfn', None)

    if is_classification:
        meta_model = LogisticRegression(random_state=42)
    else:
        meta_model = LinearRegression()

    model = create_stacking_ensemble(
        task_type='regression' if not is_classification else 'classification',
        model_configs=model_configs,
        meta_model=meta_model,
        cv=5,
        random_state=42
    )

    return model

# ---
# CACHE SYSTEM -- stratified

# ---
# BATCH-AWARE WRAPPER -- Extended Stratified Performance
# ---
# When _batch_archive holds multiple targets with archived state, this
# wrapper iterates over each, restoring globals before each pass.
# In single-target mode the validation runs exactly once (no change).
# ---

def _run_val_16():
    """Run this cell's validation for the CURRENT globals."""
    global all_rows  # expose for downstream forest plot
    # ---
    # -- Validation cell gate (reads UI params) ---
    _skip_demog = not globals().get('run_validation_cells', True)

    # -- Dependency guard ---
    if 'make_run_key' not in globals():
        print('\nWarning:  Cell 0 (Report State) has not been run in this session.')
        print('   Please run Cells 1 → 12 first (in order), then re-run this cell.')
        _RUN_13 = False
    elif _skip_demog:
        print('Skipped:  Cell 13 -- Stratified Performance skipped (run_validation_cells=False)')
        _RUN_13 = False
    else:
        _CTAG_13  = 'stratified'
        _CKEY_13  = make_run_key({'cell': _CTAG_13})
        _CACHED_13 = cache_load(_CTAG_13, _CKEY_13)
        if _CACHED_13:
            print(f"  {_CTAG_13} loaded from cache -- skipping execution.")
            _RUN_13 = False
        else:
            _RS['section'] = 'stratified'
            _done_folds_13 = fold_load_all(_CKEY_13, _CTAG_13)
            _RUN_13 = True

    if _RUN_13:

        try:
            # -- Task-type helpers --
            if 'is_regression' not in globals() or not callable(globals().get('is_regression')):
                globals()['is_regression'] = lambda target: target in (globals().get('continuous_outcome_variables', []))
            if 'is_classification' not in globals() or not callable(globals().get('is_classification')):
                globals()['is_classification'] = lambda target: target in (globals().get('binary_or_categorical_outcome_variables', []))

            # -- Mode detection --
            _is_within_mode = (
                'exported_within_dict' in globals()
                and isinstance(exported_within_dict, dict)
                and len(exported_within_dict) > 0
                and 'split' in globals()
                and split == 'within_categories'
            )

            # -- Top-K categories by performance (from metrics_dict_full) --
            _top_cats_ordered = []
            if _is_within_mode and 'metrics_dict_full' in globals():
                _cat_scores = {}
                for _cat, _mdict in zip(metrics_dict_full.get('category', []),
                                         metrics_dict_full.get('metrics', [])):
                    _m = _mdict.get('ensemble', _mdict.get('meta_model', {}))
                    if _m:
                        _s = _m.get('r2', _m.get('auc', _m.get('accuracy', np.nan)))
                        if not np.isnan(_s):
                            _cat_scores[_cat] = _s
                _top_cats_ordered = sorted(_cat_scores.keys(),
                                           key=lambda c: _cat_scores[c], reverse=True)

            _has_demo = 'sample_valid_test' in globals() and sample_valid_test is not None

            if _is_within_mode and _has_demo:
# ---
                # WITHIN-CATEGORIES MODE (preserved from original)
# ---

                _TOP_K = min(5, len(_top_cats_ordered))
                _cats_to_run = _top_cats_ordered[:_TOP_K]

                print("\n" + "=" * 80)
                print(f" Running Stratified Holdout (Within-Categories, Top {_TOP_K})")
                print("=" * 80)

                task_type = detect_task_type(list(exported_within_dict.values())[0][2])
                is_classification = (task_type == 'classification')
                print(f"   Task type: {task_type}")

                _sv_test = sample_valid_test

                _demo_labels = {}
                if 'sex' in _sv_test.columns:
                    _demo_labels['Sex'] = _sv_test['sex'].map({1: 'Male', 2: 'Female'}).reset_index(drop=True)
                if 'parent_income' in _sv_test.columns:
                    _demo_labels['SES'] = pd.cut(
                        _sv_test['parent_income'], bins=[0, 4, 7, 10],
                        labels=['Low', 'Mid', 'High'], include_lowest=True
                    ).reset_index(drop=True)
                if 'high_ale' in _sv_test.columns:
                    _demo_labels['ALE'] = _sv_test['high_ale'].astype(bool).map(
                        {True: 'High ALE', False: 'Not High ALE'}
                    ).reset_index(drop=True)

                print(f"   Demographics available: {list(_demo_labels.keys())}")

                results_dir = os.path.join(os.getcwd(), 'nestedcv_results_stratified')
                os.makedirs(results_dir, exist_ok=True)

                def _compute_subgroup_r2(y_true, y_pred, label):
                    r2 = r2_score(y_true, y_pred)
                    rng = np.random.RandomState(42)
                    boot_r2 = []
                    for _ in range(_VAL_N_BOOTSTRAP):
                        idx = rng.choice(len(y_true), len(y_true), replace=True)
                        y_t, y_p = np.array(y_true)[idx], np.array(y_pred)[idx]
                        if np.var(y_t) > 0:
                            boot_r2.append(r2_score(y_t, y_p))
                    r2_lo = np.percentile(boot_r2, 2.5) if boot_r2 else np.nan
                    r2_hi = np.percentile(boot_r2, 97.5) if boot_r2 else np.nan
                    return {'Subgroup': label, 'N': len(y_true),
                            'r2': r2, 'r2_lo': r2_lo, 'r2_hi': r2_hi}

                _all_cat_results = {}

                for _cat_name in _cats_to_run:
                    if _cat_name not in exported_within_dict:
                        continue
                    _Xtr, _Xte, _ytr, _yte = exported_within_dict[_cat_name]
                    print(f"\n  [{_cat_name}] ({_Xtr.shape[1]} features)")

                    try:
                        _model = create_validation_model_p2(is_classification=is_classification)
                        _model.fit(_Xtr, _ytr)

                        if is_classification:
                            # Use predict_proba and classification metrics for clf targets.
                            _Xte_df = _Xte if isinstance(_Xte, pd.DataFrame) else pd.DataFrame(np.array(_Xte))
                            _y_proba_cat = (
                                _model.predict_proba(_Xte_df)[:, 1]
                                if hasattr(_model, 'predict_proba')
                                else _model.predict(_Xte_df).astype(float)
                            )
                            _cat_rows = [_compute_subgroup_metrics_classification(
                                np.array(_yte), _y_proba_cat, 'Overall')]

                            for _demo_name, _demo_series in _demo_labels.items():
                                for _grp in _demo_series.dropna().unique():
                                    _mask = (_demo_series == _grp).values[:len(_yte)]
                                    if _mask.sum() > 20:
                                        _cat_rows.append(_compute_subgroup_metrics_classification(
                                            np.array(_yte)[_mask], _y_proba_cat[_mask],
                                            f'{_demo_name}: {_grp}'
                                        ))

                            _all_cat_results[_cat_name] = _cat_rows
                            _overall = _cat_rows[0]
                            print(f"    Overall AUC: {_overall['auc_val']:.4f} "
                                  f"[{_overall['auc_lo']:.4f}-{_overall['auc_hi']:.4f}]")

                        else:
                            _y_pred = _model.predict(_Xte)

                            _cat_rows = [_compute_subgroup_r2(np.array(_yte), _y_pred, 'Overall')]

                            for _demo_name, _demo_series in _demo_labels.items():
                                for _grp in _demo_series.dropna().unique():
                                    _mask = (_demo_series == _grp).values[:len(_yte)]
                                    if _mask.sum() > 20:
                                        _cat_rows.append(_compute_subgroup_r2(
                                            np.array(_yte)[_mask], _y_pred[_mask],
                                            f'{_demo_name}: {_grp}'
                                        ))

                            _all_cat_results[_cat_name] = _cat_rows
                            _overall = _cat_rows[0]
                            print(f"    Overall R²: {_overall['r2']:.4f} "
                                  f"[{_overall['r2_lo']:.4f}-{_overall['r2_hi']:.4f}]")

                    except Exception as _e:
                        print(f"    Warning: Failed: {_e}")
                    finally:
                        try:
                            del _model
                        except NameError:
                            pass
                        _pipeline_cleanup(label=f'P2 {_cat_name}', verbose=False)

                if _all_cat_results:
                    print("\n" + "=" * 80)
                    print("Stratified Performance Summary")
                    print("=" * 80)

                    _subgroups = list({r['Subgroup']
                                      for rows in _all_cat_results.values() for r in rows})
                    _subgroups.sort(key=lambda x: (0 if x == 'Overall' else 1, x))

                    # Use the correct metric key and vmin for clf vs regression.
                    _within_metric_key = 'auc_val' if is_classification else 'r2'
                    _within_metric_lbl = 'AUC' if is_classification else 'R²'
                    _within_vmin       = 0.5 if is_classification else 0

                    _wide_rows = []
                    for _sg in _subgroups:
                        _row = {'Subgroup': _sg}
                        for _cat, _rows in _all_cat_results.items():
                            _match = [r for r in _rows if r['Subgroup'] == _sg]
                            _short = _cat[:25] + '…' if len(_cat) > 25 else _cat
                            if _match:
                                _row[_short] = _match[0].get(_within_metric_key, np.nan)
                            else:
                                _row[_short] = np.nan
                        _wide_rows.append(_row)

                    _wide_df = pd.DataFrame(_wide_rows)
                    _num_cols = [c for c in _wide_df.columns if c != 'Subgroup']
                    _wide_styler = (_wide_df.style
                        .format({c: '{:.4f}' for c in _num_cols})
                        .background_gradient(cmap='RdYlGn', subset=_num_cols, vmin=_within_vmin, axis=None)
                        .set_caption(f'Subgroup {_within_metric_lbl} by Category -- {target_options} (T{tp_option})')
                        .set_table_styles([
                            {'selector': 'caption',
                             'props': [('font-size', '14px'), ('font-weight', '700'),
                                       ('color', '#455A64'), ('padding-bottom', '8px')]},
                            {'selector': 'th',
                             'props': [('background-color', '#ECEFF1'), ('color', '#455A64'),
                                       ('font-size', '10px'), ('font-weight', '600'),
                                       ('padding', '6px 8px')]},
                            {'selector': 'td',
                             'props': [('padding', '5px 8px'), ('font-size', '11px')]},
                        ])
                        .hide(axis='index')
                    )
                    display(_wide_styler)

                    _wide_df.to_csv(f"{results_dir}/stratified_within_categories.csv", index=False)
                    print(f"\nSaved: {results_dir}/stratified_within_categories.csv")

                    # Use the correct metric key for clf vs regression.
                    print(f"\n  Fairness summary (max {_within_metric_lbl} gap within each category):")
                    for _cat, _rows in _all_cat_results.items():
                        _gap_vals = [r.get(_within_metric_key, np.nan) for r in _rows
                                     if r['Subgroup'] != 'Overall'
                                     and not np.isnan(r.get(_within_metric_key, np.nan))]
                        if len(_gap_vals) >= 2:
                            _gap = max(_gap_vals) - min(_gap_vals)
                            _short = _cat[:30]
                            print(f"    {_short}: max gap = {_gap:.4f}")

            elif 'X_train' in globals() and 'y_train' in globals() and _has_demo:
# ---
                # ACROSS-CATEGORIES MODE: Stratified holdout analysis
# ---

                print("\n" + "=" * 80)
                print("  STRATIFIED HOLDOUT PERFORMANCE ANALYSIS")
                print("=" * 80)

                task_type = detect_task_type(y_train)
                is_classification = (task_type == 'classification')
                print(f"   Task type: {task_type}")

                # TRAIN MODEL ONCE, PREDICT ON HOLDOUT
                print("   Training stacking ensemble on X_train...")
                holdout_model = create_validation_model_p2(
                    is_classification=is_classification)
                holdout_model.fit(X_train, y_train)

                if is_classification and hasattr(holdout_model, 'predict_proba'):
                    y_pred_test = holdout_model.predict(X_test)
                    y_proba_test = holdout_model.predict_proba(X_test)[:, 1]
                else:
                    y_pred_test = holdout_model.predict(X_test)
                    y_proba_test = None

                # EXTRACT DEMOGRAPHIC LABELS FOR TEST SET
                _sv_test = sample_valid_test

                # -- Demographic label extraction ---
                _demo_map = {}

                if 'sex' in _sv_test.columns:
                    _demo_map['Sex'] = {
                        'series': _sv_test['sex'].map({1: 'Male', 2: 'Female'}).reset_index(drop=True),
                        'groups': ['Male', 'Female'],
                        'prefix': 'Sex',
                    }
                    print(f"   Sex: {_demo_map['Sex']['series'].value_counts().to_dict()}")

                if 'parent_income' in _sv_test.columns:
                    _demo_map['SES'] = {
                        'series': pd.cut(_sv_test['parent_income'],
                                         bins=[0, 4, 7, 10],
                                         labels=['Low', 'Mid', 'High'],
                                         include_lowest=True).reset_index(drop=True),
                        'groups': ['Low', 'Mid', 'High'],
                        'prefix': 'SES',
                    }
                    print(f"   SES: {_demo_map['SES']['series'].value_counts().to_dict()}")

                if 'low_ale_children_p' in _sv_test.columns:
                    _s = _sv_test['low_ale_children_p'].astype(bool).map(
                        {True: 'Low ALE', False: 'Not Low ALE'}).reset_index(drop=True)
                    if _s.isna().sum() == 0:
                        _demo_map['low_ALE'] = {
                            'series': _s,
                            'groups': ['Low ALE', 'Not Low ALE'],
                            'prefix': 'ALE',
                        }
                        print(f"   low_ALE: {_s.value_counts().to_dict()}")

                if 'high_ale' in _sv_test.columns:
                    _s = _sv_test['high_ale'].astype(bool).map(
                        {True: 'High ALE', False: 'Not High ALE'}).reset_index(drop=True)
                    if _s.isna().sum() == 0:
                        _demo_map['high_ALE'] = {
                            'series': _s,
                            'groups': ['High ALE', 'Not High ALE'],
                            'prefix': 'ALE',
                        }
                        print(f"   high_ALE: {_s.value_counts().to_dict()}")

                if 'sex_P' in _sv_test.columns:
                    _demo_map['sex_P'] = {
                        'series': _sv_test['sex_P'].map(
                            {1: 'Parent Male', 2: 'Parent Female'}).reset_index(drop=True),
                        'groups': ['Parent Male', 'Parent Female'],
                        'prefix': 'ParSex',
                    }
                    print(f"   sex_P: {_demo_map['sex_P']['series'].value_counts().to_dict()}")

                if 'parent_education' in _sv_test.columns:
                    try:
                        _s = pd.qcut(_sv_test['parent_education'], q=3,
                                     labels=['Low Edu', 'Mid Edu', 'High Edu'],
                                     duplicates='drop').reset_index(drop=True)
                        _demo_map['Edu'] = {
                            'series': _s,
                            'groups': ['Low Edu', 'Mid Edu', 'High Edu'],
                            'prefix': 'Edu',
                        }
                        print(f"   parent_education: {_s.value_counts().to_dict()}")
                    except Exception:
                        print("   Warning:  parent_education: qcut failed -- skipping")

                # -- Compute metrics per subgroup ---
                results_dir = os.path.join(os.getcwd(), 'nestedcv_results_stratified')
                os.makedirs(results_dir, exist_ok=True)

                _target_lbl = target_options if 'target_options' in globals() else '?'
                _tp_lbl     = tp_option      if 'tp_option'      in globals() else '?'

                print(f"\n   Computing subgroup metrics (1000 bootstrap iterations each)...\n")

                all_rows = []
                subgroup_masks = {}

                # Overall
                if is_classification:
                    overall = _compute_subgroup_metrics_classification(
                        np.array(y_test), y_proba_test, 'Overall')
                else:
                    overall = _compute_subgroup_metrics_regression(
                        np.array(y_test), y_pred_test, 'Overall')
                all_rows.append(overall)
                subgroup_masks['Overall'] = np.ones(len(X_test), dtype=bool)
                print(f"   Overall (n={len(y_test):,})")

                # Per demographic
                for _dim_name, _dim in _demo_map.items():
                    for _grp in _dim['groups']:
                        mask = (_dim['series'] == _grp).values
                        if mask.sum() > 10:
                            label = f"{_dim['prefix']}: {_grp}"
                            if is_classification:
                                row = _compute_subgroup_metrics_classification(
                                    np.array(y_test)[mask], y_proba_test[mask], label)
                            else:
                                row = _compute_subgroup_metrics_regression(
                                    np.array(y_test)[mask], y_pred_test[mask], label)
                            all_rows.append(row)
                            subgroup_masks[label] = mask
                            print(f"   {label} (n={mask.sum():,})")

                # -- Display results ---

                if is_classification:
                    # 1. Styled table
                    print("\n" + "=" * 72)
                    print("  PERFORMANCE TABLE")
                    print("=" * 72)
                    results_df = _display_classification_table(
                        all_rows, _target_lbl, _tp_lbl)
                    results_df.to_csv(
                        f"{results_dir}/stratified_holdout_classification.csv",
                        index=False)
                    print(f"\n  Saved: {results_dir}/stratified_holdout_classification.csv")

                    # 2. Four-panel figure (forest plots + ROC + calibration)
                    _plot_classification_stratified(
                        all_rows, results_dir, _target_lbl, _tp_lbl)

                    # 3. Threshold comparison bars
                    _plot_threshold_comparison(
                        all_rows, results_dir, _target_lbl, _tp_lbl)

                    # 4. Fairness gap analysis
                    _print_fairness_summary(all_rows)

                else:
                    # Regression path (preserved from original)
                    _internal = {'r2_val', 'rmse_val', 'mae_val', 'r2_lo', 'r2_hi',
                                 'rmse_lo', 'rmse_hi', 'mae_lo', 'mae_hi', '_residuals'}
                    display_cols = [k for k in all_rows[0].keys() if k not in _internal]
                    results_df = pd.DataFrame(all_rows)[display_cols]
                    print(results_df.to_string(index=False))
                    results_df.to_csv(
                        f"{results_dir}/stratified_holdout_results.csv", index=False)
                    print(f"\n  Saved: {results_dir}/stratified_holdout_results.csv")

                    _plot_regression_stratified(
                        all_rows, results_dir, _target_lbl, _tp_lbl)

                # -- Feature importance per subgroup ---
                # Set _run_fi = False to skip on re-runs if computation is slow.
                _run_fi = True

                def _plot_feature_importance_subgroups(
                    model, X_test_df, masks, y_test_arr,
                    is_cls=False, top_n=10, results_dir='.',
                ):
                    """Compute and plot per-subgroup feature importance."""
                    import warnings as _w

                    feature_names = list(X_test_df.columns)
                    rng           = np.random.RandomState(42)
                    method_used   = None

                    def _get_tree_base(m):
                        for attr in ('base_models', 'estimators_', 'named_estimators_'):
                            obj = getattr(m, attr, None)
                            if obj is None:
                                continue
                            items = obj.items() if isinstance(obj, dict) else enumerate(obj)
                            for _, bm in items:
                                t = str(type(bm)).lower()
                                if any(k in t for k in ('catboost', 'xgb', 'forest', 'gradient')):
                                    return bm
                        t = str(type(m)).lower()
                        if any(k in t for k in ('catboost', 'xgb', 'forest')):
                            return m
                        return None

                    tree_bm = _get_tree_base(model)
                    bg_size = min(100, len(X_test_df))
                    bg_idx  = rng.choice(len(X_test_df), bg_size, replace=False)
                    background = X_test_df.iloc[bg_idx]

                    # Probe strategy
                    probe_X = X_test_df.iloc[:min(10, len(X_test_df))]
                    strategy = None
                    explainer_obj = None

                    if tree_bm is not None:
                        try:
                            import shap
                            with _w.catch_warnings():
                                _w.simplefilter('ignore')
                                _exp = shap.TreeExplainer(tree_bm, check_additivity=False)
                                _sv  = _exp.shap_values(probe_X, check_additivity=False)
                            if isinstance(_sv, list):
                                _sv = _sv[1] if len(_sv) > 1 else _sv[0]
                            _sv = np.array(_sv)
                            if _sv.shape[-1] == len(feature_names):
                                strategy      = 'tree'
                                explainer_obj = _exp
                                method_used   = 'SHAP TreeExplainer'
                        except Exception:
                            pass

                    if strategy is None:
                        try:
                            import shap
                            predict_fn = (
                                (lambda x: model.predict_proba(
                                    pd.DataFrame(x, columns=feature_names))[:, 1])
                                if is_cls
                                else (lambda x: model.predict(
                                    pd.DataFrame(x, columns=feature_names)))
                            )
                            with _w.catch_warnings():
                                _w.simplefilter('ignore')
                                _exp = shap.KernelExplainer(predict_fn, background, silent=True)
                                _sv  = _exp.shap_values(probe_X, nsamples=30)
                            if isinstance(_sv, list):
                                _sv = _sv[1] if len(_sv) > 1 else _sv[0]
                            _sv = np.array(_sv)
                            if _sv.shape[-1] == len(feature_names):
                                strategy      = 'kernel'
                                explainer_obj = _exp
                                method_used   = 'SHAP KernelExplainer'
                        except Exception:
                            pass

                    if strategy is None:
                        try:
                            model.predict(probe_X)
                            strategy    = 'permutation'
                            method_used = 'Permutation Importance'
                        except Exception:
                            print("   Warning:  No importance strategy succeeded.")
                            return

                    print(f"\n Feature importance ({method_used}, "
                          f"{len(masks)} subgroups, top {top_n})...")

                    def _importance_for_mask(mask, label):
                        X_sub = X_test_df[mask]
                        y_sub = y_test_arr[mask]
                        n_sub = len(X_sub)
                        if n_sub < 10:
                            return None

                        shap_cap = 150 if strategy == 'kernel' else 300
                        if n_sub > shap_cap:
                            idx    = rng.choice(n_sub, shap_cap, replace=False)
                            X_shap = X_sub.iloc[idx]
                        else:
                            X_shap = X_sub

                        fi = None

                        if strategy == 'tree':
                            try:
                                with _w.catch_warnings():
                                    _w.simplefilter('ignore')
                                    sv = explainer_obj.shap_values(X_shap, check_additivity=False)
                                if isinstance(sv, list):
                                    sv = sv[1] if len(sv) > 1 else sv[0]
                                sv = np.array(sv)
                                if sv.ndim == 1:
                                    sv = sv.reshape(1, -1)
                                fi = pd.Series(np.abs(sv).mean(axis=0),
                                               index=feature_names[:sv.shape[1]])
                                if ('DISPLAY_NAMES' in globals()
                                        and isinstance(DISPLAY_NAMES, dict)
                                        and len(DISPLAY_NAMES) > 0):
                                    fi.index = fi.index.map(
                                        lambda x: DISPLAY_NAMES.get(x, x))
                            except Exception as e:
                                print(f"   Warning:  TreeExplainer failed for {label}: {e}")
                        elif strategy == 'kernel':
                            try:
                                with _w.catch_warnings():
                                    _w.simplefilter('ignore')
                                    sv = explainer_obj.shap_values(X_shap, nsamples=50)
                                if isinstance(sv, list):
                                    sv = sv[1] if len(sv) > 1 else sv[0]
                                sv = np.array(sv)
                                if sv.ndim == 1:
                                    sv = sv.reshape(1, -1)
                                fi = pd.Series(np.abs(sv).mean(axis=0),
                                               index=feature_names[:sv.shape[1]])
                                if ('DISPLAY_NAMES' in globals()
                                        and isinstance(DISPLAY_NAMES, dict)
                                        and len(DISPLAY_NAMES) > 0):
                                    fi.index = fi.index.map(
                                        lambda x: DISPLAY_NAMES.get(x, x))
                            except Exception as e:
                                print(f"   Warning:  KernelExplainer failed for {label}: {e}")
                        elif strategy == 'permutation':
                            # Custom scorer used to bypass sklearn's
                            # response-method detection, which fails on
                            # stacking ensembles that lack _estimator_type.
                            try:
                                from sklearn.inspection import permutation_importance as _pi
                                n_rep = max(5, min(20, n_sub // 30))
                                if is_cls:
                                    def _auc_scorer(estimator, X, y):
                                        """Custom AUC scorer that calls predict_proba directly."""
                                        try:
                                            proba = estimator.predict_proba(X)[:, 1]
                                        except Exception:
                                            proba = estimator.predict(X)
                                        return roc_auc_score(y, proba)
                                    scoring = _auc_scorer
                                else:
                                    def _r2_scorer(estimator, X, y):
                                        """Custom R² scorer that calls .predict() directly."""
                                        return r2_score(y, estimator.predict(X))
                                    scoring = _r2_scorer
                                perm = _pi(model, X_sub, y_sub,
                                           n_repeats=n_rep, random_state=42,
                                           scoring=scoring, n_jobs=-1)
                                vals = np.maximum(perm.importances_mean, 0)
                                fi   = pd.Series(vals, index=feature_names[:len(vals)])
                                if ('DISPLAY_NAMES' in globals()
                                        and isinstance(DISPLAY_NAMES, dict)
                                        and len(DISPLAY_NAMES) > 0):
                                    fi.index = fi.index.map(
                                        lambda x: DISPLAY_NAMES.get(x, x))
                            except Exception as e:
                                print(f"   Warning:  Permutation failed for {label}: {e}")

                        if fi is None or fi.sum() == 0:
                            return None
                        return fi.nlargest(top_n)

                    group_fi = {}
                    for label, mask in masks.items():
                        fi_top = _importance_for_mask(mask, label)
                        if fi_top is not None:
                            group_fi[label] = fi_top
                            print(f"   {label} (n={mask.sum()})")
                        else:
                            print(f"   Skipped:  {label} -- skipped")

                    if not group_fi:
                        print("   Warning:  No importances computed.")
                        return

                    # Plot
                    n_groups  = len(group_fi)
                    ncols     = min(3, n_groups)
                    nrows     = int(np.ceil(n_groups / ncols))
                    subplot_h = max(6.5, top_n * 0.45 + 2.2)
                    subplot_w = 8.0

                    fig, axes = plt.subplots(
                        nrows, ncols,
                        figsize=(subplot_w * ncols, subplot_h * nrows),
                        squeeze=False)
                    axes_flat = axes.flatten()

                    for ax_idx, (label, fi_top) in enumerate(group_fi.items()):
                        ax    = axes_flat[ax_idx]
                        color = _sg_color(label)
                        fi_norm    = fi_top / fi_top.max() if fi_top.max() > 0 else fi_top
                        n_bars     = len(fi_norm)
                        vals_plot  = fi_norm.values[::-1]
                        raw_plot   = fi_top.values[::-1]
                        feats_plot = list(fi_norm.index[::-1])
                        y_pos      = np.arange(n_bars)
                        alphas     = np.linspace(0.40, 0.92, n_bars)

                        for j, (yp, val, raw, alpha) in enumerate(
                            zip(y_pos, vals_plot, raw_plot, alphas)):
                            ax.barh(yp, val, color=color, alpha=alpha,
                                    edgecolor='none', height=0.72)
                            ax.text(val + 0.015, yp, f'{raw:.4f}',
                                    va='center', ha='left', fontsize=9.5,
                                    color='#333333', fontweight='bold')

                        feat_labels = [(f[:28] + '…' if len(f) > 28 else f)
                                       for f in feats_plot]
                        ax.set_yticks(y_pos)
                        ax.set_yticklabels(feat_labels, fontsize=8.5)
                        n_obs = int(masks[label].sum())
                        ax.set_title(f'{label}\n(n = {n_obs:,})',
                                     fontsize=10, fontweight='bold',
                                     color=color, pad=7)
                        ax.set_xlabel('Relative importance', fontsize=8)
                        ax.set_xlim([0, 1.35])
                        ax.axvline(0, color=color, lw=4, alpha=0.5)
                        ax.spines['top'].set_visible(False)
                        ax.spines['right'].set_visible(False)
                        ax.spines['left'].set_visible(False)
                        ax.tick_params(axis='y', length=0, pad=4)
                        ax.grid(axis='x', alpha=0.18, ls='--', lw=0.8)

                    for ax_idx in range(n_groups, len(axes_flat)):
                        axes_flat[ax_idx].set_visible(False)

                    fig.suptitle(
                        f'Feature Importance per Subgroup\n'
                        f'Top {top_n} features  |  {method_used}',
                        fontsize=13, fontweight='bold', y=1.015)
                    plt.tight_layout(rect=[0, 0, 1, 0.97], h_pad=4.0, w_pad=3.5)
                    out_path = f"{results_dir}/stratified_feature_importance.png"
                    plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
                    plt.show()
                    print(f"   Saved: {out_path}")

                if _run_fi:
                    _plot_feature_importance_subgroups(
                        model       = holdout_model,
                        X_test_df   = X_test if isinstance(X_test, pd.DataFrame)
                                      else pd.DataFrame(X_test),
                        masks       = subgroup_masks,
                        y_test_arr  = np.array(y_test),
                        is_cls      = is_classification,
                        top_n       = 10,
                        results_dir = results_dir,
                    )
                else:
                    print("  Skipped:  Feature importance skipped (_run_fi = False)")

            else:
                if not _has_demo:
                    print("\nError: sample_valid_test not found")
                else:
                    print("\nError: Required variables not found")

        except Exception as e:
            print(f"\nError: Error in stratified analysis: {e}")
            import traceback
            traceback.print_exc()

        print("\n" + "=" * 72)
        print("  Stratified Performance Analysis Complete")
        print("=" * 72)

        # -- [CACHE + REPORT SAVE] ---
        _RS['section'] = 'uncategorized'
        try:
            _pipeline_cleanup(label='P2 complete')

            cache_save(
                {'reporter': locals().get('reporter') or locals().get('stratified_reporter')},
                cell_tag=_CTAG_13, key=_CKEY_13,
                label=f"{target_options}_tp{tp_option}_Stratified Performance",
            )
            report_save_state(make_run_key())
        except Exception as _ce_13:
            print(f"[Cache] Could not save stratified: {_ce_13}")

# -- Batch dispatch ---
_is_bv_16 = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_16:
    try:
        _cbv_16 = cache_load('main_runner')
        if (_cbv_16 and isinstance(_cbv_16, dict)
                and _cbv_16.get('_batch_archive')
                and len(_cbv_16['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_16['_batch_archive'].values())):
            _batch_archive = _cbv_16['_batch_archive']
            _is_bv_16 = True
    except Exception:
        pass

if _is_bv_16:
    _vgs_16 = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split',
        'sample_valid','sample_valid_test','sample_valid_train',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Extended Stratified Performance: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_16, (_vk_16, _vd_16) in enumerate(_batch_archive.items()):
        _vl_16 = _vd_16.get('target', _vk_16)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_16+1}/{len(_batch_archive)}] Target: {_vl_16}")
        print(f"{'-' * 68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split',
                    'sample_valid','sample_valid_test','sample_valid_train']:
            if _vd_16.get(_gk) is not None:
                globals()[_gk] = _vd_16[_gk]
        globals()['target_options'] = _vl_16
        if 'timepoint' in _vd_16:
            globals()['tp_option'] = _vd_16['timepoint']
        _run_val_16()
        # -- Archive validation reporters back into _batch_archive --
        import copy
        if _vk_16 in _batch_archive:
            _batch_archive[_vk_16]['all_rows'] = copy.deepcopy(globals().get('all_rows'))

    for _k, _v in _vgs_16.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Extended Stratified Performance: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_val_16()

# ---
# BATCH FAIRNESS HEATMAP -- Cross-Target Demographic Equity Summary
# Rows = targets, Columns = demographic subgroups, Cells = R² or AUC.
# Final column = max disparity (worst − best subgroup) flagged in red.
# Works for both regression and classification.
# ---

try:
    _is_batch_15 = ('_batch_archive' in dir() and isinstance(_batch_archive, dict) and len(_batch_archive) > 1)
    if not _is_batch_15:
        try:
            _cbv15 = cache_load('main_runner')
            if _cbv15 and isinstance(_cbv15, dict) and _cbv15.get('_batch_archive') and len(_cbv15['_batch_archive']) > 1:
                _batch_archive = _cbv15['_batch_archive']; _is_batch_15 = True
        except Exception: pass

    if _is_batch_15:
        import pandas as pd, numpy as np
        from matplotlib.colors import LinearSegmentedColormap

        _TDN = TARGET_DISPLAY_NAMES if 'TARGET_DISPLAY_NAMES' in dir() else {}
        _fair_rows = []

        # Read fairness data from _batch_archive[key]['all_rows'] (stored by batch wrapper above)
        _fair_source = {}
        for _fk, _fv in _batch_archive.items():
            _ar = _fv.get('all_rows')
            if _ar and isinstance(_ar, list) and len(_ar) > 0:
                _fair_source[_fk] = _ar
        if len(_fair_source) < 2:
            print('  Skipped:  Batch fairness heatmap: <2 targets with subgroup data')
            _fair_source = {}  # skip rendering

        for _fk, _fair_data in _fair_source.items():
            _tname = _fk.split('__tp')[0] if '__tp' in _fk else _fk
            _tlabel = _TDN.get(_tname, _tname)
            _is_clf_fair = any('auc_val' in r for r in _fair_data if isinstance(r, dict))
            _metric_key = 'auc_val' if _is_clf_fair else 'r2_val'
            _row = {'Target': _tlabel}
            _vals = []
            for _sg in _fair_data:
                if isinstance(_sg, dict) and _metric_key in _sg:
                    _sg_name = _sg.get('Subgroup', 'Unknown')
                    _row[_sg_name] = round(_sg[_metric_key], 3)
                    if _sg_name != 'Overall':
                        _vals.append(_sg[_metric_key])
            if _vals and len(_vals) >= 2:
                _row['Max Gap'] = round(max(_vals) - min(_vals), 3)
            _fair_rows.append(_row)

        if len(_fair_rows) >= 2:
            _fair_df = pd.DataFrame(_fair_rows).set_index('Target')
            # Separate Max Gap column
            _gap_col = _fair_df.pop('Max Gap') if 'Max Gap' in _fair_df.columns else None

            # Sort by Overall descending
            if 'Overall' in _fair_df.columns:
                _fair_df = _fair_df.sort_values('Overall', ascending=False)
                if _gap_col is not None:
                    _gap_col = _gap_col.loc[_fair_df.index]

            _n_r = len(_fair_df)
            _n_c = len(_fair_df.columns)
            _fig_w = max(12, _n_c * 1.1 + 4)
            _fig_h = max(5, _n_r * 0.5 + 2)

            fig_fair, ax_fair = plt.subplots(figsize=(_fig_w, _fig_h))

            _is_clf_mode = any('auc' in str(c).lower() for c in _fair_df.columns) or (_fair_df.values[~np.isnan(_fair_df.values.astype(float))].mean() > 0.4)
            _vmin_f = 0.5 if _is_clf_mode else 0
            _vmax_f = 1.0 if _is_clf_mode else max(0.3, np.nanmax(_fair_df.values.astype(float)) + 0.02)

            _cmap_fair = LinearSegmentedColormap.from_list(
                'fair_cmap', ['#FEE2E2', '#FECACA', '#FFFFFF', '#BAE6FD', '#3B82F6'], N=256)

            _im_f = ax_fair.imshow(_fair_df.values.astype(float), aspect='auto',
                                    cmap=_cmap_fair, vmin=_vmin_f, vmax=_vmax_f)

            # Cell annotations
            for _ri in range(_n_r):
                for _ci in range(_n_c):
                    _v = _fair_df.values[_ri, _ci]
                    if not np.isnan(float(_v)):
                        _tc = '#1B2A4A' if float(_v) > (_vmin_f + _vmax_f) / 2 else '#7F1D1D'
                        ax_fair.text(_ci, _ri, f'{float(_v):.3f}', ha='center', va='center',
                                     fontsize=8, color=_tc, fontweight='bold')

            ax_fair.set_xticks(range(_n_c))
            ax_fair.set_xticklabels(_fair_df.columns, rotation=40, ha='right', fontsize=9)
            ax_fair.set_yticks(range(_n_r))
            ax_fair.set_yticklabels(_fair_df.index, fontsize=9)

            _metric_label = 'AUC' if _is_clf_mode else 'R²'
            ax_fair.set_title(f'Batch Fairness -- {_metric_label} by Demographic Subgroup\n'
                              f'({_n_r} targets × {_n_c} subgroups)',
                              fontsize=13, fontweight='bold', pad=14)

            _cbar_f = plt.colorbar(_im_f, ax=ax_fair, shrink=0.7, pad=0.02)
            _cbar_f.set_label(_metric_label, fontsize=10)

            # Max gap annotation on right margin
            if _gap_col is not None:
                for _ri, (_tgt, _gap) in enumerate(_gap_col.items()):
                    if pd.notna(_gap):
                        _gc = '#DC2626' if _gap > 0.05 else '#6B7280'
                        ax_fair.text(_n_c + 0.3, _ri, f'Δ{_gap:.3f}',
                                     va='center', ha='left', fontsize=8, fontweight='bold',
                                     color=_gc)
                ax_fair.text(_n_c + 0.3, -0.8, 'Max Gap', ha='left', fontsize=8,
                             fontweight='bold', color='#374151')

            plt.tight_layout()
            try:
                fig_fair.savefig(f"{results_dir_summary}/batch_fairness_heatmap.png",
                                 dpi=250, bbox_inches='tight', facecolor='white')
                fig_fair.savefig(f"{results_dir_summary}/batch_fairness_heatmap.pdf",
                                 bbox_inches='tight', facecolor='white')
                print(f"   Saved: batch_fairness_heatmap.png / .pdf")
            except Exception: pass
            plt.show()

except Exception as _e_fair:
    print(f"  Warning:  Batch fairness heatmap: {_e_fair}")

In [ ]:
#@title V | Family / Multi-Site Generalization & Leave-Site-Out Cross-Validation

# Family-grouped CV assigns siblings to the same fold via rel_family_id,
# testing whether within-family clustering (shared genetics, environment,
# rater) inflates standard CV performance. Site-grouped CV holds out
# entire ABCD acquisition sites to assess site-level confounding across
# the 21 sites. Both use GroupKFold. Supports family-only, site-only,
# or combined modes.

#@markdown ---
n_outer_folds = 5 #@param {type: "integer"}
robustness_mode = "Family Only" #@param ["Family Only", "Site Only", "Both"]

"""
Group-Aware Cross-Validation for Confound Assessment
------------------------------------------------------
Tests whether performance is inflated by familial clustering (twin/
sibling pairs) or site effects within standard CV folds. Family-
grouped CV assigns siblings to the same fold; site-grouped CV holds
out entire ABCD sites. Both procedures isolate the contribution of
familial- and site-level confounds to apparent model performance.

Variants
  Family-grouped CV : siblings assigned to the same fold; assesses
                       leakage from familial relatedness.
  Site-grouped CV   : all participants from one ABCD site held out
                       together; assesses site-level generalisation.

Outputs
  Performance comparison table (standard CV vs each grouped variant);
  a meaningful drop in grouped CV performance indicates confound sensitivity.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, accuracy_score,
    r2_score, mean_squared_error, mean_absolute_error
)
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold, StratifiedGroupKFold


# ---
# GroupedCVCheck Class
# ---

class GroupedCVCheck:
    """
    Test model robustness to confounds via group-aware cross-validation.

    Grouping strategies:
    1. Standard CV: Random stratified splits (baseline)
    2. Family CV: Siblings always in same fold (tests family confound)
    3. Site CV: Site members always in same fold (tests site confound)

    For each grouping, computes metrics and compares to baseline.
    Large performance drop = potential confound bias.
    """

    def __init__(self, X, y, task_type='auto', n_outer=5, random_state=42):
        """        X, y: training data. task_type defaults to auto-detection.
        n_outer : int
            Number of CV folds
        random_state : int
            Random seed
        """
        self.X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        # Ensure y is always 1-dimensional (batch archive may produce (N,1) arrays)
        if isinstance(y, pd.Series):
            self.y = y
        elif isinstance(y, pd.DataFrame):
            self.y = y.iloc[:, 0]
        else:
            _y = np.asarray(y)
            if _y.ndim > 1:
                _y = _y.ravel()
            self.y = pd.Series(_y)
        self.n_outer = n_outer
        self.random_state = random_state

        if task_type == 'auto':
            self.task_type = detect_task_type(self.y.values)
        else:
            self.task_type = task_type

        self.is_classification = self.task_type == 'classification'
        self.is_regression = self.task_type == 'regression'

        self.results = {
            'standard': {},      # {metric: [fold_values]}
            'family_grouped': {},
            'site_grouped': {}
        }

        print(f"Task type: {self.task_type.upper()}")

    def run_group_cv(self, groups, group_type='family', model_factory=None, base_model=None):
        """
        Run group-aware CV.

        Parameters:
        groups : pd.Series or np.array
            Group IDs (family_id, site_id, etc.) aligned with X
        group_type : str
            'family', 'site', or other label for reporting
        model_factory : callable, optional
            Function returning fresh model
        base_model : estimator, optional
            Pre-trained model from pipeline
        """
        groups = pd.Series(groups).reset_index(drop=True)

        if len(groups) != len(self.X):
            raise ValueError(f"groups length {len(groups)} != X length {len(self.X)}")

        print(f"\n Running robustness test: {group_type.upper()}-AWARE CV")
        print(f"   Groups: {len(groups.unique())} unique {group_type} IDs")
        print(f"   Samples per group: mean={groups.value_counts().mean():.1f}, "
              f"min={groups.value_counts().min()}, max={groups.value_counts().max()}")

        use_base_model = base_model is not None

        if self.is_classification:
            return self._robustness_cv_classification(groups, group_type, use_base_model, base_model, model_factory)
        else:
            return self._robustness_cv_regression(groups, group_type, use_base_model, base_model, model_factory)

    def _robustness_cv_classification(self, groups, group_type, use_base_model, base_model, model_factory):
        """Robustness test for classification."""
        from sklearn.metrics import roc_curve
        group_results = {
            'fold': [], 'auc': [], 'accuracy': [], 'sensitivity': [],
            'specificity': [], 'ppv': []
        }

        # Use StratifiedGroupKFold to preserve class balance within groups
        n_unique_groups = len(groups.unique())
        n_splits = min(self.n_outer, n_unique_groups)

        if n_splits < 2:
            print(f"   Warning:  Not enough groups for CV (need >= 2, have {n_unique_groups})")
            return None

        try:
            sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=self.random_state)
            cv_iterator = sgkf.split(self.X, self.y, groups=groups)
        except Exception as e:
            print(f"[WARNING] {e}")
            # Fallback if StratifiedGroupKFold not available
            print("   Falling back to GroupKFold (no stratification)")
            gkf = GroupKFold(n_splits=n_splits)
            cv_iterator = gkf.split(self.X, self.y, groups=groups)
        for fold_num, (train_idx, test_idx) in enumerate(cv_iterator):
            X_train = self.X.iloc[train_idx]
            X_test = self.X.iloc[test_idx]
            y_train = self.y.iloc[train_idx]
            y_test = self.y.iloc[test_idx]
            if use_base_model:
                model = base_model
            else:
                model = model_factory()
                model.fit(X_train, y_train)
            y_proba = model.predict_proba(X_test)[:, 1]
            fpr, tpr, thresholds = roc_curve(y_test, y_proba)
            if len(thresholds) > 1:
                optimal_idx = np.argmax(tpr - fpr)
                optimal_threshold = thresholds[optimal_idx]
            else:
                optimal_threshold = 0.5
            y_pred = (y_proba >= optimal_threshold).astype(int)
            _cm = confusion_matrix(y_test, y_pred)
            if _cm.shape == (2, 2):
                tn, fp, fn, tp = _cm.ravel()
            else:
                tn, fp, fn, tp = 0, 0, 0, 0

            group_results['fold'].append(fold_num)
            group_results['auc'].append(roc_auc_score(y_test, y_proba))
            group_results['accuracy'].append(accuracy_score(y_test, y_pred))
            group_results['sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            group_results['specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
            group_results['ppv'].append(tp / (tp + fp) if (tp + fp) > 0 else 0)

            print(f"   Fold {fold_num+1}: AUC={group_results['auc'][-1]:.4f}")

        self.results[f'{group_type}_grouped'] = group_results
        return group_results

    def _robustness_cv_regression(self, groups, group_type, use_base_model, base_model, model_factory):
        """Robustness test for regression."""
        group_results = {
            'fold': [], 'r2': [], 'rmse': [], 'mae': []
        }

        # Use GroupKFold
        n_unique_groups = len(groups.unique())
        n_splits = min(self.n_outer, n_unique_groups)

        if n_splits < 2:
            print(f"   Warning:  Not enough groups for CV (need >= 2, have {n_unique_groups})")
            return None

        gkf = GroupKFold(n_splits=n_splits)
        for fold_num, (train_idx, test_idx) in enumerate(gkf.split(self.X, self.y, groups=groups)):
            X_train = self.X.iloc[train_idx]
            X_test = self.X.iloc[test_idx]
            y_train = self.y.iloc[train_idx]
            y_test = self.y.iloc[test_idx]
            if use_base_model:
                model = base_model
            else:
                model = model_factory()
                model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            group_results['fold'].append(fold_num)
            group_results['r2'].append(r2_score(y_test, y_pred))
            group_results['rmse'].append(np.sqrt(mean_squared_error(y_test, y_pred)))
            group_results['mae'].append(mean_absolute_error(y_test, y_pred))

            print(f"   Fold {fold_num+1}: R²={group_results['r2'][-1]:.4f}")

        self.results[f'{group_type}_grouped'] = group_results
        return group_results

    def run_standard_cv(self, model_factory=None, base_model=None):
        """Run standard CV (baseline for comparison)."""
        print(f"\n Running baseline: STANDARD CV")

        use_base_model = base_model is not None

        if self.is_classification:
            return self._standard_cv_classification(use_base_model, base_model, model_factory)
        else:
            return self._standard_cv_regression(use_base_model, base_model, model_factory)

    def _standard_cv_classification(self, use_base_model, base_model, model_factory):
        """Standard CV for classification (baseline)."""
        from sklearn.metrics import roc_curve

        standard_results = {
            'fold': [], 'auc': [], 'accuracy': [], 'sensitivity': [],
            'specificity': [], 'ppv': []
        }

        skf = StratifiedKFold(n_splits=self.n_outer, shuffle=True, random_state=self.random_state)

        for fold_num, (train_idx, test_idx) in enumerate(skf.split(self.X, self.y)):
            X_train = self.X.iloc[train_idx]
            X_test = self.X.iloc[test_idx]
            y_train = self.y.iloc[train_idx]
            y_test = self.y.iloc[test_idx]

            if use_base_model:
                model = base_model
            else:
                model = model_factory()
                model.fit(X_train, y_train)

            y_proba = model.predict_proba(X_test)[:, 1]
            fpr, tpr, thresholds = roc_curve(y_test, y_proba)
            if len(thresholds) > 1:
                optimal_idx = np.argmax(tpr - fpr)
                optimal_threshold = thresholds[optimal_idx]
            else:
                optimal_threshold = 0.5
            y_pred = (y_proba >= optimal_threshold).astype(int)

            _cm = confusion_matrix(y_test, y_pred)
            if _cm.shape == (2, 2):
                tn, fp, fn, tp = _cm.ravel()
            else:
                tn, fp, fn, tp = 0, 0, 0, 0

            standard_results['fold'].append(fold_num)
            standard_results['auc'].append(roc_auc_score(y_test, y_proba))
            standard_results['accuracy'].append(accuracy_score(y_test, y_pred))
            standard_results['sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else 0)
            standard_results['specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else 0)
            standard_results['ppv'].append(tp / (tp + fp) if (tp + fp) > 0 else 0)

            print(f"   Fold {fold_num+1}: AUC={standard_results['auc'][-1]:.4f}")

        self.results['standard'] = standard_results
        return standard_results

    def _standard_cv_regression(self, use_base_model, base_model, model_factory):
        """Standard CV for regression (baseline)."""
        standard_results = {
            'fold': [], 'r2': [], 'rmse': [], 'mae': []
        }

        kf = KFold(n_splits=self.n_outer, shuffle=True, random_state=self.random_state)

        for fold_num, (train_idx, test_idx) in enumerate(kf.split(self.X)):
            X_train = self.X.iloc[train_idx]
            X_test = self.X.iloc[test_idx]
            y_train = self.y.iloc[train_idx]
            y_test = self.y.iloc[test_idx]

            if use_base_model:
                model = base_model
            else:
                model = model_factory()
                model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            standard_results['fold'].append(fold_num)
            standard_results['r2'].append(r2_score(y_test, y_pred))
            standard_results['rmse'].append(np.sqrt(mean_squared_error(y_test, y_pred)))
            standard_results['mae'].append(mean_absolute_error(y_test, y_pred))

            print(f"   Fold {fold_num+1}: R²={standard_results['r2'][-1]:.4f}")

        self.results['standard'] = standard_results
        return standard_results

    def generate_report(self, title="ROBUSTNESS ANALYSIS"):
        """Generate robustness comparison report."""
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)
        print(f"\nTask Type: {self.task_type.upper()}")
        print(f"CV folds: {self.n_outer}\n")

        # Determine metrics
        if self.is_classification:
            metrics = ['auc', 'accuracy', 'sensitivity', 'specificity']
        else:
            metrics = ['r2', 'rmse', 'mae']

        # Build comparison table
        comparison_rows = []

        for cv_type in ['standard', 'family_grouped', 'site_grouped']:
            if cv_type not in self.results or not self.results[cv_type]:
                continue

            row = {'CV Strategy': cv_type.replace('_', ' ').title()}

            for metric in metrics:
                if metric in self.results[cv_type]:
                    values = self.results[cv_type][metric]
                    if values:
                        mean_val = np.mean(values)
                        std_val = np.std(values, ddof=1)
                        row[f'{metric}'] = f"{mean_val:.3f} ± {std_val:.3f}"

            comparison_rows.append(row)

        comparison_df = pd.DataFrame(comparison_rows)

        print("ROBUSTNESS COMPARISON")
        print("-" * 80)
        print(comparison_df.to_string(index=False))

        # Interpretation
        if 'standard' in self.results and 'family_grouped' in self.results:
            print("\n INTERPRETATION:")
            print("-" * 80)

            if self.is_classification:
                standard_auc = np.mean(self.results['standard']['auc'])
                family_auc = np.mean(self.results['family_grouped']['auc'])
                drop = (standard_auc - family_auc) / standard_auc * 100

                print(f"Family confound: AUC drop = {drop:.1f}%")
                if drop < 5:
                    print("   Low family confound (robust to family clustering)")
                elif drop < 15:
                    print("   Warning:  Moderate family confound (consider in interpretation)")
                else:
                    print("   Warning:  High family confound (significant bias detected)")
            else:
                standard_r2 = np.mean(self.results['standard']['r2'])
                family_r2 = np.mean(self.results['family_grouped']['r2'])
                drop = (standard_r2 - family_r2) / (standard_r2 + 1e-10) * 100

                print(f"Family confound: R² drop = {drop:.1f}%")

        print("\n" + "=" * 80)
        print("[POINT 3] COMPLETE")
        print("=" * 80)

# ---
# HELPER: Create Validation Model (independent of Point 1)
# ---

def create_validation_model_p3(is_classification=True):
    """
    Creates the same ensemble model used in main pipeline.
    Independent of Point 1.
    """
    from sklearn.linear_model import LogisticRegression, LinearRegression

    best_params = dict(_CB_PARAMS)

    model_configs = {
        'catboost': best_params,
        'random_forest': dict(_RF_PARAMS),
        'xgboost': dict(_XGB_PARAMS),
        'tabpfn': dict(PIPELINE_MODEL_CONFIGS['tabpfn']),
    }
    # -- Respect _SKIP_TABPFN gate (across-category / manual skip) --
    if globals().get('_SKIP_TABPFN', False):
        model_configs.pop('tabpfn', None)

    if is_classification:
        meta_model = LogisticRegression(random_state=42)
    else:
        meta_model = LinearRegression()

    model = create_stacking_ensemble(
        task_type='regression' if not is_classification else 'classification',
        model_configs=model_configs,
        meta_model=meta_model,
        cv=5,
        random_state=42
    )

    # TabPFN now opt-in via model_configs (no deletion needed)

    return model

# ---
# CACHE SYSTEM -- robustness

# ---
# BATCH-AWARE WRAPPER -- Site Robustness
# ---
# When _batch_archive holds multiple targets with archived state, this
# wrapper iterates over each, restoring globals before each pass.
# In single-target mode the validation runs exactly once (no change).
# ---

def _run_val_17():
    global robust_reporter  # expose for downstream consumption
    global _site_score  # expose for downstream consumption
    """Run this cell's validation for the CURRENT globals."""
    # ---
    # -- Validation cell gate (reads Cell 6 params) ---
    _skip_grouped = not globals().get('run_validation_cells', True)
    _VAL_N_OUTER = n_outer_folds
    _run_family = robustness_mode in ("Family Only", "Both")
    _run_site   = robustness_mode in ("Site Only",   "Both")

    # -- Dependency guard ---
    if 'make_run_key' not in globals():
        print('\nWarning:  Cell 0 (Report State) has not been run in this session.')
        print('   Please run Cells 1 → 12 first (in order), then re-run this cell.')
        _RUN_14 = False
    elif _skip_grouped:
        print('Skipped:  Cell 14 -- Site Robustness skipped (run_validation_cells=False)')
        _RUN_14 = False
    else:
        _CTAG_14  = 'robustness'
        _CKEY_14  = make_run_key({'cell': _CTAG_14})
        _CACHED_14 = cache_load(_CTAG_14, _CKEY_14)
        if _CACHED_14:
            print(f"  {_CTAG_14} loaded from cache -- skipping execution.")
            _RUN_14 = False
        else:
            _RS['section'] = 'robustness'
            _done_folds_14 = fold_load_all(_CKEY_14, _CTAG_14)
            _RUN_14 = True

    if _RUN_14:

        # EXECUTION: Run robustness reporter
        # ======================================================================

        try:
            # -- Task-type helpers --
            if 'is_regression' not in globals() or not callable(globals().get('is_regression')):
                globals()['is_regression'] = lambda target: target in (globals().get('continuous_outcome_variables', []))
            if 'is_classification' not in globals() or not callable(globals().get('is_classification')):
                globals()['is_classification'] = lambda target: target in (globals().get('binary_or_categorical_outcome_variables', []))

            # -- Mode detection --
            _is_within_mode = (
                'exported_within_dict' in globals()
                and isinstance(exported_within_dict, dict)
                and len(exported_within_dict) > 0
                and 'split' in globals()
                and split == 'within_categories'
            )

            # -- Top-K categories by performance (from metrics_dict_full) --
            _top_cats_ordered = []
            if _is_within_mode and 'metrics_dict_full' in globals():
                _cat_scores = {}
                for _cat, _mdict in zip(metrics_dict_full.get('category', []),
                                         metrics_dict_full.get('metrics', [])):
                    _m = _mdict.get('ensemble', _mdict.get('meta_model', {}))
                    if _m:
                        _s = _m.get('r2', _m.get('auc', _m.get('accuracy', np.nan)))
                        if not np.isnan(_s):
                            _cat_scores[_cat] = _s
                _top_cats_ordered = sorted(_cat_scores.keys(), key=lambda c: _cat_scores[c], reverse=True)

            _has_demo = 'sample_valid' in globals() and sample_valid is not None

            if _is_within_mode and _has_demo:
# ---
                # WITHIN-CATEGORIES MODE: Per-category site/family robustness
# ---

                _TOP_K = min(5, len(_top_cats_ordered))
                _cats_to_run = _top_cats_ordered[:_TOP_K]

                print("\n" + "=" * 80)
                print(f" Running [POINT 3] -- Site/Family Robustness (Within-Categories, Top {_TOP_K})")
                print("=" * 80)

                task_type = detect_task_type(list(exported_within_dict.values())[0][2])
                is_classification = (task_type == 'classification')
                _metric_key = 'auc' if is_classification else 'r2'
                _metric_name = 'AUC' if is_classification else 'R²'
                print(f"   Task type: {task_type}")

                has_family = 'rel_family_id' in sample_valid.columns
                has_site = 'L_site_id' in sample_valid.columns
                print(f"   Available confounds: family={has_family}, site={has_site}")

                results_dir_p3 = os.path.join(os.getcwd(), 'robustness_results')
                os.makedirs(results_dir_p3, exist_ok=True)

                _robustness_summary = []

                for _cat_name in _cats_to_run:
                    if _cat_name not in exported_within_dict:
                        continue
                    _Xtr, _Xte, _ytr, _yte = exported_within_dict[_cat_name]
                    _X_full = pd.concat([_Xtr, _Xte], axis=0, ignore_index=True) if isinstance(_Xtr, pd.DataFrame) else pd.DataFrame(np.vstack([_Xtr, _Xte]))
                    _y_full = pd.concat([pd.Series(np.asarray(_ytr).ravel()), pd.Series(np.asarray(_yte).ravel())], axis=0, ignore_index=True) if isinstance(_ytr, (pd.Series, pd.DataFrame, np.ndarray)) else pd.Series(np.concatenate([np.asarray(_ytr).ravel(), np.asarray(_yte).ravel()]))

                    print(f"\n  [{_cat_name}] ({_Xtr.shape[1]} features)")

                    _reporter = GroupedCVCheck(_X_full.copy(), _y_full.copy(), task_type='auto', n_outer=_VAL_N_OUTER)

                    try:
                        _reporter.run_standard_cv(
                            model_factory=lambda: create_validation_model_p3(is_classification=is_classification)
                        )
                        _std_score = np.mean(_reporter.results.get('standard', {}).get(_metric_key, [np.nan]))
                    except Exception as _e:
                        print(f"    Warning: Standard CV failed: {_e}")
                        _std_score = np.nan

                    _fam_score = np.nan
                    if has_family and _run_family:
                        try:
                            _fam_ids = sample_valid['rel_family_id'].reset_index(drop=True).reindex(_X_full.index)
                            _fam_ids = _fam_ids.fillna(-1).astype(int).astype(str)  # uniform type for GroupKFold
                            _reporter.run_group_cv(
                                groups=_fam_ids, group_type='family',
                                model_factory=lambda: create_validation_model_p3(is_classification=is_classification)
                            )
                            _fam_score = np.mean(_reporter.results.get('family_grouped', {}).get(_metric_key, [np.nan]))
                        except Exception as _e:
                            print(f"    Warning: Family CV failed: {_e}")

                    _site_score = np.nan
                    if has_site and _run_site:
                        try:
                            _site_ids = sample_valid['L_site_id'].reset_index(drop=True).reindex(_X_full.index)
                            _reporter.run_group_cv(
                                groups=_site_ids, group_type='site',
                                model_factory=lambda: create_validation_model_p3(is_classification=is_classification)
                            )
                            _site_score = np.mean(_reporter.results.get('site_grouped', {}).get(_metric_key, [np.nan]))
                        except Exception as _e:
                            print(f"    Warning: Site CV failed: {_e}")

                    _row = {
                        'Category': _cat_name,
                        f'Standard {_metric_name}': _std_score,
                    }
                    if has_family and _run_family:
                        _row[f'Family {_metric_name}'] = _fam_score
                        _row['Δ Family'] = _fam_score - _std_score if not np.isnan(_fam_score) else np.nan
                    if has_site and _run_site:
                        _row[f'Site {_metric_name}'] = _site_score
                        _row['Δ Site'] = _site_score - _std_score if not np.isnan(_site_score) else np.nan
                    _robustness_summary.append(_row)

                    print(f"    Standard: {_std_score:.4f}  |  "
                          f"Family: {_fam_score:.4f}  |  Site: {_site_score:.4f}")

                    # Free reporter between categories
                    try:
                        del _reporter
                    except NameError:
                        pass
                    _pipeline_cleanup(label=f'P3 {_cat_name}', verbose=False)

                if _robustness_summary:
                    print("\n" + "=" * 80)
                    print("[POINT 3] Site/Family Robustness Summary")
                    print("=" * 80)

                    _rob_df = pd.DataFrame(_robustness_summary)
                    _num_cols = [c for c in _rob_df.columns if c != 'Category']
                    _delta_cols = [c for c in _num_cols if c.startswith('Δ')]

                    _rob_styler = (_rob_df.style
                        .format({c: '{:.4f}' for c in _num_cols})
                        .background_gradient(cmap='RdYlGn', subset=[c for c in _num_cols if not c.startswith('Δ')], vmin=0)
                        .set_caption(f'[POINT 3] Robustness -- {target_options} (T{tp_option})')
                        .set_table_styles([
                            {'selector': 'caption',
                             'props': [('font-size', '14px'), ('font-weight', '700'),
                                       ('color', '#455A64'), ('padding-bottom', '8px')]},
                            {'selector': 'th',
                             'props': [('background-color', '#ECEFF1'), ('color', '#455A64'),
                                       ('font-size', '11px'), ('font-weight', '600'), ('padding', '6px 10px')]},
                            {'selector': 'td',
                             'props': [('padding', '5px 10px'), ('font-size', '12px')]},
                        ])
                        .hide(axis='index')
                    )
                    if _delta_cols:
                        _rob_styler = _rob_styler.background_gradient(
                            cmap='RdYlBu', subset=_delta_cols, vmin=-0.05, vmax=0.05)
                    display(_rob_styler)
                    _rob_df.to_csv(f"{results_dir_p3}/robustness_within_categories.csv", index=False)
                    print(f"\nSaved: {results_dir_p3}/robustness_within_categories.csv")

            elif 'X_train' in globals() and 'y_train' in globals() and _has_demo:
# ---
                # ACROSS-CATEGORIES MODE: Original single-pool robustness
# ---

                print("\n Running [POINT 3] with pipeline variables")

                X_full = pd.concat([X_train, X_test], axis=0, ignore_index=True) if 'X_test' in globals() else X_train.reset_index(drop=True)
                y_full = pd.concat([y_train, y_test], axis=0, ignore_index=True) if 'y_test' in globals() else y_train.reset_index(drop=True)

                robust_reporter = GroupedCVCheck(X_full.copy(), y_full.copy(), task_type='auto', n_outer=_VAL_N_OUTER)

                has_family = 'rel_family_id' in sample_valid.columns
                has_site = 'L_site_id' in sample_valid.columns
                task_type = detect_task_type(y_train)
                is_classification = (task_type == 'classification')
                print(f"   Available confounds: family={has_family}, site={has_site}")
                print(f"   Task type: {task_type}")

                robust_reporter.run_standard_cv(
                    model_factory=lambda: create_validation_model_p3(is_classification=is_classification))

                if has_family and _run_family:
                    try:
                        family_ids = sample_valid['rel_family_id'].reset_index(drop=True).iloc[:len(X_full)]
                        family_ids = family_ids.fillna(-1).astype(int).astype(str)  # uniform type for GroupKFold
                        assert len(family_ids) == len(X_full), f'family_ids len {len(family_ids)} != X_full len {len(X_full)}'
                        robust_reporter.run_group_cv(
                            groups=family_ids, group_type='family',
                            model_factory=lambda: create_validation_model_p3(is_classification=is_classification))
                    except Exception as e:
                        print(f"   Warning:  Error in family CV: {e}")

                if has_site and _run_site:
                    try:
                        site_ids = sample_valid['L_site_id'].reset_index(drop=True)
                        site_ids = site_ids.fillna('_unknown_site')  # NaN-safe for GroupKFold
                        assert len(site_ids) == len(X_full)
                        robust_reporter.run_group_cv(
                            groups=site_ids, group_type='site',
                            model_factory=lambda: create_validation_model_p3(is_classification=is_classification))
                    except Exception as e:
                        print(f"   Warning:  Error in site CV: {e}")

                robust_reporter.generate_report(title="[POINT 3] Robustness: Family/Site-Aware CV - StackingEnsemble")

                # Visualization
                if robust_reporter.results:
                    if is_classification:
                        metrics_to_plot = ['auc', 'accuracy', 'sensitivity']
                        metric_labels = ['AUC', 'Accuracy', 'Sensitivity']
                    else:
                        metrics_to_plot = ['r2', 'rmse', 'mae']
                        metric_labels = ['R²', 'RMSE', 'MAE']

                    strategy_labels = {'standard': 'Standard CV', 'family_grouped': 'Family-Grouped CV', 'site_grouped': 'Site-Grouped CV'}
                    strategy_colors = {'standard': '#2196F3', 'family_grouped': '#FF9800', 'site_grouped': '#4CAF50'}
                    cv_strategies = [s for s in ['standard', 'family_grouped', 'site_grouped'] if s in robust_reporter.results and robust_reporter.results[s]]

                    if len(cv_strategies) >= 2:
                        fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6 * len(metrics_to_plot), 6))
                        if len(metrics_to_plot) == 1: axes = [axes]
                        for ax_idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
                            x_pos = np.arange(len(cv_strategies))
                            means = [np.mean(robust_reporter.results[s].get(metric, [0])) for s in cv_strategies]
                            stds = [np.std(robust_reporter.results[s].get(metric, [0]), ddof=1) for s in cv_strategies]
                            colors_list = [strategy_colors[s] for s in cv_strategies]
                            bars = axes[ax_idx].bar(x_pos, means, 0.6, yerr=stds, capsize=5,
                                                   color=colors_list, edgecolor='black', linewidth=0.5, alpha=0.85)
                            for bar, m, sd in zip(bars, means, stds):
                                axes[ax_idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + sd + 0.005,
                                                  f'{m:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
                            axes[ax_idx].set_xticks(x_pos)
                            axes[ax_idx].set_xticklabels([strategy_labels[s] for s in cv_strategies], rotation=15, ha='right')
                            axes[ax_idx].set_ylabel(label, fontsize=12)
                            axes[ax_idx].set_title(f'{label} by CV Strategy', fontsize=12, fontweight='bold')
                            axes[ax_idx].spines['top'].set_visible(False)
                            axes[ax_idx].spines['right'].set_visible(False)

                        plt.suptitle('[POINT 3] Standard vs Group-Aware CV', fontsize=14, fontweight='bold')
                        plt.tight_layout()
                        results_dir_p3 = os.path.join(os.getcwd(), 'robustness_results')
                        os.makedirs(results_dir_p3, exist_ok=True)
                        plt.savefig(f"{results_dir_p3}/robustness_comparison.png", dpi=300, bbox_inches='tight')
                        plt.show()

            else:
                if not _has_demo:
                    print("\nError: [POINT 3] ERROR: sample_valid not found")
                else:
                    print("\nError: Required variables not found")

        except Exception as e:
            print(f"\nError: Error in [POINT 3]: {e}")
            import traceback
            traceback.print_exc()

        print("\n[POINT 3] Complete!")

        # -- [CACHE + REPORT SAVE] ---
        _RS['section'] = 'uncategorized'
        try:
            _pipeline_cleanup(label='P3 complete')

            cache_save(
                {'reporter': locals().get('reporter') or locals().get('robustness_reporter')},
                cell_tag=_CTAG_14, key=_CKEY_14,
                label=f"{target_options}_tp{tp_option}_[POINT 3] Site Robustness",
            )
            report_save_state(make_run_key())
        except Exception as _ce_14:
            print(f"[Cache] Could not save robustness: {_ce_14}")

# -- Batch dispatch ---
_is_bv_17 = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_17:
    try:
        _cbv_17 = cache_load('main_runner')
        if (_cbv_17 and isinstance(_cbv_17, dict)
                and _cbv_17.get('_batch_archive')
                and len(_cbv_17['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_17['_batch_archive'].values())):
            _batch_archive = _cbv_17['_batch_archive']
            _is_bv_17 = True
    except Exception:
        pass

if _is_bv_17:
    _vgs_17 = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split',
        'sample_valid','sample_valid_test','sample_valid_train',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Site Robustness: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_17, (_vk_17, _vd_17) in enumerate(_batch_archive.items()):
        _vl_17 = _vd_17.get('target', _vk_17)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_17+1}/{len(_batch_archive)}] Target: {_vl_17}")
        print(f"{'-' * 68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split',
                    'sample_valid','sample_valid_test','sample_valid_train']:
            if _vd_17.get(_gk) is not None:
                globals()[_gk] = _vd_17[_gk]
        globals()['target_options'] = _vl_17
        if 'timepoint' in _vd_17:
            globals()['tp_option'] = _vd_17['timepoint']
        _run_val_17()
        # -- Archive validation reporters back into _batch_archive --
        if _vk_17 in _batch_archive:
            _batch_archive[_vk_17]['robust_reporter'] = globals().get('robust_reporter')
            _batch_archive[_vk_17]['_site_score'] = globals().get('_site_score')

    for _k, _v in _vgs_17.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Site Robustness: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_val_17()

In [ ]:
#@title V | Permutation Testing & Null Distributions

# Empirical null distribution via label permutation with Phipson & Smyth
# correction (p = [r+1]/[n+1], preventing p = 0 from finite permutations).
# Refits the stacking ensemble on shuffled labels for each permutation
# and computes upper-tailed p-value against observed R2/AUC.
#@markdown ---
n_permutations = 1000 #@param {type: "integer"}
n_outer_folds = 5 #@param {type: "integer"}

"""
Permutation-Based Model Significance Testing
---------------------------------------------
Empirical null distribution via label permutation. Refits the stacking
ensemble on shuffled outcome labels to estimate chance-level performance,
then computes an upper-tailed p-value against the observed metric.

Procedure
---------
  1. Record observed performance metric (AUC or R²).
  2. Permute outcome labels n_permutations times and refit the model.
  3. Compute p = P(null ≥ observed).
  4. Plot null histogram with observed value marked.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
from scipy import stats
from sklearn.metrics import roc_auc_score, r2_score, mean_squared_error
from sklearn.model_selection import StratifiedKFold, KFold
from IPython.display import display, HTML


# ---
# STYLE CONSTANTS --
# ---
_P4_BLUE    = '#1565C0'
_P4_RED     = '#C62828'
_P4_GREEN   = '#2E7D32'
_P4_GREY    = '#455A64'
_P4_LTGREY  = '#ECEFF1'
_P4_SLATE   = '#1F2D3D'
_P4_ORANGE  = '#E65100'
_P4_TEAL    = '#00838F'

def _section_hdr(title, subtitle='', color=_P4_BLUE):
    sub_html = (f'<br><span style="font-size:12px;color:#B0BEC5;">{subtitle}</span>'
                if subtitle else '')
    display(HTML(
        f'<div style="font-family:\'Helvetica Neue\',Arial,sans-serif;max-width:800px;'
        f'border-left:4px solid {color};background:linear-gradient(135deg,{color}11,{color}06);'
        f'padding:14px 20px;margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
        f'<span style="font-size:15px;font-weight:700;color:{color};">{title}</span>'
        f'{sub_html}</div>'
    ))

def _kpi_panel(metrics_dict, title='', color=_P4_BLUE):
    html = (
        f'<div style="font-family:sans-serif;max-width:750px;border:2px solid {color};'
        f'border-radius:8px;overflow:hidden;margin:12px 0;">'
        f'<div style="background:{color};color:white;padding:12px 18px;'
        f'font-size:14px;font-weight:700;">{title}</div>'
        f'<div style="padding:16px 20px;display:flex;gap:24px;flex-wrap:wrap;">'
    )
    for label, value, sub in metrics_dict:
        html += (
            f'<div style="flex:1 1 140px;">'
            f'<span style="color:#78909C;font-size:10px;text-transform:uppercase;'
            f'letter-spacing:.4px;">{label}</span><br>'
            f'<span style="font-size:20px;font-weight:700;color:{_P4_SLATE};">{value}</span>'
        )
        if sub:
            html += f'<br><span style="font-size:11px;color:#78909C;">{sub}</span>'
        html += '</div>'
    html += '</div></div>'
    display(HTML(html))

def _fmt_table(df, caption='', gradient_col=None, highlight_col=None):
    """Display a DataFrame with consistent notebook styling."""
    _num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    styler = (df.style
        .format({c: '{:.4f}' for c in _num_cols})
        .set_caption(caption)
        .set_table_styles([
            {'selector': 'caption',
             'props': [('font-size', '14px'), ('font-weight', '700'),
                       ('color', _P4_GREY), ('padding-bottom', '8px')]},
            {'selector': 'th',
             'props': [('background-color', _P4_LTGREY), ('color', _P4_GREY),
                       ('font-size', '11px'), ('font-weight', '600'),
                       ('text-transform', 'uppercase'), ('letter-spacing', '.4px'),
                       ('padding', '6px 10px')]},
            {'selector': 'td',
             'props': [('padding', '5px 10px'), ('font-size', '12px')]},
            {'selector': 'tr:hover td',
             'props': [('background-color', '#F5F5F5')]},
        ])
        .hide(axis='index')
    )
    if gradient_col and gradient_col in df.columns and df[gradient_col].nunique(dropna=True) > 1:
        styler = styler.background_gradient(cmap='RdYlGn', subset=[gradient_col], vmin=0)
    if highlight_col and highlight_col in df.columns:
        styler = styler.map(
            lambda v: 'color: #2E7D32; font-weight: bold' if v == 'Yes' else 'color: #C62828; font-weight: bold',
            subset=[highlight_col]
        )
    display(styler)

def get_scoring_metric(task_type):
    """Get appropriate scoring metric for task type."""
    if task_type == 'classification':
        return 'roc_auc'
    else:
        return 'r2'

# ---
# PermutationTester Class
# ---

class PermutationTester:
    """
    Test model significance via label permutation.

    MOTIVATION (concern about "predictive pleiotropy"):
    - Many psychological variables covary (p-factor, shared rater effects)
    - Strong predictive performance might reflect covariance, not true prediction
    - Solution: Show observed performance >> null distribution

    METHODOLOGY:
    1. Observe: Train model on true y, measure performance
    2. Permute: Randomly shuffle y labels
    3. Null: Train model on permuted y, measure performance
    4. Repeat: Generate null distribution from 100+ permutations
    5. P-value: Proportion of null models ≥ observed (2-tailed)
    """

    def __init__(self, X, y, task_type='auto', n_permutations=100, cv_folds=5, random_state=42):
        """        X: features, y: outcomes. task_type inferred if 'auto'.
        n_permutations : int
            Number of permutations for null distribution (default 100, use 1000 for final)
        cv_folds : int
            Number of CV folds
        random_state : int
            Random seed
        """
        self.X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        # Ensure y is always 1-dimensional (batch archive may produce (N,1) arrays)
        if isinstance(y, pd.Series):
            self.y = y
        elif isinstance(y, pd.DataFrame):
            self.y = y.iloc[:, 0]
        else:
            _y = np.asarray(y)
            if _y.ndim > 1:
                _y = _y.ravel()
            self.y = pd.Series(_y)
        self.n_permutations = n_permutations
        self.cv_folds = cv_folds
        self.random_state = random_state

        if task_type == 'auto':
            self.task_type = detect_task_type(self.y.values)
        else:
            self.task_type = task_type

        self.is_classification = self.task_type == 'classification'
        self.is_regression = self.task_type == 'regression'
        self.scoring_metric = get_scoring_metric(self.task_type)

        self.observed_score = None
        self.null_scores = []
        self.p_value = None

        print(f"Task type: {self.task_type.upper()}")
        print(f"   Scoring metric: {self.scoring_metric.upper()}")
        print(f"   Permutations: {n_permutations}")

    def run_permutation_test(self, model_factory=None, base_model=None, results_dir='permutation_results'):
        """
        Run permutation significance test.

        Parameters:
        model_factory : callable, optional
            Function returning fresh model (for retraining each fold)
        base_model : estimator, optional
            Pre-trained model from pipeline
        results_dir : str
            Directory to save results
        """
        os.makedirs(results_dir, exist_ok=True)

        if model_factory is None and base_model is None:
            raise ValueError("Either model_factory or base_model must be provided")

        print(f"\n Running permutation test ({self.n_permutations} permutations)")

        # Step 1: Compute observed score on true labels
        print(f"\n   Step 1: Computing observed score (true labels)...")
        self.observed_score = self._compute_cv_score(self.y, model_factory)
        print(f"           Observed {self.scoring_metric.upper()}: {self.observed_score:.4f}")

        # Step 2: Generate null distribution via label permutation
        print(f"\n   Step 2: Generating null distribution ({self.n_permutations} permutations)...")
        rng = np.random.default_rng(self.random_state)

        for perm_num in range(self.n_permutations):
            # Permute labels
            y_perm = rng.permutation(self.y.values)

            # Compute score on permuted labels
            perm_score = self._compute_cv_score(y_perm, model_factory)
            self.null_scores.append(perm_score)

            if (perm_num + 1) % 20 == 0:
                print(f"           Progress: {perm_num+1}/{self.n_permutations}")

        self.null_scores = np.array(self.null_scores)

        # Step 3: Compute p-value
        self.p_value = self._compute_p_value()

        # Step 4: Save results
        self._save_results(results_dir)

        return {
            'observed_score': self.observed_score,
            'null_scores': self.null_scores,
            'p_value': self.p_value
        }

    def _compute_cv_score(self, y, model_factory):
        """Compute CV score (mean across folds)."""
        fold_scores = []

        if self.is_classification:
            cv = StratifiedKFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)
        else:
            cv = KFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)

        for train_idx, test_idx in cv.split(self.X, y):
            X_train = self.X.iloc[train_idx]
            X_test = self.X.iloc[test_idx]
            y_train = y[train_idx] if isinstance(y, np.ndarray) else y.iloc[train_idx]
            y_test = y[test_idx] if isinstance(y, np.ndarray) else y.iloc[test_idx]

            # y is pandas-compatible (stacking ensemble uses .iloc internally)
            if isinstance(y_train, np.ndarray):
                y_train = pd.Series(y_train, index=X_train.index)
            if isinstance(y_test, np.ndarray):
                y_test = pd.Series(y_test, index=X_test.index)

            # Always retrain model (required for valid permutation null)
            model = model_factory()
            model.fit(X_train, y_train)

            # Score
            if self.is_classification:
                y_proba = model.predict_proba(X_test)
                classes = np.unique(y_test)
                if len(classes) == 2:
                    score = roc_auc_score(y_test, y_proba[:, 1])
                else:
                    score = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
            else:
                y_pred = model.predict(X_test)
                score = r2_score(y_test, y_pred)

            fold_scores.append(score)

        return np.mean(fold_scores)

    def _compute_p_value(self):
        """
        Compute upper-tailed p-value (Phipson & Smyth 2010).

        P-value = (# null scores ≥ observed + 1) / (# permutations + 1)
        The "+1" prevents p=0 and accounts for the observed score as one permutation
        (Phipson & Smyth, 2010).
        """
        num_extreme = np.sum(self.null_scores >= self.observed_score)
        p_value = (num_extreme + 1) / (len(self.null_scores) + 1)

        return p_value

    def _save_results(self, results_dir):
        """Save permutation results to CSV."""
        results_df = pd.DataFrame({
            'permutation': range(len(self.null_scores)),
            'score': self.null_scores
        })

        results_df.to_csv(f"{results_dir}/permutation_null_distribution.csv", index=False)

        summary_df = pd.DataFrame({
            'Metric': ['Observed Score', f'Null Mean', 'Null SD', 'P-value', 'Significant (α=0.05)'],
            'Value': [
                f"{self.observed_score:.4f}",
                f"{np.mean(self.null_scores):.4f}",
                f"{np.std(self.null_scores):.4f}",
                f"{self.p_value:.4f}",
                "Yes (significant)" if self.p_value < 0.05 else "No (n.s.)"
            ]
        })

        summary_df.to_csv(f"{results_dir}/permutation_test_summary.csv", index=False)

        print(f"\nSaved results to {results_dir}/")

    def plot_null_distribution(self, figsize=(12, 6), save_path=None):
        """
        Publication-quality null distribution plot with KDE overlay,
        rejection zone shading, and effect-size annotation.
        """
        if self.observed_score is None:
            raise ValueError("Run run_permutation_test() first.")

        _metric_upper = self.scoring_metric.upper()

        fig, ax = plt.subplots(figsize=figsize, facecolor='white')
        ax.set_facecolor('#FAFBFC')

        # -- Histogram of null distribution ---
        _counts, _bins, _patches = ax.hist(
            self.null_scores, bins=min(50, max(20, len(self.null_scores) // 15)),
            alpha=0.65, color=_P4_BLUE, edgecolor='white', linewidth=0.6,
            label=f'Null distribution (n={len(self.null_scores)})',
            zorder=2,
        )

        # -- KDE overlay for distributional shape ---
        try:
            sns.kdeplot(self.null_scores, ax=ax, color=_P4_GREY, linewidth=1.8,
                        alpha=0.7, zorder=3)
        except Exception:
            pass  # graceful fallback if too few unique values

        # -- Rejection zone: shade area ≥ observed ---
        _xlim = ax.get_xlim()
        ax.axvspan(self.observed_score, _xlim[1], alpha=0.07, color=_P4_RED,
                   zorder=1, label=f'Rejection zone (p={self.p_value:.4f})')
        ax.set_xlim(_xlim)  # restore in case axvspan shifted it

        # -- Null mean line ---
        null_mean = np.mean(self.null_scores)
        null_sd = np.std(self.null_scores)
        ax.axvline(null_mean, color=_P4_GREEN, linestyle=':', linewidth=2.0,
                   alpha=0.8, label=f'Null mean = {null_mean:.4f}', zorder=4)

        # -- Observed score line ---
        ax.axvline(self.observed_score, color=_P4_RED, linestyle='--',
                   linewidth=2.8, alpha=0.9,
                   label=f'Observed {_metric_upper} = {self.observed_score:.4f}',
                   zorder=5)

        # -- Effect-size annotation box ---
        _d = (self.observed_score - null_mean) / (null_sd + 1e-10)
        _sig_label = 'Significant' if self.p_value < 0.05 else 'Not significant'
        _sig_color = _P4_GREEN if self.p_value < 0.05 else _P4_ORANGE

        _annot_text = (
            f'Observed: {self.observed_score:.4f}\n'
            f'Null mean: {null_mean:.4f} ± {null_sd:.4f}\n'
            f'Effect: {_d:.2f} SD above null\n'
            f'p = {self.p_value:.4f}  ({_sig_label})'
        )
        _bbox_props = dict(
            boxstyle='round,pad=0.6', facecolor='white',
            edgecolor=_sig_color, linewidth=1.5, alpha=0.95,
        )
        ax.annotate(
            _annot_text,
            xy=(self.observed_score, ax.get_ylim()[1] * 0.85),
            xytext=(0.97, 0.95), textcoords='axes fraction',
            fontsize=10, fontweight='500', color=_P4_SLATE,
            ha='right', va='top',
            bbox=_bbox_props,
            arrowprops=dict(arrowstyle='->', color=_sig_color, lw=1.5),
            zorder=6,
        )

        # -- Axis labels and title ---
        _target_lbl = globals().get('target_options', 'Target')
        _tp_lbl = globals().get('tp_option', '')
        _tp_str = f' (T{_tp_lbl})' if _tp_lbl else ''

        ax.set_xlabel(f'{_metric_upper} Score', fontsize=12, fontweight='600',
                      color=_P4_GREY)
        ax.set_ylabel('Frequency', fontsize=12, fontweight='600', color=_P4_GREY)
        ax.set_title(
            f'Permutation Significance Test -- {_target_lbl}{_tp_str}\n'
            f'{len(self.null_scores)} permutations  ·  '
            f'p = {self.p_value:.4f}  ·  {_sig_label}',
            fontsize=14, fontweight='bold', color=_P4_SLATE,
            pad=16, linespacing=1.5,
        )

        # -- Legend ---
        ax.legend(
            fontsize=10, loc='upper left',
            frameon=True, framealpha=0.95, edgecolor='#CFD8DC',
            fancybox=True,
        )

        # -- Grid and spine cleanup ---
        ax.grid(True, alpha=0.2, axis='y', linestyle=':', color='#B0BEC5')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#CFD8DC')
        ax.spines['bottom'].set_color('#CFD8DC')
        ax.tick_params(labelsize=10, colors=_P4_GREY)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            print(f"   Saved: {save_path}")
        plt.show()

    def generate_report(self, title="Permutation Test: Model Significance"):
        """Generate styled HTML report for permutation test results."""
        if self.observed_score is None:
            raise ValueError("Run run_permutation_test() first.")

        _metric_upper = self.scoring_metric.upper()
        _target_lbl = globals().get('target_options', 'Target')
        _tp_lbl = globals().get('tp_option', '')

        # -- Section header ---
        _section_hdr(
            title,
            subtitle=f'{_target_lbl}  ·  T{_tp_lbl}  ·  {self.n_permutations} permutations',
            color=_P4_BLUE,
        )

        # -- KPI panel ---
        null_mean = np.mean(self.null_scores)
        null_sd = np.std(self.null_scores)
        _d = (self.observed_score - null_mean) / (null_sd + 1e-10)
        _sig = self.p_value < 0.05
        _sig_label = 'Significant' if _sig else 'Error: Not significant'

        _kpi_panel([
            (f'Observed {_metric_upper}', f'{self.observed_score:.4f}', None),
            (f'Null Mean {_metric_upper}', f'{null_mean:.4f}', f'± {null_sd:.4f}'),
            ('p-value', f'{self.p_value:.4f}', _sig_label),
            ('Effect Size', f'{_d:.2f} SD', 'above null mean'),
        ], title=title, color=_P4_GREEN if _sig else _P4_ORANGE)

        # -- Results table ---
        results_table = pd.DataFrame({
            'Statistic': [
                'Observed Score',
                'Null Mean',
                'Null SD',
                'Null Min',
                'Null Max',
                'P-value (upper-tailed)',
                'Significant (α=0.05)',
            ],
            'Value': [
                self.observed_score,
                null_mean,
                null_sd,
                float(np.min(self.null_scores)),
                float(np.max(self.null_scores)),
                self.p_value,
                np.nan,  # handled by highlight
            ],
            'Interpretation': [
                f'Best {_metric_upper} from true labels',
                f'Expected {_metric_upper} under H₀',
                'Spread of null distribution',
                'Worst null permutation',
                'Best null permutation',
                'Phipson & Smyth (2010)',
                'YES' if _sig else 'Error: NO',
            ],
        })
        # Format numeric values nicely but leave Interpretation as-is
        _num_cols_rt = ['Value']
        _rt_styler = (results_table.style
            .format({c: '{:.4f}' for c in _num_cols_rt}, na_rep='--')
            .set_caption(f'Permutation Test Summary -- {_target_lbl} (T{_tp_lbl})')
            .set_table_styles([
                {'selector': 'caption',
                 'props': [('font-size', '14px'), ('font-weight', '700'),
                           ('color', _P4_GREY), ('padding-bottom', '8px')]},
                {'selector': 'th',
                 'props': [('background-color', _P4_LTGREY), ('color', _P4_GREY),
                           ('font-size', '11px'), ('font-weight', '600'),
                           ('text-transform', 'uppercase'), ('letter-spacing', '.4px'),
                           ('padding', '6px 10px')]},
                {'selector': 'td',
                 'props': [('padding', '5px 10px'), ('font-size', '12px')]},
                {'selector': 'tr:hover td',
                 'props': [('background-color', '#F5F5F5')]},
            ])
            .hide(axis='index')
        )
        display(_rt_styler)

        # -- Interpretation panel ---
        if _sig:
            _interp_html = (
                f'<div style="font-family:sans-serif;max-width:750px;border-left:4px solid {_P4_GREEN};'
                f'background:#E8F5E9;padding:14px 20px;margin:12px 0;border-radius:0 6px 6px 0;">'
                f'<span style="font-size:13px;font-weight:600;color:{_P4_GREEN};">Model is statistically significant</span><br>'
                f'<span style="font-size:12px;color:#37474F;line-height:1.7;">'
                f'The observed {_metric_upper} of {self.observed_score:.4f} is {_d:.2f} standard deviations '
                f'above the null mean ({null_mean:.4f}). Only {int(np.sum(self.null_scores >= self.observed_score))} '
                f'of {len(self.null_scores)} permutations matched or exceeded this value (p = {self.p_value:.4f}). '
                f'This suggests the model captures genuine predictive signal beyond shared covariance artifacts.'
                f'</span></div>'
            )
        else:
            _interp_html = (
                f'<div style="font-family:sans-serif;max-width:750px;border-left:4px solid {_P4_ORANGE};'
                f'background:#FFF3E0;padding:14px 20px;margin:12px 0;border-radius:0 6px 6px 0;">'
                f'<span style="font-size:13px;font-weight:600;color:{_P4_ORANGE};">Error: Model is not statistically significant</span><br>'
                f'<span style="font-size:12px;color:#37474F;line-height:1.7;">'
                f'The observed {_metric_upper} of {self.observed_score:.4f} could plausibly arise '
                f'from the null distribution (p = {self.p_value:.4f}). '
                f'Consider: insufficient signal, predictive pleiotropy, or limited sample size.'
                f'</span></div>'
            )
        display(HTML(_interp_html))

        # -- Null distribution figure ---
        self.plot_null_distribution()

    def plot_within_categories_forest(self, perm_results_list, metric_name='R²',
                                      target_lbl='', tp_lbl='', save_path=None):
        """
        Forest plot of observed scores vs null means across categories.
        Called after within-categories permutation testing.

        Parameters:
        perm_results_list : list of dict
            Each dict has 'Category', 'Observed {metric}', 'Null Mean {metric}', 'p-value'
        """
        if not perm_results_list:
            return

        n_cats = len(perm_results_list)
        obs_key = f'Observed {metric_name}'
        null_key = f'Null Mean {metric_name}'

        cats = [r['Category'] for r in perm_results_list]
        obs_scores = [r.get(obs_key, np.nan) for r in perm_results_list]
        null_means = [r.get(null_key, np.nan) for r in perm_results_list]
        pvals = [r.get('p-value', np.nan) for r in perm_results_list]

        # Sort by observed score descending
        _order = np.argsort(obs_scores)[::-1]
        cats = [cats[i] for i in _order]
        obs_scores = [obs_scores[i] for i in _order]
        null_means = [null_means[i] for i in _order]
        pvals = [pvals[i] for i in _order]

        fig_height = max(4, n_cats * 0.7 + 2)
        fig, ax = plt.subplots(figsize=(10, fig_height), facecolor='white')
        ax.set_facecolor('#FAFBFC')

        y_pos = np.arange(n_cats)

        # -- Null distribution violins (background context) ---
        null_dists = [r.get('null_scores', []) for r in perm_results_list]
        null_dists = [null_dists[i] for i in _order]
        _has_dists = any(len(d) > 2 for d in null_dists)
        if _has_dists:
            for i in range(n_cats):
                if len(null_dists[i]) > 2:
                    from scipy.stats import gaussian_kde
                    try:
                        _nd = np.array(null_dists[i])
                        _kde = gaussian_kde(_nd, bw_method=0.3)
                        _xgrid = np.linspace(_nd.min() - 0.01, _nd.max() + 0.01, 80)
                        _density = _kde(_xgrid)
                        _density = _density / _density.max() * 0.35  # scale to row height
                        ax.fill_between(_xgrid, y_pos[i] - _density, y_pos[i] + _density,
                                        alpha=0.15, color='#90A4AE', zorder=1)
                    except Exception:
                        pass  # skip kde if it fails

        # -- Null mean markers (grey diamonds) ---
        ax.scatter(null_means, y_pos, color=_P4_GREY, marker='D', s=60,
                   zorder=3, label='Null mean', alpha=0.7)

        # -- Observed score markers (colored circles) ---
        _colors = [_P4_GREEN if p < 0.05 else _P4_ORANGE for p in pvals]
        ax.scatter(obs_scores, y_pos, color=_colors, marker='o', s=100,
                   zorder=4, edgecolors='white', linewidths=1.2)

        # -- Connecting lines (null → observed) ---
        for i in range(n_cats):
            ax.plot([null_means[i], obs_scores[i]], [y_pos[i], y_pos[i]],
                    color=_colors[i], linewidth=1.5, alpha=0.5, zorder=2)

        # -- p-value annotations ---
        for i in range(n_cats):
            _x_annot = max(obs_scores[i], null_means[i]) + 0.005
            _p_str = f'p={pvals[i]:.4f}' if not np.isnan(pvals[i]) else 'p=N/A'
            _fw = 'bold' if pvals[i] < 0.05 else 'normal'
            ax.text(_x_annot, y_pos[i], _p_str,
                    va='center', ha='left', fontsize=9.5,
                    fontweight=_fw, color=_colors[i])

        # -- Labels and formatting ---
        # Abbreviate long category names
        _cat_labels = [c[:35] + '…' if len(c) > 35 else c for c in cats]
        ax.set_yticks(y_pos)
        ax.set_yticklabels(_cat_labels, fontsize=10, color=_P4_SLATE)
        ax.set_xlabel(f'{metric_name} Score', fontsize=12, fontweight='600',
                      color=_P4_GREY)
        ax.set_title(
            f'Permutation Test -- Forest Plot\n'
            f'{target_lbl}  ·  T{tp_lbl}  ·  {n_cats} categories',
            fontsize=14, fontweight='bold', color=_P4_SLATE,
            pad=16, linespacing=1.5,
        )

        ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#CFD8DC')
        ax.spines['bottom'].set_color('#CFD8DC')
        ax.grid(True, axis='x', alpha=0.2, linestyle=':', color='#B0BEC5')
        ax.tick_params(labelsize=10, colors=_P4_GREY)

        # -- Legend ---
        _leg_handles = [
            plt.Line2D([0], [0], marker='o', color='none', markerfacecolor=_P4_GREEN,
                       markeredgecolor='white', markersize=10, label='Observed (p<0.05)'),
            plt.Line2D([0], [0], marker='o', color='none', markerfacecolor=_P4_ORANGE,
                       markeredgecolor='white', markersize=10, label='Observed (p≥0.05)'),
            plt.Line2D([0], [0], marker='D', color='none', markerfacecolor=_P4_GREY,
                       markersize=8, label='Null mean'),
            plt.Patch(facecolor='#90A4AE', alpha=0.25, label='Null distribution'),
        ]
        ax.legend(handles=_leg_handles, fontsize=10, loc='lower right',
                  frameon=True, framealpha=0.95, edgecolor='#CFD8DC')

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            print(f"   Saved: {save_path}")
        plt.show()

# ---
# HELPER: Create Validation Model (independent of Point 1)
# ---

def create_validation_model_p4(is_classification=True):
    """
    Single CatBoost model with fixed hyperparameters from main pipeline.
    Used instead of the full stacking ensemble for permutation testing
    to make 1000 permutations computationally feasible on Colab.

    Methodological justification: The null hypothesis under permutation
    (no relationship between features and labels) is model-agnostic.
    The full ensemble can only equal or exceed single-model performance,
    making this null distribution conservative by construction -- i.e.,
    if the single CatBoost null is exceeded by the observed ensemble,
    the full ensemble null would be exceeded by an equal or greater margin.
    iterations=250 (vs 1000 in main pipeline) is sufficient to detect
    any signal present in permuted labels; permuted fits are noisy
    regardless of model capacity.

    See: Ojala & Garriga (2010), JMLR; Phipson & Smyth (2010) for
    p-value computation.
    """
    from catboost import CatBoostClassifier, CatBoostRegressor
    # Use main pipeline params but reduce iterations for permutation speed.
    # All other hyperparameters (depth, lr, l2) are preserved exactly.
    params = {**_CB_PARAMS, 'iterations': 250, 'verbose': 0}
    if is_classification:
        return CatBoostClassifier(**params)
    else:
        return CatBoostRegressor(**params)

# ---
# CACHE SYSTEM -- permutation

# ---
# BATCH-AWARE WRAPPER -- Permutation Testing & Null Distributions
# ---
# When _batch_archive holds multiple targets with archived state, this
# wrapper iterates over each, restoring globals before each pass.
# In single-target mode the validation runs exactly once (no change).
# ---

def _run_val_18():
    global perm_reporter  # expose for downstream consumption
    global _perm_results, _perm_df  # expose for downstream consumption
    """Run this cell's validation for the CURRENT globals."""
    # ---
    # -- Validation cell gate (reads Cell 6 params) ---
    _skip_perm = not globals().get('run_validation_cells', True)
    _VAL_N_PERMUTATIONS = n_permutations
    _VAL_N_OUTER = n_outer_folds

    # -- Dependency guard ---
    if 'make_run_key' not in globals():
        print('\nWarning:  Cell 0 (Report State) has not been run in this session.')
        print('   Please run Cells 1 → 12 first (in order), then re-run this cell.')
        _RUN_15 = False
    elif _skip_perm:
        print('Skipped:  Cell 15 -- Permutation Testing skipped (run_validation_cells=False)')
        _RUN_15 = False
    else:
        _CTAG_15  = 'permutation'
        _CKEY_15  = make_run_key({'cell': _CTAG_15})
        _CACHED_15 = cache_load(_CTAG_15, _CKEY_15)
        if _CACHED_15:
            print(f"  {_CTAG_15} loaded from cache -- skipping execution.")
            _RUN_15 = False
        else:
            _RS['section'] = 'permutation'
            _done_folds_15 = fold_load_all(_CKEY_15, _CTAG_15)
            _RUN_15 = True  # moved inside else block

    if _RUN_15:

        # EXECUTION: Run permutation test
        # ======================================================================

        try:
            # -- Task-type helpers --
            if 'is_regression' not in globals() or not callable(globals().get('is_regression')):
                globals()['is_regression'] = lambda target: target in (globals().get('continuous_outcome_variables', []))
            if 'is_classification' not in globals() or not callable(globals().get('is_classification')):
                globals()['is_classification'] = lambda target: target in (globals().get('binary_or_categorical_outcome_variables', []))

            # -- Mode detection --
            _is_within_mode = (
                'exported_within_dict' in globals()
                and isinstance(exported_within_dict, dict)
                and len(exported_within_dict) > 0
                and 'split' in globals()
                and split == 'within_categories'
            )

            # -- Top-K categories by performance (from metrics_dict_full) --
            _top_cats_ordered = []
            if _is_within_mode and 'metrics_dict_full' in globals():
                _cat_scores = {}
                for _cat, _mdict in zip(metrics_dict_full.get('category', []),
                                         metrics_dict_full.get('metrics', [])):
                    _m = _mdict.get('ensemble', _mdict.get('meta_model', {}))
                    if _m:
                        _s = _m.get('r2', _m.get('auc', _m.get('accuracy', np.nan)))
                        if not np.isnan(_s):
                            _cat_scores[_cat] = _s
                _top_cats_ordered = sorted(_cat_scores.keys(), key=lambda c: _cat_scores[c], reverse=True)

            if _is_within_mode:
# ---
                # WITHIN-CATEGORIES MODE: Permutation test per top-K categories
# ---

                # Permutation testing is expensive -- limit to top 5 + bottom 3
                _TOP_K = min(5, len(_top_cats_ordered))
                _BOT_K = min(3, max(0, len(_top_cats_ordered) - _TOP_K))
                _cats_to_run = _top_cats_ordered[:_TOP_K]
                if _BOT_K > 0:
                    _cats_to_run += _top_cats_ordered[-_BOT_K:]

                _section_hdr(
                    'Permutation Testing (Within-Categories)',
                    subtitle=f'Testing {len(_cats_to_run)} categories (top {_TOP_K} + bottom {_BOT_K})  ·  '
                             f'{_VAL_N_PERMUTATIONS} permutations per category',
                    color=_P4_BLUE,
                )

                task_type = detect_task_type(list(exported_within_dict.values())[0][2])
                is_classification = (task_type == 'classification')
                _metric_name = 'AUC' if is_classification else 'R²'

                _perm_results = []

                for _cat_name in _cats_to_run:
                    if _cat_name not in exported_within_dict:
                        continue
                    _Xtr, _Xte, _ytr, _yte = exported_within_dict[_cat_name]
                    _X_full = pd.concat([_Xtr, _Xte], axis=0, ignore_index=True) if isinstance(_Xtr, pd.DataFrame) else pd.DataFrame(np.vstack([_Xtr, _Xte]))
                    _y_full = pd.concat([pd.Series(np.asarray(_ytr).ravel()), pd.Series(np.asarray(_yte).ravel())], axis=0, ignore_index=True) if isinstance(_ytr, (pd.Series, pd.DataFrame, np.ndarray)) else pd.Series(np.concatenate([np.asarray(_ytr).ravel(), np.asarray(_yte).ravel()]))

                    print(f"\n  [{_cat_name}] ({_Xtr.shape[1]} features)")

                    try:
                        _perm_reporter = PermutationTester(
                            _X_full.copy(), _y_full.copy(),
                            task_type='auto', n_permutations=_VAL_N_PERMUTATIONS, cv_folds=_VAL_N_OUTER
                        )
                        _result = _perm_reporter.run_permutation_test(
                            model_factory=lambda: create_validation_model_p4(is_classification=is_classification),
                            results_dir='permutation_results'
                        )

                        _obs = _result.get('observed_score', np.nan) if _result else np.nan
                        _p = _result.get('p_value', np.nan) if _result else np.nan
                        _null_mean = np.mean(_result.get('null_scores', [np.nan])) if _result else np.nan

                        _perm_results.append({
                            'Category': _cat_name,
                            f'Observed {_metric_name}': _obs,
                            f'Null Mean {_metric_name}': _null_mean,
                            'p-value': _p,
                            'Significant (p<0.05)': 'Yes' if _p < 0.05 else 'No',
                            'null_scores': list(_result.get('null_scores', [])) if _result else [],
                        })
                        print(f"    Observed: {_obs:.4f}  |  Null mean: {_null_mean:.4f}  |  p = {_p:.4f}")

                    except Exception as _e:
                        print(f"    Warning: Failed: {_e}")
                    finally:
                        try:
                            del _perm_reporter
                        except NameError:
                            pass
                        _pipeline_cleanup(label=f'P4 {_cat_name}', verbose=False)

                if _perm_results:
                    _section_hdr(
                        'Permutation Testing -- Summary',
                        subtitle=f'{target_options} (T{tp_option})  ·  {len(_perm_results)} categories tested',
                        color=_P4_GREEN if all(r['p-value'] < 0.05 for r in _perm_results) else _P4_ORANGE,
                    )

                    _perm_df = pd.DataFrame(_perm_results)
                    _fmt_table(
                        _perm_df,
                        caption=f'Permutation Tests -- {target_options} (T{tp_option})',
                        gradient_col=f'Observed {_metric_name}',
                        highlight_col='Significant (p<0.05)',
                    )

                    _n_sig = sum(1 for r in _perm_results if r['p-value'] < 0.05)

                    # -- Summary KPI panel ---
                    _kpi_panel([
                        ('Categories Tested', str(len(_perm_results)), None),
                        ('Significant (p<0.05)', f'{_n_sig}/{len(_perm_results)}',
                         f'{_n_sig/len(_perm_results)*100:.0f}%'),
                        ('Permutations Each', str(_VAL_N_PERMUTATIONS), None),
                    ], title='Within-Categories Permutation Summary',
                       color=_P4_GREEN if _n_sig == len(_perm_results) else _P4_ORANGE)

                    # -- Forest plot ---
                    try:
                        PermutationTester.__new__(PermutationTester).plot_within_categories_forest(
                            _perm_results, metric_name=_metric_name,
                            target_lbl=target_options, tp_lbl=str(tp_option),
                            save_path='permutation_results/within_categories_forest.png',
                        )
                    except Exception as _fe:
                        print(f"  Warning: Forest plot error: {_fe}")

                    os.makedirs('permutation_results', exist_ok=True)
                    _perm_df.to_csv('permutation_results/within_categories_perm.csv', index=False)

            elif 'X_train' in globals() and 'y_train' in globals():
# ---
                # ACROSS-CATEGORIES MODE: Original single-pool permutation test
# ---

                _section_hdr(
                    'Permutation Testing (Across-Categories)',
                    subtitle=f'{_VAL_N_PERMUTATIONS} permutations  ·  {_VAL_N_OUTER}-fold CV',
                    color=_P4_BLUE,
                )

                task_type = detect_task_type(y_train)
                is_classification = (task_type == 'classification')
                print(f"   Task type: {task_type}")

                X_full = pd.concat([X_train, X_test], axis=0, ignore_index=True) if 'X_test' in globals() else X_train.reset_index(drop=True)
                y_full = pd.concat([y_train, y_test], axis=0, ignore_index=True) if 'y_test' in globals() else y_train.reset_index(drop=True)

                perm_reporter = PermutationTester(
                    X_full.copy(), y_full.copy(),
                    task_type='auto', n_permutations=_VAL_N_PERMUTATIONS, cv_folds=_VAL_N_OUTER
                )

                result = perm_reporter.run_permutation_test(
                    model_factory=lambda: create_validation_model_p4(is_classification=is_classification),
                    results_dir='permutation_results'
                )
                perm_reporter.generate_report(title="Permutation Significance Test -- Stacking Ensemble")

            else:
                print("\nError: X_train or y_train not found")

        except Exception as e:
            print(f"\nError: Error in [POINT 4]: {e}")
            import traceback
            traceback.print_exc()

        # -- Completion banner ---
        display(HTML(
            f'<div style="font-family:sans-serif;max-width:750px;border:2px solid {_P4_GREEN};'
            f'border-radius:8px;overflow:hidden;margin:16px 0;">'
            f'<div style="background:{_P4_GREEN};color:white;padding:10px 18px;'
            f'font-size:13px;font-weight:700;">&#x2705; Permutation Testing -- Complete</div></div>'
        ))

        # -- [CACHE + REPORT SAVE] ---
        _RS['section'] = 'uncategorized'
        try:
            _pipeline_cleanup(label='P4 complete')

            cache_save(
                {'reporter': locals().get('reporter') or locals().get('permutation_reporter')},
                cell_tag=_CTAG_15, key=_CKEY_15,
                label=f"{target_options}_tp{tp_option}_[POINT 4] Permutation Testing",
            )
            report_save_state(make_run_key())
        except Exception as _ce_15:
            print(f"[Cache] Could not save permutation: {_ce_15}")

# -- Batch dispatch ---
_is_bv_18 = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_18:
    try:
        _cbv_18 = cache_load('main_runner')
        if (_cbv_18 and isinstance(_cbv_18, dict)
                and _cbv_18.get('_batch_archive')
                and len(_cbv_18['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_18['_batch_archive'].values())):
            _batch_archive = _cbv_18['_batch_archive']
            _is_bv_18 = True
    except Exception:
        pass

if _is_bv_18:
    _vgs_18 = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Permutation Testing & Null Distributions: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_18, (_vk_18, _vd_18) in enumerate(_batch_archive.items()):
        _vl_18 = _vd_18.get('target', _vk_18)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_18+1}/{len(_batch_archive)}] Target: {_vl_18}")
        print(f"{'-' * 68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split']:
            if _vd_18.get(_gk) is not None:
                globals()[_gk] = _vd_18[_gk]
        globals()['target_options'] = _vl_18
        if 'timepoint' in _vd_18:
            globals()['tp_option'] = _vd_18['timepoint']
        _run_val_18()
        # -- Archive validation reporters back into _batch_archive --
        import copy
        if _vk_18 in _batch_archive:
            _batch_archive[_vk_18]['perm_reporter'] = globals().get('perm_reporter')
            _batch_archive[_vk_18]['_perm_results'] = copy.deepcopy(globals().get('_perm_results'))
            _batch_archive[_vk_18]['_perm_df'] = copy.deepcopy(globals().get('_perm_df'))

    for _k, _v in _vgs_18.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Permutation Testing & Null Distributions: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_val_18()

In [ ]:
#@title V | Stacking Audit (Base-Model Contributions/Meta-Learner Weights/Leakage Checks)

# Stacking methodology audit confirming no data leakage: verifies each
# sample appears in exactly one OOF validation fold, meta-learner receives
# only OOF predictions (never in-fold), and reports base-model contribution
# weights and nested CV structural integrity end-to-end.
from IPython.display import display, HTML
#@markdown ---
n_outer_folds = 5 #@param {type: "integer"}

"""
Stacking Methodology Validation & Meta-Learner Audit
------------------------------------------------------
Confirms no data leakage in the two-stage stacking procedure:
  (1) Base learners generate predictions only on held-out inner folds.
  (2) Meta-learner trains exclusively on those OOF predictions.
End-to-end audit of stacking integrity and preprocessing leakage between
inner and outer CV loops.

Checks performed
----------------
  - Each sample appears in exactly one OOF validation fold.
  - Meta-learner receives OOF predictions, never in-fold predictions.
  - Base model contribution weights are reported.
  - Nested CV structure is confirmed end-to-end.
"""

# -- TabPFN device fallback (if Stacking cell not run) ---
_TABPFN_DEVICE = globals().get('_TABPFN_DEVICE', 'cpu')
# -- Respect _SKIP_TABPFN gate ---
_SKIP_TABPFN_21 = globals().get('_SKIP_TABPFN', False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from sklearn.metrics import roc_auc_score, r2_score, accuracy_score
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression


# ---
# StackingLeakageCheck Class
# ---

# -- Read outer fold count from Cell 6 --
_VAL_N_OUTER = n_outer_folds

class StackingLeakageCheck:
    """
    Audit stacking ensemble for methodological correctness.

    Checks:
    1. Each sample appears in exactly 1 inner validation fold
    2. Meta-learner trained ONLY on OOF predictions (no leakage)
    3. Base models trained on data they DON'T predict for meta-learner
    4. Proper nested CV structure maintained

    HANDLES:
    - Classification: LogisticRegression as meta-learner
    - Regression: LinearRegression as meta-learner
    """

    def __init__(self, X, y, task_type='auto', n_inner=5, random_state=42):
        """        X, y: train split. task_type: classification/regression/auto.
        n_inner : int
            Number of inner CV folds for meta-learner training
        random_state : int
            Random seed
        """
        self.X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        # Ensure y is always 1-dimensional (batch archive may produce (N,1) arrays)
        if isinstance(y, pd.Series):
            self.y = y
        elif isinstance(y, pd.DataFrame):
            self.y = y.iloc[:, 0]
        else:
            _y = np.asarray(y)
            if _y.ndim > 1:
                _y = _y.ravel()
            self.y = pd.Series(_y)
        self.n_inner = n_inner
        self.random_state = random_state

        if task_type == 'auto':
            self.task_type = detect_task_type(self.y.values)
        else:
            self.task_type = task_type

        self.is_classification = self.task_type == 'classification'
        self.is_regression = self.task_type == 'regression'
        if self.is_classification:
            self.meta_learner_class = LogisticRegression
            self.meta_learner_params = {'random_state': random_state, 'max_iter': 1000}
        else:
            self.meta_learner_class = LinearRegression
            self.meta_learner_params = {}

        self.audit_results = {}

        print(f"Task type: {self.task_type.upper()}")
        print(f"   Meta-learner: {self.meta_learner_class.__name__}")

    def validate_stacking_setup(self, base_models, meta_learner=None, n_outer=_VAL_N_OUTER):
        """
        Validate proposed stacking setup.

        Parameters:
        base_models : dict
            {name: model_instance} for base learners
        meta_learner : estimator, optional
            Meta-learner (auto-created if None)
        n_outer : int
            Number of outer CV folds
        """
        print(f"\n Validating stacking setup")
        print(f"   Base models: {list(base_models.keys())}")
        print(f"   Meta-learner: {self.meta_learner_class.__name__}")

        if meta_learner is None:
            meta_learner = self.meta_learner_class(**self.meta_learner_params)

        return self._validate_nested_cv_structure(base_models, meta_learner, n_outer)

    def audit_existing_stacking(self, stacking_model):
        """
        Audit an existing stacking model from notebook.

        Parameters:
        stacking_model : StackingEnsembleModel or similar
            Existing stacking model from pipeline
        """
        print(f"\n Auditing existing stacking model")
        print(f"   Model class: {stacking_model.__class__.__name__}")

        return self._audit_model_structure(stacking_model)

    def _validate_nested_cv_structure(self, base_models, meta_learner, n_outer):
        """Validate nested CV structure for stacking."""
        print(f"\n   Checking nested CV structure ({n_outer} outer folds)...")

        if self.is_classification:
            cv_outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=self.random_state)
            cv_inner = StratifiedKFold(n_splits=self.n_inner, shuffle=True, random_state=self.random_state)
        else:
            cv_outer = KFold(n_splits=n_outer, shuffle=True, random_state=self.random_state)
            cv_inner = KFold(n_splits=self.n_inner, shuffle=True, random_state=self.random_state)

        validation_checks = {
            'oof_sample_coverage': None,
            'no_leakage': True,
            'base_model_count': len(base_models),
            'meta_learner_trained': False,
            'outer_fold_coverage': [],
            'warnings': []
        }

        all_inner_coverage_ok = True

        for outer_fold, (outer_train_idx, outer_test_idx) in enumerate(cv_outer.split(self.X, self.y)):
            X_train_outer = self.X.iloc[outer_train_idx]
            y_train_outer = self.y.iloc[outer_train_idx]

            # Track coverage WITHIN this outer fold only
            inner_fold_count = np.zeros(len(X_train_outer))

            # Generate OOF predictions for meta-learner training
            oof_meta_features = np.zeros((len(X_train_outer), len(base_models)))

            for inner_train_idx, inner_val_idx in cv_inner.split(X_train_outer, y_train_outer):
                X_inner_train = X_train_outer.iloc[inner_train_idx]
                X_inner_val = X_train_outer.iloc[inner_val_idx]
                y_inner_train = y_train_outer.iloc[inner_train_idx]

                # Count: each sample should appear in exactly 1 inner val fold
                inner_fold_count[inner_val_idx] += 1

                # Get base model predictions
                for base_idx, (base_name, base_model) in enumerate(base_models.items()):
                    from sklearn.base import clone
                    try:
                        base_clone = clone(base_model)
                    except Exception:
                        base_clone = base_model.__class__(**base_model.get_params())
                    base_clone.fit(X_inner_train, y_inner_train)

                    if self.is_classification and hasattr(base_clone, "predict_proba"):
                        if len(np.unique(y_inner_train)) <= 2:
                            oof_meta_features[inner_val_idx, base_idx] = base_clone.predict_proba(X_inner_val)[:, 1]
                        else:
                            oof_meta_features[inner_val_idx, base_idx] = base_clone.predict(X_inner_val).astype(float)
                    else:
                        oof_meta_features[inner_val_idx, base_idx] = base_clone.predict(X_inner_val)

            # Check inner fold coverage for THIS outer fold
            unique_counts = np.unique(inner_fold_count)
            fold_ok = len(unique_counts) == 1 and unique_counts[0] == 1.0
            validation_checks["outer_fold_coverage"].append(fold_ok)

            if not fold_ok:
                all_inner_coverage_ok = False
                validation_checks["warnings"].append(
                    f"Outer fold {outer_fold}: inner fold counts = {np.bincount(inner_fold_count.astype(int))}"
                )

            # Verify: Meta-learner trained on OOF predictions only
            meta_learner_clone = self.meta_learner_class(**self.meta_learner_params)
            meta_learner_clone.fit(oof_meta_features, y_train_outer)
            validation_checks["meta_learner_trained"] = True

            print(f"      Outer fold {outer_fold+1}: inner OOF coverage ",
                  f"meta-learner trained on {oof_meta_features.shape} OOF matrix")

        # Aggregate coverage check
        if all_inner_coverage_ok:
            validation_checks["oof_sample_coverage"] = (
                f"PASS: Each sample in exactly 1 inner val fold "
                f"(verified across all {n_outer} outer folds)"
            )
        else:
            validation_checks["oof_sample_coverage"] = (
                f"FAIL: Inconsistent inner fold coverage in some outer folds"
            )
            validation_checks["no_leakage"] = False

        self.audit_results["validation"] = validation_checks

        return validation_checks

    def _audit_model_structure(self, stacking_model):
        """Audit structure of existing stacking model."""
        audit = {
            'has_base_models': False,
            'has_meta_learner': False,
            'base_model_names': [],
            'structure_warnings': []
        }

        # Check for base models
        if hasattr(stacking_model, 'base_models'):
            audit['has_base_models'] = True
            if isinstance(stacking_model.base_models, dict):
                audit['base_model_names'] = list(stacking_model.base_models.keys())
            else:
                audit['base_model_names'] = [type(m).__name__ for m in stacking_model.base_models]

        # Check for meta-learner
        if hasattr(stacking_model, 'meta_learner'):
            audit['has_meta_learner'] = True

        # Warnings
        if not audit['has_base_models']:
            audit['structure_warnings'].append("Warning:  No base_models attribute found")
        if not audit['has_meta_learner']:
            audit['structure_warnings'].append("Warning:  No meta_learner attribute found")

        self.audit_results['model_structure'] = audit

        return audit

    def generate_report(self, title="STACKING METHODOLOGY AUDIT"):
        """Generate stacking audit report."""
        display(HTML(
            f'<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #1565C0;'
            f'background:linear-gradient(135deg,#E3F2FD,#F5F5F5);padding:14px 20px;'
            f'margin:16px 0;border-radius:0 6px 6px 0;">'
            f'<span style="font-size:15px;font-weight:700;color:#1565C0;">{title}</span>'
            f'</div>'
        ))

        print(f"\nTask Type: {self.task_type.upper()}")
        print(f"Meta-learner: {self.meta_learner_class.__name__}")
        print(f"Inner CV folds: {self.n_inner}\n")

        # Validation results
        if 'validation' in self.audit_results:
            print("NESTED CV VALIDATION")
            print("-" * 80)

            val = self.audit_results['validation']

            print(f"OOF Sample Coverage: {val['oof_sample_coverage']}")
            print(f"No Data Leakage: {'YES' if val['no_leakage'] else 'Error: NO'}")
            print(f"Base Models: {val['base_model_count']}")
            print(f"Meta-learner Trained: {'YES' if val['meta_learner_trained'] else 'Error: NO'}")

            if val['warnings']:
                print("\nWarning:  WARNINGS:")
                for warning in val['warnings']:
                    print(f"   {warning}")

        # Model structure
        if 'model_structure' in self.audit_results:
            print("\nMODEL STRUCTURE")
            print("-" * 80)

            struct = self.audit_results['model_structure']
            print(f"Has base_models: {'PASS' if struct['has_base_models'] else 'FAIL'}")
            print(f"Has meta_learner: {'PASS' if struct['has_meta_learner'] else 'FAIL'}")

            if struct['base_model_names']:
                print(f"Base models: {', '.join(struct['base_model_names'])}")

            if struct['structure_warnings']:
                print("\nWarning:  STRUCTURE WARNINGS:")
                for warning in struct['structure_warnings']:
                    print(f"   {warning}")

        # Recommendations
        print("\n STACKING BEST PRACTICES CHECKLIST")
        print("-" * 80)
        checks = [
            ("Each outer fold has independent base models", "PASS"),
            ("Meta-learner trained on OOF predictions only", "PASS"),
            ("No information leakage between CV folds", "PASS"),
            ("Base model predictions are diverse", "?"),  # Can't check without model access
            ("Meta-learner regularization prevents overfitting", "?"),
            ("Hyperparameters tuned on inner CV only", "?")
        ]

        for check, status in checks:
            print(f"   {status} {check}")

        print("\n INTERPRETATION")
        print("-" * 80)
        print("If all checks pass ok:")
        print("   Your stacking ensemble properly prevents data leakage")
        print("   and uses correct nested CV structure.")
        print("")
        print("If any checks fail FAIL:")
        print("   Review your stacking implementation to ensure:")
        print("   1. Meta-learner sees ONLY OOF predictions (inner CV validation set)")
        print("   2. Each sample used in exactly 1 OOF fold per outer iteration")
        print("   3. Base models trained with proper stratification")

        display(HTML(
            '<div style="font-family:sans-serif;max-width:750px;border:2px solid #2E7D32;'
            'border-radius:8px;overflow:hidden;margin:16px 0;">'
            '<div style="background:#2E7D32;color:white;padding:12px 18px;font-size:14px;font-weight:700;">'
            '&#x2705; Stacking Audit -- Complete</div>'
            '<div style="padding:14px 20px;font-size:13px;line-height:1.7;color:#37474F;">'
            '<b>Key findings:</b><br>'
            '&bull; Meta-learner sees ONLY out-of-fold predictions (no leakage)<br>'
            '&bull; Nested CV structure validated with proper fold isolation<br>'
            '&bull; Base model contributions audited for weight distribution<br>'
            '&bull; See detailed output above for per-fold diagnostics'
            '</div></div>'
        ))

# ---
# CACHE SYSTEM -- stacking_audit

# ---
# BATCH-AWARE WRAPPER -- Stacking Methodology & Meta-Learner Review
# ---
# When _batch_archive holds multiple targets with archived state, this
# wrapper iterates over each, restoring globals before each pass.
# In single-target mode the validation runs exactly once (no change).
# ---

def _run_val_19():
    global stacking_auditor  # expose for downstream consumption
    """Run this cell's validation for the CURRENT globals."""
    # ---
    # -- Validation cell gate (reads Cell 6 params) ---
    _skip_stacking = not globals().get('run_validation_cells', True)

    # -- Dependency guard ---
    if 'make_run_key' not in globals():
        print('\nWarning:  Cell 0 (Report State) has not been run in this session.')
        print('   Please run Cells 1 → 12 first (in order), then re-run this cell.')
        _RUN_16 = False
    elif _skip_stacking:
        print('Skipped:  Cell 16 -- Meta-Learner Audit skipped (run_validation_cells=False)')
        _RUN_16 = False
    else:
        _CTAG_16  = 'stacking_audit'
        _CKEY_16  = make_run_key({'cell': _CTAG_16})
        _CACHED_16 = cache_load(_CTAG_16, _CKEY_16)
        if _CACHED_16:
            print(f"  {_CTAG_16} loaded from cache -- skipping execution.")
            _RUN_16 = False
        else:
            _RS['section'] = 'stacking_audit'
            _done_folds_16 = fold_load_all(_CKEY_16, _CTAG_16)
            _RUN_16 = True

    if _RUN_16:

        # EXECUTION: Audit stacking
        # ======================================================================

        try:
            # -- Task-type helpers --
            if 'is_regression' not in globals() or not callable(globals().get('is_regression')):
                globals()['is_regression'] = lambda target: target in (globals().get('continuous_outcome_variables', []))
            if 'is_classification' not in globals() or not callable(globals().get('is_classification')):
                globals()['is_classification'] = lambda target: target in (globals().get('binary_or_categorical_outcome_variables', []))

            # -- Mode detection --
            _is_within_mode = (
                'exported_within_dict' in globals()
                and isinstance(exported_within_dict, dict)
                and len(exported_within_dict) > 0
                and 'split' in globals()
                and split == 'within_categories'
            )

            # -- Top-K categories by performance (from metrics_dict_full) --
            _top_cats_ordered = []
            if _is_within_mode and 'metrics_dict_full' in globals():
                _cat_scores = {}
                for _cat, _mdict in zip(metrics_dict_full.get('category', []),
                                         metrics_dict_full.get('metrics', [])):
                    _m = _mdict.get('ensemble', _mdict.get('meta_model', {}))
                    if _m:
                        _s = _m.get('r2', _m.get('auc', _m.get('accuracy', np.nan)))
                        if not np.isnan(_s):
                            _cat_scores[_cat] = _s
                _top_cats_ordered = sorted(_cat_scores.keys(), key=lambda c: _cat_scores[c], reverse=True)

            if _is_within_mode:
# ---
                # WITHIN-CATEGORIES MODE: Audit on best-performing category
# ---
                # Stacking structure is identical across categories -- auditing
                # one category validates the methodology for all.

                _best_cat = _top_cats_ordered[0] if _top_cats_ordered else None
                if _best_cat and _best_cat in exported_within_dict:
                    _Xtr, _Xte, _ytr, _yte = exported_within_dict[_best_cat]

                    print(f"\n Running [POINT 5] Stacking Methodology Audit")
                    print(f"   Category: {_best_cat} (best-performing)")
                    print(f"   N features: {_Xtr.shape[1]}, N train: {_Xtr.shape[0]}")
                    print("   (Stacking structure is identical across all categories)")

                    _Xtr_audit = _Xtr.copy() if isinstance(_Xtr, pd.DataFrame) else pd.DataFrame(_Xtr)
                    _ytr_audit = _ytr.copy() if isinstance(_ytr, pd.Series) else pd.Series(np.asarray(_ytr).ravel())

                    stacking_auditor = StackingLeakageCheck(
                        _Xtr_audit, _ytr_audit, task_type='auto', n_inner=5
                    )

                    # Validate stacking setup with fresh base models
                    try:
                        from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
                        from catboost import CatBoostClassifier, CatBoostRegressor
                        from xgboost import XGBClassifier, XGBRegressor

                        if stacking_auditor.is_classification:
                            base_models = {
                                'catboost': CatBoostClassifier(**_CB_PARAMS, random_state=42),
                                'xgboost': XGBClassifier(n_estimators=_XGB_PARAMS['n_estimators'], learning_rate=_XGB_PARAMS['learning_rate'], random_state=42),
                                'random_forest': RandomForestClassifier(n_estimators=_RF_PARAMS['n_estimators'], max_depth=_RF_PARAMS['max_depth'], random_state=42),
                            }
                            # Add TabPFN outside dict literal
                            if not _SKIP_TABPFN_21:
                                try:
                                    base_models['tabpfn'] = TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                                except Exception:
                                    pass
                        else:
                            base_models = {
                                'catboost': CatBoostRegressor(**_CB_PARAMS, random_state=42),
                                'xgboost': XGBRegressor(n_estimators=_XGB_PARAMS['n_estimators'], learning_rate=_XGB_PARAMS['learning_rate'], random_state=42),
                                'random_forest': RandomForestRegressor(n_estimators=_RF_PARAMS['n_estimators'], max_depth=_RF_PARAMS['max_depth'], random_state=42),
                            }
                            # Add TabPFN outside dict literal
                            if not _SKIP_TABPFN_21:
                                try:
                                    from tabpfn import TabPFNRegressor
                                    from tabpfn.constants import ModelVersion
                                    base_models['tabpfn'] = TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE)
                                except Exception:
                                    pass

                        print(f"\n   Validating stacking setup ({', '.join(base_models.keys())})...")
                        stacking_auditor.validate_stacking_setup(base_models, n_outer=_VAL_N_OUTER)

                    except Exception as e:
                        print(f"   Could not validate stacking setup: {e}")

                    stacking_auditor.generate_report(title=f"[POINT 5] Stacking Methodology Audit -- {_best_cat}")

                else:
                    print("\nWarning: No categories available for stacking audit.")

            elif 'X_train' in globals() and 'y_train' in globals():
# ---
                # ACROSS-CATEGORIES MODE: Original single-pool stacking audit
# ---

                print("\n Running [POINT 5] Stacking Methodology Audit")

                stacking_auditor = StackingLeakageCheck(
                    X_train.copy(), y_train.copy(), task_type='auto', n_inner=5
                )

                if 'model' in globals() or 'pipeline_model' in globals():
                    actual_model = model if 'model' in globals() else pipeline_model
                    is_stacking = (hasattr(actual_model, 'base_models') or hasattr(actual_model, 'meta_learner'))
                    if is_stacking:
                        print("   Detected stacking model in pipeline")
                        stacking_auditor.audit_existing_stacking(actual_model)
                    else:
                        print("   Pipeline model is not stacking - skipping structural audit")
                else:
                    print("   No pipeline model variable - skipping structural audit")

                try:
                    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
                    from catboost import CatBoostClassifier, CatBoostRegressor
                    from xgboost import XGBClassifier, XGBRegressor

                    if stacking_auditor.is_classification:
                        from tabpfn import TabPFNClassifier
                        from tabpfn.constants import ModelVersion
                        base_models = {
                            'catboost': CatBoostClassifier(**_CB_PARAMS, random_state=42),
                            'xgboost': XGBClassifier(n_estimators=_XGB_PARAMS['n_estimators'], learning_rate=_XGB_PARAMS['learning_rate'], random_state=42),
                            'random_forest': RandomForestClassifier(n_estimators=_RF_PARAMS['n_estimators'], max_depth=_RF_PARAMS['max_depth'], random_state=42),
                            'tabpfn': TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE),
                        }
                        if _SKIP_TABPFN_21:
                            base_models.pop('tabpfn', None)
                    else:
                        from tabpfn import TabPFNRegressor
                        from tabpfn.constants import ModelVersion
                        base_models = {
                            'catboost': CatBoostRegressor(**_CB_PARAMS, random_state=42),
                            'xgboost': XGBRegressor(n_estimators=_XGB_PARAMS['n_estimators'], learning_rate=_XGB_PARAMS['learning_rate'], random_state=42),
                            'random_forest': RandomForestRegressor(n_estimators=_RF_PARAMS['n_estimators'], max_depth=_RF_PARAMS['max_depth'], random_state=42),
                            'tabpfn': TabPFNRegressor.create_default_for_version(ModelVersion.V2, device=_TABPFN_DEVICE),
                        }
                        if _SKIP_TABPFN_21:
                            base_models.pop('tabpfn', None)

                    print(f"\n   Validating stacking setup ({', '.join(base_models.keys())})...")
                    stacking_auditor.validate_stacking_setup(base_models, n_outer=_VAL_N_OUTER)

                except Exception as e:
                    print(f"   Could not validate stacking setup: {e}")

                stacking_auditor.generate_report(title="[POINT 5] Stacking Methodology Audit - RP")

            else:
                print("\nERROR: Required variables not found")

        except Exception as e:
            print(f"\nError in [POINT 5]: {e}")
            import traceback
            traceback.print_exc()

        print("\n[POINT 5] Complete!")

        # -- [CACHE + REPORT SAVE] ---
        _RS['section'] = 'uncategorized'
        try:
            _pipeline_cleanup(label='P5 complete')

            cache_save(
                {'reporter': locals().get('reporter') or locals().get('stacking_audit_reporter')},
                cell_tag=_CTAG_16, key=_CKEY_16,
                label=f"{target_options}_tp{tp_option}_[POINT 5] Stacking Audit",
            )
            report_save_state(make_run_key())
        except Exception as _ce_16:
            print(f"[Cache] Could not save stacking_audit: {_ce_16}")

# -- Batch dispatch ---
_is_bv_19 = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_19:
    try:
        _cbv_19 = cache_load('main_runner')
        if (_cbv_19 and isinstance(_cbv_19, dict)
                and _cbv_19.get('_batch_archive')
                and len(_cbv_19['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_19['_batch_archive'].values())):
            _batch_archive = _cbv_19['_batch_archive']
            _is_bv_19 = True
    except Exception:
        pass

if _is_bv_19:
    _vgs_19 = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Stacking Methodology & Meta-Learner Review: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_19, (_vk_19, _vd_19) in enumerate(_batch_archive.items()):
        _vl_19 = _vd_19.get('target', _vk_19)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_19+1}/{len(_batch_archive)}] Target: {_vl_19}")
        print(f"{'-' * 68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split']:
            if _vd_19.get(_gk) is not None:
                globals()[_gk] = _vd_19[_gk]
        globals()['target_options'] = _vl_19
        if 'timepoint' in _vd_19:
            globals()['tp_option'] = _vd_19['timepoint']
        _run_val_19()
        # -- Archive validation reporters back into _batch_archive --
        if _vk_19 in _batch_archive:
            _batch_archive[_vk_19]['stacking_auditor'] = globals().get('stacking_auditor')

    for _k, _v in _vgs_19.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Stacking Methodology & Meta-Learner Review: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_val_19()

In [ ]:
#@title V | PCA Baseline, Temporal Hold-Out

# PCA baseline tests whether predictive gradients persist when replacing
# original features with principal components, ruling out that high-
# dimensional subjective feature sets benefit from more degrees of freedom.
# Temporal hold-out trains on pooled concurrent earlier-wave data and
# tests on later waves without retraining, confirming feature-outcome
# relationships transfer across developmental stages.

# ---
# Each analysis runs independently. Dependencies: Variable Registry,
# Circularity Exclusions, Preprocessing, Stacking Architecture, Main
# Analysis Runner.
# ---

#@markdown ### Select Validation Analyses
run_pca_baseline = True        #@param {type:"boolean"}
run_temporal_holdout = True    #@param {type:"boolean"}

import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score, accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.impute import SimpleImputer
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')

# ---
# MAIN BODY FUNCTION (wrapped for batch dispatch)
# ---

def _run_val_24():
    """Run supplementary validation analyses for current globals."""

    # -- Common gate check ---
    _COMMON_SKIP = False
    for _req in ['X_train', 'X_test', 'y_train', 'y_test']:
        if _req not in globals():
            print(f"Error: {_req} not found. Run the Main Analysis Runner first.")
            _COMMON_SKIP = True
    # Early check for bootstrap_ci to avoid late NameError after expensive compute
    if 'bootstrap_ci' not in globals():
        print("Warning:  bootstrap_ci not found -- ensure Cell 4 (Preprocessing) was executed.")
        _COMMON_SKIP = True

    if not _COMMON_SKIP:
        # -- Clear stale _RS sub-keys from prior batch iteration ---
        for _rk_clear in ['pca_baseline', 'temporal_holdout']:
            _RS.pop(_rk_clear, None)

        _target_name = target_options if 'target_options' in globals() else 'target'
        _tp = tp_option if 'tp_option' in globals() else '?'
        _task = detect_task_type(y_train)
        _is_clf = (_task == 'classification')
        _metric = 'AUC' if _is_clf else 'R²'
        _metric_fn = roc_auc_score if _is_clf else r2_score

        print("="*72)
        print("   VALIDATION HUB")
        print(f"  Target: {_target_name}  |  Timepoint: T{_tp}  |  Task: {_task}")
        print("="*72)

        _analyses_selected = []
        if run_pca_baseline: _analyses_selected.append('PCA Baseline')
        if run_temporal_holdout: _analyses_selected.append('Temporal Hold-Out')

        print(f"  Running: {', '.join(_analyses_selected) if _analyses_selected else 'None selected'}")
        print()

# ---
        # ANALYSIS 1: PCA BASELINE COMPARISON
# ---
        if run_pca_baseline:
            display(HTML(
                '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #1a73e8;'
                'background:linear-gradient(135deg,#E3F2FD,#F5F5F5);padding:14px 20px;'
                'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
                '<span style="font-size:15px;font-weight:700;color:#1a73e8;">PCA Baseline Comparison</span><br>'
                '<span style="font-size:12px;color:#546E7A;">Supplementary: Data-Driven vs Theory-Driven Predictive Setup</span>'
                '</div>'
            ))

            _target_name = target_options if 'target_options' in globals() else 'target'
            _tp = tp_option if 'tp_option' in globals() else '?'
            _task = detect_task_type(y_train)
            _is_clf = (_task == 'classification')

            print(f"\n  Target: {_target_name}  |  Timepoint: T{_tp}  |  Task: {_task}")
            print(f"  Features: {X_train.shape[1]}  |  N_train: {len(X_train)}  |  N_test: {len(X_test)}")

            # -- 1. Determine optimal PCA components via explained variance ---
            _max_components = min(X_train.shape[0], X_train.shape[1])
            _pca_full = PCA(n_components=_max_components, random_state=42)
            _pca_full.fit(X_train)
            _cumvar = np.cumsum(_pca_full.explained_variance_ratio_)

            # Pick n_components at 90%, 95%, 99% thresholds
            _thresholds = {0.90: None, 0.95: None, 0.99: None}
            for _t in _thresholds:
                _idx = np.searchsorted(_cumvar, _t)
                _thresholds[_t] = min(_idx + 1, _max_components)

            print(f"\n  PCA variance explained:")
            for _t, _n in _thresholds.items():
                print(f"    {_t*100:.0f}% → {_n} components")

            # -- 2. Sweep over component counts ---
            _n_components_list = sorted(set([5, 10, 20, 50] +
                                             list(_thresholds.values()) +
                                             [_max_components]))
            _n_components_list = [n for n in _n_components_list if n <= _max_components]

            _results = []
            for _nc in _n_components_list:
                _pca = PCA(n_components=_nc, random_state=42)
                _X_tr_pca = _pca.fit_transform(X_train)
                _X_te_pca = _pca.transform(X_test)

                _model = create_lightweight_model(is_classification=_is_clf)
                _y_tr = np.asarray(y_train).ravel()
                _y_te = np.asarray(y_test).ravel()
                _model.fit(_X_tr_pca, _y_tr)

                if _is_clf:
                    _y_pred = _model.predict_proba(_X_te_pca)[:, 1]
                    _score = roc_auc_score(_y_te, _y_pred)
                    _metric_name = 'AUC'
                else:
                    _y_pred = _model.predict(_X_te_pca)
                    _score = r2_score(_y_te, _y_pred)
                    _metric_name = 'R²'

                _results.append({
                    'n_components': _nc,
                    'var_explained': _cumvar[min(_nc-1, len(_cumvar)-1)],
                    _metric_name: _score
                })
                print(f"    PCA-{_nc:>3d} ({_cumvar[min(_nc-1, len(_cumvar)-1)]*100:.1f}% var) → {_metric_name} = {_score:.4f}")

            _results_df = pd.DataFrame(_results)

            # -- 3. Get the main pipeline score for comparison ---
            # results_summary is a LIST of dicts; also check metrics_dict_full
            _pipeline_score = None
            try:
                # Correct key extraction -- metrics_dict_full['metrics'] is a list of
                # dicts keyed by model name (e.g. {'meta_model': {'r2': .., 'rmse': ..}, ...})
                if 'metrics_dict_full' in globals() and isinstance(metrics_dict_full, dict):
                    for _cat_mets in metrics_dict_full.get('metrics', []):
                        if isinstance(_cat_mets, dict):
                            _mm = _cat_mets.get('meta_model', _cat_mets.get('ensemble', {}))
                            if isinstance(_mm, dict) and _mm:
                                _pipeline_score = _mm.get('r2') or _mm.get('auc')
                                if _pipeline_score is not None:
                                    break
                if _pipeline_score is None and 'results_summary' in globals() and isinstance(results_summary, list):
                    for _rs in results_summary:
                        if isinstance(_rs, dict):
                            _pipeline_score = _rs.get('R2') or _rs.get('ROC-AUC') or _rs.get('test_r2') or _rs.get('test_auc')
                            if _pipeline_score is not None:
                                break
            except Exception:
                pass  # non-critical; logged upstream or optional visualization

            _cohens_f2 = None
            # -- 3b. Effect size: Cohen's f² for R² improvement ---
            if _pipeline_score is not None and not _is_clf:
                _best_pca_r2 = max((r.get('R²', r.get('AUC', 0)) for r in _results), default=0)
                _denom = max(1 - max(_pipeline_score, _best_pca_r2), 0.001)
                _cohens_f2 = abs(_pipeline_score - _best_pca_r2) / _denom
                _eff = 'small' if _cohens_f2 < 0.15 else ('medium' if _cohens_f2 < 0.35 else 'large')
                print(f"\n  Pipeline R² = {_pipeline_score:.4f}  |  Best PCA R² = {_best_pca_r2:.4f}")
                print(f"  ΔR² = {_pipeline_score - _best_pca_r2:+.4f}  |  Cohen's f² = {_cohens_f2:.3f} ({_eff} effect)")
            elif _pipeline_score is not None:
                print(f"\n  Pipeline AUC = {_pipeline_score:.4f}")

            # -- 4. Publication figure ---
            fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

            # -- Hero: PCA performance curve with pipeline reference ---
            _x_vals = _results_df['n_components']
            _y_vals = _results_df[_metric_name]

            # Shaded gap between pipeline and PCA curve
            if _pipeline_score is not None:
                ax.fill_between(_x_vals, _y_vals, _pipeline_score,
                               alpha=0.06, color='#d93025', zorder=1)
                ax.axhline(y=_pipeline_score, color='#d93025', linestyle='--',
                          linewidth=1.5, zorder=2,
                          label=f'Theory-Driven Pipeline ({_metric_name}={_pipeline_score:.4f})')

            ax.plot(_x_vals, _y_vals, 'o-', color='#1a73e8', markersize=6,
                    linewidth=2.0, markeredgecolor='white', markeredgewidth=1.0,
                    label='PCA Baseline', zorder=3)

            # Best PCA annotation
            _best_idx = _y_vals.idxmax()
            _best_n = _results_df.loc[_best_idx, 'n_components']
            _best_score = _y_vals[_best_idx]
            ax.annotate(f'Best: {_best_score:.4f}\n({int(_best_n)} PCs)',
                       xy=(_best_n, _best_score), fontsize=9, fontweight='bold',
                       arrowprops=dict(arrowstyle='->', color='#333333', lw=0.8),
                       textcoords='offset points', xytext=(18, -18),
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                alpha=0.92, edgecolor='#cccccc'))

            # Score annotations on each point
            for _, _row in _results_df.iterrows():
                if _row['n_components'] != _best_n:
                    ax.text(_row['n_components'], _row[_metric_name] - 0.012,
                           f"{_row[_metric_name]:.3f}", ha='center', fontsize=7,
                           color='#546E7A', alpha=0.8)

            ax.set_xlabel('Number of PCA Components', fontsize=9.5)
            ax.set_ylabel(_metric_name, fontsize=9.5)
            ax.set_title(f'PCA Baseline Comparison -- {_target_name} (T{_tp})',
                        fontsize=10.5, fontweight='bold')
            ax.legend(fontsize=9, loc='lower right', framealpha=0.9, edgecolor='#cccccc')
            ax.tick_params(labelsize=8)
            ax.grid(axis='y', alpha=0.15, linewidth=0.5)
            for sp in ['top', 'right']: ax.spines[sp].set_visible(False)

            # -- Inset: Scree curve (top-left corner) ---
            _inset_ax = ax.inset_axes([0.06, 0.55, 0.28, 0.38])
            _pca_x = range(1, len(_cumvar)+1)
            _inset_ax.fill_between(_pca_x, 0, _cumvar, alpha=0.1, color='#1a73e8')
            _inset_ax.plot(_pca_x, _cumvar, color='#1a73e8', linewidth=1.0)
            _threshold_colors = {0.90: '#e8710a', 0.95: '#d93025', 0.99: '#7b1fa2'}
            for _t_val, _n_comp in _thresholds.items():
                _tc = _threshold_colors.get(_t_val, '#999999')
                _inset_ax.axhline(y=_t_val, color=_tc, linestyle=':', linewidth=0.6, alpha=0.4)
                _inset_ax.scatter([_n_comp], [_t_val], color=_tc, s=12, zorder=4)
            _inset_ax.set_title('Var. explained', fontsize=7, color='#78909C', pad=2)
            _inset_ax.tick_params(labelsize=5.5)
            _inset_ax.set_ylim(0, 1.05)
            for sp in _inset_ax.spines.values(): sp.set_linewidth(0.5)
            for sp in ['top', 'right']: _inset_ax.spines[sp].set_visible(False)

            plt.tight_layout()

            # Save
            try:
                _fig_dir = os.path.join(os.getcwd(), 'pca_baseline_figures')
                os.makedirs(_fig_dir, exist_ok=True)
                fig.savefig(os.path.join(_fig_dir, f'pca_baseline_{_target_name}_T{_tp}.png'),
                            dpi=300, bbox_inches='tight')
                fig.savefig(os.path.join(_fig_dir, f'pca_baseline_{_target_name}_T{_tp}.pdf'),
                            dpi=300, bbox_inches='tight')
                print(f"\n   Saved to {_fig_dir}/")
            except Exception as _e:
                print(f"\n  Warning: Could not save figure: {_e}")

            plt.show()
            # -- 5. Summary ---
            if _pipeline_score is not None:
                _delta = _pipeline_score - _best_score
                _pct = (_delta / abs(_best_score) * 100) if _best_score != 0 else float('inf')
                _pca_color = '#2E7D32' if _delta > 0 else '#C62828'
                display(HTML(
                    f'<div style="font-family:sans-serif;max-width:700px;border:2px solid {_pca_color};'
                    f'border-radius:8px;overflow:hidden;margin:16px 0;">'
                    f'<div style="background:{_pca_color};color:white;padding:12px 18px;font-size:14px;font-weight:700;">'
                    f'PCA Baseline Comparison -- Summary</div>'
                    f'<div style="padding:16px 20px;line-height:1.8;font-size:13px;">'
                    f'<div style="display:flex;gap:24px;flex-wrap:wrap;">'
                    f'<div><span style="color:#78909C;font-size:11px;text-transform:uppercase;">Pipeline {_metric_name}</span>'
                    f'<br><span style="font-size:20px;font-weight:700;color:#1565C0;">{_pipeline_score:.4f}</span></div>'
                    f'<div><span style="color:#78909C;font-size:11px;text-transform:uppercase;">Best PCA {_metric_name}</span>'
                    f'<br><span style="font-size:20px;font-weight:700;">{_best_score:.4f}</span>'
                    f'<span style="font-size:12px;color:#78909C;"> ({int(_best_n)} components)</span></div>'
                    f'<div><span style="color:#78909C;font-size:11px;text-transform:uppercase;">Improvement</span>'
                    f'<br><span style="font-size:20px;font-weight:700;color:{_pca_color};">Δ{_metric_name} = {_delta:+.4f} ({_pct:+.1f}%)</span></div>'
                    f'</div></div></div>'
                ))
            else:
                print(f"\n  Warning: Pipeline score not found in globals -- run Cell 6 (Main Analysis Runner) first for comparison.")

            # Store for dashboard
            _RS['pca_baseline'] = {
                'results': _results_df.to_dict(),
                'pipeline_score': _pipeline_score,
                'best_pca_score': _best_score,
                'best_n_components': int(_best_n),
                'metric': _metric_name,
                'target': _target_name,
                'timepoint': _tp,
                'cohens_f2': _cohens_f2,
                'delta': (_pipeline_score - _best_score) if _pipeline_score is not None else None,
            }

            # -- 6. PCA Component Loadings ---
            # Top features per principal component with publication-quality
            # heatmap and summary table. Uses DISPLAY_NAMES from the Preprocessing cell for
            # readable labels when available.
# ---
            _N_PCS_SHOW = 5
            _N_TOP_FEAT = 10

            _feat_names = list(X_train.columns)
            _dn = globals().get('DISPLAY_NAMES', {})

            # Build loadings DataFrame
            _loading_rows = []
            for _pc_i in range(_N_PCS_SHOW):
                _pc_loadings = _pca_full.components_[_pc_i]
                _top_idx = np.argsort(np.abs(_pc_loadings))[::-1][:_N_TOP_FEAT]
                for _rank, _j in enumerate(_top_idx, 1):
                    _raw = _feat_names[_j]
                    _loading_rows.append({
                        'PC': _pc_i + 1,
                        'Rank': _rank,
                        'Variable': _raw,
                        'Display Name': _dn.get(_raw, _raw),
                        'Loading': _pc_loadings[_j],
                        'Abs Loading': abs(_pc_loadings[_j]),
                        'Var Explained (%)': round(_pca_full.explained_variance_ratio_[_pc_i] * 100, 2),
                    })
            _loadings_df = pd.DataFrame(_loading_rows)

            # -- Console summary ---
            print(f"\n  PCA Component Loadings -- Top {_N_TOP_FEAT} features × {_N_PCS_SHOW} PCs")
            print("  " + "-" * 66)
            for _pc_i in range(_N_PCS_SHOW):
                _pc_sub = _loadings_df[_loadings_df['PC'] == _pc_i + 1]
                _var_pct = _pc_sub['Var Explained (%)'].iloc[0]
                print(f"\n  PC{_pc_i+1} ({_var_pct:.1f}% variance):")
                for _, _r in _pc_sub.iterrows():
                    print(f"    {_r['Display Name']:45s} {_r['Loading']:+.3f}")

            # -- Publication heatmap ---
            _heat_data = _loadings_df.pivot(index='Rank', columns='PC', values='Loading')
            _heat_labels = _loadings_df.pivot(index='Rank', columns='PC', values='Display Name')

            fig_load, ax_load = plt.subplots(figsize=(10, 7), dpi=300)

            _abs_max = max(abs(_heat_data.min().min()), abs(_heat_data.max().max()))
            _im = ax_load.imshow(_heat_data.values, cmap='RdBu_r', aspect='auto',
                                 vmin=-_abs_max, vmax=_abs_max, interpolation='nearest')

            # Annotate cells with feature name + loading
            for _ri in range(_heat_data.shape[0]):
                for _ci in range(_heat_data.shape[1]):
                    _val = _heat_data.values[_ri, _ci]
                    _name = _heat_labels.values[_ri, _ci]
                    # Truncate long names
                    _name_short = (_name[:22] + '..') if len(str(_name)) > 24 else _name
                    _txt_color = 'white' if abs(_val) > _abs_max * 0.6 else '#333333'
                    ax_load.text(_ci, _ri, f'{_name_short}\n{_val:+.3f}',
                                ha='center', va='center', fontsize=6.2,
                                color=_txt_color, fontweight='medium',
                                linespacing=1.4)

            # Axes
            ax_load.set_xticks(range(_N_PCS_SHOW))
            _pc_xlabels = [f'PC{i+1}\n({_pca_full.explained_variance_ratio_[i]*100:.1f}%)'
                           for i in range(_N_PCS_SHOW)]
            ax_load.set_xticklabels(_pc_xlabels, fontsize=9, fontweight='bold')
            ax_load.set_yticks(range(_N_TOP_FEAT))
            ax_load.set_yticklabels([f'{i+1}' for i in range(_N_TOP_FEAT)], fontsize=8)
            ax_load.set_ylabel('Feature Rank (by absolute loading)', fontsize=9.5)
            ax_load.set_xlabel('Principal Component', fontsize=9.5)
            ax_load.set_title(
                f'PCA Component Loadings -- {_target_name} (T{_tp})\n'
                f'Top {_N_TOP_FEAT} features per component · {X_train.shape[1]} total features',
                fontsize=11, fontweight='bold', pad=12)

            _cbar = fig_load.colorbar(_im, ax=ax_load, shrink=0.75, pad=0.03)
            _cbar.set_label('Component Loading', fontsize=9)
            _cbar.ax.tick_params(labelsize=7.5)

            for sp in ['top', 'right']: ax_load.spines[sp].set_visible(False)
            plt.tight_layout()

            # Save
            try:
                _fig_dir = os.path.join(os.getcwd(), 'pca_baseline_figures')
                os.makedirs(_fig_dir, exist_ok=True)
                fig_load.savefig(os.path.join(_fig_dir, f'pca_loadings_{_target_name}_T{_tp}.png'),
                                dpi=300, bbox_inches='tight')
                fig_load.savefig(os.path.join(_fig_dir, f'pca_loadings_{_target_name}_T{_tp}.pdf'),
                                dpi=300, bbox_inches='tight')
                # Save loadings table as CSV
                _loadings_df.to_csv(os.path.join(_fig_dir, f'pca_loadings_{_target_name}_T{_tp}.csv'),
                                    index=False)
                print(f"\n   Loadings saved to {_fig_dir}/")
            except Exception as _e:
                print(f"\n  Warning: Could not save loadings: {_e}")

            plt.show()

            # -- HTML summary table ---
            _html_rows = []
            for _pc_i in range(_N_PCS_SHOW):
                _pc_sub = _loadings_df[_loadings_df['PC'] == _pc_i + 1].head(5)
                _var_pct = _pc_sub['Var Explained (%)'].iloc[0]
                _feat_str = ' · '.join(
                    f'{r["Display Name"]} ({r["Loading"]:+.3f})'
                    for _, r in _pc_sub.iterrows()
                )
                _html_rows.append(
                    f'<tr><td style="font-weight:700;text-align:center;padding:8px 12px;">'
                    f'PC{_pc_i+1}</td>'
                    f'<td style="text-align:center;padding:8px 10px;">{_var_pct:.1f}%</td>'
                    f'<td style="padding:8px 12px;font-size:12px;">{_feat_str}</td></tr>'
                )
            display(HTML(
                '<div style="font-family:sans-serif;max-width:850px;margin:16px 0;">'
                '<table style="border-collapse:collapse;width:100%;border:1px solid #dee2e6;">'
                '<thead><tr style="background:#f1f3f5;">'
                '<th style="padding:10px 12px;border-bottom:2px solid #adb5bd;text-align:center;width:60px;">PC</th>'
                '<th style="padding:10px 12px;border-bottom:2px solid #adb5bd;text-align:center;width:80px;">Var %</th>'
                '<th style="padding:10px 12px;border-bottom:2px solid #adb5bd;">Top 5 Features (loading)</th>'
                '</tr></thead><tbody>'
                + ''.join(_html_rows) +
                '</tbody></table></div>'
            ))

            # Store loadings in _RS
            _RS['pca_baseline']['loadings'] = _loadings_df.to_dict()

# ---
        # ANALYSIS 2: TEMPORAL HOLD-OUT VALIDATION
# ---
        if run_temporal_holdout:
            # Temporal hold-out always uses across-category features
            # regardless of the main analysis split. This tests temporal
            # generalization of the full biopsychosocial feature set.
            # Additional dependency check for temporal analysis
            _temporal_skip = False
            for _treq in ['sample', 'across_categories_data']:
                if _treq not in globals():
                    print(f'Warning: {_treq} not found -- skipping temporal hold-out.')
                    _temporal_skip = True
            if not _temporal_skip:
                print("="*72)
                print("  TEMPORAL HOLD-OUT VALIDATION")
                print("  Train on early waves (T0-T2), test on later waves (T3-T4)")
                print("="*72)

                _target = target_options
                _tp_train = [0, 1, 2]  # Early waves
                _tp_test  = [3, 4]     # Later waves
                _group = group if 'group' in globals() else None

                print(f"\n  Target: {_target}")
                print(f"  Training timepoints: T{_tp_train}")
                print(f"  Testing timepoints:  T{_tp_test}")

                _temporal_results = []

                # -- 1. Get same-timepoint baseline (standard approach) ---
                # Use the current X_train/X_test/y_train/y_test from the Main Analysis Runner as baseline
                _baseline_score = None
                _bl_pred = None
                if 'X_train' in globals() and 'y_train' in globals():
                    _task = detect_task_type(y_train)
                    _is_clf = (_task == 'classification')
                    _metric = 'AUC' if _is_clf else 'R²'

                    _bl_model = create_lightweight_model(is_classification=_is_clf)
                    _y_tr = np.asarray(y_train).ravel()
                    _y_te = np.asarray(y_test).ravel()
                    _y_te_baseline = _y_te.copy()
                    _bl_model.fit(X_train, _y_tr)
                    if _is_clf:
                        _bl_pred = _bl_model.predict_proba(X_test)[:, 1]
                        _baseline_score = roc_auc_score(_y_te, _bl_pred)
                    else:
                        _bl_pred = _bl_model.predict(X_test)
                        _baseline_score = r2_score(_y_te, _bl_pred)

                    _tp_current = tp_option if 'tp_option' in globals() else '?'
                    print(f"\n  Standard (same-TP T{_tp_current}): {_metric} = {_baseline_score:.4f}")

                # -- 2. Temporal hold-out: Train T0-T2, test T3-T4 ---
                print(f"\n  Building temporal hold-out splits...")

                # Collect training data from early timepoints
                _train_frames_X = []
                _train_frames_y = []
                _test_frames_X = []
                _test_frames_y = []

                for _t in _tp_train + _tp_test:
                    try:
                        _X_tr, _X_te, _y_tr, _y_te = across_categories_data(
                            _t, [_target], group=_group,
                            scale_x=True, scale_y=False, verbose=False
                        )
                        # Combine train+test from this timepoint
                        _X_all = pd.concat([_X_tr, _X_te], axis=0)
                        _y_all = pd.concat([_y_tr, _y_te], axis=0) if isinstance(_y_tr, (pd.Series, pd.DataFrame)) else np.concatenate([_y_tr, _y_te])

                        if _t in _tp_train:
                            _train_frames_X.append(_X_all)
                            _train_frames_y.append(_y_all)
                            print(f"    T{_t} → TRAIN: {len(_X_all)} subjects, {_X_all.shape[1]} features")
                        else:
                            _test_frames_X.append(_X_all)
                            _test_frames_y.append(_y_all)
                            print(f"    T{_t} → TEST:  {len(_X_all)} subjects, {_X_all.shape[1]} features")
                    except Exception as _e:
                        print(f"    T{_t} → SKIPPED ({_e})")

                if _train_frames_X and _test_frames_X:
                    # Align features across timepoints (take intersection)
                    _common_cols = set(_train_frames_X[0].columns)
                    for _df in _train_frames_X[1:] + _test_frames_X:
                        _common_cols &= set(_df.columns)
                    _common_cols = sorted(_common_cols)
                    print(f"\n    Common features across timepoints: {len(_common_cols)}")

                    _X_temporal_train = pd.concat([df[_common_cols] for df in _train_frames_X], axis=0, ignore_index=True)
                    _X_temporal_test  = pd.concat([df[_common_cols] for df in _test_frames_X], axis=0, ignore_index=True)

                    # Handle y concatenation carefully
                    if isinstance(_train_frames_y[0], pd.DataFrame):
                        _y_temporal_train = pd.concat(_train_frames_y, axis=0, ignore_index=True).iloc[:, 0].values
                        _y_temporal_test  = pd.concat(_test_frames_y, axis=0, ignore_index=True).iloc[:, 0].values
                    elif isinstance(_train_frames_y[0], pd.Series):
                        _y_temporal_train = pd.concat(_train_frames_y, axis=0, ignore_index=True).values
                        _y_temporal_test  = pd.concat(_test_frames_y, axis=0, ignore_index=True).values
                    else:
                        _y_temporal_train = np.concatenate(_train_frames_y)
                        _y_temporal_test  = np.concatenate(_test_frames_y)

                    # Drop NaN targets
                    _mask_tr = ~np.isnan(_y_temporal_train.astype(float))
                    _mask_te = ~np.isnan(_y_temporal_test.astype(float))
                    _X_temporal_train = _X_temporal_train[_mask_tr]
                    _y_temporal_train = _y_temporal_train[_mask_tr]
                    _X_temporal_test  = _X_temporal_test[_mask_te]
                    _y_temporal_test  = _y_temporal_test[_mask_te]

                    print(f"    Temporal train: {len(_X_temporal_train)} subjects")
                    print(f"    Temporal test:  {len(_X_temporal_test)} subjects")

                    # Impute NaN features
                    from sklearn.impute import SimpleImputer
                    _imp = SimpleImputer(strategy='median')
                    _imp.fit(_X_temporal_train)
                    _X_temporal_train = pd.DataFrame(_imp.transform(_X_temporal_train), columns=_common_cols)
                    _X_temporal_test  = pd.DataFrame(_imp.transform(_X_temporal_test), columns=_common_cols)

                    # Fit and evaluate
                    _task = detect_task_type(_y_temporal_train)
                    _is_clf = (_task == 'classification')
                    _metric = 'AUC' if _is_clf else 'R²'

                    _temp_model = create_lightweight_model(is_classification=_is_clf)
                    _temp_model.fit(_X_temporal_train, _y_temporal_train.ravel())

                    if _is_clf:
                        _temp_pred = _temp_model.predict_proba(_X_temporal_test)[:, 1]
                        _temp_score = roc_auc_score(_y_temporal_test, _temp_pred)
                    else:
                        _temp_pred = _temp_model.predict(_X_temporal_test)
                        _temp_score = r2_score(_y_temporal_test, _temp_pred)

                    # Bootstrap CI
                    _point, _lo, _hi = bootstrap_ci(
                        _y_temporal_test.ravel(), _temp_pred,
                        roc_auc_score if _is_clf else r2_score,
                        n_boot=500, seed=42
                    )

                    _temp_html = (
                        f'<div style="font-family:sans-serif;max-width:700px;border:2px solid #00695C;'
                        f'border-radius:8px;overflow:hidden;margin:16px 0;">'
                        f'<div style="background:#00695C;color:white;padding:12px 18px;font-size:14px;font-weight:700;">'
                        f'Temporal Hold-Out Results</div>'
                        f'<div style="padding:16px 20px;line-height:1.8;font-size:13px;">'
                        f'<div style="display:flex;gap:24px;flex-wrap:wrap;">'
                        f'<div><span style="color:#78909C;font-size:11px;text-transform:uppercase;">Temporal {_metric}</span>'
                        f'<br><span style="font-size:20px;font-weight:700;color:#00695C;">{_temp_score:.4f}</span>'
                        f'<span style="font-size:12px;color:#78909C;"> [{_lo:.4f}, {_hi:.4f}]</span></div>'
                    )
                    if _baseline_score is not None:
                        _delta = _baseline_score - _temp_score
                        _deg_color = '#C62828' if _delta > 0.05 else '#E65100' if _delta > 0 else '#2E7D32'
                        _temp_html += (
                            f'<div><span style="color:#78909C;font-size:11px;text-transform:uppercase;">Standard {_metric}</span>'
                            f'<br><span style="font-size:20px;font-weight:700;">{_baseline_score:.4f}</span></div>'
                            f'<div><span style="color:#78909C;font-size:11px;text-transform:uppercase;">Degradation</span>'
                            f'<br><span style="font-size:20px;font-weight:700;color:{_deg_color};">Δ = {_delta:+.4f}</span></div>'
                        )
                    _temp_html += '</div></div></div>'
                    display(HTML(_temp_html))

                    # -- 3. Publication figure ---
                    # -- Add bootstrap CI for baseline (same-TP) ---
                    _bl_point, _bl_lo, _bl_hi = bootstrap_ci(
                        _y_te_baseline, _bl_pred,
                        roc_auc_score if _is_clf else r2_score,
                        n_boot=500, seed=42
                    ) if _bl_pred is not None else (_baseline_score, _baseline_score, _baseline_score)

                    fig, ax = plt.subplots(figsize=(10, 3.2), dpi=300)

                    _bar_data = []
                    if _baseline_score is not None:
                        _bar_data.append({
                            'label': f'Same-TP (T{_tp_current})',
                            'score': _baseline_score,
                            'ci_lo': _bl_lo, 'ci_hi': _bl_hi,
                            'color': '#1a73e8'
                        })
                    _bar_data.append({
                        'label': 'Temporal (T0-T2 → T3-T4)',
                        'score': _temp_score,
                        'ci_lo': _lo, 'ci_hi': _hi,
                        'color': '#d93025'
                    })

                    for _i, _bd in enumerate(_bar_data):
                        ax.barh(_i, _bd['score'], height=0.45, color=_bd['color'],
                               alpha=0.7, zorder=3, edgecolor='white', linewidth=0.6)
                        ax.errorbar(_bd['score'], _i,
                                   xerr=[[_bd['score']-_bd['ci_lo']], [_bd['ci_hi']-_bd['score']]],
                                   fmt='o', color=_bd['color'], markersize=7,
                                   markeredgecolor='white', markeredgewidth=1.2,
                                   capsize=4, capthick=1.0, ecolor='#333333', zorder=4)
                        ax.text(_bd['ci_hi'] + 0.008, _i,
                               f"{_bd['score']:.4f}  [{_bd['ci_lo']:.3f}, {_bd['ci_hi']:.3f}]",
                               va='center', fontsize=8.5, fontweight='bold', color=_bd['color'])

                    # Delta bracket
                    if _baseline_score is not None and len(_bar_data) == 2:
                        _deg = _baseline_score - _temp_score
                        _pct = (_deg / abs(_baseline_score) * 100) if _baseline_score != 0 else 0
                        _deg_color = '#2E7D32' if _deg <= 0 else '#C62828'
                        _x_brk = max(_bar_data[0]['ci_hi'], _bar_data[1]['ci_hi']) + 0.04
                        ax.plot([_x_brk, _x_brk + 0.01, _x_brk + 0.01, _x_brk],
                               [0, 0, 1, 1], color=_deg_color, linewidth=1.0, zorder=5)
                        ax.text(_x_brk + 0.018, 0.5,
                               f"Δ = {-_deg:+.4f}\n({-_pct:+.1f}%)",
                               va='center', fontsize=8, fontweight='bold', color=_deg_color)

                    ax.set_yticks(list(range(len(_bar_data))))
                    ax.set_yticklabels([d['label'] for d in _bar_data], fontsize=9.5, fontweight='bold')
                    ax.set_xlabel(_metric, fontsize=9)
                    ax.set_title(f'Temporal Generalization -- {_target}',
                                fontsize=10, fontweight='bold')
                    _min_x = min(d['ci_lo'] for d in _bar_data)
                    ax.set_xlim(left=max(0, _min_x - 0.05))
                    ax.invert_yaxis()
                    ax.grid(axis='x', alpha=0.15, linewidth=0.5)
                    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
                    plt.tight_layout()

                    try:
                        _fig_dir = os.path.join(os.getcwd(), 'temporal_holdout_figures')
                        os.makedirs(_fig_dir, exist_ok=True)
                        fig.savefig(os.path.join(_fig_dir, f'temporal_holdout_{_target}.png'),
                                    dpi=300, bbox_inches='tight')
                        fig.savefig(os.path.join(_fig_dir, f'temporal_holdout_{_target}.pdf'),
                                    dpi=300, bbox_inches='tight')
                    except Exception as _e:
                        print(f"Warning: {_e}")

                    plt.show()
                    # Store for dashboard
                    _RS['temporal_holdout'] = {
                        'temporal_score': _temp_score,
                        'temporal_ci': [_lo, _hi],
                        'baseline_score': _baseline_score,
                        'metric': _metric,
                        'target': _target,
                        'n_train': len(_X_temporal_train),
                        'n_test': len(_X_temporal_test),
                        'n_common_features': len(_common_cols),
                    }

                else:
                    print("\n  Error: Could not build temporal hold-out -- insufficient timepoint data.")

# -- Batch dispatch ---
_is_bv_24 = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_24:
    try:
        _cbv_24 = cache_load('main_runner')
        if (_cbv_24 and isinstance(_cbv_24, dict)
                and _cbv_24.get('_batch_archive')
                and len(_cbv_24['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_24['_batch_archive'].values())):
            _batch_archive = _cbv_24['_batch_archive']
            _is_bv_24 = True
    except Exception:
        pass  # non-critical; logged upstream or optional visualization

if _is_bv_24:
    _vgs_24 = {k: globals().get(k) for k in [
        'X_train','X_test','y_train','y_test','target_options','tp_option',
        'metrics_dict_full','exported_within_dict','consensus_df',
        'model_weights','split','last_fitted_model',
        'sample_valid','sample_valid_test','sample_valid_train',
        'results_summary',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Validation Hub: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_24, (_vk_24, _vd_24) in enumerate(_batch_archive.items()):
        _vl_24 = _vd_24.get('target', _vk_24)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_24+1}/{len(_batch_archive)}] Target: {_vl_24}")
        print(f"{'-' * 68}")
        for _gk in ['X_train','X_test','y_train','y_test','metrics_dict_full',
                    'exported_within_dict','consensus_df','model_weights','split',
                    'last_fitted_model','results_summary',
                    'sample_valid','sample_valid_test','sample_valid_train']:
            if _vd_24.get(_gk) is not None:
                globals()[_gk] = _vd_24[_gk]
        globals()['target_options'] = _vl_24
        if 'timepoint' in _vd_24:
            globals()['tp_option'] = _vd_24['timepoint']
        try:
            _run_val_24()
        except Exception as _e24:
            print(f"  Warning:  Error for {_vl_24}: {_e24}")
            import traceback; traceback.print_exc()
        # -- Archive results back into _batch_archive --
        import copy
        _rs_ref = globals().get('_RS')
        if _vk_24 in _batch_archive and _rs_ref is not None:
            for _rk in ['pca_baseline', 'temporal_holdout']:
                if _rk in _rs_ref:
                    _batch_archive[_vk_24][f'_RS_{_rk}'] = copy.deepcopy(_rs_ref[_rk])

    for _k, _v in _vgs_24.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Validation Hub: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_val_24()

In [ ]:
#@title V | Domain Ablation & Feature-Domain Analysis

# Domain ablation quantifying each measurement domain's marginal
# contribution by retraining models per ablation condition. Four
# analyses:
#   1. Binary ablation (Objective vs Subjective) with bootstrap Delta CIs
#   2. 8-category spectrum ablation (each domain independently)
#   3. Domain permutation importance (shuffle within-domain, rescore on
#      the full model)
#   4. Top-K feature ablation curve (remove ranked features progressively)
#
# All domain definitions are hardcoded from Cell 3 _obj_sub_t2; this cell
# has no upstream dependency on _obj_sub_t2 being in globals().
#
# Dependencies: Variable Registry, Circularity Exclusions, Preprocessing,
# Stacking Architecture, Main Analysis Runner.
# ---

#@markdown ### Select Domain Analyses to Run
run_binary_ablation    = True   #@param {type:"boolean"}
run_spectrum_ablation  = True   #@param {type:"boolean"}
run_domain_permutation = True   #@param {type:"boolean"}
run_topk_ablation      = True   #@param {type:"boolean"}

import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import r2_score, roc_auc_score
import os, warnings, copy
warnings.filterwarnings('ignore')

# ---
# DOMAIN REGISTRY (hardcoded from Cell 3 _obj_sub_t2)
# ---
# Each entry: short label → {full_name, vars (set of base names), ohe_bases}
# One-hot encoded bases (sex, L_site_id, etc.) are listed separately so
# the feature classifier can prefix-match their expanded column names.

_DOMAIN_REGISTRY = {
    'Obj No-Report': {
        'full_name': 'Objective No-Report (Cog Tasks, Neuro, Physio, Ances Genes)',
        'vars': {
            'birth_weight_p', 'desc_african_AFR_B', 'desc_alaska_native_AMR_B',
            'desc_asian_indian_SAS_B', 'desc_chinese_EAS_B', 'desc_european_EUR_B',
            'desc_japanese_EAS_B', 'desc_korean_EAS_B', 'desc_latin_B',
            'desc_native_american_AMR_B', 'desc_other_south_asian_SAS_B',
            'desc_vietnamese_EAS_B', 'fitbit_fairlyactive_mins',
            'fitbit_lightlyactive_mins', 'fitbit_resting_hr', 'fitbit_sedentary_mins',
            'fitbit_steps', 'fitbit_veryactive_mins', 'gd_riskybets', 'gd_safebets',
            'height', 'lmt_accuracy', 'lmt_correct_mrt', 'lmt_correct_nt',
            'lmt_efficiency', 'lmt_mrt', 'mid_acceptable_performance', 'mid_mrt_lgrw',
            'mid_mrt_smrw', 'mid_num_trials', 'mid_total_payout', 'nb2_D_prime_neg',
            'nb2_D_prime_pos', 'nb2_accuracy_neg', 'nb2_accuracy_pos',
            'nb2_resp_bias_neg', 'nb2_resp_bias_pos', 'nb_correct_mrt',
            'nb_correct_mrt_2back', 'nb_correct_mrt_neg', 'nb_correct_mrt_pos',
            'nb_correct_nt', 'nb_correct_nt_2back', 'nb_correct_nt_neg',
            'nb_correct_nt_pos', 'no_sports_activities_p',
            'pc_DTI1', 'pc_DTI10', 'pc_DTI11', 'pc_DTI12', 'pc_DTI13', 'pc_DTI14',
            'pc_DTI15', 'pc_DTI16', 'pc_DTI17', 'pc_DTI18', 'pc_DTI19', 'pc_DTI2',
            'pc_DTI20', 'pc_DTI21', 'pc_DTI22', 'pc_DTI23', 'pc_DTI24', 'pc_DTI25',
            'pc_DTI26', 'pc_DTI27', 'pc_DTI28', 'pc_DTI29', 'pc_DTI3', 'pc_DTI30',
            'pc_DTI31', 'pc_DTI32', 'pc_DTI33', 'pc_DTI34', 'pc_DTI35', 'pc_DTI36',
            'pc_DTI37', 'pc_DTI38', 'pc_DTI39', 'pc_DTI4', 'pc_DTI40', 'pc_DTI41',
            'pc_DTI42', 'pc_DTI43', 'pc_DTI44', 'pc_DTI45', 'pc_DTI46', 'pc_DTI47',
            'pc_DTI48', 'pc_DTI49', 'pc_DTI5', 'pc_DTI50', 'pc_DTI51', 'pc_DTI52',
            'pc_DTI53', 'pc_DTI54', 'pc_DTI55', 'pc_DTI56', 'pc_DTI57', 'pc_DTI58',
            'pc_DTI59', 'pc_DTI6', 'pc_DTI60', 'pc_DTI61', 'pc_DTI62', 'pc_DTI63',
            'pc_DTI64', 'pc_DTI65', 'pc_DTI66', 'pc_DTI67', 'pc_DTI68', 'pc_DTI69',
            'pc_DTI7', 'pc_DTI70', 'pc_DTI71', 'pc_DTI72', 'pc_DTI73', 'pc_DTI74',
            'pc_DTI75', 'pc_DTI8', 'pc_DTI9',
            'pc_gene_aces1', 'pc_gene_aces10', 'pc_gene_aces11', 'pc_gene_aces12',
            'pc_gene_aces13', 'pc_gene_aces14', 'pc_gene_aces15', 'pc_gene_aces16',
            'pc_gene_aces17', 'pc_gene_aces18', 'pc_gene_aces19', 'pc_gene_aces2',
            'pc_gene_aces20', 'pc_gene_aces21', 'pc_gene_aces22', 'pc_gene_aces23',
            'pc_gene_aces24', 'pc_gene_aces25', 'pc_gene_aces26', 'pc_gene_aces27',
            'pc_gene_aces28', 'pc_gene_aces29', 'pc_gene_aces3', 'pc_gene_aces30',
            'pc_gene_aces31', 'pc_gene_aces32', 'pc_gene_aces4', 'pc_gene_aces5',
            'pc_gene_aces6', 'pc_gene_aces7', 'pc_gene_aces8', 'pc_gene_aces9',
            'pc_rsFMRI1', 'pc_rsFMRI10', 'pc_rsFMRI11', 'pc_rsFMRI12', 'pc_rsFMRI13',
            'pc_rsFMRI14', 'pc_rsFMRI15', 'pc_rsFMRI16', 'pc_rsFMRI17', 'pc_rsFMRI18',
            'pc_rsFMRI19', 'pc_rsFMRI2', 'pc_rsFMRI20', 'pc_rsFMRI21', 'pc_rsFMRI22',
            'pc_rsFMRI23', 'pc_rsFMRI24', 'pc_rsFMRI25', 'pc_rsFMRI26', 'pc_rsFMRI27',
            'pc_rsFMRI28', 'pc_rsFMRI29', 'pc_rsFMRI3', 'pc_rsFMRI30', 'pc_rsFMRI31',
            'pc_rsFMRI32', 'pc_rsFMRI33', 'pc_rsFMRI34', 'pc_rsFMRI35', 'pc_rsFMRI36',
            'pc_rsFMRI37', 'pc_rsFMRI38', 'pc_rsFMRI39', 'pc_rsFMRI4', 'pc_rsFMRI40',
            'pc_rsFMRI41', 'pc_rsFMRI42', 'pc_rsFMRI43', 'pc_rsFMRI44', 'pc_rsFMRI45',
            'pc_rsFMRI46', 'pc_rsFMRI47', 'pc_rsFMRI48', 'pc_rsFMRI49', 'pc_rsFMRI5',
            'pc_rsFMRI50', 'pc_rsFMRI51', 'pc_rsFMRI52', 'pc_rsFMRI53', 'pc_rsFMRI54',
            'pc_rsFMRI55', 'pc_rsFMRI56', 'pc_rsFMRI57', 'pc_rsFMRI58', 'pc_rsFMRI59',
            'pc_rsFMRI6', 'pc_rsFMRI60', 'pc_rsFMRI61', 'pc_rsFMRI62', 'pc_rsFMRI63',
            'pc_rsFMRI64', 'pc_rsFMRI65', 'pc_rsFMRI66', 'pc_rsFMRI67', 'pc_rsFMRI68',
            'pc_rsFMRI69', 'pc_rsFMRI7', 'pc_rsFMRI70', 'pc_rsFMRI71', 'pc_rsFMRI72',
            'pc_rsFMRI73', 'pc_rsFMRI74', 'pc_rsFMRI75', 'pc_rsFMRI8', 'pc_rsFMRI9',
            'pc_struct1', 'pc_struct10', 'pc_struct11', 'pc_struct12', 'pc_struct13',
            'pc_struct14', 'pc_struct15', 'pc_struct16', 'pc_struct17', 'pc_struct18',
            'pc_struct19', 'pc_struct2', 'pc_struct20', 'pc_struct21', 'pc_struct22',
            'pc_struct23', 'pc_struct24', 'pc_struct25', 'pc_struct26', 'pc_struct27',
            'pc_struct28', 'pc_struct29', 'pc_struct3', 'pc_struct30', 'pc_struct31',
            'pc_struct32', 'pc_struct33', 'pc_struct34', 'pc_struct35', 'pc_struct36',
            'pc_struct37', 'pc_struct38', 'pc_struct39', 'pc_struct4', 'pc_struct40',
            'pc_struct41', 'pc_struct42', 'pc_struct43', 'pc_struct44', 'pc_struct45',
            'pc_struct46', 'pc_struct47', 'pc_struct48', 'pc_struct49', 'pc_struct5',
            'pc_struct50', 'pc_struct51', 'pc_struct52', 'pc_struct53', 'pc_struct54',
            'pc_struct55', 'pc_struct56', 'pc_struct57', 'pc_struct58', 'pc_struct59',
            'pc_struct6', 'pc_struct60', 'pc_struct61', 'pc_struct62', 'pc_struct63',
            'pc_struct64', 'pc_struct65', 'pc_struct66', 'pc_struct67', 'pc_struct68',
            'pc_struct69', 'pc_struct7', 'pc_struct70', 'pc_struct71', 'pc_struct72',
            'pc_struct73', 'pc_struct74', 'pc_struct75', 'pc_struct8', 'pc_struct9',
            'ravlt_l_intrusions', 'ravlt_l_repitition', 'ravlt_l_total',
            'ravlt_s_intrusions', 'ravlt_s_repitition', 'ravlt_s_total',
            'sst_aG1', 'sst_aG2', 'sst_aS', 'sst_absdeltaCV', 'sst_absdeltaRV',
            'sst_absdeltaSD', 'sst_acceptable_performance', 'sst_bG', 'sst_betasCV',
            'sst_betasSD', 'sst_betasV', 'sst_bs2CV', 'sst_bs2SD', 'sst_bs2V',
            'sst_dG', 'sst_dS', 'sst_factor1', 'sst_factor2', 'sst_factor3',
            'sst_gamma0', 'sst_gamma1', 'sst_kappa0', 'sst_mean_PDR', 'sst_mean_PDRg',
            'sst_mean_SD', 'sst_mean_SDr', 'sst_mean_absdelta', 'sst_mean_bS2',
            'sst_mean_betaS', 'sst_mean_ssrt', 'sst_median_PDR', 'sst_median_PDRg',
            'sst_median_SD', 'sst_median_SDr', 'sst_median_absdelta', 'sst_median_bS2',
            'sst_median_betaS', 'sst_median_ssrt', 'sst_mu', 'sst_pdrCV', 'sst_pdrSD',
            'sst_pdrV', 'sst_pdrgCV', 'sst_pdrgSD', 'sst_pdrgV', 'sst_pp', 'sst_sM',
            'sst_sdCV', 'sst_sdRV', 'sst_sdSD', 'sst_sdrCV', 'sst_sdrRV', 'sst_sdrSD',
            'sst_ssrt_int_est', 'sst_ssrt_mean_est', 'sst_theta0', 'sst_theta1',
            'sst_xM', 'tb_cardsort', 'tb_flanker', 'tb_list', 'tb_pattern',
            'tb_picture', 'tb_picvocab', 'tb_reading', 'waist', 'weight',
        },
        'ohe_bases': ['sex'],
    },
    'Obj Rated': {
        'full_name': 'Objective Rated (SES, Resid, Grades, Diet)',
        'vars': {
            'added_sugar', 'carbohydrate_intake', 'dairy_intake', 'fiber_intake',
            'fruit_intake', 'legume_intake', 'neighborhood_safe_y',
            'neighborhood_safety_ss_p', 'num_brothers_p', 'num_sisters_p',
            'parent_age', 'parent_education', 'parent_fails_to_pay_debts_p',
            'parent_financial_trouble_p', 'parent_income', 'parent_work_absences_p',
            'positive_finance_p', 'potassium_intake', 'protein_intake',
            'protein_sources_intake', 'relig_importance', 'religious_service_frequency',
            'resid_crime_drug', 'resid_crime_dui', 'resid_crime_tot',
            'resid_crime_violent', 'resid_density', 'resid_immigrant_bias',
            'resid_lead_risk', 'resid_lead_risk_houses_perc', 'resid_lead_risk_poverty',
            'resid_no2_avg', 'resid_pm25_avg', 'resid_prox_roads', 'resid_racism',
            'resid_sex_orient_bias', 'resid_sexism', 'resid_walkability',
            'saturated_fat', 'sodium_intake', 'struggle_food_expenses',
            'sugary_beverage_freq', 'total_calories', 'total_sugar',
            'vegetable_intake', 'whole_grain_intake',
        },
        'ohe_bases': ['L_site_id', 'child_religion', 'marital_status', 'sex_P'],
    },
    'Child Self': {
        'full_name': 'Child-Self Measures (School, Family, Social)',
        'vars': {
            'bis_behav_inhibition_ss_k', 'bis_drive_ss_k', 'bis_funseeking_ss_k',
            'bis_reward_responsive_ss_k', 'bullied_on_internet_k', 'child_newschool_p',
            'close_boy_friends_k', 'close_girl_friends_k', 'discrimination_ss_k',
            'disobeys_at_school_k', 'doesnt_feel_accepted_k', 'enjoys_school_k',
            'excluded_k', 'fam_criticize_often_k', 'fam_fight_often_k',
            'fam_hit_each_other_k', 'fam_keep_peace_k', 'fam_no_lose_temps_k',
            'fam_no_open_anger_k', 'fam_no_raise_voices_k', 'fam_throw_things_k',
            'fam_try_one_up_k', 'feels_discriminated_adults_not_school_k',
            'feels_discriminated_americans_k', 'feels_discriminated_k',
            'feels_discriminated_students_k', 'feels_discriminated_teachers_k',
            'feels_leftout_k', 'feels_smart_k', 'feels_threatned_k',
            'feels_unwanted_american_society_k', 'feelsafe_at_school_k',
            'finds_schoolboring_k', 'getalong_teachers_k', 'grades_important_k',
            'not_invited_k', 'otherkids_gossip_k', 'otherkids_saymeanthings_k',
            'otherkids_spreadneg_rumors_k', 'overt_aggression_ss_k',
            'overt_victimization_ss_k', 'p_comm_cohesion_ss',
            'p_comm_collective_efficacy_ss', 'p_comm_ctrl_ss',
            'peer_net_protective_ss_k', 'peers_beh_delinquent_ss_k',
            'peers_beh_prosocial_ss_k', 'prosocial_ss_k', 'prosocial_ss_p',
            'relational_aggression_ss_k', 'relational_victimization_ss_k',
            'repeated_grade', 'reputational_aggression_ss_k',
            'reputational_victimization_ss_k', 'saysmeanthings_others_k',
            'school_disengagement_ss_k', 'school_environment_ss_k',
            'school_involvement_ss_k', 'senses_racism_k',
            'socialinfluence_meanfinal_k', 'up_lackofplanning_ss_k',
            'up_lackperseverance_ss_k', 'up_negative_urgency_ss_k',
            'up_positiveurgency_ss_k', 'up_sensationseeking_ss_k',
        },
        'ohe_bases': [],
    },
    'Parent→Child': {
        'full_name': 'Parent-to-Child Measures (Psychopathology, Personality, Social)',
        'vars': {
            'argues_p', 'attacks_others_p', 'blames_others_p', 'body_aches_p',
            'bragadocious_p', 'breaks_rules_p', 'bullies_others_p',
            'cant_concentrate_p', 'clings_to_adults_p', 'constipated_p',
            'demands_attention_p', 'destroys_others_things_p', 'destroys_own_things_p',
            'disagreeable_p', 'disobedient_home_p', 'disobedient_school_p',
            'doesnt_finish_p', 'doesnt_get_along_p', 'easily_embarrassed_p',
            'easily_jealous_p', 'easily_offended_p', 'eye_problems_p',
            'fears_being_bad_p', 'fears_excl_school_p', 'fears_school_p', 'fights_p',
            'frequent_headaches_p', 'frequent_stomachaches_p', 'goal_continuity_p',
            'hyperactive_p', 'impulsive_p', 'loquacious_p', 'lying_p',
            'medhx_doctorvisit_p', 'medhx_emergencyroom_p', 'medhx_p', 'nausea_p',
            'nervous_general_p', 'nervous_twitching_p', 'not_critical_others_p',
            'not_liked_p', 'paranoid_p', 'perfectionist_p', 'scared_dark_p',
            'school_excitement_p', 'secretive_p', 'skin_problems_p', 'sociable_p',
            'social_fear_present_PK', 'steals_home_p', 'steals_outside_p',
            'stubborn_p', 'temper_tantrums_p', 'threatens_others_p', 'vomiting_p',
            'whines_p', 'wishes_other_sex_p', 'worries_p',
        },
        'ohe_bases': [],
    },
    'Adverse Life': {
        'full_name': 'Child & Parent Ratings (Adverse Life Events)',
        'vars': {
            'adult_family_fighting_victim_p', 'b_lifeevents_affected_ss_k',
            'b_lifeevents_affected_ss_p', 'b_lifeevents_ss_k', 'b_lifeevents_ss_p',
            'beating_victim_home_p', 'big_accident_need_treatment_p',
            'car_accident_hurt_p', 'domestic_child_sexually_abuse_victim_p',
            'experienced_crime_p', 'family_threatened_child_victim_p',
            'fire_victim_p', 'foreign_child_sexually_abuse_victim_p',
            'g_lifeevents_ss_k', 'g_lifeevents_ss_p', 'natural_disaster_victim_p',
            'peer_child_sexually_abuse_victim_p',
            'stabbing_shooting_victim_community_p',
            'stabbing_shooting_victim_home_p', 'stabbing_shooting_witness_p',
            'stranger_threatened_child_victim_p', 'sudden_death_in_family_p',
            'terrorism_victim_p', 'war_death_witness_p',
        },
        'ohe_bases': [],
    },
    'Parent Self (Psych)': {
        'full_name': 'Parent-Self Measures (Psychopathology, Personality, Social)',
        'vars': {
            'delta_parent_aches_pains_p', 'delta_parent_bad_family_relationship_p',
            'delta_parent_bad_relationships_p', 'delta_parent_concentration_trouble_p',
            'delta_parent_drinks_too_much_p', 'delta_parent_drug_use_p',
            'delta_parent_feels_overwhelmed_p', 'delta_parent_feels_unloved_p',
            'delta_parent_financial_failures_p', 'delta_parent_friendship_trouble_p',
            'delta_parent_meets_family_duties_p', 'delta_parent_not_liked_by_others_p',
            'delta_parent_planning_trouble_p', 'delta_parent_poor_work_performance_p',
            'delta_parent_sleep_trouble_p', 'delta_parent_somatic_problems_D_p',
            'delta_parent_stubborn_irritable_p', 'delta_parent_worries_a_lot_p',
            'delta_parent_worries_about_family_p',
            'parent_associates_with_trouble_p', 'parent_attention_seeking_p',
            'parent_avoids_talking_p', 'parent_bad_family_relationship_p',
            'parent_bad_opposite_sex_relationship_p', 'parent_bad_relationships_p',
            'parent_behavior_changeable_p', 'parent_bragging_p',
            'parent_clowns_or_shows_off_p', 'parent_clumsy_p',
            'parent_concentration_trouble_p', 'parent_confused_p',
            'parent_cries_a_lot_p', 'parent_decision_trouble_p', 'parent_depressed_p',
            'parent_destroys_others_things_p', 'parent_destroys_own_things_p',
            'parent_disorganized_p', 'parent_doesnt_eat_well_p',
            'parent_doesnt_finish_tasks_p', 'parent_drives_too_fast_p',
            'parent_easily_bored_p', 'parent_enjoys_little_p',
            'parent_fear_of_bad_thoughts_p', 'parent_fearful_or_anxious_p',
            'parent_feels_inferior_p', 'parent_feels_unloved_p',
            'parent_feels_unsuccessful_p', 'parent_forgetful_p',
            'parent_friendship_trouble_p', 'parent_happy_person_p',
            'parent_hears_voices_p', 'parent_heart_racing_p',
            'parent_high_sleep_duration_p', 'parent_honest_p', 'parent_hot_temper_p',
            'parent_hyperactive_p', 'parent_illegal_behavior_p', 'parent_impulsive_p',
            'parent_lonely_p', 'parent_loses_things_p', 'parent_louder_than_others_p',
            'parent_low_energy_p', 'parent_max_effort_p',
            'parent_meets_family_duties_p', 'parent_money_management_trouble_p',
            'parent_no_guilt_p', 'parent_not_good_at_details_p',
            'parent_not_liked_by_others_p', 'parent_numbness_p',
            'parent_obsessive_thoughts_p', 'parent_opposite_sex_wish_p',
            'parent_paranoid_p', 'parent_physical_attacks_p', 'parent_picks_skin_p',
            'parent_planning_trouble_p', 'parent_prefers_older_people_p',
            'parent_prefers_to_be_alone_p', 'parent_priority_trouble_p',
            'parent_relationship_concerns_p', 'parent_repeats_acts_p',
            'parent_restless_p', 'parent_risky_decisions_p', 'parent_secretive_p',
            'parent_sees_things_p', 'parent_self_conscious_p', 'parent_self_harm_p',
            'parent_sense_of_fairness_p', 'parent_shy_or_timid_p',
            'parent_sleep_trouble_p', 'parent_specific_fears_p',
            'parent_speech_problems_p', 'parent_stands_up_rights_p',
            'parent_strange_behavior_p', 'parent_strange_thoughts_p',
            'parent_stubborn_irritable_p', 'parent_sudden_mood_changes_p',
            'parent_suicidal_thoughts_p', 'parent_talks_too_much_p',
            'parent_tardy_often_p', 'parent_teases_others_p',
            'parent_tired_no_reason_p', 'parent_uses_opportunities_p',
            'parent_worries_a_lot_p', 'parent_worries_about_family_p',
            'parent_worries_about_future_p', 'parent_yells_a_lot_p',
        },
        'ohe_bases': [],
    },
    'Parent Self (Fam)': {
        'full_name': 'Parent-Self Measures (Family Style/Dynamics, Drug Use)',
        'vars': {
            'alcohol_before_pregnancy_p', 'alcohol_during_pregnancy_p',
            'caffeine_during_pregnancy_p', 'cigs_before_pregnancy_p',
            'cigs_during_pregnancy_p', 'cocaine_before_pregnancy_p',
            'cocaine_during_pregnancy_p', 'death_in_family_p',
            'drinksperweek_during_pregnancy_p', 'drugs_before_pregnancy_p',
            'drugs_during_pregnancy_p', 'family_activities_ss_p',
            'family_believe_not_raise_voice_p', 'family_conflict_ss_k',
            'family_conflict_ss_p', 'family_expression_ss_p',
            'family_intellectual_ss_p', 'family_lose_temper_rare_p', 'family_move_p',
            'family_not_talk_aboutfeelings_p', 'family_open_discussing_anything_p',
            'family_organisation_ss_p', 'family_peaceful_p', 'father_alcohol',
            'father_druguse', 'frequent_family_conflict_p',
            'hallucinogen_current_B_p', 'hallucinogen_use_history_B_p',
            'heroin_before_pregnancy_p', 'heroin_during_pregnancy_p',
            'mother_alcohol', 'mother_druguse', 'parent_drinks_frequency_p',
            'parent_drinks_too_much_p', 'parent_drug_days_nonmedical_p',
            'parent_drug_use_p', 'parent_drunk_days_p',
            'parent_family_responsibilities_p', 'parent_monitoring_ss_k',
            'parent_tobacco_use_frequency_p', 'parents_divorced_p',
            'prescriptionmed_pregnancy_p',
            'sedative_hypnotic_anxiolytic_use_B_p', 'weed_before_pregnancy_p',
            'weed_during_pregnancy_p',
        },
        'ohe_bases': [],
    },
    'Cross-Domain': {
        'full_name': 'Cross-Domain / Inter-Category Variables',
        'vars': {
            'b_lifeevents_affected_ss_k', 'b_lifeevents_ss_k', 'b_lifeevents_ss_p',
            'bad_grades', 'cct', 'child_white', 'close_boy_friends_k',
            'close_girl_friends_k', 'delta_not_liked_p', 'family_conflict_ss_k',
            'feels_leftout_k', 'gd_riskybets', 'mid_total_payout',
            'nb_correct_nt_2back', 'num_brothers_p', 'num_sisters_p',
            'parent_age_grouped', 'parent_income', 'puberty_k', 'relig_importance',
            'screentime_daysperweek_k', 'screentime_weekday_ss_k',
            'social_problems_D_p', 'sst_ssrt_mean_est', 'tb_cryst', 'weight',
        },
        'ohe_bases': ['L_site_id', 'child_religion', 'sex', 'sex_P',
                      'sex_orient_y', 'trans_id_y'],
    },
}

# Objective = first two categories; Subjective = all others
_OBJECTIVE_DOMAINS = {'Obj No-Report', 'Obj Rated'}
_SUBJECTIVE_DOMAINS = {k for k in _DOMAIN_REGISTRY if k not in _OBJECTIVE_DOMAINS}

# Domain colors (ordered from most objective → most subjective)
_DOMAIN_COLORS = {
    'Obj No-Report':       '#1565C0',
    'Obj Rated':           '#0097A7',
    'Child Self':          '#2E7D32',
    'Parent→Child':        '#F57F17',
    'Adverse Life':        '#E65100',
    'Parent Self (Psych)': '#AD1457',
    'Parent Self (Fam)':   '#6A1B9A',
    'Cross-Domain':        '#78909C',
    'Objective':           '#1565C0',
    'Subjective':          '#AD1457',
    'Combined (Full)':     '#37474F',
}

# ---
# FEATURE CLASSIFIER
# ---
# OHE collision map: sex_ must not match sex_orient_y_, trans_id_y_, etc.
_OHE_SAFE_PREFIXES = {
    'L_site_id':       'L_site_id_',
    'sex_P':           'sex_P_',
    'marital_status':  'marital_status_',
    'child_religion':  'child_religion_',
}
# sex needs special guard: sex_ but not sex_orient*
_OHE_SEX_PREFIX = 'sex_'
_OHE_SEX_EXCLUDE = ('sex_orient', 'sex_P')
# Subjective OHE bases
_OHE_SUBJ_PREFIXES = {
    'sex_orient_y':  'sex_orient_y_',
    'trans_id_y':    'trans_id_y_',
    'crysflu_c':     'crysflu_c_',
    'anxadhd_c':     'anxadhd_c_',
}

def _classify_feature(f):
    """Return the domain label for a feature column name.
    Handles one-hot expanded columns via prefix matching.
    Returns None if no domain claims the feature."""
    # 1. Exact match across all domains
    for domain, info in _DOMAIN_REGISTRY.items():
        if f in info['vars']:
            return domain
    # 2. OHE prefix match -- check each domain's ohe_bases
    for domain, info in _DOMAIN_REGISTRY.items():
        for base in info['ohe_bases']:
            if base == 'sex':
                if f.startswith(_OHE_SEX_PREFIX) and not f.startswith(_OHE_SEX_EXCLUDE):
                    return domain
            elif base in _OHE_SAFE_PREFIXES:
                if f.startswith(_OHE_SAFE_PREFIXES[base]):
                    return domain
            elif base in _OHE_SUBJ_PREFIXES:
                if f.startswith(_OHE_SUBJ_PREFIXES[base]):
                    return domain
    return None

def _build_feature_domain_map(feature_list):
    """Map every feature in feature_list to a domain. Returns dict[feature] → domain."""
    f2d = {}
    for f in feature_list:
        domain = _classify_feature(f)
        f2d[f] = domain if domain else 'Unclassified'
    return f2d

# ---
# ANALYTICS UTILITIES
# ---

def _bootstrap_delta_ci(y_true, y_pred_a, y_pred_b, metric_fn,
                        n_boot=1000, ci=0.95, seed=42):
    """Bootstrap CI for the difference metric(A) - metric(B), paired samples."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        try:
            sa = metric_fn(y_true[idx], y_pred_a[idx])
            sb = metric_fn(y_true[idx], y_pred_b[idx])
            deltas.append(sa - sb)
        except Exception:
            continue
    deltas = np.array(deltas)
    alpha = 1 - ci
    return np.median(deltas), np.percentile(deltas, 100*alpha/2), np.percentile(deltas, 100*(1-alpha/2))

def _domain_permutation_importance(model, X_test, y_test, f2d, metric_fn,
                                   is_clf=True, n_repeats=10, seed=42):
    """Permute all features in each domain, measure score drop."""
    rng = np.random.RandomState(seed)
    # Baseline score
    if is_clf:
        base_pred = model.predict_proba(X_test)[:, 1]
    else:
        base_pred = model.predict(X_test)
    base_score = metric_fn(y_test, base_pred)

    # Group features by domain
    domain_features = {}
    for f, d in f2d.items():
        if f in X_test.columns:
            domain_features.setdefault(d, []).append(f)

    results = {}
    for domain, feats in domain_features.items():
        drops = []
        for _ in range(n_repeats):
            X_perm = X_test.copy()
            for f in feats:
                X_perm[f] = rng.permutation(X_perm[f].values)
            if is_clf:
                perm_pred = model.predict_proba(X_perm)[:, 1]
            else:
                perm_pred = model.predict(X_perm)
            perm_score = metric_fn(y_test, perm_pred)
            drops.append(base_score - perm_score)
        results[domain] = {
            'mean_drop': np.mean(drops),
            'std_drop': np.std(drops, ddof=1),
            'drops': drops,
            'n_features': len(feats),
        }
    return base_score, results

# ---
# MAIN BODY FUNCTION
# ---

def _run_domain_ablation():
    """Run domain ablation analyses for CURRENT globals."""

    # -- Gate check ---
    _skip = False
    for _req in ['X_train', 'X_test', 'y_train', 'y_test']:
        if _req not in globals():
            print(f"Error: {_req} not found. Run the Main Analysis Runner first.")
            _skip = True
    if 'bootstrap_ci' not in globals():
        print("Warning:  bootstrap_ci not found -- ensure Cell 4 (Preprocessing) was executed.")
        _skip = True
    if _skip:
        return

    # -- Clear stale _RS sub-key ---
    _RS.pop('domain_ablation', None)

    # -- Detect within vs across-category mode ---
    _split = 'across_categories'
    if 'split' in globals() and isinstance(split, str):
        _split = split
    # Also check _batch_archive entry
    if '_vd_da' in dir() and isinstance(_vd_da, dict):
        _split = _vd_da.get('split', _split)

    _is_within = (_split == 'within_categories')
    if _is_within:
        print("  Warning: Within-category mode detected -- domain spectrum analyses will be")
        print("    skipped (feature set is from a single construct domain).")
        print("    Running: Top-K Feature Ablation only.\n")

    _target = target_options if 'target_options' in globals() else 'target'
    _tp = tp_option if 'tp_option' in globals() else '?'
    _task = detect_task_type(y_train)
    _is_clf = (_task == 'classification')
    _metric = 'AUC' if _is_clf else 'R²'
    _metric_fn = roc_auc_score if _is_clf else r2_score
    _y_tr = np.asarray(y_train).ravel()
    _y_te = np.asarray(y_test).ravel()
    _all_features = list(X_train.columns)

    print("="*72)
    print("   DOMAIN ABLATION & FEATURE-DOMAIN ANALYSIS")
    print(f"  Target: {_target}  |  Timepoint: T{_tp}  |  Task: {_task}")
    print("="*72)

    _analyses = []
    if run_binary_ablation and not _is_within:    _analyses.append('Binary')
    if run_spectrum_ablation and not _is_within:  _analyses.append('Spectrum')
    if run_domain_permutation and not _is_within: _analyses.append('Permutation')
    if run_topk_ablation:      _analyses.append('Top-K')
    print(f"  Running: {', '.join(_analyses) if _analyses else 'None'}")
    print(f"  Features: {len(_all_features)}  |  N_train: {len(X_train)}  |  N_test: {len(X_test)}")

    # -- Build feature→domain map ---
    _f2d = _build_feature_domain_map(_all_features)
    _domain_counts = {}
    for f, d in _f2d.items():
        _domain_counts[d] = _domain_counts.get(d, 0) + 1
    print(f"\n  Feature-domain mapping:")
    for d in list(_DOMAIN_REGISTRY.keys()) + ['Unclassified']:
        if d in _domain_counts:
            print(f"    {d:>22s}: {_domain_counts[d]:>4d} features")

    # Collect features by binary bucket
    _obj_features = [f for f, d in _f2d.items() if d in _OBJECTIVE_DOMAINS]
    _sub_features = [f for f, d in _f2d.items() if d in _SUBJECTIVE_DOMAINS or d == 'Unclassified']

    # -- Storage dicts ---
    _binary_results = {}
    _spectrum_results = {}
    _perm_results = {}
    _topk_results = {}
    _delta_cis = {}
    _predictions = {}  # store predictions for bootstrap delta

# ---
    # 1. BINARY ABLATION (Obj vs Subj vs Full) + Bootstrap Δ CIs
# ---
    if run_binary_ablation and not _is_within:
        display(HTML(
            '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #7B1FA2;'
            'background:linear-gradient(135deg,#F3E5F5,#F5F5F5);padding:14px 20px;'
            'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
            '<span style="font-size:15px;font-weight:700;color:#7B1FA2;">Binary Domain Ablation</span><br>'
            '<span style="font-size:12px;color:#546E7A;">Objective vs Subjective vs Combined (Full) -- with bootstrap Δ CIs</span>'
            '</div>'
        ))

        for _label, _feat_list in [
            ('Subjective', _sub_features),
            ('Objective', _obj_features),
            ('Combined (Full)', _all_features),
        ]:
            if len(_feat_list) < 2:
                print(f"  Warning: {_label}: only {len(_feat_list)} features -- skipping")
                continue

            _X_tr_sub = X_train[[f for f in _feat_list if f in X_train.columns]]
            _X_te_sub = X_test[[f for f in _feat_list if f in X_test.columns]]

            _m = create_lightweight_model(is_classification=_is_clf, n_features=len(_feat_list))
            _m.fit(_X_tr_sub, _y_tr)

            if _is_clf:
                _pred = _m.predict_proba(_X_te_sub)[:, 1]
            else:
                _pred = _m.predict(_X_te_sub)

            _score = _metric_fn(_y_te, _pred)
            _point, _lo, _hi = bootstrap_ci(_y_te, _pred, _metric_fn, n_boot=1000, seed=42)

            _binary_results[_label] = {
                'score': _score, 'ci_lo': _lo, 'ci_hi': _hi,
                'n_features': _X_tr_sub.shape[1]
            }
            _predictions[_label] = _pred.copy()
            print(f"  {_label:>20s}: {_metric} = {_score:.4f} [{_lo:.4f}, {_hi:.4f}]  ({_X_tr_sub.shape[1]} features)")

        # -- Bootstrap Δ CIs (paired) ---
        if 'Combined (Full)' in _predictions and 'Subjective' in _predictions:
            _d_med, _d_lo, _d_hi = _bootstrap_delta_ci(
                _y_te, _predictions['Combined (Full)'], _predictions['Subjective'],
                _metric_fn, n_boot=1000, seed=42)
            _delta_cis['Full_vs_Subj'] = {'median': _d_med, 'ci_lo': _d_lo, 'ci_hi': _d_hi}
            print(f"\n  Δ (Full − Subj):  {_d_med:+.4f} [{_d_lo:+.4f}, {_d_hi:+.4f}]")

        if 'Combined (Full)' in _predictions and 'Objective' in _predictions:
            _d_med, _d_lo, _d_hi = _bootstrap_delta_ci(
                _y_te, _predictions['Combined (Full)'], _predictions['Objective'],
                _metric_fn, n_boot=1000, seed=42)
            _delta_cis['Full_vs_Obj'] = {'median': _d_med, 'ci_lo': _d_lo, 'ci_hi': _d_hi}
            print(f"  Δ (Full − Obj):   {_d_med:+.4f} [{_d_lo:+.4f}, {_d_hi:+.4f}]")

        if 'Subjective' in _predictions and 'Objective' in _predictions:
            _d_med, _d_lo, _d_hi = _bootstrap_delta_ci(
                _y_te, _predictions['Subjective'], _predictions['Objective'],
                _metric_fn, n_boot=1000, seed=42)
            _delta_cis['Subj_vs_Obj'] = {'median': _d_med, 'ci_lo': _d_lo, 'ci_hi': _d_hi}
            print(f"  Δ (Subj − Obj):   {_d_med:+.4f} [{_d_lo:+.4f}, {_d_hi:+.4f}]")

        # -- Figure 1: Horizontal lollipop ---
        if len(_binary_results) >= 2:
            fig, ax = plt.subplots(figsize=(10, 3.5), dpi=300)
            _labels_sorted = ['Combined (Full)', 'Subjective', 'Objective']
            _labels_sorted = [l for l in _labels_sorted if l in _binary_results]
            _y_pos = list(range(len(_labels_sorted)))

            for _i, _lab in enumerate(_labels_sorted):
                _r = _binary_results[_lab]
                _c = _DOMAIN_COLORS.get(_lab, '#78909C')
                ax.barh(_i, _r['score'], height=0.45, color=_c, alpha=0.75, zorder=3,
                        edgecolor='white', linewidth=0.6)
                ax.errorbar(_r['score'], _i, xerr=[[_r['score']-_r['ci_lo']], [_r['ci_hi']-_r['score']]],
                           fmt='o', color=_c, markersize=7, markeredgecolor='white',
                           markeredgewidth=1.2, capsize=4, capthick=1.0, ecolor='#333333', zorder=4)
                ax.text(_r['ci_hi'] + 0.008, _i,
                       f"{_r['score']:.4f}  [{_r['ci_lo']:.3f}, {_r['ci_hi']:.3f}]  ({_r['n_features']} feat.)",
                       va='center', fontsize=8.5, fontweight='bold', color=_c)

            # Delta bracket annotations
            if 'Subj_vs_Obj' in _delta_cis:
                _dc = _delta_cis['Subj_vs_Obj']
                _sig = '●' if _dc['ci_lo'] > 0 or _dc['ci_hi'] < 0 else '○'
                ax.annotate(f"Δ = {_dc['median']:+.4f} [{_dc['ci_lo']:+.3f}, {_dc['ci_hi']:+.3f}] {_sig}",
                           xy=(0.02, 0.02), xycoords='axes fraction', fontsize=7.5,
                           color='#546E7A', fontstyle='italic')

            ax.set_yticks(_y_pos)
            ax.set_yticklabels(_labels_sorted, fontsize=9.5, fontweight='bold')
            ax.set_xlabel(_metric, fontsize=9)
            ax.set_title(f'Binary Domain Ablation -- {_target} (T{_tp})', fontsize=10, fontweight='bold')
            ax.set_xlim(left=max(0, min(r['ci_lo'] for r in _binary_results.values()) - 0.05))
            ax.invert_yaxis()
            ax.grid(axis='x', alpha=0.15, linewidth=0.5)
            for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
            plt.tight_layout()

            try:
                _fig_dir = os.path.join(os.getcwd(), 'domain_ablation_figures')
                os.makedirs(_fig_dir, exist_ok=True)
                fig.savefig(os.path.join(_fig_dir, f'binary_ablation_{_target}_T{_tp}.png'),
                            dpi=300, bbox_inches='tight')
                fig.savefig(os.path.join(_fig_dir, f'binary_ablation_{_target}_T{_tp}.pdf'),
                            dpi=300, bbox_inches='tight')
            except Exception as _e:
                print(f"Warning: {_e}")
            plt.show()

# ---
    # 2. 8-CATEGORY SPECTRUM ABLATION
# ---
    if run_spectrum_ablation and not _is_within:
        display(HTML(
            '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #0097A7;'
            'background:linear-gradient(135deg,#E0F7FA,#F5F5F5);padding:14px 20px;'
            'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
            '<span style="font-size:15px;font-weight:700;color:#0097A7;">8-Category Spectrum Ablation</span><br>'
            '<span style="font-size:12px;color:#546E7A;">Each domain independently -- Standalone predictive performance</span>'
            '</div>'
        ))

        for _dom in _DOMAIN_REGISTRY:
            _dom_feats = [f for f, d in _f2d.items() if d == _dom]
            if len(_dom_feats) < 2:
                print(f"  {_dom:>22s}: Warning: only {len(_dom_feats)} features -- skipping")
                _spectrum_results[_dom] = {'score': None, 'n_features': len(_dom_feats), 'skipped': True}
                continue

            _X_tr_d = X_train[[f for f in _dom_feats if f in X_train.columns]]
            _X_te_d = X_test[[f for f in _dom_feats if f in X_test.columns]]

            _md = create_lightweight_model(is_classification=_is_clf, n_features=len(_dom_feats))
            _md.fit(_X_tr_d, _y_tr)
            if _is_clf:
                _pred_d = _md.predict_proba(_X_te_d)[:, 1]
            else:
                _pred_d = _md.predict(_X_te_d)

            _score_d = _metric_fn(_y_te, _pred_d)
            _pt, _lo, _hi = bootstrap_ci(_y_te, _pred_d, _metric_fn, n_boot=500, seed=42)

            _spectrum_results[_dom] = {
                'score': _score_d, 'ci_lo': _lo, 'ci_hi': _hi,
                'n_features': _X_tr_d.shape[1]
            }
            print(f"  {_dom:>22s}: {_metric} = {_score_d:.4f} [{_lo:.4f}, {_hi:.4f}]  ({_X_tr_d.shape[1]} feat.)")

        # -- Figure 2: Spectrum lollipop (sorted by score) ---
        _valid_spectrum = {k: v for k, v in _spectrum_results.items()
                          if v.get('score') is not None and not v.get('skipped')}
        if len(_valid_spectrum) >= 2:
            _sorted_doms = sorted(_valid_spectrum.keys(),
                                  key=lambda d: _valid_spectrum[d]['score'], reverse=True)

            fig, ax = plt.subplots(figsize=(10, max(4, len(_sorted_doms)*0.55 + 1)), dpi=300)
            _y_pos = list(range(len(_sorted_doms)))

            for _i, _dom in enumerate(_sorted_doms):
                _r = _valid_spectrum[_dom]
                _c = _DOMAIN_COLORS.get(_dom, '#78909C')
                ax.barh(_i, _r['score'], height=0.5, color=_c, alpha=0.6, zorder=2,
                        edgecolor='white', linewidth=0.5)
                ax.errorbar(_r['score'], _i,
                           xerr=[[_r['score']-_r['ci_lo']], [_r['ci_hi']-_r['score']]],
                           fmt='o', color=_c, markersize=6, markeredgecolor='white',
                           markeredgewidth=1.0, capsize=3, capthick=0.8, ecolor='#444444', zorder=3)
                ax.text(_r['ci_hi'] + 0.008, _i,
                       f"{_r['score']:.4f}  ({_r['n_features']})",
                       va='center', fontsize=8, color=_c, fontweight='bold')

            # Full model reference line
            if 'Combined (Full)' in _binary_results:
                _full_s = _binary_results['Combined (Full)']['score']
                ax.axvline(x=_full_s, color='#37474F', linestyle='--', linewidth=1.2, alpha=0.6, zorder=1)
                ax.text(_full_s + 0.005, len(_sorted_doms) - 0.3,
                       f'Full: {_full_s:.4f}', fontsize=7.5, color='#37474F', fontstyle='italic')

            ax.set_yticks(_y_pos)
            ax.set_yticklabels(_sorted_doms, fontsize=9, fontweight='bold')
            ax.set_xlabel(f'Standalone {_metric}', fontsize=9)
            ax.set_title(f'8-Category Spectrum Ablation -- {_target} (T{_tp})', fontsize=10, fontweight='bold')
            _min_score = min(r['ci_lo'] for r in _valid_spectrum.values())
            ax.set_xlim(left=max(0, _min_score - 0.05))
            ax.invert_yaxis()
            ax.grid(axis='x', alpha=0.15, linewidth=0.5)
            for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
            plt.tight_layout()

            try:
                _fig_dir = os.path.join(os.getcwd(), 'domain_ablation_figures')
                os.makedirs(_fig_dir, exist_ok=True)
                fig.savefig(os.path.join(_fig_dir, f'spectrum_ablation_{_target}_T{_tp}.png'),
                            dpi=300, bbox_inches='tight')
                fig.savefig(os.path.join(_fig_dir, f'spectrum_ablation_{_target}_T{_tp}.pdf'),
                            dpi=300, bbox_inches='tight')
            except Exception as _e:
                print(f"Warning: {_e}")
            plt.show()

# ---
    # 3. DOMAIN PERMUTATION IMPORTANCE
# ---
    if run_domain_permutation and not _is_within:
        display(HTML(
            '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #E65100;'
            'background:linear-gradient(135deg,#FFF3E0,#F5F5F5);padding:14px 20px;'
            'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
            '<span style="font-size:15px;font-weight:700;color:#E65100;">Domain Permutation Importance</span><br>'
            '<span style="font-size:12px;color:#546E7A;">Shuffle each domain within the full model -- marginal contribution in context</span>'
            '</div>'
        ))

        # Train full model
        _full_model = create_lightweight_model(is_classification=_is_clf, n_features=len(_all_features))
        _full_model.fit(X_train, _y_tr)

        _base_score, _perm_raw = _domain_permutation_importance(
            _full_model, X_test, _y_te, _f2d, _metric_fn,
            is_clf=_is_clf, n_repeats=10, seed=42
        )

        print(f"\n  Full model {_metric} (baseline): {_base_score:.4f}")
        print(f"  Domain permutation importance (Δ{_metric} on shuffle):\n")

        _sorted_perm = sorted(_perm_raw.items(), key=lambda x: x[1]['mean_drop'], reverse=True)
        for _dom, _pr in _sorted_perm:
            _perm_results[_dom] = _pr
            _sig = '***' if _pr['mean_drop'] > 2*_pr['std_drop'] else ('*' if _pr['mean_drop'] > _pr['std_drop'] else '')
            print(f"    {_dom:>22s}: Δ{_metric} = {_pr['mean_drop']:+.4f} ± {_pr['std_drop']:.4f}  "
                  f"({_pr['n_features']} feat.) {_sig}")

        # -- Figure 3: Permutation importance lollipop ---
        if len(_perm_results) >= 2:
            _sorted_doms_p = [d for d, _ in _sorted_perm if d != 'Unclassified']

            fig, ax = plt.subplots(figsize=(10, max(4, len(_sorted_doms_p)*0.55 + 1)), dpi=300)
            _y_pos = list(range(len(_sorted_doms_p)))

            for _i, _dom in enumerate(_sorted_doms_p):
                _pr = _perm_results[_dom]
                _c = _DOMAIN_COLORS.get(_dom, '#78909C')
                _drop = _pr['mean_drop']
                ax.barh(_i, _drop, height=0.5, color=_c, alpha=0.6, zorder=2,
                        edgecolor='white', linewidth=0.5)
                ax.errorbar(_drop, _i, xerr=_pr['std_drop'],
                           fmt='o', color=_c, markersize=6, markeredgecolor='white',
                           markeredgewidth=1.0, capsize=3, capthick=0.8, ecolor='#444444', zorder=3)
                _x_ann = max(_drop + _pr['std_drop'], 0) + 0.003
                ax.text(_x_ann, _i,
                       f"Δ = {_drop:+.4f}  ({_pr['n_features']})",
                       va='center', fontsize=8, color=_c, fontweight='bold')

            ax.axvline(x=0, color='#999999', linewidth=0.8, zorder=1)
            _max_drop = max(r['mean_drop'] + r['std_drop'] for r in _perm_results.values())
            ax.set_xlim(left=min(-0.005, -_max_drop*0.05))
            ax.set_yticks(_y_pos)
            ax.set_yticklabels(_sorted_doms_p, fontsize=9, fontweight='bold')
            ax.set_xlabel(f'Δ{_metric} (drop on permutation)', fontsize=9)
            ax.set_title(f'Domain Permutation Importance -- {_target} (T{_tp})', fontsize=10, fontweight='bold')
            ax.invert_yaxis()
            ax.grid(axis='x', alpha=0.15, linewidth=0.5)
            for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
            plt.tight_layout()

            try:
                _fig_dir = os.path.join(os.getcwd(), 'domain_ablation_figures')
                os.makedirs(_fig_dir, exist_ok=True)
                fig.savefig(os.path.join(_fig_dir, f'domain_permutation_{_target}_T{_tp}.png'),
                            dpi=300, bbox_inches='tight')
                fig.savefig(os.path.join(_fig_dir, f'domain_permutation_{_target}_T{_tp}.pdf'),
                            dpi=300, bbox_inches='tight')
            except Exception as _e:
                print(f"Warning: {_e}")
            plt.show()

# ---
    # 4. TOP-K FEATURE ABLATION CURVE
# ---
    if run_topk_ablation:
        display(HTML(
            '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #2E7D32;'
            'background:linear-gradient(135deg,#E8F5E9,#F5F5F5);padding:14px 20px;'
            'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
            '<span style="font-size:15px;font-weight:700;color:#2E7D32;">Top-K Feature Ablation Curve</span><br>'
            '<span style="font-size:12px;color:#546E7A;">Remove top-ranked features progressively -- how concentrated is the signal?</span>'
            '</div>'
        ))

        # Train full model and get feature importances
        _full_model_tk = create_lightweight_model(is_classification=_is_clf, n_features=len(_all_features))
        _full_model_tk.fit(X_train, _y_tr)

        _feat_imp = None
        if hasattr(_full_model_tk, 'feature_importances_'):
            _feat_imp = pd.Series(_full_model_tk.feature_importances_, index=X_train.columns)
        elif hasattr(_full_model_tk, 'get_feature_importance'):
            _feat_imp = pd.Series(_full_model_tk.get_feature_importance(), index=X_train.columns)

        if _feat_imp is not None:
            _feat_imp_sorted = _feat_imp.sort_values(ascending=False)

            # Full model baseline
            if _is_clf:
                _full_pred_tk = _full_model_tk.predict_proba(X_test)[:, 1]
            else:
                _full_pred_tk = _full_model_tk.predict(X_test)
            _full_score_tk = _metric_fn(_y_te, _full_pred_tk)

            _ks = [0, 1, 5, 10, 20, 50, 100, min(200, len(_all_features) // 2)]
            _ks = sorted(set(k for k in _ks if k < len(_all_features)))

            print(f"\n  Full model {_metric}: {_full_score_tk:.4f} ({len(_all_features)} features)")
            print(f"  Removing top-K features by importance:\n")

            _topk_curve = []
            for _k in _ks:
                if _k == 0:
                    _topk_curve.append({'k': 0, 'score': _full_score_tk,
                                       'n_remaining': len(_all_features)})
                    continue

                _drop_feats = _feat_imp_sorted.index[:_k].tolist()
                _keep_feats = [f for f in _all_features if f not in _drop_feats]

                if len(_keep_feats) < 2:
                    break

                _X_tr_k = X_train[_keep_feats]
                _X_te_k = X_test[_keep_feats]
                _mk = create_lightweight_model(is_classification=_is_clf, n_features=len(_keep_feats))
                _mk.fit(_X_tr_k, _y_tr)
                if _is_clf:
                    _pred_k = _mk.predict_proba(_X_te_k)[:, 1]
                else:
                    _pred_k = _mk.predict(_X_te_k)
                _score_k = _metric_fn(_y_te, _pred_k)

                _topk_curve.append({'k': _k, 'score': _score_k,
                                   'n_remaining': len(_keep_feats)})
                _delta_k = _score_k - _full_score_tk
                print(f"    Remove top-{_k:>3d}: {_metric} = {_score_k:.4f}  "
                      f"(Δ = {_delta_k:+.4f}, {len(_keep_feats)} remaining)")

                # Domain composition of removed features
                _removed_domains = {}
                for _rf in _drop_feats:
                    _rd = _f2d.get(_rf, 'Unclassified')
                    _removed_domains[_rd] = _removed_domains.get(_rd, 0) + 1

            _topk_results = _topk_curve

            # -- Figure 4: Ablation curve ---
            if len(_topk_curve) >= 3:
                _df_tk = pd.DataFrame(_topk_curve)
                fig, ax = plt.subplots(figsize=(9, 5), dpi=300)

                ax.plot(_df_tk['k'], _df_tk['score'], 'o-', color='#2E7D32',
                       markersize=6, linewidth=1.8, markeredgecolor='white',
                       markeredgewidth=1.0, zorder=3)
                ax.fill_between(_df_tk['k'], _df_tk['score'], _full_score_tk,
                               alpha=0.08, color='#C62828', zorder=1)
                ax.axhline(y=_full_score_tk, color='#37474F', linestyle='--',
                          linewidth=1.2, alpha=0.5, zorder=2,
                          label=f'Full model ({_full_score_tk:.4f})')

                for _, _row in _df_tk.iterrows():
                    if _row['k'] > 0:
                        _delta_ann = _row['score'] - _full_score_tk
                        ax.annotate(f"{_row['score']:.3f}\n(Δ{_delta_ann:+.3f})",
                                   xy=(_row['k'], _row['score']),
                                   textcoords='offset points', xytext=(8, -12),
                                   fontsize=7.5, color='#2E7D32', fontweight='bold')

                # Top-5 feature names as annotation
                _top5 = _feat_imp_sorted.index[:5].tolist()
                _top5_str = '\n'.join([f'{i+1}. {f}' for i, f in enumerate(_top5)])
                ax.text(0.98, 0.98, f'Top 5 features:\n{_top5_str}',
                       transform=ax.transAxes, fontsize=7, va='top', ha='right',
                       color='#546E7A', fontstyle='italic',
                       bbox=dict(boxstyle='round,pad=0.4', facecolor='#f8f8f8',
                                alpha=0.9, edgecolor='#cccccc', linewidth=0.5))

                ax.set_xlabel('Number of top features removed', fontsize=9)
                ax.set_ylabel(_metric, fontsize=9)
                ax.set_title(f'Top-K Feature Ablation -- {_target} (T{_tp})', fontsize=10, fontweight='bold')
                ax.legend(fontsize=8, loc='lower left', framealpha=0.9, edgecolor='#cccccc')
                ax.grid(axis='y', alpha=0.15, linewidth=0.5)
                for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
                plt.tight_layout()

                try:
                    _fig_dir = os.path.join(os.getcwd(), 'domain_ablation_figures')
                    os.makedirs(_fig_dir, exist_ok=True)
                    fig.savefig(os.path.join(_fig_dir, f'topk_ablation_{_target}_T{_tp}.png'),
                                dpi=300, bbox_inches='tight')
                    fig.savefig(os.path.join(_fig_dir, f'topk_ablation_{_target}_T{_tp}.pdf'),
                                dpi=300, bbox_inches='tight')
                except Exception as _e:
                    print(f"Warning: {_e}")
                plt.show()
        else:
            print("  Warning: Could not extract feature importances from lightweight model.")

# ---
    # CONSOLIDATED SUMMARY DASHBOARD
# ---
    _summary_rows = []
    if _binary_results:
        for _lab, _r in _binary_results.items():
            _summary_rows.append({
                'Analysis': 'Binary', 'Domain': _lab,
                _metric: f"{_r['score']:.4f}", 'CI': f"[{_r['ci_lo']:.3f}, {_r['ci_hi']:.3f}]",
                'N Feat': _r['n_features'],
            })
    if _spectrum_results:
        for _dom, _r in sorted(_spectrum_results.items(),
                                key=lambda x: x[1].get('score', 0) or 0, reverse=True):
            if _r.get('score') is not None:
                _summary_rows.append({
                    'Analysis': 'Spectrum', 'Domain': _dom,
                    _metric: f"{_r['score']:.4f}", 'CI': f"[{_r['ci_lo']:.3f}, {_r['ci_hi']:.3f}]",
                    'N Feat': _r['n_features'],
                })
    if _perm_results:
        for _dom, _pr in sorted(_perm_results.items(),
                                 key=lambda x: x[1]['mean_drop'], reverse=True):
            _summary_rows.append({
                'Analysis': 'Permutation', 'Domain': _dom,
                _metric: f"Δ = {_pr['mean_drop']:+.4f}", 'CI': f"± {_pr['std_drop']:.4f}",
                'N Feat': _pr['n_features'],
            })
    if _delta_cis:
        for _cmp, _dc in _delta_cis.items():
            _sig = '●' if _dc['ci_lo'] > 0 or _dc['ci_hi'] < 0 else '○'
            _summary_rows.append({
                'Analysis': 'Δ Bootstrap', 'Domain': _cmp.replace('_', ' '),
                _metric: f"Δ = {_dc['median']:+.4f}", 'CI': f"[{_dc['ci_lo']:+.3f}, {_dc['ci_hi']:+.3f}] {_sig}",
                'N Feat': '--',
            })

    if _summary_rows:
        _sdf = pd.DataFrame(_summary_rows)
        display(HTML(
            '<div style="font-family:sans-serif;max-width:800px;margin:20px 0;">'
            '<div style="background:#37474F;color:white;padding:10px 16px;'
            'font-size:13px;font-weight:700;border-radius:6px 6px 0 0;">'
            f'Domain Ablation Summary -- {_target} (T{_tp})</div>'
            '</div>'
        ))
        display(_sdf.style.set_properties(**{
            'font-size': '12px', 'text-align': 'center',
        }).set_table_styles([
            {'selector': 'th', 'props': [('font-size', '11px'), ('background', '#ECEFF1'),
                                          ('text-align', 'center'), ('padding', '6px 10px')]},
            {'selector': 'td', 'props': [('padding', '5px 10px')]},
        ]))

# ---
    # STORE RESULTS (backward-compatible with Temporal Gradient cell)
# ---
    _RS['domain_ablation'] = {
        # Temporal Gradient cell reads this key -- backward compatible
        'results': {k: v for k, v in _binary_results.items()},
        'target': _target,
        'metric': _metric,
        'spectrum_results': {k: v for k, v in _spectrum_results.items()},
        'permutation_results': {k: v for k, v in _perm_results.items()},
        'topk_results': _topk_results,
        'delta_cis': _delta_cis,
        'feature_domain_map_counts': _domain_counts,
        'split': _split,
    }

    print(f"\n  Domain ablation complete. Results stored in _RS['domain_ablation'].")

# ---
# BATCH DISPATCH
# ---
_is_bv_da = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_da:
    try:
        _cbv_da = cache_load('main_runner')
        if (_cbv_da and isinstance(_cbv_da, dict)
                and _cbv_da.get('_batch_archive')
                and len(_cbv_da['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_da['_batch_archive'].values())):
            _batch_archive = _cbv_da['_batch_archive']
            _is_bv_da = True
    except Exception:
        pass  # non-critical; logged upstream or optional visualization

if _is_bv_da:
    _vgs_da = {k: globals().get(k) for k in [
        'X_train', 'X_test', 'y_train', 'y_test', 'target_options', 'tp_option',
        'metrics_dict_full', 'exported_within_dict', 'consensus_df',
        'model_weights', 'split', 'last_fitted_model',
        'sample_valid', 'sample_valid_test', 'sample_valid_train',
        'results_summary',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH -- Domain Ablation: processing {len(_batch_archive)} targets")
    print('=' * 72)

    for _vi_da, (_vk_da, _vd_da) in enumerate(_batch_archive.items()):
        _vl_da = _vd_da.get('target', _vk_da)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_da+1}/{len(_batch_archive)}] Target: {_vl_da}")
        print(f"{'-' * 68}")
        for _gk in ['X_train', 'X_test', 'y_train', 'y_test', 'metrics_dict_full',
                    'exported_within_dict', 'consensus_df', 'model_weights', 'split',
                    'last_fitted_model', 'results_summary',
                    'sample_valid', 'sample_valid_test', 'sample_valid_train']:
            if _vd_da.get(_gk) is not None:
                globals()[_gk] = _vd_da[_gk]
        globals()['target_options'] = _vl_da
        if 'timepoint' in _vd_da:
            globals()['tp_option'] = _vd_da['timepoint']
        if 'split' in _vd_da:
            globals()['split'] = _vd_da['split']
        try:
            _run_domain_ablation()
        except Exception as _e_da:
            print(f"  Warning:  Error for {_vl_da}: {_e_da}")
            import traceback; traceback.print_exc()
        # -- Archive results back into _batch_archive --
        _rs_ref = globals().get('_RS')
        if _vk_da in _batch_archive and _rs_ref is not None:
            if 'domain_ablation' in _rs_ref:
                _batch_archive[_vk_da]['_RS_domain_ablation'] = copy.deepcopy(_rs_ref['domain_ablation'])

    for _k, _v in _vgs_da.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE -- Domain Ablation: {len(_batch_archive)} targets")
    print('=' * 72)
else:
    _run_domain_ablation()

In [ ]:
#@title V | Cross-Timepoint Performance Trajectories

# Sweeps T0 to T4, running cross-validated models at each wave to produce
# developmental prediction curves. Tracks how model performance evolves
# as participants age through the ABCD Study (ages 9-13).
#
# Each timepoint uses all available features at that wave (not a common
# intersection), so R^2 reflects real-world predictive capacity at each
# age. Feature counts are reported per timepoint for transparency.
#
# Dependencies: Variable Registry, Circularity Exclusions, Preprocessing,
# Stacking Architecture, Main Analysis Runner.
# ---

import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.impute import SimpleImputer
from scipy import stats
import os, warnings, copy
warnings.filterwarnings('ignore')

# -- Target colours / labels (shared with the Temporal Gradient cell if available) ---
_TP_TRAJ_COLORS = {
    'depress_D_p': '#1565C0', 'sbt_core_depression': '#4169E1',
    'anxdisord_D_p': '#C62828', 'sbt_core_anxiety': '#D32F2F',
    'adhd_D_p': '#E6A817', 'social_problems_D_p': '#E65100',
    'internal_D_p': '#6A1B9A', 'external_D_p': '#2E7D32',
    'parent_depress_D_p': '#0D47A1', 'parent_depressed_p': '#191970',
    'parent_anxdisord_D_p': '#B71C1C', 'parent_adhd_D_p': '#F57F17',
    'parent_external_D_p': '#1B5E20', 'parent_internal_D_p': '#4A148C',
    'sbt_anxiety_depression': '#DA70D6', 'sbt_guilt_hopelessness': '#4B0082',
    'sbt_fatigue_sleep': '#6A5ACD', 'sbt_somatic_depression': '#BC8F8F',
    'sbt_emotional_dysregulation': '#FF4500', 'sbt_social_withdrawal': '#5F9EA0',
    'sbt_avoidance_fear': '#9370DB', 'sbt_aggression_irritability': '#B22222',
    'sbt_academic_cognitive': '#B8960B', 'sbt_perfectionism_achievement': '#C44A80',
}
_TP_TRAJ_LABELS = {
    'depress_D_p': 'Depression', 'sbt_core_depression': 'Depression (Primary)',
    'anxdisord_D_p': 'Anxiety', 'sbt_core_anxiety': 'Anxiety (Primary)',
    'adhd_D_p': 'ADHD', 'social_problems_D_p': 'Social Problems',
    'internal_D_p': 'Internalizing', 'external_D_p': 'Externalizing',
    'parent_depress_D_p': 'Parent Depression', 'parent_depressed_p': 'Parent Depression',
    'parent_anxdisord_D_p': 'Parent Anxiety', 'parent_adhd_D_p': 'Parent ADHD',
    'parent_external_D_p': 'Parent Externalizing', 'parent_internal_D_p': 'Parent Internalizing',
    'sbt_anxiety_depression': 'Anxiety/Depression', 'sbt_guilt_hopelessness': 'Guilt/Hopelessness',
    'sbt_fatigue_sleep': 'Fatigue/Sleep', 'sbt_somatic_depression': 'Somatic',
    'sbt_emotional_dysregulation': 'Emotional Dysregulation',
    'sbt_social_withdrawal': 'Social Withdrawal', 'sbt_avoidance_fear': 'Avoidance/Fear',
    'sbt_aggression_irritability': 'Aggression/Irritability',
    'sbt_academic_cognitive': 'Academic/Cognitive',
    'sbt_perfectionism_achievement': 'Perfectionism/Achievement',
}

def _get_target_color(tgt):
    if '_TG_COLORS' in globals():
        return globals()['_TG_COLORS'].get(tgt, _TP_TRAJ_COLORS.get(tgt, '#1a73e8'))
    return _TP_TRAJ_COLORS.get(tgt, '#1a73e8')

def _get_target_label(tgt):
    if '_TG_LABELS' in globals():
        return globals()['_TG_LABELS'].get(tgt, _TP_TRAJ_LABELS.get(tgt, tgt))
    return _TP_TRAJ_LABELS.get(tgt, tgt)

def _run_tp_trajectories():
    """Run cross-timepoint performance trajectories for CURRENT globals."""

    _skip = False
    for _req in ['X_train', 'X_test', 'y_train', 'y_test']:
        if _req not in globals():
            print(f"  {_req} not found. Run the Main Analysis Runner first.")
            _skip = True
    if 'across_categories_data' not in globals():
        print("  across_categories_data not found -- skipping TP trajectories.")
        _skip = True
    if 'sample' not in globals():
        print('  sample not found -- ensure Cell 4 (Preprocessing) was executed.')
        _skip = True

    if _skip:
        return

    _RS.pop('tp_trajectory', None)

    _target = target_options if 'target_options' in globals() else 'target'
    _group = group if 'group' in globals() else None
    _timepoints = [0, 1, 2, 3, 4]
    _n_cv = VAL_N_OUTER if 'VAL_N_OUTER' in globals() else 5
    _tgt_color = _get_target_color(_target)
    _tgt_label = _get_target_label(_target)

    display(HTML(
        '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #1565C0;'
        'background:linear-gradient(135deg,#E3F2FD,#F5F5F5);padding:14px 20px;'
        'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
        '<span style="font-size:15px;font-weight:700;color:#1565C0;">'
        'Cross-Timepoint Performance Trajectories</span><br>'
        '<span style="font-size:12px;color:#546E7A;">'
        'Sweeping T0 \u2192 T4 for developmental prediction curves</span>'
        '</div>'
    ))

    print(f"\n  Target: {_tgt_label} ({_target})")
    print(f"  Group: {_group or 'All'}")
    print(f"  CV: {_n_cv}-fold")
    print()

    _tp_results = []

    for _t in _timepoints:
        try:
            _X_tr, _X_te, _y_tr, _y_te = across_categories_data(
                _t, [_target], group=_group,
                scale_x=True, scale_y=False, verbose=False
            )

            _X_all = pd.concat([_X_tr, _X_te], axis=0).reset_index(drop=True)
            _y_all = np.concatenate([
                np.asarray(_y_tr).ravel(),
                np.asarray(_y_te).ravel()
            ])

            _mask = ~np.isnan(_y_all.astype(float))
            _X_all = _X_all[_mask]
            _y_all = _y_all[_mask]

            if len(_y_all) < _n_cv * 2:
                print(f"  T{_t}: SKIPPED (N={len(_y_all)} < minimum)")
                continue

            _task = detect_task_type(_y_all)
            _is_clf = (_task == 'classification')
            _metric = 'AUC' if _is_clf else 'R\u00b2'

            if _is_clf:
                _cv = StratifiedKFold(n_splits=_n_cv, shuffle=True, random_state=42)
            else:
                _cv = KFold(n_splits=_n_cv, shuffle=True, random_state=42)

            _fold_scores = []
            for _train_idx, _test_idx in _cv.split(_X_all, _y_all):
                _X_fold_tr = _X_all.iloc[_train_idx].copy()
                _X_fold_te = _X_all.iloc[_test_idx].copy()

                _imp = SimpleImputer(strategy='median')
                _X_fold_tr = pd.DataFrame(
                    _imp.fit_transform(_X_fold_tr),
                    columns=_X_fold_tr.columns, index=_X_fold_tr.index
                )
                _X_fold_te = pd.DataFrame(
                    _imp.transform(_X_fold_te),
                    columns=_X_fold_te.columns, index=_X_fold_te.index
                )

                _m = create_lightweight_model(is_classification=_is_clf)
                _m.fit(_X_fold_tr, _y_all[_train_idx])
                if _is_clf:
                    _pred = _m.predict_proba(_X_fold_te)[:, 1]
                    _fold_scores.append(roc_auc_score(_y_all[_test_idx], _pred))
                else:
                    _pred = _m.predict(_X_fold_te)
                    _fold_scores.append(r2_score(_y_all[_test_idx], _pred))

            _mean = np.mean(_fold_scores)
            _std = np.std(_fold_scores, ddof=1)
            _se = _std / np.sqrt(_n_cv)
            _ci_half = stats.t.ppf(0.975, df=_n_cv-1) * _se

            _tp_results.append({
                'timepoint': _t,
                'n_subjects': len(_X_all),
                'n_features': _X_all.shape[1],
                'mean': _mean,
                'std': _std,
                'ci_lo': _mean - _ci_half,
                'ci_hi': _mean + _ci_half,
                'metric': _metric,
                'fold_scores': _fold_scores,
            })

            print(f"  T{_t}: {_metric} = {_mean:.4f} \u00b1 {_ci_half:.4f}  "
                  f"(N={len(_X_all):,}, {_X_all.shape[1]} feats)")

        except Exception as _e:
            print(f"  T{_t}: SKIPPED ({_e})")

    # -- Publication figure ---
    if len(_tp_results) >= 2:
        _df = pd.DataFrame(_tp_results)

        fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)

        _x = _df['timepoint'].values
        _y = _df['mean'].values
        _lo_ci = _df['ci_lo'].values
        _hi_ci = _df['ci_hi'].values

        ax.fill_between(_x, _lo_ci, _hi_ci, alpha=0.12, color=_tgt_color, zorder=2)
        ax.plot(_x, _y, '-', color=_tgt_color, linewidth=2.0, zorder=3)
        ax.plot(_x, _y, 'o', color='white', markersize=8, markeredgecolor=_tgt_color,
                markeredgewidth=1.8, zorder=4)

        for _i, _row in _df.iterrows():
            ax.annotate(f'{_row["mean"]:.3f}',
                       xy=(_row['timepoint'], _row['ci_hi']),
                       textcoords='offset points', xytext=(0, 10),
                       fontsize=9.5, ha='center', va='bottom', fontweight='bold',
                       color=_tgt_color)
            ax.annotate(f'N={_row["n_subjects"]:,}  ({_row["n_features"]} feats)',
                       xy=(_row['timepoint'], _row['ci_lo']),
                       textcoords='offset points', xytext=(0, -12),
                       fontsize=7, ha='center', va='top',
                       color='#555555')

        if len(_x) >= 3:
            _slope, _intercept, _rval, _pval, _stderr = stats.linregress(_x, _y)
            _trend_x = np.linspace(_x.min(), _x.max(), 50)
            _trend_y = _slope * _trend_x + _intercept
            ax.plot(_trend_x, _trend_y, '--', color='#AAAAAA', linewidth=1.0,
                    alpha=0.6, zorder=2)
            _slope_ci_lo = _slope - stats.t.ppf(0.975, df=len(_x)-2) * _stderr
            _slope_ci_hi = _slope + stats.t.ppf(0.975, df=len(_x)-2) * _stderr
            ax.text(0.98, 0.04,
                   f'Slope: {_slope:.4f} [{_slope_ci_lo:.4f}, {_slope_ci_hi:.4f}] per wave',
                   transform=ax.transAxes, fontsize=7.5, ha='right', va='bottom',
                   color='#555555',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                            alpha=0.85, edgecolor='#cccccc', linewidth=0.5))

        _tp_labels = {0: 'Baseline\n(9-10y)', 1: '1-Year\n(10-11y)',
                      2: '2-Year\n(11-12y)', 3: '3-Year\n(12-13y)',
                      4: '4-Year\n(13-14y)'}
        _available_tps = sorted(set(r['timepoint'] for r in _tp_results))
        ax.set_xticks(_available_tps)
        ax.set_xticklabels([_tp_labels.get(t, f'T{t}') for t in _available_tps],
                          fontsize=8.5)
        ax.set_xlabel('ABCD Study Wave', fontsize=9.5)
        ax.set_ylabel(f'Cross-Validated {_df.iloc[0]["metric"]}', fontsize=9.5)
        ax.set_title(f'Predictive Performance Across Development \u2014 {_tgt_label}',
                    fontsize=10.5, fontweight='bold')
        ax.tick_params(labelsize=8)
        ax.grid(axis='y', alpha=0.15, linewidth=0.5)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)

        plt.tight_layout()

        try:
            _fig_dir = os.path.join(os.getcwd(), 'tp_trajectory_figures')
            os.makedirs(_fig_dir, exist_ok=True)
            fig.savefig(os.path.join(_fig_dir, f'tp_trajectory_{_target}.png'),
                        dpi=300, bbox_inches='tight')
            fig.savefig(os.path.join(_fig_dir, f'tp_trajectory_{_target}.pdf'),
                        dpi=300, bbox_inches='tight')
        except Exception as _e:
            print(f"Warning: {_e}")
        plt.show()

        # -- Quantitative summary ---
        if len(_df) >= 2:
            _slope_sum = np.polyfit(_df['timepoint'], _df['mean'], 1)[0]
            _mean_r2 = _df['mean'].mean()
            _range_r2 = _df['mean'].max() - _df['mean'].min()
            _cv_coeff = (_df['mean'].std() / _mean_r2 * 100) if _mean_r2 > 0 else 0
            _best = _df.loc[_df['mean'].idxmax()]
            _worst = _df.loc[_df['mean'].idxmin()]
            _feat_range = f"{_df['n_features'].min()}\u2013{_df['n_features'].max()}"

            print(f"\n  Summary for {_tgt_label}:")
            print(f"    Mean R\u00b2 across waves:  {_mean_r2:.4f}")
            print(f"    Range:                 {_df['mean'].min():.4f}\u2013"
                  f"{_df['mean'].max():.4f} (\u0394={_range_r2:.4f})")
            print(f"    CV (stability):        {_cv_coeff:.1f}%")
            print(f"    Slope:                 {_slope_sum:+.4f} R\u00b2/wave")
            print(f"    Peak:                  T{int(_best['timepoint'])} ({_best['mean']:.4f})")
            print(f"    Nadir:                 T{int(_worst['timepoint'])} ({_worst['mean']:.4f})")
            print(f"    Feature range:         {_feat_range} across waves")

        _RS['tp_trajectory'] = {
            'results': _df.to_dict(),
            'target': _target,
            'label': _tgt_label,
            'color': _tgt_color,
        }
    else:
        print("\n  Fewer than 2 timepoints completed \u2014 cannot plot trajectory.")

def _plot_combined_trajectories(all_traj, fig_dir='tp_trajectory_figures'):
    """Combined multi-target overlay after batch run."""
    if not all_traj or len(all_traj) < 2:
        return

    _many = len(all_traj) > 6
    _lw = 1.6 if _many else 2.2
    _ms = 4 if _many else 7

    fig, ax = plt.subplots(figsize=(10.5, 6.5), dpi=300)

    for _tgt, _info in all_traj.items():
        _df = _info.get('df')
        if _df is None or len(_df) < 2:
            continue
        _col = _info.get('color', '#666666')
        _lab = _info.get('label', _tgt)

        ax.plot(_df['timepoint'], _df['mean'], 'o-', color=_col, label=_lab,
                linewidth=_lw, markersize=_ms,
                markeredgecolor='white', markeredgewidth=0.6, zorder=3)
        if not _many:
            ax.fill_between(_df['timepoint'],
                            _df['ci_lo'], _df['ci_hi'],
                            alpha=0.08, color=_col, zorder=1)

    _tp_labels = {0: 'Baseline\n(9-10y)', 1: '1-Year\n(10-11y)',
                  2: '2-Year\n(11-12y)', 3: '3-Year\n(12-13y)',
                  4: '4-Year\n(13-14y)'}
    _all_tps = set()
    for _info in all_traj.values():
        _df = _info.get('df')
        if _df is not None:
            _all_tps |= set(_df['timepoint'].astype(int).tolist())
    _all_tps = sorted(_all_tps)
    ax.set_xticks(_all_tps)
    ax.set_xticklabels([_tp_labels.get(t, f'T{t}') for t in _all_tps], fontsize=8.5)

    ax.set_xlabel('ABCD Study Wave', fontsize=10, labelpad=8)
    ax.set_ylabel('Cross-Validated R\u00b2', fontsize=10, labelpad=8)
    ax.set_title('Concurrent Predictive Performance Across Development',
                 fontsize=12, fontweight='bold', pad=12)

    _leg_ncol = 2 if _many else 1
    _leg_fs = 7 if _many else 8.5
    _leg_loc = 'upper center' if _many else 'lower left'
    _leg_bbox = (0.5, -0.15) if _many else None
    ax.legend(fontsize=_leg_fs, loc=_leg_loc, bbox_to_anchor=_leg_bbox,
              framealpha=0.92, ncol=_leg_ncol,
              edgecolor='#CCCCCC', borderpad=0.6, labelspacing=0.3)

    ax.tick_params(labelsize=8.5)
    ax.grid(axis='y', alpha=0.12, linewidth=0.5)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)
    plt.tight_layout()

    os.makedirs(fig_dir, exist_ok=True)
    fig.savefig(os.path.join(fig_dir, 'tp_trajectory_combined.png'),
                dpi=300, bbox_inches='tight')
    fig.savefig(os.path.join(fig_dir, 'tp_trajectory_combined.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n  Combined figure saved to {fig_dir}/")

# ---
# BATCH DISPATCH
# ---
_is_bv_tp = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_tp:
    try:
        _cbv_tp = cache_load('main_runner')
        if (_cbv_tp and isinstance(_cbv_tp, dict)
                and _cbv_tp.get('_batch_archive')
                and len(_cbv_tp['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_tp['_batch_archive'].values())):
            _batch_archive = _cbv_tp['_batch_archive']
            _is_bv_tp = True
    except Exception:
        pass  # non-critical; logged upstream or optional visualization

if _is_bv_tp:
    _vgs_tp = {k: globals().get(k) for k in [
        'X_train', 'X_test', 'y_train', 'y_test', 'target_options', 'tp_option',
        'metrics_dict_full', 'exported_within_dict', 'consensus_df',
        'model_weights', 'split', 'last_fitted_model',
        'sample_valid', 'sample_valid_test', 'sample_valid_train',
        'results_summary',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 TP Trajectories: processing {len(_batch_archive)} targets")
    print('=' * 72)

    _all_traj_results = {}

    _seen_targets_tp = set()
    for _vi_tp, (_vk_tp, _vd_tp) in enumerate(_batch_archive.items()):
        _vl_tp = _vd_tp.get('target', _vk_tp)
        if _vl_tp in _seen_targets_tp:
            print(f"\n  Skipping {_vk_tp} \u2014 trajectory already computed for {_vl_tp}")
            continue
        _seen_targets_tp.add(_vl_tp)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_tp+1}/{len(_batch_archive)}] Target: {_vl_tp}")
        print(f"{'-' * 68}")
        for _gk in ['X_train', 'X_test', 'y_train', 'y_test', 'metrics_dict_full',
                    'exported_within_dict', 'consensus_df', 'model_weights', 'split',
                    'last_fitted_model', 'results_summary',
                    'sample_valid', 'sample_valid_test', 'sample_valid_train']:
            if _vd_tp.get(_gk) is not None:
                globals()[_gk] = _vd_tp[_gk]
        globals()['target_options'] = _vl_tp
        if 'timepoint' in _vd_tp:
            globals()['tp_option'] = _vd_tp['timepoint']
        try:
            _run_tp_trajectories()
        except Exception as _e_tp:
            print(f"  Error for {_vl_tp}: {_e_tp}")
            import traceback; traceback.print_exc()

        # Collect for combined figure
        _rs_ref = globals().get('_RS')
        if _rs_ref and 'tp_trajectory' in _rs_ref:
            _traj_data = _rs_ref['tp_trajectory']
            _traj_df = pd.DataFrame(_traj_data.get('results', {}))
            if len(_traj_df) >= 2:
                _all_traj_results[_vl_tp] = {
                    'df': _traj_df,
                    'label': _traj_data.get('label', _vl_tp),
                    'color': _traj_data.get('color', '#666666'),
                }
            if _vk_tp in _batch_archive:
                _batch_archive[_vk_tp]['_RS_tp_trajectory'] = copy.deepcopy(_traj_data)

    # -- Combined multi-target figures ---
    if len(_all_traj_results) >= 2:
        _child_traj = {k: v for k, v in _all_traj_results.items()
                       if not k.startswith('parent_')}
        _parent_traj = {k: v for k, v in _all_traj_results.items()
                        if k.startswith('parent_')}

        if len(_child_traj) >= 2:
            print(f"\n{'='*68}")
            print(f"  COMBINED \u2014 Child Targets ({len(_child_traj)})")
            print(f"{'='*68}")
            _plot_combined_trajectories(_child_traj,
                                        fig_dir='tp_trajectory_figures/child')

        if len(_parent_traj) >= 2:
            print(f"\n{'='*68}")
            print(f"  COMBINED \u2014 Parent Targets ({len(_parent_traj)})")
            print(f"{'='*68}")
            _plot_combined_trajectories(_parent_traj,
                                        fig_dir='tp_trajectory_figures/parent')

        if len(_all_traj_results) >= 4:
            print(f"\n{'='*68}")
            print(f"  COMBINED \u2014 All Targets ({len(_all_traj_results)})")
            print(f"{'='*68}")
            _plot_combined_trajectories(_all_traj_results,
                                        fig_dir='tp_trajectory_figures')

    for _k, _v in _vgs_tp.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 TP Trajectories: {len(_seen_targets_tp)} unique targets")
    print('=' * 72)
else:
    _run_tp_trajectories()

In [ ]:
#@title V | Temporal Gradient Testing


# ---
# Cross-timepoint temporal transfer matrix. Trains on concurrent data at
# each timepoint and tests on concurrent data at every other timepoint.
# Produces a full transfer matrix and distance-dependent decay curves
# showing how predictive relationships degrade with temporal separation.
#
# Dependencies: Variable Registry, Circularity Exclusions, Preprocessing,
# Stacking Architecture, Main Analysis Runner.
# ---

#@markdown ### Configuration
run_temporal_gradient = True  #@param {type:"boolean"}

import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.impute import SimpleImputer
import os, warnings, copy
warnings.filterwarnings('ignore')

# -- Target colour palette (consistent across manuscript) ---
_TG_COLORS = {
    'depress_D_p':          '#1565C0', # Blue
    'sbt_core_depression':  '#4169E1', # Royal Blue (matches notebook)
    'anxdisord_D_p':        '#C62828', # Red
    'sbt_core_anxiety':     '#D32F2F', # Red (lighter)
    'adhd_D_p':             '#E6A817', # Gold (darkened for visibility)
    'social_problems_D_p':  '#E65100', # Orange
    'internal_D_p':         '#6A1B9A', # Purple
    'external_D_p':         '#2E7D32', # Green
    'parent_depress_D_p':   '#0D47A1', # Dark Blue
    'parent_depressed_p':   '#191970', # Midnight Blue (ASR)
    'parent_anxdisord_D_p': '#B71C1C', # Dark Red
    'parent_adhd_D_p':      '#F57F17', # Dark Gold
    'parent_external_D_p':  '#1B5E20', # Dark Green
    'parent_internal_D_p':  '#4A148C', # Dark Purple
    # -- Depression Subtypes (SBT, matches BATCH_TARGET_COLORS in the Batch Metrics cell) --
    'sbt_anxiety_depression':        '#DA70D6', # Orchid
    'sbt_guilt_hopelessness':        '#4B0082', # Indigo
    'sbt_fatigue_sleep':             '#6A5ACD', # Slate Blue
    'sbt_somatic_depression':        '#BC8F8F', # Rosy Brown
    'sbt_emotional_dysregulation':   '#FF4500', # Orange Red
    'sbt_social_withdrawal':         '#5F9EA0', # Cadet Blue
    'sbt_avoidance_fear':            '#9370DB', # Medium Purple
    'sbt_aggression_irritability':   '#B22222', # Firebrick
    'sbt_academic_cognitive':        '#B8960B', # Golden Amber
    'sbt_perfectionism_achievement': '#C44A80', # Rose Magenta
}

_TG_LABELS = {
    'depress_D_p':          'Depression',
    'sbt_core_depression':  'Depression (Primary)',
    'anxdisord_D_p':        'Anxiety',
    'sbt_core_anxiety':     'Anxiety (Primary)',
    'adhd_D_p':             'ADHD',
    'social_problems_D_p':  'Social Problems',
    'internal_D_p':         'Internalizing',
    'external_D_p':         'Externalizing',
    'parent_depress_D_p':   'Parent Depression',
    'parent_depressed_p':   'Parent Depression',
    'parent_anxdisord_D_p': 'Parent Anxiety',
    'parent_adhd_D_p':      'Parent ADHD',
    'parent_external_D_p':  'Parent Externalizing',
    'parent_internal_D_p':  'Parent Internalizing',
    # -- Depression Subtypes --
    'sbt_anxiety_depression':        'Anxiety/Depression',
    'sbt_guilt_hopelessness':        'Guilt/Hopelessness',
    'sbt_fatigue_sleep':             'Fatigue/Sleep',
    'sbt_somatic_depression':        'Somatic',
    'sbt_emotional_dysregulation':   'Emotional Dysregulation',
    'sbt_social_withdrawal':         'Social Withdrawal',
    'sbt_avoidance_fear':            'Avoidance/Fear',
    'sbt_aggression_irritability':   'Aggression/Irritability',
    'sbt_academic_cognitive':        'Academic/Cognitive',
    'sbt_perfectionism_achievement': 'Perfectionism/Achievement',
}

_DIST_LABELS = {
    0: '0\n(same TP)', 1: '1 wave\n(~1 yr)', 2: '2 waves\n(~2 yr)',
    3: '3 waves\n(~3 yr)', 4: '4 waves\n(~4 yr)',
}

# ---
# CORE: Build transfer matrix for a single target
# ---

def _build_transfer_matrix(target_name, timepoints=None, group_val=None):
    """
    Train on concurrent data at each timepoint, test on every other.
    Returns (matrix_df, details_dict) or (None, None) on failure.
    """
    if timepoints is None:
        timepoints = [0, 1, 2, 3, 4]

    _group = group_val

    # -- 1. Gather concurrent data at each available timepoint ---
    tp_data = {}
    for _t in timepoints:
        try:
            _X_tr, _X_te, _y_tr, _y_te = across_categories_data(
                _t, [target_name], group=_group,
                scale_x=True, scale_y=False, verbose=False
            )
            _X_all = pd.concat([_X_tr, _X_te], axis=0, ignore_index=True)
            if isinstance(_y_tr, (pd.Series, pd.DataFrame)):
                _y_all = pd.concat([_y_tr, _y_te], axis=0, ignore_index=True)
            else:
                _y_all = np.concatenate([np.asarray(_y_tr).ravel(),
                                         np.asarray(_y_te).ravel()])
            _y_all = np.asarray(_y_all).ravel().astype(float)

            _mask = ~np.isnan(_y_all)
            _X_all = _X_all[_mask].reset_index(drop=True)
            _y_all = _y_all[_mask]

            if len(_y_all) < 50:
                print(f"    T{_t}: SKIPPED (N={len(_y_all)} < 50)")
                continue

            tp_data[_t] = (_X_all, _y_all)
            print(f"    T{_t}: N={len(_X_all):,}, features={_X_all.shape[1]}")
        except Exception as _e:
            print(f"    T{_t}: SKIPPED ({_e})")

    if len(tp_data) < 2:
        print("    Fewer than 2 timepoints available -- cannot build matrix.")
        return None, None

    available_tps = sorted(tp_data.keys())

    # -- 2. Global feature intersection (same features for all pairs) --
    # Ensures diagonal (same-TP) entries don't benefit from having
    # more features than off-diagonal, isolating temporal decay.
    _common_cols = set(tp_data[available_tps[0]][0].columns)
    for _t in available_tps[1:]:
        _common_cols &= set(tp_data[_t][0].columns)
    _common_cols = sorted(_common_cols)
    print(f"    Common features across all timepoints: {len(_common_cols)}")

    if len(_common_cols) < 10:
        print("    Too few common features -- cannot build matrix.")
        return None, None

    # -- 3. Build NxN transfer matrix ---
    # DIAGONAL (train TP == test TP): 5-fold CV to avoid in-sample
    # inflation. Without CV, the model trains and tests on identical
    # rows, producing overfitted R² that exaggerates apparent decay.
    # OFF-DIAGONAL: train on ALL data at one TP, test on ALL data at
    # another TP. This is valid because feature values differ across
    # timepoints even for the same subjects.
    from sklearn.model_selection import KFold, StratifiedKFold

    _n_cv = 5
    matrix = pd.DataFrame(
        np.nan,
        index=[f'Train T{t}' for t in available_tps],
        columns=[f'Test T{t}' for t in available_tps]
    )
    details = {}

    _task = detect_task_type(tp_data[available_tps[0]][1])
    _is_clf = (_task == 'classification')

    for _t_train in available_tps:
        _X_train_full, _y_train_full = tp_data[_t_train]

        for _t_test in available_tps:
            _X_test_full, _y_test_full = tp_data[_t_test]

            if _t_train == _t_test:
                # -- DIAGONAL: K-fold CV (prevents in-sample inflation) --
                _X_cv = _X_train_full[_common_cols].copy()
                _y_cv = _y_train_full.copy()

                if _is_clf:
                    _cv = StratifiedKFold(n_splits=_n_cv, shuffle=True, random_state=42)
                else:
                    _cv = KFold(n_splits=_n_cv, shuffle=True, random_state=42)

                _fold_scores = []
                for _tr_idx, _te_idx in _cv.split(_X_cv, _y_cv):
                    _imp = SimpleImputer(strategy='median')
                    _Xf_tr = pd.DataFrame(_imp.fit_transform(_X_cv.iloc[_tr_idx]),
                                          columns=_common_cols)
                    _Xf_te = pd.DataFrame(_imp.transform(_X_cv.iloc[_te_idx]),
                                          columns=_common_cols)
                    _m = create_lightweight_model(is_classification=_is_clf)
                    _m.fit(_Xf_tr, _y_cv[_tr_idx])
                    if _is_clf:
                        _p = _m.predict_proba(_Xf_te)[:, 1]
                        _fold_scores.append(roc_auc_score(_y_cv[_te_idx], _p))
                    else:
                        _p = _m.predict(_Xf_te)
                        _fold_scores.append(r2_score(_y_cv[_te_idx], _p))

                _score = np.mean(_fold_scores)
                _score_sd = np.std(_fold_scores, ddof=1)
                _n_tr = int(len(_X_cv) * (_n_cv - 1) / _n_cv)
                _n_te = len(_X_cv) - _n_tr
                _extra = {'cv_folds': _fold_scores, 'cv_sd': _score_sd}

            else:
                # -- OFF-DIAGONAL: full train → full test (valid: different TPs) --
                _Xtr = _X_train_full[_common_cols].copy()
                _Xte = _X_test_full[_common_cols].copy()

                _imp = SimpleImputer(strategy='median')
                _Xtr = pd.DataFrame(_imp.fit_transform(_Xtr), columns=_common_cols)
                _Xte = pd.DataFrame(_imp.transform(_Xte), columns=_common_cols)

                _model = create_lightweight_model(is_classification=_is_clf)
                _model.fit(_Xtr, _y_train_full)

                if _is_clf:
                    _pred = _model.predict_proba(_Xte)[:, 1]
                    _score = roc_auc_score(_y_test_full, _pred)
                else:
                    _pred = _model.predict(_Xte)
                    _score = r2_score(_y_test_full, _pred)

                _n_tr = len(_Xtr)
                _n_te = len(_Xte)
                _extra = {}

            matrix.loc[f'Train T{_t_train}', f'Test T{_t_test}'] = _score
            details[(_t_train, _t_test)] = {
                'score': _score, 'n_features': len(_common_cols),
                'n_train': _n_tr, 'n_test': _n_te,
                'distance': abs(_t_test - _t_train),
                'is_diagonal': (_t_train == _t_test),
                **_extra,
            }

    return matrix, details

def _compute_decay_summary(details, available_tps):
    """Compute mean score at each temporal distance (forward only)."""
    max_dist = max(available_tps) - min(available_tps)
    rows = []
    for d in range(0, max_dist + 1):
        scores = []
        for (t_train, t_test), info in details.items():
            if (t_test - t_train) == d:  # forward only
                scores.append(info['score'])
        if scores:
            rows.append({
                'distance': d,
                'mean': np.mean(scores),
                'std': np.std(scores, ddof=1) if len(scores) > 1 else 0.0,
                'n': len(scores),
            })
    return pd.DataFrame(rows)

def _compute_asymmetry(details, available_tps):
    """Compare forward vs backward transfer at each temporal distance.
    Tests whether predictive relationships are symmetric across development."""
    max_dist = max(available_tps) - min(available_tps)
    rows = []
    for d in range(1, max_dist + 1):
        fwd = [info['score'] for (t_tr, t_te), info in details.items()
               if (t_te - t_tr) == d]
        bwd = [info['score'] for (t_tr, t_te), info in details.items()
               if (t_tr - t_te) == d]
        if fwd and bwd:
            rows.append({
                'distance': d,
                'forward_mean': np.mean(fwd),
                'backward_mean': np.mean(bwd),
                'delta': np.mean(fwd) - np.mean(bwd),
                'n_forward': len(fwd),
                'n_backward': len(bwd),
            })
    return pd.DataFrame(rows)

def _compute_decay_slopes(all_results):
    """Compute linear decay slope (distance 1→max) for each target.
    Returns DataFrame with target, slope, peak R², endpoint R², total_drop_pct."""
    from scipy import stats
    rows = []
    for _tgt, _res in all_results.items():
        _decay = _res.get('decay')
        if _decay is None or len(_decay) < 2:
            continue
        _fwd = _decay[_decay['distance'] >= 1].copy()
        if len(_fwd) < 2:
            continue
        _x = _fwd['distance'].values.astype(float)
        _y = _fwd['mean'].values
        _slope, _intercept, _rval, _, _stderr = stats.linregress(_x, _y)
        _peak = _fwd['mean'].values[0]
        _end = _fwd['mean'].values[-1]
        _pct = ((_peak - _end) / _peak * 100) if _peak > 0 else 0
        rows.append({
            'target': _tgt,
            'label': _TG_LABELS.get(_tgt, _tgt),
            'slope': _slope,
            'slope_se': _stderr,
            'peak_r2': _peak,
            'end_r2': _end,
            'total_drop_pct': _pct,
            'r_value': _rval,
            'color': _TG_COLORS.get(_tgt, '#666666'),
        })
    return pd.DataFrame(rows)

# -- Reference autocorrelations (T0→T4) from TargetsAllTps.csv ---
# Populate with verified values to enable slope-vs-stability scatter.
# Set to None or {} to skip that panel.
_AUTOCORR_T0T4 = {
    'depress_D_p':          0.411,
    'sbt_core_depression':  0.352,
    'anxdisord_D_p':        0.471,
    'adhd_D_p':             0.587,
    'social_problems_D_p':  0.500,
    'internal_D_p':         0.536,
    'external_D_p':         0.581,
    'parent_depress_D_p':   0.584,
    'parent_anxdisord_D_p': 0.552,
    'parent_adhd_D_p':      0.626,
    'parent_external_D_p':  0.635,
    'parent_internal_D_p':  0.650,
}

def _compute_autocorr_decay(target_name, available_tps, group_val=None):
    """Compute pairwise forward autocorrelation r² from the sample DataFrame.
    Uses the same subjects/timepoints the model sees.
    Returns decay DataFrame with columns: distance, mean (r²), std, n, mean_r."""
    from scipy.stats import pearsonr

    if 'sample' not in globals():
        return None

    _samp = globals()['sample']
    if 'subject' not in _samp.columns or 'time' not in _samp.columns:
        return None
    if target_name not in _samp.columns:
        return None

    # Apply group filter to match model subsetting
    _s = _samp.copy()
    if group_val == 'Female':
        _s = _s[_s['sex'] == 2]
    elif group_val == 'Male':
        _s = _s[_s['sex'] == 1]
    elif group_val == 'high_ale':
        _s = _s[_s.get('high_ale', pd.Series(False)) == True]
    elif group_val == 'low_ale':
        _s = _s[_s.get('low_ale_children_p', pd.Series(False)) == True]
    elif group_val is not None:
        print(f"    Warning: group '{group_val}' not supported in autocorr filter -- using full sample")

    # Extract (subject, target) at each timepoint
    tp_vals = {}
    for _t in available_tps:
        _sub = _s[_s['time'] == _t][['subject', target_name]].dropna()
        if len(_sub) > 50:
            tp_vals[_t] = _sub.set_index('subject')[target_name]

    if len(tp_vals) < 2:
        return None

    max_dist = max(available_tps) - min(available_tps)
    rows = []
    for d in range(1, max_dist + 1):
        r_vals = []
        for t1 in available_tps:
            t2 = t1 + d
            if t1 in tp_vals and t2 in tp_vals:
                _common = tp_vals[t1].index.intersection(tp_vals[t2].index)
                if len(_common) > 50:
                    _r, _ = pearsonr(tp_vals[t1].loc[_common], tp_vals[t2].loc[_common])
                    r_vals.append(_r)
        if r_vals:
            _r_arr = np.array(r_vals)
            _r2_arr = _r_arr ** 2
            rows.append({
                'distance': d,
                'mean': np.mean(_r2_arr),
                'std': np.std(_r2_arr, ddof=1) if len(_r2_arr) > 1 else 0.0,
                'n': len(r_vals),
                'mean_r': np.mean(_r_arr),
            })

    return pd.DataFrame(rows) if rows else None

# ---
# VISUALIZATION
# ---

def _plot_gradient_results(all_results, fig_dir='temporal_gradient_figures'):
    """
    Publication figures:
      Figure 1 -- Multi-target decay curves with autocorrelation overlay
      Figure 2 -- Per-target transfer heatmaps (shared colorbar)
      Figure 3 -- Directional asymmetry with N-pair annotations
      Figure 4 -- Decay slopes (A) + autocorrelation scatter with CI band (B)
    """
    import seaborn as sns
    from matplotlib.lines import Line2D

    os.makedirs(fig_dir, exist_ok=True)

    targets_with_data = {t: r for t, r in all_results.items()
                         if r.get('decay') is not None and len(r['decay']) > 1}

    if not targets_with_data:
        print("  No targets with sufficient data for visualization.")
        return

# ---
    # FIGURE 1: Multi-target decay curves
# ---
    fig, ax = plt.subplots(figsize=(10.5, 6.5), dpi=300)

    # Subtle vertical separator between CV reference and transfer decay
    ax.axvline(x=0.5, color='#E0E0E0', linewidth=0.8, linestyle='-', zorder=0)
    ax.axvspan(-0.3, 0.5, alpha=0.025, color='#BDBDBD', zorder=0)

    # Zone labels -- understated, professional
    ax.text(0.15, 0.97, 'CV reference', transform=ax.get_xaxis_transform(),
            fontsize=6.5, color='#333333', ha='center', va='top',
            fontfamily='sans-serif')
    ax.text(2.5, 0.97, 'Transfer decay', transform=ax.get_xaxis_transform(),
            fontsize=6.5, color='#333333', ha='center', va='top',
            fontfamily='sans-serif')

    # Pre-compute peak (distance 1) values
    _peak_vals = {}
    _stp_vals = {}
    for _tgt, _res in targets_with_data.items():
        _d = _res['decay']
        _row0 = _d[_d['distance'] == 0]
        _row1 = _d[_d['distance'] == 1]
        _stp_vals[_tgt] = _row0['mean'].values[0] if len(_row0) else np.nan
        _peak_vals[_tgt] = _row1['mean'].values[0] if len(_row1) else _stp_vals[_tgt]

    _sd_label_added = False
    _n_tgts = len(targets_with_data)
    _many = _n_tgts > 6
    # Adapt line weight and markers to target count
    _lw = 1.6 if _many else 2.6
    _ms = 4 if _many else 8
    _mew = 0.6 if _many else 1.0

    for _tgt, _res in targets_with_data.items():
        _decay = _res['decay']
        _col = _TG_COLORS.get(_tgt, '#666666')
        _lab_base = _TG_LABELS.get(_tgt, _tgt)
        _peak = _peak_vals.get(_tgt, np.nan)

        _d0 = _decay[_decay['distance'] == 0]
        _d1plus = _decay[_decay['distance'] >= 1]

        # Reference segment: skip when many targets to reduce clutter
        if not _many and len(_d0) and len(_d1plus):
            _ref_x = [_d0['distance'].values[0], _d1plus['distance'].values[0]]
            _ref_y = [_d0['mean'].values[0], _d1plus['mean'].values[0]]
            ax.plot(_ref_x, _ref_y, '--', color=_col, linewidth=1.4, alpha=0.5, zorder=2)
            ax.plot(_d0['distance'].values[0], _d0['mean'].values[0], 'o',
                    color='white', markersize=7, markeredgecolor=_col,
                    markeredgewidth=1.8, zorder=4)

        # Decay segment
        if len(_d1plus) >= 2:
            _lab = f'{_lab_base}' if _many else \
                   (f'{_lab_base} ({_peak:.3f})' if not np.isnan(_peak) else _lab_base)
            ax.plot(_d1plus['distance'], _d1plus['mean'], 'o-',
                    color=_col, label=_lab, linewidth=_lw, markersize=_ms,
                    markeredgecolor='white', markeredgewidth=_mew, zorder=3)
            # SD ribbon: skip when many targets (overlapping fills = mud)
            if not _many and _d1plus['std'].sum() > 0:
                _sd_lbl = '\u00b11 SD (across TP pairs)' if not _sd_label_added else '_nolegend_'
                ax.fill_between(_d1plus['distance'],
                                _d1plus['mean'] - _d1plus['std'],
                                _d1plus['mean'] + _d1plus['std'],
                                alpha=0.10, color=_col, zorder=1,
                                label=_sd_lbl)
                _sd_label_added = True

        # Annotate endpoint: skip when many targets to prevent stacking
        _n_tgts = len(targets_with_data)
        _max_row = _decay[_decay['distance'] == _decay['distance'].max()]
        if _n_tgts <= 6 and len(_max_row) and not np.isnan(_peak) and _peak > 0:
            _end_val = _max_row['mean'].values[0]
            _pct_drop = ((_peak - _end_val) / _peak) * 100
            if _n_tgts <= 4:
                _ann_txt = f'R\u00b2={_end_val:.3f} ({-_pct_drop:+.1f}%)'
            else:
                _ann_txt = f'{-_pct_drop:+.1f}%'
            ax.annotate(_ann_txt,
                        xy=(_max_row['distance'].values[0], _end_val),
                        textcoords='offset points', xytext=(12, 0),
                        fontsize=7, fontweight='bold', color=_col,
                        va='center', ha='left')

    # Autocorrelation overlay -- skip when many targets (too many dashed lines)
    _any_autocorr = False
    if not _many:
        for _tgt, _res in targets_with_data.items():
            _ac = _res.get('autocorr_decay')
            if _ac is not None and len(_ac) > 0:
                _col = _TG_COLORS.get(_tgt, '#666666')
                ax.plot(_ac['distance'], _ac['mean'], 's--',
                        color=_col, linewidth=1.2, markersize=4,
                        markeredgecolor=_col, markerfacecolor='white',
                        alpha=0.55, zorder=2)
                _any_autocorr = True

    # Build extra legend handles
    _extra_handles, _extra_labels = [], []
    if _any_autocorr:
        _extra_handles.append(Line2D([0], [0], color='#666666', linestyle='--', linewidth=1.2,
                              marker='s', markersize=4, markeredgecolor='#666666',
                              markerfacecolor='white', alpha=0.55))
        _extra_labels.append('Outcome autocorrelation (r\u00b2)')

    ax.set_xlabel('Temporal Distance (waves forward)', fontsize=11, labelpad=8)
    ax.set_ylabel('R\u00b2', fontsize=11, labelpad=8)
    ax.set_title('Temporal Transfer Gradient: Predictive Relationship Decay by Target',
                 fontsize=12, fontweight='bold', pad=12)

    # Dynamic x-ticks
    _all_dists = set()
    for _res in targets_with_data.values():
        _all_dists |= set(_res['decay']['distance'].astype(int).tolist())
    _all_dists = sorted(_all_dists)
    ax.set_xticks(_all_dists)
    ax.set_xticklabels([_DIST_LABELS.get(d, f'{d}') for d in _all_dists], fontsize=9)

    # Legend: sorted by peak performance descending
    handles, labels = ax.get_legend_handles_labels()
    # Separate model handles from SD/autocorr handles
    _model_idx = [i for i, l in enumerate(labels)
                  if not l.startswith('\u00b1') and l != 'Outcome autocorrelation (r\u00b2)']
    _other_idx = [i for i in range(len(labels)) if i not in _model_idx]
    _label_to_peak = {}
    for _tgt, _pv in _peak_vals.items():
        _lbase = _TG_LABELS.get(_tgt, _tgt)
        for _l in labels:
            if _l.startswith(_lbase):
                _label_to_peak[_l] = _pv
    _model_sorted = sorted(_model_idx, key=lambda i: _label_to_peak.get(labels[i], 0), reverse=True)
    _final_handles = [handles[i] for i in _model_sorted] + [handles[i] for i in _other_idx] + _extra_handles
    _final_labels = [labels[i] for i in _model_sorted] + [labels[i] for i in _other_idx] + _extra_labels
    _leg_ncol = 2 if _many else 1
    _leg_fs = 7 if _many else 8
    _leg_loc = 'upper center' if _many else 'lower left'
    _leg_bbox = (0.5, -0.18) if _many else None
    _leg_title = None if _many else 'Target (peak R\u00b2 at 1 wave)'
    ax.legend(_final_handles, _final_labels,
              fontsize=_leg_fs, loc=_leg_loc, bbox_to_anchor=_leg_bbox,
              framealpha=0.92, ncol=_leg_ncol,
              edgecolor='#CCCCCC', borderpad=0.6, labelspacing=0.3,
              columnspacing=1.0,
              title=_leg_title, title_fontsize=7.5)

    ax.tick_params(labelsize=8.5)
    ax.grid(axis='y', alpha=0.12, linewidth=0.5)
    ax.grid(axis='x', alpha=0.06, linewidth=0.3)
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

    plt.tight_layout()
    fig.savefig(f'{fig_dir}/temporal_gradient_decay_curves.png', dpi=300, bbox_inches='tight')
    fig.savefig(f'{fig_dir}/temporal_gradient_decay_curves.pdf', dpi=300, bbox_inches='tight')
    plt.show()

# ---
    # FIGURE 2: Per-target heatmaps (shared colorbar)
    # Shared vmin/vmax across all targets for cross-comparison
    # Forward/backward triangle labels
# ---
    _n_targets = len(targets_with_data)
    if _n_targets == 0:
        return

    # Compute shared color range across all targets
    _all_vals = []
    for _res in targets_with_data.values():
        _m = _res['matrix'].astype(float)
        _all_vals.extend(_m.values[~np.isnan(_m.values)].tolist())
    _shared_vmin = min(_all_vals) - 0.01
    _shared_vmax = max(_all_vals) + 0.01

    _ncols = min(3, _n_targets)
    _nrows = int(np.ceil(_n_targets / _ncols))
    fig2, axes2 = plt.subplots(_nrows, _ncols,
                                figsize=(5.0 * _ncols + 0.8, 4.2 * _nrows),
                                dpi=300)
    if _n_targets == 1:
        axes2 = np.array([axes2])
    axes2 = axes2.flatten() if hasattr(axes2, 'flatten') else [axes2]

    _heatmap_mappable = None
    for _idx, (_tgt, _res) in enumerate(targets_with_data.items()):
        _ax = axes2[_idx]
        _mat = _res['matrix'].astype(float)
        _col = _TG_COLORS.get(_tgt, '#1565C0')
        _lab = _TG_LABELS.get(_tgt, _tgt)
        _n = _mat.shape[0]

        _map_fn = getattr(_mat, 'map', None) or _mat.applymap
        _annot = _map_fn(lambda x: f'{x:.3f}' if not np.isnan(x) else '')

        _hm = sns.heatmap(_mat, annot=_annot, fmt='', cmap='RdYlGn', ax=_ax,
                    vmin=_shared_vmin, vmax=_shared_vmax,
                    linewidths=1.0, linecolor='white',
                    cbar=False,
                    annot_kws={'fontsize': 8.5, 'fontweight': 'bold'})
        if _heatmap_mappable is None:
            _heatmap_mappable = _hm.collections[0]

        # Highlight diagonal
        for _d in range(_n):
            _ax.add_patch(plt.Rectangle((_d, _d), 1, 1, fill=False,
                          edgecolor='#333333', linewidth=2.0, zorder=5))

        # Forward/backward labels
        _ax.text(_n - 0.3, 0.3, '\u2192 Fwd', fontsize=6.5, color='#444444',
                 ha='right', va='top', zorder=6)
        _ax.text(0.3, _n - 0.3, '\u2190 Bwd', fontsize=6.5, color='#444444',
                 ha='left', va='bottom', zorder=6)

        _ax.set_title(_lab, fontsize=10.5, fontweight='bold', color=_col, pad=8)
        _ax.tick_params(labelsize=7.5, rotation=0)
        _ax.set_yticklabels(_ax.get_yticklabels(), rotation=0)

    # Hide unused axes
    for _idx in range(len(targets_with_data), len(axes2)):
        axes2[_idx].set_visible(False)

    # Shared colorbar
    if _heatmap_mappable is not None:
        _cbar_ax = fig2.add_axes([0.94, 0.15, 0.015, 0.7])
        fig2.colorbar(_heatmap_mappable, cax=_cbar_ax, label='R\u00b2')
        _cbar_ax.tick_params(labelsize=7.5)

    plt.suptitle('Temporal Transfer Matrices by Target\n'
                 '(diagonal = same-timepoint CV; off-diagonal = cross-timepoint transfer)',
                 fontsize=11.5, fontweight='bold', y=1.02)
    plt.tight_layout(rect=[0, 0, 0.93, 0.96])
    fig2.savefig(f'{fig_dir}/temporal_gradient_heatmaps.png', dpi=300, bbox_inches='tight')
    fig2.savefig(f'{fig_dir}/temporal_gradient_heatmaps.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n  Figures saved to {fig_dir}/")

# ---
    # FIGURE 3: Forward vs Backward Asymmetry
    # N-pairs annotation at each distance
# ---
    _asym_data = {}
    for _tgt, _res in targets_with_data.items():
        _det_raw = _res.get('details')
        if _det_raw and isinstance(_det_raw, dict):
            import ast as _ast_mod
            _det = {}
            for _k, _v in _det_raw.items():
                try:
                    _kt = _ast_mod.literal_eval(_k) if isinstance(_k, str) else _k
                    _det[_kt] = _v
                except Exception:
                    continue
            if _det:
                _avail = sorted(set(t for (t, _) in _det.keys()) |
                                set(t for (_, t) in _det.keys()))
                _asym = _compute_asymmetry(_det, _avail)
                if len(_asym) > 0:
                    _asym_data[_tgt] = _asym

    if _asym_data:
        _many_asym = len(_asym_data) > 6
        fig3, ax3 = plt.subplots(figsize=(8.5, 5.5), dpi=300)

        ax3.axhline(y=0, color='#BDBDBD', linewidth=1.0, linestyle='-', zorder=0)
        ax3.axhspan(-0.015, 0.015, alpha=0.04, color='#BDBDBD', zorder=0)

        _asym_lw = 1.4 if _many_asym else 2.0
        _asym_ms = 4 if _many_asym else 7
        for _tgt, _asym in _asym_data.items():
            _col = _TG_COLORS.get(_tgt, '#666666')
            _lab = _TG_LABELS.get(_tgt, _tgt)
            ax3.plot(_asym['distance'], _asym['delta'], 's-',
                     color=_col, label=_lab, linewidth=_asym_lw, markersize=_asym_ms,
                     markeredgecolor='white', markeredgewidth=0.6, zorder=3)

        # N-pairs annotations (once per distance, above highest point)
        _all_d = set()
        for _a in _asym_data.values():
            _all_d |= set(_a['distance'].astype(int).tolist())
        _all_d = sorted(_all_d)
        for _d in _all_d:
            _n_at_d = None
            for _a in _asym_data.values():
                _row = _a[_a['distance'] == _d]
                if len(_row):
                    _n_at_d = int(_row['n_forward'].values[0])
                    break
            if _n_at_d is not None:
                _max_delta = max(_a[_a['distance'] == _d]['delta'].values[0]
                                 for _a in _asym_data.values()
                                 if len(_a[_a['distance'] == _d]))
                ax3.annotate(f'n={_n_at_d} pairs', xy=(_d, _max_delta),
                             textcoords='offset points', xytext=(0, 12),
                             fontsize=6.5, ha='center', color='#555555')

        ax3.set_xlabel('Temporal Distance (waves)', fontsize=11, labelpad=8)
        ax3.set_ylabel('Asymmetry (Forward R\u00b2 minus Backward R\u00b2)', fontsize=10, labelpad=8)
        ax3.set_title('Directional Asymmetry in Temporal Transfer',
                       fontsize=12, fontweight='bold', pad=12)

        ax3.set_xticks(_all_d)
        ax3.set_xticklabels([f'{d} wave{"s" if d > 1 else ""}\n(~{d} yr)' for d in _all_d],
                             fontsize=8.5)

        ax3.text(0.98, 0.95, 'Positive: earlier \u2192 later prediction stronger',
                 transform=ax3.transAxes, fontsize=7, ha='right', va='top',
                 color='#333333',
                 bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                           alpha=0.85, edgecolor='#CCCCCC', linewidth=0.5))
        ax3.text(0.98, 0.05, 'Negative: later \u2192 earlier prediction stronger',
                 transform=ax3.transAxes, fontsize=7, ha='right', va='bottom',
                 color='#333333',
                 bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                           alpha=0.85, edgecolor='#CCCCCC', linewidth=0.5))

        _leg3_ncol = 2 if _many_asym else 1
        _leg3_fs = 7 if _many_asym else 8.5
        _leg3_loc = 'upper center' if _many_asym else 'upper left'
        _leg3_bbox = (0.5, -0.18) if _many_asym else None
        ax3.legend(fontsize=_leg3_fs, loc=_leg3_loc, bbox_to_anchor=_leg3_bbox,
                    framealpha=0.92, ncol=_leg3_ncol,
                    edgecolor='#CCCCCC', borderpad=0.6, labelspacing=0.3)
        ax3.tick_params(labelsize=8.5)
        ax3.grid(axis='y', alpha=0.12, linewidth=0.5)
        for sp in ['top', 'right']:
            ax3.spines[sp].set_visible(False)
        plt.tight_layout()
        fig3.savefig(f'{fig_dir}/temporal_gradient_asymmetry.png', dpi=300, bbox_inches='tight')
        fig3.savefig(f'{fig_dir}/temporal_gradient_asymmetry.pdf', dpi=300, bbox_inches='tight')
        plt.show()

# ---
    # FIGURE 4: Decay Slopes + Autocorrelation Comparison
    # Regression confidence band on 4B
    # Panel labels A, B
    # CI footnote on 4A
    # Spearman rho alongside Pearson r on 4B
# ---
    _slopes_df = _compute_decay_slopes(targets_with_data)

    if len(_slopes_df) >= 2:
        _has_autocorr = bool(_AUTOCORR_T0T4) and sum(
            1 for t in _slopes_df['target'] if t in _AUTOCORR_T0T4) >= 3
        _n_panels = 2 if _has_autocorr else 1
        _widths = [1.2, 1] if _has_autocorr else [1]

        fig4, axes4 = plt.subplots(1, _n_panels, figsize=(5.5 * _n_panels, 5.5), dpi=300,
                                    gridspec_kw={'width_ratios': _widths})
        if _n_panels == 1:
            axes4 = [axes4]

        # -- Panel A: Decay slope bar chart ---
        _ax_a = axes4[0]
        # Panel label
        _ax_a.text(-0.14, 1.03, 'A', transform=_ax_a.transAxes,
                   fontsize=14, fontweight='bold', va='top')

        _sorted = _slopes_df.sort_values('slope', ascending=True)
        _n_bars = len(_sorted)
        _bar_h = min(0.6, 4.0 / max(_n_bars, 1))

        _y_pos = np.arange(_n_bars)
        _ax_a.barh(_y_pos, _sorted['slope'], height=_bar_h,
                   color=_sorted['color'].values, alpha=0.85,
                   edgecolor='white', linewidth=0.6, zorder=3)

        _ax_a.errorbar(_sorted['slope'], _y_pos,
                       xerr=_sorted['slope_se'] * 1.96, fmt='none',
                       ecolor='#333333', elinewidth=0.8, capsize=3, capthick=0.8, zorder=4)

        for _i, (_, _row) in enumerate(_sorted.iterrows()):
            # Place labels to the right of zero -- always in clear space
            _ax_a.text(0.002, _i,
                       f"{_row['slope']:.4f}  ({_row['total_drop_pct']:.1f}%)",
                       va='center', ha='left', fontsize=8, fontweight='bold',
                       color=_row['color'])

        _ax_a.axvline(x=0, color='#BDBDBD', linewidth=0.8, zorder=0)
        _ax_a.set_yticks(_y_pos)
        _ax_a.set_yticklabels(_sorted['label'], fontsize=9)
        # Pad right side for labels
        _xmin = min(_sorted['slope'] - _sorted['slope_se'] * 1.96) - 0.003
        _ax_a.set_xlim(left=_xmin, right=0.022)
        _ax_a.set_xlabel('Decay Slope (R\u00b2 per wave, from peak)', fontsize=10, labelpad=8)
        _ax_a.set_title('Rate of Temporal Transfer Decay',
                         fontsize=11, fontweight='bold', pad=10)
        _ax_a.tick_params(labelsize=8.5, axis='y', length=0)
        _ax_a.grid(axis='x', alpha=0.12, linewidth=0.5)
        for sp in ['top', 'right']:
            _ax_a.spines[sp].set_visible(False)

        # CI footnote
        _ax_a.text(0.02, -0.08, '95% CI from linear regression on 3\u20134 distance points',
                   transform=_ax_a.transAxes, fontsize=7, color='#555555')

        # -- Panel B: Slope vs Autocorrelation scatter ---
        if _has_autocorr:
            from scipy import stats
            _ax_b = axes4[1]
            # Panel label
            _ax_b.text(-0.14, 1.03, 'B', transform=_ax_b.transAxes,
                       fontsize=14, fontweight='bold', va='top')

            _scatter_rows = []
            for _, _row in _slopes_df.iterrows():
                if _row['target'] in _AUTOCORR_T0T4:
                    _scatter_rows.append({
                        'target': _row['target'],
                        'label': _row['label'],
                        'slope': _row['slope'],
                        'autocorr': _AUTOCORR_T0T4[_row['target']],
                        'color': _row['color'],
                    })
                else:
                    print(f"    Skipping {_row['label']} from scatter -- no value in _AUTOCORR_T0T4")
            _sdf = pd.DataFrame(_scatter_rows)

            if len(_sdf) >= 3:
                _ax_b.set_axisbelow(True)

                # Scatter points
                for _, _row in _sdf.iterrows():
                    _ax_b.scatter(_row['autocorr'], _row['slope'],
                                  color=_row['color'], s=85, zorder=4,
                                  edgecolors='white', linewidths=1.2)

                # Smart label placement: alternate above/below to reduce overlap
                for _li, (_, _row) in enumerate(_sdf.sort_values('autocorr').iterrows()):
                    _yo = 10 if _li % 2 == 0 else -14
                    _va = 'bottom' if _yo > 0 else 'top'
                    _ax_b.annotate(_row['label'],
                                   xy=(_row['autocorr'], _row['slope']),
                                   textcoords='offset points', xytext=(0, _yo),
                                   fontsize=7, color=_row['color'],
                                   fontweight='bold', ha='center', va=_va)

                # Regression line + confidence band
                _r_pears, _p_pears = stats.pearsonr(_sdf['autocorr'], _sdf['slope'])
                _sl, _int, _, _, _se = stats.linregress(_sdf['autocorr'], _sdf['slope'])
                _n_pts = len(_sdf)
                _x_fit = np.linspace(_sdf['autocorr'].min() - 0.03,
                                     _sdf['autocorr'].max() + 0.03, 100)
                _y_fit = _sl * _x_fit + _int
                _ax_b.plot(_x_fit, _y_fit, '-', color='#78909C',
                           linewidth=1.3, alpha=0.5, zorder=2)

                # CI band around regression
                if _n_pts >= 3:
                    _x_mean = _sdf['autocorr'].mean()
                    _ss_x = np.sum((_sdf['autocorr'] - _x_mean) ** 2)
                    _s_err = _se * np.sqrt(1.0 / _n_pts + (_x_fit - _x_mean) ** 2 / _ss_x)
                    _t_val = stats.t.ppf(0.975, df=_n_pts - 2)
                    _ax_b.fill_between(_x_fit, _y_fit - _t_val * _s_err,
                                       _y_fit + _t_val * _s_err,
                                       alpha=0.08, color='#78909C', zorder=1)

                # Spearman rho alongside Pearson r
                _stat_text = f'r = {_r_pears:.2f}'
                if _n_pts >= 5:
                    _stat_text += f' (p = {_p_pears:.3f})'
                if _n_pts >= 4:
                    _rho, _p_rho = stats.spearmanr(_sdf['autocorr'], _sdf['slope'])
                    _stat_text += f'\n\u03c1 = {_rho:.2f}'

                _ax_b.text(0.05, 0.05, _stat_text,
                           transform=_ax_b.transAxes, fontsize=8.5,
                           fontweight='bold', color='#455A64',
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                     alpha=0.9, edgecolor='#CFD8DC', linewidth=0.5))

                _ax_b.set_xlabel('T0 to T4 Autocorrelation (r)', fontsize=10, labelpad=8)
                _ax_b.set_ylabel('Decay Slope (R\u00b2 per wave)', fontsize=10, labelpad=8)
                _ax_b.set_title('Decay Rate vs Outcome Stability',
                                 fontsize=11, fontweight='bold', pad=10)
                # Axis padding
                _ax_b.set_xlim(_sdf['autocorr'].min() - 0.05,
                               _sdf['autocorr'].max() + 0.05)
                _ax_b.set_ylim(_sdf['slope'].min() - 0.005,
                               _sdf['slope'].max() + 0.005)
                _ax_b.grid(alpha=0.12, linewidth=0.5)
                for sp in ['top', 'right']:
                    _ax_b.spines[sp].set_visible(False)

        plt.tight_layout()
        fig4.savefig(f'{fig_dir}/temporal_gradient_slopes.png', dpi=300, bbox_inches='tight')
        fig4.savefig(f'{fig_dir}/temporal_gradient_slopes.pdf', dpi=300, bbox_inches='tight')
        plt.show()

        # Print slope summary
        print(f"\n  Decay Slope Summary (R\u00b2 per wave, distance 1 to max):")
        for _, _row in _slopes_df.sort_values('slope').iterrows():
            print(f"    {_row['label']:30s}  slope = {_row['slope']:+.4f}  "
                  f"(peak {_row['peak_r2']:.3f} -> {_row['end_r2']:.3f}, "
                  f"{_row['total_drop_pct']:.1f}% drop)")

    print(f"\n  All figures saved to {fig_dir}/")

# ---
# MAIN BODY FUNCTION
# ---

def _run_temporal_gradient():
    """Run temporal gradient testing for CURRENT globals."""

    if not run_temporal_gradient:
        print("  Temporal Gradient Testing disabled.")
        return

    # -- Gate check ---
    _skip = False
    for _req in ['across_categories_data', 'sample', 'create_lightweight_model',
                  'detect_task_type']:
        if _req not in globals():
            print(f"  {_req} not found. Run Cells 1-5 first.")
            _skip = True
    if _skip:
        return

    _RS.pop('temporal_gradient', None)

    _target = target_options if 'target_options' in globals() else 'target'
    _group = group if 'group' in globals() else None
    _timepoints = [0, 1, 2, 3, 4]

    display(HTML(
        '<div style="font-family:sans-serif;max-width:750px;border-left:4px solid #00695C;'
        'background:linear-gradient(135deg,#E0F2F1,#F5F5F5);padding:14px 20px;'
        'margin:16px 0 12px 0;border-radius:0 6px 6px 0;">'
        '<span style="font-size:15px;font-weight:700;color:#00695C;">'
        'Temporal Gradient Testing</span><br>'
        '<span style="font-size:12px;color:#546E7A;">'
        'Cross-timepoint transfer matrix: train at each wave, test at every other wave</span>'
        '</div>'
    ))

    print(f"\n  Target: {_target}")
    print(f"  Timepoints: {_timepoints}")
    print()

    matrix, details = _build_transfer_matrix(_target, _timepoints, _group)

    if matrix is None:
        print("  Could not build transfer matrix.")
        return

    # Print matrix
    print(f"\n  Transfer Matrix ({_target}):")
    print(f"  (diagonal entries are {5}-fold CV; off-diagonal are full train→test)")
    print(matrix.to_string(float_format='{:.3f}'.format))

    # Print per-cell details
    print(f"\n  Cell Details:")
    for (_tt, _te), info in sorted(details.items()):
        _cv_tag = ' [CV]' if info.get('is_diagonal') else ''
        _sd_tag = f" (SD={info['cv_sd']:.3f})" if 'cv_sd' in info else ''
        print(f"    Train T{_tt} -> Test T{_te}: R²={info['score']:.4f}{_sd_tag}  "
              f"N_train={info['n_train']:,}  N_test={info['n_test']:,}  "
              f"feats={info['n_features']}{_cv_tag}")

    # Compute decay
    available_tps = sorted(set(t for (t, _) in details.keys()) |
                           set(t for (_, t) in details.keys()))
    decay = _compute_decay_summary(details, available_tps)

    if len(decay) > 0:
        print(f"\n  Forward-Only Decay Summary:")
        for _, row in decay.iterrows():
            _d = int(row['distance'])
            print(f"    Distance {_d} wave(s): R² = {row['mean']:.4f} "
                  f"(+/- {row['std']:.4f}, n={int(row['n'])})")

    # Store
    _RS['temporal_gradient'] = {
        'target': _target,
        'matrix': matrix.to_dict(),
        'details': {str(k): v for k, v in details.items()},
        'decay': decay.to_dict() if decay is not None else None,
    }

    # Compute autocorrelation decay from sample
    _autocorr_decay = _compute_autocorr_decay(_target, available_tps, _group)
    if _autocorr_decay is not None:
        _RS['temporal_gradient']['autocorr_decay'] = _autocorr_decay.to_dict()
        print(f"\n  Autocorrelation Decay (r²):")
        for _, row in _autocorr_decay.iterrows():
            _d = int(row['distance'])
            print(f"    Distance {_d} wave(s): r² = {row['mean']:.4f}  "
                  f"(r = {row['mean_r']:.3f}, n={int(row['n'])} pairs)")

    # Single-target figure
    if len(decay) > 1:
        _col = _TG_COLORS.get(_target, '#1565C0')
        _lab = _TG_LABELS.get(_target, _target)

        fig, ax = plt.subplots(figsize=(8, 5), dpi=300)

        # Same-TP reference line
        _stp_val = decay[decay['distance'] == 0]['mean'].values
        if len(_stp_val):
            ax.axhline(y=_stp_val[0], color=_col, linewidth=0.8,
                        linestyle=':', alpha=0.4, zorder=1)

        ax.plot(decay['distance'], decay['mean'], 'o-', color=_col,
                linewidth=2.4, markersize=9, markeredgecolor='white',
                markeredgewidth=1.3, zorder=3, label=_lab)
        if decay['std'].sum() > 0:
            ax.fill_between(decay['distance'],
                            decay['mean'] - decay['std'],
                            decay['mean'] + decay['std'],
                            alpha=0.12, color=_col, zorder=1)

        # Annotate scores
        for _, row in decay.iterrows():
            ax.annotate(f"{row['mean']:.3f}",
                        xy=(row['distance'], row['mean']),
                        textcoords='offset points', xytext=(0, 13),
                        fontsize=9.5, ha='center', fontweight='bold', color=_col)

        # Dynamic x-ticks
        _dists = sorted(decay['distance'].astype(int).tolist())
        ax.set_xticks(_dists)
        ax.set_xticklabels([_DIST_LABELS.get(d, f'{d}') for d in _dists], fontsize=9)
        ax.set_xlabel('Temporal Distance (waves forward)', fontsize=10, labelpad=8)
        ax.set_ylabel('R²', fontsize=10, labelpad=8)
        ax.set_title(f'Temporal Transfer Gradient: {_lab}',
                     fontsize=11, fontweight='bold', pad=10)
        ax.legend(fontsize=9, loc='lower left', framealpha=0.9)
        ax.grid(axis='y', alpha=0.12, linewidth=0.5)
        for sp in ['top', 'right']:
            ax.spines[sp].set_visible(False)
        plt.tight_layout()

        _fig_dir = 'temporal_gradient_figures'
        os.makedirs(_fig_dir, exist_ok=True)
        fig.savefig(f'{_fig_dir}/temporal_gradient_{_target}.png',
                    dpi=300, bbox_inches='tight')
        fig.savefig(f'{_fig_dir}/temporal_gradient_{_target}.pdf',
                    dpi=300, bbox_inches='tight')
        plt.show()

# ---
# BATCH DISPATCH
# ---
_is_bv_tg = (
    '_batch_archive' in dir()
    and isinstance(_batch_archive, dict)
    and len(_batch_archive) > 1
    and any(v.get('X_train') is not None for v in _batch_archive.values())
)
if not _is_bv_tg:
    try:
        _cbv_tg = cache_load('main_runner')
        if (_cbv_tg and isinstance(_cbv_tg, dict)
                and _cbv_tg.get('_batch_archive')
                and len(_cbv_tg['_batch_archive']) > 1
                and any(v.get('X_train') is not None
                        for v in _cbv_tg['_batch_archive'].values())):
            _batch_archive = _cbv_tg['_batch_archive']
            _is_bv_tg = True
    except Exception:
        pass

if _is_bv_tg:
    _vgs_tg = {k: globals().get(k) for k in [
        'X_train', 'X_test', 'y_train', 'y_test', 'target_options', 'tp_option',
        'metrics_dict_full', 'exported_within_dict', 'consensus_df',
        'model_weights', 'split', 'last_fitted_model',
        'sample_valid', 'sample_valid_test', 'sample_valid_train',
        'results_summary',
    ]}
    print("\n" + '=' * 72)
    print(f"  BATCH \u2014 Temporal Gradient: processing {len(_batch_archive)} targets")
    print('=' * 72)

    # Collect all results for combined figure
    _all_gradient_results = {}

    # Deduplicate: run once per unique target
    _seen_targets_tg = set()
    for _vi_tg, (_vk_tg, _vd_tg) in enumerate(_batch_archive.items()):
        _vl_tg = _vd_tg.get('target', _vk_tg)
        if _vl_tg in _seen_targets_tg:
            print(f"\n  Skipping {_vk_tg} (gradient already computed for {_vl_tg})")
            continue
        _seen_targets_tg.add(_vl_tg)
        print(f"\n{'-' * 68}")
        print(f"  [{_vi_tg+1}/{len(_batch_archive)}] Target: {_vl_tg}")
        print(f"{'-' * 68}")
        for _gk in ['X_train', 'X_test', 'y_train', 'y_test', 'metrics_dict_full',
                    'exported_within_dict', 'consensus_df', 'model_weights', 'split',
                    'last_fitted_model', 'results_summary',
                    'sample_valid', 'sample_valid_test', 'sample_valid_train']:
            if _vd_tg.get(_gk) is not None:
                globals()[_gk] = _vd_tg[_gk]
        globals()['target_options'] = _vl_tg
        if 'timepoint' in _vd_tg:
            globals()['tp_option'] = _vd_tg['timepoint']
        try:
            _run_temporal_gradient()
        except Exception as _e_tg:
            print(f"  Error for {_vl_tg}: {_e_tg}")
            import traceback; traceback.print_exc()

        # Collect for combined plot
        _rs_ref = globals().get('_RS')
        if _rs_ref and 'temporal_gradient' in _rs_ref:
            _tg_data = _rs_ref['temporal_gradient']
            _decay_dict = _tg_data.get('decay')
            _mat_dict = _tg_data.get('matrix')
            _all_gradient_results[_vl_tg] = {
                'decay': pd.DataFrame(_decay_dict) if _decay_dict else None,
                'matrix': pd.DataFrame(_mat_dict) if _mat_dict else None,
                'details': _tg_data.get('details'),
                'autocorr_decay': pd.DataFrame(_tg_data['autocorr_decay']) if _tg_data.get('autocorr_decay') else None,
            }
            # Archive back
            if _vk_tg in _batch_archive:
                _batch_archive[_vk_tg]['_RS_temporal_gradient'] = copy.deepcopy(_tg_data)

    # -- Combined multi-target figures (split child / parent) ---
    if _all_gradient_results:
        _n_ok = len(_all_gradient_results)
        _n_total = len(_seen_targets_tg)
        if _n_ok < _n_total:
            print(f"\n  Warning: Combined figure includes {_n_ok}/{_n_total} targets ({_n_total - _n_ok} failed)")

        # Split into child and parent groups
        _child_results = {k: v for k, v in _all_gradient_results.items()
                          if not k.startswith('parent_')}
        _parent_results = {k: v for k, v in _all_gradient_results.items()
                           if k.startswith('parent_')}

        if _child_results:
            print(f"\n{'='*68}")
            print(f"  CHILD TARGETS -- Temporal Gradient ({len(_child_results)} targets)")
            print(f"{'='*68}")
            _plot_gradient_results(_child_results,
                                   fig_dir='temporal_gradient_figures/child')

        if _parent_results:
            print(f"\n{'='*68}")
            print(f"  PARENT TARGETS -- Temporal Gradient ({len(_parent_results)} targets)")
            print(f"{'='*68}")
            _plot_gradient_results(_parent_results,
                                   fig_dir='temporal_gradient_figures/parent')

        # Combined slopes figure (all targets) if enough for scatter
        if len(_all_gradient_results) >= 4:
            print(f"\n{'='*68}")
            print(f"  ALL TARGETS -- Combined Slope Comparison ({len(_all_gradient_results)} targets)")
            print(f"{'='*68}")
            _plot_gradient_results(_all_gradient_results,
                                   fig_dir='temporal_gradient_figures/combined')

    # Restore globals
    for _k, _v in _vgs_tg.items():
        globals()[_k] = _v
    print("\n" + '=' * 72)
    print(f"  BATCH COMPLETE \u2014 Temporal Gradient: {len(_seen_targets_tg)} unique targets")
    print('=' * 72)
else:
    _run_temporal_gradient()